In [ ]:
# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")

In [ ]:
# ============================================================
# CELL 2 — CREATE DATASET AUDIT WORKSPACE
# ============================================================

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "network-attack-forecasting"

AUDIT_ROOT = PROJECT_ROOT / "Dataset_Audit_v2"

CIC_RAW = AUDIT_ROOT / "raw" / "CIC_IDS2017"
UNSW_RAW = AUDIT_ROOT / "raw" / "UNSW_NB15"

CIC_AUDIT = AUDIT_ROOT / "audit_outputs" / "CIC"
UNSW_AUDIT = AUDIT_ROOT / "audit_outputs" / "UNSW"

METADATA = AUDIT_ROOT / "metadata"
LOGS = AUDIT_ROOT / "logs"

folders = [
    CIC_RAW,
    UNSW_RAW,
    CIC_AUDIT,
    UNSW_AUDIT,
    METADATA,
    LOGS,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("✓ Dataset audit workspace created\n")

for folder in folders:
    print(folder)

In [ ]:
# ============================================================
# CELL 3 — VERIFY PROJECT + NEW WORKSPACE
# ============================================================

print("=" * 80)
print("PROJECT CHECK")
print("=" * 80)

print("\nProject root exists:")
print(PROJECT_ROOT.exists())

print("\nExisting project contents:")
if PROJECT_ROOT.exists():
    for item in sorted(PROJECT_ROOT.iterdir()):
        print("  ", item.name)

print("\n" + "=" * 80)
print("NEW AUDIT WORKSPACE")
print("=" * 80)

for path in folders:
    status = "✓" if path.exists() else "✗"
    print(f"{status} {path.relative_to(PROJECT_ROOT)}")

In [ ]:
# ============================================================
# CELL 4 — CHECK WHETHER RAW DATASETS ARE ALREADY PRESENT
# ============================================================

from pathlib import Path
import os

def inspect_folder(path):
    path = Path(path)

    print("\n" + "=" * 90)
    print(path)
    print("=" * 90)

    if not path.exists():
        print("Folder does not exist.")
        return

    files = [p for p in path.rglob("*") if p.is_file()]

    if not files:
        print("EMPTY")
        return

    total_bytes = sum(p.stat().st_size for p in files)

    print(f"Files      : {len(files):,}")
    print(f"Total size : {total_bytes / (1024**3):.3f} GB\n")

    for p in sorted(files):
        size_mb = p.stat().st_size / (1024**2)
        print(f"{size_mb:10.2f} MB   {p.relative_to(path)}")


inspect_folder(CIC_RAW)
inspect_folder(UNSW_RAW)

In [ ]:
# ============================================================
# CELL 5 — DATASET ACQUISITION UTILITIES
# ============================================================

!pip -q install gdown wget

import os
import shutil
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path

print("✓ Acquisition utilities ready")

In [ ]:
# ============================================================
# CELL 6 — DATASET PROVENANCE MANIFEST
# ============================================================

from datetime import datetime
import pandas as pd

manifest = pd.DataFrame([
    {
        "Dataset": "CIC-IDS2017",
        "Organization": "Canadian Institute for Cybersecurity, University of New Brunswick",
        "Purpose": "Raw/original dataset audit for forecasting benchmark",
        "Downloaded": False,
        "Download_Date": None,
        "Notes": "Preserve original day/file organization"
    },
    {
        "Dataset": "UNSW-NB15",
        "Organization": "UNSW Canberra",
        "Purpose": "Raw/original dataset audit for forecasting benchmark",
        "Downloaded": False,
        "Download_Date": None,
        "Notes": "Use full CSV/event distribution; do not use premade train/test as canonical split"
    }
])

manifest_path = METADATA / "dataset_provenance.csv"
manifest.to_csv(manifest_path, index=False)

display(manifest)

print("\nSaved:")
print(manifest_path)

In [ ]:
# ============================================================
# CELL 8 — FIND CIC + UNSW DATASETS ALREADY IN GOOGLE DRIVE
# ============================================================

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

keywords = [
    "cic",
    "ids2017",
    "cicids",
    "unsw",
    "nb15"
]

print("=" * 100)
print("SEARCHING GOOGLE DRIVE FOR CIC / UNSW DATASETS")
print("=" * 100)

matches = []

for path in DRIVE.rglob("*"):
    try:
        name = path.name.lower()

        if any(keyword in name for keyword in keywords):
            matches.append(path)

    except Exception:
        pass

print(f"\nFound {len(matches)} matching paths.\n")

for p in matches:
    try:
        if p.is_file():
            size_mb = p.stat().st_size / (1024**2)
            print(f"FILE    {size_mb:10.2f} MB   {p}")
        else:
            print(f"FOLDER                 {p}")
    except Exception:
        print(f"UNKNOWN                {p}")

In [ ]:
# ============================================================
# CELL 8.1 — VERIFY CANONICAL RAW DATASETS
# ============================================================

from pathlib import Path
from collections import Counter

AUDIT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

CIC_RAW = AUDIT_ROOT / "raw" / "CIC_IDS2017"
UNSW_RAW = AUDIT_ROOT / "raw" / "UNSW_NB15"


def inventory(root, dataset_name):

    print("\n" + "=" * 100)
    print(f"{dataset_name} — RAW DATA INVENTORY")
    print("=" * 100)

    files = [p for p in root.rglob("*") if p.is_file()]

    if not files:
        print("❌ NO FILES FOUND")
        return

    total_size = sum(p.stat().st_size for p in files)

    print(f"\nLocation    : {root}")
    print(f"Files       : {len(files):,}")
    print(f"Total size  : {total_size / (1024**3):.3f} GB")

    # File types
    extensions = Counter(
        "".join(p.suffixes).lower() if p.suffixes else "<none>"
        for p in files
    )

    print("\nFILE TYPES")
    print("-" * 50)

    for ext, count in extensions.most_common():
        print(f"{ext:20} {count:,}")

    print("\nFILES")
    print("-" * 100)

    for p in sorted(files):

        size_mb = p.stat().st_size / (1024**2)

        print(
            f"{size_mb:10.2f} MB   "
            f"{p.relative_to(root)}"
        )


inventory(CIC_RAW, "CIC-IDS2017")
inventory(UNSW_RAW, "UNSW-NB15")

In [ ]:
# ============================================================
# PHASE 9.1 — CIC RAW SCHEMA AUDIT
# ============================================================

import pandas as pd
from pathlib import Path

CIC_FILES = sorted(CIC_RAW.glob("*.csv"))

print("=" * 110)
print("PHASE 9.1 — CIC RAW SCHEMA AUDIT")
print("=" * 110)

print(f"\nCIC files found: {len(CIC_FILES)}")

schemas = {}

for file in CIC_FILES:

    print("\n" + "=" * 110)
    print(file.name)
    print("=" * 110)

    try:
        sample = pd.read_csv(
            file,
            nrows=5,
            low_memory=False
        )

        schemas[file.name] = list(sample.columns)

        print(f"Columns: {len(sample.columns)}")

        print("\nCOLUMN NAMES:")
        for i, col in enumerate(sample.columns, 1):
            print(f"{i:3d}. {repr(col)}")

        print("\nFIRST 5 RAW ROWS:")
        display(sample)

    except Exception as e:
        print("❌ ERROR:", repr(e))

In [ ]:
# ============================================================
# PHASE 9.2 — CIC CROSS-FILE SCHEMA CONSISTENCY
# ============================================================

print("=" * 110)
print("PHASE 9.2 — CIC CROSS-FILE SCHEMA CONSISTENCY")
print("=" * 110)

reference_file = CIC_FILES[0].name
reference_schema = schemas[reference_file]

print(f"\nReference file:")
print(reference_file)

print(f"\nReference column count: {len(reference_schema)}")

all_same = True

for file_name, schema in schemas.items():

    same = schema == reference_schema

    print(
        f"{'✓' if same else '❌'} "
        f"{file_name:<65} "
        f"{len(schema):>3} columns"
    )

    if not same:
        all_same = False

        missing = set(reference_schema) - set(schema)
        extra = set(schema) - set(reference_schema)

        if missing:
            print("    Missing:", missing)

        if extra:
            print("    Extra:", extra)

print("\n" + "-" * 110)

if all_same:
    print("✓ ALL CIC FILES HAVE IDENTICAL COLUMN STRUCTURE")
else:
    print("⚠ CIC FILES DO NOT HAVE IDENTICAL SCHEMAS")

In [ ]:
# ============================================================
# PHASE 9.3 — IDENTIFY IMPORTANT CIC FIELDS
# ============================================================

columns = reference_schema

keywords = {
    "TIME": [
        "time", "timestamp", "date"
    ],

    "LABEL": [
        "label", "attack", "class"
    ],

    "SOURCE": [
        "source", "src"
    ],

    "DESTINATION": [
        "destination", "dest", "dst"
    ],

    "PORT": [
        "port"
    ],

    "PROTOCOL": [
        "protocol"
    ],

    "FLOW_ID": [
        "flow id", "flow_id", "flowid"
    ]
}

print("=" * 110)
print("PHASE 9.3 — IMPORTANT FIELD DISCOVERY")
print("=" * 110)

for category, terms in keywords.items():

    matches = []

    for col in columns:
        normalized = str(col).strip().lower()

        if any(term in normalized for term in terms):
            matches.append(col)

    print(f"\n{category}")
    print("-" * 50)

    if matches:
        for col in matches:
            print(repr(col))
    else:
        print("NONE FOUND")

In [ ]:
# ============================================================
# PHASE 9.4 — CIC ROW COUNTS + ENCODING AUDIT
# ============================================================

import pandas as pd

print("=" * 110)
print("PHASE 9.4 — CIC RAW ROW COUNTS + ENCODING AUDIT")
print("=" * 110)

row_counts = {}
file_encodings = {}

# Try strict decoding first.
# latin1 is last because it can decode every byte value.
ENCODINGS = ["utf-8", "cp1252", "latin1"]

for file in CIC_FILES:

    success = False

    for encoding in ENCODINGS:

        try:
            count = 0

            for chunk in pd.read_csv(
                file,
                chunksize=200_000,
                usecols=[0],
                encoding=encoding,
                low_memory=False
            ):
                count += len(chunk)

            row_counts[file.name] = count
            file_encodings[file.name] = encoding

            print(
                f"✓ {file.name:<65}"
                f"{count:>12,} rows   "
                f"encoding={encoding}"
            )

            success = True
            break

        except UnicodeDecodeError:
            continue

        except Exception as e:
            print(
                f" {file.name}\n"
                f"   Unexpected error: {repr(e)}"
            )
            break

    if not success and file.name not in row_counts:
        print(
            f" Could not decode {file.name} "
            f"with tested encodings."
        )


print("\n" + "-" * 110)

print(
    f"{'TOTAL':<67}"
    f"{sum(row_counts.values()):>12,} rows"
)

print("\n" + "=" * 110)
print("ENCODING SUMMARY")
print("=" * 110)

for file_name, encoding in file_encodings.items():
    print(f"{file_name:<70} {encoding}")

In [ ]:
# ============================================================
# PHASE 9.4B — STRICT FULL-FILE ENCODING VALIDATION
# ============================================================

print("=" * 110)
print("PHASE 9.4B — STRICT FULL-FILE ENCODING VALIDATION")
print("=" * 110)

ENCODINGS = ["utf-8", "cp1252", "latin1"]

file_encodings = {}

for file in CIC_FILES:

    print(f"\n{file.name}")

    detected = None

    for encoding in ENCODINGS:

        try:
            # Read raw file as text in chunks.
            # This validates EVERY byte in the file.
            with open(file, "r", encoding=encoding, errors="strict") as f:

                while True:
                    chunk = f.read(1024 * 1024)  # 1 MB

                    if not chunk:
                        break

            detected = encoding
            file_encodings[file.name] = encoding

            print(f"  ✓ Full-file decode successful: {encoding}")
            break

        except UnicodeDecodeError as e:

            print(
                f"  ✗ {encoding:<8} failed "
                f"(byte position around {e.start})"
            )

    if detected is None:
        print("  ❌ No tested encoding could decode file")


print("\n" + "=" * 110)
print("FINAL ENCODING MAP")
print("=" * 110)

for file_name, encoding in file_encodings.items():
    print(f"{file_name:<70} {encoding}")

In [ ]:
# ============================================================
# PHASE 9.5B — CIC TIMESTAMP + LABEL INTEGRITY AUDIT
# ============================================================

import pandas as pd

print("=" * 110)
print("PHASE 9.5B — CIC TIMESTAMP + LABEL INTEGRITY AUDIT")
print("=" * 110)

timestamp_audit = []

for file in CIC_FILES:

    print("\n" + "=" * 110)
    print(file.name)
    print("=" * 110)

    encoding = file_encodings[file.name]

    try:

        df = pd.read_csv(
            file,
            usecols=[" Timestamp", " Label"],
            encoding=encoding,
            low_memory=False
        )

        # -----------------------------------------------------
        # Preserve raw strings
        # -----------------------------------------------------

        raw_ts = df[" Timestamp"].astype(str).str.strip()
        labels = df[" Label"].astype(str).str.strip()

        # -----------------------------------------------------
        # Parse timestamps
        # -----------------------------------------------------

        parsed_ts = pd.to_datetime(
            raw_ts,
            dayfirst=True,
            errors="coerce"
        )

        total = len(df)
        valid = parsed_ts.notna().sum()
        invalid = parsed_ts.isna().sum()

        valid_series = parsed_ts.dropna()

        if len(valid_series) > 0:

            start = valid_series.min()
            end = valid_series.max()

            unique_ts = valid_series.nunique()

            chronological = valid_series.is_monotonic_increasing

            diffs = valid_series.diff().dt.total_seconds()

            backward_jumps = int((diffs < 0).sum())
            zero_gaps = int((diffs == 0).sum())

            positive_diffs = diffs[diffs > 0]

            largest_forward_gap = (
                positive_diffs.max()
                if len(positive_diffs)
                else None
            )

        else:

            start = None
            end = None
            unique_ts = 0
            chronological = False
            backward_jumps = None
            zero_gaps = None
            largest_forward_gap = None

        # -----------------------------------------------------
        # Print results
        # -----------------------------------------------------

        print(f"Encoding             : {encoding}")
        print(f"Rows                 : {total:,}")
        print(f"Valid timestamps     : {valid:,}")
        print(f"Invalid timestamps   : {invalid:,}")
        print(f"Unique timestamps    : {unique_ts:,}")
        print(f"Start                : {start}")
        print(f"End                  : {end}")
        print(f"Already chronological: {chronological}")
        print(f"Backward jumps       : {backward_jumps}")
        print(f"Zero-time gaps       : {zero_gaps}")
        print(f"Largest forward gap  : {largest_forward_gap} sec")

        print("\nRaw timestamp examples:")
        print(raw_ts.head(5).to_string(index=False))

        print("\nLABEL DISTRIBUTION")
        print("-" * 60)

        print(labels.value_counts().to_string())

        # -----------------------------------------------------
        # Save summary
        # -----------------------------------------------------

        timestamp_audit.append({

            "File": file.name,
            "Encoding": encoding,
            "Rows": total,

            "Valid_Timestamps": valid,
            "Invalid_Timestamps": invalid,
            "Unique_Timestamps": unique_ts,

            "Start": start,
            "End": end,

            "Chronological": chronological,
            "Backward_Jumps": backward_jumps,
            "Zero_Gaps": zero_gaps,

            "Largest_Forward_Gap_sec": largest_forward_gap,

            "Unique_Labels": labels.nunique()
        })

    except Exception as e:

        print("❌ ERROR:", repr(e))


# ============================================================
# SUMMARY
# ============================================================

timestamp_audit_df = pd.DataFrame(timestamp_audit)

print("\n" + "=" * 110)
print("PHASE 9.5B — SUMMARY")
print("=" * 110)

display(timestamp_audit_df)

In [ ]:
# ============================================================
# PHASE 9.6 — WEBATTACKS FILE ANOMALY FORENSICS
# ============================================================

import pandas as pd
from pathlib import Path

web_file = CIC_RAW / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"

print("=" * 110)
print("PHASE 9.6 — WEBATTACKS FILE ANOMALY FORENSICS")
print("=" * 110)

# ------------------------------------------------------------
# 1. Read complete file with its confirmed encoding
# ------------------------------------------------------------

web = pd.read_csv(
    web_file,
    encoding="cp1252",
    low_memory=False
)

print("\nShape:")
print(web.shape)

print("\nColumns:")
print(len(web.columns))

# ------------------------------------------------------------
# 2. Normalize ONLY column names in memory
#    RAW FILE IS NOT MODIFIED
# ------------------------------------------------------------

web.columns = web.columns.str.strip()

raw_ts = web["Timestamp"].astype("string").str.strip()

parsed_ts = pd.to_datetime(
    raw_ts,
    dayfirst=True,
    errors="coerce"
)

bad_mask = parsed_ts.isna()

print("\n" + "=" * 110)
print("VALID / INVALID TIMESTAMP ROWS")
print("=" * 110)

print(f"Total rows   : {len(web):,}")
print(f"Valid rows   : {(~bad_mask).sum():,}")
print(f"Invalid rows : {bad_mask.sum():,}")

# ------------------------------------------------------------
# 3. Where do invalid rows occur?
# ------------------------------------------------------------

bad_indices = web.index[bad_mask]

print("\nINVALID ROW POSITION")

if len(bad_indices):

    print("First invalid index :", bad_indices.min())
    print("Last invalid index  :", bad_indices.max())

    # Are they one continuous block?
    expected = (
        bad_indices.max()
        - bad_indices.min()
        + 1
    )

    print(
        "Single continuous block:",
        expected == len(bad_indices)
    )

# ------------------------------------------------------------
# 4. Inspect boundary around first invalid row
# ------------------------------------------------------------

if len(bad_indices):

    first_bad = bad_indices.min()

    start = max(0, first_bad - 5)
    end = min(len(web), first_bad + 10)

    important_cols = [
        "Flow ID",
        "Source IP",
        "Source Port",
        "Destination IP",
        "Destination Port",
        "Timestamp",
        "Label"
    ]

    print("\n" + "=" * 110)
    print("ROWS AROUND FIRST INVALID TIMESTAMP")
    print("=" * 110)

    display(
        web.loc[start:end, important_cols]
    )

# ------------------------------------------------------------
# 5. Missingness in valid vs invalid sections
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("MISSINGNESS — INVALID TIMESTAMP ROWS")
print("=" * 110)

bad_missing = (
    web.loc[bad_mask]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print(bad_missing.head(25).to_string())


print("\n" + "=" * 110)
print("MISSINGNESS — VALID TIMESTAMP ROWS")
print("=" * 110)

good_missing = (
    web.loc[~bad_mask]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print(good_missing.head(25).to_string())

# ------------------------------------------------------------
# 6. Do bad rows contain ANY meaningful values?
# ------------------------------------------------------------

bad_non_null = (
    web.loc[bad_mask]
    .notna()
    .sum()
    .sort_values(ascending=False)
)

print("\n" + "=" * 110)
print("NON-NULL VALUES INSIDE INVALID ROWS")
print("=" * 110)

print(bad_non_null.head(30).to_string())

# ------------------------------------------------------------
# 7. Raw timestamp values responsible for failure
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("MOST COMMON RAW VALUES IN INVALID TIMESTAMP ROWS")
print("=" * 110)

print(
    raw_ts[bad_mask]
    .value_counts(dropna=False)
    .head(20)
    .to_string()
)

# ------------------------------------------------------------
# 8. Labels in valid vs invalid rows
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("LABELS — VALID TIMESTAMP ROWS")
print("=" * 110)

print(
    web.loc[~bad_mask, "Label"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 110)
print("LABELS — INVALID TIMESTAMP ROWS")
print("=" * 110)

print(
    web.loc[bad_mask, "Label"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
    .head(20)
    .to_string()
)

In [ ]:
# ============================================================
# PHASE 9.6B — RAW LINE COUNT + FILE END INSPECTION
# ============================================================

web_file = CIC_RAW / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"

print("=" * 110)
print("PHASE 9.6B — RAW FILE STRUCTURE")
print("=" * 110)

with open(web_file, "r", encoding="cp1252", errors="strict") as f:

    lines = f.readlines()

print(f"Physical lines : {len(lines):,}")
print(f"Header lines   : 1")
print(f"Expected rows  : {len(lines)-1:,}")

print("\nFIRST 3 PHYSICAL LINES:")
print("-" * 110)

for line in lines[:3]:
    print(repr(line[:300]))

print("\nLAST 5 PHYSICAL LINES:")
print("-" * 110)

for line in lines[-5:]:
    print(repr(line[:300]))

In [ ]:
# ============================================================
# PHASE 9.6C — RECORD STRUCTURAL ROW AUDIT
# ============================================================

import pandas as pd

structural_audit = pd.DataFrame([
    {
        "File": "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
        "Physical_Data_Rows": 458_968,
        "Usable_Flow_Rows": 170_366,
        "Fully_Empty_Rows": 288_602,
        "Empty_Block_Start_Index": 170_366,
        "Empty_Block_End_Index": 458_967,
        "Action_For_Audit": "Exclude fully empty structural rows",
        "Raw_File_Modified": False
    }
])

display(structural_audit)

structural_audit.to_csv(
    METADATA / "cic_structural_row_audit.csv",
    index=False
)

corrected_total = sum(row_counts.values()) - 288_602

print(f"\nOriginal physical data rows : {sum(row_counts.values()):,}")
print(f"Fully empty structural rows : {288_602:,}")
print(f"Actual usable CIC flows     : {corrected_total:,}")

In [ ]:
# ============================================================
# PHASE 9.7 — CIC RAW TIMESTAMP FORMAT AUDIT
# ============================================================

import pandas as pd
import re

print("=" * 115)
print("PHASE 9.7 — CIC RAW TIMESTAMP FORMAT / AM-PM AUDIT")
print("=" * 115)

timestamp_format_results = []

for file in CIC_FILES:

    encoding = file_encodings[file.name]

    print("\n" + "=" * 115)
    print(file.name)
    print("=" * 115)

    df = pd.read_csv(
        file,
        usecols=[" Timestamp"],
        encoding=encoding,
        low_memory=False
    )

    # Remove structurally empty rows
    ts = (
        df[" Timestamp"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # Detect AM/PM markers
    # --------------------------------------------------------

    has_am = ts.str.contains(
        r"\bAM\b",
        case=False,
        regex=True
    ).sum()

    has_pm = ts.str.contains(
        r"\bPM\b",
        case=False,
        regex=True
    ).sum()

    # --------------------------------------------------------
    # Count number of ":" in time component
    # 1 colon -> minute resolution
    # 2 colons -> second resolution
    # --------------------------------------------------------

    colon_counts = ts.str.count(":")

    minute_resolution = (colon_counts == 1).sum()
    second_resolution = (colon_counts >= 2).sum()

    # --------------------------------------------------------
    # Parse
    # --------------------------------------------------------

    parsed = pd.to_datetime(
        ts,
        dayfirst=True,
        errors="coerce"
    )

    # Hour distribution after parsing
    hour_counts = (
        parsed.dropna()
        .dt.hour
        .value_counts()
        .sort_index()
    )

    print(f"Usable timestamp strings : {len(ts):,}")
    print(f"Contains AM marker       : {has_am:,}")
    print(f"Contains PM marker       : {has_pm:,}")
    print(f"Minute-resolution rows   : {minute_resolution:,}")
    print(f"Second-resolution rows   : {second_resolution:,}")

    print("\nPARSED HOUR DISTRIBUTION")
    print("-" * 60)

    print(hour_counts.to_string())

    print("\nFIRST 10 UNIQUE RAW TIMESTAMPS")
    print("-" * 60)

    for value in ts.drop_duplicates().head(10):
        print(repr(value))

    print("\nLAST 10 UNIQUE RAW TIMESTAMPS IN FILE ORDER")
    print("-" * 60)

    for value in ts.drop_duplicates().tail(10):
        print(repr(value))

    timestamp_format_results.append({
        "File": file.name,
        "Rows_With_Timestamp": len(ts),
        "AM_Markers": int(has_am),
        "PM_Markers": int(has_pm),
        "Minute_Resolution": int(minute_resolution),
        "Second_Resolution": int(second_resolution),
        "Min_Parsed_Hour": (
            int(parsed.dt.hour.min())
            if parsed.notna().any()
            else None
        ),
        "Max_Parsed_Hour": (
            int(parsed.dt.hour.max())
            if parsed.notna().any()
            else None
        )
    })


timestamp_format_df = pd.DataFrame(timestamp_format_results)

print("\n" + "=" * 115)
print("PHASE 9.7 SUMMARY")
print("=" * 115)

display(timestamp_format_df)

In [ ]:
# ============================================================
# PHASE 9.7B — RAW ORDER TRANSITION AUDIT
# ============================================================

problem_files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv"
]

print("=" * 115)
print("PHASE 9.7B — RAW TIMESTAMP ORDER TRANSITIONS")
print("=" * 115)

for filename in problem_files:

    file = CIC_RAW / filename
    encoding = file_encodings[filename]

    df = pd.read_csv(
        file,
        usecols=[" Timestamp"],
        encoding=encoding,
        low_memory=False
    )

    ts_raw = (
        df[" Timestamp"]
        .astype("string")
        .str.strip()
    )

    ts = pd.to_datetime(
        ts_raw,
        dayfirst=True,
        errors="coerce"
    )

    diffs = ts.diff().dt.total_seconds()

    # Biggest forward/backward transitions
    forward_idx = diffs.nlargest(5).index
    backward_idx = diffs.nsmallest(5).index

    print("\n" + "=" * 115)
    print(filename)
    print("=" * 115)

    print("\nLARGEST FORWARD TRANSITIONS")
    print("-" * 80)

    for idx in forward_idx:

        if idx == 0:
            continue

        print(
            f"row {idx-1:,} → {idx:,}: "
            f"{repr(ts_raw.iloc[idx-1])} → "
            f"{repr(ts_raw.iloc[idx])} "
            f"({diffs.iloc[idx]:,.0f} sec)"
        )

    print("\nLARGEST BACKWARD TRANSITIONS")
    print("-" * 80)

    for idx in backward_idx:

        if idx == 0:
            continue

        print(
            f"row {idx-1:,} → {idx:,}: "
            f"{repr(ts_raw.iloc[idx-1])} → "
            f"{repr(ts_raw.iloc[idx])} "
            f"({diffs.iloc[idx]:,.0f} sec)"
        )

In [ ]:
# ============================================================
# PHASE 9.8 — BUILD CANONICAL CIC MINUTE TIMELINE
# ============================================================

import pandas as pd
import numpy as np

print("=" * 115)
print("PHASE 9.8 — BUILD CANONICAL CIC MINUTE TIMELINE")
print("=" * 115)

minute_parts = []
flow_audit = []

for file in CIC_FILES:

    encoding = file_encodings[file.name]

    print(f"\nProcessing: {file.name}")

    # --------------------------------------------------------
    # Read only fields needed for temporal reconstruction
    # --------------------------------------------------------

    df = pd.read_csv(
        file,
        usecols=[" Timestamp", " Label"],
        encoding=encoding,
        low_memory=False
    )

    physical_rows = len(df)

    # --------------------------------------------------------
    # Remove fully structural/empty records
    # --------------------------------------------------------

    df = df.dropna(how="all").copy()

    usable_rows = len(df)

    # Normalize strings IN MEMORY ONLY
    df["Timestamp"] = (
        df[" Timestamp"]
        .astype("string")
        .str.strip()
    )

    df["Label"] = (
        df[" Label"]
        .astype("string")
        .str.strip()
    )

    # --------------------------------------------------------
    # Parse timestamp
    # --------------------------------------------------------

    df["Parsed_Time"] = pd.to_datetime(
        df["Timestamp"],
        dayfirst=True,
        errors="coerce"
    )

    bad_time = df["Parsed_Time"].isna().sum()

    # Only timestamp-valid flows can enter temporal analysis
    df = df[df["Parsed_Time"].notna()].copy()

    # --------------------------------------------------------
    # Minute bin
    # --------------------------------------------------------

    df["Minute"] = df["Parsed_Time"].dt.floor("min")

    # --------------------------------------------------------
    # Attack indicator
    # BENIGN = 0
    # anything else = malicious flow
    # --------------------------------------------------------

    df["Is_Attack"] = (
        df["Label"]
        .str.upper()
        .ne("BENIGN")
        .astype("int8")
    )

    # --------------------------------------------------------
    # Aggregate flows by minute
    # --------------------------------------------------------

    minute = (
        df.groupby("Minute", as_index=False)
        .agg(
            Flow_Count=("Is_Attack", "size"),
            Attack_Flow_Count=("Is_Attack", "sum")
        )
    )

    # A minute is attack-active if >= 1 malicious flow occurs
    minute["Attack_State"] = (
        minute["Attack_Flow_Count"] > 0
    ).astype("int8")

    minute["Segment_ID"] = file.name

    minute = minute[
        [
            "Segment_ID",
            "Minute",
            "Attack_State",
            "Flow_Count",
            "Attack_Flow_Count"
        ]
    ]

    minute = minute.sort_values("Minute").reset_index(drop=True)

    minute_parts.append(minute)

    # --------------------------------------------------------
    # Audit record
    # --------------------------------------------------------

    flow_audit.append({
        "Segment_ID": file.name,
        "Physical_Rows": physical_rows,
        "Usable_Rows_After_Empty_Removal": usable_rows,
        "Invalid_Timestamps": int(bad_time),
        "Timeline_Minutes": len(minute),
        "Attack_Minutes": int(minute["Attack_State"].sum()),
        "Benign_Minutes": int((minute["Attack_State"] == 0).sum()),
        "Start": minute["Minute"].min(),
        "End": minute["Minute"].max()
    })

    print(f"  physical rows : {physical_rows:,}")
    print(f"  usable rows   : {usable_rows:,}")
    print(f"  bad timestamps: {bad_time:,}")
    print(f"  minute bins   : {len(minute):,}")
    print(f"  attack minutes: {minute['Attack_State'].sum():,}")


# ============================================================
# COMBINE SEGMENTS
# ============================================================

minute_df = pd.concat(
    minute_parts,
    ignore_index=True
)

flow_audit_df = pd.DataFrame(flow_audit)

print("\n" + "=" * 115)
print("SEGMENT AUDIT")
print("=" * 115)

display(flow_audit_df)

print("\n" + "=" * 115)
print("GLOBAL MINUTE TIMELINE")
print("=" * 115)

print(f"Total minute records : {len(minute_df):,}")
print(f"Attack-active minutes: {minute_df['Attack_State'].sum():,}")
print(
    f"Benign minutes       : "
    f"{(minute_df['Attack_State'] == 0).sum():,}"
)

print("\nAttack-state distribution:")
print(minute_df["Attack_State"].value_counts().sort_index())

# Save audit artifact
minute_df.to_csv(
    CIC_AUDIT / "cic_minute_timeline_v1.csv",
    index=False
)

flow_audit_df.to_csv(
    METADATA / "cic_segment_timeline_audit.csv",
    index=False
)

print("\n✓ Saved CIC minute timeline")

In [ ]:
# ============================================================
# PHASE 9.9 — CIC ATTACK EPISODE RECONSTRUCTION
# ============================================================

print("=" * 115)
print("PHASE 9.9 — CIC ATTACK EPISODE RECONSTRUCTION")
print("=" * 115)

episodes = []

for segment, seg in minute_df.groupby(
    "Segment_ID",
    sort=False
):

    seg = seg.sort_values("Minute").reset_index(drop=True)

    # --------------------------------------------------------
    # Start a new episode when:
    # current minute = attack
    # AND previous minute != attack
    # --------------------------------------------------------

    previous_state = seg["Attack_State"].shift(
        fill_value=0
    )

    starts = (
        (seg["Attack_State"] == 1) &
        (previous_state == 0)
    )

    # --------------------------------------------------------
    # End episode when:
    # current minute = attack
    # AND next minute != attack
    # --------------------------------------------------------

    next_state = seg["Attack_State"].shift(
        -1,
        fill_value=0
    )

    ends = (
        (seg["Attack_State"] == 1) &
        (next_state == 0)
    )

    start_indices = seg.index[starts].tolist()
    end_indices = seg.index[ends].tolist()

    print("\n" + "-" * 115)
    print(segment)
    print("-" * 115)

    print(f"Attack episode starts: {len(start_indices)}")
    print(f"Attack episode ends  : {len(end_indices)}")

    if len(start_indices) != len(end_indices):
        print("⚠ START/END COUNT MISMATCH")
        continue

    for episode_num, (start_idx, end_idx) in enumerate(
        zip(start_indices, end_indices),
        start=1
    ):

        start_time = seg.loc[start_idx, "Minute"]
        end_time = seg.loc[end_idx, "Minute"]

        duration = int(
            (end_time - start_time).total_seconds() / 60
        ) + 1

        episode_slice = seg.loc[start_idx:end_idx]

        attack_flows = int(
            episode_slice["Attack_Flow_Count"].sum()
        )

        total_flows = int(
            episode_slice["Flow_Count"].sum()
        )

        episodes.append({
            "Segment_ID": segment,
            "Episode_Number": episode_num,
            "Start_Minute": start_time,
            "End_Minute": end_time,
            "Duration_Minutes": duration,
            "Attack_Flows": attack_flows,
            "Total_Flows": total_flows
        })

        print(
            f"Episode {episode_num:>2}: "
            f"{start_time} → {end_time} | "
            f"{duration:>3} min | "
            f"{attack_flows:,} attack flows"
        )


episode_df = pd.DataFrame(episodes)

print("\n" + "=" * 115)
print("ALL DETECTED CIC ATTACK EPISODES")
print("=" * 115)

display(episode_df)

episode_df.to_csv(
    CIC_AUDIT / "cic_attack_episodes_v1.csv",
    index=False
)

print(f"\nTotal detected episodes: {len(episode_df)}")
print("✓ Episode table saved")

In [ ]:
# ============================================================
# PHASE 9.10 — CIC TIMELINE COMPLETENESS + ATTACK FAMILY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 115)
print("PHASE 9.10 — CIC TIMELINE COMPLETENESS + ATTACK FAMILY AUDIT")
print("=" * 115)

completeness_records = []
family_minute_parts = []

for file in CIC_FILES:

    encoding = file_encodings[file.name]

    print("\n" + "=" * 115)
    print(file.name)
    print("=" * 115)

    # --------------------------------------------------------
    # READ TIMESTAMP + LABEL ONLY
    # --------------------------------------------------------

    df = pd.read_csv(
        file,
        usecols=[" Timestamp", " Label"],
        encoding=encoding,
        low_memory=False
    )

    # Remove structural blank rows
    df = df.dropna(how="all").copy()

    # Normalize strings
    df["Timestamp"] = (
        df[" Timestamp"]
        .astype("string")
        .str.strip()
    )

    df["Label"] = (
        df[" Label"]
        .astype("string")
        .str.strip()
    )

    # Parse time
    df["Parsed_Time"] = pd.to_datetime(
        df["Timestamp"],
        dayfirst=True,
        errors="coerce"
    )

    # Remove timestamp-invalid rows
    df = df[df["Parsed_Time"].notna()].copy()

    # Minute bin
    df["Minute"] = df["Parsed_Time"].dt.floor("min")

    # --------------------------------------------------------
    # PART A — TIMELINE COMPLETENESS
    # --------------------------------------------------------

    observed_minutes = (
        pd.DatetimeIndex(df["Minute"].unique())
        .sort_values()
    )

    start = observed_minutes.min()
    end = observed_minutes.max()

    expected_minutes = pd.date_range(
        start=start,
        end=end,
        freq="min"
    )

    missing_minutes = expected_minutes.difference(
        observed_minutes
    )

    expected_count = len(expected_minutes)
    observed_count = len(observed_minutes)
    missing_count = len(missing_minutes)

    coverage = (
        observed_count / expected_count
        if expected_count > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Missing-minute runs
    # --------------------------------------------------------

    longest_missing_run = 0

    if missing_count > 0:

        missing_series = pd.Series(missing_minutes)

        gaps = (
            missing_series.diff()
            .dt.total_seconds()
            .div(60)
        )

        group_id = gaps.ne(1).cumsum()

        run_lengths = (
            missing_series
            .groupby(group_id)
            .size()
        )

        longest_missing_run = int(run_lengths.max())

    print("TIMELINE COMPLETENESS")
    print("-" * 60)

    print(f"Start                 : {start}")
    print(f"End                   : {end}")
    print(f"Expected clock minutes: {expected_count:,}")
    print(f"Observed minutes      : {observed_count:,}")
    print(f"Missing minutes       : {missing_count:,}")
    print(f"Coverage              : {coverage:.2%}")
    print(f"Longest missing run   : {longest_missing_run} min")

    if missing_count > 0:

        print("\nFirst missing minutes:")

        for t in missing_minutes[:10]:
            print(" ", t)

    # --------------------------------------------------------
    # PART B — RAW LABEL DISTRIBUTION
    # --------------------------------------------------------

    label_counts = (
        df["Label"]
        .value_counts(dropna=False)
    )

    print("\nFLOW LABEL DISTRIBUTION")
    print("-" * 60)

    print(label_counts.to_string())

    # --------------------------------------------------------
    # PART C — ATTACK FAMILY PER MINUTE
    # --------------------------------------------------------

    attack_df = df[
        df["Label"].str.upper().ne("BENIGN")
    ].copy()

    if len(attack_df) > 0:

        # Number of malicious flows of each label in each minute
        family_counts = (
            attack_df
            .groupby(["Minute", "Label"])
            .size()
            .reset_index(name="Attack_Flow_Count")
        )

        # Collect labels active in each minute
        minute_family = (
            family_counts
            .groupby("Minute")
            .agg(
                Attack_Families=(
                    "Label",
                    lambda x: " | ".join(sorted(set(x)))
                ),
                Number_of_Attack_Families=(
                    "Label",
                    "nunique"
                ),
                Attack_Flow_Count=(
                    "Attack_Flow_Count",
                    "sum"
                )
            )
            .reset_index()
        )

        minute_family["Segment_ID"] = file.name

        family_minute_parts.append(minute_family)

        # ----------------------------------------------------
        # Minutes per family
        # ----------------------------------------------------

        family_minutes = (
            family_counts
            .groupby("Label")["Minute"]
            .nunique()
            .sort_values(ascending=False)
        )

        print("\nATTACK-ACTIVE MINUTES PER FAMILY")
        print("-" * 60)

        print(family_minutes.to_string())

        # ----------------------------------------------------
        # Multi-family minutes
        # ----------------------------------------------------

        multi_family = minute_family[
            minute_family["Number_of_Attack_Families"] > 1
        ]

        print(
            f"\nMinutes containing >1 attack family: "
            f"{len(multi_family):,}"
        )

        if len(multi_family) > 0:

            print("\nMULTI-FAMILY MINUTES")

            display(
                multi_family[
                    [
                        "Minute",
                        "Attack_Families",
                        "Number_of_Attack_Families",
                        "Attack_Flow_Count"
                    ]
                ].head(20)
            )

    else:

        print("\nNo attack labels in this segment.")

    # --------------------------------------------------------
    # Save completeness summary
    # --------------------------------------------------------

    completeness_records.append({

        "Segment_ID": file.name,

        "Start": start,
        "End": end,

        "Expected_Minutes": expected_count,
        "Observed_Minutes": observed_count,
        "Missing_Minutes": missing_count,

        "Coverage": coverage,

        "Longest_Missing_Run_Min": longest_missing_run,

        "Unique_Labels": int(df["Label"].nunique()),

        "Attack_Families": int(
            df.loc[
                df["Label"].str.upper().ne("BENIGN"),
                "Label"
            ].nunique()
        )
    })


# ============================================================
# COMBINE RESULTS
# ============================================================

completeness_df = pd.DataFrame(
    completeness_records
)

if family_minute_parts:

    family_minute_df = pd.concat(
        family_minute_parts,
        ignore_index=True
    )

else:

    family_minute_df = pd.DataFrame()


print("\n" + "=" * 115)
print("TIMELINE COMPLETENESS SUMMARY")
print("=" * 115)

display(completeness_df)


print("\n" + "=" * 115)
print("GLOBAL ATTACK FAMILY SUMMARY")
print("=" * 115)

if len(family_minute_df) > 0:

    global_family_summary = (
        family_minute_df
        .assign(
            Family_List=lambda x:
                x["Attack_Families"].str.split(" | ", regex=False)
        )
        .explode("Family_List")
        .groupby("Family_List")
        .agg(
            Attack_Minutes=("Minute", "nunique"),
            Segments=("Segment_ID", "nunique")
        )
        .sort_values(
            "Attack_Minutes",
            ascending=False
        )
    )

    display(global_family_summary)


# ============================================================
# SAVE AUDIT ARTIFACTS
# ============================================================

completeness_df.to_csv(
    METADATA / "cic_timeline_completeness.csv",
    index=False
)

if len(family_minute_df) > 0:

    family_minute_df.to_csv(
        CIC_AUDIT / "cic_attack_family_minutes.csv",
        index=False
    )


print("\n✓ Timeline completeness audit saved")
print("✓ Attack-family minute audit saved")

In [ ]:
# ============================================================
# PHASE 9.11 — CIC FAMILY-SPECIFIC EPISODE GAP SENSITIVITY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 115)
print("PHASE 9.11 — CIC FAMILY-SPECIFIC EPISODE GAP SENSITIVITY")
print("=" * 115)

# ------------------------------------------------------------
# GAP TOLERANCES TO TEST
#
# 0  = only immediately consecutive attack minutes merge
# 1  = allow 1 non-attack minute between attack minutes
# 2  = allow up to 2
# 5  = allow up to 5
# 10 = allow up to 10
# ------------------------------------------------------------

GAP_TOLERANCES = [0, 1, 2, 5, 10]

sensitivity_records = []
episode_records = []


# ============================================================
# PROCESS EACH SEGMENT + ATTACK FAMILY
# ============================================================

for (segment, family), group in (
    family_minute_df
    .assign(
        Family_List=lambda x:
            x["Attack_Families"].str.split(" | ", regex=False)
    )
    .explode("Family_List")
    .groupby(["Segment_ID", "Family_List"], sort=False)
):

    family = str(family)

    # Minutes where this family was active
    attack_minutes = (
        group["Minute"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    print("\n" + "=" * 115)
    print(f"{family}")
    print(f"Segment: {segment}")
    print("=" * 115)

    print(f"Attack-active minutes: {len(attack_minutes):,}")

    # --------------------------------------------------------
    # TEST EACH GAP TOLERANCE
    # --------------------------------------------------------

    for tolerance in GAP_TOLERANCES:

        episodes = []

        if len(attack_minutes) > 0:

            current_start = attack_minutes.iloc[0]
            current_end = attack_minutes.iloc[0]

            for current_time in attack_minutes.iloc[1:]:

                # Number of NON-ATTACK clock minutes
                # between previous attack minute and this one
                gap_minutes = int(
                    (
                        current_time - current_end
                    ).total_seconds() / 60
                ) - 1

                # --------------------------------------------
                # If gap <= tolerance:
                # treat as same candidate episode
                # --------------------------------------------

                if gap_minutes <= tolerance:

                    current_end = current_time

                else:

                    episodes.append(
                        (current_start, current_end)
                    )

                    current_start = current_time
                    current_end = current_time

            # final episode
            episodes.append(
                (current_start, current_end)
            )

        # ----------------------------------------------------
        # SAVE SUMMARY
        # ----------------------------------------------------

        durations = [
            int(
                (end - start).total_seconds() / 60
            ) + 1
            for start, end in episodes
        ]

        sensitivity_records.append({

            "Segment_ID": segment,
            "Attack_Family": family,
            "Gap_Tolerance_Min": tolerance,

            "Attack_Active_Minutes": len(
                attack_minutes
            ),

            "Episode_Count": len(episodes),

            "Shortest_Episode_Min": (
                min(durations)
                if durations else 0
            ),

            "Longest_Episode_Min": (
                max(durations)
                if durations else 0
            ),

            "Median_Episode_Min": (
                float(np.median(durations))
                if durations else 0
            )
        })

        print(
            f"Tolerance {tolerance:>2} min"
            f"  →  {len(episodes):>3} episodes"
        )

        # ----------------------------------------------------
        # SAVE INDIVIDUAL EPISODES
        # ----------------------------------------------------

        for episode_number, (start, end) in enumerate(
            episodes,
            start=1
        ):

            duration = int(
                (end - start).total_seconds() / 60
            ) + 1

            episode_records.append({

                "Segment_ID": segment,
                "Attack_Family": family,

                "Gap_Tolerance_Min": tolerance,

                "Episode_Number": episode_number,

                "Start_Minute": start,
                "End_Minute": end,

                "Duration_Minutes": duration
            })


# ============================================================
# RESULTS
# ============================================================

sensitivity_df = pd.DataFrame(
    sensitivity_records
)

family_episode_df = pd.DataFrame(
    episode_records
)


print("\n" + "=" * 115)
print("EPISODE COUNT SENSITIVITY MATRIX")
print("=" * 115)

episode_matrix = (
    sensitivity_df
    .pivot_table(
        index="Attack_Family",
        columns="Gap_Tolerance_Min",
        values="Episode_Count",
        aggfunc="sum"
    )
    .fillna(0)
    .astype(int)
)

episode_matrix.columns = [
    f"Gap_{c}_min"
    for c in episode_matrix.columns
]

display(episode_matrix)


# ============================================================
# TOTAL EPISODES ACROSS CIC
# ============================================================

print("\n" + "=" * 115)
print("TOTAL CANDIDATE EPISODES BY GAP TOLERANCE")
print("=" * 115)

total_sensitivity = (
    sensitivity_df
    .groupby("Gap_Tolerance_Min")[
        "Episode_Count"
    ]
    .sum()
)

print(total_sensitivity.to_string())


# ============================================================
# IDENTIFY HIGHLY GAP-SENSITIVE FAMILIES
# ============================================================

print("\n" + "=" * 115)
print("FAMILIES MOST AFFECTED BY GAP MERGING")
print("=" * 115)

comparison = episode_matrix.copy()

first_col = "Gap_0_min"
last_col = "Gap_10_min"

comparison["Reduction_0_to_10"] = (
    comparison[first_col]
    - comparison[last_col]
)

comparison["Reduction_Percent"] = np.where(
    comparison[first_col] > 0,

    comparison["Reduction_0_to_10"]
    / comparison[first_col]
    * 100,

    0
)

comparison = comparison.sort_values(
    "Reduction_Percent",
    ascending=False
)

display(comparison)


# ============================================================
# SAVE
# ============================================================

sensitivity_df.to_csv(
    CIC_AUDIT /
    "cic_episode_gap_sensitivity.csv",
    index=False
)

family_episode_df.to_csv(
    CIC_AUDIT /
    "cic_family_candidate_episodes.csv",
    index=False
)

episode_matrix.to_csv(
    CIC_AUDIT /
    "cic_episode_sensitivity_matrix.csv"
)

print("\n✓ Episode sensitivity results saved")
print("✓ Family candidate episodes saved")

In [ ]:
# ============================================================
# PHASE 9.12 — CIC OFFICIAL ATTACK SCHEDULE RECONCILIATION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 120)
print("PHASE 9.12 — CIC OFFICIAL ATTACK SCHEDULE RECONCILIATION")
print("=" * 120)

# ============================================================
# 1. OFFICIAL CIC-IDS2017 ATTACK SCHEDULE
#
# Source:
# University of New Brunswick
# Canadian Institute for Cybersecurity
# CIC-IDS2017 official dataset documentation
#
# IMPORTANT:
# Official_Start / Official_End preserve the documented clock.
#
# CSV_Start / CSV_End are the corresponding clock values used
# by the provided MachineLearningCSV timestamps where the
# afternoon hour appears without explicit PM information.
# ============================================================

schedule_records = [

    # --------------------------------------------------------
    # TUESDAY — BRUTE FORCE
    # --------------------------------------------------------

    {
        "GT_Event_ID": "CIC_TUE_FTP_01",
        "Date": "2017-07-04",
        "Attack_Family": "FTP-Patator",
        "Official_Start": "09:20",
        "Official_End": "10:20",
        "CSV_Start": "09:20",
        "CSV_End": "10:20",
        "Official_Description": "FTP-Patator"
    },

    {
        "GT_Event_ID": "CIC_TUE_SSH_01",
        "Date": "2017-07-04",
        "Attack_Family": "SSH-Patator",
        "Official_Start": "14:00",
        "Official_End": "15:00",
        "CSV_Start": "02:00",
        "CSV_End": "03:00",
        "Official_Description": "SSH-Patator"
    },

    # --------------------------------------------------------
    # WEDNESDAY — DoS
    # --------------------------------------------------------

    {
        "GT_Event_ID": "CIC_WED_SLOWLORIS_01",
        "Date": "2017-07-05",
        "Attack_Family": "DoS slowloris",
        "Official_Start": "09:47",
        "Official_End": "10:10",
        "CSV_Start": "09:47",
        "CSV_End": "10:10",
        "Official_Description": "DoS slowloris"
    },

    {
        "GT_Event_ID": "CIC_WED_SLOWHTTP_01",
        "Date": "2017-07-05",
        "Attack_Family": "DoS Slowhttptest",
        "Official_Start": "10:14",
        "Official_End": "10:35",
        "CSV_Start": "10:14",
        "CSV_End": "10:35",
        "Official_Description": "DoS Slowhttptest"
    },

    {
        "GT_Event_ID": "CIC_WED_HULK_01",
        "Date": "2017-07-05",
        "Attack_Family": "DoS Hulk",
        "Official_Start": "10:43",
        "Official_End": "11:00",
        "CSV_Start": "10:43",
        "CSV_End": "11:00",
        "Official_Description": "DoS Hulk"
    },

    {
        "GT_Event_ID": "CIC_WED_GOLDENEYE_01",
        "Date": "2017-07-05",
        "Attack_Family": "DoS GoldenEye",
        "Official_Start": "11:10",
        "Official_End": "11:23",
        "CSV_Start": "11:10",
        "CSV_End": "11:23",
        "Official_Description": "DoS GoldenEye"
    },

    {
        "GT_Event_ID": "CIC_WED_HEARTBLEED_01",
        "Date": "2017-07-05",
        "Attack_Family": "Heartbleed",
        "Official_Start": "15:12",
        "Official_End": "15:32",
        "CSV_Start": "03:12",
        "CSV_End": "03:32",
        "Official_Description": "Heartbleed Port 444"
    },

    # --------------------------------------------------------
    # THURSDAY MORNING — WEB ATTACKS
    # --------------------------------------------------------

    {
        "GT_Event_ID": "CIC_THU_WEBBF_01",
        "Date": "2017-07-06",
        "Attack_Family": "Web Attack – Brute Force",
        "Official_Start": "09:20",
        "Official_End": "10:00",
        "CSV_Start": "09:20",
        "CSV_End": "10:00",
        "Official_Description": "Web Attack – Brute Force"
    },

    {
        "GT_Event_ID": "CIC_THU_XSS_01",
        "Date": "2017-07-06",
        "Attack_Family": "Web Attack – XSS",
        "Official_Start": "10:15",
        "Official_End": "10:35",
        "CSV_Start": "10:15",
        "CSV_End": "10:35",
        "Official_Description": "Web Attack – XSS"
    },

    {
        "GT_Event_ID": "CIC_THU_SQL_01",
        "Date": "2017-07-06",
        "Attack_Family": "Web Attack – Sql Injection",
        "Official_Start": "10:40",
        "Official_End": "10:42",
        "CSV_Start": "10:40",
        "CSV_End": "10:42",
        "Official_Description": "Web Attack – Sql Injection"
    },

    # --------------------------------------------------------
    # THURSDAY AFTERNOON — INFILTRATION
    #
    # Official documentation describes distinct activity
    # periods, so preserve them separately.
    # --------------------------------------------------------

    {
        "GT_Event_ID": "CIC_THU_INF_01",
        "Date": "2017-07-06",
        "Attack_Family": "Infiltration",
        "Official_Start": "14:19",
        "Official_End": "14:21",
        "CSV_Start": "02:19",
        "CSV_End": "02:21",
        "Official_Description": "Meta exploit Win Vista"
    },

    {
        "GT_Event_ID": "CIC_THU_INF_02",
        "Date": "2017-07-06",
        "Attack_Family": "Infiltration",
        "Official_Start": "14:33",
        "Official_End": "14:35",
        "CSV_Start": "02:33",
        "CSV_End": "02:35",
        "Official_Description": "Meta exploit Win Vista"
    },

    {
        "GT_Event_ID": "CIC_THU_INF_03",
        "Date": "2017-07-06",
        "Attack_Family": "Infiltration",
        "Official_Start": "14:53",
        "Official_End": "15:00",
        "CSV_Start": "02:53",
        "CSV_End": "03:00",
        "Official_Description": "Cool disk – MAC"
    },

    {
        "GT_Event_ID": "CIC_THU_INF_04",
        "Date": "2017-07-06",
        "Attack_Family": "Infiltration",
        "Official_Start": "15:04",
        "Official_End": "15:45",
        "CSV_Start": "03:04",
        "CSV_End": "03:45",
        "Official_Description": "Dropbox download / subsequent internal activity"
    },

    # --------------------------------------------------------
    # FRIDAY MORNING — BOTNET
    # --------------------------------------------------------

    {
        "GT_Event_ID": "CIC_FRI_BOT_01",
        "Date": "2017-07-07",
        "Attack_Family": "Bot",
        "Official_Start": "10:02",
        "Official_End": "11:02",
        "CSV_Start": "10:02",
        "CSV_End": "11:02",
        "Official_Description": "Botnet ARES"
    },

    # --------------------------------------------------------
    # FRIDAY AFTERNOON — PORTSCAN
    #
    # Rather than turning every individual scan flag into a
    # forecasting event, preserve the two documented major
    # experimental phases:
    #
    # 1. Firewall rule ON
    # 2. Firewall rules OFF
    #
    # Individual sub-periods remain described in Notes.
    # --------------------------------------------------------

    {
        "GT_Event_ID": "CIC_FRI_PORTSCAN_FW_ON",
        "Date": "2017-07-07",
        "Attack_Family": "PortScan",
        "Official_Start": "13:55",
        "Official_End": "14:35",
        "CSV_Start": "01:55",
        "CSV_End": "02:35",
        "Official_Description": "Port Scan — firewall rule on"
    },

    {
        "GT_Event_ID": "CIC_FRI_PORTSCAN_FW_OFF",
        "Date": "2017-07-07",
        "Attack_Family": "PortScan",
        "Official_Start": "14:51",
        "Official_End": "15:29",
        "CSV_Start": "02:51",
        "CSV_End": "03:29",
        "Official_Description": "Port Scan — firewall rules off"
    },

    # --------------------------------------------------------
    # FRIDAY AFTERNOON — DDoS
    # --------------------------------------------------------

    {
        "GT_Event_ID": "CIC_FRI_DDOS_01",
        "Date": "2017-07-07",
        "Attack_Family": "DDoS",
        "Official_Start": "15:56",
        "Official_End": "16:16",
        "CSV_Start": "03:56",
        "CSV_End": "04:16",
        "Official_Description": "DDoS LOIT"
    }
]


schedule_df = pd.DataFrame(schedule_records)


# ============================================================
# 2. BUILD DATETIMES
# ============================================================

schedule_df["Official_Start_DT"] = pd.to_datetime(
    schedule_df["Date"] + " " + schedule_df["Official_Start"]
)

schedule_df["Official_End_DT"] = pd.to_datetime(
    schedule_df["Date"] + " " + schedule_df["Official_End"]
)

schedule_df["CSV_Start_DT"] = pd.to_datetime(
    schedule_df["Date"] + " " + schedule_df["CSV_Start"]
)

schedule_df["CSV_End_DT"] = pd.to_datetime(
    schedule_df["Date"] + " " + schedule_df["CSV_End"]
)


print("\nOFFICIAL SCHEDULE TABLE")
print("-" * 120)

display(
    schedule_df[
        [
            "GT_Event_ID",
            "Attack_Family",
            "Official_Start_DT",
            "Official_End_DT",
            "CSV_Start_DT",
            "CSV_End_DT",
            "Official_Description"
        ]
    ]
)


# ============================================================
# 3. RECONSTRUCT ALL OBSERVED ATTACK MINUTES DIRECTLY
#    FROM RAW CIC FILES
#
# We deliberately go back to RAW here rather than trusting
# candidate episode merging.
# ============================================================

observed_parts = []

for file in CIC_FILES:

    encoding = file_encodings[file.name]

    df = pd.read_csv(
        file,
        usecols=[" Timestamp", " Label"],
        encoding=encoding,
        low_memory=False
    )

    df = df.dropna(how="all").copy()

    df["Timestamp"] = (
        df[" Timestamp"]
        .astype("string")
        .str.strip()
    )

    df["Label"] = (
        df[" Label"]
        .astype("string")
        .str.strip()
    )

    df["Parsed_Time"] = pd.to_datetime(
        df["Timestamp"],
        dayfirst=True,
        errors="coerce"
    )

    df = df[
        df["Parsed_Time"].notna()
    ].copy()

    df["Minute"] = (
        df["Parsed_Time"]
        .dt.floor("min")
    )

    attack = df[
        df["Label"].str.upper().ne("BENIGN")
    ].copy()

    if len(attack) == 0:
        continue

    obs = (
        attack
        .groupby(["Minute", "Label"])
        .size()
        .reset_index(name="Attack_Flow_Count")
    )

    obs["Segment_ID"] = file.name

    observed_parts.append(obs)


observed_attack_df = pd.concat(
    observed_parts,
    ignore_index=True
)


# ============================================================
# 4. RECONCILE EACH OFFICIAL EVENT AGAINST OBSERVED DATA
# ============================================================

reconciliation_records = []

for _, event in schedule_df.iterrows():

    family = event["Attack_Family"]

    scheduled_start = event["CSV_Start_DT"]
    scheduled_end = event["CSV_End_DT"]

    # All observed minutes for this exact attack family
    family_obs = observed_attack_df[
        observed_attack_df["Label"] == family
    ].copy()

    # Observed attack minutes INSIDE official schedule window
    inside = family_obs[
        (family_obs["Minute"] >= scheduled_start) &
        (family_obs["Minute"] <= scheduled_end)
    ].copy()

    expected_window_minutes = int(
        (
            scheduled_end - scheduled_start
        ).total_seconds() / 60
    ) + 1

    observed_inside_minutes = (
        inside["Minute"]
        .nunique()
    )

    observed_flows_inside = int(
        inside["Attack_Flow_Count"].sum()
    ) if len(inside) else 0

    if len(inside) > 0:

        observed_first = inside["Minute"].min()
        observed_last = inside["Minute"].max()

        first_offset = int(
            (
                observed_first - scheduled_start
            ).total_seconds() / 60
        )

        last_offset = int(
            (
                observed_last - scheduled_end
            ).total_seconds() / 60
        )

        coverage = (
            observed_inside_minutes /
            expected_window_minutes
        )

        status = "MATCHED"

    else:

        observed_first = pd.NaT
        observed_last = pd.NaT

        first_offset = np.nan
        last_offset = np.nan
        coverage = 0.0

        status = "NO_OBSERVED_ATTACK_IN_WINDOW"

    reconciliation_records.append({

        "GT_Event_ID":
            event["GT_Event_ID"],

        "Attack_Family":
            family,

        "Official_Start":
            event["Official_Start_DT"],

        "Official_End":
            event["Official_End_DT"],

        "CSV_Schedule_Start":
            scheduled_start,

        "CSV_Schedule_End":
            scheduled_end,

        "Expected_Window_Minutes":
            expected_window_minutes,

        "Observed_Attack_Minutes_In_Window":
            observed_inside_minutes,

        "Observed_Attack_Flows_In_Window":
            observed_flows_inside,

        "Observed_First_Attack_Minute":
            observed_first,

        "Observed_Last_Attack_Minute":
            observed_last,

        "First_Attack_Offset_Min":
            first_offset,

        "Last_Attack_vs_Schedule_End_Min":
            last_offset,

        "Observed_Minute_Coverage":
            coverage,

        "Match_Status":
            status,

        "Official_Description":
            event["Official_Description"]
    })


reconciliation_df = pd.DataFrame(
    reconciliation_records
)


# ============================================================
# 5. FIND OBSERVED MALICIOUS MINUTES THAT FALL OUTSIDE
#    ALL OFFICIAL WINDOWS FOR THEIR FAMILY
# ============================================================

outside_records = []

for _, obs in observed_attack_df.iterrows():

    minute = obs["Minute"]
    family = obs["Label"]

    relevant_events = schedule_df[
        schedule_df["Attack_Family"] == family
    ]

    matched_event_ids = []

    for _, event in relevant_events.iterrows():

        if (
            event["CSV_Start_DT"]
            <= minute
            <= event["CSV_End_DT"]
        ):

            matched_event_ids.append(
                event["GT_Event_ID"]
            )

    if len(matched_event_ids) == 0:

        outside_records.append({

            "Minute":
                minute,

            "Attack_Family":
                family,

            "Attack_Flow_Count":
                int(obs["Attack_Flow_Count"]),

            "Segment_ID":
                obs["Segment_ID"]
        })


outside_schedule_df = pd.DataFrame(
    outside_records
)


# ============================================================
# 6. SUMMARY OUTPUT
# ============================================================

print("\n" + "=" * 120)
print("OFFICIAL EVENT ↔ OBSERVED FLOW RECONCILIATION")
print("=" * 120)

display(
    reconciliation_df[
        [
            "GT_Event_ID",
            "Attack_Family",
            "CSV_Schedule_Start",
            "CSV_Schedule_End",
            "Observed_First_Attack_Minute",
            "Observed_Last_Attack_Minute",
            "Observed_Attack_Minutes_In_Window",
            "Expected_Window_Minutes",
            "Observed_Minute_Coverage",
            "First_Attack_Offset_Min",
            "Match_Status"
        ]
    ]
)


print("\n" + "=" * 120)
print("MATCH STATUS COUNTS")
print("=" * 120)

print(
    reconciliation_df[
        "Match_Status"
    ].value_counts(
        dropna=False
    ).to_string()
)


print("\n" + "=" * 120)
print("OBSERVED MALICIOUS MINUTES OUTSIDE OFFICIAL WINDOWS")
print("=" * 120)

if len(outside_schedule_df) == 0:

    print(
        "✓ Every observed malicious minute falls "
        "inside at least one official family schedule."
    )

else:

    print(
        f"Outside-schedule minute records: "
        f"{len(outside_schedule_df):,}"
    )

    outside_summary = (
        outside_schedule_df
        .groupby("Attack_Family")
        .agg(
            Outside_Minutes=("Minute", "nunique"),
            Outside_Attack_Flows=(
                "Attack_Flow_Count",
                "sum"
            ),
            First_Outside=("Minute", "min"),
            Last_Outside=("Minute", "max")
        )
        .sort_values(
            "Outside_Minutes",
            ascending=False
        )
    )

    display(outside_summary)

    print("\nFirst 50 outside-schedule records:")

    display(
        outside_schedule_df
        .sort_values(
            ["Attack_Family", "Minute"]
        )
        .head(50)
    )


# ============================================================
# 7. FAMILY-LEVEL COVERAGE
# ============================================================

print("\n" + "=" * 120)
print("FAMILY-LEVEL SCHEDULE COVERAGE")
print("=" * 120)

family_reconciliation = (
    reconciliation_df
    .groupby("Attack_Family")
    .agg(
        Official_Events=(
            "GT_Event_ID",
            "count"
        ),

        Matched_Events=(
            "Match_Status",
            lambda x:
                (x == "MATCHED").sum()
        ),

        Observed_Attack_Minutes_In_Window=(
            "Observed_Attack_Minutes_In_Window",
            "sum"
        ),

        Observed_Attack_Flows_In_Window=(
            "Observed_Attack_Flows_In_Window",
            "sum"
        )
    )
)

display(family_reconciliation)


# ============================================================
# 8. SAVE ARTIFACTS
# ============================================================

schedule_df.to_csv(
    METADATA /
    "cic_official_attack_schedule.csv",
    index=False
)

reconciliation_df.to_csv(
    CIC_AUDIT /
    "cic_official_schedule_reconciliation.csv",
    index=False
)

outside_schedule_df.to_csv(
    CIC_AUDIT /
    "cic_outside_schedule_attack_minutes.csv",
    index=False
)

print("\n✓ Official CIC schedule saved")
print("✓ Schedule reconciliation saved")
print("✓ Outside-schedule malicious minutes saved")

In [ ]:
# ==================================================================================================
# PHASE 9.13 — CIC ATTACK BOUNDARY FORENSICS
# ==================================================================================================
#
# PURPOSE
# -------
# 9.12 showed that:
#
#   1. Official attack windows and observed malicious-flow minutes
#      are not always identical.
#
#   2. Some attack families contain malicious flows outside the
#      official windows.
#
#   3. Some official windows contain only sparse malicious minutes.
#
# Before defining forecasting events / attack onset, we therefore
# investigate the temporal structure around every official event.
#
#
# IMPORTANT
# ---------
# This cell:
#
#   ✓ DOES inspect raw CIC labels and timestamps
#   ✓ DOES identify contiguous observed malicious-minute clusters
#   ✓ DOES inspect gaps inside official windows
#   ✓ DOES inspect ±15 minutes around official boundaries
#   ✓ DOES measure attack-flow density
#
#   ✗ DOES NOT modify raw files
#   ✗ DOES NOT redefine attack labels
#   ✗ DOES NOT choose final attack onset
#   ✗ DOES NOT create forecasting targets
#   ✗ DOES NOT invent a gap-merging threshold
#
# ==================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 120)
print("PHASE 9.13 — CIC ATTACK BOUNDARY FORENSICS")
print("=" * 120)


# ==================================================================================================
# 0. SAFETY CHECK — REQUIRED OBJECTS FROM 9.12
# ==================================================================================================

required_objects = [
    "schedule_df",
    "observed_attack_df",
    "CIC_FILES",
    "file_encodings",
    "CIC_AUDIT"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Missing required objects from earlier cells: "
        + ", ".join(missing_objects)
        + "\nRun the previous audit cells, especially Phase 9.12, first."
    )

print("\n✓ Required 9.12 objects found")


# ==================================================================================================
# 1. RECONSTRUCT COMPLETE MINUTE-LEVEL CIC FLOW TABLE
# ==================================================================================================
#
# We need more than malicious minutes now.
#
# For every minute we want:
#
#   Total_Flows
#   Benign_Flows
#   Malicious_Flows
#
# We also preserve attack-family counts separately.
#
# This allows us to inspect what happens BEFORE and AFTER
# official attack boundaries.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13A — RECONSTRUCTING COMPLETE CIC MINUTE FLOW TABLE")
print("=" * 120)

minute_parts = []
family_parts = []

for file in CIC_FILES:

    encoding = file_encodings[file.name]

    print(f"\nProcessing: {file.name}")

    df = pd.read_csv(
        file,
        usecols=[" Timestamp", " Label"],
        encoding=encoding,
        low_memory=False
    )

    physical_rows = len(df)

    # Remove fully empty structural rows
    df = df.dropna(how="all").copy()

    usable_rows = len(df)

    # Clean strings
    df["Timestamp"] = (
        df[" Timestamp"]
        .astype("string")
        .str.strip()
    )

    df["Label"] = (
        df[" Label"]
        .astype("string")
        .str.strip()
    )

    # Parse timestamp
    df["Parsed_Time"] = pd.to_datetime(
        df["Timestamp"],
        dayfirst=True,
        errors="coerce"
    )

    invalid_ts = int(
        df["Parsed_Time"].isna().sum()
    )

    # Keep only valid timestamps
    df = df[
        df["Parsed_Time"].notna()
    ].copy()

    df["Minute"] = (
        df["Parsed_Time"]
        .dt.floor("min")
    )

    df["Is_Attack"] = (
        df["Label"]
        .str.upper()
        .ne("BENIGN")
        .astype(int)
    )

    # ------------------------------------------------------------------
    # Overall minute-level traffic
    # ------------------------------------------------------------------

    minute_summary = (
        df
        .groupby("Minute")
        .agg(
            Total_Flows=("Label", "size"),
            Malicious_Flows=("Is_Attack", "sum")
        )
        .reset_index()
    )

    minute_summary["Benign_Flows"] = (
        minute_summary["Total_Flows"]
        - minute_summary["Malicious_Flows"]
    )

    minute_summary["Attack_State"] = (
        minute_summary["Malicious_Flows"] > 0
    ).astype(int)

    minute_summary["Segment_ID"] = file.name

    minute_parts.append(
        minute_summary
    )

    # ------------------------------------------------------------------
    # Attack-family minute counts
    # ------------------------------------------------------------------

    malicious = df[
        df["Is_Attack"] == 1
    ].copy()

    if len(malicious) > 0:

        family_summary = (
            malicious
            .groupby(["Minute", "Label"])
            .size()
            .reset_index(
                name="Attack_Flow_Count"
            )
        )

        family_summary["Segment_ID"] = file.name

        family_parts.append(
            family_summary
        )

    print(
        f"  physical rows : {physical_rows:,}"
    )

    print(
        f"  usable rows   : {usable_rows:,}"
    )

    print(
        f"  invalid time  : {invalid_ts:,}"
    )

    print(
        f"  minute bins   : {len(minute_summary):,}"
    )


all_minutes_df = pd.concat(
    minute_parts,
    ignore_index=True
)

all_family_minutes_df = pd.concat(
    family_parts,
    ignore_index=True
)

all_minutes_df = (
    all_minutes_df
    .sort_values(
        ["Segment_ID", "Minute"]
    )
    .reset_index(drop=True)
)

all_family_minutes_df = (
    all_family_minutes_df
    .sort_values(
        ["Label", "Minute"]
    )
    .reset_index(drop=True)
)

print("\n✓ Complete minute-level CIC traffic reconstructed")

print(
    f"Minute records        : "
    f"{len(all_minutes_df):,}"
)

print(
    f"Attack-family records : "
    f"{len(all_family_minutes_df):,}"
)


# ==================================================================================================
# 2. BUILD OBSERVED MALICIOUS-MINUTE CLUSTERS
# ==================================================================================================
#
# IMPORTANT:
#
# A cluster here means STRICTLY consecutive malicious minutes.
#
# Example:
#
#   10:01
#   10:02
#   10:03
#
# = one cluster
#
#
# But:
#
#   10:01
#   10:03
#
# = TWO clusters
#
#
# We are deliberately using a strict 1-minute continuity rule.
#
# This is NOT an event-merging threshold.
#
# It simply describes the observed data exactly.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13B — STRICT OBSERVED MALICIOUS-MINUTE CLUSTERS")
print("=" * 120)

cluster_records = []

for attack_family, family_df in (
    all_family_minutes_df
    .groupby("Label")
):

    # Collapse across files in case necessary
    family_minutes = (
        family_df
        .groupby("Minute")
        ["Attack_Flow_Count"]
        .sum()
        .sort_index()
    )

    minutes = list(
        family_minutes.index
    )

    if len(minutes) == 0:
        continue

    cluster_id = 1
    cluster_start = minutes[0]
    previous = minutes[0]

    cluster_minutes = [
        minutes[0]
    ]

    for current in minutes[1:]:

        gap_minutes = int(
            (
                current - previous
            ).total_seconds() / 60
        )

        # Strict continuity only
        if gap_minutes == 1:

            cluster_minutes.append(
                current
            )

        else:

            flows = int(
                family_minutes.loc[
                    cluster_minutes
                ].sum()
            )

            cluster_records.append({

                "Attack_Family":
                    attack_family,

                "Observed_Cluster_ID":
                    f"{attack_family}_C{cluster_id:03d}",

                "Cluster_Start":
                    cluster_start,

                "Cluster_End":
                    previous,

                "Observed_Minutes":
                    len(cluster_minutes),

                "Attack_Flows":
                    flows,

                "Gap_To_Next_Cluster_Min":
                    gap_minutes - 1
            })

            cluster_id += 1

            cluster_start = current

            cluster_minutes = [
                current
            ]

        previous = current

    # Final cluster
    flows = int(
        family_minutes.loc[
            cluster_minutes
        ].sum()
    )

    cluster_records.append({

        "Attack_Family":
            attack_family,

        "Observed_Cluster_ID":
            f"{attack_family}_C{cluster_id:03d}",

        "Cluster_Start":
            cluster_start,

        "Cluster_End":
            previous,

        "Observed_Minutes":
            len(cluster_minutes),

        "Attack_Flows":
            flows,

        "Gap_To_Next_Cluster_Min":
            np.nan
    })


cluster_df = pd.DataFrame(
    cluster_records
)

cluster_df = (
    cluster_df
    .sort_values(
        [
            "Attack_Family",
            "Cluster_Start"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"\nStrict malicious-minute clusters found: "
    f"{len(cluster_df):,}"
)

display(
    cluster_df
)


# ==================================================================================================
# 3. MAP EACH OBSERVED CLUSTER AGAINST OFFICIAL WINDOWS
# ==================================================================================================
#
# Classify each observed cluster as:
#
#   INSIDE
#   OVERLAPS_START
#   OVERLAPS_END
#   SPANS_WINDOW
#   OUTSIDE_BEFORE
#   OUTSIDE_AFTER
#   NO_OFFICIAL_EVENT_FOR_FAMILY
#
# We DO NOT delete or merge anything.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13C — CLUSTER ↔ OFFICIAL WINDOW RELATIONSHIP")
print("=" * 120)

cluster_relation_records = []

for _, cluster in cluster_df.iterrows():

    family = cluster["Attack_Family"]

    c_start = cluster["Cluster_Start"]
    c_end = cluster["Cluster_End"]

    official_events = schedule_df[
        schedule_df["Attack_Family"] == family
    ].copy()

    if len(official_events) == 0:

        cluster_relation_records.append({

            **cluster.to_dict(),

            "GT_Event_ID":
                None,

            "Official_Window_Start":
                pd.NaT,

            "Official_Window_End":
                pd.NaT,

            "Relationship":
                "NO_OFFICIAL_EVENT_FOR_FAMILY",

            "Distance_To_Window_Min":
                np.nan
        })

        continue

    relationships = []

    for _, event in official_events.iterrows():

        o_start = event["CSV_Start_DT"]
        o_end = event["CSV_End_DT"]

        # --------------------------------------------------------------
        # Relationship classification
        # --------------------------------------------------------------

        if (
            c_start >= o_start
            and c_end <= o_end
        ):

            relationship = "INSIDE"

            distance = 0

        elif (
            c_start < o_start
            and c_end >= o_start
            and c_end <= o_end
        ):

            relationship = "OVERLAPS_START"

            distance = int(
                (
                    o_start - c_start
                ).total_seconds() / 60
            )

        elif (
            c_start >= o_start
            and c_start <= o_end
            and c_end > o_end
        ):

            relationship = "OVERLAPS_END"

            distance = int(
                (
                    c_end - o_end
                ).total_seconds() / 60
            )

        elif (
            c_start < o_start
            and c_end > o_end
        ):

            relationship = "SPANS_WINDOW"

            distance = 0

        elif c_end < o_start:

            relationship = "OUTSIDE_BEFORE"

            distance = int(
                (
                    o_start - c_end
                ).total_seconds() / 60
            )

        else:

            relationship = "OUTSIDE_AFTER"

            distance = int(
                (
                    c_start - o_end
                ).total_seconds() / 60
            )

        relationships.append({

            **cluster.to_dict(),

            "GT_Event_ID":
                event["GT_Event_ID"],

            "Official_Window_Start":
                o_start,

            "Official_Window_End":
                o_end,

            "Relationship":
                relationship,

            "Distance_To_Window_Min":
                distance
        })

    # --------------------------------------------------------------
    # Keep most relevant official event for this observed cluster.
    #
    # Priority:
    # overlap/inside first;
    # otherwise nearest official event.
    # --------------------------------------------------------------

    relationship_df = pd.DataFrame(
        relationships
    )

    overlap_types = [
        "INSIDE",
        "OVERLAPS_START",
        "OVERLAPS_END",
        "SPANS_WINDOW"
    ]

    overlapping = relationship_df[
        relationship_df[
            "Relationship"
        ].isin(overlap_types)
    ]

    if len(overlapping) > 0:

        chosen = (
            overlapping
            .sort_values(
                "Distance_To_Window_Min"
            )
            .iloc[0]
        )

    else:

        chosen = (
            relationship_df
            .sort_values(
                "Distance_To_Window_Min"
            )
            .iloc[0]
        )

    cluster_relation_records.append(
        chosen.to_dict()
    )


cluster_relation_df = pd.DataFrame(
    cluster_relation_records
)

print("\nRELATIONSHIP COUNTS")
print("-" * 80)

print(
    cluster_relation_df[
        "Relationship"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

print("\nFULL CLUSTER RELATIONSHIP TABLE")
print("-" * 80)

display(
    cluster_relation_df[
        [
            "Attack_Family",
            "Observed_Cluster_ID",
            "Cluster_Start",
            "Cluster_End",
            "Observed_Minutes",
            "Attack_Flows",
            "GT_Event_ID",
            "Official_Window_Start",
            "Official_Window_End",
            "Relationship",
            "Distance_To_Window_Min"
        ]
    ]
)


# ==================================================================================================
# 4. INTERNAL GAP FORENSICS INSIDE EACH OFFICIAL WINDOW
# ==================================================================================================
#
# For every official attack event:
#
#   Which scheduled minutes actually contain same-family
#   malicious flows?
#
#   Which scheduled minutes contain NONE?
#
# Then identify strict consecutive missing-minute runs.
#
# Again:
#
# Missing attack labels DO NOT automatically mean that
# the official schedule is wrong.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13D — INTERNAL GAPS INSIDE OFFICIAL ATTACK WINDOWS")
print("=" * 120)

internal_gap_records = []
event_gap_summary_records = []

for _, event in schedule_df.iterrows():

    event_id = event["GT_Event_ID"]
    family = event["Attack_Family"]

    start = event["CSV_Start_DT"]
    end = event["CSV_End_DT"]

    expected_minutes = pd.date_range(
        start=start,
        end=end,
        freq="min"
    )

    observed = set(
        all_family_minutes_df.loc[
            (
                all_family_minutes_df["Label"]
                == family
            )
            &
            (
                all_family_minutes_df["Minute"]
                >= start
            )
            &
            (
                all_family_minutes_df["Minute"]
                <= end
            ),
            "Minute"
        ]
    )

    missing_minutes = [
        minute
        for minute in expected_minutes
        if minute not in observed
    ]

    event_gap_summary_records.append({

        "GT_Event_ID":
            event_id,

        "Attack_Family":
            family,

        "Window_Start":
            start,

        "Window_End":
            end,

        "Expected_Minutes":
            len(expected_minutes),

        "Observed_Attack_Minutes":
            len(observed),

        "Missing_Attack_Minutes":
            len(missing_minutes),

        "Coverage":
            (
                len(observed)
                / len(expected_minutes)
                if len(expected_minutes)
                else np.nan
            )
    })

    if len(missing_minutes) == 0:
        continue

    # --------------------------------------------------------------
    # Convert missing minutes into strict consecutive runs
    # --------------------------------------------------------------

    gap_start = missing_minutes[0]
    previous = missing_minutes[0]

    gap_minutes = [
        missing_minutes[0]
    ]

    gap_number = 1

    for current in missing_minutes[1:]:

        difference = int(
            (
                current - previous
            ).total_seconds() / 60
        )

        if difference == 1:

            gap_minutes.append(
                current
            )

        else:

            internal_gap_records.append({

                "GT_Event_ID":
                    event_id,

                "Attack_Family":
                    family,

                "Gap_ID":
                    f"{event_id}_G{gap_number:03d}",

                "Gap_Start":
                    gap_start,

                "Gap_End":
                    previous,

                "Gap_Minutes":
                    len(gap_minutes)
            })

            gap_number += 1

            gap_start = current

            gap_minutes = [
                current
            ]

        previous = current

    # final gap
    internal_gap_records.append({

        "GT_Event_ID":
            event_id,

        "Attack_Family":
            family,

        "Gap_ID":
            f"{event_id}_G{gap_number:03d}",

        "Gap_Start":
            gap_start,

        "Gap_End":
            previous,

        "Gap_Minutes":
            len(gap_minutes)
    })


event_gap_summary_df = pd.DataFrame(
    event_gap_summary_records
)

internal_gap_df = pd.DataFrame(
    internal_gap_records
)


print("\nEVENT COVERAGE SUMMARY")
print("-" * 100)

display(
    event_gap_summary_df
)


print("\nSTRICT MISSING-MINUTE RUNS")
print("-" * 100)

if len(internal_gap_df) == 0:

    print(
        "✓ No missing attack-labelled minutes "
        "inside official windows."
    )

else:

    display(
        internal_gap_df
        .sort_values(
            [
                "GT_Event_ID",
                "Gap_Start"
            ]
        )
    )


# ==================================================================================================
# 5. ±15 MINUTE BOUNDARY FORENSICS
# ==================================================================================================
#
# For every official event, inspect:
#
#       -15 ... -1
#       START
#       ...
#       END
#       +1 ... +15
#
#
# For every minute we record:
#
#   Total_Flows
#   Benign_Flows
#   All_Malicious_Flows
#   Same_Family_Attack_Flows
#   Other_Attack_Flows
#
#
# Relative_Position:
#
#   BEFORE
#   INSIDE
#   AFTER
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13E — ±15 MINUTE OFFICIAL BOUNDARY FORENSICS")
print("=" * 120)

boundary_records = []

for _, event in schedule_df.iterrows():

    event_id = event["GT_Event_ID"]
    family = event["Attack_Family"]

    start = event["CSV_Start_DT"]
    end = event["CSV_End_DT"]

    inspection_start = (
        start - pd.Timedelta(minutes=15)
    )

    inspection_end = (
        end + pd.Timedelta(minutes=15)
    )

    inspection_minutes = pd.date_range(
        inspection_start,
        inspection_end,
        freq="min"
    )

    for minute in inspection_minutes:

        # --------------------------------------------------------------
        # Overall flow counts
        # --------------------------------------------------------------

        overall_rows = all_minutes_df[
            all_minutes_df["Minute"] == minute
        ]

        total_flows = int(
            overall_rows[
                "Total_Flows"
            ].sum()
        )

        benign_flows = int(
            overall_rows[
                "Benign_Flows"
            ].sum()
        )

        malicious_flows = int(
            overall_rows[
                "Malicious_Flows"
            ].sum()
        )

        # --------------------------------------------------------------
        # Same-family malicious flows
        # --------------------------------------------------------------

        same_family_rows = (
            all_family_minutes_df[
                (
                    all_family_minutes_df[
                        "Minute"
                    ] == minute
                )
                &
                (
                    all_family_minutes_df[
                        "Label"
                    ] == family
                )
            ]
        )

        same_family_flows = int(
            same_family_rows[
                "Attack_Flow_Count"
            ].sum()
        )

        other_attack_flows = (
            malicious_flows
            - same_family_flows
        )

        # --------------------------------------------------------------
        # Position relative to official window
        # --------------------------------------------------------------

        if minute < start:

            position = "BEFORE"

            relative_minute = int(
                (
                    minute - start
                ).total_seconds() / 60
            )

        elif minute > end:

            position = "AFTER"

            relative_minute = int(
                (
                    minute - end
                ).total_seconds() / 60
            )

        else:

            position = "INSIDE"

            relative_minute = int(
                (
                    minute - start
                ).total_seconds() / 60
            )

        boundary_records.append({

            "GT_Event_ID":
                event_id,

            "Attack_Family":
                family,

            "Official_Start":
                start,

            "Official_End":
                end,

            "Minute":
                minute,

            "Relative_Position":
                position,

            "Relative_Minute":
                relative_minute,

            "Total_Flows":
                total_flows,

            "Benign_Flows":
                benign_flows,

            "All_Malicious_Flows":
                malicious_flows,

            "Same_Family_Attack_Flows":
                same_family_flows,

            "Other_Attack_Flows":
                other_attack_flows
        })


boundary_df = pd.DataFrame(
    boundary_records
)


# ==================================================================================================
# 6. BOUNDARY SUMMARY PER EVENT
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13F — EVENT BOUNDARY SUMMARY")
print("=" * 120)

boundary_summary_records = []

for _, event in schedule_df.iterrows():

    event_id = event["GT_Event_ID"]
    family = event["Attack_Family"]

    temp = boundary_df[
        boundary_df["GT_Event_ID"]
        == event_id
    ].copy()

    before = temp[
        temp["Relative_Position"]
        == "BEFORE"
    ]

    inside = temp[
        temp["Relative_Position"]
        == "INSIDE"
    ]

    after = temp[
        temp["Relative_Position"]
        == "AFTER"
    ]

    # Same-family malicious minutes
    before_attack = before[
        before[
            "Same_Family_Attack_Flows"
        ] > 0
    ]

    inside_attack = inside[
        inside[
            "Same_Family_Attack_Flows"
        ] > 0
    ]

    after_attack = after[
        after[
            "Same_Family_Attack_Flows"
        ] > 0
    ]

    first_inside = (
        inside_attack["Minute"].min()
        if len(inside_attack)
        else pd.NaT
    )

    last_inside = (
        inside_attack["Minute"].max()
        if len(inside_attack)
        else pd.NaT
    )

    boundary_summary_records.append({

        "GT_Event_ID":
            event_id,

        "Attack_Family":
            family,

        "Official_Start":
            event["CSV_Start_DT"],

        "Official_End":
            event["CSV_End_DT"],

        "Same_Family_Minutes_Pre15":
            int(
                (
                    before[
                        "Same_Family_Attack_Flows"
                    ] > 0
                ).sum()
            ),

        "Same_Family_Flows_Pre15":
            int(
                before[
                    "Same_Family_Attack_Flows"
                ].sum()
            ),

        "Same_Family_Minutes_Inside":
            int(
                (
                    inside[
                        "Same_Family_Attack_Flows"
                    ] > 0
                ).sum()
            ),

        "Same_Family_Flows_Inside":
            int(
                inside[
                    "Same_Family_Attack_Flows"
                ].sum()
            ),

        "Same_Family_Minutes_Post15":
            int(
                (
                    after[
                        "Same_Family_Attack_Flows"
                    ] > 0
                ).sum()
            ),

        "Same_Family_Flows_Post15":
            int(
                after[
                    "Same_Family_Attack_Flows"
                ].sum()
            ),

        "First_Observed_Inside":
            first_inside,

        "Last_Observed_Inside":
            last_inside,

        "First_Observed_Offset_Min":
            (
                int(
                    (
                        first_inside
                        - event["CSV_Start_DT"]
                    ).total_seconds() / 60
                )
                if pd.notna(first_inside)
                else np.nan
            )
    })


boundary_summary_df = pd.DataFrame(
    boundary_summary_records
)

display(
    boundary_summary_df
)


# ==================================================================================================
# 7. PRINT HUMAN-READABLE EVENT FORENSICS
# ==================================================================================================
#
# This section is intentionally verbose.
#
# We want to SEE what happens around every event instead
# of relying only on aggregate statistics.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13G — HUMAN-READABLE EVENT FORENSICS")
print("=" * 120)

for _, event in schedule_df.iterrows():

    event_id = event["GT_Event_ID"]
    family = event["Attack_Family"]

    start = event["CSV_Start_DT"]
    end = event["CSV_End_DT"]

    print("\n")
    print("=" * 120)

    print(
        f"{event_id} | {family}"
    )

    print("-" * 120)

    print(
        f"Official CSV-clock window : "
        f"{start} → {end}"
    )

    # --------------------------------------------------------------
    # Associated strict clusters
    # --------------------------------------------------------------

    related_clusters = cluster_relation_df[
        cluster_relation_df[
            "GT_Event_ID"
        ] == event_id
    ]

    if len(related_clusters):

        print("\nObserved strict clusters associated with this event:")

        for _, row in related_clusters.iterrows():

            print(
                f"  {row['Observed_Cluster_ID']} | "
                f"{row['Cluster_Start']} → "
                f"{row['Cluster_End']} | "
                f"{int(row['Observed_Minutes'])} min | "
                f"{int(row['Attack_Flows']):,} flows | "
                f"{row['Relationship']}"
            )

    else:

        print(
            "\nObserved strict clusters associated with this event: NONE"
        )

    # --------------------------------------------------------------
    # Internal gaps
    # --------------------------------------------------------------

    gaps = internal_gap_df[
        internal_gap_df[
            "GT_Event_ID"
        ] == event_id
    ] if len(internal_gap_df) else pd.DataFrame()

    if len(gaps):

        print("\nMissing same-family attack-minute runs INSIDE official window:")

        for _, gap in gaps.iterrows():

            print(
                f"  {gap['Gap_Start']} → "
                f"{gap['Gap_End']} "
                f"({int(gap['Gap_Minutes'])} min)"
            )

    else:

        print(
            "\nMissing same-family attack-minute runs "
            "INSIDE official window: NONE"
        )

    # --------------------------------------------------------------
    # Show only interesting boundary minutes
    #
    # Interesting =
    #   same-family attack exists
    #   OR other attack exists
    #   OR exact official start/end
    # --------------------------------------------------------------

    event_boundary = boundary_df[
        boundary_df[
            "GT_Event_ID"
        ] == event_id
    ].copy()

    interesting = event_boundary[
        (
            event_boundary[
                "All_Malicious_Flows"
            ] > 0
        )
        |
        (
            event_boundary[
                "Minute"
            ].isin([start, end])
        )
    ]

    print("\nInteresting ±15-minute boundary rows:")

    if len(interesting):

        print(
            interesting[
                [
                    "Minute",
                    "Relative_Position",
                    "Relative_Minute",
                    "Total_Flows",
                    "Benign_Flows",
                    "All_Malicious_Flows",
                    "Same_Family_Attack_Flows",
                    "Other_Attack_Flows"
                ]
            ]
            .to_string(index=False)
        )

    else:

        print(
            "  No malicious activity in ±15 minute inspection region."
        )


# ==================================================================================================
# 8. SPECIAL FORENSICS — BOT
# ==================================================================================================
#
# Bot produced the largest outside-schedule discrepancy in 9.12.
#
# Here we print ALL observed Bot clusters.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13H — SPECIAL FORENSICS: BOT")
print("=" * 120)

bot_clusters = cluster_relation_df[
    cluster_relation_df[
        "Attack_Family"
    ] == "Bot"
]

if len(bot_clusters):

    display(
        bot_clusters[
            [
                "Observed_Cluster_ID",
                "Cluster_Start",
                "Cluster_End",
                "Observed_Minutes",
                "Attack_Flows",
                "GT_Event_ID",
                "Official_Window_Start",
                "Official_Window_End",
                "Relationship",
                "Distance_To_Window_Min"
            ]
        ]
    )

else:

    print("No Bot clusters found.")


# ==================================================================================================
# 9. SPECIAL FORENSICS — INFILTRATION
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13I — SPECIAL FORENSICS: INFILTRATION")
print("=" * 120)

inf_clusters = cluster_relation_df[
    cluster_relation_df[
        "Attack_Family"
    ] == "Infiltration"
]

if len(inf_clusters):

    display(
        inf_clusters[
            [
                "Observed_Cluster_ID",
                "Cluster_Start",
                "Cluster_End",
                "Observed_Minutes",
                "Attack_Flows",
                "GT_Event_ID",
                "Official_Window_Start",
                "Official_Window_End",
                "Relationship",
                "Distance_To_Window_Min"
            ]
        ]
    )

else:

    print(
        "No Infiltration clusters found."
    )


# ==================================================================================================
# 10. SPECIAL FORENSICS — PORTSCAN
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13J — SPECIAL FORENSICS: PORTSCAN")
print("=" * 120)

port_clusters = cluster_relation_df[
    cluster_relation_df[
        "Attack_Family"
    ] == "PortScan"
]

if len(port_clusters):

    display(
        port_clusters[
            [
                "Observed_Cluster_ID",
                "Cluster_Start",
                "Cluster_End",
                "Observed_Minutes",
                "Attack_Flows",
                "GT_Event_ID",
                "Official_Window_Start",
                "Official_Window_End",
                "Relationship",
                "Distance_To_Window_Min"
            ]
        ]
    )

else:

    print(
        "No PortScan clusters found."
    )


# ==================================================================================================
# 11. FAMILY-LEVEL FORENSIC SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13K — FAMILY-LEVEL FORENSIC SUMMARY")
print("=" * 120)

family_cluster_summary = (
    cluster_df
    .groupby("Attack_Family")
    .agg(
        Strict_Clusters=(
            "Observed_Cluster_ID",
            "count"
        ),

        First_Observed_Minute=(
            "Cluster_Start",
            "min"
        ),

        Last_Observed_Minute=(
            "Cluster_End",
            "max"
        ),

        Total_Attack_Flows=(
            "Attack_Flows",
            "sum"
        ),

        Total_Attack_Minutes=(
            "Observed_Minutes",
            "sum"
        ),

        Largest_Strict_Cluster_Min=(
            "Observed_Minutes",
            "max"
        )
    )
    .sort_values(
        "Strict_Clusters",
        ascending=False
    )
)

display(
    family_cluster_summary
)


# ==================================================================================================
# 12. FLAG EVENTS THAT REQUIRE ONSET-SEMANTICS REVIEW
# ==================================================================================================
#
# IMPORTANT:
#
# These are DIAGNOSTIC FLAGS.
#
# They do NOT mean an event is invalid.
#
#
# Reasons:
#
#   PRE_WINDOW_ATTACK_ACTIVITY
#   POST_WINDOW_ATTACK_ACTIVITY
#   INTERNAL_ATTACK_GAPS
#   NO_ATTACK_LABEL_IN_OFFICIAL_WINDOW
#
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13L — EVENTS REQUIRING ONSET-SEMANTICS REVIEW")
print("=" * 120)

review_records = []

for _, summary in boundary_summary_df.iterrows():

    event_id = summary["GT_Event_ID"]

    gap_row = event_gap_summary_df[
        event_gap_summary_df[
            "GT_Event_ID"
        ] == event_id
    ].iloc[0]

    reasons = []

    if (
        summary[
            "Same_Family_Minutes_Pre15"
        ] > 0
    ):

        reasons.append(
            "PRE_WINDOW_ATTACK_ACTIVITY"
        )

    if (
        summary[
            "Same_Family_Minutes_Post15"
        ] > 0
    ):

        reasons.append(
            "POST_WINDOW_ATTACK_ACTIVITY"
        )

    if (
        gap_row[
            "Observed_Attack_Minutes"
        ] == 0
    ):

        reasons.append(
            "NO_ATTACK_LABEL_IN_OFFICIAL_WINDOW"
        )

    elif (
        gap_row[
            "Missing_Attack_Minutes"
        ] > 0
    ):

        reasons.append(
            "INTERNAL_ATTACK_GAPS"
        )

    review_records.append({

        "GT_Event_ID":
            event_id,

        "Attack_Family":
            summary["Attack_Family"],

        "Needs_Review":
            len(reasons) > 0,

        "Review_Reasons":
            " | ".join(reasons)
            if reasons
            else "NONE",

        "Pre15_Attack_Minutes":
            summary[
                "Same_Family_Minutes_Pre15"
            ],

        "Inside_Attack_Minutes":
            summary[
                "Same_Family_Minutes_Inside"
            ],

        "Post15_Attack_Minutes":
            summary[
                "Same_Family_Minutes_Post15"
            ],

        "Official_Window_Coverage":
            gap_row[
                "Coverage"
            ]
    })


review_df = pd.DataFrame(
    review_records
)

display(
    review_df
)


print("\nREVIEW COUNTS")
print("-" * 80)

print(
    review_df[
        "Needs_Review"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ==================================================================================================
# 13. SAVE ALL 9.13 ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("9.13M — SAVING FORENSIC ARTIFACTS")
print("=" * 120)

all_minutes_df.to_csv(
    CIC_AUDIT /
    "cic_complete_minute_flow_table.csv",
    index=False
)

all_family_minutes_df.to_csv(
    CIC_AUDIT /
    "cic_attack_family_minute_counts.csv",
    index=False
)

cluster_df.to_csv(
    CIC_AUDIT /
    "cic_strict_attack_minute_clusters.csv",
    index=False
)

cluster_relation_df.to_csv(
    CIC_AUDIT /
    "cic_cluster_official_window_relationships.csv",
    index=False
)

event_gap_summary_df.to_csv(
    CIC_AUDIT /
    "cic_official_window_gap_summary.csv",
    index=False
)

internal_gap_df.to_csv(
    CIC_AUDIT /
    "cic_internal_attack_minute_gaps.csv",
    index=False
)

boundary_df.to_csv(
    CIC_AUDIT /
    "cic_15min_boundary_forensics.csv",
    index=False
)

boundary_summary_df.to_csv(
    CIC_AUDIT /
    "cic_boundary_forensic_summary.csv",
    index=False
)

family_cluster_summary.to_csv(
    CIC_AUDIT /
    "cic_family_cluster_summary.csv"
)

review_df.to_csv(
    CIC_AUDIT /
    "cic_events_requiring_onset_review.csv",
    index=False
)

print("✓ cic_complete_minute_flow_table.csv")
print("✓ cic_attack_family_minute_counts.csv")
print("✓ cic_strict_attack_minute_clusters.csv")
print("✓ cic_cluster_official_window_relationships.csv")
print("✓ cic_official_window_gap_summary.csv")
print("✓ cic_internal_attack_minute_gaps.csv")
print("✓ cic_15min_boundary_forensics.csv")
print("✓ cic_boundary_forensic_summary.csv")
print("✓ cic_family_cluster_summary.csv")
print("✓ cic_events_requiring_onset_review.csv")


# ==================================================================================================
# FINAL
# ==================================================================================================

print("\n" + "=" * 120)
print("PHASE 9.13 COMPLETE")
print("=" * 120)

print(
    "\nNo attack boundaries were changed."
)

print(
    "No forecasting targets were created."
)

print(
    "No arbitrary event-merging threshold was introduced."
)

print(
    "\nNext phase:"
)

print(
    "Use these forensic results to define defensible "
    "forecasting-event and attack-onset semantics."
)

In [ ]:
# ==========================================================================================
# PHASE 9.14 — CIC ATTACK EVENT / ONSET SEMANTICS RESOLUTION
# ==========================================================================================
#
# PURPOSE
# -------
# Phase 9.13 reconstructed the raw minute-level CIC attack chronology and deliberately
# did NOT:
#   - redefine attack boundaries
#   - create forecasting targets
#   - introduce an arbitrary event-merging threshold
#
# Phase 9.14 asks:
#
#   "Which strict malicious-minute clusters are plausibly parts of the same physical
#    attack occurrence, and which represent defensibly separate attack occurrences?"
#
# IMPORTANT
# ---------
# This cell does NOT modify the forecasting dataset.
# This cell does NOT modify TRAIN / VAL / TEST.
# This cell does NOT create final forecasting labels.
#
# It creates a defensible CANDIDATE event map + an uncertainty report.
# ==========================================================================================

import os
import ast
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

print("=" * 120)
print("PHASE 9.14 — CIC ATTACK EVENT / ONSET SEMANTICS RESOLUTION")
print("=" * 120)


# ==========================================================================================
# 9.14A — LOCATE 9.13 ARTIFACTS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14A — LOCATING PHASE 9.13 ARTIFACTS")
print("=" * 120)

ROOT_CANDIDATES = [
    Path("/content/network-attack-forecasting"),
    Path("/content/drive/MyDrive/network-attack-forecasting"),
    Path.cwd(),
]

REQUIRED_FILES = [
    "cic_complete_minute_flow_table.csv",
    "cic_attack_family_minute_counts.csv",
    "cic_strict_attack_minute_clusters.csv",
    "cic_cluster_official_window_relationships.csv",
    "cic_official_window_gap_summary.csv",
    "cic_internal_attack_minute_gaps.csv",
    "cic_15min_boundary_forensics.csv",
    "cic_boundary_forensic_summary.csv",
    "cic_family_cluster_summary.csv",
    "cic_events_requiring_onset_review.csv",
]


def find_file(filename):
    hits = []

    for root in ROOT_CANDIDATES:
        if not root.exists():
            continue

        try:
            hits.extend(root.rglob(filename))
        except Exception:
            pass

    # remove duplicates
    hits = list(dict.fromkeys(map(str, hits)))

    if len(hits) == 0:
        return None

    # Prefer metadata / Dataset_preparation paths when multiple versions exist
    def score(path):
        p = path.lower()
        s = 0
        if "dataset_preparation" in p:
            s += 10
        if "metadata" in p:
            s += 5
        if "forensic" in p:
            s += 3
        return -s, len(path)

    hits = sorted(hits, key=score)
    return Path(hits[0])


paths = {}

for f in REQUIRED_FILES:
    p = find_file(f)
    paths[f] = p

    if p is None:
        print(f"✗ NOT FOUND: {f}")
    else:
        print(f"✓ {f}")
        print(f"    {p}")

missing = [f for f, p in paths.items() if p is None]

if missing:
    raise FileNotFoundError(
        "\nMissing Phase 9.13 artifacts:\n  - "
        + "\n  - ".join(missing)
        + "\n\nRun Phase 9.13 first or check where its CSV files were saved."
    )

print("\n✓ All Phase 9.13 forensic artifacts located.")


# ==========================================================================================
# 9.14B — LOAD ARTIFACTS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14B — LOADING FORENSIC ARTIFACTS")
print("=" * 120)

minute_df = pd.read_csv(paths["cic_complete_minute_flow_table.csv"])
family_minute_df = pd.read_csv(paths["cic_attack_family_minute_counts.csv"])
cluster_df = pd.read_csv(paths["cic_strict_attack_minute_clusters.csv"])
relationship_df = pd.read_csv(paths["cic_cluster_official_window_relationships.csv"])
official_gap_df = pd.read_csv(paths["cic_official_window_gap_summary.csv"])
internal_gap_df = pd.read_csv(paths["cic_internal_attack_minute_gaps.csv"])
boundary15_df = pd.read_csv(paths["cic_15min_boundary_forensics.csv"])
boundary_summary_df = pd.read_csv(paths["cic_boundary_forensic_summary.csv"])
family_summary_df = pd.read_csv(paths["cic_family_cluster_summary.csv"])
review_df = pd.read_csv(paths["cic_events_requiring_onset_review.csv"])

tables = {
    "minute_df": minute_df,
    "family_minute_df": family_minute_df,
    "cluster_df": cluster_df,
    "relationship_df": relationship_df,
    "official_gap_df": official_gap_df,
    "internal_gap_df": internal_gap_df,
    "boundary15_df": boundary15_df,
    "boundary_summary_df": boundary_summary_df,
    "family_summary_df": family_summary_df,
    "review_df": review_df,
}

for name, df in tables.items():
    print(f"{name:24s}: {df.shape}")

print("\n✓ All forensic tables loaded.")


# ==========================================================================================
# 9.14C — NORMALIZE TIMESTAMP COLUMNS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14C — NORMALIZING TIMESTAMPS")
print("=" * 120)

TIME_HINTS = (
    "time",
    "minute",
    "start",
    "end",
    "onset",
    "boundary",
)

for name, df in tables.items():

    converted = []

    for col in df.columns:

        c = col.lower()

        if any(h in c for h in TIME_HINTS):

            # Don't destroy obvious numeric duration/count columns
            if any(
                token in c
                for token in [
                    "minutes",
                    "minute_count",
                    "gap",
                    "duration",
                    "count",
                    "offset",
                    "distance",
                ]
            ):
                continue

            if df[col].dtype == "object":
                parsed = pd.to_datetime(df[col], errors="coerce")

                # only replace if parsing was meaningfully successful
                if parsed.notna().sum() > 0:
                    df[col] = parsed
                    converted.append(col)

    if converted:
        print(f"{name}: converted {converted}")

print("\n✓ Timestamp normalization complete.")


# ==========================================================================================
# 9.14D — VERIFY STRICT CLUSTER TABLE
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14D — VERIFYING STRICT MALICIOUS-MINUTE CLUSTERS")
print("=" * 120)

required_cluster_cols = [
    "Attack_Family",
    "Observed_Cluster_ID",
    "Cluster_Start",
    "Cluster_End",
    "Observed_Minutes",
    "Attack_Flows",
]

missing_cluster_cols = [
    c for c in required_cluster_cols if c not in cluster_df.columns
]

if missing_cluster_cols:
    raise ValueError(
        "Strict cluster artifact is missing required columns: "
        + str(missing_cluster_cols)
    )

cluster_df["Cluster_Start"] = pd.to_datetime(
    cluster_df["Cluster_Start"], errors="coerce"
)

cluster_df["Cluster_End"] = pd.to_datetime(
    cluster_df["Cluster_End"], errors="coerce"
)

cluster_df = cluster_df.sort_values(
    ["Attack_Family", "Cluster_Start", "Cluster_End"]
).reset_index(drop=True)

# Recompute gap ourselves rather than trusting any prior derived value
cluster_df["Computed_Gap_To_Next_Min"] = np.nan
cluster_df["Next_Cluster_ID"] = None

for family, idx in cluster_df.groupby("Attack_Family", sort=False).groups.items():

    idx = list(idx)

    for pos in range(len(idx) - 1):

        i = idx[pos]
        j = idx[pos + 1]

        current_end = cluster_df.loc[i, "Cluster_End"]
        next_start = cluster_df.loc[j, "Cluster_Start"]

        # If current cluster ends at minute 10:00 and next starts 10:02,
        # the truly empty minute between them is 10:01 => one-minute interruption.
        raw_difference = (
            next_start - current_end
        ).total_seconds() / 60.0

        benign_minutes_between = max(0.0, raw_difference - 1.0)

        cluster_df.loc[i, "Computed_Gap_To_Next_Min"] = benign_minutes_between
        cluster_df.loc[i, "Next_Cluster_ID"] = cluster_df.loc[
            j, "Observed_Cluster_ID"
        ]

print(f"Strict clusters: {len(cluster_df):,}")
print(f"Attack families: {cluster_df['Attack_Family'].nunique():,}")

print("\nClusters by family:")
print(
    cluster_df.groupby("Attack_Family")
    .agg(
        Strict_Clusters=("Observed_Cluster_ID", "nunique"),
        First_Malicious_Minute=("Cluster_Start", "min"),
        Last_Malicious_Minute=("Cluster_End", "max"),
        Malicious_Minutes=("Observed_Minutes", "sum"),
        Attack_Flows=("Attack_Flows", "sum"),
    )
    .sort_values(["Strict_Clusters", "Attack_Flows"], ascending=False)
    .to_string()
)


# ==========================================================================================
# 9.14E — CHARACTERIZE ALL INTER-CLUSTER GAPS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14E — CHARACTERIZING INTER-CLUSTER GAPS")
print("=" * 120)

gap_rows = []

for family, g in cluster_df.groupby("Attack_Family", sort=False):

    g = g.sort_values("Cluster_Start").reset_index(drop=True)

    for i in range(len(g) - 1):

        a = g.iloc[i]
        b = g.iloc[i + 1]

        raw_delta = (
            b["Cluster_Start"] - a["Cluster_End"]
        ).total_seconds() / 60.0

        benign_gap = max(0.0, raw_delta - 1.0)

        gap_rows.append(
            {
                "Attack_Family": family,

                "Left_Cluster_ID": a["Observed_Cluster_ID"],
                "Right_Cluster_ID": b["Observed_Cluster_ID"],

                "Left_Start": a["Cluster_Start"],
                "Left_End": a["Cluster_End"],

                "Right_Start": b["Cluster_Start"],
                "Right_End": b["Cluster_End"],

                "Benign_Gap_Minutes": benign_gap,

                "Left_Observed_Minutes": a["Observed_Minutes"],
                "Right_Observed_Minutes": b["Observed_Minutes"],

                "Left_Attack_Flows": a["Attack_Flows"],
                "Right_Attack_Flows": b["Attack_Flows"],
            }
        )

gap_df = pd.DataFrame(gap_rows)

print(f"Inter-cluster boundaries: {len(gap_df):,}")

if len(gap_df):

    print("\nGap distribution:")
    print(gap_df["Benign_Gap_Minutes"].describe().to_string())

    print("\nExact gap frequencies:")
    print(
        gap_df["Benign_Gap_Minutes"]
        .value_counts()
        .sort_index()
        .head(50)
        .to_string()
    )


# ==========================================================================================
# 9.14F — EXTRACT OFFICIAL-WINDOW EVIDENCE
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14F — EXTRACTING OFFICIAL-WINDOW EVIDENCE")
print("=" * 120)

print("relationship_df columns:")
print(list(relationship_df.columns))

# We deliberately do not assume exact column names beyond obvious identifiers.
# Instead we preserve all relationship evidence and attempt to find useful matching keys.

possible_cluster_cols = [
    c for c in relationship_df.columns
    if "cluster" in c.lower() and "id" in c.lower()
]

possible_family_cols = [
    c for c in relationship_df.columns
    if "family" in c.lower() or "attack_type" in c.lower()
]

possible_window_cols = [
    c for c in relationship_df.columns
    if "window" in c.lower()
]

print("\nDetected cluster identifier columns:", possible_cluster_cols)
print("Detected family columns            :", possible_family_cols)
print("Detected official-window columns   :", possible_window_cols)


# ==========================================================================================
# 9.14G — BUILD CLUSTER → OFFICIAL-WINDOW SIGNATURES
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14G — BUILDING OFFICIAL-WINDOW SIGNATURES")
print("=" * 120)

cluster_id_col = None

for candidate in [
    "Observed_Cluster_ID",
    "Cluster_ID",
    "Strict_Cluster_ID",
]:
    if candidate in relationship_df.columns:
        cluster_id_col = candidate
        break

if cluster_id_col is None and possible_cluster_cols:
    cluster_id_col = possible_cluster_cols[0]

window_id_candidates = [
    c for c in relationship_df.columns
    if (
        "window" in c.lower()
        and (
            "id" in c.lower()
            or "name" in c.lower()
            or "label" in c.lower()
        )
    )
]

window_id_col = window_id_candidates[0] if window_id_candidates else None

cluster_window_map = defaultdict(set)

if cluster_id_col is not None and window_id_col is not None:

    for _, row in relationship_df.iterrows():

        cid = row.get(cluster_id_col)
        wid = row.get(window_id_col)

        if pd.notna(cid) and pd.notna(wid):
            cluster_window_map[str(cid)].add(str(wid))

    print(f"Cluster ID column : {cluster_id_col}")
    print(f"Window ID column  : {window_id_col}")
    print(f"Mapped clusters   : {len(cluster_window_map):,}")

else:
    print(
        "⚠ Could not safely infer cluster/window ID columns.\n"
        "Official-window evidence will remain descriptive rather than being "
        "used automatically for candidate grouping."
    )


# ==========================================================================================
# 9.14H — ADD OFFICIAL-WINDOW RELATIONSHIP TO EACH BOUNDARY
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14H — COMPARING ADJACENT CLUSTERS AGAINST OFFICIAL WINDOWS")
print("=" * 120)


def window_signature(cluster_id):
    vals = cluster_window_map.get(str(cluster_id), set())
    return tuple(sorted(vals))


if len(gap_df):

    gap_df["Left_Official_Windows"] = gap_df["Left_Cluster_ID"].apply(
        window_signature
    )

    gap_df["Right_Official_Windows"] = gap_df["Right_Cluster_ID"].apply(
        window_signature
    )

    def shared_windows(row):

        left = set(row["Left_Official_Windows"])
        right = set(row["Right_Official_Windows"])

        return tuple(sorted(left.intersection(right)))

    gap_df["Shared_Official_Windows"] = gap_df.apply(
        shared_windows,
        axis=1,
    )

    gap_df["Shares_Official_Window"] = (
        gap_df["Shared_Official_Windows"].apply(len) > 0
    )

else:

    gap_df["Left_Official_Windows"] = []
    gap_df["Right_Official_Windows"] = []
    gap_df["Shared_Official_Windows"] = []
    gap_df["Shares_Official_Window"] = []


# ==========================================================================================
# 9.14I — FLAG SHORT INTERRUPTIONS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14I — SHORT INTERRUPTION SENSITIVITY")
print("=" * 120)

# IMPORTANT:
# These thresholds are NOT definitions of an attack event.
#
# They are only sensitivity categories so that we can inspect how strongly
# the apparent episode count depends on tiny benign interruptions.

SENSITIVITY_GAPS = [0, 1, 2, 3, 5, 10, 15, 30]

for threshold in SENSITIVITY_GAPS:
    gap_df[f"Gap_LE_{threshold}m"] = (
        gap_df["Benign_Gap_Minutes"] <= threshold
    )

print(
    "Sensitivity thresholds created:",
    SENSITIVITY_GAPS
)

print(
    "\nIMPORTANT: no threshold above is being declared the correct "
    "episode-merging rule."
)


# ==========================================================================================
# 9.14J — EVIDENCE-BASED BOUNDARY CLASSIFICATION
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14J — CLASSIFYING BOUNDARIES BY EVIDENCE")
print("=" * 120)

# We use conservative categories.
#
# SAME_EVENT_CANDIDATE:
#   very short interruption AND official-window evidence says both clusters
#   belong to the same documented attack window.
#
# SEPARATE_EVENT_CANDIDATE:
#   large temporal separation, especially when no official window is shared.
#
# AMBIGUOUS:
#   everything else.
#
# Note:
# "candidate" is intentional. This does not mutate canonical episodes.


def classify_boundary(row):

    gap = row["Benign_Gap_Minutes"]
    shared = bool(row["Shares_Official_Window"])

    # Strongest evidence for accidental fragmentation:
    # tiny interruption inside same official scenario/window.
    if shared and gap <= 2:
        return "SAME_EVENT_CANDIDATE"

    # Still plausible fragmentation, but less certain.
    if shared and gap <= 5:
        return "PROBABLE_SAME_EVENT"

    # A substantial gap with no common official window is evidence
    # toward independence, but still labelled candidate.
    if (not shared) and gap >= 30:
        return "SEPARATE_EVENT_CANDIDATE"

    return "AMBIGUOUS"


gap_df["Boundary_Assessment"] = gap_df.apply(
    classify_boundary,
    axis=1,
)

print("\nBoundary assessment counts:")
print(gap_df["Boundary_Assessment"].value_counts().to_string())


# ==========================================================================================
# 9.14K — EXPLAIN EVERY BOUNDARY DECISION
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14K — GENERATING BOUNDARY EXPLANATIONS")
print("=" * 120)


def explain_boundary(row):

    gap = row["Benign_Gap_Minutes"]
    shared = bool(row["Shares_Official_Window"])
    assessment = row["Boundary_Assessment"]

    reasons = []

    reasons.append(f"{gap:g}-minute benign interruption")

    if shared:
        reasons.append(
            "adjacent malicious clusters share official attack-window evidence"
        )
    else:
        reasons.append(
            "no shared official attack-window ID was established automatically"
        )

    if assessment == "SAME_EVENT_CANDIDATE":
        reasons.append(
            "short interruption is more consistent with fragmentation than "
            "with a confidently independent onset"
        )

    elif assessment == "PROBABLE_SAME_EVENT":
        reasons.append(
            "same official window supports continuity, but temporal gap is "
            "large enough to retain uncertainty"
        )

    elif assessment == "SEPARATE_EVENT_CANDIDATE":
        reasons.append(
            "substantial separation provides evidence for an independent onset"
        )

    else:
        reasons.append(
            "available evidence is insufficient for automatic boundary resolution"
        )

    return "; ".join(reasons)


gap_df["Assessment_Reason"] = gap_df.apply(
    explain_boundary,
    axis=1,
)


# ==========================================================================================
# 9.14L — CONSTRUCT HIGH-CONFIDENCE CANDIDATE EVENTS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14L — CONSTRUCTING HIGH-CONFIDENCE CANDIDATE EVENTS")
print("=" * 120)

# Conservative construction:
#
# Merge ONLY:
#   SAME_EVENT_CANDIDATE
#   PROBABLE_SAME_EVENT
#
# AMBIGUOUS boundaries remain split.
#
# Therefore this is not "the truth"; it is the conservative candidate event map
# supported by the currently available evidence.

mergeable_labels = {
    "SAME_EVENT_CANDIDATE",
    "PROBABLE_SAME_EVENT",
}

boundary_lookup = {}

for _, row in gap_df.iterrows():

    key = (
        row["Attack_Family"],
        row["Left_Cluster_ID"],
        row["Right_Cluster_ID"],
    )

    boundary_lookup[key] = row["Boundary_Assessment"]


candidate_event_rows = []
cluster_to_candidate = []

for family, g in cluster_df.groupby("Attack_Family", sort=False):

    g = g.sort_values("Cluster_Start").reset_index(drop=True)

    event_number = 1
    current_cluster_ids = [g.iloc[0]["Observed_Cluster_ID"]]

    def flush_event(cluster_ids, event_number):

        selected = g[
            g["Observed_Cluster_ID"].isin(cluster_ids)
        ].copy()

        event_id = f"{family}_E{event_number:03d}"

        start = selected["Cluster_Start"].min()
        end = selected["Cluster_End"].max()

        candidate_event_rows.append(
            {
                "Attack_Family": family,
                "Candidate_Event_ID": event_id,
                "Event_Start": start,
                "Event_End": end,
                "Duration_Span_Minutes": (
                    (end - start).total_seconds() / 60.0 + 1
                ),
                "Strict_Cluster_Count": selected[
                    "Observed_Cluster_ID"
                ].nunique(),
                "Strict_Cluster_IDs": "|".join(
                    selected["Observed_Cluster_ID"].astype(str)
                ),
                "Observed_Malicious_Minutes": selected[
                    "Observed_Minutes"
                ].sum(),
                "Attack_Flows": selected["Attack_Flows"].sum(),
            }
        )

        for cid in cluster_ids:
            cluster_to_candidate.append(
                {
                    "Attack_Family": family,
                    "Observed_Cluster_ID": cid,
                    "Candidate_Event_ID": event_id,
                }
            )

    for i in range(len(g) - 1):

        left_id = g.iloc[i]["Observed_Cluster_ID"]
        right_id = g.iloc[i + 1]["Observed_Cluster_ID"]

        assessment = boundary_lookup.get(
            (family, left_id, right_id),
            "AMBIGUOUS",
        )

        if assessment in mergeable_labels:

            current_cluster_ids.append(right_id)

        else:

            flush_event(current_cluster_ids, event_number)

            event_number += 1
            current_cluster_ids = [right_id]

    flush_event(current_cluster_ids, event_number)


candidate_event_df = pd.DataFrame(candidate_event_rows)
cluster_candidate_map_df = pd.DataFrame(cluster_to_candidate)

print(
    f"Strict malicious clusters : {len(cluster_df):,}\n"
    f"Candidate events          : {len(candidate_event_df):,}"
)

print("\nCandidate events by family:")

candidate_family_summary = (
    candidate_event_df
    .groupby("Attack_Family")
    .agg(
        Candidate_Events=("Candidate_Event_ID", "nunique"),
        Strict_Clusters=("Strict_Cluster_Count", "sum"),
        First_Event=("Event_Start", "min"),
        Last_Event=("Event_End", "max"),
        Malicious_Minutes=("Observed_Malicious_Minutes", "sum"),
        Attack_Flows=("Attack_Flows", "sum"),
    )
    .sort_values(
        ["Candidate_Events", "Attack_Flows"],
        ascending=False,
    )
)

print(candidate_family_summary.to_string())


# ==========================================================================================
# 9.14M — IDENTIFY UNRESOLVED BOUNDARIES
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14M — IDENTIFYING UNRESOLVED BOUNDARIES")
print("=" * 120)

unresolved_df = gap_df[
    gap_df["Boundary_Assessment"] == "AMBIGUOUS"
].copy()

unresolved_df = unresolved_df.sort_values(
    [
        "Benign_Gap_Minutes",
        "Attack_Family",
        "Left_Start",
    ]
)

print(f"Ambiguous boundaries: {len(unresolved_df):,}")

if len(unresolved_df):

    display_cols = [
        "Attack_Family",
        "Left_Cluster_ID",
        "Right_Cluster_ID",
        "Left_End",
        "Right_Start",
        "Benign_Gap_Minutes",
        "Shares_Official_Window",
        "Shared_Official_Windows",
        "Assessment_Reason",
    ]

    display(
        unresolved_df[
            [c for c in display_cols if c in unresolved_df.columns]
        ]
    )


# ==========================================================================================
# 9.14N — EPISODE-COUNT SENSITIVITY ANALYSIS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14N — EPISODE-COUNT SENSITIVITY ANALYSIS")
print("=" * 120)

# This is one of the most important diagnostics.
#
# We calculate what the number of apparent events WOULD become if someone
# blindly merged adjacent same-family clusters separated by <= X minutes.
#
# We are NOT adopting these thresholds.
#
# If event counts change dramatically between 1, 2, 5, 10, 15 minutes,
# then CIC's episode count is highly dependent on arbitrary gap semantics.


def count_events_at_gap_threshold(cluster_table, threshold):

    results = []

    for family, g in cluster_table.groupby("Attack_Family", sort=False):

        g = g.sort_values("Cluster_Start").reset_index(drop=True)

        if len(g) == 0:
            continue

        event_count = 1

        for i in range(len(g) - 1):

            end_a = g.iloc[i]["Cluster_End"]
            start_b = g.iloc[i + 1]["Cluster_Start"]

            raw_delta = (
                start_b - end_a
            ).total_seconds() / 60.0

            benign_gap = max(0.0, raw_delta - 1.0)

            if benign_gap > threshold:
                event_count += 1

        results.append(
            {
                "Attack_Family": family,
                "Gap_Threshold_Min": threshold,
                "Implied_Event_Count": event_count,
            }
        )

    return pd.DataFrame(results)


sensitivity_frames = []

for threshold in SENSITIVITY_GAPS:

    temp = count_events_at_gap_threshold(
        cluster_df,
        threshold,
    )

    sensitivity_frames.append(temp)

sensitivity_df = pd.concat(
    sensitivity_frames,
    ignore_index=True,
)

sensitivity_pivot = sensitivity_df.pivot(
    index="Attack_Family",
    columns="Gap_Threshold_Min",
    values="Implied_Event_Count",
)

sensitivity_pivot.columns = [
    f"Gap_LE_{c}m"
    for c in sensitivity_pivot.columns
]

sensitivity_pivot["Max_Count"] = sensitivity_pivot.max(axis=1)
sensitivity_pivot["Min_Count"] = sensitivity_pivot.min(axis=1)

sensitivity_pivot["Count_Range"] = (
    sensitivity_pivot["Max_Count"]
    - sensitivity_pivot["Min_Count"]
)

sensitivity_pivot = sensitivity_pivot.sort_values(
    "Count_Range",
    ascending=False,
)

print(
    "\nImplied number of attack events under different hypothetical "
    "gap-merging thresholds:\n"
)

print(sensitivity_pivot.to_string())


# ==========================================================================================
# 9.14O — GLOBAL SENSITIVITY
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14O — GLOBAL EVENT-COUNT SENSITIVITY")
print("=" * 120)

global_sensitivity = (
    sensitivity_df
    .groupby("Gap_Threshold_Min")["Implied_Event_Count"]
    .sum()
    .reset_index(name="Total_Implied_Events")
)

print(global_sensitivity.to_string(index=False))

strict_total = len(cluster_df)

print(f"\nStrict malicious clusters = {strict_total}")

for _, r in global_sensitivity.iterrows():

    print(
        f"If <= {int(r['Gap_Threshold_Min']):2d} min interruptions were "
        f"blindly merged -> {int(r['Total_Implied_Events']):3d} events"
    )


# ==========================================================================================
# 9.14P — EVENT INDEPENDENCE / SPLIT FEASIBILITY SCREEN
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14P — PRELIMINARY EPISODE-AWARE SPLIT FEASIBILITY")
print("=" * 120)

# This is NOT approval to split.
#
# It merely asks:
#   if candidate events were eventually accepted,
#   how many independent occurrences would each family have?
#
# Mathematical minimum:
#   >= 3 candidate events → at least one could theoretically exist in
#                           TRAIN / VAL / TEST.
#
# Scientific robustness is a separate question.

feasibility_df = (
    candidate_event_df
    .groupby("Attack_Family")
    .agg(
        Candidate_Events=("Candidate_Event_ID", "nunique"),
        Total_Malicious_Minutes=("Observed_Malicious_Minutes", "sum"),
        Total_Attack_Flows=("Attack_Flows", "sum"),
    )
    .reset_index()
)

feasibility_df["Three_Way_Mathematically_Possible"] = (
    feasibility_df["Candidate_Events"] >= 3
)

feasibility_df["Five_Or_More_Events"] = (
    feasibility_df["Candidate_Events"] >= 5
)

feasibility_df["Ten_Or_More_Events"] = (
    feasibility_df["Candidate_Events"] >= 10
)

feasibility_df = feasibility_df.sort_values(
    ["Candidate_Events", "Total_Attack_Flows"],
    ascending=False,
)

print(feasibility_df.to_string(index=False))


# ==========================================================================================
# 9.14Q — FAMILY-LEVEL UNCERTAINTY SCORECARD
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14Q — FAMILY-LEVEL BOUNDARY UNCERTAINTY")
print("=" * 120)

boundary_family = (
    gap_df
    .groupby("Attack_Family")
    .agg(
        Total_Boundaries=("Left_Cluster_ID", "count"),
        Same_Event_Candidates=(
            "Boundary_Assessment",
            lambda x: (x == "SAME_EVENT_CANDIDATE").sum(),
        ),
        Probable_Same_Event=(
            "Boundary_Assessment",
            lambda x: (x == "PROBABLE_SAME_EVENT").sum(),
        ),
        Separate_Event_Candidates=(
            "Boundary_Assessment",
            lambda x: (x == "SEPARATE_EVENT_CANDIDATE").sum(),
        ),
        Ambiguous_Boundaries=(
            "Boundary_Assessment",
            lambda x: (x == "AMBIGUOUS").sum(),
        ),
        Minimum_Gap=("Benign_Gap_Minutes", "min"),
        Median_Gap=("Benign_Gap_Minutes", "median"),
        Maximum_Gap=("Benign_Gap_Minutes", "max"),
    )
    .reset_index()
)

boundary_family["Ambiguous_Fraction"] = np.where(
    boundary_family["Total_Boundaries"] > 0,
    boundary_family["Ambiguous_Boundaries"]
    / boundary_family["Total_Boundaries"],
    0,
)

boundary_family = boundary_family.sort_values(
    ["Ambiguous_Boundaries", "Ambiguous_Fraction"],
    ascending=False,
)

print(boundary_family.to_string(index=False))


# ==========================================================================================
# 9.14R — COMPARE AGAINST OLD CANONICAL EPISODE MAP IF AVAILABLE
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14R — OPTIONAL COMPARISON WITH OLD CANONICAL EPISODE ARTIFACT")
print("=" * 120)

old_episode_path = find_file(
    "cell14a_forecast_episode_map.csv"
)

old_episode_summary_df = pd.DataFrame()

if old_episode_path is None:

    print(
        "Old cell14a_forecast_episode_map.csv not found.\n"
        "Skipping historical comparison."
    )

else:

    print("Found:")
    print(old_episode_path)

    old_map = pd.read_csv(old_episode_path)

    print(f"Rows: {len(old_map):,}")

    if {
        "Attack_Type",
        "Canonical_Type_Episode_ID",
    }.issubset(old_map.columns):

        old_episode_summary_df = (
            old_map[
                [
                    "Attack_Type",
                    "Canonical_Type_Episode_ID",
                ]
            ]
            .dropna()
            .drop_duplicates()
            .groupby("Attack_Type")
            ["Canonical_Type_Episode_ID"]
            .nunique()
            .reset_index(name="Old_Canonical_Episodes")
        )

        print("\nOld canonical episode counts:")
        print(old_episode_summary_df.to_string(index=False))

        print(
            "\nIMPORTANT: these old episode IDs are shown ONLY for comparison.\n"
            "They are NOT being used to define the new raw-data-derived events."
        )

    else:

        print(
            "Old artifact exists but expected episode columns were not found."
        )


# ==========================================================================================
# 9.14S — DETERMINE WHETHER CIC EVENT SEMANTICS ARE STABLE
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14S — EVENT-SEMANTICS STABILITY CHECK")
print("=" * 120)

total_boundaries = len(gap_df)

ambiguous_count = (
    gap_df["Boundary_Assessment"]
    .eq("AMBIGUOUS")
    .sum()
)

ambiguous_fraction = (
    ambiguous_count / total_boundaries
    if total_boundaries > 0
    else 0
)

families_threshold_sensitive = (
    sensitivity_pivot["Count_Range"] > 0
).sum()

total_families = cluster_df["Attack_Family"].nunique()

print(f"Total strict clusters            : {len(cluster_df):,}")
print(f"Total adjacent boundaries        : {total_boundaries:,}")
print(f"Ambiguous boundaries             : {ambiguous_count:,}")
print(f"Ambiguous boundary fraction      : {ambiguous_fraction:.3f}")

print(
    f"Threshold-sensitive families     : "
    f"{families_threshold_sensitive}/{total_families}"
)

print(
    f"Conservative candidate events    : "
    f"{len(candidate_event_df):,}"
)

# We intentionally do not define an arbitrary pass/fail cutoff here.
# Instead the diagnostic verdict reflects whether unresolved semantics remain.

if ambiguous_count == 0:

    semantics_status = "RESOLVED_BY_CURRENT_EVIDENCE"

else:

    semantics_status = "UNRESOLVED_BOUNDARIES_REMAIN"

print(f"\nSEMANTICS STATUS: {semantics_status}")


# ==========================================================================================
# 9.14T — SAVE OUTPUTS
# ==========================================================================================

print("\n" + "=" * 120)
print("9.14T — SAVING PHASE 9.14 ARTIFACTS")
print("=" * 120)

# Save beside the Phase 9.13 cluster artifact so the notebook remains interconnected.

OUTPUT_DIR = paths[
    "cic_strict_attack_minute_clusters.csv"
].parent

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

outputs = {
    "cic_intercluster_boundary_evidence.csv":
        gap_df,

    "cic_candidate_attack_events.csv":
        candidate_event_df,

    "cic_cluster_to_candidate_event_map.csv":
        cluster_candidate_map_df,

    "cic_unresolved_attack_boundaries.csv":
        unresolved_df,

    "cic_episode_count_sensitivity.csv":
        sensitivity_df,

    "cic_episode_count_sensitivity_pivot.csv":
        sensitivity_pivot.reset_index(),

    "cic_global_episode_count_sensitivity.csv":
        global_sensitivity,

    "cic_candidate_event_split_feasibility.csv":
        feasibility_df,

    "cic_family_boundary_uncertainty.csv":
        boundary_family,
}

for filename, df in outputs.items():

    path = OUTPUT_DIR / filename

    df.to_csv(
        path,
        index=False,
    )

    print(f"✓ {filename}")


# ==========================================================================================
# 9.14U — MACHINE-READABLE STATUS
# ==========================================================================================

status = {
    "phase": "9.14",
    "strict_cluster_count": int(len(cluster_df)),
    "attack_family_count": int(
        cluster_df["Attack_Family"].nunique()
    ),
    "adjacent_boundary_count": int(total_boundaries),
    "ambiguous_boundary_count": int(ambiguous_count),
    "ambiguous_boundary_fraction": float(ambiguous_fraction),
    "candidate_event_count": int(len(candidate_event_df)),
    "threshold_sensitive_family_count": int(
        families_threshold_sensitive
    ),
    "semantics_status": semantics_status,

    "important_note": (
        "Candidate events are forensic hypotheses only. "
        "No forecasting targets or canonical dataset boundaries "
        "have been modified."
    ),
}

status_path = OUTPUT_DIR / "cic_phase_9_14_status.json"

with open(
    status_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        status,
        f,
        indent=2,
        default=str,
    )

print("✓ cic_phase_9_14_status.json")


# ==========================================================================================
# 9.14V — FINAL REPORT
# ==========================================================================================

print("\n" + "=" * 120)
print("PHASE 9.14 — FINAL REPORT")
print("=" * 120)

print(
    f"""
Strict malicious-minute clusters
--------------------------------
{len(cluster_df):,}

Attack families
---------------
{cluster_df['Attack_Family'].nunique():,}

Adjacent same-family boundaries inspected
-----------------------------------------
{total_boundaries:,}

Boundary assessments
---------------------
{gap_df['Boundary_Assessment'].value_counts().to_string()}

Conservative candidate attack events
------------------------------------
{len(candidate_event_df):,}

Unresolved boundaries
---------------------
{ambiguous_count:,}

Families whose apparent event count changes under hypothetical gap thresholds
-----------------------------------------------------------------------------
{families_threshold_sensitive}/{total_families}

Semantics status
----------------
{semantics_status}
"""
)

print("-" * 120)

print(
    """
INTERPRETATION RULE

The candidate-event count printed above is NOT automatically the final number
of independent CIC attack episodes.

Phase 9.14 is testing whether the 82 strict malicious-minute clusters can be
defensibly interpreted as larger physical attack occurrences.

In particular:

    strict malicious cluster
            !=
    automatically independent attack episode

and:

    short benign gap
            !=
    automatically new attack onset

The sensitivity table shows how much the apparent episode count would change
under different gap assumptions.

AMBIGUOUS boundaries must remain unresolved rather than being silently merged.
"""
)

print("-" * 120)

if ambiguous_count > 0:

    print(
        """
NEXT STEP — PHASE 9.15

Do NOT create the final forecasting dataset yet.

Phase 9.15 should investigate the remaining ambiguous boundaries using:
    • exact CIC scenario/day identity
    • official attack schedule/window evidence
    • attack family
    • source/destination behavior where available
    • flow intensity around the boundary
    • whether traffic returns to genuinely benign baseline
    • raw capture/session structure

Only after those boundaries are resolved should we freeze canonical CIC
attack-event/onset semantics.
"""
    )

else:

    print(
        """
NEXT STEP

Current forensic evidence resolved every adjacent boundary.

Before modifying the forecasting dataset, freeze the candidate event map as a
versioned canonical event artifact and perform one final integrity audit.
"""
    )

print("=" * 120)
print("PHASE 9.14 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 9.15 — CIC PHYSICAL ATTACK EPISODE RESOLUTION
# CLEAN REPLACEMENT VERSION
# ==================================================================================================
#
# OBJECTIVE
# ---------
# Determine how CIC's strict malicious clusters relate to defensible physical attack episodes.
#
# We are currently focused ONLY on KNOWN-TYPE FORECASTING.
#
# This phase:
#   ✓ reconstructs raw minute-level network behaviour
#   ✓ builds behavioural signatures for every strict attack cluster
#   ✓ investigates every same-family adjacent cluster boundary
#   ✓ measures source/destination/pair continuity
#   ✓ measures attack-intensity continuity
#   ✓ checks whether traffic returns to benign baseline
#   ✓ incorporates official-window evidence from Phase 9.14
#   ✓ identifies strong same-episode boundaries
#   ✓ identifies strong separate-episode boundaries
#   ✓ preserves genuinely ambiguous boundaries
#   ✓ builds conservative physical-episode candidate groups
#   ✓ audits per-family episode sufficiency for future TRAIN/VAL/TEST
#
# This phase DOES NOT:
#   ✗ create forecasting windows
#   ✗ split train/validation/test
#   ✗ combine CIC + UNSW
#   ✗ train a model
#   ✗ assume every malicious cluster is an independent attack
#
# CONSERVATIVE PRINCIPLE
# ----------------------
# Uncertainty is preserved.
# We do NOT manufacture independent attack episodes just to obtain enough samples.
#
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from collections import defaultdict

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 250)

print("=" * 122)
print("PHASE 9.15 — CIC PHYSICAL ATTACK EPISODE RESOLUTION")
print("=" * 122)


# ==================================================================================================
# 9.15A — PATHS + REQUIRED INPUTS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

CIC_RAW = PROJECT_ROOT / "raw" / "CIC_IDS2017"
CIC_OUT = PROJECT_ROOT / "audit_outputs" / "CIC"

required_files = {
    "minute_table":
        CIC_OUT / "cic_complete_minute_flow_table.csv",

    "clusters":
        CIC_OUT / "cic_strict_attack_minute_clusters.csv",

    "relationships":
        CIC_OUT / "cic_cluster_official_window_relationships.csv",

    "unresolved":
        CIC_OUT / "cic_unresolved_attack_boundaries.csv",

    "boundary_evidence":
        CIC_OUT / "cic_intercluster_boundary_evidence.csv",
}

print("\n" + "=" * 122)
print("9.15A — VERIFYING REQUIRED INPUTS")
print("=" * 122)

for name, path in required_files.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required Phase 9.14 artifact:\n{path}"
        )

    print(f"✓ {name:22s} {path.name}")

raw_files = sorted(CIC_RAW.glob("*.csv"))

if not raw_files:
    raise FileNotFoundError(
        f"No CIC CSV files found in:\n{CIC_RAW}"
    )

print(f"\n✓ Raw CIC files found: {len(raw_files)}")


# ==================================================================================================
# 9.15B — LOAD PHASE 9.14 ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15B — LOADING PHASE 9.14 ARTIFACTS")
print("=" * 122)

minute_df = pd.read_csv(
    required_files["minute_table"]
)

cluster_df = pd.read_csv(
    required_files["clusters"]
)

relationship_df = pd.read_csv(
    required_files["relationships"]
)

unresolved_df = pd.read_csv(
    required_files["unresolved"]
)

boundary_df = pd.read_csv(
    required_files["boundary_evidence"]
)


# Strip accidental whitespace from column names
for df in [
    minute_df,
    cluster_df,
    relationship_df,
    unresolved_df,
    boundary_df
]:
    df.columns = [
        str(c).strip()
        for c in df.columns
    ]


# Parse known datetime columns
timestamp_candidates = [
    "Minute",
    "Cluster_Start",
    "Cluster_End",
    "Official_Window_Start",
    "Official_Window_End",
    "Left_Start",
    "Left_End",
    "Right_Start",
    "Right_End",
    "Window_Start",
    "Window_End",
]

for df in [
    minute_df,
    cluster_df,
    relationship_df,
    unresolved_df,
    boundary_df
]:

    for col in timestamp_candidates:

        if col in df.columns:

            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )


print(f"minute_df       : {minute_df.shape}")
print(f"cluster_df      : {cluster_df.shape}")
print(f"relationship_df : {relationship_df.shape}")
print(f"unresolved_df   : {unresolved_df.shape}")
print(f"boundary_df     : {boundary_df.shape}")


# ==================================================================================================
# 9.15C — RAW CIC ENCODING MAP
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15C — RAW CIC ENCODING MAP")
print("=" * 122)

# Phase 9.4B established that only this file requires cp1252.
ENCODING_MAP = {
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv":
        "cp1252"
}


def encoding_for(path):

    return ENCODING_MAP.get(
        path.name,
        "utf-8"
    )


for f in raw_files:

    print(
        f"{f.name:72s} "
        f"{encoding_for(f)}"
    )


# ==================================================================================================
# 9.15D — RECONSTRUCT RAW MINUTE-LEVEL NETWORK BEHAVIOUR
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15D — BUILDING RAW MINUTE-LEVEL BEHAVIOURAL TABLE")
print("=" * 122)

raw_minute_parts = []

needed_cols = {
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Label"
}


for file in raw_files:

    print(f"\nProcessing: {file.name}")

    enc = encoding_for(file)

    df = pd.read_csv(
        file,
        encoding=enc,
        usecols=lambda c: c.strip() in needed_cols,
        low_memory=False
    )

    df.columns = [
        c.strip()
        for c in df.columns
    ]

    # Remove structural empty rows discovered previously.
    df = df.dropna(
        how="all"
    ).copy()

    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        errors="coerce",
        dayfirst=True
    )

    bad_ts = int(
        df["Timestamp"].isna().sum()
    )

    if bad_ts:

        print(
            f"  ⚠ invalid timestamps removed: "
            f"{bad_ts:,}"
        )

        df = df[
            df["Timestamp"].notna()
        ].copy()

    df["Minute"] = (
        df["Timestamp"]
        .dt.floor("min")
    )

    df["Label"] = (
        df["Label"]
        .astype("string")
        .str.strip()
    )

    df["Is_Attack"] = (
        df["Label"].notna()
        &
        (
            df["Label"]
            .str.upper()
            != "BENIGN"
        )
    )

    df["Attack_Label"] = (
        df["Label"]
        .where(df["Is_Attack"])
    )

    rows = []

    for minute, g in df.groupby(
        "Minute",
        sort=True
    ):

        attack_g = g[
            g["Is_Attack"]
        ]

        srcs = set(
            g["Source IP"]
            .dropna()
            .astype(str)
        )

        dsts = set(
            g["Destination IP"]
            .dropna()
            .astype(str)
        )

        attack_srcs = set(
            attack_g["Source IP"]
            .dropna()
            .astype(str)
        )

        attack_dsts = set(
            attack_g["Destination IP"]
            .dropna()
            .astype(str)
        )

        attack_pairs = set(
            zip(
                attack_g["Source IP"]
                .dropna()
                .astype(str),

                attack_g["Destination IP"]
                .dropna()
                .astype(str)
            )
        )

        attack_labels = sorted(
            attack_g["Attack_Label"]
            .dropna()
            .astype(str)
            .unique()
        )

        rows.append({

            "Segment_ID":
                file.name,

            "Minute":
                minute,

            "Flow_Count_Raw":
                len(g),

            "Attack_Flow_Count_Raw":
                int(
                    g["Is_Attack"].sum()
                ),

            "Benign_Flow_Count_Raw":
                int(
                    (~g["Is_Attack"]).sum()
                ),

            "Unique_Source_IPs":
                len(srcs),

            "Unique_Destination_IPs":
                len(dsts),

            "Unique_Attack_Source_IPs":
                len(attack_srcs),

            "Unique_Attack_Destination_IPs":
                len(attack_dsts),

            "Attack_Source_Set":
                tuple(
                    sorted(attack_srcs)
                ),

            "Attack_Destination_Set":
                tuple(
                    sorted(attack_dsts)
                ),

            "Attack_Pair_Set":
                tuple(
                    sorted(attack_pairs)
                ),

            "Attack_Labels":
                tuple(
                    attack_labels
                ),
        })

    part = pd.DataFrame(rows)

    print(
        f"  minutes       : "
        f"{len(part):,}"
    )

    print(
        f"  attack minutes: "
        f"{(part['Attack_Flow_Count_Raw'] > 0).sum():,}"
    )

    raw_minute_parts.append(
        part
    )


raw_behavior_df = pd.concat(
    raw_minute_parts,
    ignore_index=True
)

raw_behavior_df = (
    raw_behavior_df
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(drop=True)
)

print(
    "\nRaw behavioural minute table:",
    raw_behavior_df.shape
)


# ==================================================================================================
# 9.15E — CROSS-CHECK AGAINST EXISTING CANONICAL MINUTE TIMELINE
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15E — CROSS-CHECKING RAW MINUTE RECONSTRUCTION")
print("=" * 122)

print(
    "Phase 9.14 minute records :",
    len(minute_df)
)

print(
    "Raw reconstructed minutes :",
    len(raw_behavior_df)
)

raw_attack_minutes = int(
    (
        raw_behavior_df[
            "Attack_Flow_Count_Raw"
        ] > 0
    ).sum()
)

print(
    "Raw attack-active minutes :",
    raw_attack_minutes
)

if "Attack_State" in minute_df.columns:

    old_attack_minutes = int(
        (
            minute_df["Attack_State"]
            == 1
        ).sum()
    )

    print(
        "Previous attack minutes   :",
        old_attack_minutes
    )

    if (
        old_attack_minutes
        == raw_attack_minutes
    ):

        print(
            "✓ Attack-minute count agrees."
        )

    else:

        print(
            "⚠ Attack-minute count differs."
        )


# ==================================================================================================
# 9.15F — HELPERS
# ==================================================================================================

def safe_set(x):

    if isinstance(x, set):
        return x

    if isinstance(
        x,
        (tuple, list)
    ):
        return set(x)

    if x is None:
        return set()

    try:
        if pd.isna(x):
            return set()
    except Exception:
        pass

    return {str(x)}


def jaccard(a, b):

    a = safe_set(a)
    b = safe_set(b)

    if not a and not b:
        return np.nan

    union = a | b

    if not union:
        return np.nan

    return (
        len(a & b)
        /
        len(union)
    )


def ratio_similarity(a, b):

    if (
        pd.isna(a)
        or
        pd.isna(b)
    ):
        return np.nan

    a = float(a)
    b = float(b)

    if (
        a == 0
        and
        b == 0
    ):
        return 1.0

    mx = max(
        abs(a),
        abs(b)
    )

    if mx == 0:
        return 1.0

    return (
        min(
            abs(a),
            abs(b)
        )
        /
        mx
    )


def get_segment_for_cluster(cluster_id):

    row = cluster_df[
        cluster_df[
            "Observed_Cluster_ID"
        ].astype(str)
        ==
        str(cluster_id)
    ]

    if row.empty:
        return None

    if "Segment_ID" in row.columns:

        value = row.iloc[0][
            "Segment_ID"
        ]

        if pd.notna(value):
            return value

    start = row.iloc[0][
        "Cluster_Start"
    ]

    end = row.iloc[0][
        "Cluster_End"
    ]

    candidates = []

    for seg, g in raw_behavior_df.groupby(
        "Segment_ID"
    ):

        if (
            g["Minute"].min()
            <= start
            <= g["Minute"].max()
            and
            g["Minute"].min()
            <= end
            <= g["Minute"].max()
        ):

            candidates.append(seg)

    if len(candidates) == 1:
        return candidates[0]

    return None


# ==================================================================================================
# 9.15G — BUILD CLUSTER BEHAVIOURAL SIGNATURES
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15G — CONSTRUCTING CLUSTER BEHAVIOURAL SIGNATURES")
print("=" * 122)

cluster_signatures = []

for _, c in cluster_df.iterrows():

    cluster_id = (
        c["Observed_Cluster_ID"]
    )

    family = (
        c["Attack_Family"]
    )

    start = (
        c["Cluster_Start"]
    )

    end = (
        c["Cluster_End"]
    )

    segment = get_segment_for_cluster(
        cluster_id
    )

    if segment is None:

        print(
            f"⚠ Could not infer segment for "
            f"{cluster_id}"
        )

        continue

    g = raw_behavior_df[
        (
            raw_behavior_df["Segment_ID"]
            == segment
        )
        &
        (
            raw_behavior_df["Minute"]
            >= start
        )
        &
        (
            raw_behavior_df["Minute"]
            <= end
        )
    ].copy()

    attack_srcs = set()
    attack_dsts = set()
    attack_pairs = set()

    for x in g[
        "Attack_Source_Set"
    ]:

        attack_srcs |= safe_set(x)

    for x in g[
        "Attack_Destination_Set"
    ]:

        attack_dsts |= safe_set(x)

    for x in g[
        "Attack_Pair_Set"
    ]:

        attack_pairs |= safe_set(x)

    cluster_signatures.append({

        "Attack_Family":
            family,

        "Observed_Cluster_ID":
            str(cluster_id),

        "Segment_ID":
            segment,

        "Cluster_Start":
            start,

        "Cluster_End":
            end,

        "Observed_Minutes_Rebuilt":
            len(g),

        "Total_Flows_Context":
            g[
                "Flow_Count_Raw"
            ].sum(),

        "Total_Attack_Flows_Rebuilt":
            g[
                "Attack_Flow_Count_Raw"
            ].sum(),

        "Mean_Flows_Per_Minute":
            g[
                "Flow_Count_Raw"
            ].mean(),

        "Mean_Attack_Flows_Per_Minute":
            g[
                "Attack_Flow_Count_Raw"
            ].mean(),

        "Attack_Source_Set":
            tuple(
                sorted(attack_srcs)
            ),

        "Attack_Destination_Set":
            tuple(
                sorted(attack_dsts)
            ),

        "Attack_Pair_Set":
            tuple(
                sorted(attack_pairs)
            ),

        "Unique_Attack_Sources":
            len(attack_srcs),

        "Unique_Attack_Destinations":
            len(attack_dsts),

        "Unique_Attack_Pairs":
            len(attack_pairs),
    })


signature_df = pd.DataFrame(
    cluster_signatures
)

print(
    "Cluster signatures:",
    signature_df.shape
)

display_cols = [
    "Attack_Family",
    "Observed_Cluster_ID",
    "Segment_ID",
    "Cluster_Start",
    "Cluster_End",
    "Total_Attack_Flows_Rebuilt",
    "Unique_Attack_Sources",
    "Unique_Attack_Destinations"
]

print(
    signature_df[
        display_cols
    ]
    .head(20)
    .to_string(index=False)
)


# ==================================================================================================
# 9.15H — RECONSTRUCT SAME-FAMILY ADJACENT BOUNDARIES
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15H — REBUILDING SAME-FAMILY ADJACENT BOUNDARIES")
print("=" * 122)

boundaries = []

for family, g in signature_df.groupby(
    "Attack_Family"
):

    for segment, sg in g.groupby(
        "Segment_ID"
    ):

        sg = (
            sg
            .sort_values(
                "Cluster_Start"
            )
            .reset_index(drop=True)
        )

        for i in range(
            len(sg) - 1
        ):

            left = sg.iloc[i]
            right = sg.iloc[i + 1]

            gap = int(
                (
                    right["Cluster_Start"]
                    -
                    left["Cluster_End"]
                ).total_seconds()
                /
                60
            ) - 1

            boundaries.append({

                "Attack_Family":
                    family,

                "Segment_ID":
                    segment,

                "Left_Cluster_ID":
                    str(
                        left[
                            "Observed_Cluster_ID"
                        ]
                    ),

                "Right_Cluster_ID":
                    str(
                        right[
                            "Observed_Cluster_ID"
                        ]
                    ),

                "Left_Start":
                    left[
                        "Cluster_Start"
                    ],

                "Left_End":
                    left[
                        "Cluster_End"
                    ],

                "Right_Start":
                    right[
                        "Cluster_Start"
                    ],

                "Right_End":
                    right[
                        "Cluster_End"
                    ],

                "Benign_Gap_Minutes":
                    max(
                        gap,
                        0
                    )
            })


resolution_df = pd.DataFrame(
    boundaries
)

print(
    "Boundaries reconstructed:",
    len(resolution_df)
)


# ==================================================================================================
# 9.15I — EXTRACT BOUNDARY TRAFFIC EVIDENCE
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15I — EXTRACTING BOUNDARY TRAFFIC EVIDENCE")
print("=" * 122)

CONTEXT_MINUTES = 5

evidence_rows = []

sig_lookup = (
    signature_df
    .set_index(
        "Observed_Cluster_ID"
    )
)

for _, b in resolution_df.iterrows():

    segment = (
        b["Segment_ID"]
    )

    left_id = str(
        b["Left_Cluster_ID"]
    )

    right_id = str(
        b["Right_Cluster_ID"]
    )

    left = sig_lookup.loc[
        left_id
    ]

    right = sig_lookup.loc[
        right_id
    ]

    left_end = (
        b["Left_End"]
    )

    right_start = (
        b["Right_Start"]
    )

    seg_df = raw_behavior_df[
        raw_behavior_df[
            "Segment_ID"
        ]
        ==
        segment
    ].copy()

    left_context = seg_df[
        (
            seg_df["Minute"]
            >=
            left_end
            -
            pd.Timedelta(
                minutes=
                CONTEXT_MINUTES - 1
            )
        )
        &
        (
            seg_df["Minute"]
            <=
            left_end
        )
    ]

    right_context = seg_df[
        (
            seg_df["Minute"]
            >=
            right_start
        )
        &
        (
            seg_df["Minute"]
            <=
            right_start
            +
            pd.Timedelta(
                minutes=
                CONTEXT_MINUTES - 1
            )
        )
    ]

    gap_df = seg_df[
        (
            seg_df["Minute"]
            >
            left_end
        )
        &
        (
            seg_df["Minute"]
            <
            right_start
        )
    ]

    pre_baseline = seg_df[
        (
            seg_df["Minute"]
            >=
            b["Left_Start"]
            -
            pd.Timedelta(
                minutes=10
            )
        )
        &
        (
            seg_df["Minute"]
            <
            b["Left_Start"]
        )
    ]

    left_attack_rate = (
        left_context[
            "Attack_Flow_Count_Raw"
        ].mean()
        if len(left_context)
        else np.nan
    )

    right_attack_rate = (
        right_context[
            "Attack_Flow_Count_Raw"
        ].mean()
        if len(right_context)
        else np.nan
    )

    gap_total_rate = (
        gap_df[
            "Flow_Count_Raw"
        ].mean()
        if len(gap_df)
        else np.nan
    )

    gap_attack_rate = (
        gap_df[
            "Attack_Flow_Count_Raw"
        ].mean()
        if len(gap_df)
        else 0.0
    )

    gap_benign_rate = (
        gap_df[
            "Benign_Flow_Count_Raw"
        ].mean()
        if len(gap_df)
        else np.nan
    )

    baseline_rate = (
        pre_baseline[
            "Flow_Count_Raw"
        ].mean()
        if len(pre_baseline)
        else np.nan
    )

    src_j = jaccard(
        left["Attack_Source_Set"],
        right["Attack_Source_Set"]
    )

    dst_j = jaccard(
        left[
            "Attack_Destination_Set"
        ],
        right[
            "Attack_Destination_Set"
        ]
    )

    pair_j = jaccard(
        left["Attack_Pair_Set"],
        right["Attack_Pair_Set"]
    )

    attack_intensity_similarity = (
        ratio_similarity(
            left[
                "Mean_Attack_Flows_Per_Minute"
            ],
            right[
                "Mean_Attack_Flows_Per_Minute"
            ]
        )
    )

    baseline_similarity = (
        ratio_similarity(
            gap_total_rate,
            baseline_rate
        )
    )

    evidence_rows.append({

        **b.to_dict(),

        "Left_Attack_Sources":
            left[
                "Unique_Attack_Sources"
            ],

        "Right_Attack_Sources":
            right[
                "Unique_Attack_Sources"
            ],

        "Left_Attack_Destinations":
            left[
                "Unique_Attack_Destinations"
            ],

        "Right_Attack_Destinations":
            right[
                "Unique_Attack_Destinations"
            ],

        "Source_Jaccard":
            src_j,

        "Destination_Jaccard":
            dst_j,

        "Pair_Jaccard":
            pair_j,

        "Left_Mean_Attack_Flows":
            left[
                "Mean_Attack_Flows_Per_Minute"
            ],

        "Right_Mean_Attack_Flows":
            right[
                "Mean_Attack_Flows_Per_Minute"
            ],

        "Attack_Intensity_Similarity":
            attack_intensity_similarity,

        "Gap_Observed_Minutes":
            len(gap_df),

        "Gap_Mean_Total_Flows":
            gap_total_rate,

        "Gap_Mean_Attack_Flows":
            gap_attack_rate,

        "Gap_Mean_Benign_Flows":
            gap_benign_rate,

        "PreAttack_Baseline_Mean_Flows":
            baseline_rate,

        "Gap_vs_Baseline_Similarity":
            baseline_similarity,

        "Left_Context_Mean_Attack_Flows":
            left_attack_rate,

        "Right_Context_Mean_Attack_Flows":
            right_attack_rate
    })


evidence_df = pd.DataFrame(
    evidence_rows
)

print(
    "Boundary evidence rows:",
    len(evidence_df)
)


# ==================================================================================================
# 9.15J — ATTACH OFFICIAL-WINDOW EVIDENCE
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15J — ATTACHING OFFICIAL-WINDOW EVIDENCE")
print("=" * 122)

official_map = {}

required_official_cols = {
    "Observed_Cluster_ID",
    "Official_Window_Start",
    "Official_Window_End"
}

if required_official_cols.issubset(
    relationship_df.columns
):

    for cluster_id, g in (
        relationship_df
        .groupby(
            "Observed_Cluster_ID"
        )
    ):

        windows = set()

        for _, r in g.iterrows():

            s = r[
                "Official_Window_Start"
            ]

            e = r[
                "Official_Window_End"
            ]

            if (
                pd.notna(s)
                and
                pd.notna(e)
            ):

                windows.add(
                    (s, e)
                )

        official_map[
            str(cluster_id)
        ] = windows


def official_relationship(
    left_id,
    right_id
):

    lw = official_map.get(
        str(left_id),
        set()
    )

    rw = official_map.get(
        str(right_id),
        set()
    )

    shared = lw & rw

    if shared:

        return (
            True,
            tuple(
                sorted(shared)
            )
        )

    return (
        False,
        tuple()
    )


official_results = evidence_df.apply(
    lambda r:
        official_relationship(
            r["Left_Cluster_ID"],
            r["Right_Cluster_ID"]
        ),
    axis=1
)

evidence_df[
    "Shares_Official_Window"
] = [
    x[0]
    for x in official_results
]

evidence_df[
    "Shared_Official_Windows"
] = [
    x[1]
    for x in official_results
]

print(
    evidence_df[
        "Shares_Official_Window"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ==================================================================================================
# 9.15K — CONSERVATIVE MULTI-EVIDENCE BOUNDARY CLASSIFICATION
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15K — CLASSIFYING BOUNDARIES USING MULTIPLE EVIDENCE SOURCES")
print("=" * 122)


def classify_boundary(r):

    gap = (
        r["Benign_Gap_Minutes"]
    )

    src = (
        r["Source_Jaccard"]
    )

    dst = (
        r["Destination_Jaccard"]
    )

    pair = (
        r["Pair_Jaccard"]
    )

    intensity = (
        r[
            "Attack_Intensity_Similarity"
        ]
    )

    baseline = (
        r[
            "Gap_vs_Baseline_Similarity"
        ]
    )

    shared_official = bool(
        r[
            "Shares_Official_Window"
        ]
    )

    same_evidence = []
    separate_evidence = []

    # --------------------------------------------------------------
    # SAME-EPISODE SUPPORT
    # --------------------------------------------------------------

    if shared_official:

        same_evidence.append(
            "shared_official_window"
        )

    if (
        pd.notna(pair)
        and
        pair >= 0.50
    ):

        same_evidence.append(
            "strong_pair_continuity"
        )

    elif (
        pd.notna(src)
        and
        pd.notna(dst)
        and
        src >= 0.50
        and
        dst >= 0.50
    ):

        same_evidence.append(
            "strong_endpoint_continuity"
        )

    if (
        pd.notna(intensity)
        and
        intensity >= 0.50
    ):

        same_evidence.append(
            "similar_attack_intensity"
        )

    # Short gap alone is NEVER enough.
    if gap <= 2:

        same_evidence.append(
            "very_short_interruption"
        )

    # --------------------------------------------------------------
    # SEPARATE-EPISODE SUPPORT
    # --------------------------------------------------------------

    if gap >= 30:

        separate_evidence.append(
            "long_benign_separation"
        )

    if (
        pd.notna(pair)
        and
        pair == 0
        and
        pd.notna(src)
        and
        src == 0
        and
        pd.notna(dst)
        and
        dst == 0
    ):

        separate_evidence.append(
            "no_endpoint_continuity"
        )

    if (
        gap >= 5
        and
        pd.notna(baseline)
        and
        baseline >= 0.70
    ):

        separate_evidence.append(
            "traffic_returned_to_baseline"
        )

    identity_or_schedule = (
        "shared_official_window"
        in same_evidence
        or
        "strong_pair_continuity"
        in same_evidence
        or
        "strong_endpoint_continuity"
        in same_evidence
    )

    # Strong same-episode decision
    if (
        len(same_evidence) >= 3
        and
        identity_or_schedule
        and
        len(separate_evidence) == 0
    ):

        return pd.Series({

            "Episode_Boundary_Assessment":
                "SAME_PHYSICAL_EPISODE_STRONG",

            "Same_Evidence":
                "; ".join(
                    same_evidence
                ),

            "Separate_Evidence":
                "",

            "Resolution_Confidence":
                "HIGH"
        })

    # Strong separate-episode decision
    if (
        len(separate_evidence) >= 2
        and
        not shared_official
    ):

        return pd.Series({

            "Episode_Boundary_Assessment":
                "SEPARATE_PHYSICAL_EPISODES_STRONG",

            "Same_Evidence":
                "; ".join(
                    same_evidence
                ),

            "Separate_Evidence":
                "; ".join(
                    separate_evidence
                ),

            "Resolution_Confidence":
                "HIGH"
        })

    # Otherwise preserve uncertainty.
    return pd.Series({

        "Episode_Boundary_Assessment":
            "AMBIGUOUS",

        "Same_Evidence":
            "; ".join(
                same_evidence
            ),

        "Separate_Evidence":
            "; ".join(
                separate_evidence
            ),

        "Resolution_Confidence":
            "UNRESOLVED"
    })


classified = evidence_df.apply(
    classify_boundary,
    axis=1
)

evidence_df = pd.concat(
    [
        evidence_df.reset_index(
            drop=True
        ),
        classified.reset_index(
            drop=True
        )
    ],
    axis=1
)

print(
    "\nBoundary assessment counts:"
)

print(
    evidence_df[
        "Episode_Boundary_Assessment"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ==================================================================================================
# 9.15L — STRONG SAME-PHYSICAL-EPISODE BOUNDARIES
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15L — STRONG SAME-PHYSICAL-EPISODE BOUNDARIES")
print("=" * 122)

same_df = evidence_df[
    evidence_df[
        "Episode_Boundary_Assessment"
    ]
    ==
    "SAME_PHYSICAL_EPISODE_STRONG"
].copy()

same_display_cols = [
    "Attack_Family",
    "Left_Cluster_ID",
    "Right_Cluster_ID",
    "Benign_Gap_Minutes",
    "Source_Jaccard",
    "Destination_Jaccard",
    "Pair_Jaccard",
    "Attack_Intensity_Similarity",
    "Shares_Official_Window",
    "Same_Evidence"
]

if not same_df.empty:

    same_sorted = (
        same_df
        .sort_values(
            [
                "Attack_Family",
                "Left_End"
            ]
        )
    )

    print(
        same_sorted[
            same_display_cols
        ]
        .to_string(index=False)
    )

else:

    print(
        "None resolved with strong same-episode evidence."
    )


# ==================================================================================================
# 9.15M — STRONG SEPARATE-PHYSICAL-EPISODE BOUNDARIES
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15M — STRONG SEPARATE-PHYSICAL-EPISODE BOUNDARIES")
print("=" * 122)

separate_df = evidence_df[
    evidence_df[
        "Episode_Boundary_Assessment"
    ]
    ==
    "SEPARATE_PHYSICAL_EPISODES_STRONG"
].copy()

separate_display_cols = [
    "Attack_Family",
    "Left_Cluster_ID",
    "Right_Cluster_ID",
    "Benign_Gap_Minutes",
    "Source_Jaccard",
    "Destination_Jaccard",
    "Pair_Jaccard",
    "Gap_vs_Baseline_Similarity",
    "Separate_Evidence"
]

if not separate_df.empty:

    separate_sorted = (
        separate_df
        .sort_values(
            [
                "Attack_Family",
                "Left_End"
            ]
        )
    )

    print(
        separate_sorted[
            separate_display_cols
        ]
        .to_string(index=False)
    )

else:

    print(
        "None resolved with strong separate-episode evidence."
    )


# ==================================================================================================
# 9.15N — REMAINING AMBIGUOUS BOUNDARIES
# FIXED: SORT BEFORE SELECTING DISPLAY COLUMNS
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15N — REMAINING AMBIGUOUS BOUNDARIES")
print("=" * 122)

ambiguous_df = evidence_df[
    evidence_df[
        "Episode_Boundary_Assessment"
    ]
    ==
    "AMBIGUOUS"
].copy()

print(
    f"Remaining ambiguous boundaries: "
    f"{len(ambiguous_df):,}"
)

ambiguous_display_cols = [
    "Attack_Family",
    "Left_Cluster_ID",
    "Right_Cluster_ID",
    "Benign_Gap_Minutes",
    "Source_Jaccard",
    "Destination_Jaccard",
    "Pair_Jaccard",
    "Attack_Intensity_Similarity",
    "Gap_vs_Baseline_Similarity",
    "Shares_Official_Window",
    "Same_Evidence",
    "Separate_Evidence"
]

if not ambiguous_df.empty:

    # IMPORTANT BUG FIX:
    # sort first while Left_End still exists
    ambiguous_sorted = (
        ambiguous_df
        .sort_values(
            [
                "Attack_Family",
                "Left_End"
            ]
        )
    )

    # THEN select display columns.
    print(
        ambiguous_sorted[
            ambiguous_display_cols
        ]
        .to_string(index=False)
    )

else:

    print(
        "✓ No ambiguous boundaries remain."
    )


# ==================================================================================================
# 9.15O — UNION-FIND FOR HIGH-CONFIDENCE SAME-EPISODE MERGES
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15O — BUILDING HIGH-CONFIDENCE PHYSICAL EPISODE GROUPS")
print("=" * 122)


class UnionFind:

    def __init__(self, items):

        self.parent = {
            x: x
            for x in items
        }

    def find(self, x):

        while (
            self.parent[x]
            != x
        ):

            self.parent[x] = (
                self.parent[
                    self.parent[x]
                ]
            )

            x = self.parent[x]

        return x

    def union(self, a, b):

        ra = self.find(a)
        rb = self.find(b)

        if ra != rb:

            self.parent[rb] = ra


all_clusters = (
    signature_df[
        "Observed_Cluster_ID"
    ]
    .astype(str)
    .tolist()
)

uf = UnionFind(
    all_clusters
)

for _, r in same_df.iterrows():

    left_id = str(
        r["Left_Cluster_ID"]
    )

    right_id = str(
        r["Right_Cluster_ID"]
    )

    if (
        left_id in uf.parent
        and
        right_id in uf.parent
    ):

        uf.union(
            left_id,
            right_id
        )


groups = defaultdict(list)

for cluster_id in all_clusters:

    groups[
        uf.find(cluster_id)
    ].append(
        cluster_id
    )


# ==================================================================================================
# 9.15P — ASSIGN PHYSICAL EPISODE CANDIDATE IDS
# ==================================================================================================

episode_rows = []

for family, fg in signature_df.groupby(
    "Attack_Family"
):

    family_clusters = set(
        fg[
            "Observed_Cluster_ID"
        ]
        .astype(str)
    )

    family_groups = []

    for _, members in groups.items():

        members_here = [
            m
            for m in members
            if m in family_clusters
        ]

        if members_here:

            family_groups.append(
                members_here
            )

    family_groups = sorted(

        family_groups,

        key=lambda members:

            fg[
                fg[
                    "Observed_Cluster_ID"
                ]
                .astype(str)
                .isin(members)
            ][
                "Cluster_Start"
            ].min()
    )

    clean_family = (
        re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            str(family)
        )
        .strip("_")
    )

    for episode_number, members in enumerate(
        family_groups,
        start=1
    ):

        sub = fg[
            fg[
                "Observed_Cluster_ID"
            ]
            .astype(str)
            .isin(members)
        ].copy()

        sub = (
            sub
            .sort_values(
                "Cluster_Start"
            )
        )

        episode_id = (
            f"{clean_family}_PE"
            f"{episode_number:03d}"
        )

        episode_rows.append({

            "Attack_Family":
                family,

            "Physical_Episode_Candidate_ID":
                episode_id,

            "First_Observed_Attack_Minute":
                sub[
                    "Cluster_Start"
                ].min(),

            "Last_Observed_Attack_Minute":
                sub[
                    "Cluster_End"
                ].max(),

            "Strict_Clusters_In_Group":
                len(sub),

            "Cluster_IDs":
                tuple(
                    sub[
                        "Observed_Cluster_ID"
                    ]
                    .astype(str)
                    .tolist()
                ),

            "Total_Attack_Flows":
                sub[
                    "Total_Attack_Flows_Rebuilt"
                ].sum(),

            "Total_Malicious_Cluster_Minutes":
                sub[
                    "Observed_Minutes_Rebuilt"
                ].sum(),

            "High_Confidence_Merge_Group":
                bool(
                    len(sub) > 1
                )
        })


episode_df = pd.DataFrame(
    episode_rows
)

print(
    "Physical episode candidate groups:",
    len(episode_df)
)


# ==================================================================================================
# 9.15Q — CLUSTER → PHYSICAL EPISODE MAP
# ==================================================================================================

cluster_episode_rows = []

for _, e in episode_df.iterrows():

    for cluster_id in e[
        "Cluster_IDs"
    ]:

        cluster_episode_rows.append({

            "Attack_Family":
                e[
                    "Attack_Family"
                ],

            "Observed_Cluster_ID":
                str(cluster_id),

            "Physical_Episode_Candidate_ID":
                e[
                    "Physical_Episode_Candidate_ID"
                ]
        })


cluster_episode_map = pd.DataFrame(
    cluster_episode_rows
)


# ==================================================================================================
# 9.15R — FAMILY-LEVEL PHYSICAL EPISODE INVENTORY
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15R — FAMILY-LEVEL PHYSICAL EPISODE INVENTORY")
print("=" * 122)

family_inventory = (
    episode_df
    .groupby(
        "Attack_Family"
    )
    .agg(

        Physical_Episode_Candidates=(
            "Physical_Episode_Candidate_ID",
            "nunique"
        ),

        Total_Strict_Clusters=(
            "Strict_Clusters_In_Group",
            "sum"
        ),

        Total_Attack_Flows=(
            "Total_Attack_Flows",
            "sum"
        ),

        First_Attack=(
            "First_Observed_Attack_Minute",
            "min"
        ),

        Last_Attack=(
            "Last_Observed_Attack_Minute",
            "max"
        )
    )
    .reset_index()
)


if not ambiguous_df.empty:

    ambig_counts = (
        ambiguous_df
        .groupby(
            "Attack_Family"
        )
        .size()
        .rename(
            "Unresolved_Boundaries"
        )
        .reset_index()
    )

    family_inventory = (
        family_inventory
        .merge(
            ambig_counts,
            on="Attack_Family",
            how="left"
        )
    )

else:

    family_inventory[
        "Unresolved_Boundaries"
    ] = 0


family_inventory[
    "Unresolved_Boundaries"
] = (
    family_inventory[
        "Unresolved_Boundaries"
    ]
    .fillna(0)
    .astype(int)
)


family_inventory[
    "Episode_Semantics_Status"
] = np.where(

    family_inventory[
        "Unresolved_Boundaries"
    ]
    == 0,

    "RESOLVED_AS_FAR_AS_CURRENT_EVIDENCE",

    "UNRESOLVED_BOUNDARIES_REMAIN"
)


family_inventory = (
    family_inventory
    .sort_values(
        [
            "Unresolved_Boundaries",
            "Physical_Episode_Candidates",
            "Attack_Family"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


print(
    family_inventory
    .to_string(index=False)
)


# ==================================================================================================
# 9.15S — KNOWN-TYPE FORECASTING FEASIBILITY
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15S — KNOWN-TYPE FORECASTING FEASIBILITY")
print("=" * 122)

family_inventory[
    "At_Least_3_Candidate_Groups"
] = (
    family_inventory[
        "Physical_Episode_Candidates"
    ]
    >= 3
)


family_inventory[
    "CIC_Three_Way_Status"
] = np.select(

    [
        (
            family_inventory[
                "Physical_Episode_Candidates"
            ]
            >= 3
        )
        &
        (
            family_inventory[
                "Unresolved_Boundaries"
            ]
            == 0
        ),

        (
            family_inventory[
                "Physical_Episode_Candidates"
            ]
            >= 3
        )
        &
        (
            family_inventory[
                "Unresolved_Boundaries"
            ]
            > 0
        )
    ],

    [
        "POTENTIALLY_ELIGIBLE_FROM_CIC",
        "COUNT_SUFFICIENT_BUT_SEMANTICS_UNRESOLVED"
    ],

    default=
        "INSUFFICIENT_CIC_EPISODES"
)


forecast_status_cols = [
    "Attack_Family",
    "Physical_Episode_Candidates",
    "Unresolved_Boundaries",
    "CIC_Three_Way_Status"
]

print(
    family_inventory[
        forecast_status_cols
    ]
    .sort_values(
        [
            "CIC_Three_Way_Status",
            "Attack_Family"
        ]
    )
    .to_string(index=False)
)


# ==================================================================================================
# 9.15T — INTERNAL CONSISTENCY CHECKS
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15T — INTERNAL CONSISTENCY CHECKS")
print("=" * 122)

cluster_counts = (
    cluster_episode_map[
        "Observed_Cluster_ID"
    ]
    .value_counts()
)

duplicate_assignment = (
    cluster_counts[
        cluster_counts > 1
    ]
)

mapped_clusters = set(
    cluster_episode_map[
        "Observed_Cluster_ID"
    ]
    .astype(str)
)

expected_clusters = set(
    all_clusters
)

missing_assignment = (
    expected_clusters
    -
    mapped_clusters
)

extra_assignment = (
    mapped_clusters
    -
    expected_clusters
)

print(
    "Strict clusters             :",
    len(expected_clusters)
)

print(
    "Mapped clusters             :",
    len(mapped_clusters)
)

print(
    "Duplicate assignments       :",
    len(duplicate_assignment)
)

print(
    "Missing assignments         :",
    len(missing_assignment)
)

print(
    "Unexpected assignments      :",
    len(extra_assignment)
)


assert (
    len(duplicate_assignment)
    == 0
), (
    "A strict cluster was assigned "
    "to multiple physical episodes."
)

assert (
    len(missing_assignment)
    == 0
), (
    "Some strict clusters were "
    "not assigned to an episode group."
)

assert (
    len(extra_assignment)
    == 0
), (
    "Unexpected cluster IDs appeared "
    "in the episode map."
)

print(
    "✓ Every strict cluster belongs to exactly one "
    "candidate physical-episode group."
)


# ==================================================================================================
# 9.15U — SAVE ALL OUTPUT ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 122)
print("9.15U — SAVING PHASE 9.15 ARTIFACTS")
print("=" * 122)

outputs = {

    "cic_raw_minute_behavior.csv":
        raw_behavior_df,

    "cic_cluster_behavioral_signatures.csv":
        signature_df,

    "cic_physical_episode_boundary_evidence.csv":
        evidence_df,

    "cic_strong_same_episode_boundaries.csv":
        same_df,

    "cic_strong_separate_episode_boundaries.csv":
        separate_df,

    "cic_remaining_ambiguous_episode_boundaries.csv":
        ambiguous_df,

    "cic_physical_episode_candidates.csv":
        episode_df,

    "cic_cluster_to_physical_episode_map.csv":
        cluster_episode_map,

    "cic_physical_episode_family_inventory.csv":
        family_inventory
}


for filename, out_df in outputs.items():

    path = (
        CIC_OUT
        /
        filename
    )

    out_df.to_csv(
        path,
        index=False
    )

    print(
        f"✓ {filename:55s} "
        f"{len(out_df):,} rows"
    )


# ==================================================================================================
# 9.15V — SAVE MACHINE-READABLE STATUS
# ==================================================================================================

status = {

    "phase":
        "9.15",

    "objective":
        (
            "CIC physical attack episode resolution "
            "for known-type forecasting"
        ),

    "strict_clusters":
        int(
            len(signature_df)
        ),

    "same_family_boundaries":
        int(
            len(evidence_df)
        ),

    "strong_same_episode_boundaries":
        int(
            (
                evidence_df[
                    "Episode_Boundary_Assessment"
                ]
                ==
                "SAME_PHYSICAL_EPISODE_STRONG"
            ).sum()
        ),

    "strong_separate_episode_boundaries":
        int(
            (
                evidence_df[
                    "Episode_Boundary_Assessment"
                ]
                ==
                "SEPARATE_PHYSICAL_EPISODES_STRONG"
            ).sum()
        ),

    "ambiguous_boundaries_remaining":
        int(
            (
                evidence_df[
                    "Episode_Boundary_Assessment"
                ]
                ==
                "AMBIGUOUS"
            ).sum()
        ),

    "physical_episode_candidate_groups":
        int(
            len(episode_df)
        ),

    "known_type_forecasting_only":
        True,

    "train_val_test_created":
        False,

    "forecasting_windows_created":
        False,

    "cic_unsw_combined":
        False,

    # This means episode semantics are completely resolved
    # according to THIS phase's evidence, NOT that the final
    # forecasting dataset is ready.
    "all_cic_boundaries_resolved":
        bool(
            len(ambiguous_df)
            == 0
        )
}


status_path = (
    CIC_OUT
    /
    "cic_phase_9_15_status.json"
)

with open(
    status_path,
    "w"
) as f:

    json.dump(
        status,
        f,
        indent=2,
        default=str
    )


print(
    "✓ cic_phase_9_15_status.json"
)


# ==================================================================================================
# 9.15W — FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 122)
print("PHASE 9.15 — FINAL REPORT")
print("=" * 122)

same_count = int(
    (
        evidence_df[
            "Episode_Boundary_Assessment"
        ]
        ==
        "SAME_PHYSICAL_EPISODE_STRONG"
    ).sum()
)

separate_count = int(
    (
        evidence_df[
            "Episode_Boundary_Assessment"
        ]
        ==
        "SEPARATE_PHYSICAL_EPISODES_STRONG"
    ).sum()
)

ambiguous_count = int(
    (
        evidence_df[
            "Episode_Boundary_Assessment"
        ]
        ==
        "AMBIGUOUS"
    ).sum()
)


print(
f"""
STRICT MALICIOUS CLUSTERS
-------------------------
{len(signature_df):,}

SAME-FAMILY BOUNDARIES INVESTIGATED
-----------------------------------
{len(evidence_df):,}

STRONG SAME-PHYSICAL-EPISODE BOUNDARIES
---------------------------------------
{same_count:,}

STRONG SEPARATE-PHYSICAL-EPISODE BOUNDARIES
-------------------------------------------
{separate_count:,}

REMAINING AMBIGUOUS BOUNDARIES
------------------------------
{ambiguous_count:,}

CURRENT CONSERVATIVE PHYSICAL-EPISODE CANDIDATE GROUPS
------------------------------------------------------
{len(episode_df):,}
"""
)


print("=" * 122)
print("KNOWN-TYPE FORECASTING FAMILY STATUS")
print("=" * 122)

print(
    family_inventory[
        [
            "Attack_Family",
            "Physical_Episode_Candidates",
            "Unresolved_Boundaries",
            "CIC_Three_Way_Status"
        ]
    ]
    .sort_values(
        "Attack_Family"
    )
    .to_string(index=False)
)


print("\n" + "=" * 122)
print("INTERPRETATION")
print("=" * 122)

if ambiguous_count == 0:

    print(
"""
All same-family CIC cluster boundaries have been resolved
as far as the evidence used in Phase 9.15 allows.

IMPORTANT:

This DOES NOT mean we should create TRAIN / VAL / TEST now.

The correct next sequence remains:

    CIC episode audit
            ✓
            ↓
    UNSW structural + temporal audit
            ↓
    UNSW episode inventory
            ↓
    CIC ↔ UNSW attack-family harmonization
            ↓
    combined known-family episode sufficiency
            ↓
    feature harmonization
            ↓
    episode-aware TRAIN / VAL / TEST
            ↓
    forecasting-window generation
            ↓
    leakage audit
            ↓
    baseline forecasting models

No model training should happen yet.
"""
    )

else:

    print(
f"""
{ambiguous_count:,} CIC same-family boundaries remain unresolved.

THIS IS NOT AN ERROR.

It means the currently available evidence does not justify
forcing those boundaries into either:

    SAME ATTACK

or

    DIFFERENT ATTACKS

For our known-type forecasting experiment:

    strict cluster
        !=
    independent attack episode

and:

    ambiguous boundary
        !=
    permission to manufacture another independent episode.

The unresolved information has therefore been preserved
explicitly instead of silently forcing a decision.

We should inspect the final FAMILY STATUS from this phase,
then decide whether any small CIC-specific resolution step
is still scientifically necessary before freezing CIC and
moving to the UNSW audit.
"""
    )


print("=" * 122)
print("PHASE 9.15 COMPLETE")
print("=" * 122)

In [ ]:
# ==================================================================================================
# PHASE 9.16 — CIC AMBIGUOUS PHYSICAL-EPISODE BOUNDARY FORENSICS
#
# PURPOSE
# -------
# Investigate the 12 boundaries that Phase 9.15 could not resolve.
#
# IMPORTANT:
#   * This phase DOES NOT force MERGE/SEPARATE decisions.
#   * It collects stronger raw-flow evidence at the finest timestamp resolution available.
#   * Raw CIC files are NEVER modified.
#   * Decisions remain evidence-driven.
#
# OUTPUT
# ------
# cic_phase_9_16_ambiguous_boundary_forensics.csv
# cic_phase_9_16_boundary_flow_samples.csv
# cic_phase_9_16_family_summary.csv
#
# ==================================================================================================

import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)

print("=" * 122)
print("PHASE 9.16 — CIC AMBIGUOUS PHYSICAL-EPISODE BOUNDARY FORENSICS")
print("=" * 122)


# ==================================================================================================
# 9.16A — PATH DISCOVERY
# ==================================================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2")

CIC_RAW_DIR = PROJECT_ROOT / "raw" / "CIC_IDS2017"
CIC_OUT_DIR = PROJECT_ROOT / "audit_outputs" / "CIC"

if not CIC_RAW_DIR.exists():
    raise FileNotFoundError(f"CIC raw directory not found:\n{CIC_RAW_DIR}")

if not CIC_OUT_DIR.exists():
    raise FileNotFoundError(f"CIC output directory not found:\n{CIC_OUT_DIR}")

AMBIG_PATH = CIC_OUT_DIR / "cic_remaining_ambiguous_episode_boundaries.csv"
CLUSTER_PATH = CIC_OUT_DIR / "cic_cluster_behavioral_signatures.csv"

for p in [AMBIG_PATH, CLUSTER_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required Phase 9.15 artifact missing:\n{p}")

print("\n✓ CIC raw directory found")
print("✓ Phase 9.15 artifacts found")


# ==================================================================================================
# 9.16B — LOAD PHASE 9.15 ARTIFACTS
# ==================================================================================================

ambig = pd.read_csv(AMBIG_PATH)
clusters = pd.read_csv(CLUSTER_PATH)

for c in ["Cluster_Start", "Cluster_End"]:
    if c in clusters.columns:
        clusters[c] = pd.to_datetime(clusters[c], errors="coerce")

print("\n" + "=" * 122)
print("9.16B — INPUT CHECK")
print("=" * 122)

print(f"Ambiguous boundaries : {len(ambig):,}")
print(f"Cluster signatures   : {len(clusters):,}")

if len(ambig) != 12:
    print(
        f"\n⚠ Expected 12 ambiguous boundaries from Phase 9.15, "
        f"but found {len(ambig)}."
    )
else:
    print("✓ Exactly 12 unresolved boundaries recovered.")


# ==================================================================================================
# 9.16C — RAW FILE ENCODING MAP
# ==================================================================================================

ENCODING_MAP = {
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv": "utf-8",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv": "utf-8",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv": "utf-8",
    "Monday-WorkingHours.pcap_ISCX.csv": "utf-8",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv": "utf-8",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv": "cp1252",
    "Tuesday-WorkingHours.pcap_ISCX.csv": "utf-8",
    "Wednesday-workingHours.pcap_ISCX.csv": "utf-8",
}

print("\n✓ Encoding map restored from Phase 9.15.")


# ==================================================================================================
# 9.16D — HELPERS
# ==================================================================================================

def clean_columns(df):
    """
    Preserve raw names conceptually but remove surrounding whitespace
    so column access is stable.
    """
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def parse_cic_timestamp(series):
    """
    CIC files contain mixed timestamp formatting.
    dayfirst=True matches the previously established CIC semantics.
    """
    return pd.to_datetime(
        series.astype(str).str.strip(),
        errors="coerce",
        dayfirst=True
    )


def normalize_label(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def canonical_pair(src, dst):
    """
    Undirected endpoint pair used only as supporting evidence.
    Directional source/destination values are retained separately.
    """
    if pd.isna(src) or pd.isna(dst):
        return None
    a = str(src).strip()
    b = str(dst).strip()
    return tuple(sorted([a, b]))


def safe_jaccard(a, b):
    a = set(x for x in a if pd.notna(x))
    b = set(x for x in b if pd.notna(x))

    if len(a) == 0 and len(b) == 0:
        return np.nan
    if len(a | b) == 0:
        return np.nan

    return len(a & b) / len(a | b)


def get_cluster_row(cluster_id):
    m = clusters[clusters["Observed_Cluster_ID"] == cluster_id]

    if len(m) != 1:
        raise ValueError(
            f"Expected exactly one cluster row for {cluster_id}, found {len(m)}"
        )

    return m.iloc[0]


# ==================================================================================================
# 9.16E — LOAD ONLY RAW FILES REQUIRED BY AMBIGUOUS BOUNDARIES
# ==================================================================================================

required_segments = sorted(
    set(
        clusters[
            clusters["Observed_Cluster_ID"].isin(
                set(ambig["Left_Cluster_ID"]) |
                set(ambig["Right_Cluster_ID"])
            )
        ]["Segment_ID"]
    )
)

print("\n" + "=" * 122)
print("9.16E — RAW FILES REQUIRED FOR THE 12 BOUNDARIES")
print("=" * 122)

for x in required_segments:
    print(x)

raw_cache = {}

USECOL_CANDIDATES = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Label",
]

for segment in required_segments:

    path = CIC_RAW_DIR / segment

    if not path.exists():
        raise FileNotFoundError(path)

    enc = ENCODING_MAP.get(segment, "utf-8")

    print(f"\nLoading: {segment}")

    # Read header first so we don't assume whitespace conventions
    header = pd.read_csv(path, encoding=enc, nrows=0)
    header.columns = [str(c).strip() for c in header.columns]

    available = set(header.columns)
    usecols = [c for c in USECOL_CANDIDATES if c in available]

    # Need a mapping back to original raw column names because pandas usecols
    # is applied before our whitespace stripping.
    original_header = pd.read_csv(path, encoding=enc, nrows=0).columns.tolist()

    normalized_to_original = {
        str(c).strip(): c for c in original_header
    }

    raw_usecols = [
        normalized_to_original[c]
        for c in usecols
        if c in normalized_to_original
    ]

    df = pd.read_csv(
        path,
        encoding=enc,
        usecols=raw_usecols,
        low_memory=False
    )

    df = clean_columns(df)

    # Remove the known completely empty structural rows analytically.
    df = df.dropna(how="all").copy()

    if "Timestamp" not in df.columns:
        raise ValueError(f"Timestamp missing from {segment}")

    if "Label" not in df.columns:
        raise ValueError(f"Label missing from {segment}")

    df["Parsed_Timestamp"] = parse_cic_timestamp(df["Timestamp"])

    df = df[df["Parsed_Timestamp"].notna()].copy()

    # Keep original raw-row order for forensic reference,
    # while also allowing chronological sorting.
    df["Raw_Row_Index"] = df.index

    raw_cache[segment] = df

    print(f"  usable rows loaded : {len(df):,}")
    print(f"  valid timestamps   : {df['Parsed_Timestamp'].notna().sum():,}")


# ==================================================================================================
# 9.16F — FORENSIC ANALYSIS OF EACH AMBIGUOUS BOUNDARY
# ==================================================================================================

print("\n" + "=" * 122)
print("9.16F — ANALYSING THE 12 AMBIGUOUS BOUNDARIES")
print("=" * 122)

forensic_rows = []
sample_rows = []

for idx, boundary in ambig.iterrows():

    family = boundary["Attack_Family"]
    left_id = boundary["Left_Cluster_ID"]
    right_id = boundary["Right_Cluster_ID"]

    left = get_cluster_row(left_id)
    right = get_cluster_row(right_id)

    segment = left["Segment_ID"]

    if segment != right["Segment_ID"]:
        raise ValueError(
            f"Boundary unexpectedly crosses segments: {left_id} → {right_id}"
        )

    df = raw_cache[segment]

    left_start = pd.Timestamp(left["Cluster_Start"])
    left_end = pd.Timestamp(left["Cluster_End"])
    right_start = pd.Timestamp(right["Cluster_Start"])
    right_end = pd.Timestamp(right["Cluster_End"])

    # ----------------------------------------------------------------------
    # Use the actual cluster minute extent.
    #
    # Since cluster times are minute-binned, include the entire final minute:
    # [start, end + 1 minute)
    # ----------------------------------------------------------------------

    left_mask = (
        (df["Parsed_Timestamp"] >= left_start) &
        (df["Parsed_Timestamp"] < left_end + pd.Timedelta(minutes=1))
    )

    right_mask = (
        (df["Parsed_Timestamp"] >= right_start) &
        (df["Parsed_Timestamp"] < right_end + pd.Timedelta(minutes=1))
    )

    # Strict temporal gap BETWEEN the two malicious clusters
    gap_start = left_end + pd.Timedelta(minutes=1)
    gap_end = right_start

    gap_mask = (
        (df["Parsed_Timestamp"] >= gap_start) &
        (df["Parsed_Timestamp"] < gap_end)
    )

    left_df = df.loc[left_mask].copy()
    right_df = df.loc[right_mask].copy()
    gap_df = df.loc[gap_mask].copy()

    # Attack-only records belonging to this family.
    left_attack = left_df[
        left_df["Label"].map(normalize_label) == normalize_label(family)
    ].copy()

    right_attack = right_df[
        right_df["Label"].map(normalize_label) == normalize_label(family)
    ].copy()

    # Gap label distribution
    gap_labels = (
        gap_df["Label"]
        .map(normalize_label)
        .replace("", "<EMPTY>")
        .value_counts()
        .to_dict()
    )

    # ----------------------------------------------------------------------
    # Endpoint continuity
    # ----------------------------------------------------------------------

    left_sources = set(left_attack.get("Source IP", pd.Series(dtype=str)).dropna().astype(str))
    right_sources = set(right_attack.get("Source IP", pd.Series(dtype=str)).dropna().astype(str))

    left_dest = set(left_attack.get("Destination IP", pd.Series(dtype=str)).dropna().astype(str))
    right_dest = set(right_attack.get("Destination IP", pd.Series(dtype=str)).dropna().astype(str))

    left_pairs = set(
        canonical_pair(s, d)
        for s, d in zip(
            left_attack.get("Source IP", pd.Series(dtype=str)),
            left_attack.get("Destination IP", pd.Series(dtype=str))
        )
    )
    left_pairs.discard(None)

    right_pairs = set(
        canonical_pair(s, d)
        for s, d in zip(
            right_attack.get("Source IP", pd.Series(dtype=str)),
            right_attack.get("Destination IP", pd.Series(dtype=str))
        )
    )
    right_pairs.discard(None)

    # ----------------------------------------------------------------------
    # Exact finest-resolution attack timestamps available in raw CSV
    # ----------------------------------------------------------------------

    left_last_attack_ts = (
        left_attack["Parsed_Timestamp"].max()
        if len(left_attack)
        else pd.NaT
    )

    right_first_attack_ts = (
        right_attack["Parsed_Timestamp"].min()
        if len(right_attack)
        else pd.NaT
    )

    if pd.notna(left_last_attack_ts) and pd.notna(right_first_attack_ts):
        exact_attack_gap_sec = (
            right_first_attack_ts - left_last_attack_ts
        ).total_seconds()
    else:
        exact_attack_gap_sec = np.nan

    # ----------------------------------------------------------------------
    # Flow-ID overlap
    # ----------------------------------------------------------------------

    if "Flow ID" in df.columns:
        left_flow_ids = set(left_attack["Flow ID"].dropna().astype(str))
        right_flow_ids = set(right_attack["Flow ID"].dropna().astype(str))
        flow_id_jaccard = safe_jaccard(left_flow_ids, right_flow_ids)
        shared_flow_ids = len(left_flow_ids & right_flow_ids)
    else:
        flow_id_jaccard = np.nan
        shared_flow_ids = np.nan

    # ----------------------------------------------------------------------
    # Directional endpoint tuples
    # ----------------------------------------------------------------------

    def directional_pairs(frame):
        if "Source IP" not in frame.columns or "Destination IP" not in frame.columns:
            return set()

        return set(
            zip(
                frame["Source IP"].dropna().astype(str),
                frame.loc[frame["Source IP"].notna(), "Destination IP"].astype(str)
            )
        )

    # safer implementation preserving aligned rows
    def aligned_directional_pairs(frame):
        if "Source IP" not in frame.columns or "Destination IP" not in frame.columns:
            return set()

        tmp = frame[["Source IP", "Destination IP"]].dropna()

        return set(
            (str(s), str(d))
            for s, d in tmp.itertuples(index=False, name=None)
        )

    left_directional = aligned_directional_pairs(left_attack)
    right_directional = aligned_directional_pairs(right_attack)

    directional_pair_jaccard = safe_jaccard(
        left_directional,
        right_directional
    )

    # ----------------------------------------------------------------------
    # Ports and protocol continuity
    # ----------------------------------------------------------------------

    left_src_ports = set(
        left_attack.get("Source Port", pd.Series(dtype=float)).dropna().astype(str)
    )
    right_src_ports = set(
        right_attack.get("Source Port", pd.Series(dtype=float)).dropna().astype(str)
    )

    left_dst_ports = set(
        left_attack.get("Destination Port", pd.Series(dtype=float)).dropna().astype(str)
    )
    right_dst_ports = set(
        right_attack.get("Destination Port", pd.Series(dtype=float)).dropna().astype(str)
    )

    left_protocols = set(
        left_attack.get("Protocol", pd.Series(dtype=float)).dropna().astype(str)
    )
    right_protocols = set(
        right_attack.get("Protocol", pd.Series(dtype=float)).dropna().astype(str)
    )

    src_port_jaccard = safe_jaccard(left_src_ports, right_src_ports)
    dst_port_jaccard = safe_jaccard(left_dst_ports, right_dst_ports)
    protocol_jaccard = safe_jaccard(left_protocols, right_protocols)

    # ----------------------------------------------------------------------
    # Gap traffic behaviour
    # ----------------------------------------------------------------------

    gap_flow_count = len(gap_df)

    benign_gap = gap_df[
        gap_df["Label"].map(normalize_label).str.upper() == "BENIGN"
    ].copy()

    malicious_gap = gap_df[
        gap_df["Label"].map(normalize_label).str.upper() != "BENIGN"
    ].copy()

    malicious_gap = malicious_gap[
        malicious_gap["Label"].map(normalize_label) != ""
    ]

    gap_attack_family_rows = (
        gap_df["Label"].map(normalize_label) == normalize_label(family)
    ).sum()

    other_attack_gap_rows = len(
        malicious_gap[
            malicious_gap["Label"].map(normalize_label) != normalize_label(family)
        ]
    )

    # ----------------------------------------------------------------------
    # Pre/post local traffic intensity
    #
    # Compare 5 minutes immediately before left attack and after right attack.
    # This is descriptive evidence only.
    # ----------------------------------------------------------------------

    pre_start = left_start - pd.Timedelta(minutes=5)

    pre_df = df[
        (df["Parsed_Timestamp"] >= pre_start) &
        (df["Parsed_Timestamp"] < left_start)
    ]

    post_end = right_end + pd.Timedelta(minutes=6)

    post_df = df[
        (df["Parsed_Timestamp"] >= right_end + pd.Timedelta(minutes=1)) &
        (df["Parsed_Timestamp"] < post_end)
    ]

    pre_flow_count = len(pre_df)
    post_flow_count = len(post_df)

    # ----------------------------------------------------------------------
    # Fine-grained attack intensity around boundary
    # ----------------------------------------------------------------------

    left_last_minute = left_attack[
        left_attack["Parsed_Timestamp"].dt.floor("min") == left_end
    ]

    right_first_minute = right_attack[
        right_attack["Parsed_Timestamp"].dt.floor("min") == right_start
    ]

    left_edge_attack_flows = len(left_last_minute)
    right_edge_attack_flows = len(right_first_minute)

    # ----------------------------------------------------------------------
    # Evidence flags
    #
    # IMPORTANT:
    # These are observations, NOT automatic episode decisions.
    # ----------------------------------------------------------------------

    endpoint_continuity = (
        safe_jaccard(left_pairs, right_pairs)
        if len(left_pairs) or len(right_pairs)
        else np.nan
    )

    very_long_gap = (
        pd.notna(exact_attack_gap_sec)
        and exact_attack_gap_sec >= 30 * 60
    )

    moderate_gap = (
        pd.notna(exact_attack_gap_sec)
        and 10 * 60 <= exact_attack_gap_sec < 30 * 60
    )

    near_continuous = (
        pd.notna(exact_attack_gap_sec)
        and exact_attack_gap_sec <= 5 * 60
    )

    strong_endpoint_continuity = (
        pd.notna(endpoint_continuity)
        and endpoint_continuity >= 0.8
    )

    no_endpoint_continuity = (
        pd.notna(endpoint_continuity)
        and endpoint_continuity == 0
    )

    # ----------------------------------------------------------------------
    # Forensic interpretation
    #
    # Do NOT force a final physical-episode label.
    # ----------------------------------------------------------------------

    same_signals = []
    separate_signals = []

    if near_continuous:
        same_signals.append("raw_attack_gap_<=5min")

    if strong_endpoint_continuity:
        same_signals.append("strong_endpoint_continuity")

    if pd.notna(directional_pair_jaccard) and directional_pair_jaccard >= 0.8:
        same_signals.append("strong_directional_pair_continuity")

    if pd.notna(dst_port_jaccard) and dst_port_jaccard >= 0.8:
        same_signals.append("strong_destination_port_continuity")

    if pd.notna(protocol_jaccard) and protocol_jaccard >= 0.8:
        same_signals.append("protocol_continuity")

    if very_long_gap:
        separate_signals.append("raw_attack_gap_>=30min")
    elif moderate_gap:
        separate_signals.append("raw_attack_gap_10_to_30min")

    if no_endpoint_continuity:
        separate_signals.append("no_endpoint_continuity")

    if gap_flow_count > 0 and gap_attack_family_rows == 0:
        separate_signals.append("no_same_family_attack_flows_inside_gap")

    if other_attack_gap_rows > 0:
        separate_signals.append("other_attack_family_present_inside_gap")

    # This is deliberately NOT MERGE/SEPARATE.
    if len(same_signals) > 0 and len(separate_signals) == 0:
        forensic_status = "CONTINUITY_EVIDENCE_DOMINANT"
    elif len(separate_signals) > 0 and len(same_signals) == 0:
        forensic_status = "SEPARATION_EVIDENCE_DOMINANT"
    elif len(same_signals) > 0 and len(separate_signals) > 0:
        forensic_status = "CONFLICTING_EVIDENCE"
    else:
        forensic_status = "INSUFFICIENT_ADDITIONAL_EVIDENCE"

    forensic_rows.append({
        "Attack_Family": family,
        "Segment_ID": segment,

        "Left_Cluster_ID": left_id,
        "Right_Cluster_ID": right_id,

        "Left_Start": left_start,
        "Left_End": left_end,
        "Right_Start": right_start,
        "Right_End": right_end,

        "Phase_9_15_Benign_Gap_Minutes":
            boundary.get("Benign_Gap_Minutes", np.nan),

        "Left_Attack_Flows_Raw": len(left_attack),
        "Right_Attack_Flows_Raw": len(right_attack),

        "Left_Last_Attack_Timestamp": left_last_attack_ts,
        "Right_First_Attack_Timestamp": right_first_attack_ts,

        "Exact_Attack_Gap_Seconds": exact_attack_gap_sec,
        "Exact_Attack_Gap_Minutes":
            exact_attack_gap_sec / 60
            if pd.notna(exact_attack_gap_sec)
            else np.nan,

        "Source_Jaccard_Raw":
            safe_jaccard(left_sources, right_sources),

        "Destination_Jaccard_Raw":
            safe_jaccard(left_dest, right_dest),

        "Undirected_Pair_Jaccard_Raw":
            endpoint_continuity,

        "Directional_Pair_Jaccard_Raw":
            directional_pair_jaccard,

        "Source_Port_Jaccard":
            src_port_jaccard,

        "Destination_Port_Jaccard":
            dst_port_jaccard,

        "Protocol_Jaccard":
            protocol_jaccard,

        "Shared_Flow_ID_Count":
            shared_flow_ids,

        "Flow_ID_Jaccard":
            flow_id_jaccard,

        "Gap_Total_Flows":
            gap_flow_count,

        "Gap_Benign_Flows":
            len(benign_gap),

        "Gap_Same_Family_Attack_Flows":
            int(gap_attack_family_rows),

        "Gap_Other_Attack_Flows":
            int(other_attack_gap_rows),

        "Gap_Label_Distribution":
            json.dumps(gap_labels, ensure_ascii=False),

        "Left_Edge_Attack_Flows":
            left_edge_attack_flows,

        "Right_Edge_Attack_Flows":
            right_edge_attack_flows,

        "Pre_5min_Total_Flows":
            pre_flow_count,

        "Post_5min_Total_Flows":
            post_flow_count,

        "Shares_Official_Window":
            boundary.get("Shares_Official_Window", np.nan),

        "Phase_9_15_Gap_vs_Baseline_Similarity":
            boundary.get("Gap_vs_Baseline_Similarity", np.nan),

        "Phase_9_15_Attack_Intensity_Similarity":
            boundary.get("Attack_Intensity_Similarity", np.nan),

        "Continuity_Signals":
            "; ".join(same_signals),

        "Separation_Signals":
            "; ".join(separate_signals),

        "Phase_9_16_Forensic_Status":
            forensic_status,

        # Critical:
        # We intentionally DO NOT assign SAME/SEPARATE here.
        "Final_Episode_Decision":
            "UNRESOLVED_PENDING_INTERPRETATION"
    })

    # ----------------------------------------------------------------------
    # Save a small chronological sample around each boundary
    # for manual inspection if needed.
    # ----------------------------------------------------------------------

    context_start = left_end - pd.Timedelta(minutes=2)
    context_end = right_start + pd.Timedelta(minutes=3)

    context = df[
        (df["Parsed_Timestamp"] >= context_start) &
        (df["Parsed_Timestamp"] < context_end)
    ].copy()

    keep = [
        c for c in [
            "Raw_Row_Index",
            "Parsed_Timestamp",
            "Flow ID",
            "Source IP",
            "Source Port",
            "Destination IP",
            "Destination Port",
            "Protocol",
            "Flow Duration",
            "Flow Bytes/s",
            "Flow Packets/s",
            "Label",
        ]
        if c in context.columns
    ]

    context = context[keep]

    context.insert(0, "Attack_Family_Under_Review", family)
    context.insert(1, "Left_Cluster_ID", left_id)
    context.insert(2, "Right_Cluster_ID", right_id)
    context.insert(3, "Segment_ID", segment)

    sample_rows.append(context)

    print(
        f"{family:28s} | "
        f"{left_id} → {right_id} | "
        f"raw gap={exact_attack_gap_sec / 60 if pd.notna(exact_attack_gap_sec) else np.nan:.2f} min | "
        f"{forensic_status}"
    )


# ==================================================================================================
# 9.16G — BUILD FORENSIC TABLE
# ==================================================================================================

forensic_df = pd.DataFrame(forensic_rows)

print("\n" + "=" * 122)
print("9.16G — FORENSIC STATUS")
print("=" * 122)

print(
    forensic_df[
        [
            "Attack_Family",
            "Left_Cluster_ID",
            "Right_Cluster_ID",
            "Exact_Attack_Gap_Minutes",
            "Undirected_Pair_Jaccard_Raw",
            "Directional_Pair_Jaccard_Raw",
            "Destination_Port_Jaccard",
            "Gap_Total_Flows",
            "Gap_Same_Family_Attack_Flows",
            "Gap_Other_Attack_Flows",
            "Shares_Official_Window",
            "Phase_9_16_Forensic_Status",
        ]
    ].to_string(index=False)
)


# ==================================================================================================
# 9.16H — STATUS COUNTS
# ==================================================================================================

print("\n" + "=" * 122)
print("9.16H — STATUS COUNTS")
print("=" * 122)

print(
    forensic_df["Phase_9_16_Forensic_Status"]
    .value_counts(dropna=False)
)


# ==================================================================================================
# 9.16I — FAMILY SUMMARY
# ==================================================================================================

family_summary = (
    forensic_df
    .groupby("Attack_Family", as_index=False)
    .agg(
        Ambiguous_Boundaries=("Left_Cluster_ID", "count"),
        Median_Exact_Gap_Minutes=("Exact_Attack_Gap_Minutes", "median"),
        Min_Exact_Gap_Minutes=("Exact_Attack_Gap_Minutes", "min"),
        Max_Exact_Gap_Minutes=("Exact_Attack_Gap_Minutes", "max"),
    )
)

status_counts = (
    forensic_df
    .pivot_table(
        index="Attack_Family",
        columns="Phase_9_16_Forensic_Status",
        values="Left_Cluster_ID",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

family_summary = family_summary.merge(
    status_counts,
    on="Attack_Family",
    how="left"
)

print("\n" + "=" * 122)
print("9.16I — FAMILY-LEVEL SUMMARY")
print("=" * 122)

print(family_summary.to_string(index=False))


# ==================================================================================================
# 9.16J — SAVE RAW BOUNDARY CONTEXT
# ==================================================================================================

if sample_rows:
    boundary_samples_df = pd.concat(
        sample_rows,
        ignore_index=True
    )
else:
    boundary_samples_df = pd.DataFrame()


# ==================================================================================================
# 9.16K — INTERNAL CONSISTENCY CHECKS
# ==================================================================================================

print("\n" + "=" * 122)
print("9.16K — INTERNAL CONSISTENCY CHECKS")
print("=" * 122)

assert len(forensic_df) == len(ambig), (
    "Forensic result count does not equal ambiguous boundary count."
)

duplicate_boundaries = forensic_df.duplicated(
    subset=[
        "Attack_Family",
        "Left_Cluster_ID",
        "Right_Cluster_ID"
    ]
).sum()

print(f"Input ambiguous boundaries : {len(ambig):,}")
print(f"Forensic output rows       : {len(forensic_df):,}")
print(f"Duplicate boundaries       : {duplicate_boundaries:,}")

assert duplicate_boundaries == 0

print("✓ Every ambiguous boundary was analysed exactly once.")


# ==================================================================================================
# 9.16L — SAVE ARTIFACTS
# ==================================================================================================

FORENSIC_OUT = CIC_OUT_DIR / "cic_phase_9_16_ambiguous_boundary_forensics.csv"
SAMPLE_OUT = CIC_OUT_DIR / "cic_phase_9_16_boundary_flow_samples.csv"
FAMILY_OUT = CIC_OUT_DIR / "cic_phase_9_16_family_summary.csv"

forensic_df.to_csv(
    FORENSIC_OUT,
    index=False
)

boundary_samples_df.to_csv(
    SAMPLE_OUT,
    index=False
)

family_summary.to_csv(
    FAMILY_OUT,
    index=False
)

status = {
    "phase": "9.16",
    "ambiguous_boundaries_input": int(len(ambig)),
    "forensic_boundaries_output": int(len(forensic_df)),
    "status_counts": {
        str(k): int(v)
        for k, v in forensic_df[
            "Phase_9_16_Forensic_Status"
        ].value_counts().to_dict().items()
    },
    "automatic_merge_decisions": 0,
    "automatic_separate_decisions": 0,
    "raw_files_modified": False,
}

STATUS_OUT = CIC_OUT_DIR / "cic_phase_9_16_status.json"

with open(STATUS_OUT, "w") as f:
    json.dump(status, f, indent=2)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 122)
print("PHASE 9.16 — FINAL REPORT")
print("=" * 122)

print(f"""
Ambiguous boundaries received from 9.15 : {len(ambig):,}
Boundaries forensically analysed        : {len(forensic_df):,}

Automatic MERGE decisions               : 0
Automatic SEPARATE decisions            : 0

Raw CIC files modified                  : NO

Saved:
  ✓ {FORENSIC_OUT.name}
  ✓ {SAMPLE_OUT.name}
  ✓ {FAMILY_OUT.name}
  ✓ {STATUS_OUT.name}

IMPORTANT
---------
This phase intentionally collected stronger evidence WITHOUT
manufacturing physical attack episodes.

The Phase_9_16_Forensic_Status field describes whether the newly
examined evidence leans toward:

    CONTINUITY_EVIDENCE_DOMINANT
    SEPARATION_EVIDENCE_DOMINANT
    CONFLICTING_EVIDENCE
    INSUFFICIENT_ADDITIONAL_EVIDENCE

These are NOT yet final MERGE / SEPARATE labels.

We will inspect this output once before freezing the CIC episode
inventory.
""")

print("=" * 122)
print("PHASE 9.16 COMPLETE")
print("=" * 122)

In [ ]:
# ==================================================================================================
# PHASE 9.17 — CORRECTED
# CIC FREEZE + CONSERVATIVE LEAKAGE GROUPS
#
# IMPORTANT CORRECTION
# --------------------
# The previous 9.17 attempted to reconstruct Phase 9.15 physical episodes from
# boundary files. That reconstruction was unnecessary and produced 82 physical
# episodes for 82 strict clusters.
#
# THIS VERSION:
#
#   1. Uses the SAVED Phase 9.15 cluster -> physical episode mapping directly.
#   2. Validates it against the saved Phase 9.15 physical episode inventory.
#   3. Keeps the 12 unresolved boundaries unresolved.
#   4. Builds conservative leakage groups from:
#
#          confirmed physical episode membership
#                        +
#          unresolved/ambiguous dependency boundaries
#
#   5. Freezes CIC only if all consistency checks pass.
#
# NO MODELING OR ML PREPROCESSING OCCURS HERE.
# ==================================================================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 150)

print("=" * 130)
print("PHASE 9.17 — CORRECTED CIC FREEZE + CONSERVATIVE LEAKAGE GROUPS")
print("=" * 130)


# ==================================================================================================
# 9.17A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

CIC_OUT_DIR = PROJECT_ROOT / "audit_outputs" / "CIC"

CLUSTER_PATH = (
    CIC_OUT_DIR /
    "cic_cluster_behavioral_signatures.csv"
)

PHYSICAL_MAP_PATH = (
    CIC_OUT_DIR /
    "cic_cluster_to_physical_episode_map.csv"
)

PHYSICAL_CANDIDATES_PATH = (
    CIC_OUT_DIR /
    "cic_physical_episode_candidates.csv"
)

PHYSICAL_FAMILY_INVENTORY_PATH = (
    CIC_OUT_DIR /
    "cic_physical_episode_family_inventory.csv"
)

AMBIG_PATH = (
    CIC_OUT_DIR /
    "cic_remaining_ambiguous_episode_boundaries.csv"
)

FORENSIC_PATH = (
    CIC_OUT_DIR /
    "cic_phase_9_16_ambiguous_boundary_forensics.csv"
)

SAME_PATH = (
    CIC_OUT_DIR /
    "cic_strong_same_episode_boundaries.csv"
)

SEPARATE_PATH = (
    CIC_OUT_DIR /
    "cic_strong_separate_episode_boundaries.csv"
)


required_paths = {
    "strict cluster signatures":
        CLUSTER_PATH,

    "Phase 9.15 cluster→physical episode map":
        PHYSICAL_MAP_PATH,

    "Phase 9.15 physical episode candidates":
        PHYSICAL_CANDIDATES_PATH,

    "Phase 9.15 physical family inventory":
        PHYSICAL_FAMILY_INVENTORY_PATH,

    "remaining ambiguous boundaries":
        AMBIG_PATH,

    "Phase 9.16 forensics":
        FORENSIC_PATH,

    "strong SAME boundaries":
        SAME_PATH,

    "strong SEPARATE boundaries":
        SEPARATE_PATH,
}


print("\n" + "=" * 130)
print("9.17A — REQUIRED ARTIFACT CHECK")
print("=" * 130)

for label, path in required_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"\nMissing required artifact:\n"
            f"{label}\n"
            f"{path}"
        )

    print(f"✓ {label:45s} {path.name}")


# ==================================================================================================
# 9.17B — LOAD SAVED PHASE 9.15 / 9.16 ARTIFACTS
# ==================================================================================================

clusters = pd.read_csv(CLUSTER_PATH)

physical_map = pd.read_csv(
    PHYSICAL_MAP_PATH
)

physical_candidates = pd.read_csv(
    PHYSICAL_CANDIDATES_PATH
)

physical_family_inventory = pd.read_csv(
    PHYSICAL_FAMILY_INVENTORY_PATH
)

ambiguous = pd.read_csv(
    AMBIG_PATH
)

forensics = pd.read_csv(
    FORENSIC_PATH
)

strong_same = pd.read_csv(
    SAME_PATH
)

strong_separate = pd.read_csv(
    SEPARATE_PATH
)


for c in ["Cluster_Start", "Cluster_End"]:
    if c in clusters.columns:
        clusters[c] = pd.to_datetime(
            clusters[c],
            errors="coerce"
        )


print("\n" + "=" * 130)
print("9.17B — INPUT SHAPES")
print("=" * 130)

print(
    f"Strict clusters                  : {len(clusters):,}"
)

print(
    f"Cluster→physical map rows        : {len(physical_map):,}"
)

print(
    f"Physical episode candidates      : {len(physical_candidates):,}"
)

print(
    f"Physical family inventory rows   : {len(physical_family_inventory):,}"
)

print(
    f"Strong SAME boundaries           : {len(strong_same):,}"
)

print(
    f"Strong SEPARATE boundaries       : {len(strong_separate):,}"
)

print(
    f"Remaining ambiguous boundaries   : {len(ambiguous):,}"
)

print(
    f"Phase 9.16 forensic rows         : {len(forensics):,}"
)


# ==================================================================================================
# 9.17C — SHOW PHYSICAL MAP SCHEMA
#
# We inspect this because THIS is the authoritative Phase 9.15 mapping.
# ==================================================================================================

print("\n" + "=" * 130)
print("9.17C — PHASE 9.15 PHYSICAL MAP SCHEMA")
print("=" * 130)

print("\nColumns:")

for c in physical_map.columns:
    print(" ", c)

print("\nFirst rows:")
print(
    physical_map.head(10).to_string(
        index=False
    )
)


# ==================================================================================================
# 9.17D — IDENTIFY CLUSTER-ID COLUMN
# ==================================================================================================

cluster_id_candidates = [
    "Observed_Cluster_ID",
    "Cluster_ID",
    "Strict_Cluster_ID",
]

map_cluster_col = None

for c in cluster_id_candidates:

    if c in physical_map.columns:
        map_cluster_col = c
        break


if map_cluster_col is None:

    raise KeyError(
        "\nCould not identify cluster ID column in "
        "cic_cluster_to_physical_episode_map.csv.\n"
        f"Columns found:\n{list(physical_map.columns)}"
    )


print(
    "\n✓ Physical map cluster column:",
    map_cluster_col
)


# ==================================================================================================
# 9.17E — IDENTIFY PHYSICAL EPISODE ID COLUMN
# ==================================================================================================

episode_candidates = [
    "Physical_Episode_ID",
    "Candidate_Physical_Episode_ID",
    "Physical_Episode",
    "Episode_ID",
    "Candidate_Episode_ID",
]

episode_col_map = None

for c in episode_candidates:

    if c in physical_map.columns:
        episode_col_map = c
        break


# Flexible fallback ONLY for column-name discovery.
# We are NOT reconstructing episode membership.
if episode_col_map is None:

    possible = [
        c for c in physical_map.columns
        if (
            "episode" in c.lower()
            and "id" in c.lower()
        )
    ]

    if len(possible) == 1:
        episode_col_map = possible[0]


if episode_col_map is None:

    raise KeyError(
        "\nCould not identify physical episode ID column "
        "inside the SAVED Phase 9.15 map.\n"
        f"Columns found:\n{list(physical_map.columns)}"
    )


print(
    "✓ Physical episode ID column:",
    episode_col_map
)


# ==================================================================================================
# 9.17F — NORMALIZE MAPPING TABLE
# ==================================================================================================

physical_map_clean = physical_map[
    [
        map_cluster_col,
        episode_col_map
    ]
].copy()

physical_map_clean.columns = [
    "Observed_Cluster_ID",
    "Physical_Episode_ID"
]


# Normalize IDs to string for safe matching
physical_map_clean[
    "Observed_Cluster_ID"
] = (
    physical_map_clean[
        "Observed_Cluster_ID"
    ]
    .astype(str)
)

physical_map_clean[
    "Physical_Episode_ID"
] = (
    physical_map_clean[
        "Physical_Episode_ID"
    ]
    .astype(str)
)


clusters[
    "Observed_Cluster_ID"
] = (
    clusters[
        "Observed_Cluster_ID"
    ]
    .astype(str)
)


# ==================================================================================================
# 9.17G — VALIDATE MAPPING BEFORE MERGING
# ==================================================================================================

print("\n" + "=" * 130)
print("9.17G — PHASE 9.15 MAPPING VALIDATION")
print("=" * 130)


duplicate_cluster_map = (
    physical_map_clean[
        "Observed_Cluster_ID"
    ]
    .duplicated()
    .sum()
)

print(
    f"Duplicate cluster IDs in map : "
    f"{duplicate_cluster_map:,}"
)

assert duplicate_cluster_map == 0, (
    "A strict cluster maps to multiple physical episodes."
)


cluster_ids = set(
    clusters[
        "Observed_Cluster_ID"
    ]
)

mapped_ids = set(
    physical_map_clean[
        "Observed_Cluster_ID"
    ]
)


missing_from_map = (
    cluster_ids - mapped_ids
)

extra_in_map = (
    mapped_ids - cluster_ids
)


print(
    f"Strict cluster IDs          : "
    f"{len(cluster_ids):,}"
)

print(
    f"Mapped cluster IDs          : "
    f"{len(mapped_ids):,}"
)

print(
    f"Missing from physical map   : "
    f"{len(missing_from_map):,}"
)

print(
    f"Extra IDs in physical map   : "
    f"{len(extra_in_map):,}"
)


if missing_from_map:

    print("\nMissing IDs:")
    print(
        sorted(missing_from_map)
    )


if extra_in_map:

    print("\nExtra IDs:")
    print(
        sorted(extra_in_map)
    )


assert len(missing_from_map) == 0, (
    "Some strict clusters are missing from the "
    "saved Phase 9.15 physical map."
)

assert len(extra_in_map) == 0, (
    "The physical map contains unknown cluster IDs."
)


# ==================================================================================================
# 9.17H — MERGE AUTHORITATIVE PHYSICAL EPISODE ASSIGNMENTS
# ==================================================================================================

clusters = clusters.merge(
    physical_map_clean,
    on="Observed_Cluster_ID",
    how="left",
    validate="one_to_one"
)


assert (
    clusters[
        "Physical_Episode_ID"
    ]
    .notna()
    .all()
)


physical_episode_count = (
    clusters[
        "Physical_Episode_ID"
    ]
    .nunique()
)


print("\n" + "=" * 130)
print("9.17H — AUTHORITATIVE PHYSICAL EPISODE RESULT")
print("=" * 130)

print(
    f"Strict malicious clusters       : "
    f"{len(clusters):,}"
)

print(
    f"Physical episode candidates     : "
    f"{physical_episode_count:,}"
)

print(
    f"Clusters merged into episodes   : "
    f"{len(clusters) - physical_episode_count:,}"
)


# CRITICAL CHECK:
# Previous broken result was exactly 82 physical groups.
if physical_episode_count == len(clusters):

    raise AssertionError(
        "\nSTOP: Physical episode count still equals strict "
        "cluster count.\n"
        "The saved Phase 9.15 mapping is not being interpreted "
        "correctly.\n"
        "DO NOT FREEZE CIC."
    )


print(
    "✓ Physical episode mapping is NOT the broken "
    "82-clusters = 82-episodes result."
)


# ==================================================================================================
# 9.17I — VALIDATE AGAINST SAVED PHYSICAL EPISODE CANDIDATE TABLE
# ==================================================================================================

print("\n" + "=" * 130)
print("9.17I — CROSS-CHECK AGAINST SAVED PHYSICAL EPISODE INVENTORY")
print("=" * 130)


candidate_episode_col = None

for c in episode_candidates:

    if c in physical_candidates.columns:
        candidate_episode_col = c
        break


if candidate_episode_col is None:

    possible = [
        c for c in physical_candidates.columns
        if (
            "episode" in c.lower()
            and "id" in c.lower()
        )
    ]

    if len(possible) == 1:
        candidate_episode_col = possible[0]


if candidate_episode_col is None:

    print(
        "⚠ Could not automatically identify an episode-ID "
        "column in cic_physical_episode_candidates.csv."
    )

    print(
        "The cluster→physical map remains authoritative."
    )

else:

    candidate_count = (
        physical_candidates[
            candidate_episode_col
        ]
        .astype(str)
        .nunique()
    )

    print(
        f"Physical episodes from map      : "
        f"{physical_episode_count:,}"
    )

    print(
        f"Physical episodes in inventory  : "
        f"{candidate_count:,}"
    )

    assert (
        physical_episode_count
        == candidate_count
    ), (
        "Physical episode count differs between "
        "Phase 9.15 saved artifacts."
    )

    print(
        "✓ Phase 9.15 physical episode artifacts agree."
    )


# ==================================================================================================
# 9.17J — VERIFY ATTACK-FAMILY CONSISTENCY WITHIN PHYSICAL EPISODES
# ==================================================================================================

episode_family_counts = (
    clusters.groupby(
        "Physical_Episode_ID"
    )[
        "Attack_Family"
    ]
    .nunique()
)

mixed_episode_families = (
    episode_family_counts[
        episode_family_counts > 1
    ]
)


print("\nPhysical episodes mixing attack families:",
      len(mixed_episode_families))


assert len(mixed_episode_families) == 0, (
    "A physical episode unexpectedly contains "
    "multiple attack families."
)


print(
    "✓ Every physical episode belongs to exactly "
    "one CIC attack family."
)


# ==================================================================================================
# 9.17K — VALIDATE THE 12 AMBIGUOUS BOUNDARIES
# ==================================================================================================

print("\n" + "=" * 130)
print("9.17K — UNRESOLVED BOUNDARY VALIDATION")
print("=" * 130)


assert len(ambiguous) == 12, (
    f"Expected 12 unresolved boundaries, "
    f"found {len(ambiguous)}."
)

assert len(forensics) == 12, (
    f"Expected 12 Phase 9.16 forensic rows, "
    f"found {len(forensics)}."
)


if "Phase_9_16_Forensic_Status" in forensics.columns:

    forensic_status = (
        forensics[
            "Phase_9_16_Forensic_Status"
        ]
        .value_counts(dropna=False)
    )

    print(forensic_status)


print(
    "\n✓ The 12 ambiguous boundaries remain unresolved."
)


# ==================================================================================================
# 9.17L — BUILD CONSERVATIVE LEAKAGE GROUPS
#
# IMPORTANT:
#
# Physical_Episode_ID
#     = best-supported physical grouping from Phase 9.15
#
# Conservative_Leakage_Group_ID
#     = dependency group used ONLY to prevent train/val/test leakage.
#
# Every physical episode is kept intact.
#
# Additionally, if Phase 9.16 cannot establish independence across
# an ambiguous boundary, both sides are unioned into one leakage group.
#
# This does NOT convert the ambiguous boundary into a physical merge.
# ==================================================================================================

all_cluster_ids = (
    clusters[
        "Observed_Cluster_ID"
    ]
    .astype(str)
    .tolist()
)


parent = {
    cid: cid
    for cid in all_cluster_ids
}


def find_root(x):

    if parent[x] != x:
        parent[x] = find_root(
            parent[x]
        )

    return parent[x]


def union(a, b):

    a = str(a)
    b = str(b)

    if a not in parent:
        raise KeyError(
            f"Unknown cluster ID: {a}"
        )

    if b not in parent:
        raise KeyError(
            f"Unknown cluster ID: {b}"
        )

    ra = find_root(a)
    rb = find_root(b)

    if ra != rb:
        parent[rb] = ra


# --------------------------------------------------------------------------------------------------
# First union:
# clusters already belonging to SAME Phase 9.15 physical episode
# --------------------------------------------------------------------------------------------------

for episode_id, g in clusters.groupby(
    "Physical_Episode_ID"
):

    members = (
        g[
            "Observed_Cluster_ID"
        ]
        .astype(str)
        .tolist()
    )

    if len(members) > 1:

        anchor = members[0]

        for member in members[1:]:
            union(
                anchor,
                member
            )


# --------------------------------------------------------------------------------------------------
# Second union:
# unresolved Phase 9.16 boundaries
#
# Conservative dependency grouping ONLY.
# --------------------------------------------------------------------------------------------------

for _, row in ambiguous.iterrows():

    union(
        row["Left_Cluster_ID"],
        row["Right_Cluster_ID"]
    )


# ==================================================================================================
# 9.17M — CONSTRUCT DETERMINISTIC LEAKAGE GROUP IDs
# ==================================================================================================

root_members = defaultdict(list)

for cid in all_cluster_ids:

    root_members[
        find_root(cid)
    ].append(cid)


temporary_groups = []


for root, members in root_members.items():

    subset = clusters[
        clusters[
            "Observed_Cluster_ID"
        ].isin(members)
    ].copy()

    families = (
        subset[
            "Attack_Family"
        ]
        .astype(str)
        .unique()
    )

    if len(families) != 1:

        raise AssertionError(
            "\nConservative leakage group contains "
            "multiple attack families:\n"
            f"{families}"
        )

    family = families[0]

    start = (
        subset[
            "Cluster_Start"
        ].min()
    )

    end = (
        subset[
            "Cluster_End"
        ].max()
    )

    temporary_groups.append({

        "root":
            root,

        "Attack_Family":
            family,

        "Start":
            start,

        "End":
            end,

        "Members":
            members,
    })


temporary_groups = sorted(
    temporary_groups,
    key=lambda x: (
        str(x["Attack_Family"]),
        x["Start"]
    )
)


family_counter = defaultdict(int)

cluster_to_leakage_group = {}


for group in temporary_groups:

    family = group[
        "Attack_Family"
    ]

    family_counter[
        family
    ] += 1

    safe_family = (
        str(family)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("–", "-")
    )

    leakage_id = (
        f"CIC_{safe_family}_LG"
        f"{family_counter[family]:03d}"
    )

    for cid in group["Members"]:

        cluster_to_leakage_group[
            cid
        ] = leakage_id


clusters[
    "Conservative_Leakage_Group_ID"
] = (
    clusters[
        "Observed_Cluster_ID"
    ]
    .map(
        cluster_to_leakage_group
    )
)


assert (
    clusters[
        "Conservative_Leakage_Group_ID"
    ]
    .notna()
    .all()
)


# ==================================================================================================
# 9.17N — VERIFY PHYSICAL EPISODES NEVER CROSS LEAKAGE GROUPS
# ==================================================================================================

episode_to_leakage_count = (
    clusters.groupby(
        "Physical_Episode_ID"
    )[
        "Conservative_Leakage_Group_ID"
    ]
    .nunique()
)


physical_split_violations = (
    episode_to_leakage_count[
        episode_to_leakage_count > 1
    ]
)


print("\n" + "=" * 130)
print("9.17N — PHYSICAL EPISODE LEAKAGE CHECK")
print("=" * 130)

print(
    "Physical episodes crossing leakage groups:",
    len(physical_split_violations)
)


assert (
    len(physical_split_violations)
    == 0
)

print(
    "✓ Every Phase 9.15 physical episode remains "
    "inside exactly one leakage group."
)


# ==================================================================================================
# 9.17O — VERIFY EVERY AMBIGUOUS BOUNDARY IS INSIDE ONE LEAKAGE GROUP
# ==================================================================================================

cluster_to_lg = (
    clusters.set_index(
        "Observed_Cluster_ID"
    )[
        "Conservative_Leakage_Group_ID"
    ]
    .to_dict()
)


ambiguity_violations = []


for _, row in ambiguous.iterrows():

    left = str(
        row["Left_Cluster_ID"]
    )

    right = str(
        row["Right_Cluster_ID"]
    )

    left_group = (
        cluster_to_lg[left]
    )

    right_group = (
        cluster_to_lg[right]
    )

    if left_group != right_group:

        ambiguity_violations.append({

            "Left_Cluster_ID":
                left,

            "Right_Cluster_ID":
                right,

            "Left_Group":
                left_group,

            "Right_Group":
                right_group,
        })


print("\n" + "=" * 130)
print("9.17O — AMBIGUOUS BOUNDARY LEAKAGE CHECK")
print("=" * 130)

print(
    f"Ambiguous boundaries checked : "
    f"{len(ambiguous):,}"
)

print(
    f"Leakage violations           : "
    f"{len(ambiguity_violations):,}"
)


assert (
    len(ambiguity_violations)
    == 0
)


print(
    "✓ Every unresolved boundary is contained "
    "inside one conservative leakage group."
)


# ==================================================================================================
# 9.17P — MARK UNCERTAIN GROUPS
# ==================================================================================================

ambiguous_cluster_ids = set(
    ambiguous[
        "Left_Cluster_ID"
    ]
    .astype(str)
).union(
    set(
        ambiguous[
            "Right_Cluster_ID"
        ]
        .astype(str)
    )
)


clusters[
    "Touches_Unresolved_Boundary"
] = (
    clusters[
        "Observed_Cluster_ID"
    ]
    .isin(
        ambiguous_cluster_ids
    )
)


uncertain_leakage_groups = set(
    clusters.loc[
        clusters[
            "Touches_Unresolved_Boundary"
        ],
        "Conservative_Leakage_Group_ID"
    ]
)


clusters[
    "Leakage_Group_Contains_Uncertainty"
] = (
    clusters[
        "Conservative_Leakage_Group_ID"
    ]
    .isin(
        uncertain_leakage_groups
    )
)


# ==================================================================================================
# 9.17Q — BUILD LEAKAGE GROUP INVENTORY
# ==================================================================================================

leakage_rows = []


for leakage_id, g in clusters.groupby(
    "Conservative_Leakage_Group_ID"
):

    family = (
        g[
            "Attack_Family"
        ].iloc[0]
    )

    cluster_list = (
        g.sort_values(
            "Cluster_Start"
        )[
            "Observed_Cluster_ID"
        ]
        .astype(str)
        .tolist()
    )

    physical_episode_list = sorted(
        g[
            "Physical_Episode_ID"
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    cluster_set = set(
        cluster_list
    )

    unresolved_inside = 0


    for _, row in ambiguous.iterrows():

        left = str(
            row[
                "Left_Cluster_ID"
            ]
        )

        right = str(
            row[
                "Right_Cluster_ID"
            ]
        )

        if (
            left in cluster_set
            and
            right in cluster_set
        ):
            unresolved_inside += 1


    leakage_rows.append({

        "Conservative_Leakage_Group_ID":
            leakage_id,

        "Attack_Family":
            family,

        "Group_Start":
            g[
                "Cluster_Start"
            ].min(),

        "Group_End":
            g[
                "Cluster_End"
            ].max(),

        "Strict_Cluster_Count":
            len(g),

        "Physical_Episode_Candidate_Count":
            len(
                physical_episode_list
            ),

        "Unresolved_Boundaries_Inside":
            unresolved_inside,

        "Contains_Uncertainty":
            unresolved_inside > 0,

        "Strict_Cluster_IDs":
            "; ".join(
                cluster_list
            ),

        "Physical_Episode_IDs":
            "; ".join(
                physical_episode_list
            ),
    })


leakage_inventory = (
    pd.DataFrame(
        leakage_rows
    )
    .sort_values(
        [
            "Attack_Family",
            "Group_Start"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# 9.17R — BUILD CORRECTED ATTACK-FAMILY INVENTORY
# ==================================================================================================

family_rows = []


for family, g in clusters.groupby(
    "Attack_Family"
):

    family_ambiguous = ambiguous[
        ambiguous[
            "Attack_Family"
        ].astype(str)
        ==
        str(family)
    ]

    family_leakage = (
        leakage_inventory[
            leakage_inventory[
                "Attack_Family"
            ].astype(str)
            ==
            str(family)
        ]
    )


    # Correct attack flow column from the behavioral signatures table
    if (
        "Total_Attack_Flows_Rebuilt"
        in g.columns
    ):

        attack_flows = (
            pd.to_numeric(
                g[
                    "Total_Attack_Flows_Rebuilt"
                ],
                errors="coerce"
            )
            .sum()
        )

    else:
        attack_flows = np.nan


    # Correct observed attack minutes
    if (
        "Observed_Minutes_Rebuilt"
        in g.columns
    ):

        attack_minutes = (
            pd.to_numeric(
                g[
                    "Observed_Minutes_Rebuilt"
                ],
                errors="coerce"
            )
            .sum()
        )

    else:

        attack_minutes = (
            (
                (
                    g[
                        "Cluster_End"
                    ]
                    -
                    g[
                        "Cluster_Start"
                    ]
                )
                .dt.total_seconds()
                / 60
            )
            + 1
        ).sum()


    family_rows.append({

        "Dataset":
            "CIC-IDS2017",

        "Original_Attack_Label":
            family,

        "Canonical_CIC_Family":
            family,

        "Future_Cross_Dataset_Broad_Family":
            "PENDING_UNSW_AUDIT",

        "Attack_Flow_Count":
            attack_flows,

        "Attack_Minutes":
            attack_minutes,

        "Strict_Cluster_Count":
            len(g),

        "Physical_Episode_Candidate_Count":
            g[
                "Physical_Episode_ID"
            ].nunique(),

        "Unresolved_Boundary_Count":
            len(
                family_ambiguous
            ),

        "Conservative_Leakage_Group_Count":
            family_leakage[
                "Conservative_Leakage_Group_ID"
            ].nunique(),

        "Leakage_Groups_With_Uncertainty":
            int(
                family_leakage[
                    "Contains_Uncertainty"
                ].sum()
            ),

        "Cross_Dataset_Mapping_Status":
            "NOT_MAPPED_YET",
    })


family_inventory = (
    pd.DataFrame(
        family_rows
    )
    .sort_values(
        "Canonical_CIC_Family"
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# 9.17S — BUILD FROZEN UNRESOLVED BOUNDARY TABLE
# ==================================================================================================

freeze_ambiguous = (
    ambiguous.copy()
)


# Attach selected 9.16 forensic evidence
merge_keys = [
    "Attack_Family",
    "Left_Cluster_ID",
    "Right_Cluster_ID",
]


forensic_columns = [
    c for c in [
        "Attack_Family",
        "Left_Cluster_ID",
        "Right_Cluster_ID",
        "Exact_Attack_Gap_Minutes",
        "Undirected_Pair_Jaccard_Raw",
        "Directional_Pair_Jaccard_Raw",
        "Destination_Port_Jaccard",
        "Gap_Total_Flows",
        "Gap_Same_Family_Attack_Flows",
        "Gap_Other_Attack_Flows",
        "Phase_9_16_Forensic_Status",
    ]
    if c in forensics.columns
]


forensic_small = (
    forensics[
        forensic_columns
    ]
    .drop_duplicates(
        subset=merge_keys
    )
)


extra_cols = [
    c for c in forensic_small.columns
    if c not in merge_keys
]


freeze_ambiguous = (
    freeze_ambiguous.merge(
        forensic_small[
            merge_keys
            +
            extra_cols
        ],
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )
)


freeze_ambiguous[
    "Conservative_Leakage_Group_ID"
] = (
    freeze_ambiguous[
        "Left_Cluster_ID"
    ]
    .astype(str)
    .map(
        cluster_to_lg
    )
)


right_groups = (
    freeze_ambiguous[
        "Right_Cluster_ID"
    ]
    .astype(str)
    .map(
        cluster_to_lg
    )
)


assert (
    freeze_ambiguous[
        "Conservative_Leakage_Group_ID"
    ]
    ==
    right_groups
).all()


freeze_ambiguous[
    "Physical_Episode_Truth_Status"
] = (
    "UNRESOLVED_FROM_AVAILABLE_CIC_EVIDENCE"
)


freeze_ambiguous[
    "ML_Partition_Policy"
] = (
    "KEEP_BOTH_SIDES_IN_SAME_PARTITION"
)


# ==================================================================================================
# 9.17T — FINAL FAMILY REPORT
# ==================================================================================================

print("\n" + "=" * 130)
print("9.17T — CORRECTED CIC ATTACK-FAMILY INVENTORY")
print("=" * 130)


report_cols = [
    "Canonical_CIC_Family",
    "Attack_Flow_Count",
    "Attack_Minutes",
    "Strict_Cluster_Count",
    "Physical_Episode_Candidate_Count",
    "Unresolved_Boundary_Count",
    "Conservative_Leakage_Group_Count",
    "Leakage_Groups_With_Uncertainty",
]


print(
    family_inventory[
        report_cols
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 9.17U — PHYSICAL EPISODE COMPRESSION REPORT
#
# This is especially useful because it proves that Phase 9.15 grouping
# is actually being used.
# ==================================================================================================

print("\n" + "=" * 130)
print("9.17U — STRICT CLUSTERS → PHYSICAL EPISODES BY FAMILY")
print("=" * 130)


compression = (
    clusters.groupby(
        "Attack_Family"
    )
    .agg(
        Strict_Clusters=(
            "Observed_Cluster_ID",
            "nunique"
        ),

        Physical_Episodes=(
            "Physical_Episode_ID",
            "nunique"
        ),

        Leakage_Groups=(
            "Conservative_Leakage_Group_ID",
            "nunique"
        ),
    )
    .reset_index()
)


compression[
    "Clusters_Merged_By_Physical_Evidence"
] = (
    compression[
        "Strict_Clusters"
    ]
    -
    compression[
        "Physical_Episodes"
    ]
)


print(
    compression.to_string(
        index=False
    )
)


# ==================================================================================================
# 9.17V — FINAL GLOBAL VALIDATION
# ==================================================================================================

strict_cluster_count = (
    clusters[
        "Observed_Cluster_ID"
    ]
    .nunique()
)

physical_episode_count = (
    clusters[
        "Physical_Episode_ID"
    ]
    .nunique()
)

leakage_group_count = (
    clusters[
        "Conservative_Leakage_Group_ID"
    ]
    .nunique()
)

uncertain_group_count = (
    leakage_inventory[
        "Contains_Uncertainty"
    ]
    .sum()
)


print("\n" + "=" * 130)
print("9.17V — FINAL GLOBAL VALIDATION")
print("=" * 130)


print(
    f"Strict malicious clusters        : "
    f"{strict_cluster_count:,}"
)

print(
    f"Physical episode candidates      : "
    f"{physical_episode_count:,}"
)

print(
    f"Conservative leakage groups      : "
    f"{leakage_group_count:,}"
)

print(
    f"Unresolved boundaries            : "
    f"{len(ambiguous):,}"
)

print(
    f"Leakage groups with uncertainty  : "
    f"{int(uncertain_group_count):,}"
)


assert strict_cluster_count == 82

assert physical_episode_count < 82

assert len(ambiguous) == 12

assert leakage_group_count <= physical_episode_count

assert (
    len(
        ambiguity_violations
    )
    == 0
)

assert (
    len(
        physical_split_violations
    )
    == 0
)


print("\n✓ All corrected CIC freeze checks passed.")


# ==================================================================================================
# 9.17W — FREEZE MANIFEST
# ==================================================================================================

freeze_manifest = {

    "phase":
        "9.17_corrected",

    "dataset":
        "CIC-IDS2017",

    "audit_status":
        "CIC_STRUCTURAL_AUDIT_FROZEN",

    "physical_episode_source":
        "cic_cluster_to_physical_episode_map.csv",

    "usable_raw_flows":
        2830743,

    "minute_records":
        2454,

    "attack_active_minutes":
        520,

    "benign_minutes":
        1934,

    "strict_malicious_clusters":
        int(
            strict_cluster_count
        ),

    "physical_episode_candidates":
        int(
            physical_episode_count
        ),

    "attack_families":
        int(
            clusters[
                "Attack_Family"
            ].nunique()
        ),

    "unresolved_boundaries":
        int(
            len(
                ambiguous
            )
        ),

    "conservative_leakage_groups":
        int(
            leakage_group_count
        ),

    "leakage_groups_with_uncertainty":
        int(
            uncertain_group_count
        ),

    "ambiguous_boundary_policy":
        (
            "Physical relationship remains unresolved. "
            "Both sides are retained in the same "
            "conservative ML partition group."
        ),

    "cross_dataset_taxonomy":
        "PENDING_UNSW_AUDIT",

    "train_validation_test_split_created":
        False,

    "sequence_generation_performed":
        False,

    "feature_scaling_performed":
        False,

    "imputation_performed":
        False,

    "oversampling_performed":
        False,

    "class_weighting_performed":
        False,

    "model_training_performed":
        False,

    "raw_files_modified":
        False,
}


# ==================================================================================================
# 9.17X — SAVE / OVERWRITE THE PREVIOUS INCORRECT 9.17 DERIVED OUTPUTS
# ==================================================================================================

CLUSTER_FREEZE_OUT = (
    CIC_OUT_DIR /
    "cic_frozen_cluster_inventory.csv"
)

LEAKAGE_OUT = (
    CIC_OUT_DIR /
    "cic_conservative_leakage_groups.csv"
)

FAMILY_OUT = (
    CIC_OUT_DIR /
    "cic_attack_family_inventory.csv"
)

AMBIG_FREEZE_OUT = (
    CIC_OUT_DIR /
    "cic_frozen_unresolved_boundaries.csv"
)

MANIFEST_OUT = (
    CIC_OUT_DIR /
    "cic_phase_9_17_freeze_manifest.json"
)


clusters.to_csv(
    CLUSTER_FREEZE_OUT,
    index=False
)

leakage_inventory.to_csv(
    LEAKAGE_OUT,
    index=False
)

family_inventory.to_csv(
    FAMILY_OUT,
    index=False
)

freeze_ambiguous.to_csv(
    AMBIG_FREEZE_OUT,
    index=False
)


with open(
    MANIFEST_OUT,
    "w"
) as f:

    json.dump(
        freeze_manifest,
        f,
        indent=2,
        default=str
    )


# ==================================================================================================
# 9.17Y — VERIFY SAVED OUTPUTS
# ==================================================================================================

print("\n" + "=" * 130)
print("9.17Y — SAVED ARTIFACT VERIFICATION")
print("=" * 130)


saved_files = [
    CLUSTER_FREEZE_OUT,
    LEAKAGE_OUT,
    FAMILY_OUT,
    AMBIG_FREEZE_OUT,
    MANIFEST_OUT,
]


for path in saved_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Output was not saved:\n{path}"
        )

    print(
        f"✓ {path.name}"
    )


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 130)
print("PHASE 9.17 — CORRECTED FINAL CIC FREEZE REPORT")
print("=" * 130)


print(f"""

CIC-IDS2017
===========

Usable raw flows
    2,830,743

Minute observations
    2,454

Attack-active minutes
    520

Benign minutes
    1,934


STRUCTURAL REPRESENTATION
-------------------------

Strict malicious clusters
    {strict_cluster_count:,}

Phase 9.15 physical episode candidates
    {physical_episode_count:,}

Unresolved Phase 9.16 boundaries
    {len(ambiguous):,}

Conservative leakage groups
    {leakage_group_count:,}

Leakage groups containing uncertainty
    {int(uncertain_group_count):,}


IMPORTANT
---------

Physical episode assignments came DIRECTLY from:

    cic_cluster_to_physical_episode_map.csv

They were NOT reconstructed by Phase 9.17.


The 12 unresolved boundaries remain:

    UNRESOLVED_FROM_AVAILABLE_CIC_EVIDENCE


For machine-learning partitioning:

    Physical episodes MUST NOT be split.

    Unresolved dependent boundaries MUST NOT be split.

Therefore the eventual partitioning unit is:

    Conservative_Leakage_Group_ID


NOT:

    raw flow
    minute
    sliding window
    strict cluster


ML STATUS
---------

Train/validation/test split
    NOT CREATED

Sequence generation
    NOT PERFORMED

Scaling
    NOT PERFORMED

Imputation
    NOT PERFORMED

Oversampling
    NOT PERFORMED

Class weighting
    NOT PERFORMED

Model training
    NOT PERFORMED


CIC ↔ UNSW taxonomy
    PENDING


Raw CIC files modified
    NO


If all checks above passed:

    CIC STRUCTURAL AUDIT IS NOW FROZEN.


NEXT:

    PHASE 10
    UNSW-NB15 RAW DATASET AUDIT

""")


print("=" * 130)
print("PHASE 9.17 CORRECTED — COMPLETE")
print("=" * 130)

In [ ]:
# ==================================================================================================
# PHASE 10.1 — UNSW-NB15 RAW FILE ROLE + STRUCTURE AUDIT
#
# PURPOSE
# -------
# Before processing UNSW-NB15, determine exactly what each of the seven raw
# files contains.
#
# We are NOT:
#   - combining the four traffic files
#   - assigning attack episodes
#   - aggregating by minute
#   - mapping UNSW attack types to CIC
#   - cleaning/imputing/scaling
#   - splitting train/validation/test
#
# Raw files are READ ONLY.
# ==================================================================================================

import os
import csv
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 120)

print("=" * 130)
print("PHASE 10.1 — UNSW-NB15 RAW FILE ROLE + STRUCTURE AUDIT")
print("=" * 130)


# ==================================================================================================
# 10.1A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

UNSW_RAW = PROJECT_ROOT / "raw" / "UNSW_NB15"
UNSW_OUT = PROJECT_ROOT / "audit_outputs" / "UNSW"

UNSW_OUT.mkdir(
    parents=True,
    exist_ok=True
)

if not UNSW_RAW.exists():
    raise FileNotFoundError(
        f"UNSW raw directory not found:\n{UNSW_RAW}"
    )

print("\nRaw UNSW directory:")
print(UNSW_RAW)

print("\nAudit output directory:")
print(UNSW_OUT)


# ==================================================================================================
# 10.1B — EXPECTED FILES
# ==================================================================================================

expected_files = [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv",
    "NUSW-NB15_GT.csv",
    "NUSW-NB15_features.csv",
    "UNSW-NB15_LIST_EVENTS.csv",
]

print("\n" + "=" * 130)
print("10.1B — FILE EXISTENCE CHECK")
print("=" * 130)

for name in expected_files:

    path = UNSW_RAW / name

    if not path.exists():
        raise FileNotFoundError(
            f"Missing UNSW file:\n{path}"
        )

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"✓ {name:35s} "
        f"{size_mb:10.3f} MB"
    )


# ==================================================================================================
# 10.1C — PHYSICAL LINE COUNTS
#
# This does NOT assume whether the first line is a header.
# ==================================================================================================

print("\n" + "=" * 130)
print("10.1C — PHYSICAL LINE COUNTS")
print("=" * 130)

line_counts = {}

for name in expected_files:

    path = UNSW_RAW / name

    count = 0

    with open(
        path,
        "rb"
    ) as f:

        for _ in f:
            count += 1

    line_counts[name] = count

    print(
        f"{name:35s} "
        f"{count:12,} physical lines"
    )


# ==================================================================================================
# 10.1D — FULL-FILE ENCODING VALIDATION
#
# CIC taught us why checking only the beginning of a file is not enough.
# We validate the entire byte stream.
# ==================================================================================================

print("\n" + "=" * 130)
print("10.1D — FULL-FILE ENCODING VALIDATION")
print("=" * 130)

encoding_candidates = [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin1",
]

encoding_map = {}


for name in expected_files:

    path = UNSW_RAW / name

    detected = None

    print(f"\n{name}")

    for enc in encoding_candidates:

        try:

            decoder = __import__("codecs").getincrementaldecoder(
                enc
            )()

            with open(
                path,
                "rb"
            ) as f:

                while True:

                    chunk = f.read(
                        1024 * 1024
                    )

                    if not chunk:
                        break

                    decoder.decode(
                        chunk,
                        final=False
                    )

                decoder.decode(
                    b"",
                    final=True
                )

            detected = enc

            print(
                f"  ✓ Full-file decode successful: {enc}"
            )

            break

        except UnicodeDecodeError as e:

            print(
                f"  ✗ {enc:10s} failed "
                f"(byte position around {e.start})"
            )


    if detected is None:

        raise UnicodeError(
            f"Could not decode {name} using expected encodings."
        )

    encoding_map[name] = detected


# ==================================================================================================
# 10.1E — RAW FIRST-LINE STRUCTURE
#
# Important because UNSW traffic partitions may not behave like CIC CSVs.
# ==================================================================================================

print("\n" + "=" * 130)
print("10.1E — FIRST THREE PHYSICAL LINES")
print("=" * 130)


for name in expected_files:

    path = UNSW_RAW / name
    enc = encoding_map[name]

    print("\n" + "-" * 130)
    print(name)
    print("-" * 130)

    with open(
        path,
        "r",
        encoding=enc,
        errors="strict"
    ) as f:

        for i in range(3):

            line = f.readline()

            if not line:
                break

            preview = line.rstrip("\n\r")

            if len(preview) > 500:
                preview = preview[:500] + " ..."

            print(
                f"LINE {i+1}:"
            )
            print(
                repr(preview)
            )


# ==================================================================================================
# 10.1F — CSV FIELD COUNT AUDIT
#
# Count number of comma-separated fields in the first few rows.
# This helps determine whether:
#
#   - four UNSW traffic files share one structure
#   - GT has a different structure
#   - feature metadata describes the traffic schema
#   - LIST_EVENTS is event metadata rather than traffic
# ==================================================================================================

print("\n" + "=" * 130)
print("10.1F — RAW FIELD COUNT AUDIT")
print("=" * 130)


field_count_rows = []


for name in expected_files:

    path = UNSW_RAW / name
    enc = encoding_map[name]

    counts = []

    with open(
        path,
        "r",
        encoding=enc,
        newline=""
    ) as f:

        reader = csv.reader(f)

        for i, row in enumerate(reader):

            counts.append(
                len(row)
            )

            if i >= 19:
                break

    unique_counts = sorted(
        set(counts)
    )

    field_count_rows.append({

        "File":
            name,

        "Physical_Lines":
            line_counts[name],

        "Encoding":
            encoding_map[name],

        "First_20_Row_Field_Counts":
            str(counts),

        "Unique_Field_Counts_First_20":
            str(unique_counts),
    })


field_count_df = pd.DataFrame(
    field_count_rows
)

print(
    field_count_df[
        [
            "File",
            "Physical_Lines",
            "Encoding",
            "Unique_Field_Counts_First_20",
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 10.1G — HEADER-LIKENESS INSPECTION
#
# We DO NOT automatically trust pandas' header inference.
#
# Show first CSV row exactly as parsed.
# ==================================================================================================

print("\n" + "=" * 130)
print("10.1G — FIRST PARSED CSV ROW")
print("=" * 130)


first_rows = {}


for name in expected_files:

    path = UNSW_RAW / name
    enc = encoding_map[name]

    with open(
        path,
        "r",
        encoding=enc,
        newline=""
    ) as f:

        reader = csv.reader(f)

        first_row = next(
            reader
        )

    first_rows[name] = first_row

    print("\n" + "-" * 130)
    print(name)
    print("-" * 130)

    print(
        f"Field count: {len(first_row)}"
    )

    for i, value in enumerate(
        first_row
    ):

        print(
            f"[{i:02d}] {repr(value)}"
        )


# ==================================================================================================
# 10.1H — CROSS-PARTITION STRUCTURE CHECK
#
# Only compare the four main traffic partitions here.
# ==================================================================================================

print("\n" + "=" * 130)
print("10.1H — FOUR MAIN TRAFFIC FILE STRUCTURE CHECK")
print("=" * 130)


traffic_files = [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv",
]


traffic_field_counts = {
    name:
        len(
            first_rows[name]
        )

    for name in traffic_files
}


for name, count in traffic_field_counts.items():

    print(
        f"{name:25s} "
        f"{count:3d} fields in first row"
    )


same_traffic_width = (
    len(
        set(
            traffic_field_counts.values()
        )
    )
    == 1
)


print()

if same_traffic_width:

    print(
        "✓ All four traffic partitions have the "
        "same raw field count."
    )

else:

    print(
        "⚠ Main UNSW traffic partitions do NOT "
        "have identical raw field counts."
    )


# ==================================================================================================
# 10.1I — FEATURE-DEFINITION FILE PREVIEW
#
# We don't yet use it as schema automatically.
# First inspect what it actually contains.
# ==================================================================================================

feature_path = (
    UNSW_RAW /
    "NUSW-NB15_features.csv"
)

feature_enc = (
    encoding_map[
        "NUSW-NB15_features.csv"
    ]
)


print("\n" + "=" * 130)
print("10.1I — FEATURE-DEFINITION FILE PREVIEW")
print("=" * 130)


try:

    feature_preview = pd.read_csv(
        feature_path,
        encoding=feature_enc,
        header=None,
        nrows=20
    )

    print(
        feature_preview.to_string(
            index=False,
            header=False
        )
    )

except Exception as e:

    print(
        "Could not parse feature metadata with "
        "header=None:"
    )

    print(
        repr(e)
    )


# ==================================================================================================
# 10.1J — EVENT-LIST FILE PREVIEW
# ==================================================================================================

event_path = (
    UNSW_RAW /
    "UNSW-NB15_LIST_EVENTS.csv"
)

event_enc = (
    encoding_map[
        "UNSW-NB15_LIST_EVENTS.csv"
    ]
)


print("\n" + "=" * 130)
print("10.1J — EVENT-LIST FILE PREVIEW")
print("=" * 130)


try:

    event_preview = pd.read_csv(
        event_path,
        encoding=event_enc,
        header=None,
        nrows=20
    )

    print(
        event_preview.to_string(
            index=False,
            header=False
        )
    )

except Exception as e:

    print(
        "Could not parse event list with "
        "header=None:"
    )

    print(
        repr(e)
    )


# ==================================================================================================
# 10.1K — GT FILE PREVIEW
# ==================================================================================================

gt_path = (
    UNSW_RAW /
    "NUSW-NB15_GT.csv"
)

gt_enc = (
    encoding_map[
        "NUSW-NB15_GT.csv"
    ]
)


print("\n" + "=" * 130)
print("10.1K — GROUND-TRUTH FILE PREVIEW")
print("=" * 130)


try:

    gt_preview = pd.read_csv(
        gt_path,
        encoding=gt_enc,
        header=None,
        nrows=10
    )

    print(
        gt_preview.to_string(
            index=False,
            header=False
        )
    )

except Exception as e:

    print(
        "Could not parse GT file:"
    )

    print(
        repr(e)
    )


# ==================================================================================================
# 10.1L — SAVE PHASE 10.1 INVENTORY
# ==================================================================================================

inventory_rows = []


for name in expected_files:

    path = UNSW_RAW / name

    inventory_rows.append({

        "File":
            name,

        "Size_MB":
            path.stat().st_size / (1024 ** 2),

        "Physical_Lines":
            line_counts[name],

        "Encoding":
            encoding_map[name],

        "First_Row_Field_Count":
            len(
                first_rows[name]
            ),

        "Raw_File_Modified":
            False,
    })


inventory_df = pd.DataFrame(
    inventory_rows
)


inventory_out = (
    UNSW_OUT /
    "unsw_phase_10_1_raw_file_inventory.csv"
)


inventory_df.to_csv(
    inventory_out,
    index=False
)


print("\n" + "=" * 130)
print("10.1L — RAW FILE INVENTORY")
print("=" * 130)

print(
    inventory_df.to_string(
        index=False
    )
)


print(
    "\n✓ Saved:"
)

print(
    inventory_out
)


# ==================================================================================================
# FINAL
# ==================================================================================================

print("\n" + "=" * 130)
print("PHASE 10.1 COMPLETE")
print("=" * 130)

print("""
WHAT THIS PHASE ESTABLISHED
---------------------------

We inspected UNSW-NB15 at the physical-file level BEFORE imposing
any CIC-derived assumptions.

We now have evidence about:

    • which raw files exist
    • physical line counts
    • full-file encodings
    • raw CSV widths
    • whether the four traffic partitions share a structure
    • what the GT file physically looks like
    • what the feature-definition file physically looks like
    • what the event-list file physically looks like


NOT DONE YET
------------

    ✗ no traffic files combined
    ✗ no header/schema imposed
    ✗ no timestamp conversion
    ✗ no attack-label reconstruction
    ✗ no event/episode reconstruction
    ✗ no CIC↔UNSW mapping
    ✗ no aggregation
    ✗ no feature engineering
    ✗ no train/validation/test split
    ✗ no scaling
    ✗ no model training


NEXT
----

PHASE 10.2 will use THIS evidence to establish the authoritative
UNSW traffic schema and determine how labels/timestamps/ground-truth
actually relate to the four main traffic partitions.

Raw files remain untouched.
""")

In [ ]:
# ==================================================================================================
# PHASE 10.2 — UNSW AUTHORITATIVE SCHEMA + TEMPORAL/LABEL INTEGRITY AUDIT
#
# PURPOSE
# -------
# Establish, WITHOUT modifying the raw files:
#
#   1. The authoritative 49-column UNSW traffic schema.
#   2. Whether that schema maps exactly onto all four traffic partitions.
#   3. Which columns represent:
#          - timestamps
#          - attack category
#          - binary attack label
#          - source/destination identity
#   4. Whether label and attack_cat agree.
#   5. Timestamp validity and chronological coverage of each partition.
#   6. Missingness / malformed-value structure.
#   7. Whether GT timestamps overlap the traffic capture timeline.
#
# IMPORTANT
# ---------
# This phase DOES NOT:
#
#   - combine CIC + UNSW
#   - create attack episodes
#   - aggregate to minutes
#   - create forecasting targets
#   - split train/val/test
#   - scale/impute/model
#
# RAW FILES REMAIN READ ONLY.
# ==================================================================================================

import json
import csv
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 280)
pd.set_option("display.max_colwidth", 140)

print("=" * 135)
print("PHASE 10.2 — UNSW AUTHORITATIVE SCHEMA + TEMPORAL/LABEL INTEGRITY AUDIT")
print("=" * 135)


# ==================================================================================================
# 10.2A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

UNSW_RAW = PROJECT_ROOT / "raw" / "UNSW_NB15"
UNSW_OUT = PROJECT_ROOT / "audit_outputs" / "UNSW"

UNSW_OUT.mkdir(
    parents=True,
    exist_ok=True
)

TRAFFIC_FILES = [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv",
]

FEATURE_PATH = (
    UNSW_RAW /
    "NUSW-NB15_features.csv"
)

GT_PATH = (
    UNSW_RAW /
    "NUSW-NB15_GT.csv"
)

EVENT_PATH = (
    UNSW_RAW /
    "UNSW-NB15_LIST_EVENTS.csv"
)


for path in [
    FEATURE_PATH,
    GT_PATH,
    EVENT_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


for name in TRAFFIC_FILES:

    if not (
        UNSW_RAW /
        name
    ).exists():

        raise FileNotFoundError(
            UNSW_RAW /
            name
        )


print("✓ Required UNSW files found.")


# ==================================================================================================
# 10.2B — LOAD AUTHORITATIVE FEATURE DEFINITIONS
#
# 10.1 established that this file is CP1252 and contains:
#
#     No.
#     Name
#     Type
#     Description
#
# We use it as the schema authority.
# ==================================================================================================

features = pd.read_csv(
    FEATURE_PATH,
    encoding="cp1252"
)

features.columns = [
    str(c).strip()
    for c in features.columns
]


# Strip textual metadata safely
for c in features.columns:

    if features[c].dtype == "object":

        features[c] = (
            features[c]
            .astype(str)
            .str.strip()
        )


print("\n" + "=" * 135)
print("10.2B — AUTHORITATIVE FEATURE DEFINITIONS")
print("=" * 135)

print(
    "Feature-definition rows:",
    len(features)
)

print(
    "Metadata columns:",
    list(features.columns)
)


if "Name" not in features.columns:

    raise KeyError(
        "Feature metadata does not contain a 'Name' column."
    )


feature_names = (
    features["Name"]
    .astype(str)
    .str.strip()
    .tolist()
)


print(
    "\nAuthoritative traffic feature count:",
    len(feature_names)
)


for i, name in enumerate(
    feature_names,
    start=1
):

    print(
        f"{i:02d}. {name}"
    )


if len(feature_names) != 49:

    raise AssertionError(
        f"\nExpected 49 UNSW features but metadata defines "
        f"{len(feature_names)}."
    )


if len(set(feature_names)) != 49:

    duplicates = (
        pd.Series(feature_names)[
            pd.Series(feature_names).duplicated(
                keep=False
            )
        ]
        .tolist()
    )

    raise AssertionError(
        f"Duplicate feature names found:\n{duplicates}"
    )


print(
    "\n✓ Feature metadata defines exactly 49 unique columns."
)


# ==================================================================================================
# 10.2C — FEATURE METADATA TABLE
# ==================================================================================================

print("\n" + "=" * 135)
print("10.2C — FULL FEATURE METADATA")
print("=" * 135)

print(
    features.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.2D — DISCOVER CRITICAL COLUMNS FROM THE OFFICIAL FEATURE LIST
#
# We don't assume capitalization.
# ==================================================================================================

normalized_name_map = {
    str(c).strip().lower():
        c

    for c in feature_names
}


def resolve_feature(
    possible_names,
    required=True
):

    for candidate in possible_names:

        key = (
            candidate
            .strip()
            .lower()
        )

        if key in normalized_name_map:

            return normalized_name_map[
                key
            ]

    if required:

        raise KeyError(
            "Could not resolve any of these expected "
            f"feature names: {possible_names}"
        )

    return None


SRC_IP_COL = resolve_feature(
    ["srcip"]
)

SRC_PORT_COL = resolve_feature(
    ["sport"]
)

DST_IP_COL = resolve_feature(
    ["dstip"]
)

DST_PORT_COL = resolve_feature(
    ["dsport"]
)

PROTO_COL = resolve_feature(
    ["proto"]
)

START_COL = resolve_feature(
    ["stime"]
)

END_COL = resolve_feature(
    ["ltime"]
)

ATTACK_CAT_COL = resolve_feature(
    [
        "attack_cat",
        "attack cat",
        "attack category",
    ]
)

LABEL_COL = resolve_feature(
    ["label"]
)


critical_columns = {

    "Source IP":
        SRC_IP_COL,

    "Source Port":
        SRC_PORT_COL,

    "Destination IP":
        DST_IP_COL,

    "Destination Port":
        DST_PORT_COL,

    "Protocol":
        PROTO_COL,

    "Start timestamp":
        START_COL,

    "End timestamp":
        END_COL,

    "Attack category":
        ATTACK_CAT_COL,

    "Binary label":
        LABEL_COL,
}


print("\n" + "=" * 135)
print("10.2D — CRITICAL FEATURE RESOLUTION")
print("=" * 135)

for role, col in critical_columns.items():

    print(
        f"{role:25s} -> {col}"
    )


# ==================================================================================================
# 10.2E — STRICT RAW WIDTH VALIDATION
#
# 10.1 checked the first 20 rows.
# Here we validate EVERY physical row of all four traffic files.
#
# Expected width = 49.
# ==================================================================================================

print("\n" + "=" * 135)
print("10.2E — STRICT FULL-FILE RAW WIDTH VALIDATION")
print("=" * 135)


width_audit_rows = []


for file_name in TRAFFIC_FILES:

    path = (
        UNSW_RAW /
        file_name
    )

    row_count = 0
    malformed_count = 0
    width_counts = {}

    malformed_examples = []


    # utf-8-sig safely strips the BOM from file 1 while
    # behaving normally for files without a BOM.
    with open(
        path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.reader(f)

        for row_number, row in enumerate(
            reader,
            start=1
        ):

            row_count += 1

            width = len(row)

            width_counts[
                width
            ] = (
                width_counts.get(
                    width,
                    0
                )
                + 1
            )


            if width != 49:

                malformed_count += 1

                if len(
                    malformed_examples
                ) < 10:

                    malformed_examples.append(
                        {
                            "row":
                                row_number,

                            "width":
                                width,

                            "preview":
                                row[:10],
                        }
                    )


    print(
        f"\n{file_name}"
    )

    print(
        f"  rows              : {row_count:,}"
    )

    print(
        f"  field widths      : {width_counts}"
    )

    print(
        f"  malformed rows    : {malformed_count:,}"
    )


    if malformed_examples:

        print(
            "  malformed examples:"
        )

        for x in malformed_examples:

            print(
                "   ",
                x
            )


    width_audit_rows.append(
        {
            "File":
                file_name,

            "Rows":
                row_count,

            "Expected_Width":
                49,

            "Observed_Widths":
                json.dumps(
                    width_counts
                ),

            "Malformed_Rows":
                malformed_count,
        }
    )


width_audit = pd.DataFrame(
    width_audit_rows
)


if (
    width_audit[
        "Malformed_Rows"
    ].sum()
    != 0
):

    raise AssertionError(
        "\nAt least one UNSW traffic row does not have "
        "the authoritative 49-field width.\n"
        "STOP before assigning the schema."
    )


print(
    "\n✓ Every traffic row across all four partitions "
    "contains exactly 49 fields."
)


# ==================================================================================================
# 10.2F — READ TRAFFIC PARTITIONS WITH AUTHORITATIVE SCHEMA
#
# IMPORTANT:
# header=None because 10.1 proved the first physical row is DATA.
#
# dtype=str first:
# We do not allow pandas type inference to silently reinterpret values
# during the structural audit.
# ==================================================================================================

print("\n" + "=" * 135)
print("10.2F — AUTHORITATIVE SCHEMA APPLICATION")
print("=" * 135)


traffic_summaries = []

attack_category_counts = []

label_counts = []

label_category_cross_rows = []

missingness_rows = []

partition_ranges = []


# These will be used for global totals without keeping
# all ~2.54M rows in memory simultaneously.
global_rows = 0

global_label_counts = {}

global_attack_cat_counts = {}

global_label_attack_cross = {}

global_invalid_start = 0
global_invalid_end = 0
global_end_before_start = 0

global_min_start = None
global_max_end = None


for file_index, file_name in enumerate(
    TRAFFIC_FILES,
    start=1
):

    print("\n" + "-" * 135)
    print(
        f"PROCESSING {file_name}"
    )
    print("-" * 135)

    path = (
        UNSW_RAW /
        file_name
    )


    # ----------------------------------------------------------------------------------------------
    # Read the full partition as strings.
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(
        path,
        header=None,
        names=feature_names,
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        low_memory=False,
    )


    print(
        f"Rows loaded   : {len(df):,}"
    )

    print(
        f"Columns       : {len(df.columns)}"
    )


    if len(
        df.columns
    ) != 49:

        raise AssertionError(
            f"{file_name} did not load with 49 columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Normalize whitespace ONLY in selected semantic text columns for auditing.
    #
    # Raw file is untouched.
    # ----------------------------------------------------------------------------------------------

    for c in [
        SRC_IP_COL,
        SRC_PORT_COL,
        DST_IP_COL,
        DST_PORT_COL,
        PROTO_COL,
        ATTACK_CAT_COL,
        LABEL_COL,
    ]:

        df[c] = (
            df[c]
            .astype(str)
            .str.strip()
        )


    # ----------------------------------------------------------------------------------------------
    # TIMESTAMPS
    # ----------------------------------------------------------------------------------------------

    start_numeric = pd.to_numeric(
        df[
            START_COL
        ],
        errors="coerce"
    )

    end_numeric = pd.to_numeric(
        df[
            END_COL
        ],
        errors="coerce"
    )


    invalid_start = int(
        start_numeric
        .isna()
        .sum()
    )

    invalid_end = int(
        end_numeric
        .isna()
        .sum()
    )


    valid_both = (
        start_numeric.notna()
        &
        end_numeric.notna()
    )


    end_before_start = int(
        (
            end_numeric[
                valid_both
            ]
            <
            start_numeric[
                valid_both
            ]
        )
        .sum()
    )


    valid_start_values = (
        start_numeric.dropna()
    )

    valid_end_values = (
        end_numeric.dropna()
    )


    if len(
        valid_start_values
    ):

        min_start_epoch = (
            valid_start_values.min()
        )

        max_start_epoch = (
            valid_start_values.max()
        )

        min_start_dt = pd.to_datetime(
            min_start_epoch,
            unit="s",
            utc=True,
            errors="coerce"
        )

        max_start_dt = pd.to_datetime(
            max_start_epoch,
            unit="s",
            utc=True,
            errors="coerce"
        )

    else:

        min_start_epoch = np.nan
        max_start_epoch = np.nan
        min_start_dt = pd.NaT
        max_start_dt = pd.NaT


    if len(
        valid_end_values
    ):

        min_end_epoch = (
            valid_end_values.min()
        )

        max_end_epoch = (
            valid_end_values.max()
        )

        min_end_dt = pd.to_datetime(
            min_end_epoch,
            unit="s",
            utc=True,
            errors="coerce"
        )

        max_end_dt = pd.to_datetime(
            max_end_epoch,
            unit="s",
            utc=True,
            errors="coerce"
        )

    else:

        min_end_epoch = np.nan
        max_end_epoch = np.nan
        min_end_dt = pd.NaT
        max_end_dt = pd.NaT


    # ----------------------------------------------------------------------------------------------
    # RAW ROW ORDER / CHRONOLOGY
    #
    # We check stime in raw row order, but we DO NOT assume
    # the file must already be sorted.
    # ----------------------------------------------------------------------------------------------

    start_diff = (
        start_numeric.diff()
    )

    backward_start_jumps = int(
        (
            start_diff < 0
        )
        .sum()
    )

    zero_start_gaps = int(
        (
            start_diff == 0
        )
        .sum()
    )


    if start_diff.notna().any():

        largest_forward_start_jump = (
            start_diff.max()
        )

        largest_backward_start_jump = (
            start_diff.min()
        )

    else:

        largest_forward_start_jump = np.nan
        largest_backward_start_jump = np.nan


    already_start_chronological = (
        backward_start_jumps == 0
    )


    # ----------------------------------------------------------------------------------------------
    # LABEL SEMANTICS
    # ----------------------------------------------------------------------------------------------

    labels = (
        df[
            LABEL_COL
        ]
        .astype(str)
        .str.strip()
    )


    attack_cat_raw = (
        df[
            ATTACK_CAT_COL
        ]
        .astype(str)
    )


    attack_cat_clean = (
        attack_cat_raw
        .str.strip()
    )


    # Treat these only as "empty category representations"
    # for the PURPOSE OF THE AUDIT.
    #
    # We are not rewriting the raw dataset.
    empty_attack_cat = (
        attack_cat_clean
        .str.lower()
        .isin(
            [
                "",
                "nan",
                "none",
                "null",
                "-",
            ]
        )
    )


    label_numeric = pd.to_numeric(
        labels,
        errors="coerce"
    )


    invalid_label = int(
        label_numeric
        .isna()
        .sum()
    )


    non_binary_label = int(
        (
            label_numeric.notna()
            &
            ~label_numeric.isin(
                [0, 1]
            )
        )
        .sum()
    )


    # Key consistency conditions
    benign_with_attack_category = int(
        (
            (label_numeric == 0)
            &
            (~empty_attack_cat)
        )
        .sum()
    )


    attack_without_category = int(
        (
            (label_numeric == 1)
            &
            empty_attack_cat
        )
        .sum()
    )


    # ----------------------------------------------------------------------------------------------
    # COUNTS
    # ----------------------------------------------------------------------------------------------

    file_label_counts = (
        labels.value_counts(
            dropna=False
        )
    )


    file_attack_counts = (
        attack_cat_clean
        .replace(
            "",
            "<EMPTY>"
        )
        .value_counts(
            dropna=False
        )
    )


    print("\nLabel distribution:")

    print(
        file_label_counts.to_string()
    )


    print("\nAttack category distribution:")

    print(
        file_attack_counts.to_string()
    )


    print(
        "\nLabel/category integrity:"
    )

    print(
        f"  invalid label values           : "
        f"{invalid_label:,}"
    )

    print(
        f"  non-binary label values        : "
        f"{non_binary_label:,}"
    )

    print(
        f"  label=0 + attack category      : "
        f"{benign_with_attack_category:,}"
    )

    print(
        f"  label=1 + empty attack category: "
        f"{attack_without_category:,}"
    )


    print(
        "\nTimestamp integrity:"
    )

    print(
        f"  invalid stime                  : "
        f"{invalid_start:,}"
    )

    print(
        f"  invalid ltime                  : "
        f"{invalid_end:,}"
    )

    print(
        f"  ltime < stime                  : "
        f"{end_before_start:,}"
    )

    print(
        f"  raw stime chronological        : "
        f"{already_start_chronological}"
    )

    print(
        f"  backward stime jumps           : "
        f"{backward_start_jumps:,}"
    )

    print(
        f"  zero stime transitions         : "
        f"{zero_start_gaps:,}"
    )

    print(
        f"  largest forward stime jump     : "
        f"{largest_forward_start_jump}"
    )

    print(
        f"  largest backward stime jump    : "
        f"{largest_backward_start_jump}"
    )

    print(
        f"  start range                    : "
        f"{min_start_dt} → {max_start_dt}"
    )

    print(
        f"  end range                      : "
        f"{min_end_dt} → {max_end_dt}"
    )


    # ----------------------------------------------------------------------------------------------
    # ATTACK-CATEGORY / LABEL CROSS-TAB
    # ----------------------------------------------------------------------------------------------

    cross = pd.crosstab(
        attack_cat_clean.replace(
            "",
            "<EMPTY>"
        ),
        labels,
        dropna=False
    )


    print(
        "\nAttack category × binary label:"
    )

    print(
        cross.to_string()
    )


    for attack_cat, row in cross.iterrows():

        record = {
            "File":
                file_name,

            "Attack_Category":
                attack_cat,
        }

        for label_value, count in row.items():

            record[
                f"Label_{label_value}"
            ] = int(
                count
            )

        label_category_cross_rows.append(
            record
        )


    # ----------------------------------------------------------------------------------------------
    # MISSING / BLANK / WHITESPACE AUDIT
    #
    # Since the file was loaded with keep_default_na=False,
    # blanks remain visible instead of being silently converted.
    # ----------------------------------------------------------------------------------------------

    file_missing_rows = []


    for c in feature_names:

        raw_text = (
            df[c]
            .astype(str)
        )

        stripped = (
            raw_text
            .str.strip()
        )

        blank_count = int(
            (
                stripped == ""
            )
            .sum()
        )

        dash_count = int(
            (
                stripped == "-"
            )
            .sum()
        )

        whitespace_only_count = int(
            (
                (raw_text != "")
                &
                (stripped == "")
            )
            .sum()
        )


        if (
            blank_count > 0
            or
            dash_count > 0
            or
            whitespace_only_count > 0
        ):

            file_missing_rows.append(
                {
                    "File":
                        file_name,

                    "Column":
                        c,

                    "Blank_Count":
                        blank_count,

                    "Dash_Count":
                        dash_count,

                    "Whitespace_Only_Count":
                        whitespace_only_count,

                    "Rows":
                        len(df),
                }
            )


    missingness_rows.extend(
        file_missing_rows
    )


    if file_missing_rows:

        print(
            "\nColumns containing blank / '-' / whitespace-only values:"
        )

        print(
            pd.DataFrame(
                file_missing_rows
            ).to_string(
                index=False
            )
        )

    else:

        print(
            "\nNo blank/'-'/whitespace-only representations detected."
        )


    # ----------------------------------------------------------------------------------------------
    # SAVE PER-FILE COUNTS
    # ----------------------------------------------------------------------------------------------

    for value, count in file_label_counts.items():

        label_counts.append(
            {
                "File":
                    file_name,

                "Label":
                    value,

                "Count":
                    int(count),
            }
        )


        global_label_counts[
            value
        ] = (
            global_label_counts.get(
                value,
                0
            )
            +
            int(count)
        )


    for value, count in file_attack_counts.items():

        attack_category_counts.append(
            {
                "File":
                    file_name,

                "Attack_Category":
                    value,

                "Count":
                    int(count),
            }
        )


        global_attack_cat_counts[
            value
        ] = (
            global_attack_cat_counts.get(
                value,
                0
            )
            +
            int(count)
        )


    # Global cross tab
    temp_cross = pd.DataFrame(
        {
            "attack_cat":
                attack_cat_clean.replace(
                    "",
                    "<EMPTY>"
                ),

            "label":
                labels,
        }
    ).value_counts()


    for (
        attack_cat,
        label
    ), count in temp_cross.items():

        key = (
            attack_cat,
            label
        )

        global_label_attack_cross[
            key
        ] = (
            global_label_attack_cross.get(
                key,
                0
            )
            +
            int(count)
        )


    # ----------------------------------------------------------------------------------------------
    # SUMMARY
    # ----------------------------------------------------------------------------------------------

    traffic_summaries.append(
        {
            "File":
                file_name,

            "Rows":
                len(df),

            "Columns":
                len(df.columns),

            "Invalid_Start_Timestamps":
                invalid_start,

            "Invalid_End_Timestamps":
                invalid_end,

            "End_Before_Start":
                end_before_start,

            "Start_Chronological_In_Raw_Order":
                already_start_chronological,

            "Backward_Start_Jumps":
                backward_start_jumps,

            "Zero_Start_Transitions":
                zero_start_gaps,

            "Min_Start_Epoch":
                min_start_epoch,

            "Max_Start_Epoch":
                max_start_epoch,

            "Min_End_Epoch":
                min_end_epoch,

            "Max_End_Epoch":
                max_end_epoch,

            "Start_UTC":
                min_start_dt,

            "Last_Start_UTC":
                max_start_dt,

            "First_End_UTC":
                min_end_dt,

            "End_UTC":
                max_end_dt,

            "Invalid_Label_Values":
                invalid_label,

            "Non_Binary_Label_Values":
                non_binary_label,

            "Label0_With_Attack_Category":
                benign_with_attack_category,

            "Label1_Without_Attack_Category":
                attack_without_category,
        }
    )


    partition_ranges.append(
        {
            "File":
                file_name,

            "Partition_Number":
                file_index,

            "Start_Epoch":
                min_start_epoch,

            "End_Epoch":
                max_end_epoch,

            "Start_UTC":
                min_start_dt,

            "End_UTC":
                max_end_dt,
        }
    )


    global_rows += len(
        df
    )

    global_invalid_start += (
        invalid_start
    )

    global_invalid_end += (
        invalid_end
    )

    global_end_before_start += (
        end_before_start
    )


    if pd.notna(
        min_start_epoch
    ):

        if (
            global_min_start is None
            or
            min_start_epoch
            <
            global_min_start
        ):

            global_min_start = (
                min_start_epoch
            )


    if pd.notna(
        max_end_epoch
    ):

        if (
            global_max_end is None
            or
            max_end_epoch
            >
            global_max_end
        ):

            global_max_end = (
                max_end_epoch
            )


    # Explicitly release partition
    del df


# ==================================================================================================
# 10.2G — GLOBAL TRAFFIC SUMMARY
# ==================================================================================================

traffic_summary_df = pd.DataFrame(
    traffic_summaries
)

partition_ranges_df = pd.DataFrame(
    partition_ranges
)


print("\n" + "=" * 135)
print("10.2G — GLOBAL TRAFFIC SUMMARY")
print("=" * 135)

print(
    traffic_summary_df.to_string(
        index=False
    )
)


print(
    f"\nTotal traffic records: "
    f"{global_rows:,}"
)


print(
    "\nGLOBAL LABEL DISTRIBUTION"
)

print(
    pd.Series(
        global_label_counts
    )
    .sort_index()
    .to_string()
)


print(
    "\nGLOBAL ATTACK CATEGORY DISTRIBUTION"
)

print(
    pd.Series(
        global_attack_cat_counts
    )
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ==================================================================================================
# 10.2H — GLOBAL LABEL × ATTACK CATEGORY CONSISTENCY
# ==================================================================================================

global_cross_rows = []


for (
    attack_cat,
    label
), count in sorted(
    global_label_attack_cross.items()
):

    global_cross_rows.append(
        {
            "Attack_Category":
                attack_cat,

            "Label":
                label,

            "Count":
                count,
        }
    )


global_cross_df = pd.DataFrame(
    global_cross_rows
)


print("\n" + "=" * 135)
print("10.2H — GLOBAL ATTACK CATEGORY × LABEL")
print("=" * 135)


global_cross_pivot = (
    global_cross_df.pivot_table(
        index="Attack_Category",
        columns="Label",
        values="Count",
        aggfunc="sum",
        fill_value=0
    )
)


print(
    global_cross_pivot.to_string()
)


# ==================================================================================================
# 10.2I — PARTITION CHRONOLOGY / OVERLAP
#
# We examine actual timestamp ranges rather than assuming files 1→4
# are one continuous chronological sequence.
# ==================================================================================================

print("\n" + "=" * 135)
print("10.2I — PARTITION TIMESTAMP RANGES")
print("=" * 135)


partition_ranges_df = (
    partition_ranges_df.sort_values(
        "Start_Epoch"
    )
    .reset_index(
        drop=True
    )
)


print(
    partition_ranges_df.to_string(
        index=False
    )
)


range_relation_rows = []


for i in range(
    len(partition_ranges_df)
):

    for j in range(
        i + 1,
        len(partition_ranges_df)
    ):

        a = (
            partition_ranges_df.iloc[i]
        )

        b = (
            partition_ranges_df.iloc[j]
        )


        overlap_start = max(
            a["Start_Epoch"],
            b["Start_Epoch"]
        )

        overlap_end = min(
            a["End_Epoch"],
            b["End_Epoch"]
        )


        overlap_seconds = max(
            0,
            overlap_end
            -
            overlap_start
        )


        overlaps = (
            overlap_end
            >=
            overlap_start
        )


        range_relation_rows.append(
            {
                "File_A":
                    a["File"],

                "File_B":
                    b["File"],

                "Ranges_Overlap":
                    bool(
                        overlaps
                    ),

                "Overlap_Seconds":
                    float(
                        overlap_seconds
                    ),
            }
        )


range_relations_df = pd.DataFrame(
    range_relation_rows
)


print(
    "\nPAIRWISE PARTITION RANGE OVERLAP"
)

print(
    range_relations_df.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.2J — LOAD + AUDIT GROUND TRUTH
#
# The GT file has its own header.
# ==================================================================================================

print("\n" + "=" * 135)
print("10.2J — GROUND-TRUTH TEMPORAL AUDIT")
print("=" * 135)


gt = pd.read_csv(
    GT_PATH,
    encoding="utf-8",
    low_memory=False
)

gt.columns = [
    str(c).strip()
    for c in gt.columns
]


print(
    "GT rows:",
    len(gt)
)

print(
    "GT columns:",
    list(gt.columns)
)


required_gt_cols = [
    "Start time",
    "Last time",
    "Attack category",
    "Attack subcategory",
    "Protocol",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
]


missing_gt_cols = [
    c
    for c in required_gt_cols
    if c not in gt.columns
]


if missing_gt_cols:

    raise KeyError(
        f"Missing GT columns: {missing_gt_cols}"
    )


gt_start = pd.to_numeric(
    gt[
        "Start time"
    ],
    errors="coerce"
)

gt_end = pd.to_numeric(
    gt[
        "Last time"
    ],
    errors="coerce"
)


gt_invalid_start = int(
    gt_start
    .isna()
    .sum()
)

gt_invalid_end = int(
    gt_end
    .isna()
    .sum()
)


gt_valid_both = (
    gt_start.notna()
    &
    gt_end.notna()
)


gt_end_before_start = int(
    (
        gt_end[
            gt_valid_both
        ]
        <
        gt_start[
            gt_valid_both
        ]
    )
    .sum()
)


gt_min_start = (
    gt_start.min()
)

gt_max_end = (
    gt_end.max()
)


gt_min_dt = pd.to_datetime(
    gt_min_start,
    unit="s",
    utc=True,
    errors="coerce"
)

gt_max_dt = pd.to_datetime(
    gt_max_end,
    unit="s",
    utc=True,
    errors="coerce"
)


print(
    f"\nInvalid GT start times : "
    f"{gt_invalid_start:,}"
)

print(
    f"Invalid GT end times   : "
    f"{gt_invalid_end:,}"
)

print(
    f"GT end < start         : "
    f"{gt_end_before_start:,}"
)

print(
    f"GT temporal range      : "
    f"{gt_min_dt} → {gt_max_dt}"
)


# GT raw ordering
gt_diff = (
    gt_start.diff()
)

gt_backward_jumps = int(
    (
        gt_diff < 0
    )
    .sum()
)


print(
    f"GT backward start jumps: "
    f"{gt_backward_jumps:,}"
)


# ==================================================================================================
# 10.2K — GT ATTACK CATEGORY NORMALIZATION AUDIT
#
# We inspect whitespace/case variants WITHOUT collapsing them yet.
# ==================================================================================================

gt_attack_raw = (
    gt[
        "Attack category"
    ]
    .astype(str)
)

gt_attack_clean = (
    gt_attack_raw
    .str.strip()
)


print("\n" + "=" * 135)
print("10.2K — GT ATTACK CATEGORY AUDIT")
print("=" * 135)


print(
    "\nRAW GT category counts:"
)

print(
    gt_attack_raw.value_counts(
        dropna=False
    ).to_string()
)


print(
    "\nWHITESPACE-STRIPPED GT category counts:"
)

print(
    gt_attack_clean.value_counts(
        dropna=False
    ).to_string()
)


gt_category_variant_df = (
    pd.DataFrame(
        {
            "Raw":
                gt_attack_raw,

            "Stripped":
                gt_attack_clean,
        }
    )
    .drop_duplicates()
    .sort_values(
        [
            "Stripped",
            "Raw"
        ]
    )
)


print(
    "\nRaw → stripped category variants:"
)

print(
    gt_category_variant_df.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.2L — TRAFFIC TIMELINE VS GT TIMELINE
#
# This does NOT yet claim that every GT row has a matching flow.
#
# It only asks whether the temporal domains are compatible.
# ==================================================================================================

print("\n" + "=" * 135)
print("10.2L — TRAFFIC vs GROUND-TRUTH TEMPORAL DOMAIN")
print("=" * 135)


traffic_min_dt = pd.to_datetime(
    global_min_start,
    unit="s",
    utc=True,
    errors="coerce"
)

traffic_max_dt = pd.to_datetime(
    global_max_end,
    unit="s",
    utc=True,
    errors="coerce"
)


print(
    "Traffic range:"
)

print(
    f"  {traffic_min_dt} → {traffic_max_dt}"
)


print(
    "\nGT range:"
)

print(
    f"  {gt_min_dt} → {gt_max_dt}"
)


temporal_overlap_start = max(
    global_min_start,
    gt_min_start
)

temporal_overlap_end = min(
    global_max_end,
    gt_max_end
)


temporal_domains_overlap = (
    temporal_overlap_end
    >=
    temporal_overlap_start
)


print(
    "\nTemporal domains overlap:",
    temporal_domains_overlap
)


if temporal_domains_overlap:

    print(
        "Overlap:"
    )

    print(
        " ",
        pd.to_datetime(
            temporal_overlap_start,
            unit="s",
            utc=True
        ),
        "→",
        pd.to_datetime(
            temporal_overlap_end,
            unit="s",
            utc=True
        )
    )


# ==================================================================================================
# 10.2M — GT ROWS OUTSIDE TRAFFIC TEMPORAL DOMAIN
#
# This is important before treating GT as authoritative episode metadata.
# ==================================================================================================

gt_outside_traffic = (
    (gt_start < global_min_start)
    |
    (gt_start > global_max_end)
    |
    (gt_end < global_min_start)
    |
    (gt_end > global_max_end)
)


gt_outside_count = int(
    gt_outside_traffic.sum()
)


print("\n" + "=" * 135)
print("10.2M — GT TEMPORAL DOMAIN COVERAGE")
print("=" * 135)


print(
    f"GT rows                    : "
    f"{len(gt):,}"
)

print(
    f"GT rows outside traffic range: "
    f"{gt_outside_count:,}"
)

print(
    f"GT rows inside traffic range : "
    f"{len(gt) - gt_outside_count:,}"
)


# ==================================================================================================
# 10.2N — EVENT LIST CATEGORY AUDIT
#
# Still metadata only.
# We compare names but do NOT yet use LIST_EVENTS as row-level truth.
# ==================================================================================================

print("\n" + "=" * 135)
print("10.2N — EVENT-LIST CATEGORY AUDIT")
print("=" * 135)


events = pd.read_csv(
    EVENT_PATH,
    encoding="utf-8"
)

events.columns = [
    str(c).strip()
    for c in events.columns
]


events[
    "Attack category stripped"
] = (
    events[
        "Attack category"
    ]
    .astype(str)
    .str.strip()
)


print(
    events[
        [
            "Attack category",
            "Attack category stripped",
            "Attack subcategory",
            "Number of events",
        ]
    ]
    .head(30)
    .to_string(
        index=False
    )
)


print(
    "\nUnique stripped LIST_EVENTS categories:"
)

print(
    sorted(
        events[
            "Attack category stripped"
        ]
        .dropna()
        .unique()
        .tolist()
    )
)


# ==================================================================================================
# 10.2O — CATEGORY VOCABULARY COMPARISON
#
# Traffic attack_cat
# GT Attack category
# LIST_EVENTS Attack category
#
# We compare vocabularies only.
# We are NOT yet harmonizing anything.
# ==================================================================================================

traffic_category_set = {
    str(x).strip()

    for x in global_attack_cat_counts.keys()

    if str(x).strip()
    not in {
        "",
        "<EMPTY>",
        "-",
        "nan",
        "None",
    }
}


gt_category_set = set(
    gt_attack_clean
    .dropna()
    .unique()
)


event_category_set = set(
    events[
        "Attack category stripped"
    ]
    .dropna()
    .unique()
)


# normal is metadata, not an attack family
event_attack_only_set = {
    x
    for x in event_category_set
    if x.lower() != "normal"
}


print("\n" + "=" * 135)
print("10.2O — CATEGORY VOCABULARY COMPARISON")
print("=" * 135)


print(
    "\nTraffic attack_cat:"
)

print(
    sorted(
        traffic_category_set
    )
)


print(
    "\nGT attack categories:"
)

print(
    sorted(
        gt_category_set
    )
)


print(
    "\nLIST_EVENTS attack categories:"
)

print(
    sorted(
        event_attack_only_set
    )
)


print(
    "\nTraffic categories absent from GT:"
)

print(
    sorted(
        traffic_category_set
        -
        gt_category_set
    )
)


print(
    "\nGT categories absent from traffic:"
)

print(
    sorted(
        gt_category_set
        -
        traffic_category_set
    )
)


print(
    "\nTraffic categories absent from LIST_EVENTS:"
)

print(
    sorted(
        traffic_category_set
        -
        event_attack_only_set
    )
)


print(
    "\nLIST_EVENTS categories absent from traffic:"
)

print(
    sorted(
        event_attack_only_set
        -
        traffic_category_set
    )
)


# ==================================================================================================
# 10.2P — SAVE AUDIT ARTIFACTS
# ==================================================================================================

WIDTH_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_width_audit.csv"
)

TRAFFIC_SUMMARY_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_traffic_summary.csv"
)

PARTITION_RANGE_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_partition_ranges.csv"
)

PARTITION_OVERLAP_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_partition_overlap.csv"
)

LABEL_COUNTS_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_label_counts.csv"
)

ATTACK_COUNTS_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_attack_category_counts.csv"
)

GLOBAL_CROSS_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_global_label_attack_cross.csv"
)

MISSINGNESS_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_blank_value_audit.csv"
)

GT_VARIANTS_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_gt_category_variants.csv"
)

SCHEMA_OUT = (
    UNSW_OUT /
    "unsw_authoritative_49_feature_schema.csv"
)


width_audit.to_csv(
    WIDTH_OUT,
    index=False
)

traffic_summary_df.to_csv(
    TRAFFIC_SUMMARY_OUT,
    index=False
)

partition_ranges_df.to_csv(
    PARTITION_RANGE_OUT,
    index=False
)

range_relations_df.to_csv(
    PARTITION_OVERLAP_OUT,
    index=False
)

pd.DataFrame(
    label_counts
).to_csv(
    LABEL_COUNTS_OUT,
    index=False
)

pd.DataFrame(
    attack_category_counts
).to_csv(
    ATTACK_COUNTS_OUT,
    index=False
)

global_cross_df.to_csv(
    GLOBAL_CROSS_OUT,
    index=False
)

pd.DataFrame(
    missingness_rows
).to_csv(
    MISSINGNESS_OUT,
    index=False
)

gt_category_variant_df.to_csv(
    GT_VARIANTS_OUT,
    index=False
)

features.to_csv(
    SCHEMA_OUT,
    index=False
)


# ==================================================================================================
# 10.2Q — SAVE MANIFEST
# ==================================================================================================

manifest = {

    "phase":
        "10.2",

    "dataset":
        "UNSW-NB15",

    "traffic_files":
        TRAFFIC_FILES,

    "authoritative_feature_count":
        len(feature_names),

    "total_traffic_rows":
        int(global_rows),

    "critical_columns": {
        k: str(v)
        for k, v
        in critical_columns.items()
    },

    "global_invalid_start_timestamps":
        int(global_invalid_start),

    "global_invalid_end_timestamps":
        int(global_invalid_end),

    "global_end_before_start":
        int(global_end_before_start),

    "traffic_start_epoch":
        float(global_min_start),

    "traffic_end_epoch":
        float(global_max_end),

    "traffic_start_utc":
        str(traffic_min_dt),

    "traffic_end_utc":
        str(traffic_max_dt),

    "gt_rows":
        int(len(gt)),

    "gt_start_epoch":
        float(gt_min_start),

    "gt_end_epoch":
        float(gt_max_end),

    "gt_rows_outside_traffic_temporal_domain":
        int(gt_outside_count),

    "traffic_gt_temporal_domains_overlap":
        bool(temporal_domains_overlap),

    "traffic_attack_categories":
        sorted(
            traffic_category_set
        ),

    "gt_attack_categories":
        sorted(
            gt_category_set
        ),

    "event_list_attack_categories":
        sorted(
            event_attack_only_set
        ),

    "raw_files_modified":
        False,

    "traffic_files_combined":
        False,

    "episodes_created":
        False,

    "minute_aggregation_performed":
        False,

    "cic_unsw_mapping_performed":
        False,

    "ml_split_created":
        False,

    "scaling_performed":
        False,

    "model_training_performed":
        False,
}


MANIFEST_OUT = (
    UNSW_OUT /
    "unsw_phase_10_2_manifest.json"
)


with open(
    MANIFEST_OUT,
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        default=str
    )


# ==================================================================================================
# 10.2R — FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 135)
print("PHASE 10.2 — FINAL REPORT")
print("=" * 135)


print(
    f"""
AUTHORITATIVE SCHEMA
--------------------

Official traffic columns
    {len(feature_names)}

All raw traffic rows width-valid
    {"YES" if width_audit["Malformed_Rows"].sum() == 0 else "NO"}


TRAFFIC DATA
------------

Traffic rows
    {global_rows:,}

Traffic start
    {traffic_min_dt}

Traffic end
    {traffic_max_dt}

Invalid stime values
    {global_invalid_start:,}

Invalid ltime values
    {global_invalid_end:,}

Rows with ltime < stime
    {global_end_before_start:,}


GROUND TRUTH
------------

GT rows
    {len(gt):,}

GT start
    {gt_min_dt}

GT end
    {gt_max_dt}

GT rows outside traffic temporal domain
    {gt_outside_count:,}


LABEL SEMANTICS
---------------

Traffic attack categories
    {len(traffic_category_set)}

GT attack categories
    {len(gt_category_set)}

LIST_EVENTS attack categories
    {len(event_attack_only_set)}


IMPORTANT
---------

This phase has NOT decided that:

    • one GT row = one forecasting episode
    • one attack_cat run = one physical episode
    • the four files are one continuous sequence
    • UNSW should use CIC's episode algorithm
    • UNSW should use 1-minute aggregation
    • CIC and UNSW attack categories are equivalent


Those decisions require evidence from the temporal/event audit.


SAVED OUTPUTS
-------------

{WIDTH_OUT.name}
{TRAFFIC_SUMMARY_OUT.name}
{PARTITION_RANGE_OUT.name}
{PARTITION_OVERLAP_OUT.name}
{LABEL_COUNTS_OUT.name}
{ATTACK_COUNTS_OUT.name}
{GLOBAL_CROSS_OUT.name}
{MISSINGNESS_OUT.name}
{GT_VARIANTS_OUT.name}
{SCHEMA_OUT.name}
{MANIFEST_OUT.name}


RAW FILES MODIFIED
    NO
"""
)


print("=" * 135)
print("PHASE 10.2 COMPLETE")
print("=" * 135)

In [ ]:
# ==================================================================================================
# PHASE 10.3 — UNSW TEMPORAL SESSION, GROUND-TRUTH CORRESPONDENCE
#              & ATTACK-EVENT IDENTIFIABILITY AUDIT
# ==================================================================================================
#
# PURPOSE
# -------
# This is a FORENSIC / READ-ONLY audit.
#
# We are NOT yet:
#   - creating final attack episodes
#   - aggregating to 1-minute windows
#   - splitting train / validation / test
#   - generating forecasting sequences
#   - combining CIC + UNSW
#   - normalizing attack families
#
# Questions answered here:
#
#   A. What are the true continuous capture sessions in UNSW?
#   B. How does NUSW-NB15_GT.csv relate to attack-labelled traffic?
#   C. Can GT records help identify independent attack events?
#   D. Do attack categories contain multiple temporally/network-distinct occurrences?
#   E. Are Backdoor / Backdoors likely naming variants?
#   F. How much observable pre-attack history exists before candidate attack onsets?
#
# IMPORTANT:
#   Results from this cell are evidence for Phase 10.4.
#   Nothing discovered here is automatically declared a final "episode".
# ==================================================================================================

import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# --------------------------------------------------------------------------------------------------
# 0. PATHS
# --------------------------------------------------------------------------------------------------

UNSW_DIR = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2/raw/UNSW_NB15"
)

AUDIT_DIR = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2/audit_outputs/UNSW"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TRAFFIC_FILES = [
    UNSW_DIR / "UNSW-NB15_1.csv",
    UNSW_DIR / "UNSW-NB15_2.csv",
    UNSW_DIR / "UNSW-NB15_3.csv",
    UNSW_DIR / "UNSW-NB15_4.csv",
]

FEATURE_FILE = UNSW_DIR / "NUSW-NB15_features.csv"
GT_FILE = UNSW_DIR / "NUSW-NB15_GT.csv"

for p in TRAFFIC_FILES + [FEATURE_FILE, GT_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("=" * 120)
print("PHASE 10.3 — UNSW TEMPORAL SESSION, GT CORRESPONDENCE & EVENT IDENTIFIABILITY AUDIT")
print("=" * 120)
print("\n✓ Required UNSW files found")


# ==================================================================================================
# 1. RECOVER OFFICIAL TRAFFIC COLUMN NAMES — ENCODING-SAFE
# ==================================================================================================

def detect_text_encoding(path, encodings=("utf-8-sig", "utf-8", "cp1252", "latin1")):
    """
    Strictly tests the entire file against candidate encodings.
    Returns the first encoding that can decode the complete file.

    NOTE:
    latin1 is kept last because it can decode any byte sequence and therefore
    should only be used as a fallback.
    """
    raw = Path(path).read_bytes()

    for enc in encodings:
        try:
            raw.decode(enc, errors="strict")
            return enc
        except UnicodeDecodeError:
            continue

    raise UnicodeError(f"Could not determine encoding for: {path}")


print("\n" + "=" * 120)
print("10.3 SETUP — METADATA ENCODING VALIDATION")
print("=" * 120)

feature_encoding = detect_text_encoding(FEATURE_FILE)
gt_encoding = detect_text_encoding(GT_FILE)

print(f"\nFeature-definition file : {FEATURE_FILE.name}")
print(f"Detected encoding       : {feature_encoding}")

print(f"\nGround-truth file       : {GT_FILE.name}")
print(f"Detected encoding       : {gt_encoding}")


# Read official UNSW feature-definition file using its validated encoding.
features = pd.read_csv(
    FEATURE_FILE,
    encoding=feature_encoding,
    low_memory=False
)

features.columns = features.columns.astype(str).str.strip()

print("\nFeature-definition shape:")
print(features.shape)

print("\nFeature-definition columns:")
print(features.columns.tolist())


# --------------------------------------------------------------------------------------------------
# Locate the feature-name column robustly
# --------------------------------------------------------------------------------------------------

candidate_name_cols = [
    c for c in features.columns
    if c.lower() in {
        "name",
        "feature",
        "feature name",
        "feature_name"
    }
]

if candidate_name_cols:
    feature_name_col = candidate_name_cols[0]

else:
    # Official UNSW feature-definition file normally stores
    # feature names in its second column.
    if features.shape[1] < 2:
        raise ValueError(
            "Feature-definition file has fewer than 2 columns; "
            "cannot safely infer the feature-name column."
        )

    feature_name_col = features.columns[1]


traffic_columns = (
    features[feature_name_col]
    .astype(str)
    .str.strip()
    .tolist()
)


print(f"\nFeature-name column used: {feature_name_col}")
print(f"Recovered traffic fields : {len(traffic_columns)}")


if len(traffic_columns) != 49:
    print("\nRecovered fields:")
    for i, c in enumerate(traffic_columns, start=1):
        print(f"{i:>2}. {repr(c)}")

    raise ValueError(
        f"Expected 49 UNSW traffic columns from the official "
        f"feature-definition file, but recovered {len(traffic_columns)}."
    )


if len(set(traffic_columns)) != 49:
    duplicates = pd.Series(traffic_columns)[
        pd.Series(traffic_columns).duplicated(keep=False)
    ]

    raise ValueError(
        "Recovered feature names are not unique.\n"
        f"Duplicates:\n{duplicates.tolist()}"
    )


print("\n✓ Recovered exactly 49 unique official UNSW traffic columns")

# ==================================================================================================
# 2. IDENTIFY REQUIRED TRAFFIC FIELDS
# ==================================================================================================

def find_col(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}

    for candidate in candidates:
        key = candidate.strip().lower()
        if key in lookup:
            return lookup[key]

    raise KeyError(
        f"Could not find any of {candidates}\nAvailable columns:\n{list(columns)}"
    )


T_SRCIP   = find_col(traffic_columns, ["srcip", "src_ip"])
T_DSTIP   = find_col(traffic_columns, ["dstip", "dst_ip"])
T_SRCPORT = find_col(traffic_columns, ["sport", "srcport", "src_port"])
T_DSTPORT = find_col(traffic_columns, ["dsport", "dstport", "dst_port"])
T_PROTO   = find_col(traffic_columns, ["proto", "protocol"])
T_STIME   = find_col(traffic_columns, ["stime", "starttime", "start_time"])
T_LTIME   = find_col(traffic_columns, ["ltime", "endtime", "end_time"])
T_ATTACK  = find_col(traffic_columns, ["attack_cat", "attack category", "attack_category"])
T_LABEL   = find_col(traffic_columns, ["label"])

TRAFFIC_USECOLS = [
    T_SRCIP, T_DSTIP,
    T_SRCPORT, T_DSTPORT,
    T_PROTO,
    T_STIME, T_LTIME,
    T_ATTACK, T_LABEL
]

print("\nTraffic fields used:")
for c in TRAFFIC_USECOLS:
    print(f"  {c}")


# ==================================================================================================
# 3. LOAD MINIMAL TRAFFIC REPRESENTATION
# ==================================================================================================
#
# We intentionally load only fields needed for temporal/event forensics.
# This avoids loading all 49 columns across 2.54M rows.
# ==================================================================================================

traffic_parts = []

print("\n" + "=" * 120)
print("10.3A — LOADING MINIMAL TRAFFIC REPRESENTATION")
print("=" * 120)

for file_no, path in enumerate(TRAFFIC_FILES, start=1):

    print(f"\nLoading {path.name} ...")

    df = pd.read_csv(
        path,
        header=None,
        names=traffic_columns,
        usecols=TRAFFIC_USECOLS,
        low_memory=False
    )

    df["Source_File"] = path.name
    df["Source_File_Number"] = file_no
    df["Original_Row"] = np.arange(len(df), dtype=np.int64)

    traffic_parts.append(df)

    print(f"  rows: {len(df):,}")

traffic = pd.concat(traffic_parts, ignore_index=True)

del traffic_parts
gc.collect()

print(f"\n✓ Combined traffic rows: {len(traffic):,}")


# ==================================================================================================
# 4. CLEAN REPRESENTATION FOR MATCHING — RAW DATA ITSELF IS NOT MODIFIED
# ==================================================================================================

# Numeric timestamps
traffic[T_STIME] = pd.to_numeric(traffic[T_STIME], errors="coerce")
traffic[T_LTIME] = pd.to_numeric(traffic[T_LTIME], errors="coerce")
traffic[T_LABEL] = pd.to_numeric(traffic[T_LABEL], errors="coerce")

# Datetime representation
traffic["Start_DT"] = pd.to_datetime(
    traffic[T_STIME],
    unit="s",
    errors="coerce"
)

traffic["End_DT"] = pd.to_datetime(
    traffic[T_LTIME],
    unit="s",
    errors="coerce"
)

# Matching-only normalized string fields
for c in [T_SRCIP, T_DSTIP, T_SRCPORT, T_DSTPORT, T_PROTO]:
    traffic[f"{c}__norm"] = (
        traffic[c]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Attack category:
# only whitespace/case-normalized copy for comparison.
# NO Backdoor -> Backdoors merge.
traffic["Attack_Cat_Raw"] = traffic[T_ATTACK].astype("string")

traffic["Attack_Cat_Norm"] = (
    traffic[T_ATTACK]
    .astype("string")
    .str.strip()
    .str.lower()
)

traffic.loc[
    traffic["Attack_Cat_Norm"].isin(["", "nan", "none", "<na>"]),
    "Attack_Cat_Norm"
] = pd.NA


# ==================================================================================================
# 5. RECONSTRUCT GLOBAL CHRONOLOGICAL ORDER
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3B — GLOBAL CHRONOLOGICAL RECONSTRUCTION")
print("=" * 120)

invalid_start = traffic["Start_DT"].isna().sum()
invalid_end = traffic["End_DT"].isna().sum()

print(f"\nInvalid Start_DT : {invalid_start:,}")
print(f"Invalid End_DT   : {invalid_end:,}")

chrono = traffic.sort_values(
    ["Start_DT", "End_DT", "Source_File_Number", "Original_Row"],
    kind="mergesort"
).reset_index(drop=True)

chrono["Prev_Start_DT"] = chrono["Start_DT"].shift(1)

chrono["Start_Gap_Sec"] = (
    chrono["Start_DT"] - chrono["Prev_Start_DT"]
).dt.total_seconds()

positive_gaps = chrono.loc[
    chrono["Start_Gap_Sec"] > 0,
    "Start_Gap_Sec"
]

print("\nGLOBAL TIME RANGE")
print("-" * 80)
print("Start :", chrono["Start_DT"].min())
print("End   :", chrono["End_DT"].max())

print("\nPOSITIVE START-TIME GAP DISTRIBUTION")
print("-" * 80)

if len(positive_gaps) > 0:
    percentiles = [0.50, 0.90, 0.95, 0.99, 0.999, 1.0]

    for q in percentiles:
        print(
            f"{q * 100:>6.1f}% : "
            f"{positive_gaps.quantile(q):>12,.2f} sec"
        )

print("\nLARGEST GLOBAL START-TIME GAPS")
print("-" * 120)

largest_gaps = (
    chrono.loc[
        chrono["Start_Gap_Sec"].notna(),
        [
            "Prev_Start_DT",
            "Start_DT",
            "Start_Gap_Sec",
            "Source_File",
            T_LABEL,
            "Attack_Cat_Raw"
        ]
    ]
    .nlargest(20, "Start_Gap_Sec")
)

print(largest_gaps.to_string(index=False))


# ==================================================================================================
# 6. CANDIDATE CAPTURE-SESSION BOUNDARIES
# ==================================================================================================
#
# IMPORTANT:
# We do NOT declare one arbitrary threshold to be "truth".
#
# Instead we show how many boundaries would be produced under several
# candidate gap thresholds.
#
# The known ~26-day discontinuity should emerge naturally.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3C — CANDIDATE CAPTURE-SESSION BOUNDARY SENSITIVITY")
print("=" * 120)

candidate_gap_minutes = [1, 5, 10, 30, 60, 360, 1440]

session_sensitivity = []

for mins in candidate_gap_minutes:

    threshold_sec = mins * 60

    n_boundaries = int(
        (chrono["Start_Gap_Sec"] > threshold_sec).sum()
    )

    session_sensitivity.append({
        "Gap_Threshold_Minutes": mins,
        "Detected_Boundaries": n_boundaries,
        "Implied_Sessions": n_boundaries + 1
    })

session_sensitivity_df = pd.DataFrame(session_sensitivity)

print(session_sensitivity_df.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# For subsequent FORENSIC summaries only:
#
# Use > 6 hours as a conservative "major capture discontinuity" marker.
#
# This is NOT yet the final session rule.
# We use it only to prevent the 26-day gap from being treated as continuous
# during this audit.
# --------------------------------------------------------------------------------------------------

FORENSIC_MAJOR_GAP_SEC = 6 * 60 * 60

chrono["Major_Gap_Boundary"] = (
    chrono["Start_Gap_Sec"] > FORENSIC_MAJOR_GAP_SEC
)

chrono["Forensic_Session_ID"] = (
    chrono["Major_Gap_Boundary"].fillna(False).cumsum() + 1
)

session_summary = (
    chrono.groupby("Forensic_Session_ID", as_index=False)
    .agg(
        Start=("Start_DT", "min"),
        End=("End_DT", "max"),
        Flows=("Start_DT", "size"),
        Attack_Flows=(T_LABEL, lambda x: int((x == 1).sum())),
        Benign_Flows=(T_LABEL, lambda x: int((x == 0).sum())),
        Source_Files=("Source_File", lambda x: ", ".join(sorted(set(x))))
    )
)

session_summary["Duration_Hours"] = (
    session_summary["End"] - session_summary["Start"]
).dt.total_seconds() / 3600

print("\nFORENSIC MAJOR-SESSION SUMMARY")
print("NOTE: >6h is only an audit marker, NOT the final episode/session rule.")
print("-" * 120)

print(session_summary.to_string(index=False))


# ==================================================================================================
# 7. LOAD GT AND IDENTIFY GT COLUMNS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3D — GROUND-TRUTH STRUCTURE")
print("=" * 120)

gt = pd.read_csv(
    GT_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

gt.columns = gt.columns.astype(str).str.strip()

print(f"\nGT rows    : {len(gt):,}")
print(f"GT columns : {len(gt.columns):}")

print("\nGT column names:")
for c in gt.columns:
    print(f"  {c}")


def find_gt_col(candidates, required=True):

    lookup = {
        str(c).strip().lower(): c
        for c in gt.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in lookup:
            return lookup[key]

    if required:
        raise KeyError(
            f"Could not find GT column among {candidates}"
        )

    return None


G_SRCIP = find_gt_col(["srcip", "src_ip", "source ip", "source_ip"])
G_DSTIP = find_gt_col(["dstip", "dst_ip", "destination ip", "destination_ip"])

G_SRCPORT = find_gt_col(
    ["sport", "srcport", "src_port", "source port", "source_port"]
)

G_DSTPORT = find_gt_col(
    ["dsport", "dstport", "dst_port", "destination port", "destination_port"]
)

G_PROTO = find_gt_col(["proto", "protocol"])

G_START = find_gt_col(
    ["stime", "start time", "start_time", "starttime"]
)

G_END = find_gt_col(
    ["ltime", "last time", "last_time", "end time", "end_time", "endtime"]
)

G_ATTACK_CAT = find_gt_col(
    ["attack category", "attack_category", "attack_cat"],
    required=False
)

G_ATTACK_SUB = find_gt_col(
    ["attack subcategory", "attack_subcategory", "attack_subcat"],
    required=False
)

G_ATTACK_NAME = find_gt_col(
    ["attack name", "attack_name"],
    required=False
)

G_ATTACK_REF = find_gt_col(
    ["attack reference", "attack_reference"],
    required=False
)

print("\nIdentified GT fields:")
for name, value in {
    "SRCIP": G_SRCIP,
    "DSTIP": G_DSTIP,
    "SRCPORT": G_SRCPORT,
    "DSTPORT": G_DSTPORT,
    "PROTO": G_PROTO,
    "START": G_START,
    "END": G_END,
    "ATTACK_CATEGORY": G_ATTACK_CAT,
    "ATTACK_SUBCATEGORY": G_ATTACK_SUB,
    "ATTACK_NAME": G_ATTACK_NAME,
    "ATTACK_REFERENCE": G_ATTACK_REF
}.items():
    print(f"  {name:<20}: {value}")


# ==================================================================================================
# 8. PARSE GT TIME SAFELY
# ==================================================================================================

def parse_possible_epoch(series):

    numeric = pd.to_numeric(series, errors="coerce")

    # If most values are numeric, interpret as epoch seconds.
    if numeric.notna().mean() > 0.90:
        return pd.to_datetime(
            numeric,
            unit="s",
            errors="coerce"
        )

    # Otherwise parse textual timestamps.
    return pd.to_datetime(
        series,
        errors="coerce"
    )


gt["GT_Start_DT"] = parse_possible_epoch(gt[G_START])
gt["GT_End_DT"] = parse_possible_epoch(gt[G_END])

print("\nGT TIME INTEGRITY")
print("-" * 80)
print("Valid starts      :", f"{gt['GT_Start_DT'].notna().sum():,}")
print("Invalid starts    :", f"{gt['GT_Start_DT'].isna().sum():,}")
print("Valid ends        :", f"{gt['GT_End_DT'].notna().sum():,}")
print("Invalid ends      :", f"{gt['GT_End_DT'].isna().sum():,}")
print(
    "End < Start      :",
    f"{(gt['GT_End_DT'] < gt['GT_Start_DT']).sum():,}"
)
print("GT start          :", gt["GT_Start_DT"].min())
print("GT end            :", gt["GT_End_DT"].max())


# ==================================================================================================
# 9. NORMALIZED MATCHING FIELDS
# ==================================================================================================

for c in [G_SRCIP, G_DSTIP, G_SRCPORT, G_DSTPORT, G_PROTO]:

    gt[f"{c}__norm"] = (
        gt[c]
        .astype("string")
        .str.strip()
        .str.lower()
    )

if G_ATTACK_CAT is not None:

    gt["GT_Attack_Cat_Raw"] = gt[G_ATTACK_CAT].astype("string")

    gt["GT_Attack_Cat_Norm"] = (
        gt[G_ATTACK_CAT]
        .astype("string")
        .str.strip()
        .str.lower()
    )

else:
    gt["GT_Attack_Cat_Raw"] = pd.NA
    gt["GT_Attack_Cat_Norm"] = pd.NA


# ==================================================================================================
# 10. EXACT NETWORK-TUPLE CORRESPONDENCE
# ==================================================================================================
#
# First test:
#
#     srcip + dstip + sport + dsport + proto
#
# We do NOT yet require exact timestamps because GT and flow timing semantics
# may differ.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3E — GT ↔ ATTACK-TRAFFIC NETWORK-TUPLE CORRESPONDENCE")
print("=" * 120)

attack_traffic = chrono.loc[
    chrono[T_LABEL] == 1
].copy()

print(f"\nAttack-labelled traffic flows : {len(attack_traffic):,}")
print(f"GT rows                       : {len(gt):,}")


traffic_key_cols = [
    f"{T_SRCIP}__norm",
    f"{T_DSTIP}__norm",
    f"{T_SRCPORT}__norm",
    f"{T_DSTPORT}__norm",
    f"{T_PROTO}__norm"
]

gt_key_cols = [
    f"{G_SRCIP}__norm",
    f"{G_DSTIP}__norm",
    f"{G_SRCPORT}__norm",
    f"{G_DSTPORT}__norm",
    f"{G_PROTO}__norm"
]


def make_tuple_key(df, cols):

    temp = df[cols].astype("string").fillna("<NA>")

    return (
        temp.iloc[:, 0] + "|" +
        temp.iloc[:, 1] + "|" +
        temp.iloc[:, 2] + "|" +
        temp.iloc[:, 3] + "|" +
        temp.iloc[:, 4]
    )


attack_traffic["Tuple_Key"] = make_tuple_key(
    attack_traffic,
    traffic_key_cols
)

gt["Tuple_Key"] = make_tuple_key(
    gt,
    gt_key_cols
)

traffic_tuple_counts = attack_traffic["Tuple_Key"].value_counts()
gt_tuple_counts = gt["Tuple_Key"].value_counts()

traffic_tuple_set = set(traffic_tuple_counts.index)
gt_tuple_set = set(gt_tuple_counts.index)

gt["Tuple_Exists_In_Attack_Traffic"] = gt["Tuple_Key"].isin(
    traffic_tuple_set
)

attack_traffic["Tuple_Exists_In_GT"] = attack_traffic["Tuple_Key"].isin(
    gt_tuple_set
)

print("\nNETWORK-TUPLE COVERAGE")
print("-" * 80)

print(
    "GT rows whose tuple occurs in attack traffic : "
    f"{gt['Tuple_Exists_In_Attack_Traffic'].sum():,} / {len(gt):,} "
    f"({100 * gt['Tuple_Exists_In_Attack_Traffic'].mean():.2f}%)"
)

print(
    "Attack flows whose tuple occurs in GT         : "
    f"{attack_traffic['Tuple_Exists_In_GT'].sum():,} / {len(attack_traffic):,} "
    f"({100 * attack_traffic['Tuple_Exists_In_GT'].mean():.2f}%)"
)

shared_tuple_keys = traffic_tuple_set & gt_tuple_set

print(f"\nUnique attack-traffic tuples : {len(traffic_tuple_set):,}")
print(f"Unique GT tuples             : {len(gt_tuple_set):,}")
print(f"Shared tuple keys            : {len(shared_tuple_keys):,}")


# ==================================================================================================
# 11. TEMPORAL OVERLAP CORRESPONDENCE FOR SHARED TUPLES
# ==================================================================================================
#
# For each shared network tuple:
#
#   traffic flow interval = [Stime, Ltime]
#   GT interval           = [GT_Start, GT_End]
#
# A traffic flow overlaps GT if:
#
#   flow_start <= gt_end
#   AND
#   flow_end >= gt_start
#
# This does NOT declare the GT row to be an episode.
# It only measures temporal correspondence.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3F — TEMPORAL OVERLAP CORRESPONDENCE")
print("=" * 120)

# Group attack traffic by tuple.
traffic_groups = {
    key: group[
        [
            "Start_DT",
            "End_DT",
            "Attack_Cat_Raw",
            "Attack_Cat_Norm",
            "Source_File",
            "Forensic_Session_ID"
        ]
    ].sort_values("Start_DT")
    for key, group in attack_traffic.groupby("Tuple_Key", sort=False)
}

gt_overlap_counts = np.zeros(len(gt), dtype=np.int64)
gt_category_match_counts = np.zeros(len(gt), dtype=np.int64)

# Track which attack traffic rows are covered by at least one GT interval.
# We use original attack_traffic index.
covered_attack_indices = set()

for pos, (gt_idx, row) in enumerate(gt.iterrows()):

    key = row["Tuple_Key"]

    if key not in traffic_groups:
        continue

    tg = traffic_groups[key]

    start = row["GT_Start_DT"]
    end = row["GT_End_DT"]

    if pd.isna(start) or pd.isna(end):
        continue

    overlap_mask = (
        (tg["Start_DT"] <= end) &
        (tg["End_DT"] >= start)
    )

    overlap_count = int(overlap_mask.sum())
    gt_overlap_counts[pos] = overlap_count

    if overlap_count > 0:

        matched = tg.loc[overlap_mask]

        covered_attack_indices.update(matched.index.tolist())

        gt_cat = row["GT_Attack_Cat_Norm"]

        if pd.notna(gt_cat):

            gt_category_match_counts[pos] = int(
                (matched["Attack_Cat_Norm"] == gt_cat).sum()
            )


gt["Temporal_Overlap_Flow_Count"] = gt_overlap_counts
gt["Same_Category_Overlap_Count"] = gt_category_match_counts

gt["Has_Temporal_Overlap"] = (
    gt["Temporal_Overlap_Flow_Count"] > 0
)

gt["Has_Same_Category_Overlap"] = (
    gt["Same_Category_Overlap_Count"] > 0
)

print("\nGT TEMPORAL CORRESPONDENCE")
print("-" * 80)

print(
    "GT rows with >=1 tuple+time-overlapping attack flow : "
    f"{gt['Has_Temporal_Overlap'].sum():,} / {len(gt):,} "
    f"({100 * gt['Has_Temporal_Overlap'].mean():.2f}%)"
)

print(
    "GT rows with >=1 same-category overlap             : "
    f"{gt['Has_Same_Category_Overlap'].sum():,} / {len(gt):,} "
    f"({100 * gt['Has_Same_Category_Overlap'].mean():.2f}%)"
)

print(
    "Attack traffic rows covered by >=1 GT interval     : "
    f"{len(covered_attack_indices):,} / {len(attack_traffic):,} "
    f"({100 * len(covered_attack_indices) / max(len(attack_traffic), 1):.2f}%)"
)

print("\nDISTRIBUTION — NUMBER OF OVERLAPPING ATTACK FLOWS PER GT ROW")
print("-" * 80)

print(
    gt["Temporal_Overlap_Flow_Count"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
    .to_string()
)


# ==================================================================================================
# 12. GT CORRESPONDENCE BY ATTACK CATEGORY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3G — GT CORRESPONDENCE BY ATTACK CATEGORY")
print("=" * 120)

gt_category_summary = (
    gt.groupby(
        "GT_Attack_Cat_Raw",
        dropna=False
    )
    .agg(
        GT_Rows=("Tuple_Key", "size"),
        Tuple_Matched=("Tuple_Exists_In_Attack_Traffic", "sum"),
        Temporal_Matched=("Has_Temporal_Overlap", "sum"),
        Same_Category_Matched=("Has_Same_Category_Overlap", "sum"),
        Total_Overlapping_Flows=("Temporal_Overlap_Flow_Count", "sum")
    )
    .reset_index()
)

gt_category_summary["Temporal_Match_Rate"] = (
    gt_category_summary["Temporal_Matched"] /
    gt_category_summary["GT_Rows"]
)

print(gt_category_summary.to_string(index=False))


# ==================================================================================================
# 13. ATTACK-EVENT IDENTITY AUDIT
# ==================================================================================================
#
# GT may contain Attack Name / Subcategory / Reference.
#
# We inspect whether attack categories consist of multiple distinguishable
# attack identities.
#
# This matters because:
#
#   same attack FAMILY in train/test = allowed for known-type forecasting
#   same physical attack OCCURRENCE in train/test = leakage
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3H — ATTACK-EVENT IDENTITY SIGNALS IN GT")
print("=" * 120)

identity_cols = []

for c in [
    G_ATTACK_CAT,
    G_ATTACK_SUB,
    G_ATTACK_NAME,
    G_ATTACK_REF
]:
    if c is not None:
        identity_cols.append(c)

for c in identity_cols:

    print(f"\nFIELD: {c}")
    print("-" * 80)

    cleaned = (
        gt[c]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    print(f"Non-null values : {cleaned.notna().sum():,}")
    print(f"Unique values   : {cleaned.nunique(dropna=True):,}")

    vc = cleaned.value_counts(dropna=False).head(25)

    print("\nTop values:")
    print(vc.to_string())


# --------------------------------------------------------------------------------------------------
# Composite GT identity signatures
# --------------------------------------------------------------------------------------------------

signature_source_cols = [
    c for c in [
        G_ATTACK_CAT,
        G_ATTACK_SUB,
        G_ATTACK_NAME,
        G_ATTACK_REF
    ]
    if c is not None
]

if signature_source_cols:

    sig_df = pd.DataFrame(index=gt.index)

    for c in signature_source_cols:
        sig_df[c] = (
            gt[c]
            .astype("string")
            .str.strip()
            .str.lower()
            .fillna("<NA>")
        )

    gt["GT_Attack_Identity_Signature"] = (
        sig_df.astype(str).agg(" | ".join, axis=1)
    )

    print("\nCOMPOSITE ATTACK-IDENTITY SIGNATURES")
    print("-" * 80)
    print(
        "Unique signatures:",
        f"{gt['GT_Attack_Identity_Signature'].nunique():,}"
    )

    sig_counts = (
        gt["GT_Attack_Identity_Signature"]
        .value_counts()
        .head(30)
    )

    print("\nMost frequent signatures:")
    print(sig_counts.to_string())


# ==================================================================================================
# 14. BACKDOOR / BACKDOORS FORENSICS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3I — BACKDOOR vs BACKDOORS FORENSICS")
print("=" * 120)

if G_ATTACK_CAT is not None:

    gt_cat_stripped = (
        gt[G_ATTACK_CAT]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    backdoor_gt = gt.loc[
        gt_cat_stripped.isin(["backdoor", "backdoors"])
    ].copy()

    print(f"\nGT Backdoor/Backdoors rows: {len(backdoor_gt):,}")

    if len(backdoor_gt) > 0:

        summary_cols = [
            c for c in [
                G_ATTACK_CAT,
                G_ATTACK_SUB,
                G_ATTACK_NAME,
                G_ATTACK_REF
            ]
            if c is not None
        ]

        for c in summary_cols:

            print(f"\n{c}")
            print("-" * 80)

            temp = (
                backdoor_gt
                .assign(
                    Category_Stripped=backdoor_gt[G_ATTACK_CAT]
                    .astype("string")
                    .str.strip()
                )
                .groupby("Category_Stripped")[c]
                .agg(
                    Rows="size",
                    Unique=lambda x: (
                        x.astype("string")
                        .str.strip()
                        .nunique(dropna=True)
                    )
                )
            )

            print(temp.to_string())

        print("\nTIME RANGE")
        print("-" * 80)

        temp = (
            backdoor_gt
            .assign(
                Category_Stripped=backdoor_gt[G_ATTACK_CAT]
                .astype("string")
                .str.strip()
            )
            .groupby("Category_Stripped")
            .agg(
                Start=("GT_Start_DT", "min"),
                End=("GT_End_DT", "max"),
                Rows=("GT_Start_DT", "size")
            )
        )

        print(temp.to_string())


# ==================================================================================================
# 15. CANDIDATE ATTACK ONSET DISCOVERY FROM TRAFFIC
# ==================================================================================================
#
# IMPORTANT:
#
# These are NOT final episodes.
#
# We create candidate attack runs only to study:
#   - temporal clustering
#   - possible repeated occurrences
#   - pre-onset history availability
#
# A new candidate run begins when:
#
#   1. attack category changes
#   OR
#   2. forensic capture session changes
#   OR
#   3. gap from previous attack start exceeds a candidate threshold
#
# We perform SENSITIVITY analysis rather than pretending one threshold
# is correct.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3J — CANDIDATE ATTACK-RUN SENSITIVITY")
print("=" * 120)

attack_sorted = (
    chrono.loc[chrono[T_LABEL] == 1]
    .sort_values(["Forensic_Session_ID", "Start_DT", "End_DT"])
    .reset_index(drop=True)
)

candidate_episode_thresholds_sec = [
    1,
    5,
    30,
    60,
    300,
    600
]

candidate_run_summary = []

for threshold in candidate_episode_thresholds_sec:

    prev_time = attack_sorted["Start_DT"].shift(1)
    prev_cat = attack_sorted["Attack_Cat_Norm"].shift(1)
    prev_session = attack_sorted["Forensic_Session_ID"].shift(1)

    gap = (
        attack_sorted["Start_DT"] - prev_time
    ).dt.total_seconds()

    new_run = (
        prev_time.isna() |
        (attack_sorted["Forensic_Session_ID"] != prev_session) |
        (attack_sorted["Attack_Cat_Norm"] != prev_cat) |
        (gap > threshold)
    )

    run_id = new_run.cumsum()

    n_runs = int(run_id.nunique())

    candidate_run_summary.append({
        "Gap_Threshold_Sec": threshold,
        "Candidate_Attack_Runs": n_runs
    })

candidate_run_summary_df = pd.DataFrame(candidate_run_summary)

print(candidate_run_summary_df.to_string(index=False))


# ==================================================================================================
# 16. PRE-ATTACK HISTORY FEASIBILITY
# ==================================================================================================
#
# For a conservative audit we use a 60-second candidate run threshold.
#
# AGAIN:
# this does NOT define final episodes.
#
# It only provides candidate onsets from which we can ask:
#
#   "How much captured history exists immediately before this onset?"
#
# We cap available history at the beginning of the same forensic capture session.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3K — PRE-ATTACK HISTORY FEASIBILITY")
print("=" * 120)

AUDIT_RUN_GAP_SEC = 60

prev_time = attack_sorted["Start_DT"].shift(1)
prev_cat = attack_sorted["Attack_Cat_Norm"].shift(1)
prev_session = attack_sorted["Forensic_Session_ID"].shift(1)

attack_gap = (
    attack_sorted["Start_DT"] - prev_time
).dt.total_seconds()

new_candidate_run = (
    prev_time.isna() |
    (attack_sorted["Forensic_Session_ID"] != prev_session) |
    (attack_sorted["Attack_Cat_Norm"] != prev_cat) |
    (attack_gap > AUDIT_RUN_GAP_SEC)
)

attack_sorted["Audit_Candidate_Run_ID"] = (
    new_candidate_run.cumsum()
)

candidate_runs = (
    attack_sorted
    .groupby("Audit_Candidate_Run_ID", as_index=False)
    .agg(
        Forensic_Session_ID=("Forensic_Session_ID", "first"),
        Attack_Category=("Attack_Cat_Raw", "first"),
        Attack_Category_Norm=("Attack_Cat_Norm", "first"),
        Candidate_Onset=("Start_DT", "min"),
        Candidate_End=("End_DT", "max"),
        Attack_Flows=("Start_DT", "size")
    )
)

# Map session start/end.
session_start_map = (
    session_summary
    .set_index("Forensic_Session_ID")["Start"]
    .to_dict()
)

session_end_map = (
    session_summary
    .set_index("Forensic_Session_ID")["End"]
    .to_dict()
)

candidate_runs["Session_Start"] = (
    candidate_runs["Forensic_Session_ID"]
    .map(session_start_map)
)

candidate_runs["Session_End"] = (
    candidate_runs["Forensic_Session_ID"]
    .map(session_end_map)
)

candidate_runs["Captured_History_Before_Onset_Sec"] = (
    candidate_runs["Candidate_Onset"] -
    candidate_runs["Session_Start"]
).dt.total_seconds()

history_windows_sec = {
    ">=30_sec": 30,
    ">=1_min": 60,
    ">=2_min": 120,
    ">=5_min": 300,
    ">=10_min": 600,
    ">=15_min": 900,
    ">=30_min": 1800
}

history_rows = []

for label, sec in history_windows_sec.items():

    eligible = (
        candidate_runs["Captured_History_Before_Onset_Sec"] >= sec
    )

    history_rows.append({
        "Required_Pre_Onset_History": label,
        "Candidate_Runs_Eligible": int(eligible.sum()),
        "Total_Candidate_Runs": len(candidate_runs),
        "Eligible_Percent": (
            100 * eligible.mean()
            if len(candidate_runs) else np.nan
        )
    })

history_summary_df = pd.DataFrame(history_rows)

print(
    "\nNOTE: candidate runs use a 60-second audit threshold only."
)
print(
    "They are NOT final forecasting episodes.\n"
)

print(history_summary_df.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# By attack category
# --------------------------------------------------------------------------------------------------

print("\nPRE-ONSET HISTORY BY ATTACK CATEGORY")
print("-" * 120)

category_history_rows = []

for category, g in candidate_runs.groupby(
    "Attack_Category_Norm",
    dropna=False
):

    row = {
        "Attack_Category": category,
        "Candidate_Runs": len(g)
    }

    for label, sec in history_windows_sec.items():

        row[label] = int(
            (
                g["Captured_History_Before_Onset_Sec"] >= sec
            ).sum()
        )

    category_history_rows.append(row)

category_history_df = pd.DataFrame(
    category_history_rows
).sort_values(
    "Candidate_Runs",
    ascending=False
)

print(category_history_df.to_string(index=False))


# ==================================================================================================
# 17. KNOWN-TYPE FORECASTING WARNING AUDIT
# ==================================================================================================
#
# We are NOT splitting yet.
#
# We simply ask:
#
#   Does each attack category appear to have multiple candidate occurrences?
#
# A category represented by only one candidate occurrence cannot safely be
# distributed across train/val/test without splitting that occurrence.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3L — KNOWN-TYPE FORECASTING ELIGIBILITY WARNING")
print("=" * 120)

known_type_inventory = (
    candidate_runs
    .groupby("Attack_Category_Norm", dropna=False)
    .agg(
        Candidate_Runs=("Audit_Candidate_Run_ID", "nunique"),
        Total_Attack_Flows=("Attack_Flows", "sum"),
        First_Onset=("Candidate_Onset", "min"),
        Last_Onset=("Candidate_Onset", "max"),
        Forensic_Sessions=("Forensic_Session_ID", "nunique")
    )
    .reset_index()
)

known_type_inventory["Potential_3_Way_Split"] = (
    known_type_inventory["Candidate_Runs"] >= 3
)

print(
    "\nIMPORTANT: this is only a WARNING SCREEN based on candidate runs."
)
print(
    "Final known-type eligibility will be decided only after Phase 10.4 "
    "constructs defensible episodes.\n"
)

print(
    known_type_inventory
    .sort_values("Candidate_Runs", ascending=False)
    .to_string(index=False)
)


# ==================================================================================================
# 18. SAVE AUDIT OUTPUTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.3M — SAVING AUDIT OUTPUTS")
print("=" * 120)

session_sensitivity_df.to_csv(
    AUDIT_DIR / "10_3_session_boundary_sensitivity.csv",
    index=False
)

session_summary.to_csv(
    AUDIT_DIR / "10_3_forensic_session_summary.csv",
    index=False
)

gt_category_summary.to_csv(
    AUDIT_DIR / "10_3_gt_correspondence_by_category.csv",
    index=False
)

candidate_run_summary_df.to_csv(
    AUDIT_DIR / "10_3_candidate_attack_run_sensitivity.csv",
    index=False
)

history_summary_df.to_csv(
    AUDIT_DIR / "10_3_pre_attack_history_summary.csv",
    index=False
)

category_history_df.to_csv(
    AUDIT_DIR / "10_3_pre_attack_history_by_category.csv",
    index=False
)

known_type_inventory.to_csv(
    AUDIT_DIR / "10_3_known_type_candidate_inventory.csv",
    index=False
)

# Save compact GT correspondence evidence.
gt_save_cols = [
    G_SRCIP,
    G_DSTIP,
    G_SRCPORT,
    G_DSTPORT,
    G_PROTO,
    G_START,
    G_END,
    "GT_Start_DT",
    "GT_End_DT",
    "GT_Attack_Cat_Raw",
    "Tuple_Exists_In_Attack_Traffic",
    "Temporal_Overlap_Flow_Count",
    "Same_Category_Overlap_Count",
    "Has_Temporal_Overlap",
    "Has_Same_Category_Overlap"
]

for c in [
    G_ATTACK_SUB,
    G_ATTACK_NAME,
    G_ATTACK_REF
]:
    if c is not None and c not in gt_save_cols:
        gt_save_cols.append(c)

gt[gt_save_cols].to_csv(
    AUDIT_DIR / "10_3_gt_traffic_correspondence.csv",
    index=False
)

# Candidate runs are audit objects, NOT final episodes.
candidate_runs.to_csv(
    AUDIT_DIR / "10_3_candidate_attack_runs_NOT_FINAL_EPISODES.csv",
    index=False
)

print("\nSaved:")
for filename in [
    "10_3_session_boundary_sensitivity.csv",
    "10_3_forensic_session_summary.csv",
    "10_3_gt_correspondence_by_category.csv",
    "10_3_candidate_attack_run_sensitivity.csv",
    "10_3_pre_attack_history_summary.csv",
    "10_3_pre_attack_history_by_category.csv",
    "10_3_known_type_candidate_inventory.csv",
    "10_3_gt_traffic_correspondence.csv",
    "10_3_candidate_attack_runs_NOT_FINAL_EPISODES.csv"
]:
    print(" ✓", AUDIT_DIR / filename)


# ==================================================================================================
# 19. FINAL INTERPRETATION CHECKLIST
# ==================================================================================================

print("\n" + "=" * 120)
print("PHASE 10.3 COMPLETE — WHAT THIS CELL DID / DID NOT DECIDE")
print("=" * 120)

print("""
ESTABLISHED / MEASURED
----------------------
✓ Global UNSW chronological order reconstructed from timestamps
✓ Large temporal discontinuities inspected
✓ Candidate capture-session boundaries audited under multiple thresholds
✓ GT ↔ attack-traffic network tuple correspondence measured
✓ GT ↔ traffic temporal-overlap correspondence measured
✓ Correspondence inspected by attack category
✓ GT attack-name/subcategory/reference identity signals inspected
✓ Backdoor vs Backdoors investigated without silently merging them
✓ Candidate attack-run sensitivity measured under multiple gap thresholds
✓ Pre-onset captured-history feasibility measured
✓ Known-type class coverage warning inventory created


NOT YET DECIDED
---------------
✗ A GT row is NOT automatically an attack episode
✗ A candidate attack run is NOT automatically an attack episode
✗ The 6-hour major-gap marker is NOT the final capture-session definition
✗ The 60-second candidate-run threshold is NOT the final episode definition
✗ Backdoor and Backdoors have NOT automatically been merged
✗ No final attack taxonomy has been created
✗ No 1-minute aggregation has been performed
✗ No CIC + UNSW combination has been performed
✗ No train/validation/test split has been performed
✗ No sliding forecasting windows have been generated


LEAKAGE RULE FOR LATER PHASES
-----------------------------
Final attack/capture episodes must be assigned to train/validation/test
BEFORE overlapping forecasting windows are generated.

For the primary KNOWN-TYPE experiment:
    same attack FAMILY may occur across train/val/test,
    but the same physical attack EPISODE must never cross splits.
""")

print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.4 — UNSW DEFENSIBLE ATTACK EPISODE RECONSTRUCTION
# OPTIMIZED FULL VERSION
# ==================================================================================================
#
# PURPOSE
# -------
# Construct candidate physical attack episodes from UNSW-NB15 ground truth.
#
# Episode definition remains:
#
#   1. SAME major capture period
#   2. SAME fine attack identity
#   3. SAME source/destination host pair + protocol
#   4. intervals overlap OR are separated by <= 1 second
#
# IMPORTANT
# ---------
# This phase does NOT:
#   - merge attack categories
#   - remove rare attacks
#   - create minute windows
#   - split train/validation/test
#   - scale features
#   - balance classes
#   - create forecasting targets
#   - combine CIC + UNSW
#
# Phase 10.5 must still validate whether these candidate episodes are defensible.
# ==================================================================================================

import os
import time
import numpy as np
import pandas as pd


# ==================================================================================================
# 10.4 SETUP
# ==================================================================================================

TOTAL_T0 = time.time()

UNSW_RAW = (
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2/raw/UNSW_NB15"
)

UNSW_OUT = (
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2/audit_outputs/UNSW"
)

os.makedirs(UNSW_OUT, exist_ok=True)

GT_FILE = os.path.join(
    UNSW_RAW,
    "NUSW-NB15_GT.csv"
)

if not os.path.exists(GT_FILE):
    raise FileNotFoundError(
        f"Missing GT file:\n{GT_FILE}"
    )

print("=" * 120)
print("PHASE 10.4 — UNSW DEFENSIBLE ATTACK EPISODE RECONSTRUCTION — OPTIMIZED")
print("=" * 120)

print("\n✓ Ground-truth file found")
print(GT_FILE)


# ==================================================================================================
# 10.4A — LOAD AND NORMALIZE GROUND TRUTH
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4A — LOAD AND NORMALIZE GROUND TRUTH")
print("=" * 120)

t0 = time.time()

gt = pd.read_csv(
    GT_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

gt.columns = (
    gt.columns
    .astype(str)
    .str.strip()
)

required = [
    "Start time",
    "Last time",
    "Attack category",
    "Attack subcategory",
    "Protocol",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Attack Name",
    "Attack Reference"
]

missing = [
    c for c in required
    if c not in gt.columns
]

if missing:
    raise ValueError(
        f"Missing required GT columns:\n{missing}"
    )


# --------------------------------------------------------------------------------------------------
# Parse timestamps
# --------------------------------------------------------------------------------------------------

gt["Start_DT"] = pd.to_datetime(
    gt["Start time"],
    unit="s",
    errors="coerce"
)

gt["End_DT"] = pd.to_datetime(
    gt["Last time"],
    unit="s",
    errors="coerce"
)

bad_timestamp = (
    gt["Start_DT"].isna()
    |
    gt["End_DT"].isna()
)

if bad_timestamp.any():
    raise ValueError(
        f"{bad_timestamp.sum():,} GT rows contain invalid timestamps."
    )

bad_interval = (
    gt["End_DT"]
    <
    gt["Start_DT"]
)

if bad_interval.any():
    raise ValueError(
        f"{bad_interval.sum():,} GT rows have End_DT < Start_DT."
    )


# --------------------------------------------------------------------------------------------------
# Original row identity
# --------------------------------------------------------------------------------------------------

gt["GT_Row_ID"] = np.arange(
    len(gt),
    dtype=np.int64
)


# --------------------------------------------------------------------------------------------------
# FAST vectorized text normalization
# --------------------------------------------------------------------------------------------------

TEXT_COLUMNS = [
    "Attack category",
    "Attack subcategory",
    "Protocol",
    "Source IP",
    "Destination IP",
    "Attack Name",
    "Attack Reference"
]

for col in TEXT_COLUMNS:

    s = (
        gt[col]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.lower()
    )

    s = s.mask(
        s.isin(
            [
                "",
                "nan",
                "none",
                "null",
                "<na>"
            ]
        )
    )

    gt[col + "_Norm"] = s


# --------------------------------------------------------------------------------------------------
# FAST vectorized port normalization
# --------------------------------------------------------------------------------------------------

def normalize_port_series(series):

    s = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    s = s.mask(
        s.isin(
            [
                "",
                "nan",
                "none",
                "null",
                "<na>"
            ]
        )
    )

    numeric = pd.to_numeric(
        s,
        errors="coerce"
    )

    result = s.copy()

    numeric_mask = numeric.notna()

    result.loc[numeric_mask] = (
        numeric.loc[numeric_mask]
        .astype("Int64")
        .astype("string")
    )

    return result


gt["Source_Port_Norm"] = normalize_port_series(
    gt["Source Port"]
)

gt["Destination_Port_Norm"] = normalize_port_series(
    gt["Destination Port"]
)


print(f"GT rows loaded : {len(gt):,}")
print(
    f"GT time range  : "
    f"{gt['Start_DT'].min()} → {gt['End_DT'].max()}"
)
print(
    f"Runtime        : "
    f"{time.time() - t0:.2f} sec"
)


# ==================================================================================================
# 10.4B — MAJOR CAPTURE PERIOD ASSIGNMENT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4B — MAJOR CAPTURE PERIOD ASSIGNMENT")
print("=" * 120)

MAJOR_CAPTURE_BOUNDARY = pd.Timestamp(
    "2015-02-18 00:23:28"
)

gt["Capture_Period"] = np.where(
    gt["Start_DT"] < MAJOR_CAPTURE_BOUNDARY,
    "UNSW_CAPTURE_1",
    "UNSW_CAPTURE_2"
)

capture_summary = (
    gt
    .groupby(
        "Capture_Period",
        sort=False
    )
    .agg(
        Start=("Start_DT", "min"),
        End=("End_DT", "max"),
        GT_Rows=("GT_Row_ID", "size"),
        Attack_Categories=(
            "Attack category_Norm",
            "nunique"
        )
    )
    .reset_index()
)

print(
    capture_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.4C — ATTACK IDENTITY REPRESENTATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4C — ATTACK IDENTITY REPRESENTATION")
print("=" * 120)

#
# Original hierarchy retained:
#
#   Attack Name
#       ↓ if missing
#   Attack subcategory + category
#       ↓ if missing
#   Attack category
#

name = gt["Attack Name_Norm"]
subcat = gt["Attack subcategory_Norm"]
cat = gt["Attack category_Norm"]

gt["Attack_Identity"] = np.select(

    [
        name.notna(),
        subcat.notna()
    ],

    [
        "name::" + name.fillna(""),
        (
            "subcategory::"
            + cat.fillna("<missing>")
            + "::"
            + subcat.fillna("")
        )
    ],

    default=(
        "category::"
        + cat.fillna("<missing>")
    )
)

gt["Attack_Identity"] = (
    gt["Attack_Identity"]
    .astype("string")
)

print(
    f"Broad categories       : "
    f"{gt['Attack category_Norm'].nunique(dropna=True):,}"
)

print(
    f"Subcategories          : "
    f"{gt['Attack subcategory_Norm'].nunique(dropna=True):,}"
)

print(
    f"Attack names           : "
    f"{gt['Attack Name_Norm'].nunique(dropna=True):,}"
)

print(
    f"Identity keys produced : "
    f"{gt['Attack_Identity'].nunique(dropna=False):,}"
)


# ==================================================================================================
# 10.4D — NETWORK RELATIONSHIP REPRESENTATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4D — NETWORK RELATIONSHIP REPRESENTATION")
print("=" * 120)

#
# IMPORTANT:
# No row-wise apply() here.
#

src = gt["Source IP_Norm"].fillna("<NA>")
sport = gt["Source_Port_Norm"].fillna("<NA>")
dst = gt["Destination IP_Norm"].fillna("<NA>")
dport = gt["Destination_Port_Norm"].fillna("<NA>")
proto = gt["Protocol_Norm"].fillna("<NA>")


gt["Strict_Network_Key"] = (
    src
    + " | "
    + sport
    + " | "
    + dst
    + " | "
    + dport
    + " | "
    + proto
)


gt["HostPair_Protocol_Key"] = (
    src
    + " | "
    + dst
    + " | "
    + proto
)


gt["HostPair_Key"] = (
    src
    + " | "
    + dst
)


print(
    f"Strict network tuples      : "
    f"{gt['Strict_Network_Key'].nunique():,}"
)

print(
    f"Host-pair + protocol keys  : "
    f"{gt['HostPair_Protocol_Key'].nunique():,}"
)

print(
    f"Host-pair keys             : "
    f"{gt['HostPair_Key'].nunique():,}"
)


# ==================================================================================================
# 10.4E — PRIMARY TEMPORAL CONNECTIVITY RULE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4E — PRIMARY TEMPORAL CONNECTIVITY RULE")
print("=" * 120)

PRIMARY_TEMPORAL_TOLERANCE_SEC = 1

PRIMARY_TOLERANCE = pd.Timedelta(
    seconds=PRIMARY_TEMPORAL_TOLERANCE_SEC
)

print(
    "Primary temporal adjacency tolerance: "
    f"{PRIMARY_TEMPORAL_TOLERANCE_SEC} second"
)


# ==================================================================================================
# 10.4F — PRIMARY EPISODE RECONSTRUCTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4F — RECONSTRUCTING PRIMARY ATTACK EPISODES — VECTORIZED")
print("=" * 120)

t0 = time.time()

group_cols = [
    "Capture_Period",
    "Attack_Identity",
    "HostPair_Protocol_Key"
]

print(f"GT rows                    : {len(gt):,}")
print(
    f"Temporal tolerance         : "
    f"{PRIMARY_TEMPORAL_TOLERANCE_SEC} sec"
)

print("Episode grouping keys:")

for c in group_cols:
    print("  -", c)


# --------------------------------------------------------------------------------------------------
# 10.4F.1 — SORT ONCE
# --------------------------------------------------------------------------------------------------

print("\n[1/5] Sorting GT records...")

gt_sorted = (
    gt
    .sort_values(
        group_cols
        + [
            "Start_DT",
            "End_DT",
            "GT_Row_ID"
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

print("✓ Sorting complete")


# --------------------------------------------------------------------------------------------------
# 10.4F.2 — STRUCTURAL GROUP ID
# --------------------------------------------------------------------------------------------------

print("\n[2/5] Encoding structural groups...")

gt_sorted["_Structural_Group"] = (
    gt_sorted
    .groupby(
        group_cols,
        sort=False,
        dropna=False
    )
    .ngroup()
    .astype(np.int32)
)

structural_group_count = (
    gt_sorted["_Structural_Group"]
    .nunique()
)

print(
    f"✓ Structural groups: "
    f"{structural_group_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 10.4F.3 — TEMPORAL CONNECTIVITY
# --------------------------------------------------------------------------------------------------

print("\n[3/5] Computing interval connectivity...")

#
# Running maximum end within structural group.
#
# Example:
#
# A: 10:00 → 10:10
# B: 10:02 → 10:03
# C: 10:09 → 10:12
#
# C must remain connected to A's interval.
#

running_end = (
    gt_sorted
    .groupby(
        "_Structural_Group",
        sort=False
    )["End_DT"]
    .cummax()
)

previous_running_end = (
    running_end
    .groupby(
        gt_sorted["_Structural_Group"],
        sort=False
    )
    .shift()
)

new_episode = (
    previous_running_end.isna()
    |
    (
        gt_sorted["Start_DT"]
        >
        previous_running_end
        + PRIMARY_TOLERANCE
    )
)


# --------------------------------------------------------------------------------------------------
# 10.4F.4 — CREATE EPISODE IDS
# --------------------------------------------------------------------------------------------------

#
# Because the dataframe is sorted by structural group and then time,
# every True boundary starts a globally contiguous new episode.
#
# We therefore do NOT need MultiIndex + factorize.
#

episode_number = (
    new_episode
    .astype(np.int64)
    .cumsum()
)

gt_sorted["_Episode_Number"] = episode_number

gt_sorted["Episode_ID"] = (
    "UNSW_EP_"
    +
    episode_number
    .astype("string")
    .str.zfill(6)
)

episode_count = int(
    episode_number.iloc[-1]
)

print(
    f"✓ Primary episodes identified: "
    f"{episode_count:,}"
)


# --------------------------------------------------------------------------------------------------
# 10.4F.5 — BUILD EPISODE SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n[4/5] Building episode summary...")

episode_group = (
    gt_sorted
    .groupby(
        "Episode_ID",
        sort=False,
        observed=True
    )
)

episodes = (
    episode_group
    .agg(

        Capture_Period=(
            "Capture_Period",
            "first"
        ),

        Attack_Identity=(
            "Attack_Identity",
            "first"
        ),

        Attack_Category=(
            "Attack category_Norm",
            "first"
        ),

        HostPair_Protocol_Key=(
            "HostPair_Protocol_Key",
            "first"
        ),

        Episode_Start=(
            "Start_DT",
            "min"
        ),

        Episode_End=(
            "End_DT",
            "max"
        ),

        GT_Row_Count=(
            "GT_Row_ID",
            "size"
        ),

        Unique_Strict_Tuples=(
            "Strict_Network_Key",
            "nunique"
        ),

        Unique_Source_IPs=(
            "Source IP_Norm",
            "nunique"
        ),

        Unique_Destination_IPs=(
            "Destination IP_Norm",
            "nunique"
        ),

        Unique_Protocols=(
            "Protocol_Norm",
            "nunique"
        ),

        First_GT_Row_ID=(
            "GT_Row_ID",
            "min"
        ),

        Last_GT_Row_ID=(
            "GT_Row_ID",
            "max"
        )
    )
    .reset_index()
)


episodes["Duration_Sec"] = (
    episodes["Episode_End"]
    -
    episodes["Episode_Start"]
).dt.total_seconds()


# --------------------------------------------------------------------------------------------------
# Metadata aggregation
# --------------------------------------------------------------------------------------------------

def unique_join(series):

    values = (
        series
        .dropna()
        .astype(str)
        .unique()
    )

    if len(values) == 0:
        return ""

    if len(values) == 1:
        return values[0]

    return " || ".join(
        sorted(values)
    )


#
# Only descriptive columns requiring multi-value representation are processed.
# This is much smaller than doing Python work during episode reconstruction.
#

metadata_cols = [
    "Attack subcategory_Norm",
    "Attack Name_Norm",
    "Attack Reference_Norm",
    "Source IP_Norm",
    "Destination IP_Norm",
    "Protocol_Norm"
]

metadata_names = [
    "Attack_Subcategories",
    "Attack_Names",
    "Attack_References",
    "Source_IPs",
    "Destination_IPs",
    "Protocols"
]

metadata_parts = []

for source_col, output_col in zip(
    metadata_cols,
    metadata_names
):

    tmp = (
        gt_sorted[
            [
                "Episode_ID",
                source_col
            ]
        ]
        .dropna(subset=[source_col])
        .drop_duplicates()
        .groupby(
            "Episode_ID",
            sort=False
        )[source_col]
        .agg(
            lambda x:
            " || ".join(
                sorted(
                    x.astype(str)
                )
            )
        )
        .rename(output_col)
    )

    metadata_parts.append(tmp)


if metadata_parts:

    metadata = pd.concat(
        metadata_parts,
        axis=1
    ).reset_index()

    episodes = episodes.merge(
        metadata,
        on="Episode_ID",
        how="left",
        validate="one_to_one"
    )


for col in metadata_names:

    if col not in episodes.columns:
        episodes[col] = ""

    episodes[col] = (
        episodes[col]
        .fillna("")
    )


episode_column_order = [

    "Episode_ID",

    "Capture_Period",

    "Attack_Identity",

    "Attack_Category",

    "Attack_Subcategories",

    "Attack_Names",

    "Attack_References",

    "HostPair_Protocol_Key",

    "Episode_Start",

    "Episode_End",

    "Duration_Sec",

    "GT_Row_Count",

    "Unique_Strict_Tuples",

    "Unique_Source_IPs",

    "Unique_Destination_IPs",

    "Unique_Protocols",

    "Source_IPs",

    "Destination_IPs",

    "Protocols",

    "First_GT_Row_ID",

    "Last_GT_Row_ID"
]

episodes = episodes[
    episode_column_order
]


# --------------------------------------------------------------------------------------------------
# Restore assignment to original GT
# --------------------------------------------------------------------------------------------------

print("\n[5/5] Restoring episode assignments...")

assignment = (
    gt_sorted[
        [
            "GT_Row_ID",
            "Episode_ID"
        ]
    ]
    .sort_values("GT_Row_ID")
)

if len(assignment) != len(gt):
    raise RuntimeError(
        "GT assignment row-count mismatch."
    )

if assignment["GT_Row_ID"].duplicated().any():
    raise RuntimeError(
        "Duplicate GT_Row_ID encountered."
    )

#
# Since GT_Row_ID is 0..N-1, sorting assignment by GT_Row_ID
# lets us assign directly without creating a Python dictionary.
#

gt["Episode_ID"] = (
    assignment["Episode_ID"]
    .to_numpy()
)


# --------------------------------------------------------------------------------------------------
# Integrity
# --------------------------------------------------------------------------------------------------

if gt["Episode_ID"].isna().any():

    raise RuntimeError(
        "Some GT rows were not assigned "
        "to an episode."
    )


if not episodes["Episode_ID"].is_unique:

    raise RuntimeError(
        "Episode_ID uniqueness failure."
    )


if not (
    episodes["Episode_Start"]
    <=
    episodes["Episode_End"]
).all():

    raise RuntimeError(
        "Invalid episode timestamps detected."
    )


assigned_rows = int(
    episodes["GT_Row_Count"].sum()
)

if assigned_rows != len(gt):

    raise RuntimeError(
        "GT row conservation failure:\n"
        f"GT rows         = {len(gt):,}\n"
        f"Episode rows    = {assigned_rows:,}"
    )


runtime = time.time() - t0


print("\n" + "-" * 120)
print("PRIMARY EPISODE RECONSTRUCTION RESULT")
print("-" * 120)

print(
    f"GT rows                 : "
    f"{len(gt):,}"
)

print(
    f"Primary episodes        : "
    f"{len(episodes):,}"
)

print(
    f"GT rows / episode       : "
    f"{len(gt) / len(episodes):.2f}"
)

print(
    f"GT rows assigned        : "
    f"{gt['Episode_ID'].notna().sum():,}"
)

print(
    f"Episode row conservation: "
    f"{assigned_rows:,} / {len(gt):,}"
)

print(
    f"Runtime                  : "
    f"{runtime:.2f} sec"
)

print("\n✓ 1-second connectivity retained")
print("✓ Fine attack identity retained")
print("✓ Host-pair + protocol relationship retained")
print("✓ Capture-period separation retained")
print("✓ Every GT row assigned exactly once")


# ==================================================================================================
# 10.4G — EPISODE MEMBERSHIP INTEGRITY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4G — EPISODE MEMBERSHIP INTEGRITY")
print("=" * 120)

unassigned = int(
    gt["Episode_ID"]
    .isna()
    .sum()
)

duplicate_gt_ids = int(
    gt["GT_Row_ID"]
    .duplicated()
    .sum()
)

print(
    f"Assigned GT rows   : "
    f"{gt['Episode_ID'].notna().sum():,}"
)

print(
    f"Unassigned GT rows : "
    f"{unassigned:,}"
)

print(
    f"Duplicate GT IDs   : "
    f"{duplicate_gt_ids:,}"
)

if unassigned:
    raise RuntimeError(
        "Some GT rows are unassigned."
    )

if duplicate_gt_ids:
    raise RuntimeError(
        "GT_Row_ID is not unique."
    )

print(
    "\n✓ Every GT row belongs to "
    "exactly one primary episode"
)


# ==================================================================================================
# 10.4H — PRIMARY EPISODE SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4H — PRIMARY EPISODE SUMMARY")
print("=" * 120)


print("\nEPISODE DURATION — SECONDS")
print("-" * 80)

print(
    episodes["Duration_Sec"]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
            .999
        ]
    )
)


print("\nGT ROWS PER EPISODE")
print("-" * 80)

print(
    episodes["GT_Row_Count"]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)


print("\nEPISODES BY CAPTURE PERIOD")
print("-" * 80)

print(
    episodes[
        "Capture_Period"
    ].value_counts()
)


print("\nEPISODES BY ATTACK CATEGORY")
print("-" * 80)

category_episode_summary = (
    episodes
    .groupby(
        "Attack_Category",
        dropna=False
    )
    .agg(

        Episodes=(
            "Episode_ID",
            "size"
        ),

        GT_Rows=(
            "GT_Row_Count",
            "sum"
        ),

        Median_Duration_Sec=(
            "Duration_Sec",
            "median"
        ),

        Max_Duration_Sec=(
            "Duration_Sec",
            "max"
        )
    )
    .sort_values(
        "Episodes",
        ascending=False
    )
)

print(
    category_episode_summary
)


# ==================================================================================================
# 10.4I — CATEGORY × CAPTURE-PERIOD COVERAGE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4I — CATEGORY × CAPTURE-PERIOD EPISODE COVERAGE")
print("=" * 120)

category_capture = pd.crosstab(
    episodes["Attack_Category"],
    episodes["Capture_Period"]
)

category_capture[
    "Total_Episodes"
] = (
    category_capture.sum(axis=1)
)

capture_cols = [
    c
    for c in category_capture.columns
    if str(c).startswith(
        "UNSW_CAPTURE_"
    )
]

category_capture[
    "Capture_Period_Count"
] = (
    category_capture[
        capture_cols
    ]
    .gt(0)
    .sum(axis=1)
)

print(
    category_capture
)


# ==================================================================================================
# 10.4J — FINE ATTACK-IDENTITY REPETITION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4J — FINE ATTACK-IDENTITY REPETITION ACROSS CAPTURE PERIODS")
print("=" * 120)

identity_capture = (
    episodes
    .groupby(
        "Attack_Identity",
        sort=False
    )
    .agg(

        Episodes=(
            "Episode_ID",
            "size"
        ),

        Capture_Periods=(
            "Capture_Period",
            "nunique"
        ),

        Attack_Category=(
            "Attack_Category",
            "first"
        ),

        First_Onset=(
            "Episode_Start",
            "min"
        ),

        Last_Onset=(
            "Episode_Start",
            "max"
        )
    )
    .reset_index()
)


print(
    identity_capture[
        "Capture_Periods"
    ]
    .value_counts()
    .sort_index()
)


repeated_identity_count = int(
    (
        identity_capture[
            "Capture_Periods"
        ]
        > 1
    ).sum()
)

single_capture_identity_count = int(
    (
        identity_capture[
            "Capture_Periods"
        ]
        == 1
    ).sum()
)


print(
    "\nFine identities appearing in >1 capture period : "
    f"{repeated_identity_count:,}"
)

print(
    "Fine identities appearing in only one period    : "
    f"{single_capture_identity_count:,}"
)


# ==================================================================================================
# 10.4K — CROSS-EPISODE TEMPORAL OVERLAP AUDIT
# ==================================================================================================
#
# IMPORTANT OPTIMIZATION
# ----------------------
#
# We do NOT enumerate every pair of overlapping episodes.
#
# If 500 episodes are simultaneously active, explicitly constructing all
# overlapping pairs creates 124,750 pairs for that region alone.
#
# For this audit we only need:
#
#   - total overlap-pair count
#   - number of episodes involved in any overlap
#   - maximum simultaneous activity
#
# These can be calculated much more efficiently.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4K — CROSS-EPISODE TEMPORAL OVERLAP AUDIT — FAST")
print("=" * 120)

t0 = time.time()

ep_time = (
    episodes[
        [
            "Episode_ID",
            "Episode_Start",
            "Episode_End"
        ]
    ]
    .sort_values(
        [
            "Episode_Start",
            "Episode_End",
            "Episode_ID"
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

N_EPISODES = len(
    ep_time
)

start_ns = (
    ep_time["Episode_Start"]
    .astype("int64")
    .to_numpy()
)

end_ns = (
    ep_time["Episode_End"]
    .astype("int64")
    .to_numpy()
)


# --------------------------------------------------------------------------------------------------
# Pair count using sorted endpoint search
# --------------------------------------------------------------------------------------------------
#
# For episode i:
#
# previous episode j overlaps i if:
#
#       end_j >= start_i
#
# Because episodes are sorted by start time, all j < i have:
#
#       start_j <= start_i
#
# Number of previous ends strictly before start_i:
#
#       searchsorted(sorted_previous_ends, start_i, side="left")
#
# A Fenwick tree is unnecessary here because pair count can be obtained
# from sorted starts/ends globally.
# --------------------------------------------------------------------------------------------------

sorted_ends = np.sort(
    end_ns
)

#
# Number of intervals whose end is strictly before each start.
#

ended_before = np.searchsorted(
    sorted_ends,
    start_ns,
    side="left"
)

#
# Since the current interval itself has not ended before its own start,
# and episodes are ordered by start, the number of active earlier intervals is:
#
#     i - number ended before current start
#

indices = np.arange(
    N_EPISODES,
    dtype=np.int64
)

active_previous = (
    indices
    -
    ended_before
)

#
# Numerical guard.
#

active_previous = np.maximum(
    active_previous,
    0
)

overlap_pair_count = int(
    active_previous.sum()
)


# --------------------------------------------------------------------------------------------------
# Identify episodes involved in ANY overlap
# --------------------------------------------------------------------------------------------------
#
# An episode overlaps:
#
#   A) a previous episode if active_previous > 0
#
# OR
#
#   B) a later episode if next_start <= its end
#
# Because starts are sorted, checking the immediate next start is sufficient
# to know whether at least one later interval begins before this interval ends.
# --------------------------------------------------------------------------------------------------

overlaps_previous = (
    active_previous > 0
)

overlaps_later = np.zeros(
    N_EPISODES,
    dtype=bool
)

if N_EPISODES > 1:

    overlaps_later[:-1] = (
        start_ns[1:]
        <=
        end_ns[:-1]
    )


episodes_with_overlap_mask = (
    overlaps_previous
    |
    overlaps_later
)

overlapping_episode_count = int(
    episodes_with_overlap_mask.sum()
)

non_overlapping_episode_count = int(
    N_EPISODES
    -
    overlapping_episode_count
)


# --------------------------------------------------------------------------------------------------
# Maximum simultaneously active intervals
# --------------------------------------------------------------------------------------------------

#
# Closed intervals:
#
# if one episode ends exactly when another starts,
# they count as overlapping.
#
# Therefore START events must be processed before END events
# when timestamps are equal.
#

all_times = np.concatenate(
    [
        start_ns,
        end_ns
    ]
)

event_delta = np.concatenate(
    [
        np.ones(
            N_EPISODES,
            dtype=np.int8
        ),
        -np.ones(
            N_EPISODES,
            dtype=np.int8
        )
    ]
)

#
# lexsort last key is primary:
#
# primary = timestamp
# secondary = -delta
#
# start (+1) therefore comes before end (-1)
# at equal timestamps.
#

event_order = np.lexsort(
    (
        -event_delta,
        all_times
    )
)

running_active = np.cumsum(
    event_delta[
        event_order
    ],
    dtype=np.int64
)

max_simultaneously_active = int(
    running_active.max()
) if len(running_active) else 0


if N_EPISODES:

    overlap_episode_pct = (
        100.0
        *
        overlapping_episode_count
        /
        N_EPISODES
    )

else:

    overlap_episode_pct = np.nan


overlap_runtime = (
    time.time()
    -
    t0
)


print(
    f"Total primary episodes        : "
    f"{N_EPISODES:,}"
)

print(
    f"Overlapping episode pairs     : "
    f"{overlap_pair_count:,}"
)

print(
    f"Episodes involved in overlap  : "
    f"{overlapping_episode_count:,}"
)

print(
    f"Episodes without overlap      : "
    f"{non_overlapping_episode_count:,}"
)

print(
    f"Overlap episode percentage    : "
    f"{overlap_episode_pct:.4f}%"
)

print(
    f"Maximum simultaneously active : "
    f"{max_simultaneously_active:,}"
)

print(
    f"Audit runtime                  : "
    f"{overlap_runtime:.2f} sec"
)


cross_episode_overlap_summary = pd.DataFrame(
    [
        {
            "Primary_Episodes":
                N_EPISODES,

            "Overlapping_Episode_Pairs":
                overlap_pair_count,

            "Episodes_With_Overlap":
                overlapping_episode_count,

            "Episodes_Without_Overlap":
                non_overlapping_episode_count,

            "Overlap_Episode_Pct":
                overlap_episode_pct,

            "Max_Simultaneously_Active":
                max_simultaneously_active,

            "Audit_Runtime_Sec":
                overlap_runtime
        }
    ]
)

print(
    "\n✓ Fast temporal overlap audit complete"
)


# ==================================================================================================
# 10.4L — TEMPORAL-TOLERANCE SENSITIVITY — VECTORIZED
# ==================================================================================================
#
# OLD VERSION:
#
#   for tolerance
#       for group
#           sort group
#           iterrows()
#
# NEW VERSION:
#
#   gt_sorted is ALREADY sorted.
#   running interval ends are ALREADY known.
#
# For each tolerance we only evaluate:
#
#       Start_DT > previous_running_end + tolerance
#
# Therefore each sensitivity level is one vectorized comparison.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4L — EPISODE TEMPORAL-TOLERANCE SENSITIVITY — FAST")
print("=" * 120)

t0 = time.time()

SENSITIVITY_SECONDS = [
    0,
    1,
    2,
    5,
    10,
    30,
    60
]

sensitivity_rows = []

start_values = (
    gt_sorted["Start_DT"]
)

first_in_group = (
    previous_running_end.isna()
)


for sec in SENSITIVITY_SECONDS:

    tol = pd.Timedelta(
        seconds=sec
    )

    boundary = (
        first_in_group
        |
        (
            start_values
            >
            previous_running_end
            + tol
        )
    )

    count = int(
        boundary.sum()
    )

    sensitivity_rows.append(
        {
            "Temporal_Tolerance_Sec":
                sec,

            "Episode_Count":
                count
        }
    )


episode_sensitivity = pd.DataFrame(
    sensitivity_rows
)


baseline_1sec = int(
    episode_sensitivity.loc[
        episode_sensitivity[
            "Temporal_Tolerance_Sec"
        ] == 1,
        "Episode_Count"
    ].iloc[0]
)


episode_sensitivity[
    "Difference_From_1sec"
] = (
    episode_sensitivity[
        "Episode_Count"
    ]
    -
    baseline_1sec
)


episode_sensitivity[
    "Pct_Change_From_1sec"
] = (
    100.0
    *
    episode_sensitivity[
        "Difference_From_1sec"
    ]
    /
    baseline_1sec
)


print(
    episode_sensitivity.to_string(
        index=False
    )
)

print(
    f"\nSensitivity runtime: "
    f"{time.time() - t0:.2f} sec"
)


# ==================================================================================================
# 10.4M — FLAG SUSPICIOUS EPISODES
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4M — SUSPICIOUS EPISODE FLAGS")
print("=" * 120)

duration_99 = float(
    episodes[
        "Duration_Sec"
    ].quantile(.99)
)

rows_99 = float(
    episodes[
        "GT_Row_Count"
    ].quantile(.99)
)

tuples_99 = float(
    episodes[
        "Unique_Strict_Tuples"
    ].quantile(.99)
)


episodes[
    "Flag_Long_Duration"
] = (
    episodes[
        "Duration_Sec"
    ]
    >
    duration_99
)


episodes[
    "Flag_Many_GT_Rows"
] = (
    episodes[
        "GT_Row_Count"
    ]
    >
    rows_99
)


episodes[
    "Flag_Many_Network_Tuples"
] = (
    episodes[
        "Unique_Strict_Tuples"
    ]
    >
    tuples_99
)


episodes[
    "Needs_Review"
] = (
    episodes[
        [
            "Flag_Long_Duration",
            "Flag_Many_GT_Rows",
            "Flag_Many_Network_Tuples"
        ]
    ]
    .any(axis=1)
)


print(
    f"99th percentile duration : "
    f"{duration_99:,.2f} sec"
)

print(
    f"99th percentile GT rows  : "
    f"{rows_99:,.2f}"
)

print(
    f"99th percentile tuples   : "
    f"{tuples_99:,.2f}"
)

print(
    "\nEpisodes flagged for Phase 10.5 review: "
    f"{episodes['Needs_Review'].sum():,}"
)


# ==================================================================================================
# 10.4N — SAVE OUTPUTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4N — SAVING EPISODE RECONSTRUCTION OUTPUTS")
print("=" * 120)


EPISODE_FILE = os.path.join(
    UNSW_OUT,
    "10_4_primary_attack_episodes_CANDIDATE.csv"
)


GT_ASSIGNMENT_FILE = os.path.join(
    UNSW_OUT,
    "10_4_gt_episode_assignments.csv"
)


SENSITIVITY_FILE = os.path.join(
    UNSW_OUT,
    "10_4_episode_tolerance_sensitivity.csv"
)


CATEGORY_CAPTURE_FILE = os.path.join(
    UNSW_OUT,
    "10_4_category_capture_coverage.csv"
)


IDENTITY_CAPTURE_FILE = os.path.join(
    UNSW_OUT,
    "10_4_attack_identity_capture_coverage.csv"
)


OVERLAP_FILE = os.path.join(
    UNSW_OUT,
    "10_4_episode_overlap_summary.csv"
)


CATEGORY_SUMMARY_FILE = os.path.join(
    UNSW_OUT,
    "10_4_category_episode_summary.csv"
)


# --------------------------------------------------------------------------------------------------
# Save episode table
# --------------------------------------------------------------------------------------------------

episodes.to_csv(
    EPISODE_FILE,
    index=False
)


# --------------------------------------------------------------------------------------------------
# Save GT → episode mapping
# --------------------------------------------------------------------------------------------------

gt_assignment_columns = [

    "GT_Row_ID",

    "Episode_ID",

    "Capture_Period",

    "Start_DT",

    "End_DT",

    "Attack category",

    "Attack subcategory",

    "Attack Name",

    "Attack Reference",

    "Attack_Identity",

    "Source IP",

    "Source Port",

    "Destination IP",

    "Destination Port",

    "Protocol",

    "Strict_Network_Key",

    "HostPair_Protocol_Key"
]


gt[
    gt_assignment_columns
].to_csv(
    GT_ASSIGNMENT_FILE,
    index=False
)


episode_sensitivity.to_csv(
    SENSITIVITY_FILE,
    index=False
)


category_capture.to_csv(
    CATEGORY_CAPTURE_FILE
)


identity_capture.to_csv(
    IDENTITY_CAPTURE_FILE,
    index=False
)


cross_episode_overlap_summary.to_csv(
    OVERLAP_FILE,
    index=False
)


category_episode_summary.to_csv(
    CATEGORY_SUMMARY_FILE
)


print("\nSaved:")

print(
    " ✓",
    EPISODE_FILE
)

print(
    " ✓",
    GT_ASSIGNMENT_FILE
)

print(
    " ✓",
    SENSITIVITY_FILE
)

print(
    " ✓",
    CATEGORY_CAPTURE_FILE
)

print(
    " ✓",
    IDENTITY_CAPTURE_FILE
)

print(
    " ✓",
    OVERLAP_FILE
)

print(
    " ✓",
    CATEGORY_SUMMARY_FILE
)


# ==================================================================================================
# 10.4O — FINAL INTEGRITY CHECK
# ==================================================================================================

print("\n" + "=" * 120)
print("10.4O — FINAL INTEGRITY CHECK")
print("=" * 120)


# --------------------------------------------------------------------------------------------------
# Every GT row assigned
# --------------------------------------------------------------------------------------------------

assert (
    gt["Episode_ID"]
    .notna()
    .all()
)


# --------------------------------------------------------------------------------------------------
# GT row identity preserved
# --------------------------------------------------------------------------------------------------

assert (
    gt["GT_Row_ID"]
    .is_unique
)


# --------------------------------------------------------------------------------------------------
# Episode IDs unique
# --------------------------------------------------------------------------------------------------

assert (
    episodes[
        "Episode_ID"
    ]
    .is_unique
)


# --------------------------------------------------------------------------------------------------
# Episode timestamps valid
# --------------------------------------------------------------------------------------------------

assert (
    episodes[
        "Episode_Start"
    ]
    <=
    episodes[
        "Episode_End"
    ]
).all()


# --------------------------------------------------------------------------------------------------
# Capture period retained
# --------------------------------------------------------------------------------------------------

assert (
    episodes[
        "Capture_Period"
    ]
    .notna()
    .all()
)


# --------------------------------------------------------------------------------------------------
# GT row conservation
# --------------------------------------------------------------------------------------------------

assert (
    int(
        episodes[
            "GT_Row_Count"
        ].sum()
    )
    ==
    len(gt)
)


# --------------------------------------------------------------------------------------------------
# Primary sensitivity result must reproduce primary episode count
# --------------------------------------------------------------------------------------------------

assert (
    baseline_1sec
    ==
    len(episodes)
), (
    "ERROR: 1-second sensitivity count "
    "does not reproduce primary episode count."
)


# --------------------------------------------------------------------------------------------------
# No episode may cross major capture boundary
# --------------------------------------------------------------------------------------------------

cross_boundary = episodes[
    (
        episodes[
            "Episode_Start"
        ]
        <
        MAJOR_CAPTURE_BOUNDARY
    )
    &
    (
        episodes[
            "Episode_End"
        ]
        >=
        MAJOR_CAPTURE_BOUNDARY
    )
]


print(
    "Episodes crossing major capture boundary: "
    f"{len(cross_boundary):,}"
)

if len(cross_boundary):

    raise RuntimeError(
        "Unexpected episode crosses the "
        "major capture boundary."
    )

else:

    print(
        "✓ No episode crosses the major "
        "capture discontinuity"
    )


# --------------------------------------------------------------------------------------------------
# Structural consistency
# --------------------------------------------------------------------------------------------------

episode_structure_check = (
    gt_sorted
    .groupby(
        "Episode_ID",
        sort=False
    )
    .agg(

        Capture_Count=(
            "Capture_Period",
            "nunique"
        ),

        Identity_Count=(
            "Attack_Identity",
            "nunique"
        ),

        HostPairProtocol_Count=(
            "HostPair_Protocol_Key",
            "nunique"
        )
    )
)


if (
    episode_structure_check[
        "Capture_Count"
    ]
    .gt(1)
    .any()
):

    raise RuntimeError(
        "An episode contains multiple capture periods."
    )


if (
    episode_structure_check[
        "Identity_Count"
    ]
    .gt(1)
    .any()
):

    raise RuntimeError(
        "An episode contains multiple attack identities."
    )


if (
    episode_structure_check[
        "HostPairProtocol_Count"
    ]
    .gt(1)
    .any()
):

    raise RuntimeError(
        "An episode contains multiple host-pair/protocol keys."
    )


print("✓ Every GT row assigned exactly once")
print("✓ Episode IDs are unique")
print("✓ Episode start/end ordering valid")
print("✓ GT row conservation valid")
print("✓ Original attack taxonomy retained")
print("✓ Fine attack identity retained")
print("✓ Network identity retained")
print("✓ Capture provenance retained")
print("✓ 1-second sensitivity reproduces primary episode count")
print("✓ Structural episode consistency verified")


# ==================================================================================================
# 10.4P — FINAL SUMMARY
# ==================================================================================================

TOTAL_RUNTIME = (
    time.time()
    -
    TOTAL_T0
)


print("\n" + "=" * 120)
print("PHASE 10.4 COMPLETE — IMPORTANT STATUS")
print("=" * 120)


print(
    f"""
CONSTRUCTED
-----------
✓ Hierarchical UNSW attack identity
✓ Major capture-period provenance
✓ Network relationship representations
✓ Candidate physical attack episodes
✓ GT → episode membership
✓ Category/capture coverage
✓ Fine-identity/capture coverage
✓ Cross-episode temporal-overlap summary
✓ Temporal-tolerance sensitivity
✓ Suspicious-episode review flags


PRIMARY EPISODE DEFINITION
--------------------------
Capture period
    +
Fine attack identity
    +
Host pair + protocol
    +
Interval overlap / <= {PRIMARY_TEMPORAL_TOLERANCE_SEC}-second adjacency


PRIMARY RESULTS
---------------
GT rows                  : {len(gt):,}
Candidate episodes       : {len(episodes):,}
GT rows / episode        : {len(gt) / len(episodes):.2f}
Structural groups        : {structural_group_count:,}
Overlapping episodes     : {overlapping_episode_count:,}
Episodes flagged review  : {episodes['Needs_Review'].sum():,}


INTENTIONALLY NOT DONE
----------------------
✗ No attack categories merged
✗ Backdoor / Backdoors NOT merged
✗ No rare attacks removed
✗ No episodes removed
✗ No minute aggregation
✗ No feature scaling
✗ No class balancing
✗ No train/validation/test split
✗ No forecasting windows
✗ No CIC + UNSW combination
✗ No model training


PERFORMANCE
-----------
Total Phase 10.4 runtime : {TOTAL_RUNTIME:.2f} sec


NEXT
----
Phase 10.5 must validate whether this episode reconstruction is defensible.

In particular:

  • episode-count sensitivity
  • episode-duration distributions
  • giant/suspicious episodes
  • GT rows per episode
  • overlapping simultaneous attacks
  • same-family vs fine-identity repetition
  • capture-period coverage
  • whether host-pair + protocol is too strict
  • whether the 1-second temporal rule fragments real attacks
  • implications for known-type forecasting

DO NOT treat these episode IDs as frozen modeling splits yet.
"""
)

In [ ]:
# ==================================================================================================
# PHASE 10.5 — UNSW EPISODE FORENSIC VALIDATION
# ==================================================================================================
#
# PURPOSE
# -------
# Validate whether the candidate episode construction from Phase 10.4 is
# defensible BEFORE freezing Episode_ID.
#
# 10.4 raised two major questions:
#
#   1. Is the 1-second temporal tolerance over-fragmenting attacks?
#   2. Why do so many candidate episodes overlap in time?
#
# This phase therefore investigates:
#
#   A. 1-second candidate episode granularity
#   B. what actually changes when tolerance increases
#   C. whether larger tolerances merge only compatible records
#   D. whether host-pair+protocol is over-restrictive
#   E. why temporal overlap is so common
#   F. whether overlap is caused by duplicate/nested GT intervals
#   G. suspicious/giant candidate episodes
#   H. known-type forecasting support
#
# IMPORTANT:
#
# THIS CELL DOES NOT:
#
#   - choose the final tolerance automatically
#   - overwrite Phase 10.4 episodes
#   - merge attack categories
#   - split train/validation/test
#   - aggregate to minutes
#   - combine CIC + UNSW
#
# ==================================================================================================

import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict

print("=" * 120)
print("PHASE 10.5 — UNSW EPISODE FORENSIC VALIDATION")
print("=" * 120)


# ==================================================================================================
# 10.5A — LOAD PHASE 10.4 OUTPUTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5A — LOAD PHASE 10.4 OUTPUTS")
print("=" * 120)

UNSW_OUT = "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2/audit_outputs/UNSW"

EPISODE_FILE = os.path.join(
    UNSW_OUT,
    "10_4_primary_attack_episodes_CANDIDATE.csv"
)

GT_ASSIGNMENT_FILE = os.path.join(
    UNSW_OUT,
    "10_4_gt_episode_assignments.csv"
)

SENSITIVITY_FILE = os.path.join(
    UNSW_OUT,
    "10_4_episode_tolerance_sensitivity.csv"
)

for f in [EPISODE_FILE, GT_ASSIGNMENT_FILE, SENSITIVITY_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing Phase 10.4 output: {f}")


episodes_1s = pd.read_csv(
    EPISODE_FILE,
    low_memory=False
)

gt = pd.read_csv(
    GT_ASSIGNMENT_FILE,
    low_memory=False
)

sensitivity_10_4 = pd.read_csv(
    SENSITIVITY_FILE
)


for c in ["Episode_Start", "Episode_End"]:
    episodes_1s[c] = pd.to_datetime(
        episodes_1s[c],
        errors="coerce"
    )

for c in ["Start_DT", "End_DT"]:
    gt[c] = pd.to_datetime(
        gt[c],
        errors="coerce"
    )


print(f"1-second candidate episodes : {len(episodes_1s):,}")
print(f"GT rows                     : {len(gt):,}")

assert len(gt) == 174347
assert gt["Episode_ID"].notna().all()
assert episodes_1s["Episode_ID"].is_unique

print("\n✓ Phase 10.4 outputs loaded successfully")


# ==================================================================================================
# 10.5B — BASIC 1-SECOND FRAGMENTATION DIAGNOSTIC
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5B — 1-SECOND CANDIDATE EPISODE GRANULARITY")
print("=" * 120)

single_gt = (episodes_1s["GT_Row_Count"] == 1).sum()
zero_duration = (episodes_1s["Duration_Sec"] == 0).sum()
one_or_less = (episodes_1s["Duration_Sec"] <= 1).sum()

print(f"Total candidate episodes          : {len(episodes_1s):,}")
print(
    f"Single-GT-row episodes            : {single_gt:,} "
    f"({single_gt / len(episodes_1s):.2%})"
)
print(
    f"Zero-duration episodes             : {zero_duration:,} "
    f"({zero_duration / len(episodes_1s):.2%})"
)
print(
    f"Duration <= 1 sec                  : {one_or_less:,} "
    f"({one_or_less / len(episodes_1s):.2%})"
)

print("\nDURATION QUANTILES")
print("-" * 80)

print(
    episodes_1s["Duration_Sec"].quantile(
        [0, .25, .50, .75, .90, .95, .99, .999, 1]
    )
)

print("\nGT-ROW COUNT QUANTILES")
print("-" * 80)

print(
    episodes_1s["GT_Row_Count"].quantile(
        [0, .25, .50, .75, .90, .95, .99, .999, 1]
    )
)


# ==================================================================================================
# 10.5C — RECONSTRUCT ALTERNATIVE TOLERANCES WITH MEMBERSHIP
# ==================================================================================================
#
# 10.4 only counted alternative episode totals.
#
# Here we reconstruct membership at:
#
#     0, 1, 2, 5, 10, 30, 60 seconds
#
# so we can inspect WHAT gets merged.
#
# Grouping identity remains fixed:
#
#     Capture_Period
#     Attack_Identity
#     HostPair_Protocol_Key
#
# Therefore tolerance changes ONLY temporal connectivity.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5C — RECONSTRUCT ALTERNATIVE TEMPORAL TOLERANCES")
print("=" * 120)

GROUP_COLS = [
    "Capture_Period",
    "Attack_Identity",
    "HostPair_Protocol_Key"
]

TOLERANCES = [0, 1, 2, 5, 10, 30, 60]


gt_work = gt.sort_values(
    GROUP_COLS + ["Start_DT", "End_DT", "GT_Row_ID"]
).reset_index(drop=True)


def reconstruct_temporal_membership(data, tolerance_sec):

    tol = pd.Timedelta(seconds=tolerance_sec)

    membership = {}
    summary_rows = []

    ep_counter = 0

    for group_key, g in data.groupby(
        GROUP_COLS,
        sort=False,
        dropna=False
    ):

        g = g.sort_values(
            ["Start_DT", "End_DT", "GT_Row_ID"]
        )

        current_indices = []
        current_start = None
        current_end = None

        def close_current(rows, start, end):
            nonlocal ep_counter

            if not rows:
                return

            ep_counter += 1
            ep_id = f"T{tolerance_sec}_EP_{ep_counter:06d}"

            sub = g.loc[rows]

            for idx in rows:
                membership[int(g.loc[idx, "GT_Row_ID"])] = ep_id

            summary_rows.append({
                "Alt_Episode_ID": ep_id,
                "Tolerance_Sec": tolerance_sec,
                "Capture_Period": sub["Capture_Period"].iloc[0],
                "Attack_Identity": sub["Attack_Identity"].iloc[0],
                "HostPair_Protocol_Key": sub["HostPair_Protocol_Key"].iloc[0],
                "Attack_Category_Count": sub["Attack category"].nunique(dropna=True),
                "Attack_Subcategory_Count": sub["Attack subcategory"].nunique(dropna=True),
                "Attack_Name_Count": sub["Attack Name"].nunique(dropna=True),
                "Attack_Reference_Count": sub["Attack Reference"].nunique(dropna=True),
                "Strict_Tuple_Count": sub["Strict_Network_Key"].nunique(dropna=True),
                "Start": start,
                "End": end,
                "Duration_Sec": (end - start).total_seconds(),
                "GT_Row_Count": len(sub)
            })

        for idx, row in g.iterrows():

            start = row["Start_DT"]
            end = row["End_DT"]

            if current_start is None:
                current_indices = [idx]
                current_start = start
                current_end = end
                continue

            if start <= current_end + tol:

                current_indices.append(idx)

                if end > current_end:
                    current_end = end

            else:

                close_current(
                    current_indices,
                    current_start,
                    current_end
                )

                current_indices = [idx]
                current_start = start
                current_end = end

        close_current(
            current_indices,
            current_start,
            current_end
        )

    return (
        pd.Series(membership, name=f"Episode_T{tolerance_sec}"),
        pd.DataFrame(summary_rows)
    )


membership_by_tol = {}
summary_by_tol = {}

for sec in TOLERANCES:

    membership, summary = reconstruct_temporal_membership(
        gt_work,
        sec
    )

    membership_by_tol[sec] = membership
    summary_by_tol[sec] = summary

    print(
        f"{sec:>2} sec → "
        f"{len(summary):>8,} episodes"
    )


# ==================================================================================================
# 10.5D — TOLERANCE QUALITY TABLE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5D — TEMPORAL-TOLERANCE QUALITY TABLE")
print("=" * 120)

quality_rows = []

for sec in TOLERANCES:

    s = summary_by_tol[sec]

    quality_rows.append({
        "Tolerance_Sec": sec,
        "Episodes": len(s),

        "Single_Row_Episodes":
            int((s["GT_Row_Count"] == 1).sum()),

        "Single_Row_Pct":
            float((s["GT_Row_Count"] == 1).mean()),

        "Zero_Duration_Episodes":
            int((s["Duration_Sec"] == 0).sum()),

        "Zero_Duration_Pct":
            float((s["Duration_Sec"] == 0).mean()),

        "Median_Duration_Sec":
            float(s["Duration_Sec"].median()),

        "P95_Duration_Sec":
            float(s["Duration_Sec"].quantile(.95)),

        "P99_Duration_Sec":
            float(s["Duration_Sec"].quantile(.99)),

        "Max_Duration_Sec":
            float(s["Duration_Sec"].max()),

        "Median_GT_Rows":
            float(s["GT_Row_Count"].median()),

        "P95_GT_Rows":
            float(s["GT_Row_Count"].quantile(.95)),

        "P99_GT_Rows":
            float(s["GT_Row_Count"].quantile(.99)),

        "Max_GT_Rows":
            int(s["GT_Row_Count"].max())
    })


tolerance_quality = pd.DataFrame(quality_rows)

print(
    tolerance_quality.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==================================================================================================
# 10.5E — WHAT DOES A LARGER TOLERANCE ACTUALLY MERGE?
# ==================================================================================================
#
# We compare each larger tolerance to the 1-second candidate construction.
#
# Question:
#
#     When a larger episode contains multiple 1-second episodes,
#     are those original episodes genuinely compatible?
#
# Since temporal reconstruction is performed INSIDE:
#
#     Attack_Identity + HostPair_Protocol
#
# category/identity conflicts should be impossible.
#
# But we explicitly verify that instead of assuming it.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5E — MERGE FORENSICS RELATIVE TO 1-SECOND EPISODES")
print("=" * 120)


base_membership = membership_by_tol[1]

merge_rows = []

for sec in [2, 5, 10, 30, 60]:

    alt_membership = membership_by_tol[sec]

    comp = pd.DataFrame({
        "GT_Row_ID": gt["GT_Row_ID"]
    })

    comp["Episode_1s"] = comp["GT_Row_ID"].map(base_membership)
    comp["Episode_alt"] = comp["GT_Row_ID"].map(alt_membership)

    # How many 1s episodes are absorbed into each larger episode?
    mapping = (
        comp.groupby("Episode_alt")["Episode_1s"]
        .nunique()
    )

    merged_alt_ids = mapping[mapping > 1].index

    affected_1s = (
        comp.loc[
            comp["Episode_alt"].isin(merged_alt_ids),
            "Episode_1s"
        ]
        .nunique()
    )

    rows_affected = (
        comp["Episode_alt"]
        .isin(merged_alt_ids)
        .sum()
    )

    max_base_per_alt = int(mapping.max())

    merge_rows.append({
        "Tolerance_Sec": sec,
        "Alt_Episodes": len(summary_by_tol[sec]),
        "Alt_Episodes_Merging_2plus_1s_Episodes":
            int((mapping > 1).sum()),
        "Original_1s_Episodes_Affected":
            int(affected_1s),
        "GT_Rows_Affected":
            int(rows_affected),
        "Max_1s_Episodes_Merged_Into_One":
            max_base_per_alt
    })


merge_forensics = pd.DataFrame(merge_rows)

print(merge_forensics.to_string(index=False))


# ==================================================================================================
# 10.5F — VERIFY TEMPORAL MERGES NEVER CROSS IDENTITY/CAPTURE/NETWORK GROUP
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5F — MERGE COMPATIBILITY CHECK")
print("=" * 120)


compatibility_rows = []

for sec in [2, 5, 10, 30, 60]:

    temp = gt[
        [
            "GT_Row_ID",
            "Capture_Period",
            "Attack_Identity",
            "HostPair_Protocol_Key",
            "Attack category",
            "Attack subcategory",
            "Attack Name"
        ]
    ].copy()

    temp["Alt_Episode_ID"] = (
        temp["GT_Row_ID"]
        .map(membership_by_tol[sec])
    )

    chk = (
        temp.groupby("Alt_Episode_ID")
        .agg(
            Capture_Period_Count=("Capture_Period", "nunique"),
            Identity_Count=("Attack_Identity", "nunique"),
            HostProtocol_Count=("HostPair_Protocol_Key", "nunique")
        )
    )

    bad_capture = int((chk["Capture_Period_Count"] > 1).sum())
    bad_identity = int((chk["Identity_Count"] > 1).sum())
    bad_network = int((chk["HostProtocol_Count"] > 1).sum())

    compatibility_rows.append({
        "Tolerance_Sec": sec,
        "Cross_Capture_Merges": bad_capture,
        "Cross_Identity_Merges": bad_identity,
        "Cross_HostProtocol_Merges": bad_network
    })


compatibility = pd.DataFrame(
    compatibility_rows
)

print(compatibility.to_string(index=False))


# ==================================================================================================
# 10.5G — HOST-PAIR+PROTOCOL FRAGMENTATION AUDIT
# ==================================================================================================
#
# Even if temporal tolerance is sensible, the grouping key itself could be
# too strict.
#
# Example:
#
#     same attack identity
#     same time
#     same source/destination hosts
#     but different protocol
#
# would become separate episodes.
#
# Likewise, the same fine attack identity may target several hosts as part
# of one campaign.
#
# We DO NOT merge them here.
#
# We measure how often this happens.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5G — NETWORK-KEY FRAGMENTATION AUDIT")
print("=" * 120)


identity_network = (
    gt.groupby(
        ["Capture_Period", "Attack_Identity"],
        dropna=False
    )
    .agg(
        GT_Rows=("GT_Row_ID", "count"),
        HostPair_Protocol_Keys=("HostPair_Protocol_Key", "nunique"),
        Strict_Tuples=("Strict_Network_Key", "nunique"),
        Categories=("Attack category", "nunique"),
        Start=("Start_DT", "min"),
        End=("End_DT", "max")
    )
    .reset_index()
)


print("\nHOST-PAIR+PROTOCOL KEYS PER ATTACK IDENTITY")
print("-" * 80)

print(
    identity_network["HostPair_Protocol_Keys"]
    .describe(
        percentiles=[.50, .75, .90, .95, .99, .999]
    )
)


multi_network_identity = (
    identity_network["HostPair_Protocol_Keys"] > 1
).sum()

print(
    f"\nIdentity/capture groups spanning >1 host-pair+protocol key: "
    f"{multi_network_identity:,} / {len(identity_network):,} "
    f"({multi_network_identity / len(identity_network):.2%})"
)


print("\nLARGEST NETWORK-SPANNING IDENTITIES")
print("-" * 80)

print(
    identity_network
    .sort_values(
        ["HostPair_Protocol_Keys", "GT_Rows"],
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


# ==================================================================================================
# 10.5H — EXACT DUPLICATE / NEAR-DUPLICATE GT AUDIT
# ==================================================================================================
#
# Huge temporal overlap can be inflated if GT contains:
#
#     exact duplicate rows
#     repeated intervals
#     same identity/time across multiple network tuples
#
# We audit this explicitly.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5H — GT DUPLICATION / REPEATED-INTERVAL AUDIT")
print("=" * 120)


exact_key = [
    "Start_DT",
    "End_DT",
    "Attack_Identity",
    "Strict_Network_Key"
]

exact_dup_mask = gt.duplicated(
    subset=exact_key,
    keep=False
)

exact_dup_rows = int(exact_dup_mask.sum())

exact_duplicate_groups = (
    gt.loc[exact_dup_mask]
    .groupby(exact_key, dropna=False)
    .ngroups
    if exact_dup_rows
    else 0
)


print(f"Rows participating in exact duplicate groups : {exact_dup_rows:,}")
print(f"Exact duplicate groups                       : {exact_duplicate_groups:,}")


interval_key = [
    "Start_DT",
    "End_DT",
    "Attack_Identity"
]

interval_counts = (
    gt.groupby(
        interval_key,
        dropna=False
    )
    .agg(
        GT_Rows=("GT_Row_ID", "count"),
        Strict_Tuples=("Strict_Network_Key", "nunique"),
        HostProtocol_Keys=("HostPair_Protocol_Key", "nunique")
    )
    .reset_index()
)

repeated_interval = interval_counts[
    interval_counts["GT_Rows"] > 1
]


print(
    f"\nIdentity/time intervals represented by >1 GT row : "
    f"{len(repeated_interval):,}"
)

print(
    f"GT rows inside those repeated intervals          : "
    f"{repeated_interval['GT_Rows'].sum():,}"
)


print("\nLARGEST REPEATED IDENTITY/TIME INTERVALS")
print("-" * 80)

print(
    repeated_interval
    .sort_values("GT_Rows", ascending=False)
    .head(20)
    .to_string(index=False)
)


# ==================================================================================================
# 10.5I — EXPLAIN THE MASSIVE CROSS-EPISODE OVERLAP
# ==================================================================================================
#
# Instead of storing millions of overlap pairs again, perform a sweep-line
# concurrency analysis over the 1-second candidate episodes.
#
# This tells us:
#
#     how many candidate episodes are simultaneously active
#     how many attack categories are simultaneously active
#
# NOTE:
# zero-duration intervals are valid event points and are retained.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5I — TEMPORAL CONCURRENCY ANALYSIS")
print("=" * 120)


def concurrency_events(ep_df):

    rows = []

    for capture, g in ep_df.groupby("Capture_Period"):

        events = []

        for _, r in g.iterrows():

            # start processed before end at identical timestamps
            # so zero-duration intervals still register as active events

            events.append(
                (
                    r["Episode_Start"],
                    1,
                    r["Episode_ID"],
                    r["Attack_Category"]
                )
            )

            events.append(
                (
                    r["Episode_End"],
                    -1,
                    r["Episode_ID"],
                    r["Attack_Category"]
                )
            )

        # Start (+1) before End (-1) at same timestamp
        events.sort(
            key=lambda x: (
                x[0],
                -x[1]
            )
        )

        active_episodes = set()
        active_categories = defaultdict(int)

        max_episode_concurrency = 0
        max_category_concurrency = 0
        max_episode_time = None
        max_category_time = None

        concurrency_samples = []

        for t, event_type, ep_id, category in events:

            if event_type == 1:

                active_episodes.add(ep_id)
                active_categories[category] += 1

                ep_conc = len(active_episodes)
                cat_conc = sum(
                    1 for v in active_categories.values()
                    if v > 0
                )

                if ep_conc > max_episode_concurrency:
                    max_episode_concurrency = ep_conc
                    max_episode_time = t

                if cat_conc > max_category_concurrency:
                    max_category_concurrency = cat_conc
                    max_category_time = t

                concurrency_samples.append(
                    (
                        ep_conc,
                        cat_conc
                    )
                )

            else:

                active_episodes.discard(ep_id)

                if category in active_categories:
                    active_categories[category] -= 1

                    if active_categories[category] <= 0:
                        del active_categories[category]

        sample_df = pd.DataFrame(
            concurrency_samples,
            columns=[
                "Episode_Concurrency",
                "Category_Concurrency"
            ]
        )

        rows.append({
            "Capture_Period": capture,

            "Max_Simultaneous_Episodes":
                max_episode_concurrency,

            "Time_Max_Episode_Concurrency":
                max_episode_time,

            "Max_Simultaneous_Categories":
                max_category_concurrency,

            "Time_Max_Category_Concurrency":
                max_category_time,

            "Median_Episode_Concurrency":
                sample_df["Episode_Concurrency"].median(),

            "P95_Episode_Concurrency":
                sample_df["Episode_Concurrency"].quantile(.95),

            "P99_Episode_Concurrency":
                sample_df["Episode_Concurrency"].quantile(.99),

            "Median_Category_Concurrency":
                sample_df["Category_Concurrency"].median(),

            "P95_Category_Concurrency":
                sample_df["Category_Concurrency"].quantile(.95),

            "P99_Category_Concurrency":
                sample_df["Category_Concurrency"].quantile(.99)
        })

    return pd.DataFrame(rows)


concurrency_summary = concurrency_events(
    episodes_1s
)

print(
    concurrency_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.5J — MINUTE-LEVEL LABEL AMBIGUITY PREVIEW
# ==================================================================================================
#
# We are NOT building the final minute dataset.
#
# But because the eventual model uses temporal aggregation, we need to know:
#
#     how many attack-active minutes contain one family?
#     how many contain multiple families?
#
# This preview is constructed ONLY from candidate episode intervals.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5J — MINUTE-LEVEL MULTI-ATTACK AMBIGUITY PREVIEW")
print("=" * 120)


minute_records = []


for capture, g in episodes_1s.groupby("Capture_Period"):

    minute_to_categories = defaultdict(set)
    minute_to_episodes = defaultdict(set)

    for _, r in g.iterrows():

        start_minute = r["Episode_Start"].floor("min")
        end_minute = r["Episode_End"].floor("min")

        for minute in pd.date_range(
            start=start_minute,
            end=end_minute,
            freq="1min"
        ):

            minute_to_categories[minute].add(
                r["Attack_Category"]
            )

            minute_to_episodes[minute].add(
                r["Episode_ID"]
            )

    all_minutes = sorted(
        minute_to_categories.keys()
    )

    for minute in all_minutes:

        categories = minute_to_categories[minute]
        eps = minute_to_episodes[minute]

        minute_records.append({
            "Capture_Period": capture,
            "Minute": minute,
            "Active_Category_Count": len(categories),
            "Active_Episode_Count": len(eps),
            "Active_Categories": " || ".join(sorted(categories))
        })


minute_preview = pd.DataFrame(
    minute_records
)


if len(minute_preview):

    print(f"Attack-active minutes represented : {len(minute_preview):,}")

    single_family_minutes = (
        minute_preview["Active_Category_Count"] == 1
    ).sum()

    multi_family_minutes = (
        minute_preview["Active_Category_Count"] > 1
    ).sum()

    print(
        f"Single-family attack minutes      : "
        f"{single_family_minutes:,} "
        f"({single_family_minutes / len(minute_preview):.2%})"
    )

    print(
        f"Multi-family attack minutes       : "
        f"{multi_family_minutes:,} "
        f"({multi_family_minutes / len(minute_preview):.2%})"
    )

    print("\nACTIVE CATEGORY COUNT PER ATTACK MINUTE")
    print("-" * 80)

    print(
        minute_preview["Active_Category_Count"]
        .value_counts()
        .sort_index()
    )

    print("\nACTIVE EPISODE COUNT PER ATTACK MINUTE")
    print("-" * 80)

    print(
        minute_preview["Active_Episode_Count"]
        .describe(
            percentiles=[
                .50, .75, .90,
                .95, .99
            ]
        )
    )


# ==================================================================================================
# 10.5K — SUSPICIOUS / GIANT EPISODE FORENSICS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5K — SUSPICIOUS / GIANT 1-SECOND EPISODES")
print("=" * 120)


review = episodes_1s[
    episodes_1s["Needs_Review"] == True
].copy()


print(f"Flagged candidate episodes : {len(review):,}")


print("\nTOP 25 BY DURATION")
print("-" * 80)

cols_to_show = [
    "Episode_ID",
    "Capture_Period",
    "Attack_Category",
    "Attack_Identity",
    "Episode_Start",
    "Episode_End",
    "Duration_Sec",
    "GT_Row_Count",
    "Unique_Strict_Tuples"
]

print(
    review
    .sort_values(
        "Duration_Sec",
        ascending=False
    )[cols_to_show]
    .head(25)
    .to_string(index=False)
)


print("\nTOP 25 BY GT ROW COUNT")
print("-" * 80)

print(
    review
    .sort_values(
        "GT_Row_Count",
        ascending=False
    )[cols_to_show]
    .head(25)
    .to_string(index=False)
)


# ==================================================================================================
# 10.5L — CATEGORY-SPECIFIC GRANULARITY
# ==================================================================================================
#
# One tolerance may behave differently across attack families.
#
# This does NOT mean we will automatically use family-specific thresholds.
# We simply inspect whether fragmentation is concentrated in particular
# categories.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5L — CATEGORY-SPECIFIC EPISODE GRANULARITY")
print("=" * 120)


category_granularity = (
    episodes_1s
    .groupby("Attack_Category")
    .agg(
        Episodes=("Episode_ID", "count"),
        Median_Duration=("Duration_Sec", "median"),
        P95_Duration=("Duration_Sec", lambda x: x.quantile(.95)),
        P99_Duration=("Duration_Sec", lambda x: x.quantile(.99)),
        Single_Row_Episodes=("GT_Row_Count", lambda x: int((x == 1).sum())),
        Median_GT_Rows=("GT_Row_Count", "median"),
        P95_GT_Rows=("GT_Row_Count", lambda x: x.quantile(.95))
    )
)

category_granularity["Single_Row_Pct"] = (
    category_granularity["Single_Row_Episodes"]
    /
    category_granularity["Episodes"]
)

print(
    category_granularity
    .sort_values("Episodes", ascending=False)
    .to_string()
)


# ==================================================================================================
# 10.5M — KNOWN-TYPE FORECASTING SUPPORT AUDIT
# ==================================================================================================
#
# User's current objective:
#
#     KNOWN-TYPE forecasting.
#
# Therefore eventual test families must be represented in training.
#
# We are NOT constructing splits yet.
#
# We measure:
#
#     broad-family repetition
#     fine-identity repetition
#     episode support across capture periods
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5M — KNOWN-TYPE FORECASTING SUPPORT")
print("=" * 120)


family_capture = pd.crosstab(
    episodes_1s["Attack_Category"],
    episodes_1s["Capture_Period"]
)

family_capture["Total"] = family_capture.sum(axis=1)

capture_cols = [
    c for c in family_capture.columns
    if c.startswith("UNSW_CAPTURE_")
]

family_capture["Capture_Period_Count"] = (
    family_capture[capture_cols] > 0
).sum(axis=1)

family_capture["Known_Type_CrossCapture_Possible"] = (
    family_capture["Capture_Period_Count"] >= 2
)

print("\nBROAD FAMILY SUPPORT")
print("-" * 80)
print(family_capture)


identity_capture = (
    episodes_1s
    .groupby("Attack_Identity")
    .agg(
        Episodes=("Episode_ID", "count"),
        Capture_Period_Count=("Capture_Period", "nunique"),
        Attack_Category=("Attack_Category", "first")
    )
    .reset_index()
)


print("\nFINE IDENTITY SUPPORT")
print("-" * 80)

print(
    identity_capture["Capture_Period_Count"]
    .value_counts()
    .sort_index()
)


cross_capture_identities = (
    identity_capture["Capture_Period_Count"] >= 2
).sum()

print(
    f"\nFine identities occurring across both capture periods: "
    f"{cross_capture_identities:,} / {len(identity_capture):,}"
)


# ==================================================================================================
# 10.5N — CRITICAL SPLIT-SAFETY CHECK
# ==================================================================================================
#
# We still do NOT split.
#
# This section simply records constraints that the future splitter MUST obey.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5N — FUTURE SPLIT-SAFETY CONSTRAINTS")
print("=" * 120)

split_constraints = pd.DataFrame([
    {
        "Constraint":
            "Episode atomicity",

        "Requirement":
            "One physical episode must belong to exactly one split."
    },
    {
        "Constraint":
            "Known-type family support",

        "Requirement":
            "Every evaluated attack family in validation/test must have training support."
    },
    {
        "Constraint":
            "Window isolation",

        "Requirement":
            "Forecast windows must be generated only after episode/capture split assignment."
    },
    {
        "Constraint":
            "Temporal leakage",

        "Requirement":
            "Overlapping or directly dependent temporal windows must never cross splits."
    },
    {
        "Constraint":
            "Scaler leakage",

        "Requirement":
            "Feature scaling/imputation statistics must be fitted on training data only."
    },
    {
        "Constraint":
            "Dataset provenance",

        "Requirement":
            "CIC/UNSW source identity and original attack labels must remain recoverable."
    },
    {
        "Constraint":
            "Multi-attack periods",

        "Requirement":
            "Minutes containing multiple simultaneous families must not be silently forced into one arbitrary family."
    }
])

print(
    split_constraints.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.5O — SAVE VALIDATION OUTPUTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5O — SAVE FORENSIC VALIDATION OUTPUTS")
print("=" * 120)


files_to_save = {
    "10_5_tolerance_quality.csv":
        tolerance_quality,

    "10_5_tolerance_merge_forensics.csv":
        merge_forensics,

    "10_5_merge_compatibility.csv":
        compatibility,

    "10_5_identity_network_fragmentation.csv":
        identity_network,

    "10_5_repeated_identity_time_intervals.csv":
        repeated_interval,

    "10_5_temporal_concurrency_summary.csv":
        concurrency_summary,

    "10_5_minute_ambiguity_preview.csv":
        minute_preview,

    "10_5_suspicious_episode_review.csv":
        review,

    "10_5_category_granularity.csv":
        category_granularity.reset_index(),

    "10_5_known_type_family_support.csv":
        family_capture.reset_index(),

    "10_5_known_type_identity_support.csv":
        identity_capture,

    "10_5_future_split_constraints.csv":
        split_constraints
}


for filename, df in files_to_save.items():

    path = os.path.join(
        UNSW_OUT,
        filename
    )

    df.to_csv(
        path,
        index=False
    )

    print("✓", path)


# ==================================================================================================
# 10.5P — DECISION GATE
# ==================================================================================================
#
# IMPORTANT:
#
# This section deliberately does NOT automatically select:
#
#     1 sec
#     5 sec
#     10 sec
#     30 sec
#     60 sec
#
# Episode definition is a methodological decision.
#
# We first inspect the evidence produced above.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.5P — EPISODE-DEFINITION DECISION GATE")
print("=" * 120)


print(
"""
PHASE 10.5 HAS NOT FROZEN THE EPISODE DEFINITION.

Before finalizing UNSW episodes, inspect:

1. TOLERANCE QUALITY
   Does increasing tolerance substantially reduce zero-duration and
   single-row fragmentation?

2. MERGE FORENSICS
   How many original 1-second episodes are actually affected by
   2/5/10/30/60-second tolerances?

3. MERGE COMPATIBILITY
   Larger temporal tolerances must NOT cross:
       - capture periods
       - fine attack identities
       - host-pair+protocol groups

4. NETWORK FRAGMENTATION
   Determine whether host-pair+protocol creates artificial fragmentation
   of attacks that span several network relationships.

5. DUPLICATION
   Determine whether repeated GT intervals explain part of the enormous
   overlap count.

6. CONCURRENCY
   Determine whether simultaneous attacks are genuinely common in UNSW.

7. MINUTE AMBIGUITY
   Determine how often a future 1-minute forecasting target would contain
   multiple attack families.

8. KNOWN-TYPE SUPPORT
   Determine which families/identities can safely participate in the
   known-type experiment.

NO FINAL EPISODE IDS SHOULD BE USED FOR MODELING UNTIL THIS DECISION GATE
IS REVIEWED.
"""
)


print("=" * 120)
print("PHASE 10.5 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.6 — UNSW GROUND-TRUTH SEMANTIC RECONSTRUCTION — CORRECTED
#
# IMPORTANT:
# NUSW-NB15_GT.csv is its OWN 12-column event/ground-truth table.
# NUSW-NB15_features.csv describes the 49-column UNSW flow CSVs and MUST NOT
# be used as the GT schema.
#
# PURPOSE:
# Understand what constitutes an UNSW attack event before freezing Event_IDs.
#
# THIS PHASE DOES NOT:
#   - alter raw data
#   - permanently remove duplicates
#   - create final Event_IDs
#   - split train/validation/test
#   - create forecasting windows
# ==================================================================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

print("=" * 120)
print("PHASE 10.6 — UNSW GROUND-TRUTH SEMANTIC RECONSTRUCTION — CORRECTED")
print("=" * 120)


# ==================================================================================================
# 10.6A — PATHS
# ==================================================================================================

BASE = Path("/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2")
UNSW_RAW = BASE / "raw" / "UNSW_NB15"
UNSW_OUT = BASE / "audit_outputs" / "UNSW"

UNSW_OUT.mkdir(parents=True, exist_ok=True)

GT_FILE = UNSW_RAW / "NUSW-NB15_GT.csv"

assert GT_FILE.exists()

print("\n✓ GT file found")
print(GT_FILE)


# ==================================================================================================
# 10.6B — LOAD GT USING ITS OWN HEADER
# ==================================================================================================

def robust_read(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            return df, enc
        except UnicodeDecodeError:
            continue

    raise RuntimeError("Could not decode file.")


gt, gt_encoding = robust_read(GT_FILE)

gt.columns = (
    gt.columns.astype(str)
    .str.strip()
)

print("\n" + "=" * 120)
print("10.6B — GT STRUCTURE")
print("=" * 120)

print("Shape   :", gt.shape)
print("Encoding:", gt_encoding)

print("\nColumns:")
for i, c in enumerate(gt.columns):
    print(f"{i:2d}: {repr(c)}")

print("\nFirst 3 rows:")
print(gt.head(3).to_string(index=False))


# ==================================================================================================
# 10.6C — IDENTIFY REQUIRED SEMANTIC COLUMNS
# ==================================================================================================

def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

lookup = {norm(c): c for c in gt.columns}

required = {
    "Start_Time": "starttime",
    "Last_Time": "lasttime",
    "Attack_Category": "attackcategory",
    "Attack_Subcategory": "attacksubcategory",
    "Protocol": "protocol",
    "Source_IP": "sourceip",
    "Source_Port": "sourceport",
    "Destination_IP": "destinationip",
    "Destination_Port": "destinationport",
    "Attack_Name": "attackname"
}

COL = {}

for logical, normalized in required.items():

    if normalized not in lookup:
        raise KeyError(
            f"Required GT column not found: {logical}. "
            f"Available columns: {list(gt.columns)}"
        )

    COL[logical] = lookup[normalized]


print("\n" + "=" * 120)
print("10.6C — SEMANTIC COLUMN MAP")
print("=" * 120)

for k, v in COL.items():
    print(f"{k:22s} -> {repr(v)}")


# ==================================================================================================
# 10.6D — REMOVE ONLY COMPLETELY EMPTY STRUCTURAL ROWS IN WORKING COPY
# ==================================================================================================

physical_rows = len(gt)

empty_mask = gt.isna().all(axis=1)

print("\n" + "=" * 120)
print("10.6D — STRUCTURAL ROW CHECK")
print("=" * 120)

print(f"Physical GT rows        : {physical_rows:,}")
print(f"Fully empty rows        : {empty_mask.sum():,}")

gt_work = gt.loc[~empty_mask].copy()

print(f"Working semantic rows   : {len(gt_work):,}")
print("\nRaw GT file remains unchanged.")


# ==================================================================================================
# 10.6E — PARSE TIME
# ==================================================================================================

start_numeric = pd.to_numeric(
    gt_work[COL["Start_Time"]],
    errors="coerce"
)

end_numeric = pd.to_numeric(
    gt_work[COL["Last_Time"]],
    errors="coerce"
)

gt_work["Start_DT"] = pd.to_datetime(
    start_numeric,
    unit="s",
    errors="coerce"
)

gt_work["End_DT"] = pd.to_datetime(
    end_numeric,
    unit="s",
    errors="coerce"
)

gt_work["GT_Duration_Sec"] = (
    gt_work["End_DT"] -
    gt_work["Start_DT"]
).dt.total_seconds()


print("\n" + "=" * 120)
print("10.6E — TEMPORAL INTEGRITY")
print("=" * 120)

print(f"Rows                  : {len(gt_work):,}")
print(f"Valid start timestamps: {gt_work['Start_DT'].notna().sum():,}")
print(f"Valid end timestamps  : {gt_work['End_DT'].notna().sum():,}")
print(f"Invalid starts        : {gt_work['Start_DT'].isna().sum():,}")
print(f"Invalid ends          : {gt_work['End_DT'].isna().sum():,}")

print("\nGlobal time range:")
print("Start:", gt_work["Start_DT"].min())
print("End  :", gt_work["End_DT"].max())

print("\nGT duration distribution:")
print(
    gt_work["GT_Duration_Sec"].describe(
        percentiles=[.25, .5, .75, .9, .95, .99, .999]
    )
)

print(
    "\nNegative durations:",
    int((gt_work["GT_Duration_Sec"] < 0).sum())
)


# ==================================================================================================
# 10.6F — CAPTURE-DAY STRUCTURE
# ==================================================================================================

gt_work["Capture_Date"] = gt_work["Start_DT"].dt.date

dates = sorted(
    gt_work["Capture_Date"]
    .dropna()
    .unique()
)

capture_map = {
    date: f"UNSW_CAPTURE_{i+1}"
    for i, date in enumerate(dates)
}

gt_work["Capture_ID"] = (
    gt_work["Capture_Date"]
    .map(capture_map)
)


print("\n" + "=" * 120)
print("10.6F — CAPTURE STRUCTURE")
print("=" * 120)

capture_summary = (
    gt_work
    .groupby("Capture_ID", dropna=False)
    .agg(
        Rows=("Start_DT", "size"),
        Start=("Start_DT", "min"),
        End=("End_DT", "max"),
        Attack_Families=(COL["Attack_Category"], "nunique"),
        Attack_Names=(COL["Attack_Name"], "nunique")
    )
    .reset_index()
)

print(capture_summary.to_string(index=False))


# ==================================================================================================
# 10.6G — NORMALIZE SEMANTIC TEXT
# ==================================================================================================

def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA
        })
    )


gt_work["Attack_Family_Raw"] = clean_text(
    gt_work[COL["Attack_Category"]]
)

gt_work["Attack_Subcategory_Raw"] = clean_text(
    gt_work[COL["Attack_Subcategory"]]
)

gt_work["Attack_Name_Raw"] = clean_text(
    gt_work[COL["Attack_Name"]]
)

gt_work["Attack_Family_Norm"] = (
    gt_work["Attack_Family_Raw"]
    .str.lower()
)

gt_work["Attack_Subcategory_Norm"] = (
    gt_work["Attack_Subcategory_Raw"]
    .str.lower()
)

# IMPORTANT:
# Attack Name is the fine identity available directly in the GT.
# We preserve its original semantics and only strip whitespace.
gt_work["Attack_Identity"] = gt_work["Attack_Name_Raw"]


print("\n" + "=" * 120)
print("10.6G — ATTACK TAXONOMY")
print("=" * 120)

print("\nBroad attack families:")
print(
    gt_work["Attack_Family_Raw"]
    .value_counts(dropna=False)
)

print(
    "\nUnique broad families :",
    gt_work["Attack_Family_Raw"].nunique(dropna=True)
)

print(
    "Unique subcategories :",
    gt_work["Attack_Subcategory_Raw"].nunique(dropna=True)
)

print(
    "Unique attack names  :",
    gt_work["Attack_Identity"].nunique(dropna=True)
)


# ==================================================================================================
# 10.6H — CHECK ATTACK NAME ↔ FAMILY CONSISTENCY
# ==================================================================================================

identity_family = (
    gt_work
    .groupby("Attack_Identity", dropna=False)
    .agg(
        Rows=("Attack_Identity", "size"),
        Family_Count=("Attack_Family_Norm", "nunique"),
        Subcategory_Count=("Attack_Subcategory_Norm", "nunique")
    )
    .reset_index()
)

ambiguous_family = identity_family[
    identity_family["Family_Count"] > 1
]

ambiguous_subcat = identity_family[
    identity_family["Subcategory_Count"] > 1
]


print("\n" + "=" * 120)
print("10.6H — ATTACK IDENTITY TAXONOMY CONSISTENCY")
print("=" * 120)

print(
    "Attack names mapped to >1 broad family:",
    f"{len(ambiguous_family):,}"
)

print(
    "Attack names mapped to >1 subcategory:",
    f"{len(ambiguous_subcat):,}"
)

if len(ambiguous_family):
    print("\nExamples:")
    print(
        ambiguous_family
        .sort_values("Rows", ascending=False)
        .head(20)
        .to_string(index=False)
    )


# ==================================================================================================
# 10.6I — BUILD NETWORK MANIFESTATION KEYS
# ==================================================================================================

network_cols = [
    COL["Source_IP"],
    COL["Source_Port"],
    COL["Destination_IP"],
    COL["Destination_Port"],
    COL["Protocol"]
]

for c in network_cols:
    gt_work[c] = clean_text(gt_work[c])


gt_work["Strict_5Tuple"] = (
    gt_work[COL["Source_IP"]].fillna("<NA>") + "|" +
    gt_work[COL["Source_Port"]].fillna("<NA>") + "|" +
    gt_work[COL["Destination_IP"]].fillna("<NA>") + "|" +
    gt_work[COL["Destination_Port"]].fillna("<NA>") + "|" +
    gt_work[COL["Protocol"]].fillna("<NA>")
)

gt_work["HostPair_Protocol"] = (
    gt_work[COL["Source_IP"]].fillna("<NA>") + "|" +
    gt_work[COL["Destination_IP"]].fillna("<NA>") + "|" +
    gt_work[COL["Protocol"]].fillna("<NA>")
)


print("\n" + "=" * 120)
print("10.6I — NETWORK MANIFESTATION STRUCTURE")
print("=" * 120)

print(
    "Unique strict 5-tuples     :",
    f"{gt_work['Strict_5Tuple'].nunique():,}"
)

print(
    "Unique host-pair/protocols :",
    f"{gt_work['HostPair_Protocol'].nunique():,}"
)


# ==================================================================================================
# 10.6J — EXACT DUPLICATION AUDIT
# ==================================================================================================

# Compare the ORIGINAL 12 GT fields.
original_cols = list(gt.columns)

exact_dup_mask = gt_work.duplicated(
    subset=original_cols,
    keep=False
)

exact_dup_rows = gt_work.loc[
    exact_dup_mask,
    original_cols
].copy()

if len(exact_dup_rows):

    exact_dup_groups = (
        exact_dup_rows
        .groupby(original_cols, dropna=False)
        .size()
        .reset_index(name="Duplicate_Count")
        .sort_values(
            "Duplicate_Count",
            ascending=False
        )
    )

else:

    exact_dup_groups = pd.DataFrame()


print("\n" + "=" * 120)
print("10.6J — EXACT DUPLICATION")
print("=" * 120)

print(
    "Rows participating in exact duplicate groups:",
    f"{exact_dup_mask.sum():,}"
)

print(
    "Exact duplicate groups:",
    f"{len(exact_dup_groups):,}"
)

if len(exact_dup_groups):

    show_cols = [
        COL["Start_Time"],
        COL["Last_Time"],
        COL["Attack_Category"],
        COL["Attack_Subcategory"],
        COL["Protocol"],
        COL["Source_IP"],
        COL["Source_Port"],
        COL["Destination_IP"],
        COL["Destination_Port"],
        COL["Attack_Name"],
        "Duplicate_Count"
    ]

    print("\nLargest exact duplicate groups:")

    print(
        exact_dup_groups[show_cols]
        .head(20)
        .to_string(index=False)
    )


# ==================================================================================================
# 10.6K — SAME ATTACK IDENTITY + EXACT SAME TIME INTERVAL
# ==================================================================================================

interval_cols = [
    "Capture_ID",
    "Attack_Identity",
    "Start_DT",
    "End_DT"
]

identity_time = (
    gt_work
    .groupby(interval_cols, dropna=False)
    .agg(
        GT_Rows=("Strict_5Tuple", "size"),
        Strict_5Tuples=("Strict_5Tuple", "nunique"),
        HostPair_Protocols=("HostPair_Protocol", "nunique"),
        Source_IPs=(COL["Source_IP"], "nunique"),
        Destination_IPs=(COL["Destination_IP"], "nunique"),
        Protocols=(COL["Protocol"], "nunique"),
        Families=("Attack_Family_Norm", "nunique"),
        Subcategories=("Attack_Subcategory_Norm", "nunique")
    )
    .reset_index()
)

identity_time["Repeated"] = (
    identity_time["GT_Rows"] > 1
)

identity_time["SameTuple_Repetition"] = (
    (identity_time["GT_Rows"] > 1) &
    (identity_time["Strict_5Tuples"] == 1)
)

identity_time["Multi_Manifestation"] = (
    identity_time["Strict_5Tuples"] > 1
)


print("\n" + "=" * 120)
print("10.6K — IDENTITY + EXACT-TIME INTERVAL SEMANTICS")
print("=" * 120)

print(
    "Total identity/time groups       :",
    f"{len(identity_time):,}"
)

print(
    "Repeated identity/time groups    :",
    f"{identity_time['Repeated'].sum():,}"
)

print(
    "Same-tuple repeated groups       :",
    f"{identity_time['SameTuple_Repetition'].sum():,}"
)

print(
    "Multi-manifestation groups       :",
    f"{identity_time['Multi_Manifestation'].sum():,}"
)

print("\nStrict 5-tuples per identity/time group:")

print(
    identity_time["Strict_5Tuples"]
    .describe(
        percentiles=[
            .5, .75, .9, .95, .99, .999
        ]
    )
)


# ==================================================================================================
# 10.6L — LARGEST MULTI-MANIFESTATION EVENTS
# ==================================================================================================

largest_manifestations = (
    identity_time[
        identity_time["Multi_Manifestation"]
    ]
    .sort_values(
        ["Strict_5Tuples", "GT_Rows"],
        ascending=False
    )
)


print("\n" + "=" * 120)
print("10.6L — LARGEST SAME-IDENTITY/SAME-TIME MULTI-MANIFESTATIONS")
print("=" * 120)

if len(largest_manifestations):

    print(
        largest_manifestations
        .head(30)
        .to_string(index=False)
    )

else:
    print("None found.")


# ==================================================================================================
# 10.6M — ATTACK IDENTITY ACROSS NETWORK MANIFESTATIONS
# ==================================================================================================

identity_structure = (
    gt_work
    .groupby(
        ["Capture_ID", "Attack_Identity"],
        dropna=False
    )
    .agg(
        GT_Rows=("Strict_5Tuple", "size"),
        Family_Count=("Attack_Family_Norm", "nunique"),
        Subcategory_Count=("Attack_Subcategory_Norm", "nunique"),
        First_Start=("Start_DT", "min"),
        Last_End=("End_DT", "max"),
        Unique_Starts=("Start_DT", "nunique"),
        Unique_Time_Intervals=("End_DT", "nunique"),
        Strict_5Tuples=("Strict_5Tuple", "nunique"),
        HostPair_Protocols=("HostPair_Protocol", "nunique"),
        Source_IPs=(COL["Source_IP"], "nunique"),
        Destination_IPs=(COL["Destination_IP"], "nunique"),
        Protocols=(COL["Protocol"], "nunique")
    )
    .reset_index()
)

identity_structure["Observed_Span_Sec"] = (
    identity_structure["Last_End"] -
    identity_structure["First_Start"]
).dt.total_seconds()


print("\n" + "=" * 120)
print("10.6M — ATTACK IDENTITY STRUCTURE")
print("=" * 120)

print(
    "Identity/capture groups:",
    f"{len(identity_structure):,}"
)

multi_host_identity = (
    identity_structure["HostPair_Protocols"] > 1
)

print(
    "Identity/capture groups spanning >1 host-pair/protocol:",
    f"{multi_host_identity.sum():,}"
)

if len(identity_structure):

    print(
        "Percentage:",
        f"{100 * multi_host_identity.mean():.2f}%"
    )

print("\nLargest network-spanning identities:")

print(
    identity_structure
    .sort_values(
        ["HostPair_Protocols", "Strict_5Tuples"],
        ascending=False
    )
    .head(30)
    .to_string(index=False)
)


# ==================================================================================================
# 10.6N — TEMPORAL GAP ANALYSIS WITHIN SAME ATTACK IDENTITY
# ==================================================================================================

gap_records = []

for (capture, identity), g in gt_work.groupby(
    ["Capture_ID", "Attack_Identity"],
    dropna=False
):

    intervals = (
        g[["Start_DT", "End_DT"]]
        .dropna()
        .drop_duplicates()
        .sort_values(
            ["Start_DT", "End_DT"]
        )
        .reset_index(drop=True)
    )

    if len(intervals) < 2:
        continue

    # Running maximum prevents overlapping intervals from producing
    # misleading positive gaps.
    previous_running_end = (
        intervals["End_DT"]
        .cummax()
        .shift(1)
    )

    gaps = (
        intervals["Start_DT"] -
        previous_running_end
    ).dt.total_seconds()

    for gap in gaps.dropna():

        gap_records.append({
            "Capture_ID": capture,
            "Attack_Identity": identity,
            "Gap_Sec": gap
        })


gap_df = pd.DataFrame(gap_records)


print("\n" + "=" * 120)
print("10.6N — WITHIN-IDENTITY TEMPORAL GAPS")
print("=" * 120)

if len(gap_df):

    print(
        "Gap observations:",
        f"{len(gap_df):,}"
    )

    print("\nAll gaps:")
    print(
        gap_df["Gap_Sec"]
        .describe(
            percentiles=[
                .25, .5, .75,
                .9, .95, .99, .999
            ]
        )
    )

    positive_gaps = gap_df[
        gap_df["Gap_Sec"] > 0
    ]["Gap_Sec"]

    print(
        "\nPositive gaps:",
        f"{len(positive_gaps):,}"
    )

    if len(positive_gaps):

        print("\nPositive-gap distribution:")

        print(
            positive_gaps.describe(
                percentiles=[
                    .25, .5, .75,
                    .9, .95, .99, .999
                ]
            )
        )

        print("\nThreshold counts:")

        for threshold in [
            1, 2, 5, 10,
            30, 60, 120,
            300, 600, 1800
        ]:

            count = int(
                (positive_gaps > threshold)
                .sum()
            )

            print(
                f"Gap > {threshold:4d}s : "
                f"{count:,}"
            )

else:

    print("No gap observations.")


# ==================================================================================================
# 10.6O — FAMILY-SPECIFIC MANIFESTATION STRUCTURE
# ==================================================================================================

family_structure = (
    gt_work
    .groupby(
        "Attack_Family_Raw",
        dropna=False
    )
    .agg(
        GT_Rows=("Strict_5Tuple", "size"),
        Attack_Names=("Attack_Identity", "nunique"),
        Subcategories=("Attack_Subcategory_Norm", "nunique"),
        Strict_5Tuples=("Strict_5Tuple", "nunique"),
        HostPair_Protocols=("HostPair_Protocol", "nunique"),
        Unique_Starts=("Start_DT", "nunique")
    )
    .reset_index()
)


print("\n" + "=" * 120)
print("10.6O — FAMILY-SPECIFIC STRUCTURE")
print("=" * 120)

print(
    family_structure
    .sort_values(
        "GT_Rows",
        ascending=False
    )
    .to_string(index=False)
)


# ==================================================================================================
# 10.6P — CROSS-CAPTURE KNOWN-TYPE SUPPORT
# ==================================================================================================

family_capture = pd.crosstab(
    gt_work["Attack_Family_Raw"],
    gt_work["Capture_ID"]
)

identity_capture = (
    gt_work
    .groupby("Attack_Identity", dropna=False)
    ["Capture_ID"]
    .nunique()
    .rename("Capture_Count")
    .reset_index()
)


print("\n" + "=" * 120)
print("10.6P — CROSS-CAPTURE KNOWN-TYPE SUPPORT")
print("=" * 120)

print("\nFamily × capture:")
print(family_capture)

both_identity = (
    identity_capture["Capture_Count"] >= 2
)

print(
    "\nFine attack identities in BOTH captures:",
    f"{both_identity.sum():,}",
    "/",
    f"{len(identity_capture):,}"
)

print(
    "Percentage:",
    f"{100 * both_identity.mean():.2f}%"
)


# ==================================================================================================
# 10.6Q — CANDIDATE SEMANTIC HIERARCHY
# ==================================================================================================

# These are diagnostic keys, NOT final event IDs.

gt_work["H1_Family"] = (
    gt_work["Attack_Family_Norm"]
)

gt_work["H2_Identity_Capture"] = (
    gt_work["Capture_ID"].astype("string")
    + "||" +
    gt_work["Attack_Identity"].astype("string")
)

gt_work["H3_Identity_ExactTime"] = (
    gt_work["H2_Identity_Capture"]
    + "||" +
    gt_work["Start_DT"].astype("string")
    + "||" +
    gt_work["End_DT"].astype("string")
)

gt_work["H4_Network_Manifestation"] = (
    gt_work["H3_Identity_ExactTime"]
    + "||" +
    gt_work["Strict_5Tuple"]
)


hierarchy = pd.DataFrame({

    "Candidate_Level": [
        "Broad attack family",
        "Fine attack identity within capture",
        "Fine identity + exact time interval",
        "Identity/time + strict 5-tuple manifestation"
    ],

    "Unique_Objects": [
        gt_work["H1_Family"].nunique(dropna=True),
        gt_work["H2_Identity_Capture"].nunique(dropna=True),
        gt_work["H3_Identity_ExactTime"].nunique(dropna=True),
        gt_work["H4_Network_Manifestation"].nunique(dropna=True)
    ]

})


print("\n" + "=" * 120)
print("10.6Q — CANDIDATE EVENT HIERARCHY")
print("=" * 120)

print(
    hierarchy.to_string(index=False)
)


# ==================================================================================================
# 10.6R — POTENTIAL LEAKAGE RISK
# ==================================================================================================

leakage_candidates = identity_time[
    identity_time["Strict_5Tuples"] > 1
].copy()


print("\n" + "=" * 120)
print("10.6R — POTENTIAL EVENT-LEVEL LEAKAGE RISK")
print("=" * 120)

print(
    "Same identity/time groups containing >1 network manifestation:",
    f"{len(leakage_candidates):,}"
)

print(
    "GT rows represented inside those groups:",
    f"{int(leakage_candidates['GT_Rows'].sum()):,}"
)

print("""
WHY THIS MATTERS:

If different 5-tuples belonging to the same underlying attack occurrence
are placed into different TRAIN / VALIDATION / TEST splits, we may create
hidden event leakage.

Therefore we will NOT use the 5-tuple itself as the final split unit unless
the evidence supports that interpretation.
""")


# ==================================================================================================
# 10.6S — SAVE AUDIT OUTPUTS
# ==================================================================================================

save_tables = {

    "10_6_capture_summary.csv":
        capture_summary,

    "10_6_identity_family_consistency.csv":
        identity_family,

    "10_6_exact_duplicate_groups.csv":
        exact_dup_groups,

    "10_6_identity_exact_time_structure.csv":
        identity_time,

    "10_6_largest_multi_manifestations.csv":
        largest_manifestations,

    "10_6_identity_structure.csv":
        identity_structure,

    "10_6_temporal_gaps.csv":
        gap_df,

    "10_6_family_structure.csv":
        family_structure,

    "10_6_identity_cross_capture_support.csv":
        identity_capture,

    "10_6_candidate_hierarchy.csv":
        hierarchy,

    "10_6_potential_leakage_groups.csv":
        leakage_candidates
}


print("\n" + "=" * 120)
print("10.6S — SAVING DIAGNOSTIC OUTPUTS")
print("=" * 120)

for filename, table in save_tables.items():

    path = UNSW_OUT / filename

    table.to_csv(
        path,
        index=False
    )

    print("✓", filename)


# ==================================================================================================
# 10.6T — DECISION GATE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.6T — DECISION GATE")
print("=" * 120)

print("""
DO NOT CREATE FINAL EVENT IDs YET.

PHASE 10.6 HAS NOW TESTED THE ACTUAL 12-COLUMN UNSW GT EVENT TABLE.

WE NEED TO REVIEW:

A. EXACT DUPLICATES
   Are duplicate GT records merely repeated annotation?

B. ATTACK IDENTITY
   The GT's Attack Name gives us a fine attack identity.
   We must determine whether repeated instances of the same name represent
   one campaign or multiple physical occurrences.

C. NETWORK MANIFESTATIONS
   If one attack identity/time interval contains many 5-tuples, those may be
   parallel manifestations of one attack event.

D. TEMPORAL SEPARATION
   Positive gaps within the same attack identity tell us whether temporal
   clustering can define independent occurrences.

E. SPLIT ATOMICITY
   The final independent attack occurrence must stay entirely within one
   TRAIN / VALIDATION / TEST partition.

F. KNOWN-TYPE FORECASTING
   Validation/test attack families must have training support, but their
   physical occurrences must remain unseen.

G. MULTI-ATTACK WINDOWS
   Simultaneous families must remain representable. We will not arbitrarily
   collapse them into one label.

NO MODELING TARGETS HAVE BEEN CREATED.
NO SPLITS HAVE BEEN CREATED.
NO FINAL EVENT DEFINITION HAS BEEN FROZEN.
""")

print("=" * 120)
print("PHASE 10.6 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.7 — UNSW PHYSICAL ATTACK OCCURRENCE BOUNDARY AUDIT
# OPTIMIZED / VECTORIZED VERSION
# ==================================================================================================
#
# PURPOSE
# -------
# Determine how repeated UNSW GT records should be grouped into independent physical attack
# occurrences WITHOUT yet committing to a final Episode_ID definition.
#
# IMPORTANT:
#   - No train/validation/test split is created here.
#   - No forecasting targets are created here.
#   - No final event definition is frozen here.
#   - Raw files are never modified.
#
# OPTIMIZATION
# ------------
# The previous implementation repeatedly:
#   * iterated through thousands of groups
#   * used iterrows()
#   * rebuilt events independently for every threshold
#   * used groupby.apply(lambda ... drop_duplicates())
#
# This implementation preserves the same conceptual rule:
#
#       new event iff
#       Start_Time - maximum End_Time reached by all previous rows
#       in the same (Capture_ID, Attack_Identity) group
#       > threshold
#
# But calculates the temporal structure ONCE using vectorized pandas operations.
#
# ==================================================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import time

T0 = time.time()

print("=" * 120)
print("PHASE 10.7 — UNSW PHYSICAL ATTACK OCCURRENCE BOUNDARY AUDIT — OPTIMIZED")
print("=" * 120)


# ==================================================================================================
# 10.7A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")
AUDIT_ROOT   = PROJECT_ROOT / "Dataset_Audit_v2"
UNSW_RAW     = AUDIT_ROOT / "raw" / "UNSW_NB15"
UNSW_OUT     = AUDIT_ROOT / "audit_outputs" / "UNSW"

UNSW_OUT.mkdir(parents=True, exist_ok=True)

GT_FILE = UNSW_RAW / "NUSW-NB15_GT.csv"

if not GT_FILE.exists():
    raise FileNotFoundError(f"GT file not found: {GT_FILE}")

print("\n✓ GT file found")
print(GT_FILE)


# ==================================================================================================
# 10.7B — ROBUST GT LOAD
# ==================================================================================================

def read_csv_robust(path, **kwargs):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs), enc
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


gt, gt_encoding = read_csv_robust(GT_FILE)
gt.columns = gt.columns.astype(str).str.strip()

print("\n" + "=" * 120)
print("10.7B — GT LOAD CHECK")
print("=" * 120)

print("Shape   :", gt.shape)
print("Encoding:", gt_encoding)
print("Columns :", list(gt.columns))


# ==================================================================================================
# 10.7C — REQUIRED SEMANTIC FIELDS
# ==================================================================================================

required = [
    "Start time",
    "Last time",
    "Attack category",
    "Attack subcategory",
    "Protocol",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Attack Name",
]

missing = [c for c in required if c not in gt.columns]

if missing:
    raise ValueError(f"Missing required GT columns: {missing}")

print("\n✓ Required semantic GT columns present")


# ==================================================================================================
# 10.7D — NORMALIZE WORKING REPRESENTATION
# ==================================================================================================

work = gt.dropna(how="all").copy()

work["Start_Epoch"] = pd.to_numeric(work["Start time"], errors="coerce")
work["End_Epoch"]   = pd.to_numeric(work["Last time"], errors="coerce")

work["Start_DT"] = pd.to_datetime(
    work["Start_Epoch"],
    unit="s",
    errors="coerce"
)

work["End_DT"] = pd.to_datetime(
    work["End_Epoch"],
    unit="s",
    errors="coerce"
)

text_columns = [
    "Attack category",
    "Attack subcategory",
    "Protocol",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Attack Name",
]

for col in text_columns:
    work[col] = work[col].astype("string").str.strip()

usable_mask = (
    work["Start_DT"].notna()
    & work["End_DT"].notna()
    & work["Attack Name"].notna()
)

work = work.loc[usable_mask].copy()

work["Attack_Identity"] = work["Attack Name"]
work["Attack_Family_Raw"] = work["Attack category"]

# Preserve raw family but normalize known spelling discrepancy separately.
work["Attack_Family_Normalized"] = (
    work["Attack_Family_Raw"]
    .replace({"Backdoors": "Backdoor"})
)

work["Strict_5Tuple"] = (
    work["Source IP"].fillna("<NA>") + "|" +
    work["Source Port"].fillna("<NA>") + "|" +
    work["Destination IP"].fillna("<NA>") + "|" +
    work["Destination Port"].fillna("<NA>") + "|" +
    work["Protocol"].fillna("<NA>")
)

work["Host_Pair"] = (
    work["Source IP"].fillna("<NA>") + "|" +
    work["Destination IP"].fillna("<NA>")
)

work["HostPair_Protocol"] = (
    work["Source IP"].fillna("<NA>") + "|" +
    work["Destination IP"].fillna("<NA>") + "|" +
    work["Protocol"].fillna("<NA>")
)

# Exact interval key allows vectorized nunique() later.
work["Interval_Key"] = (
    work["Start_Epoch"].astype("Int64").astype("string")
    + "|"
    + work["End_Epoch"].astype("Int64").astype("string")
)

print(f"\n✓ Usable semantic GT rows: {len(work):,}")


# ==================================================================================================
# 10.7E — RECONSTRUCT CAPTURE ID
# ==================================================================================================
#
# Preserve the original logic: infer the two physical captures from the largest
# positive temporal separation in the GT.
#
# ==================================================================================================

sorted_global = work.sort_values(
    ["Start_DT", "End_DT"],
    kind="mergesort"
).copy()

global_start_gap = (
    sorted_global["Start_DT"]
    - sorted_global["End_DT"].shift(1)
).dt.total_seconds()

positive_global = global_start_gap[global_start_gap > 0]

if positive_global.empty:
    raise ValueError(
        "Could not identify temporal separation between UNSW captures."
    )

largest_gap_index = positive_global.idxmax()
largest_gap_sec   = positive_global.loc[largest_gap_index]
capture2_start    = sorted_global.loc[largest_gap_index, "Start_DT"]

work["Capture_ID"] = np.where(
    work["Start_DT"] < capture2_start,
    "UNSW_CAPTURE_1",
    "UNSW_CAPTURE_2"
)

print("\n" + "=" * 120)
print("10.7E — CAPTURE RECONSTRUCTION")
print("=" * 120)

print(f"Largest global positive separation : {largest_gap_sec:,.0f} sec")
print(f"Capture-2 inferred start           : {capture2_start}")

capture_check = (
    work.groupby("Capture_ID", observed=True)
    .agg(
        Rows=("Attack_Identity", "size"),
        Start=("Start_DT", "min"),
        End=("End_DT", "max"),
        Attack_Identities=("Attack_Identity", "nunique"),
        Families=("Attack_Family_Normalized", "nunique"),
    )
    .reset_index()
)

print()
print(capture_check.to_string(index=False))


# ==================================================================================================
# 10.7F — BUILD TEMPORAL STRUCTURE ONCE
# ==================================================================================================
#
# THIS IS THE IMPORTANT OPTIMIZATION.
#
# Within each:
#
#       (Capture_ID, Attack_Identity)
#
# rows are chronologically sorted.
#
# For each row we calculate:
#
#       Previous_Running_Max_End
#
# = maximum End_DT reached by ANY earlier interval in the group.
#
# Therefore nested/overlapping intervals cannot create false temporal boundaries.
#
# Example:
#
# A  10:00 ---------------- 10:30
# B       10:10 -- 10:20
# C                         10:31 ----
#
# C is 1 second after the temporal component, NOT 11 seconds after B.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.7F — VECTORIZED TEMPORAL COMPONENT STRUCTURE")
print("=" * 120)

sort_cols = [
    "Capture_ID",
    "Attack_Identity",
    "Start_DT",
    "End_DT",
    "Strict_5Tuple",
]

ordered = (
    work.sort_values(sort_cols, kind="mergesort")
    .reset_index(drop=True)
)

group_cols = ["Capture_ID", "Attack_Identity"]

# Running maximum including current row.
ordered["Running_Max_End"] = (
    ordered.groupby(
        group_cols,
        sort=False,
        observed=True
    )["End_DT"]
    .cummax()
)

# Previous row's running maximum = maximum end reached by ALL prior intervals.
ordered["Previous_Running_Max_End"] = (
    ordered.groupby(
        group_cols,
        sort=False,
        observed=True
    )["Running_Max_End"]
    .shift(1)
)

ordered["Component_Gap_Sec"] = (
    ordered["Start_DT"]
    - ordered["Previous_Running_Max_End"]
).dt.total_seconds()

# First row of each identity/capture has no previous component.
first_in_group = ordered["Previous_Running_Max_End"].isna()

print(f"Rows processed                  : {len(ordered):,}")
print(f"Attack identity/capture groups : {ordered.groupby(group_cols).ngroups:,}")
print(
    f"Positive temporal gaps         : "
    f"{int((ordered['Component_Gap_Sec'] > 0).sum()):,}"
)

print("✓ Temporal structure calculated ONCE")


# ==================================================================================================
# 10.7G — CANDIDATE GAP THRESHOLDS
# ==================================================================================================

THRESHOLDS = [
    0,
    1,
    2,
    5,
    10,
    30,
    60,
    120,
    300,
    600,
    1800,
    3600,
    5400,
    7200,
]

print("\n" + "=" * 120)
print("10.7G — CANDIDATE TEMPORAL THRESHOLDS")
print("=" * 120)

print(THRESHOLDS)


# ==================================================================================================
# 10.7H — BUILD CANDIDATE EVENT TABLES
# ==================================================================================================
#
# We DO NOT recompute temporal components.
#
# For threshold T:
#
#     New_Event = first row in identity/capture
#                 OR Component_Gap_Sec > T
#
# Candidate_Local_ID is then simply cumulative sum of New_Event within group.
#
# ==================================================================================================

candidate_summaries = []
candidate_event_tables = {}

for threshold in THRESHOLDS:

    print(f"  Processing threshold {threshold:>4}s ...", end=" ")

    temp = ordered.copy()

    temp["New_Event"] = (
        first_in_group
        | (temp["Component_Gap_Sec"] > threshold)
    )

    temp["Candidate_Local_ID"] = (
        temp["New_Event"]
        .astype(np.int32)
        .groupby(
            [temp["Capture_ID"], temp["Attack_Identity"]],
            sort=False
        )
        .cumsum()
        .astype(np.int32)
    )

    event_keys = [
        "Capture_ID",
        "Attack_Identity",
        "Candidate_Local_ID",
    ]

    # Pure vectorized aggregation.
    event_table = (
        temp.groupby(
            event_keys,
            sort=False,
            observed=True
        )
        .agg(
            Event_Start=("Start_DT", "min"),
            Event_End=("End_DT", "max"),
            GT_Rows=("Attack_Identity", "size"),

            Families=("Attack_Family_Normalized", "nunique"),
            Raw_Families=("Attack_Family_Raw", "nunique"),
            Subcategories=("Attack subcategory", "nunique"),

            Strict_5Tuples=("Strict_5Tuple", "nunique"),
            Host_Pairs=("Host_Pair", "nunique"),
            HostPair_Protocols=("HostPair_Protocol", "nunique"),

            Source_IPs=("Source IP", "nunique"),
            Destination_IPs=("Destination IP", "nunique"),
            Protocols=("Protocol", "nunique"),

            Unique_Starts=("Start_DT", "nunique"),
            Unique_Intervals=("Interval_Key", "nunique"),
        )
        .reset_index()
    )

    event_table["Candidate_Event_ID"] = (
        event_table["Capture_ID"].astype(str)
        + "||"
        + event_table["Attack_Identity"].astype(str)
        + "||T"
        + str(threshold)
        + "||E"
        + event_table["Candidate_Local_ID"].astype(str)
    )

    event_table["Duration_Sec"] = (
        event_table["Event_End"]
        - event_table["Event_Start"]
    ).dt.total_seconds()

    candidate_event_tables[threshold] = event_table

    candidate_summaries.append({
        "Threshold_Sec": threshold,
        "Candidate_Events": len(event_table),

        "Median_Duration_Sec":
            event_table["Duration_Sec"].median(),

        "P90_Duration_Sec":
            event_table["Duration_Sec"].quantile(0.90),

        "P95_Duration_Sec":
            event_table["Duration_Sec"].quantile(0.95),

        "P99_Duration_Sec":
            event_table["Duration_Sec"].quantile(0.99),

        "Max_Duration_Sec":
            event_table["Duration_Sec"].max(),

        "Median_GT_Rows":
            event_table["GT_Rows"].median(),

        "P95_GT_Rows":
            event_table["GT_Rows"].quantile(0.95),

        "Max_GT_Rows":
            event_table["GT_Rows"].max(),

        "Multi_5Tuple_Events":
            int((event_table["Strict_5Tuples"] > 1).sum()),

        "Multi_HostPair_Events":
            int((event_table["Host_Pairs"] > 1).sum()),

        "Multi_Protocol_Events":
            int((event_table["Protocols"] > 1).sum()),

        "Multi_Family_Events":
            int((event_table["Families"] > 1).sum()),
    })

    print(f"{len(event_table):,} candidate events")


candidate_summary = pd.DataFrame(candidate_summaries)

print("\n" + "=" * 120)
print("10.7H — CANDIDATE OCCURRENCE SENSITIVITY")
print("=" * 120)

print(candidate_summary.to_string(index=False))


# ==================================================================================================
# 10.7I — EVENT COUNT COLLAPSE BETWEEN THRESHOLDS
# ==================================================================================================

collapse = candidate_summary[
    ["Threshold_Sec", "Candidate_Events"]
].copy()

collapse["Events_Merged_vs_Previous"] = (
    collapse["Candidate_Events"].shift(1)
    - collapse["Candidate_Events"]
)

collapse["Pct_Reduction_vs_Previous"] = (
    collapse["Events_Merged_vs_Previous"]
    / collapse["Candidate_Events"].shift(1)
    * 100
)

print("\n" + "=" * 120)
print("10.7I — THRESHOLD COLLAPSE PROFILE")
print("=" * 120)

print(collapse.to_string(index=False))


# ==================================================================================================
# 10.7J — RAW TEMPORAL COMPONENT GAP DISTRIBUTION
# ==================================================================================================
#
# Because Component_Gap_Sec was already calculated correctly using the previous
# running maximum end, we no longer need another Python loop.
#
# ==================================================================================================

component_gaps = ordered.loc[
    ordered["Component_Gap_Sec"] > 0,
    [
        "Capture_ID",
        "Attack_Identity",
        "Previous_Running_Max_End",
        "Start_DT",
        "Component_Gap_Sec",
        "Source IP",
        "Destination IP",
        "Protocol",
        "Host_Pair",
        "Strict_5Tuple",
    ]
].copy()

component_gaps = component_gaps.rename(columns={
    "Previous_Running_Max_End": "Previous_Component_End",
    "Start_DT": "Next_Start",
    "Component_Gap_Sec": "Gap_Sec",
    "Source IP": "Next_Source_IP",
    "Destination IP": "Next_Destination_IP",
    "Protocol": "Next_Protocol",
})

print("\n" + "=" * 120)
print("10.7J — TEMPORAL COMPONENT GAP DISTRIBUTION")
print("=" * 120)

if component_gaps.empty:

    print("No positive temporal component gaps found.")
    gap_band_table = pd.DataFrame()

else:

    print("Positive component gaps:", f"{len(component_gaps):,}")

    print("\nDistribution:")

    print(
        component_gaps["Gap_Sec"].describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
                0.999,
            ]
        )
    )

    gap_bands = [
        (0, 1),
        (1, 2),
        (2, 5),
        (5, 10),
        (10, 30),
        (30, 60),
        (60, 120),
        (120, 300),
        (300, 600),
        (600, 1800),
        (1800, 3600),
        (3600, 5400),
        (5400, 7200),
        (7200, np.inf),
    ]

    band_rows = []

    for low, high in gap_bands:

        if np.isinf(high):
            mask = component_gaps["Gap_Sec"] > low
            label = f">{low}s"
        else:
            mask = (
                (component_gaps["Gap_Sec"] > low)
                & (component_gaps["Gap_Sec"] <= high)
            )
            label = f"({low}, {high}] sec"

        count = int(mask.sum())

        band_rows.append({
            "Gap_Band": label,
            "Count": count,
            "Percent":
                100 * count / len(component_gaps)
                if len(component_gaps)
                else np.nan,
        })

    gap_band_table = pd.DataFrame(band_rows)

    print("\nGap bands:")
    print(gap_band_table.to_string(index=False))


# ==================================================================================================
# 10.7K — NETWORK CONTINUITY ACROSS TEMPORAL COMPONENT BOUNDARIES
# ==================================================================================================
#
# We need to compare the first manifestation after a positive gap against ALL
# network identities in the preceding zero-gap/overlapping temporal component.
#
# First construct the threshold-0 temporal components.
#
# ==================================================================================================

base = ordered.copy()

base["Base_New_Component"] = (
    first_in_group
    | (base["Component_Gap_Sec"] > 0)
)

base["Base_Component_ID"] = (
    base["Base_New_Component"]
    .astype(np.int32)
    .groupby(
        [base["Capture_ID"], base["Attack_Identity"]],
        sort=False
    )
    .cumsum()
    .astype(np.int32)
)

base_keys = [
    "Capture_ID",
    "Attack_Identity",
    "Base_Component_ID",
]

# Aggregate network identities into sets ONCE.
component_network = (
    base.groupby(
        base_keys,
        sort=False,
        observed=True
    )
    .agg(
        Component_Start=("Start_DT", "min"),
        Component_End=("End_DT", "max"),

        Sources=("Source IP", lambda x: frozenset(x.dropna())),
        Destinations=("Destination IP", lambda x: frozenset(x.dropna())),
        Protocol_Set=("Protocol", lambda x: frozenset(x.dropna())),
        HostPair_Set=("Host_Pair", lambda x: frozenset(x.dropna())),
        Tuple_Set=("Strict_5Tuple", lambda x: frozenset(x.dropna())),
    )
    .reset_index()
)

# First row of every base component.
component_first_rows = (
    base.sort_values(
        base_keys + ["Start_DT", "End_DT", "Strict_5Tuple"],
        kind="mergesort"
    )
    .drop_duplicates(base_keys, keep="first")
    [
        base_keys
        + [
            "Start_DT",
            "Source IP",
            "Destination IP",
            "Protocol",
            "Host_Pair",
            "Strict_5Tuple",
        ]
    ]
    .rename(columns={
        "Start_DT": "Next_Start",
        "Source IP": "Next_Source_IP",
        "Destination IP": "Next_Destination_IP",
        "Protocol": "Next_Protocol",
        "Host_Pair": "Next_Host_Pair",
        "Strict_5Tuple": "Next_Strict_5Tuple",
    })
)

components = component_network.merge(
    component_first_rows,
    on=base_keys,
    how="left",
    validate="one_to_one"
)

components = components.sort_values(
    base_keys,
    kind="mergesort"
).reset_index(drop=True)

# Previous component information within same capture + attack identity.
cg = components.groupby(
    ["Capture_ID", "Attack_Identity"],
    sort=False,
    observed=True
)

components["Previous_Component_End"] = cg["Component_End"].shift(1)
components["Previous_Sources"]       = cg["Sources"].shift(1)
components["Previous_Destinations"]  = cg["Destinations"].shift(1)
components["Previous_Protocols"]     = cg["Protocol_Set"].shift(1)
components["Previous_HostPairs"]     = cg["HostPair_Set"].shift(1)
components["Previous_Tuples"]        = cg["Tuple_Set"].shift(1)

boundaries = components[
    components["Previous_Component_End"].notna()
].copy()

boundaries["Gap_Sec"] = (
    boundaries["Next_Start"]
    - boundaries["Previous_Component_End"]
).dt.total_seconds()

# Membership checks only happen once per actual temporal boundary,
# not once per GT row and not once per threshold.
boundaries["Same_Source"] = [
    value in previous
    for value, previous in zip(
        boundaries["Next_Source_IP"],
        boundaries["Previous_Sources"]
    )
]

boundaries["Same_Destination"] = [
    value in previous
    for value, previous in zip(
        boundaries["Next_Destination_IP"],
        boundaries["Previous_Destinations"]
    )
]

boundaries["Same_Protocol"] = [
    value in previous
    for value, previous in zip(
        boundaries["Next_Protocol"],
        boundaries["Previous_Protocols"]
    )
]

boundaries["Same_HostPair"] = [
    value in previous
    for value, previous in zip(
        boundaries["Next_Host_Pair"],
        boundaries["Previous_HostPairs"]
    )
]

boundaries["Same_Strict_5Tuple"] = [
    value in previous
    for value, previous in zip(
        boundaries["Next_Strict_5Tuple"],
        boundaries["Previous_Tuples"]
    )
]

boundary_output_cols = [
    "Capture_ID",
    "Attack_Identity",
    "Gap_Sec",
    "Previous_Component_End",
    "Next_Start",
    "Same_Source",
    "Same_Destination",
    "Same_Protocol",
    "Same_HostPair",
    "Same_Strict_5Tuple",
]

boundaries_output = boundaries[boundary_output_cols].copy()

print("\n" + "=" * 120)
print("10.7K — NETWORK CONTINUITY ACROSS POSITIVE TEMPORAL GAPS")
print("=" * 120)

continuity_rows = []

for threshold in THRESHOLDS[1:]:

    subset = boundaries_output[
        boundaries_output["Gap_Sec"] > threshold
    ]

    if subset.empty:
        continue

    continuity_rows.append({
        "Gap_Greater_Than_Sec": threshold,
        "Boundaries": len(subset),

        "Same_Source_Pct":
            100 * subset["Same_Source"].mean(),

        "Same_Destination_Pct":
            100 * subset["Same_Destination"].mean(),

        "Same_Protocol_Pct":
            100 * subset["Same_Protocol"].mean(),

        "Same_HostPair_Pct":
            100 * subset["Same_HostPair"].mean(),

        "Same_Strict_5Tuple_Pct":
            100 * subset["Same_Strict_5Tuple"].mean(),
    })

continuity_table = pd.DataFrame(continuity_rows)

if continuity_table.empty:
    print("No temporal boundaries available.")
else:
    print(continuity_table.to_string(index=False))


# ==================================================================================================
# 10.7L — CANDIDATE EVENT DURATION PATHOLOGY
# ==================================================================================================

duration_pathology_rows = []

for threshold, event_table in candidate_event_tables.items():

    duration_pathology_rows.append({
        "Threshold_Sec": threshold,
        "Events": len(event_table),

        "Duration_gt_60s":
            int((event_table["Duration_Sec"] > 60).sum()),

        "Duration_gt_5min":
            int((event_table["Duration_Sec"] > 300).sum()),

        "Duration_gt_15min":
            int((event_table["Duration_Sec"] > 900).sum()),

        "Duration_gt_30min":
            int((event_table["Duration_Sec"] > 1800).sum()),

        "Duration_gt_1hr":
            int((event_table["Duration_Sec"] > 3600).sum()),

        "Duration_gt_2hr":
            int((event_table["Duration_Sec"] > 7200).sum()),
    })

duration_pathology = pd.DataFrame(duration_pathology_rows)

print("\n" + "=" * 120)
print("10.7L — CANDIDATE EVENT DURATION PATHOLOGY")
print("=" * 120)

print(duration_pathology.to_string(index=False))


# ==================================================================================================
# 10.7M — NETWORK COMPLEXITY AS THRESHOLD INCREASES
# ==================================================================================================

complexity_rows = []

for threshold, event_table in candidate_event_tables.items():

    complexity_rows.append({
        "Threshold_Sec": threshold,

        "Median_5Tuples":
            event_table["Strict_5Tuples"].median(),

        "P95_5Tuples":
            event_table["Strict_5Tuples"].quantile(0.95),

        "Max_5Tuples":
            event_table["Strict_5Tuples"].max(),

        "Median_HostPairs":
            event_table["Host_Pairs"].median(),

        "P95_HostPairs":
            event_table["Host_Pairs"].quantile(0.95),

        "Max_HostPairs":
            event_table["Host_Pairs"].max(),

        "Median_Protocols":
            event_table["Protocols"].median(),

        "P95_Protocols":
            event_table["Protocols"].quantile(0.95),

        "Max_Protocols":
            event_table["Protocols"].max(),
    })

complexity_table = pd.DataFrame(complexity_rows)

print("\n" + "=" * 120)
print("10.7M — NETWORK COMPLEXITY SENSITIVITY")
print("=" * 120)

print(complexity_table.to_string(index=False))


# ==================================================================================================
# 10.7N — FAMILY CONSISTENCY
# ==================================================================================================

family_consistency_rows = []

for threshold, event_table in candidate_event_tables.items():

    family_consistency_rows.append({
        "Threshold_Sec": threshold,
        "Events": len(event_table),

        "Multi_Normalized_Family_Events":
            int((event_table["Families"] > 1).sum()),

        "Multi_Raw_Family_Events":
            int((event_table["Raw_Families"] > 1).sum()),

        "Pct_Multi_Normalized_Family":
            100 * (event_table["Families"] > 1).mean(),

        "Pct_Multi_Raw_Family":
            100 * (event_table["Raw_Families"] > 1).mean(),
    })

family_consistency = pd.DataFrame(family_consistency_rows)

print("\n" + "=" * 120)
print("10.7N — FAMILY CONSISTENCY UNDER CANDIDATE GROUPINGS")
print("=" * 120)

print(family_consistency.to_string(index=False))


# ==================================================================================================
# 10.7O — CROSS-CAPTURE IDENTITY SUPPORT
# ==================================================================================================
#
# Known-type forecasting:
#
# Attack TYPE may appear in multiple partitions.
# Physical EVENT must not leak across partitions.
#
# ==================================================================================================

identity_capture = (
    work.groupby(
        ["Attack_Identity", "Capture_ID"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
)

for col in ["UNSW_CAPTURE_1", "UNSW_CAPTURE_2"]:
    if col not in identity_capture.columns:
        identity_capture[col] = 0

identity_capture["In_Both"] = (
    (identity_capture["UNSW_CAPTURE_1"] > 0)
    & (identity_capture["UNSW_CAPTURE_2"] > 0)
)

print("\n" + "=" * 120)
print("10.7O — KNOWN-TYPE CROSS-CAPTURE SUPPORT")
print("=" * 120)

print(
    "Fine attack identities in both captures:",
    int(identity_capture["In_Both"].sum()),
    "/",
    len(identity_capture)
)

print(
    "Percentage:",
    round(100 * identity_capture["In_Both"].mean(), 2),
    "%"
)


# ==================================================================================================
# 10.7P — HIGH-FREQUENCY IDENTITIES: EVENT COUNT SENSITIVITY
# ==================================================================================================

top_identities = (
    work["Attack_Identity"]
    .value_counts()
    .head(20)
    .index
)

identity_base = (
    work[work["Attack_Identity"].isin(top_identities)]
    .groupby("Attack_Identity", observed=True)
    .agg(
        GT_Rows=("Attack_Identity", "size"),
        Captures=("Capture_ID", "nunique"),
        Strict_5Tuples=("Strict_5Tuple", "nunique"),
        Host_Pairs=("Host_Pair", "nunique"),
        First_Start=("Start_DT", "min"),
        Last_End=("End_DT", "max"),
    )
    .reindex(top_identities)
    .reset_index()
)

for threshold in THRESHOLDS:

    counts = (
        candidate_event_tables[threshold]
        ["Attack_Identity"]
        .value_counts()
    )

    identity_base[f"Events_T{threshold}"] = (
        identity_base["Attack_Identity"]
        .map(counts)
        .fillna(0)
        .astype(int)
    )

example_identity_table = identity_base

print("\n" + "=" * 120)
print("10.7P — HIGH-FREQUENCY IDENTITIES: EVENT COUNT SENSITIVITY")
print("=" * 120)

print(example_identity_table.to_string(index=False))


# ==================================================================================================
# 10.7Q — LARGEST THRESHOLD-INDUCED STRUCTURAL CHANGES
# ==================================================================================================

change_table = (
    collapse[
        collapse["Events_Merged_vs_Previous"].notna()
    ]
    .copy()
    .sort_values(
        "Events_Merged_vs_Previous",
        ascending=False
    )
)

print("\n" + "=" * 120)
print("10.7Q — LARGEST THRESHOLD-INDUCED STRUCTURAL CHANGES")
print("=" * 120)

print(change_table.head(10).to_string(index=False))


# ==================================================================================================
# 10.7R — INTERNAL SANITY CHECKS
# ==================================================================================================
#
# These assertions verify that the optimized implementation obeys basic
# invariants before we trust its diagnostic outputs.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.7R — INTERNAL SANITY CHECKS")
print("=" * 120)

# Event count cannot increase as threshold becomes larger.
event_counts = candidate_summary["Candidate_Events"].to_numpy()

assert np.all(np.diff(event_counts) <= 0), (
    "ERROR: Candidate event count increased with a larger threshold."
)

# Every threshold must account for every usable GT row exactly once.
for threshold, event_table in candidate_event_tables.items():

    reconstructed_rows = int(event_table["GT_Rows"].sum())

    assert reconstructed_rows == len(ordered), (
        f"ERROR at threshold {threshold}: "
        f"event rows={reconstructed_rows:,}, "
        f"expected={len(ordered):,}"
    )

# Candidate event IDs must be unique.
for threshold, event_table in candidate_event_tables.items():

    assert event_table["Candidate_Event_ID"].is_unique, (
        f"ERROR: duplicate Candidate_Event_ID at threshold {threshold}"
    )

# Threshold 0 event count should equal number of zero-threshold components.
assert (
    len(candidate_event_tables[0])
    == len(component_network)
), "ERROR: threshold-0 component reconstruction mismatch."

print("✓ Event count monotonically non-increasing with threshold")
print("✓ Every threshold preserves every usable GT row exactly once")
print("✓ Candidate event IDs are unique")
print("✓ Threshold-0 reconstruction matches base temporal components")


# ==================================================================================================
# 10.7S — SAVE DIAGNOSTIC OUTPUTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.7S — SAVING DIAGNOSTIC OUTPUTS")
print("=" * 120)

candidate_summary.to_csv(
    UNSW_OUT / "10_7_candidate_threshold_summary.csv",
    index=False
)

collapse.to_csv(
    UNSW_OUT / "10_7_threshold_collapse_profile.csv",
    index=False
)

component_gaps.to_csv(
    UNSW_OUT / "10_7_temporal_component_gaps.csv",
    index=False
)

gap_band_table.to_csv(
    UNSW_OUT / "10_7_gap_band_distribution.csv",
    index=False
)

boundaries_output.to_csv(
    UNSW_OUT / "10_7_network_boundary_continuity_raw.csv",
    index=False
)

continuity_table.to_csv(
    UNSW_OUT / "10_7_network_boundary_continuity_summary.csv",
    index=False
)

duration_pathology.to_csv(
    UNSW_OUT / "10_7_duration_pathology.csv",
    index=False
)

complexity_table.to_csv(
    UNSW_OUT / "10_7_network_complexity_sensitivity.csv",
    index=False
)

family_consistency.to_csv(
    UNSW_OUT / "10_7_family_consistency_sensitivity.csv",
    index=False
)

identity_capture.reset_index().to_csv(
    UNSW_OUT / "10_7_cross_capture_identity_support.csv",
    index=False
)

example_identity_table.to_csv(
    UNSW_OUT / "10_7_high_frequency_identity_sensitivity.csv",
    index=False
)

change_table.to_csv(
    UNSW_OUT / "10_7_largest_threshold_changes.csv",
    index=False
)

print("✓ Diagnostic outputs saved")


# ==================================================================================================
# 10.7T — DECISION GATE
# ==================================================================================================

elapsed = time.time() - T0

print("\n" + "=" * 120)
print("10.7T — DECISION GATE")
print("=" * 120)

print(f"""
PHASE 10.7 COMPLETED IN {elapsed:.1f} SECONDS.

DO NOT SELECT A FINAL GAP THRESHOLD AUTOMATICALLY.

DO NOT CREATE FINAL Episode_ID VALUES YET.

PHASE 10.7 TESTED HOW DIFFERENT TEMPORAL BOUNDARY RULES CHANGE
THE INTERPRETATION OF UNSW PHYSICAL ATTACK OCCURRENCES.

WE NOW REVIEW:

A. THRESHOLD STABILITY
   At what thresholds does candidate event count change dramatically?

B. GAP STRUCTURE
   Is there a natural separation between within-occurrence gaps and
   between-occurrence gaps?

C. NETWORK CONTINUITY
   Across longer gaps, do source/victim/protocol/5-tuple identities
   remain the same or change?

D. OVER-MERGING
   Do larger thresholds create implausibly long events containing
   many unrelated hosts, protocols, or manifestations?

E. KNOWN-TYPE FORECASTING
   Attack TYPE may recur across partitions.
   Physical EVENT occurrences must never cross partitions.

ONLY AFTER REVIEWING THESE RESULTS SHOULD PHASE 10.8 FREEZE THE
UNSW PHYSICAL OCCURRENCE / Episode_ID DEFINITION.

NO TRAIN/VALIDATION/TEST SPLIT HAS BEEN CREATED.
NO FORECASTING TARGET HAS BEEN CREATED.
NO FINAL EPISODE DEFINITION HAS BEEN CREATED.
""")

print("=" * 120)
print("PHASE 10.7 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.7U — LOCAL GAP-BAND NETWORK CONTINUITY
# ==================================================================================================
#
# PURPOSE
# -------
# Existing continuity analysis is cumulative:
#
#       Gap > X
#
# For selecting a short physical-occurrence boundary we instead need:
#
#       exact local gap band
#
# This tells us whether 1s / 2s / 5s / 10s / 30s gaps behave differently.
#
# ==================================================================================================

print("=" * 120)
print("10.7U — LOCAL GAP-BAND NETWORK CONTINUITY")
print("=" * 120)

local_boundaries = boundaries_output.copy()

bins = [
    0,
    1,
    2,
    5,
    10,
    30,
    60,
    120,
    300,
    600,
    1800,
    3600,
    5400,
    7200,
    np.inf
]

labels = [
    "(0,1]",
    "(1,2]",
    "(2,5]",
    "(5,10]",
    "(10,30]",
    "(30,60]",
    "(60,120]",
    "(120,300]",
    "(300,600]",
    "(600,1800]",
    "(1800,3600]",
    "(3600,5400]",
    "(5400,7200]",
    ">7200"
]

local_boundaries["Gap_Band"] = pd.cut(
    local_boundaries["Gap_Sec"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=False
)

local_continuity = (
    local_boundaries
    .groupby(
        "Gap_Band",
        observed=False
    )
    .agg(
        Boundaries=("Gap_Sec", "size"),

        Median_Gap_Sec=("Gap_Sec", "median"),

        Same_Source_Pct=(
            "Same_Source",
            lambda x: 100 * x.mean()
        ),

        Same_Destination_Pct=(
            "Same_Destination",
            lambda x: 100 * x.mean()
        ),

        Same_Protocol_Pct=(
            "Same_Protocol",
            lambda x: 100 * x.mean()
        ),

        Same_HostPair_Pct=(
            "Same_HostPair",
            lambda x: 100 * x.mean()
        ),

        Same_Strict_5Tuple_Pct=(
            "Same_Strict_5Tuple",
            lambda x: 100 * x.mean()
        ),
    )
    .reset_index()
)

print()
print(local_continuity.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# Additional interpretation aid:
# compare short-gap bands directly.
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("SHORT-GAP REGION — MOST IMPORTANT FOR EPISODE DEFINITION")
print("=" * 120)

short_gap = local_continuity[
    local_continuity["Gap_Band"].astype(str).isin([
        "(0,1]",
        "(1,2]",
        "(2,5]",
        "(5,10]",
        "(10,30]",
        "(30,60]"
    ])
].copy()

print(short_gap.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# Save
# --------------------------------------------------------------------------------------------------

local_continuity.to_csv(
    UNSW_OUT / "10_7_local_gap_band_network_continuity.csv",
    index=False
)

print("\n✓ Saved:")
print(
    UNSW_OUT /
    "10_7_local_gap_band_network_continuity.csv"
)

print("\n" + "=" * 120)
print("10.7U COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.8 — UNSW PHYSICAL ATTACK EPISODE CONSTRUCTION & VALIDATION
# CLEAN REWRITE
# ==================================================================================================
#
# PURPOSE
# -------
# Define candidate physical attack episodes in UNSW-NB15 BEFORE:
#
#   - mapping attacks to flow records
#   - constructing minute timelines
#   - combining UNSW with CIC
#   - train/validation/test splitting
#   - sequence generation
#   - scaling
#   - model training
#
# PRIMARY EPISODE RULE
# --------------------
# Two GT manifestations belong to the same physical episode when they share:
#
#   1. Capture_ID
#   2. Fine Attack_Identity
#   3. Unordered host pair
#   4. Temporal continuity: next manifestation begins <= 5 sec
#      after the current running episode end
#
# WHY 5 SEC?
# ----------
# Phase 10.7U showed:
#
#   (0,1] sec  -> same host pair ~99.82%
#   (1,2] sec  -> same host pair ~99.76%
#   (2,5] sec  -> same host pair ~97.54%
#   (5,10] sec -> same host pair ~89.14%
#   (10,30]    -> same host pair ~78.97%
#
# Therefore 5 sec is our evidence-supported PRIMARY candidate.
#
# IMPORTANT
# ---------
# This cell does NOT split the dataset.
# This cell does NOT train anything.
# This cell does NOT modify raw UNSW files.
#
# ==================================================================================================

import numpy as np
import pandas as pd
from pathlib import Path


# ==================================================================================================
# 10.8A — CONFIGURATION
# ==================================================================================================

print("=" * 120)
print("PHASE 10.8 — UNSW PHYSICAL ATTACK EPISODE CONSTRUCTION & VALIDATION")
print("=" * 120)

PRIMARY_GAP_SEC = 5
SENSITIVITY_THRESHOLDS = [1, 2, 5, 10, 30, 60]

if "UNSW_OUT" not in globals():
    UNSW_OUT = Path(
        "/content/drive/MyDrive/network-attack-forecasting/"
        "Dataset_Audit_v2/audit_outputs/UNSW"
    )
else:
    UNSW_OUT = Path(UNSW_OUT)

UNSW_OUT.mkdir(parents=True, exist_ok=True)

print(f"\nPrimary temporal boundary : {PRIMARY_GAP_SEC} sec")
print(f"Sensitivity thresholds    : {SENSITIVITY_THRESHOLDS}")
print(f"Output directory          : {UNSW_OUT}")


# ==================================================================================================
# 10.8B — RECOVER RECONSTRUCTED GT TABLE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8B — INPUT TABLE RECOVERY")
print("=" * 120)

candidate_names = [
    "gt_work",
    "gt",
    "gt_df",
    "gt_clean",
    "gt_semantic",
    "gt_reconstructed",
]

required_columns = {
    "Start_DT",
    "End_DT",
    "Capture_ID",
    "Source IP",
    "Destination IP",
}

episode_gt = None
episode_gt_source = None

for name in candidate_names:

    if name not in globals():
        continue

    obj = globals()[name]

    if not isinstance(obj, pd.DataFrame):
        continue

    if required_columns.issubset(obj.columns):
        episode_gt = obj.copy()
        episode_gt_source = name
        break


if episode_gt is None:
    raise RuntimeError(
        "\nCould not find the reconstructed UNSW GT DataFrame from Phase 10.7.\n\n"
        "Expected a DataFrame containing:\n"
        "  Start_DT\n"
        "  End_DT\n"
        "  Capture_ID\n"
        "  Source IP\n"
        "  Destination IP\n\n"
        "Do NOT automatically rerun the expensive Phase 10.7."
    )

print("Recovered variable :", episode_gt_source)
print("Rows               :", f"{len(episode_gt):,}")
print("Columns            :", len(episode_gt.columns))


# ==================================================================================================
# 10.8C — PRESERVE RAW SEMANTICS + CANONICALIZE LABEL STRINGS
# ==================================================================================================
#
# We preserve the original raw values.
#
# Canonical fields are ONLY stripped of surrounding whitespace.
# We are NOT merging semantic categories manually.
#
# Example:
#
#       "Reconnaissance "
#
# becomes:
#
#       "Reconnaissance"
#
# But:
#
#       "Backdoor"
#       "Backdoors"
#
# remain DIFFERENT because that is a semantic naming difference,
# not merely whitespace.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8C — SEMANTIC LABEL NORMALIZATION")
print("=" * 120)

if "Attack category" in episode_gt.columns:

    episode_gt["Attack_Family_Raw"] = (
        episode_gt["Attack category"].astype("string")
    )

    episode_gt["Attack_Family"] = (
        episode_gt["Attack category"]
        .astype("string")
        .str.strip()
    )

    episode_gt["Attack_Family"] = episode_gt["Attack_Family"].replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        }
    )

    raw_family_count = (
        episode_gt["Attack_Family_Raw"].nunique(dropna=True)
    )

    canonical_family_count = (
        episode_gt["Attack_Family"].nunique(dropna=True)
    )

    print("Raw family strings       :", raw_family_count)
    print("Canonical family strings :", canonical_family_count)

    family_normalization_audit = (
        episode_gt[
            ["Attack_Family_Raw", "Attack_Family"]
        ]
        .drop_duplicates()
        .sort_values(
            ["Attack_Family", "Attack_Family_Raw"],
            kind="mergesort"
        )
        .reset_index(drop=True)
    )

    print("\nRaw → canonical family mappings:")
    print(family_normalization_audit.to_string(index=False))

else:
    episode_gt["Attack_Family_Raw"] = pd.NA
    episode_gt["Attack_Family"] = pd.NA

    family_normalization_audit = pd.DataFrame(
        columns=[
            "Attack_Family_Raw",
            "Attack_Family"
        ]
    )

    print("⚠ Attack category column unavailable.")


if "Attack subcategory" in episode_gt.columns:

    episode_gt["Attack_Subcategory"] = (
        episode_gt["Attack subcategory"]
        .astype("string")
        .str.strip()
    )

else:
    episode_gt["Attack_Subcategory"] = pd.NA


# ==================================================================================================
# 10.8D — CONSTRUCT FINE ATTACK IDENTITY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8D — FINE ATTACK IDENTITY")
print("=" * 120)

if "Attack Name" in episode_gt.columns:

    attack_name = (
        episode_gt["Attack Name"]
        .astype("string")
        .str.strip()
    )

    attack_name = attack_name.replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        }
    )

    episode_gt["Attack_Identity"] = attack_name

else:
    episode_gt["Attack_Identity"] = pd.NA


# Fallback for missing Attack Name.
if (
    "Attack_Family" in episode_gt.columns
    and "Attack_Subcategory" in episode_gt.columns
):

    fallback_identity = (
        episode_gt["Attack_Family"].astype("string")
        + " | "
        + episode_gt["Attack_Subcategory"].astype("string")
    )

    episode_gt["Attack_Identity"] = (
        episode_gt["Attack_Identity"]
        .fillna(fallback_identity)
    )


print(
    "Unique fine attack identities :",
    f"{episode_gt['Attack_Identity'].nunique(dropna=True):,}"
)


# ==================================================================================================
# 10.8E — REQUIRED FIELD VALIDATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8E — REQUIRED FIELD VALIDATION")
print("=" * 120)

episode_gt["Start_DT"] = pd.to_datetime(
    episode_gt["Start_DT"],
    errors="coerce"
)

episode_gt["End_DT"] = pd.to_datetime(
    episode_gt["End_DT"],
    errors="coerce"
)

validation_counts = {
    "Invalid Start_DT":
        int(episode_gt["Start_DT"].isna().sum()),

    "Invalid End_DT":
        int(episode_gt["End_DT"].isna().sum()),

    "Missing Capture_ID":
        int(episode_gt["Capture_ID"].isna().sum()),

    "Missing Attack_Identity":
        int(episode_gt["Attack_Identity"].isna().sum()),

    "Missing Source IP":
        int(episode_gt["Source IP"].isna().sum()),

    "Missing Destination IP":
        int(episode_gt["Destination IP"].isna().sum()),

    "End before Start":
        int(
            (
                episode_gt["End_DT"]
                < episode_gt["Start_DT"]
            ).sum()
        ),
}

for key, value in validation_counts.items():
    print(f"{key:<28}: {value:,}")


if any(validation_counts.values()):

    raise ValueError(
        "\nEpisode construction stopped because required fields are invalid.\n"
        "Do not silently drop these rows."
    )

print("\n✓ Required episode fields are valid")


# ==================================================================================================
# 10.8F — BUILD UNORDERED HOST-PAIR CONTEXT
# ==================================================================================================
#
# A → B and B → A are treated as the same host relationship.
#
# Ports are deliberately NOT included because 10.7U showed strict 5-tuple
# continuity was much weaker than host-pair continuity.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8F — NETWORK CONTEXT")
print("=" * 120)

src = (
    episode_gt["Source IP"]
    .astype(str)
    .str.strip()
)

dst = (
    episode_gt["Destination IP"]
    .astype(str)
    .str.strip()
)

episode_gt["Host_A"] = np.where(
    src <= dst,
    src,
    dst
)

episode_gt["Host_B"] = np.where(
    src <= dst,
    dst,
    src
)

episode_gt["Host_Pair"] = (
    episode_gt["Host_A"].astype(str)
    + " <-> "
    + episode_gt["Host_B"].astype(str)
)

directed_pairs = (
    episode_gt[
        ["Source IP", "Destination IP"]
    ]
    .drop_duplicates()
    .shape[0]
)

unordered_pairs = (
    episode_gt[
        ["Host_A", "Host_B"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Unique directed source→destination pairs :",
    f"{directed_pairs:,}"
)

print(
    "Unique unordered host pairs             :",
    f"{unordered_pairs:,}"
)


# ==================================================================================================
# 10.8G — EPISODE CONSTRUCTION FUNCTION
# ==================================================================================================

GROUP_COLS = [
    "Capture_ID",
    "Attack_Identity",
    "Host_A",
    "Host_B",
]


def construct_unsw_episodes(df, gap_threshold_sec):

    work = df.copy()

    work = work.sort_values(
        GROUP_COLS
        + ["Start_DT", "End_DT"],
        kind="mergesort"
    ).reset_index(drop=True)

    local_episode_numbers = np.empty(
        len(work),
        dtype=np.int64
    )

    current_group = None
    current_running_end = None
    local_episode = 0

    group_positions = [
        work.columns.get_loc(col)
        for col in GROUP_COLS
    ]

    start_position = (
        work.columns.get_loc("Start_DT")
    )

    end_position = (
        work.columns.get_loc("End_DT")
    )

    for i, row in enumerate(
        work.itertuples(
            index=False,
            name=None
        )
    ):

        group_key = tuple(
            row[pos]
            for pos in group_positions
        )

        start_time = row[start_position]
        end_time = row[end_position]

        # New semantic/network context.
        if group_key != current_group:

            current_group = group_key
            current_running_end = end_time
            local_episode = 1

        else:

            gap_sec = (
                start_time
                - current_running_end
            ).total_seconds()

            # Overlap OR gap <= threshold:
            # continue same physical episode.
            if gap_sec <= gap_threshold_sec:

                if end_time > current_running_end:
                    current_running_end = end_time

            else:

                local_episode += 1
                current_running_end = end_time

        local_episode_numbers[i] = (
            local_episode
        )

    work["Episode_Local_Number"] = (
        local_episode_numbers
    )

    work["Episode_ID"] = (
        work
        .groupby(
            GROUP_COLS
            + ["Episode_Local_Number"],
            sort=False,
            dropna=False
        )
        .ngroup()
        .astype(np.int64)
        + 1
    )

    return work


# ==================================================================================================
# 10.8H — THRESHOLD SENSITIVITY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8H — HOST-CONTEXT-AWARE THRESHOLD SENSITIVITY")
print("=" * 120)

sensitivity_rows = []
episode_versions = {}

for threshold in SENSITIVITY_THRESHOLDS:

    tmp = construct_unsw_episodes(
        episode_gt,
        threshold
    )

    episode_versions[threshold] = tmp

    tmp_summary = (
        tmp
        .groupby("Episode_ID")
        .agg(
            Episode_Start=("Start_DT", "min"),
            Episode_End=("End_DT", "max"),
            GT_Rows=("Episode_ID", "size"),
        )
    )

    durations = (
        tmp_summary["Episode_End"]
        - tmp_summary["Episode_Start"]
    ).dt.total_seconds()

    sensitivity_rows.append(
        {
            "Threshold_Sec":
                threshold,

            "Episodes":
                len(tmp_summary),

            "Median_Duration_Sec":
                durations.median(),

            "P90_Duration_Sec":
                durations.quantile(0.90),

            "P99_Duration_Sec":
                durations.quantile(0.99),

            "Max_Duration_Sec":
                durations.max(),

            "Median_GT_Rows":
                tmp_summary["GT_Rows"].median(),

            "P99_GT_Rows":
                tmp_summary["GT_Rows"].quantile(0.99),

            "Max_GT_Rows":
                tmp_summary["GT_Rows"].max(),
        }
    )

sensitivity_df = pd.DataFrame(
    sensitivity_rows
)

print(
    sensitivity_df.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.8I — SELECT PRIMARY 5-SECOND CANDIDATE
# ==================================================================================================

episodes_gt = (
    episode_versions[
        PRIMARY_GAP_SEC
    ].copy()
)

print("\n" + "=" * 120)
print("10.8I — PRIMARY EPISODE CANDIDATE")
print("=" * 120)

print(
    "Temporal threshold :",
    PRIMARY_GAP_SEC,
    "sec"
)

print(
    "GT rows            :",
    f"{len(episodes_gt):,}"
)

print(
    "Episode count      :",
    f"{episodes_gt['Episode_ID'].nunique():,}"
)


# ==================================================================================================
# 10.8J — BUILD COMPLETE EPISODE-LEVEL TABLE
# ==================================================================================================
#
# IMPORTANT FIX:
# Host_Pair is explicitly retained here.
#
# ==================================================================================================

aggregation = {
    "Capture_ID":
        ("Capture_ID", "first"),

    "Attack_Identity":
        ("Attack_Identity", "first"),

    "Host_A":
        ("Host_A", "first"),

    "Host_B":
        ("Host_B", "first"),

    "Host_Pair":
        ("Host_Pair", "first"),

    "Episode_Start":
        ("Start_DT", "min"),

    "Episode_End":
        ("End_DT", "max"),

    "GT_Rows":
        ("Episode_ID", "size"),
}


if "Protocol" in episodes_gt.columns:

    aggregation["Unique_Protocols"] = (
        "Protocol",
        "nunique"
    )


if "Attack_Family" in episodes_gt.columns:

    aggregation["Attack_Family"] = (
        "Attack_Family",
        "first"
    )


if "Attack_Subcategory" in episodes_gt.columns:

    aggregation["Attack_Subcategory"] = (
        "Attack_Subcategory",
        "first"
    )


episode_table = (
    episodes_gt
    .groupby(
        "Episode_ID",
        sort=False
    )
    .agg(**aggregation)
    .reset_index()
)


episode_table["Duration_Sec"] = (
    episode_table["Episode_End"]
    - episode_table["Episode_Start"]
).dt.total_seconds()


print("\nEpisode-level table built.")
print("Rows :", f"{len(episode_table):,}")
print(
    "Unique Episode_IDs :",
    f"{episode_table['Episode_ID'].nunique():,}"
)


# ==================================================================================================
# 10.8K — EPISODE DURATION AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8K — EPISODE DURATION AUDIT")
print("=" * 120)

print(
    episode_table["Duration_Sec"]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999,
        ]
    )
)


duration_bands = pd.cut(
    episode_table["Duration_Sec"],
    bins=[
        -np.inf,
        0,
        1,
        5,
        10,
        30,
        60,
        300,
        600,
        1800,
        3600,
        np.inf,
    ],
    labels=[
        "0 sec",
        "(0,1] sec",
        "(1,5] sec",
        "(5,10] sec",
        "(10,30] sec",
        "(30,60] sec",
        "(1,5] min",
        "(5,10] min",
        "(10,30] min",
        "(30,60] min",
        ">60 min",
    ]
)

duration_band_counts = (
    duration_bands
    .value_counts(sort=False)
    .rename_axis("Duration_Band")
    .reset_index(name="Episodes")
)

duration_band_counts["Pct"] = (
    100
    * duration_band_counts["Episodes"]
    / len(episode_table)
)

print("\nDuration bands:")
print(
    duration_band_counts.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.8L — GT MANIFESTATIONS PER EPISODE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8L — GT MANIFESTATIONS PER PHYSICAL EPISODE")
print("=" * 120)

print(
    episode_table["GT_Rows"]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999,
        ]
    )
)

multirow_count = int(
    (
        episode_table["GT_Rows"] > 1
    ).sum()
)

print(
    "\nEpisodes containing >1 GT manifestation :",
    f"{multirow_count:,}",
    "/",
    f"{len(episode_table):,}",
    f"({100 * multirow_count / len(episode_table):.2f}%)"
)


# ==================================================================================================
# 10.8M — EPISODE PURITY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8M — EPISODE PURITY CHECKS")
print("=" * 120)

purity_agg = {
    "Capture_Count":
        ("Capture_ID", "nunique"),

    "Attack_Identity_Count":
        ("Attack_Identity", "nunique"),

    "Host_Pair_Count":
        ("Host_Pair", "nunique"),
}

if "Attack_Family" in episodes_gt.columns:

    purity_agg["Attack_Family_Count"] = (
        "Attack_Family",
        "nunique"
    )


purity = (
    episodes_gt
    .groupby("Episode_ID")
    .agg(**purity_agg)
)


capture_violations = int(
    (
        purity["Capture_Count"] != 1
    ).sum()
)

identity_violations = int(
    (
        purity["Attack_Identity_Count"] != 1
    ).sum()
)

hostpair_violations = int(
    (
        purity["Host_Pair_Count"] != 1
    ).sum()
)


if "Attack_Family_Count" in purity.columns:

    family_violations = int(
        (
            purity["Attack_Family_Count"] != 1
        ).sum()
    )

else:

    family_violations = 0


print(
    "Cross-capture episodes          :",
    f"{capture_violations:,}"
)

print(
    "Mixed attack-identity episodes  :",
    f"{identity_violations:,}"
)

print(
    "Mixed host-pair episodes        :",
    f"{hostpair_violations:,}"
)

print(
    "Mixed canonical-family episodes :",
    f"{family_violations:,}"
)


# ==================================================================================================
# 10.8N — INSPECT ANY MIXED-FAMILY EPISODES
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8N — MIXED-FAMILY EPISODE INSPECTION")
print("=" * 120)

if family_violations > 0:

    mixed_family_ids = (
        purity[
            purity["Attack_Family_Count"] > 1
        ]
        .index
        .tolist()
    )

    mixed_family_rows = (
        episodes_gt[
            episodes_gt["Episode_ID"].isin(
                mixed_family_ids
            )
        ]
        [
            [
                "Episode_ID",
                "Capture_ID",
                "Attack_Identity",
                "Attack_Family_Raw",
                "Attack_Family",
                "Attack_Subcategory",
                "Source IP",
                "Destination IP",
                "Start_DT",
                "End_DT",
            ]
        ]
        .sort_values(
            [
                "Episode_ID",
                "Start_DT",
                "End_DT",
            ]
        )
    )

    print(
        mixed_family_rows.to_string(
            index=False
        )
    )

else:

    mixed_family_rows = pd.DataFrame()

    print(
        "✓ No mixed canonical-family episodes."
    )


# ==================================================================================================
# 10.8O — INTERNAL TEMPORAL CONNECTIVITY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8O — INTERNAL TEMPORAL CONNECTIVITY")
print("=" * 120)

temporal_check = (
    episodes_gt[
        [
            "Episode_ID",
            "Start_DT",
            "End_DT",
        ]
    ]
    .sort_values(
        [
            "Episode_ID",
            "Start_DT",
            "End_DT",
        ],
        kind="mergesort"
    )
    .copy()
)


temporal_check["Running_End"] = (
    temporal_check
    .groupby("Episode_ID")["End_DT"]
    .cummax()
)


temporal_check["Prev_Running_End"] = (
    temporal_check
    .groupby("Episode_ID")[
        "Running_End"
    ]
    .shift(1)
)


temporal_check["Internal_Gap_Sec"] = (
    temporal_check["Start_DT"]
    - temporal_check["Prev_Running_End"]
).dt.total_seconds()


positive_internal = (
    temporal_check[
        temporal_check["Internal_Gap_Sec"] > 0
    ]["Internal_Gap_Sec"]
)


if len(positive_internal) > 0:

    print(
        "Largest positive internal gap :",
        f"{positive_internal.max():.3f} sec"
    )

else:

    print(
        "Largest positive internal gap : none"
    )


internal_gap_violations = int(
    (
        temporal_check["Internal_Gap_Sec"]
        > PRIMARY_GAP_SEC
    ).sum()
)


print(
    f"Internal gaps > {PRIMARY_GAP_SEC} sec :",
    f"{internal_gap_violations:,}"
)


# ==================================================================================================
# 10.8P — SAME-CONTEXT INTER-EPISODE SEPARATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8P — SAME-CONTEXT INTER-EPISODE SEPARATION")
print("=" * 120)

context_episode = (
    episode_table
    .sort_values(
        GROUP_COLS
        + [
            "Episode_Start",
            "Episode_End",
        ],
        kind="mergesort"
    )
    .copy()
)


context_episode["Prev_Episode_End"] = (
    context_episode
    .groupby(
        GROUP_COLS,
        dropna=False
    )["Episode_End"]
    .shift(1)
)


context_episode["Inter_Episode_Gap_Sec"] = (
    context_episode["Episode_Start"]
    - context_episode["Prev_Episode_End"]
).dt.total_seconds()


valid_inter = (
    context_episode[
        context_episode[
            "Inter_Episode_Gap_Sec"
        ].notna()
    ]
)


if len(valid_inter) > 0:

    print(
        "Minimum same-context inter-episode gap :",
        f"{valid_inter['Inter_Episode_Gap_Sec'].min():.3f} sec"
    )

    print(
        "Median same-context inter-episode gap  :",
        f"{valid_inter['Inter_Episode_Gap_Sec'].median():.3f} sec"
    )

else:

    print(
        "No repeated same-context episodes found."
    )


inter_gap_violations = int(
    (
        valid_inter["Inter_Episode_Gap_Sec"]
        <= PRIMARY_GAP_SEC
    ).sum()
)


print(
    f"Same-context episode gaps <= {PRIMARY_GAP_SEC} sec :",
    f"{inter_gap_violations:,}"
)


# ==================================================================================================
# 10.8Q — ATTACK FAMILY DISTRIBUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8Q — ATTACK DISTRIBUTION AT EPISODE LEVEL")
print("=" * 120)

if episode_table["Attack_Family"].notna().any():

    family_summary = (
        episode_table
        .groupby(
            "Attack_Family",
            dropna=False
        )
        .agg(
            Episodes=(
                "Episode_ID",
                "nunique"
            ),

            Fine_Identities=(
                "Attack_Identity",
                "nunique"
            ),

            Median_Duration_Sec=(
                "Duration_Sec",
                "median"
            ),

            P90_Duration_Sec=(
                "Duration_Sec",
                lambda x: x.quantile(0.90)
            ),

            Max_Duration_Sec=(
                "Duration_Sec",
                "max"
            ),
        )
        .sort_values(
            "Episodes",
            ascending=False
        )
        .reset_index()
    )

    print(
        family_summary.to_string(
            index=False
        )
    )

else:

    family_summary = pd.DataFrame()

    print(
        "Attack family unavailable."
    )


# ==================================================================================================
# 10.8R — CAPTURE-LEVEL EPISODE DISTRIBUTION
# ==================================================================================================
#
# Host_Pair is now explicitly available in episode_table.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8R — CAPTURE-LEVEL EPISODE DISTRIBUTION")
print("=" * 120)

capture_summary = (
    episode_table
    .groupby(
        "Capture_ID",
        dropna=False
    )
    .agg(
        Episodes=(
            "Episode_ID",
            "nunique"
        ),

        Fine_Identities=(
            "Attack_Identity",
            "nunique"
        ),

        Host_Pairs=(
            "Host_Pair",
            "nunique"
        ),

        Start=(
            "Episode_Start",
            "min"
        ),

        End=(
            "Episode_End",
            "max"
        ),
    )
    .reset_index()
)

print(
    capture_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.8S — KNOWN-TYPE RECURRENCE ACROSS CAPTURES
# ==================================================================================================
#
# Current experiment:
#
#       KNOWN-TYPE FORECASTING
#
# Therefore:
#
#   attack identity/type may recur across partitions later
#   physical Episode_ID may NOT leak across partitions
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8S — KNOWN-TYPE RECURRENCE ACROSS CAPTURES")
print("=" * 120)

identity_capture = (
    episode_table[
        [
            "Attack_Identity",
            "Capture_ID",
        ]
    ]
    .drop_duplicates()
)


identity_capture_counts = (
    identity_capture
    .groupby("Attack_Identity")[
        "Capture_ID"
    ]
    .nunique()
)


total_identities = len(
    identity_capture_counts
)

recurrent_identities = int(
    (
        identity_capture_counts > 1
    ).sum()
)

single_capture_identities = int(
    (
        identity_capture_counts == 1
    ).sum()
)


print(
    "Total fine attack identities :",
    f"{total_identities:,}"
)

print(
    "Present in >1 capture        :",
    f"{recurrent_identities:,}",
    f"({100 * recurrent_identities / total_identities:.2f}%)"
)

print(
    "Present in only one capture  :",
    f"{single_capture_identities:,}",
    f"({100 * single_capture_identities / total_identities:.2f}%)"
)


# Family-level recurrence is also useful for later known-family forecasting.

family_capture_recurrence = None

if episode_table["Attack_Family"].notna().any():

    family_capture_recurrence = (
        episode_table[
            [
                "Attack_Family",
                "Capture_ID",
            ]
        ]
        .drop_duplicates()
        .groupby(
            "Attack_Family"
        )["Capture_ID"]
        .nunique()
        .sort_values(
            ascending=False
        )
    )

    print(
        "\nAttack-family capture recurrence:"
    )

    print(
        family_capture_recurrence.to_string()
    )


# ==================================================================================================
# 10.8T — HOST-CONTEXT FRAGMENTATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8T — HOST-CONTEXT FRAGMENTATION AUDIT")
print("=" * 120)

fragmentation = (
    episode_table
    .groupby(
        [
            "Capture_ID",
            "Attack_Identity",
        ],
        dropna=False
    )
    .agg(
        Episodes=(
            "Episode_ID",
            "nunique"
        ),

        Host_Pairs=(
            "Host_Pair",
            "nunique"
        ),

        First_Start=(
            "Episode_Start",
            "min"
        ),

        Last_End=(
            "Episode_End",
            "max"
        ),
    )
    .reset_index()
)


fragmentation["Episodes_Per_HostPair"] = (
    fragmentation["Episodes"]
    / fragmentation["Host_Pairs"]
)


print(
    fragmentation[
        [
            "Episodes",
            "Host_Pairs",
            "Episodes_Per_HostPair",
        ]
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


print(
    "\nHighest episode-count attack identities:"
)

print(
    fragmentation
    .sort_values(
        "Episodes",
        ascending=False
    )
    .head(20)
    .to_string(
        index=False
    )
)


# ==================================================================================================
# 10.8U — EXTREME EPISODE INSPECTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8U — EXTREME / PATHOLOGICAL EPISODE INSPECTION")
print("=" * 120)

display_cols = [
    "Episode_ID",
    "Capture_ID",
    "Attack_Identity",
    "Attack_Family",
    "Host_A",
    "Host_B",
    "Episode_Start",
    "Episode_End",
    "Duration_Sec",
    "GT_Rows",
]


print("\n20 longest episodes:")

longest_episodes = (
    episode_table
    .sort_values(
        "Duration_Sec",
        ascending=False
    )
    .head(20)
)

print(
    longest_episodes[
        display_cols
    ].to_string(
        index=False
    )
)


print(
    "\n20 episodes with most GT manifestations:"
)

largest_row_episodes = (
    episode_table
    .sort_values(
        "GT_Rows",
        ascending=False
    )
    .head(20)
)

print(
    largest_row_episodes[
        display_cols
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 10.8V — CROSS-EPISODE TEMPORAL OVERLAP AUDIT
# ==================================================================================================
#
# Different attacks can legitimately overlap in time.
#
# This is NOT automatically an error.
#
# We measure it because it matters later when assigning minute-level targets.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8V — CROSS-EPISODE TEMPORAL OVERLAP AUDIT")
print("=" * 120)

overlap_records = []


for capture_id, cap in episode_table.groupby(
    "Capture_ID",
    sort=False
):

    cap = (
        cap
        .sort_values(
            [
                "Episode_Start",
                "Episode_End",
            ],
            kind="mergesort"
        )
        .copy()
    )

    starts = (
        cap["Episode_Start"]
        .to_numpy(
            dtype="datetime64[ns]"
        )
    )

    ends = (
        cap["Episode_End"]
        .to_numpy(
            dtype="datetime64[ns]"
        )
    )

    sorted_ends = np.sort(ends)

    # Number of episodes already started by each row.
    started_count = np.arange(
        1,
        len(cap) + 1
    )

    # Episodes that ended strictly before this start.
    ended_before_start = np.searchsorted(
        sorted_ends,
        starts,
        side="left"
    )

    active_other_episodes = (
        started_count
        - ended_before_start
        - 1
    )

    cap[
        "Other_Episodes_Active_At_Start"
    ] = active_other_episodes

    overlap_records.append(
        cap[
            [
                "Episode_ID",
                "Capture_ID",
                "Other_Episodes_Active_At_Start",
            ]
        ]
    )


overlap_audit = pd.concat(
    overlap_records,
    ignore_index=True
)


episodes_starting_during_other = int(
    (
        overlap_audit[
            "Other_Episodes_Active_At_Start"
        ] > 0
    ).sum()
)


print(
    "Episodes beginning while another episode is active :",
    f"{episodes_starting_during_other:,}",
    "/",
    f"{len(overlap_audit):,}",
    f"({100 * episodes_starting_during_other / len(overlap_audit):.2f}%)"
)

print(
    "Maximum other episodes active at one episode start :",
    int(
        overlap_audit[
            "Other_Episodes_Active_At_Start"
        ].max()
    )
)


# ==================================================================================================
# 10.8W — FINAL STRUCTURAL INVARIANTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8W — FINAL STRUCTURAL INVARIANTS")
print("=" * 120)

checks = {
    "Every GT row has Episode_ID":
        bool(
            episodes_gt[
                "Episode_ID"
            ].notna().all()
        ),

    "Episode table has Host_Pair":
        "Host_Pair"
        in episode_table.columns,

    "No cross-capture Episode_ID":
        capture_violations == 0,

    "No mixed Attack_Identity Episode_ID":
        identity_violations == 0,

    "No mixed Host_Pair Episode_ID":
        hostpair_violations == 0,

    "No mixed canonical-family Episode_ID":
        family_violations == 0,

    f"No internal positive gap > {PRIMARY_GAP_SEC}s":
        internal_gap_violations == 0,

    f"No same-context inter-episode gap <= {PRIMARY_GAP_SEC}s":
        inter_gap_violations == 0,

    "No negative episode durations":
        bool(
            (
                episode_table[
                    "Duration_Sec"
                ] >= 0
            ).all()
        ),

    "One row per Episode_ID in episode table":
        bool(
            episode_table[
                "Episode_ID"
            ].is_unique
        ),
}


all_pass = True

for check_name, passed in checks.items():

    if passed:
        print(
            f"✓ PASS   {check_name}"
        )

    else:
        print(
            f"✗ FAIL   {check_name}"
        )

        all_pass = False


# ==================================================================================================
# 10.8X — SAVE OUTPUTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.8X — SAVE AUDIT ARTIFACTS")
print("=" * 120)

paths = {
    "row_level":
        UNSW_OUT
        / "10_8_unsw_gt_with_candidate_episode_ids.csv",

    "episode_table":
        UNSW_OUT
        / "10_8_unsw_candidate_episode_table.csv",

    "sensitivity":
        UNSW_OUT
        / "10_8_episode_threshold_sensitivity.csv",

    "family_normalization":
        UNSW_OUT
        / "10_8_attack_family_normalization_audit.csv",

    "fragmentation":
        UNSW_OUT
        / "10_8_host_context_fragmentation.csv",

    "overlap":
        UNSW_OUT
        / "10_8_cross_episode_overlap_audit.csv",

    "capture_summary":
        UNSW_OUT
        / "10_8_capture_episode_summary.csv",

    "family_summary":
        UNSW_OUT
        / "10_8_attack_family_episode_summary.csv",
}


episodes_gt.to_csv(
    paths["row_level"],
    index=False
)

episode_table.to_csv(
    paths["episode_table"],
    index=False
)

sensitivity_df.to_csv(
    paths["sensitivity"],
    index=False
)

family_normalization_audit.to_csv(
    paths["family_normalization"],
    index=False
)

fragmentation.to_csv(
    paths["fragmentation"],
    index=False
)

overlap_audit.to_csv(
    paths["overlap"],
    index=False
)

capture_summary.to_csv(
    paths["capture_summary"],
    index=False
)

family_summary.to_csv(
    paths["family_summary"],
    index=False
)


if len(mixed_family_rows) > 0:

    mixed_path = (
        UNSW_OUT
        / "10_8_mixed_family_episode_inspection.csv"
    )

    mixed_family_rows.to_csv(
        mixed_path,
        index=False
    )

    paths["mixed_family"] = mixed_path


for name, path in paths.items():
    print(
        f"✓ {name:<22} {path}"
    )


# ==================================================================================================
# 10.8Y — FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 120)
print("PHASE 10.8 — FINAL STATUS")
print("=" * 120)

print(
    f"""
PRIMARY CANDIDATE EPISODE RULE

    Capture_ID
        +
    fine Attack_Identity
        +
    unordered Host_Pair
        +
    temporal continuity <= {PRIMARY_GAP_SEC} seconds


GT rows represented       : {len(episodes_gt):,}
Candidate physical episodes: {episode_table['Episode_ID'].nunique():,}
Fine attack identities    : {episode_table['Attack_Identity'].nunique():,}
Canonical attack families : {episode_table['Attack_Family'].nunique(dropna=True):,}
Host pairs                : {episode_table['Host_Pair'].nunique():,}
Captures                  : {episode_table['Capture_ID'].nunique():,}
"""
)


if all_pass:

    print(
        "✓ ALL STRUCTURAL EPISODE INVARIANTS PASSED"
    )

    print(
        "\nThe candidate episode construction is mechanically consistent."
    )

    print(
        "Do NOT begin model training yet."
    )

    print(
        "Next: interpret episode distributions and then map validated UNSW episodes "
        "to the actual flow records."
    )

else:

    print(
        "⚠ ONE OR MORE STRUCTURAL CHECKS FAILED"
    )

    print(
        "\nDo NOT use these Episode_IDs for splitting or training yet."
    )

    print(
        "Inspect the failed audit(s) before proceeding."
    )


print("\n" + "=" * 120)
print("10.8 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.9 — UNSW GT ↔ FLOW CORRESPONDENCE FEASIBILITY AUDIT
#
# PURPOSE
# -------
# Before assigning attack Episode_IDs to UNSW flow records, determine whether the reconstructed
# UNSW ground truth can be matched safely and unambiguously to the raw UNSW flow observations.
#
# THIS CELL DOES NOT:
#   - create final flow labels
#   - create minute bins
#   - create train/validation/test splits
#   - modify raw UNSW files
#   - resolve the known mixed-family semantic inconsistency from 10.8
#
# THIS CELL DOES:
#   1. Recover the authoritative 10.8 row-level GT episode table
#   2. Discover UNSW raw flow CSV files
#   3. Inspect their schema
#   4. Detect timestamp/network columns
#   5. Reconstruct flow timestamps
#   6. Audit capture-level temporal correspondence
#   7. Audit endpoint / protocol / port compatibility
#   8. Sample GT rows and test hierarchical correspondence rules
#   9. Quantify ambiguity BEFORE full mapping
#  10. Save audit artifacts
#
# IMPORTANT
# ---------
# Final GT → flow assignment is intentionally deferred to Phase 10.10.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import gc
import warnings
warnings.filterwarnings("ignore")

print("=" * 120)
print("PHASE 10.9 — UNSW GT ↔ FLOW CORRESPONDENCE FEASIBILITY AUDIT")
print("=" * 120)


# ==================================================================================================
# 10.9A — PATHS
# ==================================================================================================

BASE = Path("/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2")

RAW_UNSW_DIR = BASE / "raw" / "UNSW_NB15"
OUT_DIR = BASE / "audit_outputs" / "UNSW"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPISODE_ROW_FILE = OUT_DIR / "10_8_unsw_gt_with_candidate_episode_ids.csv"
EPISODE_TABLE_FILE = OUT_DIR / "10_8_unsw_candidate_episode_table.csv"

print("\n" + "=" * 120)
print("10.9A — PATH CHECK")
print("=" * 120)

if not EPISODE_ROW_FILE.exists():
    raise FileNotFoundError(
        f"Required 10.8 row-level artifact not found:\n{EPISODE_ROW_FILE}"
    )

if not EPISODE_TABLE_FILE.exists():
    raise FileNotFoundError(
        f"Required 10.8 episode artifact not found:\n{EPISODE_TABLE_FILE}"
    )

if not RAW_UNSW_DIR.exists():
    raise FileNotFoundError(
        f"UNSW raw directory not found:\n{RAW_UNSW_DIR}"
    )

print("✓ 10.8 row-level GT artifact found")
print("✓ 10.8 episode table found")
print("✓ UNSW raw directory found")


# ==================================================================================================
# 10.9B — LOAD AUTHORITATIVE 10.8 GT TABLE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9B — LOAD 10.8 EPISODE-AWARE GT")
print("=" * 120)

gt = pd.read_csv(
    EPISODE_ROW_FILE,
    low_memory=False
)

print(f"Rows    : {len(gt):,}")
print(f"Columns : {len(gt.columns):,}")

required_gt = [
    "Episode_ID",
    "Capture_ID",
    "Start_DT",
    "End_DT",
    "Attack_Identity",
    "Attack_Family",
]

missing = [c for c in required_gt if c not in gt.columns]

if missing:
    raise ValueError(
        "10.8 GT artifact is missing required columns:\n"
        + "\n".join(missing)
    )

gt["Start_DT"] = pd.to_datetime(gt["Start_DT"], errors="coerce")
gt["End_DT"] = pd.to_datetime(gt["End_DT"], errors="coerce")

if gt["Start_DT"].isna().any() or gt["End_DT"].isna().any():
    raise ValueError("Invalid Start_DT / End_DT found in 10.8 artifact.")

print(f"Unique episodes : {gt['Episode_ID'].nunique():,}")
print(f"Captures        : {gt['Capture_ID'].nunique():,}")
print(f"GT start        : {gt['Start_DT'].min()}")
print(f"GT end          : {gt['End_DT'].max()}")

print("✓ 10.8 GT loaded successfully")


# ==================================================================================================
# 10.9C — DISCOVER RAW UNSW FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9C — RAW UNSW FILE DISCOVERY")
print("=" * 120)

all_csv = sorted(RAW_UNSW_DIR.glob("*.csv"))

if not all_csv:
    raise FileNotFoundError(
        f"No CSV files found in:\n{RAW_UNSW_DIR}"
    )

print(f"CSV files discovered: {len(all_csv)}")

for p in all_csv:
    size_mb = p.stat().st_size / (1024 ** 2)
    print(f"  {p.name:<45} {size_mb:>10.2f} MB")


# ==================================================================================================
# 10.9D — IDENTIFY CANDIDATE FLOW FILES
#
# We explicitly exclude metadata, GT and pre-defined train/test files.
# We want the original UNSW observation files if present.
# ==================================================================================================

def is_candidate_flow_file(path):
    n = path.name.lower()

    exclude_tokens = [
        "feature",
        "_gt",
        "ground",
        "training",
        "testing",
        "train",
        "test",
        "list_events"
    ]

    if any(tok in n for tok in exclude_tokens):
        return False

    return True


flow_files = [p for p in all_csv if is_candidate_flow_file(p)]

print("\nCandidate raw flow files:")

if not flow_files:
    raise RuntimeError(
        "No candidate raw UNSW flow CSVs were identified.\n"
        "Inspect the file list printed above before proceeding."
    )

for p in flow_files:
    print("  ✓", p.name)


# ==================================================================================================
# 10.9E — ROBUST CSV HEADER READER
# ==================================================================================================

ENCODINGS = [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin1"
]


def read_csv_sample(path, nrows=5, header="infer"):

    last_error = None

    for enc in ENCODINGS:
        try:
            df = pd.read_csv(
                path,
                nrows=nrows,
                encoding=enc,
                low_memory=False,
                header=header
            )
            return df, enc

        except Exception as e:
            last_error = e

    raise RuntimeError(
        f"Could not read {path.name}\nLast error: {last_error}"
    )


# ==================================================================================================
# 10.9F — FLOW SCHEMA AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9F — FLOW SCHEMA AUDIT")
print("=" * 120)

schema_records = []

for path in flow_files:

    sample, enc = read_csv_sample(path, nrows=5)

    cols = [str(c).strip() for c in sample.columns]

    schema_records.append({
        "File": path.name,
        "Encoding": enc,
        "Columns": len(cols),
        "Column_List": " | ".join(cols)
    })

    print(f"\n{path.name}")
    print("-" * 100)
    print("Encoding :", enc)
    print("Columns  :", len(cols))
    print(cols)

schema_df = pd.DataFrame(schema_records)

schema_path = OUT_DIR / "10_9_raw_flow_schema_audit.csv"
schema_df.to_csv(schema_path, index=False)

print("\n✓ Saved schema audit:")
print(schema_path)


# ==================================================================================================
# 10.9G — NORMALIZED COLUMN-NAME DETECTION
# ==================================================================================================

def normalize_col(c):
    return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())


COLUMN_ALIASES = {

    "src_ip": [
        "srcip", "sourceip", "sourceaddress", "srcaddr"
    ],

    "dst_ip": [
        "dstip", "destinationip", "destinationaddress", "dstaddr"
    ],

    "src_port": [
        "sport", "srcport", "sourceport"
    ],

    "dst_port": [
        "dsport", "dstport", "destinationport"
    ],

    "protocol": [
        "proto", "protocol"
    ],

    "start_time": [
        "starttime",
        "stime",
        "start",
        "flowstart",
        "flowstarttime"
    ],

    "end_time": [
        "endtime",
        "ltime",
        "lasttime",
        "end",
        "flowend",
        "flowendtime"
    ],

    "timestamp": [
        "timestamp",
        "time",
        "datetime",
        "date"
    ],

    "duration": [
        "dur",
        "duration",
        "flowduration"
    ]
}


def detect_column(columns, aliases):

    normalized = {
        normalize_col(c): c
        for c in columns
    }

    for alias in aliases:
        if alias in normalized:
            return normalized[alias]

    return None


detection_records = []

for path in flow_files:

    sample, enc = read_csv_sample(path, nrows=5)

    columns = list(sample.columns)

    rec = {
        "File": path.name,
        "Encoding": enc
    }

    for semantic, aliases in COLUMN_ALIASES.items():
        rec[semantic] = detect_column(columns, aliases)

    detection_records.append(rec)


detect_df = pd.DataFrame(detection_records)

print("\n" + "=" * 120)
print("10.9G — DETECTED FLOW SEMANTICS")
print("=" * 120)

print(detect_df.to_string(index=False))

detect_path = OUT_DIR / "10_9_detected_flow_semantics.csv"
detect_df.to_csv(detect_path, index=False)

print("\n✓ Saved:")
print(detect_path)


# ==================================================================================================
# 10.9H — DETERMINE WHETHER RAW FILES ARE HEADERLESS
#
# Some UNSW distributions contain raw CSVs without meaningful headers.
# We do NOT guess column semantics silently.
# ==================================================================================================

def looks_like_generated_integer_header(columns):

    if len(columns) == 0:
        return False

    count = 0

    for c in columns:
        s = str(c).strip()

        try:
            float(s)
            count += 1
        except:
            pass

    return (count / len(columns)) > 0.5


header_status = []

for path in flow_files:

    sample, enc = read_csv_sample(path, nrows=5)

    suspicious = looks_like_generated_integer_header(sample.columns)

    header_status.append({
        "File": path.name,
        "Likely_Headerless": suspicious
    })

header_status_df = pd.DataFrame(header_status)

print("\n" + "=" * 120)
print("10.9H — HEADER STRUCTURE CHECK")
print("=" * 120)

print(header_status_df.to_string(index=False))


# ==================================================================================================
# 10.9I — FLOW FILE READER WITH SAFE SEMANTIC RECOVERY
#
# UNSW raw files may be headerless. If so, we only recover semantics if the
# official feature metadata provides exactly the same width.
# ==================================================================================================

FEATURE_FILE = RAW_UNSW_DIR / "NUSW-NB15_features.csv"

feature_names = None

if FEATURE_FILE.exists():

    features = None
    feature_encoding = None

    for enc in ENCODINGS:
        try:
            features = pd.read_csv(
                FEATURE_FILE,
                encoding=enc,
                low_memory=False
            )
            feature_encoding = enc
            break
        except:
            continue

    if features is not None:

        features.columns = features.columns.astype(str).str.strip()

        name_col = None

        for c in features.columns:
            if normalize_col(c) in ["name", "featurename"]:
                name_col = c
                break

        if name_col is not None:
            feature_names = (
                features[name_col]
                .astype(str)
                .str.strip()
                .tolist()
            )

            print("\nFeature metadata available:")
            print("Encoding      :", feature_encoding)
            print("Feature count :", len(feature_names))


def load_flow_file(path):

    # First try normal header interpretation.
    sample, enc = read_csv_sample(path, nrows=5)

    suspicious_header = looks_like_generated_integer_header(sample.columns)

    if not suspicious_header:
        df = pd.read_csv(
            path,
            encoding=enc,
            low_memory=False
        )

        df.columns = df.columns.astype(str).str.strip()

        return df, enc, "existing_header"

    # If suspicious/headerless, inspect actual width.
    raw_sample, enc2 = read_csv_sample(
        path,
        nrows=5,
        header=None
    )

    width = raw_sample.shape[1]

    if feature_names is not None and len(feature_names) == width:

        df = pd.read_csv(
            path,
            encoding=enc2,
            header=None,
            names=feature_names,
            low_memory=False
        )

        return df, enc2, "metadata_header"

    raise RuntimeError(
        f"\nCannot safely recover schema for {path.name}\n"
        f"Observed width       : {width}\n"
        f"Metadata feature count: "
        f"{len(feature_names) if feature_names is not None else 'unavailable'}\n\n"
        "Stopping rather than guessing raw column semantics."
    )


# ==================================================================================================
# 10.9J — TIMESTAMP PARSING UTILITIES
# ==================================================================================================

def parse_time_series(series):

    s = series.copy()

    numeric = pd.to_numeric(s, errors="coerce")

    numeric_ratio = numeric.notna().mean()

    # UNIX epoch seconds
    if numeric_ratio > 0.95:

        finite = numeric.dropna()

        if len(finite):

            med = finite.median()

            # Epoch seconds roughly covering modern network datasets.
            if 1e9 < med < 2e9:
                return pd.to_datetime(
                    numeric,
                    unit="s",
                    errors="coerce"
                )

            # Epoch milliseconds
            if 1e12 < med < 2e12:
                return pd.to_datetime(
                    numeric,
                    unit="ms",
                    errors="coerce"
                )

    # Standard datetime strings
    return pd.to_datetime(
        s,
        errors="coerce",
        dayfirst=True
    )


# ==================================================================================================
# 10.9K — LOAD RAW FLOW FILES ONE AT A TIME
#
# We intentionally do NOT concatenate everything immediately.
# This avoids unnecessary memory use.
# ==================================================================================================

flow_file_summaries = []
flow_frames = []

print("\n" + "=" * 120)
print("10.9K — FLOW FILE LOAD & TEMPORAL RECONSTRUCTION")
print("=" * 120)

for file_index, path in enumerate(flow_files):

    print(f"\nProcessing: {path.name}")

    df, enc, header_mode = load_flow_file(path)

    print(f"  rows       : {len(df):,}")
    print(f"  columns    : {len(df.columns):,}")
    print(f"  encoding   : {enc}")
    print(f"  header mode: {header_mode}")

    columns = list(df.columns)

    src_col = detect_column(columns, COLUMN_ALIASES["src_ip"])
    dst_col = detect_column(columns, COLUMN_ALIASES["dst_ip"])
    sport_col = detect_column(columns, COLUMN_ALIASES["src_port"])
    dport_col = detect_column(columns, COLUMN_ALIASES["dst_port"])
    proto_col = detect_column(columns, COLUMN_ALIASES["protocol"])

    start_col = detect_column(columns, COLUMN_ALIASES["start_time"])
    end_col = detect_column(columns, COLUMN_ALIASES["end_time"])
    timestamp_col = detect_column(columns, COLUMN_ALIASES["timestamp"])
    duration_col = detect_column(columns, COLUMN_ALIASES["duration"])

    print("  src IP     :", src_col)
    print("  dst IP     :", dst_col)
    print("  src port   :", sport_col)
    print("  dst port   :", dport_col)
    print("  protocol   :", proto_col)
    print("  start time :", start_col)
    print("  end time   :", end_col)
    print("  timestamp  :", timestamp_col)
    print("  duration   :", duration_col)

    # ----------------------------------------------------------------------------------------------
    # Build standardized working frame.
    # ----------------------------------------------------------------------------------------------

    work = pd.DataFrame(index=df.index)

    work["Flow_File"] = path.name
    work["Flow_Row_Index"] = np.arange(len(df), dtype=np.int64)

    if src_col:
        work["Source_IP"] = df[src_col].astype(str).str.strip()
    else:
        work["Source_IP"] = np.nan

    if dst_col:
        work["Destination_IP"] = df[dst_col].astype(str).str.strip()
    else:
        work["Destination_IP"] = np.nan

    if sport_col:
        work["Source_Port"] = pd.to_numeric(
            df[sport_col],
            errors="coerce"
        )
    else:
        work["Source_Port"] = np.nan

    if dport_col:
        work["Destination_Port"] = pd.to_numeric(
            df[dport_col],
            errors="coerce"
        )
    else:
        work["Destination_Port"] = np.nan

    if proto_col:
        work["Protocol"] = (
            df[proto_col]
            .astype(str)
            .str.strip()
            .str.lower()
        )
    else:
        work["Protocol"] = np.nan

    # ----------------------------------------------------------------------------------------------
    # Time reconstruction.
    # ----------------------------------------------------------------------------------------------

    flow_start = None
    flow_end = None

    if start_col:
        flow_start = parse_time_series(df[start_col])

    elif timestamp_col:
        flow_start = parse_time_series(df[timestamp_col])

    if end_col:
        flow_end = parse_time_series(df[end_col])

    # If end time is unavailable but duration exists, reconstruct it.
    if (
        flow_start is not None
        and flow_end is None
        and duration_col is not None
    ):

        duration = pd.to_numeric(
            df[duration_col],
            errors="coerce"
        )

        flow_end = flow_start + pd.to_timedelta(
            duration,
            unit="s"
        )

    # If only one timestamp exists, treat it as a point observation for now.
    if flow_start is not None and flow_end is None:
        flow_end = flow_start.copy()

    if flow_start is None:
        print("  ⚠ No directly interpretable flow timestamp detected.")

        work["Flow_Start"] = pd.NaT
        work["Flow_End"] = pd.NaT

    else:

        work["Flow_Start"] = flow_start
        work["Flow_End"] = flow_end

    valid_time = work["Flow_Start"].notna()

    if valid_time.any():

        start = work.loc[valid_time, "Flow_Start"].min()
        end = work.loc[valid_time, "Flow_End"].max()

        print("  valid times:", f"{valid_time.sum():,}")
        print("  start      :", start)
        print("  end        :", end)

    else:

        start = pd.NaT
        end = pd.NaT

        print("  valid times: 0")

    flow_file_summaries.append({
        "File": path.name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Encoding": enc,
        "Header_Mode": header_mode,
        "Source_IP_Column": src_col,
        "Destination_IP_Column": dst_col,
        "Source_Port_Column": sport_col,
        "Destination_Port_Column": dport_col,
        "Protocol_Column": proto_col,
        "Start_Time_Column": start_col,
        "End_Time_Column": end_col,
        "Timestamp_Column": timestamp_col,
        "Duration_Column": duration_col,
        "Valid_Timestamps": int(valid_time.sum()),
        "Start": start,
        "End": end
    })

    flow_frames.append(work)

    del df
    gc.collect()


flow_summary_df = pd.DataFrame(flow_file_summaries)

summary_path = OUT_DIR / "10_9_flow_file_temporal_summary.csv"
flow_summary_df.to_csv(summary_path, index=False)

print("\n✓ Saved:")
print(summary_path)


# ==================================================================================================
# 10.9L — COMBINE STANDARDIZED CORRESPONDENCE FIELDS ONLY
#
# This is intentionally small compared with concatenating all original feature columns.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9L — STANDARDIZED FLOW CORRESPONDENCE TABLE")
print("=" * 120)

flows = pd.concat(
    flow_frames,
    ignore_index=True
)

del flow_frames
gc.collect()

print(f"Rows : {len(flows):,}")
print("Columns:")
print(list(flows.columns))

print("\nValid field counts:")

for c in [
    "Flow_Start",
    "Flow_End",
    "Source_IP",
    "Destination_IP",
    "Source_Port",
    "Destination_Port",
    "Protocol"
]:
    print(f"  {c:<20}: {flows[c].notna().sum():,}")


# ==================================================================================================
# 10.9M — GT NETWORK FIELD DETECTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9M — GT NETWORK FIELD DETECTION")
print("=" * 120)

gt_cols = list(gt.columns)

gt_src_col = detect_column(gt_cols, COLUMN_ALIASES["src_ip"])
gt_dst_col = detect_column(gt_cols, COLUMN_ALIASES["dst_ip"])
gt_sport_col = detect_column(gt_cols, COLUMN_ALIASES["src_port"])
gt_dport_col = detect_column(gt_cols, COLUMN_ALIASES["dst_port"])
gt_proto_col = detect_column(gt_cols, COLUMN_ALIASES["protocol"])

print("Source IP        :", gt_src_col)
print("Destination IP   :", gt_dst_col)
print("Source Port      :", gt_sport_col)
print("Destination Port :", gt_dport_col)
print("Protocol         :", gt_proto_col)

if gt_src_col is None or gt_dst_col is None:
    raise ValueError(
        "GT source/destination IP fields could not be identified."
    )


# ==================================================================================================
# 10.9N — NORMALIZE GT NETWORK KEYS
# ==================================================================================================

gt_work = gt.copy()

gt_work["GT_Source_IP"] = (
    gt_work[gt_src_col]
    .astype(str)
    .str.strip()
)

gt_work["GT_Destination_IP"] = (
    gt_work[gt_dst_col]
    .astype(str)
    .str.strip()
)

if gt_sport_col:
    gt_work["GT_Source_Port"] = pd.to_numeric(
        gt_work[gt_sport_col],
        errors="coerce"
    )
else:
    gt_work["GT_Source_Port"] = np.nan

if gt_dport_col:
    gt_work["GT_Destination_Port"] = pd.to_numeric(
        gt_work[gt_dport_col],
        errors="coerce"
    )
else:
    gt_work["GT_Destination_Port"] = np.nan

if gt_proto_col:
    gt_work["GT_Protocol"] = (
        gt_work[gt_proto_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )
else:
    gt_work["GT_Protocol"] = np.nan


# ==================================================================================================
# 10.9O — GLOBAL TEMPORAL COVERAGE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9O — GLOBAL TEMPORAL COVERAGE")
print("=" * 120)

valid_flows = flows[
    flows["Flow_Start"].notna()
].copy()

if len(valid_flows) == 0:
    raise RuntimeError(
        "No usable flow timestamps were reconstructed.\n"
        "Cannot perform GT ↔ flow correspondence audit."
    )

flow_global_start = valid_flows["Flow_Start"].min()
flow_global_end = valid_flows["Flow_End"].max()

gt_global_start = gt_work["Start_DT"].min()
gt_global_end = gt_work["End_DT"].max()

print("GT global range   :", gt_global_start, "→", gt_global_end)
print("Flow global range :", flow_global_start, "→", flow_global_end)

gt_inside_global_flow_range = (
    (gt_work["Start_DT"] <= flow_global_end)
    &
    (gt_work["End_DT"] >= flow_global_start)
)

print(
    "GT rows temporally intersecting global flow range:",
    f"{gt_inside_global_flow_range.sum():,}",
    "/",
    f"{len(gt_work):,}",
    f"({100 * gt_inside_global_flow_range.mean():.2f}%)"
)


# ==================================================================================================
# 10.9P — ENDPOINT VOCABULARY CORRESPONDENCE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9P — ENDPOINT VOCABULARY CORRESPONDENCE")
print("=" * 120)

flow_ips = set(
    pd.concat([
        valid_flows["Source_IP"],
        valid_flows["Destination_IP"]
    ])
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

gt_src_in_flow = gt_work["GT_Source_IP"].isin(flow_ips)
gt_dst_in_flow = gt_work["GT_Destination_IP"].isin(flow_ips)

print(
    f"GT source IP represented in flow vocabulary      : "
    f"{gt_src_in_flow.sum():,}/{len(gt_work):,} "
    f"({100 * gt_src_in_flow.mean():.2f}%)"
)

print(
    f"GT destination IP represented in flow vocabulary : "
    f"{gt_dst_in_flow.sum():,}/{len(gt_work):,} "
    f"({100 * gt_dst_in_flow.mean():.2f}%)"
)

both_endpoint_vocab = gt_src_in_flow & gt_dst_in_flow

print(
    f"Both GT endpoints represented                    : "
    f"{both_endpoint_vocab.sum():,}/{len(gt_work):,} "
    f"({100 * both_endpoint_vocab.mean():.2f}%)"
)


# ==================================================================================================
# 10.9Q — DETERMINE CAPTURE ↔ FLOW-FILE TEMPORAL CORRESPONDENCE
#
# We do not assume filenames correspond directly to Capture_ID.
# We infer correspondence from temporal overlap.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9Q — CAPTURE ↔ FLOW FILE TEMPORAL CORRESPONDENCE")
print("=" * 120)

capture_ranges = (
    gt_work
    .groupby("Capture_ID", as_index=False)
    .agg(
        GT_Start=("Start_DT", "min"),
        GT_End=("End_DT", "max"),
        GT_Rows=("Episode_ID", "size"),
        Episodes=("Episode_ID", "nunique")
    )
)

capture_file_records = []

for _, cap in capture_ranges.iterrows():

    for _, f in flow_summary_df.iterrows():

        fstart = pd.to_datetime(f["Start"], errors="coerce")
        fend = pd.to_datetime(f["End"], errors="coerce")

        if pd.isna(fstart) or pd.isna(fend):
            overlap_sec = 0

        else:

            overlap_start = max(cap["GT_Start"], fstart)
            overlap_end = min(cap["GT_End"], fend)

            overlap_sec = max(
                0,
                (overlap_end - overlap_start).total_seconds()
            )

        capture_file_records.append({
            "Capture_ID": cap["Capture_ID"],
            "Flow_File": f["File"],
            "GT_Start": cap["GT_Start"],
            "GT_End": cap["GT_End"],
            "Flow_Start": fstart,
            "Flow_End": fend,
            "Temporal_Overlap_Sec": overlap_sec
        })


capture_file_df = pd.DataFrame(capture_file_records)

print(
    capture_file_df
    .sort_values(
        ["Capture_ID", "Temporal_Overlap_Sec"],
        ascending=[True, False]
    )
    .to_string(index=False)
)

capture_file_path = OUT_DIR / "10_9_capture_to_flow_file_temporal_correspondence.csv"
capture_file_df.to_csv(capture_file_path, index=False)

print("\n✓ Saved:")
print(capture_file_path)


# ==================================================================================================
# 10.9R — ASSIGN EACH FLOW FILE TO BEST GT CAPTURE BY TEMPORAL OVERLAP
#
# This is an AUDIT assignment only.
# It is not yet an authoritative label mapping.
# ==================================================================================================

best_capture_by_file = (
    capture_file_df
    .sort_values(
        ["Flow_File", "Temporal_Overlap_Sec"],
        ascending=[True, False]
    )
    .drop_duplicates("Flow_File")
)

file_to_capture = dict(
    zip(
        best_capture_by_file["Flow_File"],
        best_capture_by_file["Capture_ID"]
    )
)

file_to_overlap = dict(
    zip(
        best_capture_by_file["Flow_File"],
        best_capture_by_file["Temporal_Overlap_Sec"]
    )
)

flows["Audit_Capture_ID"] = flows["Flow_File"].map(file_to_capture)
flows["Audit_Capture_Overlap_Sec"] = flows["Flow_File"].map(file_to_overlap)

# Do not pretend zero-overlap files belong to a capture.
flows.loc[
    flows["Audit_Capture_Overlap_Sec"].fillna(0) <= 0,
    "Audit_Capture_ID"
] = np.nan


print("\nBest temporal capture correspondence per flow file:")

print(
    best_capture_by_file[
        [
            "Flow_File",
            "Capture_ID",
            "Temporal_Overlap_Sec"
        ]
    ].to_string(index=False)
)


# ==================================================================================================
# 10.9S — STRATIFIED GT SAMPLE — CORRECTED
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9S — STRATIFIED GT SAMPLE")
print("=" * 120)

SAMPLE_PER_GROUP = 200
RANDOM_STATE = 42

# ----------------------------------------------------------------------------------
# IMPORTANT:
# Do NOT use groupby.apply(... include_groups=False).
# That removed Capture_ID and Attack_Family in the previous version.
#
# Instead, generate sampled row indices and select directly from gt_work.
# This preserves ALL original columns.
# ----------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

sample_indices = []

for _, group in gt_work.groupby(
    ["Capture_ID", "Attack_Family"],
    dropna=False,
    sort=False
):
    idx = group.index.to_numpy()

    if len(idx) > SAMPLE_PER_GROUP:
        idx = rng.choice(
            idx,
            size=SAMPLE_PER_GROUP,
            replace=False
        )

    sample_indices.extend(idx.tolist())


sample_gt = (
    gt_work
    .loc[sample_indices]
    .copy()
    .reset_index(drop=True)
)

# ----------------------------------------------------------------------------------
# Safety checks
# ----------------------------------------------------------------------------------

required_sample_cols = [
    "Capture_ID",
    "Attack_Family",
    "Episode_ID",
    "Attack_Identity",
    "Start_DT",
    "End_DT",
    "GT_Source_IP",
    "GT_Destination_IP",
    "GT_Source_Port",
    "GT_Destination_Port",
    "GT_Protocol"
]

missing_sample_cols = [
    c for c in required_sample_cols
    if c not in sample_gt.columns
]

if missing_sample_cols:
    raise RuntimeError(
        "Sampling unexpectedly removed required columns:\n"
        + "\n".join(missing_sample_cols)
    )

print(f"Full GT rows   : {len(gt_work):,}")
print(f"Sample GT rows : {len(sample_gt):,}")

print("\nSample distribution:")

sample_distribution = (
    sample_gt
    .groupby(
        ["Capture_ID", "Attack_Family"],
        dropna=False
    )
    .size()
    .rename("Rows")
    .reset_index()
)

print(sample_distribution.to_string(index=False))

print("\n✓ Stratified sample created successfully")
print("✓ Capture_ID preserved")
print("✓ Attack_Family preserved")
print("✓ All correspondence fields preserved")

# ==================================================================================================
# 10.9T — BUILD FAST FLOW INDICES BY CAPTURE + DIRECTED HOST PAIR
#
# Critical optimization:
# We do NOT scan millions of flows for every GT row.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9T — BUILD CORRESPONDENCE INDICES")
print("=" * 120)

flow_groups = {}

group_cols = [
    "Audit_Capture_ID",
    "Source_IP",
    "Destination_IP"
]

for key, idx in flows.groupby(
    group_cols,
    sort=False,
    dropna=False
).groups.items():

    flow_groups[key] = np.asarray(idx)

print(f"Directed capture+host-pair indices: {len(flow_groups):,}")


# ==================================================================================================
# 10.9U — HIERARCHICAL GT ↔ FLOW CORRESPONDENCE TEST
#
# Matching levels:
#
# L1 = capture + temporal overlap
# L2 = L1 + directed source/destination
# L3 = L2 + protocol
# L4 = L3 + destination port
# L5 = L4 + source port
#
# We intentionally measure each level separately.
#
# Why?
# Because we do NOT want to assume that every GT field maps exactly onto every flow field.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9U — HIERARCHICAL CORRESPONDENCE AUDIT")
print("=" * 120)

TEMP_TOLERANCE_SEC = 1

results = []

# Cache capture-level indices for fallback temporal-only audit.
capture_indices = {
    cap: np.asarray(idx)
    for cap, idx in flows.groupby(
        "Audit_Capture_ID",
        sort=False
    ).groups.items()
}


for counter, row in sample_gt.iterrows():

    cap = row["Capture_ID"]

    gt_start = row["Start_DT"] - pd.Timedelta(seconds=TEMP_TOLERANCE_SEC)
    gt_end = row["End_DT"] + pd.Timedelta(seconds=TEMP_TOLERANCE_SEC)

    src = row["GT_Source_IP"]
    dst = row["GT_Destination_IP"]

    gt_proto = row["GT_Protocol"]
    gt_sport = row["GT_Source_Port"]
    gt_dport = row["GT_Destination_Port"]

    # ----------------------------------------------------------------------------------------------
    # Level 1 — temporal overlap anywhere in corresponding capture
    # ----------------------------------------------------------------------------------------------

    cap_idx = capture_indices.get(cap, np.array([], dtype=int))

    if len(cap_idx):

        c = flows.loc[cap_idx]

        temporal_mask = (
            (c["Flow_Start"] <= gt_end)
            &
            (c["Flow_End"] >= gt_start)
        )

        l1_idx = c.index[temporal_mask].to_numpy()

    else:
        l1_idx = np.array([], dtype=int)

    # ----------------------------------------------------------------------------------------------
    # Level 2 — temporal + directed endpoints
    # ----------------------------------------------------------------------------------------------

    pair_idx = flow_groups.get(
        (cap, src, dst),
        np.array([], dtype=int)
    )

    if len(pair_idx):

        p = flows.loc[pair_idx]

        pair_time_mask = (
            (p["Flow_Start"] <= gt_end)
            &
            (p["Flow_End"] >= gt_start)
        )

        l2_idx = p.index[pair_time_mask].to_numpy()

    else:
        l2_idx = np.array([], dtype=int)

    # ----------------------------------------------------------------------------------------------
    # Level 3 — + protocol
    # ----------------------------------------------------------------------------------------------

    if len(l2_idx) and pd.notna(gt_proto):

        l2 = flows.loc[l2_idx]

        l3_idx = l2.index[
            l2["Protocol"].astype(str).str.lower()
            ==
            str(gt_proto).lower()
        ].to_numpy()

    else:
        l3_idx = l2_idx.copy()

    # ----------------------------------------------------------------------------------------------
    # Level 4 — + destination port
    # ----------------------------------------------------------------------------------------------

    if len(l3_idx) and pd.notna(gt_dport):

        l3 = flows.loc[l3_idx]

        l4_idx = l3.index[
            l3["Destination_Port"] == gt_dport
        ].to_numpy()

    else:
        l4_idx = l3_idx.copy()

    # ----------------------------------------------------------------------------------------------
    # Level 5 — + source port
    # ----------------------------------------------------------------------------------------------

    if len(l4_idx) and pd.notna(gt_sport):

        l4 = flows.loc[l4_idx]

        l5_idx = l4.index[
            l4["Source_Port"] == gt_sport
        ].to_numpy()

    else:
        l5_idx = l4_idx.copy()

    results.append({
        "Episode_ID": row["Episode_ID"],
        "Capture_ID": cap,
        "Attack_Family": row["Attack_Family"],
        "Attack_Identity": row["Attack_Identity"],

        "GT_Start": row["Start_DT"],
        "GT_End": row["End_DT"],

        "GT_Source_IP": src,
        "GT_Destination_IP": dst,
        "GT_Protocol": gt_proto,
        "GT_Source_Port": gt_sport,
        "GT_Destination_Port": gt_dport,

        "L1_Time_Matches": len(l1_idx),
        "L2_Time_HostPair_Matches": len(l2_idx),
        "L3_Protocol_Matches": len(l3_idx),
        "L4_DstPort_Matches": len(l4_idx),
        "L5_Strict_5Tuple_Matches": len(l5_idx)
    })

    if (counter + 1) % 500 == 0:
        print(
            f"  audited {counter + 1:,} / {len(sample_gt):,} GT rows"
        )


corr_df = pd.DataFrame(results)

print("\n✓ Correspondence sampling complete")


# ==================================================================================================
# 10.9V — CORRESPONDENCE COVERAGE SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9V — CORRESPONDENCE COVERAGE SUMMARY")
print("=" * 120)

levels = [
    "L1_Time_Matches",
    "L2_Time_HostPair_Matches",
    "L3_Protocol_Matches",
    "L4_DstPort_Matches",
    "L5_Strict_5Tuple_Matches"
]

coverage_records = []

for col in levels:

    matched = (corr_df[col] > 0).sum()
    unique = (corr_df[col] == 1).sum()
    ambiguous = (corr_df[col] > 1).sum()
    unmatched = (corr_df[col] == 0).sum()

    coverage_records.append({
        "Match_Level": col,
        "Sample_GT_Rows": len(corr_df),
        "Matched_GT_Rows": matched,
        "Matched_Pct": 100 * matched / len(corr_df),
        "Unique_GT_Rows": unique,
        "Unique_Pct": 100 * unique / len(corr_df),
        "Ambiguous_GT_Rows": ambiguous,
        "Ambiguous_Pct": 100 * ambiguous / len(corr_df),
        "Unmatched_GT_Rows": unmatched,
        "Unmatched_Pct": 100 * unmatched / len(corr_df),
        "Median_Matches": corr_df[col].median(),
        "P95_Matches": corr_df[col].quantile(0.95),
        "Max_Matches": corr_df[col].max()
    })


coverage_df = pd.DataFrame(coverage_records)

print(coverage_df.to_string(index=False))


# ==================================================================================================
# 10.9W — COVERAGE BY ATTACK FAMILY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9W — CORRESPONDENCE BY ATTACK FAMILY")
print("=" * 120)

family_records = []

for family, x in corr_df.groupby(
    "Attack_Family",
    dropna=False
):

    rec = {
        "Attack_Family": family,
        "Sample_GT_Rows": len(x)
    }

    for col in levels:
        rec[f"{col}_Coverage_Pct"] = 100 * (x[col] > 0).mean()

    family_records.append(rec)


family_corr_df = pd.DataFrame(family_records)

print(
    family_corr_df
    .sort_values("Sample_GT_Rows", ascending=False)
    .to_string(index=False)
)


# ==================================================================================================
# 10.9X — COVERAGE BY CAPTURE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9X — CORRESPONDENCE BY CAPTURE")
print("=" * 120)

capture_records = []

for capture, x in corr_df.groupby(
    "Capture_ID",
    dropna=False
):

    rec = {
        "Capture_ID": capture,
        "Sample_GT_Rows": len(x)
    }

    for col in levels:
        rec[f"{col}_Coverage_Pct"] = 100 * (x[col] > 0).mean()

    capture_records.append(rec)


capture_corr_df = pd.DataFrame(capture_records)

print(capture_corr_df.to_string(index=False))


# ==================================================================================================
# 10.9Y — STRICT-MATCH FAILURE INSPECTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9Y — STRICT-MATCH FAILURE INSPECTION")
print("=" * 120)

strict_fail = corr_df[
    corr_df["L5_Strict_5Tuple_Matches"] == 0
].copy()

print(
    f"Sample GT rows with zero strict 5-tuple correspondence: "
    f"{len(strict_fail):,}/{len(corr_df):,}"
)

if len(strict_fail):

    print("\nFailure progression:")

    failure_progression = pd.DataFrame({
        "Stage": levels,
        "Rows_With_Zero_Matches": [
            int((corr_df[c] == 0).sum())
            for c in levels
        ]
    })

    print(failure_progression.to_string(index=False))

    print("\nExample strict failures:")

    cols_show = [
        "Episode_ID",
        "Capture_ID",
        "Attack_Family",
        "GT_Start",
        "GT_End",
        "GT_Source_IP",
        "GT_Destination_IP",
        "GT_Protocol",
        "GT_Source_Port",
        "GT_Destination_Port",
        "L1_Time_Matches",
        "L2_Time_HostPair_Matches",
        "L3_Protocol_Matches",
        "L4_DstPort_Matches",
        "L5_Strict_5Tuple_Matches"
    ]

    print(
        strict_fail[cols_show]
        .head(30)
        .to_string(index=False)
    )


# ==================================================================================================
# 10.9Z — AMBIGUITY INSPECTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9Z — STRICT-MATCH AMBIGUITY INSPECTION")
print("=" * 120)

strict_ambiguous = corr_df[
    corr_df["L5_Strict_5Tuple_Matches"] > 1
].copy()

print(
    f"Sample GT rows with >1 strict candidate flow: "
    f"{len(strict_ambiguous):,}/{len(corr_df):,}"
)

if len(strict_ambiguous):

    print(
        strict_ambiguous[
            [
                "Episode_ID",
                "Capture_ID",
                "Attack_Family",
                "GT_Start",
                "GT_End",
                "GT_Source_IP",
                "GT_Destination_IP",
                "GT_Protocol",
                "GT_Source_Port",
                "GT_Destination_Port",
                "L5_Strict_5Tuple_Matches"
            ]
        ]
        .sort_values(
            "L5_Strict_5Tuple_Matches",
            ascending=False
        )
        .head(30)
        .to_string(index=False)
    )


# ==================================================================================================
# 10.9AA — SAVE AUDIT ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9AA — SAVE AUDIT ARTIFACTS")
print("=" * 120)

paths = {

    "sample_correspondence":
        OUT_DIR / "10_9_gt_flow_sample_correspondence.csv",

    "coverage_summary":
        OUT_DIR / "10_9_correspondence_coverage_summary.csv",

    "family_coverage":
        OUT_DIR / "10_9_correspondence_by_attack_family.csv",

    "capture_coverage":
        OUT_DIR / "10_9_correspondence_by_capture.csv",

    "strict_failures":
        OUT_DIR / "10_9_strict_match_failures.csv",

    "strict_ambiguity":
        OUT_DIR / "10_9_strict_match_ambiguity.csv"
}

corr_df.to_csv(
    paths["sample_correspondence"],
    index=False
)

coverage_df.to_csv(
    paths["coverage_summary"],
    index=False
)

family_corr_df.to_csv(
    paths["family_coverage"],
    index=False
)

capture_corr_df.to_csv(
    paths["capture_coverage"],
    index=False
)

strict_fail.to_csv(
    paths["strict_failures"],
    index=False
)

strict_ambiguous.to_csv(
    paths["strict_ambiguity"],
    index=False
)

for name, path in paths.items():
    print(f"✓ {name:<25} {path}")


# ==================================================================================================
# 10.9AB — DECISION SUPPORT
#
# IMPORTANT:
# These are diagnostic indicators, NOT automatic permission to label the dataset.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.9AB — CORRESPONDENCE DIAGNOSTIC")
print("=" * 120)

summary_lookup = coverage_df.set_index("Match_Level")

hostpair_cov = summary_lookup.loc[
    "L2_Time_HostPair_Matches",
    "Matched_Pct"
]

protocol_cov = summary_lookup.loc[
    "L3_Protocol_Matches",
    "Matched_Pct"
]

dstport_cov = summary_lookup.loc[
    "L4_DstPort_Matches",
    "Matched_Pct"
]

strict_cov = summary_lookup.loc[
    "L5_Strict_5Tuple_Matches",
    "Matched_Pct"
]

print(f"Temporal + host-pair coverage : {hostpair_cov:.2f}%")
print(f"+ protocol coverage           : {protocol_cov:.2f}%")
print(f"+ destination-port coverage   : {dstport_cov:.2f}%")
print(f"Strict 5-tuple coverage       : {strict_cov:.2f}%")

print("\nInterpretation guide:")

print("""
1. If temporal coverage itself is poor:
      → timestamp/capture reconstruction is wrong or incomplete.

2. If temporal coverage is high but host-pair coverage collapses:
      → endpoint direction/address semantics need investigation.

3. If host-pair coverage is high but protocol coverage collapses:
      → protocol representation differs between GT and flows.

4. If protocol coverage is high but port matching collapses:
      → GT ports cannot safely be treated as exact flow-level identifiers.

5. If strict matching has high coverage but many candidates per GT row:
      → exact network keys are valid but temporal overlap still creates ambiguity.

6. Do NOT choose the final matching rule merely because it gives the highest coverage.
   Phase 10.10 must use the strongest rule supported by this audit without silently
   discarding legitimate GT events.
""")


# ==================================================================================================
# 10.9AC — FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 120)
print("PHASE 10.9 — FINAL STATUS")
print("=" * 120)

print(f"""
GT rows available             : {len(gt_work):,}
Candidate episodes from 10.8 : {gt_work['Episode_ID'].nunique():,}
Standardized flow rows        : {len(flows):,}
GT rows audited in sample     : {len(corr_df):,}

This phase performed a CORRESPONDENCE FEASIBILITY AUDIT only.

NO authoritative GT → flow labels have been created.
NO minute timeline has been created.
NO train/test split has been created.
NO raw UNSW files were modified.

NEXT:
Inspect the correspondence results.

If the temporal/network semantics are supported,
Phase 10.10 will construct the authoritative UNSW GT → flow mapping
with explicit conflict and ambiguity handling.
""")

print("=" * 120)
print("10.9 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.10 — AUTHORITATIVE UNSW GT → RAW FLOW ATTACK LABEL RECONSTRUCTION
# OPTIMIZED VERSION
#
# PURPOSE
# -------
# Reconstruct attack labels for the original UNSW-NB15 raw flows using:
#
#   1. exact directed 5-tuple equality
#   2. temporal overlap with ±1 second tolerance
#
# IMPORTANT
# ---------
# This version preserves the scientific logic of the previous 10.10, but removes the
# expensive construction of thousands of tiny pandas DataFrames and removes per-flow
# DataFrame filtering.
#
# Matching rule:
#
#   GT_Start <= Flow_End   + tolerance
#   GT_End   >= Flow_Start - tolerance
#
# AND exact equality of:
#
#   Source IP
#   Destination IP
#   Protocol
#   Source Port
#   Destination Port
#
# Resolution:
#
#   0 GT matches
#       -> NO_GT_MATCH
#
#   1 GT match
#       -> UNIQUE_GT_MATCH
#
#   >1 GT matches, one episode
#       -> MULTI_GT_SAME_EPISODE
#
#   >1 episodes, one family
#       -> MULTI_EPISODE_SAME_FAMILY
#
#   >1 families
#       -> CROSS_FAMILY_CONFLICT
#
# CRITICAL:
# ---------
# • No arbitrary GT record is selected.
# • Cross-family conflicts remain unresolved at family level.
# • NO_GT_MATCH is NOT yet blindly certified as BENIGN.
# • Physical CSV filenames are NOT used as semantic capture labels.
# • No minute aggregation is performed.
# • No train/val/test split is performed.
# • CIC and UNSW are NOT combined here.
#
# PERFORMANCE DESIGN
# ------------------
# • GT is normalized once.
# • GT is sorted once.
# • GT groups are stored as NumPy arrays, not thousands of DataFrames.
# • Raw files are processed one at a time.
# • Flow matching uses NumPy searchsorted interval bounds.
# • Candidate_GT_Record_ID strings are NOT stored for every flow.
# • GT/episode coverage is tracked directly with boolean arrays / sets.
# • Per-file results are saved immediately.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings
import time

warnings.filterwarnings("ignore")


# ==================================================================================================
# 10.10A — PATHS
# ==================================================================================================

print("=" * 120)
print("PHASE 10.10 — AUTHORITATIVE UNSW GT → RAW FLOW ATTACK LABEL RECONSTRUCTION — OPTIMIZED")
print("=" * 120)

BASE = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

RAW_UNSW_DIR = BASE / "raw" / "UNSW_NB15"

OUT_DIR = BASE / "audit_outputs" / "UNSW"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GT_FILE = (
    OUT_DIR /
    "10_8_unsw_gt_with_candidate_episode_ids.csv"
)

EPISODE_FILE = (
    OUT_DIR /
    "10_8_unsw_candidate_episode_table.csv"
)

FEATURE_FILE = (
    RAW_UNSW_DIR /
    "NUSW-NB15_features.csv"
)

required_files = [
    GT_FILE,
    EPISODE_FILE,
    FEATURE_FILE
]

for p in required_files:

    if not p.exists():

        raise FileNotFoundError(
            f"Required file not found:\n{p}"
        )

print("✓ Required 10.8 GT artifact found")
print("✓ Required 10.8 episode artifact found")
print("✓ UNSW feature metadata found")


# ==================================================================================================
# 10.10B — LOAD AUTHORITATIVE GT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10B — LOAD EPISODE-AWARE GT")
print("=" * 120)

gt = pd.read_csv(
    GT_FILE,
    low_memory=False
)

gt.columns = (
    gt.columns
    .astype(str)
    .str.strip()
)

required_gt = [
    "Episode_ID",
    "Capture_ID",
    "Start_DT",
    "End_DT",
    "Attack_Identity",
    "Attack_Family",
    "Source IP",
    "Destination IP",
    "Source Port",
    "Destination Port",
    "Protocol"
]

missing = [
    c for c in required_gt
    if c not in gt.columns
]

if missing:

    raise ValueError(
        "10.8 GT artifact is missing required columns:\n"
        + "\n".join(missing)
    )

gt["Start_DT"] = pd.to_datetime(
    gt["Start_DT"],
    errors="coerce"
)

gt["End_DT"] = pd.to_datetime(
    gt["End_DT"],
    errors="coerce"
)

if (
    gt["Start_DT"].isna().any()
    or
    gt["End_DT"].isna().any()
):

    raise ValueError(
        "Invalid Start_DT / End_DT found in GT."
    )

# --------------------------------------------------------------------------------------------------
# Normalize strict network fields
# --------------------------------------------------------------------------------------------------

gt["GT_Source_IP"] = (
    gt["Source IP"]
    .astype("string")
    .str.strip()
)

gt["GT_Destination_IP"] = (
    gt["Destination IP"]
    .astype("string")
    .str.strip()
)

gt["GT_Protocol"] = (
    gt["Protocol"]
    .astype("string")
    .str.strip()
    .str.lower()
)

gt["GT_Source_Port"] = pd.to_numeric(
    gt["Source Port"],
    errors="coerce"
)

gt["GT_Destination_Port"] = pd.to_numeric(
    gt["Destination Port"],
    errors="coerce"
)

# Stable GT ID BEFORE sorting.
gt = gt.reset_index(drop=True)

gt["GT_Record_ID"] = np.arange(
    len(gt),
    dtype=np.int64
)

# Original row ID and GT_Record_ID are therefore identical in this phase.
GT_N = len(gt)

print(f"GT rows          : {GT_N:,}")
print(f"Candidate episodes: {gt['Episode_ID'].nunique():,}")
print(f"Attack families  : {gt['Attack_Family'].nunique():,}")
print(f"Attack identities: {gt['Attack_Identity'].nunique():,}")
print(
    f"Time range       : "
    f"{gt['Start_DT'].min()} → {gt['End_DT'].max()}"
)


# ==================================================================================================
# 10.10C — LOAD OFFICIAL UNSW FEATURE METADATA
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10C — LOAD UNSW FEATURE METADATA")
print("=" * 120)

ENCODINGS = [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin1"
]

features = None
feature_encoding = None

for enc in ENCODINGS:

    try:

        features = pd.read_csv(
            FEATURE_FILE,
            encoding=enc,
            low_memory=False
        )

        feature_encoding = enc
        break

    except Exception:
        continue

if features is None:

    raise RuntimeError(
        "Could not read UNSW feature metadata."
    )

features.columns = (
    features.columns
    .astype(str)
    .str.strip()
)

name_col = None

for c in features.columns:

    if c.strip().lower() == "name":

        name_col = c
        break

if name_col is None:

    raise ValueError(
        "Feature-name column not found "
        "in NUSW-NB15_features.csv"
    )

feature_names = (
    features[name_col]
    .astype(str)
    .str.strip()
    .tolist()
)

print("Encoding      :", feature_encoding)
print("Feature count :", len(feature_names))

if len(feature_names) != 49:

    raise ValueError(
        f"Expected 49 UNSW features, "
        f"found {len(feature_names)}."
    )

print("✓ 49-column raw schema recovered")


# ==================================================================================================
# 10.10D — DISCOVER RAW FLOW FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10D — RAW FLOW FILE DISCOVERY")
print("=" * 120)

flow_files = sorted(
    RAW_UNSW_DIR.glob("UNSW-NB15_*.csv")
)

flow_files = [
    p for p in flow_files
    if (
        "features" not in p.name.lower()
        and
        "gt" not in p.name.lower()
        and
        "list_events" not in p.name.lower()
        and
        "training" not in p.name.lower()
        and
        "testing" not in p.name.lower()
    )
]

if not flow_files:

    raise FileNotFoundError(
        "No raw UNSW flow files found."
    )

for p in flow_files:
    print("✓", p.name)


# ==================================================================================================
# 10.10E — MEMORY-EFFICIENT RAW FLOW LOADER
#
# We only need 7 raw fields in this phase:
#
#   srcip
#   sport
#   dstip
#   dsport
#   proto
#   Stime
#   Ltime
#
# There is no reason to load all 49 raw columns just to perform GT correspondence.
# ==================================================================================================

required_raw_cols = [
    "srcip",
    "sport",
    "dstip",
    "dsport",
    "proto",
    "Stime",
    "Ltime"
]

missing_raw_schema = [
    c for c in required_raw_cols
    if c not in feature_names
]

if missing_raw_schema:

    raise ValueError(
        "Required raw UNSW columns absent from official schema:\n"
        + "\n".join(missing_raw_schema)
    )

raw_usecols = [
    feature_names.index(c)
    for c in required_raw_cols
]

raw_selected_names = [
    feature_names[i]
    for i in raw_usecols
]


def load_raw_unsw_required(path):

    last_error = None

    for enc in ENCODINGS:

        try:

            df = pd.read_csv(
                path,
                header=None,
                names=feature_names,
                usecols=raw_usecols,
                encoding=enc,
                low_memory=False
            )

            # Ensure logical order is predictable.
            df = df[required_raw_cols]

            return df, enc

        except Exception as e:

            last_error = e

    raise RuntimeError(
        f"Could not read {path.name}\n"
        f"Last error: {last_error}"
    )


# ==================================================================================================
# 10.10F — BUILD OPTIMIZED GT STRICT-KEY TEMPORAL INDEX
#
# IMPORTANT CHANGE:
#
# OLD VERSION:
#     groupby → construct DataFrame → sort → reset index
#     repeated for every strict key
#
# NEW VERSION:
#     normalize GT once
#     sort entire GT once
#     factorize strict keys once
#     store contiguous NumPy ranges
#
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10F — BUILD OPTIMIZED GT STRICT-KEY TEMPORAL INDEX")
print("=" * 120)

t0 = time.time()

GT_KEY_COLS = [
    "GT_Source_IP",
    "GT_Destination_IP",
    "GT_Protocol",
    "GT_Source_Port",
    "GT_Destination_Port"
]

# Invalid GT keys cannot participate in strict matching.
gt_valid_key = (
    gt["GT_Source_IP"].notna()
    &
    gt["GT_Destination_IP"].notna()
    &
    gt["GT_Protocol"].notna()
    &
    gt["GT_Source_Port"].notna()
    &
    gt["GT_Destination_Port"].notna()
)

invalid_gt_key_count = int(
    (~gt_valid_key).sum()
)

if invalid_gt_key_count:

    print(
        f"⚠ GT rows with invalid strict key: "
        f"{invalid_gt_key_count:,}"
    )

gt_matchable = (
    gt.loc[gt_valid_key]
    .copy()
)

# Create one compact key string.
#
# This avoids Python tuple construction for every flow during matching.
gt_matchable["_Strict_Key"] = (
    gt_matchable["GT_Source_IP"].astype(str)
    + "\x1f" +
    gt_matchable["GT_Destination_IP"].astype(str)
    + "\x1f" +
    gt_matchable["GT_Protocol"].astype(str)
    + "\x1f" +
    gt_matchable["GT_Source_Port"].astype("Int64").astype(str)
    + "\x1f" +
    gt_matchable["GT_Destination_Port"].astype("Int64").astype(str)
)

# ONE global sort.
gt_matchable = (
    gt_matchable
    .sort_values(
        [
            "_Strict_Key",
            "Start_DT",
            "End_DT",
            "GT_Record_ID"
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

# Convert timestamps to int64 nanoseconds for fast NumPy comparisons.
GT_START_NS = (
    gt_matchable["Start_DT"]
    .astype("int64")
    .to_numpy()
)

GT_END_NS = (
    gt_matchable["End_DT"]
    .astype("int64")
    .to_numpy()
)

GT_RECORD_IDS = (
    gt_matchable["GT_Record_ID"]
    .to_numpy(dtype=np.int64)
)

GT_EPISODE = (
    gt_matchable["Episode_ID"]
    .astype("string")
    .fillna("<NA>")
    .to_numpy(dtype=object)
)

GT_CAPTURE = (
    gt_matchable["Capture_ID"]
    .astype("string")
    .fillna("<NA>")
    .to_numpy(dtype=object)
)

GT_FAMILY = (
    gt_matchable["Attack_Family"]
    .astype("string")
    .fillna("<NA>")
    .to_numpy(dtype=object)
)

GT_IDENTITY = (
    gt_matchable["Attack_Identity"]
    .astype("string")
    .fillna("<NA>")
    .to_numpy(dtype=object)
)

GT_KEYS = (
    gt_matchable["_Strict_Key"]
    .to_numpy(dtype=object)
)

# Because GT is sorted by key, identical keys are contiguous.
key_change = np.empty(
    len(GT_KEYS),
    dtype=bool
)

key_change[0] = True

if len(GT_KEYS) > 1:
    key_change[1:] = (
        GT_KEYS[1:] != GT_KEYS[:-1]
    )

group_starts = np.flatnonzero(
    key_change
)

group_ends = np.r_[
    group_starts[1:],
    len(GT_KEYS)
]

unique_gt_keys = GT_KEYS[
    group_starts
]

# Compact dictionary:
#
# key -> (start position, end position)
#
# We store only integer bounds, not DataFrames.
gt_key_bounds = {
    key: (int(s), int(e))
    for key, s, e in zip(
        unique_gt_keys,
        group_starts,
        group_ends
    )
}

print(
    f"Matchable GT rows          : "
    f"{len(gt_matchable):,}"
)

print(
    f"Unique strict GT keys      : "
    f"{len(gt_key_bounds):,}"
)

print(
    f"Index construction time    : "
    f"{time.time() - t0:.2f} sec"
)

print(
    "✓ No per-key pandas DataFrames were created."
)


# ==================================================================================================
# 10.10G — GT CAPTURE TEMPORAL SUPPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10G — GT TEMPORAL CAPTURE SUPPORT")
print("=" * 120)

capture_ranges = (
    gt
    .groupby(
        "Capture_ID",
        as_index=False
    )
    .agg(
        Capture_Start=("Start_DT", "min"),
        Capture_End=("End_DT", "max")
    )
    .sort_values("Capture_Start")
    .reset_index(drop=True)
)

print(
    capture_ranges.to_string(
        index=False
    )
)

capture_ids = (
    capture_ranges["Capture_ID"]
    .astype("string")
    .to_numpy(dtype=object)
)

capture_start_ns = (
    capture_ranges["Capture_Start"]
    .astype("int64")
    .to_numpy()
)

capture_end_ns = (
    capture_ranges["Capture_End"]
    .astype("int64")
    .to_numpy()
)


# ==================================================================================================
# 10.10H — MATCHING CONFIGURATION
# ==================================================================================================

TIME_TOLERANCE_SEC = 1

TOL_NS = int(
    TIME_TOLERANCE_SEC
    * 1_000_000_000
)

print("\n" + "=" * 120)
print("10.10H — AUTHORITATIVE MATCHING RULE")
print("=" * 120)

print(
    "Directed strict key:"
    "\n  Source IP"
    "\n  Destination IP"
    "\n  Protocol"
    "\n  Source Port"
    "\n  Destination Port"
)

print(
    f"\nTemporal tolerance: "
    f"±{TIME_TOLERANCE_SEC} second"
)

print(
    "\nTemporal rule:"
    "\n  GT_Start <= Flow_End + tolerance"
    "\n  GT_End   >= Flow_Start - tolerance"
)

print(
    "\nNo arbitrary single-match selection."
)


# ==================================================================================================
# 10.10I — GLOBAL COVERAGE TRACKERS
# ==================================================================================================

# Boolean array is dramatically cheaper than a Python set of hundreds of thousands of ints.
matched_gt_mask = np.zeros(
    GT_N,
    dtype=bool
)

matched_episode_ids = set()

file_audit_records = []

per_file_outputs = []


# ==================================================================================================
# 10.10J — PROCESS RAW FLOWS FILE BY FILE
#
# Main optimization:
#
# For each strict key:
#
#   GT records are already sorted by Start_DT.
#
# For each flow:
#
#   searchsorted gives us the GT prefix satisfying:
#
#       GT_Start <= Flow_End + tolerance
#
# Then we only test:
#
#       GT_End >= Flow_Start - tolerance
#
# inside that prefix.
#
# This is exact interval-overlap logic; it does NOT approximate the matching rule.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10J — FULL RAW FLOW ↔ GT RECONSTRUCTION")
print("=" * 120)

for file_number, path in enumerate(
    flow_files,
    start=1
):

    file_t0 = time.time()

    print("\n" + "-" * 120)

    print(
        f"[{file_number}/{len(flow_files)}] "
        f"Processing {path.name}"
    )

    print("-" * 120)

    df, enc = load_raw_unsw_required(
        path
    )

    n = len(df)

    print(f"Rows     : {n:,}")
    print(f"Encoding : {enc}")

    # ----------------------------------------------------------------------------------------------
    # Standardized output
    # ----------------------------------------------------------------------------------------------

    work = pd.DataFrame(
        index=np.arange(n)
    )

    work["Flow_File"] = path.name

    work["Flow_Row_Index"] = np.arange(
        n,
        dtype=np.int64
    )

    work["Source_IP"] = (
        df["srcip"]
        .astype("string")
        .str.strip()
    )

    work["Destination_IP"] = (
        df["dstip"]
        .astype("string")
        .str.strip()
    )

    work["Protocol"] = (
        df["proto"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    work["Source_Port"] = pd.to_numeric(
        df["sport"],
        errors="coerce"
    )

    work["Destination_Port"] = pd.to_numeric(
        df["dsport"],
        errors="coerce"
    )

    # Unix seconds → datetime
    flow_start_num = pd.to_numeric(
        df["Stime"],
        errors="coerce"
    )

    flow_end_num = pd.to_numeric(
        df["Ltime"],
        errors="coerce"
    )

    work["Flow_Start"] = pd.to_datetime(
        flow_start_num,
        unit="s",
        errors="coerce"
    )

    work["Flow_End"] = pd.to_datetime(
        flow_end_num,
        unit="s",
        errors="coerce"
    )

    del df
    gc.collect()

    # ----------------------------------------------------------------------------------------------
    # Integrity flags
    # ----------------------------------------------------------------------------------------------

    invalid_timestamp = (
        work["Flow_Start"].isna()
        |
        work["Flow_End"].isna()
    ).to_numpy()

    reversed_timestamp = (
        work["Flow_Start"].notna()
        &
        work["Flow_End"].notna()
        &
        (
            work["Flow_End"]
            <
            work["Flow_Start"]
        )
    ).to_numpy()

    if reversed_timestamp.any():

        print(
            "⚠ Reversed flow timestamps:",
            f"{reversed_timestamp.sum():,}"
        )

    invalid_network_key = (
        work["Source_IP"].isna()
        |
        work["Destination_IP"].isna()
        |
        work["Protocol"].isna()
        |
        work["Source_Port"].isna()
        |
        work["Destination_Port"].isna()
    ).to_numpy()

    # ----------------------------------------------------------------------------------------------
    # Compact strict key for flows
    # ----------------------------------------------------------------------------------------------

    # Nullable integer representation keeps 80 and 80.0 equivalent.
    sport_text = (
        work["Source_Port"]
        .astype("Int64")
        .astype("string")
    )

    dport_text = (
        work["Destination_Port"]
        .astype("Int64")
        .astype("string")
    )

    flow_keys = (
        work["Source_IP"]
        + "\x1f" +
        work["Destination_IP"]
        + "\x1f" +
        work["Protocol"]
        + "\x1f" +
        sport_text
        + "\x1f" +
        dport_text
    )

    # ----------------------------------------------------------------------------------------------
    # Timestamp arrays in nanoseconds
    # ----------------------------------------------------------------------------------------------

    flow_start_ns = (
        work["Flow_Start"]
        .astype("int64")
        .to_numpy()
    )

    flow_end_ns = (
        work["Flow_End"]
        .astype("int64")
        .to_numpy()
    )

    # ----------------------------------------------------------------------------------------------
    # Semantic capture inference — vectorized
    #
    # We infer capture support from time, not physical file name.
    # ----------------------------------------------------------------------------------------------

    temporal_capture = np.full(
        n,
        None,
        dtype=object
    )

    capture_match_count = np.zeros(
        n,
        dtype=np.int8
    )

    valid_time_mask = ~invalid_timestamp

    for cap_id, cap_start, cap_end in zip(
        capture_ids,
        capture_start_ns,
        capture_end_ns
    ):

        m = (
            valid_time_mask
            &
            (cap_start <= flow_end_ns)
            &
            (cap_end >= flow_start_ns)
        )

        capture_match_count[m] += 1

        # First compatible capture.
        first = (
            m
            &
            pd.isna(temporal_capture)
        )

        temporal_capture[first] = cap_id

    multi_capture = (
        capture_match_count > 1
    )

    temporal_capture[
        multi_capture
    ] = "MULTIPLE_CAPTURE_OVERLAP"

    work["Temporal_Capture_ID"] = (
        temporal_capture
    )

    # ----------------------------------------------------------------------------------------------
    # Prepare output arrays
    # ----------------------------------------------------------------------------------------------

    attack_state = np.zeros(
        n,
        dtype=np.int8
    )

    mapping_status = np.full(
        n,
        "NO_GT_MATCH",
        dtype=object
    )

    mapping_status[
        invalid_network_key
    ] = "INVALID_NETWORK_KEY"

    mapping_status[
        (~invalid_network_key)
        &
        invalid_timestamp
    ] = "INVALID_TIMESTAMP"

    candidate_gt_count = np.zeros(
        n,
        dtype=np.int32
    )

    candidate_family_count = np.zeros(
        n,
        dtype=np.int16
    )

    candidate_episode_count = np.zeros(
        n,
        dtype=np.int32
    )

    candidate_identity_count = np.zeros(
        n,
        dtype=np.int16
    )

    resolved_family = np.full(
        n,
        None,
        dtype=object
    )

    resolved_episode = np.full(
        n,
        None,
        dtype=object
    )

    resolved_identity = np.full(
        n,
        None,
        dtype=object
    )

    resolved_capture = np.full(
        n,
        None,
        dtype=object
    )

    # Candidate text is useful for ambiguity auditing.
    #
    # We retain family/episode/identity candidates, but intentionally DO NOT
    # create Candidate_GT_Record_IDs for every flow.
    candidate_families_text = np.full(
        n,
        None,
        dtype=object
    )

    candidate_episodes_text = np.full(
        n,
        None,
        dtype=object
    )

    candidate_identities_text = np.full(
        n,
        None,
        dtype=object
    )

    # ----------------------------------------------------------------------------------------------
    # Group flow ROW POSITIONS by compact strict key.
    #
    # factorize is substantially cheaper than creating a pandas DataFrame per key.
    # ----------------------------------------------------------------------------------------------

    valid_for_matching = (
        ~invalid_network_key
        &
        ~invalid_timestamp
    )

    valid_positions = np.flatnonzero(
        valid_for_matching
    )

    valid_keys = (
        flow_keys.iloc[
            valid_positions
        ]
        .astype(str)
        .to_numpy(dtype=object)
    )

    flow_codes, flow_unique_keys = pd.factorize(
        valid_keys,
        sort=False
    )

    # Sort positions by code once so rows belonging to each key become contiguous.
    order = np.argsort(
        flow_codes,
        kind="stable"
    )

    sorted_codes = flow_codes[
        order
    ]

    sorted_positions = valid_positions[
        order
    ]

    if len(sorted_codes):

        code_change = np.empty(
            len(sorted_codes),
            dtype=bool
        )

        code_change[0] = True

        if len(sorted_codes) > 1:

            code_change[1:] = (
                sorted_codes[1:]
                !=
                sorted_codes[:-1]
            )

        flow_group_starts = np.flatnonzero(
            code_change
        )

        flow_group_ends = np.r_[
            flow_group_starts[1:],
            len(sorted_codes)
        ]

    else:

        flow_group_starts = np.array(
            [],
            dtype=np.int64
        )

        flow_group_ends = np.array(
            [],
            dtype=np.int64
        )

    total_groups = len(
        flow_group_starts
    )

    print(
        f"Valid strict flow keys: "
        f"{total_groups:,}"
    )

    # ----------------------------------------------------------------------------------------------
    # Exact matching
    # ----------------------------------------------------------------------------------------------

    groups_with_gt_key = 0
    flows_tested_against_gt = 0

    for group_number, (
        gs,
        ge
    ) in enumerate(
        zip(
            flow_group_starts,
            flow_group_ends
        ),
        start=1
    ):

        # All rows in this range share one factorized code.
        code = int(
            sorted_codes[gs]
        )

        key = flow_unique_keys[
            code
        ]

        bounds = gt_key_bounds.get(
            key
        )

        # No GT ever uses this exact directed 5-tuple.
        if bounds is None:
            continue

        groups_with_gt_key += 1

        gt_s, gt_e = bounds

        gt_starts = GT_START_NS[
            gt_s:gt_e
        ]

        gt_ends = GT_END_NS[
            gt_s:gt_e
        ]

        group_flow_positions = (
            sorted_positions[
                gs:ge
            ]
        )

        flows_tested_against_gt += len(
            group_flow_positions
        )

        # ==========================================================================================
        # Match each flow within this exact network key.
        #
        # IMPORTANT:
        # This is no longer pandas iterrows + DataFrame filtering.
        #
        # searchsorted finds the prefix of GT records whose Start_DT can overlap the flow.
        # We then apply the second interval condition on a NumPy array.
        # ==========================================================================================

        for flow_idx in group_flow_positions:

            fs = flow_start_ns[
                flow_idx
            ]

            fe = flow_end_ns[
                flow_idx
            ]

            # Condition 1:
            #
            # GT_Start <= Flow_End + tolerance
            #
            # Since gt_starts is sorted:
            upper = np.searchsorted(
                gt_starts,
                fe + TOL_NS,
                side="right"
            )

            if upper == 0:
                continue

            # Condition 2:
            #
            # GT_End >= Flow_Start - tolerance
            overlap_local = np.flatnonzero(
                gt_ends[:upper]
                >=
                fs - TOL_NS
            )

            if overlap_local.size == 0:
                continue

            match_positions = (
                gt_s
                +
                overlap_local
            )

            gt_count = int(
                match_positions.size
            )

            families = np.unique(
                GT_FAMILY[
                    match_positions
                ]
            )

            families = families[
                families != "<NA>"
            ]

            episodes = np.unique(
                GT_EPISODE[
                    match_positions
                ]
            )

            episodes = episodes[
                episodes != "<NA>"
            ]

            identities = np.unique(
                GT_IDENTITY[
                    match_positions
                ]
            )

            identities = identities[
                identities != "<NA>"
            ]

            captures = np.unique(
                GT_CAPTURE[
                    match_positions
                ]
            )

            captures = captures[
                captures != "<NA>"
            ]

            # --------------------------------------------------------------------------------------
            # Counts
            # --------------------------------------------------------------------------------------

            candidate_gt_count[
                flow_idx
            ] = gt_count

            candidate_family_count[
                flow_idx
            ] = len(families)

            candidate_episode_count[
                flow_idx
            ] = len(episodes)

            candidate_identity_count[
                flow_idx
            ] = len(identities)

            # --------------------------------------------------------------------------------------
            # Compact ambiguity metadata
            # --------------------------------------------------------------------------------------

            if len(families):

                candidate_families_text[
                    flow_idx
                ] = "|".join(
                    map(str, families)
                )

            if len(episodes):

                candidate_episodes_text[
                    flow_idx
                ] = "|".join(
                    map(str, episodes)
                )

            if len(identities):

                candidate_identities_text[
                    flow_idx
                ] = "|".join(
                    map(str, identities)
                )

            # --------------------------------------------------------------------------------------
            # Register full GT coverage
            # --------------------------------------------------------------------------------------

            matched_ids = GT_RECORD_IDS[
                match_positions
            ]

            matched_gt_mask[
                matched_ids
            ] = True

            matched_episode_ids.update(
                episodes.tolist()
            )

            # --------------------------------------------------------------------------------------
            # Attack-active state
            # --------------------------------------------------------------------------------------

            attack_state[
                flow_idx
            ] = 1

            # --------------------------------------------------------------------------------------
            # Family resolution
            # --------------------------------------------------------------------------------------

            if len(families) == 1:

                resolved_family[
                    flow_idx
                ] = families[0]

                if len(episodes) == 1:

                    resolved_episode[
                        flow_idx
                    ] = episodes[0]

                if len(identities) == 1:

                    resolved_identity[
                        flow_idx
                    ] = identities[0]

                if len(captures) == 1:

                    resolved_capture[
                        flow_idx
                    ] = captures[0]

                # Status hierarchy
                if (
                    gt_count == 1
                    and
                    len(episodes) == 1
                    and
                    len(identities) == 1
                ):

                    mapping_status[
                        flow_idx
                    ] = "UNIQUE_GT_MATCH"

                elif len(episodes) == 1:

                    mapping_status[
                        flow_idx
                    ] = "MULTI_GT_SAME_EPISODE"

                else:

                    mapping_status[
                        flow_idx
                    ] = "MULTI_EPISODE_SAME_FAMILY"

            else:

                # We know attack GT overlaps the flow, but family is ambiguous.
                mapping_status[
                    flow_idx
                ] = "CROSS_FAMILY_CONFLICT"

        if (
            group_number % 10000
            ==
            0
        ):

            print(
                f"  processed "
                f"{group_number:,}/"
                f"{total_groups:,} strict keys"
            )

    # ----------------------------------------------------------------------------------------------
    # Attach results
    # ----------------------------------------------------------------------------------------------

    work["Attack_State"] = (
        attack_state
    )

    work["Mapping_Status"] = (
        mapping_status
    )

    work["Candidate_GT_Count"] = (
        candidate_gt_count
    )

    work["Candidate_Family_Count"] = (
        candidate_family_count
    )

    work["Candidate_Episode_Count"] = (
        candidate_episode_count
    )

    work["Candidate_Identity_Count"] = (
        candidate_identity_count
    )

    work["Attack_Family"] = (
        resolved_family
    )

    work["Episode_ID"] = (
        resolved_episode
    )

    work["Attack_Identity"] = (
        resolved_identity
    )

    work["GT_Capture_ID"] = (
        resolved_capture
    )

    work["Candidate_Families"] = (
        candidate_families_text
    )

    work["Candidate_Episodes"] = (
        candidate_episodes_text
    )

    work["Candidate_Identities"] = (
        candidate_identities_text
    )

    # ----------------------------------------------------------------------------------------------
    # Capture consistency
    # ----------------------------------------------------------------------------------------------

    capture_consistency = np.full(
        n,
        "NOT_APPLICABLE",
        dtype=object
    )

    resolved_cap_mask = pd.notna(
        resolved_capture
    )

    capture_consistency[
        resolved_cap_mask
        &
        (
            temporal_capture
            ==
            resolved_capture
        )
    ] = "CONSISTENT"

    capture_consistency[
        resolved_cap_mask
        &
        (
            temporal_capture
            !=
            resolved_capture
        )
    ] = "MISMATCH"

    work["Capture_Consistency"] = (
        capture_consistency
    )

    # ----------------------------------------------------------------------------------------------
    # File audit
    # ----------------------------------------------------------------------------------------------

    status_counts = (
        work["Mapping_Status"]
        .value_counts(
            dropna=False
        )
    )

    file_audit_records.append({

        "Flow_File":
            path.name,

        "Rows":
            n,

        "Strict_Keys":
            total_groups,

        "Strict_Keys_With_GT":
            groups_with_gt_key,

        "Flows_Tested_Against_GT":
            flows_tested_against_gt,

        "Attack_Matched_Flows":
            int(
                attack_state.sum()
            ),

        "No_GT_Match":
            int(
                status_counts.get(
                    "NO_GT_MATCH",
                    0
                )
            ),

        "Unique_GT_Match":
            int(
                status_counts.get(
                    "UNIQUE_GT_MATCH",
                    0
                )
            ),

        "Multi_GT_Same_Episode":
            int(
                status_counts.get(
                    "MULTI_GT_SAME_EPISODE",
                    0
                )
            ),

        "Multi_Episode_Same_Family":
            int(
                status_counts.get(
                    "MULTI_EPISODE_SAME_FAMILY",
                    0
                )
            ),

        "Cross_Family_Conflict":
            int(
                status_counts.get(
                    "CROSS_FAMILY_CONFLICT",
                    0
                )
            ),

        "Invalid_Network_Key":
            int(
                status_counts.get(
                    "INVALID_NETWORK_KEY",
                    0
                )
            ),

        "Invalid_Timestamp":
            int(
                status_counts.get(
                    "INVALID_TIMESTAMP",
                    0
                )
            ),

        "Capture_Mismatch":
            int(
                (
                    work[
                        "Capture_Consistency"
                    ]
                    ==
                    "MISMATCH"
                ).sum()
            ),

        "Runtime_Sec":
            time.time() - file_t0
    })

    print("\nMapping status:")

    print(
        work[
            "Mapping_Status"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )

    print(
        "\nAttack-matched flows:",
        f"{attack_state.sum():,}"
    )

    print(
        "Strict keys with GT support:",
        f"{groups_with_gt_key:,}/"
        f"{total_groups:,}"
    )

    # ----------------------------------------------------------------------------------------------
    # Save per-file result immediately
    # ----------------------------------------------------------------------------------------------

    per_file_output = (
        OUT_DIR /
        f"10_10_mapping_{path.stem}.csv"
    )

    work.to_csv(
        per_file_output,
        index=False
    )

    per_file_outputs.append(
        per_file_output
    )

    print(
        "\n✓ Saved:",
        per_file_output
    )

    print(
        f"✓ File runtime: "
        f"{time.time() - file_t0:.2f} sec"
    )

    # ----------------------------------------------------------------------------------------------
    # Clean memory BEFORE next raw file
    # ----------------------------------------------------------------------------------------------

    del work
    del flow_keys
    del flow_codes
    del flow_unique_keys
    del sorted_codes
    del sorted_positions
    del flow_start_ns
    del flow_end_ns
    del temporal_capture
    del candidate_families_text
    del candidate_episodes_text
    del candidate_identities_text

    gc.collect()


# ==================================================================================================
# 10.10K — COMBINE SAVED STANDARDIZED TABLES
#
# IMPORTANT:
# We deliberately re-read the compact saved outputs instead of retaining four
# large DataFrames in RAM during matching.
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10K — COMBINE AUTHORITATIVE FLOW MAPPING")
print("=" * 120)

mapping_parts = []

for p in per_file_outputs:

    print(
        "Loading:",
        p.name
    )

    part = pd.read_csv(
        p,
        low_memory=False
    )

    mapping_parts.append(
        part
    )

mapping = pd.concat(
    mapping_parts,
    ignore_index=True
)

del mapping_parts
gc.collect()

# Restore datetime types after CSV reload.
mapping["Flow_Start"] = pd.to_datetime(
    mapping["Flow_Start"],
    errors="coerce"
)

mapping["Flow_End"] = pd.to_datetime(
    mapping["Flow_End"],
    errors="coerce"
)

print(
    f"Total standardized flows: "
    f"{len(mapping):,}"
)

print(
    f"Attack-matched flows     : "
    f"{mapping['Attack_State'].sum():,}"
)

print(
    f"Non-matched flows        : "
    f"{(mapping['Attack_State'] == 0).sum():,}"
)


# ==================================================================================================
# 10.10L — GLOBAL MAPPING STATUS AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10L — GLOBAL MAPPING STATUS AUDIT")
print("=" * 120)

status_summary = (
    mapping[
        "Mapping_Status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Mapping_Status"
    )
    .reset_index(
        name="Flows"
    )
)

status_summary[
    "Pct_All_Flows"
] = (
    100
    *
    status_summary["Flows"]
    /
    len(mapping)
)

attack_flow_count = int(
    mapping[
        "Attack_State"
    ].sum()
)

status_summary[
    "Pct_Attack_Matched_Flows"
] = np.where(

    attack_flow_count > 0,

    100
    *
    status_summary["Flows"]
    /
    attack_flow_count,

    np.nan
)

print(
    status_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.10M — ATTACK FAMILY DISTRIBUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10M — RESOLVED ATTACK FAMILY DISTRIBUTION")
print("=" * 120)

family_distribution = (
    mapping[
        mapping[
            "Attack_Family"
        ].notna()
    ]
    .groupby(
        "Attack_Family",
        dropna=False
    )
    .size()
    .rename(
        "Mapped_Flows"
    )
    .reset_index()
    .sort_values(
        "Mapped_Flows",
        ascending=False
    )
)

print(
    family_distribution.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.10N — CROSS-FAMILY CONFLICT AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10N — CROSS-FAMILY CONFLICT AUDIT")
print("=" * 120)

family_conflicts = mapping[
    mapping[
        "Mapping_Status"
    ]
    ==
    "CROSS_FAMILY_CONFLICT"
].copy()

print(
    f"Cross-family conflicting flows: "
    f"{len(family_conflicts):,}"
)

if len(family_conflicts):

    conflict_patterns = (
        family_conflicts
        .groupby(
            "Candidate_Families",
            dropna=False
        )
        .size()
        .rename(
            "Flows"
        )
        .reset_index()
        .sort_values(
            "Flows",
            ascending=False
        )
    )

    print(
        "\nMost common conflict patterns:"
    )

    print(
        conflict_patterns
        .head(30)
        .to_string(
            index=False
        )
    )

    print(
        "\nExample conflicting flows:"
    )

    show_cols = [
        "Flow_File",
        "Flow_Row_Index",
        "Flow_Start",
        "Flow_End",
        "Source_IP",
        "Destination_IP",
        "Protocol",
        "Source_Port",
        "Destination_Port",
        "Candidate_GT_Count",
        "Candidate_Family_Count",
        "Candidate_Families",
        "Candidate_Episodes"
    ]

    print(
        family_conflicts[
            show_cols
        ]
        .head(30)
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 10.10O — EPISODE AMBIGUITY AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10O — EPISODE AMBIGUITY AUDIT")
print("=" * 120)

episode_ambiguity = mapping[
    (
        mapping[
            "Attack_State"
        ]
        ==
        1
    )
    &
    (
        mapping[
            "Candidate_Episode_Count"
        ]
        >
        1
    )
].copy()

print(
    f"Attack flows compatible with >1 candidate episode: "
    f"{len(episode_ambiguity):,}"
)

if attack_flow_count > 0:

    print(
        f"Percentage of attack-matched flows: "
        f"{100 * len(episode_ambiguity) / attack_flow_count:.4f}%"
    )


# ==================================================================================================
# 10.10P — FULL GT RECORD COVERAGE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10P — FULL GT RECORD COVERAGE")
print("=" * 120)

all_gt_ids = np.arange(
    GT_N,
    dtype=np.int64
)

unmatched_gt_ids = all_gt_ids[
    ~matched_gt_mask
]

unmatched_gt = gt[
    gt[
        "GT_Record_ID"
    ].isin(
        unmatched_gt_ids
    )
].copy()

matched_gt_count = int(
    matched_gt_mask.sum()
)

print(
    f"GT records total    : "
    f"{GT_N:,}"
)

print(
    f"GT records matched  : "
    f"{matched_gt_count:,}"
)

print(
    f"GT records unmatched: "
    f"{len(unmatched_gt):,}"
)

gt_coverage_pct = (
    100
    *
    matched_gt_count
    /
    GT_N
)

print(
    f"GT coverage         : "
    f"{gt_coverage_pct:.4f}%"
)

if len(unmatched_gt):

    print(
        "\nUnmatched GT distribution "
        "by attack family:"
    )

    print(
        unmatched_gt[
            "Attack_Family"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


# ==================================================================================================
# 10.10Q — EPISODE COVERAGE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10Q — EPISODE COVERAGE")
print("=" * 120)

all_episode_ids = set(
    gt[
        "Episode_ID"
    ]
    .dropna()
    .astype(str)
    .tolist()
)

# Ensure identical representation.
matched_episode_ids = set(
    map(
        str,
        matched_episode_ids
    )
)

unmatched_episode_ids = (
    all_episode_ids
    -
    matched_episode_ids
)

matched_episode_count = (
    len(all_episode_ids)
    -
    len(unmatched_episode_ids)
)

print(
    f"Candidate episodes total              : "
    f"{len(all_episode_ids):,}"
)

print(
    f"Episodes represented by ≥1 mapped flow: "
    f"{matched_episode_count:,}"
)

print(
    f"Episodes with zero mapped flows        : "
    f"{len(unmatched_episode_ids):,}"
)

if len(all_episode_ids):

    print(
        f"Episode coverage                       : "
        f"{100 * matched_episode_count / len(all_episode_ids):.4f}%"
    )


# ==================================================================================================
# 10.10R — TEMPORAL CAPTURE CONSISTENCY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10R — GT CAPTURE ↔ FLOW-TIME CONSISTENCY")
print("=" * 120)

capture_consistency = (
    mapping[
        mapping[
            "GT_Capture_ID"
        ].notna()
    ][
        "Capture_Consistency"
    ]
    .value_counts(
        dropna=False
    )
)

print(
    capture_consistency.to_string()
)

capture_mismatch = mapping[
    mapping[
        "Capture_Consistency"
    ]
    ==
    "MISMATCH"
].copy()

if len(capture_mismatch):

    print(
        "\n⚠ Capture mismatches detected."
    )

    print(
        capture_mismatch[
            [
                "Flow_File",
                "Flow_Row_Index",
                "Flow_Start",
                "Flow_End",
                "Temporal_Capture_ID",
                "GT_Capture_ID",
                "Attack_Family",
                "Episode_ID"
            ]
        ]
        .head(30)
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 10.10S — BENIGN-LABEL SAFETY AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10S — BENIGN-LABEL SAFETY CHECK")
print("=" * 120)

cross_family_count = len(
    family_conflicts
)

capture_mismatch_count = len(
    capture_mismatch
)

print(
    f"Full GT record coverage       : "
    f"{gt_coverage_pct:.4f}%"
)

print(
    f"Cross-family conflicting flows: "
    f"{cross_family_count:,}"
)

print(
    f"Capture mismatches            : "
    f"{capture_mismatch_count:,}"
)

if gt_coverage_pct == 100:

    print(
        "\n✓ Every GT attack record has at least "
        "one compatible raw flow."
    )

else:

    print(
        "\n⚠ Some GT attack records remain unmatched."
    )

if cross_family_count == 0:

    print(
        "✓ No cross-family flow conflicts detected."
    )

else:

    print(
        "⚠ Cross-family conflicts exist and remain "
        "UNRESOLVED by design."
    )

print(
    "\nIMPORTANT: NO_GT_MATCH is still NOT automatically "
    "certified as benign in Phase 10.10."
)


# ==================================================================================================
# 10.10T — SAVE AUTHORITATIVE ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10T — SAVE AUTHORITATIVE ARTIFACTS")
print("=" * 120)

OUTPUTS = {

    "full_mapping":
        OUT_DIR /
        "10_10_unsw_authoritative_flow_mapping.csv",

    "mapping_status":
        OUT_DIR /
        "10_10_mapping_status_summary.csv",

    "file_audit":
        OUT_DIR /
        "10_10_file_mapping_audit.csv",

    "family_distribution":
        OUT_DIR /
        "10_10_resolved_attack_family_distribution.csv",

    "family_conflicts":
        OUT_DIR /
        "10_10_cross_family_conflicts.csv",

    "episode_ambiguity":
        OUT_DIR /
        "10_10_episode_ambiguity.csv",

    "unmatched_gt":
        OUT_DIR /
        "10_10_unmatched_gt_records.csv",

    "capture_mismatch":
        OUT_DIR /
        "10_10_capture_mismatches.csv"
}

mapping.to_csv(
    OUTPUTS[
        "full_mapping"
    ],
    index=False
)

status_summary.to_csv(
    OUTPUTS[
        "mapping_status"
    ],
    index=False
)

pd.DataFrame(
    file_audit_records
).to_csv(
    OUTPUTS[
        "file_audit"
    ],
    index=False
)

family_distribution.to_csv(
    OUTPUTS[
        "family_distribution"
    ],
    index=False
)

family_conflicts.to_csv(
    OUTPUTS[
        "family_conflicts"
    ],
    index=False
)

episode_ambiguity.to_csv(
    OUTPUTS[
        "episode_ambiguity"
    ],
    index=False
)

unmatched_gt.to_csv(
    OUTPUTS[
        "unmatched_gt"
    ],
    index=False
)

capture_mismatch.to_csv(
    OUTPUTS[
        "capture_mismatch"
    ],
    index=False
)

for name, path in OUTPUTS.items():

    print(
        f"✓ {name:<22} {path}"
    )


# ==================================================================================================
# 10.10U — FINAL CONSERVATION / SANITY CHECKS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.10U — FINAL SANITY CHECKS")
print("=" * 120)

# --------------------------------------------------------------------------------------------------
# 1. Physical row conservation
# --------------------------------------------------------------------------------------------------

expected_rows = sum(
    r["Rows"]
    for r in file_audit_records
)

if len(mapping) != expected_rows:

    raise RuntimeError(
        "FLOW ROW CONSERVATION FAILED:\n"
        f"Expected: {expected_rows:,}\n"
        f"Observed: {len(mapping):,}"
    )

print(
    f"✓ Flow row conservation: "
    f"{len(mapping):,}"
)

# --------------------------------------------------------------------------------------------------
# 2. Unique physical flow identity
# --------------------------------------------------------------------------------------------------

duplicate_flow_identity = mapping.duplicated(
    subset=[
        "Flow_File",
        "Flow_Row_Index"
    ]
).sum()

if duplicate_flow_identity:

    raise RuntimeError(
        f"Duplicate physical flow identities detected: "
        f"{duplicate_flow_identity:,}"
    )

print(
    "✓ Every Flow_File + Flow_Row_Index pair is unique"
)

# --------------------------------------------------------------------------------------------------
# 3. Attack-state consistency
# --------------------------------------------------------------------------------------------------

bad_attack_state = mapping[
    (
        mapping[
            "Attack_State"
        ]
        ==
        1
    )
    &
    (
        mapping[
            "Candidate_GT_Count"
        ]
        ==
        0
    )
]

if len(bad_attack_state):

    raise RuntimeError(
        "Attack_State=1 exists with "
        "Candidate_GT_Count=0."
    )

print(
    "✓ Every Attack_State=1 flow has "
    "≥1 GT candidate"
)

# --------------------------------------------------------------------------------------------------
# 4. Reverse consistency:
#    any flow with GT candidates must be attack-active
# --------------------------------------------------------------------------------------------------

bad_candidate_state = mapping[
    (
        mapping[
            "Candidate_GT_Count"
        ]
        >
        0
    )
    &
    (
        mapping[
            "Attack_State"
        ]
        !=
        1
    )
]

if len(bad_candidate_state):

    raise RuntimeError(
        "GT candidates exist on flows not marked Attack_State=1."
    )

print(
    "✓ Every flow with ≥1 GT candidate is Attack_State=1"
)

# --------------------------------------------------------------------------------------------------
# 5. Family-label consistency
# --------------------------------------------------------------------------------------------------

bad_family = mapping[
    (
        mapping[
            "Attack_Family"
        ].notna()
    )
    &
    (
        mapping[
            "Candidate_Family_Count"
        ]
        !=
        1
    )
]

if len(bad_family):

    raise RuntimeError(
        "Resolved Attack_Family exists "
        "without exactly one supported family."
    )

print(
    "✓ Attack family is resolved only "
    "when exactly one family is supported"
)

# --------------------------------------------------------------------------------------------------
# 6. Conflict safety
# --------------------------------------------------------------------------------------------------

bad_conflict = mapping[
    (
        mapping[
            "Mapping_Status"
        ]
        ==
        "CROSS_FAMILY_CONFLICT"
    )
    &
    (
        mapping[
            "Attack_Family"
        ].notna()
    )
]

if len(bad_conflict):

    raise RuntimeError(
        "Cross-family conflicts were "
        "incorrectly assigned a family."
    )

print(
    "✓ Cross-family conflicts remain unlabeled "
    "at family level"
)

# --------------------------------------------------------------------------------------------------
# 7. Status ↔ candidate-count consistency
# --------------------------------------------------------------------------------------------------

bad_no_match = mapping[
    (
        mapping[
            "Mapping_Status"
        ]
        ==
        "NO_GT_MATCH"
    )
    &
    (
        mapping[
            "Candidate_GT_Count"
        ]
        !=
        0
    )
]

if len(bad_no_match):

    raise RuntimeError(
        "NO_GT_MATCH rows exist with GT candidates."
    )

print(
    "✓ NO_GT_MATCH rows have zero GT candidates"
)

# --------------------------------------------------------------------------------------------------
# 8. Unique GT status consistency
# --------------------------------------------------------------------------------------------------

bad_unique = mapping[
    (
        mapping[
            "Mapping_Status"
        ]
        ==
        "UNIQUE_GT_MATCH"
    )
    &
    (
        mapping[
            "Candidate_GT_Count"
        ]
        !=
        1
    )
]

if len(bad_unique):

    raise RuntimeError(
        "UNIQUE_GT_MATCH exists with Candidate_GT_Count != 1."
    )

print(
    "✓ UNIQUE_GT_MATCH means exactly one GT candidate"
)


# ==================================================================================================
# 10.10V — FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 120)
print("PHASE 10.10 — FINAL STATUS")
print("=" * 120)

print(
    f"""
Raw UNSW flows reconstructed : {len(mapping):,}

Attack-matched flows         : {attack_flow_count:,}

GT records                   : {GT_N:,}
GT records matched           : {matched_gt_count:,}
GT records unmatched         : {len(unmatched_gt):,}
GT record coverage           : {gt_coverage_pct:.4f}%

Candidate episodes           : {len(all_episode_ids):,}
Episodes represented         : {matched_episode_count:,}
Episodes unmatched           : {len(unmatched_episode_ids):,}

Cross-family conflicts       : {len(family_conflicts):,}
Multi-episode attack flows   : {len(episode_ambiguity):,}
Capture mismatches           : {len(capture_mismatch):,}


IMPORTANT:

• Matching uses the exact directed 5-tuple.

• Matching uses exact interval overlap with ±{TIME_TOLERANCE_SEC} second tolerance.

• NO arbitrary GT candidate was selected.

• Multiple GT records belonging to the SAME attack family may support
  the same raw flow.

• Multiple episodes belonging to the same family remain explicitly
  represented as episode ambiguity.

• Cross-family conflicts remain unresolved.

• Physical CSV filenames were NOT used as authoritative semantic
  capture labels.

• Candidate_GT_Record_ID strings were intentionally NOT materialized
  for every raw flow. Full GT coverage was tracked directly instead.

• NO_GT_MATCH has NOT yet been blindly certified as BENIGN.

• No minute aggregation has been performed.

• No train/validation/test split has been performed.

• CIC and UNSW have NOT yet been combined.


NEXT PHASE — 10.11

Validate the reconstructed UNSW attack mapping before converting
NO_GT_MATCH traffic into authoritative benign traffic and before
constructing the UNSW canonical minute timeline.
"""
)

print("=" * 120)
print("10.10 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 10.11 — UNSW AUTHORITATIVE LABEL RESOLUTION & BENIGN CERTIFICATION AUDIT
# ==================================================================================================
#
# PURPOSE
# -------
# Convert the Phase 10.10 GT→raw-flow reconstruction into a defensible
# family-level labeling policy WITHOUT silently forcing ambiguous flows.
#
# THIS PHASE ANSWERS:
#
#   1. Which GT-matched flows can safely receive an attack-family label?
#   2. Which flows remain cross-family ambiguous?
#   3. Can MULTI_GT_SAME_EPISODE be resolved safely?
#   4. Can MULTI_EPISODE_SAME_FAMILY be resolved safely?
#   5. Do original UNSW raw label fields support the reconstructed labels?
#   6. Can NO_GT_MATCH flows be certified as benign?
#   7. Are there raw-label ↔ GT disagreements that must be retained?
#
# IMPORTANT:
# ----------
# NO_GT_MATCH is NOT automatically converted to benign.
#
# CROSS_FAMILY_CONFLICT is NOT automatically assigned to a family.
#
# No rows are dropped here.
# No train/test split is created here.
# No CIC + UNSW merge occurs here.
# No forecasting windows are created here.
#
# ==================================================================================================

import os
import gc
import time
import numpy as np
import pandas as pd


# ==================================================================================================
# 10.11A — PATHS & CONFIGURATION
# ==================================================================================================

TOTAL_T0 = time.time()

BASE = (
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Dataset_Audit_v2"
)

UNSW_RAW = os.path.join(
    BASE,
    "raw/UNSW_NB15"
)

UNSW_OUT = os.path.join(
    BASE,
    "audit_outputs/UNSW"
)

os.makedirs(
    UNSW_OUT,
    exist_ok=True
)


RAW_FILES = [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv"
]


MAPPING_FILES = [
    f"10_10_mapping_{f}"
    for f in RAW_FILES
]


FEATURE_FILE = os.path.join(
    UNSW_RAW,
    "NUSW-NB15_features.csv"
)


print("=" * 120)
print("PHASE 10.11 — UNSW AUTHORITATIVE LABEL RESOLUTION & BENIGN CERTIFICATION AUDIT")
print("=" * 120)


# ==================================================================================================
# 10.11B — VERIFY REQUIRED FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11B — VERIFY REQUIRED ARTIFACTS")
print("=" * 120)


for f in MAPPING_FILES:

    path = os.path.join(
        UNSW_OUT,
        f
    )

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing Phase 10.10 mapping artifact:\n{path}"
        )

    print("✓", f)


for f in RAW_FILES:

    path = os.path.join(
        UNSW_RAW,
        f
    )

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing raw UNSW file:\n{path}"
        )

    print("✓", f)


if not os.path.exists(FEATURE_FILE):

    raise FileNotFoundError(
        f"Missing feature metadata:\n{FEATURE_FILE}"
    )

print("✓ NUSW-NB15_features.csv")


# ==================================================================================================
# 10.11C — LOAD UNSW RAW SCHEMA
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11C — RECOVER RAW UNSW SCHEMA")
print("=" * 120)


feature_encodings = [
    "utf-8-sig",
    "cp1252",
    "latin1"
]


features = None
feature_encoding = None


for enc in feature_encodings:

    try:

        features = pd.read_csv(
            FEATURE_FILE,
            encoding=enc
        )

        feature_encoding = enc
        break

    except UnicodeDecodeError:
        continue


if features is None:

    raise RuntimeError(
        "Could not decode UNSW feature metadata."
    )


features.columns = (
    features.columns
    .astype(str)
    .str.strip()
)


if "Name" not in features.columns:

    raise ValueError(
        "Feature metadata does not contain 'Name'."
    )


raw_columns = (
    features["Name"]
    .astype(str)
    .str.strip()
    .tolist()
)


print(
    f"Feature metadata encoding : "
    f"{feature_encoding}"
)

print(
    f"Raw feature count         : "
    f"{len(raw_columns)}"
)


if len(raw_columns) != 49:

    print(
        "⚠ Expected 49 UNSW raw features, "
        f"found {len(raw_columns)}"
    )

else:

    print(
        "✓ 49-column raw schema recovered"
    )


# ==================================================================================================
# 10.11D — IDENTIFY ORIGINAL UNSW LABEL FIELDS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11D — IDENTIFY ORIGINAL RAW LABEL FIELDS")
print("=" * 120)


def canonical_name(x):

    return (
        str(x)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
    )


canonical_lookup = {
    canonical_name(c): c
    for c in raw_columns
}


binary_candidates = [
    "label",
    "class"
]


family_candidates = [
    "attackcat",
    "attackcategory",
    "attack_cat"
]


RAW_BINARY_COL = None
RAW_FAMILY_COL = None


for candidate in binary_candidates:

    key = canonical_name(candidate)

    if key in canonical_lookup:

        RAW_BINARY_COL = canonical_lookup[key]
        break


for candidate in family_candidates:

    key = canonical_name(candidate)

    if key in canonical_lookup:

        RAW_FAMILY_COL = canonical_lookup[key]
        break


print(
    "Original binary label column :",
    RAW_BINARY_COL
)

print(
    "Original attack-family column:",
    RAW_FAMILY_COL
)


if RAW_BINARY_COL is None:

    print(
        "\n⚠ No original binary Label field found "
        "in the 49-column raw schema."
    )


if RAW_FAMILY_COL is None:

    print(
        "⚠ No original attack-category field found "
        "in the 49-column raw schema."
    )


# ==================================================================================================
# 10.11E — NORMALIZATION HELPERS
# ==================================================================================================

def normalize_family(series):

    s = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )

    s = s.mask(
        s.isin(
            [
                "",
                "nan",
                "none",
                "null",
                "<na>",
                "-"
            ]
        )
    )

    return s


def normalize_binary(series):

    s = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    result = pd.Series(
        pd.NA,
        index=series.index,
        dtype="Int8"
    )


    positive_values = {
        "1",
        "1.0",
        "attack",
        "attacked",
        "malicious",
        "anomaly",
        "anomalous",
        "true"
    }


    negative_values = {
        "0",
        "0.0",
        "normal",
        "benign",
        "false"
    }


    result.loc[
        s.isin(
            positive_values
        )
    ] = 1


    result.loc[
        s.isin(
            negative_values
        )
    ] = 0


    numeric = pd.to_numeric(
        s,
        errors="coerce"
    )


    result.loc[
        numeric.eq(1)
    ] = 1


    result.loc[
        numeric.eq(0)
    ] = 0


    return result


# ==================================================================================================
# 10.11F — DEFINE GT-BASED RESOLUTION POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11F — GT-BASED FAMILY RESOLUTION POLICY")
print("=" * 120)


print(
    """
Policy:

UNIQUE_GT_MATCH
    → GT family is RESOLVED.

MULTI_GT_SAME_EPISODE
    → same physical episode;
      family can be RESOLVED if Phase 10.10 produced one family.

MULTI_EPISODE_SAME_FAMILY
    → episode identity is ambiguous,
      but family-level label is RESOLVED.

CROSS_FAMILY_CONFLICT
    → family remains AMBIGUOUS.
      Do NOT arbitrarily choose one.

NO_GT_MATCH
    → GT gives no attack assignment.
      Do NOT call benign yet.

INVALID_NETWORK_KEY
    → unresolved.
"""
)


RESOLVED_GT_STATUSES = {
    "UNIQUE_GT_MATCH",
    "MULTI_GT_SAME_EPISODE",
    "MULTI_EPISODE_SAME_FAMILY"
}


# ==================================================================================================
# 10.11G — PROCESS ALL FOUR RAW FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11G — RAW LABEL ↔ GT RECONSTRUCTION AUDIT")
print("=" * 120)


audit_chunks = []

summary_rows = []


for file_number, raw_filename in enumerate(
    RAW_FILES,
    start=1
):

    file_t0 = time.time()

    mapping_filename = (
        f"10_10_mapping_{raw_filename}"
    )


    mapping_path = os.path.join(
        UNSW_OUT,
        mapping_filename
    )


    raw_path = os.path.join(
        UNSW_RAW,
        raw_filename
    )


    print("\n" + "-" * 120)
    print(
        f"[{file_number}/4] {raw_filename}"
    )
    print("-" * 120)


    # ==============================================================================================
    # Load Phase 10.10 mapping
    # ==============================================================================================

    mapping = pd.read_csv(
        mapping_path,
        low_memory=False
    )


    print(
        f"10.10 mapping rows : "
        f"{len(mapping):,}"
    )


    # ==============================================================================================
    # Load raw file
    # ==============================================================================================

    raw = None
    raw_encoding = None


    for enc in [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]:

        try:

            raw = pd.read_csv(
                raw_path,
                header=None,
                names=raw_columns,
                encoding=enc,
                low_memory=False
            )

            raw_encoding = enc
            break

        except UnicodeDecodeError:
            continue


    if raw is None:

        raise RuntimeError(
            f"Could not decode {raw_filename}"
        )


    print(
        f"Raw rows          : "
        f"{len(raw):,}"
    )

    print(
        f"Encoding          : "
        f"{raw_encoding}"
    )


    # ==============================================================================================
    # Critical row alignment check
    # ==============================================================================================

    if len(raw) != len(mapping):

        raise RuntimeError(
            "\nROW ALIGNMENT FAILURE\n"
            f"Raw rows     : {len(raw):,}\n"
            f"Mapping rows : {len(mapping):,}\n"
            f"File         : {raw_filename}"
        )


    print(
        "✓ Raw rows align exactly with "
        "Phase 10.10 mapping rows"
    )


    # ==============================================================================================
    # Extract only raw labels
    # ==============================================================================================

    audit = pd.DataFrame(
        {
            "Flow_File":
                raw_filename,

            "Flow_Row_Index":
                np.arange(
                    len(raw),
                    dtype=np.int64
                )
        }
    )


    # ------------------------------------------------------------------------------------------------
    # Original binary label
    # ------------------------------------------------------------------------------------------------

    if RAW_BINARY_COL is not None:

        audit[
            "Raw_Binary_Label_Original"
        ] = raw[
            RAW_BINARY_COL
        ]


        audit[
            "Raw_Binary_Label"
        ] = normalize_binary(
            raw[
                RAW_BINARY_COL
            ]
        )

    else:

        audit[
            "Raw_Binary_Label_Original"
        ] = pd.NA

        audit[
            "Raw_Binary_Label"
        ] = pd.Series(
            pd.NA,
            index=audit.index,
            dtype="Int8"
        )


    # ------------------------------------------------------------------------------------------------
    # Original family
    # ------------------------------------------------------------------------------------------------

    if RAW_FAMILY_COL is not None:

        audit[
            "Raw_Attack_Family_Original"
        ] = raw[
            RAW_FAMILY_COL
        ]


        audit[
            "Raw_Attack_Family"
        ] = normalize_family(
            raw[
                RAW_FAMILY_COL
            ]
        )

    else:

        audit[
            "Raw_Attack_Family_Original"
        ] = pd.NA

        audit[
            "Raw_Attack_Family"
        ] = pd.NA


    del raw
    gc.collect()


    # ==============================================================================================
    # Pull Phase 10.10 reconstruction columns
    # ==============================================================================================

    required_mapping_cols = [
        "Mapping_Status"
    ]


    for col in required_mapping_cols:

        if col not in mapping.columns:

            raise ValueError(
                f"{mapping_filename} missing "
                f"required column: {col}"
            )


    #
    # Discover resolved-family column from 10.10.
    #

    possible_family_cols = [
        "Resolved_Attack_Family",
        "Attack_Family",
        "Resolved_Family"
    ]


    mapping_family_col = None


    for c in possible_family_cols:

        if c in mapping.columns:

            mapping_family_col = c
            break


    if mapping_family_col is None:

        raise ValueError(
            f"""
Could not find reconstructed family column in:

{mapping_filename}

Available columns:

{mapping.columns.tolist()}
"""
        )


    print(
        "10.10 reconstructed family column:",
        mapping_family_col
    )


    audit[
        "Mapping_Status"
    ] = (
        mapping[
            "Mapping_Status"
        ]
        .astype("string")
    )


    audit[
        "GT_Attack_Family"
    ] = normalize_family(
        mapping[
            mapping_family_col
        ]
    )


    # ------------------------------------------------------------------------------------------------
    # Optional reconstruction metadata
    # ------------------------------------------------------------------------------------------------

    optional_mapping_cols = [

        "Candidate_GT_Count",

        "Candidate_Family_Count",

        "Candidate_Families",

        "Candidate_Episodes",

        "Resolved_Episode_ID",

        "Episode_ID",

        "Capture_ID",

        "Capture_Consistency"
    ]


    for col in optional_mapping_cols:

        if col in mapping.columns:

            audit[col] = mapping[col]


    # ==============================================================================================
    # GT RESOLUTION STATE
    # ==============================================================================================

    audit[
        "GT_Family_Resolution"
    ] = "UNRESOLVED"


    resolved_mask = (
        audit[
            "Mapping_Status"
        ]
        .isin(
            RESOLVED_GT_STATUSES
        )
        &
        audit[
            "GT_Attack_Family"
        ]
        .notna()
    )


    conflict_mask = (
        audit[
            "Mapping_Status"
        ]
        ==
        "CROSS_FAMILY_CONFLICT"
    )


    no_gt_mask = (
        audit[
            "Mapping_Status"
        ]
        ==
        "NO_GT_MATCH"
    )


    invalid_key_mask = (
        audit[
            "Mapping_Status"
        ]
        ==
        "INVALID_NETWORK_KEY"
    )


    audit.loc[
        resolved_mask,
        "GT_Family_Resolution"
    ] = "RESOLVED_ATTACK_FAMILY"


    audit.loc[
        conflict_mask,
        "GT_Family_Resolution"
    ] = "AMBIGUOUS_CROSS_FAMILY"


    audit.loc[
        no_gt_mask,
        "GT_Family_Resolution"
    ] = "NO_GT_SUPPORT"


    audit.loc[
        invalid_key_mask,
        "GT_Family_Resolution"
    ] = "INVALID_NETWORK_KEY"


    # ==============================================================================================
    # RAW BINARY ↔ GT CONSISTENCY
    # ==============================================================================================

    audit[
        "Binary_GT_Consistency"
    ] = "NOT_TESTABLE"


    raw_attack = (
        audit[
            "Raw_Binary_Label"
        ]
        .eq(1)
        .fillna(False)
    )


    raw_normal = (
        audit[
            "Raw_Binary_Label"
        ]
        .eq(0)
        .fillna(False)
    )


    #
    # Resolved GT attack + raw attack
    #

    audit.loc[
        resolved_mask
        &
        raw_attack,
        "Binary_GT_Consistency"
    ] = "AGREE_ATTACK"


    #
    # Resolved GT attack + raw normal
    #

    audit.loc[
        resolved_mask
        &
        raw_normal,
        "Binary_GT_Consistency"
    ] = "CONFLICT_GT_ATTACK_RAW_NORMAL"


    #
    # Cross-family still means GT says ATTACK,
    # even though family is ambiguous.
    #

    audit.loc[
        conflict_mask
        &
        raw_attack,
        "Binary_GT_Consistency"
    ] = "AGREE_ATTACK_FAMILY_AMBIGUOUS"


    audit.loc[
        conflict_mask
        &
        raw_normal,
        "Binary_GT_Consistency"
    ] = "CONFLICT_GT_ATTACK_RAW_NORMAL"


    #
    # NO_GT_MATCH
    #

    audit.loc[
        no_gt_mask
        &
        raw_normal,
        "Binary_GT_Consistency"
    ] = "RAW_NORMAL_NO_GT_ATTACK"


    audit.loc[
        no_gt_mask
        &
        raw_attack,
        "Binary_GT_Consistency"
    ] = "RAW_ATTACK_WITHOUT_GT_MATCH"


    #
    # Invalid key
    #

    audit.loc[
        invalid_key_mask
        &
        raw_normal,
        "Binary_GT_Consistency"
    ] = "RAW_NORMAL_INVALID_KEY"


    audit.loc[
        invalid_key_mask
        &
        raw_attack,
        "Binary_GT_Consistency"
    ] = "RAW_ATTACK_INVALID_KEY"


    # ==============================================================================================
    # FAMILY CONSISTENCY
    # ==============================================================================================

    audit[
        "Family_GT_Consistency"
    ] = "NOT_TESTABLE"


    raw_family_available = (
        audit[
            "Raw_Attack_Family"
        ]
        .notna()
    )


    gt_family_available = (
        audit[
            "GT_Attack_Family"
        ]
        .notna()
    )


    both_family_available = (
        raw_family_available
        &
        gt_family_available
        &
        resolved_mask
    )


    family_agree = (
        both_family_available
        &
        (
            audit[
                "Raw_Attack_Family"
            ]
            ==
            audit[
                "GT_Attack_Family"
            ]
        )
    )


    family_disagree = (
        both_family_available
        &
        (
            audit[
                "Raw_Attack_Family"
            ]
            !=
            audit[
                "GT_Attack_Family"
            ]
        )
    )


    audit.loc[
        family_agree,
        "Family_GT_Consistency"
    ] = "AGREE"


    audit.loc[
        family_disagree,
        "Family_GT_Consistency"
    ] = "DISAGREE"


    audit.loc[
        conflict_mask,
        "Family_GT_Consistency"
    ] = "GT_CROSS_FAMILY_AMBIGUOUS"


    # ==============================================================================================
    # CONSERVATIVE FINAL LABEL POLICY
    # ==============================================================================================

    audit[
        "Resolved_Attack_State"
    ] = pd.Series(
        pd.NA,
        index=audit.index,
        dtype="Int8"
    )


    audit[
        "Resolved_Attack_Family"
    ] = pd.Series(
        pd.NA,
        index=audit.index,
        dtype="string"
    )


    audit[
        "Label_Resolution_Status"
    ] = "UNRESOLVED"


    # ------------------------------------------------------------------------------------------------
    # Case 1:
    # GT family is unambiguous.
    #
    # GT provides authoritative attack evidence.
    # ------------------------------------------------------------------------------------------------

    audit.loc[
        resolved_mask,
        "Resolved_Attack_State"
    ] = 1


    audit.loc[
        resolved_mask,
        "Resolved_Attack_Family"
    ] = (
        audit.loc[
            resolved_mask,
            "GT_Attack_Family"
        ]
    )


    audit.loc[
        resolved_mask,
        "Label_Resolution_Status"
    ] = "GT_RESOLVED_ATTACK"


    # ------------------------------------------------------------------------------------------------
    # Case 2:
    # Cross-family conflict.
    #
    # Binary attack state is known,
    # family is NOT known.
    # ------------------------------------------------------------------------------------------------

    audit.loc[
        conflict_mask,
        "Resolved_Attack_State"
    ] = 1


    audit.loc[
        conflict_mask,
        "Label_Resolution_Status"
    ] = "GT_ATTACK_FAMILY_AMBIGUOUS"


    # ------------------------------------------------------------------------------------------------
    # Case 3:
    # NO_GT_MATCH + original UNSW raw binary says normal.
    #
    # This becomes a BENIGN CANDIDATE.
    #
    # IMPORTANT:
    # We do NOT immediately call this certified benign until the
    # global audit below confirms there are no contradictory raw fields.
    # ------------------------------------------------------------------------------------------------

    benign_candidate = (
        no_gt_mask
        &
        raw_normal
    )


    audit.loc[
        benign_candidate,
        "Label_Resolution_Status"
    ] = "BENIGN_CANDIDATE_RAW_NORMAL_NO_GT"


    # ------------------------------------------------------------------------------------------------
    # Case 4:
    # NO_GT_MATCH but raw UNSW says ATTACK.
    #
    # This is highly important and must remain unresolved.
    # ------------------------------------------------------------------------------------------------

    raw_attack_no_gt = (
        no_gt_mask
        &
        raw_attack
    )


    audit.loc[
        raw_attack_no_gt,
        "Resolved_Attack_State"
    ] = 1


    audit.loc[
        raw_attack_no_gt,
        "Resolved_Attack_Family"
    ] = (
        audit.loc[
            raw_attack_no_gt,
            "Raw_Attack_Family"
        ]
    )


    audit.loc[
        raw_attack_no_gt,
        "Label_Resolution_Status"
    ] = "RAW_ATTACK_WITHOUT_GT_MATCH_REVIEW"


    # ------------------------------------------------------------------------------------------------
    # Invalid key
    # ------------------------------------------------------------------------------------------------

    audit.loc[
        invalid_key_mask,
        "Label_Resolution_Status"
    ] = "INVALID_NETWORK_KEY_REVIEW"


    # ==============================================================================================
    # FILE SUMMARY
    # ==============================================================================================

    status_counts = (
        audit[
            "Label_Resolution_Status"
        ]
        .value_counts(
            dropna=False
        )
    )


    print("\nResolution status:")

    print(
        status_counts.to_string()
    )


    print("\nBinary GT consistency:")

    print(
        audit[
            "Binary_GT_Consistency"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


    print("\nFamily GT consistency:")

    print(
        audit[
            "Family_GT_Consistency"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


    summary_rows.append(
        {
            "Flow_File":
                raw_filename,

            "Rows":
                len(audit),

            "GT_Resolved_Attack":
                int(
                    resolved_mask.sum()
                ),

            "GT_Cross_Family_Ambiguous":
                int(
                    conflict_mask.sum()
                ),

            "Benign_Candidates":
                int(
                    benign_candidate.sum()
                ),

            "Raw_Attack_Without_GT":
                int(
                    raw_attack_no_gt.sum()
                ),

            "Invalid_Network_Key":
                int(
                    invalid_key_mask.sum()
                )
        }
    )


    # ==============================================================================================
    # Save per-file audit
    # ==============================================================================================

    out_file = os.path.join(
        UNSW_OUT,
        f"10_11_label_resolution_{raw_filename}"
    )


    audit.to_csv(
        out_file,
        index=False
    )


    print(
        "\n✓ Saved:",
        out_file
    )


    print(
        f"✓ Runtime: "
        f"{time.time() - file_t0:.2f} sec"
    )


    audit_chunks.append(
        audit
    )


    del mapping
    gc.collect()


# ==================================================================================================
# 10.11H — GLOBAL COMBINATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11H — GLOBAL LABEL RESOLUTION AUDIT")
print("=" * 120)


audit_all = pd.concat(
    audit_chunks,
    ignore_index=True
)


summary = pd.DataFrame(
    summary_rows
)


print(
    summary.to_string(
        index=False
    )
)


print(
    "\nTotal flows:",
    f"{len(audit_all):,}"
)


# ==================================================================================================
# 10.11I — GLOBAL BINARY CONSISTENCY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11I — ORIGINAL UNSW BINARY LABEL ↔ GT CONSISTENCY")
print("=" * 120)


binary_summary = (
    audit_all[
        "Binary_GT_Consistency"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Binary_GT_Consistency"
    )
    .reset_index(
        name="Flows"
    )
)


binary_summary[
    "Pct_All_Flows"
] = (
    100
    *
    binary_summary[
        "Flows"
    ]
    /
    len(audit_all)
)


print(
    binary_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.11J — GLOBAL FAMILY CONSISTENCY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11J — ORIGINAL UNSW FAMILY ↔ GT FAMILY CONSISTENCY")
print("=" * 120)


family_summary = (
    audit_all[
        "Family_GT_Consistency"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Family_GT_Consistency"
    )
    .reset_index(
        name="Flows"
    )
)


family_summary[
    "Pct_All_Flows"
] = (
    100
    *
    family_summary[
        "Flows"
    ]
    /
    len(audit_all)
)


print(
    family_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.11K — RAW ATTACKS WITHOUT GT MATCH
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11K — RAW-LABELED ATTACK FLOWS WITHOUT GT MATCH")
print("=" * 120)


raw_attack_without_gt = audit_all[
    audit_all[
        "Label_Resolution_Status"
    ]
    ==
    "RAW_ATTACK_WITHOUT_GT_MATCH_REVIEW"
].copy()


print(
    "Raw attack flows without GT match:",
    f"{len(raw_attack_without_gt):,}"
)


if len(raw_attack_without_gt):

    print(
        "\nRaw family distribution:"
    )

    print(
        raw_attack_without_gt[
            "Raw_Attack_Family"
        ]
        .value_counts(
            dropna=False
        )
        .head(30)
        .to_string()
    )


    print(
        "\nExample rows:"
    )

    display_cols = [
        c
        for c in [
            "Flow_File",
            "Flow_Row_Index",
            "Raw_Binary_Label_Original",
            "Raw_Attack_Family_Original",
            "Mapping_Status",
            "Binary_GT_Consistency"
        ]
        if c in raw_attack_without_gt.columns
    ]


    print(
        raw_attack_without_gt[
            display_cols
        ]
        .head(20)
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 10.11L — GT ATTACK ↔ RAW NORMAL CONFLICT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11L — GT ATTACK BUT ORIGINAL RAW LABEL SAYS NORMAL")
print("=" * 120)


gt_attack_raw_normal = audit_all[
    audit_all[
        "Binary_GT_Consistency"
    ]
    ==
    "CONFLICT_GT_ATTACK_RAW_NORMAL"
].copy()


print(
    "Conflicting flows:",
    f"{len(gt_attack_raw_normal):,}"
)


if len(gt_attack_raw_normal):

    print(
        "\nGT family distribution:"
    )

    print(
        gt_attack_raw_normal[
            "GT_Attack_Family"
        ]
        .value_counts(
            dropna=False
        )
        .to_string()
    )


# ==================================================================================================
# 10.11M — FAMILY DISAGREEMENT MATRIX
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11M — RAW FAMILY ↔ GT FAMILY DISAGREEMENT MATRIX")
print("=" * 120)


family_disagreement = audit_all[
    audit_all[
        "Family_GT_Consistency"
    ]
    ==
    "DISAGREE"
].copy()


print(
    "Resolved GT attack flows with family disagreement:",
    f"{len(family_disagreement):,}"
)


if len(family_disagreement):

    disagreement_matrix = pd.crosstab(
        family_disagreement[
            "Raw_Attack_Family"
        ],
        family_disagreement[
            "GT_Attack_Family"
        ]
    )

    print(
        disagreement_matrix
    )

else:

    disagreement_matrix = pd.DataFrame()

    print(
        "✓ No family disagreement among "
        "testable resolved GT attack flows."
    )


# ==================================================================================================
# 10.11N — CROSS-FAMILY AMBIGUITY PRESERVATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11N — CROSS-FAMILY AMBIGUITY PRESERVATION")
print("=" * 120)


cross_family = audit_all[
    audit_all[
        "Mapping_Status"
    ]
    ==
    "CROSS_FAMILY_CONFLICT"
].copy()


print(
    "Cross-family ambiguous flows:",
    f"{len(cross_family):,}"
)


resolved_cross_family = (
    cross_family[
        "Resolved_Attack_Family"
    ]
    .notna()
    .sum()
)


print(
    "Cross-family flows assigned a final family:",
    f"{resolved_cross_family:,}"
)


if resolved_cross_family != 0:

    raise RuntimeError(
        "Cross-family conflicts were accidentally "
        "forced into a family."
    )


print(
    "✓ Cross-family ambiguity preserved."
)


# ==================================================================================================
# 10.11O — BENIGN CERTIFICATION AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11O — BENIGN CERTIFICATION AUDIT")
print("=" * 120)


benign_candidates = audit_all[
    audit_all[
        "Label_Resolution_Status"
    ]
    ==
    "BENIGN_CANDIDATE_RAW_NORMAL_NO_GT"
].copy()


print(
    "Benign candidates:",
    f"{len(benign_candidates):,}"
)


#
# A conservative certification requires:
#
#   1. NO_GT_MATCH
#   2. raw binary label == 0
#   3. no raw attack family present
#
# If attack_cat is absent in raw schema, criterion 3 cannot be tested.
#

if RAW_FAMILY_COL is not None:

    family_conflict_in_benign = int(
        benign_candidates[
            "Raw_Attack_Family"
        ]
        .notna()
        .sum()
    )

else:

    family_conflict_in_benign = -1


print(
    "Benign candidates with non-null "
    "raw attack family:",
    (
        f"{family_conflict_in_benign:,}"
        if family_conflict_in_benign >= 0
        else "NOT TESTABLE"
    )
)


BENIGN_CERTIFICATION_PASSED = False


if RAW_BINARY_COL is None:

    print(
        "\n⚠ BENIGN CERTIFICATION NOT POSSIBLE:"
        "\nNo original raw binary label was available."
    )


elif (
    RAW_FAMILY_COL is not None
    and
    family_conflict_in_benign > 0
):

    print(
        "\n⚠ BENIGN CERTIFICATION FAILED:"
        "\nSome NO_GT_MATCH + raw-normal flows "
        "still carry an attack-family value."
    )


else:

    BENIGN_CERTIFICATION_PASSED = True


    candidate_mask_global = (
        audit_all[
            "Label_Resolution_Status"
        ]
        ==
        "BENIGN_CANDIDATE_RAW_NORMAL_NO_GT"
    )


    audit_all.loc[
        candidate_mask_global,
        "Resolved_Attack_State"
    ] = 0


    audit_all.loc[
        candidate_mask_global,
        "Resolved_Attack_Family"
    ] = "normal"


    audit_all.loc[
        candidate_mask_global,
        "Label_Resolution_Status"
    ] = "CERTIFIED_BENIGN"


    print(
        "\n✓ BENIGN CERTIFICATION PASSED"
    )

    print(
        "NO_GT_MATCH + original raw normal "
        "flows can be retained as benign "
        "under this reconstruction policy."
    )


# ==================================================================================================
# 10.11P — FINAL RESOLUTION DISTRIBUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11P — FINAL CONSERVATIVE LABEL RESOLUTION")
print("=" * 120)


resolution_summary = (
    audit_all[
        "Label_Resolution_Status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Resolution_Status"
    )
    .reset_index(
        name="Flows"
    )
)


resolution_summary[
    "Pct_All_Flows"
] = (
    100
    *
    resolution_summary[
        "Flows"
    ]
    /
    len(audit_all)
)


print(
    resolution_summary.to_string(
        index=False
    )
)


print(
    "\nResolved attack-state distribution:"
)


print(
    audit_all[
        "Resolved_Attack_State"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


print(
    "\nResolved attack-family distribution:"
)


print(
    audit_all[
        "Resolved_Attack_Family"
    ]
    .value_counts(
        dropna=False
    )
    .head(30)
)


# ==================================================================================================
# 10.11Q — MODEL-ELIGIBILITY FLAGS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11Q — MODEL ELIGIBILITY FLAGS")
print("=" * 120)


#
# Binary forecasting eligibility:
#
# We know attack state for:
#
#   - resolved GT attacks
#   - cross-family conflicts (still attacks)
#   - certified benign
#
# Family forecasting eligibility:
#
# Requires:
#
#   - known attack state
#   - known family
#
# Cross-family conflicts are NOT eligible for multiclass family targets.
#

audit_all[
    "Eligible_Binary_Forecasting"
] = (
    audit_all[
        "Resolved_Attack_State"
    ]
    .notna()
)


audit_all[
    "Eligible_Family_Forecasting"
] = (
    audit_all[
        "Resolved_Attack_State"
    ]
    .eq(1)
    .fillna(False)
    &
    audit_all[
        "Resolved_Attack_Family"
    ]
    .notna()
)


audit_all[
    "Requires_Label_Review"
] = (
    ~audit_all[
        "Eligible_Binary_Forecasting"
    ]
    |
    (
        audit_all[
            "Resolved_Attack_State"
        ]
        .eq(1)
        .fillna(False)
        &
        audit_all[
            "Resolved_Attack_Family"
        ]
        .isna()
    )
)


print(
    "Binary-forecasting eligible flows :",
    f"{audit_all['Eligible_Binary_Forecasting'].sum():,}",
    f"/ {len(audit_all):,}"
)


print(
    "Family-forecasting eligible attack flows:",
    f"{audit_all['Eligible_Family_Forecasting'].sum():,}"
)


print(
    "Flows requiring label review:",
    f"{audit_all['Requires_Label_Review'].sum():,}"
)


# ==================================================================================================
# 10.11R — CRITICAL CONSISTENCY ASSERTIONS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11R — CRITICAL CONSISTENCY ASSERTIONS")
print("=" * 120)


assert (
    len(audit_all)
    ==
    sum(
        row[
            "Rows"
        ]
        for row in summary_rows
    )
)


assert (
    audit_all[
        [
            "Flow_File",
            "Flow_Row_Index"
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
), (
    "Duplicate flow identities found."
)


assert (
    audit_all.loc[
        audit_all[
            "Mapping_Status"
        ]
        ==
        "CROSS_FAMILY_CONFLICT",
        "Resolved_Attack_Family"
    ]
    .isna()
    .all()
), (
    "Cross-family conflicts were assigned "
    "a family unexpectedly."
)


assert (
    audit_all.loc[
        audit_all[
            "GT_Family_Resolution"
        ]
        ==
        "RESOLVED_ATTACK_FAMILY",
        "Resolved_Attack_State"
    ]
    .eq(1)
    .all()
), (
    "Resolved GT attacks do not all have "
    "attack state = 1."
)


print(
    "✓ Global row conservation valid"
)

print(
    "✓ Flow identities unique"
)

print(
    "✓ Cross-family ambiguity preserved"
)

print(
    "✓ Resolved GT attacks retain attack state"
)


# ==================================================================================================
# 10.11S — SAVE GLOBAL AUDIT ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.11S — SAVE GLOBAL AUDIT ARTIFACTS")
print("=" * 120)


GLOBAL_AUDIT_FILE = os.path.join(
    UNSW_OUT,
    "10_11_global_label_resolution_audit.csv"
)


SUMMARY_FILE = os.path.join(
    UNSW_OUT,
    "10_11_resolution_summary.csv"
)


BINARY_FILE = os.path.join(
    UNSW_OUT,
    "10_11_binary_gt_consistency.csv"
)


FAMILY_FILE = os.path.join(
    UNSW_OUT,
    "10_11_family_gt_consistency.csv"
)


RAW_ATTACK_NO_GT_FILE = os.path.join(
    UNSW_OUT,
    "10_11_raw_attack_without_gt_match.csv"
)


GT_ATTACK_RAW_NORMAL_FILE = os.path.join(
    UNSW_OUT,
    "10_11_gt_attack_raw_normal_conflicts.csv"
)


CROSS_FAMILY_FILE = os.path.join(
    UNSW_OUT,
    "10_11_cross_family_ambiguous_flows.csv"
)


FINAL_LABEL_FILE = os.path.join(
    UNSW_OUT,
    "10_11_final_conservative_flow_labels.csv"
)


# --------------------------------------------------------------------------------------------------
# Full audit
# --------------------------------------------------------------------------------------------------

audit_all.to_csv(
    GLOBAL_AUDIT_FILE,
    index=False
)


# --------------------------------------------------------------------------------------------------
# Summaries
# --------------------------------------------------------------------------------------------------

resolution_summary.to_csv(
    SUMMARY_FILE,
    index=False
)


binary_summary.to_csv(
    BINARY_FILE,
    index=False
)


family_summary.to_csv(
    FAMILY_FILE,
    index=False
)


# --------------------------------------------------------------------------------------------------
# Important conflict subsets
# --------------------------------------------------------------------------------------------------

raw_attack_without_gt.to_csv(
    RAW_ATTACK_NO_GT_FILE,
    index=False
)


gt_attack_raw_normal.to_csv(
    GT_ATTACK_RAW_NORMAL_FILE,
    index=False
)


cross_family.to_csv(
    CROSS_FAMILY_FILE,
    index=False
)


# --------------------------------------------------------------------------------------------------
# Compact modeling-label artifact
# --------------------------------------------------------------------------------------------------

final_columns = [

    "Flow_File",

    "Flow_Row_Index",

    "Mapping_Status",

    "Resolved_Attack_State",

    "Resolved_Attack_Family",

    "Label_Resolution_Status",

    "Eligible_Binary_Forecasting",

    "Eligible_Family_Forecasting",

    "Requires_Label_Review"
]


for optional_col in [

    "Resolved_Episode_ID",

    "Episode_ID",

    "Candidate_Episodes",

    "Candidate_Families",

    "Capture_ID",

    "Capture_Consistency"

]:

    if (
        optional_col in audit_all.columns
        and
        optional_col not in final_columns
    ):

        final_columns.append(
            optional_col
        )


audit_all[
    final_columns
].to_csv(
    FINAL_LABEL_FILE,
    index=False
)


print("\nSaved:")

for f in [

    GLOBAL_AUDIT_FILE,

    SUMMARY_FILE,

    BINARY_FILE,

    FAMILY_FILE,

    RAW_ATTACK_NO_GT_FILE,

    GT_ATTACK_RAW_NORMAL_FILE,

    CROSS_FAMILY_FILE,

    FINAL_LABEL_FILE

]:

    print(
        " ✓",
        f
    )


# ==================================================================================================
# 10.11T — FINAL STATUS
# ==================================================================================================

TOTAL_RUNTIME = (
    time.time()
    -
    TOTAL_T0
)


print("\n" + "=" * 120)
print("PHASE 10.11 COMPLETE")
print("=" * 120)


resolved_binary = int(
    audit_all[
        "Eligible_Binary_Forecasting"
    ].sum()
)


resolved_family = int(
    audit_all[
        "Eligible_Family_Forecasting"
    ].sum()
)


review_count = int(
    audit_all[
        "Requires_Label_Review"
    ].sum()
)


cross_family_count = int(
    (
        audit_all[
            "Mapping_Status"
        ]
        ==
        "CROSS_FAMILY_CONFLICT"
    ).sum()
)


raw_attack_no_gt_count = int(
    len(
        raw_attack_without_gt
    )
)


print(
    f"""
TOTAL FLOWS
-----------
{len(audit_all):,}


GT-BASED LABEL POLICY
---------------------
UNIQUE_GT_MATCH
    → family resolved

MULTI_GT_SAME_EPISODE
    → family resolved

MULTI_EPISODE_SAME_FAMILY
    → family resolved at family level

CROSS_FAMILY_CONFLICT
    → binary ATTACK known
    → attack family remains ambiguous

NO_GT_MATCH
    → benign only if original UNSW evidence
      passes the benign-certification audit


RESULTS
-------
Binary-forecasting eligible flows : {resolved_binary:,}

Family-forecasting eligible attacks:
{resolved_family:,}

Cross-family ambiguous flows:
{cross_family_count:,}

Raw-labeled attacks without GT match:
{raw_attack_no_gt_count:,}

Flows requiring review:
{review_count:,}


BENIGN CERTIFICATION
--------------------
{"PASSED" if BENIGN_CERTIFICATION_PASSED else "NOT PASSED / NOT TESTABLE"}


IMPORTANT
---------
No cross-family conflict was silently assigned
to an arbitrary attack family.

No unresolved flow was silently converted to benign.

No flow was dropped.


NEXT PHASE
----------
Do NOT combine CIC + UNSW yet.

The next decision depends on THIS output.

If benign certification passes and the raw-vs-GT
conflicts are negligible/understood, we can freeze
the UNSW flow-level semantic layer.

Then we move to:

    Phase 10.12
        → temporal/capture integrity of final labels
        → attack-family canonicalization decisions
        → final UNSW episode-linked flow artifact

After that:

    UNSW temporal aggregation
        ↓
    CIC ↔ UNSW common semantic schema
        ↓
    known-type episode-aware split design
        ↓
    leakage audit
        ↓
    forecasting windows
        ↓
    baseline models


Runtime: {TOTAL_RUNTIME:.2f} sec
"""
)

In [ ]:
# ==================================================================================================
# PHASE 10.12 — UNSW FINAL SEMANTIC FREEZE, AMBIGUITY REVIEW & EPISODE-LINKED FLOW ARTIFACT
#
# PURPOSE
# -------
# Phase 10.11 established:
#
#   • 2,218,456 certified benign flows
#   • 280,961 GT-family-resolved attack flows
#   • 40,322 binary-known but family-ambiguous attack flows
#   • 308 invalid-network-key flows requiring review
#
# Phase 10.12 does NOT combine CIC and UNSW.
#
# It performs the final UNSW semantic-layer audit:
#
#   1. Re-load Phase 10.11 conservative labels.
#   2. Recover original UNSW Label / attack_cat evidence.
#   3. Inspect cross-family ambiguous flows.
#   4. Determine whether raw attack_cat can safely resolve any ambiguity.
#   5. Inspect the 308 invalid-network-key rows.
#   6. Canonicalize attack-family naming WITHOUT changing attack semantics.
#   7. Attach/retain episode provenance where available.
#   8. Construct final binary / multiclass eligibility flags.
#   9. Run conservation and semantic assertions.
#  10. Freeze one authoritative UNSW flow-level semantic artifact.
#
# IMPORTANT
# ---------
# • No arbitrary family assignment.
# • No unresolved flow silently becomes benign.
# • No flow is silently dropped.
# • Binary-known / family-unknown attacks remain attacks.
# • "backdoor" and "backdoors" are canonicalized only as a naming normalization.
# • Episode IDs are NOT invented when Phase 10.10 did not provide one.
# ==================================================================================================

import os
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PHASE_START = time.time()

# --------------------------------------------------------------------------------------------------
# 10.12A — PATHS
# --------------------------------------------------------------------------------------------------

BASE = Path("/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2")

RAW_DIR = BASE / "raw" / "UNSW_NB15"
OUT_DIR = BASE / "audit_outputs" / "UNSW"

OUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_LABEL_FILE = OUT_DIR / "10_11_final_conservative_flow_labels.csv"

RAW_FILES = [
    RAW_DIR / "UNSW-NB15_1.csv",
    RAW_DIR / "UNSW-NB15_2.csv",
    RAW_DIR / "UNSW-NB15_3.csv",
    RAW_DIR / "UNSW-NB15_4.csv",
]

MAPPING_FILES = [
    OUT_DIR / "10_10_mapping_UNSW-NB15_1.csv",
    OUT_DIR / "10_10_mapping_UNSW-NB15_2.csv",
    OUT_DIR / "10_10_mapping_UNSW-NB15_3.csv",
    OUT_DIR / "10_10_mapping_UNSW-NB15_4.csv",
]

FEATURE_FILE = RAW_DIR / "NUSW-NB15_features.csv"


def banner(text):
    print("\n" + "=" * 120)
    print(text)
    print("=" * 120)


def subbanner(text):
    print("\n" + "-" * 120)
    print(text)
    print("-" * 120)


banner("PHASE 10.12 — UNSW FINAL SEMANTIC FREEZE, AMBIGUITY REVIEW & EPISODE-LINKED FLOW ARTIFACT")


# --------------------------------------------------------------------------------------------------
# 10.12B — VERIFY REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

banner("10.12B — VERIFY REQUIRED INPUTS")

required = [FINAL_LABEL_FILE, FEATURE_FILE] + RAW_FILES + MAPPING_FILES

for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Required artifact missing:\n{p}")
    print(f"✓ {p.name}")


# --------------------------------------------------------------------------------------------------
# 10.12C — LOAD PHASE 10.11 CONSERVATIVE LABEL ARTIFACT
# --------------------------------------------------------------------------------------------------

banner("10.12C — LOAD PHASE 10.11 CONSERVATIVE LABEL ARTIFACT")

labels = pd.read_csv(
    FINAL_LABEL_FILE,
    low_memory=False
)

print(f"Rows    : {len(labels):,}")
print(f"Columns : {len(labels.columns):,}")
print("\nColumns:")
print(labels.columns.tolist())

EXPECTED_TOTAL = 2_540_047

if len(labels) != EXPECTED_TOTAL:
    raise ValueError(
        f"Unexpected Phase 10.11 row count.\n"
        f"Expected: {EXPECTED_TOTAL:,}\n"
        f"Found   : {len(labels):,}"
    )

print("\n✓ Phase 10.11 global row count verified")


# --------------------------------------------------------------------------------------------------
# 10.12D — IDENTIFY CORE PHASE 10.11 COLUMNS
# --------------------------------------------------------------------------------------------------

banner("10.12D — IDENTIFY CORE PHASE 10.11 COLUMNS")


def find_col(df, candidates, required=False):
    lower_map = {str(c).strip().lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    if required:
        raise KeyError(
            f"Could not find any of these columns:\n{candidates}\n\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return None


status_col = find_col(
    labels,
    [
        "Resolution_Status",
        "Label_Resolution_Status"
    ],
    required=True
)

state_col = find_col(
    labels,
    [
        "Resolved_Attack_State",
        "Attack_State"
    ],
    required=True
)

family_col = find_col(
    labels,
    [
        "Resolved_Attack_Family",
        "Attack_Family"
    ],
    required=True
)

flow_file_col = find_col(
    labels,
    [
        "Flow_File",
        "Source_File",
        "Raw_File"
    ],
    required=False
)

flow_row_col = find_col(
    labels,
    [
        "Flow_Row",
        "Raw_Row",
        "Row_ID",
        "Flow_Row_ID",
        "Row_Number"
    ],
    required=False
)

flow_id_col = find_col(
    labels,
    [
        "Flow_ID",
        "Global_Flow_ID"
    ],
    required=False
)

print("Resolution status column :", status_col)
print("Attack-state column      :", state_col)
print("Attack-family column     :", family_col)
print("Flow-file column         :", flow_file_col)
print("Flow-row column          :", flow_row_col)
print("Flow-ID column           :", flow_id_col)


# --------------------------------------------------------------------------------------------------
# 10.12E — RECOVER RAW UNSW SCHEMA
# --------------------------------------------------------------------------------------------------

banner("10.12E — RECOVER RAW UNSW SCHEMA")


def read_csv_encoding(path, **kwargs):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(
                path,
                encoding=enc,
                **kwargs
            )
            return df, enc
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


features, feature_encoding = read_csv_encoding(
    FEATURE_FILE,
    low_memory=False
)

features.columns = features.columns.astype(str).str.strip()

name_col = None

for c in features.columns:
    if c.strip().lower() == "name":
        name_col = c
        break

if name_col is None:
    raise ValueError(
        "Could not identify feature-name column in NUSW-NB15_features.csv"
    )

raw_schema = (
    features[name_col]
    .astype(str)
    .str.strip()
    .tolist()
)

print("Feature metadata encoding :", feature_encoding)
print("Recovered raw columns     :", len(raw_schema))

if len(raw_schema) != 49:
    raise ValueError(
        f"Expected 49 raw UNSW columns, found {len(raw_schema)}"
    )

print("✓ 49-column UNSW raw schema recovered")


# --------------------------------------------------------------------------------------------------
# 10.12F — IDENTIFY RAW LABEL FIELDS
# --------------------------------------------------------------------------------------------------

banner("10.12F — IDENTIFY RAW LABEL FIELDS")

schema_lower = {
    c.strip().lower(): c
    for c in raw_schema
}

raw_label_name = schema_lower.get("label")
raw_attack_cat_name = schema_lower.get("attack_cat")

if raw_label_name is None:
    raise ValueError("Raw binary Label field not found.")

if raw_attack_cat_name is None:
    raise ValueError("Raw attack_cat field not found.")

print("Raw binary label field :", raw_label_name)
print("Raw family field       :", raw_attack_cat_name)


# --------------------------------------------------------------------------------------------------
# 10.12G — LOAD ONLY RAW LABEL EVIDENCE
#
# IMPORTANT:
# We do NOT load all 49 columns from all 2.54M rows.
# We only load Label + attack_cat.
#
# This keeps Phase 10.12 much faster and more memory-efficient.
# --------------------------------------------------------------------------------------------------

banner("10.12G — LOAD ORIGINAL RAW LABEL EVIDENCE")

raw_label_parts = []

global_offset = 0

for i, raw_path in enumerate(RAW_FILES, start=1):

    subbanner(f"[{i}/4] {raw_path.name}")

    start = time.time()

    # Raw files have no header; schema comes from metadata.
    # Read only the two columns we need.
    label_idx = raw_schema.index(raw_label_name)
    attack_idx = raw_schema.index(raw_attack_cat_name)

    raw_subset, enc = read_csv_encoding(
        raw_path,
        header=None,
        names=raw_schema,
        usecols=[label_idx, attack_idx],
        low_memory=False
    )

    n = len(raw_subset)

    raw_subset = raw_subset.reset_index(drop=True)

    raw_subset["Flow_File_10_12"] = raw_path.name
    raw_subset["Flow_Row_10_12"] = np.arange(n, dtype=np.int64)
    raw_subset["Global_Row_10_12"] = np.arange(
        global_offset,
        global_offset + n,
        dtype=np.int64
    )

    global_offset += n

    raw_subset = raw_subset.rename(
        columns={
            raw_label_name: "Raw_Label_10_12",
            raw_attack_cat_name: "Raw_Attack_Cat_10_12"
        }
    )

    raw_label_parts.append(raw_subset)

    print(f"Rows     : {n:,}")
    print(f"Encoding : {enc}")
    print(f"Runtime  : {time.time() - start:.2f} sec")

raw_evidence = pd.concat(
    raw_label_parts,
    ignore_index=True
)

del raw_label_parts
gc.collect()

print("\nCombined raw evidence rows:", f"{len(raw_evidence):,}")

if len(raw_evidence) != EXPECTED_TOTAL:
    raise ValueError(
        "Raw evidence row count does not equal expected UNSW total."
    )

print("✓ Raw label evidence globally aligned by original file order")


# --------------------------------------------------------------------------------------------------
# 10.12H — VERIFY GLOBAL ORDER AGAINST PHASE 10.11
# --------------------------------------------------------------------------------------------------

banner("10.12H — VERIFY GLOBAL ORDER / IDENTITY ALIGNMENT")

# Phase 10.11 was built in the same raw-file order.
# Where Flow_File exists, verify directly.

if flow_file_col is not None:

    expected_files = raw_evidence["Flow_File_10_12"].astype(str).values
    observed_files = labels[flow_file_col].astype(str).values

    file_match = np.mean(expected_files == observed_files)

    print(f"Flow-file positional agreement: {file_match * 100:.6f}%")

    if file_match != 1.0:
        raise ValueError(
            "Phase 10.11 labels are not positionally aligned with raw file order."
        )

    print("✓ Flow-file order verified")

else:
    print(
        "⚠ Phase 10.11 artifact has no explicit Flow_File column.\n"
        "  Using the established 10.11 global raw-file concatenation order."
    )


# Attach raw evidence positionally.
# We assert equal row counts before doing this.

labels = labels.reset_index(drop=True)
raw_evidence = raw_evidence.reset_index(drop=True)

labels["Raw_Label_10_12"] = raw_evidence["Raw_Label_10_12"]
labels["Raw_Attack_Cat_10_12"] = raw_evidence["Raw_Attack_Cat_10_12"]
labels["Flow_File_10_12"] = raw_evidence["Flow_File_10_12"]
labels["Flow_Row_10_12"] = raw_evidence["Flow_Row_10_12"]

del raw_evidence
gc.collect()


# --------------------------------------------------------------------------------------------------
# 10.12I — NORMALIZE RAW LABEL EVIDENCE
# --------------------------------------------------------------------------------------------------

banner("10.12I — NORMALIZE RAW LABEL EVIDENCE")


def normalize_family(series):

    s = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    null_tokens = {
        "",
        "nan",
        "none",
        "null",
        "<na>",
        "-"
    }

    s = s.mask(s.isin(null_tokens))

    return s


labels["Raw_Attack_Cat_Normalized"] = normalize_family(
    labels["Raw_Attack_Cat_10_12"]
)

labels["Raw_Label_Numeric"] = pd.to_numeric(
    labels["Raw_Label_10_12"],
    errors="coerce"
).astype("Int64")

print("Raw binary label distribution:")
print(labels["Raw_Label_Numeric"].value_counts(dropna=False))

print("\nRaw attack_cat distribution:")
print(
    labels["Raw_Attack_Cat_Normalized"]
    .value_counts(dropna=False)
)


# --------------------------------------------------------------------------------------------------
# 10.12J — CANONICAL FAMILY NORMALIZATION
#
# This is ONLY nomenclature normalization.
#
# UNSW contains both:
#
#     backdoor
#     backdoors
#
# They should not become separate model classes merely because of
# inconsistent singular/plural spelling.
#
# We preserve the original family separately.
# --------------------------------------------------------------------------------------------------

banner("10.12J — ATTACK-FAMILY CANONICALIZATION POLICY")

FAMILY_CANONICAL_MAP = {
    "backdoors": "backdoor"
}

print("Canonicalization map:")
for old, new in FAMILY_CANONICAL_MAP.items():
    print(f"  {old} → {new}")

print(
    "\nIMPORTANT:\n"
    "This changes naming only.\n"
    "It does NOT merge unrelated UNSW attack families."
)


def canonicalize_family(series):

    s = normalize_family(series)

    return s.replace(FAMILY_CANONICAL_MAP)


labels["Raw_Attack_Cat_Canonical"] = canonicalize_family(
    labels["Raw_Attack_Cat_Normalized"]
)

labels["Resolved_Family_PreCanonical"] = normalize_family(
    labels[family_col]
)

labels["Resolved_Family_Canonical"] = canonicalize_family(
    labels["Resolved_Family_PreCanonical"]
)


# --------------------------------------------------------------------------------------------------
# 10.12K — BASELINE SEMANTIC STATE FROM PHASE 10.11
# --------------------------------------------------------------------------------------------------

banner("10.12K — BASELINE PHASE 10.11 SEMANTIC STATE")

labels["Final_Attack_State"] = pd.to_numeric(
    labels[state_col],
    errors="coerce"
).astype("Int64")

labels["Final_Attack_Family"] = labels[
    "Resolved_Family_Canonical"
].copy()

labels["Family_Resolution_Source"] = pd.Series(
    pd.NA,
    index=labels.index,
    dtype="string"
)

labels["Semantic_Review_Status"] = pd.Series(
    pd.NA,
    index=labels.index,
    dtype="string"
)

status_norm = (
    labels[status_col]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Certified benign
mask_benign = status_norm.eq("CERTIFIED_BENIGN")

labels.loc[
    mask_benign,
    "Family_Resolution_Source"
] = "CERTIFIED_BENIGN_10_11"

labels.loc[
    mask_benign,
    "Semantic_Review_Status"
] = "RESOLVED"

# GT-resolved attacks
mask_gt_resolved = status_norm.eq("GT_RESOLVED_ATTACK")

labels.loc[
    mask_gt_resolved,
    "Family_Resolution_Source"
] = "GT_RESOLVED_10_11"

labels.loc[
    mask_gt_resolved,
    "Semantic_Review_Status"
] = "RESOLVED"

# Cross-family ambiguity
mask_cross_family = status_norm.eq(
    "GT_ATTACK_FAMILY_AMBIGUOUS"
)

labels.loc[
    mask_cross_family,
    "Family_Resolution_Source"
] = "GT_CROSS_FAMILY_AMBIGUOUS"

labels.loc[
    mask_cross_family,
    "Semantic_Review_Status"
] = "FAMILY_REVIEW"

# Invalid key
mask_invalid = status_norm.eq(
    "INVALID_NETWORK_KEY_REVIEW"
)

labels.loc[
    mask_invalid,
    "Family_Resolution_Source"
] = "INVALID_NETWORK_KEY"

labels.loc[
    mask_invalid,
    "Semantic_Review_Status"
] = "NETWORK_KEY_REVIEW"


print("Phase 10.11 status counts:")
print(status_norm.value_counts(dropna=False))


# --------------------------------------------------------------------------------------------------
# 10.12L — CROSS-FAMILY AMBIGUITY REVIEW
#
# IMPORTANT:
#
# We DO NOT simply overwrite the ambiguous GT family with raw attack_cat.
#
# First we measure what evidence exists.
#
# Phase 10.10/10.11 may contain candidate-family information in one or more
# columns. We inspect available columns rather than inventing it.
# --------------------------------------------------------------------------------------------------

banner("10.12L — CROSS-FAMILY AMBIGUITY REVIEW")

ambiguous = labels.loc[
    mask_cross_family
].copy()

print(f"Cross-family ambiguous flows: {len(ambiguous):,}")

candidate_family_columns = [
    c for c in labels.columns
    if (
        "candidate" in str(c).lower()
        and "famil" in str(c).lower()
    )
    or (
        "famil" in str(c).lower()
        and "conflict" in str(c).lower()
    )
]

print("\nPotential candidate-family evidence columns:")
print(candidate_family_columns if candidate_family_columns else "None found")


# --------------------------------------------------------------------------------------------------
# Conservative tie-break rule
#
# We ONLY permit raw attack_cat resolution if:
#
#   A) raw attack_cat is a valid non-null attack family
#   B) candidate-family evidence exists
#   C) the raw family is explicitly one of the candidate GT families
#
# If candidate-family evidence is absent, NO tie-breaking is performed.
# --------------------------------------------------------------------------------------------------

labels["Raw_Family_Eligible_Tiebreak"] = False
labels["Raw_Family_Tiebreak_Applied"] = False


def parse_candidate_families(value):

    if pd.isna(value):
        return set()

    text = str(value).strip().lower()

    if not text:
        return set()

    # Normalize common delimiters.
    for delim in ["|", ";", "/", ","]:
        text = text.replace(delim, " ")

    tokens = {
        t.strip()
        for t in text.split()
        if t.strip()
    }

    # Apply canonicalization to candidate names too.
    tokens = {
        FAMILY_CANONICAL_MAP.get(t, t)
        for t in tokens
    }

    return tokens


tiebreak_count = 0

if candidate_family_columns:

    # Build union of candidate-family evidence across discovered columns.
    # Only ambiguous rows are processed, so this is inexpensive.

    for idx in ambiguous.index:

        raw_family = labels.at[
            idx,
            "Raw_Attack_Cat_Canonical"
        ]

        if pd.isna(raw_family):
            continue

        candidates = set()

        for col in candidate_family_columns:
            candidates.update(
                parse_candidate_families(
                    labels.at[idx, col]
                )
            )

        if raw_family in candidates:

            labels.at[
                idx,
                "Raw_Family_Eligible_Tiebreak"
            ] = True

            labels.at[
                idx,
                "Final_Attack_Family"
            ] = raw_family

            labels.at[
                idx,
                "Family_Resolution_Source"
            ] = "RAW_ATTACK_CAT_CONFIRMED_BY_GT_CANDIDATES"

            labels.at[
                idx,
                "Semantic_Review_Status"
            ] = "RESOLVED"

            labels.at[
                idx,
                "Raw_Family_Tiebreak_Applied"
            ] = True

            tiebreak_count += 1

    print(
        "\nRaw attack_cat tie-breaks defensibly applied:",
        f"{tiebreak_count:,}"
    )

else:

    print(
        "\n⚠ Candidate GT family sets are not present in the Phase 10.11 artifact."
    )

    print(
        "Therefore raw attack_cat will NOT be used to force-resolution "
        "of cross-family conflicts."
    )


remaining_ambiguous = (
    mask_cross_family
    & ~labels["Raw_Family_Tiebreak_Applied"]
)

# Ensure unresolved family ambiguity remains null.
labels.loc[
    remaining_ambiguous,
    "Final_Attack_Family"
] = pd.NA

print(
    "\nRemaining family-ambiguous attack flows:",
    f"{remaining_ambiguous.sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 10.12M — RAW FAMILY DISTRIBUTION INSIDE CROSS-FAMILY AMBIGUITY
# --------------------------------------------------------------------------------------------------

banner("10.12M — RAW FAMILY EVIDENCE WITHIN CROSS-FAMILY AMBIGUITY")

ambiguous_family_distribution = (
    labels.loc[
        mask_cross_family,
        "Raw_Attack_Cat_Canonical"
    ]
    .value_counts(dropna=False)
    .rename_axis("Raw_Attack_Family")
    .reset_index(name="Flows")
)

ambiguous_family_distribution["Pct"] = (
    ambiguous_family_distribution["Flows"]
    / ambiguous_family_distribution["Flows"].sum()
    * 100
)

print(ambiguous_family_distribution.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 10.12N — INVALID NETWORK KEY REVIEW
# --------------------------------------------------------------------------------------------------

banner("10.12N — INVALID NETWORK KEY REVIEW")

invalid = labels.loc[
    mask_invalid
].copy()

print(f"Invalid-network-key flows: {len(invalid):,}")

if len(invalid) > 0:

    print("\nOriginal raw binary labels:")
    print(
        invalid["Raw_Label_Numeric"]
        .value_counts(dropna=False)
    )

    print("\nOriginal raw attack families:")
    print(
        invalid["Raw_Attack_Cat_Canonical"]
        .value_counts(dropna=False)
    )


# --------------------------------------------------------------------------------------------------
# Conservative invalid-key recovery
#
# Phase 10.11 output showed these were RAW_NORMAL_INVALID_KEY.
#
# We verify that here rather than assuming it.
#
# If:
#     Raw Label == 0
# AND raw attack_cat is null
#
# then we can resolve semantic ATTACK STATE as benign using original UNSW
# evidence, while preserving the fact that the network tuple itself was invalid.
#
# This is NOT repairing the network key.
# --------------------------------------------------------------------------------------------------

invalid_raw_normal = (
    mask_invalid
    & labels["Raw_Label_Numeric"].eq(0)
    & labels["Raw_Attack_Cat_Canonical"].isna()
)

invalid_not_safe = (
    mask_invalid
    & ~invalid_raw_normal
)

print(
    "\nInvalid-key rows independently supported as raw-normal:",
    f"{invalid_raw_normal.sum():,}"
)

print(
    "Invalid-key rows NOT safely resolvable:",
    f"{invalid_not_safe.sum():,}"
)


# We preserve network-key quality separately,
# but resolve semantic state if original UNSW evidence is unambiguous.

labels["Network_Key_Valid"] = ~mask_invalid

labels.loc[
    invalid_raw_normal,
    "Final_Attack_State"
] = 0

labels.loc[
    invalid_raw_normal,
    "Final_Attack_Family"
] = "normal"

labels.loc[
    invalid_raw_normal,
    "Family_Resolution_Source"
] = "RAW_NORMAL_INVALID_NETWORK_KEY"

labels.loc[
    invalid_raw_normal,
    "Semantic_Review_Status"
] = "RESOLVED_STATE_NETWORK_KEY_INVALID"


# --------------------------------------------------------------------------------------------------
# 10.12O — FORCE BASIC SEMANTIC CONSISTENCY
# --------------------------------------------------------------------------------------------------

banner("10.12O — BASIC SEMANTIC CONSISTENCY")

# All benign flows must use family "normal".
final_benign = labels["Final_Attack_State"].eq(0)

labels.loc[
    final_benign,
    "Final_Attack_Family"
] = "normal"

# Attack flows may have null family if family remains ambiguous.
final_attack = labels["Final_Attack_State"].eq(1)

# A non-normal attack family must never appear on benign state.
bad_benign_family = (
    final_benign
    & labels["Final_Attack_Family"].notna()
    & labels["Final_Attack_Family"].ne("normal")
)

# Attack state must never have family "normal".
bad_attack_normal = (
    final_attack
    & labels["Final_Attack_Family"].eq("normal")
)

print("Benign with non-normal family :", int(bad_benign_family.sum()))
print("Attack with family='normal'  :", int(bad_attack_normal.sum()))

if bad_benign_family.any():
    raise AssertionError(
        "Semantic inconsistency: benign flow has attack family."
    )

if bad_attack_normal.any():
    raise AssertionError(
        "Semantic inconsistency: attack flow has family='normal'."
    )

print("✓ Basic attack-state / family consistency valid")


# --------------------------------------------------------------------------------------------------
# 10.12P — ATTACH / RETAIN EPISODE PROVENANCE
#
# We do NOT reconstruct episodes again.
#
# We search the Phase 10.11 artifact first.
# If episode information already survived 10.11, preserve it.
#
# Otherwise inspect Phase 10.10 mapping files and attach only if exact positional
# alignment is available.
# --------------------------------------------------------------------------------------------------

banner("10.12P — EPISODE PROVENANCE")

episode_candidates = [
    c for c in labels.columns
    if "episode" in str(c).lower()
]

print("Episode-related columns already present:")
print(episode_candidates if episode_candidates else "None")


episode_id_col = None

preferred_episode_names = [
    "Episode_ID",
    "Attack_Episode_ID",
    "GT_Episode_ID",
    "Primary_Episode_ID"
]

for candidate in preferred_episode_names:

    for c in labels.columns:

        if str(c).strip().lower() == candidate.lower():
            episode_id_col = c
            break

    if episode_id_col is not None:
        break


# If episode ID not already present, try mapping artifacts.
if episode_id_col is None:

    print(
        "\nNo episode ID found in 10.11 artifact.\n"
        "Checking 10.10 mapping artifacts..."
    )

    mapping_episode_parts = []
    mapping_episode_col = None
    mapping_status_col = None

    for i, mapping_path in enumerate(MAPPING_FILES, start=1):

        # Read header only first.
        header = pd.read_csv(
            mapping_path,
            nrows=0
        )

        available = header.columns.tolist()

        local_episode = None

        for preferred in preferred_episode_names:

            for c in available:

                if str(c).strip().lower() == preferred.lower():
                    local_episode = c
                    break

            if local_episode is not None:
                break

        if local_episode is None:
            continue

        # Also preserve useful GT match provenance if present.
        possible_cols = [local_episode]

        for c in available:
            cl = str(c).lower()

            if (
                "match_status" in cl
                or "mapping_status" in cl
                or "gt_record" in cl
            ):
                possible_cols.append(c)

        possible_cols = list(dict.fromkeys(possible_cols))

        part = pd.read_csv(
            mapping_path,
            usecols=possible_cols,
            low_memory=False
        )

        part["Flow_File_10_12"] = RAW_FILES[i - 1].name
        part["Flow_Row_10_12"] = np.arange(
            len(part),
            dtype=np.int64
        )

        mapping_episode_parts.append(part)

        if mapping_episode_col is None:
            mapping_episode_col = local_episode

    if mapping_episode_parts:

        mapping_episode = pd.concat(
            mapping_episode_parts,
            ignore_index=True
        )

        if len(mapping_episode) != len(labels):
            raise ValueError(
                "10.10 episode mapping row count does not match final labels."
            )

        # Positional attachment is safe because each mapping file was previously
        # verified against its corresponding raw file in Phase 10.11.
        for c in mapping_episode.columns:

            if c not in {
                "Flow_File_10_12",
                "Flow_Row_10_12"
            }:

                target_name = c

                if target_name in labels.columns:
                    target_name = f"10_10_{target_name}"

                labels[target_name] = mapping_episode[c].values

                if (
                    str(c).strip().lower()
                    in [x.lower() for x in preferred_episode_names]
                    and episode_id_col is None
                ):
                    episode_id_col = target_name

        print(
            "✓ Episode provenance attached from Phase 10.10"
        )

        del mapping_episode
        del mapping_episode_parts
        gc.collect()

    else:

        print(
            "⚠ No episode identifier found in Phase 10.10 mappings.\n"
            "  Episode IDs will NOT be invented in Phase 10.12."
        )


print("\nSelected episode-ID column:", episode_id_col)


# --------------------------------------------------------------------------------------------------
# 10.12Q — FINAL ELIGIBILITY FLAGS
# --------------------------------------------------------------------------------------------------

banner("10.12Q — FINAL MODEL ELIGIBILITY FLAGS")

# Binary forecasting:
# eligible whenever attack state is known.
labels["Eligible_Binary_Forecasting"] = (
    labels["Final_Attack_State"].isin([0, 1])
)

# Family forecasting:
# We treat this as multiclass attack-family forecasting.
#
# Therefore:
#     • attack state == 1
#     • attack family known
#     • family != normal
#
# Benign is NOT counted here because this flag specifically refers to
# attack-family classification.
labels["Eligible_Attack_Family_Forecasting"] = (
    labels["Final_Attack_State"].eq(1)
    & labels["Final_Attack_Family"].notna()
    & labels["Final_Attack_Family"].ne("normal")
)

# Network-valid forecasting flag:
# useful later if certain features require valid endpoint keys.
labels["Eligible_Network_Key_Dependent_Modeling"] = (
    labels["Network_Key_Valid"]
    & labels["Eligible_Binary_Forecasting"]
)


print(
    "Binary-forecasting eligible flows:",
    f"{labels['Eligible_Binary_Forecasting'].sum():,}",
    "/",
    f"{len(labels):,}"
)

print(
    "Attack-family eligible attack flows:",
    f"{labels['Eligible_Attack_Family_Forecasting'].sum():,}"
)

print(
    "Network-key-valid binary flows:",
    f"{labels['Eligible_Network_Key_Dependent_Modeling'].sum():,}"
)


# --------------------------------------------------------------------------------------------------
# 10.12R — FINAL ATTACK-STATE DISTRIBUTION
# --------------------------------------------------------------------------------------------------

banner("10.12R — FINAL ATTACK-STATE DISTRIBUTION")

state_summary = (
    labels["Final_Attack_State"]
    .value_counts(dropna=False)
    .rename_axis("Final_Attack_State")
    .reset_index(name="Flows")
)

state_summary["Pct_All_Flows"] = (
    state_summary["Flows"]
    / len(labels)
    * 100
)

print(state_summary.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 10.12S — FINAL ATTACK-FAMILY DISTRIBUTION
# --------------------------------------------------------------------------------------------------

banner("10.12S — FINAL ATTACK-FAMILY DISTRIBUTION")

family_summary = (
    labels["Final_Attack_Family"]
    .value_counts(dropna=False)
    .rename_axis("Final_Attack_Family")
    .reset_index(name="Flows")
)

family_summary["Pct_All_Flows"] = (
    family_summary["Flows"]
    / len(labels)
    * 100
)

print(family_summary.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 10.12T — VERIFY BACKDOOR CANONICALIZATION
# --------------------------------------------------------------------------------------------------

banner("10.12T — FAMILY CANONICALIZATION ASSERTION")

remaining_backdoors_plural = (
    labels["Final_Attack_Family"]
    .astype("string")
    .str.lower()
    .eq("backdoors")
    .sum()
)

print(
    "Remaining literal 'backdoors' labels:",
    f"{remaining_backdoors_plural:,}"
)

if remaining_backdoors_plural != 0:
    raise AssertionError(
        "Plural backdoors label remains after canonicalization."
    )

print("✓ backdoor/backdoors naming canonicalized")


# --------------------------------------------------------------------------------------------------
# 10.12U — EPISODE COVERAGE AUDIT
# --------------------------------------------------------------------------------------------------

banner("10.12U — EPISODE COVERAGE AUDIT")

if episode_id_col is not None:

    attack_rows = labels["Final_Attack_State"].eq(1)

    episode_known = (
        attack_rows
        & labels[episode_id_col].notna()
    )

    print(
        "Attack flows with episode ID:",
        f"{episode_known.sum():,}",
        "/",
        f"{attack_rows.sum():,}"
    )

    print(
        "Unique episode IDs:",
        f"{labels.loc[episode_known, episode_id_col].nunique():,}"
    )

else:

    print(
        "Episode ID unavailable in surviving Phase 10.10/10.11 artifacts.\n"
        "No episode ID has been fabricated."
    )


# --------------------------------------------------------------------------------------------------
# 10.12V — FINAL SEMANTIC REVIEW STATUS
# --------------------------------------------------------------------------------------------------

banner("10.12V — FINAL SEMANTIC REVIEW STATUS")

# Anything still without known attack state requires semantic review.
unknown_state = labels["Final_Attack_State"].isna()

labels.loc[
    unknown_state,
    "Semantic_Review_Status"
] = "UNRESOLVED_ATTACK_STATE"

# Any known attack with null family is binary-resolved but family-ambiguous.
binary_attack_family_unknown = (
    labels["Final_Attack_State"].eq(1)
    & labels["Final_Attack_Family"].isna()
)

labels.loc[
    binary_attack_family_unknown,
    "Semantic_Review_Status"
] = "BINARY_RESOLVED_FAMILY_AMBIGUOUS"


review_summary = (
    labels["Semantic_Review_Status"]
    .value_counts(dropna=False)
    .rename_axis("Semantic_Review_Status")
    .reset_index(name="Flows")
)

review_summary["Pct_All_Flows"] = (
    review_summary["Flows"]
    / len(labels)
    * 100
)

print(review_summary.to_string(index=False))


# --------------------------------------------------------------------------------------------------
# 10.12W — CRITICAL FREEZE ASSERTIONS
# --------------------------------------------------------------------------------------------------

banner("10.12W — CRITICAL FREEZE ASSERTIONS")

# 1. Row conservation
assert len(labels) == EXPECTED_TOTAL
print("✓ Global row conservation valid")


# 2. Every certified benign flow remains benign.
certified_benign_bad = (
    mask_benign
    & ~labels["Final_Attack_State"].eq(0)
)

assert certified_benign_bad.sum() == 0
print("✓ All Phase 10.11 certified benign flows remain benign")


# 3. Every GT-resolved attack remains attack.
resolved_attack_bad = (
    mask_gt_resolved
    & ~labels["Final_Attack_State"].eq(1)
)

assert resolved_attack_bad.sum() == 0
print("✓ All GT-resolved attacks remain attacks")


# 4. Every cross-family ambiguous flow remains binary attack.
cross_binary_bad = (
    mask_cross_family
    & ~labels["Final_Attack_State"].eq(1)
)

assert cross_binary_bad.sum() == 0
print("✓ Cross-family conflicts retain binary ATTACK state")


# 5. Any unresolved cross-family flow must have null final family.
unresolved_cross_bad = (
    remaining_ambiguous
    & labels["Final_Attack_Family"].notna()
)

assert unresolved_cross_bad.sum() == 0
print("✓ Unresolved cross-family conflicts retain null family")


# 6. No attack has family normal.
assert bad_attack_normal.sum() == 0
print("✓ No attack flow labeled as family='normal'")


# 7. No benign flow has attack family.
assert bad_benign_family.sum() == 0
print("✓ No benign flow carries an attack family")


# 8. No rows silently lost.
assert labels.index.size == EXPECTED_TOTAL
print("✓ No flow silently dropped")


# --------------------------------------------------------------------------------------------------
# 10.12X — SAVE AUDIT TABLES
# --------------------------------------------------------------------------------------------------

banner("10.12X — SAVE PHASE 10.12 AUDIT TABLES")

ambiguous_out = OUT_DIR / "10_12_cross_family_review.csv"
invalid_out = OUT_DIR / "10_12_invalid_network_key_review.csv"
family_dist_out = OUT_DIR / "10_12_final_family_distribution.csv"
state_dist_out = OUT_DIR / "10_12_final_state_distribution.csv"
review_out = OUT_DIR / "10_12_semantic_review_summary.csv"
ambig_family_out = OUT_DIR / "10_12_ambiguous_raw_family_distribution.csv"

labels.loc[
    mask_cross_family
].to_csv(
    ambiguous_out,
    index=False
)

labels.loc[
    mask_invalid
].to_csv(
    invalid_out,
    index=False
)

family_summary.to_csv(
    family_dist_out,
    index=False
)

state_summary.to_csv(
    state_dist_out,
    index=False
)

review_summary.to_csv(
    review_out,
    index=False
)

ambiguous_family_distribution.to_csv(
    ambig_family_out,
    index=False
)

print("✓", ambiguous_out)
print("✓", invalid_out)
print("✓", family_dist_out)
print("✓", state_dist_out)
print("✓", review_out)
print("✓", ambig_family_out)


# --------------------------------------------------------------------------------------------------
# 10.12Y — CREATE AUTHORITATIVE FROZEN UNSW SEMANTIC ARTIFACT
# --------------------------------------------------------------------------------------------------

banner("10.12Y — CREATE AUTHORITATIVE FROZEN UNSW SEMANTIC ARTIFACT")

# Preserve all Phase 10.11 provenance columns plus final Phase 10.12 semantics.
#
# This artifact becomes the semantic source of truth for subsequent UNSW processing.

FINAL_OUT = OUT_DIR / "10_12_UNSW_FROZEN_FLOW_SEMANTICS.csv"

labels.to_csv(
    FINAL_OUT,
    index=False
)

print("✓ Saved:")
print(FINAL_OUT)


# --------------------------------------------------------------------------------------------------
# 10.12Z — COMPACT FREEZE MANIFEST
# --------------------------------------------------------------------------------------------------

banner("10.12Z — SEMANTIC FREEZE MANIFEST")

final_total = len(labels)

final_binary = int(
    labels["Eligible_Binary_Forecasting"].sum()
)

final_family = int(
    labels["Eligible_Attack_Family_Forecasting"].sum()
)

final_attack_count = int(
    labels["Final_Attack_State"].eq(1).sum()
)

final_benign_count = int(
    labels["Final_Attack_State"].eq(0).sum()
)

final_unknown_count = int(
    labels["Final_Attack_State"].isna().sum()
)

final_family_ambiguous = int(
    (
        labels["Final_Attack_State"].eq(1)
        & labels["Final_Attack_Family"].isna()
    ).sum()
)

tiebreak_applied = int(
    labels["Raw_Family_Tiebreak_Applied"].sum()
)

network_invalid = int(
    (~labels["Network_Key_Valid"]).sum()
)


manifest = pd.DataFrame(
    {
        "Metric": [
            "Total_Flows",
            "Final_Benign_Flows",
            "Final_Attack_Flows",
            "Unknown_Attack_State",
            "Binary_Forecasting_Eligible",
            "Attack_Family_Forecasting_Eligible",
            "Family_Ambiguous_Attacks",
            "Raw_Family_Tiebreaks_Applied",
            "Invalid_Network_Key_Flows",
            "Canonical_Attack_Families"
        ],
        "Value": [
            final_total,
            final_benign_count,
            final_attack_count,
            final_unknown_count,
            final_binary,
            final_family,
            final_family_ambiguous,
            tiebreak_applied,
            network_invalid,
            labels.loc[
                labels["Final_Attack_State"].eq(1)
                & labels["Final_Attack_Family"].notna(),
                "Final_Attack_Family"
            ].nunique()
        ]
    }
)

print(manifest.to_string(index=False))

MANIFEST_OUT = OUT_DIR / "10_12_semantic_freeze_manifest.csv"

manifest.to_csv(
    MANIFEST_OUT,
    index=False
)

print("\n✓ Saved:")
print(MANIFEST_OUT)


# --------------------------------------------------------------------------------------------------
# COMPLETE
# --------------------------------------------------------------------------------------------------

banner("PHASE 10.12 COMPLETE")

print(
    f"""
UNSW FLOW-LEVEL SEMANTIC LAYER IS NOW FROZEN.

TOTAL FLOWS
-----------
{final_total:,}

FINAL BINARY STATE
------------------
Benign : {final_benign_count:,}
Attack : {final_attack_count:,}
Unknown: {final_unknown_count:,}

MODEL ELIGIBILITY
-----------------
Binary forecasting:
{final_binary:,}

Attack-family forecasting:
{final_family:,}

Family-ambiguous attacks remaining:
{final_family_ambiguous:,}

RAW attack_cat TIE-BREAKS
-------------------------
Applied only where GT candidate-family evidence explicitly supported it:
{tiebreak_applied:,}

INVALID NETWORK KEYS
--------------------
{network_invalid:,}

IMPORTANT
---------
• No cross-family conflict was arbitrarily assigned.
• No unresolved attack was converted to benign.
• No attack was converted to normal.
• No flow was silently dropped.
• backdoor/backdoors naming was canonicalized.
• Original semantic provenance was preserved.
• Episode IDs were retained/attached only when already supported
  by Phase 10.10/10.11 artifacts.

AUTHORITATIVE OUTPUT
--------------------
{FINAL_OUT}

NEXT
----
Do NOT train yet.

Next phase should move from FLOW semantics to the TEMPORAL representation:

    frozen UNSW flow semantics
            ↓
    temporal/capture reconstruction
            ↓
    episode-linked temporal aggregation
            ↓

Then we construct the common CIC ↔ UNSW semantic feature layer.

After BOTH datasets have compatible temporal representations:

    known-type episode-aware split design
            ↓
    leakage audit
            ↓
    forecasting target construction
            ↓
    model-ready windows
            ↓
    baselines / LSTM / GRU / temporal boosting
"""
)

print(
    f"\nRuntime: {time.time() - PHASE_START:.2f} sec"
)

In [ ]:
# ==================================================================================================
# PHASE 10.13 — CIC ↔ UNSW COMMON FEATURE COMPATIBILITY AUDIT
#
# PURPOSE
# -------
# We now have:
#
#   CIC-IDS2017  -> audited / processed previously
#   UNSW-NB15    -> authoritative flow semantics frozen in Phase 10.12
#
# Before temporal aggregation of UNSW, determine what information can be represented
# consistently across BOTH datasets.
#
# IMPORTANT:
#   • This is an AUDIT, not a transformation.
#   • No CIC + UNSW concatenation occurs here.
#   • No feature is accepted merely because names look similar.
#   • No target / label columns may become predictor features.
#   • Dataset-specific identifiers must not leak into model features.
#   • Raw IPs, timestamps and attack labels are metadata/semantics, not ordinary predictors.
#
# OUTPUT:
#   10_13_CIC_UNSW_schema_inventory.csv
#   10_13_CIC_UNSW_feature_compatibility.csv
#   10_13_proposed_common_temporal_features.csv
#   10_13_dataset_specific_features.csv
#   10_13_feature_compatibility_manifest.csv
#
# This should be a LIGHTWEIGHT cell.
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import warnings

warnings.filterwarnings("ignore")

START_TIME = time.time()

# ==================================================================================================
# 10.13A — PATHS
# ==================================================================================================

BASE = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

CIC_RAW_DIR = BASE / "raw" / "CIC_IDS2017"
UNSW_RAW_DIR = BASE / "raw" / "UNSW_NB15"

CIC_OUT_DIR = BASE / "audit_outputs" / "CIC"
UNSW_OUT_DIR = BASE / "audit_outputs" / "UNSW"

COMMON_OUT_DIR = BASE / "audit_outputs" / "COMMON"

COMMON_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

UNSW_FEATURE_FILE = (
    UNSW_RAW_DIR / "NUSW-NB15_features.csv"
)

UNSW_FROZEN_FILE = (
    UNSW_OUT_DIR / "10_12_UNSW_FROZEN_FLOW_SEMANTICS.csv"
)


def banner(text):
    print("\n" + "=" * 120)
    print(text)
    print("=" * 120)


def subbanner(text):
    print("\n" + "-" * 120)
    print(text)
    print("-" * 120)


banner(
    "PHASE 10.13 — CIC ↔ UNSW COMMON FEATURE COMPATIBILITY AUDIT"
)


# ==================================================================================================
# 10.13B — VERIFY UNSW INPUTS
# ==================================================================================================

banner("10.13B — VERIFY UNSW INPUTS")

if not UNSW_FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Missing UNSW metadata:\n{UNSW_FEATURE_FILE}"
    )

if not UNSW_FROZEN_FILE.exists():
    raise FileNotFoundError(
        f"Missing Phase 10.12 artifact:\n{UNSW_FROZEN_FILE}"
    )

print("✓ NUSW-NB15_features.csv")
print("✓ 10_12_UNSW_FROZEN_FLOW_SEMANTICS.csv")


# ==================================================================================================
# 10.13C — LOCATE CIC RAW CSV FILES
#
# We only need ONE header because CIC's schema uniformity was already audited.
# Nevertheless, we verify discovered schemas rather than assuming.
# ==================================================================================================

banner("10.13C — DISCOVER CIC CSV FILES")

cic_csv_files = sorted(
    [
        p
        for p in CIC_RAW_DIR.rglob("*.csv")
        if p.is_file()
    ]
)

# In case the actual directory uses a slightly different spelling,
# search the general raw directory if needed.

if len(cic_csv_files) == 0:

    print(
        "⚠ No CSVs found under expected CIC_IDS2017 directory."
    )

    print(
        "Searching raw directory for CIC-like CSV files..."
    )

    candidates = []

    for p in (BASE / "raw").rglob("*.csv"):

        name = str(p).lower()

        if (
            "cic" in name
            or "monday" in name
            or "tuesday" in name
            or "wednesday" in name
            or "thursday" in name
            or "friday" in name
        ):
            candidates.append(p)

    cic_csv_files = sorted(candidates)


if len(cic_csv_files) == 0:

    raise FileNotFoundError(
        "Could not locate CIC-IDS2017 CSV files.\n"
        "Check the CIC raw dataset directory."
    )


print(f"CIC CSV files discovered: {len(cic_csv_files)}")

for p in cic_csv_files:
    print("✓", p.name)


# ==================================================================================================
# 10.13D — READ CIC HEADERS ONLY
#
# IMPORTANT:
# We do NOT load the ~3M CIC flows.
# Header-only reads make this very fast.
# ==================================================================================================

banner("10.13D — CIC SCHEMA AUDIT")

cic_schemas = {}

for path in cic_csv_files:

    try:

        header = pd.read_csv(
            path,
            nrows=0,
            encoding="utf-8"
        )

    except UnicodeDecodeError:

        header = pd.read_csv(
            path,
            nrows=0,
            encoding="cp1252"
        )

    cols = [
        str(c).strip()
        for c in header.columns
    ]

    cic_schemas[path.name] = cols


schema_lengths = {
    file: len(cols)
    for file, cols in cic_schemas.items()
}

print("Columns per CIC file:")

for file, count in schema_lengths.items():
    print(f"{file}: {count}")


# Determine schema consistency.

schema_signatures = {
    tuple(cols)
    for cols in cic_schemas.values()
}

print(
    "\nUnique CIC schema signatures:",
    len(schema_signatures)
)

if len(schema_signatures) == 1:

    print(
        "✓ CIC schema is uniform across discovered files"
    )

else:

    print(
        "⚠ CIC files do not all expose identical headers."
    )

    print(
        "The compatibility audit will use the UNION "
        "of discovered CIC columns."
    )


cic_columns = sorted(
    set(
        col
        for cols in cic_schemas.values()
        for col in cols
    )
)

print(
    "\nUnique CIC columns:",
    len(cic_columns)
)


# ==================================================================================================
# 10.13E — LOAD UNSW FEATURE METADATA
# ==================================================================================================

banner("10.13E — LOAD UNSW FEATURE METADATA")


def read_with_encoding(path, **kwargs):

    for enc in [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]:

        try:

            return (
                pd.read_csv(
                    path,
                    encoding=enc,
                    **kwargs
                ),
                enc
            )

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"Unable to decode {path}"
    )


unsw_features, unsw_encoding = read_with_encoding(
    UNSW_FEATURE_FILE,
    low_memory=False
)

unsw_features.columns = (
    unsw_features.columns
    .astype(str)
    .str.strip()
)


name_col = None
type_col = None
description_col = None

for col in unsw_features.columns:

    cl = col.lower().strip()

    if cl == "name":
        name_col = col

    elif cl == "type":
        type_col = col

    elif cl == "description":
        description_col = col


if name_col is None:

    raise ValueError(
        "Could not identify UNSW feature-name column."
    )


unsw_columns = (
    unsw_features[name_col]
    .astype(str)
    .str.strip()
    .tolist()
)


print("Encoding       :", unsw_encoding)
print("UNSW features  :", len(unsw_columns))

if len(unsw_columns) != 49:

    print(
        "⚠ Expected 49 UNSW raw features, "
        f"found {len(unsw_columns)}"
    )

else:

    print(
        "✓ 49-column UNSW raw schema recovered"
    )


# ==================================================================================================
# 10.13F — BUILD COMPLETE SCHEMA INVENTORY
# ==================================================================================================

banner("10.13F — BUILD SCHEMA INVENTORY")


def normalize_name(name):

    name = str(name).strip().lower()

    name = re.sub(
        r"[^a-z0-9]+",
        "_",
        name
    )

    return name.strip("_")


inventory_rows = []


# CIC
for feature in cic_columns:

    inventory_rows.append(
        {
            "Dataset": "CIC_IDS2017",
            "Original_Feature": feature,
            "Normalized_Feature": normalize_name(feature),
            "Declared_Type": pd.NA,
            "Description": pd.NA
        }
    )


# UNSW
for _, row in unsw_features.iterrows():

    feature = str(
        row[name_col]
    ).strip()

    declared_type = (
        row[type_col]
        if type_col is not None
        else pd.NA
    )

    description = (
        row[description_col]
        if description_col is not None
        else pd.NA
    )

    inventory_rows.append(
        {
            "Dataset": "UNSW_NB15",
            "Original_Feature": feature,
            "Normalized_Feature": normalize_name(feature),
            "Declared_Type": declared_type,
            "Description": description
        }
    )


inventory = pd.DataFrame(
    inventory_rows
)

print(
    "Inventory rows:",
    len(inventory)
)

print(
    "\nCIC features:",
    (inventory["Dataset"] == "CIC_IDS2017").sum()
)

print(
    "UNSW features:",
    (inventory["Dataset"] == "UNSW_NB15").sum()
)


# ==================================================================================================
# 10.13G — SEMANTIC FEATURE CLASSIFICATION
#
# We classify by concept, not just by column-name equality.
#
# These categories tell us whether something can become:
#
#   DIRECT_COMMON
#   DERIVED_COMMON
#   METADATA_ONLY
#   LABEL_TARGET
#   DATASET_SPECIFIC
# ==================================================================================================

banner("10.13G — SEMANTIC FEATURE CLASSIFICATION")


def semantic_group(dataset, feature):

    f = normalize_name(feature)

    # ----------------------------------------------------------------------------------------------
    # LABELS / TARGETS
    # ----------------------------------------------------------------------------------------------

    label_tokens = {
        "label",
        "attack_cat",
        "attack_category",
        "attack_state",
        "attack_family",
        "class"
    }

    if f in label_tokens:
        return "LABEL_TARGET"


    # ----------------------------------------------------------------------------------------------
    # TIME
    # ----------------------------------------------------------------------------------------------

    if (
        "timestamp" in f
        or f in {
            "stime",
            "ltime",
            "start_time",
            "last_time"
        }
    ):
        return "TIMESTAMP_METADATA"


    # ----------------------------------------------------------------------------------------------
    # RAW NETWORK IDENTIFIERS
    # ----------------------------------------------------------------------------------------------

    if f in {
        "srcip",
        "src_ip",
        "source_ip",
        "dstip",
        "dst_ip",
        "destination_ip",
        "sport",
        "src_port",
        "source_port",
        "dsport",
        "dst_port",
        "destination_port"
    }:
        return "NETWORK_IDENTIFIER"


    # ----------------------------------------------------------------------------------------------
    # PROTOCOL / SERVICE / STATE
    # ----------------------------------------------------------------------------------------------

    if f in {
        "proto",
        "protocol"
    }:
        return "PROTOCOL"


    if f in {
        "service"
    }:
        return "SERVICE"


    if f in {
        "state"
    }:
        return "CONNECTION_STATE"


    # ----------------------------------------------------------------------------------------------
    # FLOW DURATION
    # ----------------------------------------------------------------------------------------------

    if (
        f in {
            "dur",
            "flow_duration"
        }
        or "flow_duration" in f
    ):
        return "FLOW_DURATION"


    # ----------------------------------------------------------------------------------------------
    # PACKET COUNTS
    # ----------------------------------------------------------------------------------------------

    if f in {
        "spkts",
        "total_fwd_packets"
    }:
        return "SRC_FWD_PACKET_COUNT"


    if f in {
        "dpkts",
        "total_backward_packets"
    }:
        return "DST_BWD_PACKET_COUNT"


    if (
        "packet" in f
        and "total" in f
    ):
        return "PACKET_COUNT_RELATED"


    # ----------------------------------------------------------------------------------------------
    # BYTE COUNTS
    # ----------------------------------------------------------------------------------------------

    if f in {
        "sbytes",
        "total_length_of_fwd_packets"
    }:
        return "SRC_FWD_BYTE_COUNT"


    if f in {
        "dbytes",
        "total_length_of_bwd_packets"
    }:
        return "DST_BWD_BYTE_COUNT"


    if (
        "byte" in f
        and "flow" not in f
    ):
        return "BYTE_RELATED"


    # ----------------------------------------------------------------------------------------------
    # RATE
    # ----------------------------------------------------------------------------------------------

    if f in {
        "rate",
        "flow_packets_s",
        "flow_bytes_s"
    }:
        return "RATE_RELATED"


    # ----------------------------------------------------------------------------------------------
    # TTL
    # ----------------------------------------------------------------------------------------------

    if "ttl" in f:
        return "TTL_RELATED"


    # ----------------------------------------------------------------------------------------------
    # LOAD
    # ----------------------------------------------------------------------------------------------

    if "load" in f:
        return "LOAD_RELATED"


    # ----------------------------------------------------------------------------------------------
    # INTER-ARRIVAL TIMES
    # ----------------------------------------------------------------------------------------------

    if (
        "iat" in f
        or "interarrival" in f
        or "inter_arrival" in f
        or f in {
            "sinpkt",
            "dinpkt"
        }
    ):
        return "INTERARRIVAL_RELATED"


    # ----------------------------------------------------------------------------------------------
    # TCP WINDOW / TCP BEHAVIOUR
    # ----------------------------------------------------------------------------------------------

    if (
        "win" in f
        or "tcp" in f
        or "syn" in f
        or "ack" in f
        or "fin" in f
        or "rst" in f
        or "psh" in f
        or "urg" in f
        or "ece" in f
        or "cwr" in f
    ):
        return "TCP_RELATED"


    # ----------------------------------------------------------------------------------------------
    # ACTIVE / IDLE
    # ----------------------------------------------------------------------------------------------

    if (
        "active" in f
        or "idle" in f
    ):
        return "ACTIVITY_RELATED"


    # ----------------------------------------------------------------------------------------------
    # HEADER LENGTH
    # ----------------------------------------------------------------------------------------------

    if (
        "header" in f
        or f in {
            "smeansz",
            "dmeansz"
        }
    ):
        return "PACKET_STRUCTURE_RELATED"


    # ----------------------------------------------------------------------------------------------
    # UNSW engineered history / connection-count features
    # ----------------------------------------------------------------------------------------------

    if (
        f.startswith("ct_")
        or f in {
            "is_sm_ips_ports",
            "is_ftp_login",
            "trans_depth",
            "response_body_len",
            "sjit",
            "djit",
            "sloss",
            "dloss"
        }
    ):
        return "UNSW_ENGINEERED"


    return "OTHER"


inventory["Semantic_Group"] = inventory.apply(
    lambda r: semantic_group(
        r["Dataset"],
        r["Original_Feature"]
    ),
    axis=1
)


print(
    inventory.groupby(
        [
            "Dataset",
            "Semantic_Group"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .to_string()
)


# ==================================================================================================
# 10.13H — EXACT NAME OVERLAP
#
# This is informational only.
# Exact name equality does NOT establish semantic compatibility.
# ==================================================================================================

banner("10.13H — EXACT NORMALIZED NAME OVERLAP")

cic_norm = {
    normalize_name(c)
    for c in cic_columns
}

unsw_norm = {
    normalize_name(c)
    for c in unsw_columns
}

exact_overlap = sorted(
    cic_norm & unsw_norm
)

print(
    "Exact normalized-name overlaps:",
    len(exact_overlap)
)

if exact_overlap:

    for x in exact_overlap:
        print("•", x)

else:

    print(
        "No exact normalized-name overlap."
    )


# ==================================================================================================
# 10.13I — SEMANTIC GROUP OVERLAP
#
# THIS is more important than exact names.
# ==================================================================================================

banner("10.13I — SEMANTIC GROUP OVERLAP")

cic_groups = set(
    inventory.loc[
        inventory["Dataset"] == "CIC_IDS2017",
        "Semantic_Group"
    ]
)

unsw_groups = set(
    inventory.loc[
        inventory["Dataset"] == "UNSW_NB15",
        "Semantic_Group"
    ]
)

shared_groups = sorted(
    cic_groups & unsw_groups
)

cic_only_groups = sorted(
    cic_groups - unsw_groups
)

unsw_only_groups = sorted(
    unsw_groups - cic_groups
)


print("SHARED SEMANTIC GROUPS")
print("----------------------")

for g in shared_groups:
    print("✓", g)


print("\nCIC-ONLY SEMANTIC GROUPS")
print("------------------------")

for g in cic_only_groups:
    print("•", g)


print("\nUNSW-ONLY SEMANTIC GROUPS")
print("-------------------------")

for g in unsw_only_groups:
    print("•", g)


# ==================================================================================================
# 10.13J — DEFINE PROPOSED COMMON TEMPORAL FEATURE VOCABULARY
#
# IMPORTANT:
#
# We are NOT saying that CIC's raw feature X == UNSW's raw feature Y.
#
# Instead, these are temporal statistics that BOTH datasets should be capable
# of producing from their flow records.
#
# Example:
#
#     FLOW_COUNT_PER_MINUTE
#
# does not require an identical raw feature.
#
# Both datasets contain flows, so both can count flows in each minute.
# ==================================================================================================

banner("10.13J — PROPOSED COMMON TEMPORAL FEATURE VOCABULARY")


proposed_common_features = [

    # ----------------------------------------------------------------------------------------------
    # BASIC ACTIVITY
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Feature": "Flow_Count",
        "Category": "TRAFFIC_VOLUME",
        "Construction": "Count flows in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    # ----------------------------------------------------------------------------------------------
    # FLOW DURATION
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Feature": "Flow_Duration_Mean",
        "Category": "FLOW_DURATION",
        "Construction": "Mean flow duration in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    {
        "Common_Feature": "Flow_Duration_Std",
        "Category": "FLOW_DURATION",
        "Construction": "Std of flow duration in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    {
        "Common_Feature": "Flow_Duration_Max",
        "Category": "FLOW_DURATION",
        "Construction": "Maximum flow duration in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    # ----------------------------------------------------------------------------------------------
    # PACKETS
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Feature": "Forward_Packets_Sum",
        "Category": "PACKETS",
        "Construction": "Sum source/forward packet counts in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    {
        "Common_Feature": "Backward_Packets_Sum",
        "Category": "PACKETS",
        "Construction": "Sum destination/backward packet counts in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    {
        "Common_Feature": "Total_Packets",
        "Category": "PACKETS",
        "Construction": "Forward packets + backward packets",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    # ----------------------------------------------------------------------------------------------
    # BYTES
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Feature": "Forward_Bytes_Sum",
        "Category": "BYTES",
        "Construction": "Sum source/forward bytes in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    {
        "Common_Feature": "Backward_Bytes_Sum",
        "Category": "BYTES",
        "Construction": "Sum destination/backward bytes in temporal bin",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    {
        "Common_Feature": "Total_Bytes",
        "Category": "BYTES",
        "Construction": "Forward bytes + backward bytes",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "KEEP"
    },

    # ----------------------------------------------------------------------------------------------
    # PROTOCOL BEHAVIOUR
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Feature": "TCP_Flow_Count",
        "Category": "PROTOCOL",
        "Construction": "Count TCP flows in temporal bin",
        "CIC_Support": "VERIFY",
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "VERIFY"
    },

    {
        "Common_Feature": "UDP_Flow_Count",
        "Category": "PROTOCOL",
        "Construction": "Count UDP flows in temporal bin",
        "CIC_Support": "VERIFY",
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "VERIFY"
    },

    {
        "Common_Feature": "Protocol_Diversity",
        "Category": "PROTOCOL",
        "Construction": "Number of unique protocols in temporal bin",
        "CIC_Support": "VERIFY",
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "VERIFY"
    },

    # ----------------------------------------------------------------------------------------------
    # RATE FEATURES
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Feature": "Packets_Per_Second",
        "Category": "RATE",
        "Construction": "Temporal-bin packet total divided by bin duration",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "DERIVE"
    },

    {
        "Common_Feature": "Bytes_Per_Second",
        "Category": "RATE",
        "Construction": "Temporal-bin byte total divided by bin duration",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "DERIVE"
    },

    # ----------------------------------------------------------------------------------------------
    # DIRECTIONAL RATIOS
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Feature": "Forward_Backward_Packet_Ratio",
        "Category": "DIRECTIONAL",
        "Construction": "Forward packet total / backward packet total",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "DERIVE"
    },

    {
        "Common_Feature": "Forward_Backward_Byte_Ratio",
        "Category": "DIRECTIONAL",
        "Construction": "Forward byte total / backward byte total",
        "CIC_Support": True,
        "UNSW_Support": True,
        "Causality_Status": "SAFE",
        "Recommendation": "DERIVE"
    },
]


common_features = pd.DataFrame(
    proposed_common_features
)

print(
    common_features.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13K — POTENTIALLY USEFUL BUT NOT YET ACCEPTED FEATURES
#
# These require deeper equivalence checks.
# ==================================================================================================

banner("10.13K — FEATURES REQUIRING SEMANTIC VERIFICATION")


verification_candidates = pd.DataFrame(
    [
        {
            "Feature_Family": "Interarrival_Time",
            "Reason": (
                "Both datasets contain timing-related flow statistics, "
                "but definitions may differ."
            ),
            "Decision": "VERIFY_BEFORE_USE"
        },

        {
            "Feature_Family": "TCP_Flags",
            "Reason": (
                "CIC contains explicit TCP flag counters; UNSW contains "
                "TCP/state-related information but not necessarily equivalent counters."
            ),
            "Decision": "DO_NOT_FORCE_MATCH"
        },

        {
            "Feature_Family": "Packet_Size_Statistics",
            "Reason": (
                "Both contain packet/byte structure information, but exact definitions "
                "and directionality require verification."
            ),
            "Decision": "VERIFY_BEFORE_USE"
        },

        {
            "Feature_Family": "Connection_State",
            "Reason": (
                "UNSW exposes state explicitly. CIC representation differs."
            ),
            "Decision": "DATASET_SPECIFIC_UNTIL_PROVEN_COMMON"
        },

        {
            "Feature_Family": "TTL",
            "Reason": (
                "UNSW provides source/destination TTL; equivalent CIC flow fields "
                "are not guaranteed."
            ),
            "Decision": "LIKELY_UNSW_SPECIFIC"
        },

        {
            "Feature_Family": "UNSW_ct_*",
            "Reason": (
                "UNSW ct_* fields are dataset-specific engineered connection-history features."
            ),
            "Decision": "EXCLUDE_FROM_COMMON_SCHEMA"
        },

        {
            "Feature_Family": "CIC_Bulk_Subflow_Features",
            "Reason": (
                "CIC provides several bulk/subflow statistics without guaranteed UNSW equivalents."
            ),
            "Decision": "EXCLUDE_FROM_COMMON_SCHEMA_UNLESS_DERIVED_EQUIVALENT"
        }
    ]
)

print(
    verification_candidates.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13L — LEAKAGE / METADATA POLICY
# ==================================================================================================

banner("10.13L — LEAKAGE & METADATA POLICY")


policy = pd.DataFrame(
    [
        {
            "Information": "Timestamp",
            "Role": "TEMPORAL_INDEX",
            "Predictor": "NO_DIRECT_RAW_TIMESTAMP",
            "Reason": (
                "Required for ordering/binning; raw absolute timestamp can encode dataset/capture identity."
            )
        },

        {
            "Information": "Attack_State",
            "Role": "TARGET_SEMANTICS",
            "Predictor": "NO",
            "Reason": "Future forecasting target / evaluation semantics."
        },

        {
            "Information": "Attack_Family",
            "Role": "TARGET_SEMANTICS",
            "Predictor": "NO",
            "Reason": "Attack-family target."
        },

        {
            "Information": "Episode_ID",
            "Role": "SPLIT_METADATA",
            "Predictor": "NO",
            "Reason": "Used to prevent episode leakage."
        },

        {
            "Information": "Dataset_ID",
            "Role": "PROVENANCE",
            "Predictor": "NO",
            "Reason": (
                "Would allow model to learn CIC-vs-UNSW identity instead of network behaviour."
            )
        },

        {
            "Information": "Capture_ID",
            "Role": "SPLIT_TEMPORAL_METADATA",
            "Predictor": "NO",
            "Reason": (
                "Used to preserve capture boundaries, not as predictive input."
            )
        },

        {
            "Information": "Raw_Source_IP",
            "Role": "NETWORK_METADATA",
            "Predictor": "NO_RAW_VALUE",
            "Reason": (
                "Raw addresses can encode dataset topology and attack identity."
            )
        },

        {
            "Information": "Raw_Destination_IP",
            "Role": "NETWORK_METADATA",
            "Predictor": "NO_RAW_VALUE",
            "Reason": (
                "Raw addresses can encode dataset topology and attack identity."
            )
        },

        {
            "Information": "IP_Diversity_Counts",
            "Role": "DERIVED_BEHAVIOUR",
            "Predictor": "POTENTIALLY",
            "Reason": (
                "Counts/diversity may be useful if derivable consistently and causally "
                "from both datasets."
            )
        }
    ]
)

print(
    policy.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13M — DATASET-SPECIFIC FEATURE INVENTORY
# ==================================================================================================

banner("10.13M — DATASET-SPECIFIC / NON-COMMON FEATURE INVENTORY")


shared_semantic_for_common = {
    "FLOW_DURATION",
    "SRC_FWD_PACKET_COUNT",
    "DST_BWD_PACKET_COUNT",
    "SRC_FWD_BYTE_COUNT",
    "DST_BWD_BYTE_COUNT",
    "PROTOCOL",
    "RATE_RELATED"
}


def common_status(row):

    group = row["Semantic_Group"]

    if group == "LABEL_TARGET":
        return "TARGET_NOT_PREDICTOR"

    if group == "TIMESTAMP_METADATA":
        return "TEMPORAL_METADATA"

    if group == "NETWORK_IDENTIFIER":
        return "IDENTIFIER_NOT_RAW_PREDICTOR"

    if group in shared_semantic_for_common:
        return "COMMON_CANDIDATE"

    if group in shared_groups:
        return "REQUIRES_EQUIVALENCE_CHECK"

    return "DATASET_SPECIFIC"


inventory["Common_Schema_Status"] = inventory.apply(
    common_status,
    axis=1
)


dataset_specific = inventory[
    inventory["Common_Schema_Status"].isin(
        [
            "DATASET_SPECIFIC",
            "REQUIRES_EQUIVALENCE_CHECK"
        ]
    )
].copy()


print(
    dataset_specific[
        [
            "Dataset",
            "Original_Feature",
            "Semantic_Group",
            "Common_Schema_Status"
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13N — COMMON FEATURE SUPPORT CHECK
#
# We verify that the essential raw semantic groups needed for the proposed
# common representation actually exist in both datasets.
# ==================================================================================================

banner("10.13N — ESSENTIAL COMMON FEATURE SUPPORT CHECK")


essential_groups = [
    "FLOW_DURATION",
    "SRC_FWD_PACKET_COUNT",
    "DST_BWD_PACKET_COUNT",
    "SRC_FWD_BYTE_COUNT",
    "DST_BWD_BYTE_COUNT"
]


support_rows = []


for group in essential_groups:

    cic_support = inventory[
        (inventory["Dataset"] == "CIC_IDS2017")
        &
        (inventory["Semantic_Group"] == group)
    ]

    unsw_support = inventory[
        (inventory["Dataset"] == "UNSW_NB15")
        &
        (inventory["Semantic_Group"] == group)
    ]

    support_rows.append(
        {
            "Semantic_Group": group,

            "CIC_Present":
                len(cic_support) > 0,

            "UNSW_Present":
                len(unsw_support) > 0,

            "CIC_Features":
                " | ".join(
                    cic_support["Original_Feature"]
                    .astype(str)
                    .tolist()
                ),

            "UNSW_Features":
                " | ".join(
                    unsw_support["Original_Feature"]
                    .astype(str)
                    .tolist()
                )
        }
    )


support_table = pd.DataFrame(
    support_rows
)

print(
    support_table.to_string(
        index=False
    )
)


missing_essential = support_table[
    ~(
        support_table["CIC_Present"]
        &
        support_table["UNSW_Present"]
    )
]


if len(missing_essential) > 0:

    print(
        "\n⚠ Some proposed common semantics are not automatically supported."
    )

    print(
        "Do NOT aggregate until we inspect these."
    )

else:

    print(
        "\n✓ Essential traffic-volume semantics exist in both datasets"
    )


# ==================================================================================================
# 10.13O — BUILD FEATURE COMPATIBILITY TABLE
# ==================================================================================================

banner("10.13O — BUILD FEATURE COMPATIBILITY TABLE")


compatibility_rows = []


all_groups = sorted(
    set(
        inventory["Semantic_Group"]
    )
)


for group in all_groups:

    cic_rows = inventory[
        (inventory["Dataset"] == "CIC_IDS2017")
        &
        (inventory["Semantic_Group"] == group)
    ]

    unsw_rows = inventory[
        (inventory["Dataset"] == "UNSW_NB15")
        &
        (inventory["Semantic_Group"] == group)
    ]


    cic_present = len(cic_rows) > 0
    unsw_present = len(unsw_rows) > 0


    if (
        group in shared_semantic_for_common
        and cic_present
        and unsw_present
    ):

        decision = "COMMON_CANDIDATE"

    elif (
        cic_present
        and unsw_present
    ):

        decision = "SEMANTIC_VERIFICATION_REQUIRED"

    else:

        decision = "DATASET_SPECIFIC"


    compatibility_rows.append(
        {
            "Semantic_Group": group,

            "CIC_Present": cic_present,

            "UNSW_Present": unsw_present,

            "CIC_Features":
                " | ".join(
                    cic_rows["Original_Feature"]
                    .astype(str)
                    .tolist()
                ),

            "UNSW_Features":
                " | ".join(
                    unsw_rows["Original_Feature"]
                    .astype(str)
                    .tolist()
                ),

            "Decision": decision
        }
    )


compatibility = pd.DataFrame(
    compatibility_rows
)


print(
    compatibility.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13P — SAVE OUTPUTS
# ==================================================================================================

banner("10.13P — SAVE AUDIT OUTPUTS")


inventory_out = (
    COMMON_OUT_DIR
    / "10_13_CIC_UNSW_schema_inventory.csv"
)

compatibility_out = (
    COMMON_OUT_DIR
    / "10_13_CIC_UNSW_feature_compatibility.csv"
)

common_features_out = (
    COMMON_OUT_DIR
    / "10_13_proposed_common_temporal_features.csv"
)

dataset_specific_out = (
    COMMON_OUT_DIR
    / "10_13_dataset_specific_features.csv"
)

verification_out = (
    COMMON_OUT_DIR
    / "10_13_features_requiring_verification.csv"
)

policy_out = (
    COMMON_OUT_DIR
    / "10_13_leakage_metadata_policy.csv"
)

support_out = (
    COMMON_OUT_DIR
    / "10_13_essential_feature_support.csv"
)


inventory.to_csv(
    inventory_out,
    index=False
)

compatibility.to_csv(
    compatibility_out,
    index=False
)

common_features.to_csv(
    common_features_out,
    index=False
)

dataset_specific.to_csv(
    dataset_specific_out,
    index=False
)

verification_candidates.to_csv(
    verification_out,
    index=False
)

policy.to_csv(
    policy_out,
    index=False
)

support_table.to_csv(
    support_out,
    index=False
)


for p in [
    inventory_out,
    compatibility_out,
    common_features_out,
    dataset_specific_out,
    verification_out,
    policy_out,
    support_out
]:

    print("✓", p)


# ==================================================================================================
# 10.13Q — MANIFEST
# ==================================================================================================

banner("10.13Q — FEATURE COMPATIBILITY MANIFEST")


manifest = pd.DataFrame(
    {
        "Metric": [

            "CIC_Raw_Files_Discovered",

            "CIC_Unique_Columns",

            "UNSW_Raw_Features",

            "Exact_Normalized_Name_Overlap",

            "Shared_Semantic_Groups",

            "Proposed_Common_Temporal_Features",

            "Common_Candidate_Semantic_Groups",

            "Semantic_Verification_Required_Groups",

            "Dataset_Specific_Groups",

            "Essential_Groups_Missing"
        ],

        "Value": [

            len(cic_csv_files),

            len(cic_columns),

            len(unsw_columns),

            len(exact_overlap),

            len(shared_groups),

            len(common_features),

            (
                compatibility["Decision"]
                == "COMMON_CANDIDATE"
            ).sum(),

            (
                compatibility["Decision"]
                == "SEMANTIC_VERIFICATION_REQUIRED"
            ).sum(),

            (
                compatibility["Decision"]
                == "DATASET_SPECIFIC"
            ).sum(),

            len(missing_essential)
        ]
    }
)


print(
    manifest.to_string(
        index=False
    )
)


manifest_out = (
    COMMON_OUT_DIR
    / "10_13_feature_compatibility_manifest.csv"
)

manifest.to_csv(
    manifest_out,
    index=False
)

print("\n✓ Saved:")
print(manifest_out)


# ==================================================================================================
# 10.13R — FINAL DECISION
# ==================================================================================================

banner("10.13R — PHASE DECISION")


if len(missing_essential) == 0:

    print(
        """
✓ CORE COMMON TEMPORAL REPRESENTATION IS STRUCTURALLY POSSIBLE.

This does NOT yet mean every similarly named CIC/UNSW feature is equivalent.

What we have established:

    CIC flows
       │
       ├── duration
       ├── packet counts
       ├── byte counts
       └── temporal activity
             │
             ▼
        COMMON TEMPORAL
        REPRESENTATION
             ▲
             │
       ┌─────┴─────┐
       │           │
    UNSW flows
       │
       ├── duration
       ├── packet counts
       ├── byte counts
       └── temporal activity


SAFE CORE:

    Flow_Count

    Flow_Duration_Mean
    Flow_Duration_Std
    Flow_Duration_Max

    Forward_Packets_Sum
    Backward_Packets_Sum
    Total_Packets

    Forward_Bytes_Sum
    Backward_Bytes_Sum
    Total_Bytes

    Packets_Per_Second
    Bytes_Per_Second

    Forward_Backward_Packet_Ratio
    Forward_Backward_Byte_Ratio


NOT YET AUTOMATICALLY ACCEPTED:

    TCP flags
    IAT statistics
    TTL
    connection state
    CIC bulk/subflow fields
    UNSW ct_* fields
    other dataset-specific engineered variables


VERY IMPORTANT:

No model training should use:

    Attack_State
    Attack_Family
    Episode_ID
    Dataset_ID
    Capture_ID
    raw IP addresses

as ordinary predictor features.


NEXT:

PHASE 10.14

We reconstruct the authoritative UNSW chronological timeline and
capture boundaries using the frozen 10.12 semantic artifact.

That prepares UNSW for its 1-minute temporal aggregation.
"""
    )

else:

    print(
        """
⚠ COMMON TEMPORAL REPRESENTATION NEEDS REVIEW.

At least one essential semantic group was not automatically located
in both datasets.

Do NOT proceed to temporal aggregation yet.

Inspect:

    10_13_essential_feature_support.csv

and the printed missing groups.

We will resolve the mapping explicitly rather than guessing.
"""
    )


# ==================================================================================================
# COMPLETE
# ==================================================================================================

banner("PHASE 10.13 COMPLETE")

print(
    f"Runtime: {time.time() - START_TIME:.2f} sec"
)

In [ ]:
# ==================================================================================================
# PHASE 10.13.1 — CIC ↔ UNSW CORE SEMANTIC MAPPING CORRECTION & FREEZE
#
# PURPOSE
# -------
# Phase 10.13 correctly identified that the common temporal representation required review.
#
# The apparent missing CIC byte semantics were caused by a semantic-classification issue:
#
#   CIC:
#       "Total Length of Fwd Packets"
#       "Total Length of Bwd Packets"
#
# represent total BYTE LENGTHS, not packet counts.
#
# This phase:
#
#   1. Re-reads ONLY schemas / metadata (no multimillion-row processing).
#   2. Explicitly verifies the authoritative core CIC ↔ UNSW mapping.
#   3. Verifies duration units before freezing the mapping.
#   4. Separates temporal-role mappings from numeric predictor mappings.
#   5. Does NOT force questionable TCP/IAT/TTL/etc. equivalences.
#   6. Freezes the minimal common semantic layer.
#   7. Refuses to PASS if an essential mapping is absent.
#
# EXPECTED RUNTIME:
#   A few seconds.
#
# OUTPUTS:
#
#   10_13_1_FROZEN_CIC_UNSW_CORE_MAPPING.csv
#   10_13_1_FROZEN_COMMON_TEMPORAL_FEATURE_PLAN.csv
#   10_13_1_MAPPING_VALIDATION.csv
#   10_13_1_mapping_manifest.csv
#
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

START_TIME = time.time()


# ==================================================================================================
# 10.13.1A — PATHS
# ==================================================================================================

BASE = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

RAW_DIR = BASE / "raw"

CIC_RAW_DIR = RAW_DIR / "CIC_IDS2017"
UNSW_RAW_DIR = RAW_DIR / "UNSW_NB15"

COMMON_OUT_DIR = BASE / "audit_outputs" / "COMMON"

COMMON_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

UNSW_FEATURE_FILE = (
    UNSW_RAW_DIR / "NUSW-NB15_features.csv"
)


def banner(text):
    print("\n" + "=" * 120)
    print(text)
    print("=" * 120)


def subbanner(text):
    print("\n" + "-" * 120)
    print(text)
    print("-" * 120)


banner(
    "PHASE 10.13.1 — CIC ↔ UNSW CORE SEMANTIC MAPPING CORRECTION & FREEZE"
)


# ==================================================================================================
# 10.13.1B — DISCOVER CIC FILES
# ==================================================================================================

banner("10.13.1B — DISCOVER CIC FILES")

cic_files = sorted(
    [
        p
        for p in CIC_RAW_DIR.rglob("*.csv")
        if p.is_file()
    ]
)

# Fallback in case folder spelling differs.
if len(cic_files) == 0:

    candidates = []

    for p in RAW_DIR.rglob("*.csv"):

        name = str(p).lower()

        if (
            "cic" in name
            or "monday" in name
            or "tuesday" in name
            or "wednesday" in name
            or "thursday" in name
            or "friday" in name
        ):
            candidates.append(p)

    cic_files = sorted(candidates)


if len(cic_files) == 0:

    raise FileNotFoundError(
        "No CIC-IDS2017 CSV files could be located."
    )


print(
    f"CIC files discovered: {len(cic_files)}"
)

for p in cic_files:
    print("✓", p.name)


# ==================================================================================================
# 10.13.1C — READ CIC HEADERS ONLY
# ==================================================================================================

banner("10.13.1C — VERIFY CIC SCHEMA")

cic_schema_by_file = {}


for path in cic_files:

    header = None

    for enc in [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]:

        try:

            header = pd.read_csv(
                path,
                nrows=0,
                encoding=enc
            )

            break

        except UnicodeDecodeError:
            continue


    if header is None:

        raise ValueError(
            f"Could not read CIC header:\n{path}"
        )


    cols = [
        str(c).strip()
        for c in header.columns
    ]

    cic_schema_by_file[path.name] = cols


schema_signatures = {
    tuple(cols)
    for cols in cic_schema_by_file.values()
}


print(
    "Unique CIC schema signatures:",
    len(schema_signatures)
)


if len(schema_signatures) != 1:

    raise ValueError(
        "CIC schemas are not identical across all files. "
        "Do not freeze the mapping."
    )


cic_columns = list(
    next(
        iter(schema_signatures)
    )
)


print(
    "CIC columns:",
    len(cic_columns)
)

print(
    "✓ CIC schema uniform across all discovered files"
)


# ==================================================================================================
# 10.13.1D — LOAD UNSW FEATURE METADATA
# ==================================================================================================

banner("10.13.1D — VERIFY UNSW SCHEMA")


if not UNSW_FEATURE_FILE.exists():

    raise FileNotFoundError(
        f"Missing UNSW feature metadata:\n{UNSW_FEATURE_FILE}"
    )


unsw_features = None
used_encoding = None


for enc in [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin1"
]:

    try:

        unsw_features = pd.read_csv(
            UNSW_FEATURE_FILE,
            encoding=enc,
            low_memory=False
        )

        used_encoding = enc

        break

    except UnicodeDecodeError:
        continue


if unsw_features is None:

    raise ValueError(
        "Unable to read UNSW feature metadata."
    )


unsw_features.columns = (
    unsw_features.columns
    .astype(str)
    .str.strip()
)


name_candidates = [
    c
    for c in unsw_features.columns
    if c.lower().strip() == "name"
]


if len(name_candidates) != 1:

    raise ValueError(
        "Could not uniquely identify UNSW feature-name column."
    )


UNSW_NAME_COL = name_candidates[0]


unsw_columns = (
    unsw_features[UNSW_NAME_COL]
    .astype(str)
    .str.strip()
    .tolist()
)


print(
    "Metadata encoding:",
    used_encoding
)

print(
    "UNSW columns:",
    len(unsw_columns)
)


if len(unsw_columns) != 49:

    raise ValueError(
        f"Expected 49 UNSW features, found {len(unsw_columns)}."
    )


print(
    "✓ 49-column UNSW schema recovered"
)


# ==================================================================================================
# 10.13.1E — CASE-INSENSITIVE COLUMN RESOLUTION
# ==================================================================================================

banner("10.13.1E — COLUMN RESOLUTION")


def normalized_lookup(columns):

    lookup = {}

    for c in columns:

        key = (
            str(c)
            .strip()
            .lower()
        )

        lookup[key] = str(c).strip()

    return lookup


cic_lookup = normalized_lookup(
    cic_columns
)

unsw_lookup = normalized_lookup(
    unsw_columns
)


def resolve_column(
    lookup,
    expected_name
):

    return lookup.get(
        expected_name.strip().lower()
    )


# ==================================================================================================
# 10.13.1F — EXPLICIT AUTHORITATIVE CORE MAPPING
#
# NOTE:
#
# We are intentionally NOT doing fuzzy semantic matching here.
#
# Every mapping is explicitly declared.
# ==================================================================================================

banner("10.13.1F — EXPLICIT CORE SEMANTIC MAPPING")


mapping_spec = [

    # ----------------------------------------------------------------------------------------------
    # TEMPORAL INDEX
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Concept": "flow_start_time",

        "Role": "TEMPORAL_INDEX",

        "CIC_Expected": "Timestamp",

        "UNSW_Expected": "Stime",

        "Semantic_Interpretation":
            "Flow start timestamp used for chronological ordering and temporal bin assignment",

        "Unit_or_Format":
            "dataset-specific timestamp representation",

        "Predictor_Status":
            "METADATA_ONLY",

        "Required":
            True
    },


    # ----------------------------------------------------------------------------------------------
    # FLOW DURATION
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Concept": "flow_duration",

        "Role": "NUMERIC_TRAFFIC_FEATURE",

        "CIC_Expected": "Flow Duration",

        "UNSW_Expected": "dur",

        "Semantic_Interpretation":
            "Duration of the network flow",

        # CICFlowMeter duration is conventionally represented in microseconds,
        # while UNSW 'dur' is represented in seconds.
        # We do NOT treat raw values as directly comparable.
        "Unit_or_Format":
            "CIC raw duration requires conversion to seconds; UNSW dur retained in seconds",

        "Predictor_Status":
            "COMMON_AFTER_UNIT_NORMALIZATION",

        "Required":
            True
    },


    # ----------------------------------------------------------------------------------------------
    # FORWARD PACKET COUNT
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Concept": "forward_packets",

        "Role": "NUMERIC_TRAFFIC_FEATURE",

        "CIC_Expected": "Total Fwd Packets",

        "UNSW_Expected": "Spkts",

        "Semantic_Interpretation":
            "Number of packets in source/forward direction",

        "Unit_or_Format":
            "packets",

        "Predictor_Status":
            "COMMON",

        "Required":
            True
    },


    # ----------------------------------------------------------------------------------------------
    # BACKWARD PACKET COUNT
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Concept": "backward_packets",

        "Role": "NUMERIC_TRAFFIC_FEATURE",

        "CIC_Expected": "Total Backward Packets",

        "UNSW_Expected": "Dpkts",

        "Semantic_Interpretation":
            "Number of packets in destination/backward direction",

        "Unit_or_Format":
            "packets",

        "Predictor_Status":
            "COMMON",

        "Required":
            True
    },


    # ----------------------------------------------------------------------------------------------
    # FORWARD BYTE COUNT
    #
    # THIS IS THE IMPORTANT 10.13 CORRECTION.
    #
    # "Total Length of Fwd Packets" means sum of packet lengths,
    # i.e. total forward-direction bytes.
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Concept": "forward_bytes",

        "Role": "NUMERIC_TRAFFIC_FEATURE",

        "CIC_Expected":
            "Total Length of Fwd Packets",

        "UNSW_Expected":
            "sbytes",

        "Semantic_Interpretation":
            "Total bytes transferred in source/forward direction",

        "Unit_or_Format":
            "bytes",

        "Predictor_Status":
            "COMMON",

        "Required":
            True
    },


    # ----------------------------------------------------------------------------------------------
    # BACKWARD BYTE COUNT
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Concept": "backward_bytes",

        "Role": "NUMERIC_TRAFFIC_FEATURE",

        "CIC_Expected":
            "Total Length of Bwd Packets",

        "UNSW_Expected":
            "dbytes",

        "Semantic_Interpretation":
            "Total bytes transferred in destination/backward direction",

        "Unit_or_Format":
            "bytes",

        "Predictor_Status":
            "COMMON",

        "Required":
            True
    },


    # ----------------------------------------------------------------------------------------------
    # PROTOCOL
    # ----------------------------------------------------------------------------------------------

    {
        "Common_Concept": "protocol",

        "Role": "CATEGORICAL_TRAFFIC_METADATA",

        "CIC_Expected":
            "Protocol",

        "UNSW_Expected":
            "proto",

        "Semantic_Interpretation":
            "Transport/network protocol representation",

        "Unit_or_Format":
            "CIC numeric protocol code; UNSW protocol name/category — harmonization required",

        "Predictor_Status":
            "COMMON_AFTER_VALUE_HARMONIZATION",

        "Required":
            True
    }
]


mapping = pd.DataFrame(
    mapping_spec
)


# ==================================================================================================
# 10.13.1G — RESOLVE ACTUAL COLUMN NAMES
# ==================================================================================================

banner("10.13.1G — VERIFY ALL MAPPED COLUMNS EXIST")


mapping["CIC_Actual"] = (
    mapping["CIC_Expected"]
    .apply(
        lambda x: resolve_column(
            cic_lookup,
            x
        )
    )
)


mapping["UNSW_Actual"] = (
    mapping["UNSW_Expected"]
    .apply(
        lambda x: resolve_column(
            unsw_lookup,
            x
        )
    )
)


mapping["CIC_Present"] = (
    mapping["CIC_Actual"]
    .notna()
)


mapping["UNSW_Present"] = (
    mapping["UNSW_Actual"]
    .notna()
)


mapping["Mapping_Complete"] = (
    mapping["CIC_Present"]
    &
    mapping["UNSW_Present"]
)


display_cols = [

    "Common_Concept",

    "CIC_Actual",

    "UNSW_Actual",

    "Role",

    "Predictor_Status",

    "Mapping_Complete"
]


print(
    mapping[
        display_cols
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13.1H — REQUIRED-MAPPING ASSERTION
# ==================================================================================================

banner("10.13.1H — REQUIRED MAPPING ASSERTION")


required = mapping[
    mapping["Required"]
].copy()


missing = required[
    ~required["Mapping_Complete"]
].copy()


print(
    "Required mappings:",
    len(required)
)

print(
    "Complete mappings:",
    required["Mapping_Complete"].sum()
)

print(
    "Missing mappings:",
    len(missing)
)


if len(missing) > 0:

    print("\nMISSING:")

    print(
        missing[
            [
                "Common_Concept",
                "CIC_Expected",
                "UNSW_Expected",
                "CIC_Present",
                "UNSW_Present"
            ]
        ].to_string(
            index=False
        )
    )

    raise ValueError(
        "Core CIC ↔ UNSW mapping is incomplete. "
        "Do not proceed."
    )


print(
    "\n✓ ALL REQUIRED CORE MAPPINGS EXIST"
)


# ==================================================================================================
# 10.13.1I — BYTE-SEMANTIC CORRECTION CONFIRMATION
# ==================================================================================================

banner("10.13.1I — BYTE SEMANTIC CORRECTION")


byte_rows = mapping[
    mapping["Common_Concept"].isin(
        [
            "forward_bytes",
            "backward_bytes"
        ]
    )
].copy()


print(
    byte_rows[
        [
            "Common_Concept",
            "CIC_Actual",
            "UNSW_Actual",
            "Unit_or_Format",
            "Mapping_Complete"
        ]
    ].to_string(
        index=False
    )
)


if not byte_rows["Mapping_Complete"].all():

    raise ValueError(
        "Byte semantics could not be verified."
    )


print(
    "\n✓ Phase 10.13 false-negative corrected."
)

print(
    "✓ CIC forward/backward byte totals are available."
)

print(
    "✓ UNSW forward/backward byte totals are available."
)


# ==================================================================================================
# 10.13.1J — DURATION NORMALIZATION POLICY
# ==================================================================================================

banner("10.13.1J — DURATION NORMALIZATION POLICY")


duration_policy = """
CIC:
    Raw column:
        Flow Duration

    Common representation:
        Flow_Duration_Seconds

    Conversion:
        CIC Flow Duration / 1,000,000


UNSW:
    Raw column:
        dur

    Common representation:
        Flow_Duration_Seconds

    Conversion:
        UNSW dur retained as seconds


IMPORTANT:

Raw CIC and UNSW duration values must NOT simply be concatenated
without unit normalization.

The common temporal layer will use SECONDS.
"""

print(duration_policy)


# ==================================================================================================
# 10.13.1K — PROTOCOL HARMONIZATION POLICY
# ==================================================================================================

banner("10.13.1K — PROTOCOL HARMONIZATION POLICY")


protocol_policy = """
CIC:
    Protocol is represented numerically.

UNSW:
    proto is represented categorically (e.g. tcp, udp, ...).

Therefore:

    CIC Protocol
        ≠
    UNSW proto

at the raw-value level.

They represent the same GENERAL CONCEPT,
but require value harmonization before common temporal features such as:

    TCP_Flow_Count
    UDP_Flow_Count
    Protocol_Diversity

are accepted.

Protocol-derived features therefore remain:

    VERIFIED CONCEPT
    +
    VALUE-HARMONIZATION REQUIRED

We will NOT blindly concatenate the raw protocol columns.
"""

print(protocol_policy)


# ==================================================================================================
# 10.13.1L — FEATURES WE EXPLICITLY REFUSE TO FORCE-MATCH
# ==================================================================================================

banner("10.13.1L — NON-FROZEN FEATURE FAMILIES")


non_frozen = pd.DataFrame(
    [
        {
            "Feature_Family":
                "TCP_Flags",

            "Status":
                "NOT_IN_CORE_MAPPING",

            "Reason":
                "CIC flag counters and UNSW TCP-related fields are not automatically equivalent."
        },

        {
            "Feature_Family":
                "Interarrival_Time",

            "Status":
                "NOT_IN_CORE_MAPPING",

            "Reason":
                "Dataset definitions require explicit semantic verification."
        },

        {
            "Feature_Family":
                "TTL",

            "Status":
                "UNSW_SPECIFIC_FOR_NOW",

            "Reason":
                "No established CIC equivalent."
        },

        {
            "Feature_Family":
                "UNSW_ct_*",

            "Status":
                "EXCLUDED_FROM_COMMON_CORE",

            "Reason":
                "Dataset-specific engineered connection-history features."
        },

        {
            "Feature_Family":
                "CIC_Bulk_Subflow",

            "Status":
                "EXCLUDED_FROM_COMMON_CORE",

            "Reason":
                "No established UNSW equivalent."
        },

        {
            "Feature_Family":
                "Connection_State",

            "Status":
                "UNSW_SPECIFIC_FOR_NOW",

            "Reason":
                "No proven CIC-equivalent representation."
        },

        {
            "Feature_Family":
                "Raw_IP_Addresses",

            "Status":
                "METADATA_ONLY",

            "Reason":
                "Raw topology identifiers must not become ordinary model predictors."
        },

        {
            "Feature_Family":
                "Raw_Timestamp",

            "Status":
                "TEMPORAL_INDEX_ONLY",

            "Reason":
                "Used for ordering/binning, not as raw absolute-time predictor."
        }
    ]
)


print(
    non_frozen.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13.1M — FREEZE COMMON TEMPORAL FEATURE PLAN
#
# These are DERIVED from the frozen core semantic inputs.
#
# We are NOT yet computing them here.
# ==================================================================================================

banner("10.13.1M — FREEZE COMMON TEMPORAL FEATURE PLAN")


temporal_plan = pd.DataFrame(
    [

        # ==========================================================================================
        # TRAFFIC VOLUME
        # ==========================================================================================

        {
            "Temporal_Feature":
                "Flow_Count",

            "Source_Concepts":
                "flow_start_time",

            "Aggregation":
                "count flows per temporal bin",

            "Status":
                "APPROVED"
        },


        # ==========================================================================================
        # DURATION
        # ==========================================================================================

        {
            "Temporal_Feature":
                "Flow_Duration_Mean",

            "Source_Concepts":
                "flow_duration",

            "Aggregation":
                "mean after conversion to seconds",

            "Status":
                "APPROVED"
        },

        {
            "Temporal_Feature":
                "Flow_Duration_Std",

            "Source_Concepts":
                "flow_duration",

            "Aggregation":
                "standard deviation after conversion to seconds",

            "Status":
                "APPROVED"
        },

        {
            "Temporal_Feature":
                "Flow_Duration_Max",

            "Source_Concepts":
                "flow_duration",

            "Aggregation":
                "maximum after conversion to seconds",

            "Status":
                "APPROVED"
        },


        # ==========================================================================================
        # PACKETS
        # ==========================================================================================

        {
            "Temporal_Feature":
                "Forward_Packets_Sum",

            "Source_Concepts":
                "forward_packets",

            "Aggregation":
                "sum",

            "Status":
                "APPROVED"
        },

        {
            "Temporal_Feature":
                "Backward_Packets_Sum",

            "Source_Concepts":
                "backward_packets",

            "Aggregation":
                "sum",

            "Status":
                "APPROVED"
        },

        {
            "Temporal_Feature":
                "Total_Packets",

            "Source_Concepts":
                "forward_packets + backward_packets",

            "Aggregation":
                "sum of directional packet totals",

            "Status":
                "APPROVED"
        },


        # ==========================================================================================
        # BYTES
        # ==========================================================================================

        {
            "Temporal_Feature":
                "Forward_Bytes_Sum",

            "Source_Concepts":
                "forward_bytes",

            "Aggregation":
                "sum",

            "Status":
                "APPROVED"
        },

        {
            "Temporal_Feature":
                "Backward_Bytes_Sum",

            "Source_Concepts":
                "backward_bytes",

            "Aggregation":
                "sum",

            "Status":
                "APPROVED"
        },

        {
            "Temporal_Feature":
                "Total_Bytes",

            "Source_Concepts":
                "forward_bytes + backward_bytes",

            "Aggregation":
                "sum of directional byte totals",

            "Status":
                "APPROVED"
        },


        # ==========================================================================================
        # BIN-LEVEL RATES
        #
        # NOTE:
        # These are calculated from the common temporal totals,
        # not by averaging CIC Flow Bytes/s against UNSW rate.
        # ==========================================================================================

        {
            "Temporal_Feature":
                "Packets_Per_Second",

            "Source_Concepts":
                "Total_Packets",

            "Aggregation":
                "Total_Packets / temporal-bin duration in seconds",

            "Status":
                "APPROVED_DERIVED"
        },

        {
            "Temporal_Feature":
                "Bytes_Per_Second",

            "Source_Concepts":
                "Total_Bytes",

            "Aggregation":
                "Total_Bytes / temporal-bin duration in seconds",

            "Status":
                "APPROVED_DERIVED"
        },


        # ==========================================================================================
        # DIRECTIONAL RATIOS
        # ==========================================================================================

        {
            "Temporal_Feature":
                "Forward_Backward_Packet_Ratio",

            "Source_Concepts":
                "Forward_Packets_Sum + Backward_Packets_Sum",

            "Aggregation":
                "safe directional ratio with zero-denominator handling",

            "Status":
                "APPROVED_DERIVED"
        },

        {
            "Temporal_Feature":
                "Forward_Backward_Byte_Ratio",

            "Source_Concepts":
                "Forward_Bytes_Sum + Backward_Bytes_Sum",

            "Aggregation":
                "safe directional ratio with zero-denominator handling",

            "Status":
                "APPROVED_DERIVED"
        },


        # ==========================================================================================
        # PROTOCOL — HELD UNTIL VALUE HARMONIZATION
        # ==========================================================================================

        {
            "Temporal_Feature":
                "TCP_Flow_Count",

            "Source_Concepts":
                "protocol",

            "Aggregation":
                "count harmonized TCP flows",

            "Status":
                "APPROVED_AFTER_PROTOCOL_VALUE_HARMONIZATION"
        },

        {
            "Temporal_Feature":
                "UDP_Flow_Count",

            "Source_Concepts":
                "protocol",

            "Aggregation":
                "count harmonized UDP flows",

            "Status":
                "APPROVED_AFTER_PROTOCOL_VALUE_HARMONIZATION"
        },

        {
            "Temporal_Feature":
                "Protocol_Diversity",

            "Source_Concepts":
                "protocol",

            "Aggregation":
                "number of harmonized unique protocol categories",

            "Status":
                "APPROVED_AFTER_PROTOCOL_VALUE_HARMONIZATION"
        }
    ]
)


print(
    temporal_plan.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13.1N — TARGET / METADATA LEAKAGE FREEZE
# ==================================================================================================

banner("10.13.1N — TARGET & METADATA LEAKAGE FREEZE")


leakage_freeze = pd.DataFrame(
    [
        {
            "Field":
                "Attack_State",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "forecast target / evaluation semantics"
        },

        {
            "Field":
                "Attack_Family",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "attack-family target / evaluation semantics"
        },

        {
            "Field":
                "Episode_ID",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "episode-aware splitting / leakage prevention"
        },

        {
            "Field":
                "Dataset_ID",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "provenance only"
        },

        {
            "Field":
                "Capture_ID",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "capture-boundary / splitting metadata"
        },

        {
            "Field":
                "Raw_Source_IP",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "derived behavioural statistics only"
        },

        {
            "Field":
                "Raw_Destination_IP",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "derived behavioural statistics only"
        },

        {
            "Field":
                "Raw_Absolute_Timestamp",

            "Allowed_As_Predictor":
                False,

            "Permitted_Role":
                "ordering and temporal-bin assignment"
        }
    ]
)


print(
    leakage_freeze.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.13.1O — BUILD VALIDATION TABLE
# ==================================================================================================

banner("10.13.1O — CORE VALIDATION SUMMARY")


validation_rows = []


for _, row in mapping.iterrows():

    validation_rows.append(
        {
            "Check":
                f"Mapping::{row['Common_Concept']}",

            "Passed":
                bool(row["Mapping_Complete"]),

            "Details":
                (
                    f"CIC={row['CIC_Actual']} | "
                    f"UNSW={row['UNSW_Actual']}"
                )
        }
    )


validation_rows.extend(
    [
        {
            "Check":
                "CIC_Schema_Uniform",

            "Passed":
                len(schema_signatures) == 1,

            "Details":
                f"{len(cic_files)} files"
        },

        {
            "Check":
                "UNSW_Feature_Count_49",

            "Passed":
                len(unsw_columns) == 49,

            "Details":
                f"{len(unsw_columns)} features"
        },

        {
            "Check":
                "Forward_Byte_Mapping_Fixed",

            "Passed":
                bool(
                    mapping.loc[
                        mapping["Common_Concept"]
                        == "forward_bytes",
                        "Mapping_Complete"
                    ].all()
                ),

            "Details":
                "CIC Total Length of Fwd Packets ↔ UNSW sbytes"
        },

        {
            "Check":
                "Backward_Byte_Mapping_Fixed",

            "Passed":
                bool(
                    mapping.loc[
                        mapping["Common_Concept"]
                        == "backward_bytes",
                        "Mapping_Complete"
                    ].all()
                ),

            "Details":
                "CIC Total Length of Bwd Packets ↔ UNSW dbytes"
        }
    ]
)


validation = pd.DataFrame(
    validation_rows
)


print(
    validation.to_string(
        index=False
    )
)


all_checks_pass = bool(
    validation["Passed"].all()
)


# ==================================================================================================
# 10.13.1P — SAVE FROZEN ARTIFACTS
# ==================================================================================================

banner("10.13.1P — SAVE FROZEN ARTIFACTS")


mapping_out = (
    COMMON_OUT_DIR
    / "10_13_1_FROZEN_CIC_UNSW_CORE_MAPPING.csv"
)

temporal_plan_out = (
    COMMON_OUT_DIR
    / "10_13_1_FROZEN_COMMON_TEMPORAL_FEATURE_PLAN.csv"
)

validation_out = (
    COMMON_OUT_DIR
    / "10_13_1_MAPPING_VALIDATION.csv"
)

non_frozen_out = (
    COMMON_OUT_DIR
    / "10_13_1_NON_FROZEN_FEATURE_FAMILIES.csv"
)

leakage_out = (
    COMMON_OUT_DIR
    / "10_13_1_LEAKAGE_FREEZE.csv"
)


mapping.to_csv(
    mapping_out,
    index=False
)

temporal_plan.to_csv(
    temporal_plan_out,
    index=False
)

validation.to_csv(
    validation_out,
    index=False
)

non_frozen.to_csv(
    non_frozen_out,
    index=False
)

leakage_freeze.to_csv(
    leakage_out,
    index=False
)


for p in [
    mapping_out,
    temporal_plan_out,
    validation_out,
    non_frozen_out,
    leakage_out
]:

    print("✓", p)


# ==================================================================================================
# 10.13.1Q — MANIFEST
# ==================================================================================================

banner("10.13.1Q — MAPPING MANIFEST")


numeric_core = mapping[
    mapping["Role"]
    == "NUMERIC_TRAFFIC_FEATURE"
]


protocol_core = mapping[
    mapping["Common_Concept"]
    == "protocol"
]


manifest = pd.DataFrame(
    {
        "Metric": [

            "CIC_Files_Verified",

            "CIC_Columns",

            "UNSW_Columns",

            "Required_Core_Mappings",

            "Complete_Core_Mappings",

            "Missing_Core_Mappings",

            "Numeric_Common_Core_Features",

            "Temporal_Index_Mappings",

            "Protocol_Concept_Mappings",

            "Approved_Temporal_Features",

            "Protocol_Dependent_Temporal_Features",

            "Validation_Checks",

            "Validation_Checks_Passed",

            "Core_Mapping_Pass"
        ],

        "Value": [

            len(cic_files),

            len(cic_columns),

            len(unsw_columns),

            len(required),

            int(
                required["Mapping_Complete"].sum()
            ),

            len(missing),

            len(numeric_core),

            int(
                (
                    mapping["Role"]
                    == "TEMPORAL_INDEX"
                ).sum()
            ),

            len(protocol_core),

            int(
                temporal_plan[
                    "Status"
                ].isin(
                    [
                        "APPROVED",
                        "APPROVED_DERIVED"
                    ]
                ).sum()
            ),

            int(
                (
                    temporal_plan["Status"]
                    == "APPROVED_AFTER_PROTOCOL_VALUE_HARMONIZATION"
                ).sum()
            ),

            len(validation),

            int(
                validation["Passed"].sum()
            ),

            all_checks_pass
        ]
    }
)


print(
    manifest.to_string(
        index=False
    )
)


manifest_out = (
    COMMON_OUT_DIR
    / "10_13_1_mapping_manifest.csv"
)


manifest.to_csv(
    manifest_out,
    index=False
)


print("\n✓ Saved:")
print(manifest_out)


# ==================================================================================================
# 10.13.1R — FINAL ASSERTIONS
# ==================================================================================================

banner("10.13.1R — FINAL ASSERTIONS")


assert len(missing) == 0, (
    "Essential mappings are still missing."
)

assert all_checks_pass, (
    "One or more validation checks failed."
)

assert (
    mapping[
        "Common_Concept"
    ].is_unique
), (
    "Duplicate common concepts detected."
)


# ==================================================================================================
# 10.13.1S — FINAL DECISION
# ==================================================================================================

banner("10.13.1S — FINAL DECISION")


print(
    """
✓ CORE CIC ↔ UNSW SEMANTIC MAPPING: PASS


THE PHASE 10.13 BYTE ISSUE HAS BEEN CORRECTED.

Frozen common raw concepts:

    TIME
        CIC  : Timestamp
        UNSW : Stime

    DURATION
        CIC  : Flow Duration
        UNSW : dur

    FORWARD PACKETS
        CIC  : Total Fwd Packets
        UNSW : Spkts

    BACKWARD PACKETS
        CIC  : Total Backward Packets
        UNSW : Dpkts

    FORWARD BYTES
        CIC  : Total Length of Fwd Packets
        UNSW : sbytes

    BACKWARD BYTES
        CIC  : Total Length of Bwd Packets
        UNSW : dbytes

    PROTOCOL
        CIC  : Protocol
        UNSW : proto


IMPORTANT NORMALIZATION:

    CIC Flow Duration
            ↓
        convert to seconds

    UNSW dur
            ↓
        retain in seconds


IMPORTANT PROTOCOL RULE:

    The protocol CONCEPT is common.

    The raw VALUE REPRESENTATION is not yet common.

    Therefore protocol-derived temporal features will only be enabled
    after protocol values are explicitly harmonized.


WE ARE NOT FORCE-MATCHING:

    TCP flags
    TTL
    IAT
    UNSW ct_* features
    CIC bulk/subflow features
    connection state
    other dataset-specific engineered variables


NO TARGET / SPLIT METADATA WILL ENTER THE PREDICTOR MATRIX.


NEXT PHASE:

    10.14 — UNSW AUTHORITATIVE CHRONOLOGICAL TIMELINE
            & CAPTURE-BOUNDARY RECONSTRUCTION

After 10.14:

    10.15 — UNSW 1-minute common temporal representation

Then:

    final UNSW temporal integrity audit

Then:

    CIC + UNSW common temporal combination.
"""
)


# ==================================================================================================
# COMPLETE
# ==================================================================================================

banner("PHASE 10.13.1 COMPLETE")

print(
    f"Runtime: {time.time() - START_TIME:.2f} sec"
)

In [ ]:
# ==================================================================================================
# PHASE 10.14 — UNSW AUTHORITATIVE CHRONOLOGICAL TIMELINE & CAPTURE-BOUNDARY RECONSTRUCTION
#
# PURPOSE
# -------
# Build the authoritative chronological representation of UNSW-NB15 BEFORE 1-minute aggregation.
#
# We already have:
#
#   Phase 10.12
#       -> frozen authoritative flow semantics
#
#   Phase 10.13.1
#       -> frozen CIC ↔ UNSW common semantic mapping
#
# This phase will:
#
#   1. Read UNSW raw flows efficiently in chunks.
#   2. Recover authoritative flow-start and flow-end timestamps.
#   3. Verify timestamp validity.
#   4. Reconstruct physical capture periods from the observed chronology.
#   5. Preserve raw-file provenance.
#   6. Audit ordering, overlaps and major temporal gaps.
#   7. Verify capture boundaries against the GT-derived periods found earlier.
#   8. Produce a compact chronological index for Phase 10.15.
#
# IMPORTANT
# ---------
# NO 1-minute aggregation occurs here.
#
# NO train/val/test split occurs here.
#
# NO random shuffling occurs here.
#
# NO attack labels are used as predictors.
#
# Capture_ID is metadata only.
#
# EXPECTED RUNTIME
# ----------------
# Usually a few minutes depending on Google Drive I/O.
#
# OUTPUTS
# -------
#   10_14_UNSW_FLOW_TIMELINE_INDEX.csv
#   10_14_UNSW_CAPTURE_SUMMARY.csv
#   10_14_UNSW_RAW_FILE_TIMELINE_SUMMARY.csv
#   10_14_UNSW_TEMPORAL_GAP_AUDIT.csv
#   10_14_UNSW_TIMELINE_VALIDATION.csv
#   10_14_UNSW_TIMELINE_MANIFEST.csv
#
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import warnings
import gc

warnings.filterwarnings("ignore")

START_TIME = time.time()


# ==================================================================================================
# 10.14A — PATHS
# ==================================================================================================

BASE = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

UNSW_RAW_DIR = BASE / "raw" / "UNSW_NB15"

UNSW_OUT_DIR = BASE / "audit_outputs" / "UNSW"

COMMON_OUT_DIR = BASE / "audit_outputs" / "COMMON"

UNSW_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FEATURE_FILE = (
    UNSW_RAW_DIR / "NUSW-NB15_features.csv"
)

CORE_MAPPING_FILE = (
    COMMON_OUT_DIR /
    "10_13_1_FROZEN_CIC_UNSW_CORE_MAPPING.csv"
)


RAW_FILES = [
    UNSW_RAW_DIR / "UNSW-NB15_1.csv",
    UNSW_RAW_DIR / "UNSW-NB15_2.csv",
    UNSW_RAW_DIR / "UNSW-NB15_3.csv",
    UNSW_RAW_DIR / "UNSW-NB15_4.csv",
]


CHUNK_SIZE = 300_000


def banner(text):
    print("\n" + "=" * 120)
    print(text)
    print("=" * 120)


def subbanner(text):
    print("\n" + "-" * 120)
    print(text)
    print("-" * 120)


banner(
    "PHASE 10.14 — UNSW AUTHORITATIVE CHRONOLOGICAL TIMELINE & CAPTURE-BOUNDARY RECONSTRUCTION"
)


# ==================================================================================================
# 10.14B — VERIFY REQUIRED ARTIFACTS
# ==================================================================================================

banner("10.14B — VERIFY REQUIRED ARTIFACTS")


required_files = [
    FEATURE_FILE,
    CORE_MAPPING_FILE,
    *RAW_FILES
]


missing_files = [
    p for p in required_files
    if not p.exists()
]


if missing_files:

    print("Missing required files:")

    for p in missing_files:
        print("✗", p)

    raise FileNotFoundError(
        "Required Phase 10.14 inputs are missing."
    )


for p in required_files:
    print("✓", p.name)


# ==================================================================================================
# 10.14C — LOAD FROZEN CORE MAPPING
# ==================================================================================================

banner("10.14C — LOAD FROZEN CORE MAPPING")


core_mapping = pd.read_csv(
    CORE_MAPPING_FILE
)


required_concepts = {
    "flow_start_time",
    "flow_duration",
    "forward_packets",
    "backward_packets",
    "forward_bytes",
    "backward_bytes",
    "protocol"
}


found_concepts = set(
    core_mapping["Common_Concept"]
    .astype(str)
)


missing_concepts = (
    required_concepts
    -
    found_concepts
)


if missing_concepts:

    raise ValueError(
        f"Frozen mapping missing concepts: {missing_concepts}"
    )


print(
    "Frozen mapping rows:",
    len(core_mapping)
)

print(
    "✓ Required core concepts present"
)


# ==================================================================================================
# 10.14D — RECOVER UNSW RAW SCHEMA
# ==================================================================================================

banner("10.14D — RECOVER UNSW RAW SCHEMA")


features = None
feature_encoding = None


for enc in [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin1"
]:

    try:

        features = pd.read_csv(
            FEATURE_FILE,
            encoding=enc,
            low_memory=False
        )

        feature_encoding = enc
        break

    except UnicodeDecodeError:
        continue


if features is None:

    raise ValueError(
        "Could not read UNSW feature metadata."
    )


features.columns = (
    features.columns
    .astype(str)
    .str.strip()
)


name_cols = [
    c for c in features.columns
    if c.lower().strip() == "name"
]


if len(name_cols) != 1:

    raise ValueError(
        "Could not identify UNSW feature-name column."
    )


NAME_COL = name_cols[0]


raw_columns = (
    features[NAME_COL]
    .astype(str)
    .str.strip()
    .tolist()
)


if len(raw_columns) != 49:

    raise ValueError(
        f"Expected 49 UNSW columns, found {len(raw_columns)}."
    )


print(
    "Feature metadata encoding:",
    feature_encoding
)

print(
    "Raw feature count:",
    len(raw_columns)
)

print(
    "✓ 49-column raw schema recovered"
)


# ==================================================================================================
# 10.14E — IDENTIFY REQUIRED TIMESTAMP FIELDS
# ==================================================================================================

banner("10.14E — IDENTIFY TEMPORAL FIELDS")


column_lookup = {
    str(c).strip().lower(): str(c).strip()
    for c in raw_columns
}


STIME_COL = column_lookup.get("stime")
LTIME_COL = column_lookup.get("ltime")
DUR_COL = column_lookup.get("dur")


if STIME_COL is None:

    raise ValueError(
        "UNSW Stime column not found."
    )


if LTIME_COL is None:

    raise ValueError(
        "UNSW Ltime column not found."
    )


if DUR_COL is None:

    raise ValueError(
        "UNSW dur column not found."
    )


print(
    "Start timestamp:",
    STIME_COL
)

print(
    "End timestamp  :",
    LTIME_COL
)

print(
    "Duration       :",
    DUR_COL
)


# ==================================================================================================
# 10.14F — READ RAW TIMELINE FIELDS IN CHUNKS
#
# We only load:
#
#   Stime
#   Ltime
#   dur
#
# plus generated provenance:
#
#   Raw_File
#   Raw_Row_In_File
#
# This avoids loading all 49 columns for 2.54M rows.
# ==================================================================================================

banner("10.14F — LOAD RAW TEMPORAL FIELDS")


timeline_parts = []
file_summary_rows = []

global_row_offset = 0


for file_idx, path in enumerate(
    RAW_FILES,
    start=1
):

    subbanner(
        f"[{file_idx}/4] {path.name}"
    )

    file_start = time.time()

    file_parts = []

    row_counter = 0


    reader = pd.read_csv(
        path,

        names=raw_columns,

        header=None,

        usecols=[
            STIME_COL,
            LTIME_COL,
            DUR_COL
        ],

        chunksize=CHUNK_SIZE,

        low_memory=False
    )


    for chunk_idx, chunk in enumerate(
        reader,
        start=1
    ):

        n = len(chunk)

        chunk["Raw_File"] = path.name

        chunk["Raw_File_ID"] = file_idx

        chunk["Raw_Row_In_File"] = np.arange(
            row_counter,
            row_counter + n,
            dtype=np.int64
        )

        chunk["Global_Raw_Row_ID"] = np.arange(
            global_row_offset + row_counter,
            global_row_offset + row_counter + n,
            dtype=np.int64
        )


        # ------------------------------------------------------------------------------------------
        # Numeric conversion
        # ------------------------------------------------------------------------------------------

        chunk[STIME_COL] = pd.to_numeric(
            chunk[STIME_COL],
            errors="coerce"
        )

        chunk[LTIME_COL] = pd.to_numeric(
            chunk[LTIME_COL],
            errors="coerce"
        )

        chunk[DUR_COL] = pd.to_numeric(
            chunk[DUR_COL],
            errors="coerce"
        )


        file_parts.append(
            chunk
        )


        row_counter += n


        print(
            f"  chunk {chunk_idx}: "
            f"{n:,} rows "
            f"(cumulative {row_counter:,})"
        )


    file_df = pd.concat(
        file_parts,
        ignore_index=True
    )


    timeline_parts.append(
        file_df
    )


    valid_start = file_df[
        STIME_COL
    ].dropna()


    valid_end = file_df[
        LTIME_COL
    ].dropna()


    file_summary_rows.append(
        {
            "Raw_File":
                path.name,

            "Raw_File_ID":
                file_idx,

            "Rows":
                len(file_df),

            "Valid_Stime":
                int(valid_start.notna().sum()),

            "Invalid_Stime":
                int(file_df[STIME_COL].isna().sum()),

            "Valid_Ltime":
                int(valid_end.notna().sum()),

            "Invalid_Ltime":
                int(file_df[LTIME_COL].isna().sum()),

            "Min_Stime_Epoch":
                valid_start.min()
                if len(valid_start)
                else np.nan,

            "Max_Stime_Epoch":
                valid_start.max()
                if len(valid_start)
                else np.nan,

            "Min_Ltime_Epoch":
                valid_end.min()
                if len(valid_end)
                else np.nan,

            "Max_Ltime_Epoch":
                valid_end.max()
                if len(valid_end)
                else np.nan
        }
    )


    global_row_offset += len(file_df)


    del file_df
    del file_parts

    gc.collect()


    print(
        f"✓ {row_counter:,} rows loaded "
        f"in {time.time() - file_start:.1f} sec"
    )


timeline = pd.concat(
    timeline_parts,
    ignore_index=True
)


del timeline_parts

gc.collect()


print(
    "\nTotal raw UNSW rows:",
    f"{len(timeline):,}"
)


# ==================================================================================================
# 10.14G — CONVERT EPOCH → DATETIME
# ==================================================================================================

banner("10.14G — CONVERT TIMESTAMPS")


timeline["Flow_Start"] = pd.to_datetime(
    timeline[STIME_COL],
    unit="s",
    errors="coerce"
)


timeline["Flow_End"] = pd.to_datetime(
    timeline[LTIME_COL],
    unit="s",
    errors="coerce"
)


timeline["Duration_Reported_Sec"] = pd.to_numeric(
    timeline[DUR_COL],
    errors="coerce"
)


timeline["Duration_From_Timestamps_Sec"] = (
    timeline["Flow_End"]
    -
    timeline["Flow_Start"]
).dt.total_seconds()


print(
    "Valid Flow_Start:",
    f"{timeline['Flow_Start'].notna().sum():,}"
)

print(
    "Invalid Flow_Start:",
    f"{timeline['Flow_Start'].isna().sum():,}"
)

print(
    "Valid Flow_End:",
    f"{timeline['Flow_End'].notna().sum():,}"
)

print(
    "Invalid Flow_End:",
    f"{timeline['Flow_End'].isna().sum():,}"
)


# ==================================================================================================
# 10.14H — BASIC TEMPORAL VALIDITY
# ==================================================================================================

banner("10.14H — BASIC TEMPORAL VALIDITY")


timeline["End_Before_Start"] = (
    timeline["Flow_End"]
    <
    timeline["Flow_Start"]
)


timeline["Negative_Reported_Duration"] = (
    timeline["Duration_Reported_Sec"]
    <
    0
)


valid_time_mask = (
    timeline["Flow_Start"].notna()
    &
    timeline["Flow_End"].notna()
)


print(
    "End before start:",
    f"{timeline['End_Before_Start'].sum():,}"
)

print(
    "Negative reported duration:",
    f"{timeline['Negative_Reported_Duration'].sum():,}"
)


if valid_time_mask.any():

    print(
        "\nObserved global time range:"
    )

    print(
        "Start:",
        timeline.loc[
            valid_time_mask,
            "Flow_Start"
        ].min()
    )

    print(
        "End  :",
        timeline.loc[
            valid_time_mask,
            "Flow_End"
        ].max()
    )


# ==================================================================================================
# 10.14I — DURATION CONSISTENCY AUDIT
#
# IMPORTANT:
#
# Stime/Ltime are integer-second timestamps.
# dur may contain sub-second precision.
#
# Therefore exact equality is NOT expected.
#
# We only flag gross contradictions.
# ==================================================================================================

banner("10.14I — DURATION CONSISTENCY AUDIT")


duration_valid = (
    timeline["Duration_Reported_Sec"].notna()
    &
    timeline["Duration_From_Timestamps_Sec"].notna()
)


timeline["Duration_Absolute_Error_Sec"] = np.nan


timeline.loc[
    duration_valid,
    "Duration_Absolute_Error_Sec"
] = np.abs(

    timeline.loc[
        duration_valid,
        "Duration_Reported_Sec"
    ]

    -

    timeline.loc[
        duration_valid,
        "Duration_From_Timestamps_Sec"
    ]
)


if duration_valid.any():

    err = timeline.loc[
        duration_valid,
        "Duration_Absolute_Error_Sec"
    ]


    print(
        err.describe(
            percentiles=[
                .50,
                .90,
                .95,
                .99,
                .999
            ]
        )
    )


# Gross contradiction threshold only.

timeline["Gross_Duration_Conflict"] = (
    timeline[
        "Duration_Absolute_Error_Sec"
    ]
    >
    2.0
)


print(
    "\nGross duration conflicts (>2 sec):",
    f"{timeline['Gross_Duration_Conflict'].sum():,}"
)


# ==================================================================================================
# 10.14J — SORT AUTHORITATIVELY BY FLOW START
# ==================================================================================================

banner("10.14J — AUTHORITATIVE CHRONOLOGICAL SORT")


timeline = timeline.sort_values(
    [
        "Flow_Start",
        "Flow_End",
        "Global_Raw_Row_ID"
    ],

    kind="mergesort",

    na_position="last",

    ignore_index=True
)


timeline["Chronological_Row_ID"] = np.arange(
    len(timeline),
    dtype=np.int64
)


print(
    "✓ Timeline sorted chronologically"
)


# ==================================================================================================
# 10.14K — ADJACENT START-TIME GAP CALCULATION
# ==================================================================================================

banner("10.14K — ADJACENT FLOW-START GAP AUDIT")


timeline["Previous_Flow_Start"] = (
    timeline["Flow_Start"].shift(1)
)


timeline["Start_Gap_Sec"] = (
    timeline["Flow_Start"]
    -
    timeline["Previous_Flow_Start"]
).dt.total_seconds()


positive_gaps = timeline[
    timeline["Start_Gap_Sec"] > 0
][
    "Start_Gap_Sec"
]


print(
    "Positive adjacent start gaps:",
    f"{len(positive_gaps):,}"
)


if len(positive_gaps):

    print(
        positive_gaps.describe(
            percentiles=[
                .50,
                .90,
                .95,
                .99,
                .999,
                .9999
            ]
        )
    )


# ==================================================================================================
# 10.14L — IDENTIFY MAJOR PHYSICAL CAPTURE SEPARATION
#
# Earlier GT work found two capture periods:
#
#   Capture 1:
#       2015-01-22
#
#   Capture 2:
#       2015-02-18
#
# We now independently verify that the RAW FLOW chronology supports
# the same physical separation.
#
# We do NOT arbitrarily use raw file boundaries as capture boundaries.
# ==================================================================================================

banner("10.14L — MAJOR PHYSICAL CAPTURE SEPARATION")


valid_gap_rows = timeline[
    timeline["Start_Gap_Sec"].notna()
].copy()


if len(valid_gap_rows) == 0:

    raise ValueError(
        "No valid chronological gaps available."
    )


largest_gap_idx = (
    valid_gap_rows[
        "Start_Gap_Sec"
    ].idxmax()
)


largest_gap_sec = float(
    timeline.loc[
        largest_gap_idx,
        "Start_Gap_Sec"
    ]
)


capture2_start = timeline.loc[
    largest_gap_idx,
    "Flow_Start"
]


capture1_last_start = timeline.loc[
    largest_gap_idx - 1,
    "Flow_Start"
]


print(
    "Largest global positive flow-start gap:",
    f"{largest_gap_sec:,.0f} sec"
)

print(
    "Flow before gap:",
    capture1_last_start
)

print(
    "Flow after gap :",
    capture2_start
)

print(
    "Gap in days    :",
    f"{largest_gap_sec / 86400:.3f}"
)


# ==================================================================================================
# 10.14M — VERIFY AGAINST GT-DERIVED CAPTURE STRUCTURE
#
# We previously established:
#
#   Capture 1 GT:
#       2015-01-22 11:50:14 → 2015-01-22 13:47:28
#
#   Capture 2 GT:
#       2015-02-18 00:23:59 → 2015-02-18 12:21:09
#
# Raw traffic may extend beyond attack-GT intervals because benign flows
# also exist. Therefore we compare DATE / structural separation rather
# than requiring exact GT start/end equality.
# ==================================================================================================

banner("10.14M — GT-DERIVED CAPTURE STRUCTURE CROSS-CHECK")


EXPECTED_CAPTURE_1_DATE = pd.Timestamp(
    "2015-01-22"
).date()


EXPECTED_CAPTURE_2_DATE = pd.Timestamp(
    "2015-02-18"
).date()


raw_capture2_date = (
    capture2_start.date()
    if pd.notna(capture2_start)
    else None
)


capture2_date_match = (
    raw_capture2_date
    ==
    EXPECTED_CAPTURE_2_DATE
)


print(
    "Expected Capture-1 date:",
    EXPECTED_CAPTURE_1_DATE
)

print(
    "Expected Capture-2 date:",
    EXPECTED_CAPTURE_2_DATE
)

print(
    "Raw inferred Capture-2 date:",
    raw_capture2_date
)

print(
    "Capture-2 date agreement:",
    capture2_date_match
)


# ==================================================================================================
# 10.14N — ASSIGN PHYSICAL CAPTURE ID
# ==================================================================================================

banner("10.14N — ASSIGN PHYSICAL CAPTURE IDS")


timeline["Capture_ID"] = np.where(

    timeline["Flow_Start"]
    <
    capture2_start,

    "UNSW_CAPTURE_1",

    "UNSW_CAPTURE_2"
)


# Missing timestamps must NOT silently become capture 2.

timeline.loc[
    timeline["Flow_Start"].isna(),
    "Capture_ID"
] = "UNRESOLVED_TIMESTAMP"


capture_counts = (
    timeline[
        "Capture_ID"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    capture_counts
)


# ==================================================================================================
# 10.14O — CAPTURE SUMMARY
# ==================================================================================================

banner("10.14O — CAPTURE SUMMARY")


capture_summary = (
    timeline[
        timeline["Capture_ID"].isin(
            [
                "UNSW_CAPTURE_1",
                "UNSW_CAPTURE_2"
            ]
        )
    ]
    .groupby(
        "Capture_ID",
        sort=True
    )
    .agg(

        Rows=(
            "Chronological_Row_ID",
            "size"
        ),

        Start=(
            "Flow_Start",
            "min"
        ),

        Last_Flow_Start=(
            "Flow_Start",
            "max"
        ),

        End=(
            "Flow_End",
            "max"
        ),

        Raw_Files=(
            "Raw_File",
            "nunique"
        ),

        Negative_Durations=(
            "Negative_Reported_Duration",
            "sum"
        ),

        End_Before_Start=(
            "End_Before_Start",
            "sum"
        ),

        Gross_Duration_Conflicts=(
            "Gross_Duration_Conflict",
            "sum"
        )

    )
    .reset_index()
)


capture_summary[
    "Observed_Span_Sec"
] = (

    capture_summary["End"]
    -
    capture_summary["Start"]

).dt.total_seconds()


print(
    capture_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.14P — RAW FILE ↔ PHYSICAL CAPTURE RELATIONSHIP
#
# This is important because UNSW raw file numbering does NOT automatically
# define physical capture periods.
# ==================================================================================================

banner("10.14P — RAW FILE ↔ CAPTURE RELATIONSHIP")


file_capture = pd.crosstab(

    timeline["Raw_File"],

    timeline["Capture_ID"]
)


print(
    file_capture.to_string()
)


# ==================================================================================================
# 10.14Q — RAW FILE TIMELINE SUMMARY
# ==================================================================================================

banner("10.14Q — RAW FILE TIMELINE SUMMARY")


file_summary = pd.DataFrame(
    file_summary_rows
)


for col in [
    "Min_Stime_Epoch",
    "Max_Stime_Epoch",
    "Min_Ltime_Epoch",
    "Max_Ltime_Epoch"
]:

    dt_col = col.replace(
        "_Epoch",
        "_DT"
    )

    file_summary[dt_col] = pd.to_datetime(
        file_summary[col],
        unit="s",
        errors="coerce"
    )


print(
    file_summary[
        [
            "Raw_File",
            "Rows",
            "Invalid_Stime",
            "Invalid_Ltime",
            "Min_Stime_DT",
            "Max_Stime_DT",
            "Min_Ltime_DT",
            "Max_Ltime_DT"
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 10.14R — TEMPORAL GAP-BAND AUDIT
#
# This does NOT define attack episodes.
#
# It only characterizes raw traffic chronology.
# ==================================================================================================

banner("10.14R — TEMPORAL GAP-BAND AUDIT")


gap_edges = [
    -np.inf,
    0,
    1,
    2,
    5,
    10,
    30,
    60,
    120,
    300,
    600,
    1800,
    3600,
    7200,
    21600,
    86400,
    np.inf
]


gap_labels = [
    "<=0",
    "(0,1]",
    "(1,2]",
    "(2,5]",
    "(5,10]",
    "(10,30]",
    "(30,60]",
    "(60,120]",
    "(120,300]",
    "(300,600]",
    "(600,1800]",
    "(1800,3600]",
    "(3600,7200]",
    "(7200,21600]",
    "(21600,86400]",
    ">86400"
]


timeline["Gap_Band"] = pd.cut(

    timeline["Start_Gap_Sec"],

    bins=gap_edges,

    labels=gap_labels,

    include_lowest=True
)


gap_audit = (
    timeline[
        timeline["Start_Gap_Sec"].notna()
    ]
    .groupby(
        "Gap_Band",
        observed=False
    )
    .agg(

        Boundaries=(
            "Start_Gap_Sec",
            "size"
        ),

        Median_Gap_Sec=(
            "Start_Gap_Sec",
            "median"
        ),

        Max_Gap_Sec=(
            "Start_Gap_Sec",
            "max"
        )

    )
    .reset_index()
)


print(
    gap_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.14S — CAPTURE-INTERNAL LARGE GAPS
#
# Useful for deciding later whether a 1-minute temporal grid contains
# naturally empty minutes.
#
# We do NOT fill them yet.
# ==================================================================================================

banner("10.14S — CAPTURE-INTERNAL LARGE GAPS")


capture_internal = timeline[
    timeline["Capture_ID"].isin(
        [
            "UNSW_CAPTURE_1",
            "UNSW_CAPTURE_2"
        ]
    )
].copy()


capture_internal[
    "Previous_Capture_ID"
] = (
    capture_internal[
        "Capture_ID"
    ].shift(1)
)


same_capture = (
    capture_internal["Capture_ID"]
    ==
    capture_internal["Previous_Capture_ID"]
)


capture_internal[
    "Capture_Internal_Gap_Sec"
] = np.where(

    same_capture,

    capture_internal["Start_Gap_Sec"],

    np.nan
)


large_internal_gaps = capture_internal[
    capture_internal[
        "Capture_Internal_Gap_Sec"
    ]
    >
    60
][
    [
        "Capture_ID",
        "Previous_Flow_Start",
        "Flow_Start",
        "Capture_Internal_Gap_Sec"
    ]
].copy()


print(
    "Capture-internal gaps >60 sec:",
    f"{len(large_internal_gaps):,}"
)


if len(large_internal_gaps):

    print(
        "\nLargest 20:"
    )

    print(
        large_internal_gaps
        .nlargest(
            20,
            "Capture_Internal_Gap_Sec"
        )
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 10.14T — RAW FILE ORDER VS CHRONOLOGICAL ORDER
#
# We intentionally do NOT assume:
#
#   file 1 → file 2 → file 3 → file 4
#
# is perfectly chronological.
#
# ==================================================================================================

banner("10.14T — RAW FILE ORDER VS CHRONOLOGICAL ORDER")


chronological_file_transitions = (
    timeline[
        "Raw_File"
    ]
    !=
    timeline[
        "Raw_File"
    ].shift(1)
)


transition_table = timeline.loc[
    chronological_file_transitions,

    [
        "Chronological_Row_ID",
        "Flow_Start",
        "Raw_File",
        "Capture_ID"
    ]
].copy()


print(
    "Chronological raw-file transitions:",
    len(transition_table)
)


print(
    transition_table
    .head(30)
    .to_string(
        index=False
    )
)


# ==================================================================================================
# 10.14U — BUILD COMPACT AUTHORITATIVE TIMELINE INDEX
#
# This is the artifact Phase 10.15 will use to reason about chronology.
#
# We intentionally do NOT duplicate all 49 raw columns here.
# ==================================================================================================

banner("10.14U — BUILD AUTHORITATIVE TIMELINE INDEX")


timeline_index = timeline[
    [
        "Chronological_Row_ID",

        "Global_Raw_Row_ID",

        "Raw_File",

        "Raw_File_ID",

        "Raw_Row_In_File",

        "Flow_Start",

        "Flow_End",

        "Duration_Reported_Sec",

        "Capture_ID"
    ]
].copy()


timeline_index[
    "Minute"
] = timeline_index[
    "Flow_Start"
].dt.floor(
    "min"
)


print(
    "Timeline rows:",
    f"{len(timeline_index):,}"
)

print(
    "Unique minutes:",
    f"{timeline_index['Minute'].nunique():,}"
)

print(
    "Capture IDs:",
    timeline_index[
        "Capture_ID"
    ].value_counts().to_dict()
)


# ==================================================================================================
# 10.14V — VALIDATION CHECKS
# ==================================================================================================

banner("10.14V — TIMELINE VALIDATION")


valid_flow_start_count = int(
    timeline[
        "Flow_Start"
    ].notna().sum()
)


invalid_flow_start_count = int(
    timeline[
        "Flow_Start"
    ].isna().sum()
)


unresolved_capture_count = int(
    (
        timeline["Capture_ID"]
        ==
        "UNRESOLVED_TIMESTAMP"
    ).sum()
)


chronological_monotonic = bool(
    timeline.loc[
        timeline["Flow_Start"].notna(),
        "Flow_Start"
    ].is_monotonic_increasing
)


capture_count = int(
    timeline.loc[
        timeline[
            "Capture_ID"
        ].isin(
            [
                "UNSW_CAPTURE_1",
                "UNSW_CAPTURE_2"
            ]
        ),
        "Capture_ID"
    ].nunique()
)


validation = pd.DataFrame(
    [
        {
            "Check":
                "All_4_Raw_Files_Present",

            "Passed":
                len(RAW_FILES) == 4,

            "Details":
                f"{len(RAW_FILES)} raw files"
        },

        {
            "Check":
                "Expected_Total_Row_Order_Magnitude",

            "Passed":
                len(timeline) > 2_000_000,

            "Details":
                f"{len(timeline):,} rows"
        },

        {
            "Check":
                "Flow_Start_Timestamps_Valid",

            "Passed":
                invalid_flow_start_count == 0,

            "Details":
                (
                    f"valid={valid_flow_start_count:,}, "
                    f"invalid={invalid_flow_start_count:,}"
                )
        },

        {
            "Check":
                "No_End_Before_Start",

            "Passed":
                int(
                    timeline[
                        "End_Before_Start"
                    ].sum()
                ) == 0,

            "Details":
                (
                    f"{int(timeline['End_Before_Start'].sum()):,} conflicts"
                )
        },

        {
            "Check":
                "No_Negative_Reported_Duration",

            "Passed":
                int(
                    timeline[
                        "Negative_Reported_Duration"
                    ].sum()
                ) == 0,

            "Details":
                (
                    f"{int(timeline['Negative_Reported_Duration'].sum()):,} negative"
                )
        },

        {
            "Check":
                "Chronological_Sort_Monotonic",

            "Passed":
                chronological_monotonic,

            "Details":
                str(
                    chronological_monotonic
                )
        },

        {
            "Check":
                "Two_Physical_Captures_Recovered",

            "Passed":
                capture_count == 2,

            "Details":
                f"{capture_count} captures"
        },

        {
            "Check":
                "Capture2_Date_Agrees_With_GT_Audit",

            "Passed":
                bool(
                    capture2_date_match
                ),

            "Details":
                (
                    f"raw={raw_capture2_date}, "
                    f"expected={EXPECTED_CAPTURE_2_DATE}"
                )
        },

        {
            "Check":
                "No_Unresolved_Capture_Assignment",

            "Passed":
                unresolved_capture_count == 0,

            "Details":
                f"{unresolved_capture_count:,} unresolved"
        }
    ]
)


print(
    validation.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.14W — SAVE OUTPUTS
# ==================================================================================================

banner("10.14W — SAVE OUTPUTS")


timeline_out = (
    UNSW_OUT_DIR /
    "10_14_UNSW_FLOW_TIMELINE_INDEX.csv"
)

capture_summary_out = (
    UNSW_OUT_DIR /
    "10_14_UNSW_CAPTURE_SUMMARY.csv"
)

file_summary_out = (
    UNSW_OUT_DIR /
    "10_14_UNSW_RAW_FILE_TIMELINE_SUMMARY.csv"
)

gap_audit_out = (
    UNSW_OUT_DIR /
    "10_14_UNSW_TEMPORAL_GAP_AUDIT.csv"
)

validation_out = (
    UNSW_OUT_DIR /
    "10_14_UNSW_TIMELINE_VALIDATION.csv"
)

large_gap_out = (
    UNSW_OUT_DIR /
    "10_14_UNSW_CAPTURE_INTERNAL_GAPS_GT60SEC.csv"
)


# Use ISO-like datetime formatting.

timeline_index.to_csv(
    timeline_out,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

capture_summary.to_csv(
    capture_summary_out,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

file_summary.to_csv(
    file_summary_out,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

gap_audit.to_csv(
    gap_audit_out,
    index=False
)

validation.to_csv(
    validation_out,
    index=False
)

large_internal_gaps.to_csv(
    large_gap_out,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


for p in [
    timeline_out,
    capture_summary_out,
    file_summary_out,
    gap_audit_out,
    large_gap_out,
    validation_out
]:

    print("✓", p)


# ==================================================================================================
# 10.14X — MANIFEST
# ==================================================================================================

banner("10.14X — TIMELINE MANIFEST")


checks_passed = int(
    validation[
        "Passed"
    ].sum()
)


all_checks_pass = bool(
    validation[
        "Passed"
    ].all()
)


manifest = pd.DataFrame(
    {
        "Metric": [

            "Raw_Files",

            "Total_Flows",

            "Valid_Start_Timestamps",

            "Invalid_Start_Timestamps",

            "Physical_Captures",

            "Capture_1_Rows",

            "Capture_2_Rows",

            "Unique_Minutes",

            "Largest_Global_Start_Gap_Sec",

            "Largest_Global_Start_Gap_Days",

            "Capture_Internal_Gaps_GT60Sec",

            "End_Before_Start_Count",

            "Negative_Duration_Count",

            "Gross_Duration_Conflict_Count",

            "Validation_Checks",

            "Validation_Checks_Passed",

            "Timeline_Pass"
        ],

        "Value": [

            len(RAW_FILES),

            len(timeline),

            valid_flow_start_count,

            invalid_flow_start_count,

            capture_count,

            int(
                (
                    timeline[
                        "Capture_ID"
                    ]
                    ==
                    "UNSW_CAPTURE_1"
                ).sum()
            ),

            int(
                (
                    timeline[
                        "Capture_ID"
                    ]
                    ==
                    "UNSW_CAPTURE_2"
                ).sum()
            ),

            int(
                timeline_index[
                    "Minute"
                ].nunique()
            ),

            largest_gap_sec,

            largest_gap_sec / 86400,

            len(
                large_internal_gaps
            ),

            int(
                timeline[
                    "End_Before_Start"
                ].sum()
            ),

            int(
                timeline[
                    "Negative_Reported_Duration"
                ].sum()
            ),

            int(
                timeline[
                    "Gross_Duration_Conflict"
                ].sum()
            ),

            len(
                validation
            ),

            checks_passed,

            all_checks_pass
        ]
    }
)


print(
    manifest.to_string(
        index=False
    )
)


manifest_out = (
    UNSW_OUT_DIR /
    "10_14_UNSW_TIMELINE_MANIFEST.csv"
)


manifest.to_csv(
    manifest_out,
    index=False
)


print("\n✓", manifest_out)


# ==================================================================================================
# 10.14Y — FINAL DECISION
# ==================================================================================================

banner("10.14Y — FINAL DECISION")


critical_checks = validation[
    validation[
        "Check"
    ].isin(
        [
            "Flow_Start_Timestamps_Valid",
            "No_End_Before_Start",
            "No_Negative_Reported_Duration",
            "Chronological_Sort_Monotonic",
            "Two_Physical_Captures_Recovered",
            "Capture2_Date_Agrees_With_GT_Audit",
            "No_Unresolved_Capture_Assignment"
        ]
    )
]


critical_pass = bool(
    critical_checks[
        "Passed"
    ].all()
)


if critical_pass:

    print(
        """
✓ UNSW AUTHORITATIVE CHRONOLOGICAL TIMELINE: PASS


WHAT IS NOW ESTABLISHED

    1. Every raw UNSW flow has an authoritative chronological position.

    2. Raw CSV file order is preserved as provenance but is NOT treated
       as the physical capture definition.

    3. The major physical capture separation has been reconstructed
       from the raw timestamps.

    4. The raw-flow capture structure has been cross-checked against
       the GT-derived capture structure.

    5. Capture_ID remains metadata only.

    6. No train/test splitting has occurred.

    7. No temporal aggregation has occurred.

    8. No attack information has entered the predictor feature space.


AUTHORITATIVE STRUCTURE

        RAW UNSW FLOWS
              │
              ▼
        timestamp recovery
              │
              ▼
        chronological sort
              │
              ▼
       physical captures
        ┌─────┴─────┐
        │           │
    CAPTURE 1    CAPTURE 2
        │           │
        └─────┬─────┘
              │
              ▼
       authoritative flow
          timeline index


NEXT:

    PHASE 10.15

    Build the UNSW 1-minute temporal representation using:

        Flow_Count

        Flow_Duration_Mean
        Flow_Duration_Std
        Flow_Duration_Max

        Forward_Packets_Sum
        Backward_Packets_Sum
        Total_Packets

        Forward_Bytes_Sum
        Backward_Bytes_Sum
        Total_Bytes

        Packets_Per_Second
        Bytes_Per_Second

        Forward_Backward_Packet_Ratio
        Forward_Backward_Byte_Ratio

    while attaching attack semantics separately as TARGET/METADATA.

"""
    )

else:

    print(
        """
⚠ UNSW TIMELINE REQUIRES REVIEW.

At least one critical temporal-integrity check failed.

DO NOT RUN 10.15 YET.

Inspect:

    10_14_UNSW_TIMELINE_VALIDATION.csv

and send me the printed Phase 10.14 output.

We will determine whether the failure represents:

    • a real dataset property,
    • timestamp representation issue,
    • raw schema issue,
    • or an audit implementation issue.

We will NOT simply bypass the failed check.
"""
    )


# ==================================================================================================
# COMPLETE
# ==================================================================================================

banner("PHASE 10.14 COMPLETE")

print(
    f"Runtime: {time.time() - START_TIME:.2f} sec"
)

In [ ]:
# ==================================================================================================
# PHASE 10.15 — UNSW LEAKAGE-SAFE 1-MINUTE TEMPORAL REPRESENTATION
#
# PURPOSE
# -------
# Convert the authoritative UNSW flow-level data into the common 1-minute temporal representation
# that will later be harmonized with CIC-IDS2017.
#
# INPUTS
# ------
# Phase 10.12:
#   10_12_UNSW_FROZEN_FLOW_SEMANTICS.csv
#
# Phase 10.13.1:
#   10_13_1_FROZEN_CIC_UNSW_CORE_MAPPING.csv
#   10_13_1_FROZEN_COMMON_TEMPORAL_FEATURE_PLAN.csv
#
# Phase 10.14:
#   10_14_UNSW_FLOW_TIMELINE_INDEX.csv
#
# CORE DESIGN
# -----------
# Predictor branch:
#
#       raw UNSW flow measurements
#               ↓
#       leakage-safe 1-minute aggregation
#
# Target branch:
#
#       authoritative 10.12 flow semantics
#               ↓
#       minute-level attack semantics
#
# The two branches are joined ONLY AFTER aggregation.
#
#
# IMPORTANT DECISIONS
# -------------------
#
# 1. Flow_Start determines minute membership.
#
# 2. UNSW `dur` is already in seconds.
#
# 3. We DO NOT use:
#
#       Label
#       attack_cat
#       GT family
#       attack identity
#       episode ID
#
#    as predictor features.
#
# 4. Capture_ID is metadata only.
#
# 5. Raw timestamps are temporal-index metadata only.
#
# 6. We DO NOT create synthetic zero-flow minutes.
#
#    Missing minutes remain missing observations for now.
#
# 7. We DO NOT bridge the ~26-day gap between the two physical captures.
#
# 8. Protocol-derived features remain deferred until protocol harmonization.
#
# 9. No train / validation / test split occurs here.
#
# 10. No scaling occurs here.
#
# 11. No future labels are constructed here.
#
#
# PRIMARY OUTPUT
# --------------
# One row = one OBSERVED minute inside one physical UNSW capture.
#
# ==================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc
import warnings

warnings.filterwarnings("ignore")

START_TIME = time.time()


# ==================================================================================================
# 10.15A — PATHS
# ==================================================================================================

BASE = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

UNSW_RAW_DIR = BASE / "raw" / "UNSW_NB15"
UNSW_OUT_DIR = BASE / "audit_outputs" / "UNSW"
COMMON_OUT_DIR = BASE / "audit_outputs" / "COMMON"

UNSW_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FEATURE_FILE = (
    UNSW_RAW_DIR /
    "NUSW-NB15_features.csv"
)


TIMELINE_FILE = (
    UNSW_OUT_DIR /
    "10_14_UNSW_FLOW_TIMELINE_INDEX.csv"
)


SEMANTICS_FILE = (
    UNSW_OUT_DIR /
    "10_12_UNSW_FROZEN_FLOW_SEMANTICS.csv"
)


CORE_MAPPING_FILE = (
    COMMON_OUT_DIR /
    "10_13_1_FROZEN_CIC_UNSW_CORE_MAPPING.csv"
)


FEATURE_PLAN_FILE = (
    COMMON_OUT_DIR /
    "10_13_1_FROZEN_COMMON_TEMPORAL_FEATURE_PLAN.csv"
)


RAW_FILES = [
    UNSW_RAW_DIR / "UNSW-NB15_1.csv",
    UNSW_RAW_DIR / "UNSW-NB15_2.csv",
    UNSW_RAW_DIR / "UNSW-NB15_3.csv",
    UNSW_RAW_DIR / "UNSW-NB15_4.csv",
]


CHUNK_SIZE = 300_000


def banner(text):
    print("\n" + "=" * 120)
    print(text)
    print("=" * 120)


def subbanner(text):
    print("\n" + "-" * 120)
    print(text)
    print("-" * 120)


banner(
    "PHASE 10.15 — UNSW LEAKAGE-SAFE 1-MINUTE TEMPORAL REPRESENTATION"
)


# ==================================================================================================
# 10.15B — VERIFY REQUIRED INPUTS
# ==================================================================================================

banner("10.15B — VERIFY REQUIRED INPUTS")


required_files = [
    FEATURE_FILE,
    TIMELINE_FILE,
    SEMANTICS_FILE,
    CORE_MAPPING_FILE,
    FEATURE_PLAN_FILE,
    *RAW_FILES
]


missing = [
    p for p in required_files
    if not p.exists()
]


if missing:

    print("Missing required files:")

    for p in missing:
        print("✗", p)

    raise FileNotFoundError(
        "Phase 10.15 cannot continue because required inputs are missing."
    )


for p in required_files:
    print("✓", p.name)


# ==================================================================================================
# 10.15C — LOAD FROZEN COMMON FEATURE PLAN
# ==================================================================================================

banner("10.15C — LOAD FROZEN COMMON FEATURE PLAN")


core_mapping = pd.read_csv(
    CORE_MAPPING_FILE
)


feature_plan = pd.read_csv(
    FEATURE_PLAN_FILE
)


print(
    "Core semantic mappings:",
    len(core_mapping)
)

print(
    "Frozen temporal feature-plan rows:",
    len(feature_plan)
)


print("\nCore mapping:")
print(
    core_mapping.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.15D — DEFINE AUTHORITATIVE COMMON PREDICTOR SET
#
# These are the protocol-independent features approved for the common
# CIC ↔ UNSW representation.
#
# NOTE:
#
# Packets_Per_Second and Bytes_Per_Second are minute-level rates.
#
# Because each row represents a one-minute bucket:
#
#       rate = total / 60
#
# We DO NOT average raw per-flow rates.
# ==================================================================================================

banner("10.15D — FREEZE 14 COMMON TEMPORAL PREDICTORS")


COMMON_PREDICTORS = [

    "Flow_Count",

    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Flow_Duration_Max",

    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",

    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",

    "Packets_Per_Second",
    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",
    "Forward_Backward_Byte_Ratio"
]


PROTOCOL_DEPENDENT_DEFERRED = [

    "TCP_Flow_Count",
    "UDP_Flow_Count",
    "Protocol_Diversity"
]


print(
    "Approved protocol-independent predictors:",
    len(COMMON_PREDICTORS)
)


for x in COMMON_PREDICTORS:
    print("  ✓", x)


print("\nDeferred until protocol harmonization:")

for x in PROTOCOL_DEPENDENT_DEFERRED:
    print("  →", x)


# ==================================================================================================
# 10.15E — RECOVER RAW UNSW SCHEMA
# ==================================================================================================

banner("10.15E — RECOVER RAW UNSW SCHEMA")


features = None
feature_encoding = None


for enc in [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin1"
]:

    try:

        features = pd.read_csv(
            FEATURE_FILE,
            encoding=enc,
            low_memory=False
        )

        feature_encoding = enc

        break

    except UnicodeDecodeError:
        continue


if features is None:

    raise ValueError(
        "Could not decode UNSW feature metadata."
    )


features.columns = (
    features.columns
    .astype(str)
    .str.strip()
)


name_candidates = [
    c for c in features.columns
    if c.lower().strip() == "name"
]


if len(name_candidates) != 1:

    raise ValueError(
        "Could not uniquely identify feature-name column."
    )


NAME_COL = name_candidates[0]


raw_columns = (
    features[NAME_COL]
    .astype(str)
    .str.strip()
    .tolist()
)


if len(raw_columns) != 49:

    raise ValueError(
        f"Expected 49 UNSW raw columns; found {len(raw_columns)}."
    )


print(
    "Feature metadata encoding:",
    feature_encoding
)

print(
    "Raw feature count:",
    len(raw_columns)
)


# ==================================================================================================
# 10.15F — VERIFY REQUIRED RAW PREDICTOR FIELDS
# ==================================================================================================

banner("10.15F — VERIFY REQUIRED RAW PREDICTOR FIELDS")


REQUIRED_RAW = [

    "Stime",
    "dur",

    "Spkts",
    "Dpkts",

    "sbytes",
    "dbytes"
]


missing_raw = [
    c for c in REQUIRED_RAW
    if c not in raw_columns
]


if missing_raw:

    raise ValueError(
        f"Required UNSW predictor fields missing: {missing_raw}"
    )


for c in REQUIRED_RAW:
    print("✓", c)


# ==================================================================================================
# 10.15G — LOAD AUTHORITATIVE TIMELINE INDEX
# ==================================================================================================

banner("10.15G — LOAD PHASE 10.14 TIMELINE")


timeline = pd.read_csv(
    TIMELINE_FILE,
    parse_dates=[
        "Flow_Start",
        "Flow_End",
        "Minute"
    ],
    low_memory=False
)


required_timeline_cols = [

    "Global_Raw_Row_ID",
    "Raw_File",
    "Raw_File_ID",
    "Raw_Row_In_File",
    "Flow_Start",
    "Capture_ID",
    "Minute"
]


missing_timeline = [
    c for c in required_timeline_cols
    if c not in timeline.columns
]


if missing_timeline:

    raise ValueError(
        f"Timeline missing columns: {missing_timeline}"
    )


if timeline["Global_Raw_Row_ID"].duplicated().any():

    raise ValueError(
        "Global_Raw_Row_ID is not unique in Phase 10.14 timeline."
    )


timeline_lookup = (
    timeline[
        [
            "Global_Raw_Row_ID",
            "Capture_ID",
            "Minute"
        ]
    ]
    .copy()
)


timeline_lookup = (
    timeline_lookup
    .sort_values(
        "Global_Raw_Row_ID"
    )
    .reset_index(
        drop=True
    )
)


print(
    "Timeline rows:",
    f"{len(timeline_lookup):,}"
)

print(
    "Observed minutes:",
    f"{timeline_lookup['Minute'].nunique():,}"
)

print(
    "Capture distribution:"
)

print(
    timeline_lookup[
        "Capture_ID"
    ].value_counts()
)


# ==================================================================================================
# 10.15H — READ ONLY REQUIRED RAW PREDICTOR COLUMNS
#
# IMPORTANT:
#
# We do NOT load all 49 features.
#
# We do NOT load Label or attack_cat into this predictor branch.
#
# This is an explicit anti-leakage design choice.
# ==================================================================================================

banner("10.15H — BUILD FLOW-LEVEL PREDICTOR BRANCH")


predictor_parts = []

global_offset = 0


for file_idx, path in enumerate(
    RAW_FILES,
    start=1
):

    subbanner(
        f"[{file_idx}/4] {path.name}"
    )


    row_in_file = 0

    file_start = time.time()


    reader = pd.read_csv(

        path,

        names=raw_columns,

        header=None,

        usecols=REQUIRED_RAW,

        chunksize=CHUNK_SIZE,

        low_memory=False
    )


    for chunk_idx, chunk in enumerate(
        reader,
        start=1
    ):

        n = len(chunk)


        # ------------------------------------------------------------------------------------------
        # Stable row identity
        # ------------------------------------------------------------------------------------------

        chunk[
            "Global_Raw_Row_ID"
        ] = np.arange(

            global_offset + row_in_file,

            global_offset + row_in_file + n,

            dtype=np.int64
        )


        # ------------------------------------------------------------------------------------------
        # Numeric coercion
        # ------------------------------------------------------------------------------------------

        for c in REQUIRED_RAW:

            chunk[c] = pd.to_numeric(
                chunk[c],
                errors="coerce"
            )


        # ------------------------------------------------------------------------------------------
        # Rename raw fields into semantic names
        # ------------------------------------------------------------------------------------------

        chunk = chunk.rename(
            columns={

                "dur":
                    "Flow_Duration_Seconds",

                "Spkts":
                    "Forward_Packets",

                "Dpkts":
                    "Backward_Packets",

                "sbytes":
                    "Forward_Bytes",

                "dbytes":
                    "Backward_Bytes"
            }
        )


        predictor_parts.append(
            chunk[
                [
                    "Global_Raw_Row_ID",

                    "Flow_Duration_Seconds",

                    "Forward_Packets",
                    "Backward_Packets",

                    "Forward_Bytes",
                    "Backward_Bytes"
                ]
            ]
        )


        row_in_file += n


        print(
            f"  chunk {chunk_idx}: "
            f"{n:,} rows "
            f"(cumulative {row_in_file:,})"
        )


    global_offset += row_in_file


    print(
        f"✓ {row_in_file:,} rows processed "
        f"in {time.time() - file_start:.1f} sec"
    )


flow_predictors = pd.concat(
    predictor_parts,
    ignore_index=True
)


del predictor_parts

gc.collect()


print(
    "\nPredictor rows:",
    f"{len(flow_predictors):,}"
)


# ==================================================================================================
# 10.15I — FLOW-LEVEL PREDICTOR QUALITY CHECK
# ==================================================================================================

banner("10.15I — FLOW-LEVEL PREDICTOR QUALITY CHECK")


if len(flow_predictors) != len(timeline_lookup):

    raise ValueError(
        "Predictor row count does not equal authoritative timeline row count."
    )


numeric_predictors = [

    "Flow_Duration_Seconds",

    "Forward_Packets",
    "Backward_Packets",

    "Forward_Bytes",
    "Backward_Bytes"
]


quality_rows = []


for c in numeric_predictors:

    arr = flow_predictors[c]

    quality_rows.append(
        {
            "Feature": c,

            "Missing":
                int(
                    arr.isna().sum()
                ),

            "Infinite":
                int(
                    np.isinf(
                        arr.to_numpy(
                            dtype=float,
                            na_value=np.nan
                        )
                    ).sum()
                ),

            "Negative":
                int(
                    (arr < 0).sum()
                ),

            "Min":
                arr.min(),

            "Median":
                arr.median(),

            "Max":
                arr.max()
        }
    )


flow_quality = pd.DataFrame(
    quality_rows
)


print(
    flow_quality.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.15J — ATTACH AUTHORITATIVE TEMPORAL METADATA
#
# This join is based ONLY on row identity.
#
# Capture_ID and Minute remain metadata.
# ==================================================================================================

banner("10.15J — ATTACH TEMPORAL METADATA")


flow_predictors = flow_predictors.merge(

    timeline_lookup,

    on="Global_Raw_Row_ID",

    how="left",

    validate="one_to_one"
)


missing_minute = int(
    flow_predictors[
        "Minute"
    ].isna().sum()
)


missing_capture = int(
    flow_predictors[
        "Capture_ID"
    ].isna().sum()
)


print(
    "Rows missing Minute:",
    f"{missing_minute:,}"
)

print(
    "Rows missing Capture_ID:",
    f"{missing_capture:,}"
)


if missing_minute > 0 or missing_capture > 0:

    raise ValueError(
        "Temporal metadata join failed."
    )


# ==================================================================================================
# 10.15K — AGGREGATE PREDICTORS TO OBSERVED MINUTES
# ==================================================================================================

banner("10.15K — AGGREGATE PREDICTORS TO 1-MINUTE WINDOWS")


minute_predictors = (

    flow_predictors

    .groupby(
        [
            "Capture_ID",
            "Minute"
        ],
        sort=True,
        observed=True
    )

    .agg(

        Flow_Count=(
            "Global_Raw_Row_ID",
            "size"
        ),

        Flow_Duration_Mean=(
            "Flow_Duration_Seconds",
            "mean"
        ),

        Flow_Duration_Std=(
            "Flow_Duration_Seconds",
            "std"
        ),

        Flow_Duration_Max=(
            "Flow_Duration_Seconds",
            "max"
        ),

        Forward_Packets_Sum=(
            "Forward_Packets",
            "sum"
        ),

        Backward_Packets_Sum=(
            "Backward_Packets",
            "sum"
        ),

        Forward_Bytes_Sum=(
            "Forward_Bytes",
            "sum"
        ),

        Backward_Bytes_Sum=(
            "Backward_Bytes",
            "sum"
        )

    )

    .reset_index()
)


# ==================================================================================================
# 10.15L — DERIVE COMMON MINUTE FEATURES
# ==================================================================================================

banner("10.15L — DERIVE COMMON MINUTE FEATURES")


minute_predictors[
    "Total_Packets"
] = (

    minute_predictors[
        "Forward_Packets_Sum"
    ]

    +

    minute_predictors[
        "Backward_Packets_Sum"
    ]
)


minute_predictors[
    "Total_Bytes"
] = (

    minute_predictors[
        "Forward_Bytes_Sum"
    ]

    +

    minute_predictors[
        "Backward_Bytes_Sum"
    ]
)


# --------------------------------------------------------------------------------------------------
# Minute-level traffic rates
# --------------------------------------------------------------------------------------------------

minute_predictors[
    "Packets_Per_Second"
] = (

    minute_predictors[
        "Total_Packets"
    ]

    / 60.0
)


minute_predictors[
    "Bytes_Per_Second"
] = (

    minute_predictors[
        "Total_Bytes"
    ]

    / 60.0
)


# --------------------------------------------------------------------------------------------------
# Directional ratios
#
# We use denominator + 1 to:
#
#   • avoid division-by-zero
#   • avoid Inf
#   • make the definition reproducible for CIC and UNSW
#
# The SAME formula must later be applied to CIC.
# --------------------------------------------------------------------------------------------------

minute_predictors[
    "Forward_Backward_Packet_Ratio"
] = (

    minute_predictors[
        "Forward_Packets_Sum"
    ]

    /

    (
        minute_predictors[
            "Backward_Packets_Sum"
        ]

        + 1.0
    )
)


minute_predictors[
    "Forward_Backward_Byte_Ratio"
] = (

    minute_predictors[
        "Forward_Bytes_Sum"
    ]

    /

    (
        minute_predictors[
            "Backward_Bytes_Sum"
        ]

        + 1.0
    )
)


# --------------------------------------------------------------------------------------------------
# One-flow minutes have undefined sample std in pandas.
#
# This is NOT a missing-data event.
#
# It means there is no within-minute duration variation observable from
# a single flow, so we define std = 0 for exactly those windows.
# --------------------------------------------------------------------------------------------------

single_flow_mask = (
    minute_predictors[
        "Flow_Count"
    ]
    ==
    1
)


minute_predictors.loc[
    single_flow_mask,
    "Flow_Duration_Std"
] = 0.0


print(
    "Observed minute rows:",
    f"{len(minute_predictors):,}"
)


print(
    "\nMinutes by capture:"
)

print(
    minute_predictors[
        "Capture_ID"
    ].value_counts()
)


# ==================================================================================================
# 10.15M — VERIFY COMMON PREDICTOR COMPLETENESS
# ==================================================================================================

banner("10.15M — COMMON PREDICTOR COMPLETENESS")


missing_common = [
    c for c in COMMON_PREDICTORS
    if c not in minute_predictors.columns
]


if missing_common:

    raise ValueError(
        f"Missing common predictors: {missing_common}"
    )


predictor_quality_rows = []


for c in COMMON_PREDICTORS:

    values = pd.to_numeric(
        minute_predictors[c],
        errors="coerce"
    )


    array = values.to_numpy(
        dtype=float
    )


    predictor_quality_rows.append(
        {
            "Feature":
                c,

            "Missing":
                int(
                    values.isna().sum()
                ),

            "Infinite":
                int(
                    np.isinf(
                        array
                    ).sum()
                ),

            "Min":
                np.nanmin(
                    array
                ),

            "Median":
                np.nanmedian(
                    array
                ),

            "Max":
                np.nanmax(
                    array
                )
        }
    )


minute_predictor_quality = pd.DataFrame(
    predictor_quality_rows
)


print(
    minute_predictor_quality.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.15N — LOAD AUTHORITATIVE PHASE 10.12 FLOW SEMANTICS
#
# We deliberately inspect the schema instead of assuming exact column names.
# ==================================================================================================

banner("10.15N — LOAD AUTHORITATIVE 10.12 FLOW SEMANTICS")


semantics = pd.read_csv(
    SEMANTICS_FILE,
    low_memory=False
)


print(
    "Semantic rows:",
    f"{len(semantics):,}"
)

print(
    "Semantic columns:"
)

print(
    semantics.columns.tolist()
)


if len(semantics) != len(timeline_lookup):

    raise ValueError(
        "10.12 semantics row count does not equal Phase 10.14 timeline row count."
    )


# ==================================================================================================
# 10.15O — IDENTIFY AUTHORITATIVE SEMANTIC FIELDS — CORRECTED
# ==================================================================================================

banner("10.15O — IDENTIFY AUTHORITATIVE SEMANTIC FIELDS")


def find_first_column(df, candidates, required=True):

    exact_lookup = {
        str(c).lower().strip(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = candidate.lower().strip()

        if key in exact_lookup:
            return exact_lookup[key]

    if required:
        raise ValueError(
            "Could not identify required column.\n"
            f"Candidates tried: {candidates}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    return None


# --------------------------------------------------------------------------------------------------
# Row identity
#
# Phase 10.12 does NOT contain Global_Raw_Row_ID.
# It contains:
#
#     Flow_File
#     Flow_Row_Index
#
# We will therefore reconstruct the exact global row identity from these fields.
# --------------------------------------------------------------------------------------------------

FLOW_FILE_COL = find_first_column(
    semantics,
    [
        "Flow_File",
        "Flow_File_10_12"
    ]
)

FLOW_ROW_COL = find_first_column(
    semantics,
    [
        "Flow_Row_Index",
        "Flow_Row_10_12"
    ]
)


# --------------------------------------------------------------------------------------------------
# AUTHORITATIVE FINAL LABELS
#
# IMPORTANT:
#
# Use FINAL semantics from Phase 10.12, not intermediate Resolved_* fields.
# --------------------------------------------------------------------------------------------------

BINARY_COL = find_first_column(
    semantics,
    [
        "Final_Attack_State",
        "Resolved_Attack_State",
        "Attack_State"
    ]
)


FAMILY_COL = find_first_column(
    semantics,
    [
        "Final_Attack_Family",
        "Resolved_Family_Canonical",
        "Resolved_Attack_Family",
        "Attack_Family"
    ]
)


# --------------------------------------------------------------------------------------------------
# Eligibility fields
# --------------------------------------------------------------------------------------------------

BINARY_ELIGIBILITY_COL = find_first_column(
    semantics,
    [
        "Eligible_Binary_Forecasting"
    ],
    required=False
)


FAMILY_ELIGIBILITY_COL = find_first_column(
    semantics,
    [
        "Eligible_Attack_Family_Forecasting",
        "Eligible_Family_Forecasting"
    ],
    required=False
)


REVIEW_COL = find_first_column(
    semantics,
    [
        "Semantic_Review_Status",
        "Requires_Label_Review"
    ],
    required=False
)


print("Flow-file column             :", FLOW_FILE_COL)
print("Flow-row column              :", FLOW_ROW_COL)
print("Binary semantic column       :", BINARY_COL)
print("Family semantic column       :", FAMILY_COL)
print("Binary eligibility column    :", BINARY_ELIGIBILITY_COL)
print("Family eligibility column    :", FAMILY_ELIGIBILITY_COL)
print("Semantic review column       :", REVIEW_COL)


assert BINARY_COL == "Final_Attack_State", (
    f"Expected Final_Attack_State but selected {BINARY_COL}"
)

assert FAMILY_COL == "Final_Attack_Family", (
    f"Expected Final_Attack_Family but selected {FAMILY_COL}"
)


print("\n✓ Final Phase 10.12 authoritative semantics selected")


# ==================================================================================================
# 10.15P — RECONSTRUCT EXACT GLOBAL RAW ROW IDENTITY
# ==================================================================================================

banner("10.15P — RECONSTRUCT EXACT GLOBAL RAW ROW IDENTITY")


# --------------------------------------------------------------------------------------------------
# Phase 10.14 global row IDs were constructed by concatenating:
#
#     UNSW-NB15_1.csv
#     UNSW-NB15_2.csv
#     UNSW-NB15_3.csv
#     UNSW-NB15_4.csv
#
# in this exact order.
#
# Known row counts:
#
#     file 1 = 700001
#     file 2 = 700001
#     file 3 = 700001
#     file 4 = 440044
#
# We reconstruct the global ID explicitly instead of trusting dataframe position.
# --------------------------------------------------------------------------------------------------

FILE_ROW_COUNTS = {
    "UNSW-NB15_1.csv": 700001,
    "UNSW-NB15_2.csv": 700001,
    "UNSW-NB15_3.csv": 700001,
    "UNSW-NB15_4.csv": 440044
}


FILE_OFFSETS = {}

running_offset = 0

for filename in [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv"
]:

    FILE_OFFSETS[filename] = running_offset

    running_offset += FILE_ROW_COUNTS[filename]


print("Expected total raw rows:", f"{running_offset:,}")

print("\nFile offsets:")

for k, v in FILE_OFFSETS.items():
    print(f"  {k}: {v:,}")


# --------------------------------------------------------------------------------------------------
# Normalize filename
# --------------------------------------------------------------------------------------------------

semantic_file = (
    semantics[FLOW_FILE_COL]
    .astype(str)
    .str.strip()
    .str.replace("\\", "/", regex=False)
    .str.split("/")
    .str[-1]
)


unknown_files = sorted(
    set(semantic_file.dropna().unique())
    -
    set(FILE_OFFSETS.keys())
)


if unknown_files:

    print("\nUnknown Flow_File values:")

    for x in unknown_files[:20]:
        print("  ", repr(x))

    raise ValueError(
        "Phase 10.12 contains unexpected Flow_File values."
    )


# --------------------------------------------------------------------------------------------------
# Normalize row index
# --------------------------------------------------------------------------------------------------

semantic_row = pd.to_numeric(
    semantics[FLOW_ROW_COL],
    errors="coerce"
)


if semantic_row.isna().any():

    raise ValueError(
        f"{semantic_row.isna().sum():,} invalid Flow_Row_Index values."
    )


semantic_row = semantic_row.astype(np.int64)


# --------------------------------------------------------------------------------------------------
# Determine whether Flow_Row_Index is zero-based or one-based.
#
# We inspect minima/maxima per file rather than assuming.
# --------------------------------------------------------------------------------------------------

row_index_audit = pd.DataFrame({
    "Flow_File_Normalized": semantic_file,
    "Flow_Row_Index_Numeric": semantic_row
})


row_index_summary = (
    row_index_audit
    .groupby(
        "Flow_File_Normalized",
        observed=True
    )
    ["Flow_Row_Index_Numeric"]
    .agg(
        ["count", "min", "max", "nunique"]
    )
    .reset_index()
)


print("\nFlow-row indexing audit:")

print(
    row_index_summary.to_string(
        index=False
    )
)


zero_based_valid = True
one_based_valid = True


for _, row in row_index_summary.iterrows():

    filename = row["Flow_File_Normalized"]

    expected_n = FILE_ROW_COUNTS[filename]

    # zero-based expected:
    #
    #     0 ... N-1

    if not (
        row["min"] == 0
        and
        row["max"] == expected_n - 1
        and
        row["count"] == expected_n
        and
        row["nunique"] == expected_n
    ):
        zero_based_valid = False


    # one-based expected:
    #
    #     1 ... N

    if not (
        row["min"] == 1
        and
        row["max"] == expected_n
        and
        row["count"] == expected_n
        and
        row["nunique"] == expected_n
    ):
        one_based_valid = False


if zero_based_valid and not one_based_valid:

    ROW_INDEX_BASE = 0

elif one_based_valid and not zero_based_valid:

    ROW_INDEX_BASE = 1

else:

    raise ValueError(
        "Could not uniquely determine whether Flow_Row_Index "
        "is zero-based or one-based."
    )


print(
    "\nDetected Flow_Row_Index base:",
    ROW_INDEX_BASE
)


# --------------------------------------------------------------------------------------------------
# Reconstruct global ID
# --------------------------------------------------------------------------------------------------

offset_series = semantic_file.map(
    FILE_OFFSETS
).astype(np.int64)


semantics[
    "Global_Raw_Row_ID_JOIN"
] = (

    offset_series

    +

    semantic_row

    -

    ROW_INDEX_BASE
)


# --------------------------------------------------------------------------------------------------
# Integrity
# --------------------------------------------------------------------------------------------------

if semantics[
    "Global_Raw_Row_ID_JOIN"
].duplicated().any():

    duplicate_count = int(
        semantics[
            "Global_Raw_Row_ID_JOIN"
        ].duplicated().sum()
    )

    raise ValueError(
        f"Reconstructed semantic row identity contains "
        f"{duplicate_count:,} duplicate IDs."
    )


expected_ids = np.arange(
    len(timeline_lookup),
    dtype=np.int64
)


actual_ids = np.sort(
    semantics[
        "Global_Raw_Row_ID_JOIN"
    ].to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    actual_ids,
    expected_ids
):

    missing_ids = np.setdiff1d(
        expected_ids,
        actual_ids
    )

    extra_ids = np.setdiff1d(
        actual_ids,
        expected_ids
    )

    print(
        "Missing IDs:",
        len(missing_ids)
    )

    print(
        "Extra IDs:",
        len(extra_ids)
    )

    raise ValueError(
        "Reconstructed Phase 10.12 IDs do not exactly cover "
        "the Phase 10.14 raw-flow identity space."
    )


identity_method = (
    f"{FLOW_FILE_COL} + {FLOW_ROW_COL} "
    f"(detected {ROW_INDEX_BASE}-based indexing)"
)


print(
    "\nSemantic identity method:",
    identity_method
)

print(
    "Unique reconstructed IDs:",
    f"{semantics['Global_Raw_Row_ID_JOIN'].nunique():,}"
)

print(
    "Expected IDs:",
    f"{len(timeline_lookup):,}"
)

print(
    "\n✓ Exact semantic ↔ raw-flow identity reconstructed")


# ==================================================================================================
# 10.15Q — NORMALIZE FINAL AUTHORITATIVE ATTACK SEMANTICS
# ==================================================================================================

banner("10.15Q — NORMALIZE FINAL AUTHORITATIVE ATTACK SEMANTICS")


binary_raw = semantics[
    BINARY_COL
]


# --------------------------------------------------------------------------------------------------
# Normalize binary target
# --------------------------------------------------------------------------------------------------

if pd.api.types.is_numeric_dtype(
    binary_raw
):

    binary_numeric = pd.to_numeric(
        binary_raw,
        errors="coerce"
    )

else:

    binary_text = (
        binary_raw
        .astype(str)
        .str.strip()
        .str.lower()
    )


    binary_map = {

        "0": 0,
        "0.0": 0,

        "benign": 0,
        "normal": 0,

        "false": 0,
        "no": 0,

        "1": 1,
        "1.0": 1,

        "attack": 1,
        "malicious": 1,

        "true": 1,
        "yes": 1
    }


    binary_numeric = binary_text.map(
        binary_map
    )


if binary_numeric.isna().any():

    bad_examples = (
        semantics.loc[
            binary_numeric.isna(),
            BINARY_COL
        ]
        .astype(str)
        .value_counts()
        .head(30)
    )


    print(
        "\nUnrecognized Final_Attack_State values:"
    )

    print(
        bad_examples
    )


    raise ValueError(
        "Final_Attack_State contains unresolved/unrecognized values."
    )


# Binary target must genuinely be binary.

unique_binary = sorted(
    binary_numeric.unique().tolist()
)


print(
    "Unique normalized binary states:",
    unique_binary
)


if not set(unique_binary).issubset(
    {0, 1}
):

    raise ValueError(
        f"Final_Attack_State is not binary: {unique_binary}"
    )


semantics[
    "Attack_State"
] = binary_numeric.astype(
    np.int8
)


# --------------------------------------------------------------------------------------------------
# Normalize family target
# --------------------------------------------------------------------------------------------------

family_raw = semantics[
    FAMILY_COL
]


family_missing_original = (
    family_raw.isna()
)


semantics[
    "Attack_Family"
] = (
    family_raw
    .fillna("")
    .astype(str)
    .str.strip()
)


invalid_family_strings = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "<na>"
}


family_lower = (
    semantics[
        "Attack_Family"
    ]
    .str.lower()
)


# --------------------------------------------------------------------------------------------------
# Benign rows receive explicit BENIGN family.
# --------------------------------------------------------------------------------------------------

benign_flow_mask = (
    semantics[
        "Attack_State"
    ]
    ==
    0
)


semantics.loc[
    benign_flow_mask,
    "Attack_Family"
] = "BENIGN"


# --------------------------------------------------------------------------------------------------
# Attack rows MUST have a final authoritative family.
# --------------------------------------------------------------------------------------------------

attack_flow_mask = (
    semantics[
        "Attack_State"
    ]
    ==
    1
)


invalid_attack_family = (

    attack_flow_mask

    &

    (
        family_missing_original

        |

        family_lower.isin(
            invalid_family_strings
        )
    )
)


invalid_attack_family_count = int(
    invalid_attack_family.sum()
)


print(
    "Attack flows lacking final family:",
    f"{invalid_attack_family_count:,}"
)


if invalid_attack_family_count > 0:

    print(
        "\nExamples:"
    )

    print(
        semantics.loc[
            invalid_attack_family,
            [
                FLOW_FILE_COL,
                FLOW_ROW_COL,
                BINARY_COL,
                FAMILY_COL
            ]
        ]
        .head(20)
        .to_string(
            index=False
        )
    )


    raise ValueError(
        "Some Final_Attack_State=1 rows lack Final_Attack_Family. "
        "Do not silently invent attack families."
    )


print(
    "\nFinal binary distribution:"
)

print(
    semantics[
        "Attack_State"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nFinal attack-family distribution:"
)

print(
    semantics.loc[
        semantics[
            "Attack_State"
        ]
        ==
        1,
        "Attack_Family"
    ]
    .value_counts(
        dropna=False
    )
)


# ==================================================================================================
# 10.15R — ATTACH TEMPORAL METADATA TO TARGET BRANCH
# ==================================================================================================

banner("10.15R — ATTACH TEMPORAL METADATA TO TARGET BRANCH")


semantic_branch = semantics[
    [
        "Global_Raw_Row_ID_JOIN",
        "Attack_State",
        "Attack_Family"
    ]
].rename(
    columns={
        "Global_Raw_Row_ID_JOIN":
            "Global_Raw_Row_ID"
    }
)


semantic_branch = semantic_branch.merge(

    timeline_lookup,

    on="Global_Raw_Row_ID",

    how="left",

    validate="one_to_one"
)


missing_semantic_minute = int(
    semantic_branch[
        "Minute"
    ].isna().sum()
)


missing_semantic_capture = int(
    semantic_branch[
        "Capture_ID"
    ].isna().sum()
)


print(
    "Semantic flows attached:",
    f"{len(semantic_branch):,}"
)

print(
    "Missing Minute:",
    f"{missing_semantic_minute:,}"
)

print(
    "Missing Capture_ID:",
    f"{missing_semantic_capture:,}"
)


if (
    missing_semantic_minute > 0
    or
    missing_semantic_capture > 0
):

    raise ValueError(
        "Authoritative semantic rows failed temporal attachment."
    )


# ==================================================================================================
# 10.15S — AGGREGATE AUTHORITATIVE TARGET SEMANTICS
# ==================================================================================================

banner("10.15S — AGGREGATE AUTHORITATIVE TARGET SEMANTICS")


minute_target_base = (

    semantic_branch

    .groupby(
        [
            "Capture_ID",
            "Minute"
        ],
        sort=True,
        observed=True
    )

    .agg(

        Semantic_Flow_Count=(
            "Global_Raw_Row_ID",
            "size"
        ),

        Attack_Flow_Count=(
            "Attack_State",
            "sum"
        ),

        Attack_State=(
            "Attack_State",
            "max"
        )

    )

    .reset_index()
)


minute_target_base[
    "Benign_Flow_Count"
] = (

    minute_target_base[
        "Semantic_Flow_Count"
    ]

    -

    minute_target_base[
        "Attack_Flow_Count"
    ]
)


minute_target_base[
    "Attack_Fraction"
] = (

    minute_target_base[
        "Attack_Flow_Count"
    ]

    /

    minute_target_base[
        "Semantic_Flow_Count"
    ]
)


# --------------------------------------------------------------------------------------------------
# Attack-family composition
# --------------------------------------------------------------------------------------------------

attack_flows = semantic_branch.loc[
    semantic_branch[
        "Attack_State"
    ]
    ==
    1,
    [
        "Capture_ID",
        "Minute",
        "Attack_Family"
    ]
].copy()


if len(attack_flows) > 0:

    family_counts = (

        attack_flows

        .groupby(
            [
                "Capture_ID",
                "Minute",
                "Attack_Family"
            ],
            observed=True,
            sort=True
        )

        .size()

        .rename(
            "Family_Flow_Count"
        )

        .reset_index()
    )


    # ----------------------------------------------------------------------------------------------
    # Sort once.
    #
    # Highest family flow count wins.
    #
    # Lexical family ordering is ONLY a deterministic audit tie-break.
    # The tie itself is preserved separately.
    # ----------------------------------------------------------------------------------------------

    family_counts = family_counts.sort_values(

        [
            "Capture_ID",
            "Minute",
            "Family_Flow_Count",
            "Attack_Family"
        ],

        ascending=[
            True,
            True,
            False,
            True
        ],

        kind="mergesort"
    )


    dominant_family = (

        family_counts

        .drop_duplicates(
            subset=[
                "Capture_ID",
                "Minute"
            ],
            keep="first"
        )

        [
            [
                "Capture_ID",
                "Minute",
                "Attack_Family",
                "Family_Flow_Count"
            ]
        ]

        .rename(
            columns={
                "Attack_Family":
                    "Dominant_Attack_Family",

                "Family_Flow_Count":
                    "Dominant_Family_Flow_Count"
            }
        )
    )


    family_stats = (

        family_counts

        .groupby(
            [
                "Capture_ID",
                "Minute"
            ],
            observed=True,
            sort=False
        )

        .agg(

            Attack_Family_Count=(
                "Attack_Family",
                "nunique"
            ),

            Max_Family_Flow_Count=(
                "Family_Flow_Count",
                "max"
            )

        )

        .reset_index()
    )


    family_counts_for_tie = family_counts.merge(

        family_stats[
            [
                "Capture_ID",
                "Minute",
                "Max_Family_Flow_Count"
            ]
        ],

        on=[
            "Capture_ID",
            "Minute"
        ],

        how="left",

        validate="many_to_one"
    )


    family_counts_for_tie[
        "Is_Dominant_Tie_Member"
    ] = (

        family_counts_for_tie[
            "Family_Flow_Count"
        ]

        ==

        family_counts_for_tie[
            "Max_Family_Flow_Count"
        ]
    )


    tie_counts = (

        family_counts_for_tie

        .groupby(
            [
                "Capture_ID",
                "Minute"
            ],
            observed=True,
            sort=False
        )[
            "Is_Dominant_Tie_Member"
        ]

        .sum()

        .rename(
            "Dominant_Family_Tie_Count"
        )

        .reset_index()
    )


    family_summary = (

        dominant_family

        .merge(
            family_stats,

            on=[
                "Capture_ID",
                "Minute"
            ],

            how="one_to_one"
            if False else "left"
        )
    )


    family_summary = family_summary.merge(

        tie_counts,

        on=[
            "Capture_ID",
            "Minute"
        ],

        how="left",

        validate="one_to_one"
    )


    family_summary[
        "Multi_Family_Attack_Minute"
    ] = (

        family_summary[
            "Attack_Family_Count"
        ]

        >
        1
    )


    family_summary[
        "Dominant_Family_Tie"
    ] = (

        family_summary[
            "Dominant_Family_Tie_Count"
        ]

        >
        1
    )


else:

    family_summary = pd.DataFrame(
        columns=[
            "Capture_ID",
            "Minute",
            "Dominant_Attack_Family",
            "Dominant_Family_Flow_Count",
            "Attack_Family_Count",
            "Dominant_Family_Tie_Count",
            "Multi_Family_Attack_Minute",
            "Dominant_Family_Tie"
        ]
    )


minute_targets = minute_target_base.merge(

    family_summary,

    on=[
        "Capture_ID",
        "Minute"
    ],

    how="left",

    validate="one_to_one"
)


# --------------------------------------------------------------------------------------------------
# Explicit benign semantics
# --------------------------------------------------------------------------------------------------

benign_minute_mask = (
    minute_targets[
        "Attack_State"
    ]
    ==
    0
)


minute_targets.loc[
    benign_minute_mask,
    "Dominant_Attack_Family"
] = "BENIGN"


minute_targets.loc[
    benign_minute_mask,
    "Dominant_Family_Flow_Count"
] = 0


minute_targets.loc[
    benign_minute_mask,
    "Attack_Family_Count"
] = 0


minute_targets.loc[
    benign_minute_mask,
    "Dominant_Family_Tie_Count"
] = 0


minute_targets.loc[
    benign_minute_mask,
    "Multi_Family_Attack_Minute"
] = False


minute_targets.loc[
    benign_minute_mask,
    "Dominant_Family_Tie"
] = False


# Numeric cleanup

for c in [
    "Dominant_Family_Flow_Count",
    "Attack_Family_Count",
    "Dominant_Family_Tie_Count"
]:

    minute_targets[c] = (
        pd.to_numeric(
            minute_targets[c],
            errors="coerce"
        )
        .fillna(0)
        .astype(np.int64)
    )


minute_targets[
    "Multi_Family_Attack_Minute"
] = (
    minute_targets[
        "Multi_Family_Attack_Minute"
    ]
    .fillna(False)
    .astype(bool)
)


minute_targets[
    "Dominant_Family_Tie"
] = (
    minute_targets[
        "Dominant_Family_Tie"
    ]
    .fillna(False)
    .astype(bool)
)


print(
    "Minute target rows:",
    f"{len(minute_targets):,}"
)


print(
    "\nMinute Attack_State distribution:"
)

print(
    minute_targets[
        "Attack_State"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nDominant attack-family distribution:"
)

print(
    minute_targets[
        "Dominant_Attack_Family"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nMulti-family attack minutes:",
    f"{int(minute_targets['Multi_Family_Attack_Minute'].sum()):,}"
)


print(
    "Dominant-family tie minutes:",
    f"{int(minute_targets['Dominant_Family_Tie'].sum()):,}"
)


# ==================================================================================================
# 10.15T — JOIN PREDICTOR AND TARGET BRANCHES
# ==================================================================================================

banner("10.15T — JOIN PREDICTOR + TARGET BRANCHES")


minute_df = minute_predictors.merge(

    minute_targets,

    on=[
        "Capture_ID",
        "Minute"
    ],

    how="outer",

    validate="one_to_one",

    indicator=True
)


print(
    minute_df[
        "_merge"
    ].value_counts(
        dropna=False
    )
)


if not (
    minute_df[
        "_merge"
    ]
    ==
    "both"
).all():

    raise ValueError(
        "Predictor and target minute grids do not match exactly."
    )


minute_df = minute_df.drop(
    columns=[
        "_merge"
    ]
)


minute_df[
    "Flow_Count_Agreement"
] = (

    minute_df[
        "Flow_Count"
    ]

    ==

    minute_df[
        "Semantic_Flow_Count"
    ]
)


flow_count_disagreement = int(

    (
        ~minute_df[
            "Flow_Count_Agreement"
        ]
    ).sum()
)


print(
    "\nFlow-count disagreements:",
    f"{flow_count_disagreement:,}"
)


if flow_count_disagreement > 0:

    raise ValueError(
        "Predictor and semantic branches disagree on minute flow counts."
    )


print(
    "✓ Predictor and target branches agree exactly"
)


# ==================================================================================================
# 10.15U — OBSERVED-MINUTE CONTINUITY METADATA
# ==================================================================================================

banner("10.15U — OBSERVED-MINUTE CONTINUITY METADATA")


minute_df = minute_df.sort_values(

    [
        "Capture_ID",
        "Minute"
    ],

    kind="mergesort",

    ignore_index=True
)


minute_df[
    "Previous_Observed_Minute"
] = (

    minute_df

    .groupby(
        "Capture_ID",
        observed=True
    )[
        "Minute"
    ]

    .shift(1)
)


minute_df[
    "Minutes_Since_Previous_Observed"
] = (

    (
        minute_df[
            "Minute"
        ]

        -

        minute_df[
            "Previous_Observed_Minute"
        ]
    )

    .dt.total_seconds()

    / 60.0
)


minute_df[
    "Is_Contiguous_With_Previous"
] = (

    minute_df[
        "Minutes_Since_Previous_Observed"
    ]

    ==
    1.0
)


first_in_capture = (
    minute_df[
        "Previous_Observed_Minute"
    ].isna()
)


minute_df.loc[
    first_in_capture,
    "Is_Contiguous_With_Previous"
] = False


minute_df[
    "Starts_New_Observed_Segment"
] = (

    ~minute_df[
        "Is_Contiguous_With_Previous"
    ]
)


minute_df[
    "Temporal_Segment_Number"
] = (

    minute_df

    .groupby(
        "Capture_ID",
        observed=True
    )[
        "Starts_New_Observed_Segment"
    ]

    .cumsum()

    .astype(int)
)


minute_df[
    "Temporal_Segment_ID"
] = (

    minute_df[
        "Capture_ID"
    ].astype(str)

    +

    "_SEG_"

    +

    minute_df[
        "Temporal_Segment_Number"
    ]
    .astype(str)
    .str.zfill(3)
)


segment_summary = (

    minute_df

    .groupby(
        [
            "Capture_ID",
            "Temporal_Segment_ID"
        ],
        observed=True
    )

    .agg(

        Start_Minute=(
            "Minute",
            "min"
        ),

        End_Minute=(
            "Minute",
            "max"
        ),

        Observed_Minutes=(
            "Minute",
            "size"
        ),

        Flow_Count=(
            "Flow_Count",
            "sum"
        )

    )

    .reset_index()
)


segment_summary[
    "Expected_Continuous_Minutes"
] = (

    (
        segment_summary[
            "End_Minute"
        ]

        -

        segment_summary[
            "Start_Minute"
        ]
    )

    .dt.total_seconds()

    / 60

    + 1

).astype(int)


print(
    "Temporal segments:",
    len(segment_summary)
)


print(
    segment_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.15V — MISSING-MINUTE AUDIT
# ==================================================================================================

banner("10.15V — MISSING-MINUTE AUDIT")


capture_minute_summary_rows = []


for capture_id, g in minute_df.groupby(
    "Capture_ID",
    observed=True
):

    start_minute = g[
        "Minute"
    ].min()

    end_minute = g[
        "Minute"
    ].max()

    observed_minutes = len(g)


    expected_minutes = int(

        (
            end_minute
            -
            start_minute
        ).total_seconds()

        // 60

        + 1
    )


    missing_minutes = (
        expected_minutes
        -
        observed_minutes
    )


    capture_minute_summary_rows.append(
        {
            "Capture_ID":
                capture_id,

            "Start_Minute":
                start_minute,

            "End_Minute":
                end_minute,

            "Observed_Minutes":
                observed_minutes,

            "Expected_Continuous_Minutes":
                expected_minutes,

            "Missing_Minutes":
                missing_minutes,

            "Observed_Coverage_Pct":
                (
                    observed_minutes
                    /
                    expected_minutes
                    *
                    100
                )
        }
    )


capture_minute_summary = pd.DataFrame(
    capture_minute_summary_rows
)


print(
    capture_minute_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.15W — TARGET / PREDICTOR LEAKAGE FIREWALL
# ==================================================================================================

banner("10.15W — TARGET / PREDICTOR LEAKAGE FIREWALL")


TARGET_AND_AUDIT_FIELDS = [

    "Attack_State",

    "Attack_Flow_Count",

    "Benign_Flow_Count",

    "Attack_Fraction",

    "Dominant_Attack_Family",

    "Dominant_Family_Flow_Count",

    "Attack_Family_Count",

    "Multi_Family_Attack_Minute",

    "Dominant_Family_Tie_Count",

    "Dominant_Family_Tie"
]


METADATA_FIELDS = [

    "Capture_ID",

    "Minute",

    "Previous_Observed_Minute",

    "Minutes_Since_Previous_Observed",

    "Is_Contiguous_With_Previous",

    "Starts_New_Observed_Segment",

    "Temporal_Segment_Number",

    "Temporal_Segment_ID"
]


predictor_target_overlap = (
    set(COMMON_PREDICTORS)
    &
    set(TARGET_AND_AUDIT_FIELDS)
)


predictor_metadata_overlap = (
    set(COMMON_PREDICTORS)
    &
    set(METADATA_FIELDS)
)


print(
    "Predictor ↔ target overlap:",
    predictor_target_overlap
)


print(
    "Predictor ↔ metadata overlap:",
    predictor_metadata_overlap
)


if predictor_target_overlap:

    raise ValueError(
        "TARGET LEAKAGE DETECTED."
    )


if predictor_metadata_overlap:

    raise ValueError(
        "Metadata entered predictor set."
    )


print(
    "\n✓ Predictor/target firewall intact"
)


# ==================================================================================================
# 10.15X — ATTACK-MINUTE SEMANTIC AUDIT
# ==================================================================================================

banner("10.15X — ATTACK-MINUTE SEMANTIC AUDIT")


attack_minutes = minute_df[
    minute_df[
        "Attack_State"
    ]
    ==
    1
].copy()


benign_minutes = minute_df[
    minute_df[
        "Attack_State"
    ]
    ==
    0
].copy()


print(
    "Observed minutes:",
    f"{len(minute_df):,}"
)


print(
    "Benign minutes:",
    f"{len(benign_minutes):,}"
)


print(
    "Attack minutes:",
    f"{len(attack_minutes):,}"
)


print(
    "Attack-minute percentage:",
    f"{100 * len(attack_minutes) / len(minute_df):.4f}%"
)


print(
    "\nMulti-family attack minutes:",
    f"{int(attack_minutes['Multi_Family_Attack_Minute'].sum()):,}"
)


print(
    "Dominant-family tie minutes:",
    f"{int(attack_minutes['Dominant_Family_Tie'].sum()):,}"
)


if len(attack_minutes):

    print(
        "\nAttack-fraction distribution:"
    )

    print(
        attack_minutes[
            "Attack_Fraction"
        ].describe(
            percentiles=[
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


# ==================================================================================================
# 10.15Y — PER-CAPTURE ATTACK COVERAGE
# ==================================================================================================

banner("10.15Y — PER-CAPTURE ATTACK COVERAGE")


capture_attack_summary = (

    minute_df

    .groupby(
        "Capture_ID",
        observed=True
    )

    .agg(

        Observed_Minutes=(
            "Minute",
            "size"
        ),

        Total_Flows=(
            "Flow_Count",
            "sum"
        ),

        Attack_Minutes=(
            "Attack_State",
            "sum"
        ),

        Attack_Flows=(
            "Attack_Flow_Count",
            "sum"
        ),

        Mean_Attack_Fraction=(
            "Attack_Fraction",
            "mean"
        ),

        Max_Attack_Fraction=(
            "Attack_Fraction",
            "max"
        )

    )

    .reset_index()
)


capture_attack_summary[
    "Attack_Minute_Pct"
] = (

    capture_attack_summary[
        "Attack_Minutes"
    ]

    /

    capture_attack_summary[
        "Observed_Minutes"
    ]

    *
    100
)


print(
    capture_attack_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.15Z — DOMINANT FAMILY × CAPTURE COVERAGE
# ==================================================================================================

banner("10.15Z — DOMINANT ATTACK FAMILY × CAPTURE COVERAGE")


family_capture = pd.crosstab(

    minute_df[
        "Dominant_Attack_Family"
    ],

    minute_df[
        "Capture_ID"
    ]
)


print(
    family_capture.to_string()
)


# ==================================================================================================
# 10.15AA — FINAL NUMERIC INTEGRITY AUDIT
# ==================================================================================================

banner("10.15AA — FINAL NUMERIC INTEGRITY AUDIT")


numeric_matrix = (

    minute_df[
        COMMON_PREDICTORS
    ]

    .apply(
        pd.to_numeric,
        errors="coerce"
    )

    .to_numpy(
        dtype=float
    )
)


final_missing_predictors = int(
    np.isnan(
        numeric_matrix
    ).sum()
)


final_inf_predictors = int(
    np.isinf(
        numeric_matrix
    ).sum()
)


negative_count_fields = [

    "Flow_Count",

    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",

    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",

    "Packets_Per_Second",
    "Bytes_Per_Second"
]


negative_common_values = int(

    sum(

        (
            minute_df[c]
            <
            0
        ).sum()

        for c in negative_count_fields
    )
)


print(
    "NaN predictor cells:",
    f"{final_missing_predictors:,}"
)


print(
    "Infinite predictor cells:",
    f"{final_inf_predictors:,}"
)


print(
    "Negative count/rate cells:",
    f"{negative_common_values:,}"
)


# ==================================================================================================
# 10.15AB — VALIDATION TABLE
# ==================================================================================================

banner("10.15AB — PHASE 10.15 VALIDATION")


validation = pd.DataFrame(
    [
        {
            "Check":
                "Raw_Flow_Count_Preserved",

            "Passed":
                int(
                    minute_df[
                        "Flow_Count"
                    ].sum()
                )
                ==
                len(timeline_lookup),

            "Details":
                (
                    f"minute_sum={int(minute_df['Flow_Count'].sum()):,}, "
                    f"raw={len(timeline_lookup):,}"
                )
        },

        {
            "Check":
                "Semantic_Flow_Count_Preserved",

            "Passed":
                int(
                    minute_df[
                        "Semantic_Flow_Count"
                    ].sum()
                )
                ==
                len(semantics),

            "Details":
                (
                    f"minute_sum={int(minute_df['Semantic_Flow_Count'].sum()):,}, "
                    f"semantic={len(semantics):,}"
                )
        },

        {
            "Check":
                "Exact_Semantic_Row_Identity",

            "Passed":
                semantics[
                    "Global_Raw_Row_ID_JOIN"
                ].nunique()
                ==
                len(timeline_lookup),

            "Details":
                identity_method
        },

        {
            "Check":
                "Predictor_Target_Minute_Grid_Agreement",

            "Passed":
                flow_count_disagreement
                ==
                0,

            "Details":
                f"{flow_count_disagreement:,} disagreements"
        },

        {
            "Check":
                "Fourteen_Common_Predictors_Present",

            "Passed":
                all(
                    c in minute_df.columns
                    for c in COMMON_PREDICTORS
                ),

            "Details":
                (
                    f"{sum(c in minute_df.columns for c in COMMON_PREDICTORS)}"
                    f"/{len(COMMON_PREDICTORS)}"
                )
        },

        {
            "Check":
                "No_Common_Predictor_NaN",

            "Passed":
                final_missing_predictors
                ==
                0,

            "Details":
                f"{final_missing_predictors:,} NaN cells"
        },

        {
            "Check":
                "No_Common_Predictor_Inf",

            "Passed":
                final_inf_predictors
                ==
                0,

            "Details":
                f"{final_inf_predictors:,} Inf cells"
        },

        {
            "Check":
                "No_Negative_Count_Or_Rate_Features",

            "Passed":
                negative_common_values
                ==
                0,

            "Details":
                f"{negative_common_values:,} negative cells"
        },

        {
            "Check":
                "Predictor_Target_Firewall",

            "Passed":
                len(
                    predictor_target_overlap
                )
                ==
                0,

            "Details":
                str(
                    predictor_target_overlap
                )
        },

        {
            "Check":
                "Predictor_Metadata_Firewall",

            "Passed":
                len(
                    predictor_metadata_overlap
                )
                ==
                0,

            "Details":
                str(
                    predictor_metadata_overlap
                )
        },

        {
            "Check":
                "Two_Captures_Preserved",

            "Passed":
                minute_df[
                    "Capture_ID"
                ].nunique()
                ==
                2,

            "Details":
                f"{minute_df['Capture_ID'].nunique()} captures"
        },

        {
            "Check":
                "Missing_Minutes_Not_Synthetically_Filled",

            "Passed":
                len(
                    minute_df
                )
                ==
                timeline_lookup[
                    [
                        "Capture_ID",
                        "Minute"
                    ]
                ]
                .drop_duplicates()
                .shape[0],

            "Details":
                f"{len(minute_df):,} observed-minute rows"
        },

        {
            "Check":
                "Attack_Flows_Preserved",

            "Passed":
                int(
                    minute_df[
                        "Attack_Flow_Count"
                    ].sum()
                )
                ==
                int(
                    semantics[
                        "Attack_State"
                    ].sum()
                ),

            "Details":
                (
                    f"minute={int(minute_df['Attack_Flow_Count'].sum()):,}, "
                    f"flow={int(semantics['Attack_State'].sum()):,}"
                )
        }
    ]
)


print(
    validation.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.15AC — BUILD CLEAN OUTPUT TABLES
# ==================================================================================================

banner("10.15AC — BUILD CLEAN OUTPUT TABLES")


predictor_table = minute_df[
    [
        "Capture_ID",
        "Temporal_Segment_ID",
        "Minute",
        *COMMON_PREDICTORS
    ]
].copy()


target_table = minute_df[
    [
        "Capture_ID",
        "Temporal_Segment_ID",
        "Minute",

        "Attack_State",

        "Attack_Flow_Count",
        "Benign_Flow_Count",
        "Attack_Fraction",

        "Dominant_Attack_Family",

        "Dominant_Family_Flow_Count",

        "Attack_Family_Count",

        "Multi_Family_Attack_Minute",

        "Dominant_Family_Tie_Count",

        "Dominant_Family_Tie"
    ]
].copy()


full_table_columns = [

    "Capture_ID",

    "Temporal_Segment_ID",

    "Temporal_Segment_Number",

    "Minute",

    "Previous_Observed_Minute",

    "Minutes_Since_Previous_Observed",

    "Is_Contiguous_With_Previous",

    "Starts_New_Observed_Segment",

    *COMMON_PREDICTORS,

    "Attack_State",

    "Attack_Flow_Count",

    "Benign_Flow_Count",

    "Attack_Fraction",

    "Dominant_Attack_Family",

    "Dominant_Family_Flow_Count",

    "Attack_Family_Count",

    "Multi_Family_Attack_Minute",

    "Dominant_Family_Tie_Count",

    "Dominant_Family_Tie"
]


full_table = minute_df[
    full_table_columns
].copy()


print(
    "Predictor table:",
    predictor_table.shape
)


print(
    "Target table:",
    target_table.shape
)


print(
    "Full table:",
    full_table.shape
)


# ==================================================================================================
# 10.15AD — SAVE OUTPUTS
# ==================================================================================================

banner("10.15AD — SAVE OUTPUTS")


PREDICTOR_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_MINUTE_PREDICTORS.csv"
)


TARGET_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_MINUTE_TARGETS.csv"
)


FULL_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_MINUTE_REPRESENTATION.csv"
)


SEGMENT_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_TEMPORAL_SEGMENTS.csv"
)


CAPTURE_MINUTE_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_CAPTURE_MINUTE_COVERAGE.csv"
)


CAPTURE_ATTACK_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_CAPTURE_ATTACK_COVERAGE.csv"
)


FAMILY_CAPTURE_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_FAMILY_CAPTURE_COVERAGE.csv"
)


FLOW_QUALITY_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_FLOW_PREDICTOR_QUALITY.csv"
)


MINUTE_QUALITY_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_MINUTE_PREDICTOR_QUALITY.csv"
)


VALIDATION_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_VALIDATION.csv"
)


predictor_table.to_csv(
    PREDICTOR_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


target_table.to_csv(
    TARGET_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


full_table.to_csv(
    FULL_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


segment_summary.to_csv(
    SEGMENT_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


capture_minute_summary.to_csv(
    CAPTURE_MINUTE_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


capture_attack_summary.to_csv(
    CAPTURE_ATTACK_OUT,
    index=False
)


family_capture.to_csv(
    FAMILY_CAPTURE_OUT
)


flow_quality.to_csv(
    FLOW_QUALITY_OUT,
    index=False
)


minute_predictor_quality.to_csv(
    MINUTE_QUALITY_OUT,
    index=False
)


validation.to_csv(
    VALIDATION_OUT,
    index=False
)


for p in [
    PREDICTOR_OUT,
    TARGET_OUT,
    FULL_OUT,
    SEGMENT_OUT,
    CAPTURE_MINUTE_OUT,
    CAPTURE_ATTACK_OUT,
    FAMILY_CAPTURE_OUT,
    FLOW_QUALITY_OUT,
    MINUTE_QUALITY_OUT,
    VALIDATION_OUT
]:

    print("✓", p)


# ==================================================================================================
# 10.15AE — MANIFEST
# ==================================================================================================

banner("10.15AE — PHASE MANIFEST")


validation_passed = int(
    validation[
        "Passed"
    ].sum()
)


all_validation_pass = bool(
    validation[
        "Passed"
    ].all()
)


manifest = pd.DataFrame(
    {
        "Metric": [

            "Raw_Flows",

            "Observed_Minutes",

            "Physical_Captures",

            "Temporal_Segments",

            "Common_Predictors",

            "Deferred_Protocol_Features",

            "Benign_Minutes",

            "Attack_Minutes",

            "Attack_Flows",

            "Multi_Family_Attack_Minutes",

            "Dominant_Family_Tie_Minutes",

            "Missing_Predictor_Cells",

            "Infinite_Predictor_Cells",

            "Validation_Checks",

            "Validation_Checks_Passed",

            "Phase_10_15_Pass"
        ],

        "Value": [

            len(
                timeline_lookup
            ),

            len(
                minute_df
            ),

            minute_df[
                "Capture_ID"
            ].nunique(),

            minute_df[
                "Temporal_Segment_ID"
            ].nunique(),

            len(
                COMMON_PREDICTORS
            ),

            len(
                PROTOCOL_DEPENDENT_DEFERRED
            ),

            len(
                benign_minutes
            ),

            len(
                attack_minutes
            ),

            int(
                minute_df[
                    "Attack_Flow_Count"
                ].sum()
            ),

            int(
                attack_minutes[
                    "Multi_Family_Attack_Minute"
                ].sum()
            ),

            int(
                attack_minutes[
                    "Dominant_Family_Tie"
                ].sum()
            ),

            final_missing_predictors,

            final_inf_predictors,

            len(
                validation
            ),

            validation_passed,

            all_validation_pass
        ]
    }
)


print(
    manifest.to_string(
        index=False
    )
)


MANIFEST_OUT = (
    UNSW_OUT_DIR /
    "10_15_UNSW_MANIFEST.csv"
)


manifest.to_csv(
    MANIFEST_OUT,
    index=False
)


print(
    "\n✓",
    MANIFEST_OUT
)


# ==================================================================================================
# 10.15AF — FINAL DECISION
# ==================================================================================================

banner("10.15AF — FINAL DECISION")


if all_validation_pass:

    print(
        """
✓ PHASE 10.15 PASS

UNSW now has a leakage-safe 1-minute temporal representation.

The predictor and target branches were constructed independently and
rejoined only at minute level.

All raw flows were preserved.

Final Phase 10.12 semantics were used:

    Final_Attack_State
    Final_Attack_Family

Missing minutes were NOT synthesized.

Capture boundaries were preserved.

Temporal segment boundaries were created so future sequence models
cannot silently cross missing-minute gaps.

No scaling has occurred.

No train/validation/test split has occurred.

No future-label shifting has occurred.

NEXT:
    PHASE 10.16 — UNSW TEMPORAL TARGET FEASIBILITY AUDIT
"""
    )


else:

    print(
        """
⚠ PHASE 10.15 REQUIRES REVIEW.

At least one validation check failed.

DO NOT CONTINUE TO PHASE 10.16.

Send the complete output from:

    10.15P
    10.15Q
    10.15S
    10.15U
    10.15V
    10.15X
    10.15Y
    10.15Z
    10.15AB
    10.15AE
"""
    )


banner("PHASE 10.15 COMPLETE")

In [ ]:
# ==================================================================================================
# PHASE 10.16 — UNSW TEMPORAL TARGET FEASIBILITY & PRE-ATTACK FORECASTING AUDIT
# ==================================================================================================
#
# PURPOSE
# -------
# Determine whether the frozen UNSW 1-minute representation from Phase 10.15
# contains enough legitimate PRE-ATTACK temporal structure to support forecasting.
#
# THIS PHASE DOES NOT:
#
#   - train a model
#   - scale features
#   - split train/validation/test
#   - alter Phase 10.15 labels
#   - synthesize missing minutes
#   - bridge temporal gaps
#   - use future information as predictors
#
# CORE QUESTION
# -------------
#
# Can we construct targets of the form:
#
#     "An attack ONSET will occur within the next H minutes"
#
# while requiring the CURRENT minute to still be benign?
#
# This is deliberately different from:
#
#     Attack_State(t + H)
#
# because that naive target can reward attack persistence:
#
#     attack now -> attack later
#
# which is detection/persistence prediction rather than proactive forecasting.
#
# PRIMARY HORIZONS
# ----------------
#
#     1, 2, 5, 10, 15, 30 minutes
#
# IMPORTANT TEMPORAL RULE
# -----------------------
#
# Forecast targets may only look forward INSIDE THE SAME continuous
# Temporal_Segment_ID produced by Phase 10.15.
#
# Missing-minute gaps and capture boundaries are NEVER crossed.
#
# ==================================================================================================


from pathlib import Path

import numpy as np
import pandas as pd


# ==================================================================================================
# 10.16A — CONFIGURATION
# ==================================================================================================

print("=" * 120)
print("PHASE 10.16 — UNSW TEMPORAL TARGET FEASIBILITY & PRE-ATTACK FORECASTING AUDIT")
print("=" * 120)


BASE_DIR = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2"
)

UNSW_OUT_DIR = (
    BASE_DIR
    / "audit_outputs"
    / "UNSW"
)


INPUT_FILE = (
    UNSW_OUT_DIR
    / "10_15_UNSW_MINUTE_REPRESENTATION.csv"
)


HORIZONS = [
    1,
    2,
    5,
    10,
    15,
    30
]


# Minimum number of benign minutes immediately preceding an onset
# that we will audit.
#
# These are NOT yet model sequence lengths.
# They simply tell us how much clean history exists before each onset.

HISTORY_LENGTHS = [
    1,
    2,
    5,
    10,
    15,
    30,
    60
]


def banner(text):

    print("\n" + "=" * 120)
    print(text)
    print("=" * 120)


# ==================================================================================================
# 10.16B — VERIFY PHASE 10.15 INPUT
# ==================================================================================================

banner("10.16B — VERIFY PHASE 10.15 INPUT")


if not INPUT_FILE.exists():

    raise FileNotFoundError(
        f"Required Phase 10.15 file not found:\n{INPUT_FILE}"
    )


print("✓", INPUT_FILE)


# ==================================================================================================
# 10.16C — LOAD FROZEN MINUTE REPRESENTATION
# ==================================================================================================

banner("10.16C — LOAD FROZEN MINUTE REPRESENTATION")


minute_df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)


required_columns = [

    "Capture_ID",
    "Temporal_Segment_ID",
    "Minute",
    "Attack_State",

    "Flow_Count",
    "Attack_Flow_Count",
    "Attack_Fraction",

    "Dominant_Attack_Family"
]


missing_columns = [

    c for c in required_columns
    if c not in minute_df.columns
]


if missing_columns:

    raise ValueError(
        f"Missing required Phase 10.15 columns: {missing_columns}"
    )


minute_df["Minute"] = pd.to_datetime(
    minute_df["Minute"],
    errors="coerce"
)


if minute_df["Minute"].isna().any():

    raise ValueError(
        "Phase 10.15 contains invalid Minute timestamps."
    )


minute_df["Attack_State"] = pd.to_numeric(
    minute_df["Attack_State"],
    errors="coerce"
)


if minute_df["Attack_State"].isna().any():

    raise ValueError(
        "Attack_State contains non-numeric values."
    )


minute_df["Attack_State"] = (
    minute_df["Attack_State"]
    .astype(np.int8)
)


if not set(
    minute_df["Attack_State"].unique()
).issubset({0, 1}):

    raise ValueError(
        "Attack_State is not binary."
    )


minute_df = minute_df.sort_values(

    [
        "Capture_ID",
        "Temporal_Segment_ID",
        "Minute"
    ],

    kind="mergesort",

    ignore_index=True
)


print(
    "Rows:",
    f"{len(minute_df):,}"
)

print(
    "Captures:",
    minute_df["Capture_ID"].nunique()
)

print(
    "Temporal segments:",
    minute_df["Temporal_Segment_ID"].nunique()
)

print(
    "\nAttack-state distribution:"
)

print(
    minute_df["Attack_State"].value_counts(
        dropna=False
    )
)


# ==================================================================================================
# 10.16D — VERIFY CONTINUITY INSIDE TEMPORAL SEGMENTS
# ==================================================================================================

banner("10.16D — VERIFY CONTINUITY INSIDE TEMPORAL SEGMENTS")


continuity_rows = []


for segment_id, g in minute_df.groupby(
    "Temporal_Segment_ID",
    observed=True,
    sort=False
):

    g = g.sort_values("Minute")

    diffs = (
        g["Minute"]
        .diff()
        .dt.total_seconds()
        .div(60)
    )


    non_contiguous = int(
        (
            diffs.dropna()
            != 1
        ).sum()
    )


    continuity_rows.append(
        {
            "Temporal_Segment_ID":
                segment_id,

            "Capture_ID":
                g["Capture_ID"].iloc[0],

            "Start_Minute":
                g["Minute"].min(),

            "End_Minute":
                g["Minute"].max(),

            "Observed_Minutes":
                len(g),

            "Internal_NonContiguous_Steps":
                non_contiguous
        }
    )


continuity_audit = pd.DataFrame(
    continuity_rows
)


print(
    continuity_audit.to_string(
        index=False
    )
)


if (
    continuity_audit[
        "Internal_NonContiguous_Steps"
    ]
    >
    0
).any():

    raise ValueError(
        "A Phase 10.15 temporal segment contains an internal gap."
    )


print(
    "\n✓ Every Temporal_Segment_ID is internally contiguous"
)


# ==================================================================================================
# 10.16E — IDENTIFY TRUE BENIGN → ATTACK ONSETS
# ==================================================================================================

banner("10.16E — IDENTIFY TRUE BENIGN → ATTACK ONSETS")


# --------------------------------------------------------------------------------------------------
# An onset is counted only when:
#
#     previous observed minute = BENIGN
#     current minute           = ATTACK
#
# AND both minutes belong to the SAME continuous temporal segment.
#
# We deliberately do NOT call an attack in the first minute of a segment
# a confirmed onset because its immediately preceding minute is unavailable.
# --------------------------------------------------------------------------------------------------


minute_df[
    "Previous_Attack_State"
] = (

    minute_df

    .groupby(
        "Temporal_Segment_ID",
        observed=True
    )[
        "Attack_State"
    ]

    .shift(1)
)


minute_df[
    "Confirmed_Attack_Onset"
] = (

    (
        minute_df[
            "Attack_State"
        ]
        ==
        1
    )

    &

    (
        minute_df[
            "Previous_Attack_State"
        ]
        ==
        0
    )
)


minute_df[
    "Segment_Start_Attack"
] = (

    (
        minute_df[
            "Attack_State"
        ]
        ==
        1
    )

    &

    (
        minute_df[
            "Previous_Attack_State"
        ]
        .isna()
    )
)


onsets = minute_df.loc[
    minute_df[
        "Confirmed_Attack_Onset"
    ]
].copy()


segment_start_attacks = minute_df.loc[
    minute_df[
        "Segment_Start_Attack"
    ]
].copy()


print(
    "Confirmed benign → attack onsets:",
    f"{len(onsets):,}"
)


print(
    "Attack-active segment starts:",
    f"{len(segment_start_attacks):,}"
)


if len(onsets):

    print(
        "\nConfirmed onsets by capture:"
    )

    print(
        onsets[
            "Capture_ID"
        ].value_counts()
    )


    print(
        "\nConfirmed onset timestamps:"
    )

    print(
        onsets[
            [
                "Capture_ID",
                "Temporal_Segment_ID",
                "Minute",
                "Attack_Fraction",
                "Dominant_Attack_Family"
            ]
        ]
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 10.16F — RECONSTRUCT ATTACK RUNS
# ==================================================================================================

banner("10.16F — RECONSTRUCT CONTIGUOUS ATTACK RUNS")


# --------------------------------------------------------------------------------------------------
# Every transition in Attack_State creates a new state run.
#
# This tells us whether the dataset contains:
#
#     benign -> attack -> benign -> attack
#
# repeatedly, or whether it consists mostly of one huge attack-active region.
# --------------------------------------------------------------------------------------------------


minute_df[
    "State_Changed"
] = (

    minute_df[
        "Attack_State"
    ]

    !=

    minute_df[
        "Previous_Attack_State"
    ]
)


# First row in every segment must begin a new run.

minute_df.loc[
    minute_df[
        "Previous_Attack_State"
    ].isna(),
    "State_Changed"
] = True


minute_df[
    "State_Run_Number"
] = (

    minute_df

    .groupby(
        "Temporal_Segment_ID",
        observed=True
    )[
        "State_Changed"
    ]

    .cumsum()

    .astype(int)
)


minute_df[
    "State_Run_ID"
] = (

    minute_df[
        "Temporal_Segment_ID"
    ].astype(str)

    +

    "_RUN_"

    +

    minute_df[
        "State_Run_Number"
    ]
    .astype(str)
    .str.zfill(4)
)


state_runs = (

    minute_df

    .groupby(
        [
            "Capture_ID",
            "Temporal_Segment_ID",
            "State_Run_ID"
        ],
        observed=True,
        sort=False
    )

    .agg(

        Attack_State=(
            "Attack_State",
            "first"
        ),

        Run_Start=(
            "Minute",
            "min"
        ),

        Run_End=(
            "Minute",
            "max"
        ),

        Run_Length_Minutes=(
            "Minute",
            "size"
        ),

        Total_Flows=(
            "Flow_Count",
            "sum"
        ),

        Attack_Flows=(
            "Attack_Flow_Count",
            "sum"
        )

    )

    .reset_index()
)


print(
    "Total state runs:",
    len(state_runs)
)


print(
    "\nRun counts by state:"
)

print(
    state_runs[
        "Attack_State"
    ].value_counts()
)


attack_runs = state_runs[
    state_runs[
        "Attack_State"
    ]
    ==
    1
].copy()


benign_runs = state_runs[
    state_runs[
        "Attack_State"
    ]
    ==
    0
].copy()


print(
    "\nAttack-run duration:"
)

if len(attack_runs):

    print(
        attack_runs[
            "Run_Length_Minutes"
        ].describe(
            percentiles=[
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


print(
    "\nBenign-run duration:"
)

if len(benign_runs):

    print(
        benign_runs[
            "Run_Length_Minutes"
        ].describe(
            percentiles=[
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


# ==================================================================================================
# 10.16G — CLEAN BENIGN HISTORY BEFORE EACH CONFIRMED ONSET
# ==================================================================================================

banner("10.16G — CLEAN BENIGN HISTORY BEFORE EACH CONFIRMED ONSET")


# --------------------------------------------------------------------------------------------------
# For each confirmed onset, count how many immediately preceding minutes
# are continuously BENIGN.
#
# We stop when:
#
#   - an attack minute is encountered
#   - the segment boundary is reached
#
# Since each Phase 10.15 segment is contiguous, this produces a strict
# clean-pre-attack-history measurement.
# --------------------------------------------------------------------------------------------------


onset_history_rows = []


for segment_id, g in minute_df.groupby(
    "Temporal_Segment_ID",
    observed=True,
    sort=False
):

    g = (
        g
        .sort_values("Minute")
        .reset_index(drop=True)
    )


    states = g[
        "Attack_State"
    ].to_numpy()


    onset_positions = np.flatnonzero(
        g[
            "Confirmed_Attack_Onset"
        ].to_numpy()
    )


    for pos in onset_positions:

        clean_history = 0

        j = pos - 1


        while (
            j >= 0
            and
            states[j] == 0
        ):

            clean_history += 1

            j -= 1


        onset_row = g.iloc[pos]


        onset_history_rows.append(
            {
                "Capture_ID":
                    onset_row["Capture_ID"],

                "Temporal_Segment_ID":
                    segment_id,

                "Onset_Minute":
                    onset_row["Minute"],

                "Onset_Attack_Fraction":
                    onset_row["Attack_Fraction"],

                "Onset_Dominant_Family":
                    onset_row["Dominant_Attack_Family"],

                "Clean_Benign_History_Minutes":
                    clean_history
            }
        )


onset_history = pd.DataFrame(
    onset_history_rows
)


print(
    "Confirmed onsets audited:",
    len(onset_history)
)


if len(onset_history):

    print(
        "\nClean benign history distribution:"
    )

    print(
        onset_history[
            "Clean_Benign_History_Minutes"
        ].describe(
            percentiles=[
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


    print(
        "\nPer-onset clean history:"
    )

    print(
        onset_history.to_string(
            index=False
        )
    )


# ==================================================================================================
# 10.16H — HISTORY-LENGTH FEASIBILITY
# ==================================================================================================

banner("10.16H — PRE-ATTACK HISTORY-LENGTH FEASIBILITY")


history_feasibility_rows = []


for history_length in HISTORY_LENGTHS:

    if len(onset_history):

        eligible = (

            onset_history[
                "Clean_Benign_History_Minutes"
            ]

            >=

            history_length
        )


        eligible_count = int(
            eligible.sum()
        )


        capture_count = int(
            onset_history.loc[
                eligible,
                "Capture_ID"
            ]
            .nunique()
        )


        segment_count = int(
            onset_history.loc[
                eligible,
                "Temporal_Segment_ID"
            ]
            .nunique()
        )


    else:

        eligible_count = 0
        capture_count = 0
        segment_count = 0


    history_feasibility_rows.append(
        {
            "Required_Clean_History_Min":
                history_length,

            "Eligible_Onsets":
                eligible_count,

            "Pct_Of_Confirmed_Onsets":
                (
                    100
                    *
                    eligible_count
                    /
                    len(onset_history)
                )
                if len(onset_history)
                else np.nan,

            "Captures_Represented":
                capture_count,

            "Segments_Represented":
                segment_count
        }
    )


history_feasibility = pd.DataFrame(
    history_feasibility_rows
)


print(
    history_feasibility.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16I — BUILD STRICT HORIZON-SPECIFIC FORECAST TARGETS
# ==================================================================================================

banner("10.16I — STRICT HORIZON-SPECIFIC PRE-ATTACK TARGETS")


# --------------------------------------------------------------------------------------------------
# DEFINITION
# ----------
#
# For each CURRENT minute t:
#
#     Forecast_Target_H = 1
#
# iff:
#
#     1. Attack_State(t) == 0
#
#     2. a CONFIRMED benign -> attack onset occurs at some minute
#        t+k where:
#
#               1 <= k <= H
#
#     3. t and the onset belong to the SAME Temporal_Segment_ID
#
#
# Otherwise, among CURRENTLY BENIGN minutes:
#
#     Forecast_Target_H = 0
#
#
# ATTACK-ACTIVE CURRENT MINUTES:
#
#     Forecast_Target_H = NaN
#
# because they are not legitimate pre-attack forecasting samples.
#
#
# This avoids the major persistence leakage problem:
#
#     attack now -> attack later
#
# --------------------------------------------------------------------------------------------------


for H in HORIZONS:

    target_col = (
        f"Forecast_Onset_Within_{H}m"
    )


    minutes_to_onset_col = (
        f"Minutes_To_Next_Onset_For_{H}m"
    )


    minute_df[
        target_col
    ] = np.nan


    minute_df[
        minutes_to_onset_col
    ] = np.nan


    for segment_id, g in minute_df.groupby(
        "Temporal_Segment_ID",
        observed=True,
        sort=False
    ):

        idx = g.index.to_numpy()

        states = (
            g[
                "Attack_State"
            ]
            .to_numpy()
        )


        onset_mask = (
            g[
                "Confirmed_Attack_Onset"
            ]
            .to_numpy(
                dtype=bool
            )
        )


        n = len(g)


        # ------------------------------------------------------------------------------------------
        # Distance to NEXT confirmed onset.
        #
        # Vectorized reverse scan.
        # ------------------------------------------------------------------------------------------

        distance_to_next_onset = np.full(
            n,
            np.nan,
            dtype=float
        )


        next_onset_pos = None


        for pos in range(
            n - 1,
            -1,
            -1
        ):

            # Current onset should NOT count as a future onset
            # for itself.
            #
            # Therefore calculate distance first using the
            # previously stored future onset.

            if next_onset_pos is not None:

                distance_to_next_onset[pos] = (
                    next_onset_pos
                    -
                    pos
                )


            if onset_mask[pos]:

                next_onset_pos = pos


        benign_mask = (
            states
            ==
            0
        )


        positive_mask = (

            benign_mask

            &

            np.isfinite(
                distance_to_next_onset
            )

            &

            (
                distance_to_next_onset
                >=
                1
            )

            &

            (
                distance_to_next_onset
                <=
                H
            )
        )


        negative_mask = (

            benign_mask

            &

            ~positive_mask
        )


        # Only benign current minutes receive forecasting labels.

        target_values = np.full(
            n,
            np.nan,
            dtype=float
        )


        target_values[
            positive_mask
        ] = 1


        target_values[
            negative_mask
        ] = 0


        minute_df.loc[
            idx,
            target_col
        ] = target_values


        minute_df.loc[
            idx,
            minutes_to_onset_col
        ] = distance_to_next_onset


# ==================================================================================================
# 10.16J — HORIZON FEASIBILITY SUMMARY
# ==================================================================================================

banner("10.16J — HORIZON FEASIBILITY SUMMARY")


horizon_rows = []


for H in HORIZONS:

    target_col = (
        f"Forecast_Onset_Within_{H}m"
    )


    eligible = minute_df[
        target_col
    ].notna()


    positive = (
        minute_df[
            target_col
        ]
        ==
        1
    )


    negative = (
        minute_df[
            target_col
        ]
        ==
        0
    )


    positive_count = int(
        positive.sum()
    )


    negative_count = int(
        negative.sum()
    )


    eligible_count = int(
        eligible.sum()
    )


    positive_onset_count = 0


    if positive_count:

        # ------------------------------------------------------------------------------------------
        # Count unique future onsets represented by the positive windows.
        # ------------------------------------------------------------------------------------------

        represented_onsets = set()


        positive_rows = minute_df.loc[
            positive,
            [
                "Temporal_Segment_ID",
                "Minute",
                f"Minutes_To_Next_Onset_For_{H}m"
            ]
        ]


        for row in positive_rows.itertuples(
            index=False
        ):

            onset_time = (

                row.Minute

                +

                pd.Timedelta(
                    minutes=float(
                        getattr(
                            row,
                            f"Minutes_To_Next_Onset_For_{H}m"
                        )
                    )
                )
            )


            represented_onsets.add(
                (
                    row.Temporal_Segment_ID,
                    onset_time
                )
            )


        positive_onset_count = len(
            represented_onsets
        )


    horizon_rows.append(
        {
            "Horizon_Minutes":
                H,

            "Eligible_Currently_Benign_Minutes":
                eligible_count,

            "Positive_PreAttack_Minutes":
                positive_count,

            "Negative_Benign_Minutes":
                negative_count,

            "Positive_Pct":
                (
                    100
                    *
                    positive_count
                    /
                    eligible_count
                )
                if eligible_count
                else np.nan,

            "Unique_Onsets_Represented":
                positive_onset_count,

            "Positive_Captures":
                int(
                    minute_df.loc[
                        positive,
                        "Capture_ID"
                    ]
                    .nunique()
                ),

            "Positive_Segments":
                int(
                    minute_df.loc[
                        positive,
                        "Temporal_Segment_ID"
                    ]
                    .nunique()
                )
        }
    )


horizon_summary = pd.DataFrame(
    horizon_rows
)


print(
    horizon_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16K — PER-CAPTURE HORIZON SUPPORT
# ==================================================================================================

banner("10.16K — PER-CAPTURE HORIZON SUPPORT")


capture_horizon_rows = []


for capture_id, capture_group in minute_df.groupby(
    "Capture_ID",
    observed=True
):

    for H in HORIZONS:

        target_col = (
            f"Forecast_Onset_Within_{H}m"
        )


        eligible = (
            capture_group[
                target_col
            ]
            .notna()
        )


        positive = (
            capture_group[
                target_col
            ]
            ==
            1
        )


        negative = (
            capture_group[
                target_col
            ]
            ==
            0
        )


        capture_horizon_rows.append(
            {
                "Capture_ID":
                    capture_id,

                "Horizon_Minutes":
                    H,

                "Eligible_Benign_Minutes":
                    int(
                        eligible.sum()
                    ),

                "Positive_PreAttack_Minutes":
                    int(
                        positive.sum()
                    ),

                "Negative_Benign_Minutes":
                    int(
                        negative.sum()
                    ),

                "Positive_Pct":
                    (
                        100
                        *
                        positive.sum()
                        /
                        eligible.sum()
                    )
                    if eligible.sum()
                    else np.nan
            }
        )


capture_horizon_summary = pd.DataFrame(
    capture_horizon_rows
)


print(
    capture_horizon_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16L — NAIVE FUTURE ATTACK-STATE PERSISTENCE AUDIT
# ==================================================================================================

banner("10.16L — NAIVE FUTURE ATTACK-STATE PERSISTENCE AUDIT")


# --------------------------------------------------------------------------------------------------
# This is an IMPORTANT diagnostic.
#
# It measures what would happen if we naively defined:
#
#     target = Attack_State(t + H)
#
# without requiring the current minute to be benign.
#
# We then calculate how many positive future labels originate from a minute
# that is ALREADY under attack.
#
# High values mean naive future-state prediction would largely measure
# attack persistence rather than forecasting.
# --------------------------------------------------------------------------------------------------


persistence_rows = []


for H in HORIZONS:

    future_state = (

        minute_df

        .groupby(
            "Temporal_Segment_ID",
            observed=True
        )[
            "Attack_State"
        ]

        .shift(
            -H
        )
    )


    valid = future_state.notna()


    future_positive = (
        future_state
        ==
        1
    )


    already_attack = (

        future_positive

        &

        (
            minute_df[
                "Attack_State"
            ]
            ==
            1
        )
    )


    benign_to_future_attack = (

        future_positive

        &

        (
            minute_df[
                "Attack_State"
            ]
            ==
            0
        )
    )


    total_future_positive = int(
        future_positive.sum()
    )


    already_attack_count = int(
        already_attack.sum()
    )


    benign_to_attack_count = int(
        benign_to_future_attack.sum()
    )


    persistence_rows.append(
        {
            "Horizon_Minutes":
                H,

            "Valid_Shifted_Rows":
                int(
                    valid.sum()
                ),

            "Future_Attack_Positive_Rows":
                total_future_positive,

            "Already_Attacking_At_Current_Minute":
                already_attack_count,

            "Currently_Benign_Future_Attack":
                benign_to_attack_count,

            "Persistence_Contamination_Pct":
                (
                    100
                    *
                    already_attack_count
                    /
                    total_future_positive
                )
                if total_future_positive
                else np.nan
        }
    )


persistence_audit = pd.DataFrame(
    persistence_rows
)


print(
    persistence_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16M — PRE-ATTACK WINDOW CONTAMINATION AUDIT
# ==================================================================================================

banner("10.16M — PRE-ATTACK WINDOW CONTAMINATION AUDIT")


# --------------------------------------------------------------------------------------------------
# For each confirmed onset and each horizon H:
#
# Inspect the H minutes immediately preceding onset.
#
# We measure:
#
#   available minutes
#   benign minutes
#   attack-active minutes
#
# A proper pre-attack window should ideally be entirely benign.
# --------------------------------------------------------------------------------------------------


pre_attack_window_rows = []


for segment_id, g in minute_df.groupby(
    "Temporal_Segment_ID",
    observed=True,
    sort=False
):

    g = (
        g
        .sort_values("Minute")
        .reset_index(drop=True)
    )


    onset_positions = np.flatnonzero(
        g[
            "Confirmed_Attack_Onset"
        ].to_numpy()
    )


    states = (
        g[
            "Attack_State"
        ]
        .to_numpy()
    )


    for onset_pos in onset_positions:

        onset_row = g.iloc[
            onset_pos
        ]


        for H in HORIZONS:

            start_pos = max(
                0,
                onset_pos - H
            )


            window_states = states[
                start_pos:onset_pos
            ]


            available = len(
                window_states
            )


            benign_count = int(
                (
                    window_states
                    ==
                    0
                ).sum()
            )


            attack_count = int(
                (
                    window_states
                    ==
                    1
                ).sum()
            )


            pre_attack_window_rows.append(
                {
                    "Capture_ID":
                        onset_row[
                            "Capture_ID"
                        ],

                    "Temporal_Segment_ID":
                        segment_id,

                    "Onset_Minute":
                        onset_row[
                            "Minute"
                        ],

                    "Horizon_Minutes":
                        H,

                    "Available_PreOnset_Minutes":
                        available,

                    "Benign_PreOnset_Minutes":
                        benign_count,

                    "Attack_PreOnset_Minutes":
                        attack_count,

                    "Full_Horizon_Available":
                        available
                        ==
                        H,

                    "Completely_Benign_Window":
                        (
                            available
                            ==
                            H
                            and
                            attack_count
                            ==
                            0
                        )
                }
            )


pre_attack_windows = pd.DataFrame(
    pre_attack_window_rows
)


if len(pre_attack_windows):

    contamination_summary = (

        pre_attack_windows

        .groupby(
            "Horizon_Minutes",
            observed=True
        )

        .agg(

            Confirmed_Onsets=(
                "Onset_Minute",
                "size"
            ),

            Full_Horizon_Available=(
                "Full_Horizon_Available",
                "sum"
            ),

            Completely_Benign_Windows=(
                "Completely_Benign_Window",
                "sum"
            ),

            Mean_Available_Minutes=(
                "Available_PreOnset_Minutes",
                "mean"
            ),

            Mean_Attack_Minutes_In_PreWindow=(
                "Attack_PreOnset_Minutes",
                "mean"
            )

        )

        .reset_index()
    )


    contamination_summary[
        "Clean_Window_Pct"
    ] = (

        100

        *

        contamination_summary[
            "Completely_Benign_Windows"
        ]

        /

        contamination_summary[
            "Confirmed_Onsets"
        ]
    )


else:

    contamination_summary = pd.DataFrame()


print(
    contamination_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16N — ONSET FAMILY SUPPORT
# ==================================================================================================

banner("10.16N — ATTACK FAMILY SUPPORT AT CONFIRMED ONSET")


if len(onsets):

    onset_family_support = (

        onsets

        .groupby(
            "Dominant_Attack_Family",
            observed=True
        )

        .agg(

            Confirmed_Onsets=(
                "Minute",
                "size"
            ),

            Captures=(
                "Capture_ID",
                "nunique"
            ),

            Segments=(
                "Temporal_Segment_ID",
                "nunique"
            ),

            Median_Onset_Attack_Fraction=(
                "Attack_Fraction",
                "median"
            )

        )

        .sort_values(
            "Confirmed_Onsets",
            ascending=False
        )

        .reset_index()
    )


else:

    onset_family_support = pd.DataFrame(
        columns=[
            "Dominant_Attack_Family",
            "Confirmed_Onsets",
            "Captures",
            "Segments",
            "Median_Onset_Attack_Fraction"
        ]
    )


print(
    onset_family_support.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16O — ATTACK-FAMILY MINUTE TARGET WARNING AUDIT
# ==================================================================================================

banner("10.16O — ATTACK-FAMILY TARGET AMBIGUITY AUDIT")


attack_minutes = minute_df[
    minute_df[
        "Attack_State"
    ]
    ==
    1
].copy()


if (
    "Multi_Family_Attack_Minute"
    in
    minute_df.columns
):

    multi_family_count = int(

        attack_minutes[
            "Multi_Family_Attack_Minute"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )

else:

    multi_family_count = np.nan


if (
    "Dominant_Family_Tie"
    in
    minute_df.columns
):

    tie_count = int(

        attack_minutes[
            "Dominant_Family_Tie"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )

else:

    tie_count = np.nan


family_target_audit = pd.DataFrame(
    [
        {
            "Metric":
                "Attack_Minutes",

            "Value":
                len(
                    attack_minutes
                )
        },

        {
            "Metric":
                "Multi_Family_Attack_Minutes",

            "Value":
                multi_family_count
        },

        {
            "Metric":
                "Dominant_Family_Tie_Minutes",

            "Value":
                tie_count
        },

        {
            "Metric":
                "Multi_Family_Attack_Minute_Pct",

            "Value":
                (
                    100
                    *
                    multi_family_count
                    /
                    len(attack_minutes)
                )
                if (
                    len(attack_minutes)
                    and
                    not pd.isna(
                        multi_family_count
                    )
                )
                else np.nan
        }
    ]
)


print(
    family_target_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16P — FORECASTING SUPPORT CLASSIFICATION
# ==================================================================================================

banner("10.16P — FORECASTING SUPPORT CLASSIFICATION")


# --------------------------------------------------------------------------------------------------
# IMPORTANT:
#
# This is NOT a model-performance judgment.
#
# It is a structural dataset audit.
#
# We classify support based on the number and diversity of independently
# observable benign -> attack onset events.
# --------------------------------------------------------------------------------------------------


confirmed_onsets = len(
    onsets
)


onset_captures = int(
    onsets[
        "Capture_ID"
    ].nunique()
) if confirmed_onsets else 0


onset_segments = int(
    onsets[
        "Temporal_Segment_ID"
    ].nunique()
) if confirmed_onsets else 0


if confirmed_onsets == 0:

    support_classification = (
        "NO_CONFIRMED_ONSET_SUPPORT"
    )


elif confirmed_onsets < 5:

    support_classification = (
        "VERY_LIMITED_ONSET_SUPPORT"
    )


elif confirmed_onsets < 20:

    support_classification = (
        "LIMITED_ONSET_SUPPORT"
    )


else:

    support_classification = (
        "SUBSTANTIAL_ONSET_COUNT_PRESENT"
    )


print(
    "Confirmed onsets:",
    confirmed_onsets
)

print(
    "Captures containing confirmed onsets:",
    onset_captures
)

print(
    "Segments containing confirmed onsets:",
    onset_segments
)

print(
    "Structural support classification:",
    support_classification
)


# ==================================================================================================
# 10.16Q — AUTOMATIC RED-FLAG AUDIT
# ==================================================================================================

banner("10.16Q — AUTOMATIC RED-FLAG AUDIT")


red_flag_rows = []


# --------------------------------------------------------------------------------------------------
# Flag 1 — too few onsets
# --------------------------------------------------------------------------------------------------

red_flag_rows.append(
    {
        "Audit":
            "Confirmed_Onset_Count",

        "Value":
            confirmed_onsets,

        "Flag":
            confirmed_onsets
            <
            20,

        "Interpretation":
            (
                "Fewer than 20 confirmed benign→attack transitions "
                "limits independent forecasting-event support."
            )
    }
)


# --------------------------------------------------------------------------------------------------
# Flag 2 — onset capture diversity
# --------------------------------------------------------------------------------------------------

red_flag_rows.append(
    {
        "Audit":
            "Onset_Capture_Diversity",

        "Value":
            onset_captures,

        "Flag":
            onset_captures
            <
            2,

        "Interpretation":
            (
                "Confirmed onsets occur in fewer than two physical captures."
            )
    }
)


# --------------------------------------------------------------------------------------------------
# Flag 3 — attack-active dataset dominance
# --------------------------------------------------------------------------------------------------

attack_minute_pct = (

    100

    *

    (
        minute_df[
            "Attack_State"
        ]
        ==
        1
    ).sum()

    /

    len(
        minute_df
    )
)


red_flag_rows.append(
    {
        "Audit":
            "Attack_Minute_Dominance",

        "Value":
            attack_minute_pct,

        "Flag":
            attack_minute_pct
            >
            50,

        "Interpretation":
            (
                "More than half of observed minutes are already attack-active; "
                "naive future-state prediction may reward persistence."
            )
    }
)


# --------------------------------------------------------------------------------------------------
# Flag 4 — multi-family minute ambiguity
# --------------------------------------------------------------------------------------------------

if not pd.isna(
    multi_family_count
):

    multi_family_pct = (

        100

        *

        multi_family_count

        /

        len(
            attack_minutes
        )
    ) if len(
        attack_minutes
    ) else 0


    red_flag_rows.append(
        {
            "Audit":
                "Multi_Family_Attack_Minutes",

            "Value":
                multi_family_pct,

            "Flag":
                multi_family_pct
                >
                50,

            "Interpretation":
                (
                    "Most attack minutes contain multiple attack families; "
                    "single dominant-family minute labels may discard semantics."
                )
        }
    )


# --------------------------------------------------------------------------------------------------
# Flag 5+ — persistence contamination by horizon
# --------------------------------------------------------------------------------------------------

for row in persistence_audit.itertuples(
    index=False
):

    red_flag_rows.append(
        {
            "Audit":
                (
                    f"Persistence_Contamination_"
                    f"{row.Horizon_Minutes}m"
                ),

            "Value":
                row.Persistence_Contamination_Pct,

            "Flag":
                (
                    row.Persistence_Contamination_Pct
                    >
                    50
                )
                if not pd.isna(
                    row.Persistence_Contamination_Pct
                )
                else False,

            "Interpretation":
                (
                    "Share of naive future-attack positives whose "
                    "current minute is already under attack."
                )
        }
    )


red_flags = pd.DataFrame(
    red_flag_rows
)


print(
    red_flags.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16R — VALIDATION
# ==================================================================================================

banner("10.16R — PHASE 10.16 VALIDATION")


validation_rows = []


validation_rows.append(
    {
        "Check":
            "Input_Minutes_Preserved",

        "Passed":
            len(
                minute_df
            )
            ==
            1441,

        "Details":
            f"{len(minute_df):,} rows"
    }
)


validation_rows.append(
    {
        "Check":
            "Temporal_Segments_Internally_Contiguous",

        "Passed":
            (
                continuity_audit[
                    "Internal_NonContiguous_Steps"
                ]
                ==
                0
            ).all(),

        "Details":
            (
                f"{len(continuity_audit)} segments"
            )
    }
)


validation_rows.append(
    {
        "Check":
            "Binary_Attack_State_Valid",

        "Passed":
            set(
                minute_df[
                    "Attack_State"
                ].unique()
            ).issubset(
                {0, 1}
            ),

        "Details":
            str(
                sorted(
                    minute_df[
                        "Attack_State"
                    ].unique()
                )
            )
    }
)


validation_rows.append(
    {
        "Check":
            "Forecast_Targets_Only_On_Currently_Benign_Minutes",

        "Passed":
            all(

                not (

                    (
                        minute_df[
                            f"Forecast_Onset_Within_{H}m"
                        ]
                        .notna()
                    )

                    &

                    (
                        minute_df[
                            "Attack_State"
                        ]
                        ==
                        1
                    )

                ).any()

                for H in HORIZONS
            ),

        "Details":
            "Attack-active current minutes remain unlabeled for strict forecasting"
    }
)


validation_rows.append(
    {
        "Check":
            "No_Cross_Segment_Forecast_Shifting",

        "Passed":
            True,

        "Details":
            (
                "All target construction performed independently "
                "inside Temporal_Segment_ID"
            )
    }
)


validation_rows.append(
    {
        "Check":
            "No_Training_Performed",

        "Passed":
            True,

        "Details":
            "Audit only"
    }
)


validation_rows.append(
    {
        "Check":
            "No_Scaling_Performed",

        "Passed":
            True,

        "Details":
            "Raw Phase 10.15 predictor scale retained"
    }
)


validation_rows.append(
    {
        "Check":
            "No_Data_Split_Performed",

        "Passed":
            True,

        "Details":
            "Train/validation/test remain undefined"
    }
)


validation = pd.DataFrame(
    validation_rows
)


print(
    validation.to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16S — SAVE AUDIT ARTIFACTS
# ==================================================================================================

banner("10.16S — SAVE AUDIT ARTIFACTS")


ONSET_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_CONFIRMED_ATTACK_ONSETS.csv"
)


ONSET_HISTORY_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_ONSET_HISTORY.csv"
)


HISTORY_FEASIBILITY_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_HISTORY_FEASIBILITY.csv"
)


HORIZON_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_HORIZON_FEASIBILITY.csv"
)


CAPTURE_HORIZON_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_CAPTURE_HORIZON_FEASIBILITY.csv"
)


PERSISTENCE_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_PERSISTENCE_AUDIT.csv"
)


PREATTACK_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_PREATTACK_WINDOW_AUDIT.csv"
)


CONTAMINATION_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_PREATTACK_CONTAMINATION_SUMMARY.csv"
)


STATE_RUN_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_STATE_RUNS.csv"
)


ONSET_FAMILY_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_ONSET_FAMILY_SUPPORT.csv"
)


RED_FLAG_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_FORECASTING_RED_FLAGS.csv"
)


VALIDATION_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_VALIDATION.csv"
)


TARGET_AUDIT_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_MINUTE_FORECAST_TARGET_AUDIT.csv"
)


onsets.to_csv(
    ONSET_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


onset_history.to_csv(
    ONSET_HISTORY_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


history_feasibility.to_csv(
    HISTORY_FEASIBILITY_OUT,
    index=False
)


horizon_summary.to_csv(
    HORIZON_OUT,
    index=False
)


capture_horizon_summary.to_csv(
    CAPTURE_HORIZON_OUT,
    index=False
)


persistence_audit.to_csv(
    PERSISTENCE_OUT,
    index=False
)


pre_attack_windows.to_csv(
    PREATTACK_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


contamination_summary.to_csv(
    CONTAMINATION_OUT,
    index=False
)


state_runs.to_csv(
    STATE_RUN_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


onset_family_support.to_csv(
    ONSET_FAMILY_OUT,
    index=False
)


red_flags.to_csv(
    RED_FLAG_OUT,
    index=False
)


validation.to_csv(
    VALIDATION_OUT,
    index=False
)


# --------------------------------------------------------------------------------------------------
# Save the minute table containing AUDIT targets.
#
# IMPORTANT:
# This is NOT yet the final modeling dataset.
# --------------------------------------------------------------------------------------------------

audit_target_columns = [

    "Capture_ID",
    "Temporal_Segment_ID",
    "Minute",

    "Attack_State",
    "Confirmed_Attack_Onset",

    "Attack_Fraction",
    "Dominant_Attack_Family"
]


for H in HORIZONS:

    audit_target_columns.extend(
        [
            f"Forecast_Onset_Within_{H}m",
            f"Minutes_To_Next_Onset_For_{H}m"
        ]
    )


minute_df[
    audit_target_columns
].to_csv(
    TARGET_AUDIT_OUT,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)


for p in [

    ONSET_OUT,
    ONSET_HISTORY_OUT,
    HISTORY_FEASIBILITY_OUT,
    HORIZON_OUT,
    CAPTURE_HORIZON_OUT,
    PERSISTENCE_OUT,
    PREATTACK_OUT,
    CONTAMINATION_OUT,
    STATE_RUN_OUT,
    ONSET_FAMILY_OUT,
    RED_FLAG_OUT,
    VALIDATION_OUT,
    TARGET_AUDIT_OUT

]:

    print(
        "✓",
        p
    )


# ==================================================================================================
# 10.16T — MANIFEST
# ==================================================================================================

banner("10.16T — PHASE MANIFEST")


all_validation_pass = bool(
    validation[
        "Passed"
    ].all()
)


max_clean_history = (

    int(
        onset_history[
            "Clean_Benign_History_Minutes"
        ].max()
    )

    if len(
        onset_history
    )

    else 0
)


median_clean_history = (

    float(
        onset_history[
            "Clean_Benign_History_Minutes"
        ].median()
    )

    if len(
        onset_history
    )

    else np.nan
)


manifest = pd.DataFrame(
    {
        "Metric": [

            "Observed_Minutes",

            "Physical_Captures",

            "Temporal_Segments",

            "Confirmed_Attack_Onsets",

            "Attack_Active_Segment_Starts",

            "Onset_Captures",

            "Onset_Segments",

            "Attack_Runs",

            "Benign_Runs",

            "Median_Clean_Benign_History_Min",

            "Max_Clean_Benign_History_Min",

            "Structural_Forecasting_Support",

            "Validation_Checks",

            "Validation_Checks_Passed",

            "Phase_10_16_Pass"
        ],

        "Value": [

            len(
                minute_df
            ),

            minute_df[
                "Capture_ID"
            ].nunique(),

            minute_df[
                "Temporal_Segment_ID"
            ].nunique(),

            confirmed_onsets,

            len(
                segment_start_attacks
            ),

            onset_captures,

            onset_segments,

            len(
                attack_runs
            ),

            len(
                benign_runs
            ),

            median_clean_history,

            max_clean_history,

            support_classification,

            len(
                validation
            ),

            int(
                validation[
                    "Passed"
                ].sum()
            ),

            all_validation_pass
        ]
    }
)


print(
    manifest.to_string(
        index=False
    )
)


MANIFEST_OUT = (
    UNSW_OUT_DIR
    /
    "10_16_UNSW_MANIFEST.csv"
)


manifest.to_csv(
    MANIFEST_OUT,
    index=False
)


print(
    "\n✓",
    MANIFEST_OUT
)


# ==================================================================================================
# 10.16U — FINAL INTERPRETATION REPORT
# ==================================================================================================

banner("10.16U — FINAL TEMPORAL FEASIBILITY REPORT")


print(
    f"""
Observed minute rows                 : {len(minute_df):,}
Physical captures                    : {minute_df['Capture_ID'].nunique()}
Continuous temporal segments         : {minute_df['Temporal_Segment_ID'].nunique()}

Confirmed benign → attack onsets     : {confirmed_onsets:,}
Attack-active segment starts         : {len(segment_start_attacks):,}

Captures containing confirmed onsets : {onset_captures}
Segments containing confirmed onsets : {onset_segments}

Attack runs                          : {len(attack_runs):,}
Benign runs                          : {len(benign_runs):,}

Median clean benign history          : {median_clean_history}
Maximum clean benign history         : {max_clean_history}

Structural support classification    : {support_classification}
"""
)


print(
    "\nSTRICT FORECAST HORIZON SUPPORT"
)

print(
    horizon_summary.to_string(
        index=False
    )
)


print(
    "\nNAIVE FUTURE-STATE PERSISTENCE CONTAMINATION"
)

print(
    persistence_audit.to_string(
        index=False
    )
)


print(
    "\nPRE-ATTACK WINDOW QUALITY"
)

print(
    contamination_summary.to_string(
        index=False
    )
)


print(
    "\nAUTOMATIC RED FLAGS"
)

print(
    red_flags[
        red_flags[
            "Flag"
        ]
        ==
        True
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 10.16V — FINAL DECISION
# ==================================================================================================

banner("10.16V — FINAL DECISION")


if all_validation_pass:

    print(
        """
✓ PHASE 10.16 AUDIT COMPLETED SUCCESSFULLY

The UNSW temporal representation has now been audited for:

    ✓ confirmed benign → attack transitions
    ✓ contiguous attack/benign runs
    ✓ clean pre-attack history
    ✓ 1-minute forecasting support
    ✓ 2-minute forecasting support
    ✓ 5-minute forecasting support
    ✓ 10-minute forecasting support
    ✓ 15-minute forecasting support
    ✓ 30-minute forecasting support
    ✓ naive future-state persistence contamination
    ✓ pre-attack window contamination
    ✓ onset-family support
    ✓ minute-level family ambiguity
    ✓ capture-specific forecasting support

IMPORTANT:

Phase 10.16 is an AUDIT.

The horizon targets created here are diagnostic targets only.

Do NOT yet:

    - train LSTM
    - train GRU
    - train XGBoost
    - oversample
    - undersample
    - SMOTE
    - normalize using the full dataset
    - randomly split minutes
    - randomly split sequences

The next decision must be based on the actual 10.16 results:

    1. How many independent attack onsets exist?
    2. How much benign history exists before them?
    3. Which horizons are genuinely supported?
    4. How severe is attack-persistence contamination?
    5. Whether UNSW alone can support onset forecasting.
    6. How UNSW should later be combined with CIC without
       attack-family or temporal leakage.

NEXT:

    PHASE 10.17 — UNSW FORECAST TARGET POLICY &
                  CROSS-DATASET SPLIT FEASIBILITY

"""
    )


else:

    print(
        """
⚠ PHASE 10.16 VALIDATION FAILURE

The temporal feasibility audit itself did not complete cleanly.

Do NOT proceed.

Review the validation table above.
"""
    )


banner("PHASE 10.16 COMPLETE")

In [ ]:
# ==================================================================================================
# PHASE 10.17 — UNSW FORECAST TARGET POLICY & CROSS-DATASET SPLIT FEASIBILITY
# ==================================================================================================
#
# PURPOSE
# -------
# Phase 10.16 established the actual temporal forecasting support available in UNSW-NB15.
#
# This phase DOES NOT:
#   - reconstruct raw UNSW flows,
#   - rematch GT,
#   - create train/test splits,
#   - scale features,
#   - train models,
#   - oversample positives,
#   - manufacture new attack events.
#
# Instead, this phase freezes the forecasting methodology that MUST be used later for:
#
#       UNSW-NB15
#            +
#       CIC-IDS2017
#            ↓
#       COMMON FORECASTING DATASET
#
#
# CORE PRINCIPLE
# --------------
# We are forecasting NEW ATTACK ONSETS.
#
# We are NOT forecasting:
#
#       Attack_State(t + H)
#
# because that can reward a model simply for recognizing an attack that is already active.
#
# The forecasting question is:
#
#       Given information available up to benign time t,
#       will a NEW attack begin within the next H minutes?
#
#
# IMPORTANT
# ---------
# Multiple pre-attack windows belonging to the SAME physical attack onset are NOT
# independent attack events.
#
# Therefore:
#
#       SAME ATTACK EVENT MUST NEVER CROSS TRAIN / VALIDATION / TEST.
#
# ==================================================================================================


from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")


# ==================================================================================================
# 10.17A — PATHS
# ==================================================================================================

print("=" * 120)
print("PHASE 10.17 — UNSW FORECAST TARGET POLICY & CROSS-DATASET SPLIT FEASIBILITY")
print("=" * 120)

BASE = Path("/content/drive/MyDrive/network-attack-forecasting/Dataset_Audit_v2")
UNSW_OUT = BASE / "audit_outputs" / "UNSW"

UNSW_OUT.mkdir(parents=True, exist_ok=True)


# Required Phase 10.16 artifacts
ONSET_FILE = UNSW_OUT / "10_16_UNSW_CONFIRMED_ATTACK_ONSETS.csv"
ONSET_HISTORY_FILE = UNSW_OUT / "10_16_UNSW_ONSET_HISTORY.csv"
HISTORY_FEAS_FILE = UNSW_OUT / "10_16_UNSW_HISTORY_FEASIBILITY.csv"
HORIZON_FEAS_FILE = UNSW_OUT / "10_16_UNSW_HORIZON_FEASIBILITY.csv"
CAPTURE_HORIZON_FILE = UNSW_OUT / "10_16_UNSW_CAPTURE_HORIZON_FEASIBILITY.csv"
PERSISTENCE_FILE = UNSW_OUT / "10_16_UNSW_PERSISTENCE_AUDIT.csv"
PREATTACK_FILE = UNSW_OUT / "10_16_UNSW_PREATTACK_WINDOW_AUDIT.csv"
RED_FLAGS_FILE = UNSW_OUT / "10_16_UNSW_FORECASTING_RED_FLAGS.csv"
VALIDATION_FILE = UNSW_OUT / "10_16_UNSW_VALIDATION.csv"
MINUTE_TARGET_FILE = UNSW_OUT / "10_16_UNSW_MINUTE_FORECAST_TARGET_AUDIT.csv"


required_files = [
    ONSET_FILE,
    ONSET_HISTORY_FILE,
    HISTORY_FEAS_FILE,
    HORIZON_FEAS_FILE,
    CAPTURE_HORIZON_FILE,
    PERSISTENCE_FILE,
    PREATTACK_FILE,
    RED_FLAGS_FILE,
    VALIDATION_FILE,
    MINUTE_TARGET_FILE,
]


print("\n" + "=" * 120)
print("10.17A — VERIFY PHASE 10.16 ARTIFACTS")
print("=" * 120)

missing = []

for f in required_files:
    if f.exists():
        print(f"✓ {f.name}")
    else:
        print(f"✗ MISSING: {f.name}")
        missing.append(f)

if missing:
    raise FileNotFoundError(
        "\nPhase 10.17 requires the Phase 10.16 artifacts above.\n"
        "Missing:\n" +
        "\n".join(str(x) for x in missing)
    )

print("\n✓ All required Phase 10.16 artifacts found")


# ==================================================================================================
# 10.17B — LOAD PHASE 10.16 RESULTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17B — LOAD PHASE 10.16 RESULTS")
print("=" * 120)


onsets = pd.read_csv(ONSET_FILE)
onset_history = pd.read_csv(ONSET_HISTORY_FILE)
history_feas = pd.read_csv(HISTORY_FEAS_FILE)
horizon_feas = pd.read_csv(HORIZON_FEAS_FILE)
capture_horizon = pd.read_csv(CAPTURE_HORIZON_FILE)
persistence = pd.read_csv(PERSISTENCE_FILE)
preattack = pd.read_csv(PREATTACK_FILE)
red_flags = pd.read_csv(RED_FLAGS_FILE)
validation = pd.read_csv(VALIDATION_FILE)
minute_target = pd.read_csv(MINUTE_TARGET_FILE)


print(f"Confirmed attack onsets : {len(onsets):,}")
print(f"Onset-history rows      : {len(onset_history):,}")
print(f"History feasibility rows: {len(history_feas):,}")
print(f"Horizon feasibility rows: {len(horizon_feas):,}")
print(f"Minute target rows      : {len(minute_target):,}")


# ==================================================================================================
# 10.17C — DISCOVER AVAILABLE HORIZONS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17C — DISCOVER FORECAST HORIZONS")
print("=" * 120)


def detect_horizon_column(df):
    candidates = [
        "Horizon_Minutes",
        "Horizon_Min",
        "Horizon",
        "Forecast_Horizon_Min",
        "Forecast_Horizon_Minutes"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    return None


horizon_col = detect_horizon_column(horizon_feas)

if horizon_col is None:
    # Fallback to horizons already used in Phase 10.16
    HORIZONS = [1, 2, 5, 10, 15, 30]
    print("⚠ Horizon column not automatically detected.")
    print("Using Phase 10.16 standard horizons:", HORIZONS)

else:
    HORIZONS = (
        pd.to_numeric(horizon_feas[horizon_col], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    HORIZONS = sorted(HORIZONS)

    print("Detected horizon column :", horizon_col)
    print("Available horizons      :", HORIZONS)


# ==================================================================================================
# 10.17D — FREEZE AUTHORITATIVE FORECASTING TARGET DEFINITION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17D — AUTHORITATIVE FORECASTING TARGET POLICY")
print("=" * 120)


policy_rows = [
    {
        "Policy_ID": "P01",
        "Policy_Name": "Forecast_New_Attack_Onset",
        "Rule":
            "Forecasting target is a NEW benign-to-attack transition occurring after prediction time.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P02",
        "Policy_Name": "Current_State_Must_Be_Benign",
        "Rule":
            "A time point is eligible for onset forecasting only when Attack_State(t)=0.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P03",
        "Policy_Name": "No_Already_Active_Attacks",
        "Rule":
            "Rows where Attack_State(t)=1 cannot be positive onset-forecast samples.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P04",
        "Policy_Name": "No_Cross_Segment_Targeting",
        "Rule":
            "History and future target intervals must remain inside the same continuous temporal segment.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P05",
        "Policy_Name": "No_Cross_Capture_Targeting",
        "Rule":
            "History and target construction must never bridge independent capture periods.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P06",
        "Policy_Name": "Event_Level_Independence",
        "Rule":
            "Multiple windows preceding the same physical attack onset are correlated observations, not independent attack events.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P07",
        "Policy_Name": "Attack_Event_Split_Isolation",
        "Rule":
            "All samples associated with one physical attack onset must remain in exactly one of train, validation, or test.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P08",
        "Policy_Name": "Temporal_Context_Isolation",
        "Rule":
            "Overlapping or temporally dependent windows must not be distributed across different dataset partitions.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P09",
        "Policy_Name": "Fit_Preprocessing_On_Train_Only",
        "Rule":
            "Scalers, imputers, encoders, feature selection and learned preprocessing must be fit only on training data.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P10",
        "Policy_Name": "No_Synthetic_Event_Inflation",
        "Rule":
            "Oversampling or synthetic rows cannot be counted as new independent physical attack events.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P11",
        "Policy_Name": "Dataset_Origin_Preserved",
        "Rule":
            "Dataset origin must remain available after CIC and UNSW harmonization for cross-dataset auditing.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P12",
        "Policy_Name": "Event_Level_Evaluation",
        "Rule":
            "Forecasting performance must be evaluated at both sample level and independent attack-event level.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P13",
        "Policy_Name": "Lead_Time_From_First_Valid_Warning",
        "Rule":
            "Lead time is measured from the earliest valid pre-onset warning to the corresponding attack onset.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P14",
        "Policy_Name": "No_Attack_Family_Leakage",
        "Rule":
            "Attack labels, future attack family information, episode outcome fields, and future-derived variables cannot appear in model input features.",
        "Mandatory": True
    },

    {
        "Policy_ID": "P15",
        "Policy_Name": "No_Forced_Class_Coverage",
        "Rule":
            "Attack events must not be moved across temporal/event partitions merely to make every attack family appear in every split.",
        "Mandatory": True
    },
]


policy_df = pd.DataFrame(policy_rows)

print(policy_df[
    ["Policy_ID", "Policy_Name", "Mandatory"]
].to_string(index=False))


# ==================================================================================================
# 10.17E — DEFINE TARGET MATHEMATICALLY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17E — FORMAL TARGET DEFINITION")
print("=" * 120)


print(
"""
For a currently benign prediction time t:

    Attack_State(t) = 0

For forecasting horizon H:

    Y_H(t) = 1

iff at least one NEW attack onset occurs in:

    (t, t + H]

within the SAME valid temporal segment.

Otherwise:

    Y_H(t) = 0


IMPORTANT:

    Attack_State(t + H)

is NOT the authoritative forecasting target.

Why?

Because an attack may already be active at time t.

That would measure attack persistence / detection rather than
true pre-onset forecasting.
"""
)


# ==================================================================================================
# 10.17F — HISTORY REQUIREMENT POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17F — HISTORY WINDOW POLICY")
print("=" * 120)


history_policy = pd.DataFrame([
    {
        "History_Window_Min": 1,
        "Role": "Minimal diagnostic history",
        "Status": "AUDIT"
    },
    {
        "History_Window_Min": 5,
        "Role": "Short temporal context",
        "Status": "AUDIT"
    },
    {
        "History_Window_Min": 10,
        "Role": "Primary candidate temporal context",
        "Status": "AUDIT"
    },
    {
        "History_Window_Min": 15,
        "Role": "Extended temporal context",
        "Status": "AUDIT"
    },
    {
        "History_Window_Min": 30,
        "Role": "Long temporal context",
        "Status": "AUDIT"
    },
])


print(history_policy.to_string(index=False))

print(
"""
NOTE:
No history window is being selected as final here.

The final W must be chosen only after CIC and UNSW are compared under
the SAME onset/history definition.
"""
)


# ==================================================================================================
# 10.17G — UNSW INDEPENDENT EVENT SUPPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17G — UNSW INDEPENDENT EVENT SUPPORT")
print("=" * 120)


n_onsets = len(onsets)

print(f"Independent confirmed benign→attack onsets: {n_onsets:,}")


# Try to identify clean-history column in onset_history
history_candidates = [
    "Clean_Benign_History_Min",
    "Clean_History_Min",
    "Benign_History_Min",
    "Available_Benign_History_Min",
    "Pre_Onset_Benign_Minutes"
]

history_support_col = None

for c in history_candidates:
    if c in onset_history.columns:
        history_support_col = c
        break


history_support_rows = []

if history_support_col is not None:

    hist_values = pd.to_numeric(
        onset_history[history_support_col],
        errors="coerce"
    ).fillna(0)

    for w in [1, 2, 5, 10, 15, 30]:

        supported = int((hist_values >= w).sum())

        history_support_rows.append({
            "History_Min": w,
            "Independent_Onsets_With_Enough_History": supported,
            "Total_Confirmed_Onsets": n_onsets,
            "Support_Pct":
                100 * supported / n_onsets if n_onsets > 0 else np.nan
        })

else:

    print(
        "⚠ Could not directly identify clean-history column in onset-history artifact."
    )

    # Use 10.16 feasibility artifact instead if possible.
    print("Using 10.16 history-feasibility artifact for reporting.")

    # Preserve artifact rather than guessing semantics.
    print("\n10.16 history feasibility:")
    print(history_feas.to_string(index=False))


history_support_df = pd.DataFrame(history_support_rows)

if len(history_support_df) > 0:
    print("\nIndependent onset support by required history:\n")
    print(history_support_df.to_string(index=False))


# ==================================================================================================
# 10.17H — UNSW HORIZON SUPPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17H — UNSW FORECAST-HORIZON SUPPORT")
print("=" * 120)


print(
"""
Two quantities must NEVER be confused:

    POSITIVE WINDOWS
            ≠
    INDEPENDENT ATTACK EVENTS

Ten positive pre-attack minutes belonging to one attack still represent
only ONE independent physical attack event.
"""
)

print("\nPhase 10.16 horizon feasibility:\n")
print(horizon_feas.to_string(index=False))


# ==================================================================================================
# 10.17I — SPLIT FEASIBILITY RULES
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17I — TRAIN / VALIDATION / TEST SPLIT FEASIBILITY")
print("=" * 120)


split_rules = pd.DataFrame([
    {
        "Rule": "Random row split",
        "Allowed": False,
        "Reason":
            "Can place temporally adjacent / overlapping windows from the same event in different partitions."
    },

    {
        "Rule": "Random minute split",
        "Allowed": False,
        "Reason":
            "Violates temporal dependence and attack-event independence."
    },

    {
        "Rule": "Random sequence-window split",
        "Allowed": False,
        "Reason":
            "Overlapping windows can leak nearly identical temporal information."
    },

    {
        "Rule": "Attack-event grouped split",
        "Allowed": True,
        "Reason":
            "Keeps all windows belonging to one physical attack onset together."
    },

    {
        "Rule": "Capture-aware temporal split",
        "Allowed": True,
        "Reason":
            "Preserves chronology and independent capture boundaries."
    },

    {
        "Rule": "Dataset-origin-aware split",
        "Allowed": True,
        "Reason":
            "Allows CIC-vs-UNSW generalization and source-specific performance auditing."
    }
])


print(split_rules.to_string(index=False))


# ==================================================================================================
# 10.17J — CAN UNSW ALONE SUPPORT A NORMAL 3-WAY EVENT SPLIT?
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17J — UNSW STANDALONE SPLIT FEASIBILITY")
print("=" * 120)


# Conservative methodological thresholds.
#
# These are NOT universal statistical laws.
# They are internal audit categories preventing us from pretending that
# extremely tiny event counts support a conventional supervised experiment.

if n_onsets < 3:
    split_status = "IMPOSSIBLE_3WAY_EVENT_SPLIT"

elif n_onsets < 10:
    split_status = "EXTREMELY_WEAK_EVENT_SUPPORT"

elif n_onsets < 30:
    split_status = "LIMITED_EVENT_SUPPORT"

else:
    split_status = "POTENTIALLY_FEASIBLE"


print("Confirmed independent onsets :", n_onsets)
print("Standalone split status       :", split_status)


if n_onsets < 3:

    print(
"""
INTERPRETATION:

UNSW alone cannot create:

    TRAIN
    VALIDATION
    TEST

with at least one independent benign→attack onset in each partition.

Therefore UNSW must NOT be presented as independently supporting a
standard 3-way supervised onset-forecasting experiment.
"""
    )


# ==================================================================================================
# 10.17K — ATTACK-FAMILY SPLIT POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17K — ATTACK FAMILY COVERAGE POLICY")
print("=" * 120)


print(
"""
IMPORTANT FOR LATER CIC + UNSW COMBINATION:

We must NOT do this:

    Train:
        DoS
        Exploits
        Recon

    Test:
        Fuzzers
        Worms

and then pretend ordinary supervised multiclass performance is directly
comparable if the model never saw those target classes.

BUT we also must NOT create leakage by moving correlated attack events
around merely to force class coverage.

Therefore later we will audit THREE things separately:

1. TRAIN-SEEN FAMILIES
   Attack families represented in training.

2. TEST-SEEN FAMILIES
   Test events whose family was represented in training.

3. TEST-UNSEEN FAMILIES
   Test events whose family was absent from training.

For standard supervised attack-type metrics:

    evaluate TRAIN-SEEN families separately.

For unseen-family events:

    report them explicitly as open-set / unseen-family cases.

DO NOT silently delete them.
DO NOT move them into training.
DO NOT let class balancing override temporal integrity.
"""
)


# ==================================================================================================
# 10.17L — CROSS-DATASET COMBINATION POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17L — CIC + UNSW COMBINATION POLICY")
print("=" * 120)


combination_policy = pd.DataFrame([
    {
        "Component": "Time representation",
        "Requirement":
            "Common temporal resolution and equivalent causal feature meaning."
    },

    {
        "Component": "Forecast target",
        "Requirement":
            "Same benign→new-attack-onset definition in CIC and UNSW."
    },

    {
        "Component": "History window",
        "Requirement":
            "Same W-minute causal history definition."
    },

    {
        "Component": "Forecast horizon",
        "Requirement":
            "Same H-minute onset horizon."
    },

    {
        "Component": "Attack event",
        "Requirement":
            "Physical attack events reconstructed independently inside each source dataset."
    },

    {
        "Component": "Dataset origin",
        "Requirement":
            "Preserve Source_Dataset = CIC_IDS2017 or UNSW_NB15."
    },

    {
        "Component": "Segment identity",
        "Requirement":
            "Segment IDs must be dataset-qualified and globally unique."
    },

    {
        "Component": "Attack event identity",
        "Requirement":
            "Event IDs must be dataset-qualified and globally unique."
    },

    {
        "Component": "Feature harmonization",
        "Requirement":
            "Only semantically compatible causal features may enter the shared model matrix."
    },

    {
        "Component": "Scaling",
        "Requirement":
            "Fit using training partition only."
    },

    {
        "Component": "Splitting",
        "Requirement":
            "Event/capture/temporal grouping must occur before any learned preprocessing."
    },

    {
        "Component": "Attack family",
        "Requirement":
            "Use harmonized family taxonomy only after source-specific semantics are preserved."
    }
])


print(combination_policy.to_string(index=False))


# ==================================================================================================
# 10.17M — LEAKAGE THREAT REGISTER
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17M — LEAKAGE THREAT REGISTER")
print("=" * 120)


leakage_register = pd.DataFrame([
    {
        "Threat_ID": "L01",
        "Threat": "Same attack event across train/test",
        "Severity": "CRITICAL",
        "Prevention": "Group by physical attack event before splitting."
    },

    {
        "Threat_ID": "L02",
        "Threat": "Overlapping history windows across partitions",
        "Severity": "CRITICAL",
        "Prevention": "Split temporal groups/events before sequence generation."
    },

    {
        "Threat_ID": "L03",
        "Threat": "Future attack state used as feature",
        "Severity": "CRITICAL",
        "Prevention": "Strict causal feature whitelist."
    },

    {
        "Threat_ID": "L04",
        "Threat": "Future attack family used as feature",
        "Severity": "CRITICAL",
        "Prevention": "Target metadata excluded from model inputs."
    },

    {
        "Threat_ID": "L05",
        "Threat": "Scaler fitted before split",
        "Severity": "HIGH",
        "Prevention": "Fit scaler on training partition only."
    },

    {
        "Threat_ID": "L06",
        "Threat": "Feature selection using full dataset",
        "Severity": "HIGH",
        "Prevention": "Learn feature selection from training data only."
    },

    {
        "Threat_ID": "L07",
        "Threat": "Synthetic samples created before split",
        "Severity": "HIGH",
        "Prevention": "Any resampling occurs inside training data only."
    },

    {
        "Threat_ID": "L08",
        "Threat": "Same capture represented in incompatible partitions",
        "Severity": "HIGH",
        "Prevention": "Capture-aware grouping / chronology audit."
    },

    {
        "Threat_ID": "L09",
        "Threat": "Dataset-specific identifiers become predictive shortcuts",
        "Severity": "HIGH",
        "Prevention": "Preserve IDs for auditing but exclude identifiers from model features."
    },

    {
        "Threat_ID": "L10",
        "Threat": "Attack persistence mislabeled as forecasting",
        "Severity": "CRITICAL",
        "Prevention": "Positive forecasting samples require current benign state."
    },

    {
        "Threat_ID": "L11",
        "Threat": "Multiple windows counted as independent attacks",
        "Severity": "CRITICAL",
        "Prevention": "Report event-level metrics and independent event counts."
    },

    {
        "Threat_ID": "L12",
        "Threat": "Forced class balancing changes temporal partition",
        "Severity": "HIGH",
        "Prevention": "Never move events across partitions merely for family balance."
    }
])


print(leakage_register.to_string(index=False))


# ==================================================================================================
# 10.17N — REQUIRED EVALUATION METRICS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17N — REQUIRED FORECASTING EVALUATION")
print("=" * 120)


metrics_policy = pd.DataFrame([
    {
        "Metric": "PR-AUC",
        "Level": "Sample",
        "Required": True,
        "Reason": "Important under severe positive-class rarity."
    },

    {
        "Metric": "Precision",
        "Level": "Sample",
        "Required": True,
        "Reason": "Measures warning reliability."
    },

    {
        "Metric": "Recall",
        "Level": "Sample",
        "Required": True,
        "Reason": "Measures recovered positive forecast windows."
    },

    {
        "Metric": "F1",
        "Level": "Sample",
        "Required": True,
        "Reason": "Joint precision/recall summary."
    },

    {
        "Metric": "ROC-AUC",
        "Level": "Sample",
        "Required": False,
        "Reason": "Secondary metric; can appear optimistic under extreme imbalance."
    },

    {
        "Metric": "Event Recall",
        "Level": "Attack event",
        "Required": True,
        "Reason": "Fraction of independent attacks receiving at least one valid advance warning."
    },

    {
        "Metric": "First-Warning Lead Time",
        "Level": "Attack event",
        "Required": True,
        "Reason": "Primary forecasting utility measure."
    },

    {
        "Metric": "Median Lead Time",
        "Level": "Attack event",
        "Required": True,
        "Reason": "Robust summary of warning advance."
    },

    {
        "Metric": "False Alarms per Hour",
        "Level": "Operational",
        "Required": True,
        "Reason": "Measures alert burden."
    },

    {
        "Metric": "Event Precision",
        "Level": "Attack event",
        "Required": True,
        "Reason": "Measures how often forecast alert episodes correspond to future attacks."
    }
])


print(metrics_policy.to_string(index=False))


# ==================================================================================================
# 10.17O — UNSW FORECASTING ROLE DECISION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17O — UNSW FORECASTING ROLE DECISION")
print("=" * 120)


# Based only on empirically reconstructed onset support.
if n_onsets <= 2:

    unsw_role = "LIMITED_AUXILIARY_FORECASTING_SUPPORT"

    unsw_interpretation = (
        "UNSW contains too few independent confirmed benign-to-attack onsets "
        "to serve as the sole dataset for a conventional supervised onset-forecasting experiment. "
        "It may still contribute to a combined experiment, representation learning, "
        "robustness analysis, source-specific analysis, or auxiliary detection tasks, "
        "subject to the later CIC+UNSW feasibility audit."
    )

elif n_onsets < 10:

    unsw_role = "WEAK_STANDALONE_FORECASTING_SUPPORT"

    unsw_interpretation = (
        "UNSW has some independent onset support but event count remains too small "
        "for strong standalone supervised forecasting claims."
    )

else:

    unsw_role = "CANDIDATE_FORECASTING_SOURCE"

    unsw_interpretation = (
        "UNSW contains enough reconstructed events to proceed to a deeper split-feasibility audit."
    )


print("UNSW role:")
print("  ", unsw_role)

print("\nInterpretation:")
print(unsw_interpretation)


# ==================================================================================================
# 10.17P — WHAT CIC MUST NOW BE AUDITED FOR
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17P — REQUIRED CIC EQUIVALENT AUDIT")
print("=" * 120)


cic_requirements = pd.DataFrame([
    {
        "Audit_Item": "Continuous temporal segments",
        "Required": True
    },

    {
        "Audit_Item": "Benign→attack state transitions",
        "Required": True
    },

    {
        "Audit_Item": "Independent confirmed attack onsets",
        "Required": True
    },

    {
        "Audit_Item": "Clean benign history before each onset",
        "Required": True
    },

    {
        "Audit_Item": "1-minute history support",
        "Required": True
    },

    {
        "Audit_Item": "2-minute history support",
        "Required": True
    },

    {
        "Audit_Item": "5-minute history support",
        "Required": True
    },

    {
        "Audit_Item": "10-minute history support",
        "Required": True
    },

    {
        "Audit_Item": "15-minute history support",
        "Required": True
    },

    {
        "Audit_Item": "30-minute history support",
        "Required": True
    },

    {
        "Audit_Item": "Forecast horizon positive windows",
        "Required": True
    },

    {
        "Audit_Item": "Independent events represented by positive windows",
        "Required": True
    },

    {
        "Audit_Item": "Attack persistence contamination",
        "Required": True
    },

    {
        "Audit_Item": "Attack family support at onset",
        "Required": True
    },

    {
        "Audit_Item": "Family overlap across candidate splits",
        "Required": True
    },

    {
        "Audit_Item": "Event/campaign overlap across candidate splits",
        "Required": True
    }
])


print(cic_requirements.to_string(index=False))


# ==================================================================================================
# 10.17Q — CREATE CROSS-DATASET FEASIBILITY TEMPLATE
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17Q — CREATE CIC vs UNSW FEASIBILITY TEMPLATE")
print("=" * 120)


# We deliberately populate only facts established for UNSW here.
# CIC remains pending until evaluated under the exact same methodology.

comparison_rows = [
    {
        "Metric": "Confirmed benign→attack onsets",
        "UNSW_NB15": n_onsets,
        "CIC_IDS2017": "PENDING_EQUIVALENT_AUDIT"
    }
]


if len(history_support_df) > 0:

    for _, r in history_support_df.iterrows():

        comparison_rows.append({
            "Metric":
                f"Onsets with >= {int(r['History_Min'])} min clean benign history",
            "UNSW_NB15":
                int(r["Independent_Onsets_With_Enough_History"]),
            "CIC_IDS2017":
                "PENDING_EQUIVALENT_AUDIT"
        })


comparison_rows.extend([
    {
        "Metric": "Random row split allowed",
        "UNSW_NB15": "NO",
        "CIC_IDS2017": "NO"
    },

    {
        "Metric": "Event-grouped split required",
        "UNSW_NB15": "YES",
        "CIC_IDS2017": "YES"
    },

    {
        "Metric": "Current attack rows eligible as onset positives",
        "UNSW_NB15": "NO",
        "CIC_IDS2017": "NO"
    },

    {
        "Metric": "Dataset origin must be preserved",
        "UNSW_NB15": "YES",
        "CIC_IDS2017": "YES"
    }
])


comparison_template = pd.DataFrame(comparison_rows)

print(comparison_template.to_string(index=False))


# ==================================================================================================
# 10.17R — DEFINE CROSS-DATASET DECISION LOGIC
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17R — CROSS-DATASET DECISION LOGIC")
print("=" * 120)


decision_logic = pd.DataFrame([
    {
        "Condition": "Combined independent onset support is adequate",
        "Action":
            "Proceed to harmonized CIC+UNSW forecasting dataset."
    },

    {
        "Condition": "Combined onset support exists but is limited",
        "Action":
            "Use event-level evaluation, restricted claims, grouped validation and strong uncertainty reporting."
    },

    {
        "Condition": "Combined independent onset support remains extremely small",
        "Action":
            "Do not force conventional supervised forecasting; evaluate additional temporally suitable dataset(s) or redesign experimental target."
    },

    {
        "Condition": "Some attack families absent from training",
        "Action":
            "Report seen-family and unseen-family evaluation separately; do not leak events across partitions."
    },

    {
        "Condition": "Long forecast horizons have insufficient event support",
        "Action":
            "Restrict primary experiment to empirically supported horizons."
    },

    {
        "Condition": "History window eliminates most independent events",
        "Action":
            "Do not select that history window merely for model complexity; retain only empirically supportable W."
    }
])


print(decision_logic.to_string(index=False))


# ==================================================================================================
# 10.17S — MACHINE-READABLE METHODOLOGY CONFIG
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17S — BUILD METHODOLOGY CONFIG")
print("=" * 120)


methodology_config = {
    "phase": "10.17",

    "task": "NEW_ATTACK_ONSET_FORECASTING",

    "target_definition": {
        "prediction_state_required": "BENIGN",
        "positive_condition":
            "NEW attack onset occurs within future horizon H",
        "negative_condition":
            "No new attack onset occurs within future horizon H",
        "same_temporal_segment_required": True,
        "same_capture_required": True,
        "already_active_attack_is_forecast_positive": False
    },

    "split_policy": {
        "random_row_split": False,
        "random_minute_split": False,
        "random_window_split": False,
        "attack_event_grouping_required": True,
        "temporal_dependency_isolation_required": True,
        "capture_awareness_required": True,
        "dataset_origin_preserved": True
    },

    "preprocessing_policy": {
        "fit_scaler_on_train_only": True,
        "fit_imputer_on_train_only": True,
        "fit_encoder_on_train_only": True,
        "feature_selection_on_train_only": True,
        "resampling_training_only": True
    },

    "evaluation_policy": {
        "sample_metrics": [
            "PR_AUC",
            "Precision",
            "Recall",
            "F1"
        ],

        "event_metrics": [
            "Event_Recall",
            "Event_Precision",
            "First_Warning_Lead_Time",
            "Median_Lead_Time"
        ],

        "operational_metrics": [
            "False_Alarms_Per_Hour"
        ]
    },

    "attack_family_policy": {
        "force_family_coverage_by_moving_events": False,
        "report_seen_families_separately": True,
        "report_unseen_families_separately": True
    },

    "UNSW": {
        "confirmed_onsets": int(n_onsets),
        "standalone_split_status": split_status,
        "forecasting_role": unsw_role
    }
}


print(json.dumps(methodology_config, indent=4))


# ==================================================================================================
# 10.17T — SAVE ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17T — SAVE PHASE 10.17 ARTIFACTS")
print("=" * 120)


save_map = {
    "10_17_UNSW_FORECAST_POLICY.csv":
        policy_df,

    "10_17_UNSW_HISTORY_POLICY.csv":
        history_policy,

    "10_17_UNSW_SPLIT_RULES.csv":
        split_rules,

    "10_17_UNSW_CROSS_DATASET_COMBINATION_POLICY.csv":
        combination_policy,

    "10_17_UNSW_LEAKAGE_THREAT_REGISTER.csv":
        leakage_register,

    "10_17_UNSW_METRICS_POLICY.csv":
        metrics_policy,

    "10_17_CIC_EQUIVALENT_AUDIT_REQUIREMENTS.csv":
        cic_requirements,

    "10_17_CIC_UNSW_FEASIBILITY_TEMPLATE.csv":
        comparison_template,

    "10_17_CROSS_DATASET_DECISION_LOGIC.csv":
        decision_logic
}


if len(history_support_df) > 0:
    save_map[
        "10_17_UNSW_INDEPENDENT_HISTORY_SUPPORT.csv"
    ] = history_support_df


for filename, df in save_map.items():

    path = UNSW_OUT / filename

    df.to_csv(
        path,
        index=False
    )

    print("✓", path)


config_path = UNSW_OUT / "10_17_FORECAST_METHODOLOGY_CONFIG.json"

with open(config_path, "w") as f:
    json.dump(
        methodology_config,
        f,
        indent=4
    )

print("✓", config_path)


# ==================================================================================================
# 10.17U — FINAL VALIDATION
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17U — FINAL VALIDATION")
print("=" * 120)


validation_rows = []


def add_check(check, passed, detail):

    validation_rows.append({
        "Check": check,
        "Passed": bool(passed),
        "Detail": detail
    })


add_check(
    "Phase 10.16 artifacts available",
    len(missing) == 0,
    f"{len(required_files) - len(missing)}/{len(required_files)} required files found"
)


add_check(
    "Forecast target restricted to new attack onset",
    True,
    "Authoritative target policy frozen"
)


add_check(
    "Currently active attacks excluded from onset-positive eligibility",
    True,
    "Attack_State(t)=1 cannot define a pre-onset forecast positive"
)


add_check(
    "Random row/minute/window split prohibited",
    True,
    "Split policy explicitly forbids all three"
)


add_check(
    "Attack-event grouping required",
    True,
    "Same physical onset cannot cross partitions"
)


add_check(
    "Train-only learned preprocessing required",
    True,
    "Scaler/imputer/encoder/feature selection isolated to training"
)


add_check(
    "Attack-family coverage cannot override leakage controls",
    True,
    "Seen/unseen family reporting policy frozen"
)


add_check(
    "Event-level evaluation required",
    True,
    "Event recall, event precision and lead time required"
)


add_check(
    "CIC equivalent audit defined",
    True,
    f"{len(cic_requirements):,} CIC audit requirements specified"
)


validation_17 = pd.DataFrame(validation_rows)

print(validation_17.to_string(index=False))


validation_path = UNSW_OUT / "10_17_VALIDATION.csv"

validation_17.to_csv(
    validation_path,
    index=False
)

print("\n✓ Saved:")
print(validation_path)


# ==================================================================================================
# 10.17V — FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("10.17V — FINAL METHODOLOGICAL DECISION")
print("=" * 120)


print(f"""
UNSW confirmed independent benign→attack onsets : {n_onsets}

UNSW standalone split status:
    {split_status}

UNSW role:
    {unsw_role}


AUTHORITATIVE FORECASTING TASK
------------------------------

At benign time t:

    use only information available at or before t

and predict:

    whether a NEW attack onset occurs within H minutes.


STRICTLY PROHIBITED
-------------------

✗ random row split
✗ random minute split
✗ random sequence-window split
✗ same physical attack event across partitions
✗ future attack information in features
✗ scaler / encoder / feature selection before splitting
✗ synthetic samples before splitting
✗ counting multiple windows as independent attacks
✗ forcing attack-family coverage by moving events across partitions
✗ treating an already-active attack as successful pre-onset forecasting


REQUIRED
--------

✓ attack-event-aware splitting
✓ temporal/capture integrity
✓ train-only learned preprocessing
✓ dataset-origin preservation
✓ event-level evaluation
✓ first-warning lead-time evaluation
✓ false-alarm evaluation
✓ explicit seen-vs-unseen attack-family reporting


NEXT STEP
---------

Run the CIC-IDS2017 equivalent onset-feasibility audit using the
EXACT SAME forecasting definition.

Then construct:

                 CIC              UNSW
                  │                 │
                  └──── SAME POLICY ┘
                           │
                           ▼
                 CROSS-DATASET FEASIBILITY
                           │
             ┌─────────────┴─────────────┐
             │                           │
       enough events               insufficient events
             │                           │
             ▼                           ▼
      harmonize + combine       add suitable dataset /
                               redesign experiment

NO MODEL TRAINING SHOULD OCCUR BEFORE THAT DECISION.
""")


print("=" * 120)
print("PHASE 10.17 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 11.0 — SYNTHETIC NAF v1 DATASET SPECIFICATION, SPLIT CONTRACT & LEAKAGE FREEZE
# ==================================================================================================
#
# DATASET NAME
# ------------
# Synthetic Network Attack Forecasting Benchmark v1
# Short name: Synthetic_NAF_v1
#
#
# PURPOSE
# -------
# Freeze the complete specification BEFORE generating any synthetic observations.
#
# This phase defines:
#
#   1. Number of independent episodes
#   2. Attack-family taxonomy
#   3. Exact train / validation / test episode allocation
#   4. Exact family coverage in every split
#   5. Precursor regimes
#   6. Timestamp policy
#   7. Episode-duration / onset / attack-duration constraints
#   8. Common predictor feature schema
#   9. Derived-feature formulas
#  10. Forecasting target semantics
#  11. Metadata vs predictor separation
#  12. Leakage rules
#  13. Calibration requirements for CIC + UNSW
#  14. Structural validation requirements
#  15. Dataset freeze / reproducibility policy
#
#
# THIS PHASE DOES NOT:
#
#   ✗ generate episodes
#   ✗ generate feature values
#   ✗ train models
#   ✗ fit scalers
#   ✗ load CIC/UNSW millions of rows
#   ✗ create synthetic targets
#
#
# IF THIS CELL PASSES:
#
#   11.1 = real CIC/UNSW calibration
#   11.2 = calibration validation
#   11.3 = episode manifest generation
#   11.4 = synthetic minute generation
#   11.5 = structural + realism audit
#   11.6 = dataset freeze
#
# ==================================================================================================


from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
from collections import OrderedDict


# ==================================================================================================
# 11.0A — PATHS
# ==================================================================================================

print("=" * 120)
print("PHASE 11.0 — SYNTHETIC NAF v1 DATASET SPECIFICATION, SPLIT CONTRACT & LEAKAGE FREEZE")
print("=" * 120)


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

SPEC_DIR = (
    DATASET_ROOT
    / "spec"
)

CALIBRATION_DIR = (
    DATASET_ROOT
    / "calibration"
)

MANIFEST_DIR = (
    DATASET_ROOT
    / "manifests"
)

DATA_DIR = (
    DATASET_ROOT
    / "data"
)

AUDIT_DIR = (
    DATASET_ROOT
    / "audit"
)


for directory in [
    SPEC_DIR,
    CALIBRATION_DIR,
    MANIFEST_DIR,
    DATA_DIR,
    AUDIT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("Dataset root:")
print(DATASET_ROOT)


# ==================================================================================================
# 11.0B — GLOBAL DATASET IDENTITY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0B — DATASET IDENTITY")
print("=" * 120)


DATASET_NAME = (
    "Synthetic Network Attack Forecasting Benchmark"
)

DATASET_SHORT_NAME = (
    "Synthetic_NAF_v1"
)

DATASET_VERSION = (
    "1.0.0"
)

MASTER_SEED = (
    20260930
)


DATASET_CLAIM = (
    "Controlled synthetic temporal benchmark calibrated from "
    "CIC-IDS2017 and UNSW-NB15 real traffic statistics."
)


PROHIBITED_CLAIM = (
    "Synthetic episodes must never be described as real CIC/UNSW attack "
    "episodes or as evidence that the simulated precursor trajectories "
    "necessarily occur in real production networks."
)


print("Name          :", DATASET_NAME)
print("Short name    :", DATASET_SHORT_NAME)
print("Version       :", DATASET_VERSION)
print("Master seed   :", MASTER_SEED)
print("\nDataset claim :")
print(DATASET_CLAIM)


# ==================================================================================================
# 11.0C — EPISODE COUNTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0C — EPISODE COUNTS")
print("=" * 120)


TOTAL_EPISODES = 1200

ATTACK_EPISODES = 1000

BENIGN_EPISODES = 200


ATTACK_FAMILIES = [

    "DOS_DDOS",

    "RECON_SCAN",

    "EXPLOIT_APP",

    "BRUTE_AUTH",

    "FUZZING"
]


EPISODES_PER_ATTACK_FAMILY = 200


assert (
    len(ATTACK_FAMILIES)
    *
    EPISODES_PER_ATTACK_FAMILY
    ==
    ATTACK_EPISODES
)


assert (
    ATTACK_EPISODES
    +
    BENIGN_EPISODES
    ==
    TOTAL_EPISODES
)


print(
    "Total episodes       :",
    TOTAL_EPISODES
)

print(
    "Attack episodes      :",
    ATTACK_EPISODES
)

print(
    "Benign-only episodes :",
    BENIGN_EPISODES
)

print(
    "Attack families      :",
    len(ATTACK_FAMILIES)
)

print(
    "Episodes/family      :",
    EPISODES_PER_ATTACK_FAMILY
)


# ==================================================================================================
# 11.0D — EXACT TRAIN / VAL / TEST SPLIT CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0D — EXACT EPISODE SPLIT CONTRACT")
print("=" * 120)


SPLIT_COUNTS_PER_CLASS = {

    "TRAIN": 140,

    "VALIDATION": 30,

    "TEST": 30
}


EXPECTED_SPLIT_TOTALS = {

    "TRAIN": 840,

    "VALIDATION": 180,

    "TEST": 180
}


allocation_rows = []


# Attack classes
for family in ATTACK_FAMILIES:

    for split_name, count in SPLIT_COUNTS_PER_CLASS.items():

        allocation_rows.append(
            {
                "Episode_Type":
                    "ATTACK",

                "Attack_Family":
                    family,

                "Split":
                    split_name,

                "Episode_Count":
                    count
            }
        )


# Benign-only episodes
for split_name, count in SPLIT_COUNTS_PER_CLASS.items():

    allocation_rows.append(
        {
            "Episode_Type":
                "BENIGN_ONLY",

            "Attack_Family":
                "NONE",

            "Split":
                split_name,

            "Episode_Count":
                count
        }
    )


split_allocation = pd.DataFrame(
    allocation_rows
)


print(
    split_allocation.to_string(
        index=False
    )
)


print("\nSplit totals:")

split_totals = (
    split_allocation
    .groupby(
        "Split"
    )[
        "Episode_Count"
    ]
    .sum()
)


print(
    split_totals.to_string()
)


# ==================================================================================================
# 11.0E — ATTACK FAMILY CALIBRATION TAXONOMY
#
# These are CANDIDATE calibration mappings.
#
# 11.1 will discover actual labels.
# 11.2 must verify adequate support.
#
# Nothing here authorizes generation yet.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0E — ATTACK FAMILY CALIBRATION TAXONOMY")
print("=" * 120)


family_spec_rows = [

    {
        "Synthetic_Family":
            "DOS_DDOS",

        "Primary_Concept":
            "Denial-of-service / distributed denial-of-service behaviour",

        "CIC_Candidate_Labels":
            "DDoS | DoS Hulk | DoS GoldenEye | DoS slowloris | DoS Slowhttptest",

        "UNSW_Candidate_Labels":
            "dos",

        "Calibration_Source_Minimum":
            "AT_LEAST_ONE_REAL_DATASET",

        "Enabled":
            True
    },

    {
        "Synthetic_Family":
            "RECON_SCAN",

        "Primary_Concept":
            "Reconnaissance / scanning behaviour",

        "CIC_Candidate_Labels":
            "PortScan",

        "UNSW_Candidate_Labels":
            "reconnaissance",

        "Calibration_Source_Minimum":
            "AT_LEAST_ONE_REAL_DATASET",

        "Enabled":
            True
    },

    {
        "Synthetic_Family":
            "EXPLOIT_APP",

        "Primary_Concept":
            "Application exploitation / exploit-style behaviour",

        "CIC_Candidate_Labels":
            "Web Attack XSS | Web Attack Sql Injection | Infiltration",

        "UNSW_Candidate_Labels":
            "exploits | shellcode | backdoor",

        "Calibration_Source_Minimum":
            "AT_LEAST_ONE_REAL_DATASET",

        "Enabled":
            True
    },

    {
        "Synthetic_Family":
            "BRUTE_AUTH",

        "Primary_Concept":
            "Repeated authentication / credential attack behaviour",

        "CIC_Candidate_Labels":
            "FTP-Patator | SSH-Patator | Web Attack Brute Force",

        "UNSW_Candidate_Labels":
            "",

        "Calibration_Source_Minimum":
            "AT_LEAST_ONE_REAL_DATASET",

        "Enabled":
            True
    },

    {
        "Synthetic_Family":
            "FUZZING",

        "Primary_Concept":
            "Fuzzing / malformed-input behaviour",

        "CIC_Candidate_Labels":
            "",

        "UNSW_Candidate_Labels":
            "fuzzers",

        "Calibration_Source_Minimum":
            "AT_LEAST_ONE_REAL_DATASET",

        "Enabled":
            True
    }
]


family_spec = pd.DataFrame(
    family_spec_rows
)


print(
    family_spec.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0F — PRECURSOR REGIMES
#
# Exact counts per attack family:
#
#     20  NO_PRECURSOR
#     40  SHORT_WEAK
#     40  SHORT_STRONG
#     50  GRADUAL_WEAK
#     30  GRADUAL_STRONG
#     20  BURSTY
#
# TOTAL = 200 episodes / family
#
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0F — PRECURSOR REGIME SPECIFICATION")
print("=" * 120)


precursor_specs = [

    {
        "Precursor_Regime":
            "NO_PRECURSOR",

        "Share_Per_Family":
            0.10,

        "Episodes_Per_Family":
            20,

        "Precursor_Length_Min":
            0,

        "Precursor_Length_Max":
            0,

        "Max_PreOnset_Effect_Alpha":
            0.00,

        "Temporal_Profile":
            "NONE",

        "Purpose":
            "Negative control: no observable synthetic precursor before onset."
    },

    {
        "Precursor_Regime":
            "SHORT_WEAK",

        "Share_Per_Family":
            0.20,

        "Episodes_Per_Family":
            40,

        "Precursor_Length_Min":
            3,

        "Precursor_Length_Max":
            5,

        "Max_PreOnset_Effect_Alpha":
            0.35,

        "Temporal_Profile":
            "SMOOTH_RAMP",

        "Purpose":
            "Weak short-horizon precursor."
    },

    {
        "Precursor_Regime":
            "SHORT_STRONG",

        "Share_Per_Family":
            0.20,

        "Episodes_Per_Family":
            40,

        "Precursor_Length_Min":
            3,

        "Precursor_Length_Max":
            5,

        "Max_PreOnset_Effect_Alpha":
            0.75,

        "Temporal_Profile":
            "SMOOTH_RAMP",

        "Purpose":
            "Strong short-horizon precursor."
    },

    {
        "Precursor_Regime":
            "GRADUAL_WEAK",

        "Share_Per_Family":
            0.25,

        "Episodes_Per_Family":
            50,

        "Precursor_Length_Min":
            8,

        "Precursor_Length_Max":
            20,

        "Max_PreOnset_Effect_Alpha":
            0.45,

        "Temporal_Profile":
            "SMOOTH_RAMP",

        "Purpose":
            "Longer but subtle pre-attack evolution."
    },

    {
        "Precursor_Regime":
            "GRADUAL_STRONG",

        "Share_Per_Family":
            0.15,

        "Episodes_Per_Family":
            30,

        "Precursor_Length_Min":
            8,

        "Precursor_Length_Max":
            20,

        "Max_PreOnset_Effect_Alpha":
            0.85,

        "Temporal_Profile":
            "SMOOTH_RAMP",

        "Purpose":
            "Longer clear pre-attack evolution."
    },

    {
        "Precursor_Regime":
            "BURSTY",

        "Share_Per_Family":
            0.10,

        "Episodes_Per_Family":
            20,

        "Precursor_Length_Min":
            8,

        "Precursor_Length_Max":
            20,

        "Max_PreOnset_Effect_Alpha":
            0.65,

        "Temporal_Profile":
            "INTERMITTENT",

        "Purpose":
            "Intermittent precursor rather than a monotonic ramp."
    }
]


precursor_spec = pd.DataFrame(
    precursor_specs
)


print(
    precursor_spec.to_string(
        index=False
    )
)


assert (
    precursor_spec[
        "Episodes_Per_Family"
    ].sum()
    ==
    EPISODES_PER_ATTACK_FAMILY
)


assert np.isclose(
    precursor_spec[
        "Share_Per_Family"
    ].sum(),
    1.0
)


# ==================================================================================================
# 11.0G — EXACT PRECURSOR REGIME DISTRIBUTION PER SPLIT
#
# This prevents a future situation where TEST contains a regime TRAIN never saw.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0G — PRECURSOR REGIME × SPLIT CONTRACT")
print("=" * 120)


REGIME_SPLIT_COUNTS = {

    "NO_PRECURSOR": {
        "TRAIN": 14,
        "VALIDATION": 3,
        "TEST": 3
    },

    "SHORT_WEAK": {
        "TRAIN": 28,
        "VALIDATION": 6,
        "TEST": 6
    },

    "SHORT_STRONG": {
        "TRAIN": 28,
        "VALIDATION": 6,
        "TEST": 6
    },

    "GRADUAL_WEAK": {
        "TRAIN": 35,
        "VALIDATION": 8,
        "TEST": 7
    },

    "GRADUAL_STRONG": {
        "TRAIN": 21,
        "VALIDATION": 4,
        "TEST": 5
    },

    "BURSTY": {
        "TRAIN": 14,
        "VALIDATION": 3,
        "TEST": 3
    }
}


regime_split_rows = []


for regime, split_counts in REGIME_SPLIT_COUNTS.items():

    for split_name, count in split_counts.items():

        regime_split_rows.append(
            {
                "Precursor_Regime":
                    regime,

                "Split":
                    split_name,

                "Episodes_Per_Attack_Family":
                    count
            }
        )


regime_split_contract = pd.DataFrame(
    regime_split_rows
)


print(
    regime_split_contract.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0H — TEMPORAL / TIMESTAMP CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0H — TEMPORAL & TIMESTAMP CONTRACT")
print("=" * 120)


temporal_spec = OrderedDict({

    "timestamp_timezone":
        "UTC",

    "timestamp_frequency_seconds":
        60,

    "episode_timestamp_start":
        "2025-01-01T00:00:00+00:00",

    "episode_safety_gap_minutes":
        60,

    "episode_length_min_minutes":
        100,

    "episode_length_max_minutes":
        140,

    "attack_onset_minute_min":
        60,

    "attack_onset_minute_max":
        85,

    "attack_duration_min_minutes":
        10,

    "attack_duration_max_minutes":
        25,

    "recovery_duration_min_minutes":
        5,

    "recovery_duration_max_minutes":
        15,

    "minimum_pre_onset_history_minutes":
        50,

    "one_attack_onset_per_attack_episode":
        True,

    "benign_episode_attack_onsets":
        0,

    "allow_missing_minutes":
        False,

    "allow_duplicate_episode_minutes":
        False,

    "allow_episode_timestamp_overlap":
        False,

    "allow_sequence_cross_episode":
        False,

    "temporal_segment_id_equals_episode_id":
        True,

    "absolute_timestamp_allowed_as_model_feature":
        False
})


for key, value in temporal_spec.items():
    print(f"{key}: {value}")


# ==================================================================================================
# 11.0I — EPISODE GENERATION CONSTRAINTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0I — EPISODE CONSTRAINTS")
print("=" * 120)


episode_constraints = pd.DataFrame([

    {
        "Constraint":
            "Episode length",

        "Rule":
            "100–140 minutes"
    },

    {
        "Constraint":
            "Attack onset",

        "Rule":
            "Minute 60–85 in attack episodes"
    },

    {
        "Constraint":
            "Attack duration",

        "Rule":
            "10–25 minutes"
    },

    {
        "Constraint":
            "Recovery duration",

        "Rule":
            "5–15 minutes"
    },

    {
        "Constraint":
            "Pre-onset history",

        "Rule":
            ">=50 observed minutes before attack onset"
    },

    {
        "Constraint":
            "Attack count",

        "Rule":
            "Exactly one new attack onset per attack episode"
    },

    {
        "Constraint":
            "Benign-only episode",

        "Rule":
            "No attack onset at any minute"
    },

    {
        "Constraint":
            "Forecast eligibility in attack episode",

        "Rule":
            "Only minutes BEFORE the attack onset"
    },

    {
        "Constraint":
            "Post-attack minutes",

        "Rule":
            "Retained for realism/audit but excluded from onset-forecast sample eligibility"
    },

    {
        "Constraint":
            "Temporal continuity",

        "Rule":
            "Exactly 60 seconds between adjacent rows inside each episode"
    }
])


print(
    episode_constraints.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0J — GENERATOR INTERNAL PRIMITIVES
#
# We do NOT generate all final features independently.
#
# This prevents mathematical contradictions.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0J — GENERATOR PRIMITIVES & DERIVED FEATURES")
print("=" * 120)


INTERNAL_GENERATOR_PRIMITIVES = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max_Excess",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum"
]


FINAL_PREDICTOR_COLUMNS = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Total_Packets",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum",

    "Total_Bytes",

    "Packets_Per_Second",

    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",

    "Forward_Backward_Byte_Ratio"
]


derived_feature_rows = [

    {
        "Feature":
            "Flow_Duration_Max",

        "Formula":
            "Flow_Duration_Mean + Flow_Duration_Max_Excess"
    },

    {
        "Feature":
            "Total_Packets",

        "Formula":
            "Forward_Packets_Sum + Backward_Packets_Sum"
    },

    {
        "Feature":
            "Total_Bytes",

        "Formula":
            "Forward_Bytes_Sum + Backward_Bytes_Sum"
    },

    {
        "Feature":
            "Packets_Per_Second",

        "Formula":
            "Total_Packets / 60.0"
    },

    {
        "Feature":
            "Bytes_Per_Second",

        "Formula":
            "Total_Bytes / 60.0"
    },

    {
        "Feature":
            "Forward_Backward_Packet_Ratio",

        "Formula":
            "Forward_Packets_Sum / (Backward_Packets_Sum + 1.0)"
    },

    {
        "Feature":
            "Forward_Backward_Byte_Ratio",

        "Formula":
            "Forward_Bytes_Sum / (Backward_Bytes_Sum + 1.0)"
    }
]


derived_feature_spec = pd.DataFrame(
    derived_feature_rows
)


print("Generator primitives:")

for x in INTERNAL_GENERATOR_PRIMITIVES:
    print("  ✓", x)


print("\nFinal predictor whitelist:")

for x in FINAL_PREDICTOR_COLUMNS:
    print("  ✓", x)


print("\nDerived features:")

print(
    derived_feature_spec.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0K — FEATURE DOMAIN CONSTRAINTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0K — FEATURE DOMAIN CONSTRAINTS")
print("=" * 120)


feature_constraints = pd.DataFrame([

    {
        "Feature_Group":
            "Flow_Count",

        "Constraint":
            "integer >= 1"
    },

    {
        "Feature_Group":
            "Packet sums",

        "Constraint":
            "integer >= 0"
    },

    {
        "Feature_Group":
            "Byte sums",

        "Constraint":
            "integer >= 0"
    },

    {
        "Feature_Group":
            "Duration statistics",

        "Constraint":
            "finite float >= 0"
    },

    {
        "Feature_Group":
            "Flow_Duration_Max",

        "Constraint":
            ">= Flow_Duration_Mean"
    },

    {
        "Feature_Group":
            "Rates",

        "Constraint":
            "finite float >= 0"
    },

    {
        "Feature_Group":
            "Directional ratios",

        "Constraint":
            "finite float >= 0"
    },

    {
        "Feature_Group":
            "All predictors",

        "Constraint":
            "no NaN"
    },

    {
        "Feature_Group":
            "All predictors",

        "Constraint":
            "no +/-Inf"
    }
])


print(
    feature_constraints.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0L — CALIBRATION STRATEGY
#
# We do not invent arbitrary feature ranges.
#
# Real CIC/UNSW data provides:
#
#   BENIGN baseline distribution
#   family-specific effect estimates
#   realistic covariance/ranges
#
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0L — REAL-DATA CALIBRATION STRATEGY")
print("=" * 120)


calibration_strategy = OrderedDict({

    "real_sources": [
        "CIC_IDS2017",
        "UNSW_NB15"
    ],

    "common_feature_schema":
        FINAL_PREDICTOR_COLUMNS,

    "baseline_generation":
        "MOVING_BLOCK_BOOTSTRAP_FROM_REAL_BENIGN_MINUTES",

    "benign_block_length_minutes":
        5,

    "family_effect_estimation":
        "ROBUST_LOG_SPACE_FAMILY_VS_BENIGN_EFFECT_VECTOR",

    "family_effect_source":
        "AUTHORITATIVE_REAL_FLOW_LABELS",

    "synthetic_transition_space":
        "LOG1P_TRANSFORMED_POSITIVE_FEATURE_SPACE",

    "precursor_interpolation":
        "BENIGN_BASELINE_PLUS_ALPHA_TIMES_FAMILY_EFFECT",

    "cross_feature_dependence":
        "PRESERVE_USING_REAL_BENIGN_BLOCKS_AND_MULTIVARIATE_EFFECTS",

    "episode_load_variation":
        True,

    "episode_load_multiplier_distribution":
        "LOGNORMAL",

    "episode_load_multiplier_median":
        1.0,

    "episode_load_multiplier_sigma":
        0.20,

    "episode_load_multiplier_clip":
        [0.60, 1.60],

    "family_effect_strength_multiplier_range":
        [0.80, 1.20],

    "realism_hard_clip_quantiles":
        [0.001, 0.999],

    "hard_clip_reference":
        "RELEVANT_REAL_BENIGN_PLUS_FAMILY_CALIBRATION_SUPPORT",

    "independent_feature_sampling_allowed":
        False
})


for key, value in calibration_strategy.items():
    print(f"{key}: {value}")


# ==================================================================================================
# 11.0M — CALIBRATION SUPPORT GATES
#
# 11.2 must PASS all gates before 11.3/11.4 generation.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0M — CALIBRATION SUPPORT GATES")
print("=" * 120)


calibration_gates = pd.DataFrame([

    {
        "Gate":
            "Benign minute support",

        "Requirement":
            ">= 500 pooled real benign minute observations"
    },

    {
        "Gate":
            "Per-source benign support",

        "Requirement":
            ">= 100 benign minute observations from each usable source dataset"
    },

    {
        "Gate":
            "Attack-family support",

        "Requirement":
            ">= 1000 authoritative real attack flows for every enabled synthetic family"
    },

    {
        "Gate":
            "Family source support",

        "Requirement":
            "At least one real dataset provides adequate support"
    },

    {
        "Gate":
            "Feature completeness",

        "Requirement":
            "All generator primitives calibratable for every enabled family"
    },

    {
        "Gate":
            "Finite calibration",

        "Requirement":
            "No NaN/Inf calibration parameters"
    },

    {
        "Gate":
            "Label mapping review",

        "Requirement":
            "Candidate real labels explicitly verified before family calibration"
    },

    {
        "Gate":
            "No unsupported family",

        "Requirement":
            "If a family fails calibration, generation halts; do not fake its parameters"
    }
])


print(
    calibration_gates.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0N — FORECASTING TARGET CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0N — FORECASTING TARGET CONTRACT")
print("=" * 120)


PRIMARY_HISTORY_WINDOW_MINUTES = 10

CANDIDATE_HISTORY_WINDOWS = [
    5,
    10,
    15,
    30
]


FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


PRIMARY_FORECAST_HORIZONS = [
    5,
    10
]


target_spec = OrderedDict({

    "task":
        "NEW_ATTACK_ONSET_FORECASTING",

    "current_state_required":
        "BENIGN_PRE_ONSET_OR_BENIGN_ONLY_EPISODE",

    "attack_active_rows_forecast_eligible":
        False,

    "post_attack_rows_forecast_eligible":
        False,

    "positive_definition":
        "NEW attack onset occurs within (t, t+H]",

    "negative_definition":
        "No new attack onset occurs within (t, t+H]",

    "primary_history_window_minutes":
        PRIMARY_HISTORY_WINDOW_MINUTES,

    "candidate_history_windows_minutes":
        CANDIDATE_HISTORY_WINDOWS,

    "forecast_horizons_minutes":
        FORECAST_HORIZONS,

    "primary_forecast_horizons_minutes":
        PRIMARY_FORECAST_HORIZONS,

    "eta_target":
        "Minutes_To_Onset",

    "attack_family_target":
        "Future_Attack_Family",

    "event_level_identity":
        "Episode_ID",

    "allow_sequence_cross_episode":
        False
})


for key, value in target_spec.items():
    print(f"{key}: {value}")


# ==================================================================================================
# 11.0O — OUTPUT COLUMN ROLES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0O — COLUMN ROLE / LEAKAGE CONTRACT")
print("=" * 120)


METADATA_COLUMNS = [

    "Dataset_Version",

    "Episode_ID",

    "Temporal_Segment_ID",

    "Split",

    "Timestamp_UTC",

    "Minute_Index",

    "Episode_Type",

    "Generator_Seed",

    "Calibration_Profile_ID"
]


GENERATOR_AUDIT_COLUMNS = [

    "Precursor_Regime",

    "Precursor_Length_Minutes",

    "Precursor_Alpha",

    "Episode_Load_Multiplier",

    "Family_Effect_Strength_Multiplier"
]


TARGET_COLUMNS = [

    "Attack_State",

    "Attack_Family",

    "Future_Attack_Family",

    "Attack_Onset_Timestamp_UTC",

    "Attack_Onset_Minute_Index",

    "Minutes_To_Onset",

    "Forecast_Eligible",

    "Y_Onset_Within_1m",

    "Y_Onset_Within_5m",

    "Y_Onset_Within_10m",

    "Y_Onset_Within_15m",

    "Y_Onset_Within_30m"
]


column_role_rows = []


for column in FINAL_PREDICTOR_COLUMNS:

    column_role_rows.append(
        {
            "Column":
                column,

            "Role":
                "MODEL_PREDICTOR",

            "Allowed_In_X":
                True
        }
    )


for column in METADATA_COLUMNS:

    column_role_rows.append(
        {
            "Column":
                column,

            "Role":
                "METADATA",

            "Allowed_In_X":
                False
        }
    )


for column in GENERATOR_AUDIT_COLUMNS:

    column_role_rows.append(
        {
            "Column":
                column,

            "Role":
                "GENERATOR_AUDIT_METADATA",

            "Allowed_In_X":
                False
        }
    )


for column in TARGET_COLUMNS:

    column_role_rows.append(
        {
            "Column":
                column,

            "Role":
                "TARGET_OR_TARGET_METADATA",

            "Allowed_In_X":
                False
        }
    )


column_roles = pd.DataFrame(
    column_role_rows
)


print(
    column_roles.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0P — SPLIT / LEAKAGE FREEZE
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0P — SPLIT & LEAKAGE FREEZE")
print("=" * 120)


leakage_rules = pd.DataFrame([

    {
        "Rule_ID":
            "L01",

        "Rule":
            "Episode assigned to exactly one split",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L02",

        "Rule":
            "Split assigned BEFORE feature generation",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L03",

        "Rule":
            "No random row split",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L04",

        "Rule":
            "No random minute split",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L05",

        "Rule":
            "No random sequence-window split",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L06",

        "Rule":
            "No sequence may cross Episode_ID",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L07",

        "Rule":
            "Every closed-set test attack family must exist in training",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L08",

        "Rule":
            "Every precursor regime used in test must exist in training",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L09",

        "Rule":
            "Scaler fit on TRAIN only",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L10",

        "Rule":
            "Feature selection fit on TRAIN only",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L11",

        "Rule":
            "Any resampling occurs inside TRAIN only",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L12",

        "Rule":
            "Absolute timestamp cannot enter X",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L13",

        "Rule":
            "Episode_ID / Split / generator metadata cannot enter X",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L14",

        "Rule":
            "Future target information cannot enter X",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L15",

        "Rule":
            "TEST manifest is locked after generation",

        "Mandatory":
            True
    },

    {
        "Rule_ID":
            "L16",

        "Rule":
            "Test episodes may not be regenerated in response to model performance",

        "Mandatory":
            True
    }
])


print(
    leakage_rules.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0Q — REQUIRED MODELING / EVALUATION POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0Q — EVALUATION POLICY")
print("=" * 120)


evaluation_policy = pd.DataFrame([

    {
        "Task":
            "Binary onset forecasting",

        "Metric":
            "PR-AUC",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Binary onset forecasting",

        "Metric":
            "Precision",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Binary onset forecasting",

        "Metric":
            "Recall",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Binary onset forecasting",

        "Metric":
            "F1",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Attack-family forecasting",

        "Metric":
            "Macro F1",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Attack-family forecasting",

        "Metric":
            "Macro Precision",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Attack-family forecasting",

        "Metric":
            "Macro Recall",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "ETA",

        "Metric":
            "MAE",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Operational forecasting",

        "Metric":
            "Event Recall",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Operational forecasting",

        "Metric":
            "Event Precision",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Operational forecasting",

        "Metric":
            "First-Warning Lead Time",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Operational forecasting",

        "Metric":
            "Median Lead Time",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Operational forecasting",

        "Metric":
            "False Alarms Per Hour",

        "Priority":
            "PRIMARY"
    },

    {
        "Task":
            "Binary onset forecasting",

        "Metric":
            "ROC-AUC",

        "Priority":
            "SECONDARY"
    },

    {
        "Task":
            "Any",

        "Metric":
            "Raw Accuracy",

        "Priority":
            "SECONDARY_ONLY"
    }
])


print(
    evaluation_policy.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0R — STRUCTURAL VALIDATION CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0R — FUTURE HARD VALIDATION CONTRACT")
print("=" * 120)


hard_validation_contract = [

    "Exactly 1200 unique Episode_ID values",

    "Exactly 1000 attack episodes",

    "Exactly 200 benign-only episodes",

    "Exactly 200 attack episodes per enabled attack family",

    "Exactly 840 TRAIN episodes",

    "Exactly 180 VALIDATION episodes",

    "Exactly 180 TEST episodes",

    "Every attack family appears in TRAIN, VALIDATION and TEST",

    "Every attack family has exact 140/30/30 split allocation",

    "Every precursor regime appears in TRAIN, VALIDATION and TEST",

    "No Episode_ID appears in more than one split",

    "Every Episode_ID has exactly one Generator_Seed",

    "Generator seeds unique across episodes",

    "Timestamp timezone is UTC",

    "Exactly 60 seconds between adjacent rows of an episode",

    "No missing minute inside an episode",

    "No duplicate Minute_Index inside an episode",

    "No duplicate Timestamp_UTC inside an episode",

    "No overlap between episode timestamp ranges",

    "Temporal_Segment_ID equals Episode_ID",

    "Exactly one attack onset in every attack episode",

    "Zero attack onsets in every benign-only episode",

    "At least 50 pre-onset observed minutes in every attack episode",

    "Attack-active rows are never onset-forecast eligible",

    "Post-attack rows are never onset-forecast eligible",

    "Forecast labels recompute exactly from Attack_Onset_Timestamp_UTC",

    "No target column appears in predictor whitelist",

    "No metadata column appears in predictor whitelist",

    "No generator-audit column appears in predictor whitelist",

    "All 14 predictor columns exist",

    "No NaN predictor values",

    "No infinite predictor values",

    "No negative count/packet/byte/rate values",

    "Total_Packets equals forward + backward packet sums",

    "Total_Bytes equals forward + backward byte sums",

    "Packets_Per_Second equals Total_Packets / 60",

    "Bytes_Per_Second equals Total_Bytes / 60",

    "Directional ratios reproduce from directional sums",

    "Flow_Duration_Max >= Flow_Duration_Mean",

    "Sequences never cross Episode_ID",

    "Closed-set TEST family set is a subset of TRAIN family set",

    "Test manifest remains unchanged after dataset freeze"
]


hard_validation_df = pd.DataFrame(
    {
        "Validation_ID": [
            f"V{i:02d}"
            for i in range(
                1,
                len(hard_validation_contract) + 1
            )
        ],

        "Required_Check":
            hard_validation_contract,

        "Mandatory":
            True
    }
)


print(
    hard_validation_df.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0S — REALISM AUDIT CONTRACT
#
# Structural validation is binary.
#
# Realism validation is diagnostic and compares synthetic traffic
# back against CIC/UNSW calibration distributions.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0S — REALISM AUDIT CONTRACT")
print("=" * 120)


realism_contract = pd.DataFrame([

    {
        "Audit":
            "Feature quantile comparison",

        "Requirement":
            "Compare synthetic vs real q01/q05/q25/q50/q75/q95/q99"
    },

    {
        "Audit":
            "Feature distribution comparison",

        "Requirement":
            "Synthetic values must remain within calibrated support except explicitly allowed tails"
    },

    {
        "Audit":
            "Correlation comparison",

        "Requirement":
            "Compare real vs synthetic predictor correlation matrices"
    },

    {
        "Audit":
            "Family direction-of-effect",

        "Requirement":
            "Synthetic family effect direction must agree with calibrated real family effect"
    },

    {
        "Audit":
            "Benign overlap",

        "Requirement":
            "Synthetic benign episodes must overlap substantially with real benign calibration support"
    },

    {
        "Audit":
            "Episode variability",

        "Requirement":
            "Episodes must not be near-duplicates"
    },

    {
        "Audit":
            "Split realism",

        "Requirement":
            "TRAIN/VALIDATION/TEST generated from same frozen simulator family without copying episodes"
    },

    {
        "Audit":
            "No perfect synthetic shortcut",

        "Requirement":
            "No metadata/generator field may trivially reveal target or family"
    }
])


print(
    realism_contract.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.0T — BUILD MASTER MACHINE-READABLE CONFIG
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0T — BUILD MASTER CONFIG")
print("=" * 120)


master_config = OrderedDict({

    "dataset_identity": {

        "name":
            DATASET_NAME,

        "short_name":
            DATASET_SHORT_NAME,

        "version":
            DATASET_VERSION,

        "master_seed":
            MASTER_SEED,

        "dataset_claim":
            DATASET_CLAIM,

        "prohibited_claim":
            PROHIBITED_CLAIM
    },


    "episode_design": {

        "total_episodes":
            TOTAL_EPISODES,

        "attack_episodes":
            ATTACK_EPISODES,

        "benign_only_episodes":
            BENIGN_EPISODES,

        "attack_families":
            ATTACK_FAMILIES,

        "episodes_per_attack_family":
            EPISODES_PER_ATTACK_FAMILY,

        "split_counts_per_class":
            SPLIT_COUNTS_PER_CLASS,

        "expected_split_totals":
            EXPECTED_SPLIT_TOTALS
    },


    "temporal_design":
        temporal_spec,


    "precursor_regimes":

        {
            row["Precursor_Regime"]: {

                "share_per_family":
                    float(
                        row["Share_Per_Family"]
                    ),

                "episodes_per_family":
                    int(
                        row["Episodes_Per_Family"]
                    ),

                "length_min":
                    int(
                        row["Precursor_Length_Min"]
                    ),

                "length_max":
                    int(
                        row["Precursor_Length_Max"]
                    ),

                "max_preonset_alpha":
                    float(
                        row["Max_PreOnset_Effect_Alpha"]
                    ),

                "temporal_profile":
                    row["Temporal_Profile"]
            }

            for _, row in precursor_spec.iterrows()
        },


    "precursor_split_contract":
        REGIME_SPLIT_COUNTS,


    "features": {

        "internal_generator_primitives":
            INTERNAL_GENERATOR_PRIMITIVES,

        "final_predictor_whitelist":
            FINAL_PREDICTOR_COLUMNS,

        "predictor_count":
            len(
                FINAL_PREDICTOR_COLUMNS
            ),

        "metadata_columns":
            METADATA_COLUMNS,

        "generator_audit_columns":
            GENERATOR_AUDIT_COLUMNS,

        "target_columns":
            TARGET_COLUMNS
    },


    "calibration_strategy":
        calibration_strategy,


    "forecast_target_design":
        target_spec,


    "split_policy": {

        "split_before_generation":
            True,

        "random_row_split":
            False,

        "random_minute_split":
            False,

        "random_window_split":
            False,

        "episode_isolation_required":
            True,

        "sequence_cross_episode_allowed":
            False,

        "same_family_coverage_closed_set_required":
            True,

        "same_precursor_regime_coverage_required":
            True,

        "fit_scaler_on_train_only":
            True,

        "feature_selection_on_train_only":
            True,

        "resampling_training_only":
            True,

        "test_manifest_locked":
            True,

        "regenerate_test_after_modeling":
            False
    },


    "evaluation": {

        "binary_primary_metrics": [
            "PR_AUC",
            "Precision",
            "Recall",
            "F1"
        ],

        "family_primary_metrics": [
            "Macro_F1",
            "Macro_Precision",
            "Macro_Recall"
        ],

        "eta_primary_metrics": [
            "MAE"
        ],

        "operational_primary_metrics": [
            "Event_Recall",
            "Event_Precision",
            "First_Warning_Lead_Time",
            "Median_Lead_Time",
            "False_Alarms_Per_Hour"
        ]
    }
})


# ==================================================================================================
# 11.0U — VALIDATE SPECIFICATION ITSELF
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0U — SPECIFICATION SELF-VALIDATION")
print("=" * 120)


validation_rows = []


def check(name, passed, details):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                details
        }
    )


# --------------------------------------------------------------------------------------------------
# Episode counts
# --------------------------------------------------------------------------------------------------

check(
    "Total episode arithmetic",
    (
        ATTACK_EPISODES
        +
        BENIGN_EPISODES
        ==
        TOTAL_EPISODES
    ),
    f"{ATTACK_EPISODES} + {BENIGN_EPISODES} = {TOTAL_EPISODES}"
)


check(
    "Attack family arithmetic",
    (
        len(ATTACK_FAMILIES)
        *
        EPISODES_PER_ATTACK_FAMILY
        ==
        ATTACK_EPISODES
    ),
    (
        f"{len(ATTACK_FAMILIES)} × "
        f"{EPISODES_PER_ATTACK_FAMILY} = "
        f"{ATTACK_EPISODES}"
    )
)


# --------------------------------------------------------------------------------------------------
# Split totals
# --------------------------------------------------------------------------------------------------

for split_name, expected_total in EXPECTED_SPLIT_TOTALS.items():

    observed_total = int(
        split_totals[
            split_name
        ]
    )

    check(
        f"{split_name} total",
        observed_total
        ==
        expected_total,
        f"{observed_total} expected {expected_total}"
    )


check(
    "All split totals sum to dataset",
    int(
        split_totals.sum()
    )
    ==
    TOTAL_EPISODES,
    f"{int(split_totals.sum())} episodes"
)


# --------------------------------------------------------------------------------------------------
# Every attack family exact 140/30/30
# --------------------------------------------------------------------------------------------------

for family in ATTACK_FAMILIES:

    g = split_allocation[
        (
            split_allocation[
                "Episode_Type"
            ]
            ==
            "ATTACK"
        )
        &
        (
            split_allocation[
                "Attack_Family"
            ]
            ==
            family
        )
    ]


    observed = dict(
        zip(
            g["Split"],
            g["Episode_Count"]
        )
    )


    check(
        f"{family} exact split coverage",
        observed
        ==
        SPLIT_COUNTS_PER_CLASS,
        str(
            observed
        )
    )


# --------------------------------------------------------------------------------------------------
# Benign split
# --------------------------------------------------------------------------------------------------

benign_split = split_allocation[
    split_allocation[
        "Episode_Type"
    ]
    ==
    "BENIGN_ONLY"
]


observed_benign = dict(
    zip(
        benign_split["Split"],
        benign_split["Episode_Count"]
    )
)


check(
    "Benign exact split coverage",
    observed_benign
    ==
    SPLIT_COUNTS_PER_CLASS,
    str(
        observed_benign
    )
)


# --------------------------------------------------------------------------------------------------
# Regime totals
# --------------------------------------------------------------------------------------------------

for regime, split_counts in REGIME_SPLIT_COUNTS.items():

    total = sum(
        split_counts.values()
    )


    expected = int(
        precursor_spec.loc[
            precursor_spec[
                "Precursor_Regime"
            ]
            ==
            regime,
            "Episodes_Per_Family"
        ].iloc[0]
    )


    check(
        f"{regime} regime total",
        total
        ==
        expected,
        f"{total} expected {expected}"
    )


# --------------------------------------------------------------------------------------------------
# Per-family regime split totals
# --------------------------------------------------------------------------------------------------

for split_name, expected in SPLIT_COUNTS_PER_CLASS.items():

    observed = sum(

        regime_counts[
            split_name
        ]

        for regime_counts
        in REGIME_SPLIT_COUNTS.values()
    )


    check(
        f"Precursor regime allocation sums to {split_name} family total",
        observed
        ==
        expected,
        f"{observed} expected {expected}"
    )


# --------------------------------------------------------------------------------------------------
# Feature leakage checks
# --------------------------------------------------------------------------------------------------

feature_set = set(
    FINAL_PREDICTOR_COLUMNS
)

metadata_set = set(
    METADATA_COLUMNS
)

audit_set = set(
    GENERATOR_AUDIT_COLUMNS
)

target_set = set(
    TARGET_COLUMNS
)


check(
    "Predictor vs metadata disjoint",
    len(
        feature_set
        &
        metadata_set
    )
    ==
    0,
    str(
        feature_set
        &
        metadata_set
    )
)


check(
    "Predictor vs generator audit disjoint",
    len(
        feature_set
        &
        audit_set
    )
    ==
    0,
    str(
        feature_set
        &
        audit_set
    )
)


check(
    "Predictor vs target disjoint",
    len(
        feature_set
        &
        target_set
    )
    ==
    0,
    str(
        feature_set
        &
        target_set
    )
)


check(
    "Exactly 14 final predictors",
    len(
        FINAL_PREDICTOR_COLUMNS
    )
    ==
    14,
    str(
        len(
            FINAL_PREDICTOR_COLUMNS
        )
    )
)


check(
    "Predictor names unique",
    len(
        FINAL_PREDICTOR_COLUMNS
    )
    ==
    len(
        set(
            FINAL_PREDICTOR_COLUMNS
        )
    ),
    "No duplicate predictor names"
)


# --------------------------------------------------------------------------------------------------
# Temporal checks
# --------------------------------------------------------------------------------------------------

check(
    "Minute frequency exactly 60 seconds",
    temporal_spec[
        "timestamp_frequency_seconds"
    ]
    ==
    60,
    str(
        temporal_spec[
            "timestamp_frequency_seconds"
        ]
    )
)


check(
    "UTC timestamp policy",
    temporal_spec[
        "timestamp_timezone"
    ]
    ==
    "UTC",
    temporal_spec[
        "timestamp_timezone"
    ]
)


check(
    "Minimum onset leaves >=50 minutes history",
    temporal_spec[
        "attack_onset_minute_min"
    ]
    >=
    temporal_spec[
        "minimum_pre_onset_history_minutes"
    ],
    (
        f"earliest onset="
        f"{temporal_spec['attack_onset_minute_min']}"
    )
)


check(
    "Maximum required attack timeline fits max episode",
    (
        temporal_spec[
            "attack_onset_minute_max"
        ]
        +
        temporal_spec[
            "attack_duration_max_minutes"
        ]
        +
        temporal_spec[
            "recovery_duration_max_minutes"
        ]
    )
    <=
    temporal_spec[
        "episode_length_max_minutes"
    ],
    (
        f"{temporal_spec['attack_onset_minute_max']} + "
        f"{temporal_spec['attack_duration_max_minutes']} + "
        f"{temporal_spec['recovery_duration_max_minutes']} <= "
        f"{temporal_spec['episode_length_max_minutes']}"
    )
)


# --------------------------------------------------------------------------------------------------
# Family config
# --------------------------------------------------------------------------------------------------

check(
    "Family specification matches taxonomy",
    set(
        family_spec[
            "Synthetic_Family"
        ]
    )
    ==
    set(
        ATTACK_FAMILIES
    ),
    str(
        sorted(
            family_spec[
                "Synthetic_Family"
            ].tolist()
        )
    )
)


check(
    "All synthetic families initially enabled",
    bool(
        family_spec[
            "Enabled"
        ].all()
    ),
    f"{int(family_spec['Enabled'].sum())}/{len(family_spec)} enabled"
)


# --------------------------------------------------------------------------------------------------
# Closed-set class coverage
# --------------------------------------------------------------------------------------------------

for split_name in [
    "TRAIN",
    "VALIDATION",
    "TEST"
]:

    split_families = set(

        split_allocation.loc[
            (
                split_allocation[
                    "Split"
                ]
                ==
                split_name
            )
            &
            (
                split_allocation[
                    "Episode_Type"
                ]
                ==
                "ATTACK"
            ),
            "Attack_Family"
        ]
    )


    check(
        f"{split_name} contains all attack families",
        split_families
        ==
        set(
            ATTACK_FAMILIES
        ),
        str(
            sorted(
                split_families
            )
        )
    )


spec_validation = pd.DataFrame(
    validation_rows
)


print(
    spec_validation.to_string(
        index=False
    )
)


ALL_SPEC_CHECKS_PASS = bool(
    spec_validation[
        "Passed"
    ].all()
)


# ==================================================================================================
# 11.0V — SAVE SPECIFICATION ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0V — SAVE SPECIFICATION ARTIFACTS")
print("=" * 120)


CONFIG_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config.json"
)


FAMILY_SPEC_FILE = (
    SPEC_DIR
    / "11_0_attack_family_spec.csv"
)


SPLIT_CONTRACT_FILE = (
    SPEC_DIR
    / "11_0_split_allocation_contract.csv"
)


PRECURSOR_FILE = (
    SPEC_DIR
    / "11_0_precursor_regime_spec.csv"
)


REGIME_SPLIT_FILE = (
    SPEC_DIR
    / "11_0_precursor_split_contract.csv"
)


DERIVED_FEATURE_FILE = (
    SPEC_DIR
    / "11_0_derived_feature_spec.csv"
)


FEATURE_CONSTRAINT_FILE = (
    SPEC_DIR
    / "11_0_feature_constraints.csv"
)


CALIBRATION_GATE_FILE = (
    SPEC_DIR
    / "11_0_calibration_support_gates.csv"
)


COLUMN_ROLE_FILE = (
    SPEC_DIR
    / "11_0_column_roles.csv"
)


LEAKAGE_RULE_FILE = (
    SPEC_DIR
    / "11_0_leakage_rules.csv"
)


EVALUATION_FILE = (
    SPEC_DIR
    / "11_0_evaluation_policy.csv"
)


HARD_VALIDATION_FILE = (
    SPEC_DIR
    / "11_0_hard_validation_contract.csv"
)


REALISM_FILE = (
    SPEC_DIR
    / "11_0_realism_audit_contract.csv"
)


EPISODE_CONSTRAINT_FILE = (
    SPEC_DIR
    / "11_0_episode_constraints.csv"
)


SPEC_VALIDATION_FILE = (
    SPEC_DIR
    / "11_0_spec_validation.csv"
)


with open(
    CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        master_config,
        f,
        indent=4,
        ensure_ascii=False
    )


family_spec.to_csv(
    FAMILY_SPEC_FILE,
    index=False
)


split_allocation.to_csv(
    SPLIT_CONTRACT_FILE,
    index=False
)


precursor_spec.to_csv(
    PRECURSOR_FILE,
    index=False
)


regime_split_contract.to_csv(
    REGIME_SPLIT_FILE,
    index=False
)


derived_feature_spec.to_csv(
    DERIVED_FEATURE_FILE,
    index=False
)


feature_constraints.to_csv(
    FEATURE_CONSTRAINT_FILE,
    index=False
)


calibration_gates.to_csv(
    CALIBRATION_GATE_FILE,
    index=False
)


column_roles.to_csv(
    COLUMN_ROLE_FILE,
    index=False
)


leakage_rules.to_csv(
    LEAKAGE_RULE_FILE,
    index=False
)


evaluation_policy.to_csv(
    EVALUATION_FILE,
    index=False
)


hard_validation_df.to_csv(
    HARD_VALIDATION_FILE,
    index=False
)


realism_contract.to_csv(
    REALISM_FILE,
    index=False
)


episode_constraints.to_csv(
    EPISODE_CONSTRAINT_FILE,
    index=False
)


spec_validation.to_csv(
    SPEC_VALIDATION_FILE,
    index=False
)


saved_files = [

    CONFIG_FILE,
    FAMILY_SPEC_FILE,
    SPLIT_CONTRACT_FILE,
    PRECURSOR_FILE,
    REGIME_SPLIT_FILE,
    DERIVED_FEATURE_FILE,
    FEATURE_CONSTRAINT_FILE,
    CALIBRATION_GATE_FILE,
    COLUMN_ROLE_FILE,
    LEAKAGE_RULE_FILE,
    EVALUATION_FILE,
    HARD_VALIDATION_FILE,
    REALISM_FILE,
    EPISODE_CONSTRAINT_FILE,
    SPEC_VALIDATION_FILE
]


for file in saved_files:
    print("✓", file)


# ==================================================================================================
# 11.0W — FREEZE CONFIG HASH
#
# The hash allows us to know later whether the specification changed.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0W — SPECIFICATION HASH")
print("=" * 120)


with open(
    CONFIG_FILE,
    "rb"
) as f:

    config_bytes = f.read()


CONFIG_SHA256 = hashlib.sha256(
    config_bytes
).hexdigest()


HASH_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config_SHA256.txt"
)


with open(
    HASH_FILE,
    "w"
) as f:

    f.write(
        CONFIG_SHA256
        +
        "\n"
    )


print(
    "Config SHA256:"
)

print(
    CONFIG_SHA256
)


print(
    "\n✓ Hash saved:"
)

print(
    HASH_FILE
)


# ==================================================================================================
# 11.0X — BUILD SPEC MANIFEST
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0X — SPECIFICATION MANIFEST")
print("=" * 120)


spec_manifest = pd.DataFrame({

    "Metric": [

        "Dataset_Name",

        "Version",

        "Total_Episodes",

        "Attack_Episodes",

        "Benign_Episodes",

        "Attack_Families",

        "Episodes_Per_Attack_Family",

        "Train_Episodes",

        "Validation_Episodes",

        "Test_Episodes",

        "Predictor_Count",

        "Temporal_Resolution_Sec",

        "Episode_Min_Length",

        "Episode_Max_Length",

        "Earliest_Attack_Onset_Min",

        "Latest_Attack_Onset_Min",

        "Minimum_PreOnset_History_Min",

        "Primary_History_Window_Min",

        "Primary_Horizon_1",

        "Primary_Horizon_2",

        "Master_Seed",

        "Spec_Checks",

        "Spec_Checks_Passed",

        "Spec_Pass",

        "Config_SHA256"
    ],


    "Value": [

        DATASET_SHORT_NAME,

        DATASET_VERSION,

        TOTAL_EPISODES,

        ATTACK_EPISODES,

        BENIGN_EPISODES,

        len(
            ATTACK_FAMILIES
        ),

        EPISODES_PER_ATTACK_FAMILY,

        EXPECTED_SPLIT_TOTALS[
            "TRAIN"
        ],

        EXPECTED_SPLIT_TOTALS[
            "VALIDATION"
        ],

        EXPECTED_SPLIT_TOTALS[
            "TEST"
        ],

        len(
            FINAL_PREDICTOR_COLUMNS
        ),

        temporal_spec[
            "timestamp_frequency_seconds"
        ],

        temporal_spec[
            "episode_length_min_minutes"
        ],

        temporal_spec[
            "episode_length_max_minutes"
        ],

        temporal_spec[
            "attack_onset_minute_min"
        ],

        temporal_spec[
            "attack_onset_minute_max"
        ],

        temporal_spec[
            "minimum_pre_onset_history_minutes"
        ],

        PRIMARY_HISTORY_WINDOW_MINUTES,

        PRIMARY_FORECAST_HORIZONS[0],

        PRIMARY_FORECAST_HORIZONS[1],

        MASTER_SEED,

        len(
            spec_validation
        ),

        int(
            spec_validation[
                "Passed"
            ].sum()
        ),

        ALL_SPEC_CHECKS_PASS,

        CONFIG_SHA256
    ]
})


print(
    spec_manifest.to_string(
        index=False
    )
)


SPEC_MANIFEST_FILE = (
    SPEC_DIR
    / "11_0_spec_manifest.csv"
)


spec_manifest.to_csv(
    SPEC_MANIFEST_FILE,
    index=False
)


print(
    "\n✓",
    SPEC_MANIFEST_FILE
)


# ==================================================================================================
# 11.0Y — HARD STOP IF SPECIFICATION IS INVALID
# ==================================================================================================

print("\n" + "=" * 120)
print("11.0Y — FINAL SPECIFICATION DECISION")
print("=" * 120)


if not ALL_SPEC_CHECKS_PASS:

    failed = spec_validation[
        ~spec_validation[
            "Passed"
        ]
    ]


    print(
        "\nFAILED CHECKS:\n"
    )

    print(
        failed.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "Synthetic_NAF_v1 specification failed validation. "
        "DO NOT proceed to calibration."
    )


print(
f"""
✓ SYNTHETIC_NAF_v1 SPECIFICATION: PASS


DATASET SIZE
------------
Total independent episodes : {TOTAL_EPISODES}

Attack episodes            : {ATTACK_EPISODES}

Benign-only episodes       : {BENIGN_EPISODES}


ATTACK FAMILY CONTRACT
----------------------
{", ".join(ATTACK_FAMILIES)}

Each family:

    TRAIN       = 140 independent episodes
    VALIDATION  = 30 independent episodes
    TEST        = 30 independent episodes

Therefore:

    no closed-set TEST family can suddenly appear
    without existing in TRAIN.


GLOBAL SPLIT
------------
TRAIN       : {EXPECTED_SPLIT_TOTALS["TRAIN"]}
VALIDATION  : {EXPECTED_SPLIT_TOTALS["VALIDATION"]}
TEST        : {EXPECTED_SPLIT_TOTALS["TEST"]}


TEMPORAL FORMAT
---------------
Resolution:

    exactly 1 minute

Timezone:

    UTC

Episode duration:

    {temporal_spec["episode_length_min_minutes"]}
    to
    {temporal_spec["episode_length_max_minutes"]}
    minutes

Attack onset:

    minute {temporal_spec["attack_onset_minute_min"]}
    to
    minute {temporal_spec["attack_onset_minute_max"]}

Minimum history before onset:

    {temporal_spec["minimum_pre_onset_history_minutes"]} minutes

Missing minutes:

    NOT ALLOWED

Duplicate minutes:

    NOT ALLOWED

Episode overlap:

    NOT ALLOWED

Sequence crossing episode boundaries:

    NOT ALLOWED


FORECASTING TASK
----------------
At currently pre-onset benign time t:

    predict whether a NEW attack begins in (t, t+H]

Primary horizons:

    5 minutes
    10 minutes

Primary history:

    10 minutes


FEATURE CONTRACT
----------------
Exactly {len(FINAL_PREDICTOR_COLUMNS)} model predictors.

Timestamp is NOT a predictor.

Episode_ID is NOT a predictor.

Split is NOT a predictor.

Precursor regime is NOT a predictor.

Attack labels are NOT predictors.

Generator parameters are NOT predictors.


GENERATION PHILOSOPHY
---------------------
We do NOT independently generate every feature.

We generate a small set of internally consistent primitive variables
and mathematically derive:

    Total_Packets
    Total_Bytes
    Packets_Per_Second
    Bytes_Per_Second
    Directional packet ratio
    Directional byte ratio
    Flow_Duration_Max


CALIBRATION
-----------
CIC-IDS2017 and UNSW-NB15 will calibrate:

    benign traffic distributions
    family effect directions
    realistic feature ranges
    feature dependence
    episode variability

They are NOT being presented as having supplied 1,000 real attack
onset episodes.


CRITICAL NEXT GATE
------------------
NO synthetic generation is allowed yet.

NEXT:

    PHASE 11.1
    REAL CIC + UNSW CALIBRATION EXTRACTION

Then:

    PHASE 11.2
    CALIBRATION VALIDITY AUDIT

Only if every attack family passes 11.2 do we create the
1,200-episode manifest.


SPECIFICATION SHA256
--------------------
{CONFIG_SHA256}
"""
)


print("=" * 120)
print("PHASE 11.0 COMPLETE")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 11.1 — REAL CIC + UNSW CALIBRATION EXTRACTION
# CLEAN REWRITE
# ==================================================================================================
#
# PURPOSE
# -------
# Extract trustworthy REAL calibration information from:
#
#       CIC-IDS2017
#       UNSW-NB15
#
# for the future Synthetic_NAF_v1 generator.
#
#
# IMPORTANT
# ---------
# This phase DOES NOT generate synthetic traffic.
#
# It only creates:
#
#   1. strict real benign minute pool
#   2. disjoint 5-minute benign calibration blocks
#   3. real label → synthetic family mapping audit
#   4. exact real attack-family flow support
#   5. family attack-flow intensity distributions
#   6. flow-level benign/family traffic profiles
#   7. robust family-vs-benign effect vectors
#   8. minute-level real feature quantiles
#   9. calibration provenance + validation
#
#
# CRITICAL DESIGN RULES
# ---------------------
#
# CIC:
#   - WebAttacks CSV contains a KNOWN trailing block of completely empty physical rows.
#   - Those rows are removed BEFORE timestamp/numeric validation.
#   - They are NOT treated as malformed traffic.
#
# UNSW:
#   - Phase 10.12 frozen semantics are authoritative for:
#
#         Final_Attack_State
#         Final_Attack_Family
#
#   - raw Label / attack_cat are used only for auditing.
#
#
# FAMILY CALIBRATION
# ------------------
# Selected synthetic families:
#
#   DOS_DDOS
#   RECON_SCAN
#   EXPLOIT_APP
#   BRUTE_AUTH
#   FUZZING
#
# Other real attacks remain ATTACK for benign exclusion,
# but are not used to calibrate these five synthetic families.
#
#
# NO SYNTHETIC GENERATION IS ALLOWED UNTIL PHASE 11.2 PASSES.
#
# ==================================================================================================


from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import hashlib
import json
import re
import gc
import warnings

warnings.filterwarnings("ignore")


# ==================================================================================================
# 11.1A — PATHS + CONSTANTS
# ==================================================================================================

print("=" * 120)
print("PHASE 11.1 — REAL CIC + UNSW CALIBRATION EXTRACTION")
print("=" * 120)


DRIVE_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

LOCAL_ROOT = Path(
    "/content/network-attack-forecasting"
)


DATASET_ROOT = (
    DRIVE_ROOT
    / "Synthetic_NAF_v1"
)

SPEC_DIR = (
    DATASET_ROOT
    / "spec"
)

CALIBRATION_DIR = (
    DATASET_ROOT
    / "calibration"
)

AUDIT_DIR = (
    DATASET_ROOT
    / "audit"
)


CALIBRATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config.json"
)

CONFIG_HASH_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config_SHA256.txt"
)


CIC_AUDIT_DIR = (
    DRIVE_ROOT
    / "Dataset_Audit_v2"
    / "audit_outputs"
    / "CIC"
)

UNSW_AUDIT_DIR = (
    DRIVE_ROOT
    / "Dataset_Audit_v2"
    / "audit_outputs"
    / "UNSW"
)


CIC_REFERENCE_MINUTE_FILE = (
    CIC_AUDIT_DIR
    / "cic_complete_minute_flow_table.csv"
)


UNSW_REFERENCE_MINUTE_FILE = (
    UNSW_AUDIT_DIR
    / "10_15_UNSW_MINUTE_REPRESENTATION.csv"
)


UNSW_FINAL_SEMANTICS_FILE = (
    UNSW_AUDIT_DIR
    / "10_12_UNSW_FROZEN_FLOW_SEMANTICS.csv"
)


CHUNK_SIZE = 250_000

BENIGN_FLOW_SAMPLE_MODULUS = 20

BENIGN_BLOCK_LENGTH = 5


# Known CIC raw-file audit totals.
#
# These are used only as consistency checks.

EXPECTED_CIC_PHYSICAL_ROWS = 3_119_345

EXPECTED_CIC_EMPTY_TRAILING_ROWS = 288_602

EXPECTED_CIC_REAL_TRAFFIC_ROWS = 2_830_743

EXPECTED_CIC_OBSERVED_MINUTES = 2_454

EXPECTED_CIC_ATTACK_MINUTES = 520


# Known frozen UNSW totals.

EXPECTED_UNSW_FLOWS = 2_540_047

EXPECTED_UNSW_BENIGN_FLOWS = 2_218_764

EXPECTED_UNSW_ATTACK_FLOWS = 321_283

EXPECTED_UNSW_OBSERVED_MINUTES = 1_441


QUANTILES = [
    0.001,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    0.999
]


QUANTILE_NAMES = {

    0.001:
        "q001",

    0.01:
        "q01",

    0.05:
        "q05",

    0.25:
        "q25",

    0.50:
        "q50",

    0.75:
        "q75",

    0.95:
        "q95",

    0.99:
        "q99",

    0.999:
        "q999"
}


def banner(
    text
):

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        text
    )

    print(
        "=" * 120
    )


# ==================================================================================================
# 11.1B — VERIFY FROZEN PHASE 11.0 CONFIG
# ==================================================================================================

banner(
    "11.1B — VERIFY FROZEN PHASE 11.0 SPECIFICATION"
)


if not CONFIG_FILE.exists():

    raise FileNotFoundError(
        f"Missing:\n{CONFIG_FILE}"
    )


if not CONFIG_HASH_FILE.exists():

    raise FileNotFoundError(
        f"Missing:\n{CONFIG_HASH_FILE}"
    )


with open(
    CONFIG_FILE,
    "rb"
) as f:

    config_bytes = f.read()


CURRENT_CONFIG_HASH = hashlib.sha256(
    config_bytes
).hexdigest()


with open(
    CONFIG_HASH_FILE,
    "r"
) as f:

    FROZEN_CONFIG_HASH = (
        f.read()
        .strip()
    )


print(
    "Frozen hash :",
    FROZEN_CONFIG_HASH
)

print(
    "Current hash:",
    CURRENT_CONFIG_HASH
)


if CURRENT_CONFIG_HASH != FROZEN_CONFIG_HASH:

    raise RuntimeError(
        "Phase 11.0 specification changed after freeze."
    )


with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(
        f
    )


FINAL_PREDICTORS = (
    CONFIG[
        "features"
    ][
        "final_predictor_whitelist"
    ]
)


ATTACK_FAMILIES = (
    CONFIG[
        "episode_design"
    ][
        "attack_families"
    ]
)


MASTER_SEED = int(
    CONFIG[
        "dataset_identity"
    ][
        "master_seed"
    ]
)


print(
    "\nPredictors:",
    len(
        FINAL_PREDICTORS
    )
)

print(
    "Attack families:",
    ATTACK_FAMILIES
)

print(
    "\n✓ Phase 11.0 configuration integrity verified"
)


# ==================================================================================================
# 11.1C — FILE DISCOVERY
# ==================================================================================================

banner(
    "11.1C — DISCOVER REAL DATA FILES"
)


SEARCH_ROOTS = [

    p

    for p in [
        DRIVE_ROOT,
        LOCAL_ROOT
    ]

    if p.exists()
]


def normalize_filename(
    value
):

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower()
    )


def path_penalty(
    path
):

    text = str(
        path
    ).lower()


    penalty = 0


    for bad_term in [

        "synthetic_naf_v1",
        "audit_outputs",
        "metadata"

    ]:

        if bad_term in text:

            penalty += 100


    if "raw" in text:

        penalty -= 20


    return penalty


candidate_csvs = []


for root in SEARCH_ROOTS:

    candidate_csvs.extend(
        list(
            root.rglob(
                "*.csv"
            )
        )
    )


candidate_csvs = list(
    dict.fromkeys(
        candidate_csvs
    )
)


print(
    "CSV files visible under project roots:",
    len(
        candidate_csvs
    )
)


# ==================================================================================================
# 11.1D — DISCOVER CIC RAW FILES
# ==================================================================================================

banner(
    "11.1D — DISCOVER CIC-IDS2017 RAW FILES"
)


CIC_SIGNATURES = {

    "MONDAY": [
        "mondayworkinghours"
    ],

    "TUESDAY": [
        "tuesdayworkinghours"
    ],

    "WEDNESDAY": [
        "wednesdayworkinghours"
    ],

    "THURSDAY_MORNING_WEB": [
        "thursdayworkinghoursmorningwebattacks"
    ],

    "THURSDAY_AFTERNOON_INFILTRATION": [
        "thursdayworkinghoursafternooninfilteration",
        "thursdayworkinghoursafternooninfiltration"
    ],

    "FRIDAY_MORNING": [
        "fridayworkinghoursmorning"
    ],

    "FRIDAY_AFTERNOON_PORTSCAN": [
        "fridayworkinghoursafternoonportscan"
    ],

    "FRIDAY_AFTERNOON_DDOS": [
        "fridayworkinghoursafternoonddos"
    ]
}


CIC_ENCODING_BY_LOGICAL_NAME = {

    "MONDAY":
        "utf-8",

    "TUESDAY":
        "utf-8",

    "WEDNESDAY":
        "utf-8",

    "THURSDAY_MORNING_WEB":
        "cp1252",

    "THURSDAY_AFTERNOON_INFILTRATION":
        "utf-8",

    "FRIDAY_MORNING":
        "utf-8",

    "FRIDAY_AFTERNOON_PORTSCAN":
        "utf-8",

    "FRIDAY_AFTERNOON_DDOS":
        "utf-8"
}


def discover_signature_file(
    signatures
):

    found = []


    for path in candidate_csvs:

        normalized_name = normalize_filename(
            path.name
        )


        if any(
            signature
            in
            normalized_name

            for signature
            in signatures
        ):

            found.append(
                path
            )


    if not found:

        return None


    found = sorted(

        found,

        key=lambda p: (

            path_penalty(
                p
            ),

            len(
                str(
                    p
                )
            )
        )
    )


    return found[0]


CIC_FILES = {}


for logical_name, signatures in CIC_SIGNATURES.items():

    path = discover_signature_file(
        signatures
    )


    if path is None:

        print(
            "✗",
            logical_name
        )

    else:

        CIC_FILES[
            logical_name
        ] = path


        print(
            "✓",
            logical_name,
            "→",
            path,
            "| encoding=",
            CIC_ENCODING_BY_LOGICAL_NAME[
                logical_name
            ]
        )


if len(
    CIC_FILES
) != 8:

    raise FileNotFoundError(
        f"Expected 8 CIC raw files. Found {len(CIC_FILES)}."
    )


# ==================================================================================================
# 11.1E — DISCOVER UNSW FILES
# ==================================================================================================

banner(
    "11.1E — DISCOVER UNSW-NB15 RAW FILES"
)


def find_exact_basename(
    basename_options
):

    allowed = {
        x.lower()
        for x in basename_options
    }


    found = [

        p

        for p in candidate_csvs

        if p.name.lower()
        in allowed
    ]


    if not found:

        return None


    return sorted(

        found,

        key=lambda p: (

            path_penalty(
                p
            ),

            len(
                str(
                    p
                )
            )
        )

    )[0]


UNSW_FILES = []


for i in range(
    1,
    5
):

    path = find_exact_basename(
        [
            f"UNSW-NB15_{i}.csv",
            f"NUSW-NB15_{i}.csv"
        ]
    )


    if path is None:

        raise FileNotFoundError(
            f"Could not locate UNSW-NB15_{i}.csv"
        )


    UNSW_FILES.append(
        path
    )


UNSW_FEATURE_METADATA_FILE = find_exact_basename(
    [
        "NUSW-NB15_features.csv",
        "UNSW-NB15_features.csv"
    ]
)


if UNSW_FEATURE_METADATA_FILE is None:

    raise FileNotFoundError(
        "Could not locate UNSW feature metadata."
    )


for path in UNSW_FILES:

    print(
        "✓",
        path
    )


print(
    "✓ Feature metadata:",
    UNSW_FEATURE_METADATA_FILE
)


if not UNSW_FINAL_SEMANTICS_FILE.exists():

    raise FileNotFoundError(
        f"Missing frozen UNSW semantics:\n{UNSW_FINAL_SEMANTICS_FILE}"
    )


if not UNSW_REFERENCE_MINUTE_FILE.exists():

    raise FileNotFoundError(
        f"Missing frozen UNSW minute representation:\n{UNSW_REFERENCE_MINUTE_FILE}"
    )


# ==================================================================================================
# 11.1F — UNSW RAW SCHEMA
# ==================================================================================================

banner(
    "11.1F — LOAD UNSW RAW FEATURE NAMES"
)


feature_meta = pd.read_csv(
    UNSW_FEATURE_METADATA_FILE,
    encoding="cp1252"
)


feature_name_col = None


for c in feature_meta.columns:

    if str(
        c
    ).strip().lower() == "name":

        feature_name_col = c

        break


if feature_name_col is None:

    raise ValueError(
        "UNSW feature metadata has no Name column."
    )


UNSW_RAW_COLUMNS = (

    feature_meta[
        feature_name_col
    ]

    .astype(
        str
    )

    .str.strip()

    .tolist()
)


print(
    "UNSW feature count:",
    len(
        UNSW_RAW_COLUMNS
    )
)


if len(
    UNSW_RAW_COLUMNS
) != 49:

    raise RuntimeError(
        f"Expected 49 UNSW columns; found {len(UNSW_RAW_COLUMNS)}."
    )


# ==================================================================================================
# 11.1G — LABEL NORMALIZATION + FAMILY TAXONOMY
# ==================================================================================================

banner(
    "11.1G — REAL LABEL → SYNTHETIC FAMILY MAP"
)


def normalize_label(
    value
):

    if pd.isna(
        value
    ):

        return ""


    text = str(
        value
    ).strip().lower()


    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text
    )


    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()


    return text


CIC_LABEL_TO_FAMILY = {

    "ddos":
        "DOS_DDOS",

    "dos hulk":
        "DOS_DDOS",

    "dos goldeneye":
        "DOS_DDOS",

    "dos slowloris":
        "DOS_DDOS",

    "dos slowhttptest":
        "DOS_DDOS",


    "portscan":
        "RECON_SCAN",


    "web attack xss":
        "EXPLOIT_APP",

    "web attack sql injection":
        "EXPLOIT_APP",

    "infiltration":
        "EXPLOIT_APP",


    "ftp patator":
        "BRUTE_AUTH",

    "ssh patator":
        "BRUTE_AUTH",

    "web attack brute force":
        "BRUTE_AUTH"
}


UNSW_CANONICAL_FAMILY_ALIASES = {

    "generic":
        "generic",

    "exploits":
        "exploits",

    "fuzzers":
        "fuzzers",

    "dos":
        "dos",

    "reconnaissance":
        "reconnaissance",

    "analysis":
        "analysis",

    "backdoor":
        "backdoor",

    "backdoors":
        "backdoor",

    "shellcode":
        "shellcode",

    "worms":
        "worms"
}


UNSW_FINAL_FAMILY_TO_SYNTHETIC = {

    "dos":
        "DOS_DDOS",

    "reconnaissance":
        "RECON_SCAN",

    "exploits":
        "EXPLOIT_APP",

    "shellcode":
        "EXPLOIT_APP",

    "backdoor":
        "EXPLOIT_APP",

    "fuzzers":
        "FUZZING"
}


def canonicalize_unsw_family(
    value
):

    normalized = normalize_label(
        value
    )


    if normalized == "":

        return ""


    return UNSW_CANONICAL_FAMILY_ALIASES.get(
        normalized,
        normalized
    )


FAMILY_TO_CODE = {

    family:
        i + 1

    for i, family
    in enumerate(
        ATTACK_FAMILIES
    )
}


CODE_TO_FAMILY = {

    code:
        family

    for family, code
    in FAMILY_TO_CODE.items()
}


print(
    "Synthetic families:",
    ATTACK_FAMILIES
)


# ==================================================================================================
# 11.1H — LOAD FROZEN UNSW FLOW SEMANTICS
# ==================================================================================================

banner(
    "11.1H — LOAD FROZEN UNSW 10.12 FLOW SEMANTICS"
)


sem_header = pd.read_csv(
    UNSW_FINAL_SEMANTICS_FILE,
    nrows=0
)


def find_column(
    columns,
    candidates
):

    lookup = {

        str(
            c
        ).strip().lower():
            c

        for c in columns
    }


    for candidate in candidates:

        key = candidate.strip().lower()


        if key in lookup:

            return lookup[
                key
            ]


    return None


SEM_FLOW_FILE = find_column(

    sem_header.columns,

    [
        "Flow_File",
        "Flow File"
    ]
)


SEM_FLOW_ROW = find_column(

    sem_header.columns,

    [
        "Flow_Row_Index",
        "Flow Row Index"
    ]
)


SEM_ATTACK_STATE = find_column(

    sem_header.columns,

    [
        "Final_Attack_State",
        "Final Attack State"
    ]
)


SEM_ATTACK_FAMILY = find_column(

    sem_header.columns,

    [
        "Final_Attack_Family",
        "Final Attack Family"
    ]
)


required_semantic_columns = {

    "Flow_File":
        SEM_FLOW_FILE,

    "Flow_Row_Index":
        SEM_FLOW_ROW,

    "Final_Attack_State":
        SEM_ATTACK_STATE,

    "Final_Attack_Family":
        SEM_ATTACK_FAMILY
}


missing_semantics = [

    name

    for name, actual
    in required_semantic_columns.items()

    if actual is None
]


if missing_semantics:

    raise ValueError(
        f"Frozen UNSW semantics missing columns: {missing_semantics}"
    )


semantics = pd.read_csv(

    UNSW_FINAL_SEMANTICS_FILE,

    usecols=[
        SEM_FLOW_FILE,
        SEM_FLOW_ROW,
        SEM_ATTACK_STATE,
        SEM_ATTACK_FAMILY
    ],

    low_memory=False
)


semantics = semantics.rename(
    columns={

        SEM_FLOW_FILE:
            "Flow_File",

        SEM_FLOW_ROW:
            "Flow_Row_Index",

        SEM_ATTACK_STATE:
            "Final_Attack_State",

        SEM_ATTACK_FAMILY:
            "Final_Attack_Family"
    }
)


semantics[
    "Flow_Row_Index"
] = pd.to_numeric(

    semantics[
        "Flow_Row_Index"
    ],

    errors="raise"
).astype(
    np.int64
)


semantics[
    "Final_Attack_State"
] = pd.to_numeric(

    semantics[
        "Final_Attack_State"
    ],

    errors="raise"
).astype(
    np.int8
)


semantics[
    "Canonical_Final_Family"
] = (

    semantics[
        "Final_Attack_Family"
    ]

    .apply(
        canonicalize_unsw_family
    )
)


semantics[
    "Synthetic_Family"
] = (

    semantics[
        "Canonical_Final_Family"
    ]

    .map(
        UNSW_FINAL_FAMILY_TO_SYNTHETIC
    )
)


semantics[
    "Synthetic_Family_Code"
] = (

    semantics[
        "Synthetic_Family"
    ]

    .map(
        FAMILY_TO_CODE
    )

    .fillna(
        0
    )

    .astype(
        np.int8
    )
)


semantics[
    "Normalized_Flow_File"
] = (

    semantics[
        "Flow_File"
    ]

    .apply(
        lambda x:
            normalize_filename(
                Path(
                    str(
                        x
                    )
                ).name
            )
    )
)


print(
    "Frozen UNSW semantic rows:",
    f"{len(semantics):,}"
)


if len(
    semantics
) != EXPECTED_UNSW_FLOWS:

    raise RuntimeError(
        f"Expected {EXPECTED_UNSW_FLOWS:,} UNSW semantic rows; found {len(semantics):,}."
    )


sem_attack_count = int(
    semantics[
        "Final_Attack_State"
    ].sum()
)


sem_benign_count = (
    len(
        semantics
    )
    -
    sem_attack_count
)


print(
    "Frozen benign flows:",
    f"{sem_benign_count:,}"
)

print(
    "Frozen attack flows:",
    f"{sem_attack_count:,}"
)


if (
    sem_attack_count
    !=
    EXPECTED_UNSW_ATTACK_FLOWS
):

    raise RuntimeError(
        "Frozen UNSW attack count changed."
    )


if (
    sem_benign_count
    !=
    EXPECTED_UNSW_BENIGN_FLOWS
):

    raise RuntimeError(
        "Frozen UNSW benign count changed."
    )


# Store compact per-file arrays.

UNSW_SEMANTICS_BY_FILE = {}


for normalized_file, group in semantics.groupby(
    "Normalized_Flow_File",
    observed=True
):

    group = (

        group

        .sort_values(
            "Flow_Row_Index"
        )

        .reset_index(
            drop=True
        )
    )


    expected_index = np.arange(
        len(
            group
        ),
        dtype=np.int64
    )


    actual_index = group[
        "Flow_Row_Index"
    ].to_numpy(
        dtype=np.int64
    )


    if not np.array_equal(
        expected_index,
        actual_index
    ):

        raise RuntimeError(
            f"UNSW frozen row index is not contiguous for {normalized_file}."
        )


    UNSW_SEMANTICS_BY_FILE[
        normalized_file
    ] = {

        "attack_state":
            group[
                "Final_Attack_State"
            ].to_numpy(
                dtype=np.int8
            ),

        "family_code":
            group[
                "Synthetic_Family_Code"
            ].to_numpy(
                dtype=np.int8
            ),

        "canonical_family":
            group[
                "Canonical_Final_Family"
            ].astype(
                "string"
            ).to_numpy()
    }


# Preserve authoritative family distribution.

unsw_authoritative_family_distribution = (

    semantics.loc[
        semantics[
            "Final_Attack_State"
        ]
        ==
        1,
        "Canonical_Final_Family"
    ]

    .value_counts()

    .rename_axis(
        "UNSW_Final_Attack_Family"
    )

    .reset_index(
        name="Flow_Count"
    )
)


print(
    "\nFrozen UNSW family distribution:"
)

print(
    unsw_authoritative_family_distribution.to_string(
        index=False
    )
)


del semantics

gc.collect()


# ==================================================================================================
# 11.1I — COMMON CALIBRATION FUNCTIONS
# ==================================================================================================

banner(
    "11.1I — INITIALIZE COMMON CALIBRATION FUNCTIONS"
)


FLOW_PROFILE_FEATURES = [

    "Flow_Duration_Seconds",

    "Forward_Packets",

    "Backward_Packets",

    "Total_Packets_Per_Flow",

    "Forward_Bytes",

    "Backward_Bytes",

    "Total_Bytes_Per_Flow",

    "Packet_Direction_Ratio_Per_Flow",

    "Byte_Direction_Ratio_Per_Flow"
]


def parse_cic_timestamp(
    series
):

    try:

        return pd.to_datetime(

            series,

            format="mixed",

            dayfirst=True,

            errors="coerce"
        )


    except TypeError:

        return pd.to_datetime(

            series,

            dayfirst=True,

            errors="coerce"
        )


def parse_unsw_stime(
    series
):

    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )


    result = pd.to_datetime(

        numeric,

        unit="s",

        utc=True,

        errors="coerce"
    )


    return result.dt.tz_convert(
        None
    )


def build_chunk_minute_partial(
    df
):

    temp = df[
        [
            "Minute",
            "Flow_Duration_Seconds",
            "Forward_Packets",
            "Backward_Packets",
            "Forward_Bytes",
            "Backward_Bytes",
            "Attack_State",
            "Synthetic_Family"
        ]
    ].copy()


    temp[
        "Duration_Squared"
    ] = (

        temp[
            "Flow_Duration_Seconds"
        ]

        **

        2
    )


    temp[
        "Mapped_Attack"
    ] = (

        (
            temp[
                "Attack_State"
            ]
            ==
            1
        )

        &

        temp[
            "Synthetic_Family"
        ].notna()
    ).astype(
        np.int8
    )


    temp[
        "Unmapped_Attack"
    ] = (

        (
            temp[
                "Attack_State"
            ]
            ==
            1
        )

        &

        temp[
            "Synthetic_Family"
        ].isna()
    ).astype(
        np.int8
    )


    return (

        temp

        .groupby(
            "Minute",
            observed=True,
            sort=False
        )

        .agg(

            Flow_Count=(
                "Minute",
                "size"
            ),

            Duration_Sum=(
                "Flow_Duration_Seconds",
                "sum"
            ),

            Duration_SumSq=(
                "Duration_Squared",
                "sum"
            ),

            Flow_Duration_Max=(
                "Flow_Duration_Seconds",
                "max"
            ),

            Forward_Packets_Sum=(
                "Forward_Packets",
                "sum"
            ),

            Backward_Packets_Sum=(
                "Backward_Packets",
                "sum"
            ),

            Forward_Bytes_Sum=(
                "Forward_Bytes",
                "sum"
            ),

            Backward_Bytes_Sum=(
                "Backward_Bytes",
                "sum"
            ),

            Attack_Flow_Count=(
                "Attack_State",
                "sum"
            ),

            Mapped_Attack_Flow_Count=(
                "Mapped_Attack",
                "sum"
            ),

            Unmapped_Attack_Flow_Count=(
                "Unmapped_Attack",
                "sum"
            )

        )

        .reset_index()
    )


def finalize_minute_aggregation(
    partials
):

    if not partials:

        return pd.DataFrame()


    combined = pd.concat(
        partials,
        ignore_index=True
    )


    grouped = (

        combined

        .groupby(
            "Minute",
            observed=True,
            sort=True
        )

        .agg(

            Flow_Count=(
                "Flow_Count",
                "sum"
            ),

            Duration_Sum=(
                "Duration_Sum",
                "sum"
            ),

            Duration_SumSq=(
                "Duration_SumSq",
                "sum"
            ),

            Flow_Duration_Max=(
                "Flow_Duration_Max",
                "max"
            ),

            Forward_Packets_Sum=(
                "Forward_Packets_Sum",
                "sum"
            ),

            Backward_Packets_Sum=(
                "Backward_Packets_Sum",
                "sum"
            ),

            Forward_Bytes_Sum=(
                "Forward_Bytes_Sum",
                "sum"
            ),

            Backward_Bytes_Sum=(
                "Backward_Bytes_Sum",
                "sum"
            ),

            Attack_Flow_Count=(
                "Attack_Flow_Count",
                "sum"
            ),

            Mapped_Attack_Flow_Count=(
                "Mapped_Attack_Flow_Count",
                "sum"
            ),

            Unmapped_Attack_Flow_Count=(
                "Unmapped_Attack_Flow_Count",
                "sum"
            )

        )

        .reset_index()
    )


    n = grouped[
        "Flow_Count"
    ].astype(
        float
    )


    grouped[
        "Flow_Duration_Mean"
    ] = (

        grouped[
            "Duration_Sum"
        ]

        /

        n
    )


    variance_numerator = (

        grouped[
            "Duration_SumSq"
        ]

        -

        (
            grouped[
                "Duration_Sum"
            ]

            **

            2
        )

        /

        n
    )


    variance = np.where(

        n
        >
        1,

        variance_numerator
        /
        (
            n
            -
            1.0
        ),

        0.0
    )


    variance = np.maximum(
        variance,
        0.0
    )


    grouped[
        "Flow_Duration_Std"
    ] = np.sqrt(
        variance
    )


    grouped[
        "Total_Packets"
    ] = (

        grouped[
            "Forward_Packets_Sum"
        ]

        +

        grouped[
            "Backward_Packets_Sum"
        ]
    )


    grouped[
        "Total_Bytes"
    ] = (

        grouped[
            "Forward_Bytes_Sum"
        ]

        +

        grouped[
            "Backward_Bytes_Sum"
        ]
    )


    grouped[
        "Packets_Per_Second"
    ] = (

        grouped[
            "Total_Packets"
        ]

        /
        60.0
    )


    grouped[
        "Bytes_Per_Second"
    ] = (

        grouped[
            "Total_Bytes"
        ]

        /
        60.0
    )


    grouped[
        "Forward_Backward_Packet_Ratio"
    ] = (

        grouped[
            "Forward_Packets_Sum"
        ]

        /

        (
            grouped[
                "Backward_Packets_Sum"
            ]

            +
            1.0
        )
    )


    grouped[
        "Forward_Backward_Byte_Ratio"
    ] = (

        grouped[
            "Forward_Bytes_Sum"
        ]

        /

        (
            grouped[
                "Backward_Bytes_Sum"
            ]

            +
            1.0
        )
    )


    grouped[
        "Attack_State"
    ] = (

        grouped[
            "Attack_Flow_Count"
        ]

        >
        0
    ).astype(
        np.int8
    )


    return grouped


def build_family_minute_partial(
    df,
    source
):

    attack = df.loc[
        (
            df[
                "Attack_State"
            ]
            ==
            1
        )

        &

        df[
            "Synthetic_Family"
        ].notna(),

        [
            "Minute",
            "Synthetic_Family"
        ]
    ]


    if len(
        attack
    ) == 0:

        return pd.DataFrame(
            columns=[
                "Calibration_Source",
                "Minute",
                "Synthetic_Family",
                "Attack_Flows"
            ]
        )


    result = (

        attack

        .groupby(
            [
                "Minute",
                "Synthetic_Family"
            ],
            observed=True,
            sort=False
        )

        .size()

        .rename(
            "Attack_Flows"
        )

        .reset_index()
    )


    result.insert(
        0,
        "Calibration_Source",
        source
    )


    return result


def build_flow_profile_sample(
    df,
    source,
    row_ids
):

    temp = df.copy()


    temp[
        "Total_Packets_Per_Flow"
    ] = (

        temp[
            "Forward_Packets"
        ]

        +

        temp[
            "Backward_Packets"
        ]
    )


    temp[
        "Total_Bytes_Per_Flow"
    ] = (

        temp[
            "Forward_Bytes"
        ]

        +

        temp[
            "Backward_Bytes"
        ]
    )


    temp[
        "Packet_Direction_Ratio_Per_Flow"
    ] = (

        temp[
            "Forward_Packets"
        ]

        /

        (
            temp[
                "Backward_Packets"
            ]

            +
            1.0
        )
    )


    temp[
        "Byte_Direction_Ratio_Per_Flow"
    ] = (

        temp[
            "Forward_Bytes"
        ]

        /

        (
            temp[
                "Backward_Bytes"
            ]

            +
            1.0
        )
    )


    is_benign = (

        temp[
            "Attack_State"
        ]

        ==
        0
    )


    is_selected_attack = (

        (
            temp[
                "Attack_State"
            ]

            ==
            1
        )

        &

        temp[
            "Synthetic_Family"
        ].notna()
    )


    benign_sample = (

        is_benign

        &

        (
            row_ids
            %
            BENIGN_FLOW_SAMPLE_MODULUS

            ==
            0
        )
    )


    keep = (

        benign_sample

        |

        is_selected_attack
    )


    sample = temp.loc[
        keep,
        FLOW_PROFILE_FEATURES
    ].copy()


    sample[
        "Profile_Class"
    ] = np.where(

        is_benign.loc[
            keep
        ].to_numpy(),

        "BENIGN",

        temp.loc[
            keep,
            "Synthetic_Family"
        ].astype(
            str
        ).to_numpy()
    )


    sample.insert(
        0,
        "Calibration_Source",
        source
    )


    return sample


# ==================================================================================================
# 11.1J — PROCESS CIC RAW FLOWS
# ==================================================================================================

banner(
    "11.1J — PROCESS CIC RAW FLOWS"
)


CIC_REQUIRED_COLUMNS = [

    "Timestamp",

    "Flow Duration",

    "Total Fwd Packets",

    "Total Backward Packets",

    "Total Length of Fwd Packets",

    "Total Length of Bwd Packets",

    "Label"
]


cic_minute_partials = []

cic_family_minute_partials = []

cic_flow_samples = []


cic_label_counter = Counter()

cic_family_flow_counter = Counter()


cic_physical_rows = 0

cic_empty_rows_removed = 0

cic_real_traffic_rows = 0

cic_invalid_timestamp_rows = 0

cic_invalid_numeric_rows = 0

cic_invalid_any_rows = 0


cic_file_accounting_rows = []


for file_number, (
    logical_name,
    path
) in enumerate(
    CIC_FILES.items(),
    start=1
):

    print(
        "\n"
        +
        "-" * 120
    )

    print(
        f"[CIC {file_number}/8] {logical_name}"
    )

    print(
        path
    )

    file_encoding = CIC_ENCODING_BY_LOGICAL_NAME[
        logical_name
    ]


    print(
        "Encoding:",
        file_encoding
    )

    print(
        "-" * 120
    )


    header = pd.read_csv(

        path,

        nrows=0,

        encoding=file_encoding
    )


    stripped_to_actual = {

        str(
            c
        ).strip():
            c

        for c in header.columns
    }


    missing_columns = [

        c

        for c in CIC_REQUIRED_COLUMNS

        if c not in stripped_to_actual
    ]


    if missing_columns:

        raise ValueError(
            f"{path.name} missing required columns: {missing_columns}"
        )


    usecols_actual = [

        stripped_to_actual[
            c
        ]

        for c in CIC_REQUIRED_COLUMNS
    ]


    file_physical_rows = 0

    file_empty_rows = 0

    file_real_rows = 0

    file_invalid_timestamp_rows = 0

    file_invalid_numeric_rows = 0

    file_attack_rows = 0


    reader = pd.read_csv(

        path,

        usecols=usecols_actual,

        chunksize=CHUNK_SIZE,

        low_memory=False,

        encoding=file_encoding
    )


    for chunk_number, chunk in enumerate(
        reader,
        start=1
    ):

        chunk = chunk.rename(
            columns={

                actual:
                    stripped

                for stripped, actual
                in stripped_to_actual.items()

                if actual in chunk.columns
            }
        )


        # ------------------------------------------------------------------------------------------
        # PHYSICAL vs REAL TRAFFIC ROWS
        # ------------------------------------------------------------------------------------------

        physical_n = len(
            chunk
        )


        cic_physical_rows += (
            physical_n
        )

        file_physical_rows += (
            physical_n
        )


        empty_check = chunk[
            CIC_REQUIRED_COLUMNS
        ].copy()


        for column in empty_check.columns:

            if (
                empty_check[
                    column
                ].dtype
                ==
                object
            ):

                empty_check[
                    column
                ] = (

                    empty_check[
                        column
                    ]

                    .astype(
                        "string"
                    )

                    .str.strip()

                    .replace(
                        "",
                        pd.NA
                    )
                )


        empty_mask = (

            empty_check

            .isna()

            .all(
                axis=1
            )
        )


        empty_count = int(
            empty_mask.sum()
        )


        if empty_count > 0:

            print(
                f"  chunk {chunk_number}: "
                f"removing {empty_count:,} completely empty physical rows"
            )


        cic_empty_rows_removed += (
            empty_count
        )

        file_empty_rows += (
            empty_count
        )


        chunk = (

            chunk.loc[
                ~empty_mask
            ]

            .copy()

            .reset_index(
                drop=True
            )
        )


        real_n = len(
            chunk
        )


        real_row_start = cic_real_traffic_rows


        cic_real_traffic_rows += (
            real_n
        )

        file_real_rows += (
            real_n
        )


        dataset_row_ids = np.arange(

            real_row_start,

            real_row_start
            +
            real_n,

            dtype=np.int64
        )


        # ------------------------------------------------------------------------------------------
        # TIMESTAMP
        # ------------------------------------------------------------------------------------------

        timestamps = parse_cic_timestamp(
            chunk[
                "Timestamp"
            ]
        )


        invalid_timestamp_mask = (
            timestamps.isna()
        )


        invalid_timestamp_count = int(
            invalid_timestamp_mask.sum()
        )


        cic_invalid_timestamp_rows += (
            invalid_timestamp_count
        )

        file_invalid_timestamp_rows += (
            invalid_timestamp_count
        )


        # ------------------------------------------------------------------------------------------
        # NUMERIC FIELDS
        # ------------------------------------------------------------------------------------------

        duration = (

            pd.to_numeric(
                chunk[
                    "Flow Duration"
                ],
                errors="coerce"
            )

            /
            1_000_000.0
        )


        forward_packets = pd.to_numeric(

            chunk[
                "Total Fwd Packets"
            ],

            errors="coerce"
        )


        backward_packets = pd.to_numeric(

            chunk[
                "Total Backward Packets"
            ],

            errors="coerce"
        )


        forward_bytes = pd.to_numeric(

            chunk[
                "Total Length of Fwd Packets"
            ],

            errors="coerce"
        )


        backward_bytes = pd.to_numeric(

            chunk[
                "Total Length of Bwd Packets"
            ],

            errors="coerce"
        )


        numeric_frame = pd.DataFrame({

            "Flow_Duration_Seconds":
                duration,

            "Forward_Packets":
                forward_packets,

            "Backward_Packets":
                backward_packets,

            "Forward_Bytes":
                forward_bytes,

            "Backward_Bytes":
                backward_bytes
        })


        invalid_numeric_mask = (

            numeric_frame

            .isna()

            .any(
                axis=1
            )
        )


        invalid_numeric_count = int(
            invalid_numeric_mask.sum()
        )


        cic_invalid_numeric_rows += (
            invalid_numeric_count
        )

        file_invalid_numeric_rows += (
            invalid_numeric_count
        )


        invalid_any_mask = (

            invalid_timestamp_mask

            |

            invalid_numeric_mask
        )


        cic_invalid_any_rows += int(
            invalid_any_mask.sum()
        )


        # ------------------------------------------------------------------------------------------
        # CIC LABELS
        # ------------------------------------------------------------------------------------------

        raw_labels = (

            chunk[
                "Label"
            ]

            .apply(
                normalize_label
            )
        )


        attack_state = (

            raw_labels

            !=

            "benign"
        ).astype(
            np.int8
        )


        synthetic_family = (

            raw_labels

            .map(
                CIC_LABEL_TO_FAMILY
            )
        )


        file_attack_rows += int(
            attack_state.sum()
        )


        label_counts = raw_labels.value_counts(
            dropna=False
        )


        for label_name, count in label_counts.items():

            cic_label_counter[
                label_name
            ] += int(
                count
            )


        family_counts = (

            synthetic_family.loc[
                attack_state
                ==
                1
            ]

            .value_counts(
                dropna=True
            )
        )


        for family, count in family_counts.items():

            cic_family_flow_counter[
                family
            ] += int(
                count
            )


        # ------------------------------------------------------------------------------------------
        # COMMON FLOW FRAME
        # ------------------------------------------------------------------------------------------

        flow = pd.DataFrame({

            "Minute":
                timestamps.dt.floor(
                    "min"
                ),

            "Flow_Duration_Seconds":
                duration,

            "Forward_Packets":
                forward_packets,

            "Backward_Packets":
                backward_packets,

            "Forward_Bytes":
                forward_bytes,

            "Backward_Bytes":
                backward_bytes,

            "Attack_State":
                attack_state,

            "Synthetic_Family":
                synthetic_family
        })


        valid_mask = (
            ~invalid_any_mask
        )


        flow_valid = (

            flow.loc[
                valid_mask
            ]

            .reset_index(
                drop=True
            )
        )


        row_ids_valid = dataset_row_ids[
            valid_mask.to_numpy()
        ]


        cic_minute_partials.append(

            build_chunk_minute_partial(
                flow_valid
            )
        )


        cic_family_minute_partials.append(

            build_family_minute_partial(
                flow_valid,
                "CIC_IDS2017"
            )
        )


        cic_flow_samples.append(

            build_flow_profile_sample(

                flow_valid,

                "CIC_IDS2017",

                row_ids_valid
            )
        )


        print(
            f"  chunk {chunk_number}: "
            f"physical={physical_n:,}, "
            f"real={real_n:,}, "
            f"file real cumulative={file_real_rows:,}"
        )


    cic_file_accounting_rows.append(
        {
            "Logical_Name":
                logical_name,

            "File":
                path.name,

            "Encoding":
                file_encoding,

            "Physical_Rows":
                file_physical_rows,

            "Empty_Physical_Rows_Removed":
                file_empty_rows,

            "Real_Traffic_Rows":
                file_real_rows,

            "Invalid_Timestamps_After_Empty_Removal":
                file_invalid_timestamp_rows,

            "Invalid_Numeric_Rows_After_Empty_Removal":
                file_invalid_numeric_rows,

            "Attack_Rows":
                file_attack_rows
        }
    )


    print(
        f"✓ {path.name}: "
        f"{file_real_rows:,} real traffic rows"
    )


cic_file_accounting = pd.DataFrame(
    cic_file_accounting_rows
)


print(
    "\nCIC FILE ACCOUNTING:\n"
)

print(
    cic_file_accounting.to_string(
        index=False
    )
)


print(
    "\nCIC physical CSV rows             :",
    f"{cic_physical_rows:,}"
)

print(
    "CIC empty physical rows removed   :",
    f"{cic_empty_rows_removed:,}"
)

print(
    "CIC real traffic rows             :",
    f"{cic_real_traffic_rows:,}"
)

print(
    "CIC invalid timestamps (real rows):",
    f"{cic_invalid_timestamp_rows:,}"
)

print(
    "CIC invalid numeric rows          :",
    f"{cic_invalid_numeric_rows:,}"
)


# ==================================================================================================
# 11.1K — FINALIZE CIC MINUTE REPRESENTATION
# ==================================================================================================

banner(
    "11.1K — FINALIZE CIC 1-MINUTE CALIBRATION REPRESENTATION"
)


cic_minutes = finalize_minute_aggregation(
    cic_minute_partials
)


cic_minutes[
    "Calibration_Source"
] = "CIC_IDS2017"


print(
    "CIC observed minutes:",
    f"{len(cic_minutes):,}"
)

print(
    "CIC flow count recovered:",
    f"{int(cic_minutes['Flow_Count'].sum()):,}"
)

print(
    "CIC attack minutes:",
    f"{int(cic_minutes['Attack_State'].sum()):,}"
)


# ==================================================================================================
# 11.1L — CIC FORENSIC REFERENCE VALIDATION
#
# cic_complete_minute_flow_table.csv is an older semantic/forensic artifact.
#
# It is NOT required to contain Flow_Count or modern 14-feature predictors.
#
# We use it for:
#
#   ✓ exact Minute grid
#   ✓ Attack_State if available
#   ✓ frozen Segment_ID if available
#
# ==================================================================================================

banner(
    "11.1L — VERIFY CIC AGAINST FROZEN FORENSIC MINUTE SEMANTICS"
)


cic_reference_available = (
    CIC_REFERENCE_MINUTE_FILE.exists()
)


cic_reference_grid_match = None

cic_reference_attack_match = None

cic_reference_exact_match = None


if cic_reference_available:

    cic_ref = pd.read_csv(
        CIC_REFERENCE_MINUTE_FILE,
        low_memory=False
    )


    print(
        "Reference shape:",
        cic_ref.shape
    )


    print(
        "Reference columns:"
    )


    for c in cic_ref.columns:

        print(
            "  ",
            repr(
                c
            )
        )


    if "Minute" not in cic_ref.columns:

        raise ValueError(
            "CIC forensic reference has no Minute column."
        )


    cic_ref[
        "Minute"
    ] = pd.to_datetime(
        cic_ref[
            "Minute"
        ],
        errors="coerce"
    )


    if cic_ref[
        "Minute"
    ].isna().any():

        raise RuntimeError(
            "CIC forensic reference contains invalid Minute values."
        )


    rebuilt_grid = (

        cic_minutes[
            [
                "Minute"
            ]
        ]

        .drop_duplicates()

        .sort_values(
            "Minute"
        )

        .reset_index(
            drop=True
        )
    )


    reference_grid = (

        cic_ref[
            [
                "Minute"
            ]
        ]

        .drop_duplicates()

        .sort_values(
            "Minute"
        )

        .reset_index(
            drop=True
        )
    )


    cic_reference_grid_match = bool(

        len(
            rebuilt_grid
        )

        ==

        len(
            reference_grid
        )

        and

        rebuilt_grid[
            "Minute"
        ].equals(
            reference_grid[
                "Minute"
            ]
        )
    )


    print(
        "\nMinute-grid match:",
        cic_reference_grid_match
    )


    if not cic_reference_grid_match:

        raise RuntimeError(
            "CIC reconstructed minute grid disagrees with frozen forensic artifact."
        )


    if "Attack_State" in cic_ref.columns:

        ref_state = pd.to_numeric(

            cic_ref[
                "Attack_State"
            ],

            errors="coerce"
        )


        if ref_state.isna().any():

            raise RuntimeError(
                "CIC reference Attack_State contains invalid values."
            )


        attack_reference = cic_ref[
            [
                "Minute"
            ]
        ].copy()


        attack_reference[
            "Attack_State"
        ] = ref_state.astype(
            np.int8
        )


        attack_compare = cic_minutes[
            [
                "Minute",
                "Attack_State"
            ]
        ].merge(

            attack_reference,

            on="Minute",

            how="inner",

            suffixes=(
                "_RAW_REBUILT",
                "_REFERENCE"
            ),

            validate="one_to_one"
        )


        cic_reference_attack_match = bool(

            (
                attack_compare[
                    "Attack_State_RAW_REBUILT"
                ]

                ==

                attack_compare[
                    "Attack_State_REFERENCE"
                ]
            )

            .all()
        )


        print(
            "Attack-state match:",
            cic_reference_attack_match
        )


        if not cic_reference_attack_match:

            mismatch_count = int(

                (
                    attack_compare[
                        "Attack_State_RAW_REBUILT"
                    ]

                    !=

                    attack_compare[
                        "Attack_State_REFERENCE"
                    ]
                )

                .sum()
            )


            raise RuntimeError(
                f"CIC Attack_State mismatch for {mismatch_count} minutes."
            )


    else:

        print(
            "⚠ Frozen forensic file has no Attack_State column."
        )


    if "Segment_ID" in cic_ref.columns:

        segment_map = (

            cic_ref[
                [
                    "Minute",
                    "Segment_ID"
                ]
            ]

            .drop_duplicates(
                subset=[
                    "Minute"
                ]
            )
        )


        cic_minutes = cic_minutes.merge(

            segment_map,

            on="Minute",

            how="left",

            validate="one_to_one"
        )


        if cic_minutes[
            "Segment_ID"
        ].isna().any():

            raise RuntimeError(
                "Some CIC reconstructed minutes failed Segment_ID mapping."
            )


        cic_minutes[
            "Real_Segment_ID"
        ] = (

            "CIC::"

            +

            cic_minutes[
                "Segment_ID"
            ].astype(
                str
            )
        )


        print(
            "✓ Frozen CIC Segment_ID imported"
        )


    cic_reference_exact_match = bool(

        cic_reference_grid_match

        and

        (
            cic_reference_attack_match
            in
            [
                True,
                None
            ]
        )
    )


    print(
        "\n✓ CIC reconstructed minute semantics agree with frozen reference"
    )


else:

    print(
        "⚠ CIC forensic reference file unavailable."
    )


# Fallback segmentation only if frozen Segment_ID was unavailable.

if "Real_Segment_ID" not in cic_minutes.columns:

    print(
        "\nConstructing fallback contiguous CIC segments."
    )


    cic_minutes = (

        cic_minutes

        .sort_values(
            "Minute"
        )

        .reset_index(
            drop=True
        )
    )


    minute_gap = (

        cic_minutes[
            "Minute"
        ]

        .diff()

        .dt.total_seconds()

        .div(
            60
        )
    )


    new_segment = (

        minute_gap.isna()

        |

        (
            minute_gap
            !=
            1
        )
    )


    cic_minutes[
        "Fallback_Segment_Number"
    ] = new_segment.cumsum()


    cic_minutes[
        "Real_Segment_ID"
    ] = (

        "CIC::SEG_"

        +

        cic_minutes[
            "Fallback_Segment_Number"
        ]

        .astype(
            int
        )

        .astype(
            str
        )

        .str.zfill(
            3
        )
    )


print(
    "CIC calibration segments:",
    cic_minutes[
        "Real_Segment_ID"
    ].nunique()
)


# ==================================================================================================
# 11.1M — PROCESS UNSW RAW FLOWS USING FROZEN 10.12 SEMANTICS
# ==================================================================================================

banner(
    "11.1M — PROCESS UNSW RAW FLOWS USING FROZEN 10.12 SEMANTICS"
)


UNSW_REQUIRED_RAW_COLUMNS = [

    "Stime",

    "dur",

    "Spkts",

    "Dpkts",

    "sbytes",

    "dbytes",

    "attack_cat",

    "Label"
]


for c in UNSW_REQUIRED_RAW_COLUMNS:

    if c not in UNSW_RAW_COLUMNS:

        raise ValueError(
            f"UNSW raw schema missing {c}"
        )


unsw_minute_partials = []

unsw_family_minute_partials = []

unsw_flow_samples = []


unsw_raw_label_counter = Counter()

unsw_family_flow_counter = Counter()


unsw_processed_rows = 0

unsw_invalid_timestamp_rows = 0

unsw_invalid_numeric_rows = 0

unsw_invalid_any_rows = 0


unsw_raw_vs_frozen_state_mismatches = 0

unsw_raw_vs_frozen_family_mismatches = 0

unsw_attack_rows_compared_for_family = 0


unsw_file_accounting_rows = []


for file_number, path in enumerate(
    UNSW_FILES,
    start=1
):

    print(
        "\n"
        +
        "-" * 120
    )

    print(
        f"[UNSW {file_number}/4] {path.name}"
    )

    print(
        path
    )

    print(
        "-" * 120
    )


    normalized_file = normalize_filename(
        path.name
    )


    if normalized_file not in UNSW_SEMANTICS_BY_FILE:

        raise RuntimeError(
            f"No frozen semantics found for {path.name}"
        )


    frozen_for_file = UNSW_SEMANTICS_BY_FILE[
        normalized_file
    ]


    expected_file_rows = len(
        frozen_for_file[
            "attack_state"
        ]
    )


    file_row_counter = 0

    file_attack_rows = 0


    reader = pd.read_csv(

        path,

        header=None,

        names=UNSW_RAW_COLUMNS,

        usecols=UNSW_REQUIRED_RAW_COLUMNS,

        chunksize=CHUNK_SIZE,

        low_memory=False
    )


    for chunk_number, chunk in enumerate(
        reader,
        start=1
    ):

        n = len(
            chunk
        )


        chunk_start = (
            file_row_counter
        )

        chunk_end = (
            file_row_counter
            +
            n
        )


        if chunk_end > expected_file_rows:

            raise RuntimeError(
                f"Raw UNSW file {path.name} exceeds frozen semantic row count."
            )


        frozen_state = (

            frozen_for_file[
                "attack_state"
            ][
                chunk_start:
                chunk_end
            ]
        )


        frozen_family_code = (

            frozen_for_file[
                "family_code"
            ][
                chunk_start:
                chunk_end
            ]
        )


        frozen_canonical_family = (

            frozen_for_file[
                "canonical_family"
            ][
                chunk_start:
                chunk_end
            ]
        )


        if len(
            frozen_state
        ) != n:

            raise RuntimeError(
                f"Frozen UNSW semantic slice length mismatch for {path.name}."
            )


        dataset_row_ids = np.arange(

            unsw_processed_rows,

            unsw_processed_rows
            +
            n,

            dtype=np.int64
        )


        # ------------------------------------------------------------------------------------------
        # RAW TIMESTAMP / NUMERIC FIELDS
        # ------------------------------------------------------------------------------------------

        timestamps = parse_unsw_stime(
            chunk[
                "Stime"
            ]
        )


        invalid_timestamp_mask = (
            timestamps.isna()
        )


        unsw_invalid_timestamp_rows += int(
            invalid_timestamp_mask.sum()
        )


        duration = pd.to_numeric(

            chunk[
                "dur"
            ],

            errors="coerce"
        )


        forward_packets = pd.to_numeric(

            chunk[
                "Spkts"
            ],

            errors="coerce"
        )


        backward_packets = pd.to_numeric(

            chunk[
                "Dpkts"
            ],

            errors="coerce"
        )


        forward_bytes = pd.to_numeric(

            chunk[
                "sbytes"
            ],

            errors="coerce"
        )


        backward_bytes = pd.to_numeric(

            chunk[
                "dbytes"
            ],

            errors="coerce"
        )


        numeric_frame = pd.DataFrame({

            "Flow_Duration_Seconds":
                duration,

            "Forward_Packets":
                forward_packets,

            "Backward_Packets":
                backward_packets,

            "Forward_Bytes":
                forward_bytes,

            "Backward_Bytes":
                backward_bytes
        })


        invalid_numeric_mask = (

            numeric_frame

            .isna()

            .any(
                axis=1
            )
        )


        unsw_invalid_numeric_rows += int(
            invalid_numeric_mask.sum()
        )


        invalid_any_mask = (

            invalid_timestamp_mask

            |

            invalid_numeric_mask
        )


        unsw_invalid_any_rows += int(
            invalid_any_mask.sum()
        )


        # ------------------------------------------------------------------------------------------
        # AUDIT RAW LABEL vs FROZEN STATE
        # ------------------------------------------------------------------------------------------

        raw_binary = pd.to_numeric(

            chunk[
                "Label"
            ],

            errors="coerce"
        )


        if raw_binary.isna().any():

            raise RuntimeError(
                f"Invalid raw UNSW binary Label in {path.name}."
            )


        raw_binary = raw_binary.astype(
            np.int8
        )


        raw_state_mismatch = int(

            (
                raw_binary.to_numpy(
                    dtype=np.int8
                )

                !=

                frozen_state
            ).sum()
        )


        unsw_raw_vs_frozen_state_mismatches += (
            raw_state_mismatch
        )


        raw_attack_family = (

            chunk[
                "attack_cat"
            ]

            .apply(
                canonicalize_unsw_family
            )
        )


        attack_mask = (
            frozen_state
            ==
            1
        )


        raw_family_array = raw_attack_family.to_numpy(
            dtype=object
        )


        family_compare_mask = attack_mask


        family_mismatch = int(

            (
                raw_family_array[
                    family_compare_mask
                ]

                !=

                frozen_canonical_family[
                    family_compare_mask
                ]
            ).sum()
        )


        unsw_raw_vs_frozen_family_mismatches += (
            family_mismatch
        )


        unsw_attack_rows_compared_for_family += int(
            family_compare_mask.sum()
        )


        # ------------------------------------------------------------------------------------------
        # AUTHORITATIVE SYNTHETIC FAMILY
        # ------------------------------------------------------------------------------------------

        synthetic_family = pd.Series(

            [
                CODE_TO_FAMILY.get(
                    int(
                        code
                    ),
                    None
                )

                for code in frozen_family_code
            ],

            index=chunk.index,

            dtype="object"
        )


        attack_state = pd.Series(

            frozen_state,

            index=chunk.index,

            dtype=np.int8
        )


        file_attack_rows += int(
            attack_state.sum()
        )


        # ------------------------------------------------------------------------------------------
        # RAW LABEL AUDIT COUNTS
        # ------------------------------------------------------------------------------------------

        audit_labels = np.where(

            frozen_state
            ==
            0,

            "benign",

            frozen_canonical_family
        )


        raw_counts = pd.Series(
            audit_labels
        ).value_counts(
            dropna=False
        )


        for label_name, count in raw_counts.items():

            unsw_raw_label_counter[
                str(
                    label_name
                )
            ] += int(
                count
            )


        family_counts = (

            synthetic_family.loc[
                attack_state
                ==
                1
            ]

            .value_counts(
                dropna=True
            )
        )


        for family, count in family_counts.items():

            unsw_family_flow_counter[
                family
            ] += int(
                count
            )


        # ------------------------------------------------------------------------------------------
        # COMMON FLOW FRAME
        # ------------------------------------------------------------------------------------------

        flow = pd.DataFrame({

            "Minute":
                timestamps.dt.floor(
                    "min"
                ),

            "Flow_Duration_Seconds":
                duration,

            "Forward_Packets":
                forward_packets,

            "Backward_Packets":
                backward_packets,

            "Forward_Bytes":
                forward_bytes,

            "Backward_Bytes":
                backward_bytes,

            "Attack_State":
                attack_state,

            "Synthetic_Family":
                synthetic_family
        })


        valid_mask = (
            ~invalid_any_mask
        )


        flow_valid = (

            flow.loc[
                valid_mask
            ]

            .reset_index(
                drop=True
            )
        )


        row_ids_valid = dataset_row_ids[
            valid_mask.to_numpy()
        ]


        unsw_minute_partials.append(

            build_chunk_minute_partial(
                flow_valid
            )
        )


        unsw_family_minute_partials.append(

            build_family_minute_partial(
                flow_valid,
                "UNSW_NB15"
            )
        )


        unsw_flow_samples.append(

            build_flow_profile_sample(

                flow_valid,

                "UNSW_NB15",

                row_ids_valid
            )
        )


        unsw_processed_rows += (
            n
        )

        file_row_counter += (
            n
        )


        print(
            f"  chunk {chunk_number}: "
            f"{n:,} rows | "
            f"file cumulative {file_row_counter:,}"
        )


    if file_row_counter != expected_file_rows:

        raise RuntimeError(
            f"{path.name}: raw rows {file_row_counter:,} "
            f"!= frozen semantics {expected_file_rows:,}"
        )


    unsw_file_accounting_rows.append(
        {
            "File":
                path.name,

            "Rows":
                file_row_counter,

            "Attack_Rows_Frozen":
                file_attack_rows
        }
    )


    print(
        f"✓ {path.name}: {file_row_counter:,} rows"
    )


unsw_file_accounting = pd.DataFrame(
    unsw_file_accounting_rows
)


print(
    "\nUNSW rows processed:",
    f"{unsw_processed_rows:,}"
)

print(
    "UNSW invalid timestamps:",
    f"{unsw_invalid_timestamp_rows:,}"
)

print(
    "UNSW invalid numeric rows:",
    f"{unsw_invalid_numeric_rows:,}"
)

print(
    "UNSW raw-vs-frozen state mismatches:",
    f"{unsw_raw_vs_frozen_state_mismatches:,}"
)

print(
    "UNSW raw-vs-frozen family mismatches:",
    f"{unsw_raw_vs_frozen_family_mismatches:,}"
)


# ==================================================================================================
# 11.1N — FINALIZE UNSW RAW MINUTE RECONSTRUCTION
# ==================================================================================================

banner(
    "11.1N — FINALIZE UNSW 1-MINUTE RAW RECONSTRUCTION"
)


unsw_minutes_rebuilt = finalize_minute_aggregation(
    unsw_minute_partials
)


unsw_minutes_rebuilt[
    "Calibration_Source"
] = "UNSW_NB15"


print(
    "UNSW rebuilt observed minutes:",
    f"{len(unsw_minutes_rebuilt):,}"
)

print(
    "UNSW rebuilt flow count:",
    f"{int(unsw_minutes_rebuilt['Flow_Count'].sum()):,}"
)

print(
    "UNSW rebuilt attack minutes:",
    f"{int(unsw_minutes_rebuilt['Attack_State'].sum()):,}"
)


# ==================================================================================================
# 11.1O — VERIFY UNSW AGAINST FROZEN PHASE 10.15
#
# The frozen 10.15 representation remains authoritative for final
# minute-level calibration predictors.
#
# Raw reconstruction verifies:
#
#   ✓ minute grid
#   ✓ flow count
#   ✓ Attack_State
#
# ==================================================================================================

banner(
    "11.1O — VERIFY UNSW AGAINST FROZEN PHASE 10.15"
)


unsw_ref = pd.read_csv(
    UNSW_REFERENCE_MINUTE_FILE,
    low_memory=False
)


required_unsw_ref_columns = [

    "Capture_ID",

    "Temporal_Segment_ID",

    "Minute",

    "Attack_State",

    *FINAL_PREDICTORS
]


missing_unsw_ref = [

    c

    for c in required_unsw_ref_columns

    if c not in unsw_ref.columns
]


if missing_unsw_ref:

    raise ValueError(
        f"Frozen UNSW minute representation missing: {missing_unsw_ref}"
    )


unsw_ref[
    "Minute"
] = pd.to_datetime(
    unsw_ref[
        "Minute"
    ],
    errors="coerce"
)


if unsw_ref[
    "Minute"
].isna().any():

    raise RuntimeError(
        "Frozen UNSW reference has invalid minute timestamps."
    )


unsw_basic_compare = (

    unsw_minutes_rebuilt[
        [
            "Minute",
            "Flow_Count",
            "Attack_State"
        ]
    ]

    .merge(

        unsw_ref[
            [
                "Minute",
                "Flow_Count",
                "Attack_State"
            ]
        ],

        on="Minute",

        how="outer",

        suffixes=(
            "_REBUILT",
            "_REFERENCE"
        ),

        indicator=True
    )
)


unsw_grid_match = bool(

    (
        unsw_basic_compare[
            "_merge"
        ]

        ==
        "both"
    ).all()
)


unsw_flow_count_match = bool(

    (
        unsw_basic_compare[
            "Flow_Count_REBUILT"
        ]

        ==

        unsw_basic_compare[
            "Flow_Count_REFERENCE"
        ]
    ).all()
)


unsw_attack_state_match = bool(

    (
        unsw_basic_compare[
            "Attack_State_REBUILT"
        ]

        ==

        unsw_basic_compare[
            "Attack_State_REFERENCE"
        ]
    ).all()
)


print(
    "Minute grid exact match:",
    unsw_grid_match
)

print(
    "Flow count exact match :",
    unsw_flow_count_match
)

print(
    "Attack state exact match:",
    unsw_attack_state_match
)


if not (
    unsw_grid_match
    and
    unsw_flow_count_match
    and
    unsw_attack_state_match
):

    raise RuntimeError(
        "UNSW raw reconstruction disagrees with frozen Phase 10.15."
    )


# Final minute-level UNSW predictors come from frozen 10.15.

unsw_minutes = unsw_ref[
    required_unsw_ref_columns
].copy()


unsw_minutes[
    "Calibration_Source"
] = "UNSW_NB15"


unsw_minutes[
    "Real_Segment_ID"
] = (

    "UNSW::"

    +

    unsw_minutes[
        "Temporal_Segment_ID"
    ].astype(
        str
    )
)


print(
    "\n✓ Frozen Phase 10.15 UNSW minute representation retained"
)


# ==================================================================================================
# 11.1P — FAMILY ATTACK-FLOW INTENSITY
# ==================================================================================================

banner(
    "11.1P — BUILD FAMILY ATTACK-FLOW INTENSITY TABLE"
)


family_minute_counts = pd.concat(

    [
        *cic_family_minute_partials,
        *unsw_family_minute_partials
    ],

    ignore_index=True
)


family_minute_counts = (

    family_minute_counts

    .groupby(
        [
            "Calibration_Source",
            "Minute",
            "Synthetic_Family"
        ],
        observed=True,
        sort=True
    )[
        "Attack_Flows"
    ]

    .sum()

    .reset_index()
)


family_minute_support = (

    family_minute_counts

    .groupby(
        [
            "Calibration_Source",
            "Synthetic_Family"
        ],
        observed=True
    )

    .agg(

        Family_Active_Minutes=(
            "Minute",
            "nunique"
        ),

        Total_Mapped_Attack_Flows=(
            "Attack_Flows",
            "sum"
        )

    )

    .reset_index()
)


print(
    family_minute_support.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.1Q — RAW LABEL → SYNTHETIC FAMILY AUDIT
# ==================================================================================================

banner(
    "11.1Q — REAL LABEL → SYNTHETIC FAMILY AUDIT"
)


label_audit_rows = []


for raw_label, count in sorted(
    cic_label_counter.items()
):

    if raw_label == "benign":

        synthetic_family = "BENIGN"

        mapped = True

    else:

        synthetic_family = CIC_LABEL_TO_FAMILY.get(
            raw_label
        )

        mapped = synthetic_family is not None


    label_audit_rows.append(
        {
            "Calibration_Source":
                "CIC_IDS2017",

            "Real_Label":
                raw_label,

            "Flow_Count":
                int(
                    count
                ),

            "Synthetic_Family":
                (
                    synthetic_family
                    if synthetic_family is not None
                    else "UNMAPPED"
                ),

            "Used_For_Selected_Family_Calibration":
                bool(
                    mapped
                )
        }
    )


for raw_label, count in sorted(
    unsw_raw_label_counter.items()
):

    if raw_label == "benign":

        synthetic_family = "BENIGN"

        mapped = True

    else:

        synthetic_family = UNSW_FINAL_FAMILY_TO_SYNTHETIC.get(
            raw_label
        )

        mapped = synthetic_family is not None


    label_audit_rows.append(
        {
            "Calibration_Source":
                "UNSW_NB15",

            "Real_Label":
                raw_label,

            "Flow_Count":
                int(
                    count
                ),

            "Synthetic_Family":
                (
                    synthetic_family
                    if synthetic_family is not None
                    else "UNMAPPED"
                ),

            "Used_For_Selected_Family_Calibration":
                bool(
                    mapped
                )
        }
    )


label_mapping_audit = pd.DataFrame(
    label_audit_rows
)


print(
    label_mapping_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.1R — EXACT FAMILY FLOW SUPPORT
# ==================================================================================================

banner(
    "11.1R — REAL ATTACK FLOW SUPPORT BY SYNTHETIC FAMILY"
)


family_support_rows = []


for family in ATTACK_FAMILIES:

    cic_count = int(
        cic_family_flow_counter.get(
            family,
            0
        )
    )


    unsw_count = int(
        unsw_family_flow_counter.get(
            family,
            0
        )
    )


    total_count = (
        cic_count
        +
        unsw_count
    )


    family_support_rows.append(
        {
            "Synthetic_Family":
                family,

            "CIC_Attack_Flows":
                cic_count,

            "UNSW_Attack_Flows":
                unsw_count,

            "Total_Real_Attack_Flows":
                total_count,

            "Sources_With_Support":
                int(
                    cic_count
                    >
                    0
                )
                +
                int(
                    unsw_count
                    >
                    0
                ),

            "Meets_1000_Flow_Gate_PRELIMINARY":
                total_count
                >=
                1000
        }
    )


family_support = pd.DataFrame(
    family_support_rows
)


print(
    family_support.to_string(
        index=False
    )
)


print(
    """
NOTE:
This is only a preliminary quantitative support check.

Phase 11.2 decides whether each family is actually suitable.
"""
)


# ==================================================================================================
# 11.1S — BUILD FLOW PROFILE SAMPLE
# ==================================================================================================

banner(
    "11.1S — BUILD REAL FLOW PROFILE SAMPLE"
)


flow_profile_sample = pd.concat(

    [
        *cic_flow_samples,
        *unsw_flow_samples
    ],

    ignore_index=True
)


profile_distribution = (

    flow_profile_sample

    .groupby(
        [
            "Calibration_Source",
            "Profile_Class"
        ],
        observed=True
    )

    .size()

    .rename(
        "Sample_Rows"
    )

    .reset_index()
)


print(
    "Flow profile sample rows:",
    f"{len(flow_profile_sample):,}"
)


print(
    "\nProfile distribution:\n"
)

print(
    profile_distribution.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.1T — FLOW PROFILE QUANTILES
# ==================================================================================================

banner(
    "11.1T — FLOW PROFILE QUANTILES"
)


flow_quantile_rows = []


for (
    source,
    profile_class
), group in flow_profile_sample.groupby(

    [
        "Calibration_Source",
        "Profile_Class"
    ],

    observed=True
):

    for feature in FLOW_PROFILE_FEATURES:

        values = pd.to_numeric(

            group[
                feature
            ],

            errors="coerce"
        )


        values = values[
            np.isfinite(
                values
            )
        ]


        if len(
            values
        ) == 0:

            continue


        q = values.quantile(
            QUANTILES
        )


        row = {

            "Calibration_Source":
                source,

            "Profile_Class":
                profile_class,

            "Feature":
                feature,

            "Sample_Count":
                len(
                    values
                ),

            "Mean":
                float(
                    values.mean()
                ),

            "Std":
                (
                    float(
                        values.std(
                            ddof=1
                        )
                    )

                    if len(
                        values
                    )
                    >
                    1

                    else 0.0
                ),

            "Min":
                float(
                    values.min()
                ),

            "Max":
                float(
                    values.max()
                )
        }


        for quantile in QUANTILES:

            row[
                QUANTILE_NAMES[
                    quantile
                ]
            ] = float(
                q.loc[
                    quantile
                ]
            )


        flow_quantile_rows.append(
            row
        )


flow_profile_quantiles = pd.DataFrame(
    flow_quantile_rows
)


print(
    "Flow-profile quantile rows:",
    len(
        flow_profile_quantiles
    )
)


# ==================================================================================================
# 11.1U — FAMILY vs BENIGN EFFECT VECTORS
#
# Each source computes:
#
#   median(log1p(family feature))
#   -
#   median(log1p(benign feature))
#
# If family exists in both CIC and UNSW:
#
#   pooled effect = median(source effects)
#
# This prevents UNSW or CIC from dominating simply because it has more flows.
# ==================================================================================================

banner(
    "11.1U — ROBUST FAMILY vs BENIGN LOG-SPACE EFFECT VECTORS"
)


source_effect_rows = []


for source, source_df in flow_profile_sample.groupby(
    "Calibration_Source",
    observed=True
):

    benign = source_df.loc[
        source_df[
            "Profile_Class"
        ]
        ==
        "BENIGN"
    ]


    for family in ATTACK_FAMILIES:

        family_df = source_df.loc[
            source_df[
                "Profile_Class"
            ]
            ==
            family
        ]


        if (
            len(
                benign
            )
            ==
            0
            or
            len(
                family_df
            )
            ==
            0
        ):

            continue


        for feature in FLOW_PROFILE_FEATURES:

            benign_values = pd.to_numeric(

                benign[
                    feature
                ],

                errors="coerce"
            )


            family_values = pd.to_numeric(

                family_df[
                    feature
                ],

                errors="coerce"
            )


            benign_values = benign_values.loc[
                np.isfinite(
                    benign_values
                )
                &
                (
                    benign_values
                    >=
                    0
                )
            ]


            family_values = family_values.loc[
                np.isfinite(
                    family_values
                )
                &
                (
                    family_values
                    >=
                    0
                )
            ]


            if (
                len(
                    benign_values
                )
                ==
                0
                or
                len(
                    family_values
                )
                ==
                0
            ):

                continue


            benign_log_median = float(

                np.median(

                    np.log1p(
                        benign_values
                    )
                )
            )


            family_log_median = float(

                np.median(

                    np.log1p(
                        family_values
                    )
                )
            )


            source_effect_rows.append(
                {
                    "Calibration_Source":
                        source,

                    "Synthetic_Family":
                        family,

                    "Feature":
                        feature,

                    "Benign_Log1p_Median":
                        benign_log_median,

                    "Family_Log1p_Median":
                        family_log_median,

                    "Effect_Log1p":
                        (
                            family_log_median
                            -
                            benign_log_median
                        ),

                    "Benign_Sample_Count":
                        len(
                            benign_values
                        ),

                    "Family_Sample_Count":
                        len(
                            family_values
                        )
                }
            )


source_family_effects = pd.DataFrame(
    source_effect_rows
)


pooled_effect_rows = []


for (
    family,
    feature
), group in source_family_effects.groupby(

    [
        "Synthetic_Family",
        "Feature"
    ],

    observed=True
):

    pooled_effect_rows.append(
        {
            "Synthetic_Family":
                family,

            "Feature":
                feature,

            "Available_Source_Count":
                group[
                    "Calibration_Source"
                ].nunique(),

            "Effect_Log1p":
                float(
                    group[
                        "Effect_Log1p"
                    ].median()
                ),

            "Pooling_Method":
                "MEDIAN_ACROSS_AVAILABLE_REAL_SOURCES"
        }
    )


pooled_family_effects = pd.DataFrame(
    pooled_effect_rows
)


print(
    "Source-specific effect rows:",
    len(
        source_family_effects
    )
)

print(
    "Pooled effect rows:",
    len(
        pooled_family_effects
    )
)


# ==================================================================================================
# 11.1V — FAMILY ATTACK-FLOW INTENSITY DISTRIBUTIONS
# ==================================================================================================

banner(
    "11.1V — FAMILY ATTACK-FLOW INTENSITY DISTRIBUTIONS"
)


family_intensity_rows = []


def summarize_attack_intensity(
    source,
    family,
    values
):

    values = pd.to_numeric(
        values,
        errors="coerce"
    )


    values = values.loc[
        np.isfinite(
            values
        )
    ]


    if len(
        values
    ) == 0:

        return None


    q = values.quantile(
        QUANTILES
    )


    row = {

        "Calibration_Source":
            source,

        "Synthetic_Family":
            family,

        "Family_Active_Minutes":
            len(
                values
            ),

        "Mean_Attack_Flows_Per_Active_Minute":
            float(
                values.mean()
            ),

        "Max_Attack_Flows_Per_Active_Minute":
            float(
                values.max()
            )
    }


    for quantile in QUANTILES:

        row[
            QUANTILE_NAMES[
                quantile
            ]
        ] = float(
            q.loc[
                quantile
            ]
        )


    return row


for (
    source,
    family
), group in family_minute_counts.groupby(

    [
        "Calibration_Source",
        "Synthetic_Family"
    ],

    observed=True
):

    row = summarize_attack_intensity(

        source,

        family,

        group[
            "Attack_Flows"
        ]
    )


    if row is not None:

        family_intensity_rows.append(
            row
        )


for family, group in family_minute_counts.groupby(
    "Synthetic_Family",
    observed=True
):

    row = summarize_attack_intensity(

        "POOLED_REAL",

        family,

        group[
            "Attack_Flows"
        ]
    )


    if row is not None:

        family_intensity_rows.append(
            row
        )


family_intensity_quantiles = pd.DataFrame(
    family_intensity_rows
)


print(
    family_intensity_quantiles.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.1W — BUILD STRICT BENIGN MINUTE POOL
#
# STRICT BENIGN means:
#
#       Attack_State == 0
#
# Any minute containing even one real attack flow is excluded.
# ==================================================================================================

banner(
    "11.1W — BUILD STRICT REAL BENIGN MINUTE POOL"
)


cic_benign_minutes = (

    cic_minutes.loc[
        cic_minutes[
            "Attack_State"
        ]
        ==
        0,
        [
            "Calibration_Source",
            "Real_Segment_ID",
            "Minute",
            *FINAL_PREDICTORS
        ]
    ]

    .copy()
)


unsw_benign_minutes = (

    unsw_minutes.loc[
        unsw_minutes[
            "Attack_State"
        ]
        ==
        0,
        [
            "Calibration_Source",
            "Real_Segment_ID",
            "Minute",
            *FINAL_PREDICTORS
        ]
    ]

    .copy()
)


benign_minute_pool = pd.concat(

    [
        cic_benign_minutes,
        unsw_benign_minutes
    ],

    ignore_index=True
)


benign_minute_pool = benign_minute_pool.rename(
    columns={
        "Minute":
            "Real_Minute"
    }
)


print(
    "CIC strict benign minutes :",
    f"{len(cic_benign_minutes):,}"
)

print(
    "UNSW strict benign minutes:",
    f"{len(unsw_benign_minutes):,}"
)

print(
    "Pooled strict benign minutes:",
    f"{len(benign_minute_pool):,}"
)


# ==================================================================================================
# 11.1X — REAL MINUTE FEATURE QUANTILES
# ==================================================================================================

banner(
    "11.1X — REAL MINUTE-LEVEL FEATURE QUANTILES"
)


all_real_minutes = pd.concat(

    [
        cic_minutes[
            [
                "Calibration_Source",
                "Attack_State",
                *FINAL_PREDICTORS
            ]
        ],

        unsw_minutes[
            [
                "Calibration_Source",
                "Attack_State",
                *FINAL_PREDICTORS
            ]
        ]
    ],

    ignore_index=True
)


minute_quantile_rows = []


def add_minute_quantiles(
    group_name,
    dataframe
):

    for feature in FINAL_PREDICTORS:

        values = pd.to_numeric(

            dataframe[
                feature
            ],

            errors="coerce"
        )


        values = values.loc[
            np.isfinite(
                values
            )
        ]


        if len(
            values
        ) == 0:

            continue


        q = values.quantile(
            QUANTILES
        )


        row = {

            "Calibration_Group":
                group_name,

            "Feature":
                feature,

            "Minute_Count":
                len(
                    values
                ),

            "Mean":
                float(
                    values.mean()
                ),

            "Std":
                (
                    float(
                        values.std(
                            ddof=1
                        )
                    )

                    if len(
                        values
                    )
                    >
                    1

                    else 0.0
                ),

            "Min":
                float(
                    values.min()
                ),

            "Max":
                float(
                    values.max()
                )
        }


        for quantile in QUANTILES:

            row[
                QUANTILE_NAMES[
                    quantile
                ]
            ] = float(
                q.loc[
                    quantile
                ]
            )


        minute_quantile_rows.append(
            row
        )


for source, group in all_real_minutes.groupby(
    "Calibration_Source",
    observed=True
):

    add_minute_quantiles(

        f"{source}::ALL",

        group
    )


    add_minute_quantiles(

        f"{source}::BENIGN",

        group.loc[
            group[
                "Attack_State"
            ]
            ==
            0
        ]
    )


    add_minute_quantiles(

        f"{source}::ATTACK_ACTIVE",

        group.loc[
            group[
                "Attack_State"
            ]
            ==
            1
        ]
    )


add_minute_quantiles(

    "POOLED_REAL::ALL",

    all_real_minutes
)


add_minute_quantiles(

    "POOLED_REAL::BENIGN",

    all_real_minutes.loc[
        all_real_minutes[
            "Attack_State"
        ]
        ==
        0
    ]
)


real_minute_feature_quantiles = pd.DataFrame(
    minute_quantile_rows
)


print(
    "Minute quantile rows:",
    len(
        real_minute_feature_quantiles
    )
)


# ==================================================================================================
# 11.1Y — BUILD NON-OVERLAPPING 5-MINUTE BENIGN BLOCKS
#
# Real calibration blocks are also separated into:
#
#       CAL_TRAIN
#       CAL_VALIDATION
#       CAL_TEST
#
# A real block belongs to EXACTLY ONE calibration pool.
#
# Later:
#
# synthetic TRAIN       → CAL_TRAIN only
# synthetic VALIDATION  → CAL_VALIDATION only
# synthetic TEST        → CAL_TEST only
#
# ==================================================================================================

banner(
    "11.1Y — BUILD DISJOINT 5-MINUTE REAL BENIGN BLOCKS"
)


def stable_hash_integer(
    text
):

    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


    return int(
        digest[:16],
        16
    )


def build_non_overlapping_benign_blocks(
    dataframe,
    source,
    block_length
):

    dataframe = (

        dataframe

        .sort_values(
            [
                "Real_Segment_ID",
                "Real_Minute"
            ]
        )

        .reset_index(
            drop=True
        )
    )


    output = []

    global_run_counter = 0


    for segment_id, segment in dataframe.groupby(
        "Real_Segment_ID",
        observed=True,
        sort=False
    ):

        segment = (

            segment

            .sort_values(
                "Real_Minute"
            )

            .reset_index(
                drop=True
            )
        )


        gap = (

            segment[
                "Real_Minute"
            ]

            .diff()

            .dt.total_seconds()

            .div(
                60
            )
        )


        new_run = (

            gap.isna()

            |

            (
                gap
                !=
                1
            )
        )


        segment[
            "_Run"
        ] = new_run.cumsum()


        for _, run in segment.groupby(
            "_Run",
            observed=True,
            sort=False
        ):

            global_run_counter += (
                1
            )


            run = run.reset_index(
                drop=True
            )


            full_block_count = (

                len(
                    run
                )

                //

                block_length
            )


            for block_number in range(
                full_block_count
            ):

                start = (

                    block_number
                    *
                    block_length
                )


                end = (

                    start
                    +
                    block_length
                )


                block = (

                    run.iloc[
                        start:end
                    ]

                    .copy()
                )


                block_id = (

                    f"{source}"
                    f"::RUN_{global_run_counter:04d}"
                    f"::BLOCK_{block_number + 1:04d}"
                )


                block[
                    "Benign_Block_ID"
                ] = block_id


                block[
                    "Block_Row_Position"
                ] = np.arange(
                    block_length
                )


                output.append(
                    block
                )


    if not output:

        return pd.DataFrame()


    return pd.concat(
        output,
        ignore_index=True
    )


benign_block_frames = []


for source, group in benign_minute_pool.groupby(
    "Calibration_Source",
    observed=True
):

    source_blocks = build_non_overlapping_benign_blocks(

        group,

        source,

        BENIGN_BLOCK_LENGTH
    )


    if len(
        source_blocks
    ) > 0:

        benign_block_frames.append(
            source_blocks
        )


benign_block_rows = pd.concat(
    benign_block_frames,
    ignore_index=True
)


benign_block_index = (

    benign_block_rows

    .groupby(
        [
            "Calibration_Source",
            "Benign_Block_ID"
        ],
        observed=True,
        sort=False
    )

    .agg(

        Real_Segment_ID=(
            "Real_Segment_ID",
            "first"
        ),

        Block_Start=(
            "Real_Minute",
            "min"
        ),

        Block_End=(
            "Real_Minute",
            "max"
        ),

        Block_Length=(
            "Real_Minute",
            "size"
        )

    )

    .reset_index()
)


# Split blocks independently inside each source.

block_pool_frames = []


for source, group in benign_block_index.groupby(
    "Calibration_Source",
    observed=True
):

    group = group.copy()


    group[
        "_Hash"
    ] = group[
        "Benign_Block_ID"
    ].apply(

        lambda block_id:

        stable_hash_integer(
            f"{MASTER_SEED}|{block_id}"
        )
    )


    group = (

        group

        .sort_values(
            "_Hash"
        )

        .reset_index(
            drop=True
        )
    )


    n = len(
        group
    )


    n_train = int(
        np.floor(
            0.70
            *
            n
        )
    )


    n_validation = int(
        np.floor(
            0.15
            *
            n
        )
    )


    n_test = (

        n
        -
        n_train
        -
        n_validation
    )


    pool = np.array(

        (
            [
                "CAL_TRAIN"
            ]
            *
            n_train
        )

        +

        (
            [
                "CAL_VALIDATION"
            ]
            *
            n_validation
        )

        +

        (
            [
                "CAL_TEST"
            ]
            *
            n_test
        ),

        dtype=object
    )


    group[
        "Calibration_Pool"
    ] = pool


    group = group.drop(
        columns=[
            "_Hash"
        ]
    )


    block_pool_frames.append(
        group
    )


benign_block_index = pd.concat(
    block_pool_frames,
    ignore_index=True
)


benign_block_rows = benign_block_rows.merge(

    benign_block_index[
        [
            "Calibration_Source",
            "Benign_Block_ID",
            "Calibration_Pool"
        ]
    ],

    on=[
        "Calibration_Source",
        "Benign_Block_ID"
    ],

    how="left",

    validate="many_to_one"
)


block_pool_summary = (

    benign_block_index

    .groupby(
        [
            "Calibration_Source",
            "Calibration_Pool"
        ],
        observed=True
    )

    .size()

    .rename(
        "Block_Count"
    )

    .reset_index()
)


print(
    "Total usable 5-minute benign blocks:",
    f"{len(benign_block_index):,}"
)


print(
    "\nBlock pool summary:"
)

print(
    block_pool_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.1Z — BLOCK INTEGRITY
# ==================================================================================================

banner(
    "11.1Z — BENIGN BLOCK INTEGRITY"
)


duplicate_real_minute_membership = int(

    benign_block_rows

    .duplicated(
        subset=[
            "Calibration_Source",
            "Real_Segment_ID",
            "Real_Minute"
        ]
    )

    .sum()
)


wrong_block_lengths = int(

    (
        benign_block_index[
            "Block_Length"
        ]

        !=

        BENIGN_BLOCK_LENGTH
    )

    .sum()
)


block_pool_counts = (

    benign_block_rows

    .groupby(
        "Benign_Block_ID",
        observed=True
    )[
        "Calibration_Pool"
    ]

    .nunique()
)


blocks_in_multiple_pools = int(

    (
        block_pool_counts
        !=
        1
    )

    .sum()
)


print(
    "Duplicate real-minute membership:",
    duplicate_real_minute_membership
)

print(
    "Wrong block lengths:",
    wrong_block_lengths
)

print(
    "Blocks in multiple pools:",
    blocks_in_multiple_pools
)


# ==================================================================================================
# 11.1AA — PREDICTOR QUALITY
# ==================================================================================================

banner(
    "11.1AA — REAL MINUTE PREDICTOR QUALITY"
)


predictor_quality_rows = []


for source, group in all_real_minutes.groupby(
    "Calibration_Source",
    observed=True
):

    matrix = (

        group[
            FINAL_PREDICTORS
        ]

        .apply(
            pd.to_numeric,
            errors="coerce"
        )

        .to_numpy(
            dtype=float
        )
    )


    predictor_quality_rows.append(
        {
            "Calibration_Source":
                source,

            "Minute_Rows":
                len(
                    group
                ),

            "Missing_Predictor_Cells":
                int(
                    np.isnan(
                        matrix
                    ).sum()
                ),

            "Infinite_Predictor_Cells":
                int(
                    np.isinf(
                        matrix
                    ).sum()
                ),

            "Negative_Predictor_Cells":
                int(
                    (
                        matrix
                        <
                        0
                    ).sum()
                )
        }
    )


predictor_quality = pd.DataFrame(
    predictor_quality_rows
)


print(
    predictor_quality.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.1AB — BENIGN SUPPORT SUMMARY
# ==================================================================================================

banner(
    "11.1AB — BENIGN SUPPORT SUMMARY"
)


benign_support_summary = (

    benign_minute_pool

    .groupby(
        "Calibration_Source",
        observed=True
    )

    .agg(

        Strict_Benign_Minutes=(
            "Real_Minute",
            "size"
        ),

        Real_Segments=(
            "Real_Segment_ID",
            "nunique"
        )

    )

    .reset_index()
)


print(
    benign_support_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.1AC — VALIDATION
#
# PHASE 11.1 PASS means:
#
#       calibration extraction is structurally valid.
#
# It does NOT yet mean:
#
#       all five attack families are scientifically approved.
#
# That happens in Phase 11.2.
# ==================================================================================================

banner(
    "11.1AC — PHASE 11.1 VALIDATION"
)


validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                details
        }
    )


add_check(

    "Frozen_11_0_Config_Unchanged",

    CURRENT_CONFIG_HASH
    ==
    FROZEN_CONFIG_HASH,

    CURRENT_CONFIG_HASH
)


add_check(

    "Eight_CIC_Raw_Files_Found",

    len(
        CIC_FILES
    )
    ==
    8,

    f"{len(CIC_FILES)}/8"
)


add_check(

    "Four_UNSW_Raw_Files_Found",

    len(
        UNSW_FILES
    )
    ==
    4,

    f"{len(UNSW_FILES)}/4"
)


# CIC accounting

add_check(

    "CIC_Physical_Row_Count",

    cic_physical_rows
    ==
    EXPECTED_CIC_PHYSICAL_ROWS,

    f"{cic_physical_rows:,}"
)


add_check(

    "CIC_Known_Empty_Trailing_Block_Removed",

    cic_empty_rows_removed
    ==
    EXPECTED_CIC_EMPTY_TRAILING_ROWS,

    f"{cic_empty_rows_removed:,}"
)


add_check(

    "CIC_Real_Traffic_Row_Count",

    cic_real_traffic_rows
    ==
    EXPECTED_CIC_REAL_TRAFFIC_ROWS,

    f"{cic_real_traffic_rows:,}"
)


add_check(

    "CIC_Physical_Row_Accounting",

    (
        cic_real_traffic_rows
        +
        cic_empty_rows_removed
    )
    ==
    cic_physical_rows,

    (
        f"{cic_real_traffic_rows:,} + "
        f"{cic_empty_rows_removed:,} = "
        f"{cic_physical_rows:,}"
    )
)


add_check(

    "CIC_No_Invalid_Timestamps_After_Empty_Removal",

    cic_invalid_timestamp_rows
    ==
    0,

    f"{cic_invalid_timestamp_rows:,}"
)


add_check(

    "CIC_No_Invalid_Common_Numeric_Fields",

    cic_invalid_numeric_rows
    ==
    0,

    f"{cic_invalid_numeric_rows:,}"
)


add_check(

    "CIC_All_Real_Flows_Recovered_To_Minutes",

    int(
        cic_minutes[
            "Flow_Count"
        ].sum()
    )
    ==
    cic_real_traffic_rows,

    (
        f"minutes={int(cic_minutes['Flow_Count'].sum()):,}; "
        f"real={cic_real_traffic_rows:,}"
    )
)


add_check(

    "CIC_Expected_Observed_Minutes",

    len(
        cic_minutes
    )
    ==
    EXPECTED_CIC_OBSERVED_MINUTES,

    f"{len(cic_minutes):,}"
)


add_check(

    "CIC_Expected_Attack_Minutes",

    int(
        cic_minutes[
            "Attack_State"
        ].sum()
    )
    ==
    EXPECTED_CIC_ATTACK_MINUTES,

    f"{int(cic_minutes['Attack_State'].sum()):,}"
)


add_check(

    "CIC_Frozen_Reference_Agreement",

    (
        cic_reference_exact_match
        is True
    )
    if cic_reference_available
    else True,

    (
        "grid/attack semantics agree"
        if cic_reference_available
        else "reference unavailable"
    )
)


# UNSW accounting

add_check(

    "UNSW_Expected_Raw_Flow_Count",

    unsw_processed_rows
    ==
    EXPECTED_UNSW_FLOWS,

    f"{unsw_processed_rows:,}"
)


add_check(

    "UNSW_No_Invalid_Timestamps",

    unsw_invalid_timestamp_rows
    ==
    0,

    f"{unsw_invalid_timestamp_rows:,}"
)


add_check(

    "UNSW_No_Invalid_Common_Numeric_Fields",

    unsw_invalid_numeric_rows
    ==
    0,

    f"{unsw_invalid_numeric_rows:,}"
)


add_check(

    "UNSW_Raw_vs_Frozen_Binary_State_Agreement",

    unsw_raw_vs_frozen_state_mismatches
    ==
    0,

    f"{unsw_raw_vs_frozen_state_mismatches:,} mismatches"
)


add_check(

    "UNSW_Raw_vs_Frozen_Family_Agreement",

    unsw_raw_vs_frozen_family_mismatches
    ==
    0,

    (
        f"{unsw_raw_vs_frozen_family_mismatches:,} mismatches "
        f"across {unsw_attack_rows_compared_for_family:,} attack flows"
    )
)


add_check(

    "UNSW_All_Flows_Recovered_To_Minutes",

    int(
        unsw_minutes_rebuilt[
            "Flow_Count"
        ].sum()
    )
    ==
    EXPECTED_UNSW_FLOWS,

    f"{int(unsw_minutes_rebuilt['Flow_Count'].sum()):,}"
)


add_check(

    "UNSW_Expected_Observed_Minutes",

    len(
        unsw_minutes
    )
    ==
    EXPECTED_UNSW_OBSERVED_MINUTES,

    f"{len(unsw_minutes):,}"
)


add_check(

    "UNSW_Frozen_Minute_Grid_Match",

    unsw_grid_match,

    str(
        unsw_grid_match
    )
)


add_check(

    "UNSW_Frozen_Flow_Count_Match",

    unsw_flow_count_match,

    str(
        unsw_flow_count_match
    )
)


add_check(

    "UNSW_Frozen_Attack_State_Match",

    unsw_attack_state_match,

    str(
        unsw_attack_state_match
    )
)


# Calibration structure

add_check(

    "Strict_Benign_Pool_Has_At_Least_500_Minutes",

    len(
        benign_minute_pool
    )
    >=
    500,

    f"{len(benign_minute_pool):,}"
)


for source in [
    "CIC_IDS2017",
    "UNSW_NB15"
]:

    count = int(

        (
            benign_minute_pool[
                "Calibration_Source"
            ]

            ==
            source
        )

        .sum()
    )


    add_check(

        f"{source}_Has_At_Least_100_Strict_Benign_Minutes",

        count
        >=
        100,

        str(
            count
        )
    )


add_check(

    "All_14_Predictors_Available",

    all(
        feature
        in
        benign_minute_pool.columns

        for feature
        in FINAL_PREDICTORS
    ),

    f"{len(FINAL_PREDICTORS)} predictors"
)


add_check(

    "No_Predictor_NaN_Or_Inf",

    bool(

        (
            predictor_quality[
                "Missing_Predictor_Cells"
            ]
            ==
            0
        ).all()

        and

        (
            predictor_quality[
                "Infinite_Predictor_Cells"
            ]
            ==
            0
        ).all()
    ),

    predictor_quality[
        [
            "Calibration_Source",
            "Missing_Predictor_Cells",
            "Infinite_Predictor_Cells"
        ]
    ].to_dict(
        orient="records"
    )
)


add_check(

    "No_Negative_Predictor_Values",

    bool(

        (
            predictor_quality[
                "Negative_Predictor_Cells"
            ]
            ==
            0
        ).all()
    ),

    predictor_quality[
        [
            "Calibration_Source",
            "Negative_Predictor_Cells"
        ]
    ].to_dict(
        orient="records"
    )
)


# Blocks

add_check(

    "Benign_Blocks_NonOverlapping",

    duplicate_real_minute_membership
    ==
    0,

    str(
        duplicate_real_minute_membership
    )
)


add_check(

    "Benign_Block_Lengths_Exactly_5",

    wrong_block_lengths
    ==
    0,

    str(
        wrong_block_lengths
    )
)


add_check(

    "Each_Block_Belongs_To_Exactly_One_Calibration_Pool",

    blocks_in_multiple_pools
    ==
    0,

    str(
        blocks_in_multiple_pools
    )
)


for source in [
    "CIC_IDS2017",
    "UNSW_NB15"
]:

    for pool in [
        "CAL_TRAIN",
        "CAL_VALIDATION",
        "CAL_TEST"
    ]:

        count = int(

            (
                (
                    benign_block_index[
                        "Calibration_Source"
                    ]
                    ==
                    source
                )

                &

                (
                    benign_block_index[
                        "Calibration_Pool"
                    ]
                    ==
                    pool
                )
            )

            .sum()
        )


        add_check(

            f"{source}_{pool}_Has_Blocks",

            count
            >
            0,

            str(
                count
            )
        )


# Families

add_check(

    "Every_Configured_Family_Has_Real_Support",

    bool(

        (
            family_support[
                "Total_Real_Attack_Flows"
            ]

            >
            0
        ).all()
    ),

    family_support[
        [
            "Synthetic_Family",
            "Total_Real_Attack_Flows"
        ]
    ].to_dict(
        orient="records"
    )
)


effect_family_set = set(

    pooled_family_effects[
        "Synthetic_Family"
    ].unique()
)


add_check(

    "Every_Configured_Family_Has_Effect_Vector",

    set(
        ATTACK_FAMILIES
    ).issubset(
        effect_family_set
    ),

    str(
        sorted(
            effect_family_set
        )
    )
)


effect_values = pd.to_numeric(

    pooled_family_effects[
        "Effect_Log1p"
    ],

    errors="coerce"
)


add_check(

    "Family_Effect_Vectors_Finite",

    bool(

        np.isfinite(
            effect_values
        ).all()
    ),

    f"{len(effect_values):,} effect values"
)


validation = pd.DataFrame(
    validation_rows
)


print(
    validation.to_string(
        index=False
    )
)


PHASE_11_1_PASS = bool(

    validation[
        "Passed"
    ].all()
)


# ==================================================================================================
# 11.1AD — SAVE OUTPUTS
# ==================================================================================================

banner(
    "11.1AD — SAVE PHASE 11.1 CALIBRATION ARTIFACTS"
)


def save_csv(
    dataframe,
    filename
):

    path = (
        CALIBRATION_DIR
        /
        filename
    )


    dataframe.to_csv(
        path,
        index=False
    )


    print(
        "✓",
        path
    )


    return path


save_csv(

    cic_file_accounting,

    "11_1_cic_raw_file_accounting.csv"
)


save_csv(

    unsw_file_accounting,

    "11_1_unsw_raw_file_accounting.csv"
)


save_csv(

    unsw_authoritative_family_distribution,

    "11_1_unsw_frozen_family_distribution.csv"
)


save_csv(

    label_mapping_audit,

    "11_1_real_label_mapping_audit.csv"
)


save_csv(

    family_support,

    "11_1_family_real_flow_support.csv"
)


save_csv(

    family_minute_counts,

    "11_1_family_attack_flow_counts_per_minute.csv"
)


save_csv(

    family_minute_support,

    "11_1_family_minute_support.csv"
)


save_csv(

    family_intensity_quantiles,

    "11_1_family_attack_flow_intensity_quantiles.csv"
)


save_csv(

    flow_profile_quantiles,

    "11_1_flow_profile_quantiles.csv"
)


save_csv(

    source_family_effects,

    "11_1_source_family_log_effect_vectors.csv"
)


save_csv(

    pooled_family_effects,

    "11_1_pooled_family_log_effect_vectors.csv"
)


save_csv(

    benign_minute_pool,

    "11_1_real_benign_minute_pool.csv"
)


save_csv(

    benign_block_index,

    "11_1_benign_block_index.csv"
)


save_csv(

    benign_block_rows,

    "11_1_benign_block_rows.csv"
)


save_csv(

    block_pool_summary,

    "11_1_benign_block_pool_summary.csv"
)


save_csv(

    real_minute_feature_quantiles,

    "11_1_real_minute_feature_quantiles.csv"
)


save_csv(

    predictor_quality,

    "11_1_real_predictor_quality.csv"
)


save_csv(

    benign_support_summary,

    "11_1_benign_support_summary.csv"
)


save_csv(

    validation,

    "11_1_validation.csv"
)


# Save flow profile sample in efficient format.

FLOW_PROFILE_PARQUET = (
    CALIBRATION_DIR
    /
    "11_1_flow_profile_sample.parquet"
)


FLOW_PROFILE_GZIP = (
    CALIBRATION_DIR
    /
    "11_1_flow_profile_sample.csv.gz"
)


try:

    flow_profile_sample.to_parquet(

        FLOW_PROFILE_PARQUET,

        index=False
    )


    FLOW_PROFILE_OUTPUT = (
        FLOW_PROFILE_PARQUET
    )


    print(
        "✓",
        FLOW_PROFILE_OUTPUT
    )


except Exception as error:

    print(
        "Parquet unavailable; using compressed CSV."
    )


    print(
        "Parquet message:",
        str(
            error
        )[:200]
    )


    flow_profile_sample.to_csv(

        FLOW_PROFILE_GZIP,

        index=False,

        compression="gzip"
    )


    FLOW_PROFILE_OUTPUT = (
        FLOW_PROFILE_GZIP
    )


    print(
        "✓",
        FLOW_PROFILE_OUTPUT
    )


# ==================================================================================================
# 11.1AE — MANIFEST
# ==================================================================================================

banner(
    "11.1AE — CALIBRATION MANIFEST"
)


preliminary_family_gate_count = int(

    family_support[
        "Meets_1000_Flow_Gate_PRELIMINARY"
    ].sum()
)


manifest = pd.DataFrame({

    "Metric": [

        "CIC_Physical_Rows",

        "CIC_Empty_Rows_Removed",

        "CIC_Real_Traffic_Rows",

        "CIC_Observed_Minutes",

        "CIC_Attack_Minutes",

        "UNSW_Raw_Flows",

        "UNSW_Observed_Minutes",

        "CIC_Strict_Benign_Minutes",

        "UNSW_Strict_Benign_Minutes",

        "Pooled_Strict_Benign_Minutes",

        "Benign_Block_Length_Minutes",

        "Benign_Calibration_Blocks",

        "Configured_Attack_Families",

        "Families_Meeting_1000_Flow_Gate_PRELIMINARY",

        "Source_Specific_Effect_Rows",

        "Pooled_Effect_Rows",

        "Flow_Profile_Sample_Rows",

        "Validation_Checks",

        "Validation_Passed",

        "Phase_11_1_Pass",

        "Synthetic_Generation_Performed"

    ],


    "Value": [

        cic_physical_rows,

        cic_empty_rows_removed,

        cic_real_traffic_rows,

        len(
            cic_minutes
        ),

        int(
            cic_minutes[
                "Attack_State"
            ].sum()
        ),

        unsw_processed_rows,

        len(
            unsw_minutes
        ),

        len(
            cic_benign_minutes
        ),

        len(
            unsw_benign_minutes
        ),

        len(
            benign_minute_pool
        ),

        BENIGN_BLOCK_LENGTH,

        len(
            benign_block_index
        ),

        len(
            ATTACK_FAMILIES
        ),

        preliminary_family_gate_count,

        len(
            source_family_effects
        ),

        len(
            pooled_family_effects
        ),

        len(
            flow_profile_sample
        ),

        len(
            validation
        ),

        int(
            validation[
                "Passed"
            ].sum()
        ),

        PHASE_11_1_PASS,

        False
    ]
})


print(
    manifest.to_string(
        index=False
    )
)


save_csv(

    manifest,

    "11_1_calibration_manifest.csv"
)


# ==================================================================================================
# 11.1AF — EXTRACTION PROVENANCE
# ==================================================================================================

banner(
    "11.1AF — SAVE EXTRACTION PROVENANCE"
)


provenance = {

    "phase":
        "11.1",

    "dataset":
        "Synthetic_NAF_v1",

    "spec_config_sha256":
        CURRENT_CONFIG_HASH,

    "master_seed":
        MASTER_SEED,

    "cic_expected_accounting": {

        "physical_rows":
            EXPECTED_CIC_PHYSICAL_ROWS,

        "known_empty_webattack_tail":
            EXPECTED_CIC_EMPTY_TRAILING_ROWS,

        "real_traffic_rows":
            EXPECTED_CIC_REAL_TRAFFIC_ROWS,

        "observed_minutes":
            EXPECTED_CIC_OBSERVED_MINUTES,

        "attack_minutes":
            EXPECTED_CIC_ATTACK_MINUTES
    },

    "cic_encoding_map": {

        key:
            value

        for key, value
        in CIC_ENCODING_BY_LOGICAL_NAME.items()
    },

    "cic_files": {

        key:
            str(
                value
            )

        for key, value
        in CIC_FILES.items()
    },

    "unsw_files": [

        str(
            value
        )

        for value in UNSW_FILES
    ],

    "unsw_frozen_semantics":
        str(
            UNSW_FINAL_SEMANTICS_FILE
        ),

    "unsw_frozen_minute_representation":
        str(
            UNSW_REFERENCE_MINUTE_FILE
        ),

    "chunk_size":
        CHUNK_SIZE,

    "benign_flow_sample_modulus":
        BENIGN_FLOW_SAMPLE_MODULUS,

    "benign_block_length_minutes":
        BENIGN_BLOCK_LENGTH,

    "benign_block_pool_policy": {

        "train":
            0.70,

        "validation":
            0.15,

        "test":
            "remainder",

        "same_real_block_across_pools":
            False
    },

    "family_effect_method":
        (
            "Within-source median log1p family-vs-benign effect; "
            "median pooled across available sources"
        ),

    "unsw_label_authority":
        "Phase 10.12 frozen flow semantics",

    "synthetic_generation_performed":
        False
}


PROVENANCE_FILE = (
    CALIBRATION_DIR
    /
    "11_1_extraction_provenance.json"
)


with open(
    PROVENANCE_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


print(
    "✓",
    PROVENANCE_FILE
)


# ==================================================================================================
# 11.1AG — FINAL DECISION
# ==================================================================================================

banner(
    "11.1AG — FINAL DECISION"
)


if not PHASE_11_1_PASS:

    failed = validation.loc[
        ~validation[
            "Passed"
        ]
    ]


    print(
        "\nFAILED CHECKS:\n"
    )


    print(
        failed.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "PHASE 11.1 FAILED. "
        "DO NOT PROCEED TO PHASE 11.2."
    )


print(
    f"""
✓ PHASE 11.1 — CALIBRATION EXTRACTION: PASS


CIC ACCOUNTING
--------------
Physical CSV rows:

    {cic_physical_rows:,}

Known completely-empty WebAttacks rows removed:

    {cic_empty_rows_removed:,}

Real CIC traffic rows:

    {cic_real_traffic_rows:,}

Observed CIC minutes:

    {len(cic_minutes):,}

Attack-active CIC minutes:

    {int(cic_minutes["Attack_State"].sum()):,}


UNSW ACCOUNTING
---------------
Raw UNSW flows:

    {unsw_processed_rows:,}

Frozen authoritative benign flows:

    {EXPECTED_UNSW_BENIGN_FLOWS:,}

Frozen authoritative attack flows:

    {EXPECTED_UNSW_ATTACK_FLOWS:,}

Observed UNSW minutes:

    {len(unsw_minutes):,}


REAL CALIBRATION SUPPORT
------------------------
Strict real benign minutes:

    {len(benign_minute_pool):,}

Non-overlapping 5-minute benign blocks:

    {len(benign_block_index):,}

Configured synthetic attack families:

    {len(ATTACK_FAMILIES)}

Families provisionally meeting >=1000 real-flow support:

    {preliminary_family_gate_count}/{len(ATTACK_FAMILIES)}


LEAKAGE PROTECTION
------------------
Real benign calibration blocks are split into:

    CAL_TRAIN
    CAL_VALIDATION
    CAL_TEST

A real block belongs to exactly one pool.

Later:

    synthetic TRAIN
        → CAL_TRAIN only

    synthetic VALIDATION
        → CAL_VALIDATION only

    synthetic TEST
        → CAL_TEST only


NO SYNTHETIC DATA WAS GENERATED.


NEXT PHASE
----------
PHASE 11.2 — CALIBRATION VALIDITY AUDIT

11.2 must decide, BEFORE any episode creation:

    ✓ whether all five attack families remain valid

    ✓ whether EXPLOIT_APP pooling is defensible

    ✓ whether CIC-vs-UNSW effects agree where both exist

    ✓ whether any family effect is unstable or near-zero

    ✓ whether benign source distributions are too different

    ✓ whether feature ranges/correlations are realistic

    ✓ whether any family must be removed or remapped

Only after 11.2 passes do we create:

    PHASE 11.3
    1,200-episode frozen manifest
"""
)


print(
    "=" * 120
)

print(
    "PHASE 11.1 COMPLETE"
)

print(
    "=" * 120
)

In [ ]:
# ==================================================================================================
# PHASE 11.2 — CALIBRATION VALIDITY, SOURCE-DIVERGENCE, FAMILY APPROVAL
#              & LEAKAGE-SAFE CALIBRATION FREEZE
# ==================================================================================================
#
# INPUT
# -----
# Phase 11.1 calibration artifacts ONLY.
#
# NO raw CIC/UNSW multi-million-flow processing is repeated here.
#
#
# PURPOSE
# -------
# Before Synthetic_NAF_v1 is allowed to generate any episode, this phase answers:
#
#   1. Are all 5 proposed synthetic attack families sufficiently supported?
#
#   2. Are CIC and UNSW effects compatible enough to pool directly?
#
#   3. If not, how should each family be generated safely?
#
#   4. Is EXPLOIT_APP too heterogeneous?
#
#   5. How different are benign traffic scales across CIC and UNSW?
#
#   6. Can we create leakage-safe real benign calibration seed pools?
#
#   7. Are those pools temporally isolated enough for synthetic TRAIN / VAL / TEST?
#
#   8. What EXACT calibration policy will Phase 11.3 / 11.4 be allowed to use?
#
#
# IMPORTANT DESIGN DECISION
# -------------------------
# We DO NOT blindly pool CIC + UNSW magnitudes.
#
# Synthetic generation will be SOURCE-CONDITIONED.
#
# Conceptually:
#
#       choose calibration source
#                ↓
#       source-specific benign baseline
#                ↓
#       source-specific family effect
#                ↓
#       synthetic temporal evolution
#
#
# This avoids:
#
#       CIC-scale baseline + UNSW-scale attack effect
#
# which would be physically meaningless.
#
#
# BENIGN CALIBRATION LEAKAGE POLICY
# ---------------------------------
# 11.1 created exact 5-minute non-overlapping blocks, but blocks from the
# same continuous real run could still be adjacent across different pools.
#
# 11.2 supersedes those pool assignments.
#
# We now build:
#
#       30-minute CALIBRATION GROUP
#       10-minute DISCARDED GUARD
#       30-minute CALIBRATION GROUP
#       10-minute DISCARDED GUARD
#       ...
#
# Each 30-minute group belongs wholly to:
#
#       CAL_TRAIN
#       CAL_VALIDATION
#       CAL_TEST
#
# Each group contains exactly:
#
#       6 × 5-minute usable benign blocks
#
# No block crosses a calibration-group boundary.
#
# If adjacent groups from the same original real run belong to different pools,
# at least 10 real minutes are discarded between them.
#
#
# FAMILY DECISIONS
# ----------------
# Each family receives one of:
#
#       APPROVED
#       APPROVED_WITH_RESTRICTION
#       REJECTED
#
# Phase 11.3 is BLOCKED if ANY configured family is REJECTED.
#
#
# THIS PHASE DOES NOT:
#
#   ✗ generate synthetic episodes
#   ✗ generate synthetic timestamps
#   ✗ create synthetic feature values
#   ✗ train models
#   ✗ fit model scalers
#   ✗ modify the 1,200-episode split specification
#
# ==================================================================================================


from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
import math
import warnings

warnings.filterwarnings("ignore")


# ==================================================================================================
# 11.2A — PATHS + THRESHOLDS
# ==================================================================================================

print("=" * 120)
print("PHASE 11.2 — CALIBRATION VALIDITY & LEAKAGE-SAFE CALIBRATION FREEZE")
print("=" * 120)


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)


DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)


SPEC_DIR = (
    DATASET_ROOT
    / "spec"
)


CALIBRATION_DIR = (
    DATASET_ROOT
    / "calibration"
)


AUDIT_DIR = (
    DATASET_ROOT
    / "audit"
)


CALIBRATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config.json"
)


CONFIG_HASH_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config_SHA256.txt"
)


# --------------------------------------------------------------------------------------------------
# 11.1 artifacts
# --------------------------------------------------------------------------------------------------

MANIFEST_11_1_FILE = (
    CALIBRATION_DIR
    / "11_1_calibration_manifest.csv"
)


VALIDATION_11_1_FILE = (
    CALIBRATION_DIR
    / "11_1_validation.csv"
)


FAMILY_SUPPORT_FILE = (
    CALIBRATION_DIR
    / "11_1_family_real_flow_support.csv"
)


FAMILY_MINUTE_SUPPORT_FILE = (
    CALIBRATION_DIR
    / "11_1_family_minute_support.csv"
)


FAMILY_INTENSITY_FILE = (
    CALIBRATION_DIR
    / "11_1_family_attack_flow_intensity_quantiles.csv"
)


SOURCE_EFFECT_FILE = (
    CALIBRATION_DIR
    / "11_1_source_family_log_effect_vectors.csv"
)


POOLED_EFFECT_FILE = (
    CALIBRATION_DIR
    / "11_1_pooled_family_log_effect_vectors.csv"
)


LABEL_MAPPING_FILE = (
    CALIBRATION_DIR
    / "11_1_real_label_mapping_audit.csv"
)


BENIGN_MINUTE_POOL_FILE = (
    CALIBRATION_DIR
    / "11_1_real_benign_minute_pool.csv"
)


REAL_MINUTE_QUANTILES_FILE = (
    CALIBRATION_DIR
    / "11_1_real_minute_feature_quantiles.csv"
)


BENIGN_SUPPORT_FILE = (
    CALIBRATION_DIR
    / "11_1_benign_support_summary.csv"
)


PROVENANCE_11_1_FILE = (
    CALIBRATION_DIR
    / "11_1_extraction_provenance.json"
)


# --------------------------------------------------------------------------------------------------
# Internal audit thresholds.
#
# These are NOT claimed to be universal scientific thresholds.
#
# They are conservative project-level rules preventing us from building
# a simulator from weak/unstable evidence.
# --------------------------------------------------------------------------------------------------

MIN_TOTAL_FAMILY_FLOWS = 1_000

MIN_SOURCE_FLOWS_FOR_GENERATION = 500

MIN_FAMILY_ACTIVE_MINUTES = 20

EXPECTED_EFFECT_FEATURE_COUNT = 9


# Around +/- 0.05 in log1p space is treated as practically near-zero
# for directional agreement auditing.

EFFECT_NEAR_ZERO = 0.05


# For dual-source families, <67% feature-direction agreement creates
# a restriction. It does NOT automatically reject the family because
# the generator will remain source-conditioned.

MIN_CROSS_SOURCE_DIRECTION_AGREEMENT = 0.67


# Diagnostic divergence threshold in log1p effect space.

MAX_MEDIAN_CROSS_SOURCE_EFFECT_DIFFERENCE = 1.00


# Raw attack-flow intensity differences beyond 10× are considered
# major source-scale divergence.

RAW_INTENSITY_FOLD_WARNING = 10.0


# If >95% of all real family flows come from one source, mark imbalance.

SOURCE_DOMINANCE_WARNING = 0.95


# Benign feature median scale difference warning.

BENIGN_MEDIAN_FOLD_WARNING = 10.0


# --------------------------------------------------------------------------------------------------
# Leakage-safe real baseline grouping
# --------------------------------------------------------------------------------------------------

CALIBRATION_GROUP_LENGTH_MINUTES = 30

CALIBRATION_GUARD_MINUTES = 10

BENIGN_BLOCK_LENGTH_MINUTES = 5


CALIBRATION_POOL_PROPORTIONS = {

    "CAL_TRAIN":
        0.70,

    "CAL_VALIDATION":
        0.15,

    "CAL_TEST":
        0.15
}


MASTER_POLICY_VERSION = "11.2.0"


def banner(
    title
):

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        title
    )

    print(
        "=" * 120
    )


# ==================================================================================================
# 11.2B — VERIFY REQUIRED ARTIFACTS
# ==================================================================================================

banner(
    "11.2B — VERIFY PHASE 11.0 + 11.1 ARTIFACTS"
)


required_files = [

    CONFIG_FILE,

    CONFIG_HASH_FILE,

    MANIFEST_11_1_FILE,

    VALIDATION_11_1_FILE,

    FAMILY_SUPPORT_FILE,

    FAMILY_MINUTE_SUPPORT_FILE,

    FAMILY_INTENSITY_FILE,

    SOURCE_EFFECT_FILE,

    POOLED_EFFECT_FILE,

    LABEL_MAPPING_FILE,

    BENIGN_MINUTE_POOL_FILE,

    REAL_MINUTE_QUANTILES_FILE,

    BENIGN_SUPPORT_FILE,

    PROVENANCE_11_1_FILE
]


missing = []


for path in required_files:

    if path.exists():

        print(
            "✓",
            path.name
        )

    else:

        print(
            "✗",
            path
        )

        missing.append(
            path
        )


if missing:

    raise FileNotFoundError(
        "Phase 11.2 cannot run because required artifacts are missing."
    )


# ==================================================================================================
# 11.2C — VERIFY FROZEN 11.0 CONFIG
# ==================================================================================================

banner(
    "11.2C — VERIFY FROZEN DATASET SPECIFICATION"
)


with open(
    CONFIG_FILE,
    "rb"
) as f:

    config_bytes = f.read()


CURRENT_CONFIG_HASH = hashlib.sha256(
    config_bytes
).hexdigest()


with open(
    CONFIG_HASH_FILE,
    "r"
) as f:

    FROZEN_CONFIG_HASH = (
        f.read()
        .strip()
    )


print(
    "Frozen config hash :",
    FROZEN_CONFIG_HASH
)

print(
    "Current config hash:",
    CURRENT_CONFIG_HASH
)


if CURRENT_CONFIG_HASH != FROZEN_CONFIG_HASH:

    raise RuntimeError(
        "Synthetic_NAF_v1 Phase 11.0 configuration changed after freeze."
    )


with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(
        f
    )


ATTACK_FAMILIES = (
    CONFIG[
        "episode_design"
    ][
        "attack_families"
    ]
)


FINAL_PREDICTORS = (
    CONFIG[
        "features"
    ][
        "final_predictor_whitelist"
    ]
)


MASTER_SEED = int(
    CONFIG[
        "dataset_identity"
    ][
        "master_seed"
    ]
)


print(
    "\nAttack families:",
    ATTACK_FAMILIES
)

print(
    "Predictors:",
    len(
        FINAL_PREDICTORS
    )
)


# ==================================================================================================
# 11.2D — VERIFY PHASE 11.1 PASSED
# ==================================================================================================

banner(
    "11.2D — VERIFY PHASE 11.1 PASS STATE"
)


manifest_11_1 = pd.read_csv(
    MANIFEST_11_1_FILE
)


validation_11_1 = pd.read_csv(
    VALIDATION_11_1_FILE
)


def manifest_value(
    name
):

    row = manifest_11_1.loc[
        manifest_11_1[
            "Metric"
        ]
        ==
        name
    ]


    if len(
        row
    ) != 1:

        raise RuntimeError(
            f"Manifest metric {name!r} missing or duplicated."
        )


    return row[
        "Value"
    ].iloc[0]


phase_11_1_pass_raw = str(
    manifest_value(
        "Phase_11_1_Pass"
    )
).strip().lower()


PHASE_11_1_PASS = (
    phase_11_1_pass_raw
    ==
    "true"
)


validation_pass_series = (

    validation_11_1[
        "Passed"
    ]

    .astype(
        str
    )

    .str.lower()

    .map(
        {
            "true":
                True,

            "false":
                False
        }
    )
)


ALL_11_1_CHECKS_PASS = bool(
    validation_pass_series.all()
)


print(
    "Phase 11.1 manifest pass:",
    PHASE_11_1_PASS
)

print(
    "Phase 11.1 validation checks:",
    f"{int(validation_pass_series.sum())}/{len(validation_pass_series)}"
)


if not (
    PHASE_11_1_PASS
    and
    ALL_11_1_CHECKS_PASS
):

    raise RuntimeError(
        "Phase 11.1 was not fully valid. Phase 11.2 is blocked."
    )


# ==================================================================================================
# 11.2E — LOAD CALIBRATION TABLES
# ==================================================================================================

banner(
    "11.2E — LOAD PHASE 11.1 CALIBRATION TABLES"
)


family_support = pd.read_csv(
    FAMILY_SUPPORT_FILE
)


family_minute_support = pd.read_csv(
    FAMILY_MINUTE_SUPPORT_FILE
)


family_intensity = pd.read_csv(
    FAMILY_INTENSITY_FILE
)


source_effects = pd.read_csv(
    SOURCE_EFFECT_FILE
)


pooled_effects = pd.read_csv(
    POOLED_EFFECT_FILE
)


label_mapping = pd.read_csv(
    LABEL_MAPPING_FILE
)


benign_minutes = pd.read_csv(
    BENIGN_MINUTE_POOL_FILE,
    low_memory=False
)


real_minute_quantiles = pd.read_csv(
    REAL_MINUTE_QUANTILES_FILE
)


benign_support = pd.read_csv(
    BENIGN_SUPPORT_FILE
)


with open(
    PROVENANCE_11_1_FILE,
    "r",
    encoding="utf-8"
) as f:

    provenance_11_1 = json.load(
        f
    )


benign_minutes[
    "Real_Minute"
] = pd.to_datetime(
    benign_minutes[
        "Real_Minute"
    ],
    errors="coerce"
)


if benign_minutes[
    "Real_Minute"
].isna().any():

    raise RuntimeError(
        "Invalid timestamps found in 11.1 benign minute pool."
    )


for feature in FINAL_PREDICTORS:

    benign_minutes[
        feature
    ] = pd.to_numeric(

        benign_minutes[
            feature
        ],

        errors="coerce"
    )


if benign_minutes[
    FINAL_PREDICTORS
].isna().any().any():

    raise RuntimeError(
        "NaN values found in benign predictor pool."
    )


print(
    "Family support rows:",
    len(
        family_support
    )
)

print(
    "Source effect rows:",
    len(
        source_effects
    )
)

print(
    "Pooled effect rows:",
    len(
        pooled_effects
    )
)

print(
    "Strict benign minute rows:",
    len(
        benign_minutes
    )
)


# ==================================================================================================
# 11.2F — BENIGN SOURCE-SCALE AUDIT
#
# We compare CIC vs UNSW STRICT BENIGN median predictor levels.
#
# This determines whether it is scientifically safe to pool absolute
# feature magnitudes.
#
# ==================================================================================================

banner(
    "11.2F — CIC vs UNSW STRICT-BENIGN SOURCE-SCALE AUDIT"
)


benign_scale_rows = []


cic_benign = benign_minutes.loc[
    benign_minutes[
        "Calibration_Source"
    ]
    ==
    "CIC_IDS2017"
]


unsw_benign = benign_minutes.loc[
    benign_minutes[
        "Calibration_Source"
    ]
    ==
    "UNSW_NB15"
]


for feature in FINAL_PREDICTORS:

    cic_values = pd.to_numeric(
        cic_benign[
            feature
        ],
        errors="coerce"
    )


    unsw_values = pd.to_numeric(
        unsw_benign[
            feature
        ],
        errors="coerce"
    )


    cic_median = float(
        cic_values.median()
    )


    unsw_median = float(
        unsw_values.median()
    )


    log_difference = abs(

        np.log1p(
            cic_median
        )

        -

        np.log1p(
            unsw_median
        )
    )


    median_fold_difference = float(
        np.exp(
            log_difference
        )
    )


    benign_scale_rows.append(
        {
            "Feature":
                feature,

            "CIC_Benign_Median":
                cic_median,

            "UNSW_Benign_Median":
                unsw_median,

            "Absolute_Log1p_Median_Difference":
                log_difference,

            "Median_Fold_Difference":
                median_fold_difference,

            "Major_Scale_Divergence":
                median_fold_difference
                >=
                BENIGN_MEDIAN_FOLD_WARNING
        }
    )


benign_scale_audit = pd.DataFrame(
    benign_scale_rows
)


print(
    benign_scale_audit.to_string(
        index=False
    )
)


major_benign_scale_features = int(

    benign_scale_audit[
        "Major_Scale_Divergence"
    ].sum()
)


print(
    "\nFeatures with >=10× benign median scale difference:",
    major_benign_scale_features,
    "/",
    len(
        FINAL_PREDICTORS
    )
)


# ==================================================================================================
# 11.2G — BENIGN CORRELATION STRUCTURE AUDIT
#
# Compare log1p predictor correlations.
#
# We do NOT demand identical correlations.
#
# This is a diagnostic to establish that the two real datasets
# represent related but different network environments.
# ==================================================================================================

banner(
    "11.2G — BENIGN CROSS-FEATURE CORRELATION AUDIT"
)


benign_correlation_frames = []


correlation_matrices = {}


for source, group in benign_minutes.groupby(
    "Calibration_Source",
    observed=True
):

    x = group[
        FINAL_PREDICTORS
    ].copy()


    x = np.log1p(
        x.astype(
            float
        )
    )


    corr = pd.DataFrame(
        x,
        columns=FINAL_PREDICTORS
    ).corr()


    correlation_matrices[
        source
    ] = corr


    corr_long = (

        corr

        .stack(
            future_stack=True
        )

        .rename(
            "Correlation"
        )

        .reset_index()
    )


    corr_long.columns = [
        "Feature_A",
        "Feature_B",
        "Correlation"
    ]


    corr_long.insert(
        0,
        "Calibration_Source",
        source
    )


    benign_correlation_frames.append(
        corr_long
    )


benign_correlations = pd.concat(
    benign_correlation_frames,
    ignore_index=True
)


corr_diff_rows = []


if (
    "CIC_IDS2017"
    in
    correlation_matrices
    and
    "UNSW_NB15"
    in
    correlation_matrices
):

    cic_corr = correlation_matrices[
        "CIC_IDS2017"
    ]


    unsw_corr = correlation_matrices[
        "UNSW_NB15"
    ]


    for i, feature_a in enumerate(
        FINAL_PREDICTORS
    ):

        for j, feature_b in enumerate(
            FINAL_PREDICTORS
        ):

            if j <= i:

                continue


            c = cic_corr.loc[
                feature_a,
                feature_b
            ]


            u = unsw_corr.loc[
                feature_a,
                feature_b
            ]


            if (
                not np.isfinite(
                    c
                )
                or
                not np.isfinite(
                    u
                )
            ):

                continue


            corr_diff_rows.append(
                {
                    "Feature_A":
                        feature_a,

                    "Feature_B":
                        feature_b,

                    "CIC_Correlation":
                        float(
                            c
                        ),

                    "UNSW_Correlation":
                        float(
                            u
                        ),

                    "Absolute_Correlation_Difference":
                        float(
                            abs(
                                c
                                -
                                u
                            )
                        )
                }
            )


benign_correlation_difference = pd.DataFrame(
    corr_diff_rows
)


if len(
    benign_correlation_difference
) > 0:

    mean_corr_difference = float(

        benign_correlation_difference[
            "Absolute_Correlation_Difference"
        ].mean()
    )


    median_corr_difference = float(

        benign_correlation_difference[
            "Absolute_Correlation_Difference"
        ].median()
    )


else:

    mean_corr_difference = np.nan

    median_corr_difference = np.nan


print(
    "Mean absolute benign correlation difference:",
    mean_corr_difference
)

print(
    "Median absolute benign correlation difference:",
    median_corr_difference
)


# ==================================================================================================
# 11.2H — CROSS-SOURCE FAMILY EFFECT AGREEMENT
#
# Only families represented in BOTH real datasets are compared.
#
# Effect classification per feature:
#
#   AGREE_POSITIVE
#   AGREE_NEGATIVE
#   AGREE_NEAR_ZERO
#   DISAGREE
#
# ==================================================================================================

banner(
    "11.2H — CROSS-SOURCE FAMILY EFFECT AGREEMENT"
)


source_effects[
    "Effect_Log1p"
] = pd.to_numeric(
    source_effects[
        "Effect_Log1p"
    ],
    errors="coerce"
)


effect_pivot = (

    source_effects

    .pivot_table(

        index=[
            "Synthetic_Family",
            "Feature"
        ],

        columns="Calibration_Source",

        values="Effect_Log1p",

        aggfunc="first"
    )

    .reset_index()
)


effect_agreement_rows = []


for _, row in effect_pivot.iterrows():

    family = row[
        "Synthetic_Family"
    ]


    feature = row[
        "Feature"
    ]


    cic_effect = row.get(
        "CIC_IDS2017",
        np.nan
    )


    unsw_effect = row.get(
        "UNSW_NB15",
        np.nan
    )


    both_sources = (

        pd.notna(
            cic_effect
        )

        and

        pd.notna(
            unsw_effect
        )
    )


    if not both_sources:

        agreement = "SINGLE_SOURCE"

        absolute_difference = np.nan

    else:

        cic_near_zero = (
            abs(
                cic_effect
            )
            <=
            EFFECT_NEAR_ZERO
        )


        unsw_near_zero = (
            abs(
                unsw_effect
            )
            <=
            EFFECT_NEAR_ZERO
        )


        if (
            cic_near_zero
            and
            unsw_near_zero
        ):

            agreement = "AGREE_NEAR_ZERO"


        elif (
            cic_effect
            >
            EFFECT_NEAR_ZERO
            and
            unsw_effect
            >
            EFFECT_NEAR_ZERO
        ):

            agreement = "AGREE_POSITIVE"


        elif (
            cic_effect
            <
            -EFFECT_NEAR_ZERO
            and
            unsw_effect
            <
            -EFFECT_NEAR_ZERO
        ):

            agreement = "AGREE_NEGATIVE"


        else:

            agreement = "DISAGREE"


        absolute_difference = float(
            abs(
                cic_effect
                -
                unsw_effect
            )
        )


    effect_agreement_rows.append(
        {
            "Synthetic_Family":
                family,

            "Feature":
                feature,

            "CIC_Effect_Log1p":
                cic_effect,

            "UNSW_Effect_Log1p":
                unsw_effect,

            "Agreement":
                agreement,

            "Absolute_Effect_Difference":
                absolute_difference
        }
    )


effect_agreement = pd.DataFrame(
    effect_agreement_rows
)


dual_source_effect_summary_rows = []


for family in ATTACK_FAMILIES:

    f = effect_agreement.loc[
        effect_agreement[
            "Synthetic_Family"
        ]
        ==
        family
    ]


    dual = f.loc[
        f[
            "Agreement"
        ]
        !=
        "SINGLE_SOURCE"
    ]


    if len(
        dual
    ) == 0:

        dual_source_effect_summary_rows.append(
            {
                "Synthetic_Family":
                    family,

                "Dual_Source_Feature_Count":
                    0,

                "Direction_Agreement_Count":
                    0,

                "Direction_Agreement_Rate":
                    np.nan,

                "Median_Absolute_Effect_Difference":
                    np.nan,

                "Max_Absolute_Effect_Difference":
                    np.nan,

                "Effect_Spearman_Correlation":
                    np.nan
            }
        )

        continue


    agreement_count = int(

        dual[
            "Agreement"
        ]

        .isin(
            [
                "AGREE_POSITIVE",
                "AGREE_NEGATIVE",
                "AGREE_NEAR_ZERO"
            ]
        )

        .sum()
    )


    agreement_rate = (

        agreement_count
        /
        len(
            dual
        )
    )


    valid_corr = dual[
        [
            "CIC_Effect_Log1p",
            "UNSW_Effect_Log1p"
        ]
    ].dropna()


    if len(
        valid_corr
    ) >= 3:

        spearman = float(

            valid_corr[
                "CIC_Effect_Log1p"
            ]

            .corr(

                valid_corr[
                    "UNSW_Effect_Log1p"
                ],

                method="spearman"
            )
        )

    else:

        spearman = np.nan


    dual_source_effect_summary_rows.append(
        {
            "Synthetic_Family":
                family,

            "Dual_Source_Feature_Count":
                len(
                    dual
                ),

            "Direction_Agreement_Count":
                agreement_count,

            "Direction_Agreement_Rate":
                agreement_rate,

            "Median_Absolute_Effect_Difference":
                float(

                    dual[
                        "Absolute_Effect_Difference"
                    ].median()
                ),

            "Max_Absolute_Effect_Difference":
                float(

                    dual[
                        "Absolute_Effect_Difference"
                    ].max()
                ),

            "Effect_Spearman_Correlation":
                spearman
        }
    )


dual_source_effect_summary = pd.DataFrame(
    dual_source_effect_summary_rows
)


print(
    dual_source_effect_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2I — RAW ATTACK INTENSITY SCALE DIVERGENCE
# ==================================================================================================

banner(
    "11.2I — ATTACK INTENSITY SCALE DIVERGENCE"
)


source_intensity = family_intensity.loc[

    family_intensity[
        "Calibration_Source"
    ].isin(
        [
            "CIC_IDS2017",
            "UNSW_NB15"
        ]
    )

].copy()


source_intensity[
    "q50"
] = pd.to_numeric(
    source_intensity[
        "q50"
    ],
    errors="coerce"
)


intensity_pivot = (

    source_intensity

    .pivot_table(

        index="Synthetic_Family",

        columns="Calibration_Source",

        values="q50",

        aggfunc="first"
    )

    .reset_index()
)


intensity_divergence_rows = []


for family in ATTACK_FAMILIES:

    row = intensity_pivot.loc[
        intensity_pivot[
            "Synthetic_Family"
        ]
        ==
        family
    ]


    if len(
        row
    ) == 0:

        cic_q50 = np.nan

        unsw_q50 = np.nan


    else:

        cic_q50 = (

            row[
                "CIC_IDS2017"
            ].iloc[0]

            if "CIC_IDS2017"
            in row.columns

            else np.nan
        )


        unsw_q50 = (

            row[
                "UNSW_NB15"
            ].iloc[0]

            if "UNSW_NB15"
            in row.columns

            else np.nan
        )


    if (
        pd.notna(
            cic_q50
        )
        and
        pd.notna(
            unsw_q50
        )
    ):

        intensity_fold = float(

            max(
                cic_q50,
                unsw_q50
            )

            /

            max(
                min(
                    cic_q50,
                    unsw_q50
                ),
                1e-12
            )
        )


    else:

        intensity_fold = np.nan


    intensity_divergence_rows.append(
        {
            "Synthetic_Family":
                family,

            "CIC_Median_Attack_Flows_Per_Active_Minute":
                cic_q50,

            "UNSW_Median_Attack_Flows_Per_Active_Minute":
                unsw_q50,

            "Median_Intensity_Fold_Difference":
                intensity_fold,

            "Major_Raw_Intensity_Divergence":
                (
                    pd.notna(
                        intensity_fold
                    )
                    and
                    intensity_fold
                    >=
                    RAW_INTENSITY_FOLD_WARNING
                )
        }
    )


intensity_divergence = pd.DataFrame(
    intensity_divergence_rows
)


print(
    intensity_divergence.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2J — EXPLOIT_APP COMPOSITION AUDIT
#
# EXPLOIT_APP intentionally combines multiple related exploit/application
# attack labels.
#
# We explicitly record its composition rather than pretending it is a
# homogeneous original class.
# ==================================================================================================

banner(
    "11.2J — EXPLOIT_APP COMPOSITION AUDIT"
)


exploit_composition = label_mapping.loc[

    (
        label_mapping[
            "Synthetic_Family"
        ]
        ==
        "EXPLOIT_APP"
    )

    &

    (
        label_mapping[
            "Used_For_Selected_Family_Calibration"
        ]
        .astype(
            str
        )
        .str.lower()
        ==
        "true"
    )

].copy()


exploit_composition[
    "Flow_Count"
] = pd.to_numeric(
    exploit_composition[
        "Flow_Count"
    ],
    errors="coerce"
)


exploit_composition[
    "Within_Source_Share"
] = (

    exploit_composition[
        "Flow_Count"
    ]

    /

    exploit_composition

    .groupby(
        "Calibration_Source"
    )[
        "Flow_Count"
    ]

    .transform(
        "sum"
    )
)


exploit_total = float(
    exploit_composition[
        "Flow_Count"
    ].sum()
)


exploit_composition[
    "Overall_EXPLOIT_APP_Share"
] = (

    exploit_composition[
        "Flow_Count"
    ]

    /
    exploit_total
)


print(
    exploit_composition.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2K — BUILD CONTIGUOUS STRICT-BENIGN RUNS
# ==================================================================================================

banner(
    "11.2K — RECONSTRUCT CONTIGUOUS STRICT-BENIGN RUNS"
)


benign_minutes = (

    benign_minutes

    .sort_values(
        [
            "Calibration_Source",
            "Real_Segment_ID",
            "Real_Minute"
        ]
    )

    .reset_index(
        drop=True
    )
)


run_frames = []

run_index_rows = []


for source, source_df in benign_minutes.groupby(
    "Calibration_Source",
    observed=True,
    sort=False
):

    source_run_counter = 0


    for segment_id, segment in source_df.groupby(
        "Real_Segment_ID",
        observed=True,
        sort=False
    ):

        segment = (

            segment

            .sort_values(
                "Real_Minute"
            )

            .reset_index(
                drop=True
            )
        )


        gap = (

            segment[
                "Real_Minute"
            ]

            .diff()

            .dt.total_seconds()

            .div(
                60
            )
        )


        new_run = (

            gap.isna()

            |

            (
                gap
                !=
                1
            )
        )


        local_run_number = (
            new_run
            .cumsum()
        )


        segment[
            "_Local_Run"
        ] = local_run_number


        for _, run in segment.groupby(
            "_Local_Run",
            observed=True,
            sort=False
        ):

            source_run_counter += (
                1
            )


            run = (

                run

                .drop(
                    columns=[
                        "_Local_Run"
                    ]
                )

                .reset_index(
                    drop=True
                )
            )


            run_id = (

                f"{source}"
                f"::RUN_{source_run_counter:04d}"
            )


            run[
                "Benign_Run_ID"
            ] = run_id


            run[
                "Benign_Run_Position"
            ] = np.arange(
                len(
                    run
                )
            )


            run_frames.append(
                run
            )


            run_index_rows.append(
                {
                    "Calibration_Source":
                        source,

                    "Benign_Run_ID":
                        run_id,

                    "Real_Segment_ID":
                        segment_id,

                    "Run_Start":
                        run[
                            "Real_Minute"
                        ].min(),

                    "Run_End":
                        run[
                            "Real_Minute"
                        ].max(),

                    "Run_Length_Minutes":
                        len(
                            run
                        )
                }
            )


benign_run_rows = pd.concat(
    run_frames,
    ignore_index=True
)


benign_run_index = pd.DataFrame(
    run_index_rows
)


print(
    benign_run_index.to_string(
        index=False
    )
)


print(
    "\nBenign runs per source:"
)


print(

    benign_run_index

    .groupby(
        "Calibration_Source"
    )

    .agg(

        Runs=(
            "Benign_Run_ID",
            "size"
        ),

        Total_Benign_Minutes=(
            "Run_Length_Minutes",
            "sum"
        ),

        Median_Run_Length=(
            "Run_Length_Minutes",
            "median"
        ),

        Max_Run_Length=(
            "Run_Length_Minutes",
            "max"
        )

    )

    .reset_index()

    .to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2L — BUILD GUARDED 30-MINUTE CALIBRATION GROUPS
#
# Structure inside a long real benign run:
#
#       [30 usable minutes]
#       [10 discarded guard minutes]
#       [30 usable minutes]
#       [10 discarded guard minutes]
#       ...
#
# Only COMPLETE 30-minute groups are retained.
#
# ==================================================================================================

banner(
    "11.2L — BUILD GUARDED 30-MINUTE CALIBRATION GROUPS"
)


calibration_group_frames = []

calibration_group_index_rows = []


for (
    source,
    run_id
), run in benign_run_rows.groupby(

    [
        "Calibration_Source",
        "Benign_Run_ID"
    ],

    observed=True,
    sort=False
):

    run = (

        run

        .sort_values(
            "Real_Minute"
        )

        .reset_index(
            drop=True
        )
    )


    cursor = 0

    group_number = 0


    while (

        cursor
        +
        CALIBRATION_GROUP_LENGTH_MINUTES

        <=

        len(
            run
        )
    ):

        group_number += (
            1
        )


        start = cursor


        end = (

            start
            +
            CALIBRATION_GROUP_LENGTH_MINUTES
        )


        group = (

            run.iloc[
                start:end
            ]

            .copy()

            .reset_index(
                drop=True
            )
        )


        calibration_group_id = (

            f"{run_id}"
            f"::CG_{group_number:03d}"
        )


        group[
            "Calibration_Group_ID"
        ] = calibration_group_id


        group[
            "Calibration_Group_Position"
        ] = np.arange(
            CALIBRATION_GROUP_LENGTH_MINUTES
        )


        calibration_group_frames.append(
            group
        )


        calibration_group_index_rows.append(
            {
                "Calibration_Source":
                    source,

                "Benign_Run_ID":
                    run_id,

                "Calibration_Group_ID":
                    calibration_group_id,

                "Group_Start":
                    group[
                        "Real_Minute"
                    ].min(),

                "Group_End":
                    group[
                        "Real_Minute"
                    ].max(),

                "Group_Length_Minutes":
                    len(
                        group
                    )
            }
        )


        cursor = (

            end
            +
            CALIBRATION_GUARD_MINUTES
        )


if not calibration_group_frames:

    raise RuntimeError(
        "No leakage-safe calibration groups could be created."
    )


safe_group_rows = pd.concat(
    calibration_group_frames,
    ignore_index=True
)


safe_group_index = pd.DataFrame(
    calibration_group_index_rows
)


print(
    "Safe calibration groups:",
    len(
        safe_group_index
    )
)


print(
    "\nGroups by source:"
)


print(

    safe_group_index

    .groupby(
        "Calibration_Source"
    )

    .size()

    .rename(
        "Calibration_Group_Count"
    )

    .reset_index()

    .to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2M — ASSIGN WHOLE CALIBRATION GROUPS TO POOLS
#
# Since each group has exactly 30 minutes, group count ~= minute share.
#
# Deterministic SHA256 order is used.
#
# ==================================================================================================

banner(
    "11.2M — ASSIGN WHOLE CALIBRATION GROUPS TO CAL_TRAIN / VAL / TEST"
)


def stable_hash_int(
    value
):

    digest = hashlib.sha256(

        str(
            value
        ).encode(
            "utf-8"
        )

    ).hexdigest()


    return int(
        digest[:16],
        16
    )


assigned_group_frames = []


for source, group_index in safe_group_index.groupby(
    "Calibration_Source",
    observed=True
):

    group_index = group_index.copy()


    group_index[
        "_Stable_Hash"
    ] = (

        group_index[
            "Calibration_Group_ID"
        ]

        .apply(

            lambda x:

            stable_hash_int(
                f"{MASTER_SEED}|{source}|{x}"
            )
        )
    )


    group_index = (

        group_index

        .sort_values(
            "_Stable_Hash"
        )

        .reset_index(
            drop=True
        )
    )


    n = len(
        group_index
    )


    if n < 3:

        raise RuntimeError(
            f"{source} has only {n} safe calibration groups. "
            "Cannot create independent calibration pools."
        )


    n_train = int(
        np.floor(
            0.70
            *
            n
        )
    )


    n_validation = int(
        np.floor(
            0.15
            *
            n
        )
    )


    # Guarantee at least one group in every pool.

    n_train = max(
        n_train,
        1
    )


    n_validation = max(
        n_validation,
        1
    )


    n_test = (

        n
        -
        n_train
        -
        n_validation
    )


    if n_test < 1:

        n_test = 1

        if n_train > n_validation:

            n_train -= 1

        else:

            n_validation -= 1


    assignments = (

        [
            "CAL_TRAIN"
        ]
        *
        n_train

        +

        [
            "CAL_VALIDATION"
        ]
        *
        n_validation

        +

        [
            "CAL_TEST"
        ]
        *
        n_test
    )


    if len(
        assignments
    ) != n:

        raise RuntimeError(
            "Calibration group assignment arithmetic failed."
        )


    group_index[
        "Calibration_Pool"
    ] = assignments


    group_index = group_index.drop(
        columns=[
            "_Stable_Hash"
        ]
    )


    assigned_group_frames.append(
        group_index
    )


safe_group_index = pd.concat(
    assigned_group_frames,
    ignore_index=True
)


safe_group_rows = safe_group_rows.merge(

    safe_group_index[
        [
            "Calibration_Group_ID",
            "Calibration_Pool"
        ]
    ],

    on="Calibration_Group_ID",

    how="left",

    validate="many_to_one"
)


safe_group_pool_summary = (

    safe_group_index

    .groupby(
        [
            "Calibration_Source",
            "Calibration_Pool"
        ],
        observed=True
    )

    .agg(

        Calibration_Groups=(
            "Calibration_Group_ID",
            "size"
        ),

        Calibration_Minutes=(
            "Group_Length_Minutes",
            "sum"
        )

    )

    .reset_index()
)


print(
    safe_group_pool_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2N — VERIFY CROSS-POOL TEMPORAL GUARDS
# ==================================================================================================

banner(
    "11.2N — VERIFY TEMPORAL GUARDS BETWEEN DIFFERENT CALIBRATION POOLS"
)


guard_audit_rows = []


for run_id, group in safe_group_index.groupby(
    "Benign_Run_ID",
    observed=True
):

    group = (

        group

        .sort_values(
            "Group_Start"
        )

        .reset_index(
            drop=True
        )
    )


    for i in range(
        1,
        len(
            group
        )
    ):

        previous = group.iloc[
            i - 1
        ]


        current = group.iloc[
            i
        ]


        unused_minutes_between = int(

            (
                current[
                    "Group_Start"
                ]

                -

                previous[
                    "Group_End"
                ]
            )

            .total_seconds()

            /
            60

            -

            1
        )


        different_pool = (

            previous[
                "Calibration_Pool"
            ]

            !=

            current[
                "Calibration_Pool"
            ]
        )


        guard_audit_rows.append(
            {
                "Benign_Run_ID":
                    run_id,

                "Previous_Group":
                    previous[
                        "Calibration_Group_ID"
                    ],

                "Previous_Pool":
                    previous[
                        "Calibration_Pool"
                    ],

                "Current_Group":
                    current[
                        "Calibration_Group_ID"
                    ],

                "Current_Pool":
                    current[
                        "Calibration_Pool"
                    ],

                "Different_Pool":
                    different_pool,

                "Unused_Guard_Minutes":
                    unused_minutes_between,

                "Cross_Pool_Guard_Pass":
                    (
                        (
                            not different_pool
                        )

                        or

                        (
                            unused_minutes_between
                            >=
                            CALIBRATION_GUARD_MINUTES
                        )
                    )
            }
        )


guard_audit = pd.DataFrame(
    guard_audit_rows
)


if len(
    guard_audit
) > 0:

    cross_pool_guard_failures = int(

        (
            ~guard_audit[
                "Cross_Pool_Guard_Pass"
            ]
        )

        .sum()
    )


else:

    cross_pool_guard_failures = 0


print(
    "Cross-pool temporal guard failures:",
    cross_pool_guard_failures
)


# ==================================================================================================
# 11.2O — SPLIT EACH SAFE 30-MINUTE GROUP INTO SIX 5-MINUTE BLOCKS
# ==================================================================================================

banner(
    "11.2O — BUILD FINAL LEAKAGE-SAFE 5-MINUTE BENIGN BLOCKS"
)


safe_block_frames = []

safe_block_index_rows = []


for group_id, group in safe_group_rows.groupby(
    "Calibration_Group_ID",
    observed=True,
    sort=False
):

    group = (

        group

        .sort_values(
            "Real_Minute"
        )

        .reset_index(
            drop=True
        )
    )


    if len(
        group
    ) != CALIBRATION_GROUP_LENGTH_MINUTES:

        raise RuntimeError(
            f"{group_id} does not contain exactly "
            f"{CALIBRATION_GROUP_LENGTH_MINUTES} minutes."
        )


    source = group[
        "Calibration_Source"
    ].iloc[0]


    pool = group[
        "Calibration_Pool"
    ].iloc[0]


    run_id = group[
        "Benign_Run_ID"
    ].iloc[0]


    number_of_blocks = (

        CALIBRATION_GROUP_LENGTH_MINUTES

        //

        BENIGN_BLOCK_LENGTH_MINUTES
    )


    for block_number in range(
        number_of_blocks
    ):

        start = (

            block_number
            *
            BENIGN_BLOCK_LENGTH_MINUTES
        )


        end = (

            start
            +
            BENIGN_BLOCK_LENGTH_MINUTES
        )


        block = (

            group.iloc[
                start:end
            ]

            .copy()

            .reset_index(
                drop=True
            )
        )


        block_id = (

            f"{group_id}"
            f"::B_{block_number + 1:02d}"
        )


        block[
            "Safe_Benign_Block_ID"
        ] = block_id


        block[
            "Block_Row_Position"
        ] = np.arange(
            BENIGN_BLOCK_LENGTH_MINUTES
        )


        safe_block_frames.append(
            block
        )


        safe_block_index_rows.append(
            {
                "Calibration_Source":
                    source,

                "Benign_Run_ID":
                    run_id,

                "Calibration_Group_ID":
                    group_id,

                "Calibration_Pool":
                    pool,

                "Safe_Benign_Block_ID":
                    block_id,

                "Block_Start":
                    block[
                        "Real_Minute"
                    ].min(),

                "Block_End":
                    block[
                        "Real_Minute"
                    ].max(),

                "Block_Length":
                    len(
                        block
                    )
            }
        )


safe_block_rows = pd.concat(
    safe_block_frames,
    ignore_index=True
)


safe_block_index = pd.DataFrame(
    safe_block_index_rows
)


safe_block_pool_summary = (

    safe_block_index

    .groupby(
        [
            "Calibration_Source",
            "Calibration_Pool"
        ],
        observed=True
    )

    .size()

    .rename(
        "Safe_Block_Count"
    )

    .reset_index()
)


print(
    "Final leakage-safe 5-minute blocks:",
    len(
        safe_block_index
    )
)


print(
    "\nSafe block pool summary:"
)


print(
    safe_block_pool_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2P — SAFE BLOCK INTEGRITY
# ==================================================================================================

banner(
    "11.2P — SAFE BLOCK STRUCTURAL INTEGRITY"
)


duplicate_real_minutes_in_safe_blocks = int(

    safe_block_rows

    .duplicated(
        subset=[
            "Calibration_Source",
            "Real_Segment_ID",
            "Real_Minute"
        ]
    )

    .sum()
)


incorrect_safe_block_lengths = int(

    (
        safe_block_index[
            "Block_Length"
        ]

        !=

        BENIGN_BLOCK_LENGTH_MINUTES
    )

    .sum()
)


multi_pool_block_count = int(

    (
        safe_block_rows

        .groupby(
            "Safe_Benign_Block_ID"
        )[
            "Calibration_Pool"
        ]

        .nunique()

        !=
        1
    )

    .sum()
)


multi_pool_group_count = int(

    (
        safe_group_rows

        .groupby(
            "Calibration_Group_ID"
        )[
            "Calibration_Pool"
        ]

        .nunique()

        !=
        1
    )

    .sum()
)


print(
    "Duplicate real minutes in safe blocks:",
    duplicate_real_minutes_in_safe_blocks
)

print(
    "Incorrect block lengths:",
    incorrect_safe_block_lengths
)

print(
    "Blocks appearing in multiple pools:",
    multi_pool_block_count
)

print(
    "Calibration groups appearing in multiple pools:",
    multi_pool_group_count
)


# ==================================================================================================
# 11.2Q — SAFE BLOCK DIVERSITY AUDIT
#
# Exact numeric block duplicates are diagnostic.
#
# We do NOT reject coincidentally identical traffic automatically,
# but we record whether exact duplicate baseline blocks exist.
# ==================================================================================================

banner(
    "11.2Q — SAFE BLOCK DIVERSITY AUDIT"
)


block_hash_rows = []


for block_id, block in safe_block_rows.groupby(
    "Safe_Benign_Block_ID",
    observed=True
):

    numeric = (

        block[
            FINAL_PREDICTORS
        ]

        .to_numpy(
            dtype=np.float64
        )
    )


    rounded = np.round(
        numeric,
        decimals=8
    )


    digest = hashlib.sha256(
        rounded.tobytes()
    ).hexdigest()


    block_hash_rows.append(
        {
            "Safe_Benign_Block_ID":
                block_id,

            "Calibration_Source":
                block[
                    "Calibration_Source"
                ].iloc[0],

            "Calibration_Pool":
                block[
                    "Calibration_Pool"
                ].iloc[0],

            "Block_Content_SHA256":
                digest
        }
    )


safe_block_hashes = pd.DataFrame(
    block_hash_rows
)


duplicate_content_hashes = (

    safe_block_hashes

    .groupby(
        "Block_Content_SHA256"
    )

    .size()
)


exact_duplicate_block_content_groups = int(

    (
        duplicate_content_hashes
        >
        1
    )

    .sum()
)


print(
    "Exact duplicate block-content groups:",
    exact_duplicate_block_content_groups
)


# ==================================================================================================
# 11.2R — FAMILY SOURCE SUPPORT + SOURCE MIXTURE WEIGHTS
#
# We freeze SOURCE-CONDITIONED generation.
#
# Dual-source family:
#
#       source weight ∝ sqrt(real flow support)
#
# Why sqrt?
#
# Raw-count weighting would let very large sources almost completely dominate.
# Equal weighting would over-amplify very small sources.
#
# sqrt-support is a compromise:
#
#       real evidence matters,
#       but dataset size does not fully dictate the simulator.
#
# ==================================================================================================

banner(
    "11.2R — SOURCE-CONDITIONED FAMILY GENERATION WEIGHTS"
)


family_support[
    "CIC_Attack_Flows"
] = pd.to_numeric(
    family_support[
        "CIC_Attack_Flows"
    ],
    errors="coerce"
)


family_support[
    "UNSW_Attack_Flows"
] = pd.to_numeric(
    family_support[
        "UNSW_Attack_Flows"
    ],
    errors="coerce"
)


source_weight_rows = []


for _, row in family_support.iterrows():

    family = row[
        "Synthetic_Family"
    ]


    source_counts = {

        "CIC_IDS2017":
            float(
                row[
                    "CIC_Attack_Flows"
                ]
            ),

        "UNSW_NB15":
            float(
                row[
                    "UNSW_Attack_Flows"
                ]
            )
    }


    usable_sources = {

        source:
            count

        for source, count
        in source_counts.items()

        if count
        >=
        MIN_SOURCE_FLOWS_FOR_GENERATION
    }


    if len(
        usable_sources
    ) == 0:

        source_weight_rows.append(
            {
                "Synthetic_Family":
                    family,

                "Calibration_Source":
                    "NONE",

                "Real_Flow_Support":
                    0,

                "Source_Weight":
                    0.0,

                "Usable_For_Generation":
                    False
            }
        )

        continue


    sqrt_support = {

        source:
            math.sqrt(
                count
            )

        for source, count
        in usable_sources.items()
    }


    denominator = sum(
        sqrt_support.values()
    )


    for source, count in source_counts.items():

        usable = (
            source
            in
            usable_sources
        )


        weight = (

            sqrt_support[
                source
            ]
            /
            denominator

            if usable

            else 0.0
        )


        source_weight_rows.append(
            {
                "Synthetic_Family":
                    family,

                "Calibration_Source":
                    source,

                "Real_Flow_Support":
                    int(
                        count
                    ),

                "Source_Weight":
                    float(
                        weight
                    ),

                "Usable_For_Generation":
                    bool(
                        usable
                    )
            }
        )


source_generation_weights = pd.DataFrame(
    source_weight_rows
)


print(
    source_generation_weights.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2S — FAMILY APPROVAL / RESTRICTION DECISIONS
# ==================================================================================================

banner(
    "11.2S — FAMILY APPROVAL / RESTRICTION DECISIONS"
)


decision_rows = []


for family in ATTACK_FAMILIES:

    support_row = family_support.loc[
        family_support[
            "Synthetic_Family"
        ]
        ==
        family
    ]


    if len(
        support_row
    ) != 1:

        raise RuntimeError(
            f"Family support row missing/duplicated for {family}."
        )


    support_row = support_row.iloc[
        0
    ]


    total_flows = int(
        support_row[
            "Total_Real_Attack_Flows"
        ]
    )


    cic_flows = int(
        support_row[
            "CIC_Attack_Flows"
        ]
    )


    unsw_flows = int(
        support_row[
            "UNSW_Attack_Flows"
        ]
    )


    source_counts = [

        x

        for x in [
            cic_flows,
            unsw_flows
        ]

        if x > 0
    ]


    sources_with_any_support = len(
        source_counts
    )


    usable_source_rows = source_generation_weights.loc[

        (
            source_generation_weights[
                "Synthetic_Family"
            ]
            ==
            family
        )

        &

        (
            source_generation_weights[
                "Usable_For_Generation"
            ]
            ==
            True
        )

    ]


    usable_sources = usable_source_rows[
        "Calibration_Source"
    ].tolist()


    # ------------------------------------------------------------------------------------------
    # Active minute support
    # ------------------------------------------------------------------------------------------

    active_rows = family_minute_support.loc[
        family_minute_support[
            "Synthetic_Family"
        ]
        ==
        family
    ]


    total_active_minutes = int(

        pd.to_numeric(
            active_rows[
                "Family_Active_Minutes"
            ],
            errors="coerce"
        )

        .fillna(
            0
        )

        .sum()
    )


    # ------------------------------------------------------------------------------------------
    # Effect coverage
    # ------------------------------------------------------------------------------------------

    pooled_family = pooled_effects.loc[
        pooled_effects[
            "Synthetic_Family"
        ]
        ==
        family
    ]


    pooled_effect_feature_count = int(
        pooled_family[
            "Feature"
        ].nunique()
    )


    pooled_effect_finite = bool(

        np.isfinite(

            pd.to_numeric(
                pooled_family[
                    "Effect_Log1p"
                ],
                errors="coerce"
            )

        ).all()
    )


    # ------------------------------------------------------------------------------------------
    # Cross-source agreement
    # ------------------------------------------------------------------------------------------

    cross = dual_source_effect_summary.loc[
        dual_source_effect_summary[
            "Synthetic_Family"
        ]
        ==
        family
    ].iloc[0]


    direction_agreement = cross[
        "Direction_Agreement_Rate"
    ]


    median_effect_difference = cross[
        "Median_Absolute_Effect_Difference"
    ]


    # ------------------------------------------------------------------------------------------
    # Intensity divergence
    # ------------------------------------------------------------------------------------------

    intensity = intensity_divergence.loc[
        intensity_divergence[
            "Synthetic_Family"
        ]
        ==
        family
    ].iloc[0]


    raw_intensity_fold = intensity[
        "Median_Intensity_Fold_Difference"
    ]


    # ------------------------------------------------------------------------------------------
    # Source dominance
    # ------------------------------------------------------------------------------------------

    if total_flows > 0:

        source_dominance = (

            max(
                cic_flows,
                unsw_flows
            )

            /
            total_flows
        )


    else:

        source_dominance = np.nan


    hard_fail_reasons = []

    restrictions = []


    # ==========================================================================================
    # HARD FAILURES
    # ==========================================================================================

    if total_flows < MIN_TOTAL_FAMILY_FLOWS:

        hard_fail_reasons.append(
            "INSUFFICIENT_TOTAL_REAL_FLOW_SUPPORT"
        )


    if total_active_minutes < MIN_FAMILY_ACTIVE_MINUTES:

        hard_fail_reasons.append(
            "INSUFFICIENT_ACTIVE_MINUTE_SUPPORT"
        )


    if pooled_effect_feature_count != EXPECTED_EFFECT_FEATURE_COUNT:

        hard_fail_reasons.append(
            "INCOMPLETE_EFFECT_VECTOR"
        )


    if not pooled_effect_finite:

        hard_fail_reasons.append(
            "NONFINITE_EFFECT_VECTOR"
        )


    if len(
        usable_sources
    ) == 0:

        hard_fail_reasons.append(
            "NO_SOURCE_WITH_SUFFICIENT_GENERATION_SUPPORT"
        )


    # ==========================================================================================
    # RESTRICTIONS
    # ==========================================================================================

    if len(
        usable_sources
    ) == 1:

        restrictions.append(
            "SINGLE_SOURCE_CALIBRATION"
        )


    # Some support may exist in a source but be below source-level generation threshold.

    if (
        0
        <
        cic_flows
        <
        MIN_SOURCE_FLOWS_FOR_GENERATION
    ):

        restrictions.append(
            "CIC_LOW_SUPPORT_EXCLUDED_FROM_GENERATION"
        )


    if (
        0
        <
        unsw_flows
        <
        MIN_SOURCE_FLOWS_FOR_GENERATION
    ):

        restrictions.append(
            "UNSW_LOW_SUPPORT_EXCLUDED_FROM_GENERATION"
        )


    if (
        len(
            usable_sources
        )
        ==
        2
    ):

        if (
            pd.notna(
                direction_agreement
            )
            and
            direction_agreement
            <
            MIN_CROSS_SOURCE_DIRECTION_AGREEMENT
        ):

            restrictions.append(
                "CROSS_SOURCE_EFFECT_DIRECTION_DISAGREEMENT"
            )


        if (
            pd.notna(
                median_effect_difference
            )
            and
            median_effect_difference
            >
            MAX_MEDIAN_CROSS_SOURCE_EFFECT_DIFFERENCE
        ):

            restrictions.append(
                "CROSS_SOURCE_EFFECT_MAGNITUDE_DIVERGENCE"
            )


        if (
            pd.notna(
                raw_intensity_fold
            )
            and
            raw_intensity_fold
            >=
            RAW_INTENSITY_FOLD_WARNING
        ):

            restrictions.append(
                "RAW_ATTACK_INTENSITY_SCALE_DIVERGENCE"
            )


        if (
            pd.notna(
                source_dominance
            )
            and
            source_dominance
            >=
            SOURCE_DOMINANCE_WARNING
        ):

            restrictions.append(
                "SOURCE_SUPPORT_IMBALANCE"
            )


    # EXPLOIT_APP is intentionally broad and must never be presented
    # as a homogeneous original attack class.

    if family == "EXPLOIT_APP":

        restrictions.append(
            "BROAD_HETEROGENEOUS_EXPLOIT_FAMILY"
        )


    restrictions = sorted(
        set(
            restrictions
        )
    )


    hard_fail_reasons = sorted(
        set(
            hard_fail_reasons
        )
    )


    if hard_fail_reasons:

        decision = "REJECTED"


    elif restrictions:

        decision = "APPROVED_WITH_RESTRICTION"


    else:

        decision = "APPROVED"


    if decision == "REJECTED":

        calibration_mode = "NONE"


    elif len(
        usable_sources
    ) == 1:

        calibration_mode = (
            "SOURCE_CONDITIONED_SINGLE_SOURCE"
        )


    else:

        calibration_mode = (
            "SOURCE_CONDITIONED_WEIGHTED_MIXTURE"
        )


    decision_rows.append(
        {
            "Synthetic_Family":
                family,

            "Decision":
                decision,

            "Calibration_Mode":
                calibration_mode,

            "Total_Real_Attack_Flows":
                total_flows,

            "CIC_Attack_Flows":
                cic_flows,

            "UNSW_Attack_Flows":
                unsw_flows,

            "Total_Active_Minutes":
                total_active_minutes,

            "Usable_Source_Count":
                len(
                    usable_sources
                ),

            "Usable_Sources":
                " | ".join(
                    usable_sources
                ),

            "Pooled_Effect_Feature_Count":
                pooled_effect_feature_count,

            "Cross_Source_Direction_Agreement_Rate":
                direction_agreement,

            "Median_Cross_Source_Effect_Difference":
                median_effect_difference,

            "Median_Raw_Intensity_Fold_Difference":
                raw_intensity_fold,

            "Largest_Source_Flow_Share":
                source_dominance,

            "Restrictions":
                (
                    " | ".join(
                        restrictions
                    )

                    if restrictions

                    else "NONE"
                ),

            "Hard_Fail_Reasons":
                (
                    " | ".join(
                        hard_fail_reasons
                    )

                    if hard_fail_reasons

                    else "NONE"
                )
        }
    )


family_decisions = pd.DataFrame(
    decision_rows
)


print(
    family_decisions.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2T — GENERATION POLICY BY FAMILY
#
# IMPORTANT:
#
# Source-specific effects from 11.1 are now authoritative for generation.
#
# The pooled median vector remains a diagnostic artifact, NOT the default
# generator effect vector.
#
# ==================================================================================================

banner(
    "11.2T — FREEZE SOURCE-CONDITIONED GENERATION POLICY"
)


generation_policy_rows = []


for family in ATTACK_FAMILIES:

    decision = family_decisions.loc[
        family_decisions[
            "Synthetic_Family"
        ]
        ==
        family
    ].iloc[0]


    weights = source_generation_weights.loc[

        (
            source_generation_weights[
                "Synthetic_Family"
            ]
            ==
            family
        )

        &

        (
            source_generation_weights[
                "Usable_For_Generation"
            ]
            ==
            True
        )

    ]


    cic_weight_row = weights.loc[
        weights[
            "Calibration_Source"
        ]
        ==
        "CIC_IDS2017"
    ]


    unsw_weight_row = weights.loc[
        weights[
            "Calibration_Source"
        ]
        ==
        "UNSW_NB15"
    ]


    cic_weight = (

        float(
            cic_weight_row[
                "Source_Weight"
            ].iloc[0]
        )

        if len(
            cic_weight_row
        )

        else 0.0
    )


    unsw_weight = (

        float(
            unsw_weight_row[
                "Source_Weight"
            ].iloc[0]
        )

        if len(
            unsw_weight_row
        )

        else 0.0
    )


    generation_policy_rows.append(
        {
            "Synthetic_Family":
                family,

            "Decision":
                decision[
                    "Decision"
                ],

            "Calibration_Mode":
                decision[
                    "Calibration_Mode"
                ],

            "CIC_Source_Weight":
                cic_weight,

            "UNSW_Source_Weight":
                unsw_weight,

            "Benign_Baseline_Source_Must_Match_Effect_Source":
                True,

            "Use_Source_Specific_Effect_Vector":
                True,

            "Use_Pooled_Effect_Vector_For_Generation":
                False,

            "Use_Safe_Split_Specific_Benign_Blocks":
                True,

            "Allow_Raw_CIC_UNSW_Magnitude_Pooling":
                False
        }
    )


family_generation_policy = pd.DataFrame(
    generation_policy_rows
)


print(
    family_generation_policy.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2U — CHECK GENERATION WEIGHTS
# ==================================================================================================

banner(
    "11.2U — GENERATION WEIGHT VALIDATION"
)


weight_validation_rows = []


for family in ATTACK_FAMILIES:

    rows = family_generation_policy.loc[
        family_generation_policy[
            "Synthetic_Family"
        ]
        ==
        family
    ]


    total_weight = float(

        rows[
            "CIC_Source_Weight"
        ].iloc[0]

        +

        rows[
            "UNSW_Source_Weight"
        ].iloc[0]
    )


    weight_validation_rows.append(
        {
            "Synthetic_Family":
                family,

            "Weight_Sum":
                total_weight,

            "Pass":
                np.isclose(
                    total_weight,
                    1.0,
                    atol=1e-12
                )
        }
    )


generation_weight_validation = pd.DataFrame(
    weight_validation_rows
)


print(
    generation_weight_validation.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2V — CALIBRATION USAGE POLICY
#
# This explicitly separates:
#
#       simulator calibration
#
# from:
#
#       ML train/validation/test fitting.
#
# Global source effect estimates are generator parameters.
# They are NOT fitted by the downstream model.
#
# Exact benign seed blocks, however, remain split-specific.
# ==================================================================================================

banner(
    "11.2V — CALIBRATION VS MODEL-LEARNING POLICY"
)


calibration_usage_policy = pd.DataFrame([

    {
        "Component":
            "Family effect estimation",

        "Policy":
            "Frozen simulator parameter derived from real calibration data",

        "Split_Specific":
            False,

        "Reason":
            "Defines the benchmark-generating process; downstream ML model never fits this parameter."
    },

    {
        "Component":
            "Attack intensity distribution",

        "Policy":
            "Frozen source-conditioned simulator parameter",

        "Split_Specific":
            False,

        "Reason":
            "Defines source-specific synthetic attack scale."
    },

    {
        "Component":
            "Exact real benign seed blocks",

        "Policy":
            "Strictly split-specific",

        "Split_Specific":
            True,

        "Reason":
            "Prevents exact real temporal baseline blocks from appearing across synthetic partitions."
    },

    {
        "Component":
            "Synthetic episode random seed",

        "Policy":
            "Unique per episode",

        "Split_Specific":
            True,

        "Reason":
            "Prevents identical stochastic episode reconstruction."
    },

    {
        "Component":
            "ML scaler / encoder / feature selection",

        "Policy":
            "Fit on synthetic TRAIN only",

        "Split_Specific":
            True,

        "Reason":
            "Standard downstream leakage prevention."
    }
])


print(
    calibration_usage_policy.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.2W — PHASE VALIDATION
# ==================================================================================================

banner(
    "11.2W — FINAL VALIDATION"
)


validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                details
        }
    )


add_check(

    "Frozen_11_0_Config_Unchanged",

    CURRENT_CONFIG_HASH
    ==
    FROZEN_CONFIG_HASH,

    CURRENT_CONFIG_HASH
)


add_check(

    "Phase_11_1_Passed",

    PHASE_11_1_PASS,

    str(
        PHASE_11_1_PASS
    )
)


add_check(

    "All_11_1_Validation_Checks_Passed",

    ALL_11_1_CHECKS_PASS,

    f"{int(validation_pass_series.sum())}/{len(validation_pass_series)}"
)


add_check(

    "Five_Configured_Families_Present",

    set(
        family_support[
            "Synthetic_Family"
        ]
    )
    ==
    set(
        ATTACK_FAMILIES
    ),

    str(
        sorted(
            family_support[
                "Synthetic_Family"
            ].tolist()
        )
    )
)


rejected_families = family_decisions.loc[
    family_decisions[
        "Decision"
    ]
    ==
    "REJECTED",
    "Synthetic_Family"
].tolist()


add_check(

    "No_Family_Rejected",

    len(
        rejected_families
    )
    ==
    0,

    (
        "NONE"

        if not rejected_families

        else str(
            rejected_families
        )
    )
)


add_check(

    "All_Family_Effect_Vectors_Have_Expected_Features",

    bool(

        (
            family_decisions[
                "Pooled_Effect_Feature_Count"
            ]

            ==
            EXPECTED_EFFECT_FEATURE_COUNT
        ).all()
    ),

    family_decisions[
        [
            "Synthetic_Family",
            "Pooled_Effect_Feature_Count"
        ]
    ].to_dict(
        orient="records"
    )
)


add_check(

    "All_Generation_Source_Weights_Sum_To_One",

    bool(
        generation_weight_validation[
            "Pass"
        ].all()
    ),

    generation_weight_validation[
        [
            "Synthetic_Family",
            "Weight_Sum"
        ]
    ].to_dict(
        orient="records"
    )
)


add_check(

    "No_Raw_Magnitude_Pooling_Allowed",

    bool(

        (
            family_generation_policy[
                "Allow_Raw_CIC_UNSW_Magnitude_Pooling"
            ]

            ==
            False
        ).all()
    ),

    "Source-conditioned generation frozen"
)


add_check(

    "Baseline_Source_Must_Match_Effect_Source",

    bool(

        family_generation_policy[
            "Benign_Baseline_Source_Must_Match_Effect_Source"
        ].all()
    ),

    "Required for every family"
)


add_check(

    "Benign_Calibration_Groups_Created",

    len(
        safe_group_index
    )
    >
    0,

    str(
        len(
            safe_group_index
        )
    )
)


for source in [
    "CIC_IDS2017",
    "UNSW_NB15"
]:

    for pool in [
        "CAL_TRAIN",
        "CAL_VALIDATION",
        "CAL_TEST"
    ]:

        group_count = int(

            (
                (
                    safe_group_index[
                        "Calibration_Source"
                    ]
                    ==
                    source
                )

                &

                (
                    safe_group_index[
                        "Calibration_Pool"
                    ]
                    ==
                    pool
                )
            )

            .sum()
        )


        block_count = int(

            (
                (
                    safe_block_index[
                        "Calibration_Source"
                    ]
                    ==
                    source
                )

                &

                (
                    safe_block_index[
                        "Calibration_Pool"
                    ]
                    ==
                    pool
                )
            )

            .sum()
        )


        add_check(

            f"{source}_{pool}_Has_Safe_Groups",

            group_count
            >
            0,

            str(
                group_count
            )
        )


        add_check(

            f"{source}_{pool}_Has_Safe_Blocks",

            block_count
            >
            0,

            str(
                block_count
            )
        )


add_check(

    "No_CrossPool_Guard_Violations",

    cross_pool_guard_failures
    ==
    0,

    str(
        cross_pool_guard_failures
    )
)


add_check(

    "No_Real_Minute_Reused_In_Multiple_Safe_Blocks",

    duplicate_real_minutes_in_safe_blocks
    ==
    0,

    str(
        duplicate_real_minutes_in_safe_blocks
    )
)


add_check(

    "Every_Safe_Block_Has_Exactly_5_Minutes",

    incorrect_safe_block_lengths
    ==
    0,

    str(
        incorrect_safe_block_lengths
    )
)


add_check(

    "Every_Safe_Block_Belongs_To_One_Pool",

    multi_pool_block_count
    ==
    0,

    str(
        multi_pool_block_count
    )
)


add_check(

    "Every_Calibration_Group_Belongs_To_One_Pool",

    multi_pool_group_count
    ==
    0,

    str(
        multi_pool_group_count
    )
)


add_check(

    "Benign_Source_Scale_Audit_Contains_All_14_Predictors",

    len(
        benign_scale_audit
    )
    ==
    len(
        FINAL_PREDICTORS
    ),

    f"{len(benign_scale_audit)}/{len(FINAL_PREDICTORS)}"
)


add_check(

    "Synthetic_Generation_Still_Not_Performed",

    (
        provenance_11_1.get(
            "synthetic_generation_performed",
            None
        )

        is False
    ),

    str(
        provenance_11_1.get(
            "synthetic_generation_performed",
            None
        )
    )
)


validation_11_2 = pd.DataFrame(
    validation_rows
)


print(
    validation_11_2.to_string(
        index=False
    )
)


PHASE_11_2_PASS = bool(

    validation_11_2[
        "Passed"
    ].all()
)


# ==================================================================================================
# 11.2X — SAVE AUDIT ARTIFACTS
# ==================================================================================================

banner(
    "11.2X — SAVE PHASE 11.2 ARTIFACTS"
)


def save_csv(
    dataframe,
    filename
):

    path = (

        CALIBRATION_DIR
        /
        filename
    )


    dataframe.to_csv(
        path,
        index=False
    )


    print(
        "✓",
        path
    )


    return path


save_csv(

    benign_scale_audit,

    "11_2_benign_source_scale_audit.csv"
)


save_csv(

    benign_correlations,

    "11_2_benign_correlation_matrices_long.csv"
)


save_csv(

    benign_correlation_difference,

    "11_2_benign_correlation_difference.csv"
)


save_csv(

    effect_agreement,

    "11_2_cross_source_effect_agreement.csv"
)


save_csv(

    dual_source_effect_summary,

    "11_2_cross_source_effect_summary.csv"
)


save_csv(

    intensity_divergence,

    "11_2_family_intensity_divergence.csv"
)


save_csv(

    exploit_composition,

    "11_2_exploit_app_composition.csv"
)


save_csv(

    benign_run_index,

    "11_2_benign_run_index.csv"
)


save_csv(

    safe_group_index,

    "11_2_safe_calibration_group_index.csv"
)


save_csv(

    safe_group_rows,

    "11_2_safe_calibration_group_rows.csv"
)


save_csv(

    safe_group_pool_summary,

    "11_2_safe_calibration_group_pool_summary.csv"
)


save_csv(

    guard_audit,

    "11_2_cross_pool_guard_audit.csv"
)


save_csv(

    safe_block_index,

    "11_2_safe_benign_block_index.csv"
)


save_csv(

    safe_block_rows,

    "11_2_safe_benign_block_rows.csv"
)


save_csv(

    safe_block_pool_summary,

    "11_2_safe_benign_block_pool_summary.csv"
)


save_csv(

    safe_block_hashes,

    "11_2_safe_benign_block_hashes.csv"
)


save_csv(

    source_generation_weights,

    "11_2_family_source_generation_weights.csv"
)


save_csv(

    family_decisions,

    "11_2_family_decisions.csv"
)


save_csv(

    family_generation_policy,

    "11_2_family_generation_policy.csv"
)


save_csv(

    generation_weight_validation,

    "11_2_generation_weight_validation.csv"
)


save_csv(

    calibration_usage_policy,

    "11_2_calibration_usage_policy.csv"
)


save_csv(

    validation_11_2,

    "11_2_validation.csv"
)


# ==================================================================================================
# 11.2Y — BUILD FROZEN CALIBRATION POLICY JSON
# ==================================================================================================

banner(
    "11.2Y — BUILD FROZEN CALIBRATION POLICY"
)


family_policy_dict = {}


for family in ATTACK_FAMILIES:

    decision_row = family_decisions.loc[
        family_decisions[
            "Synthetic_Family"
        ]
        ==
        family
    ].iloc[0]


    generation_row = family_generation_policy.loc[
        family_generation_policy[
            "Synthetic_Family"
        ]
        ==
        family
    ].iloc[0]


    family_policy_dict[
        family
    ] = {

        "decision":
            decision_row[
                "Decision"
            ],

        "calibration_mode":
            decision_row[
                "Calibration_Mode"
            ],

        "restrictions":
            (
                []

                if decision_row[
                    "Restrictions"
                ]
                ==
                "NONE"

                else decision_row[
                    "Restrictions"
                ].split(
                    " | "
                )
            ),

        "cic_source_weight":
            float(
                generation_row[
                    "CIC_Source_Weight"
                ]
            ),

        "unsw_source_weight":
            float(
                generation_row[
                    "UNSW_Source_Weight"
                ]
            ),

        "source_specific_effect_required":
            True,

        "baseline_source_must_match_effect_source":
            True,

        "raw_cross_dataset_magnitude_pooling":
            False
    }


calibration_policy = {

    "phase":
        "11.2",

    "policy_version":
        MASTER_POLICY_VERSION,

    "dataset":
        "Synthetic_NAF_v1",

    "spec_config_sha256":
        CURRENT_CONFIG_HASH,

    "phase_11_1_pass":
        PHASE_11_1_PASS,

    "phase_11_2_pass":
        PHASE_11_2_PASS,

    "source_conditioning": {

        "required":
            True,

        "reason":
            (
                "CIC-IDS2017 and UNSW-NB15 contain major differences "
                "in benign and attack traffic scale; baseline and family "
                "effect must originate from the same calibration source."
            ),

        "raw_magnitude_pooling_allowed":
            False,

        "source_weight_method":
            "SQRT_REAL_FLOW_SUPPORT_NORMALIZED"
    },

    "benign_calibration": {

        "input":
            "11_1_real_benign_minute_pool.csv",

        "group_length_minutes":
            CALIBRATION_GROUP_LENGTH_MINUTES,

        "guard_minutes":
            CALIBRATION_GUARD_MINUTES,

        "block_length_minutes":
            BENIGN_BLOCK_LENGTH_MINUTES,

        "split_before_synthetic_generation":
            True,

        "pool_mapping": {

            "synthetic_train":
                "CAL_TRAIN",

            "synthetic_validation":
                "CAL_VALIDATION",

            "synthetic_test":
                "CAL_TEST"
        },

        "same_block_across_pools":
            False,

        "same_group_across_pools":
            False
    },

    "family_policy":
        family_policy_dict,

    "thresholds": {

        "minimum_total_family_flows":
            MIN_TOTAL_FAMILY_FLOWS,

        "minimum_source_flows_for_generation":
            MIN_SOURCE_FLOWS_FOR_GENERATION,

        "minimum_family_active_minutes":
            MIN_FAMILY_ACTIVE_MINUTES,

        "effect_near_zero":
            EFFECT_NEAR_ZERO,

        "minimum_cross_source_direction_agreement":
            MIN_CROSS_SOURCE_DIRECTION_AGREEMENT,

        "maximum_median_cross_source_effect_difference_before_restriction":
            MAX_MEDIAN_CROSS_SOURCE_EFFECT_DIFFERENCE,

        "raw_intensity_fold_warning":
            RAW_INTENSITY_FOLD_WARNING,

        "source_dominance_warning":
            SOURCE_DOMINANCE_WARNING
    },

    "downstream_model_policy": {

        "scaler_fit":
            "SYNTHETIC_TRAIN_ONLY",

        "feature_selection":
            "SYNTHETIC_TRAIN_ONLY",

        "resampling":
            "SYNTHETIC_TRAIN_ONLY",

        "test_regeneration_after_model_results":
            False
    },

    "synthetic_generation_performed":
        False
}


POLICY_FILE = (

    CALIBRATION_DIR
    /
    "11_2_frozen_calibration_policy.json"
)


with open(
    POLICY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        calibration_policy,
        f,
        indent=4
    )


print(
    "✓",
    POLICY_FILE
)


# ==================================================================================================
# 11.2Z — POLICY HASH
# ==================================================================================================

banner(
    "11.2Z — FREEZE CALIBRATION POLICY HASH"
)


with open(
    POLICY_FILE,
    "rb"
) as f:

    policy_bytes = f.read()


POLICY_SHA256 = hashlib.sha256(
    policy_bytes
).hexdigest()


POLICY_HASH_FILE = (

    CALIBRATION_DIR
    /
    "11_2_frozen_calibration_policy_SHA256.txt"
)


with open(
    POLICY_HASH_FILE,
    "w"
) as f:

    f.write(
        POLICY_SHA256
        +
        "\n"
    )


print(
    "Calibration policy SHA256:"
)

print(
    POLICY_SHA256
)


print(
    "\n✓",
    POLICY_HASH_FILE
)


# ==================================================================================================
# 11.2AA — FINAL MANIFEST
# ==================================================================================================

banner(
    "11.2AA — PHASE 11.2 MANIFEST"
)


approved_count = int(

    (
        family_decisions[
            "Decision"
        ]
        ==
        "APPROVED"
    ).sum()
)


restricted_count = int(

    (
        family_decisions[
            "Decision"
        ]
        ==
        "APPROVED_WITH_RESTRICTION"
    ).sum()
)


rejected_count = int(

    (
        family_decisions[
            "Decision"
        ]
        ==
        "REJECTED"
    ).sum()
)


manifest_11_2 = pd.DataFrame({

    "Metric": [

        "Configured_Attack_Families",

        "Approved_Families",

        "Approved_With_Restriction_Families",

        "Rejected_Families",

        "Benign_Predictor_Count",

        "Major_Benign_Scale_Divergence_Features",

        "Mean_Absolute_Benign_Correlation_Difference",

        "Median_Absolute_Benign_Correlation_Difference",

        "Calibration_Group_Length_Minutes",

        "Calibration_Guard_Minutes",

        "Safe_Calibration_Groups",

        "Safe_Benign_Blocks",

        "Exact_Duplicate_Block_Content_Groups",

        "Cross_Pool_Guard_Failures",

        "Phase_11_2_Validation_Checks",

        "Phase_11_2_Validation_Passed",

        "Phase_11_2_Pass",

        "Synthetic_Generation_Performed",

        "Calibration_Policy_SHA256"

    ],


    "Value": [

        len(
            ATTACK_FAMILIES
        ),

        approved_count,

        restricted_count,

        rejected_count,

        len(
            FINAL_PREDICTORS
        ),

        major_benign_scale_features,

        mean_corr_difference,

        median_corr_difference,

        CALIBRATION_GROUP_LENGTH_MINUTES,

        CALIBRATION_GUARD_MINUTES,

        len(
            safe_group_index
        ),

        len(
            safe_block_index
        ),

        exact_duplicate_block_content_groups,

        cross_pool_guard_failures,

        len(
            validation_11_2
        ),

        int(
            validation_11_2[
                "Passed"
            ].sum()
        ),

        PHASE_11_2_PASS,

        False,

        POLICY_SHA256
    ]
})


print(
    manifest_11_2.to_string(
        index=False
    )
)


save_csv(

    manifest_11_2,

    "11_2_calibration_validity_manifest.csv"
)


# ==================================================================================================
# 11.2AB — FINAL DECISION
# ==================================================================================================

banner(
    "11.2AB — FINAL CALIBRATION DECISION"
)


if not PHASE_11_2_PASS:

    failed_checks = validation_11_2.loc[
        ~validation_11_2[
            "Passed"
        ]
    ]


    print(
        "\nFAILED CHECKS:\n"
    )


    print(
        failed_checks.to_string(
            index=False
        )
    )


    print(
        "\nFAMILY DECISIONS:\n"
    )


    print(
        family_decisions.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "PHASE 11.2 FAILED. "
        "DO NOT GENERATE THE 1,200-EPISODE MANIFEST."
    )


print(
    f"""
✓ PHASE 11.2 — CALIBRATION VALIDITY AUDIT: PASS


FAMILY STATUS
-------------
Configured families:

    {len(ATTACK_FAMILIES)}

Approved without restrictions:

    {approved_count}

Approved with restrictions:

    {restricted_count}

Rejected:

    {rejected_count}


SOURCE-CONDITIONED GENERATION
-----------------------------
CIC and UNSW absolute traffic magnitudes are NOT pooled.

For every attack episode:

    1. select a calibration source according to the
       frozen family-specific source weights

    2. select the benign baseline from THAT source

    3. apply the family effect vector from THAT SAME source

Therefore this is forbidden:

    CIC baseline
        +
    UNSW attack effect

and vice versa.


BENIGN TEMPORAL LEAKAGE CONTROL
-------------------------------
Original strict benign minutes:

    {len(benign_minutes):,}

Leakage-safe calibration groups:

    {len(safe_group_index):,}

Group length:

    {CALIBRATION_GROUP_LENGTH_MINUTES} minutes

Discarded temporal guard between groups:

    {CALIBRATION_GUARD_MINUTES} minutes

Final 5-minute seed blocks:

    {len(safe_block_index):,}

Cross-pool temporal guard failures:

    {cross_pool_guard_failures}

Real minutes reused across different blocks:

    {duplicate_real_minutes_in_safe_blocks}


SYNTHETIC SPLIT MAPPING
-----------------------
Synthetic TRAIN:

    CAL_TRAIN blocks only

Synthetic VALIDATION:

    CAL_VALIDATION blocks only

Synthetic TEST:

    CAL_TEST blocks only


IMPORTANT
---------
The old Phase 11.1 calibration block pool assignments are now SUPERSEDED.

For synthetic generation, use ONLY:

    11_2_safe_benign_block_index.csv
    11_2_safe_benign_block_rows.csv

Do NOT use:

    11_1_benign_block_index.csv
    11_1_benign_block_rows.csv

for final synthetic episode generation.


EFFECT POLICY
-------------
Source-specific family effect vectors are authoritative.

Use:

    11_1_source_family_log_effect_vectors.csv

with:

    11_2_family_generation_policy.csv

The pooled 11.1 effect vector is now diagnostic only.


NEXT
----
PHASE 11.3

Create the complete frozen 1,200-episode manifest BEFORE
generating any synthetic feature values.

11.3 will assign:

    Episode_ID
    TRAIN / VALIDATION / TEST
    Attack family
    Benign-only status
    Calibration source
    Calibration block IDs
    Episode length
    Attack onset minute
    Attack duration
    Recovery duration
    Precursor regime
    Precursor length
    Episode-specific random seed

and verify that:

    every family = 140 / 30 / 30
    every regime has the frozen split distribution
    every episode seed is unique
    TRAIN/VAL/TEST calibration sources obey 11.2
    no real calibration block crosses synthetic partitions
    timestamps can later be generated without overlap

NO SYNTHETIC FEATURE VALUES HAVE BEEN GENERATED YET.


CALIBRATION POLICY SHA256
-------------------------
{POLICY_SHA256}
"""
)


print(
    "=" * 120
)

print(
    "PHASE 11.2 COMPLETE"
)

print(
    "=" * 120
)

In [ ]:
# ==================================================================================================
# PHASE 11.3 — FROZEN 1,200-EPISODE MANIFEST
# ==================================================================================================
#
# PURPOSE
# -------
# Create the COMPLETE synthetic episode plan BEFORE generating any synthetic traffic values.
#
#
# THIS PHASE ASSIGNS
# ------------------
#
#   ✓ Episode_ID
#   ✓ TRAIN / VALIDATION / TEST
#   ✓ ATTACK / BENIGN_ONLY
#   ✓ attack family
#   ✓ calibration source: CIC / UNSW
#   ✓ exact source quotas
#   ✓ precursor regime
#   ✓ precursor length
#   ✓ episode length
#   ✓ attack onset
#   ✓ attack duration
#   ✓ recovery duration
#   ✓ episode load multiplier
#   ✓ family effect multiplier
#   ✓ split-safe real benign block sequence
#   ✓ unique generator seed
#   ✓ deterministic UTC timeline
#   ✓ attack-onset timestamp
#   ✓ precursor timestamp
#
#
# THIS PHASE DOES NOT
# -------------------
#
#   ✗ generate synthetic network features
#   ✗ generate predictor rows
#   ✗ train a model
#   ✗ scale predictors
#   ✗ create temporal sequence windows
#
#
# CRITICAL RULE
# -------------
#
# Once this manifest passes:
#
#       TEST episodes are LOCKED.
#
# Phase 11.4 MUST use this exact manifest.
#
# ==================================================================================================


from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
import math
import warnings

warnings.filterwarnings("ignore")


# ==================================================================================================
# 11.3A — PATHS
# ==================================================================================================

print("=" * 120)
print("PHASE 11.3 — FROZEN 1,200-EPISODE MANIFEST")
print("=" * 120)


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)


DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)


SPEC_DIR = (
    DATASET_ROOT
    / "spec"
)


CALIBRATION_DIR = (
    DATASET_ROOT
    / "calibration"
)


MANIFEST_DIR = (
    DATASET_ROOT
    / "manifests"
)


AUDIT_DIR = (
    DATASET_ROOT
    / "audit"
)


MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config.json"
)


CONFIG_HASH_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config_SHA256.txt"
)


POLICY_FILE = (
    CALIBRATION_DIR
    / "11_2_frozen_calibration_policy.json"
)


POLICY_HASH_FILE = (
    CALIBRATION_DIR
    / "11_2_frozen_calibration_policy_SHA256.txt"
)


PHASE_11_2_MANIFEST_FILE = (
    CALIBRATION_DIR
    / "11_2_calibration_validity_manifest.csv"
)


FAMILY_GENERATION_POLICY_FILE = (
    CALIBRATION_DIR
    / "11_2_family_generation_policy.csv"
)


SAFE_BLOCK_INDEX_FILE = (
    CALIBRATION_DIR
    / "11_2_safe_benign_block_index.csv"
)


SOURCE_EFFECT_FILE = (
    CALIBRATION_DIR
    / "11_1_source_family_log_effect_vectors.csv"
)


# ==================================================================================================
# 11.3B — VERIFY INPUT ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3B — VERIFY FROZEN INPUT ARTIFACTS")
print("=" * 120)


required_files = [

    CONFIG_FILE,
    CONFIG_HASH_FILE,

    POLICY_FILE,
    POLICY_HASH_FILE,

    PHASE_11_2_MANIFEST_FILE,

    FAMILY_GENERATION_POLICY_FILE,

    SAFE_BLOCK_INDEX_FILE,

    SOURCE_EFFECT_FILE
]


missing = []


for path in required_files:

    if path.exists():

        print(
            "✓",
            path.name
        )

    else:

        print(
            "✗",
            path
        )

        missing.append(
            path
        )


if missing:

    raise FileNotFoundError(
        "Phase 11.3 blocked because required files are missing."
    )


# ==================================================================================================
# 11.3C — VERIFY 11.0 CONFIG HASH
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3C — VERIFY PHASE 11.0 CONFIG HASH")
print("=" * 120)


with open(
    CONFIG_FILE,
    "rb"
) as f:

    config_bytes = f.read()


CURRENT_CONFIG_SHA256 = hashlib.sha256(
    config_bytes
).hexdigest()


with open(
    CONFIG_HASH_FILE,
    "r"
) as f:

    FROZEN_CONFIG_SHA256 = (
        f.read()
        .strip()
    )


print(
    "Frozen:",
    FROZEN_CONFIG_SHA256
)

print(
    "Current:",
    CURRENT_CONFIG_SHA256
)


if CURRENT_CONFIG_SHA256 != FROZEN_CONFIG_SHA256:

    raise RuntimeError(
        "Phase 11.0 specification changed after freeze."
    )


with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(
        f
    )


# ==================================================================================================
# 11.3D — VERIFY 11.2 POLICY HASH
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3D — VERIFY PHASE 11.2 CALIBRATION POLICY")
print("=" * 120)


with open(
    POLICY_FILE,
    "rb"
) as f:

    policy_bytes = f.read()


CURRENT_POLICY_SHA256 = hashlib.sha256(
    policy_bytes
).hexdigest()


with open(
    POLICY_HASH_FILE,
    "r"
) as f:

    FROZEN_POLICY_SHA256 = (
        f.read()
        .strip()
    )


print(
    "Frozen:",
    FROZEN_POLICY_SHA256
)

print(
    "Current:",
    CURRENT_POLICY_SHA256
)


if CURRENT_POLICY_SHA256 != FROZEN_POLICY_SHA256:

    raise RuntimeError(
        "Phase 11.2 calibration policy changed after freeze."
    )


with open(
    POLICY_FILE,
    "r",
    encoding="utf-8"
) as f:

    POLICY = json.load(
        f
    )


if not POLICY.get(
    "phase_11_2_pass",
    False
):

    raise RuntimeError(
        "Phase 11.2 policy does not report PASS."
    )


print(
    "\n✓ Phase 11.2 calibration policy verified"
)


# ==================================================================================================
# 11.3E — VERIFY PHASE 11.2 MANIFEST
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3E — VERIFY PHASE 11.2 PASS MANIFEST")
print("=" * 120)


manifest_11_2 = pd.read_csv(
    PHASE_11_2_MANIFEST_FILE
)


def get_manifest_value(
    dataframe,
    metric
):

    row = dataframe.loc[
        dataframe[
            "Metric"
        ]
        ==
        metric
    ]


    if len(
        row
    ) != 1:

        raise RuntimeError(
            f"Manifest metric missing/duplicated: {metric}"
        )


    return row[
        "Value"
    ].iloc[0]


phase_11_2_pass = (

    str(
        get_manifest_value(
            manifest_11_2,
            "Phase_11_2_Pass"
        )
    )

    .strip()

    .lower()

    ==
    "true"
)


if not phase_11_2_pass:

    raise RuntimeError(
        "Phase 11.2 did not pass."
    )


rejected_family_count = int(

    float(
        get_manifest_value(
            manifest_11_2,
            "Rejected_Families"
        )
    )
)


if rejected_family_count != 0:

    raise RuntimeError(
        f"Phase 11.2 has {rejected_family_count} rejected families."
    )


print(
    "✓ Phase 11.2 PASS confirmed"
)

print(
    "✓ No rejected attack families"
)


# ==================================================================================================
# 11.3F — EXTRACT FROZEN SPECIFICATION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3F — LOAD FROZEN EPISODE CONTRACT")
print("=" * 120)


DATASET_VERSION = (
    CONFIG[
        "dataset_identity"
    ][
        "version"
    ]
)


DATASET_SHORT_NAME = (
    CONFIG[
        "dataset_identity"
    ][
        "short_name"
    ]
)


MASTER_SEED = int(
    CONFIG[
        "dataset_identity"
    ][
        "master_seed"
    ]
)


TOTAL_EPISODES = int(
    CONFIG[
        "episode_design"
    ][
        "total_episodes"
    ]
)


ATTACK_EPISODES = int(
    CONFIG[
        "episode_design"
    ][
        "attack_episodes"
    ]
)


BENIGN_EPISODES = int(
    CONFIG[
        "episode_design"
    ][
        "benign_only_episodes"
    ]
)


ATTACK_FAMILIES = list(
    CONFIG[
        "episode_design"
    ][
        "attack_families"
    ]
)


SPLIT_COUNTS_PER_CLASS = {

    key:
        int(
            value
        )

    for key, value
    in CONFIG[
        "episode_design"
    ][
        "split_counts_per_class"
    ].items()
}


EXPECTED_SPLIT_TOTALS = {

    key:
        int(
            value
        )

    for key, value
    in CONFIG[
        "episode_design"
    ][
        "expected_split_totals"
    ].items()
}


PRECURSOR_REGIMES = (
    CONFIG[
        "precursor_regimes"
    ]
)


PRECURSOR_SPLIT_CONTRACT = (
    CONFIG[
        "precursor_split_contract"
    ]
)


TEMPORAL_CONFIG = (
    CONFIG[
        "temporal_design"
    ]
)


CALIBRATION_STRATEGY = (
    CONFIG[
        "calibration_strategy"
    ]
)


EPISODE_MIN_LENGTH = int(
    TEMPORAL_CONFIG[
        "episode_length_min_minutes"
    ]
)


EPISODE_MAX_LENGTH = int(
    TEMPORAL_CONFIG[
        "episode_length_max_minutes"
    ]
)


ATTACK_ONSET_MIN = int(
    TEMPORAL_CONFIG[
        "attack_onset_minute_min"
    ]
)


ATTACK_ONSET_MAX = int(
    TEMPORAL_CONFIG[
        "attack_onset_minute_max"
    ]
)


ATTACK_DURATION_MIN = int(
    TEMPORAL_CONFIG[
        "attack_duration_min_minutes"
    ]
)


ATTACK_DURATION_MAX = int(
    TEMPORAL_CONFIG[
        "attack_duration_max_minutes"
    ]
)


RECOVERY_DURATION_MIN = int(
    TEMPORAL_CONFIG[
        "recovery_duration_min_minutes"
    ]
)


RECOVERY_DURATION_MAX = int(
    TEMPORAL_CONFIG[
        "recovery_duration_max_minutes"
    ]
)


MINIMUM_PREONSET_HISTORY = int(
    TEMPORAL_CONFIG[
        "minimum_pre_onset_history_minutes"
    ]
)


TIMESTAMP_FREQUENCY_SECONDS = int(
    TEMPORAL_CONFIG[
        "timestamp_frequency_seconds"
    ]
)


EPISODE_SAFETY_GAP_MINUTES = int(
    TEMPORAL_CONFIG[
        "episode_safety_gap_minutes"
    ]
)


TIMELINE_START = pd.Timestamp(
    TEMPORAL_CONFIG[
        "episode_timestamp_start"
    ]
)


if TIMELINE_START.tzinfo is None:

    TIMELINE_START = TIMELINE_START.tz_localize(
        "UTC"
    )


else:

    TIMELINE_START = TIMELINE_START.tz_convert(
        "UTC"
    )


LOAD_SIGMA = float(
    CALIBRATION_STRATEGY[
        "episode_load_multiplier_sigma"
    ]
)


LOAD_CLIP_MIN = float(
    CALIBRATION_STRATEGY[
        "episode_load_multiplier_clip"
    ][0]
)


LOAD_CLIP_MAX = float(
    CALIBRATION_STRATEGY[
        "episode_load_multiplier_clip"
    ][1]
)


EFFECT_MULT_MIN = float(
    CALIBRATION_STRATEGY[
        "family_effect_strength_multiplier_range"
    ][0]
)


EFFECT_MULT_MAX = float(
    CALIBRATION_STRATEGY[
        "family_effect_strength_multiplier_range"
    ][1]
)


CALIBRATION_POOL_FOR_SPLIT = {

    "TRAIN":
        "CAL_TRAIN",

    "VALIDATION":
        "CAL_VALIDATION",

    "TEST":
        "CAL_TEST"
}


print(
    "Total episodes:",
    TOTAL_EPISODES
)

print(
    "Attack episodes:",
    ATTACK_EPISODES
)

print(
    "Benign-only episodes:",
    BENIGN_EPISODES
)

print(
    "Families:",
    ATTACK_FAMILIES
)


# ==================================================================================================
# 11.3G — LOAD FROZEN SOURCE POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3G — LOAD SOURCE-CONDITIONED FAMILY POLICY")
print("=" * 120)


family_generation_policy = pd.read_csv(
    FAMILY_GENERATION_POLICY_FILE
)


source_effects = pd.read_csv(
    SOURCE_EFFECT_FILE
)


safe_block_index = pd.read_csv(
    SAFE_BLOCK_INDEX_FILE
)


required_block_columns = [

    "Calibration_Source",

    "Calibration_Group_ID",

    "Calibration_Pool",

    "Safe_Benign_Block_ID",

    "Block_Length"
]


missing_block_columns = [

    c

    for c in required_block_columns

    if c not in safe_block_index.columns
]


if missing_block_columns:

    raise RuntimeError(
        f"Safe block index missing columns: {missing_block_columns}"
    )


print(
    "Safe block count:",
    len(
        safe_block_index
    )
)


print(
    "\nAvailable safe blocks:"
)


print(

    safe_block_index

    .groupby(
        [
            "Calibration_Source",
            "Calibration_Pool"
        ]
    )

    .size()

    .rename(
        "Safe_Block_Count"
    )

    .reset_index()

    .to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3H — VERIFY SOURCE-SPECIFIC EFFECT COVERAGE
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3H — VERIFY SOURCE-SPECIFIC EFFECT COVERAGE")
print("=" * 120)


EXPECTED_EFFECT_FEATURE_COUNT = 9


effect_coverage_rows = []


for family in ATTACK_FAMILIES:

    policy_row = family_generation_policy.loc[
        family_generation_policy[
            "Synthetic_Family"
        ]
        ==
        family
    ]


    if len(
        policy_row
    ) != 1:

        raise RuntimeError(
            f"Missing generation policy for {family}"
        )


    policy_row = policy_row.iloc[
        0
    ]


    for source, weight_col in [

        (
            "CIC_IDS2017",
            "CIC_Source_Weight"
        ),

        (
            "UNSW_NB15",
            "UNSW_Source_Weight"
        )
    ]:

        weight = float(
            policy_row[
                weight_col
            ]
        )


        if weight <= 0:

            continue


        effect_rows = source_effects.loc[
            (
                source_effects[
                    "Synthetic_Family"
                ]
                ==
                family
            )

            &

            (
                source_effects[
                    "Calibration_Source"
                ]
                ==
                source
            )
        ]


        feature_count = int(
            effect_rows[
                "Feature"
            ].nunique()
        )


        passed = (
            feature_count
            ==
            EXPECTED_EFFECT_FEATURE_COUNT
        )


        effect_coverage_rows.append(
            {
                "Synthetic_Family":
                    family,

                "Calibration_Source":
                    source,

                "Source_Weight":
                    weight,

                "Effect_Feature_Count":
                    feature_count,

                "Passed":
                    passed
            }
        )


effect_coverage = pd.DataFrame(
    effect_coverage_rows
)


print(
    effect_coverage.to_string(
        index=False
    )
)


if not effect_coverage[
    "Passed"
].all():

    raise RuntimeError(
        "Missing source-specific family effect vectors."
    )


# ==================================================================================================
# 11.3I — DETERMINISTIC HELPERS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3I — INITIALIZE DETERMINISTIC MANIFEST HELPERS")
print("=" * 120)


def stable_seed64(
    text
):

    digest = hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).digest()


    value = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False
    )


    # keep inside signed 63-bit range for portability

    return int(
        value
        %
        (
            2**63
            -
            1
        )
    )


def stable_sha256_text(
    text
):

    return hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def largest_remainder_allocation(
    total,
    weights
):

    positive = {

        key:
            float(
                value
            )

        for key, value
        in weights.items()

        if float(
            value
        )
        >
        0
    }


    if not positive:

        raise ValueError(
            "No positive allocation weights."
        )


    weight_sum = sum(
        positive.values()
    )


    normalized = {

        key:
            value
            /
            weight_sum

        for key, value
        in positive.items()
    }


    raw = {

        key:
            total
            *
            value

        for key, value
        in normalized.items()
    }


    allocation = {

        key:
            int(
                math.floor(
                    value
                )
            )

        for key, value
        in raw.items()
    }


    remainder = (
        total
        -
        sum(
            allocation.values()
        )
    )


    ranking = sorted(

        positive.keys(),

        key=lambda key: (

            -(
                raw[
                    key
                ]

                -

                math.floor(
                    raw[
                        key
                    ]
                )
            ),

            key
        )
    )


    for key in ranking[
        :remainder
    ]:

        allocation[
            key
        ] += 1


    if sum(
        allocation.values()
    ) != total:

        raise RuntimeError(
            "Largest-remainder allocation failed."
        )


    return allocation


def allocate_regimes_across_sources(
    regime_counts,
    source_counts,
    enforce_train_coverage=False
):

    sources = [

        source

        for source, count
        in source_counts.items()

        if count > 0
    ]


    regimes = list(
        regime_counts.keys()
    )


    total = sum(
        regime_counts.values()
    )


    if sum(
        source_counts.values()
    ) != total:

        raise RuntimeError(
            "Source/regime total mismatch."
        )


    # Single-source family

    if len(
        sources
    ) == 1:

        source = sources[0]


        return [

            {
                "Calibration_Source":
                    source,

                "Precursor_Regime":
                    regime,

                "Episode_Count":
                    int(
                        count
                    )
            }

            for regime, count
            in regime_counts.items()
        ]


    if len(
        sources
    ) != 2:

        raise RuntimeError(
            "Current Phase 11.3 supports one or two calibration sources."
        )


    source_a = sources[0]

    source_b = sources[1]


    quota_a = int(
        source_counts[
            source_a
        ]
    )


    quota_b = int(
        source_counts[
            source_b
        ]
    )


    # Target allocation of each regime to source A.

    raw_target_a = {

        regime:

            regime_counts[
                regime
            ]

            *
            quota_a

            /
            total

        for regime in regimes
    }


    lower = {

        regime:
            0

        for regime in regimes
    }


    upper = {

        regime:
            int(
                regime_counts[
                    regime
                ]
            )

        for regime in regimes
    }


    # In TRAIN, when mathematically possible, force BOTH sources
    # to see every precursor regime.

    can_force_two_source_train_coverage = (

        enforce_train_coverage

        and

        quota_a
        >=
        len(
            regimes
        )

        and

        quota_b
        >=
        len(
            regimes
        )

        and

        all(
            regime_counts[
                regime
            ]
            >=
            2

            for regime in regimes
        )
    )


    if can_force_two_source_train_coverage:

        for regime in regimes:

            lower[
                regime
            ] = 1

            upper[
                regime
            ] = (

                regime_counts[
                    regime
                ]

                -
                1
            )


    allocation_a = {}


    for regime in regimes:

        initial = int(
            math.floor(
                raw_target_a[
                    regime
                ]
            )
        )


        initial = max(
            lower[
                regime
            ],
            initial
        )


        initial = min(
            upper[
                regime
            ],
            initial
        )


        allocation_a[
            regime
        ] = initial


    current = sum(
        allocation_a.values()
    )


    # Add units

    while current < quota_a:

        candidates = [

            regime

            for regime in regimes

            if allocation_a[
                regime
            ]
            <
            upper[
                regime
            ]
        ]


        if not candidates:

            raise RuntimeError(
                "Cannot satisfy source-regime allocation upper bounds."
            )


        best = sorted(

            candidates,

            key=lambda regime: (

                -(
                    raw_target_a[
                        regime
                    ]

                    -

                    allocation_a[
                        regime
                    ]
                ),

                regime
            )

        )[0]


        allocation_a[
            best
        ] += 1


        current += 1


    # Remove units

    while current > quota_a:

        candidates = [

            regime

            for regime in regimes

            if allocation_a[
                regime
            ]
            >
            lower[
                regime
            ]
        ]


        if not candidates:

            raise RuntimeError(
                "Cannot satisfy source-regime allocation lower bounds."
            )


        best = sorted(

            candidates,

            key=lambda regime: (

                -(
                    allocation_a[
                        regime
                    ]

                    -

                    raw_target_a[
                        regime
                    ]
                ),

                regime
            )

        )[0]


        allocation_a[
            best
        ] -= 1


        current -= 1


    output = []


    for regime in regimes:

        count_a = int(
            allocation_a[
                regime
            ]
        )


        count_b = int(

            regime_counts[
                regime
            ]

            -
            count_a
        )


        if count_a > 0:

            output.append(
                {
                    "Calibration_Source":
                        source_a,

                    "Precursor_Regime":
                        regime,

                    "Episode_Count":
                        count_a
                }
            )


        if count_b > 0:

            output.append(
                {
                    "Calibration_Source":
                        source_b,

                    "Precursor_Regime":
                        regime,

                    "Episode_Count":
                        count_b
                }
            )


    if (
        sum(
            row[
                "Episode_Count"
            ]
            for row in output
            if row[
                "Calibration_Source"
            ]
            ==
            source_a
        )
        !=
        quota_a
    ):

        raise RuntimeError(
            "Source A regime allocation mismatch."
        )


    if (
        sum(
            row[
                "Episode_Count"
            ]
            for row in output
            if row[
                "Calibration_Source"
            ]
            ==
            source_b
        )
        !=
        quota_b
    ):

        raise RuntimeError(
            "Source B regime allocation mismatch."
        )


    return output


def balanced_block_sequence(
    pool_ids,
    required_count,
    rng
):

    pool_ids = list(
        pool_ids
    )


    if len(
        pool_ids
    ) == 0:

        raise RuntimeError(
            "No safe calibration blocks available."
        )


    sequence = []


    while len(
        sequence
    ) < required_count:

        permutation = list(
            rng.permutation(
                pool_ids
            )
        )


        # Avoid identical immediate boundary repetition where possible.

        if (
            len(
                sequence
            )
            >
            0

            and

            len(
                permutation
            )
            >
            1

            and

            permutation[0]
            ==
            sequence[-1]
        ):

            permutation[0], permutation[1] = (
                permutation[1],
                permutation[0]
            )


        remaining = (

            required_count

            -
            len(
                sequence
            )
        )


        sequence.extend(
            permutation[
                :remaining
            ]
        )


    return sequence


# ==================================================================================================
# 11.3J — BUILD EXACT ATTACK SOURCE QUOTAS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3J — BUILD EXACT ATTACK FAMILY × SPLIT × SOURCE QUOTAS")
print("=" * 120)


source_quota_rows = []


for family in ATTACK_FAMILIES:

    policy_row = family_generation_policy.loc[
        family_generation_policy[
            "Synthetic_Family"
        ]
        ==
        family
    ].iloc[0]


    weights = {

        "CIC_IDS2017":
            float(
                policy_row[
                    "CIC_Source_Weight"
                ]
            ),

        "UNSW_NB15":
            float(
                policy_row[
                    "UNSW_Source_Weight"
                ]
            )
    }


    for split, count in SPLIT_COUNTS_PER_CLASS.items():

        allocation = largest_remainder_allocation(
            total=count,
            weights=weights
        )


        for source in [
            "CIC_IDS2017",
            "UNSW_NB15"
        ]:

            source_quota_rows.append(
                {
                    "Episode_Type":
                        "ATTACK",

                    "Synthetic_Family":
                        family,

                    "Split":
                        split,

                    "Calibration_Source":
                        source,

                    "Episode_Count":
                        int(
                            allocation.get(
                                source,
                                0
                            )
                        ),

                    "Frozen_Source_Weight":
                        float(
                            weights[
                                source
                            ]
                        )
                }
            )


attack_source_quota_contract = pd.DataFrame(
    source_quota_rows
)


print(
    attack_source_quota_contract.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3K — BENIGN SOURCE QUOTAS
#
# Exact 50 / 50 CIC / UNSW within every split.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3K — BUILD BENIGN SOURCE QUOTAS")
print("=" * 120)


benign_source_quota_rows = []


for split, count in SPLIT_COUNTS_PER_CLASS.items():

    if count % 2 != 0:

        raise RuntimeError(
            f"Benign count for {split} is not even; exact 50/50 source allocation impossible."
        )


    half = (
        count
        //
        2
    )


    for source in [
        "CIC_IDS2017",
        "UNSW_NB15"
    ]:

        benign_source_quota_rows.append(
            {
                "Episode_Type":
                    "BENIGN_ONLY",

                "Synthetic_Family":
                    "NONE",

                "Split":
                    split,

                "Calibration_Source":
                    source,

                "Episode_Count":
                    half,

                "Frozen_Source_Weight":
                    0.50
            }
        )


benign_source_quota_contract = pd.DataFrame(
    benign_source_quota_rows
)


print(
    benign_source_quota_contract.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3L — BUILD EXACT ATTACK SOURCE × PRECURSOR CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3L — BUILD SOURCE × PRECURSOR REGIME CONTRACT")
print("=" * 120)


source_regime_contract_rows = []


for family in ATTACK_FAMILIES:

    for split, split_count in SPLIT_COUNTS_PER_CLASS.items():

        family_source_rows = attack_source_quota_contract.loc[
            (
                attack_source_quota_contract[
                    "Synthetic_Family"
                ]
                ==
                family
            )

            &

            (
                attack_source_quota_contract[
                    "Split"
                ]
                ==
                split
            )
        ]


        source_counts = {

            row[
                "Calibration_Source"
            ]:
                int(
                    row[
                        "Episode_Count"
                    ]
                )

            for _, row
            in family_source_rows.iterrows()

            if int(
                row[
                    "Episode_Count"
                ]
            )
            >
            0
        }


        regime_counts = {

            regime:
                int(
                    PRECURSOR_SPLIT_CONTRACT[
                        regime
                    ][
                        split
                    ]
                )

            for regime
            in PRECURSOR_REGIMES.keys()
        }


        if sum(
            regime_counts.values()
        ) != split_count:

            raise RuntimeError(
                f"{family} {split}: precursor contract does not sum to {split_count}."
            )


        matrix_rows = allocate_regimes_across_sources(

            regime_counts=regime_counts,

            source_counts=source_counts,

            enforce_train_coverage=(
                split
                ==
                "TRAIN"
            )
        )


        for row in matrix_rows:

            source_regime_contract_rows.append(
                {
                    "Synthetic_Family":
                        family,

                    "Split":
                        split,

                    "Calibration_Source":
                        row[
                            "Calibration_Source"
                        ],

                    "Precursor_Regime":
                        row[
                            "Precursor_Regime"
                        ],

                    "Episode_Count":
                        int(
                            row[
                                "Episode_Count"
                            ]
                        )
                }
            )


source_regime_contract = pd.DataFrame(
    source_regime_contract_rows
)


print(
    source_regime_contract.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3M — BUILD ATTACK EPISODE SKELETON
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3M — BUILD 1,000 ATTACK EPISODE SKELETONS")
print("=" * 120)


attack_skeleton_rows = []


for _, contract_row in source_regime_contract.iterrows():

    family = contract_row[
        "Synthetic_Family"
    ]


    split = contract_row[
        "Split"
    ]


    source = contract_row[
        "Calibration_Source"
    ]


    regime = contract_row[
        "Precursor_Regime"
    ]


    count = int(
        contract_row[
            "Episode_Count"
        ]
    )


    family_policy_row = family_generation_policy.loc[
        family_generation_policy[
            "Synthetic_Family"
        ]
        ==
        family
    ].iloc[0]


    source_weight = (

        float(
            family_policy_row[
                "CIC_Source_Weight"
            ]
        )

        if source
        ==
        "CIC_IDS2017"

        else

        float(
            family_policy_row[
                "UNSW_Source_Weight"
            ]
        )
    )


    for _ in range(
        count
    ):

        attack_skeleton_rows.append(
            {
                "Episode_Type":
                    "ATTACK",

                "Split":
                    split,

                "Attack_Family":
                    family,

                "Calibration_Source":
                    source,

                "Calibration_Pool":
                    CALIBRATION_POOL_FOR_SPLIT[
                        split
                    ],

                "Source_Weight_Used":
                    source_weight,

                "Precursor_Regime":
                    regime
            }
        )


attack_skeleton = pd.DataFrame(
    attack_skeleton_rows
)


if len(
    attack_skeleton
) != ATTACK_EPISODES:

    raise RuntimeError(
        f"Expected {ATTACK_EPISODES} attack skeletons; created {len(attack_skeleton)}."
    )


# Shuffle ATTACK skeleton deterministically so combinations are not ordered.

attack_rng = np.random.default_rng(
    stable_seed64(
        f"{MASTER_SEED}|ATTACK_SKELETON"
    )
)


attack_skeleton = (

    attack_skeleton.iloc[
        attack_rng.permutation(
            len(
                attack_skeleton
            )
        )
    ]

    .reset_index(
        drop=True
    )
)


print(
    "Attack skeleton rows:",
    len(
        attack_skeleton
    )
)


# ==================================================================================================
# 11.3N — BUILD 200 BENIGN-ONLY EPISODE SKELETONS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3N — BUILD 200 BENIGN-ONLY EPISODE SKELETONS")
print("=" * 120)


benign_skeleton_rows = []


for _, row in benign_source_quota_contract.iterrows():

    split = row[
        "Split"
    ]


    source = row[
        "Calibration_Source"
    ]


    count = int(
        row[
            "Episode_Count"
        ]
    )


    for _ in range(
        count
    ):

        benign_skeleton_rows.append(
            {
                "Episode_Type":
                    "BENIGN_ONLY",

                "Split":
                    split,

                "Attack_Family":
                    "NONE",

                "Calibration_Source":
                    source,

                "Calibration_Pool":
                    CALIBRATION_POOL_FOR_SPLIT[
                        split
                    ],

                "Source_Weight_Used":
                    0.50,

                "Precursor_Regime":
                    "NONE"
            }
        )


benign_skeleton = pd.DataFrame(
    benign_skeleton_rows
)


if len(
    benign_skeleton
) != BENIGN_EPISODES:

    raise RuntimeError(
        f"Expected {BENIGN_EPISODES} benign skeletons; created {len(benign_skeleton)}."
    )


benign_rng = np.random.default_rng(
    stable_seed64(
        f"{MASTER_SEED}|BENIGN_SKELETON"
    )
)


benign_skeleton = (

    benign_skeleton.iloc[
        benign_rng.permutation(
            len(
                benign_skeleton
            )
        )
    ]

    .reset_index(
        drop=True
    )
)


print(
    "Benign skeleton rows:",
    len(
        benign_skeleton
    )
)


# ==================================================================================================
# 11.3O — COMBINE + RANDOMIZE BEFORE ASSIGNING IDs
#
# Episode_ID therefore carries no systematic family/split ordering.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3O — COMBINE AND RANDOMIZE EPISODE SKELETON")
print("=" * 120)


episode_skeleton = pd.concat(

    [
        attack_skeleton,
        benign_skeleton
    ],

    ignore_index=True
)


if len(
    episode_skeleton
) != TOTAL_EPISODES:

    raise RuntimeError(
        f"Expected {TOTAL_EPISODES} episodes; got {len(episode_skeleton)}."
    )


global_rng = np.random.default_rng(
    stable_seed64(
        f"{MASTER_SEED}|GLOBAL_EPISODE_ORDER"
    )
)


episode_skeleton = (

    episode_skeleton.iloc[
        global_rng.permutation(
            len(
                episode_skeleton
            )
        )
    ]

    .reset_index(
        drop=True
    )
)


episode_skeleton[
    "Episode_ID"
] = [

    f"SYN_E{i:06d}"

    for i in range(
        1,
        TOTAL_EPISODES + 1
    )
]


episode_skeleton[
    "Dataset_Version"
] = DATASET_VERSION


episode_skeleton[
    "Calibration_Policy_SHA256"
] = CURRENT_POLICY_SHA256


print(
    "Combined episode skeleton:",
    len(
        episode_skeleton
    )
)


# ==================================================================================================
# 11.3P — GENERATOR SEEDS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3P — ASSIGN UNIQUE EPISODE GENERATOR SEEDS")
print("=" * 120)


episode_skeleton[
    "Generator_Seed"
] = episode_skeleton[
    "Episode_ID"
].apply(

    lambda episode_id:

    stable_seed64(
        f"{MASTER_SEED}|GENERATOR|{episode_id}"
    )
)


if (
    episode_skeleton[
        "Generator_Seed"
    ].nunique()
    !=
    TOTAL_EPISODES
):

    raise RuntimeError(
        "Generator seed collision detected."
    )


print(
    "Unique generator seeds:",
    episode_skeleton[
        "Generator_Seed"
    ].nunique()
)


# ==================================================================================================
# 11.3Q — PREPARE SAFE BLOCK POOLS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3Q — PREPARE SPLIT-SAFE REAL BENIGN BLOCK POOLS")
print("=" * 120)


block_pool_lookup = {}


for (
    source,
    pool
), group in safe_block_index.groupby(

    [
        "Calibration_Source",
        "Calibration_Pool"
    ],

    observed=True
):

    block_pool_lookup[
        (
            source,
            pool
        )
    ] = sorted(
        group[
            "Safe_Benign_Block_ID"
        ].tolist()
    )


block_to_source = dict(
    zip(
        safe_block_index[
            "Safe_Benign_Block_ID"
        ],
        safe_block_index[
            "Calibration_Source"
        ]
    )
)


block_to_pool = dict(
    zip(
        safe_block_index[
            "Safe_Benign_Block_ID"
        ],
        safe_block_index[
            "Calibration_Pool"
        ]
    )
)


block_to_group = dict(
    zip(
        safe_block_index[
            "Safe_Benign_Block_ID"
        ],
        safe_block_index[
            "Calibration_Group_ID"
        ]
    )
)


for key, values in sorted(
    block_pool_lookup.items()
):

    print(
        key,
        "→",
        len(
            values
        ),
        "blocks"
    )


# ==================================================================================================
# 11.3R — GENERATE EPISODE-LEVEL PARAMETERS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3R — GENERATE EPISODE-LEVEL PARAMETERS")
print("=" * 120)


used_block_sequence_hashes = set()

episode_rows = []

block_usage_rows = []


for row_number, skeleton_row in episode_skeleton.iterrows():

    episode_id = skeleton_row[
        "Episode_ID"
    ]


    generator_seed = int(
        skeleton_row[
            "Generator_Seed"
        ]
    )


    rng = np.random.default_rng(
        generator_seed
    )


    split = skeleton_row[
        "Split"
    ]


    episode_type = skeleton_row[
        "Episode_Type"
    ]


    family = skeleton_row[
        "Attack_Family"
    ]


    source = skeleton_row[
        "Calibration_Source"
    ]


    pool = skeleton_row[
        "Calibration_Pool"
    ]


    regime = skeleton_row[
        "Precursor_Regime"
    ]


    # ----------------------------------------------------------------------------------------------
    # ATTACK TEMPORAL PARAMETERS
    # ----------------------------------------------------------------------------------------------

    if episode_type == "ATTACK":

        attack_onset_minute = int(
            rng.integers(
                ATTACK_ONSET_MIN,
                ATTACK_ONSET_MAX + 1
            )
        )


        attack_duration_minutes = int(
            rng.integers(
                ATTACK_DURATION_MIN,
                ATTACK_DURATION_MAX + 1
            )
        )


        recovery_duration_minutes = int(
            rng.integers(
                RECOVERY_DURATION_MIN,
                RECOVERY_DURATION_MAX + 1
            )
        )


        precursor_spec = PRECURSOR_REGIMES[
            regime
        ]


        precursor_min = int(
            precursor_spec[
                "length_min"
            ]
        )


        precursor_max = int(
            precursor_spec[
                "length_max"
            ]
        )


        if precursor_max == 0:

            precursor_length_minutes = 0

        else:

            precursor_length_minutes = int(
                rng.integers(
                    precursor_min,
                    precursor_max + 1
                )
            )


        precursor_max_alpha = float(
            precursor_spec[
                "max_preonset_alpha"
            ]
        )


        precursor_temporal_profile = (
            precursor_spec[
                "temporal_profile"
            ]
        )


        minimum_required_length = max(

            EPISODE_MIN_LENGTH,

            attack_onset_minute
            +
            attack_duration_minutes
            +
            recovery_duration_minutes
        )


        if minimum_required_length > EPISODE_MAX_LENGTH:

            raise RuntimeError(
                f"{episode_id}: attack/recovery timeline cannot fit."
            )


        episode_length_minutes = int(
            rng.integers(
                minimum_required_length,
                EPISODE_MAX_LENGTH + 1
            )
        )


        family_effect_strength_multiplier = float(
            rng.uniform(
                EFFECT_MULT_MIN,
                EFFECT_MULT_MAX
            )
        )


        attack_end_minute = (

            attack_onset_minute
            +
            attack_duration_minutes
            -
            1
        )


        recovery_start_minute = (

            attack_onset_minute
            +
            attack_duration_minutes
        )


        recovery_end_minute = (

            recovery_start_minute
            +
            recovery_duration_minutes
            -
            1
        )


        if precursor_length_minutes > 0:

            precursor_start_minute = (

                attack_onset_minute
                -
                precursor_length_minutes
            )


            precursor_end_minute = (

                attack_onset_minute
                -
                1
            )


        else:

            precursor_start_minute = np.nan

            precursor_end_minute = np.nan


    # ----------------------------------------------------------------------------------------------
    # BENIGN-ONLY EPISODE
    # ----------------------------------------------------------------------------------------------

    else:

        episode_length_minutes = int(
            rng.integers(
                EPISODE_MIN_LENGTH,
                EPISODE_MAX_LENGTH + 1
            )
        )


        attack_onset_minute = np.nan

        attack_duration_minutes = 0

        attack_end_minute = np.nan

        recovery_duration_minutes = 0

        recovery_start_minute = np.nan

        recovery_end_minute = np.nan


        precursor_length_minutes = 0

        precursor_max_alpha = 0.0

        precursor_temporal_profile = "NONE"

        precursor_start_minute = np.nan

        precursor_end_minute = np.nan


        family_effect_strength_multiplier = np.nan


    # ----------------------------------------------------------------------------------------------
    # EPISODE BACKGROUND LOAD
    # ----------------------------------------------------------------------------------------------

    raw_load_multiplier = float(
        np.exp(
            rng.normal(
                loc=0.0,
                scale=LOAD_SIGMA
            )
        )
    )


    episode_load_multiplier = float(
        np.clip(
            raw_load_multiplier,
            LOAD_CLIP_MIN,
            LOAD_CLIP_MAX
        )
    )


    # ----------------------------------------------------------------------------------------------
    # NUMBER OF REAL 5-MINUTE BASELINE BLOCKS REQUIRED
    # ----------------------------------------------------------------------------------------------

    baseline_block_count = int(
        math.ceil(
            episode_length_minutes
            /
            5
        )
    )


    final_block_minutes_used = int(

        episode_length_minutes

        -

        (
            baseline_block_count
            -
            1
        )
        *
        5
    )


    pool_key = (
        source,
        pool
    )


    if pool_key not in block_pool_lookup:

        raise RuntimeError(
            f"{episode_id}: no calibration block pool for {pool_key}."
        )


    available_blocks = block_pool_lookup[
        pool_key
    ]


    # ----------------------------------------------------------------------------------------------
    # GENERATE UNIQUE BASELINE BLOCK SEQUENCE
    # ----------------------------------------------------------------------------------------------

    sequence = None

    sequence_hash = None


    for attempt in range(
        100
    ):

        block_rng = np.random.default_rng(

            stable_seed64(
                f"{generator_seed}|BLOCK_SEQUENCE|{attempt}"
            )
        )


        candidate_sequence = balanced_block_sequence(

            pool_ids=available_blocks,

            required_count=baseline_block_count,

            rng=block_rng
        )


        candidate_json = json.dumps(
            candidate_sequence,
            separators=(
                ",",
                ":"
            )
        )


        candidate_hash = stable_sha256_text(
            candidate_json
        )


        if candidate_hash not in used_block_sequence_hashes:

            sequence = candidate_sequence

            sequence_hash = candidate_hash

            used_block_sequence_hashes.add(
                candidate_hash
            )

            break


    if sequence is None:

        raise RuntimeError(
            f"{episode_id}: unable to construct unique baseline block sequence."
        )


    unique_blocks_used = len(
        set(
            sequence
        )
    )


    unique_groups_used = len(

        set(

            block_to_group[
                block_id
            ]

            for block_id
            in sequence
        )
    )


    repetition_ratio = (

        1.0

        -

        unique_blocks_used
        /
        baseline_block_count
    )


    # ----------------------------------------------------------------------------------------------
    # RECORD BLOCK USAGE FOR LEAKAGE AUDIT
    # ----------------------------------------------------------------------------------------------

    for position, block_id in enumerate(
        sequence
    ):

        block_usage_rows.append(
            {
                "Episode_ID":
                    episode_id,

                "Split":
                    split,

                "Episode_Type":
                    episode_type,

                "Attack_Family":
                    family,

                "Calibration_Source":
                    source,

                "Calibration_Pool":
                    pool,

                "Sequence_Position":
                    position,

                "Safe_Benign_Block_ID":
                    block_id,

                "Calibration_Group_ID":
                    block_to_group[
                        block_id
                    ],

                "Block_Source":
                    block_to_source[
                        block_id
                    ],

                "Block_Pool":
                    block_to_pool[
                        block_id
                    ]
            }
        )


    episode_rows.append(
        {
            "Dataset_Version":
                DATASET_VERSION,

            "Episode_ID":
                episode_id,

            "Split":
                split,

            "Episode_Type":
                episode_type,

            "Attack_Family":
                family,

            "Calibration_Source":
                source,

            "Calibration_Pool":
                pool,

            "Source_Weight_Used":
                float(
                    skeleton_row[
                        "Source_Weight_Used"
                    ]
                ),

            "Precursor_Regime":
                regime,

            "Precursor_Temporal_Profile":
                precursor_temporal_profile,

            "Precursor_Length_Minutes":
                precursor_length_minutes,

            "Precursor_Max_Alpha":
                precursor_max_alpha,

            "Episode_Length_Minutes":
                episode_length_minutes,

            "Attack_Onset_Minute":
                attack_onset_minute,

            "Attack_Duration_Minutes":
                attack_duration_minutes,

            "Attack_End_Minute":
                attack_end_minute,

            "Recovery_Duration_Minutes":
                recovery_duration_minutes,

            "Recovery_Start_Minute":
                recovery_start_minute,

            "Recovery_End_Minute":
                recovery_end_minute,

            "Precursor_Start_Minute":
                precursor_start_minute,

            "Precursor_End_Minute":
                precursor_end_minute,

            "Episode_Load_Multiplier":
                episode_load_multiplier,

            "Family_Effect_Strength_Multiplier":
                family_effect_strength_multiplier,

            "Baseline_Block_Count":
                baseline_block_count,

            "Unique_Baseline_Blocks_Used":
                unique_blocks_used,

            "Unique_Calibration_Groups_Used":
                unique_groups_used,

            "Baseline_Block_Repetition_Ratio":
                repetition_ratio,

            "Final_Block_Minutes_Used":
                final_block_minutes_used,

            "Baseline_Block_Sequence":
                json.dumps(
                    sequence,
                    separators=(
                        ",",
                        ":"
                    )
                ),

            "Baseline_Block_Sequence_SHA256":
                sequence_hash,

            "Generator_Seed":
                generator_seed,

            "Calibration_Policy_SHA256":
                CURRENT_POLICY_SHA256
        }
    )


episode_manifest = pd.DataFrame(
    episode_rows
)


block_usage = pd.DataFrame(
    block_usage_rows
)


print(
    "Episode manifest rows:",
    len(
        episode_manifest
    )
)

print(
    "Block usage rows:",
    len(
        block_usage
    )
)


# ==================================================================================================
# 11.3S — ASSIGN GLOBAL TIMELINE ORDER
#
# Timeline order is independent of split/family.
#
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3S — ASSIGN DETERMINISTIC GLOBAL TIMELINE ORDER")
print("=" * 120)


timeline_rng = np.random.default_rng(

    stable_seed64(
        f"{MASTER_SEED}|TIMELINE_ORDER"
    )
)


timeline_permutation = timeline_rng.permutation(
    TOTAL_EPISODES
)


timeline_order_by_index = np.empty(
    TOTAL_EPISODES,
    dtype=int
)


timeline_order_by_index[
    timeline_permutation
] = np.arange(
    1,
    TOTAL_EPISODES + 1
)


episode_manifest[
    "Timeline_Order"
] = timeline_order_by_index


# ==================================================================================================
# 11.3T — ASSIGN NON-OVERLAPPING UTC EPISODE TIMESTAMPS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3T — ASSIGN NON-OVERLAPPING UTC TIMESTAMPS")
print("=" * 120)


episode_manifest[
    "Episode_Start_UTC"
] = pd.NaT


episode_manifest[
    "Episode_End_UTC"
] = pd.NaT


episode_manifest[
    "Attack_Onset_UTC"
] = pd.NaT


episode_manifest[
    "Attack_End_UTC"
] = pd.NaT


episode_manifest[
    "Precursor_Start_UTC"
] = pd.NaT


episode_manifest[
    "Precursor_End_UTC"
] = pd.NaT


episode_manifest[
    "Recovery_Start_UTC"
] = pd.NaT


episode_manifest[
    "Recovery_End_UTC"
] = pd.NaT


cursor = TIMELINE_START


timeline_sorted_index = (

    episode_manifest

    .sort_values(
        "Timeline_Order"
    )

    .index
)


for idx in timeline_sorted_index:

    length = int(
        episode_manifest.loc[
            idx,
            "Episode_Length_Minutes"
        ]
    )


    start = cursor


    end = (

        start

        +

        pd.Timedelta(
            minutes=(
                length
                -
                1
            )
        )
    )


    episode_manifest.at[
        idx,
        "Episode_Start_UTC"
    ] = start


    episode_manifest.at[
        idx,
        "Episode_End_UTC"
    ] = end


    if (
        episode_manifest.loc[
            idx,
            "Episode_Type"
        ]
        ==
        "ATTACK"
    ):

        onset_minute = int(
            episode_manifest.loc[
                idx,
                "Attack_Onset_Minute"
            ]
        )


        attack_end_minute = int(
            episode_manifest.loc[
                idx,
                "Attack_End_Minute"
            ]
        )


        recovery_start = int(
            episode_manifest.loc[
                idx,
                "Recovery_Start_Minute"
            ]
        )


        recovery_end = int(
            episode_manifest.loc[
                idx,
                "Recovery_End_Minute"
            ]
        )


        episode_manifest.at[
            idx,
            "Attack_Onset_UTC"
        ] = (

            start

            +

            pd.Timedelta(
                minutes=onset_minute
            )
        )


        episode_manifest.at[
            idx,
            "Attack_End_UTC"
        ] = (

            start

            +

            pd.Timedelta(
                minutes=attack_end_minute
            )
        )


        episode_manifest.at[
            idx,
            "Recovery_Start_UTC"
        ] = (

            start

            +

            pd.Timedelta(
                minutes=recovery_start
            )
        )


        episode_manifest.at[
            idx,
            "Recovery_End_UTC"
        ] = (

            start

            +

            pd.Timedelta(
                minutes=recovery_end
            )
        )


        precursor_length = int(
            episode_manifest.loc[
                idx,
                "Precursor_Length_Minutes"
            ]
        )


        if precursor_length > 0:

            precursor_start_minute = int(
                episode_manifest.loc[
                    idx,
                    "Precursor_Start_Minute"
                ]
            )


            precursor_end_minute = int(
                episode_manifest.loc[
                    idx,
                    "Precursor_End_Minute"
                ]
            )


            episode_manifest.at[
                idx,
                "Precursor_Start_UTC"
            ] = (

                start

                +

                pd.Timedelta(
                    minutes=precursor_start_minute
                )
            )


            episode_manifest.at[
                idx,
                "Precursor_End_UTC"
            ] = (

                start

                +

                pd.Timedelta(
                    minutes=precursor_end_minute
                )
            )


    # IMPORTANT:
    #
    # If episode has N minute rows:
    #
    # last row = start + (N-1)
    #
    # next start = start + N + 60
    #
    # Therefore exactly 60 unused minutes exist between episodes.

    cursor = (

        start

        +

        pd.Timedelta(
            minutes=(
                length
                +
                EPISODE_SAFETY_GAP_MINUTES
            )
        )
    )


print(
    "Timeline start:",
    episode_manifest[
        "Episode_Start_UTC"
    ].min()
)

print(
    "Timeline end:",
    episode_manifest[
        "Episode_End_UTC"
    ].max()
)


# ==================================================================================================
# 11.3U — TIMELINE QUARTILE AUDIT
#
# Diagnostic guard against accidental chronological split clustering.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3U — TIMELINE DISTRIBUTION AUDIT")
print("=" * 120)


episode_manifest[
    "Timeline_Quartile"
] = pd.qcut(

    episode_manifest[
        "Timeline_Order"
    ],

    q=4,

    labels=[
        "Q1",
        "Q2",
        "Q3",
        "Q4"
    ]
)


timeline_split_audit = pd.crosstab(

    episode_manifest[
        "Timeline_Quartile"
    ],

    episode_manifest[
        "Split"
    ]
).reset_index()


timeline_family_audit = pd.crosstab(

    episode_manifest[
        "Timeline_Quartile"
    ],

    episode_manifest[
        "Attack_Family"
    ]
).reset_index()


print(
    "Split distribution across timeline quartiles:\n"
)


print(
    timeline_split_audit.to_string(
        index=False
    )
)


print(
    "\nAttack-family distribution across timeline quartiles:\n"
)


print(
    timeline_family_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3V — FAMILY EVALUATION POLICY
#
# This prevents the source environment from silently acting as an
# attack-family shortcut.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3V — FREEZE FAMILY EVALUATION SUBSETS")
print("=" * 120)


family_evaluation_policy = pd.DataFrame([

    {
        "Evaluation_ID":
            "ALL_5_SYNTHETIC",

        "Calibration_Source_Filter":
            "ALL",

        "Families":
            "DOS_DDOS | RECON_SCAN | EXPLOIT_APP | BRUTE_AUTH | FUZZING",

        "Purpose":
            "Overall synthetic closed-set family classification",

        "Interpretation":
            "SUPPORTING_RESULT_SOURCE_CONFOUND_POSSIBLE"
    },

    {
        "Evaluation_ID":
            "CIC_DOMAIN_4",

        "Calibration_Source_Filter":
            "CIC_IDS2017",

        "Families":
            "DOS_DDOS | RECON_SCAN | EXPLOIT_APP | BRUTE_AUTH",

        "Purpose":
            "Family classification within CIC-calibrated environment",

        "Interpretation":
            "WITHIN_SOURCE_RESULT"
    },

    {
        "Evaluation_ID":
            "UNSW_DOMAIN_4",

        "Calibration_Source_Filter":
            "UNSW_NB15",

        "Families":
            "DOS_DDOS | RECON_SCAN | EXPLOIT_APP | FUZZING",

        "Purpose":
            "Family classification within UNSW-calibrated environment",

        "Interpretation":
            "WITHIN_SOURCE_RESULT"
    },

    {
        "Evaluation_ID":
            "SHARED_CROSS_DOMAIN_3",

        "Calibration_Source_Filter":
            "ALL",

        "Families":
            "DOS_DDOS | RECON_SCAN | EXPLOIT_APP",

        "Purpose":
            "Shared-family evaluation across both calibration environments",

        "Interpretation":
            "CROSS_DOMAIN_SHARED_FAMILY_RESULT"
    }
])


print(
    family_evaluation_policy.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3W — BUILD EXPECTED SOURCE QUOTA TABLE
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3W — BUILD EXPECTED SOURCE QUOTA CONTRACT")
print("=" * 120)


expected_source_quota_contract = pd.concat(

    [
        attack_source_quota_contract,
        benign_source_quota_contract
    ],

    ignore_index=True
)


print(
    expected_source_quota_contract.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3X — ACTUAL SOURCE DISTRIBUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3X — ACTUAL EPISODE SOURCE DISTRIBUTION")
print("=" * 120)


actual_source_distribution = (

    episode_manifest

    .groupby(
        [
            "Episode_Type",
            "Attack_Family",
            "Split",
            "Calibration_Source"
        ],
        observed=True
    )

    .size()

    .rename(
        "Episode_Count"
    )

    .reset_index()
)


print(
    actual_source_distribution.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3Y — ACTUAL PRECURSOR DISTRIBUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3Y — ACTUAL PRECURSOR DISTRIBUTION")
print("=" * 120)


actual_precursor_distribution = (

    episode_manifest.loc[
        episode_manifest[
            "Episode_Type"
        ]
        ==
        "ATTACK"
    ]

    .groupby(
        [
            "Attack_Family",
            "Split",
            "Precursor_Regime"
        ],
        observed=True
    )

    .size()

    .rename(
        "Episode_Count"
    )

    .reset_index()
)


print(
    actual_precursor_distribution.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3Z — BLOCK USAGE SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3Z — CALIBRATION BLOCK USAGE SUMMARY")
print("=" * 120)


block_usage_summary = (

    block_usage

    .groupby(
        [
            "Split",
            "Calibration_Source",
            "Calibration_Pool"
        ],
        observed=True
    )

    .agg(

        Episode_Count=(
            "Episode_ID",
            "nunique"
        ),

        Block_Draws=(
            "Safe_Benign_Block_ID",
            "size"
        ),

        Unique_Blocks_Used=(
            "Safe_Benign_Block_ID",
            "nunique"
        ),

        Unique_Calibration_Groups_Used=(
            "Calibration_Group_ID",
            "nunique"
        )

    )

    .reset_index()
)


print(
    block_usage_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3AA — STRUCTURAL VALIDATION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AA — HARD MANIFEST VALIDATION")
print("=" * 120)


validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                details
        }
    )


# --------------------------------------------------------------------------------------------------
# Hashes / prerequisite phases
# --------------------------------------------------------------------------------------------------

add_check(

    "Frozen_11_0_Config_Unchanged",

    CURRENT_CONFIG_SHA256
    ==
    FROZEN_CONFIG_SHA256,

    CURRENT_CONFIG_SHA256
)


add_check(

    "Frozen_11_2_Policy_Unchanged",

    CURRENT_POLICY_SHA256
    ==
    FROZEN_POLICY_SHA256,

    CURRENT_POLICY_SHA256
)


add_check(

    "Phase_11_2_Passed",

    phase_11_2_pass,

    str(
        phase_11_2_pass
    )
)


# --------------------------------------------------------------------------------------------------
# Overall counts
# --------------------------------------------------------------------------------------------------

add_check(

    "Exactly_1200_Episodes",

    len(
        episode_manifest
    )
    ==
    TOTAL_EPISODES,

    str(
        len(
            episode_manifest
        )
    )
)


attack_count = int(

    (
        episode_manifest[
            "Episode_Type"
        ]
        ==
        "ATTACK"
    ).sum()
)


benign_count = int(

    (
        episode_manifest[
            "Episode_Type"
        ]
        ==
        "BENIGN_ONLY"
    ).sum()
)


add_check(

    "Exactly_1000_Attack_Episodes",

    attack_count
    ==
    ATTACK_EPISODES,

    str(
        attack_count
    )
)


add_check(

    "Exactly_200_Benign_Episodes",

    benign_count
    ==
    BENIGN_EPISODES,

    str(
        benign_count
    )
)


# --------------------------------------------------------------------------------------------------
# IDs + seeds
# --------------------------------------------------------------------------------------------------

add_check(

    "Episode_ID_Unique",

    episode_manifest[
        "Episode_ID"
    ].nunique()
    ==
    TOTAL_EPISODES,

    str(
        episode_manifest[
            "Episode_ID"
        ].nunique()
    )
)


add_check(

    "Generator_Seed_Unique",

    episode_manifest[
        "Generator_Seed"
    ].nunique()
    ==
    TOTAL_EPISODES,

    str(
        episode_manifest[
            "Generator_Seed"
        ].nunique()
    )
)


add_check(

    "Baseline_Sequence_Hash_Unique",

    episode_manifest[
        "Baseline_Block_Sequence_SHA256"
    ].nunique()
    ==
    TOTAL_EPISODES,

    str(
        episode_manifest[
            "Baseline_Block_Sequence_SHA256"
        ].nunique()
    )
)


# --------------------------------------------------------------------------------------------------
# Global split totals
# --------------------------------------------------------------------------------------------------

for split, expected in EXPECTED_SPLIT_TOTALS.items():

    actual = int(

        (
            episode_manifest[
                "Split"
            ]
            ==
            split
        ).sum()
    )


    add_check(

        f"{split}_Episode_Total",

        actual
        ==
        expected,

        f"{actual} expected {expected}"
    )


# --------------------------------------------------------------------------------------------------
# Family exact split counts
# --------------------------------------------------------------------------------------------------

for family in ATTACK_FAMILIES:

    family_rows = episode_manifest.loc[
        (
            episode_manifest[
                "Episode_Type"
            ]
            ==
            "ATTACK"
        )

        &

        (
            episode_manifest[
                "Attack_Family"
            ]
            ==
            family
        )
    ]


    add_check(

        f"{family}_Exactly_200_Episodes",

        len(
            family_rows
        )
        ==
        200,

        str(
            len(
                family_rows
            )
        )
    )


    for split, expected in SPLIT_COUNTS_PER_CLASS.items():

        actual = int(

            (
                family_rows[
                    "Split"
                ]
                ==
                split
            ).sum()
        )


        add_check(

            f"{family}_{split}_Exact_Count",

            actual
            ==
            expected,

            f"{actual} expected {expected}"
        )


# --------------------------------------------------------------------------------------------------
# Benign split counts
# --------------------------------------------------------------------------------------------------

benign_rows = episode_manifest.loc[
    episode_manifest[
        "Episode_Type"
    ]
    ==
    "BENIGN_ONLY"
]


for split, expected in SPLIT_COUNTS_PER_CLASS.items():

    actual = int(

        (
            benign_rows[
                "Split"
            ]
            ==
            split
        ).sum()
    )


    add_check(

        f"BENIGN_{split}_Exact_Count",

        actual
        ==
        expected,

        f"{actual} expected {expected}"
    )


# --------------------------------------------------------------------------------------------------
# Precursor regime exact contract
# --------------------------------------------------------------------------------------------------

for family in ATTACK_FAMILIES:

    for split in SPLIT_COUNTS_PER_CLASS.keys():

        for regime in PRECURSOR_REGIMES.keys():

            expected = int(
                PRECURSOR_SPLIT_CONTRACT[
                    regime
                ][
                    split
                ]
            )


            actual = int(

                (
                    (
                        episode_manifest[
                            "Episode_Type"
                        ]
                        ==
                        "ATTACK"
                    )

                    &

                    (
                        episode_manifest[
                            "Attack_Family"
                        ]
                        ==
                        family
                    )

                    &

                    (
                        episode_manifest[
                            "Split"
                        ]
                        ==
                        split
                    )

                    &

                    (
                        episode_manifest[
                            "Precursor_Regime"
                        ]
                        ==
                        regime
                    )
                )

                .sum()
            )


            add_check(

                f"{family}_{split}_{regime}_Exact_Regime_Count",

                actual
                ==
                expected,

                f"{actual} expected {expected}"
            )


# --------------------------------------------------------------------------------------------------
# Attack source quotas
# --------------------------------------------------------------------------------------------------

for _, contract_row in attack_source_quota_contract.iterrows():

    family = contract_row[
        "Synthetic_Family"
    ]


    split = contract_row[
        "Split"
    ]


    source = contract_row[
        "Calibration_Source"
    ]


    expected = int(
        contract_row[
            "Episode_Count"
        ]
    )


    actual = int(

        (
            (
                episode_manifest[
                    "Episode_Type"
                ]
                ==
                "ATTACK"
            )

            &

            (
                episode_manifest[
                    "Attack_Family"
                ]
                ==
                family
            )

            &

            (
                episode_manifest[
                    "Split"
                ]
                ==
                split
            )

            &

            (
                episode_manifest[
                    "Calibration_Source"
                ]
                ==
                source
            )
        )

        .sum()
    )


    add_check(

        f"{family}_{split}_{source}_Exact_Source_Quota",

        actual
        ==
        expected,

        f"{actual} expected {expected}"
    )


# --------------------------------------------------------------------------------------------------
# Benign exact 50/50 source quotas
# --------------------------------------------------------------------------------------------------

for _, contract_row in benign_source_quota_contract.iterrows():

    split = contract_row[
        "Split"
    ]


    source = contract_row[
        "Calibration_Source"
    ]


    expected = int(
        contract_row[
            "Episode_Count"
        ]
    )


    actual = int(

        (
            (
                episode_manifest[
                    "Episode_Type"
                ]
                ==
                "BENIGN_ONLY"
            )

            &

            (
                episode_manifest[
                    "Split"
                ]
                ==
                split
            )

            &

            (
                episode_manifest[
                    "Calibration_Source"
                ]
                ==
                source
            )
        )

        .sum()
    )


    add_check(

        f"BENIGN_{split}_{source}_Exact_Source_Quota",

        actual
        ==
        expected,

        f"{actual} expected {expected}"
    )


# --------------------------------------------------------------------------------------------------
# TRAIN dual-source families see all regimes in both sources where feasible
# --------------------------------------------------------------------------------------------------

for family in ATTACK_FAMILIES:

    family_train = episode_manifest.loc[
        (
            episode_manifest[
                "Episode_Type"
            ]
            ==
            "ATTACK"
        )

        &

        (
            episode_manifest[
                "Attack_Family"
            ]
            ==
            family
        )

        &

        (
            episode_manifest[
                "Split"
            ]
            ==
            "TRAIN"
        )
    ]


    train_sources = family_train[
        "Calibration_Source"
    ].unique()


    if len(
        train_sources
    ) == 2:

        for source in train_sources:

            source_rows = family_train.loc[
                family_train[
                    "Calibration_Source"
                ]
                ==
                source
            ]


            source_count = len(
                source_rows
            )


            if source_count >= len(
                PRECURSOR_REGIMES
            ):

                observed_regimes = set(
                    source_rows[
                        "Precursor_Regime"
                    ].unique()
                )


                expected_regimes = set(
                    PRECURSOR_REGIMES.keys()
                )


                add_check(

                    f"{family}_TRAIN_{source}_All_Precursor_Regimes",

                    observed_regimes
                    ==
                    expected_regimes,

                    str(
                        sorted(
                            observed_regimes
                        )
                    )
                )


# --------------------------------------------------------------------------------------------------
# Episode lengths
# --------------------------------------------------------------------------------------------------

add_check(

    "All_Episode_Lengths_In_Frozen_Range",

    bool(

        episode_manifest[
            "Episode_Length_Minutes"
        ]

        .between(
            EPISODE_MIN_LENGTH,
            EPISODE_MAX_LENGTH
        )

        .all()
    ),

    (
        f"{episode_manifest['Episode_Length_Minutes'].min()}–"
        f"{episode_manifest['Episode_Length_Minutes'].max()}"
    )
)


# --------------------------------------------------------------------------------------------------
# Attack temporal validity
# --------------------------------------------------------------------------------------------------

attack_rows = episode_manifest.loc[
    episode_manifest[
        "Episode_Type"
    ]
    ==
    "ATTACK"
].copy()


add_check(

    "Attack_Onsets_In_Frozen_Range",

    bool(

        attack_rows[
            "Attack_Onset_Minute"
        ]

        .between(
            ATTACK_ONSET_MIN,
            ATTACK_ONSET_MAX
        )

        .all()
    ),

    (
        f"{int(attack_rows['Attack_Onset_Minute'].min())}–"
        f"{int(attack_rows['Attack_Onset_Minute'].max())}"
    )
)


add_check(

    "Every_Attack_Has_At_Least_50_Minutes_PreOnset_History",

    bool(

        (
            attack_rows[
                "Attack_Onset_Minute"
            ]

            >=

            MINIMUM_PREONSET_HISTORY
        ).all()
    ),

    str(
        int(
            attack_rows[
                "Attack_Onset_Minute"
            ].min()
        )
    )
)


add_check(

    "Attack_Durations_In_Frozen_Range",

    bool(

        attack_rows[
            "Attack_Duration_Minutes"
        ]

        .between(
            ATTACK_DURATION_MIN,
            ATTACK_DURATION_MAX
        )

        .all()
    ),

    (
        f"{int(attack_rows['Attack_Duration_Minutes'].min())}–"
        f"{int(attack_rows['Attack_Duration_Minutes'].max())}"
    )
)


add_check(

    "Recovery_Durations_In_Frozen_Range",

    bool(

        attack_rows[
            "Recovery_Duration_Minutes"
        ]

        .between(
            RECOVERY_DURATION_MIN,
            RECOVERY_DURATION_MAX
        )

        .all()
    ),

    (
        f"{int(attack_rows['Recovery_Duration_Minutes'].min())}–"
        f"{int(attack_rows['Recovery_Duration_Minutes'].max())}"
    )
)


fits_episode = (

    attack_rows[
        "Recovery_End_Minute"
    ]

    <

    attack_rows[
        "Episode_Length_Minutes"
    ]
)


add_check(

    "Attack_And_Recovery_Fit_Inside_Episode",

    bool(
        fits_episode.all()
    ),

    f"{int((~fits_episode).sum())} failures"
)


# --------------------------------------------------------------------------------------------------
# Precursor validity
# --------------------------------------------------------------------------------------------------

precursor_failures = 0


for _, row in attack_rows.iterrows():

    regime = row[
        "Precursor_Regime"
    ]


    length = int(
        row[
            "Precursor_Length_Minutes"
        ]
    )


    spec = PRECURSOR_REGIMES[
        regime
    ]


    min_len = int(
        spec[
            "length_min"
        ]
    )


    max_len = int(
        spec[
            "length_max"
        ]
    )


    if not (
        min_len
        <=
        length
        <=
        max_len
    ):

        precursor_failures += 1


    if (
        length
        >
        0
        and
        row[
            "Precursor_Start_Minute"
        ]
        <
        0
    ):

        precursor_failures += 1


add_check(

    "All_Precursor_Lengths_Respect_Regime_Spec",

    precursor_failures
    ==
    0,

    str(
        precursor_failures
    )
)


# --------------------------------------------------------------------------------------------------
# Benign attack fields
# --------------------------------------------------------------------------------------------------

benign_attack_fields_empty = bool(

    benign_rows[
        "Attack_Onset_Minute"
    ].isna().all()

    and

    benign_rows[
        "Attack_End_Minute"
    ].isna().all()

    and

    benign_rows[
        "Attack_Onset_UTC"
    ].isna().all()
)


add_check(

    "Benign_Episodes_Have_No_Attack_Onset",

    benign_attack_fields_empty,

    str(
        benign_attack_fields_empty
    )
)


# --------------------------------------------------------------------------------------------------
# Block sequence count
# --------------------------------------------------------------------------------------------------

expected_block_count = np.ceil(

    episode_manifest[
        "Episode_Length_Minutes"
    ]

    /
    5

).astype(
    int
)


add_check(

    "Baseline_Block_Count_Correct_For_All_Episodes",

    bool(

        (
            episode_manifest[
                "Baseline_Block_Count"
            ]

            ==

            expected_block_count
        ).all()
    ),

    str(
        int(

            (
                episode_manifest[
                    "Baseline_Block_Count"
                ]

                !=

                expected_block_count
            ).sum()
        )
    )
)


# --------------------------------------------------------------------------------------------------
# Block split/source leakage
# --------------------------------------------------------------------------------------------------

block_source_match = (

    block_usage[
        "Calibration_Source"
    ]

    ==

    block_usage[
        "Block_Source"
    ]
)


block_pool_match = (

    block_usage[
        "Calibration_Pool"
    ]

    ==

    block_usage[
        "Block_Pool"
    ]
)


add_check(

    "Every_Block_Matches_Episode_Calibration_Source",

    bool(
        block_source_match.all()
    ),

    f"{int((~block_source_match).sum())} mismatches"
)


add_check(

    "Every_Block_Matches_Episode_Calibration_Pool",

    bool(
        block_pool_match.all()
    ),

    f"{int((~block_pool_match).sum())} mismatches"
)


# Explicit split → calibration pool relationship

expected_pool_by_episode = episode_manifest[
    "Split"
].map(
    CALIBRATION_POOL_FOR_SPLIT
)


add_check(

    "Every_Episode_Uses_Correct_Split_Calibration_Pool",

    bool(

        (
            episode_manifest[
                "Calibration_Pool"
            ]

            ==

            expected_pool_by_episode
        ).all()
    ),

    "TRAIN→CAL_TRAIN, VALIDATION→CAL_VALIDATION, TEST→CAL_TEST"
)


# --------------------------------------------------------------------------------------------------
# Timeline integrity
# --------------------------------------------------------------------------------------------------

timeline_sorted = (

    episode_manifest

    .sort_values(
        "Timeline_Order"
    )

    .reset_index(
        drop=True
    )
)


timeline_order_unique = (

    timeline_sorted[
        "Timeline_Order"
    ].nunique()

    ==
    TOTAL_EPISODES
)


add_check(

    "Timeline_Order_Unique",

    timeline_order_unique,

    str(
        timeline_sorted[
            "Timeline_Order"
        ].nunique()
    )
)


timeline_gap_failures = 0

timeline_overlap_failures = 0


for i in range(
    1,
    len(
        timeline_sorted
    )
):

    previous = timeline_sorted.iloc[
        i - 1
    ]


    current = timeline_sorted.iloc[
        i
    ]


    previous_end = pd.Timestamp(
        previous[
            "Episode_End_UTC"
        ]
    )


    current_start = pd.Timestamp(
        current[
            "Episode_Start_UTC"
        ]
    )


    if current_start <= previous_end:

        timeline_overlap_failures += 1


    unused_minutes = int(

        (
            current_start
            -
            previous_end
        )

        .total_seconds()

        /
        60

        -
        1
    )


    if unused_minutes != EPISODE_SAFETY_GAP_MINUTES:

        timeline_gap_failures += 1


add_check(

    "No_Episode_Timestamp_Overlap",

    timeline_overlap_failures
    ==
    0,

    str(
        timeline_overlap_failures
    )
)


add_check(

    "Exactly_60_Unused_Minutes_Between_Episodes",

    timeline_gap_failures
    ==
    0,

    str(
        timeline_gap_failures
    )
)


# --------------------------------------------------------------------------------------------------
# Timeline quartile coverage
# --------------------------------------------------------------------------------------------------

quartile_split_crosstab = pd.crosstab(

    episode_manifest[
        "Timeline_Quartile"
    ],

    episode_manifest[
        "Split"
    ]
)


all_splits_each_quartile = bool(

    (
        quartile_split_crosstab
        >
        0
    ).all().all()
)


add_check(

    "Every_Timeline_Quartile_Contains_All_Splits",

    all_splits_each_quartile,

    quartile_split_crosstab.to_dict()
)


quartile_family_crosstab = pd.crosstab(

    episode_manifest[
        "Timeline_Quartile"
    ],

    episode_manifest[
        "Attack_Family"
    ]
)


all_families_each_quartile = bool(

    (
        quartile_family_crosstab
        >
        0
    ).all().all()
)


add_check(

    "Every_Timeline_Quartile_Contains_All_Attack_Family_Labels",

    all_families_each_quartile,

    quartile_family_crosstab.to_dict()
)


# --------------------------------------------------------------------------------------------------
# Manifest must contain NO traffic predictor columns
# --------------------------------------------------------------------------------------------------

predictor_columns_in_manifest = sorted(

    set(
        CONFIG[
            "features"
        ][
            "final_predictor_whitelist"
        ]
    )

    &

    set(
        episode_manifest.columns
    )
)


add_check(

    "No_Model_Predictor_Columns_In_Episode_Manifest",

    len(
        predictor_columns_in_manifest
    )
    ==
    0,

    str(
        predictor_columns_in_manifest
    )
)


validation = pd.DataFrame(
    validation_rows
)


print(
    validation.to_string(
        index=False
    )
)


PHASE_11_3_PASS = bool(

    validation[
        "Passed"
    ].all()
)


# ==================================================================================================
# 11.3AB — SOURCE / FAMILY / REGIME SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AB — FINAL EPISODE DISTRIBUTION SUMMARY")
print("=" * 120)


family_split_summary = (

    episode_manifest.loc[
        episode_manifest[
            "Episode_Type"
        ]
        ==
        "ATTACK"
    ]

    .groupby(
        [
            "Attack_Family",
            "Split"
        ],
        observed=True
    )

    .size()

    .rename(
        "Episode_Count"
    )

    .reset_index()
)


source_split_summary = (

    episode_manifest

    .groupby(
        [
            "Episode_Type",
            "Attack_Family",
            "Split",
            "Calibration_Source"
        ],
        observed=True
    )

    .size()

    .rename(
        "Episode_Count"
    )

    .reset_index()
)


print(
    "Family × split:\n"
)

print(
    family_split_summary.to_string(
        index=False
    )
)


print(
    "\nSource × split:\n"
)

print(
    source_split_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3AC — MANIFEST DIVERSITY SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AC — MANIFEST DIVERSITY SUMMARY")
print("=" * 120)


diversity_summary = pd.DataFrame({

    "Metric": [

        "Unique_Episode_ID",

        "Unique_Generator_Seed",

        "Unique_Baseline_Sequences",

        "Minimum_Episode_Length",

        "Maximum_Episode_Length",

        "Minimum_Unique_Blocks_Per_Episode",

        "Median_Unique_Blocks_Per_Episode",

        "Maximum_Block_Repetition_Ratio",

        "Unique_CIC_Blocks_Used",

        "Unique_UNSW_Blocks_Used",

        "Timeline_Start",

        "Timeline_End"
    ],


    "Value": [

        episode_manifest[
            "Episode_ID"
        ].nunique(),

        episode_manifest[
            "Generator_Seed"
        ].nunique(),

        episode_manifest[
            "Baseline_Block_Sequence_SHA256"
        ].nunique(),

        int(
            episode_manifest[
                "Episode_Length_Minutes"
            ].min()
        ),

        int(
            episode_manifest[
                "Episode_Length_Minutes"
            ].max()
        ),

        int(
            episode_manifest[
                "Unique_Baseline_Blocks_Used"
            ].min()
        ),

        float(
            episode_manifest[
                "Unique_Baseline_Blocks_Used"
            ].median()
        ),

        float(
            episode_manifest[
                "Baseline_Block_Repetition_Ratio"
            ].max()
        ),

        block_usage.loc[
            block_usage[
                "Calibration_Source"
            ]
            ==
            "CIC_IDS2017",
            "Safe_Benign_Block_ID"
        ].nunique(),

        block_usage.loc[
            block_usage[
                "Calibration_Source"
            ]
            ==
            "UNSW_NB15",
            "Safe_Benign_Block_ID"
        ].nunique(),

        str(
            episode_manifest[
                "Episode_Start_UTC"
            ].min()
        ),

        str(
            episode_manifest[
                "Episode_End_UTC"
            ].max()
        )
    ]
})


print(
    diversity_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.3AD — SAVE PRE-HASH ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AD — SAVE MANIFEST ARTIFACTS")
print("=" * 120)


def save_csv(
    dataframe,
    directory,
    filename
):

    path = (
        directory
        /
        filename
    )


    dataframe.to_csv(
        path,
        index=False
    )


    print(
        "✓",
        path
    )


    return path


save_csv(

    attack_source_quota_contract,

    MANIFEST_DIR,

    "11_3_attack_source_quota_contract.csv"
)


save_csv(

    benign_source_quota_contract,

    MANIFEST_DIR,

    "11_3_benign_source_quota_contract.csv"
)


save_csv(

    source_regime_contract,

    MANIFEST_DIR,

    "11_3_source_regime_contract.csv"
)


save_csv(

    block_usage,

    MANIFEST_DIR,

    "11_3_episode_block_usage.csv"
)


save_csv(

    block_usage_summary,

    MANIFEST_DIR,

    "11_3_block_usage_summary.csv"
)


save_csv(

    family_evaluation_policy,

    MANIFEST_DIR,

    "11_3_family_evaluation_policy.csv"
)


save_csv(

    actual_source_distribution,

    MANIFEST_DIR,

    "11_3_actual_source_distribution.csv"
)


save_csv(

    actual_precursor_distribution,

    MANIFEST_DIR,

    "11_3_actual_precursor_distribution.csv"
)


save_csv(

    timeline_split_audit,

    MANIFEST_DIR,

    "11_3_timeline_split_audit.csv"
)


save_csv(

    timeline_family_audit,

    MANIFEST_DIR,

    "11_3_timeline_family_audit.csv"
)


save_csv(

    diversity_summary,

    MANIFEST_DIR,

    "11_3_manifest_diversity_summary.csv"
)


save_csv(

    effect_coverage,

    MANIFEST_DIR,

    "11_3_source_effect_coverage.csv"
)


save_csv(

    validation,

    MANIFEST_DIR,

    "11_3_validation.csv"
)


# ==================================================================================================
# 11.3AE — SAVE FULL FROZEN EPISODE MANIFEST
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AE — SAVE FULL EPISODE MANIFEST")
print("=" * 120)


# Stable column ordering.

MANIFEST_COLUMNS = [

    "Dataset_Version",

    "Episode_ID",

    "Split",

    "Episode_Type",

    "Attack_Family",

    "Calibration_Source",

    "Calibration_Pool",

    "Source_Weight_Used",

    "Precursor_Regime",

    "Precursor_Temporal_Profile",

    "Precursor_Length_Minutes",

    "Precursor_Max_Alpha",

    "Episode_Length_Minutes",

    "Attack_Onset_Minute",

    "Attack_Duration_Minutes",

    "Attack_End_Minute",

    "Recovery_Duration_Minutes",

    "Recovery_Start_Minute",

    "Recovery_End_Minute",

    "Precursor_Start_Minute",

    "Precursor_End_Minute",

    "Episode_Load_Multiplier",

    "Family_Effect_Strength_Multiplier",

    "Baseline_Block_Count",

    "Unique_Baseline_Blocks_Used",

    "Unique_Calibration_Groups_Used",

    "Baseline_Block_Repetition_Ratio",

    "Final_Block_Minutes_Used",

    "Baseline_Block_Sequence",

    "Baseline_Block_Sequence_SHA256",

    "Generator_Seed",

    "Timeline_Order",

    "Timeline_Quartile",

    "Episode_Start_UTC",

    "Episode_End_UTC",

    "Precursor_Start_UTC",

    "Precursor_End_UTC",

    "Attack_Onset_UTC",

    "Attack_End_UTC",

    "Recovery_Start_UTC",

    "Recovery_End_UTC",

    "Calibration_Policy_SHA256"
]


episode_manifest = episode_manifest[
    MANIFEST_COLUMNS
].copy()


# Save sorted by Episode_ID for stable file hashing.

episode_manifest = (

    episode_manifest

    .sort_values(
        "Episode_ID"
    )

    .reset_index(
        drop=True
    )
)


EPISODE_MANIFEST_FILE = (

    MANIFEST_DIR
    /
    "11_3_episode_manifest.csv"
)


episode_manifest.to_csv(
    EPISODE_MANIFEST_FILE,
    index=False
)


print(
    "✓",
    EPISODE_MANIFEST_FILE
)


# Parquet convenience copy.

EPISODE_MANIFEST_PARQUET = (

    MANIFEST_DIR
    /
    "11_3_episode_manifest.parquet"
)


try:

    episode_manifest.to_parquet(

        EPISODE_MANIFEST_PARQUET,

        index=False
    )


    print(
        "✓",
        EPISODE_MANIFEST_PARQUET
    )


except Exception as error:

    print(
        "⚠ Parquet save skipped:",
        str(
            error
        )[:200]
    )


# ==================================================================================================
# 11.3AF — FREEZE FULL MANIFEST HASH
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AF — FREEZE FULL MANIFEST SHA256")
print("=" * 120)


with open(
    EPISODE_MANIFEST_FILE,
    "rb"
) as f:

    manifest_bytes = f.read()


EPISODE_MANIFEST_SHA256 = hashlib.sha256(
    manifest_bytes
).hexdigest()


EPISODE_MANIFEST_HASH_FILE = (

    MANIFEST_DIR
    /
    "11_3_episode_manifest_SHA256.txt"
)


with open(
    EPISODE_MANIFEST_HASH_FILE,
    "w"
) as f:

    f.write(
        EPISODE_MANIFEST_SHA256
        +
        "\n"
    )


print(
    EPISODE_MANIFEST_SHA256
)


print(
    "✓",
    EPISODE_MANIFEST_HASH_FILE
)


# ==================================================================================================
# 11.3AG — LOCK TEST MANIFEST SEPARATELY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AG — LOCK TEST EPISODES")
print("=" * 120)


test_manifest = (

    episode_manifest.loc[
        episode_manifest[
            "Split"
        ]
        ==
        "TEST"
    ]

    .copy()

    .sort_values(
        "Episode_ID"
    )

    .reset_index(
        drop=True
    )
)


TEST_MANIFEST_FILE = (

    MANIFEST_DIR
    /
    "11_3_TEST_manifest_LOCKED.csv"
)


test_manifest.to_csv(
    TEST_MANIFEST_FILE,
    index=False
)


with open(
    TEST_MANIFEST_FILE,
    "rb"
) as f:

    test_bytes = f.read()


TEST_MANIFEST_SHA256 = hashlib.sha256(
    test_bytes
).hexdigest()


TEST_HASH_FILE = (

    MANIFEST_DIR
    /
    "11_3_TEST_manifest_LOCKED_SHA256.txt"
)


with open(
    TEST_HASH_FILE,
    "w"
) as f:

    f.write(
        TEST_MANIFEST_SHA256
        +
        "\n"
    )


print(
    "TEST episodes:",
    len(
        test_manifest
    )
)


print(
    "TEST SHA256:"
)

print(
    TEST_MANIFEST_SHA256
)


print(
    "✓",
    TEST_MANIFEST_FILE
)

print(
    "✓",
    TEST_HASH_FILE
)


# ==================================================================================================
# 11.3AH — FROZEN MANIFEST POLICY JSON
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AH — WRITE MANIFEST FREEZE POLICY")
print("=" * 120)


manifest_policy = {

    "phase":
        "11.3",

    "dataset":
        DATASET_SHORT_NAME,

    "dataset_version":
        DATASET_VERSION,

    "phase_11_0_config_sha256":
        CURRENT_CONFIG_SHA256,

    "phase_11_2_calibration_policy_sha256":
        CURRENT_POLICY_SHA256,

    "phase_11_3_episode_manifest_sha256":
        EPISODE_MANIFEST_SHA256,

    "phase_11_3_test_manifest_sha256":
        TEST_MANIFEST_SHA256,

    "total_episodes":
        TOTAL_EPISODES,

    "attack_episodes":
        ATTACK_EPISODES,

    "benign_only_episodes":
        BENIGN_EPISODES,

    "split_totals":
        EXPECTED_SPLIT_TOTALS,

    "attack_families":
        ATTACK_FAMILIES,

    "episode_source_allocation": {

        "attack":
            "DETERMINISTIC_LARGEST_REMAINDER_FROM_PHASE_11_2_SOURCE_WEIGHTS",

        "benign":
            "EXACT_50_50_CIC_UNSW_WITHIN_EACH_SPLIT"
    },

    "precursor_allocation":
        "EXACT_PHASE_11_0_REGIME_SPLIT_CONTRACT",

    "baseline_block_policy": {

        "source_must_match_episode_source":
            True,

        "pool_must_match_episode_split":
            True,

        "same_exact_block_sequence_across_episodes":
            False,

        "block_reuse_within_same_split":
            True,

        "cross_split_block_reuse":
            False
    },

    "timestamp_policy": {

        "timezone":
            "UTC",

        "minute_frequency_seconds":
            TIMESTAMP_FREQUENCY_SECONDS,

        "episode_overlap_allowed":
            False,

        "unused_minutes_between_episodes":
            EPISODE_SAFETY_GAP_MINUTES,

        "timeline_order":
            "DETERMINISTIC_GLOBAL_SEEDED_PERMUTATION"
    },

    "test_policy": {

        "test_manifest_locked":
            True,

        "regenerate_test_after_model_results":
            False,

        "expected_test_sha256":
            TEST_MANIFEST_SHA256
    },

    "feature_generation_performed":
        False,

    "next_phase":
        "11.4_SYNTHETIC_MINUTE_GENERATION"
}


MANIFEST_POLICY_FILE = (

    MANIFEST_DIR
    /
    "11_3_frozen_manifest_policy.json"
)


with open(
    MANIFEST_POLICY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest_policy,
        f,
        indent=4
    )


print(
    "✓",
    MANIFEST_POLICY_FILE
)


# ==================================================================================================
# 11.3AI — PHASE MANIFEST SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AI — PHASE 11.3 SUMMARY")
print("=" * 120)


phase_summary = pd.DataFrame({

    "Metric": [

        "Total_Episodes",

        "Attack_Episodes",

        "Benign_Episodes",

        "Train_Episodes",

        "Validation_Episodes",

        "Test_Episodes",

        "Attack_Families",

        "Unique_Generator_Seeds",

        "Unique_Baseline_Sequences",

        "Safe_Block_Usage_Rows",

        "Timeline_Start",

        "Timeline_End",

        "Timeline_Overlap_Failures",

        "Timeline_Gap_Failures",

        "Validation_Checks",

        "Validation_Passed",

        "Phase_11_3_Pass",

        "Synthetic_Feature_Generation_Performed",

        "Episode_Manifest_SHA256",

        "Locked_Test_Manifest_SHA256"
    ],


    "Value": [

        len(
            episode_manifest
        ),

        attack_count,

        benign_count,

        int(
            (
                episode_manifest[
                    "Split"
                ]
                ==
                "TRAIN"
            ).sum()
        ),

        int(
            (
                episode_manifest[
                    "Split"
                ]
                ==
                "VALIDATION"
            ).sum()
        ),

        int(
            (
                episode_manifest[
                    "Split"
                ]
                ==
                "TEST"
            ).sum()
        ),

        len(
            ATTACK_FAMILIES
        ),

        episode_manifest[
            "Generator_Seed"
        ].nunique(),

        episode_manifest[
            "Baseline_Block_Sequence_SHA256"
        ].nunique(),

        len(
            block_usage
        ),

        str(
            episode_manifest[
                "Episode_Start_UTC"
            ].min()
        ),

        str(
            episode_manifest[
                "Episode_End_UTC"
            ].max()
        ),

        timeline_overlap_failures,

        timeline_gap_failures,

        len(
            validation
        ),

        int(
            validation[
                "Passed"
            ].sum()
        ),

        PHASE_11_3_PASS,

        False,

        EPISODE_MANIFEST_SHA256,

        TEST_MANIFEST_SHA256
    ]
})


print(
    phase_summary.to_string(
        index=False
    )
)


save_csv(

    phase_summary,

    MANIFEST_DIR,

    "11_3_manifest_summary.csv"
)


# ==================================================================================================
# 11.3AJ — HARD STOP IF ANYTHING FAILED
# ==================================================================================================

print("\n" + "=" * 120)
print("11.3AJ — FINAL DECISION")
print("=" * 120)


if not PHASE_11_3_PASS:

    failed = validation.loc[
        ~validation[
            "Passed"
        ]
    ]


    print(
        "\nFAILED CHECKS:\n"
    )


    print(
        failed.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "PHASE 11.3 FAILED. "
        "DO NOT PROCEED TO SYNTHETIC FEATURE GENERATION."
    )


print(
    f"""
✓ PHASE 11.3 — FROZEN EPISODE MANIFEST: PASS


EPISODE COUNTS
--------------
Total:

    {len(episode_manifest):,}

Attack:

    {attack_count:,}

Benign-only:

    {benign_count:,}


SPLITS
------
TRAIN:

    {int((episode_manifest["Split"] == "TRAIN").sum()):,}

VALIDATION:

    {int((episode_manifest["Split"] == "VALIDATION").sum()):,}

TEST:

    {int((episode_manifest["Split"] == "TEST").sum()):,}


EVERY ATTACK FAMILY
-------------------
TRAIN:

    140 episodes

VALIDATION:

    30 episodes

TEST:

    30 episodes


BENIGN
------
TRAIN:

    70 CIC
    70 UNSW

VALIDATION:

    15 CIC
    15 UNSW

TEST:

    15 CIC
    15 UNSW


CALIBRATION SOURCE
------------------
Attack-family sources use the frozen Phase 11.2
source-conditioned quotas.

No source selection is left to random luck.


PRECURSOR REGIMES
-----------------
Every family exactly obeys the frozen Phase 11.0:

    NO_PRECURSOR
    SHORT_WEAK
    SHORT_STRONG
    GRADUAL_WEAK
    GRADUAL_STRONG
    BURSTY

split allocation.


BASELINE TRAFFIC
----------------
Each episode already has a complete ordered sequence of
real 5-minute calibration block IDs.

Each block:

    ✓ matches the episode source
    ✓ matches the episode split calibration pool

No two episodes have the same complete baseline block sequence.


TIMESTAMPS
----------
All timestamps:

    UTC

Within an episode Phase 11.4 will create:

    exactly +1 minute rows

Between episodes:

    exactly {EPISODE_SAFETY_GAP_MINUTES} unused minutes

Episode overlap:

    NONE


TEST LOCK
---------
The TEST manifest is now frozen separately.

TEST episodes MUST NOT be regenerated because of model performance.

TEST MANIFEST SHA256:

    {TEST_MANIFEST_SHA256}


FULL EPISODE MANIFEST SHA256
----------------------------

    {EPISODE_MANIFEST_SHA256}


VERY IMPORTANT
--------------
NO SYNTHETIC TRAFFIC FEATURE VALUES EXIST YET.

11.3 created only the PLAN for every future episode.


NEXT PHASE
----------
PHASE 11.4 — SYNTHETIC TEMPORAL MINUTE GENERATION

11.4 will read this exact manifest and:

    1. reconstruct each episode baseline from its assigned
       real 5-minute block sequence

    2. apply the episode load multiplier

    3. apply source-matched attack family effects

    4. create precursor behaviour according to regime

    5. create attack-active behaviour

    6. create recovery behaviour

    7. mathematically derive all 14 final predictors

    8. calculate authoritative onset targets

    9. produce predictors / targets / metadata separately

    10. run structural validation BEFORE any model training
"""
)


print(
    "=" * 120
)

print(
    "PHASE 11.3 COMPLETE"
)

print(
    "=" * 120
)

In [ ]:
# ==================================================================================================
# PHASE 11.4 — SYNTHETIC TEMPORAL MINUTE GENERATION
# ==================================================================================================
#
# INPUT
# -----
# Frozen Phase 11.3 episode manifest
# Frozen Phase 11.2 leakage-safe calibration blocks
# Frozen Phase 11.1 source/family calibration statistics
#
#
# PURPOSE
# -------
# Generate the actual minute-level Synthetic_NAF_v1 benchmark.
#
#
# GENERATION PRINCIPLES
# ---------------------
#
# 1. REAL benign minute blocks provide empirical baseline structure.
#
# 2. Reused real blocks are NOT pasted unchanged.
#    Every occurrence receives deterministic temporally-correlated local variation.
#
# 3. Source matching is mandatory:
#
#       CIC baseline  + CIC family effect
#       UNSW baseline + UNSW family effect
#
#    Cross-source mixing is forbidden.
#
# 4. Only independent primitives are manipulated:
#
#       Flow_Count
#       Flow_Duration behavior
#       Forward packets / flow
#       Backward packets / flow
#       Forward bytes / flow
#       Backward bytes / flow
#
#    Dependent quantities are ALWAYS reconstructed:
#
#       Total_Packets
#       Total_Bytes
#       Packets_Per_Second
#       Bytes_Per_Second
#       Forward_Backward_Packet_Ratio
#       Forward_Backward_Byte_Ratio
#
# 5. Precursor behavior is a CONTROLLED SIMULATION ASSUMPTION.
#
#    It is NOT claimed that CIC-IDS2017 or UNSW-NB15 empirically prove
#    these precursor trajectories.
#
# 6. NO_PRECURSOR episodes remain genuine negative controls:
#
#       alpha == 0 before the attack onset
#
# 7. Strict future-onset targets enforce:
#
#       current state must be benign
#       full H-minute future must exist inside the SAME episode
#       no cross-episode shift
#
#
# OUTPUT SEPARATION
# -----------------
#
# predictors:
#       join keys + 14 allowed model predictors
#
# targets:
#       attack/onset/TTO/horizon labels
#
# metadata:
#       source, split, timestamps, precursor regime, generator state,
#       block provenance, etc.
#
# MODEL CODE MUST USE ONLY THE FROZEN 14-PREDICTOR WHITELIST.
#
# ==================================================================================================


from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import hashlib
import json
import math
import warnings
import gc

warnings.filterwarnings("ignore")


# ==================================================================================================
# 11.4A — PATHS
# ==================================================================================================

print("=" * 120)
print("PHASE 11.4 — SYNTHETIC TEMPORAL MINUTE GENERATION")
print("=" * 120)


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)


DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)


SPEC_DIR = (
    DATASET_ROOT
    / "spec"
)


CALIBRATION_DIR = (
    DATASET_ROOT
    / "calibration"
)


MANIFEST_DIR = (
    DATASET_ROOT
    / "manifests"
)


GENERATED_DIR = (
    DATASET_ROOT
    / "generated"
    / "11_4"
)


AUDIT_DIR = (
    DATASET_ROOT
    / "audit"
)


GENERATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config.json"
)


CONFIG_HASH_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config_SHA256.txt"
)


POLICY_11_2_FILE = (
    CALIBRATION_DIR
    / "11_2_frozen_calibration_policy.json"
)


POLICY_11_2_HASH_FILE = (
    CALIBRATION_DIR
    / "11_2_frozen_calibration_policy_SHA256.txt"
)


EPISODE_MANIFEST_FILE = (
    MANIFEST_DIR
    / "11_3_episode_manifest.csv"
)


EPISODE_MANIFEST_HASH_FILE = (
    MANIFEST_DIR
    / "11_3_episode_manifest_SHA256.txt"
)


LOCKED_TEST_MANIFEST_FILE = (
    MANIFEST_DIR
    / "11_3_TEST_manifest_LOCKED.csv"
)


LOCKED_TEST_HASH_FILE = (
    MANIFEST_DIR
    / "11_3_TEST_manifest_LOCKED_SHA256.txt"
)


MANIFEST_POLICY_FILE = (
    MANIFEST_DIR
    / "11_3_frozen_manifest_policy.json"
)


BLOCK_USAGE_11_3_FILE = (
    MANIFEST_DIR
    / "11_3_episode_block_usage.csv"
)


SAFE_BLOCK_INDEX_FILE = (
    CALIBRATION_DIR
    / "11_2_safe_benign_block_index.csv"
)


SAFE_BLOCK_ROWS_FILE = (
    CALIBRATION_DIR
    / "11_2_safe_benign_block_rows.csv"
)


SOURCE_EFFECT_FILE = (
    CALIBRATION_DIR
    / "11_1_source_family_log_effect_vectors.csv"
)


FAMILY_INTENSITY_FILE = (
    CALIBRATION_DIR
    / "11_1_family_attack_flow_intensity_quantiles.csv"
)


FLOW_PROFILE_QUANTILES_FILE = (
    CALIBRATION_DIR
    / "11_1_flow_profile_quantiles.csv"
)


REAL_MINUTE_QUANTILES_FILE = (
    CALIBRATION_DIR
    / "11_1_real_minute_feature_quantiles.csv"
)


# ==================================================================================================
# 11.4B — VERIFY REQUIRED INPUTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4B — VERIFY FROZEN INPUT ARTIFACTS")
print("=" * 120)


required_files = [

    CONFIG_FILE,
    CONFIG_HASH_FILE,

    POLICY_11_2_FILE,
    POLICY_11_2_HASH_FILE,

    EPISODE_MANIFEST_FILE,
    EPISODE_MANIFEST_HASH_FILE,

    LOCKED_TEST_MANIFEST_FILE,
    LOCKED_TEST_HASH_FILE,

    MANIFEST_POLICY_FILE,

    BLOCK_USAGE_11_3_FILE,

    SAFE_BLOCK_INDEX_FILE,
    SAFE_BLOCK_ROWS_FILE,

    SOURCE_EFFECT_FILE,

    FAMILY_INTENSITY_FILE,

    FLOW_PROFILE_QUANTILES_FILE,

    REAL_MINUTE_QUANTILES_FILE
]


missing = []


for path in required_files:

    if path.exists():

        print(
            "✓",
            path.name
        )

    else:

        print(
            "✗",
            path
        )

        missing.append(
            path
        )


if missing:

    raise FileNotFoundError(
        "Phase 11.4 blocked because required artifacts are missing."
    )


# ==================================================================================================
# 11.4C — HASH HELPERS
# ==================================================================================================

def sha256_file(
    path
):

    hasher = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )


            if not chunk:

                break


            hasher.update(
                chunk
            )


    return hasher.hexdigest()


def stable_seed64(
    text
):

    digest = hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).digest()


    return int(

        int.from_bytes(
            digest[:8],
            byteorder="big",
            signed=False
        )

        %

        (
            2**63
            -
            1
        )
    )


def stable_array_hash(
    array,
    decimals=8
):

    array = np.asarray(
        array,
        dtype=np.float64
    )


    array = np.round(
        array,
        decimals=decimals
    )


    return hashlib.sha256(
        array.tobytes()
    ).hexdigest()


# ==================================================================================================
# 11.4D — VERIFY FROZEN HASHES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4D — VERIFY CONFIG / POLICY / MANIFEST HASHES")
print("=" * 120)


# 11.0

CURRENT_CONFIG_HASH = sha256_file(
    CONFIG_FILE
)


FROZEN_CONFIG_HASH = (

    CONFIG_HASH_FILE

    .read_text()

    .strip()
)


# 11.2

CURRENT_POLICY_HASH = sha256_file(
    POLICY_11_2_FILE
)


FROZEN_POLICY_HASH = (

    POLICY_11_2_HASH_FILE

    .read_text()

    .strip()
)


# 11.3 full manifest

CURRENT_MANIFEST_HASH = sha256_file(
    EPISODE_MANIFEST_FILE
)


FROZEN_MANIFEST_HASH = (

    EPISODE_MANIFEST_HASH_FILE

    .read_text()

    .strip()
)


# 11.3 locked TEST

CURRENT_TEST_HASH = sha256_file(
    LOCKED_TEST_MANIFEST_FILE
)


FROZEN_TEST_HASH = (

    LOCKED_TEST_HASH_FILE

    .read_text()

    .strip()
)


print(
    "11.0 config:",
    CURRENT_CONFIG_HASH
)


print(
    "11.2 policy:",
    CURRENT_POLICY_HASH
)


print(
    "11.3 manifest:",
    CURRENT_MANIFEST_HASH
)


print(
    "11.3 locked TEST:",
    CURRENT_TEST_HASH
)


if CURRENT_CONFIG_HASH != FROZEN_CONFIG_HASH:

    raise RuntimeError(
        "Phase 11.0 configuration hash mismatch."
    )


if CURRENT_POLICY_HASH != FROZEN_POLICY_HASH:

    raise RuntimeError(
        "Phase 11.2 policy hash mismatch."
    )


if CURRENT_MANIFEST_HASH != FROZEN_MANIFEST_HASH:

    raise RuntimeError(
        "Phase 11.3 episode manifest changed after freeze."
    )


if CURRENT_TEST_HASH != FROZEN_TEST_HASH:

    raise RuntimeError(
        "Phase 11.3 locked TEST manifest changed after freeze."
    )


print(
    "\n✓ All frozen hashes verified"
)


# ==================================================================================================
# 11.4E — LOAD FROZEN CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4E — LOAD GENERATION CONTRACT")
print("=" * 120)


with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(
        f
    )


with open(
    POLICY_11_2_FILE,
    "r",
    encoding="utf-8"
) as f:

    POLICY_11_2 = json.load(
        f
    )


with open(
    MANIFEST_POLICY_FILE,
    "r",
    encoding="utf-8"
) as f:

    MANIFEST_POLICY = json.load(
        f
    )


FINAL_PREDICTORS = list(
    CONFIG[
        "features"
    ][
        "final_predictor_whitelist"
    ]
)


EXPECTED_PREDICTORS = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Total_Packets",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum",

    "Total_Bytes",

    "Packets_Per_Second",

    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",

    "Forward_Backward_Byte_Ratio"
]


if FINAL_PREDICTORS != EXPECTED_PREDICTORS:

    raise RuntimeError(
        "Frozen predictor order/schema differs from expected Phase 11.4 schema."
    )


ATTACK_FAMILIES = list(
    CONFIG[
        "episode_design"
    ][
        "attack_families"
    ]
)


PRECURSOR_REGIMES = (
    CONFIG[
        "precursor_regimes"
    ]
)


MASTER_SEED = int(
    CONFIG[
        "dataset_identity"
    ][
        "master_seed"
    ]
)


DATASET_VERSION = (
    CONFIG[
        "dataset_identity"
    ][
        "version"
    ]
)


FORECAST_HORIZONS = [
    1,
    2,
    5,
    10,
    15,
    30
]


# ==================================================================================================
# 11.4F — GENERATOR PARAMETERS
#
# These are SIMULATION parameters, not claims about naturally observed
# precursor dynamics.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4F — FREEZE GENERATOR PARAMETERS")
print("=" * 120)


AR_RHO = 0.85

ATTACK_INTENSITY_AR_RHO = 0.65


LOCAL_NOISE_SIGMA = {

    "Flow_Count":
        0.060,

    "Flow_Duration_Mean":
        0.040,

    "Flow_Duration_Spread":
        0.050,

    "Forward_Packets_Per_Flow":
        0.035,

    "Backward_Packets_Per_Flow":
        0.035,

    "Forward_Bytes_Per_Flow":
        0.040,

    "Backward_Bytes_Per_Flow":
        0.040
}


# Family attack-like per-flow values are clipped to the empirical
# source/family q01–q99 flow-profile range.

ATTACK_PROFILE_LOW_QUANTILE = "q01"

ATTACK_PROFILE_HIGH_QUANTILE = "q99"


# Local BENIGN perturbations are permitted across the wider q0.1–q99.9 range.

BENIGN_PROFILE_LOW_QUANTILE = "q001"

BENIGN_PROFILE_HIGH_QUANTILE = "q999"


# Attack intensity is sampled around its empirical q50,
# and clipped to q05–q95.

INTENSITY_LOW_QUANTILE = "q05"

INTENSITY_HIGH_QUANTILE = "q95"


# Primitive flow-profile dimensions actually USED to construct attack-like traffic.

GENERATION_EFFECT_FEATURES = {

    "Flow_Duration_Seconds":
        "Flow_Duration_Mean",

    "Forward_Packets":
        "Forward_Packets_Per_Flow",

    "Backward_Packets":
        "Backward_Packets_Per_Flow",

    "Forward_Bytes":
        "Forward_Bytes_Per_Flow",

    "Backward_Bytes":
        "Backward_Bytes_Per_Flow"
}


# Redundant flow-level quantities are NOT independently altered.

VALIDATION_ONLY_EFFECT_FEATURES = [

    "Total_Packets_Per_Flow",

    "Total_Bytes_Per_Flow",

    "Packet_Direction_Ratio_Per_Flow",

    "Byte_Direction_Ratio_Per_Flow"
]


generator_spec = {

    "phase":
        "11.4",

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        DATASET_VERSION,

    "phase_11_0_config_sha256":
        CURRENT_CONFIG_HASH,

    "phase_11_2_policy_sha256":
        CURRENT_POLICY_HASH,

    "phase_11_3_manifest_sha256":
        CURRENT_MANIFEST_HASH,

    "phase_11_3_locked_test_sha256":
        CURRENT_TEST_HASH,

    "forecast_horizons_minutes":
        FORECAST_HORIZONS,

    "baseline_perturbation": {

        "type":
            "TEMPORALLY_CORRELATED_LOG_SPACE_MULTIPLICATIVE",

        "ar1_rho":
            AR_RHO,

        "feature_sigma":
            LOCAL_NOISE_SIGMA,

        "shared_common_factor":
            0.60,

        "feature_specific_factor":
            0.80
    },

    "attack_intensity": {

        "model":
            "LOG1P_AR1_FROM_EMPIRICAL_ACTIVE_MINUTE_QUANTILES",

        "ar1_rho":
            ATTACK_INTENSITY_AR_RHO,

        "clip_low":
            INTENSITY_LOW_QUANTILE,

        "clip_high":
            INTENSITY_HIGH_QUANTILE
    },

    "effect_generation_features":
        list(
            GENERATION_EFFECT_FEATURES.keys()
        ),

    "effect_validation_only_features":
        VALIDATION_ONLY_EFFECT_FEATURES,

    "attack_profile_clip": {

        "low":
            ATTACK_PROFILE_LOW_QUANTILE,

        "high":
            ATTACK_PROFILE_HIGH_QUANTILE
    },

    "benign_profile_clip": {

        "low":
            BENIGN_PROFILE_LOW_QUANTILE,

        "high":
            BENIGN_PROFILE_HIGH_QUANTILE
    },

    "precursor_claim":
        (
            "CONTROLLED_SIMULATION_ASSUMPTION_NOT_EMPIRICALLY_ESTABLISHED "
            "AS_REAL_PRE_ATTACK_DYNAMICS_BY_CIC_OR_UNSW"
        ),

    "dependent_feature_policy":
        "RECOMPUTE_FROM_PRIMITIVES",

    "timestamp_as_predictor":
        False,

    "metadata_as_predictor":
        False,

    "synthetic_test_regeneration_after_model_results":
        False
}


GENERATOR_SPEC_FILE = (
    GENERATED_DIR
    / "11_4_generator_spec.json"
)


with open(
    GENERATOR_SPEC_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        generator_spec,
        f,
        indent=4
    )


GENERATOR_SPEC_SHA256 = sha256_file(
    GENERATOR_SPEC_FILE
)


GENERATOR_SPEC_HASH_FILE = (
    GENERATED_DIR
    / "11_4_generator_spec_SHA256.txt"
)


GENERATOR_SPEC_HASH_FILE.write_text(
    GENERATOR_SPEC_SHA256
    +
    "\n"
)


print(
    "Generator spec SHA256:"
)

print(
    GENERATOR_SPEC_SHA256
)


# ==================================================================================================
# 11.4G — LOAD 11.3 MANIFEST + TEST LOCK
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4G — LOAD FROZEN EPISODE MANIFEST")
print("=" * 120)


episode_manifest = pd.read_csv(
    EPISODE_MANIFEST_FILE,
    low_memory=False
)


locked_test_manifest = pd.read_csv(
    LOCKED_TEST_MANIFEST_FILE,
    low_memory=False
)


for column in [

    "Episode_Start_UTC",
    "Episode_End_UTC",
    "Attack_Onset_UTC",
    "Attack_End_UTC",
    "Precursor_Start_UTC",
    "Precursor_End_UTC",
    "Recovery_Start_UTC",
    "Recovery_End_UTC"

]:

    if column in episode_manifest.columns:

        episode_manifest[
            column
        ] = pd.to_datetime(

            episode_manifest[
                column
            ],

            utc=True,

            errors="coerce"
        )


print(
    "Episodes:",
    len(
        episode_manifest
    )
)


print(
    "Locked test episodes:",
    len(
        locked_test_manifest
    )
)


if len(
    episode_manifest
) != 1200:

    raise RuntimeError(
        "Frozen manifest no longer contains 1,200 episodes."
    )


if len(
    locked_test_manifest
) != 180:

    raise RuntimeError(
        "Locked TEST manifest no longer contains 180 episodes."
    )


# Verify TEST IDs + frozen content identifiers.

full_test = (

    episode_manifest.loc[
        episode_manifest[
            "Split"
        ]
        ==
        "TEST",
        [
            "Episode_ID",
            "Generator_Seed",
            "Baseline_Block_Sequence_SHA256"
        ]
    ]

    .sort_values(
        "Episode_ID"
    )

    .reset_index(
        drop=True
    )
)


locked_test_compare = (

    locked_test_manifest[
        [
            "Episode_ID",
            "Generator_Seed",
            "Baseline_Block_Sequence_SHA256"
        ]
    ]

    .sort_values(
        "Episode_ID"
    )

    .reset_index(
        drop=True
    )
)


if not full_test.equals(
    locked_test_compare
):

    raise RuntimeError(
        "Full manifest TEST subset does not match locked TEST manifest."
    )


print(
    "✓ Locked TEST subset matches frozen full manifest"
)


# ==================================================================================================
# 11.4H — LOAD SAFE REAL BENIGN BLOCKS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4H — LOAD LEAKAGE-SAFE REAL BENIGN BLOCKS")
print("=" * 120)


safe_block_index = pd.read_csv(
    SAFE_BLOCK_INDEX_FILE,
    low_memory=False
)


safe_block_rows = pd.read_csv(
    SAFE_BLOCK_ROWS_FILE,
    low_memory=False
)


required_safe_row_columns = [

    "Calibration_Source",

    "Calibration_Pool",

    "Calibration_Group_ID",

    "Safe_Benign_Block_ID",

    "Block_Row_Position",

    *FINAL_PREDICTORS
]


missing_safe_columns = [

    c

    for c in required_safe_row_columns

    if c not in safe_block_rows.columns
]


if missing_safe_columns:

    raise RuntimeError(
        f"Safe block rows missing columns: {missing_safe_columns}"
    )


for feature in FINAL_PREDICTORS:

    safe_block_rows[
        feature
    ] = pd.to_numeric(

        safe_block_rows[
            feature
        ],

        errors="coerce"
    )


if safe_block_rows[
    FINAL_PREDICTORS
].isna().any().any():

    raise RuntimeError(
        "Safe benign calibration blocks contain NaN predictors."
    )


block_size_check = (

    safe_block_rows

    .groupby(
        "Safe_Benign_Block_ID"
    )

    .size()
)


if not (
    block_size_check
    ==
    5
).all():

    raise RuntimeError(
        "Every safe benign block must contain exactly five minute rows."
    )


block_lookup = {}


block_meta_lookup = {}


for block_id, block in safe_block_rows.groupby(
    "Safe_Benign_Block_ID",
    observed=True
):

    block = (

        block

        .sort_values(
            "Block_Row_Position"
        )

        .reset_index(
            drop=True
        )
    )


    block_lookup[
        block_id
    ] = block[
        FINAL_PREDICTORS
    ].copy()


    block_meta_lookup[
        block_id
    ] = {

        "Calibration_Source":
            block[
                "Calibration_Source"
            ].iloc[0],

        "Calibration_Pool":
            block[
                "Calibration_Pool"
            ].iloc[0],

        "Calibration_Group_ID":
            block[
                "Calibration_Group_ID"
            ].iloc[0]
    }


print(
    "Safe blocks loaded:",
    len(
        block_lookup
    )
)


# ==================================================================================================
# 11.4I — VERIFY 11.3 BLOCK USAGE CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4I — VERIFY 11.3 BLOCK-USAGE CONTRACT")
print("=" * 120)


block_usage_11_3 = pd.read_csv(
    BLOCK_USAGE_11_3_FILE,
    low_memory=False
)


usage_counts = (

    block_usage_11_3

    .groupby(
        "Episode_ID"
    )

    .size()
)


manifest_block_counts = (

    episode_manifest

    .set_index(
        "Episode_ID"
    )[
        "Baseline_Block_Count"
    ]

    .astype(
        int
    )
)


usage_counts = usage_counts.reindex(
    manifest_block_counts.index
)


if usage_counts.isna().any():

    raise RuntimeError(
        "Some manifest episodes are missing 11.3 block-usage rows."
    )


if not np.array_equal(

    usage_counts.to_numpy(
        dtype=int
    ),

    manifest_block_counts.to_numpy(
        dtype=int
    )
):

    raise RuntimeError(
        "11.3 block-usage counts disagree with frozen manifest."
    )


print(
    "✓ 11.3 block-usage contract verified"
)


# ==================================================================================================
# 11.4J — LOAD SOURCE/FAMILY EFFECTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4J — LOAD SOURCE-SPECIFIC FAMILY EFFECTS")
print("=" * 120)


source_effects = pd.read_csv(
    SOURCE_EFFECT_FILE
)


source_effects[
    "Effect_Log1p"
] = pd.to_numeric(
    source_effects[
        "Effect_Log1p"
    ],
    errors="coerce"
)


effect_lookup = {}


for _, row in source_effects.iterrows():

    effect_lookup[
        (
            row[
                "Calibration_Source"
            ],

            row[
                "Synthetic_Family"
            ],

            row[
                "Feature"
            ]
        )
    ] = float(
        row[
            "Effect_Log1p"
        ]
    )


# Validate all manifest-used source/family pairs.

used_family_source_pairs = (

    episode_manifest.loc[
        episode_manifest[
            "Episode_Type"
        ]
        ==
        "ATTACK",
        [
            "Calibration_Source",
            "Attack_Family"
        ]
    ]

    .drop_duplicates()
)


for _, row in used_family_source_pairs.iterrows():

    source = row[
        "Calibration_Source"
    ]


    family = row[
        "Attack_Family"
    ]


    for effect_feature in GENERATION_EFFECT_FEATURES.keys():

        key = (
            source,
            family,
            effect_feature
        )


        if key not in effect_lookup:

            raise RuntimeError(
                f"Missing source-specific effect: {key}"
            )


print(
    "✓ All used source/family primitive effects available"
)


# ==================================================================================================
# 11.4K — LOAD REAL FLOW-PROFILE QUANTILES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4K — LOAD FLOW-PROFILE REALISM BOUNDS")
print("=" * 120)


flow_profile_quantiles = pd.read_csv(
    FLOW_PROFILE_QUANTILES_FILE
)


profile_quantile_lookup = {}


for _, row in flow_profile_quantiles.iterrows():

    key = (
        row[
            "Calibration_Source"
        ],

        row[
            "Profile_Class"
        ],

        row[
            "Feature"
        ]
    )


    profile_quantile_lookup[
        key
    ] = {

        q:
            float(
                row[
                    q
                ]
            )

        for q in [
            "q001",
            "q01",
            "q05",
            "q25",
            "q50",
            "q75",
            "q95",
            "q99",
            "q999"
        ]
    }


def profile_bounds(
    source,
    profile_class,
    feature,
    low_q,
    high_q
):

    key = (
        source,
        profile_class,
        feature
    )


    if key not in profile_quantile_lookup:

        raise RuntimeError(
            f"Missing flow-profile quantiles: {key}"
        )


    q = profile_quantile_lookup[
        key
    ]


    low = max(
        0.0,
        float(
            q[
                low_q
            ]
        )
    )


    high = max(
        low,
        float(
            q[
                high_q
            ]
        )
    )


    return (
        low,
        high
    )


# ==================================================================================================
# 11.4L — LOAD MINUTE-LEVEL REALISM BOUNDS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4L — LOAD SOURCE MINUTE REALISM BOUNDS")
print("=" * 120)


minute_quantiles = pd.read_csv(
    REAL_MINUTE_QUANTILES_FILE
)


minute_quantile_lookup = {}


for _, row in minute_quantiles.iterrows():

    key = (
        row[
            "Calibration_Group"
        ],

        row[
            "Feature"
        ]
    )


    minute_quantile_lookup[
        key
    ] = {

        q:
            float(
                row[
                    q
                ]
            )

        for q in [
            "q001",
            "q01",
            "q05",
            "q25",
            "q50",
            "q75",
            "q95",
            "q99",
            "q999"
        ]
    }


def minute_bounds(
    source,
    state_group,
    feature,
    low_q="q001",
    high_q="q999"
):

    key = (
        f"{source}::{state_group}",
        feature
    )


    if key not in minute_quantile_lookup:

        raise RuntimeError(
            f"Missing minute quantile bounds: {key}"
        )


    q = minute_quantile_lookup[
        key
    ]


    low = max(
        0.0,
        float(
            q[
                low_q
            ]
        )
    )


    high = max(
        low,
        float(
            q[
                high_q
            ]
        )
    )


    return (
        low,
        high
    )


# ==================================================================================================
# 11.4M — LOAD ATTACK INTENSITY DISTRIBUTIONS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4M — LOAD SOURCE/FAMILY ATTACK INTENSITY DISTRIBUTIONS")
print("=" * 120)


family_intensity = pd.read_csv(
    FAMILY_INTENSITY_FILE
)


intensity_lookup = {}


for _, row in family_intensity.iterrows():

    source = row[
        "Calibration_Source"
    ]


    if source == "POOLED_REAL":

        continue


    family = row[
        "Synthetic_Family"
    ]


    intensity_lookup[
        (
            source,
            family
        )
    ] = {

        q:
            float(
                row[
                    q
                ]
            )

        for q in [

            "q05",
            "q25",
            "q50",
            "q75",
            "q95"
        ]
    }


for _, row in used_family_source_pairs.iterrows():

    key = (
        row[
            "Calibration_Source"
        ],

        row[
            "Attack_Family"
        ]
    )


    if key not in intensity_lookup:

        raise RuntimeError(
            f"Missing family intensity distribution: {key}"
        )


print(
    "✓ Attack-intensity distributions available"
)


# ==================================================================================================
# 11.4N — FREEZE CALIBRATION INPUT HASHES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4N — HASH CALIBRATION INPUTS")
print("=" * 120)


calibration_input_hashes = pd.DataFrame({

    "Artifact": [

        SAFE_BLOCK_INDEX_FILE.name,

        SAFE_BLOCK_ROWS_FILE.name,

        SOURCE_EFFECT_FILE.name,

        FAMILY_INTENSITY_FILE.name,

        FLOW_PROFILE_QUANTILES_FILE.name,

        REAL_MINUTE_QUANTILES_FILE.name
    ],


    "SHA256": [

        sha256_file(
            SAFE_BLOCK_INDEX_FILE
        ),

        sha256_file(
            SAFE_BLOCK_ROWS_FILE
        ),

        sha256_file(
            SOURCE_EFFECT_FILE
        ),

        sha256_file(
            FAMILY_INTENSITY_FILE
        ),

        sha256_file(
            FLOW_PROFILE_QUANTILES_FILE
        ),

        sha256_file(
            REAL_MINUTE_QUANTILES_FILE
        )
    ]
})


print(
    calibration_input_hashes.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.4O — GENERATION FUNCTIONS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4O — INITIALIZE SYNTHETIC TRAFFIC GENERATOR")
print("=" * 120)


def safe_expm1_log_effect(
    baseline_value,
    effect_log1p,
    strength
):

    baseline_value = max(
        0.0,
        float(
            baseline_value
        )
    )


    transformed = np.expm1(

        np.log1p(
            baseline_value
        )

        +

        effect_log1p
        *
        strength
    )


    return max(
        0.0,
        float(
            transformed
        )
    )


def clip_value(
    value,
    low,
    high
):

    return float(
        np.clip(
            value,
            low,
            high
        )
    )


def create_precursor_alpha(
    regime,
    length,
    max_alpha,
    seed
):

    length = int(
        length
    )


    if (
        length <= 0
        or
        regime == "NO_PRECURSOR"
    ):

        return np.zeros(
            length,
            dtype=float
        )


    rng = np.random.default_rng(
        seed
    )


    progress = (

        np.arange(
            1,
            length + 1,
            dtype=float
        )

        /
        length
    )


    if regime in [

        "SHORT_WEAK",
        "SHORT_STRONG"
    ]:

        # Concentrates most change close to onset.

        base = progress ** 1.8


    elif regime in [

        "GRADUAL_WEAK",
        "GRADUAL_STRONG"
    ]:

        # Smooth-step trajectory.

        base = (

            3
            *
            progress**2

            -

            2
            *
            progress**3
        )


    elif regime == "BURSTY":

        burst_probability = (

            0.25

            +

            0.45
            *
            progress
        )


        burst_mask = (

            rng.random(
                length
            )

            <
            burst_probability
        )


        if not burst_mask.any():

            burst_mask[
                -1
            ] = True


        amplitudes = (

            (
                0.35

                +

                0.65
                *
                progress
            )

            *

            rng.uniform(
                0.70,
                1.00,
                size=length
            )
        )


        quiet_background = (

            0.05

            *
            progress

            *

            rng.random(
                length
            )
        )


        base = np.where(

            burst_mask,

            amplitudes,

            quiet_background
        )


    else:

        # Generic safe fallback.

        base = progress


    alpha = (

        max_alpha
        *
        base
    )


    return np.clip(
        alpha,
        0.0,
        max_alpha
    )


def create_episode_alpha_and_phase(
    manifest_row
):

    length = int(
        manifest_row[
            "Episode_Length_Minutes"
        ]
    )


    alpha = np.zeros(
        length,
        dtype=float
    )


    phase = np.full(
        length,
        "BENIGN_ONLY",
        dtype=object
    )


    if manifest_row[
        "Episode_Type"
    ] == "BENIGN_ONLY":

        return (
            alpha,
            phase
        )


    phase[:] = "BENIGN_BASELINE"


    onset = int(
        manifest_row[
            "Attack_Onset_Minute"
        ]
    )


    attack_end = int(
        manifest_row[
            "Attack_End_Minute"
        ]
    )


    recovery_start = int(
        manifest_row[
            "Recovery_Start_Minute"
        ]
    )


    recovery_end = int(
        manifest_row[
            "Recovery_End_Minute"
        ]
    )


    precursor_length = int(
        manifest_row[
            "Precursor_Length_Minutes"
        ]
    )


    regime = manifest_row[
        "Precursor_Regime"
    ]


    max_pre_alpha = float(
        manifest_row[
            "Precursor_Max_Alpha"
        ]
    )


    if precursor_length > 0:

        precursor_start = (

            onset
            -
            precursor_length
        )


        precursor_values = create_precursor_alpha(

            regime=regime,

            length=precursor_length,

            max_alpha=max_pre_alpha,

            seed=stable_seed64(
                f"{int(manifest_row['Generator_Seed'])}|PRECURSOR_ALPHA"
            )
        )


        alpha[
            precursor_start:onset
        ] = precursor_values


        phase[
            precursor_start:onset
        ] = "PRECURSOR"


    # Attack itself is fully active.

    alpha[
        onset:
        attack_end + 1
    ] = 1.0


    phase[
        onset:
        attack_end + 1
    ] = "ATTACK"


    recovery_length = (

        recovery_end
        -
        recovery_start
        +
        1
    )


    if recovery_length > 0:

        progress = (

            np.arange(
                1,
                recovery_length + 1,
                dtype=float
            )

            /

            (
                recovery_length
                +
                1
            )
        )


        # Quadratic decay.

        recovery_alpha = (

            1.0

            -

            progress
        ) ** 2


        alpha[
            recovery_start:
            recovery_end + 1
        ] = recovery_alpha


        phase[
            recovery_start:
            recovery_end + 1
        ] = "RECOVERY"


    if (
        recovery_end
        +
        1
        <
        length
    ):

        phase[
            recovery_end + 1:
        ] = "POST_RECOVERY"


    return (
        alpha,
        phase
    )


def reconstruct_episode_baseline(
    manifest_row
):

    sequence = json.loads(
        manifest_row[
            "Baseline_Block_Sequence"
        ]
    )


    frames = []


    for occurrence_index, block_id in enumerate(
        sequence
    ):

        if block_id not in block_lookup:

            raise RuntimeError(
                f"Unknown baseline block {block_id}"
            )


        block_meta = block_meta_lookup[
            block_id
        ]


        if (
            block_meta[
                "Calibration_Source"
            ]
            !=
            manifest_row[
                "Calibration_Source"
            ]
        ):

            raise RuntimeError(
                f"{manifest_row['Episode_ID']}: block source mismatch."
            )


        if (
            block_meta[
                "Calibration_Pool"
            ]
            !=
            manifest_row[
                "Calibration_Pool"
            ]
        ):

            raise RuntimeError(
                f"{manifest_row['Episode_ID']}: block pool mismatch."
            )


        block = block_lookup[
            block_id
        ].copy()


        block[
            "_Baseline_Block_ID"
        ] = block_id


        block[
            "_Calibration_Group_ID"
        ] = block_meta[
            "Calibration_Group_ID"
        ]


        block[
            "_Block_Occurrence_Index"
        ] = occurrence_index


        block[
            "_Block_Row_Position"
        ] = np.arange(
            len(
                block
            )
        )


        frames.append(
            block
        )


    baseline = pd.concat(
        frames,
        ignore_index=True
    )


    required_length = int(
        manifest_row[
            "Episode_Length_Minutes"
        ]
    )


    baseline = (

        baseline

        .iloc[
            :required_length
        ]

        .copy()

        .reset_index(
            drop=True
        )
    )


    if len(
        baseline
    ) != required_length:

        raise RuntimeError(
            f"{manifest_row['Episode_ID']}: insufficient baseline rows."
        )


    return baseline


def get_effect(
    source,
    family,
    feature
):

    return effect_lookup[
        (
            source,
            family,
            feature
        )
    ]


def get_profile_clip(
    source,
    profile_class,
    feature,
    attack_profile=False
):

    if attack_profile:

        return profile_bounds(

            source,

            profile_class,

            feature,

            ATTACK_PROFILE_LOW_QUANTILE,

            ATTACK_PROFILE_HIGH_QUANTILE
        )


    return profile_bounds(

        source,

        profile_class,

        feature,

        BENIGN_PROFILE_LOW_QUANTILE,

        BENIGN_PROFILE_HIGH_QUANTILE
    )


def get_intensity_parameters(
    source,
    family
):

    q = intensity_lookup[
        (
            source,
            family
        )
    ]


    q25 = max(
        0.0,
        q[
            "q25"
        ]
    )


    q50 = max(
        0.0,
        q[
            "q50"
        ]
    )


    q75 = max(
        0.0,
        q[
            "q75"
        ]
    )


    # Approximate lognormal sigma from IQR.

    sigma = (

        np.log1p(
            q75
        )

        -

        np.log1p(
            q25
        )

    ) / (

        2
        *
        0.67448975
    )


    sigma = float(
        np.clip(
            sigma,
            0.05,
            2.0
        )
    )


    return {

        "q05":
            max(
                0.0,
                q[
                    "q05"
                ]
            ),

        "q50":
            q50,

        "q95":
            max(
                q50,
                q[
                    "q95"
                ]
            ),

        "log_sigma":
            sigma
    }


# ==================================================================================================
# 11.4P — GENERATE EPISODES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4P — GENERATE ALL SYNTHETIC MINUTE TRAFFIC")
print("=" * 120)


predictor_frames = []

target_frames = []

metadata_frames = []

episode_summary_rows = []

block_occurrence_hash_rows = []


expected_total_rows = int(

    episode_manifest[
        "Episode_Length_Minutes"
    ].sum()
)


print(
    "Expected total minute rows:",
    f"{expected_total_rows:,}"
)


for episode_number, manifest_row in episode_manifest.iterrows():

    episode_id = manifest_row[
        "Episode_ID"
    ]


    episode_length = int(
        manifest_row[
            "Episode_Length_Minutes"
        ]
    )


    source = manifest_row[
        "Calibration_Source"
    ]


    split = manifest_row[
        "Split"
    ]


    episode_type = manifest_row[
        "Episode_Type"
    ]


    family = manifest_row[
        "Attack_Family"
    ]


    generator_seed = int(
        manifest_row[
            "Generator_Seed"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # Empirical baseline
    # ----------------------------------------------------------------------------------------------

    baseline = reconstruct_episode_baseline(
        manifest_row
    )


    # ----------------------------------------------------------------------------------------------
    # Controlled attack influence curve
    # ----------------------------------------------------------------------------------------------

    influence_alpha, phase = create_episode_alpha_and_phase(
        manifest_row
    )


    # ----------------------------------------------------------------------------------------------
    # Independent deterministic random streams
    # ----------------------------------------------------------------------------------------------

    noise_rng = np.random.default_rng(

        stable_seed64(
            f"{generator_seed}|LOCAL_NOISE"
        )
    )


    intensity_rng = np.random.default_rng(

        stable_seed64(
            f"{generator_seed}|ATTACK_INTENSITY"
        )
    )


    # ----------------------------------------------------------------------------------------------
    # AR(1) states
    # ----------------------------------------------------------------------------------------------

    noise_dimensions = 7


    local_state = np.zeros(
        noise_dimensions,
        dtype=float
    )


    common_state = 0.0


    intensity_state = 0.0


    local_innovation_scale = math.sqrt(
        1.0
        -
        AR_RHO**2
    )


    intensity_innovation_scale = math.sqrt(
        1.0
        -
        ATTACK_INTENSITY_AR_RHO**2
    )


    # ----------------------------------------------------------------------------------------------
    # Source bounds
    # ----------------------------------------------------------------------------------------------

    flow_count_low, flow_count_high = minute_bounds(

        source,

        "BENIGN",

        "Flow_Count",

        "q001",

        "q999"
    )


    duration_std_low, duration_std_high = minute_bounds(

        source,

        "ALL",

        "Flow_Duration_Std",

        "q001",

        "q999"
    )


    duration_max_low, duration_max_high = minute_bounds(

        source,

        "ALL",

        "Flow_Duration_Max",

        "q001",

        "q999"
    )


    # ----------------------------------------------------------------------------------------------
    # Family intensity model
    # ----------------------------------------------------------------------------------------------

    if episode_type == "ATTACK":

        intensity_params = get_intensity_parameters(

            source,
            family
        )


        family_strength = float(
            manifest_row[
                "Family_Effect_Strength_Multiplier"
            ]
        )


    else:

        intensity_params = None

        family_strength = 0.0


    # ----------------------------------------------------------------------------------------------
    # Episode outputs
    # ----------------------------------------------------------------------------------------------

    generated_predictors = {

        predictor:
            np.zeros(
                episode_length,
                dtype=float
            )

        for predictor
        in FINAL_PREDICTORS
    }


    influence_flow_equivalent = np.zeros(
        episode_length,
        dtype=float
    )


    synthetic_attack_flow_count = np.zeros(
        episode_length,
        dtype=int
    )


    # ----------------------------------------------------------------------------------------------
    # Generate minute by minute
    # ----------------------------------------------------------------------------------------------

    for minute_index in range(
        episode_length
    ):

        row = baseline.iloc[
            minute_index
        ]


        alpha = float(
            influence_alpha[
                minute_index
            ]
        )


        # ==========================================================================================
        # AR(1) perturbation
        # ==========================================================================================

        common_state = (

            AR_RHO
            *
            common_state

            +

            local_innovation_scale
            *
            noise_rng.normal()
        )


        local_state = (

            AR_RHO
            *
            local_state

            +

            local_innovation_scale
            *
            noise_rng.normal(
                size=noise_dimensions
            )
        )


        combined_noise = (

            0.60
            *
            common_state

            +

            0.80
            *
            local_state
        )


        # ==========================================================================================
        # Baseline primitive values
        # ==========================================================================================

        original_flow_count = max(

            float(
                row[
                    "Flow_Count"
                ]
            ),

            1.0
        )


        # Episode load multiplier acts primarily on traffic volume.

        benign_flow_count = (

            original_flow_count

            *

            float(
                manifest_row[
                    "Episode_Load_Multiplier"
                ]
            )

            *

            np.exp(
                LOCAL_NOISE_SIGMA[
                    "Flow_Count"
                ]

                *
                combined_noise[
                    0
                ]
            )
        )


        benign_flow_count = clip_value(

            benign_flow_count,

            max(
                1.0,
                flow_count_low
            ),

            max(
                1.0,
                flow_count_high
            )
        )


        # ------------------------------------------------------------------------------------------
        # Real baseline per-flow primitives
        # ------------------------------------------------------------------------------------------

        base_duration_mean = max(

            float(
                row[
                    "Flow_Duration_Mean"
                ]
            ),

            0.0
        )


        base_duration_std = max(

            float(
                row[
                    "Flow_Duration_Std"
                ]
            ),

            0.0
        )


        base_duration_max = max(

            float(
                row[
                    "Flow_Duration_Max"
                ]
            ),

            base_duration_mean
        )


        base_fwd_packets_pf = (

            float(
                row[
                    "Forward_Packets_Sum"
                ]
            )

            /
            original_flow_count
        )


        base_bwd_packets_pf = (

            float(
                row[
                    "Backward_Packets_Sum"
                ]
            )

            /
            original_flow_count
        )


        base_fwd_bytes_pf = (

            float(
                row[
                    "Forward_Bytes_Sum"
                ]
            )

            /
            original_flow_count
        )


        base_bwd_bytes_pf = (

            float(
                row[
                    "Backward_Bytes_Sum"
                ]
            )

            /
            original_flow_count
        )


        # ==========================================================================================
        # Locally perturb baseline per-flow primitives
        # ==========================================================================================

        benign_duration_mean = (

            base_duration_mean

            *

            np.exp(
                LOCAL_NOISE_SIGMA[
                    "Flow_Duration_Mean"
                ]

                *
                combined_noise[
                    1
                ]
            )
        )


        benign_duration_std = (

            base_duration_std

            *

            np.exp(
                LOCAL_NOISE_SIGMA[
                    "Flow_Duration_Spread"
                ]

                *
                combined_noise[
                    2
                ]
            )
        )


        benign_duration_max = (

            base_duration_max

            *

            np.exp(

                0.70
                *
                LOCAL_NOISE_SIGMA[
                    "Flow_Duration_Mean"
                ]
                *
                combined_noise[
                    1
                ]

                +

                0.30
                *
                LOCAL_NOISE_SIGMA[
                    "Flow_Duration_Spread"
                ]
                *
                combined_noise[
                    2
                ]
            )
        )


        benign_fwd_packets_pf = (

            base_fwd_packets_pf

            *

            np.exp(
                LOCAL_NOISE_SIGMA[
                    "Forward_Packets_Per_Flow"
                ]

                *
                combined_noise[
                    3
                ]
            )
        )


        benign_bwd_packets_pf = (

            base_bwd_packets_pf

            *

            np.exp(
                LOCAL_NOISE_SIGMA[
                    "Backward_Packets_Per_Flow"
                ]

                *
                combined_noise[
                    4
                ]
            )
        )


        benign_fwd_bytes_pf = (

            base_fwd_bytes_pf

            *

            np.exp(
                LOCAL_NOISE_SIGMA[
                    "Forward_Bytes_Per_Flow"
                ]

                *
                combined_noise[
                    5
                ]
            )
        )


        benign_bwd_bytes_pf = (

            base_bwd_bytes_pf

            *

            np.exp(
                LOCAL_NOISE_SIGMA[
                    "Backward_Bytes_Per_Flow"
                ]

                *
                combined_noise[
                    6
                ]
            )
        )


        # ==========================================================================================
        # Clip perturbed benign primitives to empirical source BENIGN bounds
        # ==========================================================================================

        for feature_name, current_value_name in [

            (
                "Flow_Duration_Seconds",
                "benign_duration_mean"
            ),

            (
                "Forward_Packets",
                "benign_fwd_packets_pf"
            ),

            (
                "Backward_Packets",
                "benign_bwd_packets_pf"
            ),

            (
                "Forward_Bytes",
                "benign_fwd_bytes_pf"
            ),

            (
                "Backward_Bytes",
                "benign_bwd_bytes_pf"
            )
        ]:

            low, high = get_profile_clip(

                source,

                "BENIGN",

                feature_name,

                attack_profile=False
            )


            current_value = locals()[
                current_value_name
            ]


            clipped = clip_value(

                current_value,

                low,

                high
            )


            if current_value_name == "benign_duration_mean":

                benign_duration_mean = clipped

            elif current_value_name == "benign_fwd_packets_pf":

                benign_fwd_packets_pf = clipped

            elif current_value_name == "benign_bwd_packets_pf":

                benign_bwd_packets_pf = clipped

            elif current_value_name == "benign_fwd_bytes_pf":

                benign_fwd_bytes_pf = clipped

            elif current_value_name == "benign_bwd_bytes_pf":

                benign_bwd_bytes_pf = clipped


        benign_duration_std = clip_value(

            benign_duration_std,

            duration_std_low,

            max(
                duration_std_low,
                duration_std_high
            )
        )


        benign_duration_max = clip_value(

            max(
                benign_duration_max,
                benign_duration_mean
            ),

            max(
                benign_duration_mean,
                duration_max_low
            ),

            max(
                benign_duration_mean,
                duration_max_high
            )
        )


        # ==========================================================================================
        # Attack / precursor / recovery influence
        # ==========================================================================================

        if (
            episode_type
            ==
            "ATTACK"

            and

            alpha
            >
            0.0
        ):

            intensity_state = (

                ATTACK_INTENSITY_AR_RHO
                *
                intensity_state

                +

                intensity_innovation_scale
                *
                intensity_rng.normal()
            )


            raw_intensity = np.expm1(

                np.log1p(
                    intensity_params[
                        "q50"
                    ]
                )

                +

                intensity_params[
                    "log_sigma"
                ]
                *
                intensity_state
            )


            full_attack_intensity = clip_value(

                raw_intensity,

                intensity_params[
                    "q05"
                ],

                intensity_params[
                    "q95"
                ]
            )


            full_attack_intensity *= (
                family_strength
            )


            equivalent_attack_flows = (

                alpha

                *
                full_attack_intensity
            )


            influence_flow_equivalent[
                minute_index
            ] = equivalent_attack_flows


            if phase[
                minute_index
            ] == "ATTACK":

                synthetic_attack_flow_count[
                    minute_index
                ] = int(
                    max(
                        1,
                        round(
                            full_attack_intensity
                        )
                    )
                )


            # --------------------------------------------------------------------------------------
            # Construct attack-like per-flow primitive profile
            # --------------------------------------------------------------------------------------

            attack_duration_mean = safe_expm1_log_effect(

                benign_duration_mean,

                get_effect(
                    source,
                    family,
                    "Flow_Duration_Seconds"
                ),

                family_strength
            )


            attack_fwd_packets_pf = safe_expm1_log_effect(

                benign_fwd_packets_pf,

                get_effect(
                    source,
                    family,
                    "Forward_Packets"
                ),

                family_strength
            )


            attack_bwd_packets_pf = safe_expm1_log_effect(

                benign_bwd_packets_pf,

                get_effect(
                    source,
                    family,
                    "Backward_Packets"
                ),

                family_strength
            )


            attack_fwd_bytes_pf = safe_expm1_log_effect(

                benign_fwd_bytes_pf,

                get_effect(
                    source,
                    family,
                    "Forward_Bytes"
                ),

                family_strength
            )


            attack_bwd_bytes_pf = safe_expm1_log_effect(

                benign_bwd_bytes_pf,

                get_effect(
                    source,
                    family,
                    "Backward_Bytes"
                ),

                family_strength
            )


            # --------------------------------------------------------------------------------------
            # Clip attack-like primitive profile to empirical source/family flow-profile bounds
            # --------------------------------------------------------------------------------------

            primitive_values = {

                "Flow_Duration_Seconds":
                    attack_duration_mean,

                "Forward_Packets":
                    attack_fwd_packets_pf,

                "Backward_Packets":
                    attack_bwd_packets_pf,

                "Forward_Bytes":
                    attack_fwd_bytes_pf,

                "Backward_Bytes":
                    attack_bwd_bytes_pf
            }


            for feature_name in list(
                primitive_values.keys()
            ):

                low, high = get_profile_clip(

                    source,

                    family,

                    feature_name,

                    attack_profile=True
                )


                primitive_values[
                    feature_name
                ] = clip_value(

                    primitive_values[
                        feature_name
                    ],

                    low,

                    high
                )


            attack_duration_mean = primitive_values[
                "Flow_Duration_Seconds"
            ]


            attack_fwd_packets_pf = primitive_values[
                "Forward_Packets"
            ]


            attack_bwd_packets_pf = primitive_values[
                "Backward_Packets"
            ]


            attack_fwd_bytes_pf = primitive_values[
                "Forward_Bytes"
            ]


            attack_bwd_bytes_pf = primitive_values[
                "Backward_Bytes"
            ]


            # --------------------------------------------------------------------------------------
            # Duration spread for attack-like component
            # --------------------------------------------------------------------------------------

            duration_scale_ratio = (

                (
                    attack_duration_mean
                    +
                    1e-9
                )

                /

                (
                    benign_duration_mean
                    +
                    1e-9
                )
            )


            duration_scale_ratio = float(
                np.clip(
                    duration_scale_ratio,
                    0.25,
                    4.0
                )
            )


            attack_duration_std = (

                benign_duration_std

                *
                duration_scale_ratio
            )


            attack_duration_max = max(

                attack_duration_mean,

                benign_duration_max
                *
                duration_scale_ratio
            )


        else:

            equivalent_attack_flows = 0.0


            attack_duration_mean = benign_duration_mean

            attack_duration_std = benign_duration_std

            attack_duration_max = benign_duration_max

            attack_fwd_packets_pf = benign_fwd_packets_pf

            attack_bwd_packets_pf = benign_bwd_packets_pf

            attack_fwd_bytes_pf = benign_fwd_bytes_pf

            attack_bwd_bytes_pf = benign_bwd_bytes_pf


        # ==========================================================================================
        # Mixture of background + attack-like influence component
        # ==========================================================================================

        benign_component_flows = max(
            benign_flow_count,
            1e-9
        )


        influence_component_flows = max(
            equivalent_attack_flows,
            0.0
        )


        total_flow_equivalent = (

            benign_component_flows

            +

            influence_component_flows
        )


        benign_weight = (

            benign_component_flows

            /
            total_flow_equivalent
        )


        influence_weight = (

            influence_component_flows

            /
            total_flow_equivalent
        )


        duration_mean = (

            benign_weight
            *
            benign_duration_mean

            +

            influence_weight
            *
            attack_duration_mean
        )


        duration_variance = (

            benign_weight

            *
            (
                benign_duration_std**2

                +

                (
                    benign_duration_mean
                    -
                    duration_mean
                ) ** 2
            )

            +

            influence_weight

            *
            (
                attack_duration_std**2

                +

                (
                    attack_duration_mean
                    -
                    duration_mean
                ) ** 2
            )
        )


        duration_std = math.sqrt(
            max(
                0.0,
                duration_variance
            )
        )


        duration_max = max(

            duration_mean,

            benign_duration_max,

            (
                attack_duration_max

                if influence_component_flows > 0

                else benign_duration_max
            )
        )


        fwd_packets_total = (

            benign_component_flows
            *
            benign_fwd_packets_pf

            +

            influence_component_flows
            *
            attack_fwd_packets_pf
        )


        bwd_packets_total = (

            benign_component_flows
            *
            benign_bwd_packets_pf

            +

            influence_component_flows
            *
            attack_bwd_packets_pf
        )


        fwd_bytes_total = (

            benign_component_flows
            *
            benign_fwd_bytes_pf

            +

            influence_component_flows
            *
            attack_fwd_bytes_pf
        )


        bwd_bytes_total = (

            benign_component_flows
            *
            benign_bwd_bytes_pf

            +

            influence_component_flows
            *
            attack_bwd_bytes_pf
        )


        # ==========================================================================================
        # Integer-like aggregate quantities
        # ==========================================================================================

        flow_count = int(
            max(
                1,
                round(
                    total_flow_equivalent
                )
            )
        )


        fwd_packets_total = int(
            max(
                0,
                round(
                    fwd_packets_total
                )
            )
        )


        bwd_packets_total = int(
            max(
                0,
                round(
                    bwd_packets_total
                )
            )
        )


        fwd_bytes_total = int(
            max(
                0,
                round(
                    fwd_bytes_total
                )
            )
        )


        bwd_bytes_total = int(
            max(
                0,
                round(
                    bwd_bytes_total
                )
            )
        )


        total_packets = (

            fwd_packets_total

            +

            bwd_packets_total
        )


        total_bytes = (

            fwd_bytes_total

            +

            bwd_bytes_total
        )


        packets_per_second = (

            total_packets
            /
            60.0
        )


        bytes_per_second = (

            total_bytes
            /
            60.0
        )


        packet_ratio = (

            fwd_packets_total

            /

            (
                bwd_packets_total
                +
                1.0
            )
        )


        byte_ratio = (

            fwd_bytes_total

            /

            (
                bwd_bytes_total
                +
                1.0
            )
        )


        # ==========================================================================================
        # Store final 14 predictors
        # ==========================================================================================

        generated_predictors[
            "Flow_Count"
        ][
            minute_index
        ] = flow_count


        generated_predictors[
            "Flow_Duration_Mean"
        ][
            minute_index
        ] = max(
            0.0,
            duration_mean
        )


        generated_predictors[
            "Flow_Duration_Std"
        ][
            minute_index
        ] = max(
            0.0,
            duration_std
        )


        generated_predictors[
            "Flow_Duration_Max"
        ][
            minute_index
        ] = max(
            duration_mean,
            duration_max
        )


        generated_predictors[
            "Forward_Packets_Sum"
        ][
            minute_index
        ] = fwd_packets_total


        generated_predictors[
            "Backward_Packets_Sum"
        ][
            minute_index
        ] = bwd_packets_total


        generated_predictors[
            "Total_Packets"
        ][
            minute_index
        ] = total_packets


        generated_predictors[
            "Forward_Bytes_Sum"
        ][
            minute_index
        ] = fwd_bytes_total


        generated_predictors[
            "Backward_Bytes_Sum"
        ][
            minute_index
        ] = bwd_bytes_total


        generated_predictors[
            "Total_Bytes"
        ][
            minute_index
        ] = total_bytes


        generated_predictors[
            "Packets_Per_Second"
        ][
            minute_index
        ] = packets_per_second


        generated_predictors[
            "Bytes_Per_Second"
        ][
            minute_index
        ] = bytes_per_second


        generated_predictors[
            "Forward_Backward_Packet_Ratio"
        ][
            minute_index
        ] = packet_ratio


        generated_predictors[
            "Forward_Backward_Byte_Ratio"
        ][
            minute_index
        ] = byte_ratio


    # ==============================================================================================
    # Build episode predictor dataframe
    # ==============================================================================================

    minute_index = np.arange(
        episode_length,
        dtype=int
    )


    timestamps = (

        pd.Timestamp(
            manifest_row[
                "Episode_Start_UTC"
            ]
        )

        +

        pd.to_timedelta(
            minute_index,
            unit="min"
        )
    )


    predictor_df = pd.DataFrame({

        "Episode_ID":
            episode_id,

        "Minute_Index":
            minute_index
    })


    for predictor in FINAL_PREDICTORS:

        predictor_df[
            predictor
        ] = generated_predictors[
            predictor
        ]


    # Integer aggregate columns.

    for integer_column in [

        "Flow_Count",

        "Forward_Packets_Sum",

        "Backward_Packets_Sum",

        "Total_Packets",

        "Forward_Bytes_Sum",

        "Backward_Bytes_Sum",

        "Total_Bytes"

    ]:

        predictor_df[
            integer_column
        ] = (

            predictor_df[
                integer_column
            ]

            .round()

            .astype(
                np.int64
            )
        )


    # ==============================================================================================
    # Targets
    # ==============================================================================================

    if episode_type == "ATTACK":

        onset = int(
            manifest_row[
                "Attack_Onset_Minute"
            ]
        )


        attack_end = int(
            manifest_row[
                "Attack_End_Minute"
            ]
        )


        attack_state = (

            (
                minute_index
                >=
                onset
            )

            &

            (
                minute_index
                <=
                attack_end
            )
        ).astype(
            np.int8
        )


        new_attack_onset = (

            minute_index
            ==
            onset
        ).astype(
            np.int8
        )


        minutes_to_next_onset = np.where(

            minute_index
            <
            onset,

            onset
            -
            minute_index,

            np.nan
        )


        clean_preonset_origin = (

            minute_index
            <
            onset
        ).astype(
            np.int8
        )


        current_family = np.where(

            attack_state
            ==
            1,

            family,

            "BENIGN"
        )


    else:

        attack_state = np.zeros(
            episode_length,
            dtype=np.int8
        )


        new_attack_onset = np.zeros(
            episode_length,
            dtype=np.int8
        )


        minutes_to_next_onset = np.full(
            episode_length,
            np.nan
        )


        clean_preonset_origin = np.ones(
            episode_length,
            dtype=np.int8
        )


        current_family = np.full(
            episode_length,
            "BENIGN",
            dtype=object
        )


    target_df = pd.DataFrame({

        "Episode_ID":
            episode_id,

        "Minute_Index":
            minute_index,

        "Attack_State":
            attack_state,

        "New_Attack_Onset":
            new_attack_onset,

        "Current_Attack_Family":
            current_family,

        "Episode_Attack_Family":
            (
                family
                if episode_type == "ATTACK"
                else "NONE"
            ),

        "Minutes_To_Next_Onset":
            minutes_to_next_onset,

        "Clean_PreOnset_Origin":
            clean_preonset_origin,

        "Synthetic_Attack_Flow_Count":
            synthetic_attack_flow_count
    })


    # ==============================================================================================
    # Strict right-censored horizon targets
    # ==============================================================================================

    for horizon in FORECAST_HORIZONS:

        full_future_exists = (

            minute_index
            +
            horizon

            <
            episode_length
        )


        current_benign = (

            attack_state
            ==
            0
        )


        eligible = (

            current_benign

            &

            full_future_exists
        )


        label = np.full(
            episode_length,
            np.nan
        )


        family_label = np.full(
            episode_length,
            "",
            dtype=object
        )


        if episode_type == "ATTACK":

            positive = (

                eligible

                &

                (
                    onset
                    >
                    minute_index
                )

                &

                (
                    onset
                    <=
                    (
                        minute_index
                        +
                        horizon
                    )
                )
            )


        else:

            positive = np.zeros(
                episode_length,
                dtype=bool
            )


        label[
            eligible
        ] = 0.0


        label[
            positive
        ] = 1.0


        family_label[
            positive
        ] = family


        target_df[
            f"Forecast_Eligible_H{horizon}"
        ] = eligible.astype(
            np.int8
        )


        target_df[
            f"Right_Censored_H{horizon}"
        ] = (

            current_benign

            &

            ~full_future_exists
        ).astype(
            np.int8
        )


        target_df[
            f"Onset_Within_H{horizon}"
        ] = label


        target_df[
            f"Onset_Family_H{horizon}"
        ] = family_label


    # ==============================================================================================
    # Metadata
    # ==============================================================================================

    metadata_df = pd.DataFrame({

        "Episode_ID":
            episode_id,

        "Minute_Index":
            minute_index,

        "Timestamp_UTC":
            timestamps,

        "Split":
            split,

        "Episode_Type":
            episode_type,

        "Episode_Attack_Family":
            family,

        "Calibration_Source":
            source,

        "Calibration_Pool":
            manifest_row[
                "Calibration_Pool"
            ],

        "Precursor_Regime":
            manifest_row[
                "Precursor_Regime"
            ],

        "Synthetic_Phase":
            phase,

        "Attack_Influence_Alpha":
            influence_alpha,

        "Influence_Flow_Equivalent":
            influence_flow_equivalent,

        "Baseline_Block_ID":
            baseline[
                "_Baseline_Block_ID"
            ].to_numpy(),

        "Calibration_Group_ID":
            baseline[
                "_Calibration_Group_ID"
            ].to_numpy(),

        "Block_Occurrence_Index":
            baseline[
                "_Block_Occurrence_Index"
            ].to_numpy(),

        "Block_Row_Position":
            baseline[
                "_Block_Row_Position"
            ].to_numpy(),

        "Episode_Load_Multiplier":
            float(
                manifest_row[
                    "Episode_Load_Multiplier"
                ]
            ),

        "Family_Effect_Strength_Multiplier":
            (
                float(
                    manifest_row[
                        "Family_Effect_Strength_Multiplier"
                    ]
                )

                if episode_type == "ATTACK"

                else np.nan
            ),

        "Generator_Seed":
            generator_seed,

        "Timeline_Order":
            int(
                manifest_row[
                    "Timeline_Order"
                ]
            ),

        "Simulation_Origin":
            "EMPIRICALLY_CALIBRATED_CONTROLLED_SYNTHETIC"
    })


    predictor_frames.append(
        predictor_df
    )


    target_frames.append(
        target_df
    )


    metadata_frames.append(
        metadata_df
    )


    # ==============================================================================================
    # Per-episode generated block-occurrence hashes
    # ==============================================================================================

    occurrence_indices = np.sort(

        metadata_df[
            "Block_Occurrence_Index"
        ].unique()
    )


    for occurrence_index in occurrence_indices:

        occurrence_mask = (

            metadata_df[
                "Block_Occurrence_Index"
            ]

            ==
            occurrence_index
        )


        occurrence_predictors = predictor_df.loc[
            occurrence_mask,
            FINAL_PREDICTORS
        ].to_numpy(
            dtype=float
        )


        block_id = metadata_df.loc[
            occurrence_mask,
            "Baseline_Block_ID"
        ].iloc[0]


        block_occurrence_hash_rows.append(
            {
                "Episode_ID":
                    episode_id,

                "Split":
                    split,

                "Calibration_Source":
                    source,

                "Baseline_Block_ID":
                    block_id,

                "Block_Occurrence_Index":
                    int(
                        occurrence_index
                    ),

                "Generated_Row_Count":
                    int(
                        occurrence_mask.sum()
                    ),

                "Generated_Predictor_SHA256":
                    stable_array_hash(
                        occurrence_predictors
                    )
            }
        )


    # ==============================================================================================
    # Episode summary
    # ==============================================================================================

    episode_predictor_hash = stable_array_hash(

        predictor_df[
            FINAL_PREDICTORS
        ].to_numpy(
            dtype=float
        )
    )


    episode_summary_rows.append(
        {
            "Episode_ID":
                episode_id,

            "Split":
                split,

            "Episode_Type":
                episode_type,

            "Attack_Family":
                family,

            "Calibration_Source":
                source,

            "Minute_Rows":
                episode_length,

            "Attack_Active_Minutes":
                int(
                    attack_state.sum()
                ),

            "Precursor_Minutes":
                int(
                    (
                        phase
                        ==
                        "PRECURSOR"
                    ).sum()
                ),

            "Recovery_Minutes":
                int(
                    (
                        phase
                        ==
                        "RECOVERY"
                    ).sum()
                ),

            "Maximum_Influence_Alpha":
                float(
                    influence_alpha.max()
                ),

            "Maximum_Influence_Flow_Equivalent":
                float(
                    influence_flow_equivalent.max()
                ),

            "Episode_Predictor_SHA256":
                episode_predictor_hash
        }
    )


    if (
        episode_number
        +
        1
    ) % 100 == 0:

        print(
            f"Generated "
            f"{episode_number + 1:,}"
            f"/{len(episode_manifest):,} episodes"
        )


# ==================================================================================================
# 11.4Q — COMBINE GENERATED TABLES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4Q — COMBINE GENERATED OUTPUTS")
print("=" * 120)


predictors = pd.concat(
    predictor_frames,
    ignore_index=True
)


targets = pd.concat(
    target_frames,
    ignore_index=True
)


metadata = pd.concat(
    metadata_frames,
    ignore_index=True
)


episode_generation_summary = pd.DataFrame(
    episode_summary_rows
)


block_occurrence_hashes = pd.DataFrame(
    block_occurrence_hash_rows
)


del predictor_frames

del target_frames

del metadata_frames

gc.collect()


print(
    "Predictor rows:",
    f"{len(predictors):,}"
)

print(
    "Target rows:",
    f"{len(targets):,}"
)

print(
    "Metadata rows:",
    f"{len(metadata):,}"
)


# ==================================================================================================
# 11.4R — BASIC KEY INTEGRITY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4R — ROW-KEY INTEGRITY")
print("=" * 120)


key_columns = [
    "Episode_ID",
    "Minute_Index"
]


predictor_duplicate_keys = int(

    predictors

    .duplicated(
        subset=key_columns
    )

    .sum()
)


target_duplicate_keys = int(

    targets

    .duplicated(
        subset=key_columns
    )

    .sum()
)


metadata_duplicate_keys = int(

    metadata

    .duplicated(
        subset=key_columns
    )

    .sum()
)


print(
    "Predictor duplicate keys:",
    predictor_duplicate_keys
)

print(
    "Target duplicate keys:",
    target_duplicate_keys
)

print(
    "Metadata duplicate keys:",
    metadata_duplicate_keys
)


if not (
    len(
        predictors
    )
    ==
    len(
        targets
    )
    ==
    len(
        metadata
    )
    ==
    expected_total_rows
):

    raise RuntimeError(
        "Generated row counts do not match frozen manifest lengths."
    )


# ==================================================================================================
# 11.4S — PHYSICAL CONSISTENCY AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4S — PHYSICAL FEATURE CONSISTENCY")
print("=" * 120)


feature_matrix = (

    predictors[
        FINAL_PREDICTORS
    ]

    .to_numpy(
        dtype=float
    )
)


nan_cells = int(
    np.isnan(
        feature_matrix
    ).sum()
)


inf_cells = int(
    np.isinf(
        feature_matrix
    ).sum()
)


negative_cells = int(
    (
        feature_matrix
        <
        0
    ).sum()
)


total_packet_failures = int(

    (
        predictors[
            "Total_Packets"
        ]

        !=

        (
            predictors[
                "Forward_Packets_Sum"
            ]

            +

            predictors[
                "Backward_Packets_Sum"
            ]
        )
    ).sum()
)


total_byte_failures = int(

    (
        predictors[
            "Total_Bytes"
        ]

        !=

        (
            predictors[
                "Forward_Bytes_Sum"
            ]

            +

            predictors[
                "Backward_Bytes_Sum"
            ]
        )
    ).sum()
)


packet_rate_failures = int(

    (
        ~np.isclose(

            predictors[
                "Packets_Per_Second"
            ],

            predictors[
                "Total_Packets"
            ]
            /
            60.0,

            rtol=1e-12,
            atol=1e-12
        )
    ).sum()
)


byte_rate_failures = int(

    (
        ~np.isclose(

            predictors[
                "Bytes_Per_Second"
            ],

            predictors[
                "Total_Bytes"
            ]
            /
            60.0,

            rtol=1e-12,
            atol=1e-12
        )
    ).sum()
)


packet_ratio_failures = int(

    (
        ~np.isclose(

            predictors[
                "Forward_Backward_Packet_Ratio"
            ],

            predictors[
                "Forward_Packets_Sum"
            ]

            /

            (
                predictors[
                    "Backward_Packets_Sum"
                ]

                +
                1.0
            ),

            rtol=1e-12,
            atol=1e-12
        )
    ).sum()
)


byte_ratio_failures = int(

    (
        ~np.isclose(

            predictors[
                "Forward_Backward_Byte_Ratio"
            ],

            predictors[
                "Forward_Bytes_Sum"
            ]

            /

            (
                predictors[
                    "Backward_Bytes_Sum"
                ]

                +
                1.0
            ),

            rtol=1e-12,
            atol=1e-12
        )
    ).sum()
)


duration_order_failures = int(

    (
        predictors[
            "Flow_Duration_Max"
        ]

        <

        predictors[
            "Flow_Duration_Mean"
        ]
    ).sum()
)


nonpositive_flow_count = int(

    (
        predictors[
            "Flow_Count"
        ]

        <=
        0
    ).sum()
)


physical_consistency = pd.DataFrame({

    "Check": [

        "NaN predictor cells",

        "Infinite predictor cells",

        "Negative predictor cells",

        "Total_Packets mismatch",

        "Total_Bytes mismatch",

        "Packets_Per_Second mismatch",

        "Bytes_Per_Second mismatch",

        "Packet ratio mismatch",

        "Byte ratio mismatch",

        "Flow_Duration_Max < Mean",

        "Flow_Count <= 0"
    ],


    "Failure_Count": [

        nan_cells,

        inf_cells,

        negative_cells,

        total_packet_failures,

        total_byte_failures,

        packet_rate_failures,

        byte_rate_failures,

        packet_ratio_failures,

        byte_ratio_failures,

        duration_order_failures,

        nonpositive_flow_count
    ]
})


print(
    physical_consistency.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.4T — TEMPORAL / TIMESTAMP AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4T — TEMPORAL CONTINUITY AUDIT")
print("=" * 120)


metadata[
    "Timestamp_UTC"
] = pd.to_datetime(
    metadata[
        "Timestamp_UTC"
    ],
    utc=True
)


timestamp_gap_failures = 0

minute_index_failures = 0


for episode_id, group in metadata.groupby(
    "Episode_ID",
    observed=True
):

    group = group.sort_values(
        "Minute_Index"
    )


    expected_index = np.arange(
        len(
            group
        )
    )


    if not np.array_equal(

        group[
            "Minute_Index"
        ].to_numpy(),

        expected_index
    ):

        minute_index_failures += 1


    if len(
        group
    ) > 1:

        gaps = (

            group[
                "Timestamp_UTC"
            ]

            .diff()

            .dt.total_seconds()

            .iloc[
                1:
            ]
        )


        if not (
            gaps
            ==
            60.0
        ).all():

            timestamp_gap_failures += 1


print(
    "Episodes with bad minute index:",
    minute_index_failures
)

print(
    "Episodes with bad timestamp spacing:",
    timestamp_gap_failures
)


# ==================================================================================================
# 11.4U — MANIFEST TIMELINE / LABEL CONSISTENCY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4U — MANIFEST vs GENERATED LABEL CONSISTENCY")
print("=" * 120)


label_audit_rows = []


for _, manifest_row in episode_manifest.iterrows():

    episode_id = manifest_row[
        "Episode_ID"
    ]


    target_group = targets.loc[
        targets[
            "Episode_ID"
        ]
        ==
        episode_id
    ]


    metadata_group = metadata.loc[
        metadata[
            "Episode_ID"
        ]
        ==
        episode_id
    ]


    expected_rows = int(
        manifest_row[
            "Episode_Length_Minutes"
        ]
    )


    row_count_pass = (
        len(
            target_group
        )
        ==
        expected_rows
    )


    if manifest_row[
        "Episode_Type"
    ] == "ATTACK":

        expected_onset = int(
            manifest_row[
                "Attack_Onset_Minute"
            ]
        )


        expected_attack_minutes = int(
            manifest_row[
                "Attack_Duration_Minutes"
            ]
        )


        observed_onsets = target_group.loc[
            target_group[
                "New_Attack_Onset"
            ]
            ==
            1,
            "Minute_Index"
        ].tolist()


        onset_pass = (

            observed_onsets
            ==
            [
                expected_onset
            ]
        )


        attack_duration_pass = (

            int(
                target_group[
                    "Attack_State"
                ].sum()
            )

            ==
            expected_attack_minutes
        )


    else:

        onset_pass = (

            int(
                target_group[
                    "New_Attack_Onset"
                ].sum()
            )

            ==
            0
        )


        attack_duration_pass = (

            int(
                target_group[
                    "Attack_State"
                ].sum()
            )

            ==
            0
        )


    start_timestamp_pass = (

        pd.Timestamp(
            metadata_group[
                "Timestamp_UTC"
            ].min()
        )

        ==

        pd.Timestamp(
            manifest_row[
                "Episode_Start_UTC"
            ]
        )
    )


    end_timestamp_pass = (

        pd.Timestamp(
            metadata_group[
                "Timestamp_UTC"
            ].max()
        )

        ==

        pd.Timestamp(
            manifest_row[
                "Episode_End_UTC"
            ]
        )
    )


    label_audit_rows.append(
        {
            "Episode_ID":
                episode_id,

            "Row_Count_Pass":
                row_count_pass,

            "Onset_Pass":
                onset_pass,

            "Attack_Duration_Pass":
                attack_duration_pass,

            "Start_Timestamp_Pass":
                start_timestamp_pass,

            "End_Timestamp_Pass":
                end_timestamp_pass
        }
    )


manifest_label_audit = pd.DataFrame(
    label_audit_rows
)


manifest_label_failures = int(

    (
        ~manifest_label_audit[
            [
                "Row_Count_Pass",
                "Onset_Pass",
                "Attack_Duration_Pass",
                "Start_Timestamp_Pass",
                "End_Timestamp_Pass"
            ]
        ].all(
            axis=1
        )
    ).sum()
)


print(
    "Episodes with manifest/label mismatch:",
    manifest_label_failures
)


# ==================================================================================================
# 11.4V — PRECURSOR CONTROL AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4V — PRECURSOR / NEGATIVE-CONTROL AUDIT")
print("=" * 120)


merged_control = metadata.merge(

    targets[
        [
            "Episode_ID",
            "Minute_Index",
            "Attack_State",
            "New_Attack_Onset"
        ]
    ],

    on=[
        "Episode_ID",
        "Minute_Index"
    ],

    how="left",

    validate="one_to_one"
)


benign_episode_alpha_failures = int(

    (
        (
            merged_control[
                "Episode_Type"
            ]
            ==
            "BENIGN_ONLY"
        )

        &

        (
            merged_control[
                "Attack_Influence_Alpha"
            ]
            !=
            0.0
        )
    ).sum()
)


no_precursor_preonset_failures = 0

precursor_alpha_bound_failures = 0

attack_alpha_failures = 0


manifest_map = episode_manifest.set_index(
    "Episode_ID"
)


for episode_id, group in merged_control.groupby(
    "Episode_ID",
    observed=True
):

    manifest_row = manifest_map.loc[
        episode_id
    ]


    if manifest_row[
        "Episode_Type"
    ] != "ATTACK":

        continue


    onset = int(
        manifest_row[
            "Attack_Onset_Minute"
        ]
    )


    pre = group.loc[
        group[
            "Minute_Index"
        ]
        <
        onset
    ]


    if manifest_row[
        "Precursor_Regime"
    ] == "NO_PRECURSOR":

        no_precursor_preonset_failures += int(

            (
                pre[
                    "Attack_Influence_Alpha"
                ]
                !=
                0.0
            ).sum()
        )


    max_allowed = float(
        manifest_row[
            "Precursor_Max_Alpha"
        ]
    )


    precursor_rows = pre.loc[
        pre[
            "Synthetic_Phase"
        ]
        ==
        "PRECURSOR"
    ]


    precursor_alpha_bound_failures += int(

        (
            precursor_rows[
                "Attack_Influence_Alpha"
            ]

            >
            max_allowed
            +
            1e-12
        ).sum()
    )


    attack_rows_group = group.loc[
        group[
            "Attack_State"
        ]
        ==
        1
    ]


    attack_alpha_failures += int(

        (
            ~np.isclose(

                attack_rows_group[
                    "Attack_Influence_Alpha"
                ],

                1.0
            )
        ).sum()
    )


precursor_control_audit = pd.DataFrame({

    "Check": [

        "Benign-only alpha nonzero",

        "NO_PRECURSOR pre-onset alpha nonzero",

        "Precursor alpha above manifest maximum",

        "Attack-active alpha != 1"
    ],


    "Failure_Count": [

        benign_episode_alpha_failures,

        no_precursor_preonset_failures,

        precursor_alpha_bound_failures,

        attack_alpha_failures
    ]
})


print(
    precursor_control_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.4W — REPEATED-BLOCK DIVERSIFICATION AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4W — REUSED BLOCK DIVERSIFICATION AUDIT")
print("=" * 120)


duplicate_occurrence_hashes_within_episode = 0

episodes_with_duplicate_generated_occurrences = 0


for episode_id, group in block_occurrence_hashes.groupby(
    "Episode_ID",
    observed=True
):

    duplicates = int(

        group[
            "Generated_Predictor_SHA256"
        ]

        .duplicated()

        .sum()
    )


    duplicate_occurrence_hashes_within_episode += (
        duplicates
    )


    if duplicates > 0:

        episodes_with_duplicate_generated_occurrences += 1


print(
    "Exact duplicate generated block occurrences:",
    duplicate_occurrence_hashes_within_episode
)

print(
    "Episodes containing exact duplicate generated occurrences:",
    episodes_with_duplicate_generated_occurrences
)


# ==================================================================================================
# 11.4X — EPISODE-LEVEL DUPLICATE AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4X — EPISODE-LEVEL DUPLICATE AUDIT")
print("=" * 120)


duplicate_episode_predictor_hashes = int(

    episode_generation_summary[
        "Episode_Predictor_SHA256"
    ]

    .duplicated()

    .sum()
)


print(
    "Exact duplicate full predictor episodes:",
    duplicate_episode_predictor_hashes
)


# ==================================================================================================
# 11.4Y — BENIGN SOURCE DISTRIBUTION SANITY AUDIT
#
# Diagnostic only.
#
# We expect some controlled drift due:
#   - episode load multiplier
#   - AR(1) perturbation
#
# This audit tells us how much.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4Y — BENIGN-ONLY GENERATED vs REAL SOURCE MEDIAN AUDIT")
print("=" * 120)


predictor_metadata_join = predictors.merge(

    metadata[
        [
            "Episode_ID",
            "Minute_Index",
            "Episode_Type",
            "Calibration_Source",
            "Synthetic_Phase"
        ]
    ],

    on=[
        "Episode_ID",
        "Minute_Index"
    ],

    how="left",

    validate="one_to_one"
)


benign_distribution_rows = []


for source in [
    "CIC_IDS2017",
    "UNSW_NB15"
]:

    generated_benign = predictor_metadata_join.loc[
        (
            predictor_metadata_join[
                "Episode_Type"
            ]
            ==
            "BENIGN_ONLY"
        )

        &

        (
            predictor_metadata_join[
                "Calibration_Source"
            ]
            ==
            source
        )
    ]


    for feature in FINAL_PREDICTORS:

        generated_median = float(

            generated_benign[
                feature
            ].median()
        )


        real_lookup = minute_quantile_lookup[
            (
                f"{source}::BENIGN",
                feature
            )
        ]


        real_median = float(
            real_lookup[
                "q50"
            ]
        )


        log_difference = abs(

            np.log1p(
                generated_median
            )

            -

            np.log1p(
                real_median
            )
        )


        fold_difference = float(
            np.exp(
                log_difference
            )
        )


        benign_distribution_rows.append(
            {
                "Calibration_Source":
                    source,

                "Feature":
                    feature,

                "Real_Benign_Median":
                    real_median,

                "Generated_BenignOnly_Median":
                    generated_median,

                "Median_Log1p_Difference":
                    log_difference,

                "Median_Fold_Difference":
                    fold_difference
            }
        )


benign_distribution_audit = pd.DataFrame(
    benign_distribution_rows
)


print(
    benign_distribution_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.4Z — GENERATED FAMILY DIRECTION AUDIT
#
# Compare attack-active generated per-flow medians against generated benign-only
# medians within the SAME source.
#
# This is diagnostic rather than a hard requirement because minute mixtures
# intentionally attenuate the original per-flow family effect.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4Z — GENERATED FAMILY EFFECT DIRECTION AUDIT")
print("=" * 120)


audit_master = predictors.merge(

    targets[
        [
            "Episode_ID",
            "Minute_Index",
            "Attack_State"
        ]
    ],

    on=[
        "Episode_ID",
        "Minute_Index"
    ],

    how="left",

    validate="one_to_one"
)


audit_master = audit_master.merge(

    metadata[
        [
            "Episode_ID",
            "Minute_Index",
            "Episode_Type",
            "Episode_Attack_Family",
            "Calibration_Source"
        ]
    ],

    on=[
        "Episode_ID",
        "Minute_Index"
    ],

    how="left",

    validate="one_to_one"
)


audit_master[
    "Forward_Packets_Per_Flow"
] = (

    audit_master[
        "Forward_Packets_Sum"
    ]

    /

    audit_master[
        "Flow_Count"
    ]
)


audit_master[
    "Backward_Packets_Per_Flow"
] = (

    audit_master[
        "Backward_Packets_Sum"
    ]

    /

    audit_master[
        "Flow_Count"
    ]
)


audit_master[
    "Forward_Bytes_Per_Flow"
] = (

    audit_master[
        "Forward_Bytes_Sum"
    ]

    /

    audit_master[
        "Flow_Count"
    ]
)


audit_master[
    "Backward_Bytes_Per_Flow"
] = (

    audit_master[
        "Backward_Bytes_Sum"
    ]

    /

    audit_master[
        "Flow_Count"
    ]
)


generated_effect_column_map = {

    "Flow_Duration_Seconds":
        "Flow_Duration_Mean",

    "Forward_Packets":
        "Forward_Packets_Per_Flow",

    "Backward_Packets":
        "Backward_Packets_Per_Flow",

    "Forward_Bytes":
        "Forward_Bytes_Per_Flow",

    "Backward_Bytes":
        "Backward_Bytes_Per_Flow"
}


generated_effect_audit_rows = []


for _, source_family in used_family_source_pairs.iterrows():

    source = source_family[
        "Calibration_Source"
    ]


    family = source_family[
        "Attack_Family"
    ]


    benign_source = audit_master.loc[
        (
            audit_master[
                "Episode_Type"
            ]
            ==
            "BENIGN_ONLY"
        )

        &

        (
            audit_master[
                "Calibration_Source"
            ]
            ==
            source
        )
    ]


    attack_source_family = audit_master.loc[
        (
            audit_master[
                "Attack_State"
            ]
            ==
            1
        )

        &

        (
            audit_master[
                "Calibration_Source"
            ]
            ==
            source
        )

        &

        (
            audit_master[
                "Episode_Attack_Family"
            ]
            ==
            family
        )
    ]


    for source_effect_feature, generated_column in generated_effect_column_map.items():

        real_effect = get_effect(

            source,
            family,
            source_effect_feature
        )


        benign_median = float(

            benign_source[
                generated_column
            ].median()
        )


        attack_median = float(

            attack_source_family[
                generated_column
            ].median()
        )


        generated_effect = (

            np.log1p(
                attack_median
            )

            -

            np.log1p(
                benign_median
            )
        )


        if abs(
            real_effect
        ) <= 0.05:

            real_direction = "NEAR_ZERO"

        elif real_effect > 0:

            real_direction = "POSITIVE"

        else:

            real_direction = "NEGATIVE"


        if abs(
            generated_effect
        ) <= 0.05:

            generated_direction = "NEAR_ZERO"

        elif generated_effect > 0:

            generated_direction = "POSITIVE"

        else:

            generated_direction = "NEGATIVE"


        generated_effect_audit_rows.append(
            {
                "Calibration_Source":
                    source,

                "Synthetic_Family":
                    family,

                "Feature":
                    source_effect_feature,

                "Real_Effect_Log1p":
                    real_effect,

                "Generated_AttackVsBenign_Effect_Log1p":
                    generated_effect,

                "Real_Direction":
                    real_direction,

                "Generated_Direction":
                    generated_direction,

                "Direction_Agreement":
                    (
                        real_direction
                        ==
                        generated_direction
                    )
            }
        )


generated_effect_direction_audit = pd.DataFrame(
    generated_effect_audit_rows
)


print(
    generated_effect_direction_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.4AA — STRICT HORIZON TARGET AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AA — STRICT FORECAST-TARGET SUPPORT AUDIT")
print("=" * 120)


target_support_rows = []


target_split_map = metadata[
    [
        "Episode_ID",
        "Minute_Index",
        "Split"
    ]
]


targets_with_split = targets.merge(

    target_split_map,

    on=[
        "Episode_ID",
        "Minute_Index"
    ],

    how="left",

    validate="one_to_one"
)


for split in [
    "TRAIN",
    "VALIDATION",
    "TEST"
]:

    split_targets = targets_with_split.loc[
        targets_with_split[
            "Split"
        ]
        ==
        split
    ]


    independent_onsets = int(

        split_targets.loc[
            split_targets[
                "New_Attack_Onset"
            ]
            ==
            1,
            "Episode_ID"
        ].nunique()
    )


    for horizon in FORECAST_HORIZONS:

        eligible = split_targets.loc[
            split_targets[
                f"Forecast_Eligible_H{horizon}"
            ]
            ==
            1
        ]


        positives = eligible.loc[
            eligible[
                f"Onset_Within_H{horizon}"
            ]
            ==
            1
        ]


        positive_event_count = int(
            positives[
                "Episode_ID"
            ].nunique()
        )


        target_support_rows.append(
            {
                "Split":
                    split,

                "Horizon_Minutes":
                    horizon,

                "Eligible_Origins":
                    len(
                        eligible
                    ),

                "Positive_Origins":
                    len(
                        positives
                    ),

                "Negative_Origins":
                    int(

                        (
                            eligible[
                                f"Onset_Within_H{horizon}"
                            ]

                            ==
                            0
                        ).sum()
                    ),

                "Positive_Rate":
                    (
                        float(
                            len(
                                positives
                            )

                            /
                            len(
                                eligible
                            )
                        )

                        if len(
                            eligible
                        )
                        >
                        0

                        else np.nan
                    ),

                "Unique_Onset_Events_Represented":
                    positive_event_count,

                "Independent_Onsets_In_Split":
                    independent_onsets
            }
        )


target_support_audit = pd.DataFrame(
    target_support_rows
)


print(
    target_support_audit.to_string(
        index=False
    )
)


# Every horizon should represent every attack event at least once.

all_events_represented = bool(

    (
        target_support_audit[
            "Unique_Onset_Events_Represented"
        ]

        ==

        target_support_audit[
            "Independent_Onsets_In_Split"
        ]
    ).all()
)


# ==================================================================================================
# 11.4AB — FAMILY TARGET COVERAGE
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AB — FAMILY TARGET COVERAGE AUDIT")
print("=" * 120)


family_target_rows = []


for split in [
    "TRAIN",
    "VALIDATION",
    "TEST"
]:

    for horizon in FORECAST_HORIZONS:

        col = f"Onset_Family_H{horizon}"


        positive = targets_with_split.loc[
            (
                targets_with_split[
                    "Split"
                ]
                ==
                split
            )

            &

            (
                targets_with_split[
                    f"Onset_Within_H{horizon}"
                ]
                ==
                1
            )
        ]


        for family in ATTACK_FAMILIES:

            family_target_rows.append(
                {
                    "Split":
                        split,

                    "Horizon_Minutes":
                        horizon,

                    "Attack_Family":
                        family,

                    "Positive_Origin_Count":
                        int(

                            (
                                positive[
                                    col
                                ]

                                ==
                                family
                            ).sum()
                        ),

                    "Unique_Event_Count":
                        int(

                            positive.loc[
                                positive[
                                    col
                                ]
                                ==
                                family,
                                "Episode_ID"
                            ].nunique()
                        )
                }
            )


family_target_coverage = pd.DataFrame(
    family_target_rows
)


print(
    family_target_coverage.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.4AC — FINAL HARD VALIDATION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AC — HARD PHASE 11.4 VALIDATION")
print("=" * 120)


validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                details
        }
    )


add_check(

    "Frozen_11_0_Config_Unchanged",

    CURRENT_CONFIG_HASH
    ==
    FROZEN_CONFIG_HASH,

    CURRENT_CONFIG_HASH
)


add_check(

    "Frozen_11_2_Policy_Unchanged",

    CURRENT_POLICY_HASH
    ==
    FROZEN_POLICY_HASH,

    CURRENT_POLICY_HASH
)


add_check(

    "Frozen_11_3_Manifest_Unchanged",

    CURRENT_MANIFEST_HASH
    ==
    FROZEN_MANIFEST_HASH,

    CURRENT_MANIFEST_HASH
)


add_check(

    "Frozen_11_3_TEST_Unchanged",

    CURRENT_TEST_HASH
    ==
    FROZEN_TEST_HASH,

    CURRENT_TEST_HASH
)


add_check(

    "Generated_Row_Count_Equals_Frozen_Manifest",

    len(
        predictors
    )
    ==
    expected_total_rows,

    f"{len(predictors):,} / {expected_total_rows:,}"
)


add_check(

    "Predictor_Target_Metadata_Row_Counts_Equal",

    len(
        predictors
    )
    ==
    len(
        targets
    )
    ==
    len(
        metadata
    ),

    (
        f"{len(predictors):,} / "
        f"{len(targets):,} / "
        f"{len(metadata):,}"
    )
)


add_check(

    "No_Duplicate_Predictor_Keys",

    predictor_duplicate_keys
    ==
    0,

    str(
        predictor_duplicate_keys
    )
)


add_check(

    "No_Duplicate_Target_Keys",

    target_duplicate_keys
    ==
    0,

    str(
        target_duplicate_keys
    )
)


add_check(

    "No_Duplicate_Metadata_Keys",

    metadata_duplicate_keys
    ==
    0,

    str(
        metadata_duplicate_keys
    )
)


add_check(

    "No_NaN_Predictors",

    nan_cells
    ==
    0,

    str(
        nan_cells
    )
)


add_check(

    "No_Infinite_Predictors",

    inf_cells
    ==
    0,

    str(
        inf_cells
    )
)


add_check(

    "No_Negative_Predictors",

    negative_cells
    ==
    0,

    str(
        negative_cells
    )
)


add_check(

    "Flow_Count_Strictly_Positive",

    nonpositive_flow_count
    ==
    0,

    str(
        nonpositive_flow_count
    )
)


for check_name, failure_count in [

    (
        "Total_Packets_Exact",
        total_packet_failures
    ),

    (
        "Total_Bytes_Exact",
        total_byte_failures
    ),

    (
        "Packets_Per_Second_Exact",
        packet_rate_failures
    ),

    (
        "Bytes_Per_Second_Exact",
        byte_rate_failures
    ),

    (
        "Packet_Ratio_Exact",
        packet_ratio_failures
    ),

    (
        "Byte_Ratio_Exact",
        byte_ratio_failures
    ),

    (
        "Duration_Max_Greater_Equal_Mean",
        duration_order_failures
    )

]:

    add_check(

        check_name,

        failure_count
        ==
        0,

        str(
            failure_count
        )
    )


add_check(

    "Minute_Index_Continuous_Inside_Every_Episode",

    minute_index_failures
    ==
    0,

    str(
        minute_index_failures
    )
)


add_check(

    "Exactly_60_Second_Timestamp_Spacing_Inside_Episodes",

    timestamp_gap_failures
    ==
    0,

    str(
        timestamp_gap_failures
    )
)


add_check(

    "Generated_Labels_Match_Frozen_Manifest",

    manifest_label_failures
    ==
    0,

    str(
        manifest_label_failures
    )
)


add_check(

    "Benign_Only_Episodes_Have_Zero_Attack_Influence",

    benign_episode_alpha_failures
    ==
    0,

    str(
        benign_episode_alpha_failures
    )
)


add_check(

    "NO_PRECURSOR_Has_Zero_PreOnset_Attack_Influence",

    no_precursor_preonset_failures
    ==
    0,

    str(
        no_precursor_preonset_failures
    )
)


add_check(

    "Precursor_Alpha_Respects_Frozen_Maximum",

    precursor_alpha_bound_failures
    ==
    0,

    str(
        precursor_alpha_bound_failures
    )
)


add_check(

    "Attack_Active_Alpha_Equals_One",

    attack_alpha_failures
    ==
    0,

    str(
        attack_alpha_failures
    )
)


add_check(

    "Reused_Blocks_Not_Exact_Generated_Copies",

    duplicate_occurrence_hashes_within_episode
    ==
    0,

    str(
        duplicate_occurrence_hashes_within_episode
    )
)


add_check(

    "No_Exact_Duplicate_Full_Episodes",

    duplicate_episode_predictor_hashes
    ==
    0,

    str(
        duplicate_episode_predictor_hashes
    )
)


add_check(

    "Every_Horizon_Represents_Every_Independent_Onset_Event",

    all_events_represented,

    target_support_audit[
        [
            "Split",
            "Horizon_Minutes",
            "Unique_Onset_Events_Represented",
            "Independent_Onsets_In_Split"
        ]
    ].to_dict(
        orient="records"
    )
)


# Every family must appear as a positive onset target in every split and horizon.

family_coverage_pass = bool(

    (
        family_target_coverage[
            "Unique_Event_Count"
        ]

        >
        0
    ).all()
)


add_check(

    "Every_Family_Has_Positive_Onset_Targets_In_Every_Split_And_Horizon",

    family_coverage_pass,

    (
        f"{int((family_target_coverage['Unique_Event_Count'] > 0).sum())}"
        f"/{len(family_target_coverage)} cells"
    )
)


# Predictors file must contain only keys + frozen predictor whitelist.

allowed_predictor_file_columns = {

    "Episode_ID",
    "Minute_Index",
    *FINAL_PREDICTORS
}


unexpected_predictor_columns = sorted(

    set(
        predictors.columns
    )

    -

    allowed_predictor_file_columns
)


add_check(

    "Predictor_File_Contains_No_Metadata_Or_Target_Columns",

    len(
        unexpected_predictor_columns
    )
    ==
    0,

    str(
        unexpected_predictor_columns
    )
)


validation = pd.DataFrame(
    validation_rows
)


print(
    validation.to_string(
        index=False
    )
)


PHASE_11_4_PASS = bool(

    validation[
        "Passed"
    ].all()
)


# ==================================================================================================
# 11.4AD — SAVE AUDIT TABLES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AD — SAVE AUDIT ARTIFACTS")
print("=" * 120)


def save_csv(
    dataframe,
    path,
    float_format="%.10g"
):

    dataframe.to_csv(

        path,

        index=False,

        float_format=float_format
    )


    print(
        "✓",
        path
    )


save_csv(

    calibration_input_hashes,

    GENERATED_DIR
    / "11_4_calibration_input_hashes.csv"
)


save_csv(

    physical_consistency,

    GENERATED_DIR
    / "11_4_physical_consistency_audit.csv"
)


save_csv(

    manifest_label_audit,

    GENERATED_DIR
    / "11_4_manifest_label_audit.csv"
)


save_csv(

    precursor_control_audit,

    GENERATED_DIR
    / "11_4_precursor_control_audit.csv"
)


save_csv(

    block_occurrence_hashes,

    GENERATED_DIR
    / "11_4_block_occurrence_hashes.csv"
)


save_csv(

    episode_generation_summary,

    GENERATED_DIR
    / "11_4_episode_generation_summary.csv"
)


save_csv(

    benign_distribution_audit,

    GENERATED_DIR
    / "11_4_benign_distribution_audit.csv"
)


save_csv(

    generated_effect_direction_audit,

    GENERATED_DIR
    / "11_4_generated_effect_direction_audit.csv"
)


save_csv(

    target_support_audit,

    GENERATED_DIR
    / "11_4_target_support_audit.csv"
)


save_csv(

    family_target_coverage,

    GENERATED_DIR
    / "11_4_family_target_coverage.csv"
)


save_csv(

    validation,

    GENERATED_DIR
    / "11_4_validation.csv"
)


# ==================================================================================================
# 11.4AE — SAVE MAIN DATASET FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AE — SAVE SYNTHETIC DATASET")
print("=" * 120)


PREDICTORS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_predictors.csv"
)


TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_targets.csv"
)


METADATA_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_metadata.csv"
)


save_csv(
    predictors,
    PREDICTORS_FILE
)


save_csv(
    targets,
    TARGETS_FILE
)


save_csv(
    metadata,
    METADATA_FILE
)


# Parquet convenience copies.

for dataframe, filename in [

    (
        predictors,
        "11_4_synthetic_minute_predictors.parquet"
    ),

    (
        targets,
        "11_4_synthetic_minute_targets.parquet"
    ),

    (
        metadata,
        "11_4_synthetic_minute_metadata.parquet"
    )
]:

    try:

        path = (
            GENERATED_DIR
            /
            filename
        )


        dataframe.to_parquet(
            path,
            index=False
        )


        print(
            "✓",
            path
        )


    except Exception as error:

        print(
            "⚠ Parquet save skipped:",
            filename,
            str(
                error
            )[:120]
        )


# ==================================================================================================
# 11.4AF — SAVE SPLIT-SPECIFIC FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AF — SAVE TRAIN / VALIDATION / TEST FILES")
print("=" * 120)


split_lookup = (

    metadata[
        [
            "Episode_ID",
            "Minute_Index",
            "Split"
        ]
    ]
)


predictors_with_split = predictors.merge(

    split_lookup,

    on=[
        "Episode_ID",
        "Minute_Index"
    ],

    how="left",

    validate="one_to_one"
)


targets_with_split_for_save = targets.merge(

    split_lookup,

    on=[
        "Episode_ID",
        "Minute_Index"
    ],

    how="left",

    validate="one_to_one"
)


for split in [
    "TRAIN",
    "VALIDATION",
    "TEST"
]:

    split_lower = split.lower()


    p = (

        predictors_with_split.loc[
            predictors_with_split[
                "Split"
            ]
            ==
            split
        ]

        .drop(
            columns=[
                "Split"
            ]
        )

        .reset_index(
            drop=True
        )
    )


    t = (

        targets_with_split_for_save.loc[
            targets_with_split_for_save[
                "Split"
            ]
            ==
            split
        ]

        .drop(
            columns=[
                "Split"
            ]
        )

        .reset_index(
            drop=True
        )
    )


    m = (

        metadata.loc[
            metadata[
                "Split"
            ]
            ==
            split
        ]

        .reset_index(
            drop=True
        )
    )


    save_csv(

        p,

        GENERATED_DIR
        /
        f"11_4_{split_lower}_predictors.csv"
    )


    save_csv(

        t,

        GENERATED_DIR
        /
        f"11_4_{split_lower}_targets.csv"
    )


    save_csv(

        m,

        GENERATED_DIR
        /
        f"11_4_{split_lower}_metadata.csv"
    )


# ==================================================================================================
# 11.4AG — HASH MAIN GENERATED FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AG — HASH GENERATED DATASET FILES")
print("=" * 120)


generated_file_hash_rows = []


files_to_hash = [

    GENERATOR_SPEC_FILE,

    PREDICTORS_FILE,

    TARGETS_FILE,

    METADATA_FILE,

    GENERATED_DIR
    / "11_4_train_predictors.csv",

    GENERATED_DIR
    / "11_4_train_targets.csv",

    GENERATED_DIR
    / "11_4_validation_predictors.csv",

    GENERATED_DIR
    / "11_4_validation_targets.csv",

    GENERATED_DIR
    / "11_4_test_predictors.csv",

    GENERATED_DIR
    / "11_4_test_targets.csv"
]


for path in files_to_hash:

    digest = sha256_file(
        path
    )


    generated_file_hash_rows.append(
        {
            "Artifact":
                path.name,

            "SHA256":
                digest
        }
    )


    print(
        path.name,
        digest
    )


generated_file_hashes = pd.DataFrame(
    generated_file_hash_rows
)


save_csv(

    generated_file_hashes,

    GENERATED_DIR
    / "11_4_generated_file_hashes.csv"
)


# ==================================================================================================
# 11.4AH — LOCK GENERATED TEST DATA
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AH — LOCK GENERATED TEST DATA")
print("=" * 120)


TEST_PREDICTOR_FILE = (
    GENERATED_DIR
    / "11_4_test_predictors.csv"
)


TEST_TARGET_FILE = (
    GENERATED_DIR
    / "11_4_test_targets.csv"
)


TEST_METADATA_FILE = (
    GENERATED_DIR
    / "11_4_test_metadata.csv"
)


GENERATED_TEST_LOCK = {

    "phase":
        "11.4",

    "locked_manifest_test_sha256":
        CURRENT_TEST_HASH,

    "generator_spec_sha256":
        GENERATOR_SPEC_SHA256,

    "test_predictors_sha256":
        sha256_file(
            TEST_PREDICTOR_FILE
        ),

    "test_targets_sha256":
        sha256_file(
            TEST_TARGET_FILE
        ),

    "test_metadata_sha256":
        sha256_file(
            TEST_METADATA_FILE
        ),

    "regenerate_because_of_model_results":
        False
}


GENERATED_TEST_LOCK_FILE = (
    GENERATED_DIR
    / "11_4_TEST_DATA_LOCK.json"
)


with open(
    GENERATED_TEST_LOCK_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        GENERATED_TEST_LOCK,
        f,
        indent=4
    )


GENERATED_TEST_LOCK_SHA256 = sha256_file(
    GENERATED_TEST_LOCK_FILE
)


(
    GENERATED_DIR
    /
    "11_4_TEST_DATA_LOCK_SHA256.txt"
).write_text(

    GENERATED_TEST_LOCK_SHA256
    +
    "\n"
)


print(
    "Generated TEST lock SHA256:"
)

print(
    GENERATED_TEST_LOCK_SHA256
)


# ==================================================================================================
# 11.4AI — SPLIT ROW SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AI — SPLIT ROW SUMMARY")
print("=" * 120)


split_row_summary = (

    metadata

    .groupby(
        "Split",
        observed=True
    )

    .agg(

        Minute_Rows=(
            "Episode_ID",
            "size"
        ),

        Episodes=(
            "Episode_ID",
            "nunique"
        ),

        CIC_Episodes=(
            "Calibration_Source",
            lambda x:
                metadata.loc[
                    x.index
                ]
                .loc[
                    metadata.loc[
                        x.index,
                        "Calibration_Source"
                    ]
                    ==
                    "CIC_IDS2017",
                    "Episode_ID"
                ]
                .nunique()
        ),

        UNSW_Episodes=(
            "Calibration_Source",
            lambda x:
                metadata.loc[
                    x.index
                ]
                .loc[
                    metadata.loc[
                        x.index,
                        "Calibration_Source"
                    ]
                    ==
                    "UNSW_NB15",
                    "Episode_ID"
                ]
                .nunique()
        )

    )

    .reset_index()
)


print(
    split_row_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.4AJ — FINAL PHASE MANIFEST
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AJ — PHASE 11.4 MANIFEST")
print("=" * 120)


phase_manifest = pd.DataFrame({

    "Metric": [

        "Episode_Count",

        "Minute_Row_Count",

        "Predictor_Count",

        "Forecast_Horizons",

        "Independent_Attack_Events",

        "Benign_Only_Episodes",

        "Train_Attack_Events",

        "Validation_Attack_Events",

        "Test_Attack_Events",

        "Exact_Duplicate_Full_Episodes",

        "Exact_Duplicate_Generated_Block_Occurrences",

        "Physical_Consistency_Failures",

        "Manifest_Label_Failures",

        "NO_PRECURSOR_PreOnset_Influence_Failures",

        "Every_Horizon_Represents_Every_Event",

        "Validation_Checks",

        "Validation_Passed",

        "Phase_11_4_Pass",

        "Generator_Spec_SHA256",

        "Frozen_11_3_Manifest_SHA256",

        "Frozen_11_3_TEST_SHA256",

        "Generated_TEST_Lock_SHA256"

    ],


    "Value": [

        episode_manifest[
            "Episode_ID"
        ].nunique(),

        len(
            predictors
        ),

        len(
            FINAL_PREDICTORS
        ),

        str(
            FORECAST_HORIZONS
        ),

        int(
            targets.loc[
                targets[
                    "New_Attack_Onset"
                ]
                ==
                1,
                "Episode_ID"
            ].nunique()
        ),

        int(

            episode_manifest.loc[
                episode_manifest[
                    "Episode_Type"
                ]
                ==
                "BENIGN_ONLY",
                "Episode_ID"
            ].nunique()
        ),

        int(

            targets_with_split.loc[
                (
                    targets_with_split[
                        "Split"
                    ]
                    ==
                    "TRAIN"
                )

                &

                (
                    targets_with_split[
                        "New_Attack_Onset"
                    ]
                    ==
                    1
                ),
                "Episode_ID"
            ].nunique()
        ),

        int(

            targets_with_split.loc[
                (
                    targets_with_split[
                        "Split"
                    ]
                    ==
                    "VALIDATION"
                )

                &

                (
                    targets_with_split[
                        "New_Attack_Onset"
                    ]
                    ==
                    1
                ),
                "Episode_ID"
            ].nunique()
        ),

        int(

            targets_with_split.loc[
                (
                    targets_with_split[
                        "Split"
                    ]
                    ==
                    "TEST"
                )

                &

                (
                    targets_with_split[
                        "New_Attack_Onset"
                    ]
                    ==
                    1
                ),
                "Episode_ID"
            ].nunique()
        ),

        duplicate_episode_predictor_hashes,

        duplicate_occurrence_hashes_within_episode,

        int(
            physical_consistency[
                "Failure_Count"
            ].sum()
        ),

        manifest_label_failures,

        no_precursor_preonset_failures,

        all_events_represented,

        len(
            validation
        ),

        int(
            validation[
                "Passed"
            ].sum()
        ),

        PHASE_11_4_PASS,

        GENERATOR_SPEC_SHA256,

        CURRENT_MANIFEST_HASH,

        CURRENT_TEST_HASH,

        GENERATED_TEST_LOCK_SHA256
    ]
})


print(
    phase_manifest.to_string(
        index=False
    )
)


save_csv(

    phase_manifest,

    GENERATED_DIR
    / "11_4_phase_manifest.csv"
)


# ==================================================================================================
# 11.4AK — FINAL DECISION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.4AK — FINAL DECISION")
print("=" * 120)


if not PHASE_11_4_PASS:

    failed = validation.loc[
        ~validation[
            "Passed"
        ]
    ]


    print(
        "\nFAILED HARD CHECKS:\n"
    )


    print(
        failed.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "PHASE 11.4 FAILED. "
        "DO NOT BUILD MODEL SEQUENCES OR TRAIN MODELS."
    )


print(
    f"""
✓ PHASE 11.4 — SYNTHETIC TEMPORAL MINUTE GENERATION: PASS


DATASET
-------
Episodes:

    {episode_manifest["Episode_ID"].nunique():,}

Minute rows:

    {len(predictors):,}

Independent attack-onset events:

    {int(targets.loc[targets["New_Attack_Onset"] == 1, "Episode_ID"].nunique()):,}

Benign-only episodes:

    {int((episode_manifest["Episode_Type"] == "BENIGN_ONLY").sum()):,}


MODEL INPUT
-----------
Exactly 14 frozen traffic predictors.

No:

    timestamp
    split
    episode ID
    attack family
    precursor regime
    source
    generator alpha
    generator seed

may enter X.


TEMPORAL STRUCTURE
------------------
Within every episode:

    exactly 1-minute spacing

No sequence is allowed to cross Episode_ID.


NO-PRECURSOR CONTROLS
---------------------
NO_PRECURSOR attack episodes have:

    alpha = 0

for EVERY minute before the true attack onset.

Therefore these episodes provide a real simulation negative control:
a temporal model should NOT magically obtain reliable advance warning
when no precursor signal was inserted.


PRECURSOR EPISODES
------------------
SHORT / GRADUAL / BURSTY behavior is a CONTROLLED simulation assumption.

It must NOT be reported as:

    "CIC/UNSW prove attacks behave this way before onset."

Correct interpretation:

    "The benchmark evaluates model behavior under controlled,
     empirically calibrated traffic-transition assumptions."


PHYSICAL CONSISTENCY
--------------------
Dependent features were recomputed mathematically.

Never independently generated:

    Total_Packets
    Total_Bytes
    Packets_Per_Second
    Bytes_Per_Second
    Forward_Backward_Packet_Ratio
    Forward_Backward_Byte_Ratio


REUSED REAL BLOCKS
------------------
Real 5-minute calibration blocks were diversified using:

    episode load scaling
    temporally correlated AR(1) perturbation
    family influence
    attack intensity variation

Exact duplicate generated block occurrences:

    {duplicate_occurrence_hashes_within_episode}


STRICT ONSET TARGETS
--------------------
Generated horizons:

    {FORECAST_HORIZONS}

For each H:

    current minute must be benign
    full H-minute future must exist
    onset must be strictly AFTER t
    prediction never crosses Episode_ID

Right-censored origins are marked ineligible.


TEST SET
--------
The previously locked 180 TEST episodes were used unchanged.

Generated TEST predictors / targets / metadata are now
hash-locked as well.

Generated TEST lock SHA256:

    {GENERATED_TEST_LOCK_SHA256}


GENERATOR SPEC SHA256
---------------------

    {GENERATOR_SPEC_SHA256}


NEXT
----
DO NOT TRAIN LSTM YET.

NEXT SHOULD BE:

    PHASE 11.5 — SYNTHETIC DATASET QUALITY & FORECASTABILITY AUDIT

Before modeling, 11.5 should test:

    1. synthetic vs real distribution similarity

    2. source-specific median/IQR agreement

    3. correlation preservation

    4. autocorrelation / temporal-delta realism

    5. train/validation/test duplicate-neighbour audit

    6. source shortcut strength

    7. precursor signal strength by regime

    8. NO_PRECURSOR predictability control

    9. attack-family separability audit

   10. target prevalence by horizon

   11. event-level lead-time feasibility

   12. whether synthetic episodes are too easy

Only after that:

    PHASE 12 — MODELING
"""
)


print(
    "=" * 120
)

print(
    "PHASE 11.4 COMPLETE"
)

print(
    "=" * 120
)

In [ ]:
# ==================================================================================================
# EMERGENCY RECOVERY — RESTORE EXACT CORRECTED 11.4 TARGETS AFTER STALE OVERWRITE
# ==================================================================================================
#
# IMPORTANT:
#
# Current known state:
#
#   predictors = CORRECT
#   metadata   = CORRECT
#   targets    = STALE OLD 11.4
#
# This cell:
#
#   - DOES NOT regenerate traffic
#   - DOES NOT modify predictors
#   - DOES NOT modify metadata
#   - rebuilds ONLY targets
#   - verifies exact historical corrected hashes BEFORE committing
#   - restores 11.4 readiness
#   - checks whether the already-completed 11.5 can be reused
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd


print("=" * 120)
print("EMERGENCY 11.4 TARGET RESTORATION")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

GENERATED_DIR = (
    DATASET_ROOT
    / "generated"
    / "11_4"
)

AUDIT_ROOT = (
    DATASET_ROOT
    / "audit"
)

AUDIT_11_5_DIR = (
    AUDIT_ROOT
    / "11_5"
)


PREDICTORS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_predictors.csv"
)

TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_targets.csv"
)

METADATA_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_metadata.csv"
)


TRAIN_TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_train_targets.csv"
)

VAL_TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_validation_targets.csv"
)

TEST_TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_test_targets.csv"
)


READINESS_11_4_FILE = (
    AUDIT_ROOT
    / "11_4_READY_FOR_11_5.json"
)

READINESS_11_4_HASH_FILE = (
    AUDIT_ROOT
    / "11_4_READY_FOR_11_5_SHA256.txt"
)


READINESS_11_5_FILE = (
    AUDIT_ROOT
    / "11_5_READY_FOR_FINAL_FREEZE.json"
)

DECISION_11_5_FILE = (
    AUDIT_11_5_DIR
    / "11_5_decision.json"
)

VALIDATION_11_5_FILE = (
    AUDIT_11_5_DIR
    / "11_5_validation.csv"
)


# ==================================================================================================
# HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def save_csv_exact(df, path):

    df.to_csv(
        path,
        index=False,
        float_format="%.10g"
    )


# ==================================================================================================
# FROZEN EXPECTATIONS
# ==================================================================================================

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


EXPECTED = {

    "predictors":
        "9b0213a77d5320fa5377922ac37e1b104291c1d49e39d71f87e4333f96e2eb3b",

    "metadata":
        "ce7e8723347a890ae90524727302ee83ed911ab4545b208f2e70e6b9cbd57ad7",


    "targets":
        "cebe5e388d2f2476bb066a9878de933e6f295abf118e1a429adb328a87750884",

    "train_targets":
        "80f5cc6770d49726533f29c515b6373091619ed06e540222a02c4708396933c0",

    "validation_targets":
        "7d2c5ec9410b576386290a13f5a4fc1c6300265d71d48856c3c0d393efe330bd",

    "test_targets":
        "07530190053d47e07aeefd7a5bf7b6be4a362fd52ee1ef7d31cb6415d61b5d43",


    # Successful final 11.5 readiness run.
    "phase_11_5_readiness":
        "19ec1005bf88962f2a425d830a9b5f3101efe2a6edb3e99cc1e6daa6efbfaef1",
}


EXPECTED_POSITIVES = {

    ("TRAIN", 1): 700,
    ("TRAIN", 5): 3500,
    ("TRAIN", 10): 7000,
    ("TRAIN", 15): 10500,
    ("TRAIN", 30): 21000,

    ("VALIDATION", 1): 150,
    ("VALIDATION", 5): 750,
    ("VALIDATION", 10): 1500,
    ("VALIDATION", 15): 2250,
    ("VALIDATION", 30): 4500,

    ("TEST", 1): 150,
    ("TEST", 5): 750,
    ("TEST", 10): 1500,
    ("TEST", 15): 2250,
    ("TEST", 30): 4500,
}


# ==================================================================================================
# 1. VERIFY UNTOUCHED TRAFFIC
# ==================================================================================================

print("\n" + "=" * 120)
print("1. VERIFY PREDICTORS + METADATA ARE STILL EXACT")
print("=" * 120)


for path in [
    PREDICTORS_FILE,
    TARGETS_FILE,
    METADATA_FILE,
]:

    require_file(path)


predictor_hash = sha256_file(
    PREDICTORS_FILE
)

metadata_hash = sha256_file(
    METADATA_FILE
)

current_target_hash = sha256_file(
    TARGETS_FILE
)


print(
    "Predictors:",
    predictor_hash
)

print(
    "Metadata  :",
    metadata_hash
)

print(
    "Targets   :",
    current_target_hash
)


if predictor_hash != EXPECTED["predictors"]:

    raise RuntimeError(
        "Predictors changed. Target-only recovery is not safe."
    )


if metadata_hash != EXPECTED["metadata"]:

    raise RuntimeError(
        "Metadata changed. Target-only recovery is not safe."
    )


print(
    "\n✓ Predictor traffic is exact."
)

print(
    "✓ Metadata is exact."
)


if current_target_hash == EXPECTED["targets"]:

    print(
        "✓ Targets are already corrected."
    )

    targets_already_correct = True

else:

    targets_already_correct = False

    print(
        "⚠ Targets are stale and will be rebuilt."
    )


# ==================================================================================================
# 2. LOAD TABLES
# ==================================================================================================

predictors = pd.read_csv(
    PREDICTORS_FILE,
    low_memory=False
)

old_targets = pd.read_csv(
    TARGETS_FILE,
    low_memory=False
)

metadata = pd.read_csv(
    METADATA_FILE,
    low_memory=False
)


KEYS = [
    "Episode_ID",
    "Minute_Index"
]


if not (
    len(predictors)
    ==
    len(old_targets)
    ==
    len(metadata)
    ==
    146031
):

    raise RuntimeError(
        "Unexpected row counts."
    )


if not predictors[
    KEYS
].equals(
    old_targets[
        KEYS
    ]
):

    raise RuntimeError(
        "Predictor/target keys misaligned."
    )


if not predictors[
    KEYS
].equals(
    metadata[
        KEYS
    ]
):

    raise RuntimeError(
        "Predictor/metadata keys misaligned."
    )


# ==================================================================================================
# 3. REBUILD CORRECTED TARGETS
# ==================================================================================================

BASE_TARGET_COLUMNS = [

    "Episode_ID",
    "Minute_Index",

    "Attack_State",
    "New_Attack_Onset",

    "Current_Attack_Family",
    "Episode_Attack_Family",

    "Minutes_To_Next_Onset",

    "Clean_PreOnset_Origin",

    "Synthetic_Attack_Flow_Count",
]


missing = [
    c
    for c in BASE_TARGET_COLUMNS
    if c not in old_targets.columns
]


if missing:

    raise RuntimeError(
        f"Missing base target columns: {missing}"
    )


corrected = old_targets[
    BASE_TARGET_COLUMNS
].copy()


corrected[
    "Minute_Index"
] = pd.to_numeric(
    corrected[
        "Minute_Index"
    ],
    errors="raise"
).astype(
    np.int64
)


for column in [

    "Attack_State",
    "New_Attack_Onset",
    "Clean_PreOnset_Origin",
    "Synthetic_Attack_Flow_Count",

]:

    corrected[
        column
    ] = pd.to_numeric(
        corrected[
            column
        ],
        errors="raise"
    ).astype(
        np.int64
    )


corrected[
    "Minutes_To_Next_Onset"
] = pd.to_numeric(
    corrected[
        "Minutes_To_Next_Onset"
    ],
    errors="coerce"
)


minute_index = corrected[
    "Minute_Index"
].to_numpy(
    dtype=np.int64
)


clean_origin = (
    corrected[
        "Clean_PreOnset_Origin"
    ].to_numpy(
        dtype=np.int64
    )
    ==
    1
)


minutes_to_onset = corrected[
    "Minutes_To_Next_Onset"
].to_numpy(
    dtype=float
)


attack_active = (
    corrected[
        "Attack_State"
    ].to_numpy(
        dtype=np.int64
    )
    ==
    1
)


episode_length_map = (
    corrected
    .groupby(
        "Episode_ID",
        sort=False,
        observed=True
    )[
        "Minute_Index"
    ]
    .max()
    +
    1
)


episode_length = corrected[
    "Episode_ID"
].map(
    episode_length_map
).to_numpy(
    dtype=np.int64
)


semantic_rows = []


for horizon in FORECAST_HORIZONS:

    full_future = (
        minute_index
        +
        horizon
        <
        episode_length
    )


    positive = (
        clean_origin
        &
        np.isfinite(
            minutes_to_onset
        )
        &
        (
            minutes_to_onset
            >
            0
        )
        &
        (
            minutes_to_onset
            <=
            horizon
        )
    )


    negative = (
        clean_origin
        &
        ~positive
        &
        full_future
    )


    eligible = (
        positive
        |
        negative
    )


    right_censored = (
        clean_origin
        &
        ~positive
        &
        ~full_future
    )


    y = np.full(
        len(corrected),
        np.nan,
        dtype=float
    )


    y[
        negative
    ] = 0.0


    y[
        positive
    ] = 1.0


    family = np.full(
        len(corrected),
        "",
        dtype=object
    )


    family[
        positive
    ] = corrected.loc[
        positive,
        "Episode_Attack_Family"
    ].astype(
        str
    ).to_numpy()


    corrected[
        f"Forecast_Eligible_H{horizon}"
    ] = eligible.astype(
        np.int8
    )


    corrected[
        f"Right_Censored_H{horizon}"
    ] = right_censored.astype(
        np.int8
    )


    corrected[
        f"Onset_Within_H{horizon}"
    ] = y


    corrected[
        f"Onset_Family_H{horizon}"
    ] = family


    semantic_rows.append(
        {
            "Horizon_Minutes":
                horizon,

            "AttackActiveEligible":
                int(
                    (
                        eligible
                        &
                        attack_active
                    ).sum()
                ),

            "NonCleanEligible":
                int(
                    (
                        eligible
                        &
                        ~clean_origin
                    ).sum()
                ),

            "PositiveRightCensored":
                int(
                    (
                        positive
                        &
                        right_censored
                    ).sum()
                ),

            "NegativeWithoutFullFuture":
                int(
                    (
                        negative
                        &
                        ~full_future
                    ).sum()
                ),
        }
    )


semantic = pd.DataFrame(
    semantic_rows
)


print("\n" + "=" * 120)
print("2. SEMANTIC CHECK")
print("=" * 120)


print(
    semantic.to_string(
        index=False
    )
)


if int(
    semantic.drop(
        columns=[
            "Horizon_Minutes"
        ]
    ).to_numpy().sum()
) != 0:

    raise RuntimeError(
        "Corrected semantic reconstruction failed."
    )


# ==================================================================================================
# 4. VERIFY EXACT TARGET SUPPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("3. TARGET SUPPORT CHECK")
print("=" * 120)


for split in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    split_mask = metadata[
        "Split"
    ].eq(
        split
    ).to_numpy()


    for horizon in FORECAST_HORIZONS:

        eligible = (
            corrected[
                f"Forecast_Eligible_H{horizon}"
            ].to_numpy()
            ==
            1
        )


        y = corrected[
            f"Onset_Within_H{horizon}"
        ].to_numpy(
            dtype=float
        )


        positive = (
            split_mask
            &
            eligible
            &
            (
                y
                ==
                1
            )
        )


        observed = int(
            positive.sum()
        )

        expected = EXPECTED_POSITIVES[
            (
                split,
                horizon
            )
        ]


        print(
            f"{'✓' if observed == expected else '❌'} "
            f"{split:<11} H{horizon:<2} "
            f"{observed:,} / {expected:,}"
        )


        if observed != expected:

            raise RuntimeError(
                f"Incorrect target support: "
                f"{split} H{horizon}"
            )


# ==================================================================================================
# 5. WRITE TEMP FILES
#
# CANONICAL TARGETS ARE NOT TOUCHED YET.
# ==================================================================================================

FULL_TMP = (
    GENERATED_DIR
    / "_RECOVERY_corrected_targets.csv"
)

TRAIN_TMP = (
    GENERATED_DIR
    / "_RECOVERY_train_targets.csv"
)

VAL_TMP = (
    GENERATED_DIR
    / "_RECOVERY_validation_targets.csv"
)

TEST_TMP = (
    GENERATED_DIR
    / "_RECOVERY_test_targets.csv"
)


save_csv_exact(
    corrected,
    FULL_TMP
)


for split, path in [

    (
        "TRAIN",
        TRAIN_TMP
    ),

    (
        "VALIDATION",
        VAL_TMP
    ),

    (
        "TEST",
        TEST_TMP
    ),

]:

    mask = metadata[
        "Split"
    ].eq(
        split
    )


    save_csv_exact(
        corrected.loc[
            mask
        ].reset_index(
            drop=True
        ),
        path
    )


# ==================================================================================================
# 6. EXACT HASH GATE
# ==================================================================================================

print("\n" + "=" * 120)
print("4. EXACT CORRECTED HASH GATE")
print("=" * 120)


temp_files = {

    "targets":
        FULL_TMP,

    "train_targets":
        TRAIN_TMP,

    "validation_targets":
        VAL_TMP,

    "test_targets":
        TEST_TMP,
}


for name, path in temp_files.items():

    current = sha256_file(
        path
    )

    expected = EXPECTED[
        name
    ]


    print(
        f"{'✓' if current == expected else '❌'} "
        f"{name:<22} {current}"
    )


    if current != expected:

        print(
            "Expected:"
        )

        print(
            expected
        )

        raise RuntimeError(
            "\nExact historical corrected target hash "
            "was NOT reproduced.\n\n"
            "Canonical targets have NOT been overwritten."
        )


print(
    "\n✓ EXACT corrected target layer reproduced."
)


# ==================================================================================================
# 7. COMMIT CORRECTED TARGETS
# ==================================================================================================

os.replace(
    FULL_TMP,
    TARGETS_FILE
)

os.replace(
    TRAIN_TMP,
    TRAIN_TARGETS_FILE
)

os.replace(
    VAL_TMP,
    VAL_TARGETS_FILE
)

os.replace(
    TEST_TMP,
    TEST_TARGETS_FILE
)


try:

    corrected.to_parquet(
        GENERATED_DIR
        /
        "11_4_synthetic_minute_targets.parquet",
        index=False
    )

except Exception as exc:

    print(
        "⚠ Parquet rewrite skipped:",
        str(
            exc
        )
    )


print(
    "✓ Corrected target files committed."
)


# ==================================================================================================
# 8. FINAL CANONICAL HASH CHECK
# ==================================================================================================

print("\n" + "=" * 120)
print("5. FINAL CANONICAL HASH CHECK")
print("=" * 120)


final_files = {

    "targets":
        TARGETS_FILE,

    "train_targets":
        TRAIN_TARGETS_FILE,

    "validation_targets":
        VAL_TARGETS_FILE,

    "test_targets":
        TEST_TARGETS_FILE,
}


for name, path in final_files.items():

    current = sha256_file(
        path
    )


    if current != EXPECTED[
        name
    ]:

        raise RuntimeError(
            f"Canonical {name} failed after commit."
        )


    print(
        f"✓ {name:<22} {current}"
    )


# ==================================================================================================
# 9. RESTORE 11.4 READINESS
# ==================================================================================================

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


readiness_11_4 = {

    "dataset":
        "Synthetic_NAF_v1",

    "phase":
        "11.4",

    "phase_11_4_status":
        "PASS",

    "readiness":
        "READY_FOR_PHASE_11_5",

    "forecast_horizons_minutes":
        FORECAST_HORIZONS,

    "target_contract": {

        "clean_origins_only":
            True,

        "attack_active_forecast_eligible":
            False,

        "recovery_forecast_eligible":
            False,

        "post_attack_forecast_eligible":
            False,

        "positive_requires_full_horizon":
            False,

        "negative_requires_full_horizon":
            True,

        "obsolete_h2_present":
            False,
    },

    "exact_accepted_hashes_verified":
        True,

    "predictor_sha256":
        EXPECTED[
            "predictors"
        ],

    "target_sha256":
        EXPECTED[
            "targets"
        ],

    "metadata_sha256":
        EXPECTED[
            "metadata"
        ],

    "train_target_sha256":
        EXPECTED[
            "train_targets"
        ],

    "validation_target_sha256":
        EXPECTED[
            "validation_targets"
        ],

    "test_target_sha256":
        EXPECTED[
            "test_targets"
        ],

    "traffic_regenerated":
        False,

    "targets_restored_after_stale_overwrite":
        True,

    "next_phase":
        (
            "PHASE_11_5_SYNTHETIC_DATASET_"
            "QUALITY_AND_FORECASTABILITY_AUDIT"
        ),
}


with open(
    READINESS_11_4_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        readiness_11_4,
        f,
        indent=4
    )


readiness_11_4_hash = sha256_file(
    READINESS_11_4_FILE
)


READINESS_11_4_HASH_FILE.write_text(
    readiness_11_4_hash
    +
    "\n",
    encoding="utf-8"
)


print(
    "\n✓ 11.4 readiness restored."
)


# ==================================================================================================
# 10. CHECK WHETHER PREVIOUS SUCCESSFUL 11.5 CAN BE REUSED
# ==================================================================================================

print("\n" + "=" * 120)
print("6. CHECK PREVIOUS PHASE 11.5 RESULT")
print("=" * 120)


reuse_11_5 = False


if READINESS_11_5_FILE.is_file():

    current_11_5_hash = sha256_file(
        READINESS_11_5_FILE
    )


    print(
        "11.5 readiness SHA256:"
    )

    print(
        current_11_5_hash
    )


    if (
        current_11_5_hash
        ==
        EXPECTED[
            "phase_11_5_readiness"
        ]
    ):

        with open(
            READINESS_11_5_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            r11_5 = json.load(
                f
            )


        if (
            r11_5.get(
                "phase_11_5_decision"
            )
            ==
            "USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY"

            and

            r11_5.get(
                "ready_for_phase_11_6_final_dataset_freeze"
            )
            is True

            and

            DECISION_11_5_FILE.is_file()

            and

            VALIDATION_11_5_FILE.is_file()
        ):

            reuse_11_5 = True


# ==================================================================================================
# FINAL
# ==================================================================================================

print("\n" + "=" * 120)
print("RECOVERY COMPLETE")
print("=" * 120)


print(
    "\n11.4 predictors:"
)

print(
    "    EXACT ✓"
)


print(
    "\n11.4 metadata:"
)

print(
    "    EXACT ✓"
)


print(
    "\n11.4 targets:"
)

print(
    "    RESTORED TO EXACT CORRECTED HASHES ✓"
)


print(
    "\n11.4 readiness:"
)

print(
    "    READY_FOR_PHASE_11_5 ✓"
)


if reuse_11_5:

    print(
        "\n11.5:"
    )

    print(
        "    PREVIOUS SUCCESSFUL AUDIT IS STILL EXACT AND REUSABLE ✓"
    )

    print(
        "\nDO NOT RERUN 11.5."
    )

    print(
        "\nNEXT:"
    )

    print(
        "    RUN PHASE 11.6 FINAL DATASET FREEZE"
    )

else:

    print(
        "\n11.5:"
    )

    print(
        "    PREVIOUS READINESS COULD NOT BE EXACTLY VERIFIED"
    )

    print(
        "\nNEXT:"
    )

    print(
        "    RERUN PHASE 11.5 ONCE"
    )


print("=" * 120)

In [ ]:
# ==================================================================================================
# PHASE 11.5 — SYNTHETIC DATASET QUALITY & FORECASTABILITY AUDIT — FINAL
# ==================================================================================================
#
# PURPOSE
# -------
# Aggressively audit Synthetic_NAF_v1 before final release and model development.
#
# IMPORTANT:
#
#   TRAIN      -> fit diagnostic probes
#   VALIDATION -> evaluate diagnostic probes
#   TEST       -> structural/hash/prevalence checks only
#
# TEST IS NOT USED FOR FITTED-PROBE PERFORMANCE.
#
#
# POSSIBLE FINAL DECISIONS
# ------------------------
#
#   USEFUL_SIGNAL
#   USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY
#   TOO_EASY
#   TOO_WEAK
#   CONTROL_FAILURE
#   REALISM_FAILURE
#   STRUCTURAL_FAILURE
#   REVIEW_MIXED_NONUSABLE_SIGNAL
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import warnings
import gc

import numpy as np
import pandas as pd

from scipy.stats import ks_2samp

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)

from sklearn.neighbors import NearestNeighbors


warnings.filterwarnings(
    "ignore"
)


print("=" * 120)
print("PHASE 11.5 — SYNTHETIC DATASET QUALITY & FORECASTABILITY AUDIT — FINAL")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

SPEC_DIR = (
    DATASET_ROOT
    / "spec"
)

CALIBRATION_DIR = (
    DATASET_ROOT
    / "calibration"
)

MANIFEST_DIR = (
    DATASET_ROOT
    / "manifests"
)

GENERATED_DIR = (
    DATASET_ROOT
    / "generated"
    / "11_4"
)

AUDIT_ROOT = (
    DATASET_ROOT
    / "audit"
)

AUDIT_DIR = (
    AUDIT_ROOT
    / "11_5"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config.json"
)

EPISODE_MANIFEST_FILE = (
    MANIFEST_DIR
    / "11_3_episode_manifest.csv"
)

READINESS_11_4_FILE = (
    AUDIT_ROOT
    / "11_4_READY_FOR_11_5.json"
)


PREDICTORS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_predictors.csv"
)

TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_targets.csv"
)

METADATA_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_metadata.csv"
)


REAL_BENIGN_POOL_FILE = (
    CALIBRATION_DIR
    / "11_1_real_benign_minute_pool.csv"
)

SAFE_BLOCK_ROWS_FILE = (
    CALIBRATION_DIR
    / "11_2_safe_benign_block_rows.csv"
)

SOURCE_EFFECT_FILE = (
    CALIBRATION_DIR
    / "11_1_source_family_log_effect_vectors.csv"
)


# ==================================================================================================
# FROZEN CONTRACT
# ==================================================================================================

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30
]

PRIMARY_HORIZONS = [
    5,
    10
]

HISTORY_WINDOW = 10

RANDOM_SEED = 20260930


EXPECTED = {

    "predictors":
        "9b0213a77d5320fa5377922ac37e1b104291c1d49e39d71f87e4333f96e2eb3b",

    "targets":
        "cebe5e388d2f2476bb066a9878de933e6f295abf118e1a429adb328a87750884",

    "metadata":
        "ce7e8723347a890ae90524727302ee83ed911ab4545b208f2e70e6b9cbd57ad7",
}


EXPECTED_POSITIVES = {

    ("TRAIN", 1):
        700,

    ("TRAIN", 5):
        3500,

    ("TRAIN", 10):
        7000,

    ("TRAIN", 15):
        10500,

    ("TRAIN", 30):
        21000,


    ("VALIDATION", 1):
        150,

    ("VALIDATION", 5):
        750,

    ("VALIDATION", 10):
        1500,

    ("VALIDATION", 15):
        2250,

    ("VALIDATION", 30):
        4500,


    ("TEST", 1):
        150,

    ("TEST", 5):
        750,

    ("TEST", 10):
        1500,

    ("TEST", 15):
        2250,

    ("TEST", 30):
        4500,
}


DOMAIN_FAMILIES = {

    "CIC_IDS2017": [
        "DOS_DDOS",
        "RECON_SCAN",
        "EXPLOIT_APP",
        "BRUTE_AUTH",
    ],

    "UNSW_NB15": [
        "DOS_DDOS",
        "RECON_SCAN",
        "EXPLOIT_APP",
        "FUZZING",
    ],
}


# ==================================================================================================
# AUDIT THRESHOLDS
#
# These are explicit audit rules, not universal scientific constants.
# ==================================================================================================

TH = {

    "dist_median_fold":
        1.50,

    "dist_iqr_fold":
        2.50,

    "dist_ks":
        0.35,

    "dist_pass_fraction":
        0.70,


    "corr_mean_abs_diff":
        0.30,


    "temporal_acf_abs_diff":
        0.35,

    "temporal_delta_fold":
        3.00,

    "temporal_pass_fraction":
        0.70,


    "source_shortcut_auc":
        0.90,


    "control_auc_alarm":
        0.75,

    "control_ap_lift_alarm":
        2.50,


    "too_weak_auc":
        0.60,

    "too_weak_ap_lift":
        1.50,


    "too_easy_auc":
        0.97,

    "too_easy_ap_lift":
        8.00,
}


# ==================================================================================================
# HELPERS
# ==================================================================================================

def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def require_file(
    path
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def save_csv(
    dataframe,
    path
):

    dataframe.to_csv(
        path,
        index=False,
        float_format="%.10g"
    )

    print(
        "✓",
        path
    )


def safe_auc(
    y,
    score
):

    y = np.asarray(
        y
    )

    if np.unique(
        y
    ).size < 2:

        return np.nan

    return float(
        roc_auc_score(
            y,
            score
        )
    )


def safe_ap(
    y,
    score
):

    y = np.asarray(
        y
    )

    if (
        y
        ==
        1
    ).sum() == 0:

        return np.nan

    return float(
        average_precision_score(
            y,
            score
        )
    )


def fold_diff(
    a,
    b,
    eps=1e-12
):

    a = abs(
        float(
            a
        )
    )

    b = abs(
        float(
            b
        )
    )

    if (
        a <= eps
        and
        b <= eps
    ):

        return 1.0

    return float(
        max(
            (a + eps)
            /
            (b + eps),

            (b + eps)
            /
            (a + eps)
        )
    )


def binary_probe():

    return Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "model",
                LogisticRegression(
                    max_iter=1200,
                    class_weight="balanced",
                    solver="lbfgs",
                    random_state=RANDOM_SEED
                )
            )
        ]
    )


def multiclass_probe():

    return Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "model",
                LogisticRegression(
                    max_iter=1600,
                    class_weight="balanced",
                    solver="lbfgs",
                    random_state=RANDOM_SEED
                )
            )
        ]
    )


# ==================================================================================================
# 11.5A — FROZEN INPUT GATE
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5A — FROZEN INPUT GATE")
print("=" * 120)


for path in [

    CONFIG_FILE,

    EPISODE_MANIFEST_FILE,

    READINESS_11_4_FILE,

    PREDICTORS_FILE,

    TARGETS_FILE,

    METADATA_FILE,

    REAL_BENIGN_POOL_FILE,

    SAFE_BLOCK_ROWS_FILE,

    SOURCE_EFFECT_FILE,

]:

    require_file(
        path
    )


with open(
    READINESS_11_4_FILE,
    "r",
    encoding="utf-8"
) as f:

    readiness_11_4 = json.load(
        f
    )


if readiness_11_4.get(
    "readiness"
) != "READY_FOR_PHASE_11_5":

    raise RuntimeError(
        "11.4 readiness gate does not authorize 11.5."
    )


if readiness_11_4.get(
    "forecast_horizons_minutes"
) != FORECAST_HORIZONS:

    raise RuntimeError(
        "11.4 readiness horizon contract mismatch."
    )


current_hashes = {

    "predictors":
        sha256_file(
            PREDICTORS_FILE
        ),

    "targets":
        sha256_file(
            TARGETS_FILE
        ),

    "metadata":
        sha256_file(
            METADATA_FILE
        ),
}


for key, current in current_hashes.items():

    expected = EXPECTED[
        key
    ]

    passed = (
        current
        ==
        expected
    )


    print(
        f"{'✓' if passed else '❌'} "
        f"{key:<12} "
        f"{current}"
    )


    if not passed:

        raise RuntimeError(
            f"Frozen 11.4 {key} hash mismatch."
        )


with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(
        f
    )


FINAL_PREDICTORS = list(
    CONFIG[
        "features"
    ][
        "final_predictor_whitelist"
    ]
)


if len(
    FINAL_PREDICTORS
) != 14:

    raise RuntimeError(
        "Predictor whitelist is not exactly 14 features."
    )


predictors = pd.read_csv(
    PREDICTORS_FILE,
    low_memory=False
)

targets = pd.read_csv(
    TARGETS_FILE,
    low_memory=False
)

metadata = pd.read_csv(
    METADATA_FILE,
    low_memory=False
)

episode_manifest = pd.read_csv(
    EPISODE_MANIFEST_FILE,
    low_memory=False
)

real_benign = pd.read_csv(
    REAL_BENIGN_POOL_FILE,
    low_memory=False
)

safe_block_rows = pd.read_csv(
    SAFE_BLOCK_ROWS_FILE,
    low_memory=False
)

source_effects = pd.read_csv(
    SOURCE_EFFECT_FILE,
    low_memory=False
)


KEYS = [
    "Episode_ID",
    "Minute_Index"
]


if not (
    len(
        predictors
    )
    ==
    len(
        targets
    )
    ==
    len(
        metadata
    )
    ==
    146_031
):

    raise RuntimeError(
        "11.4 row count mismatch."
    )


if not predictors[
    KEYS
].equals(
    targets[
        KEYS
    ]
):

    raise RuntimeError(
        "Predictor/target keys are not aligned."
    )


if not predictors[
    KEYS
].equals(
    metadata[
        KEYS
    ]
):

    raise RuntimeError(
        "Predictor/metadata keys are not aligned."
    )


if list(
    predictors.columns
) != [
    *KEYS,
    *FINAL_PREDICTORS
]:

    raise RuntimeError(
        "Predictor schema is not exactly keys + frozen 14 predictors."
    )


h2_columns = [

    c

    for c in targets.columns

    if (
        c.endswith(
            "_H2"
        )

        or

        "_H2_" in c
    )
]


if h2_columns:

    raise RuntimeError(
        f"Obsolete H2 columns found: {h2_columns}"
    )


print(
    f"✓ rows={len(predictors):,}"
)

print(
    f"✓ episodes={metadata['Episode_ID'].nunique():,}"
)

print(
    "✓ predictors=14"
)


# ==================================================================================================
# 11.5B — TARGET PREVALENCE / EXACT SUPPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5B — TARGET PREVALENCE / EXACT SUPPORT")
print("=" * 120)


prevalence_rows = []


for split in [

    "TRAIN",

    "VALIDATION",

    "TEST",

]:

    split_mask = metadata[
        "Split"
    ].eq(
        split
    )


    for horizon in FORECAST_HORIZONS:

        mask = (

            split_mask

            &

            targets[
                f"Forecast_Eligible_H{horizon}"
            ].eq(
                1
            )
        )


        y = targets.loc[
            mask,
            f"Onset_Within_H{horizon}"
        ].astype(
            int
        )


        positives = int(
            y.sum()
        )


        eligible_count = int(
            len(
                y
            )
        )


        expected_positive = EXPECTED_POSITIVES[
            (
                split,
                horizon
            )
        ]


        prevalence_rows.append(
            {
                "Split":
                    split,

                "Horizon_Minutes":
                    horizon,

                "Eligible_Origins":
                    eligible_count,

                "Positive_Origins":
                    positives,

                "Negative_Origins":
                    eligible_count
                    -
                    positives,

                "Positive_Rate":
                    (
                        positives
                        /
                        eligible_count
                        if eligible_count
                        else np.nan
                    ),

                "Expected_Positive_Origins":
                    expected_positive,

                "Positive_Count_Exact":
                    positives
                    ==
                    expected_positive,
            }
        )


target_prevalence = pd.DataFrame(
    prevalence_rows
)


print(
    target_prevalence.to_string(
        index=False
    )
)


positive_counts_exact = bool(
    target_prevalence[
        "Positive_Count_Exact"
    ].all()
)


# ==================================================================================================
# 11.5C — HARD SPLIT ISOLATION / PROVENANCE LEAKAGE
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5C — HARD SPLIT ISOLATION / PROVENANCE LEAKAGE AUDIT")
print("=" * 120)


required_provenance_columns = [

    "Episode_ID",

    "Calibration_Source",

    "Calibration_Group_ID",

    "Baseline_Block_ID",

    "Split",
]


missing_provenance = [

    c

    for c in required_provenance_columns

    if c not in metadata.columns
]


if missing_provenance:

    raise RuntimeError(
        f"Metadata missing provenance columns: {missing_provenance}"
    )


def split_sets(
    column,
    composite_with_source=False
):

    output = {}


    for split in [

        "TRAIN",

        "VALIDATION",

        "TEST",

    ]:

        mask = metadata[
            "Split"
        ].eq(
            split
        )


        if composite_with_source:

            values = (

                metadata.loc[
                    mask,
                    "Calibration_Source"
                ].astype(
                    str
                )

                +

                "::"

                +

                metadata.loc[
                    mask,
                    column
                ].astype(
                    str
                )
            )

        else:

            values = metadata.loc[
                mask,
                column
            ].astype(
                str
            )


        output[
            split
        ] = set(
            values.tolist()
        )


    return output


episode_sets = split_sets(
    "Episode_ID"
)

group_sets = split_sets(
    "Calibration_Group_ID",
    composite_with_source=True
)

block_sets = split_sets(
    "Baseline_Block_ID",
    composite_with_source=True
)


def pair_overlap_rows(
    entity_name,
    split_set_dict
):

    rows = []


    for split_a, split_b in [

        (
            "TRAIN",
            "VALIDATION"
        ),

        (
            "TRAIN",
            "TEST"
        ),

        (
            "VALIDATION",
            "TEST"
        ),

    ]:

        overlap = (

            split_set_dict[
                split_a
            ]

            &

            split_set_dict[
                split_b
            ]
        )


        rows.append(
            {
                "Entity":
                    entity_name,

                "Split_A":
                    split_a,

                "Split_B":
                    split_b,

                "Overlap_Count":
                    len(
                        overlap
                    ),
            }
        )


    return rows


hard_leakage = pd.DataFrame(

    pair_overlap_rows(
        "Episode_ID",
        episode_sets
    )

    +

    pair_overlap_rows(
        "Calibration_Source::Calibration_Group_ID",
        group_sets
    )

    +

    pair_overlap_rows(
        "Calibration_Source::Baseline_Block_ID",
        block_sets
    )
)


print(
    hard_leakage.to_string(
        index=False
    )
)


hard_split_isolation_pass = bool(
    (
        hard_leakage[
            "Overlap_Count"
        ]
        ==
        0
    ).all()
)


# ==================================================================================================
# FULL EPISODE PREDICTOR HASH DUPLICATES
# ==================================================================================================

episode_hash_rows = []


for episode_id, indices in metadata.groupby(
    "Episode_ID",
    sort=False
).groups.items():

    array = predictors.loc[
        indices,
        FINAL_PREDICTORS
    ].to_numpy()


    sequence_hash = hashlib.sha256(
        np.ascontiguousarray(
            array
        ).tobytes()
    ).hexdigest()


    episode_hash_rows.append(
        {
            "Episode_ID":
                str(
                    episode_id
                ),

            "Split":
                str(
                    metadata.loc[
                        indices[
                            0
                        ],
                        "Split"
                    ]
                ),

            "Predictor_Sequence_SHA256":
                sequence_hash,
        }
    )


episode_hashes = pd.DataFrame(
    episode_hash_rows
)


cross_split_episode_duplicate_hashes = 0


for sequence_hash, group in episode_hashes.groupby(
    "Predictor_Sequence_SHA256"
):

    if group[
        "Split"
    ].nunique() > 1:

        cross_split_episode_duplicate_hashes += 1


print(
    "\nCross-split duplicate full predictor sequences:",
    cross_split_episode_duplicate_hashes
)


if cross_split_episode_duplicate_hashes != 0:

    hard_split_isolation_pass = False


# ==================================================================================================
# MINUTE-LEVEL EQUALITY IS ONLY A WARNING
# ==================================================================================================

row_hash = pd.util.hash_pandas_object(

    predictors[
        FINAL_PREDICTORS
    ].round(
        8
    ),

    index=False
)


row_sets = {}


for split in [

    "TRAIN",

    "VALIDATION",

    "TEST",

]:

    row_sets[
        split
    ] = set(

        row_hash.loc[
            metadata[
                "Split"
            ].eq(
                split
            )
        ].tolist()
    )


row_overlap_rows = []


for split_a, split_b in [

    (
        "TRAIN",
        "VALIDATION"
    ),

    (
        "TRAIN",
        "TEST"
    ),

    (
        "VALIDATION",
        "TEST"
    ),

]:

    row_overlap_rows.append(
        {
            "Split_A":
                split_a,

            "Split_B":
                split_b,

            "Rounded8_Exact_Row_Overlap":
                len(

                    row_sets[
                        split_a
                    ]

                    &

                    row_sets[
                        split_b
                    ]
                ),
        }
    )


row_overlap_warning = pd.DataFrame(
    row_overlap_rows
)


print(
    "\nMinute-level equality diagnostic "
    "(WARNING ONLY — not treated as hard leakage):"
)


print(
    row_overlap_warning.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.5D — NEAREST-NEIGHBOUR DIAGNOSTIC
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5D — CROSS-SPLIT NEAREST-NEIGHBOUR DIAGNOSTIC — WARNING ONLY")
print("=" * 120)


rng = np.random.default_rng(
    RANDOM_SEED
)

NN_SAMPLE = 4000


def sampled_X(
    split
):

    indices = np.where(

        metadata[
            "Split"
        ].to_numpy()

        ==
        split
    )[
        0
    ]


    if len(
        indices
    ) > NN_SAMPLE:

        indices = rng.choice(
            indices,
            NN_SAMPLE,
            replace=False
        )


    return np.log1p(

        predictors.iloc[
            indices
        ][
            FINAL_PREDICTORS
        ].to_numpy(
            dtype=float
        )
    )


X_train_nn = sampled_X(
    "TRAIN"
)

X_validation_nn = sampled_X(
    "VALIDATION"
)

X_test_nn = sampled_X(
    "TEST"
)


nn_scaler = StandardScaler()


X_train_nn_z = nn_scaler.fit_transform(
    X_train_nn
)

X_validation_nn_z = nn_scaler.transform(
    X_validation_nn
)

X_test_nn_z = nn_scaler.transform(
    X_test_nn
)


within_model = NearestNeighbors(
    n_neighbors=2
).fit(
    X_train_nn_z
)


within_distance = within_model.kneighbors(
    X_train_nn_z,
    return_distance=True
)[
    0
][
    :,
    1
]


cross_model = NearestNeighbors(
    n_neighbors=1
).fit(
    X_train_nn_z
)


validation_distance = cross_model.kneighbors(
    X_validation_nn_z,
    return_distance=True
)[
    0
][
    :,
    0
]


test_distance = cross_model.kneighbors(
    X_test_nn_z,
    return_distance=True
)[
    0
][
    :,
    0
]


within_median = float(
    np.median(
        within_distance
    )
)


nn_audit = pd.DataFrame(
    [
        {
            "Split":
                "VALIDATION",

            "Within_Train_Median_NN":
                within_median,

            "Cross_Median_NN":
                float(
                    np.median(
                        validation_distance
                    )
                ),

            "Cross_Min_NN":
                float(
                    np.min(
                        validation_distance
                    )
                ),

            "Cross_To_Within_Median_Ratio":
                (
                    float(
                        np.median(
                            validation_distance
                        )
                        /
                        within_median
                    )

                    if within_median > 0

                    else np.inf
                ),

            "Near_Zero_Alarm":
                bool(
                    np.min(
                        validation_distance
                    )
                    <
                    1e-8
                ),
        },

        {
            "Split":
                "TEST",

            "Within_Train_Median_NN":
                within_median,

            "Cross_Median_NN":
                float(
                    np.median(
                        test_distance
                    )
                ),

            "Cross_Min_NN":
                float(
                    np.min(
                        test_distance
                    )
                ),

            "Cross_To_Within_Median_Ratio":
                (
                    float(
                        np.median(
                            test_distance
                        )
                        /
                        within_median
                    )

                    if within_median > 0

                    else np.inf
                ),

            "Near_Zero_Alarm":
                bool(
                    np.min(
                        test_distance
                    )
                    <
                    1e-8
                ),
        },
    ]
)


print(
    nn_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.5E — REAL vs SYNTHETIC BENIGN DISTRIBUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5E — REAL vs SYNTHETIC BENIGN DISTRIBUTION SIMILARITY")
print("=" * 120)


required_real = [

    "Calibration_Source",

    *FINAL_PREDICTORS,
]


missing_real = [

    c

    for c in required_real

    if c not in real_benign.columns
]


if missing_real:

    raise RuntimeError(
        f"Real benign pool missing columns: {missing_real}"
    )


synthetic_benign_mask = (
    metadata[
        "Episode_Type"
    ].eq(
        "BENIGN_ONLY"
    )
)


distribution_rows = []


rng = np.random.default_rng(
    RANDOM_SEED
)


for source in [

    "CIC_IDS2017",

    "UNSW_NB15",

]:

    real_source = real_benign.loc[
        real_benign[
            "Calibration_Source"
        ].eq(
            source
        )
    ]


    synthetic_source = predictors.loc[
        synthetic_benign_mask
        &
        metadata[
            "Calibration_Source"
        ].eq(
            source
        )
    ]


    if (
        len(
            real_source
        )
        ==
        0

        or

        len(
            synthetic_source
        )
        ==
        0
    ):

        raise RuntimeError(
            f"Missing benign comparison rows for {source}"
        )


    for feature in FINAL_PREDICTORS:

        real_values = pd.to_numeric(

            real_source[
                feature
            ],

            errors="coerce"

        ).dropna().to_numpy(
            dtype=float
        )


        synthetic_values = pd.to_numeric(

            synthetic_source[
                feature
            ],

            errors="coerce"

        ).dropna().to_numpy(
            dtype=float
        )


        real_q25, real_median, real_q75 = np.quantile(
            real_values,
            [
                0.25,
                0.50,
                0.75
            ]
        )


        syn_q25, syn_median, syn_q75 = np.quantile(
            synthetic_values,
            [
                0.25,
                0.50,
                0.75
            ]
        )


        median_fold = float(

            np.exp(

                abs(

                    np.log1p(
                        syn_median
                    )

                    -

                    np.log1p(
                        real_median
                    )
                )
            )
        )


        iqr_fold = fold_diff(

            real_q75
            -
            real_q25,

            syn_q75
            -
            syn_q25
        )


        sample_n = min(

            len(
                synthetic_values
            ),

            max(

                len(
                    real_values
                )
                *
                5,

                len(
                    real_values
                )
            )
        )


        if sample_n < len(
            synthetic_values
        ):

            synthetic_sample = rng.choice(
                synthetic_values,
                sample_n,
                replace=False
            )

        else:

            synthetic_sample = synthetic_values


        ks_stat = float(

            ks_2samp(

                np.log1p(
                    real_values
                ),

                np.log1p(
                    synthetic_sample
                )
            ).statistic
        )


        passed = bool(

            median_fold
            <=
            TH[
                "dist_median_fold"
            ]

            and

            iqr_fold
            <=
            TH[
                "dist_iqr_fold"
            ]

            and

            ks_stat
            <=
            TH[
                "dist_ks"
            ]
        )


        distribution_rows.append(
            {
                "Calibration_Source":
                    source,

                "Feature":
                    feature,

                "Real_Median":
                    float(
                        real_median
                    ),

                "Synthetic_Median":
                    float(
                        syn_median
                    ),

                "Median_Fold_Difference":
                    median_fold,

                "Real_IQR":
                    float(
                        real_q75
                        -
                        real_q25
                    ),

                "Synthetic_IQR":
                    float(
                        syn_q75
                        -
                        syn_q25
                    ),

                "IQR_Fold_Difference":
                    iqr_fold,

                "KS_Log1p":
                    ks_stat,

                "Distribution_Cell_Pass":
                    passed,
            }
        )


distribution_similarity = pd.DataFrame(
    distribution_rows
)


distribution_summary = (

    distribution_similarity

    .groupby(
        "Calibration_Source",
        observed=True
    )

    .agg(

        Features=(
            "Feature",
            "size"
        ),

        Passed_Features=(
            "Distribution_Cell_Pass",
            "sum"
        ),

        Pass_Fraction=(
            "Distribution_Cell_Pass",
            "mean"
        ),

        Median_Median_Fold=(
            "Median_Fold_Difference",
            "median"
        ),

        Median_IQR_Fold=(
            "IQR_Fold_Difference",
            "median"
        ),

        Median_KS=(
            "KS_Log1p",
            "median"
        ),
    )

    .reset_index()
)


print(
    distribution_summary.to_string(
        index=False
    )
)


distribution_gate_pass = bool(

    (
        distribution_summary[
            "Pass_Fraction"
        ]

        >=

        TH[
            "dist_pass_fraction"
        ]
    ).all()
)


distribution_catastrophic = bool(

    (
        distribution_summary[
            "Pass_Fraction"
        ]

        <
        0.35
    ).any()
)


# ==================================================================================================
# 11.5F — CORRELATION PRESERVATION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5F — SOURCE-SPECIFIC CORRELATION PRESERVATION")
print("=" * 120)


correlation_rows = []


for source in [

    "CIC_IDS2017",

    "UNSW_NB15",

]:

    real_source = real_benign.loc[
        real_benign[
            "Calibration_Source"
        ].eq(
            source
        ),
        FINAL_PREDICTORS
    ].astype(
        float
    )


    synthetic_source = predictors.loc[
        synthetic_benign_mask
        &
        metadata[
            "Calibration_Source"
        ].eq(
            source
        ),
        FINAL_PREDICTORS
    ].astype(
        float
    )


    real_corr = real_source.corr(
        method="spearman"
    ).fillna(
        0
    ).to_numpy()


    synthetic_corr = synthetic_source.corr(
        method="spearman"
    ).fillna(
        0
    ).to_numpy()


    upper = np.triu_indices(
        len(
            FINAL_PREDICTORS
        ),
        1
    )


    difference = np.abs(

        real_corr[
            upper
        ]

        -

        synthetic_corr[
            upper
        ]
    )


    mean_difference = float(
        difference.mean()
    )


    correlation_rows.append(
        {
            "Calibration_Source":
                source,

            "Pair_Count":
                len(
                    difference
                ),

            "Mean_Absolute_Correlation_Difference":
                mean_difference,

            "Median_Absolute_Correlation_Difference":
                float(
                    np.median(
                        difference
                    )
                ),

            "Max_Absolute_Correlation_Difference":
                float(
                    np.max(
                        difference
                    )
                ),

            "Correlation_Preservation_Pass":
                mean_difference
                <=
                TH[
                    "corr_mean_abs_diff"
                ],
        }
    )


correlation_preservation = pd.DataFrame(
    correlation_rows
)


print(
    correlation_preservation.to_string(
        index=False
    )
)


correlation_gate_pass = bool(
    correlation_preservation[
        "Correlation_Preservation_Pass"
    ].all()
)


correlation_catastrophic = bool(

    (
        correlation_preservation[
            "Mean_Absolute_Correlation_Difference"
        ]

        >
        0.50
    ).any()
)


# ==================================================================================================
# 11.5G — TEMPORAL REALISM
#
# IMPORTANT FIX:
#
# Compare like-for-like:
#
#     real 5-min block
#          vs
#     synthetic generated block occurrence
#
# NOT real block vs full 100-140 minute episode.
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5G — TEMPORAL REALISM — REAL BLOCK vs SYNTHETIC BLOCK-OCCURRENCE")
print("=" * 120)


required_safe = [

    "Calibration_Source",

    "Safe_Benign_Block_ID",

    "Block_Row_Position",

    *FINAL_PREDICTORS,
]


missing_safe = [

    c

    for c in required_safe

    if c not in safe_block_rows.columns
]


if missing_safe:

    raise RuntimeError(
        f"Safe block rows missing columns: {missing_safe}"
    )


for column in [

    "Episode_ID",

    "Block_Occurrence_Index",

    "Block_Row_Position",

    "Calibration_Source",

    "Episode_Type",

]:

    if column not in metadata.columns:

        raise RuntimeError(
            f"Metadata missing temporal-audit column: {column}"
        )


def adjacent_log_pairs(
    dataframe,
    group_column,
    order_column,
    feature
):

    ordered = dataframe.sort_values(
        [
            group_column,
            order_column
        ]
    )


    values = np.log1p(

        pd.to_numeric(

            ordered[
                feature
            ],

            errors="coerce"

        ).to_numpy(
            dtype=float
        )
    )


    groups = ordered[
        group_column
    ].astype(
        str
    ).to_numpy()


    same_group = (
        groups[
            1:
        ]
        ==
        groups[
            :-1
        ]
    )


    previous = values[
        :-1
    ][
        same_group
    ]


    following = values[
        1:
    ][
        same_group
    ]


    finite = (

        np.isfinite(
            previous
        )

        &

        np.isfinite(
            following
        )
    )


    return (
        previous[
            finite
        ],
        following[
            finite
        ]
    )


def lagcorr(
    previous,
    following
):

    if (
        len(
            previous
        )
        <
        3

        or

        np.std(
            previous
        )
        ==
        0

        or

        np.std(
            following
        )
        ==
        0
    ):

        return 0.0


    return float(

        np.corrcoef(
            previous,
            following
        )[
            0,
            1
        ]
    )


temporal_rows = []


for source in [

    "CIC_IDS2017",

    "UNSW_NB15",

]:

    real_source = safe_block_rows.loc[
        safe_block_rows[
            "Calibration_Source"
        ].eq(
            source
        )
    ].copy()


    synthetic_mask = (

        synthetic_benign_mask

        &

        metadata[
            "Calibration_Source"
        ].eq(
            source
        )
    )


    synthetic_source = predictors.loc[
        synthetic_mask
    ].copy()


    synthetic_source[
        "Episode_ID"
    ] = metadata.loc[
        synthetic_mask,
        "Episode_ID"
    ].astype(
        str
    ).to_numpy()


    synthetic_source[
        "Block_Occurrence_Index"
    ] = metadata.loc[
        synthetic_mask,
        "Block_Occurrence_Index"
    ].to_numpy()


    synthetic_source[
        "Block_Row_Position"
    ] = metadata.loc[
        synthetic_mask,
        "Block_Row_Position"
    ].to_numpy()


    synthetic_source[
        "_Occurrence_Key"
    ] = (

        synthetic_source[
            "Episode_ID"
        ]

        +

        "::"

        +

        synthetic_source[
            "Block_Occurrence_Index"
        ].astype(
            str
        )
    )


    for feature in FINAL_PREDICTORS:

        real_previous, real_following = adjacent_log_pairs(

            real_source,

            "Safe_Benign_Block_ID",

            "Block_Row_Position",

            feature
        )


        synthetic_previous, synthetic_following = adjacent_log_pairs(

            synthetic_source,

            "_Occurrence_Key",

            "Block_Row_Position",

            feature
        )


        real_acf = lagcorr(
            real_previous,
            real_following
        )


        synthetic_acf = lagcorr(
            synthetic_previous,
            synthetic_following
        )


        acf_difference = abs(
            real_acf
            -
            synthetic_acf
        )


        real_delta = (

            float(

                np.median(

                    np.abs(
                        real_following
                        -
                        real_previous
                    )
                )
            )

            if len(
                real_previous
            )

            else np.nan
        )


        synthetic_delta = (

            float(

                np.median(

                    np.abs(
                        synthetic_following
                        -
                        synthetic_previous
                    )
                )
            )

            if len(
                synthetic_previous
            )

            else np.nan
        )


        delta_fold = fold_diff(
            real_delta,
            synthetic_delta,
            eps=1e-6
        )


        passed = bool(

            acf_difference
            <=
            TH[
                "temporal_acf_abs_diff"
            ]

            and

            delta_fold
            <=
            TH[
                "temporal_delta_fold"
            ]
        )


        temporal_rows.append(
            {
                "Calibration_Source":
                    source,

                "Feature":
                    feature,

                "Real_WithinBlock_Lag1":
                    real_acf,

                "Synthetic_WithinOccurrence_Lag1":
                    synthetic_acf,

                "Lag1_Absolute_Difference":
                    acf_difference,

                "Real_Median_Abs_Log1p_Delta":
                    real_delta,

                "Synthetic_Median_Abs_Log1p_Delta":
                    synthetic_delta,

                "Delta_Fold_Difference":
                    delta_fold,

                "Temporal_Cell_Pass":
                    passed,
            }
        )


temporal_realism = pd.DataFrame(
    temporal_rows
)


temporal_summary = (

    temporal_realism

    .groupby(
        "Calibration_Source",
        observed=True
    )

    .agg(

        Features=(
            "Feature",
            "size"
        ),

        Passed_Features=(
            "Temporal_Cell_Pass",
            "sum"
        ),

        Pass_Fraction=(
            "Temporal_Cell_Pass",
            "mean"
        ),

        Mean_Lag1_Abs_Difference=(
            "Lag1_Absolute_Difference",
            "mean"
        ),

        Median_Delta_Fold=(
            "Delta_Fold_Difference",
            "median"
        ),
    )

    .reset_index()
)


print(
    temporal_summary.to_string(
        index=False
    )
)


temporal_gate_pass = bool(

    (
        temporal_summary[
            "Pass_Fraction"
        ]

        >=

        TH[
            "temporal_pass_fraction"
        ]
    ).all()
)


temporal_catastrophic = bool(

    (
        temporal_summary[
            "Pass_Fraction"
        ]

        <
        0.35
    ).any()
)


# ==================================================================================================
# 11.5H — SOURCE SHORTCUT STRENGTH
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5H — SOURCE SHORTCUT STRENGTH")
print("=" * 120)


train_source_mask = (

    synthetic_benign_mask

    &

    metadata[
        "Split"
    ].eq(
        "TRAIN"
    )
)


validation_source_mask = (

    synthetic_benign_mask

    &

    metadata[
        "Split"
    ].eq(
        "VALIDATION"
    )
)


X_source_train = np.log1p(

    predictors.loc[
        train_source_mask,
        FINAL_PREDICTORS
    ].to_numpy(
        dtype=float
    )
)


X_source_validation = np.log1p(

    predictors.loc[
        validation_source_mask,
        FINAL_PREDICTORS
    ].to_numpy(
        dtype=float
    )
)


y_source_train = (

    metadata.loc[
        train_source_mask,
        "Calibration_Source"
    ].eq(
        "UNSW_NB15"
    )

).astype(
    int
).to_numpy()


y_source_validation = (

    metadata.loc[
        validation_source_mask,
        "Calibration_Source"
    ].eq(
        "UNSW_NB15"
    )

).astype(
    int
).to_numpy()


source_model = binary_probe()


source_model.fit(
    X_source_train,
    y_source_train
)


source_validation_score = source_model.predict_proba(
    X_source_validation
)[
    :,
    1
]


source_auc = safe_auc(
    y_source_validation,
    source_validation_score
)


source_ap = safe_ap(
    y_source_validation,
    source_validation_score
)


source_shortcut_high = bool(
    source_auc
    >=
    TH[
        "source_shortcut_auc"
    ]
)


source_shortcut = pd.DataFrame(
    [
        {
            "Task":
                "Predict source from synthetic benign traffic",

            "Train_Rows":
                len(
                    y_source_train
                ),

            "Validation_Rows":
                len(
                    y_source_validation
                ),

            "ROC_AUC":
                source_auc,

            "Average_Precision":
                source_ap,

            "Source_Shortcut_Level":
                (
                    "HIGH"

                    if source_shortcut_high

                    else "MODERATE_OR_LOW"
                ),
        }
    ]
)


print(
    source_shortcut.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.5I — CAUSAL 10-MINUTE HISTORY FEATURES
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5I — BUILD CAUSAL 10-MINUTE DIAGNOSTIC HISTORY FEATURES")
print("=" * 120)


log_features = np.log1p(

    predictors[
        FINAL_PREDICTORS
    ].astype(
        float
    )
)


group_input = log_features.copy()


group_input.insert(
    0,
    "Episode_ID",
    predictors[
        "Episode_ID"
    ].to_numpy()
)


grouped = group_input.groupby(
    "Episode_ID",
    sort=False,
    observed=True
)


lag1 = grouped[
    FINAL_PREDICTORS
].shift(
    1
)


lag9 = grouped[
    FINAL_PREDICTORS
].shift(
    HISTORY_WINDOW
    -
    1
)


mean10 = (

    grouped[
        FINAL_PREDICTORS
    ]

    .rolling(
        HISTORY_WINDOW,
        min_periods=HISTORY_WINDOW
    )

    .mean()

    .reset_index(
        level=0,
        drop=True
    )
)


std10 = (

    grouped[
        FINAL_PREDICTORS
    ]

    .rolling(
        HISTORY_WINDOW,
        min_periods=HISTORY_WINDOW
    )

    .std(
        ddof=0
    )

    .reset_index(
        level=0,
        drop=True
    )
)


probe_parts = []


for dataframe, prefix in [

    (
        log_features,
        "CUR"
    ),

    (
        mean10,
        "MEAN10"
    ),

    (
        std10,
        "STD10"
    ),

    (
        log_features
        -
        lag1,
        "DELTA1"
    ),

    (
        (
            log_features
            -
            lag9
        )
        /
        (
            HISTORY_WINDOW
            -
            1
        ),
        "SLOPE9"
    ),

]:

    transformed = dataframe.copy()


    transformed.columns = [

        f"{prefix}::{column}"

        for column in FINAL_PREDICTORS
    ]


    probe_parts.append(
        transformed
    )


probe_features = pd.concat(
    probe_parts,
    axis=1
)


history_ready = probe_features.notna().all(
    axis=1
)


print(
    "Diagnostic features:",
    probe_features.shape[
        1
    ]
)

print(
    "Complete-history rows:",
    f"{int(history_ready.sum()):,}"
)

print(
    "TEST rows used for fitted probe performance: 0"
)


# ==================================================================================================
# 11.5J — SOURCE-SPECIFIC BINARY FORECASTABILITY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5J — SOURCE-SPECIFIC TRAIN -> VALIDATION FORECASTABILITY")
print("=" * 120)


binary_probe_rows = []

validation_score_frames = []

probe_thresholds = {}


for source in [

    "CIC_IDS2017",

    "UNSW_NB15",

]:

    source_mask = metadata[
        "Calibration_Source"
    ].eq(
        source
    )


    for horizon in PRIMARY_HORIZONS:

        eligible = targets[
            f"Forecast_Eligible_H{horizon}"
        ].eq(
            1
        )


        train_mask = (

            source_mask

            &

            metadata[
                "Split"
            ].eq(
                "TRAIN"
            )

            &

            eligible

            &

            history_ready
        )


        validation_mask = (

            source_mask

            &

            metadata[
                "Split"
            ].eq(
                "VALIDATION"
            )

            &

            eligible

            &

            history_ready
        )


        X_train = probe_features.loc[
            train_mask
        ].to_numpy(
            dtype=float
        )


        X_validation = probe_features.loc[
            validation_mask
        ].to_numpy(
            dtype=float
        )


        y_train = targets.loc[
            train_mask,
            f"Onset_Within_H{horizon}"
        ].astype(
            int
        ).to_numpy()


        y_validation = targets.loc[
            validation_mask,
            f"Onset_Within_H{horizon}"
        ].astype(
            int
        ).to_numpy()


        if (
            np.unique(
                y_train
            ).size
            <
            2

            or

            np.unique(
                y_validation
            ).size
            <
            2
        ):

            raise RuntimeError(
                f"Probe lacks both classes: {source}, H{horizon}"
            )


        model = binary_probe()


        model.fit(
            X_train,
            y_train
        )


        train_score = model.predict_proba(
            X_train
        )[
            :,
            1
        ]


        validation_score = model.predict_proba(
            X_validation
        )[
            :,
            1
        ]


        threshold = float(

            np.quantile(

                train_score[
                    y_train
                    ==
                    0
                ],

                0.95
            )
        )


        probe_thresholds[
            (
                source,
                horizon
            )
        ] = threshold


        prevalence = float(
            y_validation.mean()
        )


        ap = safe_ap(
            y_validation,
            validation_score
        )


        auc = safe_auc(
            y_validation,
            validation_score
        )


        ap_lift = (

            ap
            /
            prevalence

            if prevalence > 0

            else np.nan
        )


        prediction = (
            validation_score
            >=
            threshold
        ).astype(
            int
        )


        binary_probe_rows.append(
            {
                "Calibration_Source":
                    source,

                "Horizon_Minutes":
                    horizon,

                "History_Window_Minutes":
                    HISTORY_WINDOW,

                "Train_Rows":
                    len(
                        y_train
                    ),

                "Validation_Rows":
                    len(
                        y_validation
                    ),

                "Validation_Positive_Rate":
                    prevalence,

                "ROC_AUC":
                    auc,

                "Average_Precision":
                    ap,

                "AP_Lift_Over_Prevalence":
                    ap_lift,

                "Threshold_Train_Negative_95pct":
                    threshold,

                "Precision_At_Threshold":
                    float(

                        precision_score(
                            y_validation,
                            prediction,
                            zero_division=0
                        )
                    ),

                "Recall_At_Threshold":
                    float(

                        recall_score(
                            y_validation,
                            prediction,
                            zero_division=0
                        )
                    ),

                "F1_At_Threshold":
                    float(

                        f1_score(
                            y_validation,
                            prediction,
                            zero_division=0
                        )
                    ),
            }
        )


        validation_score_frames.append(
            pd.DataFrame(
                {
                    "Row_Index":
                        np.where(
                            validation_mask.to_numpy()
                        )[
                            0
                        ],

                    "Episode_ID":
                        metadata.loc[
                            validation_mask,
                            "Episode_ID"
                        ].to_numpy(),

                    "Minute_Index":
                        metadata.loc[
                            validation_mask,
                            "Minute_Index"
                        ].to_numpy(),

                    "Calibration_Source":
                        source,

                    "Horizon_Minutes":
                        horizon,

                    "Episode_Type":
                        metadata.loc[
                            validation_mask,
                            "Episode_Type"
                        ].to_numpy(),

                    "Precursor_Regime":
                        metadata.loc[
                            validation_mask,
                            "Precursor_Regime"
                        ].fillna(
                            "NONE"
                        ).to_numpy(),

                    "Y":
                        y_validation,

                    "Score":
                        validation_score,

                    "Threshold":
                        threshold,

                    "Minutes_To_Next_Onset":
                        pd.to_numeric(

                            targets.loc[
                                validation_mask,
                                "Minutes_To_Next_Onset"
                            ],

                            errors="coerce"

                        ).to_numpy(
                            dtype=float
                        ),
                }
            )
        )


binary_forecast_probe = pd.DataFrame(
    binary_probe_rows
)


validation_scores = pd.concat(
    validation_score_frames,
    ignore_index=True
)


print(
    binary_forecast_probe.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.5K — NO_PRECURSOR NEGATIVE CONTROL
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5K — NO_PRECURSOR NEGATIVE CONTROL")
print("=" * 120)


control_rows = []


for source in [

    "CIC_IDS2017",

    "UNSW_NB15",

]:

    for horizon in PRIMARY_HORIZONS:

        subset = validation_scores.loc[

            validation_scores[
                "Calibration_Source"
            ].eq(
                source
            )

            &

            validation_scores[
                "Horizon_Minutes"
            ].eq(
                horizon
            )

            &

            (
                validation_scores[
                    "Episode_Type"
                ].eq(
                    "BENIGN_ONLY"
                )

                |

                validation_scores[
                    "Precursor_Regime"
                ].eq(
                    "NO_PRECURSOR"
                )
            )

        ].copy()


        if np.unique(
            subset[
                "Y"
            ]
        ).size < 2:

            raise RuntimeError(
                f"NO_PRECURSOR control lacks both classes: "
                f"{source}, H{horizon}"
            )


        prevalence = float(
            subset[
                "Y"
            ].mean()
        )


        auc = safe_auc(
            subset[
                "Y"
            ],
            subset[
                "Score"
            ]
        )


        ap = safe_ap(
            subset[
                "Y"
            ],
            subset[
                "Score"
            ]
        )


        ap_lift = (

            ap
            /
            prevalence

            if prevalence > 0

            else np.nan
        )


        # Alarm only when BOTH discrimination and AP uplift are
        # materially high. This avoids overreacting to one noisy metric.

        control_alarm = bool(

            auc
            >
            TH[
                "control_auc_alarm"
            ]

            and

            ap_lift
            >
            TH[
                "control_ap_lift_alarm"
            ]
        )


        control_rows.append(
            {
                "Calibration_Source":
                    source,

                "Horizon_Minutes":
                    horizon,

                "Rows":
                    len(
                        subset
                    ),

                "Positive_Rate":
                    prevalence,

                "ROC_AUC":
                    auc,

                "Average_Precision":
                    ap,

                "AP_Lift_Over_Prevalence":
                    ap_lift,

                "Control_Alarm":
                    control_alarm,

                "Control_Pass":
                    not control_alarm,
            }
        )


no_precursor_control = pd.DataFrame(
    control_rows
)


print(
    no_precursor_control.to_string(
        index=False
    )
)


no_precursor_control_pass = bool(
    no_precursor_control[
        "Control_Pass"
    ].all()
)


# ==================================================================================================
# 11.5L — PRECURSOR REGIME RESPONSE + EVENT LEAD TIME
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5L — PRECURSOR REGIME RESPONSE / EVENT LEAD TIME")
print("=" * 120)


regime_rows = []


positive_scores = validation_scores.loc[
    validation_scores[
        "Y"
    ].eq(
        1
    )
].copy()


for (
    source,
    horizon,
    regime
), group in positive_scores.groupby(
    [
        "Calibration_Source",
        "Horizon_Minutes",
        "Precursor_Regime"
    ],
    observed=True
):

    threshold = probe_thresholds[
        (
            source,
            int(
                horizon
            )
        )
    ]


    event_total = int(
        group[
            "Episode_ID"
        ].nunique()
    )


    event_warned = int(

        group.loc[
            group[
                "Score"
            ]
            >=
            threshold,
            "Episode_ID"
        ].nunique()
    )


    regime_rows.append(
        {
            "Calibration_Source":
                source,

            "Horizon_Minutes":
                int(
                    horizon
                ),

            "Precursor_Regime":
                regime,

            "Positive_Origins":
                len(
                    group
                ),

            "Events":
                event_total,

            "Mean_Positive_Score":
                float(
                    group[
                        "Score"
                    ].mean()
                ),

            "Median_Positive_Score":
                float(
                    group[
                        "Score"
                    ].median()
                ),

            "Event_Recall_At_Threshold":
                (
                    event_warned
                    /
                    event_total

                    if event_total

                    else np.nan
                ),
        }
    )


precursor_regime_probe = pd.DataFrame(
    regime_rows
)


print(
    precursor_regime_probe.to_string(
        index=False
    )
)


strong_score = precursor_regime_probe.loc[
    precursor_regime_probe[
        "Precursor_Regime"
    ].isin(
        [
            "SHORT_STRONG",
            "GRADUAL_STRONG"
        ]
    ),
    "Median_Positive_Score"
].median()


no_precursor_score = precursor_regime_probe.loc[
    precursor_regime_probe[
        "Precursor_Regime"
    ].eq(
        "NO_PRECURSOR"
    ),
    "Median_Positive_Score"
].median()


strong_vs_no_precursor_score_gap = float(
    strong_score
    -
    no_precursor_score
)


event_rows = []


for source in [

    "CIC_IDS2017",

    "UNSW_NB15",

]:

    for horizon in PRIMARY_HORIZONS:

        subset = validation_scores.loc[

            validation_scores[
                "Calibration_Source"
            ].eq(
                source
            )

            &

            validation_scores[
                "Horizon_Minutes"
            ].eq(
                horizon
            )

        ].copy()


        threshold = probe_thresholds[
            (
                source,
                horizon
            )
        ]


        negatives = subset.loc[
            subset[
                "Y"
            ].eq(
                0
            )
        ]


        false_warning_count = int(

            (
                negatives[
                    "Score"
                ]
                >=
                threshold
            ).sum()
        )


        false_alerts_per_hour = (

            false_warning_count
            /
            len(
                negatives
            )
            *
            60

            if len(
                negatives
            )

            else np.nan
        )


        positives = subset.loc[
            subset[
                "Y"
            ].eq(
                1
            )
        ].copy()


        per_event = []


        for episode_id, group in positives.groupby(
            "Episode_ID",
            observed=True
        ):

            warned = group.loc[
                group[
                    "Score"
                ]
                >=
                threshold
            ]


            per_event.append(
                {
                    "Episode_ID":
                        episode_id,

                    "Detected":
                        len(
                            warned
                        )
                        >
                        0,

                    "First_Warning_Lead_Minutes":
                        (
                            float(

                                warned[
                                    "Minutes_To_Next_Onset"
                                ].max()
                            )

                            if len(
                                warned
                            )

                            else np.nan
                        ),
                }
            )


        per_event = pd.DataFrame(
            per_event
        )


        detected = per_event.loc[
            per_event[
                "Detected"
            ]
        ]


        event_rows.append(
            {
                "Calibration_Source":
                    source,

                "Horizon_Minutes":
                    horizon,

                "Validation_Attack_Events":
                    len(
                        per_event
                    ),

                "Detected_Events":
                    int(
                        per_event[
                            "Detected"
                        ].sum()
                    ),

                "Event_Recall":
                    float(
                        per_event[
                            "Detected"
                        ].mean()
                    ),

                "Median_First_Warning_Lead_Minutes":
                    (
                        float(

                            detected[
                                "First_Warning_Lead_Minutes"
                            ].median()
                        )

                        if len(
                            detected
                        )

                        else np.nan
                    ),

                "Mean_First_Warning_Lead_Minutes":
                    (
                        float(

                            detected[
                                "First_Warning_Lead_Minutes"
                            ].mean()
                        )

                        if len(
                            detected
                        )

                        else np.nan
                    ),

                "False_Alerts_Per_Hour_Of_Eligible_Negatives":
                    false_alerts_per_hour,
            }
        )


event_lead_time_probe = pd.DataFrame(
    event_rows
)


print(
    "\nEvent-level:"
)


print(
    event_lead_time_probe.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.5M — ATTACK-FAMILY SEPARABILITY
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5M — FAMILY SEPARABILITY AUDIT")
print("=" * 120)


family_rows = []


attack_active = targets[
    "Attack_State"
].eq(
    1
)


# --------------------------------------------------------------------------------------------------
# Current attack-active family separability
# --------------------------------------------------------------------------------------------------

for source, families in DOMAIN_FAMILIES.items():

    source_mask = metadata[
        "Calibration_Source"
    ].eq(
        source
    )


    train_mask = (

        attack_active

        &

        source_mask

        &

        metadata[
            "Split"
        ].eq(
            "TRAIN"
        )

        &

        targets[
            "Episode_Attack_Family"
        ].isin(
            families
        )
    )


    validation_mask = (

        attack_active

        &

        source_mask

        &

        metadata[
            "Split"
        ].eq(
            "VALIDATION"
        )

        &

        targets[
            "Episode_Attack_Family"
        ].isin(
            families
        )
    )


    X_train = np.log1p(

        predictors.loc[
            train_mask,
            FINAL_PREDICTORS
        ].to_numpy(
            dtype=float
        )
    )


    X_validation = np.log1p(

        predictors.loc[
            validation_mask,
            FINAL_PREDICTORS
        ].to_numpy(
            dtype=float
        )
    )


    y_train = targets.loc[
        train_mask,
        "Episode_Attack_Family"
    ].astype(
        str
    ).to_numpy()


    y_validation = targets.loc[
        validation_mask,
        "Episode_Attack_Family"
    ].astype(
        str
    ).to_numpy()


    model = multiclass_probe()


    model.fit(
        X_train,
        y_train
    )


    prediction = model.predict(
        X_validation
    )


    family_rows.append(
        {
            "Audit_Task":
                "ATTACK_ACTIVE_CURRENT_MINUTE",

            "Calibration_Source":
                source,

            "Horizon_Minutes":
                np.nan,

            "Train_Rows":
                len(
                    y_train
                ),

            "Validation_Rows":
                len(
                    y_validation
                ),

            "Class_Count":
                len(
                    np.unique(
                        y_train
                    )
                ),

            "Macro_F1":
                float(

                    f1_score(
                        y_validation,
                        prediction,
                        average="macro",
                        zero_division=0
                    )
                ),

            "Balanced_Accuracy":
                float(

                    balanced_accuracy_score(
                        y_validation,
                        prediction
                    )
                ),
        }
    )


# --------------------------------------------------------------------------------------------------
# Pre-onset H10 family separability
# --------------------------------------------------------------------------------------------------

horizon = 10


for source, families in DOMAIN_FAMILIES.items():

    source_mask = metadata[
        "Calibration_Source"
    ].eq(
        source
    )


    positive = targets[
        f"Onset_Within_H{horizon}"
    ].eq(
        1
    )


    allowed = targets[
        f"Onset_Family_H{horizon}"
    ].isin(
        families
    )


    train_mask = (

        positive

        &

        source_mask

        &

        allowed

        &

        metadata[
            "Split"
        ].eq(
            "TRAIN"
        )

        &

        history_ready
    )


    validation_mask = (

        positive

        &

        source_mask

        &

        allowed

        &

        metadata[
            "Split"
        ].eq(
            "VALIDATION"
        )

        &

        history_ready
    )


    X_train = probe_features.loc[
        train_mask
    ].to_numpy(
        dtype=float
    )


    X_validation = probe_features.loc[
        validation_mask
    ].to_numpy(
        dtype=float
    )


    y_train = targets.loc[
        train_mask,
        f"Onset_Family_H{horizon}"
    ].astype(
        str
    ).to_numpy()


    y_validation = targets.loc[
        validation_mask,
        f"Onset_Family_H{horizon}"
    ].astype(
        str
    ).to_numpy()


    model = multiclass_probe()


    model.fit(
        X_train,
        y_train
    )


    prediction = model.predict(
        X_validation
    )


    family_rows.append(
        {
            "Audit_Task":
                "PRE_ONSET_FAMILY_H10_HISTORY10",

            "Calibration_Source":
                source,

            "Horizon_Minutes":
                horizon,

            "Train_Rows":
                len(
                    y_train
                ),

            "Validation_Rows":
                len(
                    y_validation
                ),

            "Class_Count":
                len(
                    np.unique(
                        y_train
                    )
                ),

            "Macro_F1":
                float(

                    f1_score(
                        y_validation,
                        prediction,
                        average="macro",
                        zero_division=0
                    )
                ),

            "Balanced_Accuracy":
                float(

                    balanced_accuracy_score(
                        y_validation,
                        prediction
                    )
                ),
        }
    )


family_separability = pd.DataFrame(
    family_rows
)


print(
    family_separability.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.5N — GENERATED FAMILY-EFFECT DILUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5N — GENERATED FAMILY-EFFECT DILUTION")
print("=" * 120)


effect_feature_map = {

    "Flow_Duration_Seconds":
        "Flow_Duration_Mean",

    "Forward_Packets":
        "Forward_Packets_Per_Flow",

    "Backward_Packets":
        "Backward_Packets_Per_Flow",

    "Forward_Bytes":
        "Forward_Bytes_Per_Flow",

    "Backward_Bytes":
        "Backward_Bytes_Per_Flow",
}


effect_master = predictors.copy()


effect_master[
    "Forward_Packets_Per_Flow"
] = (

    effect_master[
        "Forward_Packets_Sum"
    ]

    /

    effect_master[
        "Flow_Count"
    ]
)


effect_master[
    "Backward_Packets_Per_Flow"
] = (

    effect_master[
        "Backward_Packets_Sum"
    ]

    /

    effect_master[
        "Flow_Count"
    ]
)


effect_master[
    "Forward_Bytes_Per_Flow"
] = (

    effect_master[
        "Forward_Bytes_Sum"
    ]

    /

    effect_master[
        "Flow_Count"
    ]
)


effect_master[
    "Backward_Bytes_Per_Flow"
] = (

    effect_master[
        "Backward_Bytes_Sum"
    ]

    /

    effect_master[
        "Flow_Count"
    ]
)


source_effects[
    "Effect_Log1p"
] = pd.to_numeric(

    source_effects[
        "Effect_Log1p"
    ],

    errors="coerce"
)


source_family_pairs = (

    episode_manifest.loc[
        episode_manifest[
            "Episode_Type"
        ].eq(
            "ATTACK"
        ),
        [
            "Calibration_Source",
            "Attack_Family"
        ]
    ]

    .drop_duplicates()
)


effect_rows = []


for _, row in source_family_pairs.iterrows():

    source = row[
        "Calibration_Source"
    ]


    family = row[
        "Attack_Family"
    ]


    benign_mask = (

        synthetic_benign_mask

        &

        metadata[
            "Calibration_Source"
        ].eq(
            source
        )
    )


    attack_mask = (

        targets[
            "Attack_State"
        ].eq(
            1
        )

        &

        metadata[
            "Calibration_Source"
        ].eq(
            source
        )

        &

        targets[
            "Episode_Attack_Family"
        ].eq(
            family
        )
    )


    for real_feature, generated_feature in effect_feature_map.items():

        real_match = source_effects.loc[

            source_effects[
                "Calibration_Source"
            ].eq(
                source
            )

            &

            source_effects[
                "Synthetic_Family"
            ].eq(
                family
            )

            &

            source_effects[
                "Feature"
            ].eq(
                real_feature
            )

        ]


        if len(
            real_match
        ) != 1:

            raise RuntimeError(
                f"Missing/duplicate real effect: "
                f"{source}, {family}, {real_feature}"
            )


        real_effect = float(
            real_match[
                "Effect_Log1p"
            ].iloc[
                0
            ]
        )


        benign_median = float(

            effect_master.loc[
                benign_mask,
                generated_feature
            ].median()
        )


        attack_median = float(

            effect_master.loc[
                attack_mask,
                generated_feature
            ].median()
        )


        generated_effect = float(

            np.log1p(
                attack_median
            )

            -

            np.log1p(
                benign_median
            )
        )


        if abs(
            real_effect
        ) <= 0.05:

            classification = (
                "REAL_NEAR_ZERO"
            )

            attenuation_ratio = np.nan

        else:

            attenuation_ratio = (

                abs(
                    generated_effect
                )

                /

                abs(
                    real_effect
                )
            )


            if abs(
                generated_effect
            ) < 0.05:

                classification = (
                    "VANISHED"
                )


            elif np.sign(
                generated_effect
            ) != np.sign(
                real_effect
            ):

                classification = (
                    "DIRECTION_FLIP"
                )


            elif attenuation_ratio < 0.10:

                classification = (
                    "HEAVILY_DILUTED"
                )


            else:

                classification = (
                    "RETAINED"
                )


        effect_rows.append(
            {
                "Calibration_Source":
                    source,

                "Synthetic_Family":
                    family,

                "Feature":
                    real_feature,

                "Real_Effect_Log1p":
                    real_effect,

                "Generated_Effect_Log1p":
                    generated_effect,

                "Absolute_Attenuation_Ratio":
                    attenuation_ratio,

                "Effect_Dilution_Class":
                    classification,
            }
        )


effect_dilution = pd.DataFrame(
    effect_rows
)


effect_dilution_summary = (

    effect_dilution

    .groupby(
        [
            "Calibration_Source",
            "Synthetic_Family"
        ],
        observed=True
    )

    .agg(

        Feature_Count=(
            "Feature",
            "size"
        ),

        Retained=(
            "Effect_Dilution_Class",
            lambda x:
                int(
                    (
                        x
                        ==
                        "RETAINED"
                    ).sum()
                )
        ),

        Vanished=(
            "Effect_Dilution_Class",
            lambda x:
                int(
                    (
                        x
                        ==
                        "VANISHED"
                    ).sum()
                )
        ),

        Heavily_Diluted=(
            "Effect_Dilution_Class",
            lambda x:
                int(
                    (
                        x
                        ==
                        "HEAVILY_DILUTED"
                    ).sum()
                )
        ),

        Direction_Flip=(
            "Effect_Dilution_Class",
            lambda x:
                int(
                    (
                        x
                        ==
                        "DIRECTION_FLIP"
                    ).sum()
                )
        ),
    )

    .reset_index()
)


print(
    effect_dilution_summary.to_string(
        index=False
    )
)


unsw_informative = effect_dilution.loc[

    effect_dilution[
        "Calibration_Source"
    ].eq(
        "UNSW_NB15"
    )

    &

    ~effect_dilution[
        "Effect_Dilution_Class"
    ].eq(
        "REAL_NEAR_ZERO"
    )
]


unsw_vanished_fraction = (

    float(

        (
            unsw_informative[
                "Effect_Dilution_Class"
            ]
            ==
            "VANISHED"
        ).mean()
    )

    if len(
        unsw_informative
    )

    else 0.0
)


# ==================================================================================================
# 11.5O — SOURCE-LEVEL FORECASTABILITY CLASSIFICATION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5O — SOURCE-LEVEL FORECASTABILITY CLASSIFICATION")
print("=" * 120)


source_signal_rows = []


for source in [

    "CIC_IDS2017",

    "UNSW_NB15",

]:

    subset = binary_forecast_probe.loc[
        binary_forecast_probe[
            "Calibration_Source"
        ].eq(
            source
        )
    ]


    median_auc = float(
        subset[
            "ROC_AUC"
        ].median()
    )


    median_lift = float(
        subset[
            "AP_Lift_Over_Prevalence"
        ].median()
    )


    control_pass = bool(

        no_precursor_control.loc[
            no_precursor_control[
                "Calibration_Source"
            ].eq(
                source
            ),
            "Control_Pass"
        ].all()
    )


    if not control_pass:

        signal_class = (
            "CONTROL_FAILURE"
        )


    elif (
        median_auc
        <
        TH[
            "too_weak_auc"
        ]

        and

        median_lift
        <
        TH[
            "too_weak_ap_lift"
        ]
    ):

        signal_class = (
            "TOO_WEAK"
        )


    elif (
        median_auc
        >
        TH[
            "too_easy_auc"
        ]

        and

        median_lift
        >
        TH[
            "too_easy_ap_lift"
        ]
    ):

        signal_class = (
            "TOO_EASY"
        )


    else:

        signal_class = (
            "USEFUL_SIGNAL"
        )


    source_signal_rows.append(
        {
            "Calibration_Source":
                source,

            "Median_Primary_Horizon_ROC_AUC":
                median_auc,

            "Median_Primary_Horizon_AP_Lift":
                median_lift,

            "NO_PRECURSOR_Control_Pass":
                control_pass,

            "Signal_Class":
                signal_class,
        }
    )


source_signal_classification = pd.DataFrame(
    source_signal_rows
)


print(
    source_signal_classification.to_string(
        index=False
    )
)


# ==================================================================================================
# 11.5P — FINAL QUALITY / FORECASTABILITY DECISION
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5P — FINAL QUALITY / FORECASTABILITY DECISION")
print("=" * 120)


predictor_values = predictors[
    FINAL_PREDICTORS
].to_numpy(
    dtype=float
)


structure_pass = bool(

    positive_counts_exact

    and

    hard_split_isolation_pass

    and

    predictors[
        KEYS
    ].duplicated().sum()
    ==
    0

    and

    targets[
        KEYS
    ].duplicated().sum()
    ==
    0

    and

    metadata[
        KEYS
    ].duplicated().sum()
    ==
    0

    and

    np.isfinite(
        predictor_values
    ).all()

    and

    (
        predictor_values
        >=
        0
    ).all()
)


realism_catastrophic = bool(

    distribution_catastrophic

    or

    correlation_catastrophic

    or

    temporal_catastrophic
)


realism_status = (

    "PASS"

    if (
        distribution_gate_pass

        and

        correlation_gate_pass

        and

        temporal_gate_pass
    )

    else "WARNING"
)


signal_classes = set(
    source_signal_classification[
        "Signal_Class"
    ].tolist()
)


has_useful_signal = (
    "USEFUL_SIGNAL"
    in
    signal_classes
)


if not structure_pass:

    FINAL_DECISION = (
        "STRUCTURAL_FAILURE"
    )


elif realism_catastrophic:

    FINAL_DECISION = (
        "REALISM_FAILURE"
    )


elif (
    "CONTROL_FAILURE"
    in
    signal_classes
):

    FINAL_DECISION = (
        "CONTROL_FAILURE"
    )


elif signal_classes == {
    "TOO_WEAK"
}:

    FINAL_DECISION = (
        "TOO_WEAK"
    )


elif signal_classes == {
    "TOO_EASY"
}:

    FINAL_DECISION = (
        "TOO_EASY"
    )


elif not has_useful_signal:

    # Example:
    #
    # CIC  = TOO_EASY
    # UNSW = TOO_WEAK
    #
    # This MUST NOT silently become "useful heterogeneity".

    FINAL_DECISION = (
        "REVIEW_MIXED_NONUSABLE_SIGNAL"
    )


else:

    domain_heterogeneity = bool(

        len(
            signal_classes
        )
        >
        1

        or

        source_shortcut_high

        or

        unsw_vanished_fraction
        >=
        0.50
    )


    FINAL_DECISION = (

        "USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY"

        if domain_heterogeneity

        else "USEFUL_SIGNAL"
    )


print(
    "Structural gate:",
    structure_pass
)

print(
    "Distribution gate:",
    distribution_gate_pass
)

print(
    "Correlation gate:",
    correlation_gate_pass
)

print(
    "Temporal realism gate:",
    temporal_gate_pass
)

print(
    "Realism status:",
    realism_status
)

print(
    "Source shortcut:",
    (
        "HIGH"

        if source_shortcut_high

        else "MODERATE_OR_LOW"
    )
)

print(
    "NO_PRECURSOR control pass:",
    no_precursor_control_pass
)

print(
    "Strong - NO_PRECURSOR median score gap:",
    strong_vs_no_precursor_score_gap
)

print(
    "UNSW informative-effect vanished fraction:",
    unsw_vanished_fraction
)

print()

print(
    "FINAL 11.5 DECISION:",
    FINAL_DECISION
)


# ==================================================================================================
# 11.5Q — SAVE AUDIT ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("11.5Q — SAVE AUDIT ARTIFACTS")
print("=" * 120)


artifacts = {

    "11_5_target_prevalence.csv":
        target_prevalence,

    "11_5_hard_split_isolation.csv":
        hard_leakage,

    "11_5_episode_sequence_hashes.csv":
        episode_hashes,

    "11_5_exact_row_overlap_warning.csv":
        row_overlap_warning,

    "11_5_nearest_neighbor_warning.csv":
        nn_audit,

    "11_5_distribution_similarity.csv":
        distribution_similarity,

    "11_5_distribution_summary.csv":
        distribution_summary,

    "11_5_correlation_preservation.csv":
        correlation_preservation,

    "11_5_temporal_realism_within_blocks.csv":
        temporal_realism,

    "11_5_temporal_summary.csv":
        temporal_summary,

    "11_5_source_shortcut_probe.csv":
        source_shortcut,

    "11_5_binary_forecastability_probe.csv":
        binary_forecast_probe,

    "11_5_no_precursor_control.csv":
        no_precursor_control,

    "11_5_precursor_regime_probe.csv":
        precursor_regime_probe,

    "11_5_event_lead_time_probe.csv":
        event_lead_time_probe,

    "11_5_family_separability.csv":
        family_separability,

    "11_5_effect_dilution.csv":
        effect_dilution,

    "11_5_effect_dilution_summary.csv":
        effect_dilution_summary,

    "11_5_source_signal_classification.csv":
        source_signal_classification,
}


for filename, dataframe in artifacts.items():

    save_csv(
        dataframe,
        AUDIT_DIR
        /
        filename
    )


# ==================================================================================================
# 11.5R — VALIDATION TABLE
# ==================================================================================================

validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                details,
        }
    )


add_check(
    "11_4_Readiness_Gate",

    readiness_11_4.get(
        "readiness"
    )
    ==
    "READY_FOR_PHASE_11_5",

    readiness_11_4.get(
        "readiness"
    )
)


add_check(
    "Predictor_Hash_Exact",

    current_hashes[
        "predictors"
    ]
    ==
    EXPECTED[
        "predictors"
    ],

    current_hashes[
        "predictors"
    ]
)


add_check(
    "Target_Hash_Exact",

    current_hashes[
        "targets"
    ]
    ==
    EXPECTED[
        "targets"
    ],

    current_hashes[
        "targets"
    ]
)


add_check(
    "Metadata_Hash_Exact",

    current_hashes[
        "metadata"
    ]
    ==
    EXPECTED[
        "metadata"
    ],

    current_hashes[
        "metadata"
    ]
)


add_check(
    "Positive_Counts_Exact",

    positive_counts_exact,

    "15/15 split×horizon cells expected"
)


add_check(
    "Hard_Split_Isolation",

    hard_split_isolation_pass,

    hard_leakage.to_dict(
        orient="records"
    )
)


add_check(
    "No_CrossSplit_Duplicate_Full_Episodes",

    cross_split_episode_duplicate_hashes
    ==
    0,

    str(
        cross_split_episode_duplicate_hashes
    )
)


add_check(
    "Distribution_Audit_Completed",

    len(
        distribution_similarity
    )
    ==
    28,

    str(
        len(
            distribution_similarity
        )
    )
)


add_check(
    "Correlation_Audit_Completed",

    len(
        correlation_preservation
    )
    ==
    2,

    str(
        len(
            correlation_preservation
        )
    )
)


add_check(
    "LikeForLike_Temporal_Audit_Completed",

    len(
        temporal_realism
    )
    ==
    28,

    str(
        len(
            temporal_realism
        )
    )
)


add_check(
    "Source_Shortcut_Audit_Completed",

    np.isfinite(
        source_auc
    ),

    f"AUC={source_auc:.6f}"
)


add_check(
    "Binary_Probe_Completed",

    len(
        binary_forecast_probe
    )
    ==
    4,

    str(
        len(
            binary_forecast_probe
        )
    )
)


add_check(
    "NO_PRECURSOR_Control_Completed",

    len(
        no_precursor_control
    )
    ==
    4,

    str(
        len(
            no_precursor_control
        )
    )
)


add_check(
    "Family_Separability_Completed",

    len(
        family_separability
    )
    ==
    4,

    str(
        len(
            family_separability
        )
    )
)


add_check(
    "Effect_Dilution_Completed",

    len(
        effect_dilution
    )
    >
    0,

    str(
        len(
            effect_dilution
        )
    )
)


add_check(
    "Probe_Training_Uses_No_TEST",

    True,

    "All fitted probes TRAIN -> VALIDATION only"
)


add_check(
    "Final_Decision_Not_Structural_Failure",

    FINAL_DECISION
    !=
    "STRUCTURAL_FAILURE",

    FINAL_DECISION
)


add_check(
    "Final_Decision_Not_Realism_Failure",

    FINAL_DECISION
    !=
    "REALISM_FAILURE",

    FINAL_DECISION
)


validation = pd.DataFrame(
    validation_rows
)


print("\n" + "=" * 120)
print("11.5R — VALIDATION")
print("=" * 120)


print(
    validation.to_string(
        index=False
    )
)


save_csv(
    validation,
    AUDIT_DIR
    /
    "11_5_validation.csv"
)


# ==================================================================================================
# 11.5S — DECISION + READINESS
# ==================================================================================================

decision = {

    "phase":
        "11.5",

    "dataset":
        "Synthetic_NAF_v1",

    "input_hashes":
        current_hashes,

    "forecast_horizons_minutes":
        FORECAST_HORIZONS,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,

    "history_window_minutes":
        HISTORY_WINDOW,

    "test_labels_used_for_trainable_probe_performance":
        False,

    "structural_gate_pass":
        structure_pass,

    "realism_status":
        realism_status,

    "distribution_gate_pass":
        distribution_gate_pass,

    "correlation_gate_pass":
        correlation_gate_pass,

    "temporal_gate_pass":
        temporal_gate_pass,

    "source_shortcut_high":
        source_shortcut_high,

    "source_shortcut_auc":
        source_auc,

    "no_precursor_control_pass":
        no_precursor_control_pass,

    "strong_vs_no_precursor_median_score_gap":
        strong_vs_no_precursor_score_gap,

    "unsw_informative_effect_vanished_fraction":
        unsw_vanished_fraction,

    "source_signal_classification":
        source_signal_classification.to_dict(
            orient="records"
        ),

    "final_decision":
        FINAL_DECISION,

    "evaluation_policy": {

        "ALL_5_SYNTHETIC":
            "SUPPORTING_ONLY_SOURCE_CONFOUND_POSSIBLE",

        "CIC_DOMAIN_4":
            DOMAIN_FAMILIES[
                "CIC_IDS2017"
            ],

        "UNSW_DOMAIN_4":
            DOMAIN_FAMILIES[
                "UNSW_NB15"
            ],

        "SHARED_CROSS_DOMAIN_3":
            [
                "DOS_DDOS",
                "RECON_SCAN",
                "EXPLOIT_APP"
            ],
    },
}


DECISION_FILE = (
    AUDIT_DIR
    /
    "11_5_decision.json"
)


with open(
    DECISION_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        decision,
        f,
        indent=4
    )


decision_hash = sha256_file(
    DECISION_FILE
)


(
    AUDIT_DIR
    /
    "11_5_decision_SHA256.txt"
).write_text(
    decision_hash
    +
    "\n",
    encoding="utf-8"
)


PASS_DECISIONS = {

    "USEFUL_SIGNAL",

    "USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY",
}


hard_validation_pass = bool(
    validation[
        "Passed"
    ].all()
)


READY_FOR_11_6 = bool(

    hard_validation_pass

    and

    FINAL_DECISION
    in
    PASS_DECISIONS
)


readiness_11_5 = {

    "dataset":
        "Synthetic_NAF_v1",

    "phase_11_5_decision":
        FINAL_DECISION,

    "hard_validation_pass":
        hard_validation_pass,

    "ready_for_phase_11_6_final_dataset_freeze":
        READY_FOR_11_6,

    "phase_11_5_decision_sha256":
        decision_hash,

    "next_step":
        (
            "PHASE_11_6_FINAL_DATASET_FREEZE"

            if READY_FOR_11_6

            else "STOP_AND_REVIEW_PHASE_11_5"
        ),
}


READINESS_11_5_FILE = (
    AUDIT_ROOT
    /
    "11_5_READY_FOR_FINAL_FREEZE.json"
)


with open(
    READINESS_11_5_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        readiness_11_5,
        f,
        indent=4
    )


readiness_11_5_hash = sha256_file(
    READINESS_11_5_FILE
)


(
    AUDIT_ROOT
    /
    "11_5_READY_FOR_FINAL_FREEZE_SHA256.txt"
).write_text(
    readiness_11_5_hash
    +
    "\n",
    encoding="utf-8"
)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("PHASE 11.5 — FINAL REPORT")
print("=" * 120)


print(
    "\nFINAL DECISION:"
)

print(
    FINAL_DECISION
)


print(
    "\nHard split isolation:",
    hard_split_isolation_pass
)


print(
    "Distribution gate:",
    distribution_gate_pass
)


print(
    "Correlation gate:",
    correlation_gate_pass
)


print(
    "Temporal realism gate:",
    temporal_gate_pass
)


print(
    "Overall realism status:",
    realism_status
)


print(
    "Source shortcut AUC:",
    source_auc
)


print(
    "NO_PRECURSOR control pass:",
    no_precursor_control_pass
)


print(
    "Strong-vs-NO_PRECURSOR score gap:",
    strong_vs_no_precursor_score_gap
)


print(
    "UNSW vanished-effect fraction:",
    unsw_vanished_fraction
)


print(
    "\nSOURCE CLASSIFICATION"
)


print(
    source_signal_classification.to_string(
        index=False
    )
)


print(
    "\nReady for Phase 11.6:",
    READY_FOR_11_6
)


print(
    "Readiness file:"
)

print(
    READINESS_11_5_FILE
)


print(
    "Readiness SHA256:"
)

print(
    readiness_11_5_hash
)


if READY_FOR_11_6:

    print(
        "\n✓ PHASE 11.5 PASS — PROCEED TO PHASE 11.6 FINAL DATASET FREEZE"
    )

else:

    print(
        "\n⚠ PHASE 11.5 DOES NOT AUTHORIZE FINAL FREEZE"
    )

    print(
        "DO NOT regenerate, retune, or train final models yet."
    )

    print(
        "Review the failing diagnostic first."
    )


print("=" * 120)
print("PHASE 11.5 COMPLETE")
print("=" * 120)


# ==================================================================================================
# CLEAN TEMPORARY LARGE OBJECTS
# ==================================================================================================

del log_features
del group_input
del lag1
del lag9
del mean10
del std10

gc.collect()

In [ ]:
# ==================================================================================================
# PHASE 11.6 — FINAL DATASET RELEASE / IMMUTABLE FREEZE
# DEFINITIVE VERSION — HANDLES KNOWN STALE 11.4 TEST LOCK
# ==================================================================================================
#
# ROOT CAUSE CONFIRMED FROM NOTEBOOK:
#
# Old Phase 11.4 produced:
#
#   TEST target SHA256:
#   7228b088007d836a095cd854f9f8337a62e890216b60503b136e48959c7938ba
#
# and wrote that value into:
#
#   11_4_TEST_DATA_LOCK.json
#
# Emergency target recovery later correctly changed the TEST target to:
#
#   07530190053d47e07aeefd7a5bf7b6be4a362fd52ee1ef7d31cb6415d61b5d43
#
# but DID NOT rewrite 11_4_TEST_DATA_LOCK.json.
#
# Therefore the old 11.4 TEST lock is stale provenance.
#
# THIS CELL:
#
#   - DOES NOT CHANGE 11.3
#   - DOES NOT CHANGE 11.4 data
#   - DOES NOT CHANGE 11.5
#   - DOES NOT REGENERATE TEST
#
# It verifies the corrected TEST files directly and creates the FINAL release lock.
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import shutil
import gc

import numpy as np
import pandas as pd


print("=" * 120)
print("PHASE 11.6 — FINAL DATASET RELEASE / IMMUTABLE FREEZE")
print("DEFINITIVE STALE-LOCK-AWARE VERSION")
print("=" * 120)


# ==================================================================================================
# 1 — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

SPEC_DIR = (
    DATASET_ROOT
    / "spec"
)

MANIFEST_DIR = (
    DATASET_ROOT
    / "manifests"
)

GENERATED_DIR = (
    DATASET_ROOT
    / "generated"
    / "11_4"
)

AUDIT_ROOT = (
    DATASET_ROOT
    / "audit"
)

AUDIT_11_5_DIR = (
    AUDIT_ROOT
    / "11_5"
)


PREDICTORS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_predictors.csv"
)

TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_targets.csv"
)

METADATA_FILE = (
    GENERATED_DIR
    / "11_4_synthetic_minute_metadata.csv"
)


TRAIN_PREDICTORS_FILE = (
    GENERATED_DIR
    / "11_4_train_predictors.csv"
)

TRAIN_TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_train_targets.csv"
)

TRAIN_METADATA_FILE = (
    GENERATED_DIR
    / "11_4_train_metadata.csv"
)


VAL_PREDICTORS_FILE = (
    GENERATED_DIR
    / "11_4_validation_predictors.csv"
)

VAL_TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_validation_targets.csv"
)

VAL_METADATA_FILE = (
    GENERATED_DIR
    / "11_4_validation_metadata.csv"
)


TEST_PREDICTORS_FILE = (
    GENERATED_DIR
    / "11_4_test_predictors.csv"
)

TEST_TARGETS_FILE = (
    GENERATED_DIR
    / "11_4_test_targets.csv"
)

TEST_METADATA_FILE = (
    GENERATED_DIR
    / "11_4_test_metadata.csv"
)


OLD_TEST_LOCK_FILE = (
    GENERATED_DIR
    / "11_4_TEST_DATA_LOCK.json"
)


EPISODE_MANIFEST_CSV = (
    MANIFEST_DIR
    / "11_3_episode_manifest.csv"
)

EPISODE_MANIFEST_PARQUET = (
    MANIFEST_DIR
    / "11_3_episode_manifest.parquet"
)

LOCKED_TEST_MANIFEST = (
    MANIFEST_DIR
    / "11_3_TEST_manifest_LOCKED.csv"
)


READINESS_11_5_FILE = (
    AUDIT_ROOT
    / "11_5_READY_FOR_FINAL_FREEZE.json"
)

DECISION_11_5_FILE = (
    AUDIT_11_5_DIR
    / "11_5_decision.json"
)

VALIDATION_11_5_FILE = (
    AUDIT_11_5_DIR
    / "11_5_validation.csv"
)


CONFIG_FILE = (
    SPEC_DIR
    / "Synthetic_NAF_v1_config.json"
)


# ==================================================================================================
# RELEASE
# ==================================================================================================

RELEASE_ROOT = (
    DATASET_ROOT
    / "release"
)

RELEASE_NAME = (
    "Synthetic_NAF_v1_v1.0.0"
)

FINAL_RELEASE_DIR = (
    RELEASE_ROOT
    / RELEASE_NAME
)

STAGING_DIR = (
    RELEASE_ROOT
    / f"._staging_{RELEASE_NAME}"
)


RELEASE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 2 — EXACT ACCEPTED HASHES
# ==================================================================================================

EXPECTED = {

    "predictors":
        "9b0213a77d5320fa5377922ac37e1b104291c1d49e39d71f87e4333f96e2eb3b",

    "targets":
        "cebe5e388d2f2476bb066a9878de933e6f295abf118e1a429adb328a87750884",

    "metadata":
        "ce7e8723347a890ae90524727302ee83ed911ab4545b208f2e70e6b9cbd57ad7",


    "train_predictors":
        "8e478d8a8b299a4f2a9df22d106270082a27b06a73c698a1e1d2212592c90ace",

    "train_targets":
        "80f5cc6770d49726533f29c515b6373091619ed06e540222a02c4708396933c0",


    "validation_predictors":
        "f9a35f4cac4f632475ffbd660b19d12bb4221e9e4c48a4718357a27f8da6b99f",

    "validation_targets":
        "7d2c5ec9410b576386290a13f5a4fc1c6300265d71d48856c3c0d393efe330bd",


    "test_predictors":
        "572eac0cc102c582d87b1c1a70f33a9a8e2f35c95bb4beacbc5311aebb10c6c2",

    "test_targets":
        "07530190053d47e07aeefd7a5bf7b6be4a362fd52ee1ef7d31cb6415d61b5d43",
}


# The known stale TEST target that was written into the old 11.4 lock.

KNOWN_STALE_TEST_TARGET_SHA256 = (
    "7228b088007d836a095cd854f9f8337a62e890216b60503b136e48959c7938ba"
)


PHASE_11_5_READINESS_SHA256 = (
    "e68357c29b8e5fa88690020962a1625cc418c06dd6014e0098e1774b3809979c"
)


HISTORICAL_MANIFEST_SHA256 = (
    "514e335576d110ab7712548e2f5c0213fa080e82162606ecb2b97ffcbe9f8560"
)

HISTORICAL_TEST_MANIFEST_SHA256 = (
    "369f95936f545f34adcb039487a7423196fc4c2a9e664386ca2bba7181044851"
)


# ==================================================================================================
# 3 — CONTRACT
# ==================================================================================================

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30
]

PRIMARY_HORIZONS = [
    5,
    10
]

HISTORY_WINDOWS = [
    5,
    10,
    15,
    30
]

PRIMARY_HISTORY = 10


FINAL_PREDICTORS = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Total_Packets",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum",

    "Total_Bytes",

    "Packets_Per_Second",

    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",

    "Forward_Backward_Byte_Ratio",
]


FAMILIES = [

    "DOS_DDOS",

    "RECON_SCAN",

    "EXPLOIT_APP",

    "BRUTE_AUTH",

    "FUZZING",
]


EXPECTED_ROWS = {

    "TRAIN":
        102187,

    "VALIDATION":
        21786,

    "TEST":
        22058,
}


EXPECTED_EPISODES = {

    "TRAIN":
        840,

    "VALIDATION":
        180,

    "TEST":
        180,
}


EXPECTED_ATTACK_EPISODES = {

    "TRAIN":
        700,

    "VALIDATION":
        150,

    "TEST":
        150,
}


EXPECTED_BENIGN_EPISODES = {

    "TRAIN":
        140,

    "VALIDATION":
        30,

    "TEST":
        30,
}


EXPECTED_POSITIVES = {

    ("TRAIN", 1): 700,
    ("TRAIN", 5): 3500,
    ("TRAIN", 10): 7000,
    ("TRAIN", 15): 10500,
    ("TRAIN", 30): 21000,

    ("VALIDATION", 1): 150,
    ("VALIDATION", 5): 750,
    ("VALIDATION", 10): 1500,
    ("VALIDATION", 15): 2250,
    ("VALIDATION", 30): 4500,

    ("TEST", 1): 150,
    ("TEST", 5): 750,
    ("TEST", 10): 1500,
    ("TEST", 15): 2250,
    ("TEST", 30): 4500,
}


LEGACY_TIMESTAMP_COLUMNS = [

    "Episode_Start_UTC",

    "Episode_End_UTC",

    "Precursor_Start_UTC",

    "Precursor_End_UTC",

    "Attack_Onset_UTC",

    "Attack_End_UTC",

    "Recovery_Start_UTC",

    "Recovery_End_UTC",
]


KEYS = [
    "Episode_ID",
    "Minute_Index"
]


# ==================================================================================================
# 4 — HELPERS
# ==================================================================================================

def require_file(
    path
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def read_json(
    path
):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def save_csv(
    df,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        path,
        index=False,
        float_format="%.10g"
    )


def copy_required(
    src,
    dst
):

    require_file(
        src
    )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        src,
        dst
    )


def copy_optional(
    src,
    dst
):

    if src.is_file():

        dst.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            src,
            dst
        )

        return True

    return False


def normalize_text(
    series
):

    return (

        series

        .astype(
            "string"
        )

        .fillna(
            "<NA>"
        )
    )


def compare_non_timestamp_frames(
    left,
    right
):

    if "Episode_ID" not in left.columns:

        raise RuntimeError(
            "Left manifest missing Episode_ID."
        )


    if "Episode_ID" not in right.columns:

        raise RuntimeError(
            "Right manifest missing Episode_ID."
        )


    left_work = left.copy()

    right_work = right.copy()


    left_work[
        "__KEY__"
    ] = left_work[
        "Episode_ID"
    ].astype(
        str
    )


    right_work[
        "__KEY__"
    ] = right_work[
        "Episode_ID"
    ].astype(
        str
    )


    if set(
        left_work[
            "__KEY__"
        ]
    ) != set(
        right_work[
            "__KEY__"
        ]
    ):

        raise RuntimeError(
            "Manifest Episode_ID sets differ."
        )


    left_work = (

        left_work

        .sort_values(
            "__KEY__"
        )

        .reset_index(
            drop=True
        )
    )


    right_work = (

        right_work

        .sort_values(
            "__KEY__"
        )

        .reset_index(
            drop=True
        )
    )


    common_columns = [

        c

        for c in left.columns

        if (
            c in right.columns

            and

            c not in LEGACY_TIMESTAMP_COLUMNS
        )
    ]


    results = []


    for column in common_columns:

        a = left_work[
            column
        ]

        b = right_work[
            column
        ]


        if not np.array_equal(

            a.isna().to_numpy(),

            b.isna().to_numpy()

        ):

            passed = False


        else:

            an = pd.to_numeric(
                a,
                errors="coerce"
            )

            bn = pd.to_numeric(
                b,
                errors="coerce"
            )


            numeric = (

                an.notna().sum()
                ==
                a.notna().sum()

                and

                bn.notna().sum()
                ==
                b.notna().sum()
            )


            if numeric:

                passed = bool(

                    np.allclose(

                        an.to_numpy(
                            dtype=float
                        ),

                        bn.to_numpy(
                            dtype=float
                        ),

                        atol=1e-9,

                        rtol=1e-12,

                        equal_nan=True
                    )
                )


            else:

                passed = bool(

                    (
                        normalize_text(
                            a
                        )

                        ==

                        normalize_text(
                            b
                        )
                    ).all()
                )


        results.append(
            {
                "Column":
                    column,

                "Passed":
                    passed,
            }
        )


    audit = pd.DataFrame(
        results
    )


    return (
        bool(
            audit[
                "Passed"
            ].all()
        ),

        audit
    )


def verify_split_against_full(
    full_df,
    split_df
):

    full_index = full_df.set_index(
        KEYS,
        drop=False
    )

    split_index = split_df.set_index(
        KEYS,
        drop=False
    )


    missing = split_index.index.difference(
        full_index.index
    )


    if len(
        missing
    ):

        return False


    expected = full_index.loc[
        split_index.index
    ].reset_index(
        drop=True
    )


    observed = split_index.reset_index(
        drop=True
    )


    return expected.equals(
        observed
    )


def build_hash_table(
    root
):

    excluded = {

        "FINAL_FILE_HASHES.csv",

        "RELEASE_LOCK.json",

        "RELEASE_LOCK_SHA256.txt",
    }


    rows = []


    for path in sorted(
        root.rglob(
            "*"
        )
    ):

        if not path.is_file():

            continue


        if path.name in excluded:

            continue


        rows.append(
            {
                "Relative_Path":
                    str(
                        path.relative_to(
                            root
                        )
                    ),

                "Bytes":
                    int(
                        path.stat().st_size
                    ),

                "SHA256":
                    sha256_file(
                        path
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ==================================================================================================
# 5 — VERIFY PHASE 11.5
# ==================================================================================================

print("\n" + "=" * 120)
print("5 — VERIFY PHASE 11.5")
print("=" * 120)


current_11_5_hash = sha256_file(
    READINESS_11_5_FILE
)


if current_11_5_hash != PHASE_11_5_READINESS_SHA256:

    raise RuntimeError(
        "Successful Phase 11.5 readiness changed."
    )


readiness_11_5 = read_json(
    READINESS_11_5_FILE
)


if (
    readiness_11_5.get(
        "phase_11_5_decision"
    )
    !=
    "USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY"
):

    raise RuntimeError(
        "Unexpected 11.5 decision."
    )


if (
    readiness_11_5.get(
        "ready_for_phase_11_6_final_dataset_freeze"
    )
    is not True
):

    raise RuntimeError(
        "11.5 does not authorize 11.6."
    )


validation_11_5 = pd.read_csv(
    VALIDATION_11_5_FILE
)


passed_11_5 = (

    validation_11_5[
        "Passed"
    ]

    .astype(
        str
    )

    .str.lower()

    .str.strip()

    .isin(
        [
            "true",
            "1",
            "yes"
        ]
    )
)


if not passed_11_5.all():

    raise RuntimeError(
        "11.5 has failed validation checks."
    )


print(
    "✓ 11.5 readiness exact"
)

print(
    "✓ USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY"
)

print(
    f"✓ {int(passed_11_5.sum())}/{len(passed_11_5)} PASS"
)


# ==================================================================================================
# 6 — VERIFY EXACT BENCHMARK FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("6 — VERIFY EXACT BENCHMARK DATA")
print("=" * 120)


hash_files = {

    "predictors":
        PREDICTORS_FILE,

    "targets":
        TARGETS_FILE,

    "metadata":
        METADATA_FILE,

    "train_predictors":
        TRAIN_PREDICTORS_FILE,

    "train_targets":
        TRAIN_TARGETS_FILE,

    "validation_predictors":
        VAL_PREDICTORS_FILE,

    "validation_targets":
        VAL_TARGETS_FILE,

    "test_predictors":
        TEST_PREDICTORS_FILE,

    "test_targets":
        TEST_TARGETS_FILE,
}


for name, path in hash_files.items():

    require_file(
        path
    )


    current = sha256_file(
        path
    )


    expected = EXPECTED[
        name
    ]


    print(
        f"{'✓' if current == expected else '❌'} "
        f"{name:<24} "
        f"{current}"
    )


    if current != expected:

        raise RuntimeError(
            f"Benchmark file changed: {name}"
        )


print(
    "\n✓ ACTUAL BENCHMARK DATA EXACT"
)


# ==================================================================================================
# 7 — LOAD DATA
# ==================================================================================================

predictors = pd.read_csv(
    PREDICTORS_FILE,
    low_memory=False
)

targets = pd.read_csv(
    TARGETS_FILE,
    low_memory=False
)

metadata = pd.read_csv(
    METADATA_FILE,
    low_memory=False
)


train_predictors = pd.read_csv(
    TRAIN_PREDICTORS_FILE,
    low_memory=False
)

train_targets = pd.read_csv(
    TRAIN_TARGETS_FILE,
    low_memory=False
)

train_metadata = pd.read_csv(
    TRAIN_METADATA_FILE,
    low_memory=False
)


val_predictors = pd.read_csv(
    VAL_PREDICTORS_FILE,
    low_memory=False
)

val_targets = pd.read_csv(
    VAL_TARGETS_FILE,
    low_memory=False
)

val_metadata = pd.read_csv(
    VAL_METADATA_FILE,
    low_memory=False
)


test_predictors = pd.read_csv(
    TEST_PREDICTORS_FILE,
    low_memory=False
)

test_targets = pd.read_csv(
    TEST_TARGETS_FILE,
    low_memory=False
)

test_metadata = pd.read_csv(
    TEST_METADATA_FILE,
    low_memory=False
)


if not (
    len(
        predictors
    )
    ==
    len(
        targets
    )
    ==
    len(
        metadata
    )
    ==
    146031
):

    raise RuntimeError(
        "Full dataset row-count mismatch."
    )


if not predictors[
    KEYS
].equals(
    targets[
        KEYS
    ]
):

    raise RuntimeError(
        "Predictor/target key mismatch."
    )


if not predictors[
    KEYS
].equals(
    metadata[
        KEYS
    ]
):

    raise RuntimeError(
        "Predictor/metadata key mismatch."
    )


if list(
    predictors.columns
) != [
    *KEYS,
    *FINAL_PREDICTORS
]:

    raise RuntimeError(
        "Predictor schema changed."
    )


print(
    "✓ 146,031 aligned rows"
)

print(
    "✓ 14 model predictors"
)


# ==================================================================================================
# 8 — MANIFEST CONTENT
# ==================================================================================================

print("\n" + "=" * 120)
print("8 — VERIFY EPISODE MANIFEST CONTENT")
print("=" * 120)


manifest = pd.read_csv(
    EPISODE_MANIFEST_CSV,
    low_memory=False
)

manifest_parquet = pd.read_parquet(
    EPISODE_MANIFEST_PARQUET
)

locked_test_manifest = pd.read_csv(
    LOCKED_TEST_MANIFEST,
    low_memory=False
)


manifest_hash = sha256_file(
    EPISODE_MANIFEST_CSV
)

test_manifest_hash = sha256_file(
    LOCKED_TEST_MANIFEST
)

manifest_parquet_hash = sha256_file(
    EPISODE_MANIFEST_PARQUET
)


manifest_pass, manifest_audit = compare_non_timestamp_frames(

    manifest,

    manifest_parquet
)


if not manifest_pass:

    print(
        manifest_audit.loc[
            ~manifest_audit[
                "Passed"
            ]
        ]
    )


    raise RuntimeError(
        "Non-timestamp manifest content differs."
    )


full_test_manifest = manifest.loc[
    manifest[
        "Split"
    ].eq(
        "TEST"
    )
].copy()


test_manifest_pass, test_manifest_audit = compare_non_timestamp_frames(

    full_test_manifest,

    locked_test_manifest
)


if not test_manifest_pass:

    print(
        test_manifest_audit.loc[
            ~test_manifest_audit[
                "Passed"
            ]
        ]
    )


    raise RuntimeError(
        "Non-timestamp TEST manifest content differs."
    )


print(
    "✓ full manifest non-time content verified"
)

print(
    "✓ TEST manifest non-time content verified"
)


# ==================================================================================================
# 9 — EPISODE SAMPLE SIZE / LEAKAGE CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("9 — VERIFY EPISODE COUNTS / SPLIT ISOLATION")
print("=" * 120)


attack_manifest = manifest.loc[
    manifest[
        "Episode_Type"
    ].eq(
        "ATTACK"
    )
].copy()


benign_manifest = manifest.loc[
    manifest[
        "Episode_Type"
    ].eq(
        "BENIGN_ONLY"
    )
].copy()


if len(
    manifest
) != 1200:

    raise RuntimeError(
        "Expected 1,200 episodes."
    )


if len(
    attack_manifest
) != 1000:

    raise RuntimeError(
        "Expected 1,000 independent attack episodes."
    )


if len(
    benign_manifest
) != 200:

    raise RuntimeError(
        "Expected 200 benign-only episodes."
    )


if manifest[
    "Episode_ID"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Episode_ID."
    )


if manifest[
    "Generator_Seed"
].nunique() != 1200:

    raise RuntimeError(
        "Generator seeds not unique."
    )


if manifest[
    "Baseline_Block_Sequence_SHA256"
].nunique() != 1200:

    raise RuntimeError(
        "Baseline block sequences not unique."
    )


family_counts = (

    attack_manifest

    .groupby(
        "Attack_Family",
        observed=True
    )[
        "Episode_ID"
    ]

    .nunique()

    .reindex(
        FAMILIES,
        fill_value=0
    )
)


if not (
    family_counts
    ==
    200
).all():

    raise RuntimeError(
        "Expected exactly 200 attack episodes per family."
    )


for split in [

    "TRAIN",

    "VALIDATION",

    "TEST",

]:

    total = int(
        manifest[
            "Split"
        ].eq(
            split
        ).sum()
    )


    attacks = int(
        attack_manifest[
            "Split"
        ].eq(
            split
        ).sum()
    )


    benign = int(
        benign_manifest[
            "Split"
        ].eq(
            split
        ).sum()
    )


    if total != EXPECTED_EPISODES[
        split
    ]:

        raise RuntimeError(
            f"{split} total episode mismatch."
        )


    if attacks != EXPECTED_ATTACK_EPISODES[
        split
    ]:

        raise RuntimeError(
            f"{split} attack episode mismatch."
        )


    if benign != EXPECTED_BENIGN_EPISODES[
        split
    ]:

        raise RuntimeError(
            f"{split} benign episode mismatch."
        )


    print(
        f"✓ {split:<10} "
        f"{total} total / "
        f"{attacks} attack / "
        f"{benign} benign"
    )


for split, expected_family_count in [

    (
        "TRAIN",
        140
    ),

    (
        "VALIDATION",
        30
    ),

    (
        "TEST",
        30
    ),

]:

    counts = (

        attack_manifest.loc[
            attack_manifest[
                "Split"
            ].eq(
                split
            )
        ]

        .groupby(
            "Attack_Family",
            observed=True
        )[
            "Episode_ID"
        ]

        .nunique()

        .reindex(
            FAMILIES,
            fill_value=0
        )
    )


    if not (
        counts
        ==
        expected_family_count
    ).all():

        raise RuntimeError(
            f"{split} family allocation mismatch."
        )


print(
    "\n✓ 1,000 independent attack episodes"
)

print(
    "✓ 200 benign-only episodes"
)

print(
    "✓ 5 families × 200 attacks"
)

print(
    "✓ each family = 140 TRAIN / 30 VALIDATION / 30 TEST"
)

print(
    "✓ episode-level split contract preserved"
)


# ==================================================================================================
# 10 — VERIFY SPLIT TABLES
# ==================================================================================================

print("\n" + "=" * 120)
print("10 — VERIFY GENERATED TRAIN / VALIDATION / TEST")
print("=" * 120)


split_tables = {

    "TRAIN":
        (
            train_predictors,
            train_targets,
            train_metadata
        ),

    "VALIDATION":
        (
            val_predictors,
            val_targets,
            val_metadata
        ),

    "TEST":
        (
            test_predictors,
            test_targets,
            test_metadata
        ),
}


all_split_keys = []


for split, (
    split_predictors,
    split_targets,
    split_metadata
) in split_tables.items():

    if not (
        len(
            split_predictors
        )
        ==
        len(
            split_targets
        )
        ==
        len(
            split_metadata
        )
        ==
        EXPECTED_ROWS[
            split
        ]
    ):

        raise RuntimeError(
            f"{split} row-count mismatch."
        )


    if not split_predictors[
        KEYS
    ].equals(
        split_targets[
            KEYS
        ]
    ):

        raise RuntimeError(
            f"{split} predictor/target misalignment."
        )


    if not split_predictors[
        KEYS
    ].equals(
        split_metadata[
            KEYS
        ]
    ):

        raise RuntimeError(
            f"{split} predictor/metadata misalignment."
        )


    if not verify_split_against_full(

        predictors,

        split_predictors

    ):

        raise RuntimeError(
            f"{split} predictors differ from full table."
        )


    if not verify_split_against_full(

        targets,

        split_targets

    ):

        raise RuntimeError(
            f"{split} targets differ from full table."
        )


    if not verify_split_against_full(

        metadata,

        split_metadata

    ):

        raise RuntimeError(
            f"{split} metadata differ from full table."
        )


    all_split_keys.append(
        split_predictors[
            KEYS
        ]
    )


    print(
        f"✓ {split:<10} "
        f"{len(split_predictors):,} rows"
    )


combined_keys = pd.concat(
    all_split_keys,
    ignore_index=True
)


if combined_keys.duplicated(
    subset=KEYS
).any():

    raise RuntimeError(
        "Cross-split row overlap."
    )


full_keys = set(
    map(
        tuple,
        predictors[
            KEYS
        ].to_numpy()
    )
)


partition_keys = set(
    map(
        tuple,
        combined_keys[
            KEYS
        ].to_numpy()
    )
)


if full_keys != partition_keys:

    raise RuntimeError(
        "Split partition does not equal full dataset."
    )


print(
    "✓ exact non-overlapping row partition"
)


# ==================================================================================================
# 11 — TARGET SUPPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("11 — VERIFY CORRECTED TARGET SUPPORT")
print("=" * 120)


h2_columns = [

    column

    for column in targets.columns

    if (
        column.endswith(
            "_H2"
        )

        or

        "_H2_" in column
    )
]


if h2_columns:

    raise RuntimeError(
        f"Obsolete H2 target exists: {h2_columns}"
    )


target_support_rows = []


for split, (
    split_predictors,
    split_targets,
    split_metadata
) in split_tables.items():

    for horizon in FORECAST_HORIZONS:

        eligible = split_targets[
            f"Forecast_Eligible_H{horizon}"
        ].eq(
            1
        )


        y = pd.to_numeric(

            split_targets.loc[
                eligible,
                f"Onset_Within_H{horizon}"
            ],

            errors="raise"

        ).astype(
            int
        )


        positives = int(
            y.sum()
        )


        expected = EXPECTED_POSITIVES[
            (
                split,
                horizon
            )
        ]


        if positives != expected:

            raise RuntimeError(
                f"{split} H{horizon} positive count mismatch."
            )


        target_support_rows.append(
            {
                "Split":
                    split,

                "Horizon_Minutes":
                    horizon,

                "Eligible_Origins":
                    int(
                        len(
                            y
                        )
                    ),

                "Positive_Origins":
                    positives,

                "Negative_Origins":
                    int(
                        len(
                            y
                        )
                        -
                        positives
                    ),

                "Positive_Rate":
                    float(
                        positives
                        /
                        len(
                            y
                        )
                    ),
            }
        )


target_support = pd.DataFrame(
    target_support_rows
)


print(
    target_support.to_string(
        index=False
    )
)


print(
    "\n✓ 15/15 support checks exact"
)

print(
    "✓ H2 absent"
)


# ==================================================================================================
# 12 — RESOLVE KNOWN STALE 11.4 TEST LOCK
# ==================================================================================================

print("\n" + "=" * 120)
print("12 — RESOLVE KNOWN STALE 11.4 TEST LOCK")
print("=" * 120)


require_file(
    OLD_TEST_LOCK_FILE
)


old_test_lock = read_json(
    OLD_TEST_LOCK_FILE
)


old_lock_file_sha256 = sha256_file(
    OLD_TEST_LOCK_FILE
)


current_test_predictor_hash = sha256_file(
    TEST_PREDICTORS_FILE
)

current_test_target_hash = sha256_file(
    TEST_TARGETS_FILE
)

current_test_metadata_hash = sha256_file(
    TEST_METADATA_FILE
)


old_locked_predictor_hash = old_test_lock.get(
    "test_predictors_sha256"
)

old_locked_target_hash = old_test_lock.get(
    "test_targets_sha256"
)

old_locked_metadata_hash = old_test_lock.get(
    "test_metadata_sha256"
)


print(
    "Old lock predictor:",
    old_locked_predictor_hash
)

print(
    "Current predictor :",
    current_test_predictor_hash
)


print(
    "\nOld lock target   :",
    old_locked_target_hash
)

print(
    "Current target    :",
    current_test_target_hash
)


print(
    "\nOld lock metadata :",
    old_locked_metadata_hash
)

print(
    "Current metadata  :",
    current_test_metadata_hash
)


# Predictor never changed and MUST still agree.

if old_locked_predictor_hash != current_test_predictor_hash:

    raise RuntimeError(
        "Unexpected TEST predictor-lock mismatch."
    )


# Metadata never changed and MUST still agree when recorded.

if (
    old_locked_metadata_hash is not None

    and

    old_locked_metadata_hash
    !=
    current_test_metadata_hash
):

    raise RuntimeError(
        "Unexpected TEST metadata-lock mismatch."
    )


# Target is allowed to be either:
#
# 1. already corrected
# 2. the single KNOWN stale hash created by the old 11.4 target semantics

if old_locked_target_hash == current_test_target_hash:

    TEST_LOCK_STATUS = (
        "CURRENT_AND_EXACT"
    )


elif old_locked_target_hash == KNOWN_STALE_TEST_TARGET_SHA256:

    TEST_LOCK_STATUS = (
        "KNOWN_STALE_PRE_CORRECTION_LOCK"
    )


else:

    raise RuntimeError(
        "\nUnexpected TEST lock target hash.\n\n"
        f"Old lock contains:\n{old_locked_target_hash}\n\n"
        f"Known stale hash:\n{KNOWN_STALE_TEST_TARGET_SHA256}\n\n"
        f"Correct current hash:\n{current_test_target_hash}"
    )


if current_test_target_hash != EXPECTED[
    "test_targets"
]:

    raise RuntimeError(
        "Current TEST target is not the accepted corrected target."
    )


if current_test_predictor_hash != EXPECTED[
    "test_predictors"
]:

    raise RuntimeError(
        "Current TEST predictor is not accepted."
    )


print(
    "\n✓ current TEST predictor is exact"
)

print(
    "✓ current corrected TEST target is exact"
)

print(
    "✓ current TEST metadata unchanged"
)

print(
    "✓ old lock status:",
    TEST_LOCK_STATUS
)


if TEST_LOCK_STATUS == "KNOWN_STALE_PRE_CORRECTION_LOCK":

    print(
        "\n✓ stale lock explained by the known Phase 11.4 target correction"
    )

    print(
        "✓ NO TEST data regeneration required"
    )

    print(
        "✓ old lock will be retained as provenance only"
    )


# ==================================================================================================
# 13 — FINAL VALIDATION
# ==================================================================================================

print("\n" + "=" * 120)
print("13 — FINAL PRE-RELEASE VALIDATION")
print("=" * 120)


validation_rows = [

    {
        "Check":
            "Phase_11_5_Exact",

        "Passed":
            current_11_5_hash
            ==
            PHASE_11_5_READINESS_SHA256,
    },

    {
        "Check":
            "Phase_11_5_18_of_18",

        "Passed":
            bool(
                passed_11_5.all()
            ),
    },

    {
        "Check":
            "Predictors_Exact",

        "Passed":
            sha256_file(
                PREDICTORS_FILE
            )
            ==
            EXPECTED[
                "predictors"
            ],
    },

    {
        "Check":
            "Targets_Exact",

        "Passed":
            sha256_file(
                TARGETS_FILE
            )
            ==
            EXPECTED[
                "targets"
            ],
    },

    {
        "Check":
            "Metadata_Exact",

        "Passed":
            sha256_file(
                METADATA_FILE
            )
            ==
            EXPECTED[
                "metadata"
            ],
    },

    {
        "Check":
            "Manifest_NonTimestamp_Content_Verified",

        "Passed":
            manifest_pass,
    },

    {
        "Check":
            "TEST_Manifest_Content_Verified",

        "Passed":
            test_manifest_pass,
    },

    {
        "Check":
            "Total_Episodes_1200",

        "Passed":
            len(
                manifest
            )
            ==
            1200,
    },

    {
        "Check":
            "Attack_Episodes_1000",

        "Passed":
            len(
                attack_manifest
            )
            ==
            1000,
    },

    {
        "Check":
            "Benign_Episodes_200",

        "Passed":
            len(
                benign_manifest
            )
            ==
            200,
    },

    {
        "Check":
            "Five_Families_200_Each",

        "Passed":
            bool(
                (
                    family_counts
                    ==
                    200
                ).all()
            ),
    },

    {
        "Check":
            "Unique_Generator_Seeds_1200",

        "Passed":
            manifest[
                "Generator_Seed"
            ].nunique()
            ==
            1200,
    },

    {
        "Check":
            "Unique_Baseline_Sequences_1200",

        "Passed":
            manifest[
                "Baseline_Block_Sequence_SHA256"
            ].nunique()
            ==
            1200,
    },

    {
        "Check":
            "Exact_Split_Partition",

        "Passed":
            full_keys
            ==
            partition_keys,
    },

    {
        "Check":
            "No_H2",

        "Passed":
            len(
                h2_columns
            )
            ==
            0,
    },

    {
        "Check":
            "Target_Support_15_of_15",

        "Passed":
            True,
    },

    {
        "Check":
            "Current_TEST_Predictor_Exact",

        "Passed":
            current_test_predictor_hash
            ==
            EXPECTED[
                "test_predictors"
            ],
    },

    {
        "Check":
            "Current_TEST_Target_Exact",

        "Passed":
            current_test_target_hash
            ==
            EXPECTED[
                "test_targets"
            ],
    },

    {
        "Check":
            "Old_TEST_Lock_Explained",

        "Passed":
            TEST_LOCK_STATUS
            in
            {
                "CURRENT_AND_EXACT",
                "KNOWN_STALE_PRE_CORRECTION_LOCK",
            },
    },
]


validation_11_6 = pd.DataFrame(
    validation_rows
)


if not validation_11_6[
    "Passed"
].all():

    failed = validation_11_6.loc[
        ~validation_11_6[
            "Passed"
        ],
        "Check"
    ].tolist()


    raise RuntimeError(
        f"Final 11.6 validation failed:\n{failed}"
    )


print(
    validation_11_6.to_string(
        index=False
    )
)


print(
    "\n✓ ALL FINAL 11.6 GATES PASS"
)


# ==================================================================================================
# 14 — RELEASE DIRECTORY SAFETY
# ==================================================================================================

print("\n" + "=" * 120)
print("14 — RELEASE DIRECTORY")
print("=" * 120)


if STAGING_DIR.exists():

    shutil.rmtree(
        STAGING_DIR
    )


ALREADY_FROZEN = False


if FINAL_RELEASE_DIR.exists():

    existing_lock_file = (
        FINAL_RELEASE_DIR
        / "RELEASE_LOCK.json"
    )


    existing_lock_hash_file = (
        FINAL_RELEASE_DIR
        / "RELEASE_LOCK_SHA256.txt"
    )


    if (
        existing_lock_file.is_file()

        and

        existing_lock_hash_file.is_file()
    ):

        actual = sha256_file(
            existing_lock_file
        )


        recorded = (

            existing_lock_hash_file

            .read_text(
                encoding="utf-8"
            )

            .strip()
        )


        if actual == recorded:

            existing_lock = read_json(
                existing_lock_file
            )


            if (
                existing_lock.get(
                    "release_status"
                )
                ==
                "IMMUTABLY_FROZEN"
            ):

                ALREADY_FROZEN = True


    if ALREADY_FROZEN:

        print(
            "✓ final immutable release already exists"
        )


    else:

        i = 1


        while True:

            backup = (

                RELEASE_ROOT
                /
                f"_incomplete_11_6_attempt_{i}"
            )


            if not backup.exists():

                break


            i += 1


        shutil.move(
            str(
                FINAL_RELEASE_DIR
            ),
            str(
                backup
            )
        )


        print(
            "⚠ incomplete previous attempt preserved:"
        )

        print(
            backup
        )


# ==================================================================================================
# 15 — CREATE FINAL RELEASE
# ==================================================================================================

if not ALREADY_FROZEN:

    FULL_DIR = (
        STAGING_DIR
        / "data"
        / "full"
    )

    TRAIN_DIR = (
        STAGING_DIR
        / "data"
        / "train"
    )

    VAL_DIR = (
        STAGING_DIR
        / "data"
        / "validation"
    )

    TEST_DIR = (
        STAGING_DIR
        / "data"
        / "test"
    )

    MANIFEST_RELEASE_DIR = (
        STAGING_DIR
        / "manifests"
    )

    CONTRACT_DIR = (
        STAGING_DIR
        / "contracts"
    )

    DOC_DIR = (
        STAGING_DIR
        / "documentation"
    )

    PROVENANCE_DIR = (
        STAGING_DIR
        / "provenance"
    )

    RELEASE_AUDIT_DIR = (
        STAGING_DIR
        / "audit"
    )


    for directory in [

        FULL_DIR,

        TRAIN_DIR,

        VAL_DIR,

        TEST_DIR,

        MANIFEST_RELEASE_DIR,

        CONTRACT_DIR,

        DOC_DIR,

        PROVENANCE_DIR,

        RELEASE_AUDIT_DIR,

    ]:

        directory.mkdir(
            parents=True,
            exist_ok=True
        )


    # ==============================================================================================
    # COPY ACTUAL BENCHMARK
    # ==============================================================================================

    copy_required(
        PREDICTORS_FILE,
        FULL_DIR
        / "predictors.csv"
    )

    copy_required(
        TARGETS_FILE,
        FULL_DIR
        / "targets.csv"
    )

    copy_required(
        METADATA_FILE,
        FULL_DIR
        / "metadata.csv"
    )


    copy_required(
        TRAIN_PREDICTORS_FILE,
        TRAIN_DIR
        / "predictors.csv"
    )

    copy_required(
        TRAIN_TARGETS_FILE,
        TRAIN_DIR
        / "targets.csv"
    )

    copy_required(
        TRAIN_METADATA_FILE,
        TRAIN_DIR
        / "metadata.csv"
    )


    copy_required(
        VAL_PREDICTORS_FILE,
        VAL_DIR
        / "predictors.csv"
    )

    copy_required(
        VAL_TARGETS_FILE,
        VAL_DIR
        / "targets.csv"
    )

    copy_required(
        VAL_METADATA_FILE,
        VAL_DIR
        / "metadata.csv"
    )


    copy_required(
        TEST_PREDICTORS_FILE,
        TEST_DIR
        / "predictors.csv"
    )

    copy_required(
        TEST_TARGETS_FILE,
        TEST_DIR
        / "targets.csv"
    )

    copy_required(
        TEST_METADATA_FILE,
        TEST_DIR
        / "metadata.csv"
    )


    # ==============================================================================================
    # MANIFESTS / PROVENANCE
    # ==============================================================================================

    copy_required(
        EPISODE_MANIFEST_CSV,
        MANIFEST_RELEASE_DIR
        / "episode_manifest.csv"
    )


    copy_required(
        EPISODE_MANIFEST_PARQUET,
        MANIFEST_RELEASE_DIR
        / "episode_manifest.parquet"
    )


    copy_required(
        LOCKED_TEST_MANIFEST,
        MANIFEST_RELEASE_DIR
        / "TEST_manifest.csv"
    )


    copy_required(
        OLD_TEST_LOCK_FILE,
        PROVENANCE_DIR
        / "11_4_TEST_DATA_LOCK_PRE_CORRECTION.json"
    )


    copy_required(
        READINESS_11_5_FILE,
        PROVENANCE_DIR
        / "11_5_READY_FOR_FINAL_FREEZE.json"
    )


    copy_required(
        DECISION_11_5_FILE,
        PROVENANCE_DIR
        / "11_5_decision.json"
    )


    copy_optional(
        CONFIG_FILE,
        PROVENANCE_DIR
        / "Synthetic_NAF_v1_config.json"
    )


    shutil.copytree(

        AUDIT_11_5_DIR,

        RELEASE_AUDIT_DIR
        / "11_5",

        dirs_exist_ok=True
    )


    save_csv(

        validation_11_6,

        RELEASE_AUDIT_DIR
        / "11_6_validation.csv"
    )


    save_csv(

        manifest_audit,

        RELEASE_AUDIT_DIR
        / "11_6_manifest_non_timestamp_audit.csv"
    )


    save_csv(

        test_manifest_audit,

        RELEASE_AUDIT_DIR
        / "11_6_TEST_manifest_non_timestamp_audit.csv"
    )


    # ==============================================================================================
    # FINAL TEST LOCK — CORRECTED
    # ==============================================================================================

    final_test_lock = {

        "dataset":
            "Synthetic_NAF_v1",

        "version":
            "1.0.0",

        "lock_scope":
            "FINAL_RELEASE_TEST_DATA",

        "test_predictors_sha256":
            current_test_predictor_hash,

        "test_targets_sha256":
            current_test_target_hash,

        "test_metadata_sha256":
            current_test_metadata_hash,

        "test_rows":
            int(
                len(
                    test_predictors
                )
            ),

        "test_episodes":
            int(
                test_predictors[
                    "Episode_ID"
                ].nunique()
            ),

        "old_phase_11_4_lock_sha256":
            old_lock_file_sha256,

        "old_phase_11_4_lock_status":
            TEST_LOCK_STATUS,

        "known_old_target_sha256":
            old_locked_target_hash,

        "target_corrected_after_original_11_4_lock":
            TEST_LOCK_STATUS
            ==
            "KNOWN_STALE_PRE_CORRECTION_LOCK",

        "test_regenerated":
            False,

        "test_episode_membership_changed":
            False,

        "test_predictor_content_changed":
            False,

        "final_lock_is_authoritative":
            True,
    }


    write_json(

        final_test_lock,

        PROVENANCE_DIR
        / "FINAL_TEST_DATA_LOCK.json"
    )


    # ==============================================================================================
    # MODEL CONTRACT
    # ==============================================================================================

    model_contract = {

        "dataset":
            "Synthetic_NAF_v1",

        "version":
            "1.0.0",

        "allowed_predictors":
            FINAL_PREDICTORS,

        "predictor_count":
            14,

        "history_windows_minutes":
            HISTORY_WINDOWS,

        "primary_history_minutes":
            PRIMARY_HISTORY,

        "forecast_horizons_minutes":
            FORECAST_HORIZONS,

        "primary_forecast_horizons_minutes":
            PRIMARY_HORIZONS,

        "sequence_cross_episode_allowed":
            False,

        "preprocessing_fit":
            "TRAIN_ONLY",

        "metadata_as_predictor":
            False,

        "episode_id_as_predictor":
            False,

        "source_as_predictor":
            False,

        "precursor_regime_as_predictor":
            False,

        "target_information_as_predictor":
            False,

        "future_information_as_predictor":
            False,
    }


    write_json(

        model_contract,

        CONTRACT_DIR
        / "MODEL_INPUT_CONTRACT.json"
    )


    # ==============================================================================================
    # EVALUATION POLICY
    # ==============================================================================================

    evaluation_policy = {

        "binary_primary_metrics": [

            "Average Precision / PR-AUC",

            "Precision",

            "Recall",

            "F1",
        ],

        "binary_secondary_metrics": [

            "ROC-AUC",

            "Accuracy",
        ],

        "event_metrics": [

            "Event Recall",

            "Event Precision",

            "First-warning lead time",

            "Median lead time",

            "Mean lead time",

            "False alerts per hour",
        ],

        "family_metrics": [

            "Macro F1",

            "Macro Recall",

            "Macro Precision",

            "Per-class metrics",

            "Confusion matrix",
        ],

        "threshold_selection":
            "VALIDATION_ONLY",

        "model_selection_on_test":
            False,

        "random_row_split":
            False,

        "random_window_split":
            False,

        "train_only_sampling":
            True,

        "validation_oversampling":
            False,

        "test_oversampling":
            False,

        "allowed_train_imbalance_methods": [

            "class weighting",

            "positive-class weighting",

            "episode-aware sampling",

            "balanced mini-batches",

            "focal loss",

            "negative downsampling",
        ],

        "primary_reporting": [

            "CIC_DOMAIN_4",

            "UNSW_DOMAIN_4",

            "SHARED_CROSS_DOMAIN_3",
        ],

        "ALL_5_SYNTHETIC":
            "SUPPORTING_ONLY",
    }


    write_json(

        evaluation_policy,

        CONTRACT_DIR
        / "EVALUATION_POLICY.json"
    )


    # ==============================================================================================
    # TARGET CONTRACT
    # ==============================================================================================

    target_contract = {

        "task":
            "NEW_ATTACK_ONSET_FORECASTING",

        "positive":
            "new onset in same Episode_ID within (t,t+H]",

        "positive_requires_full_future":
            False,

        "negative":
            "no onset in (t,t+H]",

        "negative_requires_full_future":
            True,

        "attack_active_origin_allowed":
            False,

        "post_attack_origin_allowed":
            False,

        "cross_episode_target":
            False,

        "horizons":
            FORECAST_HORIZONS,
    }


    write_json(

        target_contract,

        CONTRACT_DIR
        / "TARGET_CONTRACT.json"
    )


    # ==============================================================================================
    # DOCUMENTATION
    # ==============================================================================================

    save_csv(

        target_support,

        DOC_DIR
        / "TARGET_PREVALENCE.csv"
    )


    split_summary = pd.DataFrame(
        [
            {
                "Split":
                    "TRAIN",

                "Minute_Rows":
                    102187,

                "Episodes":
                    840,

                "Attack_Episodes":
                    700,

                "Benign_Only_Episodes":
                    140,
            },

            {
                "Split":
                    "VALIDATION",

                "Minute_Rows":
                    21786,

                "Episodes":
                    180,

                "Attack_Episodes":
                    150,

                "Benign_Only_Episodes":
                    30,
            },

            {
                "Split":
                    "TEST",

                "Minute_Rows":
                    22058,

                "Episodes":
                    180,

                "Attack_Episodes":
                    150,

                "Benign_Only_Episodes":
                    30,
            },
        ]
    )


    save_csv(

        split_summary,

        DOC_DIR
        / "SPLIT_SUMMARY.csv"
    )


    dataset_card = f"""# Synthetic_NAF_v1 v1.0.0

## Final status

IMMUTABLY FROZEN BENCHMARK.

## Scale

- 146,031 minute rows
- 1,200 independent episodes
- 1,000 attack episodes
- 200 benign-only episodes
- 5 attack families
- 200 attack episodes per family

## Split

TRAIN:
- 840 total episodes
- 700 attack episodes
- 140 benign-only episodes

VALIDATION:
- 180 total episodes
- 150 attack episodes
- 30 benign-only episodes

TEST:
- 180 total episodes
- 150 attack episodes
- 30 benign-only episodes

Each attack family has:

- 140 TRAIN events
- 30 VALIDATION events
- 30 TEST events

Splitting is performed at Episode_ID level.

Sequences must never cross Episode_ID.

## Forecasting

Horizons:

{FORECAST_HORIZONS}

Primary horizons:

{PRIMARY_HORIZONS}

Primary history:

{PRIMARY_HISTORY} minutes

## Phase 11.5

USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY

18/18 audit validation checks passed.

## TEST-lock provenance

The original Phase 11.4 TEST lock was created before the corrected target
semantics were restored.

Old locked TEST target SHA256:

{old_locked_target_hash}

Correct final TEST target SHA256:

{current_test_target_hash}

Status:

{TEST_LOCK_STATUS}

The TEST predictor data, episode membership, metadata and synthetic traffic
were NOT regenerated.

Only the target semantics were corrected to the historically accepted target
hash before this final release.

The FINAL_TEST_DATA_LOCK.json included in this release is authoritative.

## Scientific interpretation

Synthetic_NAF_v1 is a controlled synthetic forecasting benchmark calibrated
from CIC-IDS2017 and UNSW-NB15 statistics.

It is not a real captured attack forecasting dataset.

## Training imbalance

Imbalance handling may be performed on TRAIN only.

Allowed examples:

- class weighting
- positive-class weighting
- episode-aware sampling
- balanced mini-batches
- focal loss
- negative downsampling

VALIDATION and TEST remain untouched.

## Next phase

Phase 12.0 builds model-ready causal sequences.

The benchmark itself must not be modified.
"""


    (
        DOC_DIR
        / "DATASET_CARD.md"
    ).write_text(
        dataset_card,
        encoding="utf-8"
    )


    (
        STAGING_DIR
        / "README.md"
    ).write_text(
        """# Synthetic_NAF_v1 v1.0.0

FINAL IMMUTABLE BENCHMARK RELEASE.

Do not modify this directory in place.

Any benchmark-content change requires a new version.

Next:
Phase 12.0 — model-ready causal sequence construction.
""",
        encoding="utf-8"
    )


    # ==============================================================================================
    # FINAL DATASET MANIFEST
    # ==============================================================================================

    final_dataset_manifest = {

        "dataset":
            "Synthetic_NAF_v1",

        "version":
            "1.0.0",

        "status":
            "FINAL_FROZEN_BENCHMARK",


        "minute_rows":
            146031,

        "episodes":
            1200,

        "attack_episodes":
            1000,

        "benign_only_episodes":
            200,

        "attack_families":
            5,

        "attack_episodes_per_family":
            200,


        "train_attack_episodes":
            700,

        "validation_attack_episodes":
            150,

        "test_attack_episodes":
            150,


        "predictor_count":
            14,


        "history_windows_minutes":
            HISTORY_WINDOWS,

        "primary_history_minutes":
            PRIMARY_HISTORY,

        "forecast_horizons_minutes":
            FORECAST_HORIZONS,

        "primary_horizons_minutes":
            PRIMARY_HORIZONS,


        "canonical_hashes":
            EXPECTED,


        "phase_11_5": {

            "decision":
                "USEFUL_SIGNAL_WITH_DOMAIN_HETEROGENEITY",

            "readiness_sha256":
                current_11_5_hash,

            "validation":
                "18/18 PASS",
        },


        "manifest": {

            "historical_csv_sha256":
                HISTORICAL_MANIFEST_SHA256,

            "release_csv_sha256":
                manifest_hash,

            "parquet_sha256":
                manifest_parquet_hash,

            "non_timestamp_semantic_verification":
                True,
        },


        "test_manifest": {

            "historical_csv_sha256":
                HISTORICAL_TEST_MANIFEST_SHA256,

            "release_csv_sha256":
                test_manifest_hash,

            "semantic_verification":
                True,
        },


        "old_phase_11_4_test_lock": {

            "sha256":
                old_lock_file_sha256,

            "status":
                TEST_LOCK_STATUS,

            "old_target_sha256":
                old_locked_target_hash,

            "final_corrected_target_sha256":
                current_test_target_hash,
        },


        "next_phase":
            "12.0_MODEL_READY_CAUSAL_SEQUENCE_BUILDING",
    }


    write_json(

        final_dataset_manifest,

        STAGING_DIR
        / "FINAL_DATASET_MANIFEST.json"
    )


    # ==============================================================================================
    # HASH PAYLOAD
    # ==============================================================================================

    print("\n" + "=" * 120)
    print("16 — HASH RELEASE PAYLOAD")
    print("=" * 120)


    hash_table = build_hash_table(
        STAGING_DIR
    )


    save_csv(

        hash_table,

        STAGING_DIR
        / "FINAL_FILE_HASHES.csv"
    )


    hash_table_sha256 = sha256_file(

        STAGING_DIR
        / "FINAL_FILE_HASHES.csv"
    )


    dataset_manifest_sha256 = sha256_file(

        STAGING_DIR
        / "FINAL_DATASET_MANIFEST.json"
    )


    # ==============================================================================================
    # FINAL RELEASE LOCK
    # ==============================================================================================

    release_lock = {

        "dataset":
            "Synthetic_NAF_v1",

        "version":
            "1.0.0",

        "release_status":
            "IMMUTABLY_FROZEN",


        "canonical_generated_data_hashes":
            EXPECTED,


        "final_test_lock": {

            "test_predictors_sha256":
                current_test_predictor_hash,

            "test_targets_sha256":
                current_test_target_hash,

            "test_metadata_sha256":
                current_test_metadata_hash,
        },


        "old_phase_11_4_test_lock": {

            "sha256":
                old_lock_file_sha256,

            "status":
                TEST_LOCK_STATUS,

            "old_test_target_sha256":
                old_locked_target_hash,
        },


        "final_dataset_manifest_sha256":
            dataset_manifest_sha256,

        "final_file_hashes_sha256":
            hash_table_sha256,


        "phase_11_5_readiness_sha256":
            current_11_5_hash,


        "immutability": {

            "modify_release_in_place":
                False,

            "regenerate_test":
                False,

            "change_episode_membership":
                False,

            "retune_generator_after_results":
                False,

            "new_content_requires_new_version":
                True,
        },


        "next_phase":
            "12.0_MODEL_READY_CAUSAL_SEQUENCE_BUILDING",
    }


    write_json(

        release_lock,

        STAGING_DIR
        / "RELEASE_LOCK.json"
    )


    release_lock_sha256 = sha256_file(

        STAGING_DIR
        / "RELEASE_LOCK.json"
    )


    (
        STAGING_DIR
        / "RELEASE_LOCK_SHA256.txt"
    ).write_text(
        release_lock_sha256
        +
        "\n",
        encoding="utf-8"
    )


    # ==============================================================================================
    # VERIFY RELEASE PAYLOAD
    # ==============================================================================================

    recorded_hashes = pd.read_csv(

        STAGING_DIR
        / "FINAL_FILE_HASHES.csv"
    )


    failures = []


    for row in recorded_hashes.itertuples(
        index=False
    ):

        path = (
            STAGING_DIR
            / row.Relative_Path
        )


        if not path.is_file():

            failures.append(
                row.Relative_Path
            )

            continue


        if sha256_file(
            path
        ) != row.SHA256:

            failures.append(
                row.Relative_Path
            )


    if failures:

        raise RuntimeError(
            f"Release payload verification failed:\n"
            f"{failures[:10]}"
        )


    # ==============================================================================================
    # VERIFY CORE COPIES
    # ==============================================================================================

    release_core = {

        "predictors":
            FULL_DIR
            / "predictors.csv",

        "targets":
            FULL_DIR
            / "targets.csv",

        "metadata":
            FULL_DIR
            / "metadata.csv",

        "train_predictors":
            TRAIN_DIR
            / "predictors.csv",

        "train_targets":
            TRAIN_DIR
            / "targets.csv",

        "validation_predictors":
            VAL_DIR
            / "predictors.csv",

        "validation_targets":
            VAL_DIR
            / "targets.csv",

        "test_predictors":
            TEST_DIR
            / "predictors.csv",

        "test_targets":
            TEST_DIR
            / "targets.csv",
    }


    for name, path in release_core.items():

        if sha256_file(
            path
        ) != EXPECTED[
            name
        ]:

            raise RuntimeError(
                f"Release copy mismatch: {name}"
            )


    print(
        "✓ release payload verified"
    )

    print(
        "✓ all benchmark data copies exact"
    )


    # ==============================================================================================
    # FINALIZE
    # ==============================================================================================

    if FINAL_RELEASE_DIR.exists():

        raise RuntimeError(
            "Final release directory unexpectedly exists."
        )


    shutil.move(
        str(
            STAGING_DIR
        ),
        str(
            FINAL_RELEASE_DIR
        )
    )


    print(
        "✓ release finalized"
    )


# ==================================================================================================
# 17 — VERIFY FINAL RELEASE
# ==================================================================================================

print("\n" + "=" * 120)
print("17 — VERIFY FINAL RELEASE")
print("=" * 120)


FINAL_LOCK_FILE = (
    FINAL_RELEASE_DIR
    / "RELEASE_LOCK.json"
)

FINAL_LOCK_HASH_FILE = (
    FINAL_RELEASE_DIR
    / "RELEASE_LOCK_SHA256.txt"
)

FINAL_HASHES_FILE = (
    FINAL_RELEASE_DIR
    / "FINAL_FILE_HASHES.csv"
)


require_file(
    FINAL_LOCK_FILE
)

require_file(
    FINAL_LOCK_HASH_FILE
)

require_file(
    FINAL_HASHES_FILE
)


actual_final_lock_hash = sha256_file(
    FINAL_LOCK_FILE
)


recorded_final_lock_hash = (

    FINAL_LOCK_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if actual_final_lock_hash != recorded_final_lock_hash:

    raise RuntimeError(
        "Final release lock hash mismatch."
    )


final_lock = read_json(
    FINAL_LOCK_FILE
)


if (
    final_lock.get(
        "release_status"
    )
    !=
    "IMMUTABLY_FROZEN"
):

    raise RuntimeError(
        "Release is not marked IMMUTABLY_FROZEN."
    )


final_hashes = pd.read_csv(
    FINAL_HASHES_FILE
)


bad_files = []


for row in final_hashes.itertuples(
    index=False
):

    path = (
        FINAL_RELEASE_DIR
        / row.Relative_Path
    )


    if not path.is_file():

        bad_files.append(
            row.Relative_Path
        )

        continue


    if sha256_file(
        path
    ) != row.SHA256:

        bad_files.append(
            row.Relative_Path
        )


if bad_files:

    raise RuntimeError(
        f"Final frozen release verification failed:\n"
        f"{bad_files[:10]}"
    )


print(
    "✓ RELEASE_LOCK verified"
)

print(
    "✓ entire release payload verified"
)


# ==================================================================================================
# 18 — SOURCE-SIDE FINAL POINTER
# ==================================================================================================

release_pointer = {

    "dataset":
        "Synthetic_NAF_v1",

    "version":
        "1.0.0",

    "phase":
        "11.6",

    "phase_11_6_status":
        "PASS",

    "release_status":
        "IMMUTABLY_FROZEN",

    "release_path":
        str(
            FINAL_RELEASE_DIR
        ),

    "release_lock_sha256":
        actual_final_lock_hash,


    "attack_episodes":
        1000,

    "train_attack_episodes":
        700,

    "validation_attack_episodes":
        150,

    "test_attack_episodes":
        150,


    "test_lock_status":
        TEST_LOCK_STATUS,


    "ready_for_phase_12_0":
        True,

    "next_phase":
        "12.0_MODEL_READY_CAUSAL_SEQUENCE_BUILDING",
}


RELEASE_POINTER_FILE = (
    AUDIT_ROOT
    / "11_6_FINAL_RELEASE_FROZEN.json"
)


write_json(

    release_pointer,

    RELEASE_POINTER_FILE
)


release_pointer_sha256 = sha256_file(
    RELEASE_POINTER_FILE
)


(
    AUDIT_ROOT
    / "11_6_FINAL_RELEASE_FROZEN_SHA256.txt"
).write_text(
    release_pointer_sha256
    +
    "\n",
    encoding="utf-8"
)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("✓ PHASE 11.6 — FINAL DATASET RELEASE / IMMUTABLE FREEZE: PASS")
print("=" * 120)


print(
    "\nROOT CAUSE RESOLVED:"
)

print(
    "Old Phase 11.4 TEST lock target:"
)

print(
    old_locked_target_hash
)

print(
    "\nFinal corrected TEST target:"
)

print(
    current_test_target_hash
)

print(
    "\nOld lock status:"
)

print(
    TEST_LOCK_STATUS
)


print(
    "\nDATASET:"
)

print(
    "    Synthetic_NAF_v1 v1.0.0"
)


print(
    "\nSTATUS:"
)

print(
    "    IMMUTABLY_FROZEN"
)


print(
    "\nEPISODE SAMPLE SIZE:"
)

print(
    "    1,200 total independent episodes"
)

print(
    "    1,000 independent attack episodes"
)

print(
    "    200 benign-only episodes"
)


print(
    "\nTRAIN / VALIDATION / TEST:"
)

print(
    "    TRAIN      700 attacks + 140 benign"
)

print(
    "    VALIDATION 150 attacks + 30 benign"
)

print(
    "    TEST       150 attacks + 30 benign"
)


print(
    "\nFAMILY SUPPORT:"
)

print(
    "    5 attack families"
)

print(
    "    200 events per family"
)

print(
    "    140 TRAIN / 30 VALIDATION / 30 TEST per family"
)


print(
    "\nLEAKAGE:"
)

print(
    "    Episode-level splitting ✓"
)

print(
    "    TRAIN/VALIDATION/TEST non-overlapping ✓"
)

print(
    "    No sequence will cross Episode_ID ✓"
)

print(
    "    TEST permanently locked ✓"
)


print(
    "\nIMPORTANT:"
)

print(
    "    11.3 was NOT changed ✓"
)

print(
    "    attack episodes were NOT regenerated ✓"
)

print(
    "    TEST was NOT regenerated ✓"
)

print(
    "    corrected target semantics preserved ✓"
)

print(
    "    stale old lock retained only as provenance ✓"
)


print(
    "\nDATASET GENERATION:"
)

print(
    "    FINISHED ✓"
)


print(
    "\nNEXT:"
)

print(
    "    PHASE 12.0 — MODEL-READY CAUSAL SEQUENCE BUILDING"
)


print("=" * 120)
print("Synthetic_NAF_v1 v1.0.0 IS NOW THE FINAL FROZEN BENCHMARK")
print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# PHASE 12.0 — MODEL-READY CAUSAL SEQUENCE BUILD
# ==================================================================================================
#
# PURPOSE
# -------
# Convert the immutable Synthetic_NAF_v1 v1.0.0 minute-level benchmark into
# leakage-safe temporal sequence datasets.
#
#
# INPUT
# -----
# ONLY the final immutable Phase 11.6 release.
#
#
# OUTPUT
# ------
# History windows:
#
#     W = [5, 10, 15, 30] minutes
#
# Primary:
#
#     W = 10
#
# Forecast horizons:
#
#     H = [1, 5, 10, 15, 30]
#
# Primary:
#
#     H = [5, 10]
#
#
# Sequence definition:
#
#     X_t = [x_(t-W+1), ..., x_t]
#
# Shape:
#
#     [N, W, 14]
#
#
# IMPORTANT
# ---------
#
# THIS PHASE DOES NOT:
#
#   - train models
#   - scale features
#   - fit preprocessing
#   - oversample positives
#   - downsample negatives
#   - compute class weights
#   - balance mini-batches
#   - use SMOTE
#   - modify the frozen benchmark
#
#
# Phase 12.1 will independently audit the generated sequences BEFORE
# preprocessing or model training is allowed.
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import shutil
import gc

import numpy as np
import pandas as pd


print("=" * 120)
print("PHASE 12.0 — MODEL-READY CAUSAL SEQUENCE BUILD")
print("=" * 120)


# ==================================================================================================
# 12.0A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

AUDIT_ROOT = (
    DATASET_ROOT
    / "audit"
)


RELEASE_POINTER_FILE = (
    AUDIT_ROOT
    / "11_6_FINAL_RELEASE_FROZEN.json"
)


MODEL_READY_ROOT = (
    DATASET_ROOT
    / "model_ready"
)

FINAL_OUT = (
    MODEL_READY_ROOT
    / "12_0"
)

STAGING = (
    MODEL_READY_ROOT
    / "._staging_12_0"
)


# ==================================================================================================
# FROZEN SEQUENCE CONTRACT
# ==================================================================================================

WINDOWS = [
    5,
    10,
    15,
    30
]


PRIMARY_WINDOW = 10


HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


PRIMARY_HORIZONS = [
    5,
    10
]


FEATURES = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Total_Packets",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum",

    "Total_Bytes",

    "Packets_Per_Second",

    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",

    "Forward_Backward_Byte_Ratio",
]


FAMILY_ORDER = [

    "DOS_DDOS",

    "RECON_SCAN",

    "EXPLOIT_APP",

    "BRUTE_AUTH",

    "FUZZING",
]


FAMILY_TO_CODE = {

    family:
        i

    for i, family in enumerate(
        FAMILY_ORDER
    )
}


SPLIT_DIRS = {

    "TRAIN":
        "train",

    "VALIDATION":
        "validation",

    "TEST":
        "test",
}


# ==================================================================================================
# EXACT POSITIVE SUPPORT FROM FROZEN BENCHMARK
# ==================================================================================================

EXPECTED_POS = {

    ("TRAIN", 1):
        700,

    ("TRAIN", 5):
        3500,

    ("TRAIN", 10):
        7000,

    ("TRAIN", 15):
        10500,

    ("TRAIN", 30):
        21000,


    ("VALIDATION", 1):
        150,

    ("VALIDATION", 5):
        750,

    ("VALIDATION", 10):
        1500,

    ("VALIDATION", 15):
        2250,

    ("VALIDATION", 30):
        4500,


    ("TEST", 1):
        150,

    ("TEST", 5):
        750,

    ("TEST", 10):
        1500,

    ("TEST", 15):
        2250,

    ("TEST", 30):
        4500,
}


EXPECTED_EVENTS = {

    "TRAIN":
        700,

    "VALIDATION":
        150,

    "TEST":
        150,
}


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(
    path
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def read_json(
    path
):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def save_csv(
    df,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        path,
        index=False,
        float_format="%.10g"
    )


def build_hash_table(
    root
):

    # Control files are verified separately because including them would
    # produce circular hashes.

    excluded = {

        "12_0_FILE_HASHES.csv",

        "12_0_BUILD_LOCK.json",

        "12_0_BUILD_LOCK_SHA256.txt",

        "12_0_READY_FOR_SEQUENCE_AUDIT.json",
    }


    rows = []


    for path in sorted(
        root.rglob(
            "*"
        )
    ):

        if not path.is_file():

            continue


        if path.name in excluded:

            continue


        rows.append(
            {
                "Relative_Path":
                    str(
                        path.relative_to(
                            root
                        )
                    ),

                "Bytes":
                    int(
                        path.stat().st_size
                    ),

                "SHA256":
                    sha256_file(
                        path
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ==================================================================================================
# 12.0B — VERIFY IMMUTABLE 11.6 RELEASE
# ==================================================================================================

print("\n" + "=" * 120)
print("12.0B — VERIFY IMMUTABLE 11.6 RELEASE")
print("=" * 120)


release_pointer = read_json(
    RELEASE_POINTER_FILE
)


if release_pointer.get(
    "phase_11_6_status"
) != "PASS":

    raise RuntimeError(
        "Phase 11.6 final-release pointer is not PASS."
    )


if release_pointer.get(
    "release_status"
) != "IMMUTABLY_FROZEN":

    raise RuntimeError(
        "Phase 11.6 release is not IMMUTABLY_FROZEN."
    )


if release_pointer.get(
    "ready_for_phase_12_0"
) is not True:

    raise RuntimeError(
        "Phase 11.6 does not authorize Phase 12.0."
    )


RELEASE_DIR = Path(
    release_pointer[
        "release_path"
    ]
)


RELEASE_LOCK_FILE = (
    RELEASE_DIR
    / "RELEASE_LOCK.json"
)

RELEASE_LOCK_HASH_FILE = (
    RELEASE_DIR
    / "RELEASE_LOCK_SHA256.txt"
)

FINAL_FILE_HASHES = (
    RELEASE_DIR
    / "FINAL_FILE_HASHES.csv"
)

MODEL_INPUT_CONTRACT_FILE = (
    RELEASE_DIR
    / "contracts"
    / "MODEL_INPUT_CONTRACT.json"
)

TARGET_CONTRACT_FILE = (
    RELEASE_DIR
    / "contracts"
    / "TARGET_CONTRACT.json"
)


for path in [

    RELEASE_LOCK_FILE,

    RELEASE_LOCK_HASH_FILE,

    FINAL_FILE_HASHES,

    MODEL_INPUT_CONTRACT_FILE,

    TARGET_CONTRACT_FILE,

]:

    require_file(
        path
    )


release_lock_hash = sha256_file(
    RELEASE_LOCK_FILE
)


recorded_release_lock_hash = (

    RELEASE_LOCK_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if (
    release_lock_hash
    !=
    recorded_release_lock_hash
):

    raise RuntimeError(
        "Phase 11.6 RELEASE_LOCK hash verification failed."
    )


if (
    release_pointer.get(
        "release_lock_sha256"
    )
    !=
    release_lock_hash
):

    raise RuntimeError(
        "Phase 11.6 pointer disagrees with the release lock."
    )


release_lock = read_json(
    RELEASE_LOCK_FILE
)


if release_lock.get(
    "release_status"
) != "IMMUTABLY_FROZEN":

    raise RuntimeError(
        "Release lock does not mark dataset IMMUTABLY_FROZEN."
    )


# ==================================================================================================
# Verify every recorded Phase 11.6 payload file
# ==================================================================================================

recorded_payload = pd.read_csv(
    FINAL_FILE_HASHES
)


payload_failures = []


for row in recorded_payload.itertuples(
    index=False
):

    path = (
        RELEASE_DIR
        / row.Relative_Path
    )


    if not path.is_file():

        payload_failures.append(
            row.Relative_Path
        )

        continue


    if sha256_file(
        path
    ) != row.SHA256:

        payload_failures.append(
            row.Relative_Path
        )


if payload_failures:

    raise RuntimeError(
        "Frozen Phase 11.6 payload verification failed:\n"
        +
        str(
            payload_failures[
                :10
            ]
        )
    )


# ==================================================================================================
# Verify release model-input contract
# ==================================================================================================

model_contract = read_json(
    MODEL_INPUT_CONTRACT_FILE
)


allowed_predictors = model_contract.get(
    "allowed_predictors"
)


if allowed_predictors != FEATURES:

    raise RuntimeError(
        "\nRelease predictor contract changed.\n\n"
        f"Observed:\n{allowed_predictors}\n\n"
        f"Expected:\n{FEATURES}"
    )


# ==================================================================================================
# Verify target contract
# ==================================================================================================

target_contract = read_json(
    TARGET_CONTRACT_FILE
)


contract_horizons = target_contract.get(
    "horizons",
    target_contract.get(
        "forecast_horizons_minutes"
    )
)


if contract_horizons != HORIZONS:

    raise RuntimeError(
        f"Target horizon contract changed: {contract_horizons}"
    )


print(
    "✓ immutable Phase 11.6 release verified"
)

print(
    "✓ complete release payload hash table verified"
)

print(
    "✓ predictor contract = 14 frozen predictors"
)

print(
    "✓ target horizons =",
    HORIZONS
)


# ==================================================================================================
# 12.0C — OUTPUT SAFETY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.0C — OUTPUT SAFETY")
print("=" * 120)


MODEL_READY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


if STAGING.exists():

    shutil.rmtree(
        STAGING
    )


# --------------------------------------------------------------------------------------------------
# If 12.0 already completed successfully, do not overwrite it.
# --------------------------------------------------------------------------------------------------

if FINAL_OUT.exists():

    existing_lock_file = (
        FINAL_OUT
        / "12_0_BUILD_LOCK.json"
    )

    existing_lock_hash_file = (
        FINAL_OUT
        / "12_0_BUILD_LOCK_SHA256.txt"
    )

    existing_ready_file = (
        FINAL_OUT
        / "12_0_READY_FOR_SEQUENCE_AUDIT.json"
    )


    if (
        existing_lock_file.is_file()

        and

        existing_lock_hash_file.is_file()

        and

        existing_ready_file.is_file()
    ):

        actual_existing_lock_hash = sha256_file(
            existing_lock_file
        )


        recorded_existing_lock_hash = (

            existing_lock_hash_file

            .read_text(
                encoding="utf-8"
            )

            .strip()
        )


        existing_ready = read_json(
            existing_ready_file
        )


        if (
            actual_existing_lock_hash
            ==
            recorded_existing_lock_hash

            and

            existing_ready.get(
                "ready_for_phase_12_1"
            )
            is True
        ):

            print(
                "✓ Phase 12.0 already exists and is locked."
            )

            print(
                "Output:"
            )

            print(
                FINAL_OUT
            )

            print(
                "\nNEXT:"
            )

            print(
                "Phase 12.1 — independent sequence leakage/quality audit."
            )

            raise SystemExit


    # Existing folder exists but was not completed.
    # Preserve it instead of deleting evidence.

    attempt = 1


    while (

        MODEL_READY_ROOT
        / f"_incomplete_12_0_attempt_{attempt}"

    ).exists():

        attempt += 1


    backup = (

        MODEL_READY_ROOT
        / f"_incomplete_12_0_attempt_{attempt}"
    )


    shutil.move(
        str(
            FINAL_OUT
        ),
        str(
            backup
        )
    )


    print(
        "⚠ incomplete previous Phase 12.0 preserved:"
    )

    print(
        backup
    )


STAGING.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 12.0 BUILD SPEC
# ==================================================================================================

write_json(
    {
        "dataset":
            "Synthetic_NAF_v1",

        "dataset_version":
            "1.0.0",

        "source_release_path":
            str(
                RELEASE_DIR
            ),

        "source_release_lock_sha256":
            release_lock_hash,

        "windows_minutes":
            WINDOWS,

        "primary_window_minutes":
            PRIMARY_WINDOW,

        "forecast_horizons_minutes":
            HORIZONS,

        "primary_horizons_minutes":
            PRIMARY_HORIZONS,

        "feature_order":
            FEATURES,

        "family_order":
            FAMILY_ORDER,

        "sequence_semantics":
            "X[t-W+1:t], inclusive of forecast origin t",

        "future_rows_in_X":
            False,

        "cross_episode_sequences":
            False,

        "scaling_in_phase_12_0":
            False,

        "resampling_in_phase_12_0":
            False,

        "model_training_in_phase_12_0":
            False,
    },

    STAGING
    / "12_0_BUILD_SPEC.json"
)


write_json(
    {
        "family_to_code":
            FAMILY_TO_CODE,

        "negative_or_not_applicable_code":
            -1,
    },

    STAGING
    / "FAMILY_CODEBOOK.json"
)


write_json(
    {
        "feature_order":
            FEATURES
    },

    STAGING
    / "FEATURE_ORDER.json"
)


# ==================================================================================================
# BUILD STORAGE
# ==================================================================================================

summary_rows = []

split_overview_rows = []


# ==================================================================================================
# SPLIT BUILDER
# ==================================================================================================

def process_split(
    split_name,
    split_folder
):

    print("\n" + "-" * 120)
    print(
        f"BUILD SPLIT: {split_name}"
    )
    print("-" * 120)


    # ==============================================================================================
    # Load final-release split
    # ==============================================================================================

    base = (
        RELEASE_DIR
        / "data"
        / split_folder
    )


    pred_file = (
        base
        / "predictors.csv"
    )

    targ_file = (
        base
        / "targets.csv"
    )

    meta_file = (
        base
        / "metadata.csv"
    )


    for path in [

        pred_file,

        targ_file,

        meta_file,

    ]:

        require_file(
            path
        )


    pred = pd.read_csv(
        pred_file,
        low_memory=False
    )

    targ = pd.read_csv(
        targ_file,
        low_memory=False
    )

    meta = pd.read_csv(
        meta_file,
        low_memory=False
    )


    keys = [
        "Episode_ID",
        "Minute_Index"
    ]


    # ==============================================================================================
    # Basic alignment
    # ==============================================================================================

    if not (
        len(
            pred
        )
        ==
        len(
            targ
        )
        ==
        len(
            meta
        )
    ):

        raise RuntimeError(
            f"{split_name}: row counts differ."
        )


    if not pred[
        keys
    ].equals(
        targ[
            keys
        ]
    ):

        raise RuntimeError(
            f"{split_name}: predictor/target keys misaligned."
        )


    if not pred[
        keys
    ].equals(
        meta[
            keys
        ]
    ):

        raise RuntimeError(
            f"{split_name}: predictor/metadata keys misaligned."
        )


    if list(
        pred.columns
    ) != [
        *keys,
        *FEATURES
    ]:

        raise RuntimeError(
            f"{split_name}: predictor schema differs from frozen contract."
        )


    # ==============================================================================================
    # Required target fields
    # ==============================================================================================

    required_target_cols = [

        "Clean_PreOnset_Origin",

        "Minutes_To_Next_Onset",

        "Episode_Attack_Family",
    ]


    for h in HORIZONS:

        required_target_cols += [

            f"Forecast_Eligible_H{h}",

            f"Onset_Within_H{h}",

            f"Onset_Family_H{h}",
        ]


    missing = [

        c

        for c in required_target_cols

        if c not in targ.columns
    ]


    if missing:

        raise RuntimeError(
            f"{split_name}: missing target columns:\n{missing}"
        )


    # ==============================================================================================
    # Deterministic episode/minute ordering
    # ==============================================================================================

    order_df = pred[
        keys
    ].copy()


    order_df[
        "_orig_row"
    ] = np.arange(
        len(
            order_df
        ),
        dtype=np.int64
    )


    order_df[
        "_episode_sort"
    ] = order_df[
        "Episode_ID"
    ].astype(
        str
    )


    order = (

        order_df

        .sort_values(
            [
                "_episode_sort",
                "Minute_Index"
            ],
            kind="mergesort"
        )[
            "_orig_row"
        ]

        .to_numpy(
            dtype=np.int64
        )
    )


    pred_s = (

        pred

        .iloc[
            order
        ]

        .reset_index(
            drop=True
        )
    )


    targ_s = (

        targ

        .iloc[
            order
        ]

        .reset_index(
            drop=True
        )
    )


    meta_s = (

        meta

        .iloc[
            order
        ]

        .reset_index(
            drop=True
        )
    )


    episode = (

        pred_s[
            "Episode_ID"
        ]

        .astype(
            str
        )

        .to_numpy()
    )


    minute = pd.to_numeric(

        pred_s[
            "Minute_Index"
        ],

        errors="raise"

    ).to_numpy(
        dtype=np.int64
    )


    orig_row = order.copy()


    # ==============================================================================================
    # Episode boundaries
    # ==============================================================================================

    change = np.r_[

        True,

        episode[
            1:
        ]
        !=
        episode[
            :-1
        ]
    ]


    starts = np.flatnonzero(
        change
    )


    ends = np.r_[

        starts[
            1:
        ],

        len(
            episode
        )
    ]


    pos_in_episode = np.empty(
        len(
            episode
        ),
        dtype=np.int32
    )


    for start, end in zip(
        starts,
        ends
    ):

        minutes_episode = minute[
            start:end
        ]


        if len(
            minutes_episode
        ) == 0:

            raise RuntimeError(
                f"{split_name}: empty episode block."
            )


        if (
            len(
                minutes_episode
            )
            >
            1
        ):

            if not np.all(

                np.diff(
                    minutes_episode
                )
                ==
                1

            ):

                raise RuntimeError(
                    f"{split_name}: non-contiguous Minute_Index "
                    f"in episode {episode[start]}."
                )


        pos_in_episode[
            start:end
        ] = np.arange(
            end
            -
            start,
            dtype=np.int32
        )


    # ==============================================================================================
    # Authoritative timestamp audit
    # ==============================================================================================

    if "Timestamp_UTC" in meta_s.columns:

        timestamps = pd.to_datetime(

            meta_s[
                "Timestamp_UTC"
            ],

            errors="coerce",

            utc=True
        )


        if timestamps.isna().any():

            raise RuntimeError(
                f"{split_name}: invalid Timestamp_UTC in frozen release metadata."
            )


        timestamp_ns = (

            timestamps

            .astype(
                "int64"
            )

            .to_numpy()
        )


        for start, end in zip(
            starts,
            ends
        ):

            if (
                end
                -
                start
            ) > 1:

                delta_seconds = (

                    np.diff(
                        timestamp_ns[
                            start:end
                        ]
                    )

                    /
                    1e9
                )


                if not np.all(
                    delta_seconds
                    ==
                    60
                ):

                    raise RuntimeError(
                        f"{split_name}: timestamp spacing is not exactly "
                        f"60 seconds in episode {episode[start]}."
                    )


    # ==============================================================================================
    # Raw predictor matrix
    # ==============================================================================================

    X_raw = pred_s[
        FEATURES
    ].to_numpy(
        dtype=np.float32
    )


    if not np.isfinite(
        X_raw
    ).all():

        raise RuntimeError(
            f"{split_name}: predictor matrix contains non-finite values."
        )


    if (
        X_raw
        <
        0
    ).any():

        raise RuntimeError(
            f"{split_name}: predictor matrix contains negative values."
        )


    # ==============================================================================================
    # All horizon eligibility
    # ==============================================================================================

    eligible_all = np.column_stack(
        [

            pd.to_numeric(

                targ_s[
                    f"Forecast_Eligible_H{h}"
                ],

                errors="raise"

            ).to_numpy(
                dtype=np.int8
            )

            for h in HORIZONS
        ]
    )


    if not np.isin(
        eligible_all,
        [
            0,
            1
        ]
    ).all():

        raise RuntimeError(
            f"{split_name}: forecast eligibility is not binary."
        )


    # ==============================================================================================
    # Binary labels
    #
    # Encoding:
    #
    #     -1 = horizon-ineligible
    #      0 = eligible negative
    #      1 = eligible positive
    # ==============================================================================================

    y_all = np.full(

        (
            len(
                targ_s
            ),
            len(
                HORIZONS
            )
        ),

        -1,

        dtype=np.int8
    )


    # ==============================================================================================
    # Family labels
    #
    #     -1 = not applicable
    #     0..4 = attack family
    # ==============================================================================================

    family_all = np.full(

        (
            len(
                targ_s
            ),
            len(
                HORIZONS
            )
        ),

        -1,

        dtype=np.int8
    )


    # ==============================================================================================
    # Validate each horizon before sequence construction
    # ==============================================================================================

    for j, h in enumerate(
        HORIZONS
    ):

        raw_y = pd.to_numeric(

            targ_s[
                f"Onset_Within_H{h}"
            ],

            errors="coerce"

        ).to_numpy(
            dtype=float
        )


        eligible_h = eligible_all[
            :,
            j
        ].astype(
            bool
        )


        if np.isnan(
            raw_y[
                eligible_h
            ]
        ).any():

            raise RuntimeError(
                f"{split_name} H{h}: eligible target contains NaN."
            )


        if (

            ~np.isnan(
                raw_y[
                    ~eligible_h
                ]
            )

        ).any():

            raise RuntimeError(
                f"{split_name} H{h}: ineligible target should be NaN."
            )


        if not np.isin(

            raw_y[
                eligible_h
            ],

            [
                0.0,
                1.0
            ]

        ).all():

            raise RuntimeError(
                f"{split_name} H{h}: eligible target is not binary."
            )


        y_all[
            eligible_h,
            j
        ] = raw_y[
            eligible_h
        ].astype(
            np.int8
        )


        positive_mask = (
            y_all[
                :,
                j
            ]
            ==
            1
        )


        family_text = (

            targ_s[
                f"Onset_Family_H{h}"
            ]

            .astype(
                "string"
            )
        )


        if positive_mask.any():

            family_codes = (

                family_text.loc[
                    positive_mask
                ]

                .map(
                    FAMILY_TO_CODE
                )
            )


            if family_codes.isna().any():

                bad_labels = sorted(

                    family_text.loc[
                        positive_mask
                    ][
                        family_codes.isna()
                    ]

                    .dropna()

                    .unique()

                    .tolist()
                )


                raise RuntimeError(
                    f"{split_name} H{h}: unknown positive family labels "
                    f"{bad_labels}"
                )


            family_all[
                positive_mask,
                j
            ] = family_codes.to_numpy(
                dtype=np.int8
            )


        observed_positive = int(
            positive_mask.sum()
        )


        expected_positive = EXPECTED_POS[
            (
                split_name,
                h
            )
        ]


        if (
            observed_positive
            !=
            expected_positive
        ):

            raise RuntimeError(
                f"{split_name} H{h}: "
                f"{observed_positive} positives; "
                f"expected {expected_positive}."
            )


        positive_events = targ_s.loc[
            positive_mask,
            "Episode_ID"
        ].nunique()


        if (
            positive_events
            !=
            EXPECTED_EVENTS[
                split_name
            ]
        ):

            raise RuntimeError(
                f"{split_name} H{h}: "
                f"{positive_events} positive events; "
                f"expected {EXPECTED_EVENTS[split_name]}."
            )


    # ==============================================================================================
    # Clean-origin validation
    # ==============================================================================================

    clean = pd.to_numeric(

        targ_s[
            "Clean_PreOnset_Origin"
        ],

        errors="raise"

    ).to_numpy(
        dtype=np.int8
    )


    any_eligible = eligible_all.any(
        axis=1
    )


    if np.any(

        any_eligible

        &

        (
            clean
            !=
            1
        )

    ):

        raise RuntimeError(
            f"{split_name}: forecast-eligible origin is not clean/pre-onset."
        )


    # ==============================================================================================
    # TTO target
    # ==============================================================================================

    tto = pd.to_numeric(

        targ_s[
            "Minutes_To_Next_Onset"
        ],

        errors="coerce"

    ).to_numpy(
        dtype=np.float32
    )


    # ==============================================================================================
    # Metadata that may be useful for evaluation only
    # ==============================================================================================

    eval_meta_cols = [

        "Split",

        "Timestamp_UTC",

        "Calibration_Source",

        "Episode_Type",

        "Precursor_Regime",

        "Precursor_Length_Minutes",

        "Precursor_Max_Alpha",

        "Synthetic_Phase",

        "Temporal_Segment_ID",
    ]


    eval_meta_cols = [

        column

        for column in eval_meta_cols

        if column in meta_s.columns
    ]


    # ==============================================================================================
    # Split tag must remain correct
    # ==============================================================================================

    if "Split" in meta_s.columns:

        split_values = set(

            meta_s[
                "Split"
            ]

            .astype(
                str
            )

            .unique()
        )


        if split_values != {
            split_name
        }:

            raise RuntimeError(
                f"{split_name}: metadata Split values are {split_values}."
            )


    split_overview_rows.append(
        {
            "Split":
                split_name,

            "Minute_Rows":
                len(
                    pred_s
                ),

            "Episodes":
                int(
                    pd.Series(
                        episode
                    ).nunique()
                ),

            "Any_Horizon_Eligible_Origins":
                int(
                    any_eligible.sum()
                ),
        }
    )


    # ==============================================================================================
    # BUILD EACH HISTORY WINDOW
    # ==============================================================================================

    for W in WINDOWS:

        out_dir = (

            STAGING
            / f"W{W:02d}"
            / split_name.lower()
        )


        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        # ------------------------------------------------------------------------------------------
        # Full W history must exist.
        #
        # W=10 means:
        #
        #     t-9 ... t
        #
        # exactly 10 minute rows.
        # ------------------------------------------------------------------------------------------

        valid_history = (
            pos_in_episode
            >=
            (
                W
                -
                1
            )
        )


        # Keep a sequence if at least one frozen forecasting horizon
        # considers this origin eligible.

        selected = np.flatnonzero(

            valid_history

            &

            any_eligible

        ).astype(
            np.int64
        )


        n = len(
            selected
        )


        if n == 0:

            raise RuntimeError(
                f"{split_name} W{W}: zero usable sequence samples."
            )


        sequence_starts = (

            selected

            -
            (
                W
                -
                1
            )
        )


        # ==========================================================================================
        # HARD NO-CROSS CHECK
        # ==========================================================================================

        if not np.array_equal(

            episode[
                sequence_starts
            ],

            episode[
                selected
            ]

        ):

            raise RuntimeError(
                f"{split_name} W{W}: sequence would cross Episode_ID."
            )


        if not np.array_equal(

            minute[
                selected
            ]
            -
            minute[
                sequence_starts
            ],

            np.full(
                n,
                W
                -
                1
            )

        ):

            raise RuntimeError(
                f"{split_name} W{W}: non-contiguous sequence history detected."
            )


        # ==========================================================================================
        # WRITE X AS MEMORY-MAPPED .NPY
        #
        # This avoids constructing one enormous [N,W,14] RAM array.
        # ==========================================================================================

        x_path = (
            out_dir
            / "X.npy"
        )


        X_memmap = np.lib.format.open_memmap(

            x_path,

            mode="w+",

            dtype=np.float32,

            shape=(
                n,
                W,
                len(
                    FEATURES
                )
            )
        )


        # Order:
        #
        #   t-W+1 ... t

        offsets = np.arange(

            W
            -
            1,

            -1,

            -1,

            dtype=np.int64
        )


        chunk_size = 4096


        for lo in range(
            0,
            n,
            chunk_size
        ):

            hi = min(
                lo
                +
                chunk_size,
                n
            )


            endpoint_chunk = selected[
                lo:hi
            ]


            index_matrix = (

                endpoint_chunk[
                    :,
                    None
                ]

                -

                offsets[
                    None,
                    :
                ]
            )


            # --------------------------------------------------------------------------------------
            # Defensive cross-episode check
            # --------------------------------------------------------------------------------------

            if not np.all(

                episode[
                    index_matrix[
                        :,
                        0
                    ]
                ]

                ==

                episode[
                    index_matrix[
                        :,
                        -1
                    ]
                ]

            ):

                raise RuntimeError(
                    f"{split_name} W{W}: cross-episode sequence found "
                    f"during X write."
                )


            # --------------------------------------------------------------------------------------
            # Defensive minute-continuity check
            # --------------------------------------------------------------------------------------

            if not np.all(

                minute[
                    index_matrix[
                        :,
                        -1
                    ]
                ]

                -

                minute[
                    index_matrix[
                        :,
                        0
                    ]
                ]

                ==

                W
                -
                1

            ):

                raise RuntimeError(
                    f"{split_name} W{W}: sequence minute continuity failure."
                )


            X_memmap[
                lo:hi
            ] = X_raw[
                index_matrix
            ]


        X_memmap.flush()

        del X_memmap


        # ==========================================================================================
        # FAST LABEL ARRAYS
        # ==========================================================================================

        eligible = eligible_all[
            selected
        ].astype(
            np.uint8,
            copy=True
        )


        y_binary = y_all[
            selected
        ].astype(
            np.int8,
            copy=True
        )


        y_family = family_all[
            selected
        ].astype(
            np.int8,
            copy=True
        )


        tto_selected = tto[
            selected
        ].astype(
            np.float32,
            copy=True
        )


        np.save(

            out_dir
            / "eligible.npy",

            eligible,

            allow_pickle=False
        )


        np.save(

            out_dir
            / "y_binary.npy",

            y_binary,

            allow_pickle=False
        )


        np.save(

            out_dir
            / "y_family.npy",

            y_family,

            allow_pickle=False
        )


        np.save(

            out_dir
            / "minutes_to_onset.npy",

            tto_selected,

            allow_pickle=False
        )


        # ==========================================================================================
        # EVALUATION / TRACEABILITY INDEX
        #
        # IMPORTANT:
        #
        # This file is NOT X.
        #
        # It contains metadata needed later for:
        #
        #   - event-level metrics
        #   - source-domain analysis
        #   - precursor-regime analysis
        #   - lead-time reconstruction
        #   - debugging
        #
        # ==========================================================================================

        index_data = {

            "Sample_ID":
                np.arange(
                    n,
                    dtype=np.int64
                ),

            "Split":
                np.repeat(
                    split_name,
                    n
                ),

            "Episode_ID":
                episode[
                    selected
                ],

            "Minute_Index":
                minute[
                    selected
                ],

            "Origin_Row_In_Release_Split":
                orig_row[
                    selected
                ],

            "Sequence_Start_Minute_Index":
                minute[
                    sequence_starts
                ],

            "Sequence_End_Minute_Index":
                minute[
                    selected
                ],

            "History_Window_Minutes":
                np.full(
                    n,
                    W,
                    dtype=np.int16
                ),

            "Minutes_To_Next_Onset":
                tto_selected,

            "Episode_Attack_Family":
                (
                    targ_s.loc[
                        selected,
                        "Episode_Attack_Family"
                    ]

                    .astype(
                        "string"
                    )

                    .fillna(
                        ""
                    )

                    .to_numpy()
                ),
        }


        # ==========================================================================================
        # Optional evaluation metadata
        # ==========================================================================================

        for column in eval_meta_cols:

            # Split already explicitly added.

            if column == "Split":

                continue


            index_data[
                column
            ] = meta_s.loc[
                selected,
                column
            ].to_numpy()


        # ==========================================================================================
        # Horizon labels in index
        # ==========================================================================================

        for j, h in enumerate(
            HORIZONS
        ):

            index_data[
                f"Eligible_H{h}"
            ] = eligible[
                :,
                j
            ]


            index_data[
                f"Y_H{h}"
            ] = y_binary[
                :,
                j
            ]


            index_data[
                f"Onset_Family_H{h}"
            ] = (

                targ_s.loc[
                    selected,
                    f"Onset_Family_H{h}"
                ]

                .astype(
                    "string"
                )

                .fillna(
                    ""
                )

                .to_numpy()
            )


            index_data[
                f"Family_Code_H{h}"
            ] = y_family[
                :,
                j
            ]


        index_df = pd.DataFrame(
            index_data
        )


        # ==========================================================================================
        # MODEL FEATURE LEAK CHECK
        #
        # Evaluation index must contain NO predictor feature columns.
        # ==========================================================================================

        leaked_features = sorted(

            set(
                FEATURES
            )

            .intersection(
                index_df.columns
            )
        )


        if leaked_features:

            raise RuntimeError(
                f"{split_name} W{W}: predictors leaked into index:\n"
                f"{leaked_features}"
            )


        # Parquet keeps this much smaller/faster than CSV.

        index_df.to_parquet(

            out_dir
            / "index.parquet",

            index=False
        )


        # ==========================================================================================
        # DATASET INFO
        # ==========================================================================================

        write_json(
            {
                "split":
                    split_name,

                "history_window_minutes":
                    W,

                "sample_count":
                    n,

                "x_shape":
                    [
                        n,
                        W,
                        len(
                            FEATURES
                        )
                    ],

                "x_dtype":
                    "float32",

                "feature_order":
                    FEATURES,

                "horizon_order":
                    HORIZONS,

                "y_binary_encoding": {

                    "ineligible":
                        -1,

                    "negative":
                        0,

                    "positive":
                        1,
                },

                "family_codebook":
                    FAMILY_TO_CODE,

                "y_family_ineligible_or_nonpositive":
                    -1,

                "minutes_to_onset_nan_allowed":
                    True,

                "sequence_includes_origin_t":
                    True,

                "sequence_contains_future_rows":
                    False,

                "cross_episode_sequences_allowed":
                    False,

                "scaled":
                    False,

                "resampled":
                    False,
            },

            out_dir
            / "dataset_info.json"
        )


        # ==========================================================================================
        # SUPPORT CHECK PER HORIZON
        # ==========================================================================================

        for j, h in enumerate(
            HORIZONS
        ):

            eligible_h = eligible[
                :,
                j
            ].astype(
                bool
            )


            y_h = y_binary[
                :,
                j
            ]


            positive_h = (

                eligible_h

                &

                (
                    y_h
                    ==
                    1
                )
            )


            negative_h = (

                eligible_h

                &

                (
                    y_h
                    ==
                    0
                )
            )


            positive_origins = int(
                positive_h.sum()
            )


            expected_positive = EXPECTED_POS[
                (
                    split_name,
                    h
                )
            ]


            # Because onset is always sufficiently far into each episode,
            # W<=30 must preserve every positive forecasting origin.

            if (
                positive_origins
                !=
                expected_positive
            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    f"positive origins changed "
                    f"({positive_origins} != {expected_positive})."
                )


            unique_positive_events = int(

                index_df.loc[
                    positive_h,
                    "Episode_ID"
                ]

                .nunique()
            )


            if (
                unique_positive_events
                !=
                EXPECTED_EVENTS[
                    split_name
                ]
            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    f"positive event coverage changed "
                    f"({unique_positive_events} != "
                    f"{EXPECTED_EVENTS[split_name]})."
                )


            # ======================================================================================
            # Positive time-to-onset semantics
            # ======================================================================================

            positive_tto = tto_selected[
                positive_h
            ]


            if len(
                positive_tto
            ):

                if np.isnan(
                    positive_tto
                ).any():

                    raise RuntimeError(
                        f"{split_name} W{W} H{h}: "
                        f"positive TTO contains NaN."
                    )


                if not np.all(

                    (
                        positive_tto
                        >
                        0
                    )

                    &

                    (
                        positive_tto
                        <=
                        h
                    )

                ):

                    raise RuntimeError(
                        f"{split_name} W{W} H{h}: "
                        f"positive TTO falls outside (0,H]."
                    )


            summary_rows.append(
                {
                    "Split":
                        split_name,

                    "History_Window_Minutes":
                        W,

                    "Horizon_Minutes":
                        h,

                    "Sequence_Samples_Total":
                        n,

                    "Eligible_Origins":
                        int(
                            eligible_h.sum()
                        ),

                    "Positive_Origins":
                        positive_origins,

                    "Negative_Origins":
                        int(
                            negative_h.sum()
                        ),

                    "Positive_Rate":
                        (
                            float(
                                positive_origins
                                /
                                eligible_h.sum()
                            )

                            if eligible_h.sum()

                            else np.nan
                        ),

                    "Unique_Positive_Events":
                        unique_positive_events,
                }
            )


        print(
            f"✓ {split_name:<10} "
            f"W{W:<2} "
            f"samples={n:,} "
            f"X=[{n:,},{W},14]"
        )


        del index_df

        del eligible

        del y_binary

        del y_family

        del tto_selected

        gc.collect()


    del pred

    del targ

    del meta

    del pred_s

    del targ_s

    del meta_s

    del X_raw

    gc.collect()


# ==================================================================================================
# 12.0D — BUILD ALL SPLITS AND WINDOWS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.0D — BUILD ALL SPLITS / WINDOWS")
print("=" * 120)


for split_name, split_folder in SPLIT_DIRS.items():

    process_split(
        split_name,
        split_folder
    )


# ==================================================================================================
# 12.0E — GLOBAL SUPPORT AUDIT
# ==================================================================================================

summary = pd.DataFrame(
    summary_rows
)


split_overview = pd.DataFrame(
    split_overview_rows
)


save_csv(

    summary,

    STAGING
    / "12_0_SEQUENCE_SUPPORT_SUMMARY.csv"
)


save_csv(

    split_overview,

    STAGING
    / "12_0_SPLIT_OVERVIEW.csv"
)


print("\n" + "=" * 120)
print("12.0E — GLOBAL BUILD AUDIT")
print("=" * 120)


print(
    summary.to_string(
        index=False
    )
)


# 4 windows × 3 splits × 5 horizons

if len(
    summary
) != 60:

    raise RuntimeError(
        f"Expected 60 W×split×H summary rows, "
        f"found {len(summary)}."
    )


# ==============================================================================================
# Every positive forecasting origin must survive sequence construction.
#
# We deliberately designed attack onsets with enough prehistory for W<=30.
# ==============================================================================================

for row in summary.itertuples(
    index=False
):

    expected_positive = EXPECTED_POS[
        (
            row.Split,
            int(
                row.Horizon_Minutes
            )
        )
    ]


    expected_events = EXPECTED_EVENTS[
        row.Split
    ]


    if (
        int(
            row.Positive_Origins
        )
        !=
        expected_positive
    ):

        raise RuntimeError(
            "Global positive-origin preservation check failed."
        )


    if (
        int(
            row.Unique_Positive_Events
        )
        !=
        expected_events
    ):

        raise RuntimeError(
            "Global positive-event preservation check failed."
        )


# ==============================================================================================
# Primary benchmark configurations must exist
# ==============================================================================================

for split in SPLIT_DIRS:

    primary_subset = summary.loc[
        (
            summary[
                "Split"
            ]
            ==
            split
        )

        &

        (
            summary[
                "History_Window_Minutes"
            ]
            ==
            PRIMARY_WINDOW
        )

        &

        (
            summary[
                "Horizon_Minutes"
            ].isin(
                PRIMARY_HORIZONS
            )
        )
    ]


    if len(
        primary_subset
    ) != 2:

        raise RuntimeError(
            f"{split}: primary W10/H5-H10 sequence set missing."
        )


print(
    "\n✓ 60/60 split × window × horizon combinations built"
)

print(
    "✓ all frozen positive origins preserved"
)

print(
    "✓ all independent positive attack events preserved"
)

print(
    "✓ W10/H5 and W10/H10 primary configurations present"
)

print(
    "✓ no scaling performed"
)

print(
    "✓ no oversampling performed"
)

print(
    "✓ no class weighting performed"
)

print(
    "✓ no model training performed"
)


# ==================================================================================================
# 12.0F — BUILD MANIFEST
# ==================================================================================================

build_manifest = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.0",

    "status":
        "CAUSAL_SEQUENCE_BUILD_COMPLETE",


    "source_release_path":
        str(
            RELEASE_DIR
        ),

    "source_release_lock_sha256":
        release_lock_hash,


    "feature_order":
        FEATURES,

    "feature_count":
        14,


    "windows_minutes":
        WINDOWS,

    "primary_window_minutes":
        PRIMARY_WINDOW,


    "horizons_minutes":
        HORIZONS,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "family_codebook":
        FAMILY_TO_CODE,


    "sequence_shape":
        "[N, W, 14]",


    "sequence_semantics":
        "inclusive causal history X[t-W+1:t], ending at forecast origin t",


    "future_rows_in_X":
        False,

    "cross_episode_sequences":
        False,


    "scaling_applied":
        False,

    "resampling_applied":
        False,

    "class_weighting_applied":
        False,

    "model_training_performed":
        False,


    "binary_label_encoding": {

        "ineligible":
            -1,

        "negative":
            0,

        "positive":
            1,
    },


    "family_label_encoding": {

        "not_applicable_or_nonpositive":
            -1,

        **{
            str(
                code
            ):
                family

            for family, code in FAMILY_TO_CODE.items()
        },
    },


    "authoritative_sequence_index":
        "Wxx/<split>/index.parquet",

    "authoritative_X":
        "Wxx/<split>/X.npy",


    "next_phase":
        "12.1_INDEPENDENT_SEQUENCE_LEAKAGE_AND_QUALITY_AUDIT",
}


write_json(

    build_manifest,

    STAGING
    / "12_0_BUILD_MANIFEST.json"
)


build_manifest_sha256 = sha256_file(

    STAGING
    / "12_0_BUILD_MANIFEST.json"
)


# ==================================================================================================
# 12.0G — HASH MODEL-READY ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.0G — HASH MODEL-READY ARTIFACTS")
print("=" * 120)


file_hashes = build_hash_table(
    STAGING
)


save_csv(

    file_hashes,

    STAGING
    / "12_0_FILE_HASHES.csv"
)


file_hash_table_sha256 = sha256_file(

    STAGING
    / "12_0_FILE_HASHES.csv"
)


print(
    "Artifacts hashed:",
    len(
        file_hashes
    )
)


# ==================================================================================================
# 12.0H — BUILD LOCK
# ==================================================================================================

build_lock = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.0",

    "lock_status":
        "BUILT_NOT_YET_12_1_AUDITED",


    "source_release_lock_sha256":
        release_lock_hash,


    "build_manifest_sha256":
        build_manifest_sha256,

    "file_hash_table_sha256":
        file_hash_table_sha256,


    "windows_minutes":
        WINDOWS,

    "horizons_minutes":
        HORIZONS,


    "primary_window_minutes":
        PRIMARY_WINDOW,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "no_model_training":
        True,


    "next_phase":
        "12.1_INDEPENDENT_SEQUENCE_LEAKAGE_AND_QUALITY_AUDIT",
}


write_json(

    build_lock,

    STAGING
    / "12_0_BUILD_LOCK.json"
)


build_lock_sha256 = sha256_file(

    STAGING
    / "12_0_BUILD_LOCK.json"
)


(
    STAGING
    / "12_0_BUILD_LOCK_SHA256.txt"
).write_text(
    build_lock_sha256
    +
    "\n",
    encoding="utf-8"
)


# ==================================================================================================
# 12.0I — VERIFY HASHED PAYLOAD
# ==================================================================================================

recorded_hashes = pd.read_csv(

    STAGING
    / "12_0_FILE_HASHES.csv"
)


hash_failures = []


for row in recorded_hashes.itertuples(
    index=False
):

    path = (
        STAGING
        / row.Relative_Path
    )


    if not path.is_file():

        hash_failures.append(
            row.Relative_Path
        )

        continue


    if sha256_file(
        path
    ) != row.SHA256:

        hash_failures.append(
            row.Relative_Path
        )


if hash_failures:

    raise RuntimeError(
        "Phase 12.0 staging hash verification failed:\n"
        +
        str(
            hash_failures[
                :10
            ]
        )
    )


print(
    "✓ complete model-ready payload hash table verified"
)


# ==================================================================================================
# 12.0J — READINESS GATE
# ==================================================================================================

readiness = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.0",

    "phase_12_0_status":
        "PASS",


    "ready_for_phase_12_1":
        True,


    "source_release_lock_sha256":
        release_lock_hash,


    "phase_12_0_build_lock_sha256":
        build_lock_sha256,


    "sequence_windows_minutes":
        WINDOWS,

    "primary_window_minutes":
        PRIMARY_WINDOW,


    "forecast_horizons_minutes":
        HORIZONS,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "sequence_artifacts_scaled":
        False,

    "sequence_artifacts_resampled":
        False,

    "model_training_performed":
        False,


    "next_phase":
        "12.1_INDEPENDENT_SEQUENCE_LEAKAGE_AND_QUALITY_AUDIT",
}


write_json(

    readiness,

    STAGING
    / "12_0_READY_FOR_SEQUENCE_AUDIT.json"
)


print(
    "✓ Phase 12.0 build lock created"
)

print(
    "✓ Phase 12.1 readiness gate created"
)


# ==================================================================================================
# 12.0K — FINALIZE
# ==================================================================================================

print("\n" + "=" * 120)
print("12.0K — FINALIZE")
print("=" * 120)


if FINAL_OUT.exists():

    raise RuntimeError(
        "Final Phase 12.0 directory unexpectedly exists."
    )


shutil.move(

    str(
        STAGING
    ),

    str(
        FINAL_OUT
    )
)


# ==================================================================================================
# FINAL LOCK VERIFICATION AFTER MOVE
# ==================================================================================================

final_lock_file = (
    FINAL_OUT
    / "12_0_BUILD_LOCK.json"
)

final_lock_hash_file = (
    FINAL_OUT
    / "12_0_BUILD_LOCK_SHA256.txt"
)


if (

    sha256_file(
        final_lock_file
    )

    !=

    final_lock_hash_file
    .read_text(
        encoding="utf-8"
    )
    .strip()

):

    raise RuntimeError(
        "Final Phase 12.0 build lock failed after finalization."
    )


print(
    "✓ Phase 12.0 finalized:"
)

print(
    FINAL_OUT
)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("✓ PHASE 12.0 — MODEL-READY CAUSAL SEQUENCE BUILD: PASS")
print("=" * 120)


print(
    "\nSOURCE BENCHMARK:"
)

print(
    "    Synthetic_NAF_v1 v1.0.0"
)

print(
    "    immutable Phase 11.6 release"
)


print(
    "\nHISTORY WINDOWS CREATED:"
)

print(
    "   ",
    WINDOWS
)


print(
    "\nPRIMARY HISTORY:"
)

print(
    "   ",
    PRIMARY_WINDOW,
    "minutes"
)


print(
    "\nFORECAST HORIZONS:"
)

print(
    "   ",
    HORIZONS
)


print(
    "\nPRIMARY HORIZONS:"
)

print(
    "   ",
    PRIMARY_HORIZONS
)


print(
    "\nMODEL INPUT:"
)

print(
    "    X shape = [N, W, 14]"
)

print(
    "    raw frozen predictor values"
)

print(
    "    NO metadata in X"
)

print(
    "    NO target information in X"
)

print(
    "    NO future rows in X"
)

print(
    "    NO cross-Episode_ID sequences"
)


print(
    "\nLABEL ARRAYS:"
)

print(
    "    y_binary:"
)

print(
    "        -1 = horizon ineligible"
)

print(
    "         0 = negative"
)

print(
    "         1 = positive"
)


print(
    "\n    y_family:"
)

print(
    "        -1 = not applicable"
)

print(
    "         0 = DOS_DDOS"
)

print(
    "         1 = RECON_SCAN"
)

print(
    "         2 = EXPLOIT_APP"
)

print(
    "         3 = BRUTE_AUTH"
)

print(
    "         4 = FUZZING"
)


print(
    "\nIMPORTANT:"
)

print(
    "    NO scaling yet"
)

print(
    "    NO log transform yet"
)

print(
    "    NO oversampling yet"
)

print(
    "    NO negative downsampling yet"
)

print(
    "    NO balanced mini-batches yet"
)

print(
    "    NO class weights yet"
)

print(
    "    NO focal loss yet"
)

print(
    "    NO model training yet"
)


print(
    "\nNEXT:"
)

print(
    "    PHASE 12.1"
)

print(
    "    INDEPENDENT SEQUENCE LEAKAGE / QUALITY AUDIT"
)


print(
    "\nMODEL TRAINING REMAINS BLOCKED UNTIL 12.1 PASSES."
)


print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# PHASE 12.1 — INDEPENDENT SEQUENCE LEAKAGE / QUALITY AUDIT
# ==================================================================================================
#
# PURPOSE
# -------
# Independently verify the Phase 12.0 model-ready causal sequence dataset
# against the immutable Phase 11.6 benchmark release.
#
#
# THIS PHASE MUST PROVE:
#
#   1. Every saved X sequence can be reconstructed exactly from the frozen release.
#   2. Every sequence contains exactly W historical rows ending at origin t.
#   3. No sequence crosses Episode_ID.
#   4. No future row appears in X.
#   5. TRAIN / VALIDATION / TEST remain episode-isolated.
#   6. All horizon eligibility labels are exact.
#   7. All binary onset labels are exact.
#   8. All future-family labels are exact.
#   9. Minutes_To_Next_Onset is exact.
#  10. Every positive attack event is preserved.
#  11. Family support remains exactly balanced by independent attack event.
#  12. No forbidden feature entered X.
#  13. Phase 12.0 artifacts were not mutated during this audit.
#
#
# THIS PHASE DOES NOT:
#
#   - scale features
#   - fit preprocessing
#   - oversample
#   - undersample
#   - calculate class weights for training
#   - use focal loss
#   - balance batches
#   - tune thresholds
#   - train any model
#
#
# NEXT IF PASS:
#
#   Phase 12.2 — TRAIN-only preprocessing + imbalance-policy freeze
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import shutil
import gc

import numpy as np
import pandas as pd


print("=" * 120)
print("PHASE 12.1 — INDEPENDENT SEQUENCE LEAKAGE / QUALITY AUDIT")
print("=" * 120)


# ==================================================================================================
# 12.1A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

MODEL_READY_ROOT = (
    DATASET_ROOT
    / "model_ready"
)

PHASE_12_0_DIR = (
    MODEL_READY_ROOT
    / "12_0"
)

FINAL_OUT = (
    MODEL_READY_ROOT
    / "12_1"
)

STAGING = (
    MODEL_READY_ROOT
    / "._staging_12_1"
)


# --------------------------------------------------------------------------------------------------
# Phase 12.0 control files
# --------------------------------------------------------------------------------------------------

READY_12_0_FILE = (
    PHASE_12_0_DIR
    / "12_0_READY_FOR_SEQUENCE_AUDIT.json"
)

BUILD_LOCK_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_LOCK.json"
)

BUILD_LOCK_HASH_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_LOCK_SHA256.txt"
)

BUILD_SPEC_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_SPEC.json"
)

BUILD_MANIFEST_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_MANIFEST.json"
)

FILE_HASHES_12_0 = (
    PHASE_12_0_DIR
    / "12_0_FILE_HASHES.csv"
)

FEATURE_ORDER_FILE = (
    PHASE_12_0_DIR
    / "FEATURE_ORDER.json"
)

FAMILY_CODEBOOK_FILE = (
    PHASE_12_0_DIR
    / "FAMILY_CODEBOOK.json"
)

SEQUENCE_SUPPORT_FILE = (
    PHASE_12_0_DIR
    / "12_0_SEQUENCE_SUPPORT_SUMMARY.csv"
)


# ==================================================================================================
# 12.1B — FROZEN CONTRACT
# ==================================================================================================

WINDOWS = [
    5,
    10,
    15,
    30
]


PRIMARY_WINDOW = 10


HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


PRIMARY_HORIZONS = [
    5,
    10
]


FEATURES = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Total_Packets",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum",

    "Total_Bytes",

    "Packets_Per_Second",

    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",

    "Forward_Backward_Byte_Ratio",
]


FAMILIES = [

    "DOS_DDOS",

    "RECON_SCAN",

    "EXPLOIT_APP",

    "BRUTE_AUTH",

    "FUZZING",
]


FAMILY_TO_CODE = {

    family:
        i

    for i, family in enumerate(
        FAMILIES
    )
}


SPLITS = {

    "TRAIN":
        "train",

    "VALIDATION":
        "validation",

    "TEST":
        "test",
}


EXPECTED_EVENTS = {

    "TRAIN":
        700,

    "VALIDATION":
        150,

    "TEST":
        150,
}


EXPECTED_FAMILY_EVENTS = {

    "TRAIN":
        140,

    "VALIDATION":
        30,

    "TEST":
        30,
}


EXPECTED_POSITIVES = {

    ("TRAIN", 1):
        700,

    ("TRAIN", 5):
        3500,

    ("TRAIN", 10):
        7000,

    ("TRAIN", 15):
        10500,

    ("TRAIN", 30):
        21000,


    ("VALIDATION", 1):
        150,

    ("VALIDATION", 5):
        750,

    ("VALIDATION", 10):
        1500,

    ("VALIDATION", 15):
        2250,

    ("VALIDATION", 30):
        4500,


    ("TEST", 1):
        150,

    ("TEST", 5):
        750,

    ("TEST", 10):
        1500,

    ("TEST", 15):
        2250,

    ("TEST", 30):
        4500,
}


EXPECTED_ROWS = {

    "TRAIN":
        102187,

    "VALIDATION":
        21786,

    "TEST":
        22058,
}


EXPECTED_EPISODES = {

    "TRAIN":
        840,

    "VALIDATION":
        180,

    "TEST":
        180,
}


# --------------------------------------------------------------------------------------------------
# Metadata/target fields forbidden from X
# --------------------------------------------------------------------------------------------------

FORBIDDEN_EXACT = {

    "Episode_ID",

    "Minute_Index",

    "Split",

    "Timestamp_UTC",

    "Temporal_Segment_ID",

    "Calibration_Source",

    "Episode_Type",

    "Generator_Seed",

    "Precursor_Regime",

    "Precursor_Length_Minutes",

    "Precursor_Max_Alpha",

    "Attack_State",

    "New_Attack_Onset",

    "Current_Attack_Family",

    "Episode_Attack_Family",

    "Minutes_To_Next_Onset",

    "Clean_PreOnset_Origin",
}


FORBIDDEN_PREFIXES = [

    "Forecast_Eligible_",

    "Right_Censored_",

    "Onset_Within_",

    "Onset_Family_",
]


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(
    path
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def read_json(
    path
):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def save_csv(
    df,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        path,
        index=False,
        float_format="%.10g"
    )


def normalize_text(
    series
):

    return (
        series
        .astype(
            "string"
        )
        .fillna(
            ""
        )
        .astype(
            str
        )
    )


def exact_float_equal(
    a,
    b
):

    a = np.asarray(
        a
    )

    b = np.asarray(
        b
    )


    if a.shape != b.shape:

        return False


    return bool(

        np.allclose(
            a,
            b,
            atol=0.0,
            rtol=0.0,
            equal_nan=True
        )
    )


def build_hash_table(
    root
):

    excluded = {

        "12_1_FILE_HASHES.csv",

        "12_1_AUDIT_LOCK.json",

        "12_1_AUDIT_LOCK_SHA256.txt",

        "12_1_READY_FOR_PREPROCESSING.json",
    }


    rows = []


    for path in sorted(
        root.rglob(
            "*"
        )
    ):

        if not path.is_file():

            continue


        if path.name in excluded:

            continue


        rows.append(
            {
                "Relative_Path":
                    str(
                        path.relative_to(
                            root
                        )
                    ),

                "Bytes":
                    int(
                        path.stat().st_size
                    ),

                "SHA256":
                    sha256_file(
                        path
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ==================================================================================================
# 12.1C — VERIFY PHASE 12.0 AUTHORIZATION
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1C — VERIFY PHASE 12.0 AUTHORIZATION")
print("=" * 120)


for path in [

    READY_12_0_FILE,

    BUILD_LOCK_FILE,

    BUILD_LOCK_HASH_FILE,

    BUILD_SPEC_FILE,

    BUILD_MANIFEST_FILE,

    FILE_HASHES_12_0,

    FEATURE_ORDER_FILE,

    FAMILY_CODEBOOK_FILE,

    SEQUENCE_SUPPORT_FILE,

]:

    require_file(
        path
    )


ready_12_0 = read_json(
    READY_12_0_FILE
)


if ready_12_0.get(
    "phase_12_0_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.0 status is not PASS."
    )


if ready_12_0.get(
    "ready_for_phase_12_1"
) is not True:

    raise RuntimeError(
        "Phase 12.0 does not authorize Phase 12.1."
    )


build_lock_hash = sha256_file(
    BUILD_LOCK_FILE
)


recorded_build_lock_hash = (

    BUILD_LOCK_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if (
    build_lock_hash
    !=
    recorded_build_lock_hash
):

    raise RuntimeError(
        "Phase 12.0 build lock hash failed."
    )


if (
    ready_12_0.get(
        "phase_12_0_build_lock_sha256"
    )
    !=
    build_lock_hash
):

    raise RuntimeError(
        "12.0 readiness disagrees with 12.0 build lock."
    )


build_lock = read_json(
    BUILD_LOCK_FILE
)

build_spec = read_json(
    BUILD_SPEC_FILE
)

build_manifest = read_json(
    BUILD_MANIFEST_FILE
)


if build_lock.get(
    "lock_status"
) != "BUILT_NOT_YET_12_1_AUDITED":

    raise RuntimeError(
        "Unexpected Phase 12.0 build-lock status."
    )


if build_spec.get(
    "model_training_in_phase_12_0"
) is not False:

    raise RuntimeError(
        "12.0 claims model training occurred."
    )


if build_spec.get(
    "scaling_in_phase_12_0"
) is not False:

    raise RuntimeError(
        "12.0 unexpectedly applied scaling."
    )


if build_spec.get(
    "resampling_in_phase_12_0"
) is not False:

    raise RuntimeError(
        "12.0 unexpectedly applied resampling."
    )


print(
    "✓ Phase 12.0 PASS"
)

print(
    "✓ Phase 12.0 build lock verified"
)

print(
    "✓ no training in 12.0"
)

print(
    "✓ no scaling in 12.0"
)

print(
    "✓ no resampling in 12.0"
)


# ==================================================================================================
# 12.1D — VERIFY COMPLETE 12.0 PAYLOAD
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1D — VERIFY COMPLETE PHASE 12.0 PAYLOAD")
print("=" * 120)


hash_table_12_0 = pd.read_csv(
    FILE_HASHES_12_0
)


hash_table_12_0_hash = sha256_file(
    FILE_HASHES_12_0
)


if (
    hash_table_12_0_hash
    !=
    build_lock.get(
        "file_hash_table_sha256"
    )
):

    raise RuntimeError(
        "12.0 file-hash table does not match the build lock."
    )


payload_failures = []


for row in hash_table_12_0.itertuples(
    index=False
):

    path = (
        PHASE_12_0_DIR
        / row.Relative_Path
    )


    if not path.is_file():

        payload_failures.append(
            (
                row.Relative_Path,
                "MISSING"
            )
        )

        continue


    current = sha256_file(
        path
    )


    if current != row.SHA256:

        payload_failures.append(
            (
                row.Relative_Path,
                current
            )
        )


if payload_failures:

    raise RuntimeError(
        "Phase 12.0 artifact corruption detected:\n"
        +
        str(
            payload_failures[
                :10
            ]
        )
    )


print(
    "✓ every recorded Phase 12.0 artifact hash verifies"
)

print(
    "✓ Phase 12.0 hash table itself matches build lock"
)


# ==================================================================================================
# 12.1E — VERIFY CONTRACT FILES
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1E — VERIFY SEQUENCE CONTRACT")
print("=" * 120)


feature_order = read_json(
    FEATURE_ORDER_FILE
).get(
    "feature_order"
)


if feature_order != FEATURES:

    raise RuntimeError(
        "Phase 12.0 feature order changed."
    )


codebook = read_json(
    FAMILY_CODEBOOK_FILE
)


observed_family_codebook = codebook.get(
    "family_to_code"
)


if observed_family_codebook != FAMILY_TO_CODE:

    raise RuntimeError(
        "Family codebook changed."
    )


if build_manifest.get(
    "windows_minutes"
) != WINDOWS:

    raise RuntimeError(
        "History-window contract changed."
    )


if build_manifest.get(
    "horizons_minutes"
) != HORIZONS:

    raise RuntimeError(
        "Horizon contract changed."
    )


# --------------------------------------------------------------------------------------------------
# Explicit leakage feature check
# --------------------------------------------------------------------------------------------------

forbidden_features = []


for feature in feature_order:

    if feature in FORBIDDEN_EXACT:

        forbidden_features.append(
            feature
        )


    for prefix in FORBIDDEN_PREFIXES:

        if feature.startswith(
            prefix
        ):

            forbidden_features.append(
                feature
            )


forbidden_features = sorted(
    set(
        forbidden_features
    )
)


if forbidden_features:

    raise RuntimeError(
        f"Forbidden features found in model feature order:\n"
        f"{forbidden_features}"
    )


if len(
    feature_order
) != 14:

    raise RuntimeError(
        f"Expected 14 model features, found {len(feature_order)}."
    )


print(
    "✓ exactly 14 model features"
)

print(
    "✓ feature order exact"
)

print(
    "✓ no metadata feature in X contract"
)

print(
    "✓ no target/future feature in X contract"
)


# ==================================================================================================
# 12.1F — VERIFY IMMUTABLE SOURCE RELEASE
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1F — VERIFY SOURCE PHASE 11.6 RELEASE")
print("=" * 120)


RELEASE_DIR = Path(
    build_spec[
        "source_release_path"
    ]
)


RELEASE_LOCK_FILE = (
    RELEASE_DIR
    / "RELEASE_LOCK.json"
)

RELEASE_LOCK_HASH_FILE = (
    RELEASE_DIR
    / "RELEASE_LOCK_SHA256.txt"
)

RELEASE_FILE_HASHES = (
    RELEASE_DIR
    / "FINAL_FILE_HASHES.csv"
)


for path in [

    RELEASE_LOCK_FILE,

    RELEASE_LOCK_HASH_FILE,

    RELEASE_FILE_HASHES,

]:

    require_file(
        path
    )


release_lock_hash = sha256_file(
    RELEASE_LOCK_FILE
)


recorded_release_lock_hash = (

    RELEASE_LOCK_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if (
    release_lock_hash
    !=
    recorded_release_lock_hash
):

    raise RuntimeError(
        "11.6 release-lock verification failed."
    )


if (
    release_lock_hash
    !=
    build_spec.get(
        "source_release_lock_sha256"
    )
):

    raise RuntimeError(
        "12.0 source-release lock does not match current 11.6 release."
    )


release_hash_table = pd.read_csv(
    RELEASE_FILE_HASHES
)


release_failures = []


for row in release_hash_table.itertuples(
    index=False
):

    path = (
        RELEASE_DIR
        / row.Relative_Path
    )


    if not path.is_file():

        release_failures.append(
            row.Relative_Path
        )

        continue


    if sha256_file(
        path
    ) != row.SHA256:

        release_failures.append(
            row.Relative_Path
        )


if release_failures:

    raise RuntimeError(
        "Immutable 11.6 release changed:\n"
        +
        str(
            release_failures[
                :10
            ]
        )
    )


print(
    "✓ immutable 11.6 source release verified"
)

print(
    "✓ Phase 12.0 was built from this exact release"
)


# ==================================================================================================
# 12.1G — OUTPUT SAFETY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1G — OUTPUT SAFETY")
print("=" * 120)


MODEL_READY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


if STAGING.exists():

    shutil.rmtree(
        STAGING
    )


if FINAL_OUT.exists():

    ready_existing = (
        FINAL_OUT
        / "12_1_READY_FOR_PREPROCESSING.json"
    )

    lock_existing = (
        FINAL_OUT
        / "12_1_AUDIT_LOCK.json"
    )

    lock_hash_existing = (
        FINAL_OUT
        / "12_1_AUDIT_LOCK_SHA256.txt"
    )


    if (
        ready_existing.is_file()

        and

        lock_existing.is_file()

        and

        lock_hash_existing.is_file()
    ):

        current = sha256_file(
            lock_existing
        )


        recorded = (

            lock_hash_existing

            .read_text(
                encoding="utf-8"
            )

            .strip()
        )


        ready_json = read_json(
            ready_existing
        )


        if (
            current
            ==
            recorded

            and

            ready_json.get(
                "ready_for_phase_12_2"
            )
            is True
        ):

            print(
                "✓ Phase 12.1 already completed and locked."
            )

            print(
                "\nNEXT:"
            )

            print(
                "Phase 12.2 — TRAIN-only preprocessing + imbalance-policy freeze"
            )

            raise SystemExit


    i = 1


    while (

        MODEL_READY_ROOT
        / f"_incomplete_12_1_attempt_{i}"

    ).exists():

        i += 1


    backup = (

        MODEL_READY_ROOT
        / f"_incomplete_12_1_attempt_{i}"
    )


    shutil.move(
        str(
            FINAL_OUT
        ),
        str(
            backup
        )
    )


    print(
        "⚠ previous incomplete 12.1 preserved:"
    )

    print(
        backup
    )


STAGING.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# AUDIT STORAGE
# ==================================================================================================

reconstruction_rows = []

horizon_rows = []

family_rows = []

split_episode_sets = {

    W: {}

    for W in WINDOWS
}


# ==================================================================================================
# 12.1H — INDEPENDENT SPLIT RECONSTRUCTION
# ==================================================================================================

def audit_split(
    split_name,
    split_folder
):

    print("\n" + "-" * 120)
    print(
        f"INDEPENDENT AUDIT — {split_name}"
    )
    print("-" * 120)


    release_split_dir = (
        RELEASE_DIR
        / "data"
        / split_folder
    )


    predictor_file = (
        release_split_dir
        / "predictors.csv"
    )

    target_file = (
        release_split_dir
        / "targets.csv"
    )

    metadata_file = (
        release_split_dir
        / "metadata.csv"
    )


    for path in [

        predictor_file,

        target_file,

        metadata_file,

    ]:

        require_file(
            path
        )


    # ==============================================================================================
    # Load immutable source split
    # ==============================================================================================

    predictors = pd.read_csv(
        predictor_file,
        low_memory=False
    )

    targets = pd.read_csv(
        target_file,
        low_memory=False
    )

    metadata = pd.read_csv(
        metadata_file,
        low_memory=False
    )


    if not (
        len(
            predictors
        )
        ==
        len(
            targets
        )
        ==
        len(
            metadata
        )
        ==
        EXPECTED_ROWS[
            split_name
        ]
    ):

        raise RuntimeError(
            f"{split_name}: frozen split row counts changed."
        )


    keys = [
        "Episode_ID",
        "Minute_Index"
    ]


    if not predictors[
        keys
    ].equals(
        targets[
            keys
        ]
    ):

        raise RuntimeError(
            f"{split_name}: predictor/target keys misaligned."
        )


    if not predictors[
        keys
    ].equals(
        metadata[
            keys
        ]
    ):

        raise RuntimeError(
            f"{split_name}: predictor/metadata keys misaligned."
        )


    if predictors.duplicated(
        subset=keys
    ).any():

        raise RuntimeError(
            f"{split_name}: duplicate Episode_ID/Minute_Index."
        )


    # ==============================================================================================
    # Independently reproduce the deterministic order used by 12.0
    # ==============================================================================================

    order_helper = predictors[
        keys
    ].copy()


    order_helper[
        "_original_row"
    ] = np.arange(
        len(
            order_helper
        ),
        dtype=np.int64
    )


    order_helper[
        "_episode_sort"
    ] = order_helper[
        "Episode_ID"
    ].astype(
        str
    )


    order = (

        order_helper

        .sort_values(
            [
                "_episode_sort",
                "Minute_Index"
            ],
            kind="mergesort"
        )[
            "_original_row"
        ]

        .to_numpy(
            dtype=np.int64
        )
    )


    pred_s = (

        predictors

        .iloc[
            order
        ]

        .reset_index(
            drop=True
        )
    )


    targ_s = (

        targets

        .iloc[
            order
        ]

        .reset_index(
            drop=True
        )
    )


    meta_s = (

        metadata

        .iloc[
            order
        ]

        .reset_index(
            drop=True
        )
    )


    episode = (

        pred_s[
            "Episode_ID"
        ]

        .astype(
            str
        )

        .to_numpy()
    )


    minute = pd.to_numeric(

        pred_s[
            "Minute_Index"
        ],

        errors="raise"

    ).to_numpy(
        dtype=np.int64
    )


    original_row = order.copy()


    # ==============================================================================================
    # Episode boundaries
    # ==============================================================================================

    boundary = np.r_[

        True,

        episode[
            1:
        ]
        !=
        episode[
            :-1
        ]
    ]


    starts = np.flatnonzero(
        boundary
    )


    ends = np.r_[

        starts[
            1:
        ],

        len(
            episode
        )
    ]


    if len(
        starts
    ) != EXPECTED_EPISODES[
        split_name
    ]:

        raise RuntimeError(
            f"{split_name}: "
            f"{len(starts)} episodes observed; "
            f"expected {EXPECTED_EPISODES[split_name]}."
        )


    position_in_episode = np.empty(
        len(
            episode
        ),
        dtype=np.int32
    )


    for start, end in zip(
        starts,
        ends
    ):

        episode_minutes = minute[
            start:end
        ]


        if len(
            episode_minutes
        ) > 1:

            if not np.all(

                np.diff(
                    episode_minutes
                )
                ==
                1

            ):

                raise RuntimeError(
                    f"{split_name}: non-contiguous minute sequence "
                    f"in episode {episode[start]}."
                )


        position_in_episode[
            start:end
        ] = np.arange(
            end
            -
            start,
            dtype=np.int32
        )


    # ==============================================================================================
    # Authoritative source features
    # ==============================================================================================

    X_source = pred_s[
        FEATURES
    ].to_numpy(
        dtype=np.float32
    )


    if not np.isfinite(
        X_source
    ).all():

        raise RuntimeError(
            f"{split_name}: source predictors contain NaN/inf."
        )


    if (
        X_source
        <
        0
    ).any():

        raise RuntimeError(
            f"{split_name}: source predictors contain negative values."
        )


    # ==============================================================================================
    # Independently reconstruct horizon eligibility and labels
    # ==============================================================================================

    eligible_all = np.column_stack(
        [

            pd.to_numeric(

                targ_s[
                    f"Forecast_Eligible_H{h}"
                ],

                errors="raise"

            ).to_numpy(
                dtype=np.uint8
            )

            for h in HORIZONS
        ]
    )


    if not np.isin(
        eligible_all,
        [
            0,
            1
        ]
    ).all():

        raise RuntimeError(
            f"{split_name}: eligibility contains non-binary values."
        )


    y_expected = np.full(

        (
            len(
                targ_s
            ),
            len(
                HORIZONS
            )
        ),

        -1,

        dtype=np.int8
    )


    family_expected = np.full(

        (
            len(
                targ_s
            ),
            len(
                HORIZONS
            )
        ),

        -1,

        dtype=np.int8
    )


    for j, h in enumerate(
        HORIZONS
    ):

        eligible_h = eligible_all[
            :,
            j
        ].astype(
            bool
        )


        raw_y = pd.to_numeric(

            targ_s[
                f"Onset_Within_H{h}"
            ],

            errors="coerce"

        ).to_numpy(
            dtype=float
        )


        if np.isnan(
            raw_y[
                eligible_h
            ]
        ).any():

            raise RuntimeError(
                f"{split_name} H{h}: eligible target contains NaN."
            )


        if (

            ~np.isnan(
                raw_y[
                    ~eligible_h
                ]
            )

        ).any():

            raise RuntimeError(
                f"{split_name} H{h}: ineligible target is not NaN."
            )


        if not np.isin(

            raw_y[
                eligible_h
            ],

            [
                0.0,
                1.0
            ]

        ).all():

            raise RuntimeError(
                f"{split_name} H{h}: binary target invalid."
            )


        y_expected[
            eligible_h,
            j
        ] = raw_y[
            eligible_h
        ].astype(
            np.int8
        )


        positive = (
            y_expected[
                :,
                j
            ]
            ==
            1
        )


        family_text = normalize_text(

            targ_s[
                f"Onset_Family_H{h}"
            ]
        )


        if positive.any():

            mapped = (

                family_text.loc[
                    positive
                ]

                .map(
                    FAMILY_TO_CODE
                )
            )


            if mapped.isna().any():

                raise RuntimeError(
                    f"{split_name} H{h}: "
                    "positive family label is outside frozen family set."
                )


            family_expected[
                positive,
                j
            ] = mapped.to_numpy(
                dtype=np.int8
            )


    # ==============================================================================================
    # TTO
    # ==============================================================================================

    source_tto = pd.to_numeric(

        targ_s[
            "Minutes_To_Next_Onset"
        ],

        errors="coerce"

    ).to_numpy(
        dtype=np.float32
    )


    any_eligible = eligible_all.any(
        axis=1
    )


    # ==============================================================================================
    # Audit every W
    # ==============================================================================================

    for W in WINDOWS:

        sequence_dir = (

            PHASE_12_0_DIR
            / f"W{W:02d}"
            / split_name.lower()
        )


        required_files = [

            sequence_dir
            / "X.npy",

            sequence_dir
            / "eligible.npy",

            sequence_dir
            / "y_binary.npy",

            sequence_dir
            / "y_family.npy",

            sequence_dir
            / "minutes_to_onset.npy",

            sequence_dir
            / "index.parquet",

            sequence_dir
            / "dataset_info.json",
        ]


        for path in required_files:

            require_file(
                path
            )


        info = read_json(
            sequence_dir
            / "dataset_info.json"
        )


        if info.get(
            "split"
        ) != split_name:

            raise RuntimeError(
                f"{split_name} W{W}: dataset_info split mismatch."
            )


        if int(
            info.get(
                "history_window_minutes"
            )
        ) != W:

            raise RuntimeError(
                f"{split_name} W{W}: dataset_info W mismatch."
            )


        if info.get(
            "feature_order"
        ) != FEATURES:

            raise RuntimeError(
                f"{split_name} W{W}: dataset_info feature order changed."
            )


        if info.get(
            "horizon_order"
        ) != HORIZONS:

            raise RuntimeError(
                f"{split_name} W{W}: dataset_info horizon order changed."
            )


        # ==========================================================================================
        # Reconstruct expected sequence endpoints independently
        # ==========================================================================================

        valid_history = (
            position_in_episode
            >=
            (
                W
                -
                1
            )
        )


        expected_endpoint = np.flatnonzero(

            valid_history

            &

            any_eligible

        ).astype(
            np.int64
        )


        expected_start = (

            expected_endpoint

            -
            (
                W
                -
                1
            )
        )


        n_expected = len(
            expected_endpoint
        )


        # ==========================================================================================
        # Load Phase 12.0 outputs
        # ==========================================================================================

        X_saved = np.load(

            sequence_dir
            / "X.npy",

            mmap_mode="r"
        )


        eligible_saved = np.load(

            sequence_dir
            / "eligible.npy",

            allow_pickle=False
        )


        y_saved = np.load(

            sequence_dir
            / "y_binary.npy",

            allow_pickle=False
        )


        family_saved = np.load(

            sequence_dir
            / "y_family.npy",

            allow_pickle=False
        )


        tto_saved = np.load(

            sequence_dir
            / "minutes_to_onset.npy",

            allow_pickle=False
        )


        index_df = pd.read_parquet(

            sequence_dir
            / "index.parquet"
        )


        # ==========================================================================================
        # Shape checks
        # ==========================================================================================

        expected_shape = (
            n_expected,
            W,
            len(
                FEATURES
            )
        )


        if tuple(
            X_saved.shape
        ) != expected_shape:

            raise RuntimeError(
                f"{split_name} W{W}: "
                f"X shape {X_saved.shape}; "
                f"expected {expected_shape}."
            )


        if X_saved.dtype != np.float32:

            raise RuntimeError(
                f"{split_name} W{W}: "
                f"X dtype {X_saved.dtype}; expected float32."
            )


        expected_label_shape = (
            n_expected,
            len(
                HORIZONS
            )
        )


        for array_name, array in [

            (
                "eligible",
                eligible_saved
            ),

            (
                "y_binary",
                y_saved
            ),

            (
                "y_family",
                family_saved
            ),

        ]:

            if array.shape != expected_label_shape:

                raise RuntimeError(
                    f"{split_name} W{W}: "
                    f"{array_name} shape {array.shape}; "
                    f"expected {expected_label_shape}."
                )


        if tto_saved.shape != (
            n_expected,
        ):

            raise RuntimeError(
                f"{split_name} W{W}: TTO shape mismatch."
            )


        if len(
            index_df
        ) != n_expected:

            raise RuntimeError(
                f"{split_name} W{W}: index row count mismatch."
            )


        if int(
            info.get(
                "sample_count"
            )
        ) != n_expected:

            raise RuntimeError(
                f"{split_name} W{W}: dataset_info sample count mismatch."
            )


        # ==========================================================================================
        # Index reconstruction
        # ==========================================================================================

        required_index_columns = [

            "Sample_ID",

            "Split",

            "Episode_ID",

            "Minute_Index",

            "Origin_Row_In_Release_Split",

            "Sequence_Start_Minute_Index",

            "Sequence_End_Minute_Index",

            "History_Window_Minutes",

            "Minutes_To_Next_Onset",

            "Episode_Attack_Family",
        ]


        for h in HORIZONS:

            required_index_columns += [

                f"Eligible_H{h}",

                f"Y_H{h}",

                f"Onset_Family_H{h}",

                f"Family_Code_H{h}",
            ]


        missing_index_columns = [

            c

            for c in required_index_columns

            if c not in index_df.columns
        ]


        if missing_index_columns:

            raise RuntimeError(
                f"{split_name} W{W}: missing index fields:\n"
                f"{missing_index_columns}"
            )


        if not np.array_equal(

            pd.to_numeric(
                index_df[
                    "Sample_ID"
                ],
                errors="raise"
            ).to_numpy(
                dtype=np.int64
            ),

            np.arange(
                n_expected,
                dtype=np.int64
            )

        ):

            raise RuntimeError(
                f"{split_name} W{W}: Sample_ID ordering mismatch."
            )


        if set(

            index_df[
                "Split"
            ].astype(
                str
            )

        ) != {
            split_name
        }:

            raise RuntimeError(
                f"{split_name} W{W}: wrong Split value in index."
            )


        observed_episode = (

            index_df[
                "Episode_ID"
            ]

            .astype(
                str
            )

            .to_numpy()
        )


        expected_episode = episode[
            expected_endpoint
        ]


        if not np.array_equal(
            observed_episode,
            expected_episode
        ):

            raise RuntimeError(
                f"{split_name} W{W}: Episode_ID sequence-index mismatch."
            )


        observed_minute = pd.to_numeric(

            index_df[
                "Minute_Index"
            ],

            errors="raise"

        ).to_numpy(
            dtype=np.int64
        )


        expected_minute = minute[
            expected_endpoint
        ]


        if not np.array_equal(
            observed_minute,
            expected_minute
        ):

            raise RuntimeError(
                f"{split_name} W{W}: origin Minute_Index mismatch."
            )


        observed_original_rows = pd.to_numeric(

            index_df[
                "Origin_Row_In_Release_Split"
            ],

            errors="raise"

        ).to_numpy(
            dtype=np.int64
        )


        expected_original_rows = original_row[
            expected_endpoint
        ]


        if not np.array_equal(
            observed_original_rows,
            expected_original_rows
        ):

            raise RuntimeError(
                f"{split_name} W{W}: source-row mapping mismatch."
            )


        expected_start_minute = minute[
            expected_start
        ]


        observed_start_minute = pd.to_numeric(

            index_df[
                "Sequence_Start_Minute_Index"
            ],

            errors="raise"

        ).to_numpy(
            dtype=np.int64
        )


        observed_end_minute = pd.to_numeric(

            index_df[
                "Sequence_End_Minute_Index"
            ],

            errors="raise"

        ).to_numpy(
            dtype=np.int64
        )


        if not np.array_equal(
            observed_start_minute,
            expected_start_minute
        ):

            raise RuntimeError(
                f"{split_name} W{W}: sequence-start mismatch."
            )


        if not np.array_equal(
            observed_end_minute,
            expected_minute
        ):

            raise RuntimeError(
                f"{split_name} W{W}: sequence-end mismatch."
            )


        if not np.all(

            (
                observed_end_minute
                -
                observed_start_minute
            )

            ==
            (
                W
                -
                1
            )

        ):

            raise RuntimeError(
                f"{split_name} W{W}: "
                "history length does not equal W consecutive minutes."
            )


        if not np.all(

            pd.to_numeric(

                index_df[
                    "History_Window_Minutes"
                ],

                errors="raise"

            ).to_numpy(
                dtype=np.int64
            )

            ==
            W

        ):

            raise RuntimeError(
                f"{split_name} W{W}: History_Window_Minutes mismatch."
            )


        # ==========================================================================================
        # Duplicate endpoint check
        # ==========================================================================================

        if index_df.duplicated(
            subset=[
                "Episode_ID",
                "Minute_Index"
            ]
        ).any():

            raise RuntimeError(
                f"{split_name} W{W}: duplicate sequence endpoint."
            )


        # ==========================================================================================
        # No-cross / no-future reconstruction
        # ==========================================================================================

        if not np.array_equal(

            episode[
                expected_start
            ],

            episode[
                expected_endpoint
            ]

        ):

            raise RuntimeError(
                f"{split_name} W{W}: reconstructed history crosses Episode_ID."
            )


        if not np.array_equal(

            minute[
                expected_endpoint
            ]

            -
            minute[
                expected_start
            ],

            np.full(
                n_expected,
                W
                -
                1,
                dtype=np.int64
            )

        ):

            raise RuntimeError(
                f"{split_name} W{W}: reconstructed history is non-contiguous."
            )


        # ==========================================================================================
        # Independently verify EVERY X tensor element
        #
        # Chunked so Colab RAM does not explode.
        # ==========================================================================================

        offsets = np.arange(

            W
            -
            1,

            -1,

            -1,

            dtype=np.int64
        )


        chunk_size = 2048


        X_exact = True

        max_abs_difference = 0.0

        finite_ok = True

        nonnegative_ok = True

        no_cross_ok = True

        no_future_ok = True


        for lo in range(
            0,
            n_expected,
            chunk_size
        ):

            hi = min(
                lo
                +
                chunk_size,
                n_expected
            )


            endpoints = expected_endpoint[
                lo:hi
            ]


            row_matrix = (

                endpoints[
                    :,
                    None
                ]

                -

                offsets[
                    None,
                    :
                ]
            )


            # Last timestep must be exactly the forecast origin.

            if not np.array_equal(

                row_matrix[
                    :,
                    -1
                ],

                endpoints

            ):

                no_future_ok = False


            # No row after t is permitted.

            if np.any(

                row_matrix

                >

                endpoints[
                    :,
                    None
                ]

            ):

                no_future_ok = False


            # First and last row must have the same Episode_ID.

            if not np.all(

                episode[
                    row_matrix[
                        :,
                        0
                    ]
                ]

                ==

                episode[
                    row_matrix[
                        :,
                        -1
                    ]
                ]

            ):

                no_cross_ok = False


            expected_X_chunk = X_source[
                row_matrix
            ]


            saved_X_chunk = np.asarray(

                X_saved[
                    lo:hi
                ]

            )


            if not np.isfinite(
                saved_X_chunk
            ).all():

                finite_ok = False


            if (
                saved_X_chunk
                <
                0
            ).any():

                nonnegative_ok = False


            if not np.array_equal(

                saved_X_chunk,

                expected_X_chunk

            ):

                X_exact = False


                difference = np.abs(

                    saved_X_chunk.astype(
                        np.float64
                    )

                    -

                    expected_X_chunk.astype(
                        np.float64
                    )
                )


                if difference.size:

                    max_abs_difference = max(

                        max_abs_difference,

                        float(
                            difference.max()
                        )
                    )


        if not no_cross_ok:

            raise RuntimeError(
                f"{split_name} W{W}: X crosses episode boundary."
            )


        if not no_future_ok:

            raise RuntimeError(
                f"{split_name} W{W}: future timestep leaked into X."
            )


        if not X_exact:

            raise RuntimeError(
                f"{split_name} W{W}: X does not reconstruct exactly. "
                f"Max absolute difference={max_abs_difference}"
            )


        if not finite_ok:

            raise RuntimeError(
                f"{split_name} W{W}: X contains NaN/inf."
            )


        if not nonnegative_ok:

            raise RuntimeError(
                f"{split_name} W{W}: X contains negative values."
            )


        # ==========================================================================================
        # Eligibility arrays
        # ==========================================================================================

        expected_eligible = eligible_all[
            expected_endpoint
        ]


        if not np.array_equal(

            eligible_saved,

            expected_eligible

        ):

            raise RuntimeError(
                f"{split_name} W{W}: eligible.npy mismatch."
            )


        # ==========================================================================================
        # Binary labels
        # ==========================================================================================

        expected_y_selected = y_expected[
            expected_endpoint
        ]


        if not np.array_equal(

            y_saved,

            expected_y_selected

        ):

            raise RuntimeError(
                f"{split_name} W{W}: y_binary.npy mismatch."
            )


        # ==========================================================================================
        # Family labels
        # ==========================================================================================

        expected_family_selected = family_expected[
            expected_endpoint
        ]


        if not np.array_equal(

            family_saved,

            expected_family_selected

        ):

            raise RuntimeError(
                f"{split_name} W{W}: y_family.npy mismatch."
            )


        # ==========================================================================================
        # TTO
        # ==========================================================================================

        expected_tto = source_tto[
            expected_endpoint
        ]


        if not exact_float_equal(

            tto_saved,

            expected_tto

        ):

            raise RuntimeError(
                f"{split_name} W{W}: Minutes_To_Next_Onset mismatch."
            )


        index_tto = pd.to_numeric(

            index_df[
                "Minutes_To_Next_Onset"
            ],

            errors="coerce"

        ).to_numpy(
            dtype=np.float32
        )


        if not exact_float_equal(

            index_tto,

            expected_tto

        ):

            raise RuntimeError(
                f"{split_name} W{W}: index TTO mismatch."
            )


        # ==========================================================================================
        # Episode family in evaluation index
        # ==========================================================================================

        expected_episode_family = normalize_text(

            targ_s.loc[
                expected_endpoint,
                "Episode_Attack_Family"
            ]
        ).to_numpy()


        observed_episode_family = normalize_text(

            index_df[
                "Episode_Attack_Family"
            ]
        ).to_numpy()


        if not np.array_equal(

            observed_episode_family,

            expected_episode_family

        ):

            raise RuntimeError(
                f"{split_name} W{W}: Episode_Attack_Family index mismatch."
            )


        # ==========================================================================================
        # Evaluation metadata consistency
        # ==========================================================================================

        text_metadata_fields = [

            "Calibration_Source",

            "Episode_Type",

            "Precursor_Regime",

            "Synthetic_Phase",

            "Temporal_Segment_ID",
        ]


        for column in text_metadata_fields:

            if (
                column in meta_s.columns

                and

                column in index_df.columns
            ):

                expected_values = normalize_text(

                    meta_s.loc[
                        expected_endpoint,
                        column
                    ]
                ).to_numpy()


                observed_values = normalize_text(

                    index_df[
                        column
                    ]
                ).to_numpy()


                if not np.array_equal(

                    observed_values,

                    expected_values

                ):

                    raise RuntimeError(
                        f"{split_name} W{W}: "
                        f"evaluation metadata mismatch: {column}"
                    )


        numeric_metadata_fields = [

            "Precursor_Length_Minutes",

            "Precursor_Max_Alpha",
        ]


        for column in numeric_metadata_fields:

            if (
                column in meta_s.columns

                and

                column in index_df.columns
            ):

                expected_values = pd.to_numeric(

                    meta_s.loc[
                        expected_endpoint,
                        column
                    ],

                    errors="coerce"

                ).to_numpy(
                    dtype=float
                )


                observed_values = pd.to_numeric(

                    index_df[
                        column
                    ],

                    errors="coerce"

                ).to_numpy(
                    dtype=float
                )


                if not np.allclose(

                    observed_values,

                    expected_values,

                    atol=0.0,

                    rtol=0.0,

                    equal_nan=True

                ):

                    raise RuntimeError(
                        f"{split_name} W{W}: "
                        f"numeric metadata mismatch: {column}"
                    )


        if (
            "Timestamp_UTC"
            in meta_s.columns

            and

            "Timestamp_UTC"
            in index_df.columns
        ):

            expected_timestamp = pd.to_datetime(

                meta_s.loc[
                    expected_endpoint,
                    "Timestamp_UTC"
                ],

                errors="coerce",

                utc=True
            )


            observed_timestamp = pd.to_datetime(

                index_df[
                    "Timestamp_UTC"
                ],

                errors="coerce",

                utc=True
            )


            if (
                expected_timestamp.isna().any()

                or

                observed_timestamp.isna().any()
            ):

                raise RuntimeError(
                    f"{split_name} W{W}: invalid timestamp in sequence index."
                )


            if not np.array_equal(

                expected_timestamp.astype(
                    "int64"
                ).to_numpy(),

                observed_timestamp.astype(
                    "int64"
                ).to_numpy()

            ):

                raise RuntimeError(
                    f"{split_name} W{W}: authoritative timestamp mismatch."
                )


        # ==========================================================================================
        # Horizon-by-horizon labels and support
        # ==========================================================================================

        for j, h in enumerate(
            HORIZONS
        ):

            expected_eligible_h = expected_eligible[
                :,
                j
            ]


            expected_y_h = expected_y_selected[
                :,
                j
            ]


            expected_family_h = expected_family_selected[
                :,
                j
            ]


            # --------------------------------------------------------------------------------------
            # Index copies must equal .npy arrays
            # --------------------------------------------------------------------------------------

            observed_index_eligible = pd.to_numeric(

                index_df[
                    f"Eligible_H{h}"
                ],

                errors="raise"

            ).to_numpy(
                dtype=np.uint8
            )


            observed_index_y = pd.to_numeric(

                index_df[
                    f"Y_H{h}"
                ],

                errors="raise"

            ).to_numpy(
                dtype=np.int8
            )


            observed_index_family_code = pd.to_numeric(

                index_df[
                    f"Family_Code_H{h}"
                ],

                errors="raise"

            ).to_numpy(
                dtype=np.int8
            )


            if not np.array_equal(

                observed_index_eligible,

                expected_eligible_h

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: index eligibility mismatch."
                )


            if not np.array_equal(

                observed_index_y,

                expected_y_h

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: index binary target mismatch."
                )


            if not np.array_equal(

                observed_index_family_code,

                expected_family_h

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: index family-code mismatch."
                )


            # --------------------------------------------------------------------------------------
            # Text family target
            # --------------------------------------------------------------------------------------

            expected_family_text = normalize_text(

                targ_s.loc[
                    expected_endpoint,
                    f"Onset_Family_H{h}"
                ]
            ).to_numpy()


            observed_family_text = normalize_text(

                index_df[
                    f"Onset_Family_H{h}"
                ]
            ).to_numpy()


            if not np.array_equal(

                observed_family_text,

                expected_family_text

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: text family target mismatch."
                )


            # --------------------------------------------------------------------------------------
            # Eligibility / target encoding relationship
            # --------------------------------------------------------------------------------------

            if np.any(

                (
                    expected_eligible_h
                    ==
                    0
                )

                &

                (
                    expected_y_h
                    !=
                    -1
                )

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    "ineligible sample does not have y=-1."
                )


            if np.any(

                (
                    expected_eligible_h
                    ==
                    1
                )

                &

                ~np.isin(
                    expected_y_h,
                    [
                        0,
                        1
                    ]
                )

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    "eligible sample target is not 0/1."
                )


            # --------------------------------------------------------------------------------------
            # Positive family code relationship
            # --------------------------------------------------------------------------------------

            positive = (
                expected_y_h
                ==
                1
            )


            nonpositive = (
                expected_y_h
                !=
                1
            )


            if np.any(

                positive

                &

                (
                    expected_family_h
                    <
                    0
                )

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    "positive sample lacks family code."
                )


            if np.any(

                nonpositive

                &

                (
                    expected_family_h
                    !=
                    -1
                )

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    "non-positive sample unexpectedly has family code."
                )


            # --------------------------------------------------------------------------------------
            # Positive support
            # --------------------------------------------------------------------------------------

            eligible_mask = (
                expected_eligible_h
                ==
                1
            )


            negative = (
                expected_y_h
                ==
                0
            )


            positive_count = int(
                positive.sum()
            )


            negative_count = int(
                negative.sum()
            )


            eligible_count = int(
                eligible_mask.sum()
            )


            expected_positive_count = EXPECTED_POSITIVES[
                (
                    split_name,
                    h
                )
            ]


            if (
                positive_count
                !=
                expected_positive_count
            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    f"{positive_count} positives; "
                    f"expected {expected_positive_count}."
                )


            positive_events = int(

                pd.Series(

                    expected_episode[
                        positive
                    ]

                ).nunique()
            )


            if (
                positive_events
                !=
                EXPECTED_EVENTS[
                    split_name
                ]
            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: "
                    f"{positive_events} independent positive events; "
                    f"expected {EXPECTED_EVENTS[split_name]}."
                )


            # --------------------------------------------------------------------------------------
            # Positive TTO must lie inside (0,H]
            # --------------------------------------------------------------------------------------

            positive_tto = expected_tto[
                positive
            ]


            if np.isnan(
                positive_tto
            ).any():

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: positive TTO contains NaN."
                )


            if not np.all(

                (
                    positive_tto
                    >
                    0
                )

                &

                (
                    positive_tto
                    <=
                    h
                )

            ):

                raise RuntimeError(
                    f"{split_name} W{W} H{h}: positive TTO outside (0,H]."
                )


            horizon_rows.append(
                {
                    "Split":
                        split_name,

                    "History_Window_Minutes":
                        W,

                    "Horizon_Minutes":
                        h,

                    "Sequence_Samples":
                        n_expected,

                    "Eligible_Origins":
                        eligible_count,

                    "Positive_Origins":
                        positive_count,

                    "Negative_Origins":
                        negative_count,

                    "Unique_Positive_Events":
                        positive_events,

                    "Positive_Rate":
                        (
                            positive_count
                            /
                            eligible_count
                        ),
                }
            )


            # --------------------------------------------------------------------------------------
            # FAMILY SUPPORT
            # --------------------------------------------------------------------------------------

            for family in FAMILIES:

                family_code = FAMILY_TO_CODE[
                    family
                ]


                family_positive = (

                    positive

                    &

                    (
                        expected_family_h
                        ==
                        family_code
                    )
                )


                family_positive_count = int(
                    family_positive.sum()
                )


                family_event_count = int(

                    pd.Series(

                        expected_episode[
                            family_positive
                        ]

                    ).nunique()
                )


                expected_family_events = EXPECTED_FAMILY_EVENTS[
                    split_name
                ]


                # One onset event contributes exactly H positive origins.

                expected_family_positive_origins = (

                    expected_family_events

                    *
                    h
                )


                if (
                    family_event_count
                    !=
                    expected_family_events
                ):

                    raise RuntimeError(
                        f"{split_name} W{W} H{h} {family}: "
                        f"{family_event_count} events; "
                        f"expected {expected_family_events}."
                    )


                if (
                    family_positive_count
                    !=
                    expected_family_positive_origins
                ):

                    raise RuntimeError(
                        f"{split_name} W{W} H{h} {family}: "
                        f"{family_positive_count} positive origins; "
                        f"expected {expected_family_positive_origins}."
                    )


                family_rows.append(
                    {
                        "Split":
                            split_name,

                        "History_Window_Minutes":
                            W,

                        "Horizon_Minutes":
                            h,

                        "Family":
                            family,

                        "Positive_Origins":
                            family_positive_count,

                        "Unique_Positive_Events":
                            family_event_count,
                    }
                )


        # ==========================================================================================
        # Record episode set for cross-split isolation
        # ==========================================================================================

        split_episode_sets[
            W
        ][
            split_name
        ] = set(

            index_df[
                "Episode_ID"
            ].astype(
                str
            )
        )


        # ==========================================================================================
        # Per-W reconstruction audit
        # ==========================================================================================

        reconstruction_rows.append(
            {
                "Split":
                    split_name,

                "History_Window_Minutes":
                    W,

                "Sequence_Samples":
                    n_expected,

                "X_Shape_Exact":
                    True,

                "X_Dtype_Float32":
                    True,

                "X_Exact_Reconstruction":
                    X_exact,

                "X_Max_Absolute_Difference":
                    max_abs_difference,

                "X_All_Finite":
                    finite_ok,

                "X_All_Nonnegative":
                    nonnegative_ok,

                "No_Cross_Episode":
                    no_cross_ok,

                "No_Future_Row":
                    no_future_ok,

                "Index_Exact":
                    True,

                "Eligibility_Exact":
                    True,

                "Binary_Targets_Exact":
                    True,

                "Family_Targets_Exact":
                    True,

                "TTO_Exact":
                    True,

                "Overall_PASS":
                    True,
            }
        )


        print(
            f"✓ {split_name:<10} "
            f"W{W:<2} "
            f"samples={n_expected:,} "
            f"X exact / labels exact / no-cross / no-future"
        )


        del X_saved

        del eligible_saved

        del y_saved

        del family_saved

        del tto_saved

        del index_df

        gc.collect()


    del predictors

    del targets

    del metadata

    del pred_s

    del targ_s

    del meta_s

    del X_source

    gc.collect()


# ==================================================================================================
# 12.1I — RUN INDEPENDENT AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1I — RECONSTRUCT EVERY SEQUENCE FROM IMMUTABLE RELEASE")
print("=" * 120)


for split_name, split_folder in SPLITS.items():

    audit_split(
        split_name,
        split_folder
    )


# ==================================================================================================
# 12.1J — CROSS-SPLIT EPISODE ISOLATION
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1J — CROSS-SPLIT EPISODE ISOLATION")
print("=" * 120)


split_isolation_rows = []


pairs = [

    (
        "TRAIN",
        "VALIDATION"
    ),

    (
        "TRAIN",
        "TEST"
    ),

    (
        "VALIDATION",
        "TEST"
    ),
]


for W in WINDOWS:

    for left_name, right_name in pairs:

        left_ids = split_episode_sets[
            W
        ][
            left_name
        ]


        right_ids = split_episode_sets[
            W
        ][
            right_name
        ]


        overlap = sorted(
            left_ids.intersection(
                right_ids
            )
        )


        overlap_count = len(
            overlap
        )


        split_isolation_rows.append(
            {
                "History_Window_Minutes":
                    W,

                "Split_A":
                    left_name,

                "Split_B":
                    right_name,

                "Episodes_A":
                    len(
                        left_ids
                    ),

                "Episodes_B":
                    len(
                        right_ids
                    ),

                "Overlap_Episodes":
                    overlap_count,

                "PASS":
                    overlap_count
                    ==
                    0,
            }
        )


        if overlap_count:

            raise RuntimeError(
                f"W{W}: {left_name}/{right_name} share "
                f"{overlap_count} Episode_ID values."
            )


        print(
            f"✓ W{W:<2} "
            f"{left_name:<10} vs {right_name:<10} "
            f"overlap=0"
        )


split_isolation = pd.DataFrame(
    split_isolation_rows
)


# ==================================================================================================
# 12.1K — RESULT TABLES
# ==================================================================================================

reconstruction_audit = pd.DataFrame(
    reconstruction_rows
)

horizon_audit = pd.DataFrame(
    horizon_rows
)

family_audit = pd.DataFrame(
    family_rows
)


save_csv(

    reconstruction_audit,

    STAGING
    / "12_1_RECONSTRUCTION_AUDIT.csv"
)


save_csv(

    horizon_audit,

    STAGING
    / "12_1_HORIZON_SUPPORT_AUDIT.csv"
)


save_csv(

    family_audit,

    STAGING
    / "12_1_FAMILY_SUPPORT_AUDIT.csv"
)


save_csv(

    split_isolation,

    STAGING
    / "12_1_SPLIT_ISOLATION_AUDIT.csv"
)


print("\n" + "=" * 120)
print("12.1K — RECONSTRUCTION SUMMARY")
print("=" * 120)


print(
    reconstruction_audit.to_string(
        index=False
    )
)


# ==================================================================================================
# 12.1L — GLOBAL CONTRACT CHECKS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1L — GLOBAL CONTRACT CHECKS")
print("=" * 120)


# 3 splits × 4 windows
if len(
    reconstruction_audit
) != 12:

    raise RuntimeError(
        "Expected 12 split×window reconstruction rows."
    )


# 3 splits × 4 windows × 5 horizons
if len(
    horizon_audit
) != 60:

    raise RuntimeError(
        "Expected 60 split×window×horizon audit rows."
    )


# 3 splits × 4 windows × 5 horizons × 5 families
if len(
    family_audit
) != 300:

    raise RuntimeError(
        "Expected 300 split×window×horizon×family audit rows."
    )


if not reconstruction_audit[
    "Overall_PASS"
].all():

    raise RuntimeError(
        "Not all sequence reconstructions passed."
    )


if not split_isolation[
    "PASS"
].all():

    raise RuntimeError(
        "Cross-split episode isolation failed."
    )


# --------------------------------------------------------------------------------------------------
# Compare independently reconstructed support with the original 12.0 summary.
# --------------------------------------------------------------------------------------------------

reported_12_0 = pd.read_csv(
    SEQUENCE_SUPPORT_FILE
)


compare_columns = [

    "Split",

    "History_Window_Minutes",

    "Horizon_Minutes",

    "Sequence_Samples_Total",

    "Eligible_Origins",

    "Positive_Origins",

    "Negative_Origins",

    "Unique_Positive_Events",
]


reported_cmp = reported_12_0[
    compare_columns
].copy()


audited_cmp = (

    horizon_audit

    .rename(
        columns={
            "Sequence_Samples":
                "Sequence_Samples_Total"
        }
    )[
        compare_columns
    ]

    .copy()
)


sort_cols = [

    "Split",

    "History_Window_Minutes",

    "Horizon_Minutes",
]


reported_cmp = (

    reported_cmp

    .sort_values(
        sort_cols
    )

    .reset_index(
        drop=True
    )
)


audited_cmp = (

    audited_cmp

    .sort_values(
        sort_cols
    )

    .reset_index(
        drop=True
    )
)


if not reported_cmp.equals(
    audited_cmp
):

    raise RuntimeError(
        "Independent 12.1 support reconstruction disagrees with 12.0 summary."
    )


print(
    "✓ 12/12 split×window sequence reconstructions exact"
)

print(
    "✓ 60/60 split×window×horizon support checks exact"
)

print(
    "✓ 300/300 family-support checks exact"
)

print(
    "✓ TRAIN / VALIDATION / TEST episode overlap = 0"
)

print(
    "✓ independently reconstructed support equals 12.0 reported support"
)

print(
    "✓ every X value independently reproduced from immutable 11.6 data"
)

print(
    "✓ no sequence crosses Episode_ID"
)

print(
    "✓ no sequence uses a row after forecast origin t"
)

print(
    "✓ labels / family labels / TTO exact"
)


# ==================================================================================================
# 12.1M — PRIMARY W10/H5-H10 SNAPSHOT
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1M — PRIMARY W10 / H5-H10 DATASET")
print("=" * 120)


primary = horizon_audit.loc[

    (
        horizon_audit[
            "History_Window_Minutes"
        ]
        ==
        PRIMARY_WINDOW
    )

    &

    (
        horizon_audit[
            "Horizon_Minutes"
        ].isin(
            PRIMARY_HORIZONS
        )
    )

].copy()


primary = (

    primary

    .sort_values(
        [
            "Split",
            "Horizon_Minutes"
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    primary.to_string(
        index=False
    )
)


save_csv(

    primary,

    STAGING
    / "12_1_PRIMARY_W10_H5_H10_SUMMARY.csv"
)


# ==================================================================================================
# 12.1N — HARD VALIDATION TABLE
# ==================================================================================================

validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                str(
                    details
                ),
        }
    )


add_check(
    "Phase_12_0_Status_PASS",
    ready_12_0.get(
        "phase_12_0_status"
    )
    ==
    "PASS",
    ready_12_0.get(
        "phase_12_0_status"
    )
)


add_check(
    "Phase_12_0_Build_Lock_Exact",
    build_lock_hash
    ==
    recorded_build_lock_hash,
    build_lock_hash
)


add_check(
    "Phase_12_0_Payload_Hashes_Exact",
    len(
        payload_failures
    )
    ==
    0,
    f"failures={len(payload_failures)}"
)


add_check(
    "Phase_11_6_Release_Still_Exact",
    len(
        release_failures
    )
    ==
    0,
    f"failures={len(release_failures)}"
)


add_check(
    "Feature_Count_14",
    len(
        feature_order
    )
    ==
    14,
    len(
        feature_order
    )
)


add_check(
    "No_Forbidden_Features",
    len(
        forbidden_features
    )
    ==
    0,
    forbidden_features
)


add_check(
    "Twelve_Sequence_Sets_Reconstructed",
    len(
        reconstruction_audit
    )
    ==
    12,
    len(
        reconstruction_audit
    )
)


add_check(
    "Every_X_Sequence_Exact",
    bool(
        reconstruction_audit[
            "X_Exact_Reconstruction"
        ].all()
    ),
    "all 12 split×W datasets exact"
)


add_check(
    "No_Cross_Episode",
    bool(
        reconstruction_audit[
            "No_Cross_Episode"
        ].all()
    ),
    "all sequence sets"
)


add_check(
    "No_Future_Row",
    bool(
        reconstruction_audit[
            "No_Future_Row"
        ].all()
    ),
    "all sequence sets"
)


add_check(
    "Eligibility_Exact",
    bool(
        reconstruction_audit[
            "Eligibility_Exact"
        ].all()
    ),
    "all sequence sets"
)


add_check(
    "Binary_Targets_Exact",
    bool(
        reconstruction_audit[
            "Binary_Targets_Exact"
        ].all()
    ),
    "all sequence sets"
)


add_check(
    "Family_Targets_Exact",
    bool(
        reconstruction_audit[
            "Family_Targets_Exact"
        ].all()
    ),
    "all sequence sets"
)


add_check(
    "TTO_Exact",
    bool(
        reconstruction_audit[
            "TTO_Exact"
        ].all()
    ),
    "all sequence sets"
)


add_check(
    "Cross_Split_Episode_Overlap_Zero",
    bool(
        split_isolation[
            "PASS"
        ].all()
    ),
    "all W / split pairs"
)


add_check(
    "Sixty_Horizon_Checks",
    len(
        horizon_audit
    )
    ==
    60,
    len(
        horizon_audit
    )
)


add_check(
    "ThreeHundred_Family_Checks",
    len(
        family_audit
    )
    ==
    300,
    len(
        family_audit
    )
)


add_check(
    "Primary_W10_H5_H10_Present",
    len(
        primary
    )
    ==
    6,
    len(
        primary
    )
)


add_check(
    "No_Scaling_Performed",
    build_spec.get(
        "scaling_in_phase_12_0"
    )
    is False,
    build_spec.get(
        "scaling_in_phase_12_0"
    )
)


add_check(
    "No_Resampling_Performed",
    build_spec.get(
        "resampling_in_phase_12_0"
    )
    is False,
    build_spec.get(
        "resampling_in_phase_12_0"
    )
)


add_check(
    "No_Model_Training_Performed",
    build_spec.get(
        "model_training_in_phase_12_0"
    )
    is False,
    build_spec.get(
        "model_training_in_phase_12_0"
    )
)


validation = pd.DataFrame(
    validation_rows
)


if not validation[
    "Passed"
].all():

    failed = validation.loc[
        ~validation[
            "Passed"
        ],
        [
            "Check",
            "Details"
        ]
    ]


    print(
        failed.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "PHASE 12.1 HARD VALIDATION FAILED."
    )


save_csv(

    validation,

    STAGING
    / "12_1_VALIDATION.csv"
)


print("\n" + "=" * 120)
print("12.1N — HARD VALIDATION")
print("=" * 120)


print(
    validation.to_string(
        index=False
    )
)


# ==================================================================================================
# 12.1O — DECISION
# ==================================================================================================

decision = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.1",

    "decision":
        "PASS_SEQUENCE_INTEGRITY",

    "hard_validation_pass":
        True,


    "sequence_integrity": {

        "all_sequences_independently_reconstructed":
            True,

        "x_exact":
            True,

        "no_cross_episode_sequences":
            True,

        "no_future_rows":
            True,

        "split_episode_overlap":
            0,

        "eligibility_exact":
            True,

        "binary_targets_exact":
            True,

        "family_targets_exact":
            True,

        "minutes_to_onset_exact":
            True,
    },


    "support": {

        "independent_attack_events_train":
            700,

        "independent_attack_events_validation":
            150,

        "independent_attack_events_test":
            150,

        "attack_families":
            5,

        "family_events_train_each":
            140,

        "family_events_validation_each":
            30,

        "family_events_test_each":
            30,
    },


    "primary_configuration": {

        "history_window_minutes":
            10,

        "forecast_horizons_minutes":
            [
                5,
                10
            ],
    },


    "still_not_performed": [

        "feature scaling",

        "feature transformation",

        "oversampling",

        "negative downsampling",

        "class weighting",

        "balanced mini-batches",

        "focal loss",

        "model training",
    ],


    "next_phase":
        "12.2_TRAIN_ONLY_PREPROCESSING_AND_IMBALANCE_POLICY_FREEZE",
}


write_json(

    decision,

    STAGING
    / "12_1_DECISION.json"
)


decision_sha256 = sha256_file(

    STAGING
    / "12_1_DECISION.json"
)


# ==================================================================================================
# 12.1P — HASH AUDIT OUTPUT
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1P — HASH AUDIT OUTPUT")
print("=" * 120)


audit_hash_table = build_hash_table(
    STAGING
)


save_csv(

    audit_hash_table,

    STAGING
    / "12_1_FILE_HASHES.csv"
)


audit_hash_table_sha256 = sha256_file(

    STAGING
    / "12_1_FILE_HASHES.csv"
)


print(
    "Audit artifacts hashed:",
    len(
        audit_hash_table
    )
)


# ==================================================================================================
# 12.1Q — AUDIT LOCK
# ==================================================================================================

audit_lock = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.1",

    "audit_status":
        "PASS_SEQUENCE_INTEGRITY",


    "source_phase_12_0_build_lock_sha256":
        build_lock_hash,

    "source_phase_11_6_release_lock_sha256":
        release_lock_hash,


    "decision_sha256":
        decision_sha256,

    "audit_file_hash_table_sha256":
        audit_hash_table_sha256,


    "windows_minutes":
        WINDOWS,

    "horizons_minutes":
        HORIZONS,


    "primary_window_minutes":
        PRIMARY_WINDOW,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "model_training_allowed_after_12_1":
        False,


    "next_required_phase":
        "12.2_TRAIN_ONLY_PREPROCESSING_AND_IMBALANCE_POLICY_FREEZE",
}


write_json(

    audit_lock,

    STAGING
    / "12_1_AUDIT_LOCK.json"
)


audit_lock_sha256 = sha256_file(

    STAGING
    / "12_1_AUDIT_LOCK.json"
)


(
    STAGING
    / "12_1_AUDIT_LOCK_SHA256.txt"
).write_text(
    audit_lock_sha256
    +
    "\n",
    encoding="utf-8"
)


# ==================================================================================================
# 12.1R — READINESS FOR 12.2
# ==================================================================================================

readiness = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.1",

    "phase_12_1_status":
        "PASS",

    "decision":
        "PASS_SEQUENCE_INTEGRITY",

    "ready_for_phase_12_2":
        True,


    "phase_12_1_audit_lock_sha256":
        audit_lock_sha256,


    "primary_history_window_minutes":
        PRIMARY_WINDOW,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "model_training_authorized":
        False,


    "next_phase":
        "12.2_TRAIN_ONLY_PREPROCESSING_AND_IMBALANCE_POLICY_FREEZE",
}


write_json(

    readiness,

    STAGING
    / "12_1_READY_FOR_PREPROCESSING.json"
)


# ==================================================================================================
# 12.1S — FINAL SOURCE MUTATION CHECK
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1S — VERIFY AUDIT DID NOT MUTATE SOURCE DATA")
print("=" * 120)


if sha256_file(
    BUILD_LOCK_FILE
) != build_lock_hash:

    raise RuntimeError(
        "Phase 12.0 build lock changed during Phase 12.1."
    )


if sha256_file(
    FILE_HASHES_12_0
) != hash_table_12_0_hash:

    raise RuntimeError(
        "Phase 12.0 file-hash table changed during Phase 12.1."
    )


if sha256_file(
    RELEASE_LOCK_FILE
) != release_lock_hash:

    raise RuntimeError(
        "Phase 11.6 release lock changed during Phase 12.1."
    )


# Recheck all recorded 12.0 payload hashes.

post_audit_12_0_failures = []


for row in hash_table_12_0.itertuples(
    index=False
):

    path = (
        PHASE_12_0_DIR
        / row.Relative_Path
    )


    if (
        not path.is_file()

        or

        sha256_file(
            path
        )
        !=
        row.SHA256
    ):

        post_audit_12_0_failures.append(
            row.Relative_Path
        )


if post_audit_12_0_failures:

    raise RuntimeError(
        "Phase 12.0 changed during audit."
    )


print(
    "✓ Phase 12.0 source artifacts unchanged"
)

print(
    "✓ immutable Phase 11.6 release unchanged"
)


# ==================================================================================================
# 12.1T — FINALIZE
# ==================================================================================================

print("\n" + "=" * 120)
print("12.1T — FINALIZE")
print("=" * 120)


if FINAL_OUT.exists():

    raise RuntimeError(
        "Final Phase 12.1 directory unexpectedly exists."
    )


shutil.move(
    str(
        STAGING
    ),
    str(
        FINAL_OUT
    )
)


# ==============================================================================================
# Verify final audit lock after move
# ==============================================================================================

final_lock_file = (
    FINAL_OUT
    / "12_1_AUDIT_LOCK.json"
)

final_lock_hash_file = (
    FINAL_OUT
    / "12_1_AUDIT_LOCK_SHA256.txt"
)


if (

    sha256_file(
        final_lock_file
    )

    !=

    final_lock_hash_file
    .read_text(
        encoding="utf-8"
    )
    .strip()

):

    raise RuntimeError(
        "Final Phase 12.1 audit lock failed verification."
    )


print(
    "✓ Phase 12.1 finalized:"
)

print(
    FINAL_OUT
)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("✓ PHASE 12.1 — INDEPENDENT SEQUENCE LEAKAGE / QUALITY AUDIT: PASS")
print("=" * 120)


print(
    "\nSEQUENCE RECONSTRUCTION:"
)

print(
    "    12/12 split × history-window datasets exact ✓"
)

print(
    "    every X tensor value reconstructed from immutable 11.6 release ✓"
)


print(
    "\nLEAKAGE:"
)

print(
    "    no cross-Episode_ID history ✓"
)

print(
    "    no future row in X ✓"
)

print(
    "    TRAIN / VALIDATION / TEST Episode_ID overlap = 0 ✓"
)

print(
    "    no metadata/target/future feature in X ✓"
)


print(
    "\nTARGETS:"
)

print(
    "    eligibility exact ✓"
)

print(
    "    binary onset labels exact ✓"
)

print(
    "    future-family labels exact ✓"
)

print(
    "    Minutes_To_Next_Onset exact ✓"
)


print(
    "\nEVENT SUPPORT:"
)

print(
    "    TRAIN      700 independent attack events ✓"
)

print(
    "    VALIDATION 150 independent attack events ✓"
)

print(
    "    TEST       150 independent attack events ✓"
)

print(
    "    every family = 140 / 30 / 30 events ✓"
)


print(
    "\nPRIMARY EXPERIMENT:"
)

print(
    "    W10 / H5"
)

print(
    "    W10 / H10"
)


print(
    "\nSTILL NOT DONE:"
)

print(
    "    NO scaling"
)

print(
    "    NO oversampling"
)

print(
    "    NO downsampling"
)

print(
    "    NO class weights"
)

print(
    "    NO balanced batches"
)

print(
    "    NO focal loss"
)

print(
    "    NO model training"
)


print(
    "\nNEXT:"
)

print(
    "    PHASE 12.2"
)

print(
    "    TRAIN-ONLY PREPROCESSING + IMBALANCE-POLICY FREEZE"
)


print(
    "\nMODEL TRAINING IS STILL BLOCKED."
)


print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# PHASE 12.2 — TRAIN-ONLY PREPROCESSING + IMBALANCE-POLICY FREEZE
# ==================================================================================================
#
# PURPOSE
# -------
# Freeze preprocessing and imbalance handling BEFORE any model training.
#
#
# PREPROCESSING
# -------------
#
# Frozen transform:
#
#     raw nonnegative feature
#           ↓
#        log1p(x)
#           ↓
#   StandardScaler
#
#
# IMPORTANT:
#
# The StandardScaler is fitted ONLY on unique TRAIN minute rows that actually
# participate in at least one Phase 12.0 training sequence.
#
# Repeated appearances of the same minute in overlapping windows do NOT get
# counted repeatedly when fitting the scaler.
#
#
# MODEL REPRESENTATIONS
# ---------------------
#
# RAW X.npy:
#     stays untouched in Phase 12.0
#
# TRANSFORMED:
#     X_log1p_standardized.npy
#
#
# TREE MODELS
# -----------
#
# XGBoost may later use raw X.
#
#
# LINEAR / NEURAL MODELS
# ----------------------
#
# Logistic regression / GRU / LSTM / TCN / CNN / Transformer may use the
# frozen log1p + standardized representation.
#
#
# IMBALANCE POLICY
# ----------------
#
# PRIMARY:
#
#     Natural TRAIN distribution + TRAIN-derived class weighting.
#
# Binary:
#
#     balanced negative weight
#     balanced positive weight
#     BCE pos_weight = N_negative / N_positive
#
#
# SECONDARY ABLATION:
#
#     episode-aware balanced sampling
#
# Sampling probability:
#
#     50% positive-class probability mass
#     50% negative-class probability mass
#
# Within each class:
#
#     equal probability mass per Episode_ID
#
# Within each episode:
#
#     equal probability across its eligible origins
#
#
# NOT USED / NOT PERMITTED HERE:
#
#     SMOTE
#     synthetic window interpolation
#     validation oversampling
#     test oversampling
#     validation downsampling
#     test downsampling
#     model training
#     threshold tuning
#
#
# NEXT IF PASS:
#
#     PHASE 12.3
#     FINAL PRE-MODEL EXPERIMENTAL PROTOCOL FREEZE
#
# MODEL TRAINING REMAINS BLOCKED.
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import shutil
import gc

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler


print("=" * 120)
print("PHASE 12.2 — TRAIN-ONLY PREPROCESSING + IMBALANCE-POLICY FREEZE")
print("=" * 120)


# ==================================================================================================
# 12.2A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

MODEL_READY_ROOT = (
    DATASET_ROOT
    / "model_ready"
)


PHASE_12_0_DIR = (
    MODEL_READY_ROOT
    / "12_0"
)

PHASE_12_1_DIR = (
    MODEL_READY_ROOT
    / "12_1"
)


FINAL_OUT = (
    MODEL_READY_ROOT
    / "12_2"
)

STAGING = (
    MODEL_READY_ROOT
    / "._staging_12_2"
)


# --------------------------------------------------------------------------------------------------
# 12.0
# --------------------------------------------------------------------------------------------------

BUILD_SPEC_12_0_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_SPEC.json"
)

BUILD_LOCK_12_0_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_LOCK.json"
)

BUILD_LOCK_12_0_HASH_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_LOCK_SHA256.txt"
)

HASHES_12_0_FILE = (
    PHASE_12_0_DIR
    / "12_0_FILE_HASHES.csv"
)


# --------------------------------------------------------------------------------------------------
# 12.1
# --------------------------------------------------------------------------------------------------

READY_12_1_FILE = (
    PHASE_12_1_DIR
    / "12_1_READY_FOR_PREPROCESSING.json"
)

AUDIT_LOCK_12_1_FILE = (
    PHASE_12_1_DIR
    / "12_1_AUDIT_LOCK.json"
)

AUDIT_LOCK_12_1_HASH_FILE = (
    PHASE_12_1_DIR
    / "12_1_AUDIT_LOCK_SHA256.txt"
)

HASHES_12_1_FILE = (
    PHASE_12_1_DIR
    / "12_1_FILE_HASHES.csv"
)

DECISION_12_1_FILE = (
    PHASE_12_1_DIR
    / "12_1_DECISION.json"
)


# ==================================================================================================
# 12.2B — CONTRACT
# ==================================================================================================

WINDOWS = [
    5,
    10,
    15,
    30
]


PRIMARY_WINDOW = 10


HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


PRIMARY_HORIZONS = [
    5,
    10
]


FEATURES = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Total_Packets",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum",

    "Total_Bytes",

    "Packets_Per_Second",

    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",

    "Forward_Backward_Byte_Ratio",
]


FAMILIES = [

    "DOS_DDOS",

    "RECON_SCAN",

    "EXPLOIT_APP",

    "BRUTE_AUTH",

    "FUZZING",
]


FAMILY_TO_CODE = {

    family:
        i

    for i, family in enumerate(
        FAMILIES
    )
}


SPLITS = {

    "TRAIN":
        "train",

    "VALIDATION":
        "validation",

    "TEST":
        "test",
}


DOMAIN_FAMILIES = {

    "ALL_5_SYNTHETIC": [

        "DOS_DDOS",

        "RECON_SCAN",

        "EXPLOIT_APP",

        "BRUTE_AUTH",

        "FUZZING",
    ],


    "CIC_DOMAIN_4": [

        "DOS_DDOS",

        "RECON_SCAN",

        "EXPLOIT_APP",

        "BRUTE_AUTH",
    ],


    "UNSW_DOMAIN_4": [

        "DOS_DDOS",

        "RECON_SCAN",

        "EXPLOIT_APP",

        "FUZZING",
    ],


    "SHARED_CROSS_DOMAIN_3": [

        "DOS_DDOS",

        "RECON_SCAN",

        "EXPLOIT_APP",
    ],
}


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(
    path
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def read_json(
    path
):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def save_csv(
    df,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        path,
        index=False,
        float_format="%.12g"
    )


def normalize_source(
    value
):

    text = str(
        value
    ).upper()


    if "CIC" in text:

        return "CIC"


    if "UNSW" in text:

        return "UNSW"


    return text


def build_hash_table(
    root
):

    excluded = {

        "12_2_FILE_HASHES.csv",

        "12_2_LOCK.json",

        "12_2_LOCK_SHA256.txt",

        "12_2_READY_FOR_PROTOCOL_FREEZE.json",
    }


    rows = []


    for path in sorted(
        root.rglob(
            "*"
        )
    ):

        if not path.is_file():

            continue


        if path.name in excluded:

            continue


        rows.append(
            {
                "Relative_Path":
                    str(
                        path.relative_to(
                            root
                        )
                    ),

                "Bytes":
                    int(
                        path.stat().st_size
                    ),

                "SHA256":
                    sha256_file(
                        path
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


def verify_hash_table(
    root,
    hash_table
):

    failures = []


    for row in hash_table.itertuples(
        index=False
    ):

        path = (
            root
            / row.Relative_Path
        )


        if not path.is_file():

            failures.append(
                (
                    row.Relative_Path,
                    "MISSING"
                )
            )

            continue


        digest = sha256_file(
            path
        )


        if digest != row.SHA256:

            failures.append(
                (
                    row.Relative_Path,
                    digest
                )
            )


    return failures


# ==================================================================================================
# 12.2C — VERIFY PHASE 12.1 AUTHORIZATION
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2C — VERIFY PHASE 12.1 AUTHORIZATION")
print("=" * 120)


for path in [

    READY_12_1_FILE,

    AUDIT_LOCK_12_1_FILE,

    AUDIT_LOCK_12_1_HASH_FILE,

    HASHES_12_1_FILE,

    DECISION_12_1_FILE,

    BUILD_SPEC_12_0_FILE,

    BUILD_LOCK_12_0_FILE,

    BUILD_LOCK_12_0_HASH_FILE,

    HASHES_12_0_FILE,

]:

    require_file(
        path
    )


ready_12_1 = read_json(
    READY_12_1_FILE
)


if ready_12_1.get(
    "phase_12_1_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.1 status is not PASS."
    )


if ready_12_1.get(
    "decision"
) != "PASS_SEQUENCE_INTEGRITY":

    raise RuntimeError(
        "Phase 12.1 sequence-integrity decision is not PASS."
    )


if ready_12_1.get(
    "ready_for_phase_12_2"
) is not True:

    raise RuntimeError(
        "Phase 12.1 does not authorize Phase 12.2."
    )


if ready_12_1.get(
    "model_training_authorized"
) is not False:

    raise RuntimeError(
        "Phase 12.1 unexpectedly authorized model training."
    )


audit_lock_12_1_hash = sha256_file(
    AUDIT_LOCK_12_1_FILE
)


recorded_audit_lock_hash = (

    AUDIT_LOCK_12_1_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if (
    audit_lock_12_1_hash
    !=
    recorded_audit_lock_hash
):

    raise RuntimeError(
        "Phase 12.1 audit lock hash failed."
    )


if (
    ready_12_1.get(
        "phase_12_1_audit_lock_sha256"
    )
    !=
    audit_lock_12_1_hash
):

    raise RuntimeError(
        "12.1 readiness disagrees with audit lock."
    )


decision_12_1 = read_json(
    DECISION_12_1_FILE
)


if decision_12_1.get(
    "decision"
) != "PASS_SEQUENCE_INTEGRITY":

    raise RuntimeError(
        "Phase 12.1 decision file is not PASS_SEQUENCE_INTEGRITY."
    )


print(
    "✓ Phase 12.1 PASS_SEQUENCE_INTEGRITY"
)

print(
    "✓ Phase 12.1 audit lock verified"
)

print(
    "✓ Phase 12.2 authorized"
)

print(
    "✓ model training still blocked"
)


# ==================================================================================================
# 12.2D — VERIFY 12.0 + 12.1 SOURCE PAYLOADS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2D — VERIFY SOURCE PAYLOADS")
print("=" * 120)


hashes_12_0 = pd.read_csv(
    HASHES_12_0_FILE
)

hashes_12_1 = pd.read_csv(
    HASHES_12_1_FILE
)


failures_12_0 = verify_hash_table(

    PHASE_12_0_DIR,

    hashes_12_0
)


failures_12_1 = verify_hash_table(

    PHASE_12_1_DIR,

    hashes_12_1
)


if failures_12_0:

    raise RuntimeError(
        "Phase 12.0 payload changed:\n"
        +
        str(
            failures_12_0[
                :10
            ]
        )
    )


if failures_12_1:

    raise RuntimeError(
        "Phase 12.1 payload changed:\n"
        +
        str(
            failures_12_1[
                :10
            ]
        )
    )


build_lock_12_0_hash = sha256_file(
    BUILD_LOCK_12_0_FILE
)


recorded_build_lock_12_0_hash = (

    BUILD_LOCK_12_0_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if (
    build_lock_12_0_hash
    !=
    recorded_build_lock_12_0_hash
):

    raise RuntimeError(
        "Phase 12.0 build lock changed."
    )


print(
    "✓ Phase 12.0 payload unchanged"
)

print(
    "✓ Phase 12.1 payload unchanged"
)


# ==================================================================================================
# 12.2E — LOAD SOURCE RELEASE
# ==================================================================================================

build_spec_12_0 = read_json(
    BUILD_SPEC_12_0_FILE
)


RELEASE_DIR = Path(
    build_spec_12_0[
        "source_release_path"
    ]
)


RELEASE_TRAIN_DIR = (
    RELEASE_DIR
    / "data"
    / "train"
)


TRAIN_PREDICTORS_FILE = (
    RELEASE_TRAIN_DIR
    / "predictors.csv"
)


require_file(
    TRAIN_PREDICTORS_FILE
)


# ==================================================================================================
# 12.2F — OUTPUT SAFETY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2F — OUTPUT SAFETY")
print("=" * 120)


MODEL_READY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


if STAGING.exists():

    shutil.rmtree(
        STAGING
    )


if FINAL_OUT.exists():

    ready_existing = (
        FINAL_OUT
        / "12_2_READY_FOR_PROTOCOL_FREEZE.json"
    )

    lock_existing = (
        FINAL_OUT
        / "12_2_LOCK.json"
    )

    lock_hash_existing = (
        FINAL_OUT
        / "12_2_LOCK_SHA256.txt"
    )


    if (
        ready_existing.is_file()

        and

        lock_existing.is_file()

        and

        lock_hash_existing.is_file()
    ):

        current_lock_hash = sha256_file(
            lock_existing
        )


        recorded_lock_hash = (

            lock_hash_existing

            .read_text(
                encoding="utf-8"
            )

            .strip()
        )


        ready_existing_json = read_json(
            ready_existing
        )


        if (
            current_lock_hash
            ==
            recorded_lock_hash

            and

            ready_existing_json.get(
                "ready_for_phase_12_3"
            )
            is True
        ):

            print(
                "✓ Phase 12.2 already completed and locked."
            )

            print(
                "\nNEXT:"
            )

            print(
                "Phase 12.3 — final pre-model experimental protocol freeze"
            )

            raise SystemExit


    attempt = 1


    while (

        MODEL_READY_ROOT
        / f"_incomplete_12_2_attempt_{attempt}"

    ).exists():

        attempt += 1


    backup = (

        MODEL_READY_ROOT
        / f"_incomplete_12_2_attempt_{attempt}"
    )


    shutil.move(
        str(
            FINAL_OUT
        ),
        str(
            backup
        )
    )


    print(
        "⚠ previous incomplete 12.2 preserved:"
    )

    print(
        backup
    )


STAGING.mkdir(
    parents=True,
    exist_ok=True
)


PREPROCESSING_DIR = (
    STAGING
    / "preprocessing"
)

TRANSFORMED_DIR = (
    STAGING
    / "transformed"
)

SAMPLING_DIR = (
    STAGING
    / "sampling"
)

POLICY_DIR = (
    STAGING
    / "policy"
)

AUDIT_DIR = (
    STAGING
    / "audit"
)


for directory in [

    PREPROCESSING_DIR,

    TRANSFORMED_DIR,

    SAMPLING_DIR,

    POLICY_DIR,

    AUDIT_DIR,

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 12.2G — RECONSTRUCT UNIQUE TRAIN MINUTES USED BY SEQUENCES
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2G — IDENTIFY UNIQUE TRAIN MINUTES USED BY SEQUENCES")
print("=" * 120)


train_predictors = pd.read_csv(
    TRAIN_PREDICTORS_FILE,
    low_memory=False
)


KEYS = [
    "Episode_ID",
    "Minute_Index"
]


if list(
    train_predictors.columns
) != [
    *KEYS,
    *FEATURES
]:

    raise RuntimeError(
        "Frozen TRAIN predictor schema changed."
    )


if train_predictors.duplicated(
    subset=KEYS
).any():

    raise RuntimeError(
        "Duplicate TRAIN Episode_ID/Minute_Index."
    )


# --------------------------------------------------------------------------------------------------
# Reproduce deterministic sorted order.
# --------------------------------------------------------------------------------------------------

train_sorted = train_predictors.copy()


train_sorted[
    "__EPISODE_SORT__"
] = train_sorted[
    "Episode_ID"
].astype(
    str
)


train_sorted = (

    train_sorted

    .sort_values(
        [
            "__EPISODE_SORT__",
            "Minute_Index"
        ],
        kind="mergesort"
    )

    .reset_index(
        drop=True
    )
)


train_sorted[
    "__SORTED_POSITION__"
] = np.arange(
    len(
        train_sorted
    ),
    dtype=np.int64
)


train_episode = (

    train_sorted[
        "Episode_ID"
    ]

    .astype(
        str
    )

    .to_numpy()
)


train_minute = pd.to_numeric(

    train_sorted[
        "Minute_Index"
    ],

    errors="raise"

).to_numpy(
    dtype=np.int64
)


# --------------------------------------------------------------------------------------------------
# Map (Episode_ID, Minute_Index) → sorted TRAIN row position.
# --------------------------------------------------------------------------------------------------

train_multi_index = pd.MultiIndex.from_arrays(
    [
        train_sorted[
            "Episode_ID"
        ].astype(
            str
        ),

        pd.to_numeric(
            train_sorted[
                "Minute_Index"
            ],
            errors="raise"
        ).astype(
            np.int64
        ),
    ],
    names=[
        "Episode_ID",
        "Minute_Index"
    ]
)


position_lookup = pd.Series(

    train_sorted[
        "__SORTED_POSITION__"
    ].to_numpy(
        dtype=np.int64
    ),

    index=train_multi_index
)


if not position_lookup.index.is_unique:

    raise RuntimeError(
        "TRAIN sequence row lookup is not unique."
    )


used_train_rows = np.zeros(
    len(
        train_sorted
    ),
    dtype=bool
)


fit_usage_rows = []


for W in WINDOWS:

    index_file = (

        PHASE_12_0_DIR
        / f"W{W:02d}"
        / "train"
        / "index.parquet"
    )


    require_file(
        index_file
    )


    index_df = pd.read_parquet(
        index_file,
        columns=[
            "Episode_ID",
            "Minute_Index"
        ]
    )


    origin_keys = pd.MultiIndex.from_arrays(
        [
            index_df[
                "Episode_ID"
            ].astype(
                str
            ),

            pd.to_numeric(
                index_df[
                    "Minute_Index"
                ],
                errors="raise"
            ).astype(
                np.int64
            ),
        ],
        names=[
            "Episode_ID",
            "Minute_Index"
        ]
    )


    endpoint_positions = position_lookup.reindex(
        origin_keys
    )


    if endpoint_positions.isna().any():

        raise RuntimeError(
            f"W{W}: sequence origin absent from frozen TRAIN predictors."
        )


    endpoint_positions = endpoint_positions.to_numpy(
        dtype=np.int64
    )


    offsets = np.arange(

        W
        -
        1,

        -1,

        -1,

        dtype=np.int64
    )


    row_matrix = (

        endpoint_positions[
            :,
            None
        ]

        -

        offsets[
            None,
            :
        ]
    )


    if np.any(
        row_matrix
        <
        0
    ):

        raise RuntimeError(
            f"W{W}: negative source row during scaler-fit reconstruction."
        )


    if not np.all(

        train_episode[
            row_matrix[
                :,
                0
            ]
        ]

        ==

        train_episode[
            row_matrix[
                :,
                -1
            ]
        ]

    ):

        raise RuntimeError(
            f"W{W}: scaler-fit reconstruction crosses Episode_ID."
        )


    if not np.all(

        train_minute[
            row_matrix[
                :,
                -1
            ]
        ]

        -

        train_minute[
            row_matrix[
                :,
                0
            ]
        ]

        ==

        W
        -
        1

    ):

        raise RuntimeError(
            f"W{W}: scaler-fit reconstruction is not contiguous."
        )


    unique_positions_this_W = np.unique(
        row_matrix.reshape(
            -1
        )
    )


    used_train_rows[
        unique_positions_this_W
    ] = True


    fit_usage_rows.append(
        {
            "History_Window_Minutes":
                W,

            "Sequence_Count":
                len(
                    index_df
                ),

            "Unique_Train_Minutes_Used":
                len(
                    unique_positions_this_W
                ),
        }
    )


    del index_df

    del row_matrix

    gc.collect()


fit_usage = pd.DataFrame(
    fit_usage_rows
)


unique_fit_count = int(
    used_train_rows.sum()
)


if unique_fit_count <= 0:

    raise RuntimeError(
        "No TRAIN rows selected for preprocessing fit."
    )


fit_rows = train_sorted.loc[
    used_train_rows,
    [
        "Episode_ID",
        "Minute_Index",
        "__SORTED_POSITION__",
        *FEATURES
    ]
].copy()


fit_keys = fit_rows[
    [
        "Episode_ID",
        "Minute_Index",
        "__SORTED_POSITION__"
    ]
].copy()


fit_keys.to_parquet(

    PREPROCESSING_DIR
    / "TRAIN_SCALER_FIT_ROWS.parquet",

    index=False
)


save_csv(

    fit_usage,

    AUDIT_DIR
    / "12_2_TRAIN_FIT_ROW_USAGE_BY_WINDOW.csv"
)


print(
    fit_usage.to_string(
        index=False
    )
)


print(
    "\nUnique TRAIN minute rows used to fit preprocessing:",
    f"{unique_fit_count:,}"
)

print(
    "✓ overlapping sequence rows counted only once"
)

print(
    "✓ VALIDATION rows used for fit: 0"
)

print(
    "✓ TEST rows used for fit: 0"
)


# ==================================================================================================
# 12.2H — FIT LOG1P + STANDARDSCALER ON TRAIN ONLY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2H — FIT TRAIN-ONLY LOG1P + STANDARDSCALER")
print("=" * 120)


X_fit_raw = fit_rows[
    FEATURES
].to_numpy(
    dtype=np.float64
)


if not np.isfinite(
    X_fit_raw
).all():

    raise RuntimeError(
        "TRAIN scaler-fit rows contain NaN/inf."
    )


if (
    X_fit_raw
    <
    0
).any():

    raise RuntimeError(
        "TRAIN scaler-fit rows contain negative values."
    )


X_fit_log = np.log1p(
    X_fit_raw
)


if not np.isfinite(
    X_fit_log
).all():

    raise RuntimeError(
        "log1p produced non-finite TRAIN values."
    )


scaler = StandardScaler(
    with_mean=True,
    with_std=True
)


scaler.fit(
    X_fit_log
)


SCALER_MEAN = scaler.mean_.astype(
    np.float64
)

SCALER_VAR = scaler.var_.astype(
    np.float64
)

SCALER_SCALE = scaler.scale_.astype(
    np.float64
)


if not np.isfinite(
    SCALER_MEAN
).all():

    raise RuntimeError(
        "Scaler mean contains non-finite values."
    )


if not np.isfinite(
    SCALER_SCALE
).all():

    raise RuntimeError(
        "Scaler scale contains non-finite values."
    )


if np.any(
    SCALER_SCALE
    <=
    0
):

    raise RuntimeError(
        "Scaler contains non-positive scale."
    )


X_fit_transformed = (

    X_fit_log

    -

    SCALER_MEAN[
        None,
        :
    ]

) / SCALER_SCALE[
    None,
    :
]


fit_mean_after = X_fit_transformed.mean(
    axis=0
)

fit_std_after = X_fit_transformed.std(
    axis=0,
    ddof=0
)


zero_variance = (

    SCALER_VAR
    ==
    0
)


for j in range(
    len(
        FEATURES
    )
):

    if zero_variance[
        j
    ]:

        if not np.isclose(
            fit_std_after[
                j
            ],
            0.0,
            atol=1e-12
        ):

            raise RuntimeError(
                f"Zero-variance feature audit failed: {FEATURES[j]}"
            )


    else:

        if not np.isclose(
            fit_mean_after[
                j
            ],
            0.0,
            atol=1e-10
        ):

            raise RuntimeError(
                f"Scaler mean audit failed: {FEATURES[j]}"
            )


        if not np.isclose(
            fit_std_after[
                j
            ],
            1.0,
            atol=1e-10
        ):

            raise RuntimeError(
                f"Scaler std audit failed: {FEATURES[j]}"
            )


# --------------------------------------------------------------------------------------------------
# Persist exact parameters.
# --------------------------------------------------------------------------------------------------

np.savez(

    PREPROCESSING_DIR
    / "SCALER_PARAMETERS.npz",

    feature_names=np.array(
        FEATURES,
        dtype="U64"
    ),

    mean=SCALER_MEAN,

    var=SCALER_VAR,

    scale=SCALER_SCALE,

    n_samples_seen=np.array(
        [
            unique_fit_count
        ],
        dtype=np.int64
    )
)


scaler_json = {

    "transform":
        "log1p_then_standardize",

    "fit_split":
        "TRAIN_ONLY",

    "fit_unit":
        "unique TRAIN minute rows participating in >=1 model-ready sequence",

    "fit_rows":
        unique_fit_count,

    "feature_order":
        FEATURES,

    "mean_log1p":
        {
            feature:
                float(
                    SCALER_MEAN[
                        i
                    ]
                )

            for i, feature in enumerate(
                FEATURES
            )
        },

    "variance_log1p":
        {
            feature:
                float(
                    SCALER_VAR[
                        i
                    ]
                )

            for i, feature in enumerate(
                FEATURES
            )
        },

    "scale_log1p":
        {
            feature:
                float(
                    SCALER_SCALE[
                        i
                    ]
                )

            for i, feature in enumerate(
                FEATURES
            )
        },

    "zero_variance_features":
        [
            FEATURES[
                i
            ]

            for i in np.flatnonzero(
                zero_variance
            )
        ],

    "validation_used_for_fit":
        False,

    "test_used_for_fit":
        False,
}


write_json(

    scaler_json,

    PREPROCESSING_DIR
    / "SCALER_PARAMETERS.json"
)


scaler_audit = pd.DataFrame(
    {
        "Feature":
            FEATURES,

        "Fit_Raw_Min":
            X_fit_raw.min(
                axis=0
            ),

        "Fit_Raw_Max":
            X_fit_raw.max(
                axis=0
            ),

        "Fit_Log1p_Mean":
            SCALER_MEAN,

        "Fit_Log1p_Variance":
            SCALER_VAR,

        "Fit_Log1p_Scale":
            SCALER_SCALE,

        "Transformed_Fit_Mean":
            fit_mean_after,

        "Transformed_Fit_Std":
            fit_std_after,

        "Zero_Variance":
            zero_variance,
    }
)


save_csv(

    scaler_audit,

    AUDIT_DIR
    / "12_2_SCALER_AUDIT.csv"
)


print(
    scaler_audit.to_string(
        index=False
    )
)


print(
    "\n✓ scaler fitted on TRAIN only"
)

print(
    "✓ log1p parameters frozen"
)

print(
    "✓ StandardScaler parameters frozen"
)


# Free temporary full transformed fit matrix.

del X_fit_transformed

del X_fit_log

gc.collect()


# ==================================================================================================
# 12.2I — TRANSFORM ALL SEQUENCE ARRAYS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2I — APPLY FROZEN TRAIN TRANSFORM TO ALL SPLITS")
print("=" * 120)


transform_audit_rows = []


for W in WINDOWS:

    for split_name, split_folder in SPLITS.items():

        source_dir = (

            PHASE_12_0_DIR
            / f"W{W:02d}"
            / split_name.lower()
        )


        raw_x_file = (
            source_dir
            / "X.npy"
        )


        require_file(
            raw_x_file
        )


        raw_hash_before = sha256_file(
            raw_x_file
        )


        X_raw = np.load(

            raw_x_file,

            mmap_mode="r"
        )


        if X_raw.ndim != 3:

            raise RuntimeError(
                f"{split_name} W{W}: X must be 3D."
            )


        if X_raw.shape[
            1
        ] != W:

            raise RuntimeError(
                f"{split_name} W{W}: X history dimension mismatch."
            )


        if X_raw.shape[
            2
        ] != len(
            FEATURES
        ):

            raise RuntimeError(
                f"{split_name} W{W}: X feature dimension mismatch."
            )


        out_dir = (

            TRANSFORMED_DIR
            / f"W{W:02d}"
            / split_name.lower()
        )


        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        transformed_file = (
            out_dir
            / "X_log1p_standardized.npy"
        )


        X_out = np.lib.format.open_memmap(

            transformed_file,

            mode="w+",

            dtype=np.float32,

            shape=X_raw.shape
        )


        chunk_size = 2048


        all_finite = True

        global_min = np.full(
            len(
                FEATURES
            ),
            np.inf,
            dtype=np.float64
        )

        global_max = np.full(
            len(
                FEATURES
            ),
            -np.inf,
            dtype=np.float64
        )


        for lo in range(
            0,
            X_raw.shape[
                0
            ],
            chunk_size
        ):

            hi = min(
                lo
                +
                chunk_size,
                X_raw.shape[
                    0
                ]
            )


            chunk_raw = np.asarray(

                X_raw[
                    lo:hi
                ],

                dtype=np.float64
            )


            if not np.isfinite(
                chunk_raw
            ).all():

                raise RuntimeError(
                    f"{split_name} W{W}: raw X contains NaN/inf."
                )


            if (
                chunk_raw
                <
                0
            ).any():

                raise RuntimeError(
                    f"{split_name} W{W}: raw X contains negative values."
                )


            transformed = (

                np.log1p(
                    chunk_raw
                )

                -

                SCALER_MEAN[
                    None,
                    None,
                    :
                ]

            ) / SCALER_SCALE[
                None,
                None,
                :
            ]


            if not np.isfinite(
                transformed
            ).all():

                all_finite = False


            flattened = transformed.reshape(
                -1,
                len(
                    FEATURES
                )
            )


            global_min = np.minimum(

                global_min,

                flattened.min(
                    axis=0
                )
            )


            global_max = np.maximum(

                global_max,

                flattened.max(
                    axis=0
                )
            )


            X_out[
                lo:hi
            ] = transformed.astype(
                np.float32
            )


        X_out.flush()


        del X_out


        if not all_finite:

            raise RuntimeError(
                f"{split_name} W{W}: transformed X contains NaN/inf."
            )


        X_check = np.load(

            transformed_file,

            mmap_mode="r"
        )


        if X_check.shape != X_raw.shape:

            raise RuntimeError(
                f"{split_name} W{W}: transformed shape changed."
            )


        if X_check.dtype != np.float32:

            raise RuntimeError(
                f"{split_name} W{W}: transformed dtype is not float32."
            )


        # ------------------------------------------------------------------------------------------
        # Independent formula spot check:
        #
        # first / middle / last samples.
        # ------------------------------------------------------------------------------------------

        check_indices = sorted(
            set(
                [
                    0,

                    X_raw.shape[
                        0
                    ]
                    //
                    2,

                    X_raw.shape[
                        0
                    ]
                    -
                    1,
                ]
            )
        )


        for sample_idx in check_indices:

            expected = (

                np.log1p(

                    np.asarray(
                        X_raw[
                            sample_idx
                        ],
                        dtype=np.float64
                    )
                )

                -

                SCALER_MEAN[
                    None,
                    :
                ]

            ) / SCALER_SCALE[
                None,
                :
            ]


            expected = expected.astype(
                np.float32
            )


            observed = np.asarray(

                X_check[
                    sample_idx
                ]
            )


            if not np.array_equal(
                observed,
                expected
            ):

                raise RuntimeError(
                    f"{split_name} W{W}: transformed formula check failed."
                )


        raw_hash_after = sha256_file(
            raw_x_file
        )


        if raw_hash_after != raw_hash_before:

            raise RuntimeError(
                f"{split_name} W{W}: raw Phase 12.0 X was modified."
            )


        write_json(
            {
                "split":
                    split_name,

                "history_window_minutes":
                    W,

                "shape":
                    list(
                        X_check.shape
                    ),

                "dtype":
                    "float32",

                "transform":
                    "log1p_then_train_only_standardize",

                "scaler_fit_split":
                    "TRAIN_ONLY",

                "source_raw_X":
                    str(
                        raw_x_file
                    ),

                "labels_source":
                    str(
                        source_dir
                    ),

                "source_raw_X_sha256":
                    raw_hash_before,

                "feature_order":
                    FEATURES,

                "resampled":
                    False,

                "labels_modified":
                    False,
            },

            out_dir
            / "dataset_info.json"
        )


        transform_audit_rows.append(
            {
                "Split":
                    split_name,

                "History_Window_Minutes":
                    W,

                "Samples":
                    int(
                        X_check.shape[
                            0
                        ]
                    ),

                "Window":
                    int(
                        X_check.shape[
                            1
                        ]
                    ),

                "Features":
                    int(
                        X_check.shape[
                            2
                        ]
                    ),

                "Output_Float32":
                    X_check.dtype
                    ==
                    np.float32,

                "All_Finite":
                    all_finite,

                "Raw_Source_Unchanged":
                    raw_hash_before
                    ==
                    raw_hash_after,

                "PASS":
                    True,
            }
        )


        print(
            f"✓ {split_name:<10} "
            f"W{W:<2} "
            f"{X_check.shape} "
            f"log1p + TRAIN scaler"
        )


        del X_raw

        del X_check

        gc.collect()


transform_audit = pd.DataFrame(
    transform_audit_rows
)


save_csv(

    transform_audit,

    AUDIT_DIR
    / "12_2_TRANSFORM_AUDIT.csv"
)


if len(
    transform_audit
) != 12:

    raise RuntimeError(
        "Expected 12 transformed split×window datasets."
    )


if not transform_audit[
    "PASS"
].all():

    raise RuntimeError(
        "At least one transformed sequence set failed."
    )


print(
    "\n✓ 12/12 sequence datasets transformed"
)

print(
    "✓ same sequence ordering preserved"
)

print(
    "✓ raw Phase 12.0 tensors unchanged"
)


# ==================================================================================================
# 12.2J — BINARY TRAIN CLASS WEIGHTS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2J — COMPUTE TRAIN-ONLY BINARY CLASS WEIGHTS")
print("=" * 120)


binary_weight_rows = []


for W in WINDOWS:

    sequence_dir = (

        PHASE_12_0_DIR
        / f"W{W:02d}"
        / "train"
    )


    eligible = np.load(

        sequence_dir
        / "eligible.npy",

        allow_pickle=False
    )


    y_binary = np.load(

        sequence_dir
        / "y_binary.npy",

        allow_pickle=False
    )


    index_df = pd.read_parquet(

        sequence_dir
        / "index.parquet"
    )


    if len(
        index_df
    ) != eligible.shape[
        0
    ]:

        raise RuntimeError(
            f"W{W}: TRAIN index/eligibility length mismatch."
        )


    if y_binary.shape != eligible.shape:

        raise RuntimeError(
            f"W{W}: TRAIN y/eligibility shape mismatch."
        )


    for j, H in enumerate(
        HORIZONS
    ):

        eligible_mask = (
            eligible[
                :,
                j
            ]
            ==
            1
        )


        y = y_binary[
            eligible_mask,
            j
        ].astype(
            np.int8
        )


        if not np.isin(
            y,
            [
                0,
                1
            ]
        ).all():

            raise RuntimeError(
                f"W{W} H{H}: eligible TRAIN label is not binary."
            )


        n_total = int(
            len(
                y
            )
        )

        n_negative = int(
            (
                y
                ==
                0
            ).sum()
        )

        n_positive = int(
            (
                y
                ==
                1
            ).sum()
        )


        if (
            n_negative
            <=
            0

            or

            n_positive
            <=
            0
        ):

            raise RuntimeError(
                f"W{W} H{H}: both classes are required."
            )


        balanced_negative = (

            n_total

            /

            (
                2.0
                *
                n_negative
            )
        )


        balanced_positive = (

            n_total

            /

            (
                2.0
                *
                n_positive
            )
        )


        bce_pos_weight = (

            n_negative

            /

            n_positive
        )


        binary_weight_rows.append(
            {
                "History_Window_Minutes":
                    W,

                "Horizon_Minutes":
                    H,

                "Eligible_Train_Origins":
                    n_total,

                "Negative_Train_Origins":
                    n_negative,

                "Positive_Train_Origins":
                    n_positive,

                "Positive_Rate":
                    n_positive
                    /
                    n_total,

                "Balanced_Class_Weight_0":
                    balanced_negative,

                "Balanced_Class_Weight_1":
                    balanced_positive,

                "BCE_Pos_Weight":
                    bce_pos_weight,

                "Derived_From":
                    "TRAIN_ONLY",
            }
        )


binary_weights = pd.DataFrame(
    binary_weight_rows
)


if len(
    binary_weights
) != 20:

    raise RuntimeError(
        "Expected 20 W×H binary weight configurations."
    )


save_csv(

    binary_weights,

    POLICY_DIR
    / "BINARY_CLASS_WEIGHTS.csv"
)


print(
    binary_weights.to_string(
        index=False
    )
)


# ==================================================================================================
# 12.2K — EPISODE-AWARE BALANCED TRAIN SAMPLING WEIGHTS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2K — PRECOMPUTE OPTIONAL EPISODE-BALANCED TRAIN SAMPLING")
print("=" * 120)


sampling_summary_rows = []


for W in WINDOWS:

    train_dir = (

        PHASE_12_0_DIR
        / f"W{W:02d}"
        / "train"
    )


    eligible = np.load(

        train_dir
        / "eligible.npy",

        allow_pickle=False
    )


    y_binary = np.load(

        train_dir
        / "y_binary.npy",

        allow_pickle=False
    )


    index_df = pd.read_parquet(

        train_dir
        / "index.parquet",

        columns=[
            "Episode_ID"
        ]
    )


    episode_ids_all = (

        index_df[
            "Episode_ID"
        ]

        .astype(
            str
        )

        .to_numpy()
    )


    for j, H in enumerate(
        HORIZONS
    ):

        eligible_mask = (
            eligible[
                :,
                j
            ]
            ==
            1
        )


        eligible_indices = np.flatnonzero(
            eligible_mask
        ).astype(
            np.int64
        )


        y = y_binary[
            eligible_mask,
            j
        ].astype(
            np.int8
        )


        episodes = episode_ids_all[
            eligible_mask
        ]


        weights = np.zeros(
            len(
                y
            ),
            dtype=np.float64
        )


        class_episode_counts = {}


        for class_value in [
            0,
            1
        ]:

            class_mask = (
                y
                ==
                class_value
            )


            class_episodes = episodes[
                class_mask
            ]


            if len(
                class_episodes
            ) == 0:

                raise RuntimeError(
                    f"W{W} H{H}: empty class during sampling-weight build."
                )


            episode_counts = (

                pd.Series(
                    class_episodes
                )

                .value_counts()
            )


            n_episodes_class = len(
                episode_counts
            )


            class_episode_counts[
                class_value
            ] = n_episodes_class


            # Each class receives 50% total probability mass.
            #
            # Within that class, every Episode_ID receives equal total mass.
            # Within the episode, its samples divide that episode mass equally.

            counts_per_sample = (

                pd.Series(
                    class_episodes
                )

                .map(
                    episode_counts
                )

                .to_numpy(
                    dtype=np.float64
                )
            )


            class_sample_weights = (

                0.5

                /

                (
                    n_episodes_class
                    *
                    counts_per_sample
                )
            )


            weights[
                class_mask
            ] = class_sample_weights


        if not np.isfinite(
            weights
        ).all():

            raise RuntimeError(
                f"W{W} H{H}: non-finite sampling weights."
            )


        if np.any(
            weights
            <=
            0
        ):

            raise RuntimeError(
                f"W{W} H{H}: sampling weights must be positive."
            )


        weight_sum = float(
            weights.sum()
        )


        if not np.isclose(
            weight_sum,
            1.0,
            atol=1e-12
        ):

            raise RuntimeError(
                f"W{W} H{H}: sampling weights sum to {weight_sum}."
            )


        positive_mass = float(
            weights[
                y
                ==
                1
            ].sum()
        )


        negative_mass = float(
            weights[
                y
                ==
                0
            ].sum()
        )


        if not np.isclose(
            positive_mass,
            0.5,
            atol=1e-12
        ):

            raise RuntimeError(
                f"W{W} H{H}: positive sampling mass != 0.5."
            )


        if not np.isclose(
            negative_mass,
            0.5,
            atol=1e-12
        ):

            raise RuntimeError(
                f"W{W} H{H}: negative sampling mass != 0.5."
            )


        out_dir = (

            SAMPLING_DIR
            / f"W{W:02d}"
            / f"H{H:02d}"
        )


        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        np.save(

            out_dir
            / "eligible_sample_indices.npy",

            eligible_indices,

            allow_pickle=False
        )


        np.save(

            out_dir
            / "episode_balanced_sampling_weights.npy",

            weights.astype(
                np.float64
            ),

            allow_pickle=False
        )


        write_json(
            {
                "history_window_minutes":
                    W,

                "horizon_minutes":
                    H,

                "split":
                    "TRAIN",

                "eligible_samples":
                    int(
                        len(
                            y
                        )
                    ),

                "negative_samples":
                    int(
                        (
                            y
                            ==
                            0
                        ).sum()
                    ),

                "positive_samples":
                    int(
                        (
                            y
                            ==
                            1
                        ).sum()
                    ),

                "negative_episode_count":
                    int(
                        class_episode_counts[
                            0
                        ]
                    ),

                "positive_episode_count":
                    int(
                        class_episode_counts[
                            1
                        ]
                    ),

                "negative_total_probability_mass":
                    negative_mass,

                "positive_total_probability_mass":
                    positive_mass,

                "policy":
                    (
                        "50/50 class mass; equal mass per Episode_ID "
                        "within class; equal mass per eligible origin "
                        "within Episode_ID"
                    ),

                "primary_training_strategy":
                    False,

                "role":
                    "SECONDARY_ABLATION_ONLY",

                "replacement_sampling":
                    True,

                "validation_sampling":
                    False,

                "test_sampling":
                    False,
            },

            out_dir
            / "sampling_info.json"
        )


        sampling_summary_rows.append(
            {
                "History_Window_Minutes":
                    W,

                "Horizon_Minutes":
                    H,

                "Eligible_Train_Origins":
                    len(
                        y
                    ),

                "Negative_Origins":
                    int(
                        (
                            y
                            ==
                            0
                        ).sum()
                    ),

                "Positive_Origins":
                    int(
                        (
                            y
                            ==
                            1
                        ).sum()
                    ),

                "Negative_Episodes":
                    int(
                        class_episode_counts[
                            0
                        ]
                    ),

                "Positive_Episodes":
                    int(
                        class_episode_counts[
                            1
                        ]
                    ),

                "Negative_Probability_Mass":
                    negative_mass,

                "Positive_Probability_Mass":
                    positive_mass,

                "Weight_Sum":
                    weight_sum,
            }
        )


sampling_summary = pd.DataFrame(
    sampling_summary_rows
)


if len(
    sampling_summary
) != 20:

    raise RuntimeError(
        "Expected 20 W×H sampling configurations."
    )


save_csv(

    sampling_summary,

    POLICY_DIR
    / "EPISODE_BALANCED_SAMPLING_SUMMARY.csv"
)


print(
    sampling_summary.to_string(
        index=False
    )
)


print(
    "\n✓ optional episode-aware sampler prepared for TRAIN only"
)

print(
    "✓ validation sampling artifacts created: 0"
)

print(
    "✓ test sampling artifacts created: 0"
)


# ==================================================================================================
# 12.2L — FAMILY CLASS-WEIGHT TABLES
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2L — COMPUTE TRAIN-ONLY FAMILY CLASS WEIGHTS")
print("=" * 120)


family_weight_rows = []


for W in WINDOWS:

    train_dir = (

        PHASE_12_0_DIR
        / f"W{W:02d}"
        / "train"
    )


    eligible = np.load(

        train_dir
        / "eligible.npy",

        allow_pickle=False
    )


    y_binary = np.load(

        train_dir
        / "y_binary.npy",

        allow_pickle=False
    )


    index_df = pd.read_parquet(

        train_dir
        / "index.parquet"
    )


    source_normalized = (

        index_df[
            "Calibration_Source"
        ]

        .map(
            normalize_source
        )

        .to_numpy()
    )


    for j, H in enumerate(
        HORIZONS
    ):

        positive_mask = (

            (
                eligible[
                    :,
                    j
                ]
                ==
                1
            )

            &

            (
                y_binary[
                    :,
                    j
                ]
                ==
                1
            )
        )


        family_text = (

            index_df[
                f"Onset_Family_H{H}"
            ]

            .astype(
                "string"
            )

            .fillna(
                ""
            )

            .astype(
                str
            )

            .to_numpy()
        )


        for domain_name, domain_families in DOMAIN_FAMILIES.items():

            if domain_name == "CIC_DOMAIN_4":

                domain_mask = (

                    positive_mask

                    &

                    (
                        source_normalized
                        ==
                        "CIC"
                    )

                    &

                    np.isin(
                        family_text,
                        domain_families
                    )
                )


            elif domain_name == "UNSW_DOMAIN_4":

                domain_mask = (

                    positive_mask

                    &

                    (
                        source_normalized
                        ==
                        "UNSW"
                    )

                    &

                    np.isin(
                        family_text,
                        domain_families
                    )
                )


            else:

                domain_mask = (

                    positive_mask

                    &

                    np.isin(
                        family_text,
                        domain_families
                    )
                )


            domain_labels = family_text[
                domain_mask
            ]


            n_total = len(
                domain_labels
            )


            if n_total <= 0:

                raise RuntimeError(
                    f"W{W} H{H} {domain_name}: zero positive family samples."
                )


            n_classes = len(
                domain_families
            )


            for family in domain_families:

                n_family = int(
                    (
                        domain_labels
                        ==
                        family
                    ).sum()
                )


                if n_family <= 0:

                    raise RuntimeError(
                        f"W{W} H{H} {domain_name}: "
                        f"family {family} has zero TRAIN samples."
                    )


                balanced_weight = (

                    n_total

                    /

                    (
                        n_classes
                        *
                        n_family
                    )
                )


                unique_events = int(

                    index_df.loc[
                        domain_mask
                        &
                        (
                            family_text
                            ==
                            family
                        ),
                        "Episode_ID"
                    ]

                    .nunique()
                )


                family_weight_rows.append(
                    {
                        "History_Window_Minutes":
                            W,

                        "Horizon_Minutes":
                            H,

                        "Domain":
                            domain_name,

                        "Family":
                            family,

                        "Positive_Train_Origins":
                            n_family,

                        "Independent_Train_Events":
                            unique_events,

                        "Balanced_Class_Weight":
                            balanced_weight,

                        "Derived_From":
                            "TRAIN_ONLY",
                    }
                )


family_weights = pd.DataFrame(
    family_weight_rows
)


# Per W/H:
#
# ALL5 = 5
# CIC4 = 4
# UNSW4 = 4
# SHARED3 = 3
#
# Total = 16
#
# 4 W × 5 H × 16 = 320 rows.

if len(
    family_weights
) != 320:

    raise RuntimeError(
        f"Expected 320 family-weight rows, found {len(family_weights)}."
    )


save_csv(

    family_weights,

    POLICY_DIR
    / "FAMILY_CLASS_WEIGHTS.csv"
)


print(
    "✓ family-weight configurations:",
    len(
        family_weights
    )
)

print(
    "✓ ALL_5_SYNTHETIC weights frozen"
)

print(
    "✓ CIC_DOMAIN_4 weights frozen"
)

print(
    "✓ UNSW_DOMAIN_4 weights frozen"
)

print(
    "✓ SHARED_CROSS_DOMAIN_3 weights frozen"
)


# ==================================================================================================
# 12.2M — FREEZE IMBALANCE POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2M — FREEZE IMBALANCE POLICY")
print("=" * 120)


imbalance_policy = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.2",


    "principle":
        (
            "Independent attack episodes are already sufficient. "
            "Do not generate additional synthetic episodes merely to "
            "balance forecasting-origin classes."
        ),


    "binary_primary_training_policy": {

        "training_distribution":
            "NATURAL_TRAIN",

        "class_weighting":
            True,

        "weight_source":
            "TRAIN_ONLY",

        "weight_table":
            "policy/BINARY_CLASS_WEIGHTS.csv",

        "neural_default_loss":
            "BCEWithLogitsLoss_with_TRAIN_pos_weight",

        "logistic_regression":
            "TRAIN-derived balanced class weights",

        "xgboost":
            "TRAIN-derived scale_pos_weight = N_negative / N_positive",

        "oversampling":
            False,

        "SMOTE":
            False,

        "synthetic_window_interpolation":
            False,
    },


    "secondary_sampling_ablation": {

        "enabled_as_optional_ablation":
            True,

        "primary_strategy":
            False,

        "split":
            "TRAIN_ONLY",

        "sampling":
            "episode-aware balanced replacement sampling",

        "positive_probability_mass":
            0.5,

        "negative_probability_mass":
            0.5,

        "episode_weighting":
            "equal probability mass per Episode_ID within each class",

        "within_episode_weighting":
            "equal probability per eligible origin",

        "combine_with_primary_class_weighting":
            False,

        "validation_sampling":
            False,

        "test_sampling":
            False,
    },


    "focal_loss": {

        "primary_strategy":
            False,

        "allowed_only_as_predeclared_ablation":
            True,

        "tuning_on_test":
            False,
    },


    "negative_downsampling": {

        "primary_strategy":
            False,

        "performed_in_phase_12_2":
            False,

        "allowed_only_if_predeclared_in_phase_12_3":
            True,

        "validation_downsampling":
            False,

        "test_downsampling":
            False,
    },


    "validation_policy": {

        "preserve_natural_prevalence":
            True,

        "oversampling":
            False,

        "downsampling":
            False,

        "class_balance_mutation":
            False,
    },


    "test_policy": {

        "preserve_natural_prevalence":
            True,

        "oversampling":
            False,

        "downsampling":
            False,

        "class_balance_mutation":
            False,

        "threshold_selection":
            False,

        "model_selection":
            False,
    },


    "family_classification": {

        "weight_source":
            "TRAIN_ONLY",

        "class_weight_table":
            "policy/FAMILY_CLASS_WEIGHTS.csv",

        "ordinary_metrics":
            [
                "Macro F1",
                "Macro Recall",
                "Macro Precision",
                "Per-class metrics",
                "Confusion matrix"
            ],
    },
}


write_json(

    imbalance_policy,

    POLICY_DIR
    / "IMBALANCE_POLICY.json"
)


print(
    "✓ natural TRAIN distribution + class weighting = PRIMARY"
)

print(
    "✓ episode-balanced sampling = SECONDARY ABLATION"
)

print(
    "✓ SMOTE = PROHIBITED"
)

print(
    "✓ validation/test resampling = PROHIBITED"
)

print(
    "✓ focal loss = optional predeclared ablation only"
)


# ==================================================================================================
# 12.2N — FREEZE PREPROCESSING POLICY
# ==================================================================================================

preprocessing_policy = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.2",


    "raw_representation": {

        "source":
            "Phase 12.0 X.npy",

        "modified":
            False,

        "recommended_for":
            [
                "tree models",
                "XGBoost",
                "raw-feature ablations"
            ],
    },


    "standardized_representation": {

        "transform_order": [

            "log1p",

            "TRAIN-fitted StandardScaler",
        ],

        "fit_split":
            "TRAIN_ONLY",

        "fit_rows":
            unique_fit_count,

        "fit_unit":
            (
                "unique TRAIN minute rows participating in at least "
                "one model-ready sequence"
            ),

        "repeated_overlapping_window_rows_counted_multiple_times":
            False,

        "validation_used_for_fit":
            False,

        "test_used_for_fit":
            False,

        "output_dtype":
            "float32",

        "recommended_for":
            [
                "logistic regression",
                "GRU",
                "LSTM",
                "TCN",
                "1D CNN",
                "Transformer"
            ],
    },


    "feature_order":
        FEATURES,


    "per_split_scalers":
        False,


    "single_global_train_scaler":
        True,


    "scaler_parameters":
        "preprocessing/SCALER_PARAMETERS.npz",
}


write_json(

    preprocessing_policy,

    POLICY_DIR
    / "PREPROCESSING_POLICY.json"
)


# ==================================================================================================
# 12.2O — PRIMARY W10/H5-H10 SNAPSHOT
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2O — PRIMARY W10 / H5-H10 IMBALANCE SNAPSHOT")
print("=" * 120)


primary_binary = binary_weights.loc[

    (
        binary_weights[
            "History_Window_Minutes"
        ]
        ==
        PRIMARY_WINDOW
    )

    &

    (
        binary_weights[
            "Horizon_Minutes"
        ].isin(
            PRIMARY_HORIZONS
        )
    )

].copy()


print(
    primary_binary.to_string(
        index=False
    )
)


save_csv(

    primary_binary,

    STAGING
    / "12_2_PRIMARY_W10_H5_H10_WEIGHTS.csv"
)


# ==================================================================================================
# 12.2P — HARD VALIDATION
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2P — HARD VALIDATION")
print("=" * 120)


validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                str(
                    details
                ),
        }
    )


add_check(
    "Phase_12_1_PASS",
    ready_12_1.get(
        "phase_12_1_status"
    )
    ==
    "PASS",
    ready_12_1.get(
        "phase_12_1_status"
    )
)


add_check(
    "Phase_12_1_Sequence_Integrity_PASS",
    ready_12_1.get(
        "decision"
    )
    ==
    "PASS_SEQUENCE_INTEGRITY",
    ready_12_1.get(
        "decision"
    )
)


add_check(
    "Phase_12_0_Source_Unchanged",
    len(
        failures_12_0
    )
    ==
    0,
    f"failures={len(failures_12_0)}"
)


add_check(
    "Phase_12_1_Source_Unchanged",
    len(
        failures_12_1
    )
    ==
    0,
    f"failures={len(failures_12_1)}"
)


add_check(
    "Scaler_Fit_On_TRAIN_Only",
    True,
    f"unique_train_rows={unique_fit_count}"
)


add_check(
    "Scaler_VALIDATION_Fit_Rows_Zero",
    True,
    0
)


add_check(
    "Scaler_TEST_Fit_Rows_Zero",
    True,
    0
)


add_check(
    "Feature_Count_14",
    len(
        FEATURES
    )
    ==
    14,
    len(
        FEATURES
    )
)


add_check(
    "Scaler_Parameters_Finite",
    bool(
        np.isfinite(
            SCALER_MEAN
        ).all()

        and

        np.isfinite(
            SCALER_SCALE
        ).all()
    ),
    "mean+scale finite"
)


add_check(
    "Twelve_Transformed_Datasets",
    len(
        transform_audit
    )
    ==
    12,
    len(
        transform_audit
    )
)


add_check(
    "All_Transformed_Datasets_PASS",
    bool(
        transform_audit[
            "PASS"
        ].all()
    ),
    "12/12"
)


add_check(
    "Raw_X_Remained_Unchanged",
    bool(
        transform_audit[
            "Raw_Source_Unchanged"
        ].all()
    ),
    "12/12"
)


add_check(
    "Binary_Weight_Configurations_20",
    len(
        binary_weights
    )
    ==
    20,
    len(
        binary_weights
    )
)


add_check(
    "Sampling_Configurations_20",
    len(
        sampling_summary
    )
    ==
    20,
    len(
        sampling_summary
    )
)


add_check(
    "Sampling_Positive_Mass_Half",
    bool(
        np.allclose(

            sampling_summary[
                "Positive_Probability_Mass"
            ].to_numpy(
                dtype=float
            ),

            0.5,

            atol=1e-12,

            rtol=0.0
        )
    ),
    "all 20 configs"
)


add_check(
    "Sampling_Negative_Mass_Half",
    bool(
        np.allclose(

            sampling_summary[
                "Negative_Probability_Mass"
            ].to_numpy(
                dtype=float
            ),

            0.5,

            atol=1e-12,

            rtol=0.0
        )
    ),
    "all 20 configs"
)


add_check(
    "Family_Weight_Rows_320",
    len(
        family_weights
    )
    ==
    320,
    len(
        family_weights
    )
)


add_check(
    "SMOTE_Disabled",
    imbalance_policy[
        "binary_primary_training_policy"
    ][
        "SMOTE"
    ]
    is False,
    False
)


add_check(
    "Validation_Resampling_Disabled",
    (
        imbalance_policy[
            "validation_policy"
        ][
            "oversampling"
        ]
        is False

        and

        imbalance_policy[
            "validation_policy"
        ][
            "downsampling"
        ]
        is False
    ),
    "disabled"
)


add_check(
    "Test_Resampling_Disabled",
    (
        imbalance_policy[
            "test_policy"
        ][
            "oversampling"
        ]
        is False

        and

        imbalance_policy[
            "test_policy"
        ][
            "downsampling"
        ]
        is False
    ),
    "disabled"
)


add_check(
    "No_Model_Training",
    True,
    "Phase 12.2 performs preprocessing/policy only"
)


validation = pd.DataFrame(
    validation_rows
)


if not validation[
    "Passed"
].all():

    failed = validation.loc[
        ~validation[
            "Passed"
        ],
        [
            "Check",
            "Details"
        ]
    ]


    print(
        failed.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "PHASE 12.2 HARD VALIDATION FAILED."
    )


save_csv(

    validation,

    AUDIT_DIR
    / "12_2_VALIDATION.csv"
)


print(
    validation.to_string(
        index=False
    )
)


# ==================================================================================================
# 12.2Q — DECISION
# ==================================================================================================

decision = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.2",

    "decision":
        "PASS_PREPROCESSING_AND_IMBALANCE_FREEZE",


    "preprocessing": {

        "transform":
            "log1p_then_standardize",

        "fit_split":
            "TRAIN_ONLY",

        "fit_unique_train_minute_rows":
            unique_fit_count,

        "validation_fit_rows":
            0,

        "test_fit_rows":
            0,

        "raw_X_preserved":
            True,

        "transformed_X_created":
            True,
    },


    "binary_imbalance": {

        "primary":
            "natural TRAIN + TRAIN-derived class weighting",

        "secondary_ablation":
            "episode-balanced TRAIN sampling",

        "SMOTE":
            False,

        "validation_resampling":
            False,

        "test_resampling":
            False,
    },


    "family_imbalance": {

        "train_only_weight_table":
            "policy/FAMILY_CLASS_WEIGHTS.csv",

        "domains": [
            "ALL_5_SYNTHETIC",
            "CIC_DOMAIN_4",
            "UNSW_DOMAIN_4",
            "SHARED_CROSS_DOMAIN_3"
        ],
    },


    "primary_experiment": {

        "history_window_minutes":
            10,

        "horizons_minutes":
            [
                5,
                10
            ],
    },


    "model_training_performed":
        False,


    "model_training_authorized":
        False,


    "next_phase":
        "12.3_FINAL_PRE_MODEL_EXPERIMENT_PROTOCOL_FREEZE",
}


write_json(

    decision,

    STAGING
    / "12_2_DECISION.json"
)


decision_sha256 = sha256_file(

    STAGING
    / "12_2_DECISION.json"
)


# ==================================================================================================
# 12.2R — SOURCE MUTATION CHECK
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2R — VERIFY 12.0 / 12.1 WERE NOT MODIFIED")
print("=" * 120)


post_failures_12_0 = verify_hash_table(

    PHASE_12_0_DIR,

    hashes_12_0
)


post_failures_12_1 = verify_hash_table(

    PHASE_12_1_DIR,

    hashes_12_1
)


if post_failures_12_0:

    raise RuntimeError(
        "Phase 12.0 changed during 12.2."
    )


if post_failures_12_1:

    raise RuntimeError(
        "Phase 12.1 changed during 12.2."
    )


print(
    "✓ Phase 12.0 unchanged"
)

print(
    "✓ Phase 12.1 unchanged"
)


# ==================================================================================================
# 12.2S — BUILD MANIFEST
# ==================================================================================================

build_manifest = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.2",

    "status":
        "PREPROCESSING_AND_IMBALANCE_POLICY_FROZEN",


    "source_phase_12_0_build_lock_sha256":
        build_lock_12_0_hash,

    "source_phase_12_1_audit_lock_sha256":
        audit_lock_12_1_hash,


    "features":
        FEATURES,

    "windows_minutes":
        WINDOWS,

    "horizons_minutes":
        HORIZONS,


    "primary_window_minutes":
        PRIMARY_WINDOW,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "preprocessing": {

        "raw_representation_retained":
            True,

        "standardized_representation_created":
            True,

        "transform":
            "log1p_then_standardize",

        "fit":
            "TRAIN_ONLY",

        "unique_fit_rows":
            unique_fit_count,

        "parameters":
            "preprocessing/SCALER_PARAMETERS.npz",
    },


    "imbalance_policy": {

        "primary":
            "natural TRAIN + TRAIN-derived weighting",

        "secondary":
            "episode-aware balanced TRAIN sampling",

        "SMOTE":
            False,

        "validation_resampling":
            False,

        "test_resampling":
            False,
    },


    "model_training_performed":
        False,


    "next_phase":
        "12.3_FINAL_PRE_MODEL_EXPERIMENT_PROTOCOL_FREEZE",
}


write_json(

    build_manifest,

    STAGING
    / "12_2_MANIFEST.json"
)


manifest_sha256 = sha256_file(

    STAGING
    / "12_2_MANIFEST.json"
)


# ==================================================================================================
# 12.2T — HASH PHASE 12.2
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2T — HASH PHASE 12.2 ARTIFACTS")
print("=" * 120)


file_hashes = build_hash_table(
    STAGING
)


save_csv(

    file_hashes,

    STAGING
    / "12_2_FILE_HASHES.csv"
)


file_hash_table_sha256 = sha256_file(

    STAGING
    / "12_2_FILE_HASHES.csv"
)


print(
    "Artifacts hashed:",
    len(
        file_hashes
    )
)


# ==================================================================================================
# 12.2U — LOCK
# ==================================================================================================

lock = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.2",

    "lock_status":
        "PASS_PREPROCESSING_AND_IMBALANCE_FREEZE",


    "source_phase_12_0_build_lock_sha256":
        build_lock_12_0_hash,

    "source_phase_12_1_audit_lock_sha256":
        audit_lock_12_1_hash,


    "decision_sha256":
        decision_sha256,

    "manifest_sha256":
        manifest_sha256,

    "file_hash_table_sha256":
        file_hash_table_sha256,


    "preprocessing_fit_split":
        "TRAIN_ONLY",

    "validation_fit_rows":
        0,

    "test_fit_rows":
        0,


    "model_training_allowed_after_12_2":
        False,


    "next_required_phase":
        "12.3_FINAL_PRE_MODEL_EXPERIMENT_PROTOCOL_FREEZE",
}


write_json(

    lock,

    STAGING
    / "12_2_LOCK.json"
)


lock_sha256 = sha256_file(

    STAGING
    / "12_2_LOCK.json"
)


(
    STAGING
    / "12_2_LOCK_SHA256.txt"
).write_text(
    lock_sha256
    +
    "\n",
    encoding="utf-8"
)


# ==================================================================================================
# 12.2V — READINESS FOR 12.3
# ==================================================================================================

readiness = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.2",

    "phase_12_2_status":
        "PASS",

    "decision":
        "PASS_PREPROCESSING_AND_IMBALANCE_FREEZE",


    "ready_for_phase_12_3":
        True,


    "phase_12_2_lock_sha256":
        lock_sha256,


    "primary_window_minutes":
        PRIMARY_WINDOW,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "model_training_authorized":
        False,


    "next_phase":
        "12.3_FINAL_PRE_MODEL_EXPERIMENT_PROTOCOL_FREEZE",
}


write_json(

    readiness,

    STAGING
    / "12_2_READY_FOR_PROTOCOL_FREEZE.json"
)


# ==================================================================================================
# 12.2W — VERIFY CURRENT OUTPUT PAYLOAD
# ==================================================================================================

recorded_hashes = pd.read_csv(

    STAGING
    / "12_2_FILE_HASHES.csv"
)


output_failures = verify_hash_table(

    STAGING,

    recorded_hashes
)


if output_failures:

    raise RuntimeError(
        "Phase 12.2 staging hash verification failed:\n"
        +
        str(
            output_failures[
                :10
            ]
        )
    )


print(
    "✓ Phase 12.2 payload hashes verified"
)


# ==================================================================================================
# 12.2X — FINALIZE
# ==================================================================================================

print("\n" + "=" * 120)
print("12.2X — FINALIZE")
print("=" * 120)


if FINAL_OUT.exists():

    raise RuntimeError(
        "Final Phase 12.2 directory unexpectedly exists."
    )


shutil.move(

    str(
        STAGING
    ),

    str(
        FINAL_OUT
    )
)


FINAL_LOCK = (
    FINAL_OUT
    / "12_2_LOCK.json"
)

FINAL_LOCK_HASH = (
    FINAL_OUT
    / "12_2_LOCK_SHA256.txt"
)


if (

    sha256_file(
        FINAL_LOCK
    )

    !=

    FINAL_LOCK_HASH
    .read_text(
        encoding="utf-8"
    )
    .strip()

):

    raise RuntimeError(
        "Final Phase 12.2 lock verification failed."
    )


print(
    "✓ Phase 12.2 finalized:"
)

print(
    FINAL_OUT
)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("✓ PHASE 12.2 — TRAIN-ONLY PREPROCESSING + IMBALANCE-POLICY FREEZE: PASS")
print("=" * 120)


print(
    "\nPREPROCESSING:"
)

print(
    "    raw Phase 12.0 X preserved ✓"
)

print(
    "    log1p transform ✓"
)

print(
    "    StandardScaler ✓"
)

print(
    "    scaler fitted on TRAIN only ✓"
)

print(
    "    unique TRAIN minutes used for fit:",
    f"{unique_fit_count:,}"
)

print(
    "    VALIDATION fit rows: 0 ✓"
)

print(
    "    TEST fit rows: 0 ✓"
)


print(
    "\nTRANSFORMED DATA:"
)

print(
    "    W5 / W10 / W15 / W30 ✓"
)

print(
    "    TRAIN / VALIDATION / TEST ✓"
)

print(
    "    12/12 transformed tensors ✓"
)

print(
    "    output dtype float32 ✓"
)


print(
    "\nBINARY IMBALANCE POLICY:"
)

print(
    "    PRIMARY:"
)

print(
    "        natural TRAIN distribution"
)

print(
    "        + TRAIN-derived class weighting"
)

print(
    "\n    SECONDARY ABLATION:"
)

print(
    "        episode-aware balanced TRAIN sampler"
)

print(
    "\n    SMOTE:"
)

print(
    "        DISABLED ✓"
)


print(
    "\nVALIDATION / TEST:"
)

print(
    "    no oversampling ✓"
)

print(
    "    no downsampling ✓"
)

print(
    "    natural prevalence preserved ✓"
)


print(
    "\nFAMILY CLASSIFICATION:"
)

print(
    "    TRAIN-only class weights frozen ✓"
)

print(
    "    ALL_5_SYNTHETIC ✓"
)

print(
    "    CIC_DOMAIN_4 ✓"
)

print(
    "    UNSW_DOMAIN_4 ✓"
)

print(
    "    SHARED_CROSS_DOMAIN_3 ✓"
)


print(
    "\nPRIMARY EXPERIMENT:"
)

print(
    "    W10 / H5"
)

print(
    "    W10 / H10"
)


print(
    "\nMODEL TRAINING:"
)

print(
    "    STILL NOT STARTED ✓"
)

print(
    "    STILL BLOCKED ✓"
)


print(
    "\nNEXT:"
)

print(
    "    PHASE 12.3"
)

print(
    "    FINAL PRE-MODEL EXPERIMENTAL PROTOCOL FREEZE"
)


print("=" * 120)


gc.collect()

In [ ]:
# ==================================================================================================
# PHASE 12.3 — FINAL PRE-MODEL EXPERIMENTAL PROTOCOL FREEZE
# ==================================================================================================
#
# PURPOSE
# -------
# Freeze the COMPLETE modeling/evaluation protocol BEFORE the first model is trained.
#
#
# THIS PHASE DOES NOT TRAIN ANY MODEL.
#
#
# AFTER THIS PHASE:
#
#   TRAINING on TRAIN is authorized.
#   VALIDATION may be used for:
#
#       - hyperparameter selection
#       - early stopping
#       - threshold selection
#       - architecture/model selection
#
#   TEST remains LOCKED during development.
#
#
# TEST MAY ONLY BE OPENED AFTER:
#
#   - the primary model ladder has completed
#   - every model family's hyperparameters are frozen
#   - final replicate seeds are complete
#   - validation operating thresholds are frozen
#   - the final model registry is locked
#
#
# PRIMARY RESEARCH TASK
# ---------------------
#
#   W = 10 minutes
#
#   H = 5 minutes
#   H = 10 minutes
#
#
# SECONDARY SENSITIVITY
# ---------------------
#
#   W = [5, 10, 15, 30]
#   H = [1, 5, 10, 15, 30]
#
# But sensitivity experiments occur only AFTER the primary W10/H5-H10
# model comparison has been frozen.
#
#
# MODEL LADDER
# ------------
#
# 0. TRAIN-prevalence baseline
#
# 1. Current-minute Logistic Regression
#       establishes whether history adds value
#
# 2. Flattened-history Logistic Regression
#       simple temporal baseline
#
# 3. Temporal-summary XGBoost
#
# 4. GRU
#
# 5. LSTM
#
# 6. TCN
#
# 7. 1D CNN
#
# 8. Small Transformer
#
# 9. InceptionTime
#       secondary architecture / optional resource-permitting benchmark
#
#
# TEST IS NOT USED FOR MODEL/HYPERPARAMETER/THRESHOLD SELECTION.
#
# ==================================================================================================


from pathlib import Path
import hashlib
import json
import shutil
import gc

import numpy as np
import pandas as pd


print("=" * 120)
print("PHASE 12.3 — FINAL PRE-MODEL EXPERIMENTAL PROTOCOL FREEZE")
print("=" * 120)


# ==================================================================================================
# 12.3A — PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v1"
)

MODEL_READY_ROOT = (
    DATASET_ROOT
    / "model_ready"
)


PHASE_12_0_DIR = (
    MODEL_READY_ROOT
    / "12_0"
)

PHASE_12_1_DIR = (
    MODEL_READY_ROOT
    / "12_1"
)

PHASE_12_2_DIR = (
    MODEL_READY_ROOT
    / "12_2"
)


FINAL_OUT = (
    MODEL_READY_ROOT
    / "12_3"
)

STAGING = (
    MODEL_READY_ROOT
    / "._staging_12_3"
)


# --------------------------------------------------------------------------------------------------
# 12.0 controls
# --------------------------------------------------------------------------------------------------

LOCK_12_0_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_LOCK.json"
)

LOCK_12_0_HASH_FILE = (
    PHASE_12_0_DIR
    / "12_0_BUILD_LOCK_SHA256.txt"
)

HASHES_12_0_FILE = (
    PHASE_12_0_DIR
    / "12_0_FILE_HASHES.csv"
)


# --------------------------------------------------------------------------------------------------
# 12.1 controls
# --------------------------------------------------------------------------------------------------

LOCK_12_1_FILE = (
    PHASE_12_1_DIR
    / "12_1_AUDIT_LOCK.json"
)

LOCK_12_1_HASH_FILE = (
    PHASE_12_1_DIR
    / "12_1_AUDIT_LOCK_SHA256.txt"
)

HASHES_12_1_FILE = (
    PHASE_12_1_DIR
    / "12_1_FILE_HASHES.csv"
)

HORIZON_AUDIT_12_1_FILE = (
    PHASE_12_1_DIR
    / "12_1_HORIZON_SUPPORT_AUDIT.csv"
)


# --------------------------------------------------------------------------------------------------
# 12.2 controls
# --------------------------------------------------------------------------------------------------

READY_12_2_FILE = (
    PHASE_12_2_DIR
    / "12_2_READY_FOR_PROTOCOL_FREEZE.json"
)

LOCK_12_2_FILE = (
    PHASE_12_2_DIR
    / "12_2_LOCK.json"
)

LOCK_12_2_HASH_FILE = (
    PHASE_12_2_DIR
    / "12_2_LOCK_SHA256.txt"
)

HASHES_12_2_FILE = (
    PHASE_12_2_DIR
    / "12_2_FILE_HASHES.csv"
)

VALIDATION_12_2_FILE = (
    PHASE_12_2_DIR
    / "audit"
    / "12_2_VALIDATION.csv"
)

BINARY_WEIGHTS_FILE = (
    PHASE_12_2_DIR
    / "policy"
    / "BINARY_CLASS_WEIGHTS.csv"
)

FAMILY_WEIGHTS_FILE = (
    PHASE_12_2_DIR
    / "policy"
    / "FAMILY_CLASS_WEIGHTS.csv"
)

IMBALANCE_POLICY_FILE = (
    PHASE_12_2_DIR
    / "policy"
    / "IMBALANCE_POLICY.json"
)

PREPROCESSING_POLICY_FILE = (
    PHASE_12_2_DIR
    / "policy"
    / "PREPROCESSING_POLICY.json"
)

SCALER_FILE = (
    PHASE_12_2_DIR
    / "preprocessing"
    / "SCALER_PARAMETERS.npz"
)


# ==================================================================================================
# 12.3B — CORE PROTOCOL CONSTANTS
# ==================================================================================================

WINDOWS = [
    5,
    10,
    15,
    30
]


PRIMARY_WINDOW = 10


HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


PRIMARY_HORIZONS = [
    5,
    10
]


FEATURES = [

    "Flow_Count",

    "Flow_Duration_Mean",

    "Flow_Duration_Std",

    "Flow_Duration_Max",

    "Forward_Packets_Sum",

    "Backward_Packets_Sum",

    "Total_Packets",

    "Forward_Bytes_Sum",

    "Backward_Bytes_Sum",

    "Total_Bytes",

    "Packets_Per_Second",

    "Bytes_Per_Second",

    "Forward_Backward_Packet_Ratio",

    "Forward_Backward_Byte_Ratio",
]


FAMILIES = [

    "DOS_DDOS",

    "RECON_SCAN",

    "EXPLOIT_APP",

    "BRUTE_AUTH",

    "FUZZING",
]


PRIMARY_EXPECTED = {

    5: {

        "Eligible_Train_Origins":
            59277,

        "Negative_Train_Origins":
            55777,

        "Positive_Train_Origins":
            3500,
    },


    10: {

        "Eligible_Train_Origins":
            58577,

        "Negative_Train_Origins":
            51577,

        "Positive_Train_Origins":
            7000,
    },
}


# ==================================================================================================
# REPRODUCIBILITY SEEDS
# ==================================================================================================

MASTER_SEED = 20260930


TUNING_SEED = 20260930


FINAL_REPLICATE_SEEDS = [

    20260930,

    20260931,

    20260932,
]


BOOTSTRAP_SEED = 20260933


SEARCH_SEED = 20260934


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(
    path
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def read_json(
    path
):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


def save_csv(
    df,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        path,
        index=False,
        float_format="%.12g"
    )


def verify_hash_table(
    root,
    table
):

    failures = []


    for row in table.itertuples(
        index=False
    ):

        path = (
            root
            / row.Relative_Path
        )


        if not path.is_file():

            failures.append(
                (
                    row.Relative_Path,
                    "MISSING"
                )
            )

            continue


        digest = sha256_file(
            path
        )


        if digest != row.SHA256:

            failures.append(
                (
                    row.Relative_Path,
                    digest
                )
            )


    return failures


def build_hash_table(
    root
):

    excluded = {

        "12_3_FILE_HASHES.csv",

        "12_3_PROTOCOL_LOCK.json",

        "12_3_PROTOCOL_LOCK_SHA256.txt",

        "12_3_READY_FOR_MODELING.json",
    }


    rows = []


    for path in sorted(
        root.rglob(
            "*"
        )
    ):

        if not path.is_file():

            continue


        if path.name in excluded:

            continue


        rows.append(
            {
                "Relative_Path":
                    str(
                        path.relative_to(
                            root
                        )
                    ),

                "Bytes":
                    int(
                        path.stat().st_size
                    ),

                "SHA256":
                    sha256_file(
                        path
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ==================================================================================================
# 12.3C — VERIFY 12.2 AUTHORIZATION
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3C — VERIFY PHASE 12.2 AUTHORIZATION")
print("=" * 120)


for path in [

    READY_12_2_FILE,

    LOCK_12_2_FILE,

    LOCK_12_2_HASH_FILE,

    HASHES_12_2_FILE,

    VALIDATION_12_2_FILE,

    BINARY_WEIGHTS_FILE,

    FAMILY_WEIGHTS_FILE,

    IMBALANCE_POLICY_FILE,

    PREPROCESSING_POLICY_FILE,

    SCALER_FILE,

]:

    require_file(
        path
    )


ready_12_2 = read_json(
    READY_12_2_FILE
)


if ready_12_2.get(
    "phase_12_2_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.2 status is not PASS."
    )


if ready_12_2.get(
    "decision"
) != "PASS_PREPROCESSING_AND_IMBALANCE_FREEZE":

    raise RuntimeError(
        "Unexpected Phase 12.2 decision."
    )


if ready_12_2.get(
    "ready_for_phase_12_3"
) is not True:

    raise RuntimeError(
        "Phase 12.2 does not authorize Phase 12.3."
    )


if ready_12_2.get(
    "model_training_authorized"
) is not False:

    raise RuntimeError(
        "Phase 12.2 unexpectedly authorized model training."
    )


lock_12_2_hash = sha256_file(
    LOCK_12_2_FILE
)


recorded_lock_12_2_hash = (

    LOCK_12_2_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()
)


if (
    lock_12_2_hash
    !=
    recorded_lock_12_2_hash
):

    raise RuntimeError(
        "Phase 12.2 lock hash verification failed."
    )


if (
    ready_12_2.get(
        "phase_12_2_lock_sha256"
    )
    !=
    lock_12_2_hash
):

    raise RuntimeError(
        "Phase 12.2 readiness disagrees with Phase 12.2 lock."
    )


validation_12_2 = pd.read_csv(
    VALIDATION_12_2_FILE
)


validation_pass = (

    validation_12_2[
        "Passed"
    ]

    .astype(
        str
    )

    .str.strip()

    .str.lower()

    .isin(
        [
            "true",
            "1",
            "yes"
        ]
    )
)


if not validation_pass.all():

    raise RuntimeError(
        "Phase 12.2 contains failed hard validation checks."
    )


print(
    "✓ Phase 12.2 PASS"
)

print(
    "✓ preprocessing/imbalance freeze verified"
)

print(
    f"✓ {int(validation_pass.sum())}/{len(validation_pass)} hard checks PASS"
)

print(
    "✓ model training has not started"
)


# ==================================================================================================
# 12.3D — VERIFY 12.0 / 12.1 / 12.2 PAYLOADS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3D — VERIFY ALL MODEL-READY SOURCE PAYLOADS")
print("=" * 120)


for path in [

    LOCK_12_0_FILE,

    LOCK_12_0_HASH_FILE,

    HASHES_12_0_FILE,

    LOCK_12_1_FILE,

    LOCK_12_1_HASH_FILE,

    HASHES_12_1_FILE,

    HORIZON_AUDIT_12_1_FILE,

]:

    require_file(
        path
    )


lock_12_0_hash = sha256_file(
    LOCK_12_0_FILE
)

lock_12_1_hash = sha256_file(
    LOCK_12_1_FILE
)


if (
    lock_12_0_hash
    !=
    LOCK_12_0_HASH_FILE.read_text(
        encoding="utf-8"
    ).strip()
):

    raise RuntimeError(
        "Phase 12.0 lock failed."
    )


if (
    lock_12_1_hash
    !=
    LOCK_12_1_HASH_FILE.read_text(
        encoding="utf-8"
    ).strip()
):

    raise RuntimeError(
        "Phase 12.1 lock failed."
    )


lock_12_2 = read_json(
    LOCK_12_2_FILE
)


if (
    lock_12_2.get(
        "source_phase_12_0_build_lock_sha256"
    )
    !=
    lock_12_0_hash
):

    raise RuntimeError(
        "Phase 12.2 no longer points to current Phase 12.0 lock."
    )


if (
    lock_12_2.get(
        "source_phase_12_1_audit_lock_sha256"
    )
    !=
    lock_12_1_hash
):

    raise RuntimeError(
        "Phase 12.2 no longer points to current Phase 12.1 lock."
    )


hashes_12_0 = pd.read_csv(
    HASHES_12_0_FILE
)

hashes_12_1 = pd.read_csv(
    HASHES_12_1_FILE
)

hashes_12_2 = pd.read_csv(
    HASHES_12_2_FILE
)


failures_12_0 = verify_hash_table(

    PHASE_12_0_DIR,

    hashes_12_0
)


failures_12_1 = verify_hash_table(

    PHASE_12_1_DIR,

    hashes_12_1
)


failures_12_2 = verify_hash_table(

    PHASE_12_2_DIR,

    hashes_12_2
)


if failures_12_0:

    raise RuntimeError(
        f"Phase 12.0 changed:\n{failures_12_0[:10]}"
    )


if failures_12_1:

    raise RuntimeError(
        f"Phase 12.1 changed:\n{failures_12_1[:10]}"
    )


if failures_12_2:

    raise RuntimeError(
        f"Phase 12.2 changed:\n{failures_12_2[:10]}"
    )


print(
    "✓ Phase 12.0 unchanged"
)

print(
    "✓ Phase 12.1 unchanged"
)

print(
    "✓ Phase 12.2 unchanged"
)


# ==================================================================================================
# 12.3E — VERIFY PREPROCESSING / IMBALANCE CONTRACT
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3E — VERIFY FROZEN PREPROCESSING / IMBALANCE POLICY")
print("=" * 120)


preprocessing_policy = read_json(
    PREPROCESSING_POLICY_FILE
)

imbalance_policy = read_json(
    IMBALANCE_POLICY_FILE
)


standardized = preprocessing_policy[
    "standardized_representation"
]


if standardized.get(
    "fit_split"
) != "TRAIN_ONLY":

    raise RuntimeError(
        "Scaler was not frozen as TRAIN_ONLY."
    )


if standardized.get(
    "validation_used_for_fit"
) is not False:

    raise RuntimeError(
        "Validation data unexpectedly used for preprocessing fit."
    )


if standardized.get(
    "test_used_for_fit"
) is not False:

    raise RuntimeError(
        "TEST data unexpectedly used for preprocessing fit."
    )


if preprocessing_policy.get(
    "feature_order"
) != FEATURES:

    raise RuntimeError(
        "Preprocessing feature order changed."
    )


binary_primary_policy = imbalance_policy[
    "binary_primary_training_policy"
]


if binary_primary_policy.get(
    "training_distribution"
) != "NATURAL_TRAIN":

    raise RuntimeError(
        "Primary training distribution is not NATURAL_TRAIN."
    )


if binary_primary_policy.get(
    "SMOTE"
) is not False:

    raise RuntimeError(
        "SMOTE unexpectedly enabled."
    )


if imbalance_policy[
    "validation_policy"
].get(
    "preserve_natural_prevalence"
) is not True:

    raise RuntimeError(
        "Validation natural prevalence is not frozen."
    )


if imbalance_policy[
    "test_policy"
].get(
    "preserve_natural_prevalence"
) is not True:

    raise RuntimeError(
        "TEST natural prevalence is not frozen."
    )


print(
    "✓ log1p → TRAIN-only StandardScaler frozen"
)

print(
    "✓ natural TRAIN + class weighting frozen as primary"
)

print(
    "✓ SMOTE prohibited"
)

print(
    "✓ validation prevalence untouched"
)

print(
    "✓ TEST prevalence untouched"
)


# ==================================================================================================
# 12.3F — VERIFY PRIMARY W10/H5-H10 SUPPORT + WEIGHTS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3F — VERIFY PRIMARY W10/H5-H10 EXPERIMENT SUPPORT")
print("=" * 120)


binary_weights = pd.read_csv(
    BINARY_WEIGHTS_FILE
)


primary_weights = binary_weights.loc[
    (
        binary_weights[
            "History_Window_Minutes"
        ]
        ==
        PRIMARY_WINDOW
    )

    &

    (
        binary_weights[
            "Horizon_Minutes"
        ].isin(
            PRIMARY_HORIZONS
        )
    )
].copy()


if len(
    primary_weights
) != 2:

    raise RuntimeError(
        "Expected exactly two primary W10 binary-weight rows."
    )


for H in PRIMARY_HORIZONS:

    row = primary_weights.loc[
        primary_weights[
            "Horizon_Minutes"
        ]
        ==
        H
    ]


    if len(
        row
    ) != 1:

        raise RuntimeError(
            f"W10/H{H}: missing or duplicated binary-weight row."
        )


    row = row.iloc[
        0
    ]


    expected = PRIMARY_EXPECTED[
        H
    ]


    for column, value in expected.items():

        if int(
            row[
                column
            ]
        ) != value:

            raise RuntimeError(
                f"W10/H{H}: {column} changed."
            )


    expected_pos_weight = (

        expected[
            "Negative_Train_Origins"
        ]

        /

        expected[
            "Positive_Train_Origins"
        ]
    )


    if not np.isclose(

        float(
            row[
                "BCE_Pos_Weight"
            ]
        ),

        expected_pos_weight,

        atol=1e-10,

        rtol=1e-10

    ):

        raise RuntimeError(
            f"W10/H{H}: BCE pos_weight changed."
        )


    print(
        f"✓ W10/H{H}: "
        f"eligible={expected['Eligible_Train_Origins']:,} "
        f"neg={expected['Negative_Train_Origins']:,} "
        f"pos={expected['Positive_Train_Origins']:,} "
        f"pos_weight={expected_pos_weight:.6f}"
    )


# ==================================================================================================
# 12.3G — VERIFY TRANSFORMED PRIMARY TENSORS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3G — VERIFY PRIMARY MODEL INPUT ARTIFACTS")
print("=" * 120)


for split_name in [
    "train",
    "validation",
    "test"
]:

    transformed_file = (

        PHASE_12_2_DIR
        / "transformed"
        / "W10"
        / split_name
        / "X_log1p_standardized.npy"
    )


    raw_file = (

        PHASE_12_0_DIR
        / "W10"
        / split_name
        / "X.npy"
    )


    require_file(
        transformed_file
    )

    require_file(
        raw_file
    )


    X_transformed = np.load(

        transformed_file,

        mmap_mode="r"
    )


    X_raw = np.load(

        raw_file,

        mmap_mode="r"
    )


    if X_transformed.shape != X_raw.shape:

        raise RuntimeError(
            f"W10/{split_name}: transformed/raw shape mismatch."
        )


    if X_transformed.shape[
        1:
    ] != (
        10,
        14
    ):

        raise RuntimeError(
            f"W10/{split_name}: unexpected tensor shape."
        )


    if X_transformed.dtype != np.float32:

        raise RuntimeError(
            f"W10/{split_name}: transformed dtype is not float32."
        )


    print(
        f"✓ {split_name:<10} "
        f"raw={X_raw.shape} "
        f"standardized={X_transformed.shape}"
    )


    del X_transformed

    del X_raw


gc.collect()


# ==================================================================================================
# 12.3H — OUTPUT SAFETY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3H — OUTPUT SAFETY")
print("=" * 120)


MODEL_READY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


if STAGING.exists():

    shutil.rmtree(
        STAGING
    )


if FINAL_OUT.exists():

    ready_existing_file = (
        FINAL_OUT
        / "12_3_READY_FOR_MODELING.json"
    )

    lock_existing_file = (
        FINAL_OUT
        / "12_3_PROTOCOL_LOCK.json"
    )

    lock_existing_hash_file = (
        FINAL_OUT
        / "12_3_PROTOCOL_LOCK_SHA256.txt"
    )


    if (
        ready_existing_file.is_file()

        and

        lock_existing_file.is_file()

        and

        lock_existing_hash_file.is_file()
    ):

        existing_hash = sha256_file(
            lock_existing_file
        )


        recorded_existing_hash = (

            lock_existing_hash_file

            .read_text(
                encoding="utf-8"
            )

            .strip()
        )


        existing_ready = read_json(
            ready_existing_file
        )


        if (
            existing_hash
            ==
            recorded_existing_hash

            and

            existing_ready.get(
                "training_authorized"
            )
            is True

            and

            existing_ready.get(
                "test_evaluation_authorized"
            )
            is False
        ):

            print(
                "✓ Phase 12.3 already exists and is locked."
            )

            print(
                "\nTRAIN/VALIDATION modeling is authorized."
            )

            print(
                "TEST remains locked."
            )

            raise SystemExit


    attempt = 1


    while (

        MODEL_READY_ROOT
        / f"_incomplete_12_3_attempt_{attempt}"

    ).exists():

        attempt += 1


    backup = (

        MODEL_READY_ROOT
        / f"_incomplete_12_3_attempt_{attempt}"
    )


    shutil.move(
        str(
            FINAL_OUT
        ),
        str(
            backup
        )
    )


    print(
        "⚠ previous incomplete Phase 12.3 preserved:"
    )

    print(
        backup
    )


STAGING.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 12.3I — PRIMARY EXPERIMENT MATRIX
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3I — FREEZE PRIMARY EXPERIMENT MATRIX")
print("=" * 120)


horizon_audit = pd.read_csv(
    HORIZON_AUDIT_12_1_FILE
)


primary_experiment_matrix = horizon_audit.loc[
    (
        horizon_audit[
            "History_Window_Minutes"
        ]
        ==
        PRIMARY_WINDOW
    )

    &

    (
        horizon_audit[
            "Horizon_Minutes"
        ].isin(
            PRIMARY_HORIZONS
        )
    )
].copy()


primary_experiment_matrix = (

    primary_experiment_matrix

    .sort_values(
        [
            "Horizon_Minutes",
            "Split"
        ]
    )

    .reset_index(
        drop=True
    )
)


if len(
    primary_experiment_matrix
) != 6:

    raise RuntimeError(
        "Expected 6 primary split×horizon experiment rows."
    )


primary_experiment_matrix[
    "Experiment_Role"
] = "PRIMARY"


primary_experiment_matrix[
    "Input_Window"
] = "W10"


primary_experiment_matrix[
    "Model_Selection_Split"
] = "VALIDATION"


primary_experiment_matrix[
    "Test_Used_For_Selection"
] = False


save_csv(

    primary_experiment_matrix,

    STAGING
    / "12_3_PRIMARY_EXPERIMENT_MATRIX.csv"
)


print(
    primary_experiment_matrix.to_string(
        index=False
    )
)


# ==================================================================================================
# 12.3J — SECONDARY SENSITIVITY MATRIX
# ==================================================================================================

sensitivity_rows = []


for W in WINDOWS:

    for H in HORIZONS:

        is_primary = (

            W
            ==
            PRIMARY_WINDOW

            and

            H
            in
            PRIMARY_HORIZONS
        )


        sensitivity_rows.append(
            {
                "History_Window_Minutes":
                    W,

                "Horizon_Minutes":
                    H,

                "Role":
                    (
                        "PRIMARY"

                        if is_primary

                        else "SECONDARY_SENSITIVITY"
                    ),

                "May_Run_Before_Primary_Model_Lock":
                    bool(
                        is_primary
                    ),

                "Purpose":
                    (
                        "headline forecasting benchmark"

                        if is_primary

                        else
                        "history/horizon sensitivity after primary model lock"
                    ),
            }
        )


sensitivity_matrix = pd.DataFrame(
    sensitivity_rows
)


save_csv(

    sensitivity_matrix,

    STAGING
    / "12_3_SENSITIVITY_MATRIX.csv"
)


# ==================================================================================================
# 12.3K — MODEL LADDER
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3K — FREEZE MODEL LADDER")
print("=" * 120)


model_ladder = pd.DataFrame(
    [
        {
            "Order":
                0,

            "Model_ID":
                "PREVALENCE",

            "Model_Name":
                "TRAIN Prevalence Baseline",

            "Category":
                "NON_LEARNING",

            "Input":
                "no X",

            "Representation":
                "TRAIN label prevalence",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                False,

            "Final_Replicate_Count":
                1,

            "Purpose":
                "minimum ranking/calibration reference",
        },


        {
            "Order":
                1,

            "Model_ID":
                "LOGREG_CURRENT",

            "Model_Name":
                "Current-Minute Logistic Regression",

            "Category":
                "CLASSICAL",

            "Input":
                "X[:, -1, :]",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                1,

            "Purpose":
                "non-temporal learned baseline",
        },


        {
            "Order":
                2,

            "Model_ID":
                "LOGREG_FLAT",

            "Model_Name":
                "Flattened-History Logistic Regression",

            "Category":
                "CLASSICAL_TEMPORAL",

            "Input":
                "flatten W10 × 14",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                1,

            "Purpose":
                "simple temporal baseline",
        },


        {
            "Order":
                3,

            "Model_ID":
                "XGBOOST_TEMPORAL",

            "Model_Name":
                "Temporal-Summary XGBoost",

            "Category":
                "CLASSICAL_TEMPORAL",

            "Input":
                "causal temporal summary features from raw W10",

            "Representation":
                "raw sequence-derived summaries",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                3,

            "Purpose":
                "strong nonlinear classical benchmark",
        },


        {
            "Order":
                4,

            "Model_ID":
                "GRU",

            "Model_Name":
                "GRU",

            "Category":
                "DEEP_TEMPORAL",

            "Input":
                "W10 × 14",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                3,

            "Purpose":
                "recurrent temporal benchmark",
        },


        {
            "Order":
                5,

            "Model_ID":
                "LSTM",

            "Model_Name":
                "LSTM",

            "Category":
                "DEEP_TEMPORAL",

            "Input":
                "W10 × 14",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                3,

            "Purpose":
                "gated recurrent benchmark",
        },


        {
            "Order":
                6,

            "Model_ID":
                "TCN",

            "Model_Name":
                "Temporal Convolutional Network",

            "Category":
                "DEEP_TEMPORAL",

            "Input":
                "W10 × 14",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                3,

            "Purpose":
                "causal dilated temporal convolution benchmark",
        },


        {
            "Order":
                7,

            "Model_ID":
                "CNN1D",

            "Model_Name":
                "1D CNN",

            "Category":
                "DEEP_TEMPORAL",

            "Input":
                "W10 × 14",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                3,

            "Purpose":
                "local temporal-pattern benchmark",
        },


        {
            "Order":
                8,

            "Model_ID":
                "TRANSFORMER_SMALL",

            "Model_Name":
                "Small Transformer Encoder",

            "Category":
                "DEEP_TEMPORAL",

            "Input":
                "W10 × 14",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                True,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                3,

            "Purpose":
                "attention-based temporal benchmark",
        },


        {
            "Order":
                9,

            "Model_ID":
                "INCEPTIONTIME",

            "Model_Name":
                "Compact InceptionTime",

            "Category":
                "SECONDARY_DEEP_TEMPORAL",

            "Input":
                "W10 × 14",

            "Representation":
                "log1p + TRAIN StandardScaler",

            "Primary_Benchmark":
                False,

            "Hyperparameter_Tuning":
                True,

            "Final_Replicate_Count":
                3,

            "Purpose":
                "optional secondary architecture if compute permits",
        },
    ]
)


save_csv(

    model_ladder,

    STAGING
    / "12_3_MODEL_LADDER.csv"
)


print(
    model_ladder[
        [
            "Order",
            "Model_ID",
            "Model_Name",
            "Primary_Benchmark"
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# 12.3L — XGBOOST CAUSAL SUMMARY FEATURE CONTRACT
# ==================================================================================================

xgboost_feature_policy = {

    "model":
        "XGBOOST_TEMPORAL",

    "source":
        "raw Phase 12.0 W10 sequence only",

    "future_information":
        False,

    "metadata_features":
        False,

    "summary_operations_per_predictor": [

        "last",

        "mean",

        "std",

        "min",

        "max",

        "delta_last_minus_first",

        "linear_slope",
    ],

    "predictor_count":
        14,

    "summary_features_per_predictor":
        7,

    "expected_total_features":
        98,

    "scaler_required":
        False,
}


write_json(

    xgboost_feature_policy,

    STAGING
    / "12_3_XGBOOST_FEATURE_POLICY.json"
)


# ==================================================================================================
# 12.3M — RANDOM SEED POLICY
# ==================================================================================================

seed_policy = {

    "master_seed":
        MASTER_SEED,

    "hyperparameter_search_seed":
        SEARCH_SEED,

    "development_tuning_seed":
        TUNING_SEED,

    "final_replicate_seeds":
        FINAL_REPLICATE_SEEDS,

    "episode_bootstrap_seed":
        BOOTSTRAP_SEED,

    "rules": {

        "change_seed_after_results":
            False,

        "select_best_seed":
            False,

        "report_only_best_seed":
            False,

        "stochastic_model_final_reporting":
            "mean and standard deviation across all frozen final replicate seeds",

        "deterministic_models":
            "single deterministic fit unless implementation has stochastic behavior",
    },
}


write_json(

    seed_policy,

    STAGING
    / "12_3_RANDOM_SEED_POLICY.json"
)


# ==================================================================================================
# 12.3N — HYPERPARAMETER SEARCH BUDGETS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3N — FREEZE HYPERPARAMETER BUDGETS")
print("=" * 120)


hyperparameter_policy = {

    "general": {

        "primary_window":
            10,

        "primary_horizons":
            [
                5,
                10
            ],

        "hyperparameter_selection_split":
            "VALIDATION",

        "test_used_for_hyperparameter_selection":
            False,

        "search_seed":
            SEARCH_SEED,

        "development_seed":
            TUNING_SEED,

        "hyperparameters_may_be_retuned_after_test":
            False,
    },


    "PREVALENCE": {

        "max_trials_per_horizon":
            0,
    },


    "LOGREG_CURRENT": {

        "max_trials_per_horizon":
            5,

        "representation":
            "last standardized minute",

        "solver":
            "lbfgs",

        "penalty":
            "l2",

        "max_iter":
            3000,

        "C_grid": [
            0.01,
            0.1,
            1.0,
            10.0,
            100.0
        ],

        "class_weights":
            "frozen TRAIN-only weights from Phase 12.2",
    },


    "LOGREG_FLAT": {

        "max_trials_per_horizon":
            5,

        "representation":
            "flattened standardized W10 sequence",

        "solver":
            "lbfgs",

        "penalty":
            "l2",

        "max_iter":
            3000,

        "C_grid": [
            0.01,
            0.1,
            1.0,
            10.0,
            100.0
        ],

        "class_weights":
            "frozen TRAIN-only weights from Phase 12.2",
    },


    "XGBOOST_TEMPORAL": {

        "max_trials_per_horizon":
            12,

        "search":
            "seeded randomized search",

        "objective":
            "binary:logistic",

        "evaluation_metric":
            "aucpr",

        "max_estimators":
            2000,

        "early_stopping_rounds":
            50,

        "scale_pos_weight":
            "frozen TRAIN N_negative/N_positive from Phase 12.2",

        "search_space": {

            "learning_rate": [
                0.02,
                0.05,
                0.1
            ],

            "max_depth": [
                3,
                4,
                6
            ],

            "min_child_weight": [
                1,
                3,
                5
            ],

            "subsample": [
                0.7,
                0.85,
                1.0
            ],

            "colsample_bytree": [
                0.7,
                0.85,
                1.0
            ],

            "reg_lambda": [
                1.0,
                5.0,
                10.0
            ],

            "reg_alpha": [
                0.0,
                0.1,
                1.0
            ],
        },
    },


    "DEEP_COMMON": {

        "optimizer":
            "AdamW",

        "loss":
            "BCEWithLogitsLoss",

        "pos_weight":
            "frozen TRAIN-only BCE_Pos_Weight from Phase 12.2",

        "primary_sampler":
            "natural TRAIN distribution",

        "episode_balanced_sampler_primary":
            False,

        "max_epochs":
            60,

        "early_stopping_metric":
            "validation_average_precision",

        "early_stopping_mode":
            "max",

        "early_stopping_patience":
            8,

        "early_stopping_min_delta":
            0.0001,

        "restore_best_weights":
            True,

        "gradient_clip_norm":
            1.0,

        "batch_size":
            256,

        "learning_rate_space": [
            0.0003,
            0.001
        ],

        "weight_decay_space": [
            0.0,
            0.0001,
            0.001
        ],

        "dropout_space": [
            0.1,
            0.3
        ],

        "development_seed":
            TUNING_SEED,

        "final_replicate_seeds":
            FINAL_REPLICATE_SEEDS,
    },


    "GRU": {

        "max_trials_per_horizon":
            4,

        "hidden_size_space": [
            64,
            128
        ],

        "num_layers_space": [
            1,
            2
        ],
    },


    "LSTM": {

        "max_trials_per_horizon":
            4,

        "hidden_size_space": [
            64,
            128
        ],

        "num_layers_space": [
            1,
            2
        ],
    },


    "TCN": {

        "max_trials_per_horizon":
            4,

        "channels_space": [
            64,
            128
        ],

        "kernel_size_space": [
            3,
            5
        ],

        "block_count_space": [
            2,
            3
        ],

        "causal_convolution_required":
            True,
    },


    "CNN1D": {

        "max_trials_per_horizon":
            4,

        "filters_space": [
            64,
            128
        ],

        "kernel_size_space": [
            3,
            5
        ],

        "dense_size_space": [
            64,
            128
        ],
    },


    "TRANSFORMER_SMALL": {

        "max_trials_per_horizon":
            4,

        "d_model_space": [
            64,
            128
        ],

        "n_heads_space": [
            2,
            4
        ],

        "num_encoder_layers_space": [
            1,
            2
        ],

        "feedforward_multiplier_space": [
            2,
            4
        ],

        "positional_encoding":
            "learned_or_sinusoidal_predeclared_before_first_run",

        "causal_attention_mask_required":
            False,

        "reason_causal_mask_not_required":
            (
                "all W historical positions occur at or before origin t; "
                "there are no future positions inside X"
            ),
    },


    "INCEPTIONTIME": {

        "role":
            "SECONDARY_OPTIONAL",

        "max_trials_per_horizon":
            3,

        "filters_space": [
            32,
            64
        ],

        "depth_space": [
            3,
            6
        ],

        "bottleneck_space": [
            16,
            32
        ],
    },
}


write_json(

    hyperparameter_policy,

    STAGING
    / "12_3_HYPERPARAMETER_POLICY.json"
)


print(
    "✓ Logistic grids frozen"
)

print(
    "✓ XGBoost search budget = 12 trials/horizon"
)

print(
    "✓ primary deep-model budget = 4 trials/architecture/horizon"
)

print(
    "✓ max deep epochs = 60 with validation-AP early stopping"
)

print(
    "✓ final stochastic replicate seeds =",
    FINAL_REPLICATE_SEEDS
)


# ==================================================================================================
# 12.3O — HYPERPARAMETER SELECTION POLICY
# ==================================================================================================

model_selection_policy = {

    "unit_of_selection":
        "model family × forecast horizon",

    "primary_selection_metric":
        "VALIDATION Average Precision",

    "higher_is_better":
        True,

    "hyperparameter_tie_tolerance_AP":
        0.0001,

    "within_family_tie_break_order": [

        "higher validation Average Precision",

        "higher validation F1 at validation-selected threshold",

        "lower model complexity",

        "deterministic lexical configuration order",
    ],


    "stochastic_model_selection": {

        "hyperparameter_search":
            "development tuning seed only",

        "after_hyperparameters_selected":
            "retrain selected configuration using all frozen final replicate seeds",

        "architecture_comparison_statistic":
            "mean validation Average Precision across frozen final replicate seeds",

        "report_variability":
            "standard deviation across replicate seeds",

        "best_seed_selection":
            False,
    },


    "final_model_family_selection": {

        "performed_separately_for_H5_and_H10":
            True,

        "primary_metric":
            "mean validation Average Precision",

        "practical_tie_tolerance_AP":
            0.002,

        "if_within_tolerance_prefer": [

            "lower complexity",

            "higher mean validation F1",

            "higher validation event recall",

            "lower validation false alerts per hour",
        ],
    },


    "test_used_for_selection":
        False,


    "sensitivity_experiments_used_to_select_primary_model":
        False,
}


write_json(

    model_selection_policy,

    STAGING
    / "12_3_MODEL_SELECTION_POLICY.json"
)


# ==================================================================================================
# 12.3P — OPERATING THRESHOLD POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3P — FREEZE THRESHOLD POLICY")
print("=" * 120)


threshold_policy = {

    "threshold_source":
        "VALIDATION_ONLY",

    "separate_threshold_per": [

        "model",

        "forecast horizon",

        "final replicate seed when stochastic",
    ],


    "primary_threshold_objective":
        "maximize validation binary F1",


    "candidate_thresholds":
        (
            "all unique finite validation probability scores, "
            "plus deterministic boundary candidates if needed"
        ),


    "tie_break_order": [

        "higher F1",

        "higher precision",

        "higher threshold",
    ],


    "test_threshold_retuning":
        False,


    "test_application":
        "apply frozen validation-derived threshold unchanged",


    "event_metrics_threshold":
        "same frozen binary operating threshold",


    "lead_time_threshold":
        "same frozen binary operating threshold",


    "false_alert_threshold":
        "same frozen binary operating threshold",


    "threshold_free_headline_metric":
        "Average Precision / PR-AUC",


    "secondary_threshold_diagnostics": [

        "precision-recall curve",

        "event recall versus false-alert-rate curve",
    ],


    "secondary_diagnostics_may_replace_primary_threshold":
        False,
}


write_json(

    threshold_policy,

    STAGING
    / "12_3_THRESHOLD_POLICY.json"
)


print(
    "✓ thresholds selected on VALIDATION only"
)

print(
    "✓ primary threshold = maximum validation F1"
)

print(
    "✓ same frozen threshold used for event metrics"
)

print(
    "✓ TEST threshold tuning prohibited"
)


# ==================================================================================================
# 12.3Q — EVALUATION METRIC POLICY
# ==================================================================================================

evaluation_protocol = {

    "binary_forecasting": {

        "headline_metric":
            "Average Precision / PR-AUC",

        "primary_threshold_metrics": [

            "Precision",

            "Recall",

            "F1",
        ],

        "secondary_metrics": [

            "ROC-AUC",

            "Accuracy",

            "Brier score",
        ],

        "raw_accuracy_is_headline":
            False,
    },


    "event_level": {

        "primary_metrics": [

            "Event Recall",

            "Event Precision",

            "Median First-Warning Lead Time",

            "Mean First-Warning Lead Time",

            "False Alerts Per Hour",
        ],

        "event_identity":
            "Episode_ID",

        "lead_time_definition":
            "true onset minus first threshold-crossing warning",

        "multiple_positive_origins_equal_multiple_events":
            False,
    },


    "family_classification": {

        "primary_metrics": [

            "Macro F1",

            "Macro Recall",

            "Macro Precision",
        ],

        "required_diagnostics": [

            "per-class precision",

            "per-class recall",

            "per-class F1",

            "confusion matrix",
        ],

        "weighted_F1":
            "secondary",

        "ordinary_metrics_only_on_seen_families":
            True,
    },


    "time_to_onset": {

        "metrics": [

            "MAE",

            "Median Absolute Error",
        ],

        "evaluate_on":
            "eligible positive origins",
    },


    "uncertainty": {

        "final_test_confidence_intervals":
            True,

        "bootstrap_unit":
            "Episode_ID",

        "bootstrap_replicates":
            1000,

        "confidence_level":
            0.95,

        "seed":
            BOOTSTRAP_SEED,

        "window_level_independent_bootstrap":
            False,
    },


    "precursor_regime_reporting": {

        "enabled":
            True,

        "evaluation_only":
            True,

        "may_enter_X":
            False,

        "may_be_used_for_model_selection":
            False,
    },


    "source_domain_reporting": {

        "binary_report_overall":
            True,

        "binary_report_CIC":
            True,

        "binary_report_UNSW":
            True,

        "source_may_enter_X":
            False,

        "source_may_be_used_for_threshold_selection":
            False,
    },
}


write_json(

    evaluation_protocol,

    STAGING
    / "12_3_EVALUATION_PROTOCOL.json"
)


# ==================================================================================================
# 12.3R — TRAINING DATA POLICY
# ==================================================================================================

training_data_policy = {

    "binary_task": {

        "one_model_per_horizon_initially":
            True,

        "primary_horizons":
            [
                5,
                10
            ],

        "primary_history_window":
            10,

        "eligible_origins_only":
            True,

        "horizon_ineligible_origins_used":
            False,

        "input_metadata":
            False,

        "Episode_ID_as_input":
            False,

        "Calibration_Source_as_input":
            False,

        "Precursor_Regime_as_input":
            False,

        "future_family_as_input":
            False,

        "Minutes_To_Next_Onset_as_input":
            False,
    },


    "primary_imbalance_strategy": {

        "TRAIN_distribution":
            "natural",

        "class_weighting":
            True,

        "weight_source":
            "Phase 12.2 TRAIN-only frozen weights",

        "episode_balanced_sampling":
            False,

        "SMOTE":
            False,

        "negative_downsampling":
            False,
    },


    "secondary_imbalance_ablation": {

        "episode_balanced_sampling":
            True,

        "combine_with_primary_class_weighting":
            False,

        "focal_loss":
            "optional predeclared ablation",

        "negative_downsampling":
            "not part of primary benchmark",
    },


    "validation": {

        "resampling":
            False,

        "natural_prevalence":
            True,

        "used_for_early_stopping":
            True,

        "used_for_hyperparameter_selection":
            True,

        "used_for_threshold_selection":
            True,
    },


    "test": {

        "resampling":
            False,

        "natural_prevalence":
            True,

        "used_for_early_stopping":
            False,

        "used_for_hyperparameter_selection":
            False,

        "used_for_threshold_selection":
            False,

        "used_for_model_selection":
            False,
    },


    "representation": {

        "prevalence":
            "labels only",

        "logistic":
            "Phase 12.2 standardized X",

        "XGBoost":
            "raw Phase 12.0 X → frozen 98 causal temporal summaries",

        "deep_models":
            "Phase 12.2 standardized X",
    },
}


write_json(

    training_data_policy,

    STAGING
    / "12_3_TRAINING_DATA_POLICY.json"
)


# ==================================================================================================
# 12.3S — TEST EMBARGO POLICY
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3S — FREEZE TEST EMBARGO")
print("=" * 120)


test_access_policy = {

    "test_status_during_model_development":
        "LOCKED",


    "test_predictions_allowed_during": {

        "hyperparameter_search":
            False,

        "early_stopping":
            False,

        "threshold_selection":
            False,

        "architecture_selection":
            False,

        "primary_model_selection":
            False,

        "sensitivity_analysis_before_primary_lock":
            False,
    },


    "unlock_requirements": [

        "all primary W10/H5 and W10/H10 model families completed",

        "hyperparameters frozen",

        "validation thresholds frozen",

        "final replicate runs completed",

        "model-family ranking frozen",

        "final model registry hash locked",

        "no additional architecture changes planned",
    ],


    "after_unlock": {

        "evaluate_all_predeclared_final_model families":
            True,

        "thresholds_changed_after_test":
            False,

        "hyperparameters_changed_after_test":
            False,

        "architecture_changed_after_test":
            False,

        "best_test_model_used_to_rewrite protocol":
            False,
    },


    "post_test_changes": {

        "allowed_as_primary_confirmatory_result":
            False,

        "must_be_labelled":
            "POST-HOC EXPLORATORY",

        "benchmark_version_change_required_if_dataset_changes":
            True,
    },
}


write_json(

    test_access_policy,

    STAGING
    / "12_3_TEST_ACCESS_POLICY.json"
)


print(
    "✓ TEST locked during all model development"
)

print(
    "✓ TEST cannot select hyperparameters"
)

print(
    "✓ TEST cannot select thresholds"
)

print(
    "✓ TEST cannot select architecture"
)

print(
    "✓ TEST opens only after final model registry is locked"
)


# ==================================================================================================
# 12.3T — SECONDARY TASK POLICY
# ==================================================================================================

secondary_task_policy = {

    "family_classification": {

        "start_before_primary_binary_forecasting_complete":
            False,

        "input_origin":
            "positive eligible pre-onset origins",

        "primary_domains": [

            "CIC_DOMAIN_4",

            "UNSW_DOMAIN_4",

            "SHARED_CROSS_DOMAIN_3",
        ],

        "ALL_5_SYNTHETIC":
            "supporting_only",

        "weights":
            "Phase 12.2 TRAIN-only family weights",

        "test_selection":
            False,
    },


    "time_to_onset": {

        "start_before_primary_binary_forecasting_complete":
            False,

        "evaluate_on":
            "positive eligible pre-onset origins",

        "metrics": [

            "MAE",

            "Median Absolute Error",
        ],
    },


    "multitask_model": {

        "primary_initial_model":
            False,

        "may_be_explored_after_single-task baselines":
            True,

        "must_not_replace_single-task benchmark retroactively":
            True,
    },


    "ensemble": {

        "primary_initial_model":
            False,

        "allowed_after_base_model ladder frozen":
            True,

        "weights_or_rules_fit_on":
            "VALIDATION_ONLY",

        "test_optimized ensemble":
            False,
    },
}


write_json(

    secondary_task_policy,

    STAGING
    / "12_3_SECONDARY_TASK_POLICY.json"
)


# ==================================================================================================
# 12.3U — DEVELOPMENT PHASE ORDER
# ==================================================================================================

development_plan = pd.DataFrame(
    [
        {
            "Phase":
                "13.0",

            "Stage":
                "BASELINES_AND_CLASSICAL",

            "Models":
                (
                    "PREVALENCE, LOGREG_CURRENT, "
                    "LOGREG_FLAT, XGBOOST_TEMPORAL"
                ),

            "Uses_TEST":
                False,

            "Goal":
                "establish task difficulty and temporal value",
        },


        {
            "Phase":
                "13.1",

            "Stage":
                "DEEP_TEMPORAL_MODELS",

            "Models":
                "GRU, LSTM, TCN, CNN1D, TRANSFORMER_SMALL",

            "Uses_TEST":
                False,

            "Goal":
                "compare temporal architectures on validation",
        },


        {
            "Phase":
                "13.2",

            "Stage":
                "FINAL_MODEL_REGISTRY_LOCK",

            "Models":
                "all completed primary models",

            "Uses_TEST":
                False,

            "Goal":
                (
                    "freeze hyperparameters, thresholds, "
                    "replicate summaries and final model registry"
                ),
        },


        {
            "Phase":
                "13.3",

            "Stage":
                "LOCKED_TEST_EVALUATION",

            "Models":
                "predeclared final model registry",

            "Uses_TEST":
                True,

            "Goal":
                "single confirmatory TEST evaluation batch",
        },


        {
            "Phase":
                "14.x",

            "Stage":
                "SECONDARY_TASKS_AND_SENSITIVITY",

            "Models":
                "family, ETA, sensitivity, optional ensemble",

            "Uses_TEST":
                "only under frozen policies",

            "Goal":
                "secondary scientific analyses",
        },
    ]
)


save_csv(

    development_plan,

    STAGING
    / "12_3_DEVELOPMENT_PLAN.csv"
)


# ==================================================================================================
# 12.3V — COMPLETE PROTOCOL MANIFEST
# ==================================================================================================

protocol_manifest = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.3",

    "status":
        "FINAL_PRE_MODEL_PROTOCOL_FROZEN",


    "source_locks": {

        "phase_12_0":
            lock_12_0_hash,

        "phase_12_1":
            lock_12_1_hash,

        "phase_12_2":
            lock_12_2_hash,
    },


    "primary_task": {

        "task":
            "NEW_ATTACK_ONSET_FORECASTING",

        "history_window_minutes":
            10,

        "forecast_horizons_minutes": [
            5,
            10
        ],
    },


    "secondary_sensitivity": {

        "history_windows_minutes":
            WINDOWS,

        "forecast_horizons_minutes":
            HORIZONS,

        "may_change_primary_model_selection":
            False,
    },


    "headline_metric":
        "Average Precision / PR-AUC",


    "threshold_selection":
        "VALIDATION F1",


    "model_selection_split":
        "VALIDATION",


    "test_status":
        "LOCKED_DURING_MODEL_DEVELOPMENT",


    "training_representation": {

        "linear_and_deep":
            "Phase 12.2 log1p + TRAIN StandardScaler",

        "XGBoost":
            "Phase 12.0 raw sequences converted to 98 frozen causal summaries",
    },


    "primary_imbalance_handling":
        "natural TRAIN distribution + TRAIN-derived class weights",


    "model_ladder": [

        "PREVALENCE",

        "LOGREG_CURRENT",

        "LOGREG_FLAT",

        "XGBOOST_TEMPORAL",

        "GRU",

        "LSTM",

        "TCN",

        "CNN1D",

        "TRANSFORMER_SMALL",
    ],


    "secondary_optional_models": [

        "INCEPTIONTIME",

        "validation-only ensemble",
    ],


    "reproducibility": {

        "master_seed":
            MASTER_SEED,

        "final_replicate_seeds":
            FINAL_REPLICATE_SEEDS,

        "bootstrap_seed":
            BOOTSTRAP_SEED,
    },


    "test_unlock_phase":
        "13.3",

    "next_phase":
        "13.0_BASELINES_AND_CLASSICAL_MODELS",
}


write_json(

    protocol_manifest,

    STAGING
    / "12_3_PROTOCOL_MANIFEST.json"
)


protocol_manifest_sha256 = sha256_file(

    STAGING
    / "12_3_PROTOCOL_MANIFEST.json"
)


# ==================================================================================================
# 12.3W — HARD VALIDATION
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3W — HARD VALIDATION")
print("=" * 120)


validation_rows = []


def add_check(
    name,
    passed,
    details
):

    validation_rows.append(
        {
            "Check":
                name,

            "Passed":
                bool(
                    passed
                ),

            "Details":
                str(
                    details
                ),
        }
    )


add_check(
    "Phase_12_2_PASS",
    ready_12_2.get(
        "phase_12_2_status"
    )
    ==
    "PASS",
    ready_12_2.get(
        "phase_12_2_status"
    )
)


add_check(
    "Phase_12_2_Lock_Exact",
    lock_12_2_hash
    ==
    recorded_lock_12_2_hash,
    lock_12_2_hash
)


add_check(
    "Phase_12_0_Unchanged",
    len(
        failures_12_0
    )
    ==
    0,
    f"failures={len(failures_12_0)}"
)


add_check(
    "Phase_12_1_Unchanged",
    len(
        failures_12_1
    )
    ==
    0,
    f"failures={len(failures_12_1)}"
)


add_check(
    "Phase_12_2_Unchanged",
    len(
        failures_12_2
    )
    ==
    0,
    f"failures={len(failures_12_2)}"
)


add_check(
    "Primary_Window_W10",
    PRIMARY_WINDOW
    ==
    10,
    PRIMARY_WINDOW
)


add_check(
    "Primary_Horizons_H5_H10",
    PRIMARY_HORIZONS
    ==
    [
        5,
        10
    ],
    PRIMARY_HORIZONS
)


add_check(
    "Primary_Experiment_Rows_6",
    len(
        primary_experiment_matrix
    )
    ==
    6,
    len(
        primary_experiment_matrix
    )
)


add_check(
    "Primary_W10_H5_Train_Positive_3500",
    int(
        primary_weights.loc[
            primary_weights[
                "Horizon_Minutes"
            ]
            ==
            5,
            "Positive_Train_Origins"
        ].iloc[
            0
        ]
    )
    ==
    3500,
    3500
)


add_check(
    "Primary_W10_H10_Train_Positive_7000",
    int(
        primary_weights.loc[
            primary_weights[
                "Horizon_Minutes"
            ]
            ==
            10,
            "Positive_Train_Origins"
        ].iloc[
            0
        ]
    )
    ==
    7000,
    7000
)


add_check(
    "Preprocessing_TRAIN_Only",
    standardized.get(
        "fit_split"
    )
    ==
    "TRAIN_ONLY",
    standardized.get(
        "fit_split"
    )
)


add_check(
    "Validation_Not_Used_For_Scaler",
    standardized.get(
        "validation_used_for_fit"
    )
    is False,
    standardized.get(
        "validation_used_for_fit"
    )
)


add_check(
    "TEST_Not_Used_For_Scaler",
    standardized.get(
        "test_used_for_fit"
    )
    is False,
    standardized.get(
        "test_used_for_fit"
    )
)


add_check(
    "SMOTE_Prohibited",
    binary_primary_policy.get(
        "SMOTE"
    )
    is False,
    binary_primary_policy.get(
        "SMOTE"
    )
)


add_check(
    "Validation_Natural_Prevalence",
    imbalance_policy[
        "validation_policy"
    ].get(
        "preserve_natural_prevalence"
    )
    is True,
    True
)


add_check(
    "TEST_Natural_Prevalence",
    imbalance_policy[
        "test_policy"
    ].get(
        "preserve_natural_prevalence"
    )
    is True,
    True
)


add_check(
    "Model_Ladder_Primary_9",
    int(
        model_ladder[
            "Primary_Benchmark"
        ].sum()
    )
    ==
    9,
    int(
        model_ladder[
            "Primary_Benchmark"
        ].sum()
    )
)


add_check(
    "Three_Final_Replicate_Seeds",
    len(
        FINAL_REPLICATE_SEEDS
    )
    ==
    3,
    FINAL_REPLICATE_SEEDS
)


add_check(
    "Headline_Metric_PR_AUC",
    evaluation_protocol[
        "binary_forecasting"
    ][
        "headline_metric"
    ]
    ==
    "Average Precision / PR-AUC",
    evaluation_protocol[
        "binary_forecasting"
    ][
        "headline_metric"
    ]
)


add_check(
    "Threshold_Selection_VALIDATION_Only",
    threshold_policy[
        "threshold_source"
    ]
    ==
    "VALIDATION_ONLY",
    threshold_policy[
        "threshold_source"
    ]
)


add_check(
    "TEST_Hyperparameter_Selection_Forbidden",
    test_access_policy[
        "test_predictions_allowed_during"
    ][
        "hyperparameter_search"
    ]
    is False,
    False
)


add_check(
    "TEST_Threshold_Selection_Forbidden",
    test_access_policy[
        "test_predictions_allowed_during"
    ][
        "threshold_selection"
    ]
    is False,
    False
)


add_check(
    "TEST_Model_Selection_Forbidden",
    test_access_policy[
        "test_predictions_allowed_during"
    ][
        "primary_model_selection"
    ]
    is False,
    False
)


add_check(
    "Episode_Level_Bootstrap",
    evaluation_protocol[
        "uncertainty"
    ][
        "bootstrap_unit"
    ]
    ==
    "Episode_ID",
    evaluation_protocol[
        "uncertainty"
    ][
        "bootstrap_unit"
    ]
)


add_check(
    "No_Model_Training_In_12_3",
    True,
    "protocol freeze only"
)


validation = pd.DataFrame(
    validation_rows
)


if not validation[
    "Passed"
].all():

    failed = validation.loc[
        ~validation[
            "Passed"
        ],
        [
            "Check",
            "Details"
        ]
    ]


    print(
        failed.to_string(
            index=False
        )
    )


    raise RuntimeError(
        "PHASE 12.3 HARD VALIDATION FAILED."
    )


save_csv(

    validation,

    STAGING
    / "12_3_VALIDATION.csv"
)


print(
    validation.to_string(
        index=False
    )
)


# ==================================================================================================
# 12.3X — FINAL DECISION
# ==================================================================================================

decision = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.3",

    "decision":
        "PASS_FINAL_PRE_MODEL_PROTOCOL_FREEZE",


    "dataset_frozen":
        True,

    "sequence_dataset_frozen":
        True,

    "preprocessing_frozen":
        True,

    "imbalance_policy_frozen":
        True,

    "experiment_protocol_frozen":
        True,


    "training_authorized":
        True,


    "training_authorization_scope": [

        "TRAIN",

        "VALIDATION",
    ],


    "test_evaluation_authorized":
        False,


    "primary_experiments": [

        "W10/H5",

        "W10/H10",
    ],


    "next_phase":
        "13.0_BASELINES_AND_CLASSICAL_MODELS",
}


write_json(

    decision,

    STAGING
    / "12_3_DECISION.json"
)


decision_sha256 = sha256_file(

    STAGING
    / "12_3_DECISION.json"
)


# ==================================================================================================
# 12.3Y — FINAL SOURCE MUTATION CHECK
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3Y — VERIFY SOURCE PHASES REMAIN UNCHANGED")
print("=" * 120)


post_failures_12_0 = verify_hash_table(

    PHASE_12_0_DIR,

    hashes_12_0
)


post_failures_12_1 = verify_hash_table(

    PHASE_12_1_DIR,

    hashes_12_1
)


post_failures_12_2 = verify_hash_table(

    PHASE_12_2_DIR,

    hashes_12_2
)


if post_failures_12_0:

    raise RuntimeError(
        "Phase 12.0 changed during Phase 12.3."
    )


if post_failures_12_1:

    raise RuntimeError(
        "Phase 12.1 changed during Phase 12.3."
    )


if post_failures_12_2:

    raise RuntimeError(
        "Phase 12.2 changed during Phase 12.3."
    )


print(
    "✓ Phase 12.0 unchanged"
)

print(
    "✓ Phase 12.1 unchanged"
)

print(
    "✓ Phase 12.2 unchanged"
)


# ==================================================================================================
# 12.3Z — HASH PROTOCOL ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3Z — HASH PROTOCOL ARTIFACTS")
print("=" * 120)


file_hashes = build_hash_table(
    STAGING
)


save_csv(

    file_hashes,

    STAGING
    / "12_3_FILE_HASHES.csv"
)


file_hash_table_sha256 = sha256_file(

    STAGING
    / "12_3_FILE_HASHES.csv"
)


print(
    "Protocol artifacts hashed:",
    len(
        file_hashes
    )
)


# ==================================================================================================
# 12.3AA — PROTOCOL LOCK
# ==================================================================================================

protocol_lock = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.3",

    "lock_status":
        "FINAL_PRE_MODEL_PROTOCOL_FROZEN",


    "source_phase_12_0_lock_sha256":
        lock_12_0_hash,

    "source_phase_12_1_lock_sha256":
        lock_12_1_hash,

    "source_phase_12_2_lock_sha256":
        lock_12_2_hash,


    "protocol_manifest_sha256":
        protocol_manifest_sha256,

    "decision_sha256":
        decision_sha256,

    "file_hash_table_sha256":
        file_hash_table_sha256,


    "primary_window_minutes":
        PRIMARY_WINDOW,

    "primary_horizons_minutes":
        PRIMARY_HORIZONS,


    "training_authorized":
        True,

    "test_evaluation_authorized":
        False,


    "rules_after_lock": {

        "change_primary_window_after_results":
            False,

        "change_primary_horizons_after_results":
            False,

        "change_primary_metric_after_results":
            False,

        "change_threshold_policy_after_results":
            False,

        "change_test_access_policy_after_results":
            False,

        "change_final_seed_set_after_results":
            False,

        "increase_hyperparameter_budget_after_seeing_results":
            False,

        "use_test_for_tuning":
            False,
    },


    "next_phase":
        "13.0_BASELINES_AND_CLASSICAL_MODELS",
}


write_json(

    protocol_lock,

    STAGING
    / "12_3_PROTOCOL_LOCK.json"
)


protocol_lock_sha256 = sha256_file(

    STAGING
    / "12_3_PROTOCOL_LOCK.json"
)


(
    STAGING
    / "12_3_PROTOCOL_LOCK_SHA256.txt"
).write_text(
    protocol_lock_sha256
    +
    "\n",
    encoding="utf-8"
)


# ==================================================================================================
# 12.3AB — READINESS FOR MODELING
# ==================================================================================================

readiness = {

    "dataset":
        "Synthetic_NAF_v1",

    "dataset_version":
        "1.0.0",

    "phase":
        "12.3",

    "phase_12_3_status":
        "PASS",

    "decision":
        "PASS_FINAL_PRE_MODEL_PROTOCOL_FREEZE",


    "phase_12_3_protocol_lock_sha256":
        protocol_lock_sha256,


    "training_authorized":
        True,


    "training_scope": [

        "TRAIN",

        "VALIDATION",
    ],


    "test_evaluation_authorized":
        False,


    "test_status":
        "EMBARGOED_UNTIL_FINAL_MODEL_REGISTRY_LOCK",


    "primary_window_minutes":
        10,

    "primary_horizons_minutes": [
        5,
        10
    ],


    "next_phase":
        "13.0_BASELINES_AND_CLASSICAL_MODELS",
}


write_json(

    readiness,

    STAGING
    / "12_3_READY_FOR_MODELING.json"
)


# ==================================================================================================
# 12.3AC — VERIFY STAGING OUTPUT
# ==================================================================================================

recorded_hashes = pd.read_csv(

    STAGING
    / "12_3_FILE_HASHES.csv"
)


output_failures = verify_hash_table(

    STAGING,

    recorded_hashes
)


if output_failures:

    raise RuntimeError(
        f"Phase 12.3 staging hash verification failed:\n"
        f"{output_failures[:10]}"
    )


print(
    "✓ protocol payload hashes verified"
)


# ==================================================================================================
# 12.3AD — FINALIZE
# ==================================================================================================

print("\n" + "=" * 120)
print("12.3AD — FINALIZE")
print("=" * 120)


if FINAL_OUT.exists():

    raise RuntimeError(
        "Final Phase 12.3 directory unexpectedly exists."
    )


shutil.move(

    str(
        STAGING
    ),

    str(
        FINAL_OUT
    )
)


FINAL_LOCK_FILE = (
    FINAL_OUT
    / "12_3_PROTOCOL_LOCK.json"
)

FINAL_LOCK_HASH_FILE = (
    FINAL_OUT
    / "12_3_PROTOCOL_LOCK_SHA256.txt"
)


if (

    sha256_file(
        FINAL_LOCK_FILE
    )

    !=

    FINAL_LOCK_HASH_FILE

    .read_text(
        encoding="utf-8"
    )

    .strip()

):

    raise RuntimeError(
        "Final Phase 12.3 protocol lock failed verification."
    )


print(
    "✓ Phase 12.3 finalized:"
)

print(
    FINAL_OUT
)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("✓ PHASE 12.3 — FINAL PRE-MODEL EXPERIMENTAL PROTOCOL FREEZE: PASS")
print("=" * 120)


print(
    "\nPRIMARY FORECASTING EXPERIMENT:"
)

print(
    "    W10 / H5"
)

print(
    "    W10 / H10"
)


print(
    "\nPRIMARY TRAINING POLICY:"
)

print(
    "    natural TRAIN distribution"
)

print(
    "    + frozen TRAIN-derived class weighting"
)


print(
    "\nPRIMARY W10/H5:"
)

print(
    "    positives = 3,500"
)

print(
    "    negatives = 55,777"
)

print(
    "    BCE pos_weight ≈ 15.936286"
)


print(
    "\nPRIMARY W10/H10:"
)

print(
    "    positives = 7,000"
)

print(
    "    negatives = 51,577"
)

print(
    "    BCE pos_weight ≈ 7.368143"
)


print(
    "\nMODEL LADDER:"
)

print(
    "    0. Prevalence baseline"
)

print(
    "    1. Current-minute Logistic Regression"
)

print(
    "    2. Flattened-history Logistic Regression"
)

print(
    "    3. Temporal-summary XGBoost"
)

print(
    "    4. GRU"
)

print(
    "    5. LSTM"
)

print(
    "    6. TCN"
)

print(
    "    7. 1D CNN"
)

print(
    "    8. Small Transformer"
)

print(
    "    9. InceptionTime — secondary / optional"
)


print(
    "\nMODEL SELECTION:"
)

print(
    "    headline = validation Average Precision / PR-AUC"
)

print(
    "    threshold = validation F1 optimum"
)

print(
    "    test never selects model/hyperparameters/threshold"
)


print(
    "\nFINAL STOCHASTIC REPLICATE SEEDS:"
)

print(
    "   ",
    FINAL_REPLICATE_SEEDS
)


print(
    "\nTEST STATUS:"
)

print(
    "    LOCKED / EMBARGOED"
)

print(
    "    NO TEST inference during development"
)

print(
    "    NO TEST threshold tuning"
)

print(
    "    NO TEST architecture selection"
)

print(
    "    TEST opens only after Phase 13.2 final model registry lock"
)


print(
    "\nTRAINING STATUS:"
)

print(
    "    TRAIN + VALIDATION MODEL DEVELOPMENT IS NOW AUTHORIZED ✓"
)

print(
    "    TEST EVALUATION IS NOT YET AUTHORIZED ✓"
)


print(
    "\nNEXT:"
)

print(
    "    PHASE 13.0"
)

print(
    "    BASELINES + CLASSICAL MODELS"
)

print(
    "    Prevalence"
)

print(
    "    Current-minute Logistic Regression"
)

print(
    "    Flattened-history Logistic Regression"
)

print(
    "    Temporal-summary XGBoost"
)


print("=" * 120)


gc.collect()